# Utah Wildland–Urban Interface Exposure and Wildfire Hazard Dashboard

**Geospatial data science portfolio project**  
**Study area:** Salt Lake, Utah, and Washington counties, Utah  
**Analytical CRS:** NAD83 / UTM zone 12N (EPSG:26912)  
**Core technologies:** Python, GeoPandas, Rasterio, Shapely, pandas, NumPy, Microsoft Planetary Computer, ArcGIS REST services, and Folium

This publication notebook contains the complete executed workflow and the rendered results used in the portfolio release. It integrates demographic exposure, critical infrastructure, satellite-derived vegetation conditions, fuels, terrain, historical fire occurrence, and human ignition potential into an interactive decision-support dashboard.


## 1. Project Objective

Develop a reproducible geospatial workflow that identifies where mapped high-risk Wildland–Urban Interface (WUI), human exposure, critical infrastructure, and elevated relative wildfire hazard occur within the same landscape. The project is designed as a regional screening and prioritization tool for planners, emergency managers, fire agencies, and risk analysts.

### Headline results

- **111 of 1,240** Census Block Groups contain measurable WUI overlap.
- The exposed portions of those block groups represent an estimated **19,192 residents** and **6,933 housing units**.
- The analysis measures **407.24 km²** of WUI overlap.
- **27 critical facilities** fall inside the mapped WUI and **178 additional facilities** are outside the WUI but within one mile.
- The 30-meter composite grid passed its final alignment, range, masking, and component validation checks.


## 2. Research Questions

1. Which Census Block Groups intersect mapped high-risk WUI areas in the three-county study area?
2. How much population, housing, and land area is potentially exposed within those intersections?
3. Which critical facilities occur inside or near the WUI?
4. How can vegetation dryness, fuels, terrain, historical fire occurrence, and human ignition potential be combined into a consistent relative-hazard surface?
5. Where do elevated relative hazard and human exposure coincide, and how can those results be communicated through an interactive dashboard?


## 3. Study Area

The analysis covers **Salt Lake County (FIPS 035), Utah County (049), and Washington County (053)**. These counties represent distinct Utah landscapes and development patterns, including Wasatch Front foothills, mountain communities, and rapidly growing desert WUI areas. Vector and raster analysis uses EPSG:26912; dashboard layers are converted to EPSG:4326 for web mapping.


## 4. Data Sources

| Analytical role | Dataset and vintage | Provider |
|---|---|---|
| Census geography | 2020 Census Block Groups | U.S. Census Bureau / ArcGIS Feature Service |
| Demographics | 2024 ACS 5-Year population and housing estimates | U.S. Census Bureau API |
| Administrative boundaries | Utah county boundaries | Utah GIS / ArcGIS Feature Service |
| WUI exposure | Utah High-Risk WUI polygons | Utah GIS / ArcGIS Feature Service |
| Critical infrastructure | Hospitals, fire stations, law enforcement, schools, and emergency shelters | Utah GIS feature services |
| Vegetation condition | HLS L30/S30 Version 2.0, July–September 2025 | NASA / Microsoft Planetary Computer |
| Surface and canopy fuels | LF2025 FBFM40, canopy cover, and canopy bulk density | LANDFIRE |
| Terrain | USGS 3DEP 1 arc-second DEM | U.S. Geological Survey |
| Historical fire | InFORM FODR and MTBS burned-area boundaries | NIFC, USFS, and USGS |
| Human ignition indicators | 2025 TIGER/Line roads and Annual NLCD 2025 land cover | Census Bureau and MRLC |

Source URLs, versions, access methods, expected formats, and licensing notes are recorded programmatically in the source catalog below.


## 5. Environment and Reproducibility

The verified run used **Python 3.14.7**, **NumPy 2.4.6**, and **Rasterio 1.5.1**, with NumPy constrained below 2.5. The accompanying `environment.yml` records the principal package versions reported by the executed notebook.

Publication defaults use `census_geography_data_mode = 'snapshot'`, `acs_data_mode = 'snapshot'`, and `fire_hazard_data_mode = 'snapshot'` so the saved results remain tied to validated local inputs and derived products. In `snapshot` mode, each workflow loads its validated local snapshot and does not contact its upstream source. The corresponding `refresh` mode intentionally reacquires that source, validates the result, and replaces the local snapshot; refreshed inputs may change the analytical results.


## Notebook Roadmap

The executable workflow is ordered by data dependency while covering the full portfolio narrative:

| Portfolio section | Notebook implementation |
|---|---|
| 1–5. Objective, questions, study area, sources, environment | Opening sections and project setup |
| 6. Data acquisition and preparation | WEA Phases 1–2 and CHRG acquisition stages |
| 7. WUI processing | WEA Phases 1 and 3 |
| 8. Composite fire-hazard modeling | CHRG Phases 6–12 |
| 9. Exposure analysis | WEA Phase 3 |
| 10. Population, housing, and facility summaries | WEA Phases 3–4 |
| 11. Validation and quality control | Embedded assertions and QA summaries throughout |
| 12. Interactive dashboard | WEA Phase 5 and CHRG Phase 13 |
| 13–15. Findings, limitations, and next steps | Closing sections |


# WEA Phase 1 – Data Ingestion

## Purpose

Configure the project environment and acquire, validate, clip, repair, and align the Census Block Group, county-boundary, and Utah High-Risk WUI datasets.


## Project Setup


In [1]:
# ----------------------------------------------------
# PROJECT SETUP
# ----------------------------------------------------
# Import the libraries, verify the Python 3.14 runtime, and configure the
# notebook display before running the Phase 1 spatial data-ingestion workflow.
print('\n=== PHASE 1: CORE SPATIAL DATA INGESTION ===')

# Import the libraries required for spatial data ingestion, validation, and processing.
import geopandas as gpd     # Work with geospatial vector data
import pandas as pd         # Work with tabular data
import numpy as np          # Perform numerical calculations
import requests             # Send REST API requests
import urllib.parse         # Encode ArcGIS REST query parameters
import folium               # Create interactive web maps
import os                   # Work with environment variables
import getpass              # Enter credentials without displaying them
import json                 # Read and write JSON-formatted data
import time                 # Support retry delays and timed operations
import hashlib              # Create file-integrity hash values
import platform             # Inspect the active Python runtime
from datetime import datetime, timezone   # Create timezone-aware UTC timestamps
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path    # Build portable file-system paths
import re                   # Apply regular-expression text replacements
from rasterio.enums import Resampling     # Define raster resampling methods
from IPython.display import HTML, display # Display notebook HTML and tables

# ----------------------------------------------------
# VERIFY PYTHON 3.14 RUNTIME
# ----------------------------------------------------

# Parse the active runtime without relying on deprecated version-string parsing.
active_python_version = tuple(
    int(version_part)
    for version_part
    in platform.python_version_tuple()
)

# Require Python 3.14 while allowing later Python 3.14 maintenance releases.
if active_python_version[:2] != (3, 14):
    raise RuntimeError(
        'This portfolio notebook requires Python 3.14.x. '
        f'Active runtime: {platform.python_version()}'
    )

# Map import names to distribution names for reproducible version reporting.
runtime_distribution_names = {
    'GeoPandas': 'geopandas',
    'pandas': 'pandas',
    'NumPy': 'numpy',
    'Shapely': 'shapely',
    'Rasterio': 'rasterio',
    'PyProj': 'pyproj',
    'Folium': 'folium',
    'Branca': 'branca',
    'Requests': 'requests',
    'SciPy': 'scipy',
}

# Collect installed versions without importing package-private version attributes.
runtime_version_records = [
    {
        'COMPONENT': 'Python',
        'VERSION': platform.python_version(),
    }
]

for component_name, distribution_name in runtime_distribution_names.items():
    try:
        installed_version = version(distribution_name)
    except PackageNotFoundError:
        installed_version = 'Not installed'

    runtime_version_records.append(
        {
            'COMPONENT': component_name,
            'VERSION': installed_version,
        }
    )

runtime_version_summary = pd.DataFrame(runtime_version_records)

print(f'-> Active Python runtime: {platform.python_version()}')
print('-> Required Python runtime: current stable Python 3.14.x release')
print('-> Compatibility constraint: NumPy <2.5 for the verified Rasterio workflow')
print('\n--- RUNTIME VERSION SUMMARY ---')
display(runtime_version_summary)

# Expand the notebook display area so wide code and spatial outputs are easier to review.
HTML('\n<style>\n.container {\n    width: 95% !important;\n}\n</style>\n')



=== PHASE 1: CORE SPATIAL DATA INGESTION ===
-> Active Python runtime: 3.14.7
-> Required Python runtime: current stable Python 3.14.x release
-> Compatibility constraint: NumPy <2.5 for the verified Rasterio workflow

--- RUNTIME VERSION SUMMARY ---


,COMPONENT,VERSION
0,Python,3.14.7
1,GeoPandas,1.1.4
2,pandas,3.0.5
3,NumPy,2.4.6
4,Shapely,2.1.2
5,Rasterio,1.5.1
6,PyProj,3.8.0
7,Folium,0.20.0
8,Branca,0.8.2
9,Requests,2.34.2


## Census Block Group Ingestion


In [2]:
# ----------------------------------------------------
# CENSUS BLOCK GROUP INGESTION
# ----------------------------------------------------
# Load the validated 2020 Census Block Group snapshot used for publication,
# or intentionally refresh the snapshot from the upstream ArcGIS service.
# Both paths apply the same schema, county-coverage, geometry, and identifier
# validation before returning the GeoDataFrame.

census_geography_data_mode = 'snapshot'

valid_census_geography_data_modes = [
    'snapshot',
    'refresh',
]

if census_geography_data_mode not in valid_census_geography_data_modes:
    raise ValueError(
        "census_geography_data_mode must be either "
        "'snapshot' or 'refresh'."
    )

census_geography_snapshot_directory = Path(
    'data/processed'
)

census_geography_snapshot_path = (
    census_geography_snapshot_directory
    / 'utah_three_county_2020_census_block_groups.gpkg'
)

print(
    f'-> Census-geography data mode: '
    f'{census_geography_data_mode}'
)
print(
    f'-> Census-geography snapshot: '
    f'{census_geography_snapshot_path}'
)

def load_census_block_groups():
    """
    Load or refresh the validated 2020 Census Block Group snapshot for
    the three-county Utah WUI study area.

    Returns
    -------
    geopandas.GeoDataFrame
        Census Block Groups in the service-response CRS (EPSG:4326).
    """

    # Define the exact county identifiers required by the study design.
    expected_census_county_fips = {
        '035',
        '049',
        '053',
    }

    # Define the fields required by later joins, validation, and mapping.
    required_census_geography_fields = {
        'GEOID20',
        'COUNTYFP20',
        'geometry',
    }

    # Load the publication snapshot without contacting the upstream service.
    if census_geography_data_mode == 'snapshot':
        if not census_geography_snapshot_path.exists():
            raise FileNotFoundError(
                'The validated Census geography snapshot is missing:\n'
                f'{census_geography_snapshot_path}\n\n'
                'Snapshot mode does not contact the upstream service. '
                "Set census_geography_data_mode = 'refresh', rerun this "
                'cell and the Phase 1 execution cell once to create the '
                'validated snapshot, then restore snapshot mode.'
            )

        print(
            'Loading validated Census Block Group snapshot...'
        )

        gdf_census = gpd.read_file(
            census_geography_snapshot_path,
            layer='census_block_groups',
        )

        census_geography_source_action = (
            'Loaded validated local snapshot'
        )

    # Refresh mode intentionally reacquires the three county subsets and
    # writes a validated replacement snapshot only after all checks pass.
    else:
        # Define the ArcGIS FeatureServer endpoint used to retrieve 2020
        # Census Block Group geometry and attributes.
        census_base = (
            "https://services1.arcgis.com/"
            "99lidPhWCzftIe9K/arcgis/rest/services/"
            "CensusBlockGroups2020/FeatureServer/0/query"
        )

        # Store the independently requested county responses before combining
        # them into the complete study-area geography.
        census_county_frames = []

        # Use smaller county-specific requests to reduce FeatureServer load
        # and avoid the large three-county request that can time out.
        for county_fips in sorted(expected_census_county_fips):
            census_params = {
                'where': f"COUNTYFP20 = '{county_fips}'",
                'outFields': '*',
                'returnGeometry': 'true',
                'outSR': '4326',
                'f': 'geojson',
            }

            print(
                f'Streaming Census Block Groups for FIPS '
                f'{county_fips}...'
            )

            census_response = None
            census_request_error = None

            # Retry transient upstream failures with exponential backoff.
            for request_attempt in range(1, 5):
                try:
                    census_response = requests.get(
                        census_base,
                        params=census_params,
                        timeout=120,
                    )

                    census_response.raise_for_status()
                    census_request_error = None
                    break

                except requests.RequestException as error:
                    census_request_error = error

                    if request_attempt == 4:
                        break

                    retry_delay_seconds = 2 ** request_attempt

                    print(
                        f'-> Attempt {request_attempt} failed for FIPS '
                        f'{county_fips}; retrying in '
                        f'{retry_delay_seconds} seconds.'
                    )

                    time.sleep(retry_delay_seconds)

            if census_request_error is not None:
                raise RuntimeError(
                    f'The Census FeatureServer request for county FIPS '
                    f'{county_fips} failed after four attempts.'
                ) from census_request_error

            # Parse the response before creating the county GeoDataFrame.
            try:
                census_geojson = census_response.json()

            except ValueError as error:
                raise ValueError(
                    'The Census FeatureServer did not return valid '
                    f'GeoJSON for county FIPS {county_fips}.\n\n'
                    f'Response status: {census_response.status_code}\n'
                    f'Content type: '
                    f"{census_response.headers.get('Content-Type')}\n"
                    f'Response preview:\n'
                    f'{census_response.text[:500]}'
                ) from error

            if 'error' in census_geojson:
                raise RuntimeError(
                    'The Census FeatureServer returned an ArcGIS service '
                    f'error for county FIPS {county_fips}:\n'
                    f"{census_geojson['error']}"
                )

            if (
                census_geojson.get('type') != 'FeatureCollection'
                or not isinstance(census_geojson.get('features'), list)
            ):
                raise ValueError(
                    'The Census FeatureServer did not return a valid '
                    f'GeoJSON FeatureCollection for county FIPS '
                    f'{county_fips}.'
                )

            if not census_geojson['features']:
                raise ValueError(
                    f'The Census query returned zero block groups for '
                    f'county FIPS {county_fips}.'
                )

            county_census_gdf = gpd.GeoDataFrame.from_features(
                census_geojson['features'],
                crs='EPSG:4326',
            )

            census_county_frames.append(
                county_census_gdf
            )

            print(
                f'-> Loaded {len(county_census_gdf):,} block groups '
                f'for FIPS {county_fips}.'
            )

        gdf_census = gpd.GeoDataFrame(
            pd.concat(
                census_county_frames,
                ignore_index=True,
            ),
            geometry='geometry',
            crs='EPSG:4326',
        )

        census_geography_source_action = (
            'Refreshed upstream source and replaced local snapshot'
        )

    # Apply identical validation to snapshot and refresh results.
    missing_census_geography_fields = (
        required_census_geography_fields
        - set(gdf_census.columns)
    )

    if missing_census_geography_fields:
        raise ValueError(
            'The Census geography is missing required fields: '
            f'{sorted(missing_census_geography_fields)}'
        )

    if gdf_census.empty:
        raise ValueError(
            'The Census geography contains zero block groups.'
        )

    if gdf_census.crs is None:
        raise ValueError(
            'The Census geography does not define a CRS.'
        )

    census_returned_county_fips = set(
        gdf_census['COUNTYFP20']
        .astype(str)
        .str.zfill(3)
    )

    if census_returned_county_fips != expected_census_county_fips:
        raise ValueError(
            'The Census geography does not contain exactly the required '
            'county FIPS codes.\n'
            f'Expected: {sorted(expected_census_county_fips)}\n'
            f'Returned: {sorted(census_returned_county_fips)}'
        )

    duplicate_census_geoid_count = int(
        gdf_census['GEOID20'].astype(str).duplicated().sum()
    )

    if duplicate_census_geoid_count != 0:
        raise ValueError(
            f'The Census geography contains '
            f'{duplicate_census_geoid_count:,} duplicate GEOID20 values.'
        )

    invalid_census_geometry_count = int(
        (
            gdf_census.geometry.isna()
            | gdf_census.geometry.is_empty
        ).sum()
    )

    if invalid_census_geometry_count != 0:
        raise ValueError(
            f'The Census geography contains '
            f'{invalid_census_geometry_count:,} missing or empty geometries.'
        )

    # Write a refreshed result only after it passes every validation check.
    if census_geography_data_mode == 'refresh':
        census_geography_snapshot_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        temporary_census_geography_snapshot_path = (
            census_geography_snapshot_path.with_name(
                f'{census_geography_snapshot_path.stem}.tmp.gpkg'
            )
        )

        temporary_census_geography_snapshot_path.unlink(
            missing_ok=True
        )

        gdf_census.to_file(
            temporary_census_geography_snapshot_path,
            layer='census_block_groups',
            driver='GPKG',
            index=False,
        )

        os.replace(
            temporary_census_geography_snapshot_path,
            census_geography_snapshot_path,
        )

        print(
            f'-> Validated Census geography snapshot written: '
            f'{census_geography_snapshot_path}'
        )

    print(
        f"-> Successfully loaded "
        f"{len(gdf_census):,} Census Block Groups."
    )
    print(
        f"-> Native Census projection: "
        f"{gdf_census.crs}"
    )
    print(
        f'-> Census-geography source action: '
        f'{census_geography_source_action}'
    )

    return gdf_census


-> Census-geography data mode: snapshot
-> Census-geography snapshot: data\processed\utah_three_county_2020_census_block_groups.gpkg


## County-Boundary Ingestion


In [3]:
# ----------------------------------------------------
# COUNTY-BOUNDARY INGESTION
# ----------------------------------------------------
# Retrieve the three county boundaries that define the authoritative study
# area used for later spatial filtering and clipping.
def load_county_boundaries():
    """
    Retrieve Salt Lake, Utah, and Washington County boundaries.

    Returns
    -------
    geopandas.GeoDataFrame
        County boundaries in the source-response CRS.
    """

    # Define the ArcGIS FeatureServer endpoint used to retrieve Utah county boundaries.
    county_boundaries_base = (
        "https://services1.arcgis.com/"
        "99lidPhWCzftIe9K/ArcGIS/rest/services/"
        "UtahCountyBoundaries/FeatureServer/0/query"
    )

    # Restrict the county query to Salt Lake, Utah, and Washington Counties.
    county_where_clause = (
        "NAME IN ('SALT LAKE', 'UTAH', 'WASHINGTON')"
    )

    # Request all county attributes and polygon geometry as GeoJSON in EPSG:4326.
    county_boundaries_params = {
        "where": county_where_clause,
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "geojson"
    }

    # Encode the county query parameters into a FeatureServer URL that GeoPandas can read directly.
    county_boundaries_url = (
        f"{county_boundaries_base}?"
        f"{urllib.parse.urlencode(county_boundaries_params)}"
    )

    print(
        "\nStreaming Salt Lake, Utah, and Washington "
        "County boundaries..."
    )

    # Load the returned county polygons into a GeoDataFrame for study-area construction.
    gdf_county_boundaries = gpd.read_file(
        county_boundaries_url
    )

    print(
        f"-> Successfully loaded "
        f"{len(gdf_county_boundaries):,} county boundaries."
    )
    print(
        f"-> Native county-boundary projection: "
        f"{gdf_county_boundaries.crs}"
    )

    return gdf_county_boundaries


## Statewide High-Risk WUI Ingestion

In [4]:
# ----------------------------------------------------
# STATEWIDE HIGH-RISK WUI INGESTION
# ----------------------------------------------------
# Retrieve the complete statewide High-Risk WUI layer using paginated
# ArcGIS FeatureServer requests and verify that all reported records are
# downloaded.

def load_statewide_wui():
    """
    Download the complete Utah High-Risk WUI layer using paginated
    ArcGIS FeatureServer requests.

    Returns
    -------
    geopandas.GeoDataFrame
        Complete statewide High-Risk WUI polygon layer.
    """

    # Define the ArcGIS FeatureServer endpoint for the statewide
    # High-Risk WUI polygon layer.
    wui_base = (
        "https://services.arcgis.com/"
        "ZzrwjTRez6FJiOq4/arcgis/rest/services/"
        "Utah_High_Risk_WUI_Properties/"
        "FeatureServer/1/query"
    )

    print(
        "\nStreaming Utah High-Risk WUI boundaries..."
    )

    # Request the total WUI record count first so the complete layer
    # can be downloaded with controlled pagination.
    count_response = requests.get(
        wui_base,
        params={
            "where": "1=1",
            "returnCountOnly": "true",
            "f": "json"
        },
        timeout=60
    )

    # Stop the workflow if the WUI count request fails at the HTTP level.
    count_response.raise_for_status()

    # Parse the WUI count response.
    count_response_json = count_response.json()

    # Check for an ArcGIS service-level error before using the count.
    if "error" in count_response_json:
        raise RuntimeError(
            "The WUI FeatureServer returned an "
            "ArcGIS service error:\n"
            f"{count_response_json['error']}"
        )

    # Store the total number of records reported by the service.
    total_wui_records = int(
        count_response_json["count"]
    )

    print(
        f"-> FeatureServer reports "
        f"{total_wui_records:,} total WUI records."
    )

    # Use 2,000-record pages because the service has a
    # maximum record count of 2,000.
    page_size = 2000

    # Store each downloaded page before combining them.
    wui_pages = []

    # Iterate through the statewide records using result offsets.
    for offset in range(
        0,
        total_wui_records,
        page_size
    ):

        # Define the parameters for the current FeatureServer page.
        wui_params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "orderByFields": "FID",
            "f": "geojson"
        }

        # Calculate the record range for progress reporting.
        first_record = offset + 1

        last_record = min(
            offset + page_size,
            total_wui_records
        )

        print(
            f"-> Downloading WUI records "
            f"{first_record:,} through "
            f"{last_record:,}..."
        )

        # Download the current GeoJSON page with requests instead of
        # asking Fiona/GDAL to stream the remote URL directly.
        page_response = requests.get(
            wui_base,
            params=wui_params,
            timeout=120
        )

        # Stop if the HTTP request fails.
        page_response.raise_for_status()

        # Parse the response as JSON so ArcGIS service errors can be
        # detected before GeoPandas attempts to read the features.
        page_json = page_response.json()

        # Stop if ArcGIS returned a service-level error.
        if "error" in page_json:
            raise RuntimeError(
                "The WUI FeatureServer returned an "
                "ArcGIS service error while downloading "
                f"records {first_record:,} through "
                f"{last_record:,}:\n"
                f"{page_json['error']}"
            )

        # Confirm that a GeoJSON feature collection was returned.
        if page_json.get("type") != "FeatureCollection":
            raise ValueError(
                "The WUI service did not return a valid "
                "GeoJSON FeatureCollection."
            )

        # Convert the returned GeoJSON features directly into a
        # GeoDataFrame without requiring Fiona to open a remote URL.
        page_gdf = gpd.GeoDataFrame.from_features(
            page_json["features"],
            crs="EPSG:4326"
        )

        # Store the page so all results can be combined afterward.
        wui_pages.append(
            page_gdf
        )

    # Stop if the pagination loop returned no data.
    if not wui_pages:
        raise ValueError(
            "No WUI download pages were returned."
        )

    # Combine all downloaded WUI pages into one GeoDataFrame.
    gdf_wui = gpd.GeoDataFrame(
        pd.concat(
            wui_pages,
            ignore_index=True
        ),
        geometry="geometry",
        crs=wui_pages[0].crs
    )

    print(
        f"-> Successfully loaded "
        f"{len(gdf_wui):,} WUI hazard polygons."
    )

    print(
        f"-> Native WUI projection: "
        f"{gdf_wui.crs}"
    )

    # Verify that pagination retrieved exactly the number of records
    # reported by the FeatureServer.
    if len(gdf_wui) != total_wui_records:
        raise ValueError(
            f"Expected {total_wui_records:,} WUI records, "
            f"but downloaded {len(gdf_wui):,}."
        )

    print(
        "-> All WUI records successfully downloaded."
    )

    return gdf_wui


## CRS Alignment and Study-Area Validation


In [5]:
# ----------------------------------------------------
# CRS ALIGNMENT AND STUDY-AREA VALIDATION
# ----------------------------------------------------
# Reproject all source layers to NAD83 / UTM Zone 12N and verify that the
# county layer contains exactly the three intended study counties.
def align_and_validate_study_layers(
    gdf_census,
    gdf_wui,
    gdf_county_boundaries
):
    """
    Reproject the source layers and confirm that the county query returned
    exactly the intended three-county study area.

    Returns
    -------
    tuple
        Reprojected Census, WUI, and county-boundary GeoDataFrames.
    """

    print(
        "\nReprojecting layers to EPSG:26912..."
    )

    # Reproject all source layers to NAD83 / UTM Zone 12N so spatial overlays and area measurements
    # use a common meter-based CRS.
    gdf_census = gdf_census.to_crs(
        "EPSG:26912"
    )
    # Derive wui with to_crs for use in the next processing or validation step.
    gdf_wui = gdf_wui.to_crs(
        "EPSG:26912"
    )
    # Calculate county boundaries for validation and workflow QA.
    gdf_county_boundaries = gdf_county_boundaries.to_crs(
        "EPSG:26912"
    )

    print(
        "-> Layers successfully reprojected "
        "to EPSG:26912."
    )

    # Define the exact county-name set expected from the study-area boundary query.
    expected_counties = {
        "SALT LAKE",
        "UTAH",
        "WASHINGTON"
    }

    # Standardize returned county names before validation to avoid false mismatches from case or
    # whitespace differences.
    returned_counties = set(
        gdf_county_boundaries["NAME"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    print(
        "\nCounties returned:"
    )
    print(
        sorted(returned_counties)
    )

    # Validate that the boundary query returned exactly the three intended study counties.
    if returned_counties != expected_counties:
        raise ValueError(
            "The county query did not return exactly "
            "Salt Lake, Utah, and Washington Counties."
        )

    return (
        gdf_census,
        gdf_wui,
        gdf_county_boundaries
    )


## WUI Geometry Preparation and Clipping


In [6]:
# ----------------------------------------------------
# WUI GEOMETRY PREPARATION AND CLIPPING
# ----------------------------------------------------
# Prepare the statewide WUI geometry for analysis by dissolving the study
# boundary, repairing invalid geometry, filtering intersections, clipping,
# and validating the result.
def prepare_wui_for_study_area(
    gdf_wui,
    gdf_county_boundaries
):
    """
    Repair, filter, and clip statewide WUI polygons to the three-county
    project study area.

    Returns
    -------
    geopandas.GeoDataFrame
        WUI polygons retained within the dissolved study-area boundary.
    """

    # Dissolve the three county polygons with the current GeoPandas union API.
    study_area_geometry = (
        gdf_county_boundaries.geometry.union_all()
    )

    # Repair the dissolved boundary if the union operation creates invalid polygon topology.
    if not study_area_geometry.is_valid:
        # Derive study area geometry used to define or validate the spatial analysis area.
        study_area_geometry = (
            study_area_geometry.buffer(0)
        )

    # Store the original statewide WUI count so later QA output can document how many records were
    # removed.
    statewide_wui_count = len(
        gdf_wui
    )

    # Explode multipart WUI geometry so each polygon part can be validated, filtered, and clipped
    # independently.
    gdf_wui = gdf_wui.explode(
        index_parts=False,
        ignore_index=True
    )

    # Calculate exploded wui count for validation and workflow QA.
    exploded_wui_count = len(
        gdf_wui
    )

    # Repair invalid WUI geometry before intersection and clipping to reduce topology-related
    # spatial errors.
    if not gdf_wui.geometry.is_valid.all():
        # Branch on hasattr so the workflow follows the appropriate processing path.
        if hasattr(
            gdf_wui.geometry,
            "make_valid"
        ):
            # Derive wui with make_valid for use in the next processing or validation step.
            gdf_wui["geometry"] = (
                gdf_wui.geometry.make_valid()
            )
        else:
            # Derive wui with buffer for use in the next processing or validation step.
            gdf_wui["geometry"] = (
                gdf_wui.geometry.buffer(0)
            )

    # Spatially filter the statewide WUI layer so only polygon parts intersecting the three-county
    # study area are retained.
    gdf_wui = gdf_wui[
        gdf_wui.geometry.intersects(
            study_area_geometry
        )
    ].copy()

    # Calculate intersecting wui count for validation and workflow QA.
    intersecting_wui_count = len(
        gdf_wui
    )

    # Clip each retained WUI geometry to the dissolved study-area boundary so no hazard area extends
    # beyond the project extent.
    gdf_wui["geometry"] = (
        gdf_wui.geometry.intersection(
            study_area_geometry
        )
    )

    # Remove null or empty geometries created when features only touch the study-area boundary.
    gdf_wui = gdf_wui[
        gdf_wui.geometry.notna()
        & ~gdf_wui.geometry.is_empty
    ].copy()

    # Retain only polygonal geometry because later exposure calculations depend on valid area
    # features.
    gdf_wui = gdf_wui[
        gdf_wui.geometry.geom_type.isin(
            [
                "Polygon",
                "MultiPolygon"
            ]
        )
    ].copy().reset_index(
        drop=True
    )

    # Measure any residual WUI geometry outside the study boundary as a post-clipping QA check.
    outside_geometry = (
        gdf_wui.geometry.difference(
            study_area_geometry
        )
    )

    # Derive outside area m2 with float for use in the next processing or validation step.
    outside_area_m2 = float(
        outside_geometry.area.sum()
    )

    # Allow a 1 m² tolerance for minor numerical artifacts produced by polygon overlay operations.
    area_tolerance_m2 = 1.0

    # Calculate outside feature count for validation and workflow QA.
    outside_feature_count = int(
        (
            outside_geometry.area
            > area_tolerance_m2
        ).sum()
    )

    # Report feature counts from each major geometry-processing stage to document how the statewide
    # layer was reduced.
    print(
        f"-> WUI records initially downloaded: "
        f"{statewide_wui_count:,}"
    )
    print(
        f"-> Polygon parts after exploding multipart "
        f"features: {exploded_wui_count:,}"
    )
    print(
        f"-> Polygon parts intersecting study area: "
        f"{intersecting_wui_count:,}"
    )
    print(
        f"-> WUI polygon parts retained after clipping: "
        f"{len(gdf_wui):,}"
    )
    print(
        f"-> WUI geometries with more than "
        f"{area_tolerance_m2} m² outside the study area: "
        f"{outside_feature_count:,}"
    )
    print(
        f"-> Total WUI area outside study area: "
        f"{outside_area_m2:.6f} m²"
    )

    # Fail validation if residual WUI area exceeds the tolerance, indicating that clipping did not
    # fully constrain the dataset.
    if outside_area_m2 > area_tolerance_m2:
        raise ValueError(
            f"{outside_area_m2:.6f} m² of WUI geometry "
            "remains outside the study-area boundary "
            "after clipping."
        )

    return gdf_wui


## Execute Phase 1 Data-Ingestion Workflow


In [7]:
# ----------------------------------------------------
# EXECUTE PHASE 1 DATA-INGESTION WORKFLOW
# ----------------------------------------------------
# Run the source-ingestion, CRS-alignment, validation, and WUI clipping
# functions in sequence to create the aligned Phase 1 output
# GeoDataFrames.
print(
    "=== STARTING DATA INGESTION PIPELINE ==="
)

# Derive census with load_census_block_groups for use in the next processing or validation step.
gdf_census = load_census_block_groups()
# Calculate county boundaries for validation and workflow QA.
gdf_county_boundaries = load_county_boundaries()
# Derive wui with load_statewide_wui for use in the next processing or validation step.
gdf_wui = load_statewide_wui()

# Align the source layers to the project CRS and validate the county study-area definition.
(
    gdf_census_aligned,
    gdf_wui_aligned,
    gdf_county_boundaries_aligned
) = align_and_validate_study_layers(
    gdf_census,
    gdf_wui,
    gdf_county_boundaries
)

# Repair, filter, and clip the statewide WUI layer to the validated three-county study area.
gdf_wui_aligned = prepare_wui_for_study_area(
    gdf_wui_aligned,
    gdf_county_boundaries_aligned
)

print(
    "\n=== PIPELINE INITIALIZATION COMPLETE ==="
)


=== STARTING DATA INGESTION PIPELINE ===
Loading validated Census Block Group snapshot...
-> Successfully loaded 1,240 Census Block Groups.
-> Native Census projection: EPSG:4326
-> Census-geography source action: Loaded validated local snapshot

Streaming Salt Lake, Utah, and Washington County boundaries...
-> Successfully loaded 3 county boundaries.
-> Native county-boundary projection: EPSG:4326

Streaming Utah High-Risk WUI boundaries...
-> FeatureServer reports 2,002 total WUI records.
-> Downloading WUI records 1 through 2,000...
-> Downloading WUI records 2,001 through 2,002...
-> Successfully loaded 2,002 WUI hazard polygons.
-> Native WUI projection: EPSG:4326
-> All WUI records successfully downloaded.

Reprojecting layers to EPSG:26912...
-> Layers successfully reprojected to EPSG:26912.

Counties returned:
['SALT LAKE', 'UTAH', 'WASHINGTON']
-> WUI records initially downloaded: 2,002
-> Polygon parts after exploding multipart features: 2,002
-> Polygon parts intersecting st

# WEA Phase 2 – ACS Integration

## Purpose

Download, clean, validate, and join ACS 5-year population and housing estimates to the aligned Census Block Group layer.


## Inspect Census and ACS Inputs


In [8]:
# ----------------------------------------------------
# INSPECT FINAL ACS-ENHANCED CENSUS DATASET
# ----------------------------------------------------
# Inspect the final joined dataset and confirm its structure before
# passing the results to the next sequential notebook.
# ACS configuration, download, validation, and final table.
# Retrieve ACS population and housing estimates for the three-county study area and prepare
# them for the Census Block Group join.
print('\n=== PHASE 2: ACS 5-YEAR DEMOGRAPHIC INTEGRATION ===')
print('=== CONFIGURING ACS RELEASE AND SNAPSHOT LOCATION ===')
from pathlib import Path

# Select the ACS 5-Year release used for population and housing-unit estimates.
# Use one ACS 5-Year release throughout the project so all demographic estimates refer to the
# same reference period.
acs_year = 2024

# Build the Census API endpoint dynamically from the configured release year.
acs_base_url = f'https://api.census.gov/data/{acs_year}/acs/acs5'

# Store a processed snapshot locally so later runs can reproduce the analysis without
# repeatedly querying the Census API.
acs_snapshot_directory = Path('data/processed')

# Create the processed-data directory when it does not already exist.
acs_snapshot_directory.mkdir(parents=True, exist_ok=True)

# Include the ACS release year in the filename so cached tables from different releases cannot
# be confused.
acs_snapshot_filename = f'utah_three_county_acs5_{acs_year}_block_groups_processed.csv'
# Build acs snapshot path used to read, cache, or save this workflow product.
acs_snapshot_path = acs_snapshot_directory / acs_snapshot_filename
# Derive acs snapshot exists with exists for use in the next processing or validation step.
acs_snapshot_exists = acs_snapshot_path.exists()
print(f'-> ACS release year: {acs_year}')
print(f'-> Processed snapshot exists: {acs_snapshot_exists}')
print(f'-> Snapshot path: {acs_snapshot_path.resolve()}')
print('\nNOTE:')
print("The dedicated ACS Data Source "
    "cell that follows selects "
    "either 'snapshot' or 'refresh' "
    "mode.")

print('\n=== ACS RELEASE AND SNAPSHOT CONFIGURATION COMPLETE ===')



=== PHASE 2: ACS 5-YEAR DEMOGRAPHIC INTEGRATION ===
=== CONFIGURING ACS RELEASE AND SNAPSHOT LOCATION ===
-> ACS release year: 2024
-> Processed snapshot exists: True
-> Snapshot path: C:\Users\adamd\Projects\WUI\data\processed\utah_three_county_acs5_2024_block_groups_processed.csv

NOTE:
The dedicated ACS Data Source cell that follows selects either 'snapshot' or 'refresh' mode.

=== ACS RELEASE AND SNAPSHOT CONFIGURATION COMPLETE ===


## Define Live ACS Download and Processing Function


In [9]:
# ----------------------------------------------------
# DEFINE LIVE ACS DOWNLOAD AND PROCESSING FUNCTION
# ----------------------------------------------------
# Define the reusable Census API workflow used to retrieve, clean,
# standardize, and validate Block Group-level demographic estimates.
print('=== DEFINING LIVE ACS DOWNLOAD AND PROCESSING FUNCTION ===')

# Encapsulate the live ACS acquisition, cleaning, and validation steps in one reusable
# function.
def download_and_prepare_acs_data(acs_year, acs_base_url, census_api_key):
    """
    Download, prepare, and validate ACS Block Group data

    for Salt Lake, Utah, and Washington counties.

    The function:
      1. downloads county-level ACS Block Group records;
      2. combines the three county tables;
      3. creates a 12-digit Census Block Group GEOID;
      4. renames ACS variables to readable project fields;
      5. converts estimates and margins of error to numeric;
      6. removes invalid negative Census sentinel values;
      7. validates the processed ACS table;
      8. returns one processed pandas DataFrame.
    """

    # Request total population, population margin of error, total housing units, and housing-unit
    # margin of error. The NAME field is retained for source-level context.
    acs_variables = ['NAME', 'B01003_001E', 'B01003_001M', 'B25001_001E', 'B25001_001M']

    # The Census API expects requested variables as one comma-separated string.
    acs_get_fields = ','.join(acs_variables)

    # Utah's state FIPS code is required by the Census API geography hierarchy.
    utah_state_fips = '49'

    # Map readable county names to their three-digit county FIPS codes for the three project
    # counties.
    study_counties = {
        'Salt Lake County': '035',
        'Utah County': '049',
        'Washington County': '053',
    }

    # Retain each county response until the three tables can be combined into one project-wide ACS
    # dataset.
    acs_county_tables = []

    # Download one county at a time because the Census API requires county-specific geography
    # filters for Block Group requests.
    for county_name, county_fips in study_counties.items():
        print(f'\nDownloading {acs_year} ACS 5-Year data for {county_name}...')

        # Request every Block Group within every tract for the current county.
        acs_params = {
            'get': acs_get_fields,
            'for': 'block group:*',
            'in': f'state:{utah_state_fips} county:{county_fips} tract:*',
            'key': census_api_key,
        }

        # Submit the county-level ACS request and validate the response before constructing the
        # table.
        # Submit the parameterized request and allow enough time for a county-wide Block Group
        # response.
        acs_response = requests.get(acs_base_url, params=acs_params, timeout=120)

        # Raise immediately for HTTP failures so an error page is not processed as demographic data.
        acs_response.raise_for_status()

        # Confirm that the server returned JSON rather than an HTML error message or another
        # unexpected format.
        content_type = acs_response.headers.get('Content-Type', '').lower()

        # Stop execution when content type fails the requirement needed for reliable analysis.
        if 'json' not in content_type:
            raise ValueError(f'''The Census API did not return JSON.
County: {county_name}
Status code: {acs_response.status_code}
Response preview: {acs_response.text[:500]}''')

        # Decode the validated response. The Census API returns a list whose first row contains
        # field
        # names.
        acs_json = acs_response.json()

        # Stop execution when len fails the requirement needed for reliable analysis.
        if len(acs_json) <= 1:
            raise ValueError(f'No ACS Block Group records were returned for {county_name}.')

        # Convert the response rows to a DataFrame using the first API row as the column schema.
        county_acs = pd.DataFrame(acs_json[1:], columns=acs_json[0])

        # Add a readable county label because the raw response contains only numeric geographic
        # codes.
        county_acs['COUNTY_NAME'] = county_name
        acs_county_tables.append(county_acs)
        print(f'-> Downloaded {len(county_acs):,} Block Groups for {county_name}.')

    # Combine the three county tables into one continuous Block Group dataset.
    acs_df = pd.concat(acs_county_tables, ignore_index=True)
    # Derive acs df with reset_index for use in the next processing or validation step.
    acs_df = acs_df.reset_index(drop=True)
    print(f'\n-> Combined ACS records: {len(acs_df):,}')

    # Concatenate the state, county, tract, and Block Group codes into the standard 12-digit
    # Census Block Group GEOID used by the spatial layer.
    acs_df['GEOID'] = acs_df['state'].astype(str) + acs_df['county'].astype(str) + \
        acs_df['tract'].astype(str) + acs_df['block group'].astype(str)

    # Replace Census variable codes with descriptive project field names while preserving
    # estimates and margins of error as separate attributes.
    acs_df = acs_df.rename(columns={'B01003_001E': 'ACS_POPULATION',
        'B01003_001M': 'ACS_POPULATION_MOE', 'B25001_001E': 'ACS_HOUSING_UNITS',
        'B25001_001M': 'ACS_HOUSING_UNITS_MOE'})
    # Define acs numeric fields needed to keep the analysis schema consistent.
    acs_numeric_fields = [
        'ACS_POPULATION',
        'ACS_POPULATION_MOE',
        'ACS_HOUSING_UNITS',
        'ACS_HOUSING_UNITS_MOE',
    ]

    # Convert estimates and margins of error to numeric values; unexpected text becomes NaN for
    # later validation.
    for field in acs_numeric_fields:
        # Derive acs df with to_numeric for use in the next processing or validation step.
        acs_df[field] = pd.to_numeric(acs_df[field], errors='coerce')

    # Iterate through field records to apply the required processing consistently.
    for field in acs_numeric_fields:

        # Replace negative Census sentinel values with NaN so unavailable estimates are not treated
        # as
        # valid counts.
        acs_df.loc[acs_df[field] < 0, field] = np.nan

    # Retain only the standardized fields required by the downstream Census Block Group join.
    final_acs_fields = [
        'GEOID',
        'COUNTY_NAME',
        'ACS_POPULATION',
        'ACS_POPULATION_MOE',
        'ACS_HOUSING_UNITS',
        'ACS_HOUSING_UNITS_MOE',
    ]

    # Apply the final schema and create a clean sequential index after combining the county
    # tables.
    acs_df = acs_df[final_acs_fields].copy().reset_index(drop=True)

    # Stop before caching or joining when processing produces no usable ACS records.
    if acs_df.empty:
        raise ValueError('The processed ACS table contains no records.')

    # Store GEOIDs as strings so leading zeros are preserved and regular-expression validation is
    # reliable.
    acs_df['GEOID'] = acs_df['GEOID'].astype(str)

    # Every Census Block Group GEOID must contain exactly 12 numeric characters.
    invalid_geoid_mask = ~acs_df['GEOID'].str.fullmatch('\\d{12}')
    # Calculate invalid geoid count for validation and workflow QA.
    invalid_geoid_count = invalid_geoid_mask.sum()

    # Stop execution when invalid geoid count fails the requirement needed for reliable analysis.
    if invalid_geoid_count > 0:
        raise ValueError(f'{invalid_geoid_count} ACS '
            f'GEOIDs do not contain exactly '
            f'12 numeric characters.')

    # A duplicate GEOID would break the required one-to-one relationship with the Census geometry
    # layer.
    duplicate_geoid_count = acs_df['GEOID'].duplicated().sum()

    # Stop execution when duplicate geoid count fails the requirement needed for reliable analysis.
    if duplicate_geoid_count > 0:
        raise ValueError(f'The processed ACS table '
            f'contains '
            f'{duplicate_geoid_count} '
            f'duplicate GEOIDs.')

    # Verify that the processed table still contains the full output schema after cleaning and
    # field selection.
    missing_final_fields = [field for field in final_acs_fields if field not in acs_df.columns]

    # Stop execution when required final fields inputs are unavailable.
    if missing_final_fields:
        raise ValueError(f'The processed ACS table is '
            f'missing required fields: '
            f'{missing_final_fields}')
    print(f'-> Processed ACS records validated: {len(acs_df):,}')
    print(f'-> Duplicate GEOIDs: {duplicate_geoid_count}')
    print(f'-> Invalid GEOIDs: {invalid_geoid_count}')

    # Return one validated table that can be cached or joined directly to the Census Block Group
    # GeoDataFrame.
    return acs_df
print('-> Live ACS download and processing function created.')

print('\n=== LIVE ACS DOWNLOAD AND PROCESSING FUNCTION READY ===')


=== DEFINING LIVE ACS DOWNLOAD AND PROCESSING FUNCTION ===
-> Live ACS download and processing function created.

=== LIVE ACS DOWNLOAD AND PROCESSING FUNCTION READY ===


## Validate ACS-Census Join Results


In [10]:
# ----------------------------------------------------
# VALIDATE ACS-CENSUS JOIN RESULTS
# ----------------------------------------------------
# Check join completeness, duplicate identifiers, and missing demographic
# values so data-integration issues are identified before later analysis.
# Use snapshot mode for reproducibility or refresh mode to request and validate a new Census
# API extract.
print('=== CONFIGURING ACS DATA SOURCE ===')

# Change this value to 'refresh' only when a new Census API download is intentionally
# required.
acs_data_mode = 'snapshot'
# Evaluate valid acs data modes so invalid inputs or outputs can be rejected before continuing.
valid_acs_data_modes = ['snapshot', 'refresh']

# Stop execution when acs data mode fails the requirement needed for reliable analysis.
if acs_data_mode not in valid_acs_data_modes:
    raise ValueError("acs_data_mode must be either 'snapshot' or 'refresh'.")

# Confirm that release settings, paths, and the live-download function exist before source
# selection continues.
required_acs_configuration_objects = [
    'acs_year',
    'acs_base_url',
    'acs_snapshot_path',
    'download_and_prepare_acs_data',
]
# Identify missing acs configuration objects inputs before the workflow continues.
missing_acs_configuration_objects = [object_name for object_name in \
    required_acs_configuration_objects if object_name not in globals()]

# Stop execution when required acs configuration objects inputs are unavailable.
if missing_acs_configuration_objects:
    raise NameError(f'The following ACS '
        f'configuration objects are '
        f'missing: '
        f'{missing_acs_configuration_objects}. '
        f'Run the previous ACS setup '
        f'cells first.')
# Derive acs snapshot exists with exists for use in the next processing or validation step.
acs_snapshot_exists = acs_snapshot_path.exists()

# Convert the selected mode into a Boolean used by the loading branch below.
use_live_acs_api = acs_data_mode == 'refresh'
print(f'-> ACS release year: {acs_year}')
print(f'-> ACS data mode: {acs_data_mode}')
print(f'-> Processed snapshot exists: {acs_snapshot_exists}')
print(f'-> Snapshot location: {acs_snapshot_path.resolve()}')
print('\nNOTE:')

# Branch on acs data mode so the workflow follows the appropriate processing path.
if acs_data_mode == 'snapshot':
    print('The notebook will load the processed ACS snapshot. No Census API key will be requested.')

# Branch on acs data mode so the workflow follows the appropriate processing path.
if acs_data_mode == 'refresh':
    print('The notebook will download '
        'fresh ACS data from the Census '
        'API and update the processed '
        'snapshot. A Census API key '
        'will be required.')

print('\n=== ACS DATA SOURCE CONFIGURATION COMPLETE ===')


=== CONFIGURING ACS DATA SOURCE ===
-> ACS release year: 2024
-> ACS data mode: snapshot
-> Processed snapshot exists: True
-> Snapshot location: C:\Users\adamd\Projects\WUI\data\processed\utah_three_county_acs5_2024_block_groups_processed.csv

NOTE:
The notebook will load the processed ACS snapshot. No Census API key will be requested.

=== ACS DATA SOURCE CONFIGURATION COMPLETE ===


## Configure ACS Data Source Mode


In [11]:
# ----------------------------------------------------
# CONFIGURE ACS DATA SOURCE MODE
# ----------------------------------------------------
# Choose between the validated local snapshot and a live Census API
# refresh so the workflow can balance reproducibility with data updates.
print('=== LOADING OR REFRESHING ACS DATA ===')

# Refresh mode downloads and validates the source again, while snapshot mode reuses the
# processed local table.
if use_live_acs_api:
    print('-> Live Census API refresh selected.')

    # Prefer an environment variable so the API key is not written directly into the notebook.
    census_api_key = os.getenv('CENSUS_API_KEY')

    # Branch on census api key so the workflow follows the appropriate processing path.
    if not census_api_key:

        # Prompt securely only when the key is not already available in the environment.
        census_api_key = getpass.getpass('Enter your Census API key: ').strip()

    # Stop execution when census api key fails the requirement needed for reliable analysis.
    if not census_api_key:
        raise ValueError("A Census API key is required when acs_data_mode is set to 'refresh'.")
    # Derive acs df with download_and_prepare_acs_data for use in the next processing or validation
    # step.
    acs_df = download_and_prepare_acs_data(

        # Select the ACS 5-Year release used for population and housing-unit estimates.
        acs_year=acs_year,
        acs_base_url=acs_base_url,
        census_api_key=census_api_key,
    )

    # Stop execution when acs df fails the requirement needed for reliable analysis.
    if acs_df.empty:
        raise ValueError('The refreshed ACS dataset contains no records.')

    # Write refreshed ACS data to a temporary file first so a failed write cannot corrupt the last
    # valid snapshot.
    temporary_acs_snapshot_path = acs_snapshot_path.with_suffix('.temporary.csv')
    # Export the table so this workflow result is available to later phases.
    acs_df.to_csv(temporary_acs_snapshot_path, index=False)
    # Set temporary snapshot created used by the current processing stage.
    temporary_snapshot_created = temporary_acs_snapshot_path.exists() and \
        temporary_acs_snapshot_path.stat().st_size > 0

    # Stop execution when temporary snapshot created fails the requirement needed for reliable
    # analysis.
    if not temporary_snapshot_created:
        raise IOError('The temporary ACS snapshot could not be created.')

    # Replace the saved snapshot only after the temporary file is confirmed to exist and contain
    # data.
    temporary_acs_snapshot_path.replace(acs_snapshot_path)
    print('-> Processed ACS snapshot updated.')
    print(f'-> Snapshot location: {acs_snapshot_path.resolve()}')
    del census_api_key
    # Set acs data source used by the current processing stage.
    acs_data_source = 'Live Census API'
else:
    print('-> Loading processed ACS snapshot.')

    # Stop execution when acs snapshot exists fails the requirement needed for reliable analysis.
    if not acs_snapshot_exists:
        raise FileNotFoundError("The processed ACS snapshot was "
            "not found.\n\nChange\n"
            "acs_data_mode = 'refresh'\nand "
            "rerun this cell to create it.")

    # Load GEOID and county name as string columns so leading zeros and text formatting are
    # preserved.
    acs_df = pd.read_csv(
        acs_snapshot_path,
        dtype={'GEOID': 'string', 'COUNTY_NAME': 'string'},
    )
    # Set acs data source used by the current processing stage.
    acs_data_source = 'Processed ACS snapshot'
print(f'-> ACS data source: {acs_data_source}')
print(f'-> ACS records loaded: {len(acs_df):,}')
print(f'-> Snapshot location: {acs_snapshot_path.resolve()}')

print('\n=== ACS DATA LOADED ===')


=== LOADING OR REFRESHING ACS DATA ===
-> Loading processed ACS snapshot.
-> ACS data source: Processed ACS snapshot
-> ACS records loaded: 1,240
-> Snapshot location: C:\Users\adamd\Projects\WUI\data\processed\utah_three_county_acs5_2024_block_groups_processed.csv

=== ACS DATA LOADED ===


## Validate Loaded ACS Data


In [12]:
# ----------------------------------------------------
# VALIDATE LOADED ACS DATA
# ----------------------------------------------------
# Validate identifiers, required fields, numeric values, and record
# integrity before the ACS table is used in the spatial workflow.
# Apply the same validation checks whether the data came from the live API or the saved
# snapshot.
print('=== VALIDATING LOADED ACS DATA ===')

# Define the exact fields required by the downstream spatial join and exposure analysis.
required_loaded_acs_fields = [
    'GEOID',
    'COUNTY_NAME',
    'ACS_POPULATION',
    'ACS_POPULATION_MOE',
    'ACS_HOUSING_UNITS',
    'ACS_HOUSING_UNITS_MOE',
]
# Identify missing loaded acs fields inputs before the workflow continues.
missing_loaded_acs_fields = [field for field in required_loaded_acs_fields if field not in \
    acs_df.columns]

# Stop execution when required loaded acs fields inputs are unavailable.
if missing_loaded_acs_fields:
    raise ValueError(f'''The loaded ACS dataset is missing the following required fields:
{missing_loaded_acs_fields}''')

# Stop execution when acs df fails the requirement needed for reliable analysis.
if acs_df.empty:
    raise ValueError('The loaded ACS dataset contains no records.')

# Normalize GEOIDs as 12-character strings and restore leading zeros that CSV loading may
# remove.
acs_df['GEOID'] = acs_df['GEOID'].astype('string').str.strip().str.zfill(12)

# Standardize county labels so summaries and comparisons are not affected by surrounding
# whitespace.
acs_df['COUNTY_NAME'] = acs_df['COUNTY_NAME'].astype('string').str.strip()
# Define loaded acs numeric fields needed to keep the analysis schema consistent.
loaded_acs_numeric_fields = [
    'ACS_POPULATION',
    'ACS_POPULATION_MOE',
    'ACS_HOUSING_UNITS',
    'ACS_HOUSING_UNITS_MOE',
]

# Reconfirm numeric data types after either API processing or CSV loading.
for field in loaded_acs_numeric_fields:
    # Derive acs df with to_numeric for use in the next processing or validation step.
    acs_df[field] = pd.to_numeric(acs_df[field], errors='coerce')
# Calculate loaded duplicate geoid count for validation and workflow QA.
loaded_duplicate_geoid_count = acs_df['GEOID'].duplicated().sum()

# Stop execution when loaded duplicate geoid count fails the requirement needed for reliable
# analysis.
if loaded_duplicate_geoid_count > 0:
    raise ValueError(f'The loaded ACS dataset '
        f'contains '
        f'{loaded_duplicate_geoid_count} '
        f'duplicate GEOIDs.')
# Calculate loaded invalid geoid count for validation and workflow QA.
loaded_invalid_geoid_count = (~acs_df['GEOID'].str.fullmatch('\\d{12}')).sum()

# Stop execution when loaded invalid geoid count fails the requirement needed for reliable analysis.
if loaded_invalid_geoid_count > 0:
    raise ValueError(f'The loaded ACS dataset '
        f'contains '
        f'{loaded_invalid_geoid_count} '
        f'invalid GEOIDs.')

# Count missing estimates by field so incomplete source data cannot silently enter the
# exposure calculations.
loaded_acs_missing_value_counts = acs_df[loaded_acs_numeric_fields].isna().sum()

# Verify that no negative Census sentinel values remain after the cleaning process.
loaded_acs_negative_value_counts = acs_df[loaded_acs_numeric_fields].lt(0).sum()
# Define loaded acs fields with negative values needed to keep the analysis schema consistent.
loaded_acs_fields_with_negative_values = \
    loaded_acs_negative_value_counts[loaded_acs_negative_value_counts > 0]

# Stop execution when loaded acs fields with negative values fails the requirement needed for
# reliable analysis.
if not loaded_acs_fields_with_negative_values.empty:
    raise ValueError \
        (f'''The loaded ACS dataset contains invalid negative values in the following fields:
{loaded_acs_fields_with_negative_values}''')
# Identify missing loaded acs fields with values inputs before the workflow continues.
loaded_acs_fields_with_missing_values = \
    loaded_acs_missing_value_counts[loaded_acs_missing_value_counts > 0]

# Stop execution when loaded acs fields with missing values fails the requirement needed for
# reliable analysis.
if not loaded_acs_fields_with_missing_values.empty:
    raise ValueError(f'''The loaded ACS dataset contains missing values in the following fields:
{loaded_acs_fields_with_missing_values}''')
print(f'-> ACS data source: {acs_data_source}')
print(f'-> ACS Block Group records: {len(acs_df):,}')
print(f'-> Duplicate GEOIDs: {loaded_duplicate_geoid_count}')
print(f'-> Invalid GEOIDs: {loaded_invalid_geoid_count}')
print('\n--- ACS Missing Values by Field ---')
display(loaded_acs_missing_value_counts.rename('MISSING_VALUES').to_frame())
print('\n--- ACS Negative Values by Field ---')
display(loaded_acs_negative_value_counts.rename('NEGATIVE_VALUES').to_frame())
print('\nNOTE:')
print('The loaded ACS dataset passed '
    'all structural, data-type, '
    'GEOID, and numeric validation '
    'checks and is ready for '
    'downstream processing.')

print('\n=== ACS DATA VALIDATION COMPLETE ===')


=== VALIDATING LOADED ACS DATA ===
-> ACS data source: Processed ACS snapshot
-> ACS Block Group records: 1,240
-> Duplicate GEOIDs: 0
-> Invalid GEOIDs: 0

--- ACS Missing Values by Field ---


,MISSING_VALUES
ACS_POPULATION,0
ACS_POPULATION_MOE,0
ACS_HOUSING_UNITS,0
ACS_HOUSING_UNITS_MOE,0



--- ACS Negative Values by Field ---


,NEGATIVE_VALUES
ACS_POPULATION,0
ACS_POPULATION_MOE,0
ACS_HOUSING_UNITS,0
ACS_HOUSING_UNITS_MOE,0



NOTE:
The loaded ACS dataset passed all structural, data-type, GEOID, and numeric validation checks and is ready for downstream processing.

=== ACS DATA VALIDATION COMPLETE ===


## Create Standardized ACS Output Table


In [13]:
# ----------------------------------------------------
# CREATE FINAL ACS OUTPUT TABLE
# ----------------------------------------------------

print('=== CREATING FINAL ACS OUTPUT TABLE ===')
# Define the final acs input objects inputs required by this workflow stage.
required_final_acs_input_objects = [
    'acs_df',
    'acs_data_source',
    'required_loaded_acs_fields',
    'loaded_duplicate_geoid_count',
    'loaded_invalid_geoid_count',
    'loaded_acs_missing_value_counts',
    'loaded_acs_negative_value_counts',
]
# Identify missing final acs input objects inputs before the workflow continues.
missing_final_acs_input_objects = [object_name for object_name in \
    required_final_acs_input_objects if object_name not in globals()]

# Stop execution when required final acs input objects inputs are unavailable.
if missing_final_acs_input_objects:
    raise NameError(f'The following required ACS '
        f'objects are missing: '
        f'{missing_final_acs_input_objects}. '
        f'Run the ACS loading and '
        f'validation cells first.')

# Apply the required field order, sort by GEOID for deterministic output, and reset the index.
acs_final = acs_df[required_loaded_acs_fields].copy().sort_values('GEOID').reset_index(drop=True)

# Stop execution when acs final fails the requirement needed for reliable analysis.
if acs_final.empty:
    raise ValueError('The final ACS output table contains no records.')

# Confirm that field selection and sorting did not add or remove Block Group records.
acs_final_record_count_matches = len(acs_final) == len(acs_df)

# Stop execution when acs final record count matches fails the requirement needed for reliable
# analysis.
if not acs_final_record_count_matches:
    raise ValueError('The final ACS output table '
        'record count does not match '
        'the validated ACS dataset.')
# Identify missing final acs fields inputs before the workflow continues.
missing_final_acs_fields = [field for field in required_loaded_acs_fields if field not in \
    acs_final.columns]

# Stop execution when required final acs fields inputs are unavailable.
if missing_final_acs_fields:
    raise ValueError(f'''The final ACS output table is missing the following required fields:
{missing_final_acs_fields}''')

# Preserve a fixed field order so downstream code receives the expected output schema.
acs_final_field_order_valid = acs_final.columns.tolist() == required_loaded_acs_fields

# Stop execution when acs final field order valid fails the requirement needed for reliable
# analysis.
if not acs_final_field_order_valid:
    raise ValueError('The final ACS output fields are not arranged in the required order.')
# Calculate acs final duplicate geoid count for validation and workflow QA.
acs_final_duplicate_geoid_count = acs_final['GEOID'].duplicated().sum()

# Stop execution when acs final duplicate geoid count fails the requirement needed for reliable
# analysis.
if acs_final_duplicate_geoid_count > 0:
    raise ValueError(f'The final ACS output table '
        f'contains '
        f'{acs_final_duplicate_geoid_count} '
        f'duplicate GEOIDs.')
# Calculate acs final invalid geoid count for validation and workflow QA.
acs_final_invalid_geoid_count = (~acs_final['GEOID'].str.fullmatch('\\d{12}')).sum()

# Stop execution when acs final invalid geoid count fails the requirement needed for reliable
# analysis.
if acs_final_invalid_geoid_count > 0:
    raise ValueError(f'The final ACS output table '
        f'contains '
        f'{acs_final_invalid_geoid_count} '
        f'invalid GEOIDs.')

# Verify deterministic ascending GEOID order for reproducible inspection, caching, and
# joining.
acs_final_geoid_sort_valid = acs_final['GEOID'].is_monotonic_increasing

# Stop execution when acs final geoid sort valid fails the requirement needed for reliable analysis.
if not acs_final_geoid_sort_valid:
    raise ValueError('The final ACS output table is not sorted in ascending GEOID order.')
# Identify missing acs final value counts inputs before the workflow continues.
acs_final_missing_value_counts = acs_final[loaded_acs_numeric_fields].isna().sum()
# Calculate acs final negative value counts for validation and workflow QA.
acs_final_negative_value_counts = acs_final[loaded_acs_numeric_fields].lt(0).sum()
# Identify missing acs final fields with values inputs before the workflow continues.
acs_final_fields_with_missing_values = \
    acs_final_missing_value_counts[acs_final_missing_value_counts > 0]

# Stop execution when acs final fields with missing values fails the requirement needed for reliable
# analysis.
if not acs_final_fields_with_missing_values.empty:
    raise ValueError(f'''The final ACS output table contains missing values in the following fields:
{acs_final_fields_with_missing_values}''')
# Define acs final fields with negative values needed to keep the analysis schema consistent.
acs_final_fields_with_negative_values = \
    acs_final_negative_value_counts[acs_final_negative_value_counts > 0]

# Stop execution when acs final fields with negative values fails the requirement needed for
# reliable analysis.
if not acs_final_fields_with_negative_values.empty:
    raise ValueError \
        (f'''The final ACS output table contains invalid negative values in the following fields:
{acs_final_fields_with_negative_values}''')

# Summarize the final structural checks in a compact table for the project audit trail.
acs_final_output_summary = pd.DataFrame([{'METRIC': 'ACS Data Source',
    'VALUE': acs_data_source}, {'METRIC': 'Final ACS Records',
    'VALUE': len(acs_final)}, {'METRIC': 'Final ACS Fields',
    'VALUE': len(acs_final.columns)}, {'METRIC': 'Duplicate GEOIDs',
    'VALUE': acs_final_duplicate_geoid_count}, {'METRIC': 'Invalid GEOIDs',
    'VALUE': acs_final_invalid_geoid_count}, {'METRIC': 'Record Count Matches Source',
    'VALUE': acs_final_record_count_matches}, {'METRIC': 'Field Order Valid',
    'VALUE': acs_final_field_order_valid}, {'METRIC': 'GEOID Sort Valid',
    'VALUE': acs_final_geoid_sort_valid}])
print(f'-> ACS data source: {acs_data_source}')
print(f'-> Final ACS table prepared: {len(acs_final):,} records.')
print(f'-> Final ACS fields: {len(acs_final.columns)}')
print(f'-> Final ACS duplicate GEOIDs: {acs_final_duplicate_geoid_count}')
print(f'-> Final ACS invalid GEOIDs: {acs_final_invalid_geoid_count}')
print(f'-> Record count matches validated ACS data: {acs_final_record_count_matches}')
print(f'-> Final ACS field order valid: {acs_final_field_order_valid}')
print(f'-> Final ACS GEOID sort valid: {acs_final_geoid_sort_valid}')
print('\n--- FINAL ACS OUTPUT SUMMARY ---')
display(acs_final_output_summary)
print('\n--- SAMPLE FINAL ACS RECORDS ---')
display(acs_final.head(5))
print('\nNOTE:')
print('The final ACS output contains '
    'the standardized population, '
    'housing, and margin-of-error '
    'fields required by the '
    'downstream Census Block Group '
    'join.')
print('The validated source table '
    'remains available as acs_df, '
    'while acs_final is the ordered '
    'project output used by the '
    'remaining analysis workflow.')

print('\n=== FINAL ACS OUTPUT TABLE READY ===')


=== CREATING FINAL ACS OUTPUT TABLE ===
-> ACS data source: Processed ACS snapshot
-> Final ACS table prepared: 1,240 records.
-> Final ACS fields: 6
-> Final ACS duplicate GEOIDs: 0
-> Final ACS invalid GEOIDs: 0
-> Record count matches validated ACS data: True
-> Final ACS field order valid: True
-> Final ACS GEOID sort valid: True

--- FINAL ACS OUTPUT SUMMARY ---


,METRIC,VALUE
0,ACS Data Source,Processed ACS snapshot
1,Final ACS Records,1240
2,Final ACS Fields,6
3,Duplicate GEOIDs,0
4,Invalid GEOIDs,0
5,Record Count Matches Source,True
6,Field Order Valid,True
7,GEOID Sort Valid,True



--- SAMPLE FINAL ACS RECORDS ---


,GEOID,COUNTY_NAME,ACS_POPULATION,ACS_POPULATION_MOE,ACS_HOUSING_UNITS,ACS_HOUSING_UNITS_MOE
0,490351001001,Salt Lake County,473.0,225.0,334.0,174.0
1,490351001002,Salt Lake County,3395.0,458.0,2137.0,247.0
2,490351002001,Salt Lake County,1456.0,236.0,626.0,68.0
3,490351003061,Salt Lake County,2500.0,1380.0,1169.0,300.0
4,490351003062,Salt Lake County,2924.0,904.0,1290.0,254.0



NOTE:
The final ACS output contains the standardized population, housing, and margin-of-error fields required by the downstream Census Block Group join.
The validated source table remains available as acs_df, while acs_final is the ordered project output used by the remaining analysis workflow.

=== FINAL ACS OUTPUT TABLE READY ===


## Prepare ACS and Census Join Fields


In [14]:
# ----------------------------------------------------
# PREPARE ACS AND CENSUS JOIN FIELDS
# ----------------------------------------------------
# GEOID20 from the Census geometry layer and GEOID from the ACS table must use the same
# 12-character format.
print('=== PREPARING ACS AND CENSUS JOIN FIELDS ===')

# Create working copies so join-field standardization does not modify the original Phase 1 or
# ACS outputs.
census_join = gdf_census_aligned.copy()
# Derive acs join with copy for use in the next processing or validation step.
acs_join = acs_final.copy()

# Convert Census GEOID20 values to stripped strings while preserving leading zeros.
census_join['GEOID20'] = census_join['GEOID20'].astype(str).str.strip()

# Apply the same normalization to the ACS join field.
acs_join['GEOID'] = acs_join['GEOID'].astype(str).str.strip()

# Validate both join keys before merging so malformed identifiers cannot create false
# unmatched records.
invalid_census_geoids = (~census_join['GEOID20'].str.fullmatch('\\d{12}')).sum()
# Evaluate invalid acs geoids so invalid inputs or outputs can be rejected before continuing.
invalid_acs_geoids = (~acs_join['GEOID'].str.fullmatch('\\d{12}')).sum()
print(f'-> Invalid Census GEOID20 values: {invalid_census_geoids}')
print(f'-> Invalid ACS GEOID values: {invalid_acs_geoids}')

# Stop execution when invalid census geoids fails the requirement needed for reliable analysis.
if invalid_census_geoids > 0:
    raise ValueError(f'{invalid_census_geoids} Census '
        f'GEOID20 values do not contain '
        f'exactly 12 numeric characters.')

# Stop execution when invalid acs geoids fails the requirement needed for reliable analysis.
if invalid_acs_geoids > 0:
    raise ValueError(f'{invalid_acs_geoids} ACS GEOID '
        f'values do not contain exactly '
        f'12 numeric characters.')
print('-> Census and ACS join fields are ready.')

print('\n=== JOIN FIELD PREPARATION COMPLETE ===')


=== PREPARING ACS AND CENSUS JOIN FIELDS ===
-> Invalid Census GEOID20 values: 0
-> Invalid ACS GEOID values: 0
-> Census and ACS join fields are ready.

=== JOIN FIELD PREPARATION COMPLETE ===


## Join ACS Attributes to Census Block Groups


In [15]:
# ----------------------------------------------------
# JOIN ACS ATTRIBUTES TO CENSUS BLOCK GROUPS
# ----------------------------------------------------
print('=== JOINING ACS ATTRIBUTES TO CENSUS BLOCK GROUPS ===')

# Preserve the pre-join feature count so the merge can be verified not to duplicate or remove
# spatial features.
census_count_before_join = len(census_join)

# Limit the right-hand table to the demographic fields needed by the WUI exposure analysis.
acs_join_fields = acs_join[['GEOID', 'COUNTY_NAME',
    'ACS_POPULATION', 'ACS_POPULATION_MOE', 'ACS_HOUSING_UNITS',
    'ACS_HOUSING_UNITS_MOE']].copy()

# Use a one-to-one left merge so every Census Block Group is retained and duplicate ACS keys
# are rejected.
gdf_census_acs = census_join.merge(
    acs_join_fields,
    left_on='GEOID20',
    right_on='GEOID',
    how='left',
    validate='one_to_one',
)

# Restore the GeoDataFrame structure while preserving the Census geometry column and projected
# CRS.
gdf_census_acs = gpd.GeoDataFrame(gdf_census_acs, geometry='geometry', crs=census_join.crs)
print(f'-> Census Block Groups before join: {census_count_before_join:,}')
print(f'-> Census Block Groups after join: {len(gdf_census_acs):,}')

print('\n=== ACS-CENSUS JOIN COMPLETE ===')


=== JOINING ACS ATTRIBUTES TO CENSUS BLOCK GROUPS ===
-> Census Block Groups before join: 1,240
-> Census Block Groups after join: 1,240

=== ACS-CENSUS JOIN COMPLETE ===


## Validate ACS-Census Join Results


In [16]:
# ----------------------------------------------------
# VALIDATE ACS-CENSUS JOIN RESULTS
# ----------------------------------------------------
print('=== VALIDATING ACS-CENSUS JOIN RESULTS ===')

# The feature count should remain unchanged because the join adds attributes rather than
# creating or deleting geometries.
feature_count_difference = len(gdf_census_acs) - census_count_before_join
print(f'-> Change in feature count after join: {feature_count_difference}')

# Stop execution when feature count difference fails the requirement needed for reliable analysis.
if feature_count_difference != 0:
    raise ValueError('The ACS join changed the Census Block Group feature count.')

# Count missing estimates separately from missing GEOID matches to distinguish source-data
# gaps from join failures.
unmatched_population_count = gdf_census_acs['ACS_POPULATION'].isna().sum()
# Calculate unmatched housing count for validation and workflow QA.
unmatched_housing_count = gdf_census_acs['ACS_HOUSING_UNITS'].isna().sum()
# Calculate unmatched geoid count for validation and workflow QA.
unmatched_geoid_count = gdf_census_acs['GEOID'].isna().sum()

# Calculate the number and percentage of spatial records that received an ACS match.
matched_record_count = len(gdf_census_acs) - unmatched_geoid_count
# Set join match rate used by the current processing stage.
join_match_rate = matched_record_count / len(gdf_census_acs) * 100
print(f'-> Census Block Groups matched to ACS: {matched_record_count:,}')
print(f'-> Census Block Groups without ACS match: {unmatched_geoid_count:,}')
print(f'-> ACS join match rate: {join_match_rate:.2f}%')
print(f'-> Missing ACS population estimates: {unmatched_population_count:,}')
print(f'-> Missing ACS housing estimates: {unmatched_housing_count:,}')

# Verify that the merge did not create duplicate Census Block Group geometries.
duplicate_joined_geoids = gdf_census_acs['GEOID20'].duplicated().sum()
print(f'-> Duplicate Census GEOID20 values after join: {duplicate_joined_geoids}')

# Stop execution when duplicate joined geoids fails the requirement needed for reliable analysis.
if duplicate_joined_geoids > 0:
    raise ValueError('Duplicate Census Block Group GEOIDs were created during the ACS join.')

print('\n=== ACS-CENSUS JOIN VALIDATION COMPLETE ===')


=== VALIDATING ACS-CENSUS JOIN RESULTS ===
-> Change in feature count after join: 0
-> Census Block Groups matched to ACS: 1,240
-> Census Block Groups without ACS match: 0
-> ACS join match rate: 100.00%
-> Missing ACS population estimates: 0
-> Missing ACS housing estimates: 0
-> Duplicate Census GEOID20 values after join: 0

=== ACS-CENSUS JOIN VALIDATION COMPLETE ===


## Inspect Census and ACS Inputs


In [17]:
# ----------------------------------------------------
# INSPECT FINAL ACS-ENHANCED CENSUS DATASET
# ----------------------------------------------------
# Validate identifiers, required fields, numeric values, and record
# integrity before the ACS table is used in the spatial workflow.
print('=== INSPECTING FINAL ACS-ENHANCED CENSUS DATASET ===')

# Isolate unmatched Census records for targeted review without displaying full geometry.
unmatched_census_acs = gdf_census_acs[gdf_census_acs['GEOID'].isna()][['GEOID20',
    'COUNTYFP20']].copy()

# Branch on len so the workflow follows the appropriate processing path.
if len(unmatched_census_acs) > 0:
    print(f'-> Unmatched Census Block Groups found: {len(unmatched_census_acs):,}')
    display(unmatched_census_acs.head(5))
else:
    print('-> Every Census Block Group successfully matched an ACS demographic record.')

# Display a concise sample containing both join keys, source geography, demographic estimates,
# margins of error, and geometry.
joined_output_sample = gdf_census_acs[['GEOID20',
    'GEOID', 'COUNTYFP20', 'COUNTY_NAME', 'ACS_POPULATION',
    'ACS_POPULATION_MOE', 'ACS_HOUSING_UNITS', 'ACS_HOUSING_UNITS_MOE',
    'geometry']].head(5)
print('\n--- FINAL ACS-ENHANCED CENSUS SAMPLE ---')
display(joined_output_sample)
print(f'\n-> Final analytical GeoDataFrame contains {len(gdf_census_acs):,} Census Block Groups.')
print(f'-> Final analytical CRS: {gdf_census_acs.crs}')

print('\n=== ACS-ENHANCED CENSUS DATASET READY FOR EXPOSURE ANALYSIS ===')


=== INSPECTING FINAL ACS-ENHANCED CENSUS DATASET ===
-> Every Census Block Group successfully matched an ACS demographic record.

--- FINAL ACS-ENHANCED CENSUS SAMPLE ---


,GEOID20,GEOID,COUNTYFP20,COUNTY_NAME,ACS_POPULATION,ACS_POPULATION_MOE,ACS_HOUSING_UNITS,ACS_HOUSING_UNITS_MOE,geometry
0,490351113061,490351113061,035,Salt Lake County,1353.0,269.0,696.0,86.0,"MULTIPOLYGON (((429467.222 4496405.559, 429498..."
1,490351113062,490351113062,035,Salt Lake County,722.0,212.0,274.0,63.0,"MULTIPOLYGON (((430494.109 4495540.762, 430520..."
2,490351114001,490351114001,035,Salt Lake County,1525.0,504.0,924.0,230.0,"MULTIPOLYGON (((425456.78 4507874.317, 425457...."
3,490351114002,490351114002,035,Salt Lake County,756.0,445.0,220.0,128.0,"MULTIPOLYGON (((425447.585 4507253.601, 425448..."
4,490351114003,490351114003,035,Salt Lake County,1307.0,510.0,528.0,194.0,"MULTIPOLYGON (((425434.423 4506381.029, 425434..."



-> Final analytical GeoDataFrame contains 1,240 Census Block Groups.
-> Final analytical CRS: EPSG:26912

=== ACS-ENHANCED CENSUS DATASET READY FOR EXPOSURE ANALYSIS ===


# WEA Phase 3 – WUI Exposure Analysis

## Purpose

Calculate block-group WUI exposure metrics, including exposed land area, WUI percentage, estimated exposed population, estimated exposed housing units, and intersecting WUI polygons.


## Enhanced WUI Exposure Assessment


In [18]:
# ----------------------------------------------------
# ENHANCED WUI EXPOSURE ASSESSMENT
# ----------------------------------------------------
# This phase measures how much of each Census Block Group
# intersects the clipped High-Risk WUI layer. The resulting
# area proportion is then used to estimate exposed population
# and housing units from ACS demographic totals.
print(
    "\n=== PHASE 3: ENHANCED WILDFIRE "
    "EXPOSURE ASSESSMENT ==="
)

# ----------------------------------------------------
# PREPARE ACS-ENHANCED EXPOSURE INPUTS
# ----------------------------------------------------
print(
    "=== PREPARING ACS-ENHANCED EXPOSURE INPUTS ==="
)

# Create working copies so the ACS-enriched Census layer and
# clipped WUI layer from earlier phases remain unchanged.
gdf_exposure_acs = (
    gdf_census_acs.copy()
)

# Derive wui exposure zones with copy for use in the next processing or validation step.
wui_exposure_zones = (
    gdf_wui_aligned.copy()
)

# Define the minimum fields required to calculate land-area,
# population, housing, and margin-of-error exposure metrics.
required_exposure_fields = [
    "GEOID20",
    "COUNTYFP20",
    "ACS_POPULATION",
    "ACS_POPULATION_MOE",
    "ACS_HOUSING_UNITS",
    "ACS_HOUSING_UNITS_MOE",
    "geometry"
]

# Identify missing fields before any spatial or demographic
# calculations are performed.
missing_exposure_fields = [
    field
    for field in required_exposure_fields
    if field not in gdf_exposure_acs.columns
]

# Stop when the ACS-enriched Census layer does not contain the
# complete schema required by the exposure workflow.
if missing_exposure_fields:

    raise ValueError(
        "The ACS-enhanced Census layer is missing the "
        "following required fields: "
        f"{missing_exposure_fields}"
    )

# Area and overlay operations require both spatial layers to use
# the same projected coordinate system.
if (
    gdf_exposure_acs.crs
    != wui_exposure_zones.crs
):

    raise ValueError(
        "The ACS Census Block Groups and WUI layers must "
        "use the same CRS before exposure analysis."
    )

# Define the ACS estimate and margin-of-error fields that must
# be numeric before area-weighted calculations are applied.
acs_analysis_fields = [
    "ACS_POPULATION",
    "ACS_POPULATION_MOE",
    "ACS_HOUSING_UNITS",
    "ACS_HOUSING_UNITS_MOE"
]

# Convert the ACS fields to numeric values. Unexpected text is
# converted to NaN so it can be counted and reported instead of
# causing an arithmetic error later.
for field in acs_analysis_fields:

    # Derive exposure acs with to_numeric for use in the next processing or validation step.
    gdf_exposure_acs[
        field
    ] = pd.to_numeric(
        gdf_exposure_acs[
            field
        ],
        errors="coerce"
    )

# Count missing primary ACS estimates before exposure is
# calculated so source-data limitations remain visible.
missing_population_count = (
    gdf_exposure_acs[
        "ACS_POPULATION"
    ]
    # Check required fields for missing values before using them in exposure calculations.
    .isna()
    .sum()
)

# Identify missing housing count inputs before the workflow continues.
missing_housing_count = (
    gdf_exposure_acs[
        "ACS_HOUSING_UNITS"
    ]
    # Check required fields for missing values before using them in exposure calculations.
    .isna()
    .sum()
)

print(
    f"-> Census Block Groups available for analysis: "
    f"{len(gdf_exposure_acs):,}"
)

print(
    f"-> Clipped WUI polygon parts available: "
    f"{len(wui_exposure_zones):,}"
)

print(
    f"-> Missing ACS population estimates: "
    f"{missing_population_count:,}"
)

print(
    f"-> Missing ACS housing estimates: "
    f"{missing_housing_count:,}"
)

print(
    f"-> Exposure analysis CRS: "
    f"{gdf_exposure_acs.crs}"
)

print(
    "\n=== ACS EXPOSURE INPUT PREPARATION COMPLETE ==="
)



=== PHASE 3: ENHANCED WILDFIRE EXPOSURE ASSESSMENT ===
=== PREPARING ACS-ENHANCED EXPOSURE INPUTS ===
-> Census Block Groups available for analysis: 1,240
-> Clipped WUI polygon parts available: 391
-> Missing ACS population estimates: 0
-> Missing ACS housing estimates: 0
-> Exposure analysis CRS: EPSG:26912

=== ACS EXPOSURE INPUT PREPARATION COMPLETE ===


### Calculate Block Group and WUI Intersections


In [19]:
# ----------------------------------------------------
# CALCULATE BLOCK GROUP AND WUI INTERSECTIONS
# ----------------------------------------------------
print(
    "=== CALCULATING BLOCK GROUP AND "
    "WUI INTERSECTIONS ==="
)

# Calculate the complete area of every Census Block Group.
# EPSG:26912 uses meters, so GeoPandas returns square meters.
gdf_exposure_acs[
    "BLOCK_GROUP_AREA_M2"
] = (
    # Calculate polygon area in the projected CRS so the exposure metric uses meter-based geometry.
    gdf_exposure_acs.geometry.area
)

# Convert total Block Group area to square kilometers for
# reporting and map popups.
gdf_exposure_acs[
    "BLOCK_GROUP_AREA_KM2"
] = (
    gdf_exposure_acs[
        "BLOCK_GROUP_AREA_M2"
    ]
    / 1_000_000
)

# Reset the WUI index and assign a unique identifier to every
# individual polygon part. The identifier is used to count how
# many distinct WUI polygons intersect each Block Group.
wui_exposure_zones = (
    wui_exposure_zones
    .reset_index(
        drop=True
    )
    .copy()
)

# Set wui exposure zones used by the current processing stage.
wui_exposure_zones[
    "WUI_POLYGON_ID"
] = (
    wui_exposure_zones.index
    + 1
)

# Create exact shared geometries between Census Block Groups and
# WUI polygon parts. Overlay is required because a spatial join
# identifies relationships but does not create measurable
# intersection geometry.
wui_block_group_intersections = (
    gdf_exposure_acs.overlay(
        wui_exposure_zones[
            [
                "WUI_POLYGON_ID",
                "geometry"
            ]
        ],
        how="intersection",
        keep_geom_type=True,
    )
)

# Remove null or empty geometries that can result from
# boundary-only contacts or small geometry artifacts.
wui_block_group_intersections = (
    wui_block_group_intersections[
        wui_block_group_intersections.geometry.notna()
        & ~wui_block_group_intersections.geometry.is_empty
    ]
    .copy()
)

# Stop when no measurable intersection geometry is created,
# because all later exposure calculations depend on it.
if wui_block_group_intersections.empty:

    raise ValueError(
        "No Census Block Groups intersect the clipped "
        "WUI layer."
    )

# Measure the area of each Block Group-WUI intersection piece.
wui_block_group_intersections[
    "INTERSECTION_AREA_M2"
] = (
    # Calculate polygon area in the projected CRS so the exposure metric uses meter-based geometry.
    wui_block_group_intersections.geometry.area
)

print(
    f"-> Created "
    f"{len(wui_block_group_intersections):,} "
    "Block Group-WUI intersection features."
)

print(
    "\n=== BLOCK GROUP AND WUI "
    "INTERSECTIONS COMPLETE ==="
)


=== CALCULATING BLOCK GROUP AND WUI INTERSECTIONS ===
-> Created 538 Block Group-WUI intersection features.

=== BLOCK GROUP AND WUI INTERSECTIONS COMPLETE ===


## Calculate Exposed Area and WUI Polygon Counts


In [20]:
# ----------------------------------------------------
# CALCULATE EXPOSED AREA AND WUI POLYGON COUNTS
# ----------------------------------------------------
print(
    "=== CALCULATING EXPOSED AREA AND "
    "WUI POLYGON COUNTS ==="
)

# Count unique WUI polygon parts associated with each Census
# Block Group. nunique prevents duplicate counts when geometry
# processing creates more than one intersection piece.
wui_polygon_counts = (
    wui_block_group_intersections
    .groupby(
        "GEOID20",
        as_index=False
    )
    .agg(
        INTERSECTING_WUI_POLYGONS=(
            "WUI_POLYGON_ID",
            "nunique"
        )
    )
)

# Dissolve all intersection pieces belonging to the same Block
# Group before measuring exposed area. This prevents overlapping
# WUI pieces from being added as separate geometries.
dissolved_block_group_exposure = (
    wui_block_group_intersections[
        [
            "GEOID20",
            "geometry"
        ]
    ]
    .dissolve(
        by="GEOID20"
    )
    .reset_index()
)

# Calculate the total WUI-exposed area for each Block Group in
# square meters and square kilometers.
dissolved_block_group_exposure[
    "EXPOSED_AREA_M2"
] = (
    # Calculate polygon area in the projected CRS so the exposure metric uses meter-based geometry.
    dissolved_block_group_exposure.geometry.area
)

# Set dissolved block group exposure used by the current processing stage.
dissolved_block_group_exposure[
    "EXPOSED_AREA_KM2"
] = (
    dissolved_block_group_exposure[
        "EXPOSED_AREA_M2"
    ]
    / 1_000_000
)

# Combine exposed area and distinct WUI polygon counts into one
# GEOID-level table. validate="one_to_one" confirms that each
# Block Group appears once in both summarized inputs.
exposure_by_geoid = (
    dissolved_block_group_exposure[
        [
            "GEOID20",
            "EXPOSED_AREA_M2",
            "EXPOSED_AREA_KM2"
        ]
    ]
    # Join the prepared attributes using the shared key required for the exposure workflow.
    .merge(
        wui_polygon_counts,
        on="GEOID20",
        how="left",
        validate="one_to_one"
    )
)

print(
    f"-> Block Groups with measurable WUI exposure: "
    f"{len(exposure_by_geoid):,}"
)

print(
    "\n=== EXPOSED AREA AND WUI COUNTS COMPLETE ==="
)


=== CALCULATING EXPOSED AREA AND WUI POLYGON COUNTS ===
-> Block Groups with measurable WUI exposure: 111

=== EXPOSED AREA AND WUI COUNTS COMPLETE ===


## Estimate ACS Population and Housing Exposure


In [21]:
# ----------------------------------------------------
# ESTIMATE ACS POPULATION AND HOUSING EXPOSURE
# ----------------------------------------------------
print(
    "=== ESTIMATING ACS POPULATION AND "
    "HOUSING EXPOSURE ==="
)

# Join the summarized exposure values back to the complete
# ACS-enhanced Block Group layer. A left join retains both
# exposed and non-exposed Block Groups.
gdf_exposure_acs = (
    gdf_exposure_acs
    # Join the prepared attributes using the shared key required for the exposure workflow.
    .merge(
        exposure_by_geoid,
        on="GEOID20",
        how="left",
        validate="one_to_one"
    )
)

# Block Groups without a WUI intersection receive zero exposed
# area and zero intersecting polygons rather than missing values.
zero_fill_fields = [
    "EXPOSED_AREA_M2",
    "EXPOSED_AREA_KM2",
    "INTERSECTING_WUI_POLYGONS"
]

# Derive exposure acs with fillna for use in the next processing or validation step.
gdf_exposure_acs[
    zero_fill_fields
] = (
    gdf_exposure_acs[
        zero_fill_fields
    ]
    .fillna(
        0
    )
)

# Convert the WUI polygon count to an integer because it
# represents a discrete number of intersecting features.
gdf_exposure_acs[
    "INTERSECTING_WUI_POLYGONS"
] = (
    gdf_exposure_acs[
        "INTERSECTING_WUI_POLYGONS"
    ]
    .astype(
        int
    )
)

# Calculate the fraction of each Block Group located inside the
# WUI. np.where prevents division by zero for any geometry with
# a calculated total area of zero.
gdf_exposure_acs[
    "EXPOSURE_PROPORTION"
] = np.where(
    gdf_exposure_acs[
        "BLOCK_GROUP_AREA_M2"
    ] > 0,
    (
        gdf_exposure_acs[
            "EXPOSED_AREA_M2"
        ]
        / gdf_exposure_acs[
            "BLOCK_GROUP_AREA_M2"
        ]
    ),
    0
)

# Restrict exposure proportions to the valid range of zero to
# one. This protects against tiny floating-point differences
# produced by polygon overlay operations.
gdf_exposure_acs[
    "EXPOSURE_PROPORTION"
] = (
    gdf_exposure_acs[
        "EXPOSURE_PROPORTION"
    ]
    # Clip the spatial layer to the project study area before calculating exposure metrics.
    .clip(
        lower=0,
        upper=1
    )
)

# Convert the exposure proportion to a percentage for readable
# reporting and interactive map popups.
gdf_exposure_acs[
    "PERCENT_WUI"
] = (
    gdf_exposure_acs[
        "EXPOSURE_PROPORTION"
    ]
    * 100
).round(
    2
)

# Estimate exposed population using area-weighted allocation.
# This assumes population is distributed evenly throughout each
# Census Block Group.
gdf_exposure_acs[
    "EST_ACS_POP_EXPOSED"
] = (
    gdf_exposure_acs[
        "ACS_POPULATION"
    ]
    * gdf_exposure_acs[
        "EXPOSURE_PROPORTION"
    ]
)

# Estimate exposed housing units using the same area-weighted
# allocation assumption.
gdf_exposure_acs[
    "EST_ACS_HOUSING_EXPOSED"
] = (
    gdf_exposure_acs[
        "ACS_HOUSING_UNITS"
    ]
    * gdf_exposure_acs[
        "EXPOSURE_PROPORTION"
    ]
)

# Apply the same area proportion to the ACS margins of error.
# These are approximate allocated MOEs and are not formally
# recalculated Census margins of error.
gdf_exposure_acs[
    "EST_ACS_POP_EXPOSED_MOE"
] = (
    gdf_exposure_acs[
        "ACS_POPULATION_MOE"
    ]
    * gdf_exposure_acs[
        "EXPOSURE_PROPORTION"
    ]
)

# Set exposure acs used by the current processing stage.
gdf_exposure_acs[
    "EST_ACS_HOUSING_EXPOSED_MOE"
] = (
    gdf_exposure_acs[
        "ACS_HOUSING_UNITS_MOE"
    ]
    * gdf_exposure_acs[
        "EXPOSURE_PROPORTION"
    ]
)

# Round estimated people, housing units, and approximate MOEs to
# whole values. Pandas nullable Int64 preserves missing ACS data.
integer_exposure_fields = [
    "EST_ACS_POP_EXPOSED",
    "EST_ACS_HOUSING_EXPOSED",
    "EST_ACS_POP_EXPOSED_MOE",
    "EST_ACS_HOUSING_EXPOSED_MOE"
]

# Iterate through field records to apply the required processing consistently.
for field in integer_exposure_fields:

    # Derive exposure acs with astype for use in the next processing or validation step.
    gdf_exposure_acs[
        field
    ] = (
        gdf_exposure_acs[
            field
        ]
        .round()
        .astype(
            "Int64"
        )
    )

# Create a Boolean flag that identifies Block Groups with
# measurable WUI exposure.
gdf_exposure_acs[
    "IS_EXPOSED"
] = (
    gdf_exposure_acs[
        "EXPOSED_AREA_M2"
    ] > 0
)

print(
    f"-> Exposed Census Block Groups: "
    f"{gdf_exposure_acs['IS_EXPOSED'].sum():,}"
)

print(
    "\n=== ACS EXPOSURE ESTIMATION COMPLETE ==="
)


=== ESTIMATING ACS POPULATION AND HOUSING EXPOSURE ===
-> Exposed Census Block Groups: 111

=== ACS EXPOSURE ESTIMATION COMPLETE ===


## Create ACS Exposure Summary Statistics


In [22]:
# ----------------------------------------------------
# CREATE ACS EXPOSURE SUMMARY STATISTICS
# ----------------------------------------------------
print(
    "=== CREATING ACS EXPOSURE "
    "SUMMARY STATISTICS ==="
)

# Calculate project-wide totals across the complete
# three-county study area.
total_exposed_block_groups = int(
    gdf_exposure_acs[
        "IS_EXPOSED"
    ].sum()
)

# Derive total exposed area km2 with sum for use in the next processing or validation step.
total_exposed_area_km2 = (
    gdf_exposure_acs[
        "EXPOSED_AREA_KM2"
    ].sum()
)

# Derive total acs population exposed with sum for use in the next processing or validation step.
total_acs_population_exposed = (
    gdf_exposure_acs[
        "EST_ACS_POP_EXPOSED"
    ].sum()
)

# Derive total acs housing exposed with sum for use in the next processing or validation step.
total_acs_housing_exposed = (
    gdf_exposure_acs[
        "EST_ACS_HOUSING_EXPOSED"
    ].sum()
)

# Derive total intersecting wui relationships with sum for use in the next processing or validation
# step.
total_intersecting_wui_relationships = (
    gdf_exposure_acs[
        "INTERSECTING_WUI_POLYGONS"
    ].sum()
)

print(
    "\n--- THREE-COUNTY ACS EXPOSURE TOTALS ---"
)

print(
    f"Exposed Census Block Groups: "
    f"{total_exposed_block_groups:,}"
)

print(
    f"Estimated ACS Population Exposed: "
    f"{total_acs_population_exposed:,.0f}"
)

print(
    f"Estimated ACS Housing Units Exposed: "
    f"{total_acs_housing_exposed:,.0f}"
)

print(
    f"Exposed Land Area: "
    f"{total_exposed_area_km2:,.2f} km²"
)

print(
    f"Total Block Group-WUI Polygon Relationships: "
    f"{total_intersecting_wui_relationships:,}"
)

# Aggregate exposure results by county so the three study
# counties can be compared using the same output metrics.
county_exposure_summary_acs = (
    gdf_exposure_acs
    .groupby(
        "COUNTY_NAME",
        as_index=False
    )
    .agg(
        TOTAL_ACS_POPULATION=(
            "ACS_POPULATION",
            "sum"
        ),
        TOTAL_ACS_HOUSING_UNITS=(
            "ACS_HOUSING_UNITS",
            "sum"
        ),
        EXPOSED_BLOCK_GROUPS=(
            "IS_EXPOSED",
            "sum"
        ),
        EST_ACS_POP_EXPOSED=(
            "EST_ACS_POP_EXPOSED",
            "sum"
        ),
        EST_ACS_HOUSING_EXPOSED=(
            "EST_ACS_HOUSING_EXPOSED",
            "sum"
        ),
        EXPOSED_AREA_KM2=(
            "EXPOSED_AREA_KM2",
            "sum"
        ),
        WUI_INTERSECTION_COUNT=(
            "INTERSECTING_WUI_POLYGONS",
            "sum"
        )
    )
)

# Calculate the estimated percentage of each county's total ACS
# population located within exposed Block Group area.
county_exposure_summary_acs[
    "PERCENT_ACS_POP_EXPOSED"
] = np.where(
    county_exposure_summary_acs[
        "TOTAL_ACS_POPULATION"
    ] > 0,
    (
        county_exposure_summary_acs[
            "EST_ACS_POP_EXPOSED"
        ]
        / county_exposure_summary_acs[
            "TOTAL_ACS_POPULATION"
        ]
    )
    * 100,
    0
)

# Calculate the estimated percentage of each county's ACS
# housing units located within exposed Block Group area.
county_exposure_summary_acs[
    "PERCENT_ACS_HOUSING_EXPOSED"
] = np.where(
    county_exposure_summary_acs[
        "TOTAL_ACS_HOUSING_UNITS"
    ] > 0,
    (
        county_exposure_summary_acs[
            "EST_ACS_HOUSING_EXPOSED"
        ]
        / county_exposure_summary_acs[
            "TOTAL_ACS_HOUSING_UNITS"
        ]
    )
    * 100,
    0
)

# Round percentage and area fields for clear presentation.
county_exposure_summary_acs[
    [
        "PERCENT_ACS_POP_EXPOSED",
        "PERCENT_ACS_HOUSING_EXPOSED",
        "EXPOSED_AREA_KM2"
    ]
] = county_exposure_summary_acs[
    [
        "PERCENT_ACS_POP_EXPOSED",
        "PERCENT_ACS_HOUSING_EXPOSED",
        "EXPOSED_AREA_KM2"
    ]
].round(
    2
)

print(
    "\n--- COUNTY ACS EXPOSURE SUMMARY ---"
)

display(
    county_exposure_summary_acs
)

print(
    "\n=== ACS EXPOSURE SUMMARY "
    "STATISTICS COMPLETE ==="
)


=== CREATING ACS EXPOSURE SUMMARY STATISTICS ===

--- THREE-COUNTY ACS EXPOSURE TOTALS ---
Exposed Census Block Groups: 111
Estimated ACS Population Exposed: 19,192
Estimated ACS Housing Units Exposed: 6,933
Exposed Land Area: 407.24 km²
Total Block Group-WUI Polygon Relationships: 538

--- COUNTY ACS EXPOSURE SUMMARY ---


,COUNTY_NAME,TOTAL_ACS_POPULATION,TOTAL_ACS_HOUSING_UNITS,EXPOSED_BLOCK_GROUPS,EST_ACS_POP_EXPOSED,EST_ACS_HOUSING_EXPOSED,EXPOSED_AREA_KM2,WUI_INTERSECTION_COUNT,PERCENT_ACS_POP_EXPOSED,PERCENT_ACS_HOUSING_EXPOSED
0,Salt Lake County,1196523.0,449662.0,20,4552,1523,63.60,72,0.38,0.34
1,Utah County,705400.0,211066.0,45,6464,1640,93.26,188,0.92,0.78
2,Washington County,196431.0,82189.0,46,8176,3770,250.38,278,4.16,4.59



=== ACS EXPOSURE SUMMARY STATISTICS COMPLETE ===


## Create Final ACS Block Group Exposure Table


In [23]:
# ----------------------------------------------------
# CREATE FINAL ACS BLOCK GROUP EXPOSURE TABLE
# ----------------------------------------------------
print(
    "=== CREATING FINAL ACS BLOCK GROUP "
    "EXPOSURE TABLE ==="
)

# Select the final nonspatial reporting fields while preserving
# the full analytical GeoDataFrame as gdf_exposure_acs.
block_group_exposure_acs = (
    gdf_exposure_acs[
        [
            "GEOID20",
            "COUNTY_NAME",
            "ACS_POPULATION",
            "ACS_POPULATION_MOE",
            "ACS_HOUSING_UNITS",
            "ACS_HOUSING_UNITS_MOE",
            "BLOCK_GROUP_AREA_KM2",
            "EXPOSED_AREA_KM2",
            "EXPOSURE_PROPORTION",
            "PERCENT_WUI",
            "EST_ACS_POP_EXPOSED",
            "EST_ACS_POP_EXPOSED_MOE",
            "EST_ACS_HOUSING_EXPOSED",
            "EST_ACS_HOUSING_EXPOSED_MOE",
            "INTERSECTING_WUI_POLYGONS",
            "IS_EXPOSED"
        ]
    ]
    .copy()
)

# Round land-area values to two decimal places for reporting.
block_group_exposure_acs[
    [
        "BLOCK_GROUP_AREA_KM2",
        "EXPOSED_AREA_KM2"
    ]
] = block_group_exposure_acs[
    [
        "BLOCK_GROUP_AREA_KM2",
        "EXPOSED_AREA_KM2"
    ]
].round(
    2
)

# Retain four decimal places for the zero-to-one exposure
# proportion because small differences matter at this scale.
block_group_exposure_acs[
    "EXPOSURE_PROPORTION"
] = (
    block_group_exposure_acs[
        "EXPOSURE_PROPORTION"
    ]
    .round(
        4
    )
)

# Sort the reporting table from highest to lowest estimated
# population exposure so the most affected Block Groups appear
# first in the final inspection.
block_group_exposure_acs = (
    block_group_exposure_acs
    .sort_values(
        by="EST_ACS_POP_EXPOSED",
        ascending=False,
        na_position="last"
    )
    .reset_index(
        drop=True
    )
)

print(
    f"-> Final exposure GeoDataFrame records: "
    f"{len(gdf_exposure_acs):,}"
)

print(
    f"-> Final reporting-table records: "
    f"{len(block_group_exposure_acs):,}"
)

print(
    "\n--- HIGHEST ACS POPULATION "
    "EXPOSURE BLOCK GROUPS ---"
)

display(
    block_group_exposure_acs.head(
        5
    )
)

print(
    "\n=== ENHANCED WILDFIRE EXPOSURE "
    "ASSESSMENT COMPLETE ==="
)


=== CREATING FINAL ACS BLOCK GROUP EXPOSURE TABLE ===
-> Final exposure GeoDataFrame records: 1,240
-> Final reporting-table records: 1,240

--- HIGHEST ACS POPULATION EXPOSURE BLOCK GROUPS ---


,GEOID20,COUNTY_NAME,ACS_POPULATION,ACS_POPULATION_MOE,ACS_HOUSING_UNITS,ACS_HOUSING_UNITS_MOE,BLOCK_GROUP_AREA_KM2,EXPOSED_AREA_KM2,EXPOSURE_PROPORTION,PERCENT_WUI,EST_ACS_POP_EXPOSED,EST_ACS_POP_EXPOSED_MOE,EST_ACS_HOUSING_EXPOSED,EST_ACS_HOUSING_EXPOSED_MOE,INTERSECTING_WUI_POLYGONS,IS_EXPOSED
0,490351151071,Salt Lake County,4063.0,687.0,1185.0,235.0,17.92,5.67,0.3168,31.68,1287,218,375,74,2,True
1,490532704011,Washington County,3272.0,676.0,1539.0,278.0,15.68,5.26,0.3356,33.56,1098,227,516,93,3,True
2,490351151082,Salt Lake County,5535.0,847.0,1871.0,412.0,16.61,2.46,0.1478,14.78,818,125,277,61,1,True
3,490532705021,Washington County,2735.0,768.0,897.0,219.0,8.37,2.42,0.2888,28.88,790,222,259,63,2,True
4,490490101141,Utah County,4706.0,1032.0,1131.0,269.0,9.60,1.54,0.1604,16.04,755,166,181,43,1,True



=== ENHANCED WILDFIRE EXPOSURE ASSESSMENT COMPLETE ===


# WEA Phase 4 – Critical Facilities

## Purpose

Acquire, standardize, combine, and evaluate critical facilities for WUI exposure, nearest-WUI proximity, and surrounding ACS demographic context.


## Critical-Facility Analysis


In [24]:
# ----------------------------------------------------
# PHASE 4: CRITICAL FACILITIES ANALYSIS
# ----------------------------------------------------
print('\n=== PHASE 4: CRITICAL FACILITIES ANALYSIS ===')

print('=== CREATING REUSABLE STUDY-AREA GEOMETRY ===')

# Calculate study area counties for validation and workflow QA.
study_area_counties = gdf_county_boundaries_aligned.copy()

# Define the exact three county names required for the Phase 4 study area.
expected_counties = {'SALT LAKE', 'UTAH', 'WASHINGTON'}

# Normalize county names before confirming that the input contains only the three study counties.
returned_counties = set(study_area_counties['NAME'].astype(str).str.upper().str.strip())

print(f'-> Counties available for study-area creation: {sorted(returned_counties)}')

# Stop processing unless the county input contains exactly Salt Lake, Utah, and Washington counties.
if returned_counties != expected_counties:
    raise ValueError('The aligned county dataset '
        'does not contain exactly Salt '
        'Lake, Utah, and Washington '
        'counties.')

# Dissolve the three county polygons with the current GeoPandas union API.
study_area_geometry = study_area_counties.geometry.union_all()

# Check geometry validity before running the next spatial operation.
if not study_area_geometry.is_valid:
    # Use a zero-distance buffer as a compatibility fallback for repairing invalid geometry.
    study_area_geometry = study_area_geometry.buffer(0)

# Stop the workflow if dissolving the county polygons does not produce a usable study-area geometry.
if study_area_geometry is None or study_area_geometry.is_empty:
    raise ValueError('The combined study-area geometry is empty.')

print(f'-> Study-area geometry type: {study_area_geometry.geom_type}')

print(f'-> Study-area CRS: {study_area_counties.crs}')



=== PHASE 4: CRITICAL FACILITIES ANALYSIS ===
=== CREATING REUSABLE STUDY-AREA GEOMETRY ===
-> Counties available for study-area creation: ['SALT LAKE', 'UTAH', 'WASHINGTON']
-> Study-area geometry type: MultiPolygon
-> Study-area CRS: EPSG:26912


## Build Reusable Study-Area Geometry


In [25]:
# ----------------------------------------------------
# REUSABLE STUDY-AREA GEOMETRY READY
# ----------------------------------------------------
print('\n=== REUSABLE STUDY-AREA GEOMETRY READY ===')

print('=== DEFINING CRITICAL FACILITY DATA SOURCES ===')

# Map each critical-facility category to its authoritative ArcGIS Feature Service layer.
facility_sources = {
    'Hospitals and Medical Facilities': 'https://services1.arcgis.com/' \
        '99lidPhWCzftIe9K/ArcGIS/rest/' \
        'services/' \
        'LicensedHealthCareFacilities/' \
        'FeatureServer/0',
    'Fire Stations': 'https://services1.arcgis.com/' \
        '99lidPhWCzftIe9K/ArcGIS/rest/' \
        'services/FireStations/' \
        'FeatureServer/0',
    'Police and Sheriff Stations': 'https://services1.arcgis.com/' \
        '99lidPhWCzftIe9K/ArcGIS/rest/' \
        'services/' \
        'society_law_enforcement_locations/' \
        'FeatureServer/0',
    'Public Schools': 'https://services1.arcgis.com/' \
        '99lidPhWCzftIe9K/ArcGIS/rest/' \
        'services/Schools_PreKto12/' \
        'FeatureServer/0',
    'Emergency Shelters': 'https://gis.fema.gov/arcgis/rest/services/NSS/FEMA_NSS/FeatureServer/5',
}

print(f'-> Defined {len(facility_sources)} critical facility datasets.')

print('\nFacility Categories:')

# Apply this processing sequence consistently across each item in the collection.
for category in facility_sources:
    print(f'   • {category}')



=== REUSABLE STUDY-AREA GEOMETRY READY ===
=== DEFINING CRITICAL FACILITY DATA SOURCES ===
-> Defined 5 critical facility datasets.

Facility Categories:
   • Hospitals and Medical Facilities
   • Fire Stations
   • Police and Sheriff Stations
   • Public Schools
   • Emergency Shelters


## Define Critical-Facility Data Sources


In [26]:
# ----------------------------------------------------
# CRITICAL FACILITY DATA SOURCES DEFINED
# ----------------------------------------------------
# Define the critical-facility data sources and study-area query parameters
# used for acquisition.
print('\n=== CRITICAL FACILITY DATA SOURCES DEFINED ===')

print('=== INSPECTING FACILITY SERVICE SCHEMAS ===')

# List the source attributes expected from each facility service before downloading records.
expected_facility_fields = {
    'Hospitals and Medical Facilities': ['OBJECTID',
        'FACILITY_NAME', 'ADDRESS', 'CITY', 'COUNTY', 'LICENSE_TYPE'],
    'Fire Stations': ['OBJECTID', 'NAME', 'ADDRESS', 'CITY', 'COUNTY', 'FIPS'],
    'Police and Sheriff Stations': ['OBJECTID', 'name', 'address', 'city', 'county'],
    'Public Schools': ['OBJECTID', 'SchoolName',
        'PrivateSchool', 'CharterSchool', 'Address', 'City'],
    'Emergency Shelters': ['shelter_id', 'shelter_name',
        'address_1', 'city', 'county_parish', 'fips_code',
        'state'],
}

# Store validated service metadata for reuse by the download routine.
facility_service_metadata = {}

# Inspect each facility service with the same schema-validation procedure.
for facility_category, layer_url in facility_sources.items():
    print(f'\nInspecting {facility_category}...')

    # Send the source-data request and retain the response for validation before processing.
    metadata_response = requests.get(layer_url, params={'f': 'json'}, timeout=60)

    # Stop the workflow if the remote service returns an unsuccessful HTTP response.
    metadata_response.raise_for_status()

    # Parse the service response before extracting or standardizing the returned records.
    metadata = metadata_response.json()

    # Stop when the ArcGIS service returns an application-level error in its JSON response.
    if 'error' in metadata:
        raise ValueError(f"ArcGIS metadata error for {facility_category}: {metadata['error']}")

    # Extract the field names advertised by the current ArcGIS service.
    returned_fields = [field['name'] for field in metadata.get('fields', [])]

    # Derive object id field with get for use in the next processing or validation step.
    object_id_field = metadata.get('objectIdField')

    # Search the service field definitions for an object-ID field when metadata does not provide
    # one.
    if not object_id_field:
        # Fall back to fields typed as ArcGIS object IDs when the metadata does not name one
        # explicitly.
        object_id_candidates = [field['name'] for field in metadata.get('fields',
            []) if field.get('type') == 'esriFieldTypeOID']

        # Set object id field used by the current processing stage.
        object_id_field = object_id_candidates[0] if object_id_candidates else None

    # Identify expected source fields that are absent from the current service schema.
    missing_expected_fields = [field for field in expected_facility_fields[facility_category] if \
        field not in returned_fields]

    print(f"-> Layer name: {metadata.get('name')}")

    print(f"-> Geometry type: {metadata.get('geometryType')}")

    print(f'-> Object ID field: {object_id_field}')

    print(f"-> Maximum records per response: {metadata.get('maxRecordCount')}")

    print(f'-> Missing expected fields: {missing_expected_fields}')

    # Stop before downloading records if the source schema is missing fields required by this
    # workflow.
    if missing_expected_fields:
        raise ValueError(f'The {facility_category} source '
            f'is missing expected fields: '
            f'{missing_expected_fields}')

    # Define facility service metadata used to configure this workflow stage.
    facility_service_metadata[facility_category] = {'layer_url': layer_url,
        'object_id_field': object_id_field, 'available_fields': returned_fields,
        'max_record_count': metadata.get('maxRecordCount',
        1000)}



=== CRITICAL FACILITY DATA SOURCES DEFINED ===
=== INSPECTING FACILITY SERVICE SCHEMAS ===

Inspecting Hospitals and Medical Facilities...
-> Layer name: LicensedHealthCareFacilities
-> Geometry type: esriGeometryPoint
-> Object ID field: OBJECTID
-> Maximum records per response: 2000
-> Missing expected fields: []

Inspecting Fire Stations...
-> Layer name: FireStations
-> Geometry type: esriGeometryPoint
-> Object ID field: OBJECTID
-> Maximum records per response: 2000
-> Missing expected fields: []

Inspecting Police and Sheriff Stations...
-> Layer name: law_enforcement_locations
-> Geometry type: esriGeometryPoint
-> Object ID field: OBJECTID
-> Maximum records per response: 2000
-> Missing expected fields: []

Inspecting Public Schools...
-> Layer name: Schools_PreKto12
-> Geometry type: esriGeometryPoint
-> Object ID field: OBJECTID
-> Maximum records per response: 2000
-> Missing expected fields: []

Inspecting Emergency Shelters...
-> Layer name: Shelter Locations
-> Geometr

## Validate Facility-Service Schemas


In [27]:
# ----------------------------------------------------
# FACILITY SERVICE SCHEMA VALIDATION COMPLETE
# ----------------------------------------------------
print('\n=== FACILITY SERVICE SCHEMA VALIDATION COMPLETE ===')

print('=== CREATING COUNTY QUERY ENVELOPES ===')

# Reproject the layer to the common project CRS before spatial comparison, overlay, or distance
# analysis.
county_boundaries_web = gdf_county_boundaries_aligned.to_crs('EPSG:4326').copy()

# Calculate county boundaries web for validation and workflow QA.
county_boundaries_web['COUNTY_QUERY_NAME'] = \
    county_boundaries_web['NAME'].astype(str).str.upper().str.strip()

# Define the three counties that must be represented in the REST query envelopes.
expected_query_counties = {'SALT LAKE', 'UTAH', 'WASHINGTON'}

# Calculate returned query counties for validation and workflow QA.
returned_query_counties = set(county_boundaries_web['COUNTY_QUERY_NAME'])

# Stop envelope creation unless all three study counties are present.
if returned_query_counties != expected_query_counties:
    raise ValueError('The county boundary layer does '
        'not contain exactly Salt Lake, '
        'Utah, and Washington counties.')

# Store one WGS 84 bounding envelope per study county for server-side spatial filtering.
county_query_envelopes = {}

# Build a REST query envelope from the bounds of each study county.
for _, county_row in county_boundaries_web.iterrows():
    # Set min x and min y and max x and max y used by the current processing stage.
    min_x, min_y, max_x, max_y = county_row.geometry.bounds

    # Define county query envelopes used to configure this workflow stage.
    county_query_envelopes[county_row['COUNTY_QUERY_NAME']] = {'xmin': min_x,
        'ymin': min_y, 'xmax': max_x, 'ymax': max_y, 'spatialReference': {'wkid': 4326}}

    print(f"-> Created query envelope for {county_row['COUNTY_QUERY_NAME']} County.")



=== FACILITY SERVICE SCHEMA VALIDATION COMPLETE ===
=== CREATING COUNTY QUERY ENVELOPES ===
-> Created query envelope for WASHINGTON County.
-> Created query envelope for SALT LAKE County.
-> Created query envelope for UTAH County.


## Build County Query Envelopes


In [28]:
# ----------------------------------------------------
# COUNTY QUERY ENVELOPES COMPLETE
# ----------------------------------------------------
# Build county-based query envelopes so remote facility services can be
# filtered to the three-county study area.
print('\n=== COUNTY QUERY ENVELOPES COMPLETE ===')

print('=== CREATING SERVER-FILTERED FACILITY DOWNLOAD FUNCTION ===')

print('=== CREATING RETRY-ENABLED HTTP SESSION ===')

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Set facility batch size used by the current processing stage.
facility_batch_size = 500

# Define facility request timeout used to configure this workflow stage.
facility_request_timeout = (15, 120)

# Derive facility request retry strategy with Retry for use in the next processing or validation
# step.
facility_request_retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    status=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=['GET', 'POST'],
    respect_retry_after_header=True,
    raise_on_status=True,
)

# Configure facility request session used for reliable requests to the source service.
facility_request_session = requests.Session()

# Derive facility request adapter with HTTPAdapter for use in the next processing or validation
# step.
facility_request_adapter = HTTPAdapter(max_retries=facility_request_retry_strategy)

facility_request_session.mount('https://', facility_request_adapter)

facility_request_session.mount('http://', facility_request_adapter)

print('-> Retry-enabled HTTP session created.')

print('-> Temporary HTTP 429, 500, 502, 503, and 504 responses will be retried automatically.')

print(f'-> Default facility batch size: {facility_batch_size:,} records.')

print('-> Request timeout: 15-second connection timeout and 120-second response timeout.')



=== COUNTY QUERY ENVELOPES COMPLETE ===
=== CREATING SERVER-FILTERED FACILITY DOWNLOAD FUNCTION ===
=== CREATING RETRY-ENABLED HTTP SESSION ===
-> Retry-enabled HTTP session created.
-> Temporary HTTP 429, 500, 502, 503, and 504 responses will be retried automatically.
-> Default facility batch size: 500 records.
-> Request timeout: 15-second connection timeout and 120-second response timeout.


## Configure a Retry-Enabled HTTP Session


In [29]:
# ----------------------------------------------------
# RETRY-ENABLED HTTP SESSION READY
# ----------------------------------------------------
print('\n=== RETRY-ENABLED HTTP SESSION READY ===')

# Define a reusable helper so the same logic is applied consistently throughout the notebook.
def download_facilities_for_study_area(
    layer_url,
    layer_name,
    county_envelopes,
    where_clause='1=1',
    batch_size=facility_batch_size,
):
    """
    Download facilities intersecting the three study-county
    envelopes from an ArcGIS FeatureServer.
    The function:
      1. queries object IDs separately for each county envelope;
      2. combines and deduplicates the returned object IDs;
      3. downloads selected features in retry-enabled GET batches;
      4. validates the requested and returned record counts;
      5. returns one GeoDataFrame in EPSG:4326.
    Parameters
    ----------
    layer_url : str
        ArcGIS FeatureServer layer URL.
    layer_name : str
        Readable source name used in progress messages.
    county_envelopes : dict
        ArcGIS envelope geometry for each study county.
    where_clause : str, default "1=1"
        Optional attribute filter applied by the FeatureServer.
    batch_size : int, default facility_batch_size
        Number of object IDs requested in each feature batch.
    Returns
    -------
    geopandas.GeoDataFrame
        Facilities potentially located within the study area.
    """

    # Validate the service URL type before building ArcGIS REST requests.
    if not isinstance(layer_url, str):
        raise TypeError('layer_url must be provided as a string.')

    # Reject an empty service URL before attempting any network request.
    if not layer_url.strip():
        raise ValueError('layer_url cannot be empty.')

    # Require county query envelopes to be supplied as a dictionary keyed by county.
    if not isinstance(county_envelopes, dict):
        raise TypeError('county_envelopes must be a dictionary.')

    # Stop the download routine when no county envelopes are available for spatial filtering.
    if not county_envelopes:
        raise ValueError('county_envelopes contains no study counties.')

    # Require a positive integer batch size so service pagination behaves predictably.
    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError('batch_size must be a positive integer.')

    # Prevent per-request batches from exceeding the notebook's configured service limit.
    if batch_size > facility_batch_size:
        print(f'-> Requested batch size of {batch_size:,} exceeds the recommended maximum.')

        print(f'-> Batch size reduced to {facility_batch_size:,} records.')

        # Set batch size used by the current processing stage.
        batch_size = facility_batch_size

    # Define query url used to access the configured source service.
    query_url = f"{layer_url.rstrip('/')}/query"

    print(f'\nDownloading {layer_name}...')

    print(f'-> Query endpoint: {query_url}')

    # Derive study area object ids with set for use in the next processing or validation step.
    study_area_object_ids = set()

    # Define county candidate counts used to configure this workflow stage.
    county_candidate_counts = {}

    # Apply this processing sequence consistently across each item in the collection.
    for county_name, envelope in county_envelopes.items():
        print(f'-> Querying candidate object IDs for {county_name} County...')

        # Derive envelope json with dumps for use in the next processing or validation step.
        envelope_json = json.dumps(envelope)

        # Define id params used to configure this workflow stage.
        id_params = {
            'where': where_clause,
            'geometry': envelope_json,
            'geometryType': 'esriGeometryEnvelope',
            'inSR': '4326',
            'spatialRel': 'esriSpatialRelIntersects',
            'returnIdsOnly': 'true',
            'returnGeometry': 'false',
            'f': 'json',
        }

        # Derive id response with get for use in the next processing or validation step.
        id_response = facility_request_session.get(
            query_url,
            params=id_params,
            timeout=facility_request_timeout,
        )

        # Stop the workflow if the remote service returns an unsuccessful HTTP response.
        id_response.raise_for_status()

        # Attempt the operation so expected data, file, or service failures can be handled
        # explicitly.
        try:
            # Parse the service response before extracting or standardizing the returned records.
            id_data = id_response.json()
        # Handle the expected failure without leaving the workflow in an inconsistent state.
        except ValueError as error:
            raise ValueError \
                (f'''{layer_name} returned non-JSON content while querying {county_name} County.
Response preview: {id_response.text[:500]}''') from error

        # Stop this county query when ArcGIS returns an application-level error instead of object
        # IDs.
        if 'error' in id_data:
            raise ValueError(f"ArcGIS query error for "
                f"{layer_name}, {county_name} "
                f"County: {id_data['error']}")

        # Calculate county object ids for validation and workflow QA.
        county_object_ids = id_data.get('objectIds', [])

        # Treat a missing objectIds array as an invalid ArcGIS response rather than as an empty
        # result.
        if county_object_ids is None:
            # Define county object ids used to configure this workflow stage.
            county_object_ids = []

        # Calculate county candidate counts for validation and workflow QA.
        county_candidate_counts[county_name] = len(county_object_ids)

        study_area_object_ids.update(county_object_ids)

        print(f'-> {county_name} County '
            f'envelope returned '
            f'{len(county_object_ids):,} '
            f'candidate records.')

    # Derive study area object ids with sorted for use in the next processing or validation step.
    study_area_object_ids = sorted(study_area_object_ids)

    # Stop the facility download when the county queries return no object IDs for the study area.
    if not study_area_object_ids:
        print(f'-> No {layer_name} records were found within the three study-county envelopes.')

        # Return the validated result to the calling workflow.
        return gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

    print(f'-> Unique candidate records across all three counties: {len(study_area_object_ids):,}')

    # Define downloaded batches used to configure this workflow stage.
    downloaded_batches = []

    # Calculate total batch count for validation and workflow QA.
    total_batch_count = (len(study_area_object_ids) + batch_size - 1) // batch_size

    # Iterate through batch number and batch start records to apply the required processing
    # consistently.
    for batch_number, batch_start in enumerate(
        range(0, len(study_area_object_ids), batch_size),
        start=1,
    ):
        # Set batch ids used by the current processing stage.
        batch_ids = study_area_object_ids[batch_start:batch_start + batch_size]

        # Derive batch end with min for use in the next processing or validation step.
        batch_end = min(batch_start + batch_size, len(study_area_object_ids))

        print(f'-> Downloading batch '
            f'{batch_number:,} of '
            f'{total_batch_count:,}: records '
            f'{batch_start + 1:,} through '
            f'{batch_end:,}...')

        # Define batch query params used to configure this workflow stage.
        batch_query_params = {
            'objectIds': ','.join((str(object_id) for object_id in batch_ids)),
            'outFields': '*',
            'returnGeometry': 'true',
            'outSR': '4326',
            'f': 'geojson',
        }

        # Derive batch response with post for use in the next processing or validation step.
        batch_response = facility_request_session.post(
            query_url,
            data=batch_query_params,
            timeout=facility_request_timeout,
        )

        # Stop the workflow if the remote service returns an unsuccessful HTTP response.
        batch_response.raise_for_status()

        # Attempt the operation so expected data, file, or service failures can be handled
        # explicitly.
        try:
            # Parse the service response before extracting or standardizing the returned records.
            batch_geojson = batch_response.json()
        # Handle the expected failure without leaving the workflow in an inconsistent state.
        except ValueError as error:
            raise ValueError(f'''{layer_name} returned non-JSON content for batch {batch_number:,}.
Response preview: {batch_response.text[:500]}''') from error

        # Stop processing the current feature batch if ArcGIS returns an error payload.
        if 'error' in batch_geojson:
            raise ValueError(f"ArcGIS download error for "
                f"{layer_name}, batch "
                f"{batch_number:,}: "
                f"{batch_geojson['error']}")

        # Derive batch features with get for use in the next processing or validation step.
        batch_features = batch_geojson.get('features', [])

        # Skip empty feature batches while continuing to process any valid batches already returned.
        if not batch_features:
            raise ValueError(f'No features were returned for '
                f'requested {layer_name} batch '
                f'{batch_number:,}.')

        # Derive batch gdf with from_features for use in the next processing or validation step.
        batch_gdf = gpd.GeoDataFrame.from_features(batch_features, crs='EPSG:4326')

        # Stop when filtering leaves no usable records for the next analysis step.
        if batch_gdf.empty:
            raise ValueError(f'The converted {layer_name} '
                f'GeoDataFrame is empty for '
                f'batch {batch_number:,}.')

        downloaded_batches.append(batch_gdf)

        print(f'-> Batch {batch_number:,} returned {len(batch_gdf):,} features.')

    # Stop the download routine if no feature batch produced usable records.
    if not downloaded_batches:
        raise ValueError(f'No {layer_name} feature batches were successfully downloaded.')

    # Derive complete gdf with GeoDataFrame for use in the next processing or validation step.
    complete_gdf = gpd.GeoDataFrame(
        # Combine the prepared facility datasets into one table after standardizing their schemas.
        pd.concat(downloaded_batches, ignore_index=True, sort=False),
        geometry='geometry',
        crs='EPSG:4326',
    )

    # Calculate combined feature count for validation and workflow QA.
    combined_feature_count = len(complete_gdf)

    # Define possible object id fields needed to keep the analysis schema consistent.
    possible_object_id_fields = ['OBJECTID', 'ObjectID', 'objectid', 'FID', 'fid']

    # Derive object id field with next for use in the next processing or validation step.
    object_id_field = next(
        (field for field in possible_object_id_fields if field in complete_gdf.columns),
        None,
    )

    # Deduplicate on the service object-ID field when one is available.
    if object_id_field is not None:
        # Remove duplicate records before continuing so counts and joins remain one-to-one where
        # expected.
        complete_gdf = complete_gdf.drop_duplicates(subset=[object_id_field]).reset_index(drop=True)

        print(f'-> Object-ID field used for deduplication: {object_id_field}')
    # Use the fallback path when the preferred field or geometry method is unavailable.
    else:
        # Reset the index after filtering or deduplication to maintain a clean sequential index.
        complete_gdf = complete_gdf.reset_index(drop=True)

        print('-> No standard object-ID field was available for attribute-based deduplication.')

    # Calculate duplicate feature count for validation and workflow QA.
    duplicate_feature_count = combined_feature_count - len(complete_gdf)

    # Calculate requested object id count for validation and workflow QA.
    requested_object_id_count = len(study_area_object_ids)

    # Calculate final downloaded feature count for validation and workflow QA.
    final_downloaded_feature_count = len(complete_gdf)

    print(f'-> Requested unique object IDs: {requested_object_id_count:,}')

    print(f'-> Combined features before deduplication: {combined_feature_count:,}')

    print(f'-> Duplicate features removed: {duplicate_feature_count:,}')

    print(f'-> Final downloaded features: {final_downloaded_feature_count:,}')

    # Treat an over-count as a download-integrity failure because more features were returned than
    # requested.
    if final_downloaded_feature_count > requested_object_id_count:
        raise ValueError(f'{layer_name} returned more '
            f'final features than the number '
            f'of unique object IDs '
            f'requested.')

    # Warn when the service returns fewer features than the unique object IDs requested.
    if final_downloaded_feature_count < requested_object_id_count:
        # Identify missing feature count inputs before the workflow continues.
        missing_feature_count = requested_object_id_count - final_downloaded_feature_count

        print('\nWARNING:')

        print(f'{missing_feature_count:,} '
            f'requested {layer_name} records '
            f'were not present in the final '
            f'downloaded dataset.')

        print('The exact county-boundary '
            'validation step should still '
            'be run, but the source service '
            'response should be reviewed '
            'for completeness.')

    # Confirm the assembled service result is still a GeoDataFrame before returning it.
    if not isinstance(complete_gdf, gpd.GeoDataFrame):
        raise TypeError(f'The completed {layer_name} output is not a GeoDataFrame.')

    # Store complete gdf crs so spatial operations use the required coordinate reference system.
    complete_gdf_crs = complete_gdf.crs.to_string() if complete_gdf.crs is not None else None

    # Require downloaded service geometry to remain in WGS 84 before project reprojection.
    if complete_gdf_crs != 'EPSG:4326':
        raise ValueError(f'The downloaded {layer_name} layer does not use EPSG:4326.')

    print(f'-> Downloaded {final_downloaded_feature_count:,} filtered {layer_name} records.')

    print(f'-> Downloaded layer CRS: {complete_gdf_crs}')

    # Return the validated result to the calling workflow.
    return complete_gdf

print('-> Server-filtered facility download function created.')

print('-> Object-ID GET queries and '
    'feature-batch POST queries '
    'will use automatic retries and '
    'separate connection/read '
    'timeouts.')

# Load the spatial source into a GeoDataFrame for geospatial processing.
print('-> GeoJSON responses will be converted directly into GeoDataFrames without gpd.read_file().')



=== RETRY-ENABLED HTTP SESSION READY ===
-> Server-filtered facility download function created.
-> Object-ID GET queries and feature-batch POST queries will use automatic retries and separate connection/read timeouts.
-> GeoJSON responses will be converted directly into GeoDataFrames without gpd.read_file().


## Define the Server-Filtered Facility Download Function


In [30]:
# ----------------------------------------------------
# SERVER-FILTERED FACILITY DOWNLOAD FUNCTION READY
# ----------------------------------------------------
# Define the reusable server-filtered download function used to retrieve
# facility records consistently from each ArcGIS service.
print('\n=== SERVER-FILTERED FACILITY DOWNLOAD FUNCTION READY ===')

print('=== DEFINING SOURCE-SPECIFIC FACILITY FILTERS ===')

# Store the study-county FIPS codes used by sources that support attribute filtering.
study_county_fips = ['49035', '49049', '49053']

# Define source-specific server-side filters to reduce downloads before exact spatial validation.
facility_where_clauses = {
    'Hospitals and Medical Facilities': '1=1',
    'Fire Stations': "FIPS IN ('49035', '49049', '49053')",
    'Police and Sheriff Stations': '1=1',
    'Public Schools': '1=1',
    'Emergency Shelters': "state = 'UT'",
}

# Apply this processing sequence consistently across each item in the collection.
for category, where_clause in facility_where_clauses.items():
    print(f'-> {category}: {where_clause}')



=== SERVER-FILTERED FACILITY DOWNLOAD FUNCTION READY ===
=== DEFINING SOURCE-SPECIFIC FACILITY FILTERS ===
-> Hospitals and Medical Facilities: 1=1
-> Fire Stations: FIPS IN ('49035', '49049', '49053')
-> Police and Sheriff Stations: 1=1
-> Public Schools: 1=1
-> Emergency Shelters: state = 'UT'


## Apply Source-Specific Facility Filters


In [31]:
# ----------------------------------------------------
# SOURCE-SPECIFIC FACILITY FILTERS COMPLETE
# ----------------------------------------------------
# Define the critical-facility data sources and study-area query parameters
# used for acquisition.
print('\n=== SOURCE-SPECIFIC FACILITY FILTERS COMPLETE ===')

print('=== DOWNLOADING STUDY-AREA CRITICAL FACILITIES ===')

# Derive medical facilities raw with download_facilities_for_study_area for use in the next
# processing or validation step.
gdf_medical_facilities_raw = download_facilities_for_study_area(
    layer_url=facility_sources['Hospitals and Medical Facilities'],
    layer_name='Hospitals and Medical Facilities',
    county_envelopes=county_query_envelopes,
    where_clause=facility_where_clauses['Hospitals and Medical Facilities'],
)

# Derive fire stations raw with download_facilities_for_study_area for use in the next processing or
# validation step.
gdf_fire_stations_raw = download_facilities_for_study_area(
    layer_url=facility_sources['Fire Stations'],
    layer_name='Fire Stations',
    county_envelopes=county_query_envelopes,
    where_clause=facility_where_clauses['Fire Stations'],
)

# Derive law enforcement raw with download_facilities_for_study_area for use in the next processing
# or validation step.
gdf_law_enforcement_raw = download_facilities_for_study_area(
    layer_url=facility_sources['Police and Sheriff Stations'],
    layer_name='Police and Sheriff Stations',
    county_envelopes=county_query_envelopes,
    where_clause=facility_where_clauses['Police and Sheriff Stations'],
)

# Derive schools raw with download_facilities_for_study_area for use in the next processing or
# validation step.
gdf_schools_raw = download_facilities_for_study_area(
    layer_url=facility_sources['Public Schools'],
    layer_name='PreK-12 Schools',
    county_envelopes=county_query_envelopes,
    where_clause=facility_where_clauses['Public Schools'],
)

# Derive shelters raw with download_facilities_for_study_area for use in the next processing or
# validation step.
gdf_shelters_raw = download_facilities_for_study_area(
    layer_url=facility_sources['Emergency Shelters'],
    layer_name='Emergency Shelters',
    county_envelopes=county_query_envelopes,
    where_clause=facility_where_clauses['Emergency Shelters'],
)

print('\n--- FILTERED FACILITY DOWNLOAD COUNTS ---')

print(f'Medical facilities: {len(gdf_medical_facilities_raw):,}')

print(f'Fire stations: {len(gdf_fire_stations_raw):,}')

print(f'Police and sheriff stations: {len(gdf_law_enforcement_raw):,}')

print(f'PreK-12 school candidates: {len(gdf_schools_raw):,}')

print(f'Emergency shelters: {len(gdf_shelters_raw):,}')



=== SOURCE-SPECIFIC FACILITY FILTERS COMPLETE ===
=== DOWNLOADING STUDY-AREA CRITICAL FACILITIES ===

-> Query endpoint: https://services1.arcgis.com/99lidPhWCzftIe9K/ArcGIS/rest/services/LicensedHealthCareFacilities/FeatureServer/0/query
-> Querying candidate object IDs for WASHINGTON County...
-> WASHINGTON County envelope returned 97 candidate records.
-> Querying candidate object IDs for SALT LAKE County...
-> SALT LAKE County envelope returned 353 candidate records.
-> Querying candidate object IDs for UTAH County...
-> UTAH County envelope returned 240 candidate records.
-> Unique candidate records across all three counties: 599
-> Downloading batch 1 of 2: records 1 through 500...
-> Batch 1 returned 500 features.
-> Downloading batch 2 of 2: records 501 through 599...
-> Batch 2 returned 99 features.
-> Object-ID field used for deduplication: OBJECTID
-> Requested unique object IDs: 599
-> Combined features before deduplication: 599
-> Duplicate features removed: 0
-> Final do

## Download Study-Area Facilities


In [32]:
# ----------------------------------------------------
# STUDY-AREA FACILITY DOWNLOADS COMPLETE
# ----------------------------------------------------
print('\n=== STUDY-AREA FACILITY DOWNLOADS COMPLETE ===')

print('=== FILTERING SCHOOL DATASET TO PUBLIC SCHOOLS ===')

# Summarize the source PrivateSchool values before deciding which records represent public schools.
private_school_values = \
    gdf_schools_raw['PrivateSchool'].astype('string').str.strip().value_counts(dropna=False)

print('\nPrivateSchool source values:')

display(private_school_values.rename('RECORD_COUNT').to_frame())

# Normalize PrivateSchool values so equivalent private-school flags are treated consistently.
private_school_normalized = \
    gdf_schools_raw['PrivateSchool'].astype('string').str.upper().str.strip()

# Define the source values interpreted as a positive private-school indicator.
private_value_labels = {'YES', 'Y', 'TRUE', '1', 'PRIVATE'}

# Retain schools not identified as private so the facility layer represents public schools.
gdf_public_schools_raw = \
    gdf_schools_raw[~private_school_normalized.isin(private_value_labels)].copy()

print(f'-> PreK-12 school candidates downloaded: {len(gdf_schools_raw):,}')

print(f'-> Schools retained as public or not marked private: {len(gdf_public_schools_raw):,}')



=== STUDY-AREA FACILITY DOWNLOADS COMPLETE ===
=== FILTERING SCHOOL DATASET TO PUBLIC SCHOOLS ===

PrivateSchool source values:


,RECORD_COUNT
PrivateSchool,
False,617
True,184


-> PreK-12 school candidates downloaded: 801
-> Schools retained as public or not marked private: 617


## Apply Exact Facility Filters


In [33]:
# ----------------------------------------------------
# EXACT FACILITY FILTER COMPLETE
# ----------------------------------------------------
# Define source-specific filters so only facility records relevant to the
# project study area are requested or retained.
print('\n=== PUBLIC SCHOOL FILTER COMPLETE ===')

print('=== APPLYING EXACT STUDY-AREA FACILITY FILTER ===')

# Store analysis crs so spatial operations use the required coordinate reference system.
analysis_crs = gdf_county_boundaries_aligned.crs

# Stop exact spatial filtering if the county layer has no defined CRS.
if analysis_crs is None:
    raise ValueError('The county boundary GeoDataFrame does not have a CRS.')

# Organize the downloaded facility layers so each category receives the same spatial filter.
facility_candidate_layers = {
    'Hospitals and Medical Facilities': gdf_medical_facilities_raw,
    'Fire Stations': gdf_fire_stations_raw,
    'Police and Sheriff Stations': gdf_law_enforcement_raw,
    'Public Schools': gdf_public_schools_raw,
    'Emergency Shelters': gdf_shelters_raw,
}

# Store the facility layers that pass the exact three-county boundary test.
facility_layers_clipped = {}

# Apply the same CRS, geometry, and exact-boundary filter to every facility category.
for facility_category, candidate_gdf in facility_candidate_layers.items():
    # Derive facility working with copy for use in the next processing or validation step.
    facility_working = candidate_gdf.copy()

    # Derive facility working with copy for use in the next processing or validation step.
    facility_working = facility_working[facility_working.geometry.notna() & \
        ~facility_working.geometry.is_empty].copy()

    # Stop processing this facility category if its source layer has no defined CRS.
    if facility_working.crs is None:
        raise ValueError(f'{facility_category} has no defined CRS.')

    # Reproject the layer to the common project CRS before spatial comparison, overlay, or distance
    # analysis.
    facility_working = facility_working.to_crs(analysis_crs)

    # Save the pre-filter facility count for comparison with the exact county-boundary result.
    candidate_count = len(facility_working)

    # Test spatial intersection so records can be filtered or classified by their relationship to
    # the study area or WUI.
    facility_working = \
        facility_working[facility_working.geometry.intersects(study_area_geometry)].copy()

    # Set facility working used by the current processing stage.
    facility_working['FACILITY_TYPE'] = facility_category

    # Reset the index after filtering or deduplication to maintain a clean sequential index.
    facility_working = facility_working.reset_index(drop=True)

    # Set facility layers clipped used by the current processing stage.
    facility_layers_clipped[facility_category] = facility_working

    print(f'-> {facility_category}: '
        f'{candidate_count:,} envelope '
        f'candidates, '
        f'{len(facility_working):,} '
        f'retained after exact '
        f'county-boundary validation.')

print('\n=== EXACT FACILITY FILTER COMPLETE ===')



=== PUBLIC SCHOOL FILTER COMPLETE ===
=== APPLYING EXACT STUDY-AREA FACILITY FILTER ===
-> Hospitals and Medical Facilities: 599 envelope candidates, 553 retained after exact county-boundary validation.
-> Fire Stations: 151 envelope candidates, 151 retained after exact county-boundary validation.
-> Police and Sheriff Stations: 100 envelope candidates, 89 retained after exact county-boundary validation.
-> Public Schools: 617 envelope candidates, 572 retained after exact county-boundary validation.
-> Emergency Shelters: 507 envelope candidates, 458 retained after exact county-boundary validation.

=== EXACT FACILITY FILTER COMPLETE ===


## Defining Critical Facility Field Mappings


In [34]:
# ----------------------------------------------------
# DEFINING CRITICAL FACILITY FIELD MAPPINGS
# ----------------------------------------------------
# Define source-field mappings so facility attributes from different services
# can be standardized to a common schema.
print('=== DEFINING CRITICAL FACILITY FIELD MAPPINGS ===')

# Map inconsistent source field names to the common facility schema used by the analysis.
facility_field_mapping = {
    'Hospitals and Medical Facilities': {'name': ['FACILITY_NAME',
        'FacilityName', 'NAME'], 'address': ['ADDRESS',
        'Address'], 'city': ['CITY', 'City'], 'county': ['COUNTY',
        'County']},
    'Fire Stations': {'name': ['NAME', 'Name'],
        'address': ['ADDRESS', 'Address'], 'city': ['CITY',
        'City'], 'county': ['COUNTY', 'County']},
    'Police and Sheriff Stations': {'name': ['name',
        'NAME', 'AgencyName', 'AGENCY'], 'address': ['address',
        'ADDRESS'], 'city': ['city', 'CITY'], 'county': ['county',
        'COUNTY']},
    'Public Schools': {'name': ['SchoolName', 'SCHOOL_NAME',
        'NAME'], 'address': ['Address', 'ADDRESS'], 'city': ['City',
        'CITY'], 'county': ['County', 'COUNTY']},
    'Emergency Shelters': {'name': ['shelter_name',
        'SHELTER_NAME', 'NAME'], 'address': ['address_1',
        'ADDRESS', 'Address'], 'city': ['city', 'CITY'],
        'county': ['county_parish', 'COUNTY', 'County']},
}

# Identify facility categories that lack a source-to-standard field mapping.
missing_mapping_categories = [facility_category for facility_category in facility_layers_clipped \
    if facility_category not in facility_field_mapping]

# Stop standardization if any downloaded facility category lacks a field mapping.
if missing_mapping_categories:
    raise ValueError(f'The following facility '
        f'categories do not have field '
        f'mappings: '
        f'{missing_mapping_categories}')

print(f'-> Field mappings defined for {len(facility_field_mapping)} facility categories.')

# Review each configured facility category after validating the mapping dictionary.
for facility_category in facility_field_mapping:
    print(f'  • {facility_category}')


=== DEFINING CRITICAL FACILITY FIELD MAPPINGS ===
-> Field mappings defined for 5 facility categories.
  • Hospitals and Medical Facilities
  • Fire Stations
  • Police and Sheriff Stations
  • Public Schools
  • Emergency Shelters


## Validate Critical-Facility Field Mappings


In [35]:
# ----------------------------------------------------
# CRITICAL FACILITY FIELD MAPPINGS COMPLETE
# ----------------------------------------------------
# Define source-field mappings so facility attributes from different services
# can be standardized to a common schema.
print('\n=== CRITICAL FACILITY FIELD MAPPINGS COMPLETE ===')

print('=== CREATING FACILITY FIELD-LOOKUP FUNCTION ===')

# Define a reusable helper so the same logic is applied consistently throughout the notebook.
def find_existing_field(dataframe, candidate_fields):
    """
    Return the first candidate field found in a DataFrame.
    Field matching is case-insensitive so source-field
    variations such as NAME, Name, and name can be handled
    consistently.
    Parameters
    ----------
    dataframe : pandas.DataFrame or geopandas.GeoDataFrame
        Dataset whose columns will be searched.
    candidate_fields : list
        Possible source-field names listed in priority order.
    Returns
    -------
    str or None
        Original column name when a match is found.
        None when none of the candidate fields exist.
    """

    # Build a case-insensitive lookup from normalized field names to their original column names.
    column_lookup = {str(column).lower(): column for column in dataframe.columns}

    # Apply this processing sequence consistently across each item in the collection.
    for candidate in candidate_fields:
        # Derive candidate lower with lower for use in the next processing or validation step.
        candidate_lower = str(candidate).lower()

        # Return the first case-insensitive source-field match in the requested priority order.
        if candidate_lower in column_lookup:
            # Return the validated result to the calling workflow.
            return column_lookup[candidate_lower]

    # Return the validated result to the calling workflow.
    return None

# Derive test facility category with next for use in the next processing or validation step.
test_facility_category = next(iter(facility_layers_clipped))

# Set test facility gdf used by the current processing stage.
test_facility_gdf = facility_layers_clipped[test_facility_category]

# Derive test name field with find_existing_field for use in the next processing or validation step.
test_name_field = find_existing_field(
    test_facility_gdf,
    facility_field_mapping[test_facility_category]['name'],
)

print(f'-> Test facility category: {test_facility_category}')

print(f'-> Detected facility-name field: {test_name_field}')



=== CRITICAL FACILITY FIELD MAPPINGS COMPLETE ===
=== CREATING FACILITY FIELD-LOOKUP FUNCTION ===
-> Test facility category: Hospitals and Medical Facilities
-> Detected facility-name field: FACILITY_NAME


## Define the Facility Field-Lookup Function


In [36]:
# ----------------------------------------------------
# FACILITY FIELD-LOOKUP FUNCTION READY
# ----------------------------------------------------
# Define a helper function that resolves equivalent source-field names across
# facility datasets with inconsistent schemas.
print('\n=== FACILITY FIELD-LOOKUP FUNCTION READY ===')

print('=== STANDARDIZING CRITICAL FACILITY ATTRIBUTES ===')

# Define standardized facility layers used to configure this workflow stage.
standardized_facility_layers = []

# Apply this processing sequence consistently across each item in the collection.
for facility_type, facility_gdf in facility_layers_clipped.items():
    print(f'\nStandardizing {facility_type}...')

    # Derive facility standardized with copy for use in the next processing or validation step.
    facility_standardized = facility_gdf.copy()

    # Set field config used by the current processing stage.
    field_config = facility_field_mapping[facility_type]

    # Derive source name field with find_existing_field for use in the next processing or validation
    # step.
    source_name_field = find_existing_field(facility_standardized, field_config['name'])

    # Derive source address field with find_existing_field for use in the next processing or
    # validation step.
    source_address_field = find_existing_field(
        facility_standardized,
        field_config['address'],
    )

    # Derive source city field with find_existing_field for use in the next processing or validation
    # step.
    source_city_field = find_existing_field(facility_standardized, field_config['city'])

    # Calculate source county field for validation and workflow QA.
    source_county_field = find_existing_field(facility_standardized, field_config['county'])

    # Populate the standardized facility name from the first available source-name field.
    if source_name_field is not None:
        # Derive facility standardized with strip for use in the next processing or validation step.
        facility_standardized['FACILITY_NAME'] = \
            facility_standardized[source_name_field].astype('string').str.strip()
    # Use the fallback path when the preferred field or geometry method is unavailable.
    else:
        # Set facility standardized used by the current processing stage.
        facility_standardized['FACILITY_NAME'] = 'Unnamed Facility'

        print('  [WARNING] No recognized facility-name field was found.')

    # Populate the standardized address when the source provides a mapped address field.
    if source_address_field is not None:
        # Derive facility standardized with strip for use in the next processing or validation step.
        facility_standardized['ADDRESS'] = \
            facility_standardized[source_address_field].astype('string').str.strip()
    # Use the fallback path when the preferred field or geometry method is unavailable.
    else:
        # Set facility standardized used by the current processing stage.
        facility_standardized['ADDRESS'] = pd.NA

        print('  [WARNING] No recognized address field was found.')

    # Populate the standardized city when the source provides a mapped city field.
    if source_city_field is not None:
        # Derive facility standardized with strip for use in the next processing or validation step.
        facility_standardized['CITY'] = \
            facility_standardized[source_city_field].astype('string').str.strip()
    # Use the fallback path when the preferred field or geometry method is unavailable.
    else:
        # Set facility standardized used by the current processing stage.
        facility_standardized['CITY'] = pd.NA

        print('  [WARNING] No recognized city field was found.')

    # Populate the standardized county attribute when the source provides a mapped county field.
    if source_county_field is not None:
        # Derive facility standardized with strip for use in the next processing or validation step.
        facility_standardized['COUNTY_NAME'] = \
            facility_standardized[source_county_field].astype('string').str.upper().str.strip()
    # Use the fallback path when the preferred field or geometry method is unavailable.
    else:
        # Set facility standardized used by the current processing stage.
        facility_standardized['COUNTY_NAME'] = pd.NA

        print('  [WARNING] No recognized county field was found.')

    # Set facility standardized used by the current processing stage.
    facility_standardized['FACILITY_TYPE'] = facility_type

    # Set facility standardized used by the current processing stage.
    facility_standardized['SOURCE_DATASET'] = facility_type

    # Derive facility standardized with astype for use in the next processing or validation step.
    facility_standardized['SOURCE_ROW_ID'] = facility_standardized.index.astype(str)

    # Build the blank name mask mask used to isolate records required for this analysis.
    blank_name_mask = facility_standardized['FACILITY_NAME'].isna() | \
        facility_standardized['FACILITY_NAME'].str.strip().eq('')

    # Set loc used by the current processing stage.
    facility_standardized.loc[blank_name_mask, 'FACILITY_NAME'] = 'Unnamed Facility'

    standardized_facility_layers.append(facility_standardized)

    print(f'-> Standardized {len(facility_standardized):,} {facility_type} records.')

    print(f'-> Name field used: {source_name_field}')

    print(f'-> Address field used: {source_address_field}')

    print(f'-> City field used: {source_city_field}')

    print(f'-> County field used: {source_county_field}')

# Define the standardized fields inputs required by this workflow stage.
required_standardized_fields = [
    'FACILITY_NAME',
    'FACILITY_TYPE',
    'ADDRESS',
    'CITY',
    'COUNTY_NAME',
    'SOURCE_DATASET',
    'SOURCE_ROW_ID',
    'geometry',
]

# Apply this processing sequence consistently across each item in the collection.
for layer_number, standardized_gdf in enumerate(standardized_facility_layers, start=1):
    # Identify missing standardized fields inputs before the workflow continues.
    missing_standardized_fields = [field for field in required_standardized_fields if field not \
        in standardized_gdf.columns]

    # Stop processing when required fields or records are missing.
    if missing_standardized_fields:
        raise ValueError(f'Standardized facility layer '
            f'{layer_number} is missing '
            f'required fields: '
            f'{missing_standardized_fields}')

# Confirm every clipped facility category produced a standardized output layer.
if len(standardized_facility_layers) != len(facility_layers_clipped):
    raise ValueError('The number of standardized '
        'facility layers does not match '
        'the number of clipped facility '
        'layers.')

print(f'''
-> Successfully standardized {len(standardized_facility_layers)} facility-category layers.''')



=== FACILITY FIELD-LOOKUP FUNCTION READY ===
=== STANDARDIZING CRITICAL FACILITY ATTRIBUTES ===

Standardizing Hospitals and Medical Facilities...
-> Standardized 553 Hospitals and Medical Facilities records.
-> Name field used: FACILITY_NAME
-> Address field used: ADDRESS
-> City field used: CITY
-> County field used: COUNTY

Standardizing Fire Stations...
-> Standardized 151 Fire Stations records.
-> Name field used: NAME
-> Address field used: ADDRESS
-> City field used: CITY
-> County field used: COUNTY

Standardizing Police and Sheriff Stations...
-> Standardized 89 Police and Sheriff Stations records.
-> Name field used: name
-> Address field used: address
-> City field used: city
-> County field used: county

Standardizing Public Schools...
  [WARNING] No recognized county field was found.
-> Standardized 572 Public Schools records.
-> Name field used: SchoolName
-> Address field used: Address
-> City field used: City
-> County field used: None

Standardizing Emergency Shelters

## Standardize Critical-Facility Attributes


In [37]:
# ----------------------------------------------------
# CRITICAL FACILITY ATTRIBUTE STANDARDIZATION COMPLETE
# ----------------------------------------------------
print('\n=== CRITICAL FACILITY ATTRIBUTE STANDARDIZATION COMPLETE ===')

print('=== COMBINING CRITICAL FACILITY CATEGORIES ===')

# Stop combination if no standardized facility layers were created.
if not standardized_facility_layers:
    raise ValueError('No standardized facility layers are available to combine.')

# Store expected facility crs so spatial operations use the required coordinate reference system.
expected_facility_crs = standardized_facility_layers[0].crs

# Stop before combining facilities if the reference analysis CRS is undefined.
if expected_facility_crs is None:
    raise ValueError('The first standardized facility layer does not have a defined CRS.')

# Apply this processing sequence consistently across each item in the collection.
for layer_number, facility_layer in enumerate(standardized_facility_layers, start=1):
    # Stop processing because the compared spatial layers must use the same CRS.
    if facility_layer.crs != expected_facility_crs:
        raise ValueError(f'Standardized facility layer '
            f'{layer_number} does not use '
            f'the expected CRS.')

print(f'-> All standardized facility layers use: {expected_facility_crs}')

# Define combined facility fields needed to keep the analysis schema consistent.
combined_facility_fields = [
    'FACILITY_NAME',
    'FACILITY_TYPE',
    'ADDRESS',
    'CITY',
    'COUNTY_NAME',
    'SOURCE_DATASET',
    'SOURCE_ROW_ID',
    'geometry',
]

# Define facility layers for combination used to configure this workflow stage.
facility_layers_for_combination = []

# Apply this processing sequence consistently across each item in the collection.
for facility_layer in standardized_facility_layers:
    facility_layers_for_combination.append(facility_layer[combined_facility_fields].copy())

# Derive critical facilities with GeoDataFrame for use in the next processing or validation step.
gdf_critical_facilities = gpd.GeoDataFrame(
    # Combine the prepared facility datasets into one table after standardizing their schemas.
    pd.concat(facility_layers_for_combination, ignore_index=True, sort=False),
    geometry='geometry',
    crs=expected_facility_crs,
)

# Combine the standardized facility categories into one critical-facilities GeoDataFrame.
gdf_critical_facilities = gdf_critical_facilities.reset_index(drop=True)

# Set critical facilities used by the current processing stage.
gdf_critical_facilities['FACILITY_ID'] = 'FAC-' + (gdf_critical_facilities.index + \
    1).astype(str).str.zfill(5)

# Define final facility column order used to configure this workflow stage.
final_facility_column_order = [
    'FACILITY_ID',
    'FACILITY_NAME',
    'FACILITY_TYPE',
    'ADDRESS',
    'CITY',
    'COUNTY_NAME',
    'SOURCE_DATASET',
    'SOURCE_ROW_ID',
    'geometry',
]

# Set critical facilities used by the current processing stage.
gdf_critical_facilities = gdf_critical_facilities[final_facility_column_order]

# Check identifier uniqueness because duplicate records can distort counts or joins.
duplicate_facility_id_count = gdf_critical_facilities['FACILITY_ID'].duplicated().sum()

# Stop the workflow if generated FACILITY_ID values are not unique.
if duplicate_facility_id_count > 0:
    raise ValueError(f'{duplicate_facility_id_count} duplicate FACILITY_ID values were created.')

# Calculate expected combined count for validation and workflow QA.
expected_combined_count = sum((len(facility_layer) for facility_layer in \
    standardized_facility_layers))

# Calculate actual combined count for validation and workflow QA.
actual_combined_count = len(gdf_critical_facilities)

# Verify that concatenating the standardized categories neither lost nor duplicated facility
# records.
if actual_combined_count != expected_combined_count:
    raise ValueError(f'Expected '
        f'{expected_combined_count:,} '
        f'combined facility records, but '
        f'created '
        f'{actual_combined_count:,}.')

print(f'-> Combined {len(standardized_facility_layers)} facility-category layers.')

print(f'-> Master critical-facility dataset contains {actual_combined_count:,} records.')

print(f'-> Duplicate FACILITY_ID values: {duplicate_facility_id_count}')

# Group records by the relevant identifier or category before calculating summary values.
facility_counts_by_type = gdf_critical_facilities.groupby(
    'FACILITY_TYPE',
    as_index=False,
# Sort the output so review and downstream comparisons remain deterministic.
).agg(FACILITY_COUNT=('FACILITY_ID', 'count')).sort_values(by='FACILITY_COUNT',
    ascending=False).reset_index(drop=True)

print('\n--- FACILITY COUNTS BY TYPE ---')

display(facility_counts_by_type)

print('\n--- COMBINED FACILITY DATA SAMPLE ---')

display(gdf_critical_facilities[['FACILITY_ID',
    'FACILITY_NAME', 'FACILITY_TYPE', 'CITY', 'COUNTY_NAME',
    'geometry']].head(5))



=== CRITICAL FACILITY ATTRIBUTE STANDARDIZATION COMPLETE ===
=== COMBINING CRITICAL FACILITY CATEGORIES ===
-> All standardized facility layers use: EPSG:26912
-> Combined 5 facility-category layers.
-> Master critical-facility dataset contains 1,823 records.
-> Duplicate FACILITY_ID values: 0

--- FACILITY COUNTS BY TYPE ---


,FACILITY_TYPE,FACILITY_COUNT
0,Public Schools,572
1,Hospitals and Medical Facilities,553
2,Emergency Shelters,458
3,Fire Stations,151
4,Police and Sheriff Stations,89



--- COMBINED FACILITY DATA SAMPLE ---


,FACILITY_ID,FACILITY_NAME,FACILITY_TYPE,CITY,COUNTY_NAME,geometry
0,FAC-00001,Abbington Manor AL I,Hospitals and Medical Facilities,Lehi,UTAH,POINT (427872.095 4471453.257)
1,FAC-00002,Abbington Manor AL II,Hospitals and Medical Facilities,Lehi,UTAH,POINT (427872.095 4471453.257)
2,FAC-00003,Abbington Manor Memory Care,Hospitals and Medical Facilities,Lehi,UTAH,POINT (426811.348 4473146.991)
3,FAC-00004,Ability Home Health,Hospitals and Medical Facilities,South Jordan,SALT LAKE,POINT (423435.4 4490242.023)
4,FAC-00005,Ability Hospice,Hospitals and Medical Facilities,South Jordan,SALT LAKE,POINT (423435.4 4490242.023)


## Assign Facilities to Counties Spatially


In [38]:
# ----------------------------------------------------
# SPATIAL FACILITY COUNTY ASSIGNMENT COMPLETE
# ----------------------------------------------------
print('\n=== CRITICAL FACILITY CATEGORY COMBINATION COMPLETE ===')

print('=== ASSIGNING FACILITY COUNTIES USING SPATIAL LOCATION ===')

# Derive critical facilities with strip for use in the next processing or validation step.
gdf_critical_facilities['SOURCE_COUNTY_NAME'] = \
    gdf_critical_facilities['COUNTY_NAME'].astype('string').str.upper().str.strip()

# Calculate facility county boundaries for validation and workflow QA.
facility_county_boundaries = gdf_county_boundaries_aligned[['NAME', 'geometry']].copy()

# Calculate facility county boundaries for validation and workflow QA.
facility_county_boundaries = facility_county_boundaries.rename(columns={'NAME': \
    'SPATIAL_COUNTY_NAME'})

# Calculate facility county boundaries for validation and workflow QA.
facility_county_boundaries['SPATIAL_COUNTY_NAME'] = \
    facility_county_boundaries['SPATIAL_COUNTY_NAME'].astype('string').str.upper().str.strip()

# Stop processing because the compared spatial layers must use the same CRS.
if gdf_critical_facilities.crs != facility_county_boundaries.crs:
    raise ValueError('The critical-facility and county-boundary layers do not use the same CRS.')

# Remove a stale spatial-join index before assigning counties to facilities.
if 'index_right' in gdf_critical_facilities.columns:
    # Derive critical facilities with drop for use in the next processing or validation step.
    gdf_critical_facilities = gdf_critical_facilities.drop(columns='index_right')

# Use a spatial join to associate facility records with the polygon layer that contains or
# intersects them.
gdf_critical_facilities = gpd.sjoin(
    gdf_critical_facilities,
    facility_county_boundaries,
    how='left',
    predicate='intersects',
)

# Set critical facilities used by the current processing stage.
gdf_critical_facilities['COUNTY_NAME'] = gdf_critical_facilities['SPATIAL_COUNTY_NAME']

# Derive critical facilities with drop for use in the next processing or validation step.
gdf_critical_facilities = gdf_critical_facilities.drop(
    columns='index_right',
    errors='ignore',
)

# Identify missing spatial county count inputs before the workflow continues.
missing_spatial_county_count = gdf_critical_facilities['COUNTY_NAME'].isna().sum()

print(f'-> Facilities without a spatial county assignment: {missing_spatial_county_count}')

# Stop processing when required fields or records are missing.
if missing_spatial_county_count > 0:
    raise ValueError(f'{missing_spatial_county_count} '
        f'facilities could not be '
        f'assigned to a study-area '
        f'county.')

# Calculate county attribute mismatch for validation and workflow QA.
county_attribute_mismatch = gdf_critical_facilities['SOURCE_COUNTY_NAME'].notna() & \
    (gdf_critical_facilities['SOURCE_COUNTY_NAME'] != gdf_critical_facilities['COUNTY_NAME'])

# Calculate county attribute mismatch count for validation and workflow QA.
county_attribute_mismatch_count = county_attribute_mismatch.sum()

print(f'-> Source and spatial county mismatches: {county_attribute_mismatch_count}')

# Flag facilities whose source county attribute disagrees with the spatially assigned county.
if county_attribute_mismatch_count > 0:
    print('\n--- SOURCE COUNTY ATTRIBUTE MISMATCHES ---')

    display(gdf_critical_facilities.loc[county_attribute_mismatch,
        ['FACILITY_ID', 'FACILITY_NAME', 'FACILITY_TYPE',
        'CITY', 'SOURCE_COUNTY_NAME', 'COUNTY_NAME']].head(5))

print('\n=== SPATIAL FACILITY COUNTY ASSIGNMENT COMPLETE ===')



=== CRITICAL FACILITY CATEGORY COMBINATION COMPLETE ===
=== ASSIGNING FACILITY COUNTIES USING SPATIAL LOCATION ===
-> Facilities without a spatial county assignment: 0
-> Source and spatial county mismatches: 4

--- SOURCE COUNTY ATTRIBUTE MISMATCHES ---


,FACILITY_ID,FACILITY_NAME,FACILITY_TYPE,CITY,SOURCE_COUNTY_NAME,COUNTY_NAME
791,FAC-00792,UINTA WASATCH CACHE NATIONAL FOREST SUPERVISOR,Police and Sheriff Stations,PROVO,UTAH,SALT LAKE
1471,FAC-01472,HOBBLE CREEK ELEMENTARY,Emergency Shelters,MAPLETON,MILLARD,UTAH
1802,FAC-01803,Mapleton Elementary,Emergency Shelters,MAPLETON,CUYAHOGA,UTAH
1819,FAC-01820,TEP-LDS Church,Emergency Shelters,Salt Lake City,SALT LAKE CITY,SALT LAKE



=== SPATIAL FACILITY COUNTY ASSIGNMENT COMPLETE ===


## Prepare WUI Layer for Facility Analysis


In [39]:
# ----------------------------------------------------
# PREPARING WUI LAYER FOR FACILITY ANALYSIS
# ----------------------------------------------------
# Prepare the facility and WUI layers for direct spatial comparison using
# valid geometry and a common projected CRS.
print('=== PREPARING WUI LAYER FOR FACILITY ANALYSIS ===')

# Create the working facility layer used for direct WUI intersection analysis.
gdf_facilities_wui_analysis = gdf_critical_facilities.copy()

# Derive wui facility analysis with copy for use in the next processing or validation step.
wui_facility_analysis = gdf_wui_aligned.copy()

# Stop WUI intersection analysis if the facility layer has no defined CRS.
if gdf_facilities_wui_analysis.crs is None:
    raise ValueError('The critical-facility dataset does not have a defined CRS.')

# Stop processing because the next spatial operation requires a defined CRS.
if wui_facility_analysis.crs is None:
    raise ValueError('The WUI dataset does not have a defined CRS.')

# Stop processing because the compared spatial layers must use the same CRS.
if gdf_facilities_wui_analysis.crs != wui_facility_analysis.crs:
    raise ValueError('The critical-facility and WUI '
        'datasets do not use the same '
        'coordinate reference system.')

print(f'-> Facility analysis CRS: {gdf_facilities_wui_analysis.crs}')

# Derive wui facility analysis with copy for use in the next processing or validation step.
wui_facility_analysis = wui_facility_analysis[wui_facility_analysis.geometry.notna() & \
    ~wui_facility_analysis.geometry.is_empty].copy()

# Check geometry validity before running the next spatial operation.
if not wui_facility_analysis.geometry.is_valid.all():
    print('-> Repairing invalid WUI geometries...')

    # Repair invalid geometry so later clipping and spatial joins can complete reliably.
    wui_facility_analysis['geometry'] = wui_facility_analysis.geometry.make_valid()

# Reset the index after filtering or deduplication to maintain a clean sequential index.
wui_facility_analysis = \
    wui_facility_analysis[wui_facility_analysis.geometry.geom_type.isin(['Polygon',
    'MultiPolygon'])].reset_index(drop=True)

# Set wui facility analysis used by the current processing stage.
wui_facility_analysis['WUI_POLYGON_ID'] = 'WUI-' + (wui_facility_analysis.index + \
    1).astype(str).str.zfill(5)

# Check identifier uniqueness because duplicate records can distort counts or joins.
duplicate_wui_id_count = wui_facility_analysis['WUI_POLYGON_ID'].duplicated().sum()

# Stop WUI analysis if duplicate polygon identifiers would inflate facility intersection counts.
if duplicate_wui_id_count > 0:
    raise ValueError(f'{duplicate_wui_id_count} duplicate WUI_POLYGON_ID values were created.')

print(f'-> Critical facilities available for analysis: {len(gdf_facilities_wui_analysis):,}')

print(f'-> WUI polygon parts available for analysis: {len(wui_facility_analysis):,}')


=== PREPARING WUI LAYER FOR FACILITY ANALYSIS ===
-> Facility analysis CRS: EPSG:26912
-> Critical facilities available for analysis: 1,823
-> WUI polygon parts available for analysis: 391


## Build the WUI Facility Analysis Layer


In [40]:
# ----------------------------------------------------
# WUI FACILITY ANALYSIS LAYER READY
# ----------------------------------------------------
print('\n=== WUI FACILITY ANALYSIS LAYER READY ===')

print('=== IDENTIFYING CRITICAL FACILITIES INSIDE THE WUI ===')

# Derive facilities wui analysis with drop for use in the next processing or validation step.
gdf_facilities_wui_analysis = gdf_facilities_wui_analysis.drop(
    columns='index_right',
    errors='ignore',
)

# Use a spatial join to associate facility records with the polygon layer that contains or
# intersects them.
facility_wui_matches = gpd.sjoin(
    gdf_facilities_wui_analysis[['FACILITY_ID', 'geometry']],
    wui_facility_analysis[['WUI_POLYGON_ID', 'geometry']],
    how='inner',
    predicate='intersects',
)

# Group records by the relevant identifier or category before calculating summary values.
facility_wui_counts = facility_wui_matches.groupby(
    'FACILITY_ID',
    as_index=False,
).agg(INTERSECTING_WUI_POLYGONS=('WUI_POLYGON_ID', 'nunique'))

# Merge the prepared attributes using the shared identifier required by this stage.
gdf_facilities_wui_analysis = gdf_facilities_wui_analysis.merge(
    facility_wui_counts,
    on='FACILITY_ID',
    how='left',
    validate='one_to_one',
)

# Derive facilities wui analysis with astype for use in the next processing or validation step.
gdf_facilities_wui_analysis['INTERSECTING_WUI_POLYGONS'] = \
    gdf_facilities_wui_analysis['INTERSECTING_WUI_POLYGONS'].fillna(0).astype(int)

# Set facilities wui analysis used by the current processing stage.
gdf_facilities_wui_analysis['IS_INSIDE_WUI'] = \
    gdf_facilities_wui_analysis['INTERSECTING_WUI_POLYGONS'] > 0

# Derive facilities wui analysis with where for use in the next processing or validation step.
gdf_facilities_wui_analysis['WUI_EXPOSURE_STATUS'] = \
    np.where(gdf_facilities_wui_analysis['IS_INSIDE_WUI'],
    'Inside High-Risk WUI', 'Outside High-Risk WUI')

# Reset the index after filtering or deduplication to maintain a clean sequential index.
gdf_facilities_wui = \
    gdf_facilities_wui_analysis[gdf_facilities_wui_analysis['IS_INSIDE_WUI']].copy().reset_index \
    (drop=True)

print(f'-> Facilities inside the high-risk WUI: {len(gdf_facilities_wui):,}')

# Calculate outside wui count for validation and workflow QA.
outside_wui_count = (~gdf_facilities_wui_analysis['IS_INSIDE_WUI']).sum()

print(f'-> Facilities outside the high-risk WUI: {outside_wui_count:,}')



=== WUI FACILITY ANALYSIS LAYER READY ===
=== IDENTIFYING CRITICAL FACILITIES INSIDE THE WUI ===
-> Facilities inside the high-risk WUI: 27
-> Facilities outside the high-risk WUI: 1,796


## Classify Critical-Facility WUI Exposure


In [41]:
# ----------------------------------------------------
# CRITICAL FACILITY WUI CLASSIFICATION COMPLETE
# ----------------------------------------------------
print('\n=== CRITICAL FACILITY WUI CLASSIFICATION COMPLETE ===')

print('=== PREPARING WUI GEOMETRY FOR DISTANCE ANALYSIS ===')

# Create the working facility layer used for projected distance-to-WUI calculations.
gdf_facility_proximity = gdf_facilities_wui_analysis.copy()

# Create a separate WUI layer for geometry cleanup and distance calculations.
wui_distance_layer = wui_facility_analysis.copy()

# Stop distance analysis if the facility layer has no defined CRS.
if gdf_facility_proximity.crs is None:
    raise ValueError('The critical-facility analysis layer does not have a defined CRS.')

# Stop distance analysis if the WUI layer has no defined CRS.
if wui_distance_layer.crs is None:
    raise ValueError('The WUI distance layer does not have a defined CRS.')

# Require facility and WUI layers to share the same projected CRS before measuring distance.
if gdf_facility_proximity.crs != wui_distance_layer.crs:
    raise ValueError('The critical-facility and WUI layers do not use the same CRS.')

# Require a projected CRS because facility-to-WUI distance is measured in linear map units.
if not gdf_facility_proximity.crs.is_projected:
    raise ValueError('The facility proximity analysis requires a projected CRS.')

print(f'-> Distance-analysis CRS: {gdf_facility_proximity.crs}')

# Create a separate WUI layer for geometry cleanup and distance calculations.
wui_distance_layer = wui_distance_layer[wui_distance_layer.geometry.notna() & \
    ~wui_distance_layer.geometry.is_empty].copy()

# Repair invalid WUI polygons before dissolving them for distance calculations.
if not wui_distance_layer.geometry.is_valid.all():
    print('-> Repairing invalid WUI geometries...')

    # Repair invalid geometry so later clipping and spatial joins can complete reliably.
    wui_distance_layer['geometry'] = wui_distance_layer.geometry.make_valid()

# Create a separate WUI layer for geometry cleanup and distance calculations.
wui_distance_layer = wui_distance_layer[wui_distance_layer.geometry.geom_type.isin(['Polygon',
    'MultiPolygon'])].reset_index(drop=True)

# Stop distance analysis when no usable WUI polygons remain after geometry cleanup.
if wui_distance_layer.empty:
    raise ValueError('No valid WUI polygons are available for facility distance analysis.')

# Dissolve the WUI polygons with the current GeoPandas union API.
combined_wui_geometry = wui_distance_layer.geometry.union_all()

# Repair the dissolved WUI geometry before measuring facility distances.
if not combined_wui_geometry.is_valid:
    # Use a zero-distance buffer as a compatibility fallback for repairing invalid geometry.
    combined_wui_geometry = combined_wui_geometry.buffer(0)

# Stop distance analysis if dissolving the WUI layer produces no usable geometry.
if combined_wui_geometry is None or combined_wui_geometry.is_empty:
    raise ValueError('The combined WUI geometry is empty.')

print(f'-> Facilities available for proximity analysis: {len(gdf_facility_proximity):,}')

print(f'-> WUI polygon parts used: {len(wui_distance_layer):,}')

print(f'-> Combined WUI geometry type: {combined_wui_geometry.geom_type}')



=== CRITICAL FACILITY WUI CLASSIFICATION COMPLETE ===
=== PREPARING WUI GEOMETRY FOR DISTANCE ANALYSIS ===
-> Distance-analysis CRS: EPSG:26912
-> Facilities available for proximity analysis: 1,823
-> WUI polygon parts used: 391
-> Combined WUI geometry type: MultiPolygon


## Calculate Facility Distance and Proximity


In [42]:
# ----------------------------------------------------
# FACILITY DISTANCE AND PROXIMITY CALCULATION COMPLETE
# ----------------------------------------------------
print('\n=== WUI DISTANCE GEOMETRY READY ===')

print('=== CALCULATING FACILITY DISTANCE AND PROXIMITY TO WUI ===')

# Set meters per mile used by the current processing stage.
meters_per_mile = 1609.344

# Set one mile m used by the current processing stage.
one_mile_m = meters_per_mile

# Set five miles m used by the current processing stage.
five_miles_m = 5 * meters_per_mile

# Calculate projected distance so facility proximity to the WUI is measured in consistent map units.
gdf_facility_proximity['DISTANCE_TO_WUI_M'] = \
    gdf_facility_proximity.geometry.distance(combined_wui_geometry)

# Set facility proximity used by the current processing stage.
gdf_facility_proximity['DISTANCE_TO_WUI_MI'] = gdf_facility_proximity['DISTANCE_TO_WUI_M'] / \
    meters_per_mile

# Set loc used by the current processing stage.
gdf_facility_proximity.loc[
    gdf_facility_proximity['IS_INSIDE_WUI'],
    ['DISTANCE_TO_WUI_M', 'DISTANCE_TO_WUI_MI'],
] = 0.0

# Define mutually exclusive distance conditions from highest to lowest WUI exposure.
proximity_conditions = [
    gdf_facility_proximity['IS_INSIDE_WUI'],
    ~gdf_facility_proximity['IS_INSIDE_WUI'] & (gdf_facility_proximity['DISTANCE_TO_WUI_M'] <= \
        one_mile_m),
    (gdf_facility_proximity['DISTANCE_TO_WUI_M'] > one_mile_m) & \
        (gdf_facility_proximity['DISTANCE_TO_WUI_M'] <= five_miles_m),
    gdf_facility_proximity['DISTANCE_TO_WUI_M'] > five_miles_m,
]

# Pair each distance condition with the proximity class reported in the final dataset.
proximity_labels = [
    'Inside High-Risk WUI',
    'Within 1 Mile of WUI',
    'More Than 1 Mile and Within 5 Miles',
    'More Than 5 Miles from WUI',
]

# Assign the facility proximity class from the ordered distance conditions.
gdf_facility_proximity['WUI_PROXIMITY_CLASS'] = np.select(proximity_conditions,
    proximity_labels, default='Unclassified')

# Derive facility proximity with isin for use in the next processing or validation step.
gdf_facility_proximity['REQUIRES_MITIGATION_REVIEW'] = \
    gdf_facility_proximity['WUI_PROXIMITY_CLASS'].isin(['Inside High-Risk WUI',
    'Within 1 Mile of WUI'])

# Derive facility proximity with round for use in the next processing or validation step.
gdf_facility_proximity['DISTANCE_TO_WUI_M'] = gdf_facility_proximity['DISTANCE_TO_WUI_M'].round(1)

# Derive facility proximity with round for use in the next processing or validation step.
gdf_facility_proximity['DISTANCE_TO_WUI_MI'] = gdf_facility_proximity['DISTANCE_TO_WUI_MI'].round(2)

# Calculate inside wui count for validation and workflow QA.
inside_wui_count = gdf_facility_proximity['IS_INSIDE_WUI'].sum()

# Calculate within one mile count for validation and workflow QA.
within_one_mile_count = \
    gdf_facility_proximity['WUI_PROXIMITY_CLASS'].eq('Within 1 Mile of WUI').sum()

# Calculate one to five mile count for validation and workflow QA.
one_to_five_mile_count = \
    gdf_facility_proximity['WUI_PROXIMITY_CLASS'].eq('More Than 1 Mile and Within 5 Miles').sum()

# Calculate more than five mile count for validation and workflow QA.
more_than_five_mile_count = \
    gdf_facility_proximity['WUI_PROXIMITY_CLASS'].eq('More Than 5 Miles from WUI').sum()

print(f'-> Facilities inside the WUI: {inside_wui_count:,}')

print(f'-> Facilities outside but within 1 mile: {within_one_mile_count:,}')

print(f'-> Facilities more than 1 mile and within 5 miles: {one_to_five_mile_count:,}')

print(f'-> Facilities more than 5 miles away: {more_than_five_mile_count:,}')

print('\n=== FACILITY DISTANCE AND PROXIMITY CALCULATION COMPLETE ===')



=== WUI DISTANCE GEOMETRY READY ===
=== CALCULATING FACILITY DISTANCE AND PROXIMITY TO WUI ===
-> Facilities inside the WUI: 27
-> Facilities outside but within 1 mile: 178
-> Facilities more than 1 mile and within 5 miles: 1,074
-> Facilities more than 5 miles away: 544

=== FACILITY DISTANCE AND PROXIMITY CALCULATION COMPLETE ===


## Prepare ACS Block Group Context for Facilities


In [43]:
# ----------------------------------------------------
# PREPARING ACS BLOCK GROUP CONTEXT FOR FACILITIES
# ----------------------------------------------------
# Prepare ACS-enriched Block Groups so demographic and WUI-exposure context
# can be attached to each facility.
print('=== PREPARING ACS BLOCK GROUP CONTEXT FOR FACILITIES ===')

# Create the facility working layer that will receive ACS Block Group context.
gdf_facility_acs_context = gdf_facility_proximity.copy()

# Create a working copy of the ACS-enriched exposure layer for the facility context join.
acs_block_group_context = gdf_exposure_acs.copy()

# Define the Block Group demographic and exposure fields required for the facility context join.
required_block_group_context_fields = [
    'GEOID20',
    'COUNTY_NAME',
    'ACS_POPULATION',
    'ACS_HOUSING_UNITS',
    'EXPOSED_AREA_KM2',
    'PERCENT_WUI',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED',
    'INTERSECTING_WUI_POLYGONS',
    'geometry',
]

# Identify required ACS context fields that are absent before the spatial join.
missing_context_fields = [field for field in required_block_group_context_fields if field not in \
    acs_block_group_context.columns]

# Stop before the ACS join if required demographic or exposure fields are missing.
if missing_context_fields:
    raise ValueError(f'The ACS Block Group context '
        f'layer is missing required '
        f'fields: '
        f'{missing_context_fields}')

# Stop the ACS context join if the facility layer has no defined CRS.
if gdf_facility_acs_context.crs is None:
    raise ValueError('The facility proximity layer does not have a defined CRS.')

# Stop the ACS context join if the Block Group layer has no defined CRS.
if acs_block_group_context.crs is None:
    raise ValueError('The ACS Block Group context layer does not have a defined CRS.')

# Require facility and Block Group layers to share a CRS before the spatial join.
if gdf_facility_acs_context.crs != acs_block_group_context.crs:
    raise ValueError('The facility and ACS Block '
        'Group layers do not use the '
        'same coordinate reference '
        'system.')

# Check identifier uniqueness because duplicate records can distort counts or joins.
duplicate_context_geoids = acs_block_group_context['GEOID20'].duplicated().sum()

# Stop the ACS join if duplicate GEOID20 values would make Block Group context ambiguous.
if duplicate_context_geoids > 0:
    raise ValueError(f'{duplicate_context_geoids} '
        f'duplicate GEOID20 values exist '
        f'in the ACS context layer.')

# Derive acs block group context with rename for use in the next processing or validation step.
acs_block_group_context = acs_block_group_context.rename(columns={'COUNTY_NAME': \
    'BLOCK_GROUP_COUNTY_NAME'})

# Create a working copy of the ACS-enriched exposure layer for the facility context join.
acs_block_group_context = acs_block_group_context[['GEOID20',
    'BLOCK_GROUP_COUNTY_NAME', 'ACS_POPULATION', 'ACS_HOUSING_UNITS',
    'EXPOSED_AREA_KM2', 'PERCENT_WUI', 'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED', 'INTERSECTING_WUI_POLYGONS',
    'geometry']].copy()

print(f'-> Facilities available for ACS context join: {len(gdf_facility_acs_context):,}')

print(f'-> ACS Block Groups available: {len(acs_block_group_context):,}')

print(f'-> Spatial join CRS: {gdf_facility_acs_context.crs}')


=== PREPARING ACS BLOCK GROUP CONTEXT FOR FACILITIES ===
-> Facilities available for ACS context join: 1,823
-> ACS Block Groups available: 1,240
-> Spatial join CRS: EPSG:26912


## Prepare ACS Block-Group Context


In [44]:
# ----------------------------------------------------
# ACS BLOCK GROUP CONTEXT READY
# ----------------------------------------------------
# Prepare ACS-enriched Block Groups so demographic and WUI-exposure context
# can be attached to each facility.
print('\n=== ACS BLOCK GROUP CONTEXT READY ===')

print('=== ATTACHING ACS BLOCK GROUP CONTEXT TO FACILITIES ===')

# Save the facility count so the Block Group join can be checked for record loss or duplication.
facility_count_before_context_join = len(gdf_facility_acs_context)

# Derive facility acs context with drop for use in the next processing or validation step.
gdf_facility_acs_context = gdf_facility_acs_context.drop(
    columns='index_right',
    errors='ignore',
)

# Use a spatial join to associate facility records with the polygon layer that contains or
# intersects them.
gdf_facility_acs_context = gpd.sjoin(
    gdf_facility_acs_context,
    acs_block_group_context,
    how='left',
    predicate='within',
)

# Derive facility acs context with drop for use in the next processing or validation step.
gdf_facility_acs_context = gdf_facility_acs_context.drop(
    columns='index_right',
    errors='ignore',
)

# Count facilities not assigned to a Block Group by the primary within predicate.
unmatched_block_group_count = gdf_facility_acs_context['GEOID20'].isna().sum()

print(f"-> Facilities matched using "
    f"'within': "
    f"{facility_count_before_context_join - unmatched_block_group_count:,}")

print(f"-> Facilities not matched using 'within': {unmatched_block_group_count:,}")

# Use an intersects fallback only for facilities not matched by the primary within predicate.
if unmatched_block_group_count > 0:
    print('-> Checking unmatched facilities against Block Group boundaries...')

    # Isolate only unmatched facility points for the boundary-intersection fallback.
    unmatched_facilities = \
        gdf_facility_acs_context[gdf_facility_acs_context['GEOID20'].isna()][['FACILITY_ID',
        'geometry']].copy()

    # Use a spatial join to associate facility records with the polygon layer that contains or
    # intersects them.
    boundary_matches = gpd.sjoin(
        unmatched_facilities,
        acs_block_group_context,
        how='inner',
        predicate='intersects',
    )

    # Sort the output so review and downstream comparisons remain deterministic.
    boundary_matches = boundary_matches.sort_values(['FACILITY_ID', 'GEOID20'])

    # Derive boundary matches with drop_duplicates for use in the next processing or validation
    # step.
    boundary_matches = boundary_matches.drop_duplicates(subset='FACILITY_ID', keep='first')

    # List the Block Group attributes copied into facilities recovered by the boundary fallback.
    context_update_fields = [
        'GEOID20',
        'BLOCK_GROUP_COUNTY_NAME',
        'ACS_POPULATION',
        'ACS_HOUSING_UNITS',
        'EXPOSED_AREA_KM2',
        'PERCENT_WUI',
        'EST_ACS_POP_EXPOSED',
        'EST_ACS_HOUSING_EXPOSED',
        'INTERSECTING_WUI_POLYGONS',
    ]

    # Derive boundary matches with set_index for use in the next processing or validation step.
    boundary_matches = boundary_matches.set_index('FACILITY_ID')

    # Derive facility acs context with set_index for use in the next processing or validation step.
    gdf_facility_acs_context = gdf_facility_acs_context.set_index('FACILITY_ID')

    # Copy each recovered Block Group attribute back to the corresponding unmatched facility.
    for field in context_update_fields:
        # Set loc used by the current processing stage.
        gdf_facility_acs_context.loc[
            boundary_matches.index,
            field,
        ] = boundary_matches[field]

    # Create the facility working layer that will receive ACS Block Group context.
    gdf_facility_acs_context = gdf_facility_acs_context.reset_index()

# Recount unmatched facilities after applying the boundary-intersection fallback.
remaining_unmatched_count = gdf_facility_acs_context['GEOID20'].isna().sum()

print(f'-> Facilities still without Block Group context: {remaining_unmatched_count:,}')



=== ACS BLOCK GROUP CONTEXT READY ===
=== ATTACHING ACS BLOCK GROUP CONTEXT TO FACILITIES ===
-> Facilities matched using 'within': 1,823
-> Facilities not matched using 'within': 0
-> Facilities still without Block Group context: 0


## Join ACS Block-Group Context


In [45]:
# ----------------------------------------------------
# ACS BLOCK GROUP CONTEXT JOIN COMPLETE
# ----------------------------------------------------
# Prepare ACS-enriched Block Groups so demographic and WUI-exposure context
# can be attached to each facility.
print('\n=== ACS BLOCK GROUP CONTEXT JOIN COMPLETE ===')

# Detect the facility-level and Block Group-level WUI count fields when the join used _left/_right
# suffixes.
if 'INTERSECTING_WUI_POLYGONS_left' in gdf_facility_acs_context.columns and \
    'INTERSECTING_WUI_POLYGONS_right' in gdf_facility_acs_context.columns:
    # Set facility wui count field used by the current processing stage.
    facility_wui_count_field = 'INTERSECTING_WUI_POLYGONS_left'

    # Set block group wui count field used by the current processing stage.
    block_group_wui_count_field = 'INTERSECTING_WUI_POLYGONS_right'
# Check the alternate merge-suffix pattern used by another pandas/GeoPandas version.
elif 'INTERSECTING_WUI_POLYGONS_x' in gdf_facility_acs_context.columns and \
    'INTERSECTING_WUI_POLYGONS_y' in gdf_facility_acs_context.columns:
    # Set facility wui count field used by the current processing stage.
    facility_wui_count_field = 'INTERSECTING_WUI_POLYGONS_x'

    # Set block group wui count field used by the current processing stage.
    block_group_wui_count_field = 'INTERSECTING_WUI_POLYGONS_y'
# Check the alternate merge-suffix pattern used by another pandas/GeoPandas version.
elif 'FACILITY_INTERSECTING_WUI_POLYGONS' in gdf_facility_acs_context.columns and \
    'BLOCK_GROUP_INTERSECTING_WUI_POLYGONS' in gdf_facility_acs_context.columns:
    # Set facility wui count field used by the current processing stage.
    facility_wui_count_field = 'FACILITY_INTERSECTING_WUI_POLYGONS'

    # Set block group wui count field used by the current processing stage.
    block_group_wui_count_field = 'BLOCK_GROUP_INTERSECTING_WUI_POLYGONS'
# Use the fallback path when the preferred field or geometry method is unavailable.
else:
    raise ValueError('Unable to identify the '
        'facility-level and Block '
        'Group-level WUI intersection '
        'count fields.')

print(f'-> Facility-level WUI count field: {facility_wui_count_field}')

print(f'-> Block Group-level WUI count field: {block_group_wui_count_field}')

# Define the ordered fields retained in the final critical-facility exposure dataset.
final_facility_exposure_fields = [
    'FACILITY_ID',
    'FACILITY_NAME',
    'FACILITY_TYPE',
    'ADDRESS',
    'CITY',
    'COUNTY_NAME',
    'SOURCE_DATASET',
    'WUI_EXPOSURE_STATUS',
    'WUI_PROXIMITY_CLASS',
    'DISTANCE_TO_WUI_M',
    'DISTANCE_TO_WUI_MI',
    'REQUIRES_MITIGATION_REVIEW',
    facility_wui_count_field,
    'GEOID20',
    'BLOCK_GROUP_COUNTY_NAME',
    'ACS_POPULATION',
    'ACS_HOUSING_UNITS',
    'EXPOSED_AREA_KM2',
    'PERCENT_WUI',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED',
    block_group_wui_count_field,
    'geometry',
]

# Verify that every required final output field exists before subsetting the dataset.
missing_final_exposure_fields = [field for field in final_facility_exposure_fields if field not \
    in gdf_facility_acs_context.columns]

# Stop final dataset creation if any required facility-exposure field is missing.
if missing_final_exposure_fields:
    raise ValueError(f'The facility ACS context layer '
        f'is missing the following final '
        f'output fields: '
        f'{missing_final_exposure_fields}')

print('-> All required final facility-exposure fields are present.')

# Create the final facility-exposure GeoDataFrame from the validated output fields.
gdf_critical_facilities_exposure = gdf_facility_acs_context[final_facility_exposure_fields].copy()

# Derive critical facilities exposure with rename for use in the next processing or validation step.
gdf_critical_facilities_exposure = \
    gdf_critical_facilities_exposure.rename(columns={facility_wui_count_field: \
    'FACILITY_INTERSECTING_WUI_POLYGONS',
    block_group_wui_count_field: 'BLOCK_GROUP_INTERSECTING_WUI_POLYGONS'})

# Derive critical facilities exposure with GeoDataFrame for use in the next processing or validation
# step.
gdf_critical_facilities_exposure = gpd.GeoDataFrame(
    gdf_critical_facilities_exposure,
    geometry='geometry',
    crs=gdf_facility_acs_context.crs,
)

# Sort the output so review and downstream comparisons remain deterministic.
gdf_critical_facilities_exposure = gdf_critical_facilities_exposure.sort_values(['COUNTY_NAME',
    'FACILITY_TYPE', 'FACILITY_NAME']).reset_index(drop=True)



=== ACS BLOCK GROUP CONTEXT JOIN COMPLETE ===
-> Facility-level WUI count field: INTERSECTING_WUI_POLYGONS_left
-> Block Group-level WUI count field: INTERSECTING_WUI_POLYGONS_right
-> All required final facility-exposure fields are present.


## Verify the Final Analytical CRS


In [46]:
# ----------------------------------------------------
# FINAL ANALYTICAL CRS VERIFICATION COMPLETE
# ----------------------------------------------------
print('=== VERIFYING FINAL ANALYTICAL CRS VALUES ===')

# Collect every Phase 4 analytical GeoDataFrame for a final CRS consistency audit.
analytical_layers = {
    'Census Block Groups': gdf_census_aligned,
    'County Boundaries': gdf_county_boundaries_aligned,
    'Clipped WUI': gdf_wui_aligned,
    'ACS-Enhanced Census': gdf_census_acs,
    'ACS Exposure': gdf_exposure_acs,
    'Combined Critical Facilities': gdf_critical_facilities,
    'Facility WUI Analysis': gdf_facilities_wui_analysis,
    'Facility Proximity': gdf_facility_proximity,
    'Final Facility Exposure': gdf_critical_facilities_exposure,
}

# Store expected analysis crs so spatial operations use the required coordinate reference system.
expected_analysis_crs = 'EPSG:26912'

# Collect any analytical layers that fail the final CRS requirement.
incorrect_crs_layers = []

# Audit every analytical layer against the required project CRS.
for layer_name, layer_gdf in analytical_layers.items():
    # Store current crs so spatial operations use the required coordinate reference system.
    current_crs = layer_gdf.crs.to_string() if layer_gdf.crs is not None else None

    print(f'-> {layer_name}: {current_crs}')

    # Record this layer if it does not use the required EPSG:26912 analytical CRS.
    if current_crs != expected_analysis_crs:
        incorrect_crs_layers.append({'LAYER': layer_name, 'CRS': current_crs})

# Stop the workflow if any final analytical layer fails the CRS audit.
if incorrect_crs_layers:
    raise ValueError(f'The following analytical '
        f'layers do not use '
        f'{expected_analysis_crs}: '
        f'{incorrect_crs_layers}')

print('\n-> All final analytical layers use EPSG:26912.')

print('\nNOTE: Dashboard copies will '
    'be converted to EPSG:4326 for '
    'the Folium map to display as '
    'web-map geometry in longitude '
    'and latitude.')

print('\n=== FINAL ANALYTICAL CRS VERIFICATION COMPLETE ===')


=== VERIFYING FINAL ANALYTICAL CRS VALUES ===
-> Census Block Groups: EPSG:26912
-> County Boundaries: EPSG:26912
-> Clipped WUI: EPSG:26912
-> ACS-Enhanced Census: EPSG:26912
-> ACS Exposure: EPSG:26912
-> Combined Critical Facilities: EPSG:26912
-> Facility WUI Analysis: EPSG:26912
-> Facility Proximity: EPSG:26912
-> Final Facility Exposure: EPSG:26912

-> All final analytical layers use EPSG:26912.

NOTE: Dashboard copies will be converted to EPSG:4326 for the Folium map to display as web-map geometry in longitude and latitude.

=== FINAL ANALYTICAL CRS VERIFICATION COMPLETE ===


# WEA Phase 5 – Folium Dashboard

## Purpose

Build and export the interactive WUI exposure decision-support dashboard containing county, WUI, block-group exposure, and critical-facility layers.


## Interactive Dashboard Setup and Base Map


In [47]:
# ----------------------------------------------------
# PREPARE AND VALIDATE DASHBOARD SOURCE LAYERS
# ----------------------------------------------------
# Phase 5 uses the analytical outputs created in earlier phases; no source layers are modified here.
print('\n=== PHASE 5: INTERACTIVE FOLIUM DECISION-SUPPORT DASHBOARD ===')

print('=== DEFINING AND VALIDATING DASHBOARD FIELDS ===')

# Fields retained from the county layer for dashboard validation and mapping.
county_dashboard_fields = ['NAME', 'geometry']

# The WUI dashboard only needs polygon geometry at this preparation stage.
wui_dashboard_fields = ['geometry']

# ACS exposure attributes required for block-group symbology, popups, and QA checks.
exposure_dashboard_fields = [
    'GEOID20',
    'COUNTY_NAME',
    'ACS_POPULATION',
    'ACS_POPULATION_MOE',
    'ACS_HOUSING_UNITS',
    'ACS_HOUSING_UNITS_MOE',
    'BLOCK_GROUP_AREA_KM2',
    'EXPOSED_AREA_KM2',
    'EXPOSURE_PROPORTION',
    'PERCENT_WUI',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_POP_EXPOSED_MOE',
    'EST_ACS_HOUSING_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED_MOE',
    'INTERSECTING_WUI_POLYGONS',
    'IS_EXPOSED',
    'geometry',
]

# Critical-facility attributes required for exposure/proximity popups and mitigation review.
facility_dashboard_fields = [
    'FACILITY_ID',
    'FACILITY_NAME',
    'FACILITY_TYPE',
    'ADDRESS',
    'CITY',
    'COUNTY_NAME',
    'WUI_EXPOSURE_STATUS',
    'WUI_PROXIMITY_CLASS',
    'DISTANCE_TO_WUI_M',
    'DISTANCE_TO_WUI_MI',
    'REQUIRES_MITIGATION_REVIEW',
    'FACILITY_INTERSECTING_WUI_POLYGONS',
    'GEOID20',
    'BLOCK_GROUP_COUNTY_NAME',
    'ACS_POPULATION',
    'ACS_HOUSING_UNITS',
    'EXPOSED_AREA_KM2',
    'PERCENT_WUI',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED',
    'BLOCK_GROUP_INTERSECTING_WUI_POLYGONS',
    'geometry',
]

# Pair each analytical GeoDataFrame with the fields it must contain before dashboard preparation.
dashboard_source_layers = {
    'County Boundaries': {'data': gdf_county_boundaries_aligned,
        'required_fields': county_dashboard_fields},
    'High-Risk WUI': {'data': gdf_wui_aligned, 'required_fields': wui_dashboard_fields},
    'ACS Block Group Exposure': {'data': gdf_exposure_acs,
        'required_fields': exposure_dashboard_fields},
    'Critical Facility Exposure': {'data': gdf_critical_facilities_exposure,
        'required_fields': facility_dashboard_fields},
}

# Validate every source as a spatial layer with a defined CRS and the attributes required by the
# dashboard.
for layer_name, layer_config in dashboard_source_layers.items():
    # Set layer gdf used by the current processing stage.
    layer_gdf = layer_config['data']
    # Define the fields inputs required by this workflow stage.
    required_fields = layer_config['required_fields']

    print(f'\nValidating {layer_name}...')

    # Stop execution when isinstance fails the requirement needed for reliable analysis.
    if not isinstance(layer_gdf, gpd.GeoDataFrame):
        raise TypeError(f'{layer_name} is not a GeoDataFrame.')

    # Stop execution when layer gdf fails the requirement needed for reliable analysis.
    if layer_gdf.crs is None:
        raise ValueError(f'{layer_name} does not have a defined CRS.')

    # Identify any required fields that are absent from the current source layer.
    missing_dashboard_fields = [field for field in required_fields if field not in \
        layer_gdf.columns]

    # Stop execution when required dashboard fields inputs are unavailable.
    if missing_dashboard_fields:
        raise ValueError(f'{layer_name} is missing dashboard fields: {missing_dashboard_fields}')

    print(f'-> Records available: {len(layer_gdf):,}')

    print(f'-> Source CRS: {layer_gdf.crs}')

    print(f'-> Required dashboard fields present: {len(required_fields)}')

print('\n=== DASHBOARD FIELD DEFINITION AND VALIDATION COMPLETE ===')

print('=== CREATING CLEAN DASHBOARD LAYER COPIES ===')

# Copy only the county fields needed by the dashboard so upstream analysis remains unchanged.
gdf_dashboard_counties = gdf_county_boundaries_aligned[county_dashboard_fields].copy()

# Standardize county labels, then remove the original NAME field so the dashboard uses one county-
# name field.
gdf_dashboard_counties['COUNTY_NAME'] = \
    gdf_dashboard_counties['NAME'].astype('string').str.title().str.strip()
# Calculate dashboard counties for validation and workflow QA.
gdf_dashboard_counties = gdf_dashboard_counties.drop(columns='NAME')

# Create a dashboard-only WUI copy and reset its index for stable polygon IDs.
gdf_dashboard_wui = gdf_wui_aligned[wui_dashboard_fields].copy().reset_index(drop=True)

# Assign a stable polygon ID and calculate WUI area while the geometry is still in projected
# EPSG:26912.
gdf_dashboard_wui['WUI_POLYGON_ID'] = 'WUI-' + (gdf_dashboard_wui.index + \
    1).astype(str).str.zfill(5)
# Set dashboard wui used by the current processing stage.
gdf_dashboard_wui['WUI_AREA_M2'] = gdf_dashboard_wui.geometry.area
# Derive dashboard wui with round for use in the next processing or validation step.
gdf_dashboard_wui['WUI_AREA_KM2'] = (gdf_dashboard_wui['WUI_AREA_M2'] / 1000000).round(2)

# Copy the ACS exposure fields used by dashboard symbology and popups.
gdf_dashboard_exposure = gdf_exposure_acs[exposure_dashboard_fields].copy()

# Normalize the exposure flag and derive a readable status used in block-group popups.
gdf_dashboard_exposure['IS_EXPOSED'] = \
    gdf_dashboard_exposure['IS_EXPOSED'].fillna(False).astype(bool)
# Derive dashboard exposure with where for use in the next processing or validation step.
gdf_dashboard_exposure['EXPOSURE_STATUS'] = np.where(gdf_dashboard_exposure['IS_EXPOSED'],
    'WUI Exposure Present', 'No WUI Exposure')

# Display precision for continuous exposure metrics shown in dashboard tables and popups.
exposure_rounding_rules = {
    'BLOCK_GROUP_AREA_KM2': 2,
    'EXPOSED_AREA_KM2': 2,
    'EXPOSURE_PROPORTION': 4,
    'PERCENT_WUI': 2,
}

# Apply display precision without changing the underlying analytical source GeoDataFrame.
gdf_dashboard_exposure = gdf_dashboard_exposure.round(exposure_rounding_rules)

# Copy critical-facility exposure results into a dashboard-specific working layer.
gdf_dashboard_facilities = gdf_critical_facilities_exposure[facility_dashboard_fields].copy()

# Text attributes normalized below so popup values remain readable and consistent.
facility_text_fields = [
    'FACILITY_NAME',
    'FACILITY_TYPE',
    'ADDRESS',
    'CITY',
    'COUNTY_NAME',
    'WUI_EXPOSURE_STATUS',
    'WUI_PROXIMITY_CLASS',
]

# Strip whitespace from facility text so tooltips and popups use consistent labels.
for field in facility_text_fields:
    # Derive dashboard facilities with strip for use in the next processing or validation step.
    gdf_dashboard_facilities[field] = gdf_dashboard_facilities[field].astype('string').str.strip()

# Supply explicit address/city fallbacks instead of displaying null values in popups.
gdf_dashboard_facilities['ADDRESS'] = \
    gdf_dashboard_facilities['ADDRESS'].fillna('Address unavailable')
# Derive dashboard facilities with fillna for use in the next processing or validation step.
gdf_dashboard_facilities['CITY'] = gdf_dashboard_facilities['CITY'].fillna('City unavailable')

# Normalize the mitigation flag and derive the human-readable review status shown in facility
# popups.
gdf_dashboard_facilities['REQUIRES_MITIGATION_REVIEW'] = \
    gdf_dashboard_facilities['REQUIRES_MITIGATION_REVIEW'].fillna(False).astype(bool)
# Derive dashboard facilities with where for use in the next processing or validation step.
gdf_dashboard_facilities['MITIGATION_REVIEW_STATUS'] = \
    np.where(gdf_dashboard_facilities['REQUIRES_MITIGATION_REVIEW'],
    'Mitigation Review Recommended', 'No Immediate Review Flag')

# Display precision for facility distance and exposure metrics.
facility_rounding_rules = {
    'DISTANCE_TO_WUI_M': 1,
    'DISTANCE_TO_WUI_MI': 2,
    'EXPOSED_AREA_KM2': 2,
    'PERCENT_WUI': 2,
}

# Apply display precision to distance and exposure values used by facility popups.
gdf_dashboard_facilities = gdf_dashboard_facilities.round(facility_rounding_rules)

# Layers checked for null or empty geometries before reprojection and map rendering.
dashboard_layers_for_geometry_cleaning = {
    'County Boundaries': gdf_dashboard_counties,
    'High-Risk WUI': gdf_dashboard_wui,
    'ACS Block Group Exposure': gdf_dashboard_exposure,
    'Critical Facilities': gdf_dashboard_facilities,
}

# Remove null and empty geometries from each dashboard copy before reprojection and web rendering.
for layer_name, layer_gdf in dashboard_layers_for_geometry_cleaning.items():
    # Calculate original record count for validation and workflow QA.
    original_record_count = len(layer_gdf)

    # Retain only records with usable geometry; .copy() keeps this cleanup isolated from source
    # data.
    cleaned_layer = layer_gdf[layer_gdf.geometry.notna() & ~layer_gdf.geometry.is_empty].copy()

    # Calculate removed geometry count for validation and workflow QA.
    removed_geometry_count = original_record_count - len(cleaned_layer)

    # Branch on layer name so the workflow follows the appropriate processing path.
    if layer_name == 'County Boundaries':

        # Set dashboard counties used by the current processing stage.
        gdf_dashboard_counties = cleaned_layer
    # Branch on layer name so the workflow follows the appropriate processing path.
    elif layer_name == 'High-Risk WUI':
        # Set dashboard wui used by the current processing stage.
        gdf_dashboard_wui = cleaned_layer
    # Branch on layer name so the workflow follows the appropriate processing path.
    elif layer_name == 'ACS Block Group Exposure':
        # Set dashboard exposure used by the current processing stage.
        gdf_dashboard_exposure = cleaned_layer
    # Branch on layer name so the workflow follows the appropriate processing path.
    elif layer_name == 'Critical Facilities':
        # Set dashboard facilities used by the current processing stage.
        gdf_dashboard_facilities = cleaned_layer

    print(f'-> {layer_name}: {len(cleaned_layer):,} retained, {removed_geometry_count:,} removed.')

print('\n=== CLEAN DASHBOARD LAYER COPIES CREATED ===')

print('=== CONVERTING DASHBOARD LAYERS TO EPSG:4326 ===')

# Store dashboard crs so spatial operations use the required coordinate reference system.
dashboard_crs = 'EPSG:4326'

# Reproject dashboard copies to WGS 84 because Folium/Leaflet expects longitude-latitude web-map
# coordinates.
gdf_dashboard_counties = gdf_dashboard_counties.to_crs(dashboard_crs)
# Derive dashboard wui with to_crs for use in the next processing or validation step.
gdf_dashboard_wui = gdf_dashboard_wui.to_crs(dashboard_crs)
# Derive dashboard exposure with to_crs for use in the next processing or validation step.
gdf_dashboard_exposure = gdf_dashboard_exposure.to_crs(dashboard_crs)
# Derive dashboard facilities with to_crs for use in the next processing or validation step.
gdf_dashboard_facilities = gdf_dashboard_facilities.to_crs(dashboard_crs)

# Store cleaned, EPSG:4326 dashboard copies by layer name for common QA processing.
prepared_dashboard_layers = {
    'County Boundaries': gdf_dashboard_counties,
    'High-Risk WUI': gdf_dashboard_wui,
    'ACS Block Group Exposure': gdf_dashboard_exposure,
    'Critical Facilities': gdf_dashboard_facilities,
}

# Collect layer-level record counts and CRS values for the preparation summary table.
dashboard_layer_summary_records = []

# Summarize the prepared layers to verify record counts and CRS consistency.
for layer_name, layer_gdf in prepared_dashboard_layers.items():
    # Store current crs so spatial operations use the required coordinate reference system.
    current_crs = layer_gdf.crs.to_string() if layer_gdf.crs is not None else None

    # Count null, empty, and invalid geometries as a final spatial-integrity check after
    # reprojection.
    null_geometry_count = layer_gdf.geometry.isna().sum()
    # Calculate empty geometry count for validation and workflow QA.
    empty_geometry_count = layer_gdf.geometry.is_empty.sum()
    # Calculate invalid geometry count for validation and workflow QA.
    invalid_geometry_count = (~layer_gdf.geometry.is_valid).sum()

    print(f'\n{layer_name}:')

    print(f'-> Records: {len(layer_gdf):,}')

    print(f'-> CRS: {current_crs}')

    print(f'-> Missing geometries: {null_geometry_count}')

    print(f'-> Empty geometries: {empty_geometry_count}')

    print(f'-> Invalid geometries: {invalid_geometry_count}')

    # Stop execution when current crs fails the requirement needed for reliable analysis.
    if current_crs != dashboard_crs:
        raise ValueError(f'{layer_name} does not use {dashboard_crs}.')

    # Stop execution when null geometry count fails the requirement needed for reliable analysis.
    if null_geometry_count > 0:
        raise ValueError(f'{layer_name} contains {null_geometry_count} missing geometries.')

    # Stop execution when empty geometry count fails the requirement needed for reliable analysis.
    if empty_geometry_count > 0:
        raise ValueError(f'{layer_name} contains {empty_geometry_count} empty geometries.')

    # Stop execution when invalid geometry count fails the requirement needed for reliable analysis.
    if invalid_geometry_count > 0:
        raise ValueError(f'{layer_name} contains {invalid_geometry_count} invalid geometries.')

    dashboard_layer_summary_records.append({'DASHBOARD_LAYER': layer_name,
        'RECORD_COUNT': len(layer_gdf), 'CRS': current_crs,
        'NULL_GEOMETRIES': null_geometry_count, 'EMPTY_GEOMETRIES': empty_geometry_count,
        'INVALID_GEOMETRIES': invalid_geometry_count})

# Create dashboard layer summary so the workflow can report and validate the resulting metrics.
dashboard_layer_summary = pd.DataFrame(dashboard_layer_summary_records)

print('\n--- DASHBOARD LAYER PREPARATION SUMMARY ---')

# Display the preparation summary so record counts, CRS, and geometry QA can be reviewed before
# mapping.
display(dashboard_layer_summary)

# Derive the map extent from the prepared county boundaries rather than hard-coding coordinates.
dashboard_min_x, dashboard_min_y, dashboard_max_x, dashboard_max_y = \
    gdf_dashboard_counties.total_bounds

# Convert GeoPandas bounds order (x/y) to Folium bounds order ([lat, lon]).
dashboard_study_area_bounds = [
    [dashboard_min_y, dashboard_min_x],
    [dashboard_max_y, dashboard_max_x],
]

# Calculate the midpoint of the three-county extent for the initial Folium map center.
dashboard_center_lat = (dashboard_min_y + dashboard_max_y) / 2
# Set dashboard center lon used by the current processing stage.
dashboard_center_lon = (dashboard_min_x + dashboard_max_x) / 2
# Define dashboard center used to configure this workflow stage.
dashboard_center = [dashboard_center_lat, dashboard_center_lon]

print(f'\n-> Dashboard center: {dashboard_center}')

print(f'-> Dashboard study-area bounds: {dashboard_study_area_bounds}')

print('\nNOTE:')

print('The original analytical layers '
    'remain in EPSG:26912. Only the '
    'dashboard copies were '
    'converted to EPSG:4326 for '
    'Folium and Leaflet web-map '
    'rendering.')

print('\n=== DASHBOARD LAYER PREPARATION COMPLETE ===')

print('=== CREATING THE FOLIUM DECISION-SUPPORT MAP ===')

# Initialize an empty Folium map; custom basemaps are added explicitly below.
wui_dashboard_map = folium.Map(
    location=dashboard_center,
    zoom_start=7,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
    width='100%',
    height='100%',
)

# Add Esri World Topographic as the default basemap shown when the dashboard opens.
esri_topographic_basemap = folium.TileLayer(
    tiles='https://' \
        'server.arcgisonline.com/ArcGIS/' \
        'rest/services/World_Topo_Map/' \
        'MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri',
    name='Esri World Topographic',
    overlay=False,
    control=True,
    show=True,
    max_zoom=19,
)

# Add the configured map element to the Folium dashboard.
esri_topographic_basemap.add_to(wui_dashboard_map)




# Add Esri imagery for aerial/satellite context.
esri_imagery_basemap = folium.TileLayer(
    tiles='https://' \
        'server.arcgisonline.com/ArcGIS/' \
        'rest/services/World_Imagery/' \
        'MapServer/tile/{z}/{y}/{x}',
    attr='Imagery © Esri',
    name='Esri World Imagery',
    overlay=False,
    control=True,
    show=False,
    max_zoom=19,
)

# Add the configured map element to the Folium dashboard.
esri_imagery_basemap.add_to(wui_dashboard_map)

# Track expected basemap names so the map configuration can be verified after creation.
configured_basemap_names = [
    'Esri World Topographic',
    'Esri World Imagery',
]

print('-> Main Folium map created.')

print('-> Default basemap: Esri World Topographic.')

print('-> Optional basemap: Esri World Imagery.')

print(f'-> Selectable basemaps configured: {len(configured_basemap_names)}')

print('\nNOTE:')

print('The Layer Control added during '
    'the next step will display the '
    'basemaps as mutually exclusive '
    'radio-button options.')

print('\n=== FOLIUM MAP AND BASEMAP CREATION COMPLETE ===')

print('=== FITTING THE MAP TO THE THREE-COUNTY STUDY AREA ===')

# Fit the map extent to the study-area bounds so all project features are visible on initial
# display.
wui_dashboard_map.fit_bounds(
    dashboard_study_area_bounds,
    # Reserve upper-left map space when fitting bounds so controls do not obscure the study area.
    padding_top_left=[20, 20],
    # Reserve lower-right map space when fitting bounds so controls and legend do not obscure data.
    padding_bottom_right=[20, 20],
)

# Confirm that map creation and basemap attachment completed as expected.
if not isinstance(wui_dashboard_map, folium.Map):
    raise TypeError('wui_dashboard_map was not created as a Folium Map.')

# Calculate basemap layer count for validation and workflow QA.
basemap_layer_count = sum((isinstance(child, folium.raster_layers.TileLayer) for child in \
    wui_dashboard_map._children.values()))

print(f'-> Folium map object type: {type(wui_dashboard_map).__name__}')

print(f'-> Basemap layers added: {basemap_layer_count}')

print(f'-> Initial dashboard center: {dashboard_center}')

print(f'-> Study-area bounds applied: {dashboard_study_area_bounds}')

# Require exactly the two publication basemaps configured above.
if basemap_layer_count != len(configured_basemap_names):
    raise ValueError(
        f'Expected {len(configured_basemap_names)} Esri basemaps but found '
        f'{basemap_layer_count}.'
    )

print('\nNOTE:')

print('Only the basemap is displayed. '
    'The map is initially fitted to '
    'all three counties rather than '
    'centered on only one county. '
    'Users will still be able to '
    'zoom and pan normally after '
    'the dashboard loads.')

print('\n=== FOLIUM MAP CREATION AND VALIDATION COMPLETE ===')



=== PHASE 5: INTERACTIVE FOLIUM DECISION-SUPPORT DASHBOARD ===
=== DEFINING AND VALIDATING DASHBOARD FIELDS ===

Validating County Boundaries...
-> Records available: 3
-> Source CRS: EPSG:26912
-> Required dashboard fields present: 2

Validating High-Risk WUI...
-> Records available: 391
-> Source CRS: EPSG:26912
-> Required dashboard fields present: 1

Validating ACS Block Group Exposure...
-> Records available: 1,240
-> Source CRS: EPSG:26912
-> Required dashboard fields present: 17

Validating Critical Facility Exposure...
-> Records available: 1,823
-> Source CRS: EPSG:26912
-> Required dashboard fields present: 22

=== DASHBOARD FIELD DEFINITION AND VALIDATION COMPLETE ===
=== CREATING CLEAN DASHBOARD LAYER COPIES ===
-> County Boundaries: 3 retained, 0 removed.
-> High-Risk WUI: 391 retained, 0 removed.
-> ACS Block Group Exposure: 1,240 retained, 0 removed.
-> Critical Facilities: 1,823 retained, 0 removed.

=== CLEAN DASHBOARD LAYER COPIES CREATED ===
=== CONVERTING DASHBOARD

,DASHBOARD_LAYER,RECORD_COUNT,CRS,NULL_GEOMETRIES,EMPTY_GEOMETRIES,INVALID_GEOMETRIES
0,County Boundaries,3,EPSG:4326,0,0,0
1,High-Risk WUI,391,EPSG:4326,0,0,0
2,ACS Block Group Exposure,1240,EPSG:4326,0,0,0
3,Critical Facilities,1823,EPSG:4326,0,0,0



-> Dashboard center: [np.float64(38.96091422349325), np.float64(-112.45527739679102)]
-> Dashboard study-area bounds: [[np.float64(36.9999623124971), np.float64(-114.05289681150204)], [np.float64(40.921866134489406), np.float64(-110.85765798208)]]

NOTE:
The original analytical layers remain in EPSG:26912. Only the dashboard copies were converted to EPSG:4326 for Folium and Leaflet web-map rendering.

=== DASHBOARD LAYER PREPARATION COMPLETE ===
=== CREATING THE FOLIUM DECISION-SUPPORT MAP ===
-> Main Folium map created.
-> Default basemap: Esri World Topographic.
-> Optional basemap: Esri World Imagery.
-> Selectable basemaps configured: 2

NOTE:
The Layer Control added during the next step will display the basemaps as mutually exclusive radio-button options.

=== FOLIUM MAP AND BASEMAP CREATION COMPLETE ===
=== FITTING THE MAP TO THE THREE-COUNTY STUDY AREA ===
-> Folium map object type: Map
-> Basemap layers added: 2
-> Initial dashboard center: [np.float64(38.96091422349325), np.f

## County and WUI Dashboard Layers


In [48]:
# ----------------------------------------------------
# BUILD COUNTY BOUNDARY AND HIGH-RISK WUI LAYERS
# ----------------------------------------------------
# These layers provide geographic context for the exposure and critical-facility overlays added
# later.
print('=== CREATING INDIVIDUAL COUNTY BOUNDARY LAYERS ===')

# Work from the prepared county layer without modifying the source dashboard dataset.
gdf_county_boundaries_map = gdf_dashboard_counties.copy()

# Accept either normalized COUNTY_NAME or the original NAME field so the cell remains compatible
# with prior outputs.
if 'COUNTY_NAME' in gdf_county_boundaries_map.columns:
    # Set county name field used by the current processing stage.
    county_name_field = 'COUNTY_NAME'
# Branch on county boundaries map so the workflow follows the appropriate processing path.
elif 'NAME' in gdf_county_boundaries_map.columns:
    # Set county name field used by the current processing stage.
    county_name_field = 'NAME'
else:
    raise KeyError(f"The dashboard county layer "
        f"does not contain 'COUNTY_NAME' "
        f"or 'NAME'. Available fields: "
        f"{gdf_county_boundaries_map.columns.tolist()}")

print(f'-> County-name field used: {county_name_field}')

# Normalize county names to uppercase so they match the project county keys.
gdf_county_boundaries_map['COUNTY_NAME'] = \
    gdf_county_boundaries_map[county_name_field].astype('string').str.upper().str.strip()

# Limit the county-boundary controls to the three project counties.
expected_dashboard_counties = ['SALT LAKE', 'UTAH', 'WASHINGTON']

# Compare the counties present in the prepared layer against the expected three-county study area.
returned_dashboard_counties = \
    sorted(gdf_county_boundaries_map['COUNTY_NAME'].dropna().unique().tolist())

print(f'-> Counties available: {returned_dashboard_counties}')

# Confirm that all three study counties survived dashboard preparation.
missing_dashboard_counties = [county_name for county_name in expected_dashboard_counties if \
    county_name not in returned_dashboard_counties]

# Stop execution when required dashboard counties inputs are unavailable.
if missing_dashboard_counties:
    raise ValueError(f'The dashboard county layer is '
        f'missing the following '
        f'counties: '
        f'{missing_dashboard_counties}')

# County boundaries must already be in WGS 84 before being serialized to Folium GeoJSON.
county_boundary_map_crs = gdf_county_boundaries_map.crs.to_string() if \
    gdf_county_boundaries_map.crs is not None else None

# Stop execution when county boundary map crs fails the requirement needed for reliable analysis.
if county_boundary_map_crs != 'EPSG:4326':
    raise ValueError('The county dashboard layer must use EPSG:4326.')

print(f'-> County boundary CRS: {county_boundary_map_crs}')

# Human-readable county names used in the layer control and popups.
county_boundary_labels = {
    'SALT LAKE': 'Salt Lake County',
    'UTAH': 'Utah County',
    'WASHINGTON': 'Washington County',
}

# Use one neutral outline color so county context remains subordinate to analytical layers.
county_boundary_colors = {
    'SALT LAKE': '#374151',
    'UTAH': '#374151',
    'WASHINGTON': '#374151',
}

# Return the default outline-only style for each county polygon.
def individual_county_boundary_style(feature):
    """
    Style an individual county boundary polygon.
    """

    # Calculate county name for validation and workflow QA.
    county_name = feature['properties'].get('COUNTY_NAME', '')

    # Calculate county color for validation and workflow QA.
    county_color = county_boundary_colors.get(county_name, '#1f2937')

    return {
        'fillColor': 'transparent',
        'fillOpacity': 0.0,
        'color': county_color,
        'weight': 2.0,
        'opacity': 1.0,
    }

# Increase line weight and add light fill when a county is hovered.
def individual_county_boundary_highlight(feature):
    """
    Highlight an individual county boundary polygon.
    """

    # Calculate county name for validation and workflow QA.
    county_name = feature['properties'].get('COUNTY_NAME', '')

    # Calculate county color for validation and workflow QA.
    county_color = county_boundary_colors.get(county_name, '#111827')

    return {
        'fillColor': county_color,
        'fillOpacity': 0.1,
        'color': county_color,
        'weight': 5.0,
        'opacity': 1.0,
    }

# Keep references to county FeatureGroups for later validation and layer-control checks.
county_boundary_groups = {}

# Keep references to the GeoJson objects added for each county.
county_boundary_geojson_layers = {}

# Record the number of boundary features rendered for each county.
county_boundary_feature_counts = {}

# Build a separate toggleable boundary layer for each study county.
for county_name in expected_dashboard_counties:
    # Set county label used by the current processing stage.
    county_label = county_boundary_labels[county_name]

    print(f'\nCreating boundary layer for {county_label}...')

    # Isolate one county geometry for its own toggleable Folium layer.
    county_layer = \
        gdf_county_boundaries_map[gdf_county_boundaries_map['COUNTY_NAME'].eq(county_name)][ \
        ['COUNTY_NAME',
        'geometry']].copy()

    # Store the readable county name used by both the tooltip and popup.
    county_layer['COUNTY_DISPLAY_NAME'] = county_label

    # Calculate county feature count for validation and workflow QA.
    county_feature_count = len(county_layer)

    # Stop execution when county feature count fails the requirement needed for reliable analysis.
    if county_feature_count == 0:
        raise ValueError(f'No boundary geometry was found for {county_label}.')

    # Give each county its own FeatureGroup so users can toggle individual boundaries in Layer
    # Control.
    county_feature_group = folium.FeatureGroup(
        name=f'{county_label} Boundary',
        overlay=True,
        control=True,
        show=True,
    )

    # Render the county polygon as GeoJSON with hover highlighting and a simple county label.
    county_geojson_layer = folium.GeoJson(
        data=county_layer.__geo_interface__,
        name=f'{county_label} Boundary',
        style_function=individual_county_boundary_style,
        highlight_function=individual_county_boundary_highlight,
        # Show the county name on hover and again in the click popup.
        tooltip=folium.GeoJsonTooltip(fields=['COUNTY_DISPLAY_NAME'],
            aliases=['County:'], localize=True, sticky=False,
            labels=True, style='background-color: white; '
            'color: #111827; font-family: '
            'Arial; font-size: 12px; '
            'padding: 6px;'),
        popup=folium.GeoJsonPopup(fields=['COUNTY_DISPLAY_NAME'],
            aliases=['County:'], localize=True, labels=True,
            style='background-color: white; '
            'font-family: Arial; font-size: '
            '12px;'),
    )

    # Add the configured map element to the Folium dashboard.
    county_geojson_layer.add_to(county_feature_group)

    # Add the configured map element to the Folium dashboard.
    county_feature_group.add_to(wui_dashboard_map)

    # Set county boundary groups used by the current processing stage.
    county_boundary_groups[county_name] = county_feature_group

    # Set county boundary geojson layers used by the current processing stage.
    county_boundary_geojson_layers[county_name] = county_geojson_layer

    # Set county boundary feature counts used by the current processing stage.
    county_boundary_feature_counts[county_name] = county_feature_count

    print(f'-> Boundary records added: {county_feature_count:,}')

    print(f'-> Layer Control name: {county_label} Boundary')

print('\n=== INDIVIDUAL COUNTY BOUNDARY LAYERS CREATED ===')

print('=== ATTACHING COUNTY CONTEXT TO WUI POLYGONS ===')

# Work from dashboard copies while attaching county names to WUI polygons for display.
wui_county_context = gdf_dashboard_wui.copy()
# Calculate county context source for validation and workflow QA.
county_context_source = gdf_dashboard_counties.copy()

# Branch on county context source so the workflow follows the appropriate processing path.
if 'COUNTY_NAME' in county_context_source.columns:
    # Set county name source field used by the current processing stage.
    county_name_source_field = 'COUNTY_NAME'
# Branch on county context source so the workflow follows the appropriate processing path.
elif 'NAME' in county_context_source.columns:
    # Set county name source field used by the current processing stage.
    county_name_source_field = 'NAME'
else:
    raise KeyError(f"The dashboard county layer "
        f"does not contain either "
        f"'COUNTY_NAME' or 'NAME'. "
        f"Available fields: "
        f"{county_context_source.columns.tolist()}")

print(f'-> County-name source field: {county_name_source_field}')

# Normalize county names and retain only the county label plus geometry needed by the spatial join.
county_context_source['WUI_COUNTY_NAME'] = \
    county_context_source[county_name_source_field].astype('string').str.strip().str.title()
# Calculate county context layer for validation and workflow QA.
county_context_layer = county_context_source[['WUI_COUNTY_NAME', 'geometry']].copy()

# Both join inputs must use the same WGS 84 CRS for reliable spatial matching.
if wui_county_context.crs is None or wui_county_context.crs.to_string() != 'EPSG:4326':
    raise ValueError('The WUI dashboard layer must use EPSG:4326.')

# Stop execution when county context layer fails the requirement needed for reliable analysis.
if county_context_layer.crs is None or county_context_layer.crs.to_string() != 'EPSG:4326':
    raise ValueError('The county dashboard layer must use EPSG:4326.')

# Fields needed to assign each WUI polygon to a county for popup display.
required_wui_county_fields = ['WUI_POLYGON_ID', 'geometry']

# Verify the county-tagging source contains the fields used by the spatial join.
missing_wui_county_fields = [field for field in required_wui_county_fields if field not in \
    wui_county_context.columns]

# Stop execution when required wui county fields inputs are unavailable.
if missing_wui_county_fields:
    raise ValueError(f'The WUI dashboard layer is missing fields: {missing_wui_county_fields}')

# Remove fields created by an earlier run so the spatial join can be rerun without column conflicts.
wui_county_context = wui_county_context.drop(
    columns=['index_right', 'COUNTY_NAME', 'WUI_COUNTY_NAME'],
    errors='ignore',
)

# Spatially join WUI polygons to counties using intersection so cross-boundary polygons retain all
# matching counties.
wui_county_matches = gpd.sjoin(
    wui_county_context[['WUI_POLYGON_ID', 'geometry']],
    county_context_layer,
    how='left',
    predicate='intersects',
)

# Collapse one-to-many spatial-join matches to one record per WUI polygon, listing multiple counties
# when needed.
wui_county_names = wui_county_matches.groupby(
    'WUI_POLYGON_ID',
    as_index=False,
).agg(COUNTY_NAME=('WUI_COUNTY_NAME', lambda values: \
    ', '.join(sorted(set(values.dropna().astype(str))))))

# Merge the aggregated county labels back to the original WUI geometry by stable polygon ID.
gdf_dashboard_wui = wui_county_context.merge(
    wui_county_names,
    on='WUI_POLYGON_ID',
    how='left',
    validate='one_to_one',
)

# Derive dashboard wui with GeoDataFrame for use in the next processing or validation step.
gdf_dashboard_wui = gpd.GeoDataFrame(
    gdf_dashboard_wui,
    geometry='geometry',
    crs=wui_county_context.crs,
)

# Replace unmatched county labels with an explicit popup fallback rather than a blank value.
gdf_dashboard_wui['COUNTY_NAME'] = gdf_dashboard_wui['COUNTY_NAME'].astype('string').replace('',
    pd.NA).fillna('County unavailable')

# Identify missing wui county count inputs before the workflow continues.
missing_wui_county_count = gdf_dashboard_wui['COUNTY_NAME'].eq('County unavailable').sum()

print(f'-> WUI polygons processed: {len(gdf_dashboard_wui):,}')

print(f'-> WUI polygons without county context: {missing_wui_county_count:,}')

# Stop execution when required wui county count inputs are unavailable.
if missing_wui_county_count == len(gdf_dashboard_wui):
    raise ValueError('No WUI polygons received county context.')

print('\n=== WUI COUNTY CONTEXT COMPLETE ===')

print('=== CREATING HIGH-RISK WUI MAP LAYER ===')

# Place all High-Risk WUI polygons in one toggleable FeatureGroup.
wui_feature_group = folium.FeatureGroup(
    name='High-Risk WUI',
    overlay=True,
    control=True,
    show=True,
)

# Use a violet outline and minimal fill so WUI remains distinct from hazard classes.
def wui_polygon_style(feature):
    """
    Return the normal map style for WUI polygons.
    """

    return {
        'fillColor': '#8b5cf6',
        'fillOpacity': 0.10,
        'color': '#6d28d9',
        'weight': 1.8,
        'opacity': 0.9,
    }

# Emphasize a WUI polygon on hover without changing its classification.
def wui_polygon_highlight(feature):
    """
    Return the mouse-over highlight style for WUI polygons.
    """

    return {
        'fillColor': '#a78bfa',
        'fillOpacity': 0.24,
        'color': '#4c1d95',
        'weight': 2.5,
        'opacity': 1.0,
    }

print(f'-> WUI polygons available for mapping: {len(gdf_dashboard_wui):,}')

print('-> WUI FeatureGroup and polygon styles created.')

print('\n=== HIGH-RISK WUI FEATURE GROUP READY ===')

print('=== ADDING HIGH-RISK WUI POLYGONS TO THE DASHBOARD ===')

# WUI attributes required to render the layer and populate its tooltip/popup.
required_wui_map_fields = ['WUI_POLYGON_ID', 'COUNTY_NAME', 'WUI_AREA_KM2', 'geometry']

# Stop the map build if any WUI display fields are unavailable.
missing_wui_map_fields = [field for field in required_wui_map_fields if field not in \
    gdf_dashboard_wui.columns]

# Stop execution when required wui map fields inputs are unavailable.
if missing_wui_map_fields:
    raise ValueError(f'The WUI dashboard layer is '
        f'missing required fields: '
        f'{missing_wui_map_fields}')

# Calculate wui feature count for validation and workflow QA.
wui_feature_count = len(gdf_dashboard_wui)

# Stop execution when wui feature count fails the requirement needed for reliable analysis.
if wui_feature_count == 0:
    raise ValueError('The prepared WUI dashboard layer contains no records.')

# Render the prepared WUI polygons as interactive GeoJSON using the normal and hover styles above.
wui_geojson_layer = folium.GeoJson(
    data=gdf_dashboard_wui.__geo_interface__,
    name='High-Risk WUI',
    style_function=wui_polygon_style,
    highlight_function=wui_polygon_highlight,
    # Keep the tooltip concise; the popup provides the full polygon identifier, county context, and
    # area.
    tooltip=folium.GeoJsonTooltip(fields=['WUI_POLYGON_ID',
        'COUNTY_NAME', 'WUI_AREA_KM2'], aliases=['WUI Polygon:',
        'County:', 'Area (km²):'], localize=True, sticky=False,
        labels=True, style='background-color: white; '
        'color: #4c1d95; font-family: '
        'Arial; font-size: 12px; '
        'padding: 6px;'),
    popup=folium.GeoJsonPopup(fields=['WUI_POLYGON_ID',
        'COUNTY_NAME', 'WUI_AREA_KM2'], aliases=['WUI Polygon ID:',
        'Intersecting County:', 'Polygon Area (km²):'],
        localize=True, labels=True, style='background-color: white; '
        'font-family: Arial; font-size: '
        '12px;'),
)

# Add the configured map element to the Folium dashboard.
wui_geojson_layer.add_to(wui_feature_group)

# Add the configured map element to the Folium dashboard.
wui_feature_group.add_to(wui_dashboard_map)

# Set wui group added used by the current processing stage.
wui_group_added = wui_feature_group.get_name() in wui_dashboard_map._children

# Store wui dashboard crs so spatial operations use the required coordinate reference system.
wui_dashboard_crs = gdf_dashboard_wui.crs.to_string() if gdf_dashboard_wui.crs is not None else None

# Verify the rendered WUI layer still has usable geometry after all display preparation.
missing_wui_geometry_count = gdf_dashboard_wui.geometry.isna().sum()

# Calculate empty wui geometry count for validation and workflow QA.
empty_wui_geometry_count = gdf_dashboard_wui.geometry.is_empty.sum()

print(f'-> WUI polygons added: {wui_feature_count:,}')

print(f'-> WUI layer added to map: {wui_group_added}')

print(f'-> WUI dashboard CRS: {wui_dashboard_crs}')

print(f'-> Missing WUI geometries: {missing_wui_geometry_count}')

print(f'-> Empty WUI geometries: {empty_wui_geometry_count}')

# Stop execution when wui group added fails the requirement needed for reliable analysis.
if not wui_group_added:
    raise ValueError('The High-Risk WUI FeatureGroup was not added to the Folium dashboard.')

# Stop execution when wui dashboard crs fails the requirement needed for reliable analysis.
if wui_dashboard_crs != 'EPSG:4326':
    raise ValueError('The WUI dashboard layer must use EPSG:4326.')

# Stop execution when required wui geometry count inputs are unavailable.
if missing_wui_geometry_count > 0:
    raise ValueError(f'{missing_wui_geometry_count} WUI polygons have missing geometries.')

# Stop execution when empty wui geometry count fails the requirement needed for reliable analysis.
if empty_wui_geometry_count > 0:
    raise ValueError(f'{empty_wui_geometry_count} WUI polygons have empty geometries.')

print('\nNOTE:')

print('WUI polygons that intersect '
    'more than one county display '
    'all intersecting county names '
    'in their tooltip and popup.')

print('The WUI fill is partially '
    'transparent so roads, terrain, '
    'and later exposure layers '
    'remain visible.')

print('\n=== HIGH-RISK WUI LAYER COMPLETE ===')


=== CREATING INDIVIDUAL COUNTY BOUNDARY LAYERS ===
-> County-name field used: COUNTY_NAME
-> Counties available: ['SALT LAKE', 'UTAH', 'WASHINGTON']
-> County boundary CRS: EPSG:4326

Creating boundary layer for Salt Lake County...
-> Boundary records added: 1
-> Layer Control name: Salt Lake County Boundary

Creating boundary layer for Utah County...
-> Boundary records added: 1
-> Layer Control name: Utah County Boundary

Creating boundary layer for Washington County...
-> Boundary records added: 1
-> Layer Control name: Washington County Boundary

=== INDIVIDUAL COUNTY BOUNDARY LAYERS CREATED ===
=== ATTACHING COUNTY CONTEXT TO WUI POLYGONS ===
-> County-name source field: COUNTY_NAME
-> WUI polygons processed: 391
-> WUI polygons without county context: 0

=== WUI COUNTY CONTEXT COMPLETE ===
=== CREATING HIGH-RISK WUI MAP LAYER ===
-> WUI polygons available for mapping: 391
-> WUI FeatureGroup and polygon styles created.

=== HIGH-RISK WUI FEATURE GROUP READY ===
=== ADDING HIGH-RI

## ACS Exposure Dashboard Layer


In [49]:
# ----------------------------------------------------
# BUILD ACS BLOCK GROUP WUI EXPOSURE LAYER
# ----------------------------------------------------
# Classify block groups by percent WUI overlap and render the associated population/housing exposure
# estimates.
print('=== PREPARING ACS BLOCK GROUP EXPOSURE VALUES ===')

# Work from a separate ACS exposure copy so display formatting does not alter analysis values.
gdf_dashboard_exposure_map = gdf_dashboard_exposure.copy()

# Block-group fields required for exposure classification, symbology, and popup content.
required_exposure_map_fields = [
    'GEOID20',
    'COUNTY_NAME',
    'ACS_POPULATION',
    'ACS_HOUSING_UNITS',
    'BLOCK_GROUP_AREA_KM2',
    'EXPOSED_AREA_KM2',
    'PERCENT_WUI',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED',
    'INTERSECTING_WUI_POLYGONS',
    'EXPOSURE_STATUS',
    'geometry',
]

# Check that every exposure mapping field is present before classification.
missing_exposure_map_fields = [field for field in required_exposure_map_fields if field not in \
    gdf_dashboard_exposure_map.columns]

# Stop execution when required exposure map fields inputs are unavailable.
if missing_exposure_map_fields:
    raise ValueError(f'The ACS exposure dashboard '
        f'layer is missing required '
        f'fields: '
        f'{missing_exposure_map_fields}')

# Coerce the classification variable to numeric so invalid text values can be handled consistently.
gdf_dashboard_exposure_map['PERCENT_WUI'] = pd.to_numeric(gdf_dashboard_exposure_map['PERCENT_WUI'],
    errors='coerce')

# Treat missing exposure percentages as zero and constrain values to the valid 0–100% range.
gdf_dashboard_exposure_map['PERCENT_WUI'] = \
    gdf_dashboard_exposure_map['PERCENT_WUI'].fillna(0.0).clip(lower=0.0,
    upper=100.0)

# Convert exposure metrics used in popups to numeric values before rounding and formatting.
numeric_exposure_popup_fields = [
    'ACS_POPULATION',
    'ACS_HOUSING_UNITS',
    'BLOCK_GROUP_AREA_KM2',
    'EXPOSED_AREA_KM2',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED',
    'INTERSECTING_WUI_POLYGONS',
]

# Coerce popup metrics to numeric values; invalid strings become NaN for controlled cleanup.
for field in numeric_exposure_popup_fields:
    # Derive dashboard exposure map with to_numeric for use in the next processing or validation
    # step.
    gdf_dashboard_exposure_map[field] = pd.to_numeric(gdf_dashboard_exposure_map[field],
        errors='coerce')

# Replace missing numeric popup values with zero after numeric coercion.
gdf_dashboard_exposure_map[numeric_exposure_popup_fields] = \
    gdf_dashboard_exposure_map[numeric_exposure_popup_fields].fillna(0)

# Round continuous area and percent metrics to the precision shown in dashboard popups.
gdf_dashboard_exposure_map['BLOCK_GROUP_AREA_KM2'] = \
    gdf_dashboard_exposure_map['BLOCK_GROUP_AREA_KM2'].round(2)
# Derive dashboard exposure map with round for use in the next processing or validation step.
gdf_dashboard_exposure_map['EXPOSED_AREA_KM2'] = \
    gdf_dashboard_exposure_map['EXPOSED_AREA_KM2'].round(2)
# Derive dashboard exposure map with round for use in the next processing or validation step.
gdf_dashboard_exposure_map['PERCENT_WUI'] = gdf_dashboard_exposure_map['PERCENT_WUI'].round(2)

# Round estimated exposed population to whole persons for dashboard display.
gdf_dashboard_exposure_map['EST_ACS_POP_EXPOSED'] = \
    gdf_dashboard_exposure_map['EST_ACS_POP_EXPOSED'].round(0).astype(int)

# Round estimated exposed housing units to whole units for dashboard display.
gdf_dashboard_exposure_map['EST_ACS_HOUSING_EXPOSED'] = \
    gdf_dashboard_exposure_map['EST_ACS_HOUSING_EXPOSED'].round(0).astype(int)

# Store ACS population as whole persons for clean popup formatting.
gdf_dashboard_exposure_map['ACS_POPULATION'] = \
    gdf_dashboard_exposure_map['ACS_POPULATION'].round(0).astype(int)

# Store ACS housing counts as whole units for clean popup formatting.
gdf_dashboard_exposure_map['ACS_HOUSING_UNITS'] = \
    gdf_dashboard_exposure_map['ACS_HOUSING_UNITS'].round(0).astype(int)

# Percentage breakpoints used to classify block groups by WUI exposure.
exposure_class_bins = [-0.01, 0, 10, 25, 50, 75, 100]

# Descriptive labels paired with the WUI exposure percentage bins.
exposure_class_labels = [
    'No WUI Exposure',
    'Greater Than 0% to 10%',
    'Greater Than 10% to 25%',
    'Greater Than 25% to 50%',
    'Greater Than 50% to 75%',
    'Greater Than 75% to 100%',
]

# Assign each block group to one mutually exclusive WUI-percentage class using the defined
# breakpoints.
gdf_dashboard_exposure_map['WUI_EXPOSURE_CLASS'] = pd.cut(gdf_dashboard_exposure_map['PERCENT_WUI'],
    bins=exposure_class_bins, labels=exposure_class_labels,
    include_lowest=True, right=True)

# Convert categorical class labels to strings so Folium can serialize them reliably.
gdf_dashboard_exposure_map['WUI_EXPOSURE_CLASS'] = \
    gdf_dashboard_exposure_map['WUI_EXPOSURE_CLASS'].astype('string')

# Confirm every block group received a class before the classification is used for symbology.
unclassified_exposure_count = gdf_dashboard_exposure_map['WUI_EXPOSURE_CLASS'].isna().sum()

print(f'-> Block Groups available for mapping: {len(gdf_dashboard_exposure_map):,}')

print(f'-> Unclassified exposure records: {unclassified_exposure_count}')

# Stop execution when unclassified exposure count fails the requirement needed for reliable
# analysis.
if unclassified_exposure_count > 0:
    raise ValueError(f'{unclassified_exposure_count} '
        f'Block Groups were not assigned '
        f'an exposure class.')

print('\nExposure class counts:')

# Summarize the mapped categories as a QA check before rendering the dashboard.
display(gdf_dashboard_exposure_map['WUI_EXPOSURE_CLASS'].value_counts(dropna=False).rename \
    ('BLOCK_GROUP_COUNT').to_frame())

print('\n=== ACS BLOCK GROUP EXPOSURE VALUES READY ===')

print('=== CREATING ACS BLOCK GROUP EXPOSURE MAP LAYER ===')

# Store the choropleth-style block-group polygons in one toggleable operational layer.
acs_exposure_feature_group = folium.FeatureGroup(
    name='ACS Block Group WUI Exposure',
    overlay=True,
    control=True,
    show=True,
)

# Color ramp used to distinguish increasing block-group WUI exposure.
exposure_class_colors = {
    'No WUI Exposure': '#f7fbff',
    'Greater Than 0% to 10%': '#fee8c8',
    'Greater Than 10% to 25%': '#fdbb84',
    'Greater Than 25% to 50%': '#fc8d59',
    'Greater Than 50% to 75%': '#e34a33',
    'Greater Than 75% to 100%': '#b30000',
}

# Style each block group from its assigned WUI exposure class.
def exposure_polygon_style(feature):
    """
    Style Block Groups according to percent WUI exposure.
    """

    # Derive exposure class with get for use in the next processing or validation step.
    exposure_class = feature['properties'].get('WUI_EXPOSURE_CLASS', 'No WUI Exposure')

    # Derive fill color with get for use in the next processing or validation step.
    fill_color = exposure_class_colors.get(exposure_class, '#f7fbff')

    return {
        'fillColor': fill_color,
        'fillOpacity': 0.6,
        'color': '#4b5563',
        'weight': 0.6,
        'opacity': 0.75,
    }

# Increase outline emphasis when the user hovers over a block group.
def exposure_polygon_highlight(feature):
    """
    Highlight a Block Group when the cursor moves over it.
    """

    # Derive exposure class with get for use in the next processing or validation step.
    exposure_class = feature['properties'].get('WUI_EXPOSURE_CLASS', 'No WUI Exposure')

    # Derive fill color with get for use in the next processing or validation step.
    fill_color = exposure_class_colors.get(exposure_class, '#f7fbff')

    return {
        'fillColor': fill_color,
        'fillOpacity': 0.8,
        'color': '#111827',
        'weight': 2.0,
        'opacity': 1.0,
    }

print(f'-> Exposure polygons available: {len(gdf_dashboard_exposure_map):,}')

print(f'-> Exposure classes defined: {len(exposure_class_colors)}')

print('\n=== ACS EXPOSURE FEATURE GROUP READY ===')

print('=== ADDING ACS BLOCK GROUP EXPOSURE TO THE DASHBOARD ===')

# Calculate exposure feature count for validation and workflow QA.
exposure_feature_count = len(gdf_dashboard_exposure_map)

# Stop execution when exposure feature count fails the requirement needed for reliable analysis.
if exposure_feature_count == 0:
    raise ValueError('The ACS Block Group exposure layer contains no records.')

# Render classified block groups as GeoJSON with class-based fill colors and detailed exposure
# attributes.
acs_exposure_geojson = folium.GeoJson(
    data=gdf_dashboard_exposure_map.__geo_interface__,
    name='ACS Block Group WUI Exposure',
    style_function=exposure_polygon_style,
    highlight_function=exposure_polygon_highlight,
    # Use a concise hover summary; the popup retains the full ACS and WUI exposure context.
    tooltip=folium.GeoJsonTooltip(fields=['GEOID20',
        'COUNTY_NAME', 'PERCENT_WUI', 'EST_ACS_POP_EXPOSED',
        'EST_ACS_HOUSING_EXPOSED'], aliases=['Block Group GEOID:',
        'County:', 'WUI Exposure (%):', 'Estimated Population Exposed:',
        'Estimated Housing Units Exposed:'], localize=True,
        sticky=False, labels=True, style='background-color: white; '
        'color: #111827; font-family: '
        'Arial; font-size: 12px; '
        'padding: 7px;'),
    popup=folium.GeoJsonPopup(fields=['GEOID20',
        'COUNTY_NAME', 'EXPOSURE_STATUS', 'WUI_EXPOSURE_CLASS',
        'ACS_POPULATION', 'ACS_HOUSING_UNITS', 'BLOCK_GROUP_AREA_KM2',
        'EXPOSED_AREA_KM2', 'PERCENT_WUI', 'EST_ACS_POP_EXPOSED',
        'EST_ACS_HOUSING_EXPOSED', 'INTERSECTING_WUI_POLYGONS'],
        aliases=['Block Group GEOID:', 'County:', 'Exposure Status:',
        'Exposure Class:', 'ACS Population Estimate:',
        'ACS Housing Unit Estimate:', 'Block Group Area (km²):',
        'WUI-Exposed Area (km²):', 'Percent Inside WUI:',
        'Estimated Population Exposed:', 'Estimated Housing Units Exposed:',
        'Intersecting WUI Polygons:'], localize=True, labels=True,
        style='background-color: white; '
        'font-family: Arial; font-size: '
        '12px;'),
)

# Add the configured map element to the Folium dashboard.
acs_exposure_geojson.add_to(acs_exposure_feature_group)

# Add the configured map element to the Folium dashboard.
acs_exposure_feature_group.add_to(wui_dashboard_map)

# Set acs exposure group added used by the current processing stage.
acs_exposure_group_added = acs_exposure_feature_group.get_name() in wui_dashboard_map._children

# Store exposure dashboard crs so spatial operations use the required coordinate reference system.
exposure_dashboard_crs = gdf_dashboard_exposure_map.crs.to_string() if \
    gdf_dashboard_exposure_map.crs is not None else None

# Recheck geometry integrity on the exact GeoDataFrame passed to Folium.
missing_exposure_geometry_count = gdf_dashboard_exposure_map.geometry.isna().sum()
# Calculate empty exposure geometry count for validation and workflow QA.
empty_exposure_geometry_count = gdf_dashboard_exposure_map.geometry.is_empty.sum()
# Calculate invalid exposure geometry count for validation and workflow QA.
invalid_exposure_geometry_count = (~gdf_dashboard_exposure_map.geometry.is_valid).sum()

print(f'-> Block Group polygons added: {exposure_feature_count:,}')

print(f'-> Exposure layer added to map: {acs_exposure_group_added}')

print(f'-> Exposure dashboard CRS: {exposure_dashboard_crs}')

print(f'-> Missing exposure geometries: {missing_exposure_geometry_count}')

print(f'-> Empty exposure geometries: {empty_exposure_geometry_count}')

print(f'-> Invalid exposure geometries: {invalid_exposure_geometry_count}')

# Stop execution when acs exposure group added fails the requirement needed for reliable analysis.
if not acs_exposure_group_added:
    raise ValueError('The ACS exposure FeatureGroup was not added to the Folium dashboard.')

# Stop execution when exposure dashboard crs fails the requirement needed for reliable analysis.
if exposure_dashboard_crs != 'EPSG:4326':
    raise ValueError('The ACS exposure dashboard layer must use EPSG:4326.')

# Stop execution when required exposure geometry count inputs are unavailable.
if missing_exposure_geometry_count > 0:
    raise ValueError(f'{missing_exposure_geometry_count} '
        f'exposure features have missing '
        f'geometries.')

# Stop execution when empty exposure geometry count fails the requirement needed for reliable
# analysis.
if empty_exposure_geometry_count > 0:
    raise ValueError(f'{empty_exposure_geometry_count} exposure features have empty geometries.')

# Stop execution when invalid exposure geometry count fails the requirement needed for reliable
# analysis.
if invalid_exposure_geometry_count > 0:
    raise ValueError(f'{invalid_exposure_geometry_count} '
        f'exposure features have invalid '
        f'geometries.')

print('\nNOTE:')

print('The Block Group color '
    'represents the percentage of '
    'its land area intersecting the '
    'mapped high-risk WUI.')

print('The exposed population and '
    'housing values are '
    'area-weighted ACS estimates '
    'and should be interpreted as '
    'screening estimates.')

print('\n=== ACS BLOCK GROUP EXPOSURE LAYER COMPLETE ===')


=== PREPARING ACS BLOCK GROUP EXPOSURE VALUES ===
-> Block Groups available for mapping: 1,240
-> Unclassified exposure records: 0

Exposure class counts:


,BLOCK_GROUP_COUNT
WUI_EXPOSURE_CLASS,
No WUI Exposure,1129
Greater Than 0% to 10%,82
Greater Than 10% to 25%,24
Greater Than 25% to 50%,5



=== ACS BLOCK GROUP EXPOSURE VALUES READY ===
=== CREATING ACS BLOCK GROUP EXPOSURE MAP LAYER ===
-> Exposure polygons available: 1,240
-> Exposure classes defined: 6

=== ACS EXPOSURE FEATURE GROUP READY ===
=== ADDING ACS BLOCK GROUP EXPOSURE TO THE DASHBOARD ===
-> Block Group polygons added: 1,240
-> Exposure layer added to map: True
-> Exposure dashboard CRS: EPSG:4326
-> Missing exposure geometries: 0
-> Empty exposure geometries: 0
-> Invalid exposure geometries: 0

NOTE:
The Block Group color represents the percentage of its land area intersecting the mapped high-risk WUI.
The exposed population and housing values are area-weighted ACS estimates and should be interpreted as screening estimates.

=== ACS BLOCK GROUP EXPOSURE LAYER COMPLETE ===


## Critical-Facility Dashboard Markers


In [50]:
# ----------------------------------------------------
# BUILD CRITICAL FACILITY MARKER LAYERS
# ----------------------------------------------------
# Prepare facility attributes, create one toggleable layer per facility type, and attach
# exposure/proximity popups.
print('=== DEFINING CRITICAL FACILITY MARKER STYLES ===')

# Marker colors distinguish the five critical-facility categories.
facility_marker_colors = {
    'Hospitals and Medical Facilities': 'red',
    'Fire Stations': 'orange',
    'Police and Sheriff Stations': 'blue',
    'Public Schools': 'green',
    'Emergency Shelters': 'purple',
}

# Font Awesome icons identify each critical-facility category on the map.
facility_marker_icons = {
    'Hospitals and Medical Facilities': 'plus-square',
    'Fire Stations': 'fire-extinguisher',
    'Police and Sheriff Stations': 'shield',
    'Public Schools': 'graduation-cap',
    'Emergency Shelters': 'home',
}

# Labels used to keep facility names consistent between markers and the legend.
facility_legend_labels = {
    'Hospitals and Medical Facilities': 'Hospitals/Medical Facilities',
    'Fire Stations': 'Fire Stations',
    'Police and Sheriff Stations': 'Police/Sheriff Stations',
    'Public Schools': 'Public Schools',
    'Emergency Shelters': 'Emergency Shelters',
}

# Fixed facility order used for layer creation, reporting, and legend consistency.
facility_category_order = [
    'Hospitals and Medical Facilities',
    'Fire Stations',
    'Police and Sheriff Stations',
    'Public Schools',
    'Emergency Shelters',
]

print(f'-> Marker styles defined for {len(facility_category_order)} facility categories.')

print('\nFacility marker styles:')

# Report the marker style assigned to each facility category.
for facility_type in facility_category_order:
    print(f'  • {facility_type}: '
        f'color={facility_marker_colors[facility_type]},'
        f' '
        f'icon={facility_marker_icons[facility_type]}')

print('\n=== CRITICAL FACILITY MARKER STYLE DEFINITIONS COMPLETE ===')

print('=== PREPARING CRITICAL FACILITY MARKER ATTRIBUTES ===')

# Escape facility text before inserting it into custom HTML popups.
import html

# Use MarkerCluster when available, but fall back to individual markers in environments where the
# plugin cannot load.
try:
    from folium.plugins import MarkerCluster

    # Set marker clustering available used by the current processing stage.
    marker_clustering_available = True

    print('-> Folium MarkerCluster successfully loaded.')
# Handle the expected failure without leaving the workflow in an inconsistent state.
except (ImportError, OSError) as cluster_import_error:
    # Set MarkerCluster used by the current processing stage.
    MarkerCluster = None

    # Set marker clustering available used by the current processing stage.
    marker_clustering_available = False

    print('-> MarkerCluster could not be loaded from the CyberGISX environment.')

    print(f'-> Plugin import message: {cluster_import_error}')

    print('-> The dashboard will use individual facility markers instead.')

# Create a marker-specific copy of the facility layer for display cleanup and coordinate fields.
gdf_dashboard_facility_markers = gdf_dashboard_facilities.copy()

# Facility attributes required for marker placement and exposure/proximity popup content.
required_facility_marker_fields = [
    'FACILITY_ID',
    'FACILITY_NAME',
    'FACILITY_TYPE',
    'ADDRESS',
    'CITY',
    'COUNTY_NAME',
    'WUI_EXPOSURE_STATUS',
    'WUI_PROXIMITY_CLASS',
    'DISTANCE_TO_WUI_MI',
    'REQUIRES_MITIGATION_REVIEW',
    'MITIGATION_REVIEW_STATUS',
    'GEOID20',
    'PERCENT_WUI',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED',
    'geometry',
]

# Confirm all facility marker fields exist before building map points.
missing_facility_marker_fields = [field for field in required_facility_marker_fields if field not \
    in gdf_dashboard_facility_markers.columns]

# Stop execution when required facility marker fields inputs are unavailable.
if missing_facility_marker_fields:
    raise ValueError(f'The facility dashboard layer '
        f'is missing required marker '
        f'fields: '
        f'{missing_facility_marker_fields}')

print('-> All required facility marker fields are present.')

# Store facility marker crs so spatial operations use the required coordinate reference system.
facility_marker_crs = gdf_dashboard_facility_markers.crs.to_string() if \
    gdf_dashboard_facility_markers.crs is not None else None

# Stop execution when facility marker crs fails the requirement needed for reliable analysis.
if facility_marker_crs != 'EPSG:4326':
    raise ValueError('The facility marker layer must use EPSG:4326.')

print(f'-> Facility marker CRS: {facility_marker_crs}')

# Calculate facility count before marker cleaning for validation and workflow QA.
facility_count_before_marker_cleaning = len(gdf_dashboard_facility_markers)

# Remove null or empty facility geometries before calculating marker coordinates.
gdf_dashboard_facility_markers = \
    gdf_dashboard_facility_markers[gdf_dashboard_facility_markers.geometry.notna() & \
    ~gdf_dashboard_facility_markers.geometry.is_empty].copy()

# Calculate removed facility geometry count for validation and workflow QA.
removed_facility_geometry_count = facility_count_before_marker_cleaning - \
    len(gdf_dashboard_facility_markers)

print(f'-> Facility records before geometry cleaning: {facility_count_before_marker_cleaning:,}')

print(f'-> Facility records removed: {removed_facility_geometry_count:,}')

# Derive a guaranteed in-geometry point for marker placement; this also works if a facility geometry
# is not already a Point.
facility_marker_points = gdf_dashboard_facility_markers.geometry.representative_point()
# Set dashboard facility markers used by the current processing stage.
gdf_dashboard_facility_markers['MARKER_LONGITUDE'] = facility_marker_points.x
# Set dashboard facility markers used by the current processing stage.
gdf_dashboard_facility_markers['MARKER_LATITUDE'] = facility_marker_points.y

# Numeric facility metrics converted before rounding and HTML popup formatting.
facility_marker_numeric_fields = [
    'DISTANCE_TO_WUI_MI',
    'PERCENT_WUI',
    'EST_ACS_POP_EXPOSED',
    'EST_ACS_HOUSING_EXPOSED',
]

# Coerce facility exposure/proximity metrics to numeric values before display formatting.
for field in facility_marker_numeric_fields:
    # Derive dashboard facility markers with to_numeric for use in the next processing or validation
    # step.
    gdf_dashboard_facility_markers[field] = pd.to_numeric(gdf_dashboard_facility_markers[field],
        errors='coerce')

# Replace missing facility metrics with zero so popup formatting does not fail.
gdf_dashboard_facility_markers[facility_marker_numeric_fields] = \
    gdf_dashboard_facility_markers[facility_marker_numeric_fields].fillna(0)

# Match the distance and percent precision used in the facility popup.
gdf_dashboard_facility_markers['DISTANCE_TO_WUI_MI'] = \
    gdf_dashboard_facility_markers['DISTANCE_TO_WUI_MI'].round(2)
# Derive dashboard facility markers with round for use in the next processing or validation step.
gdf_dashboard_facility_markers['PERCENT_WUI'] = \
    gdf_dashboard_facility_markers['PERCENT_WUI'].round(2)

# Round estimated exposed population to whole persons for dashboard display.
gdf_dashboard_facility_markers['EST_ACS_POP_EXPOSED'] = \
    gdf_dashboard_facility_markers['EST_ACS_POP_EXPOSED'].round(0).astype(int)

# Round estimated exposed housing units to whole units for dashboard display.
gdf_dashboard_facility_markers['EST_ACS_HOUSING_EXPOSED'] = \
    gdf_dashboard_facility_markers['EST_ACS_HOUSING_EXPOSED'].round(0).astype(int)

# Fallback text prevents blank or null values from appearing in facility popups.
facility_marker_text_defaults = {
    'FACILITY_NAME': 'Unnamed Facility',
    'FACILITY_TYPE': 'Facility type unavailable',
    'ADDRESS': 'Address unavailable',
    'CITY': 'City unavailable',
    'COUNTY_NAME': 'County unavailable',
    'WUI_EXPOSURE_STATUS': 'Exposure status unavailable',
    'WUI_PROXIMITY_CLASS': 'Proximity unavailable',
    'MITIGATION_REVIEW_STATUS': 'Review status unavailable',
    'GEOID20': 'GEOID unavailable',
}

# Normalize popup text and replace missing/blank values with explicit display labels.
for field, default_value in facility_marker_text_defaults.items():
    # Derive dashboard facility markers with strip for use in the next processing or validation
    # step.
    gdf_dashboard_facility_markers[field] = \
        gdf_dashboard_facility_markers[field].astype('string').fillna(default_value).replace('',
        default_value).str.strip()

# Derive facility types in marker data with set for use in the next processing or validation step.
facility_types_in_marker_data = \
    set(gdf_dashboard_facility_markers['FACILITY_TYPE'].dropna().unique())

# Set unconfigured marker types used by the current processing stage.
unconfigured_marker_types = facility_types_in_marker_data - set(facility_category_order)

# Stop execution when unconfigured marker types fails the requirement needed for reliable analysis.
if unconfigured_marker_types:
    raise ValueError(f'The following facility types '
        f'do not have marker styles: '
        f'{sorted(unconfigured_marker_types)}')

# Count missing coordinates before marker creation so unusable records are visible in QA output.
missing_marker_latitude_count = gdf_dashboard_facility_markers['MARKER_LATITUDE'].isna().sum()
# Identify missing marker longitude count inputs before the workflow continues.
missing_marker_longitude_count = gdf_dashboard_facility_markers['MARKER_LONGITUDE'].isna().sum()

print(f'-> Facilities prepared for marker creation: {len(gdf_dashboard_facility_markers):,}')

print(f'-> Missing marker latitudes: {missing_marker_latitude_count}')

print(f'-> Missing marker longitudes: {missing_marker_longitude_count}')

print(f'-> Marker clustering available: {marker_clustering_available}')

print('\n=== CRITICAL FACILITY MARKER ATTRIBUTES READY ===')

# ----------------------------------------------------
# REMOVE PREVIOUS FACILITY FEATURE GROUPS
# ----------------------------------------------------

print('=== REMOVING PREVIOUS FACILITY MAP LAYERS ===')

# Identify the five visible facility layer names used by the dashboard.
facility_layer_names = set(facility_legend_labels.values())

# Remove facility FeatureGroups created by an earlier execution of this cell.
# Matching by visible layer name makes this rerun-safe even if a previous
# facility_feature_groups dictionary was overwritten or lost.
previous_facility_group_keys = []

for child_key, child_object in list(wui_dashboard_map._children.items()):
    child_layer_name = getattr(child_object, 'layer_name', None)

    if (
        isinstance(child_object, folium.map.FeatureGroup)
        and child_layer_name in facility_layer_names
    ):
        previous_facility_group_keys.append(child_key)

for child_key in previous_facility_group_keys:
    del wui_dashboard_map._children[child_key]

previous_facility_groups_removed = len(previous_facility_group_keys)

print(
    f'-> Previous facility category layers removed: '
    f'{previous_facility_groups_removed}'
)

print('\n=== PREVIOUS FACILITY MAP LAYER CLEANUP COMPLETE ===')

print('=== CREATING FACILITY FEATURE GROUPS AND MARKER CONTAINERS ===')

# Store one toggleable Folium FeatureGroup for each facility category.
facility_feature_groups = {}

# Store MarkerCluster objects when the Folium plugin is available.
facility_marker_clusters = {}

# Map each facility type to the object that receives its markers (cluster or FeatureGroup).
facility_marker_containers = {}

# Count source records by facility type for marker QA.
facility_records_expected_by_type = {}

# Count successfully rendered markers by facility type.
facility_markers_added_by_type = {facility_type: 0 for facility_type in facility_category_order}

# Count records skipped because their coordinates are unusable.
facility_markers_skipped_by_type = {facility_type: 0 for facility_type in facility_category_order}

# Create one FeatureGroup per facility category and optionally place a MarkerCluster inside it.
for facility_type in facility_category_order:
    print(f'\nCreating layer for {facility_type}...')

    # Calculate expected category marker count for validation and workflow QA.
    expected_category_marker_count = \
        gdf_dashboard_facility_markers['FACILITY_TYPE'].eq(facility_type).sum()

    # Build facility records expected by type used to track the records included in this processing
    # stage.
    facility_records_expected_by_type[facility_type] = expected_category_marker_count

    # Keep each facility type independently selectable in the dashboard Layer Control.
    facility_group = folium.FeatureGroup(
        name=facility_legend_labels[facility_type],
        overlay=True,
        control=True,
        show=False,
    )

    # Set facility feature groups used by the current processing stage.
    facility_feature_groups[facility_type] = facility_group

    # Attach the facility category FeatureGroup to the dashboard map so its markers
    # render and the category becomes available to Layer Control.
    facility_group.add_to(wui_dashboard_map)

    # Confirm the FeatureGroup is registered with the current map.
    facility_group_added = (
        facility_group.get_name()
        in wui_dashboard_map._children
    )

    if not facility_group_added:
        raise ValueError(
            f'The facility FeatureGroup for {facility_type} '
            f'was not added to wui_dashboard_map.'
        )

    # Cluster dense facility points when the plugin is available; otherwise markers attach directly
    # to the FeatureGroup.
    if marker_clustering_available:
        # Derive marker cluster with MarkerCluster for use in the next processing or validation
        # step.
        marker_cluster = MarkerCluster(
            name=f'{facility_legend_labels[facility_type]} Clusters',
            overlay=True,
            control=False,
            show=True,
            disableClusteringAtZoom=13,
        )

        # Add the configured map element to the Folium dashboard.
        marker_cluster.add_to(facility_group)

        # Set facility marker clusters used by the current processing stage.
        facility_marker_clusters[facility_type] = marker_cluster

        # Set facility marker containers used by the current processing stage.
        facility_marker_containers[facility_type] = marker_cluster

        # Set marker container type used by the current processing stage.
        marker_container_type = 'MarkerCluster'
    else:
        # Set facility marker clusters used by the current processing stage.
        facility_marker_clusters[facility_type] = None

        # Set facility marker containers used by the current processing stage.
        facility_marker_containers[facility_type] = facility_group

        # Set marker container type used by the current processing stage.
        marker_container_type = 'FeatureGroup'

    print(f'-> Expected facility records: {expected_category_marker_count:,}')

    print(f'-> Marker container: {marker_container_type}')

    print(f'-> Marker color: {facility_marker_colors[facility_type]}')

    print(f'-> Marker icon: {facility_marker_icons[facility_type]}')

# Verify that exactly one FeatureGroup for each facility category is attached to the map.
facility_groups_added_to_map = {
    facility_type: (
        facility_group.get_name()
        in wui_dashboard_map._children
    )
    for facility_type, facility_group
    in facility_feature_groups.items()
}

missing_facility_groups = [
    facility_type
    for facility_type, group_added
    in facility_groups_added_to_map.items()
    if not group_added
]

facility_group_count_added = sum(
    facility_groups_added_to_map.values()
)

facility_group_count_expected = len(
    facility_category_order
)

# Count facility FeatureGroups directly from the map to detect duplicates.
facility_layer_count_on_map = sum(
    1
    for child_object
    in wui_dashboard_map._children.values()
    if (
        isinstance(child_object, folium.map.FeatureGroup)
        and getattr(child_object, 'layer_name', None)
        in facility_layer_names
    )
)

print(
    f'\n-> Facility category layers attached to map: '
    f'{facility_group_count_added} of '
    f'{facility_group_count_expected}'
)

print(
    f'-> Facility FeatureGroups currently on map: '
    f'{facility_layer_count_on_map}'
)

if missing_facility_groups:
    raise ValueError(
        f'The following critical-facility FeatureGroups '
        f'were not added to wui_dashboard_map:\n'
        f'{missing_facility_groups}'
    )

if facility_layer_count_on_map != facility_group_count_expected:
    raise ValueError(
        f'The dashboard contains {facility_layer_count_on_map} '
        f'critical-facility FeatureGroups; exactly '
        f'{facility_group_count_expected} are required.'
    )

print('\n=== FACILITY FEATURE GROUPS AND MARKER CONTAINERS READY ===')

print('=== CREATING CRITICAL FACILITY MARKERS AND POPUPS ===')

# Build markers category by category so each marker is routed to the correct FeatureGroup or
# cluster.
for facility_type in facility_category_order:
    print(f'\nCreating {facility_type} markers...')

    # Isolate the current facility type without modifying the prepared marker dataset.
    category_facilities = \
        gdf_dashboard_facility_markers[gdf_dashboard_facility_markers['FACILITY_TYPE'].eq \
        (facility_type)].copy()

    # Set marker container used by the current processing stage.
    marker_container = facility_marker_containers[facility_type]

    # Calculate category marker count for validation and workflow QA.
    category_marker_count = 0

    # Calculate skipped category marker count for validation and workflow QA.
    skipped_category_marker_count = 0

    # Convert each facility record into one interactive CircleMarker.
    for _, facility in category_facilities.iterrows():
        # Set facility latitude used by the current processing stage.
        facility_latitude = facility['MARKER_LATITUDE']

        # Set facility longitude used by the current processing stage.
        facility_longitude = facility['MARKER_LONGITUDE']

        # Skip records that still lack valid map coordinates rather than creating malformed Leaflet
        # markers.
        if pd.isna(facility_latitude) or pd.isna(facility_longitude):
            # Calculate skipped category marker count for validation and workflow QA.
            skipped_category_marker_count += 1

            continue

        # HTML-escape text fields because these values are inserted directly into custom popup
        # markup.
        facility_name = html.escape(str(facility['FACILITY_NAME']))

        # Derive facility type label with escape for use in the next processing or validation step.
        facility_type_label = html.escape(str(facility_legend_labels[facility_type]))

        # Derive facility address with escape for use in the next processing or validation step.
        facility_address = html.escape(str(facility['ADDRESS']))

        # Derive facility city with escape for use in the next processing or validation step.
        facility_city = html.escape(str(facility['CITY']))

        # Calculate facility county for validation and workflow QA.
        facility_county = html.escape(str(facility['COUNTY_NAME']))

        # Derive exposure status with escape for use in the next processing or validation step.
        exposure_status = html.escape(str(facility['WUI_EXPOSURE_STATUS']))

        # Derive proximity class with escape for use in the next processing or validation step.
        proximity_class = html.escape(str(facility['WUI_PROXIMITY_CLASS']))

        # Derive mitigation status with escape for use in the next processing or validation step.
        mitigation_status = html.escape(str(facility['MITIGATION_REVIEW_STATUS']))

        # Derive block group geoid with escape for use in the next processing or validation step.
        block_group_geoid = html.escape(str(facility['GEOID20']))

        # Set facility tooltip text used by the current processing stage.
        facility_tooltip_text = f'{facility_name} | {facility_type_label}'

        # Build a structured popup containing facility identity, WUI proximity, mitigation flag, and
        # block-group exposure context.
        facility_popup_html = f"""
        <div style="
            width: 310px;
            font-family: Arial, sans-serif;
            font-size: 12px;
            line-height: 1.35;
        ">

            <h4 style="
                margin: 0 0 8px 0;
                color: #111827;
            ">
                {facility_name}
            </h4>

            <table style="
                width: 100%;
                border-collapse: collapse;
            ">

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>Facility Type:</strong>
                    </td>
                    <td>
                        {facility_type_label}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>Address:</strong>
                    </td>
                    <td>
                        {facility_address}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>City:</strong>
                    </td>
                    <td>
                        {facility_city}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>County:</strong>
                    </td>
                    <td>
                        {facility_county}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>WUI Status:</strong>
                    </td>
                    <td>
                        {exposure_status}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>WUI Proximity:</strong>
                    </td>
                    <td>
                        {proximity_class}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>Distance to WUI:</strong>
                    </td>
                    <td>
                        {facility['DISTANCE_TO_WUI_MI']:.2f} miles
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>Mitigation Review:</strong>
                    </td>
                    <td>
                        {mitigation_status}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>Block Group:</strong>
                    </td>
                    <td>
                        {block_group_geoid}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>Block Group WUI:</strong>
                    </td>
                    <td>
                        {facility['PERCENT_WUI']:.2f}%
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>
                            Estimated Population Exposed:
                        </strong>
                    </td>
                    <td>
                        {facility['EST_ACS_POP_EXPOSED']:,}
                    </td>
                </tr>

                <tr>
                    <td style="padding-right: 8px;">
                        <strong>
                            Estimated Housing Exposed:
                        </strong>
                    </td>
                    <td>
                        {facility['EST_ACS_HOUSING_EXPOSED']:,}
                    </td>
                </tr>

            </table>
        </div>
        """

        # Render the facility as a compact CircleMarker using the category-specific color.
        facility_marker = folium.CircleMarker(
            # Folium expects point coordinates in [latitude, longitude] order.
            location=[facility_latitude, facility_longitude],
            radius=4,
            color=facility_marker_colors[facility_type],
            weight=1,
            opacity=1.0,
            fill=True,
            fill_color=facility_marker_colors[facility_type],
            fill_opacity=0.8,
            # Show facility name/type on hover and the detailed HTML table on click.
            tooltip=folium.Tooltip(facility_tooltip_text, sticky=False),
            popup=folium.Popup(facility_popup_html, max_width=350),
        )

        # Add the configured map element to the Folium dashboard.
        facility_marker.add_to(marker_container)

        # Calculate category marker count for validation and workflow QA.
        category_marker_count += 1

    # Set facility markers added by type used by the current processing stage.
    facility_markers_added_by_type[facility_type] = category_marker_count

    # Set facility markers skipped by type used by the current processing stage.
    facility_markers_skipped_by_type[facility_type] = skipped_category_marker_count

    print(f'-> Markers created: {category_marker_count:,}')

    print(f'-> Markers skipped: {skipped_category_marker_count:,}')

# Derive total facility markers created with sum for use in the next processing or validation step.
total_facility_markers_created = sum(facility_markers_added_by_type.values())

# Derive total facility markers skipped with sum for use in the next processing or validation step.
total_facility_markers_skipped = sum(facility_markers_skipped_by_type.values())

# Calculate the total number of facility source records expected.
total_facility_records_expected = sum(
    facility_records_expected_by_type.values()
)

# Confirm that created plus skipped records reconcile with the prepared facility records.
facility_marker_counts_reconcile = (
    total_facility_markers_created
    + total_facility_markers_skipped
    ==
    total_facility_records_expected
)

# Reconfirm that all five facility FeatureGroups remain attached after marker creation.
facility_groups_still_attached = {
    facility_type: (
        facility_group.get_name()
        in wui_dashboard_map._children
    )
    for facility_type, facility_group
    in facility_feature_groups.items()
}

detached_facility_groups = [
    facility_type
    for facility_type, group_attached
    in facility_groups_still_attached.items()
    if not group_attached
]

# Recount facility FeatureGroups directly from the map so reruns cannot leave duplicates.
final_facility_layer_count_on_map = sum(
    1
    for child_object
    in wui_dashboard_map._children.values()
    if (
        isinstance(child_object, folium.map.FeatureGroup)
        and getattr(child_object, 'layer_name', None)
        in facility_layer_names
    )
)

print(
    f'\n-> Total facility records expected: '
    f'{total_facility_records_expected:,}'
)

print(
    f'-> Total facility markers created: '
    f'{total_facility_markers_created:,}'
)

print(
    f'-> Total facility markers skipped: '
    f'{total_facility_markers_skipped:,}'
)

print(
    f'-> Facility marker counts reconcile: '
    f'{facility_marker_counts_reconcile}'
)

print(
    f'-> Facility category layers active on map: '
    f'{sum(facility_groups_still_attached.values())} '
    f'of {len(facility_feature_groups)}'
)

print(
    f'-> Final facility FeatureGroup count on map: '
    f'{final_facility_layer_count_on_map}'
)

if not facility_marker_counts_reconcile:
    raise ValueError(
        'Critical-facility marker counts do not reconcile '
        'with the prepared facility records.'
    )

if detached_facility_groups:
    raise ValueError(
        f'The following critical-facility layers became '
        f'detached from the dashboard:\n'
        f'{detached_facility_groups}'
    )

if final_facility_layer_count_on_map != len(facility_category_order):
    raise ValueError(
        f'The final dashboard contains '
        f'{final_facility_layer_count_on_map} '
        f'critical-facility FeatureGroups; exactly '
        f'{len(facility_category_order)} are required.'
    )

print('\n=== CRITICAL FACILITY MARKER CREATION COMPLETE ===')


=== DEFINING CRITICAL FACILITY MARKER STYLES ===
-> Marker styles defined for 5 facility categories.

Facility marker styles:
  • Hospitals and Medical Facilities: color=red, icon=plus-square
  • Fire Stations: color=orange, icon=fire-extinguisher
  • Police and Sheriff Stations: color=blue, icon=shield
  • Public Schools: color=green, icon=graduation-cap
  • Emergency Shelters: color=purple, icon=home

=== CRITICAL FACILITY MARKER STYLE DEFINITIONS COMPLETE ===
=== PREPARING CRITICAL FACILITY MARKER ATTRIBUTES ===
-> Folium MarkerCluster successfully loaded.
-> All required facility marker fields are present.
-> Facility marker CRS: EPSG:4326
-> Facility records before geometry cleaning: 1,823
-> Facility records removed: 0
-> Facilities prepared for marker creation: 1,823
-> Missing marker latitudes: 0
-> Missing marker longitudes: 0
-> Marker clustering available: True

=== CRITICAL FACILITY MARKER ATTRIBUTES READY ===
=== REMOVING PREVIOUS FACILITY MAP LAYERS ===
-> Previous facili

## Dashboard Navigation Controls


In [51]:
# ----------------------------------------------------
# ADD CUSTOM DASHBOARD NAVIGATION CONTROLS
# ----------------------------------------------------
# Add custom navigation controls to the shared Folium map object. The single
# final legend and TreeLayerControl are added during Phase 13, after the
# Composite Fire Hazard layer and every other selectable layer are available.

print('=== CONFIGURING WUI DASHBOARD NAVIGATION CONTROLS ===')

# Report the selectable layers already registered with the shared Folium map.
# These layers will be exposed through the single final TreeLayerControl created
# during Phase 13.
print('\nRegistered basemaps:')

print('   • Esri World Topographic')
print('   • Esri World Imagery')

print('\nRegistered WUI operational layers:')

print('   • Salt Lake County Boundary')
print('   • Utah County Boundary')
print('   • Washington County Boundary')
print('   • High-Risk WUI')
print('   • ACS Block Group WUI Exposure')
print('   • Hospitals/Medical Facilities')
print('   • Fire Stations')
print('   • Police/Sheriff Stations')
print('   • Public Schools')
print('   • Emergency Shelters')

print(
    '\n-> Final legend and TreeLayerControl deferred to Phase 13 after all '
    'selectable dashboard layers are assembled.'
)


# ----------------------------------------------------
# REMOVE LEGACY PHASE 5 LEGEND
# ----------------------------------------------------
# The former lower-left legend duplicated layer-control information, displayed
# facility symbols while facility layers were hidden, and used a single color
# for a five-class exposure layer. Remove it so Phase 13 can create one accurate
# and compact portfolio legend.

dashboard_legend_name = 'dashboard_map_legend'

previous_dashboard_legend_found = (
    dashboard_legend_name
    in wui_dashboard_map._children
)

wui_dashboard_map._children.pop(
    dashboard_legend_name,
    None,
)

dashboard_legend_html = ''

print(
    f'-> Legacy Phase 5 legend removed: '
    f'{previous_dashboard_legend_found}'
)


# ----------------------------------------------------
# ADD CUSTOM FULLSCREEN, RESET-VIEW, AND MOUSE-POSITION CONTROLS
# ----------------------------------------------------

print(
    '=== ADDING CUSTOM FULLSCREEN, RESET VIEW, AND MOUSE POSITION CONTROLS ==='
)

# Import Branca elements used to attach custom CSS and JavaScript.
from branca.element import MacroElement, Template

# Retrieve Folium's generated JavaScript map variable so the custom Leaflet
# controls reference this specific map instance.
dashboard_map_javascript_name = (
    wui_dashboard_map.get_name()
)

# Serialize the original three-county extent for the JavaScript reset-view control.
dashboard_reset_view_bounds_json = json.dumps(
    dashboard_study_area_bounds
)

# Define the custom CSS and JavaScript used for fullscreen behavior, reset-view
# behavior, and live EPSG:4326 cursor coordinates.
custom_dashboard_controls_template = f'''
{{% macro header(this, kwargs) %}}

<style>

    /* Apply the standard Leaflet appearance to custom navigation controls. */
    .dashboard-fullscreen-control,
    .dashboard-reset-view-control {{
        background-color: white;
        border: 2px solid rgba(0, 0, 0, 0.20);
        border-radius: 4px;
        box-shadow: 0 1px 5px rgba(0, 0, 0, 0.40);
    }}

    /* Apply one button layout to fullscreen and reset-view actions. */
    .dashboard-fullscreen-button,
    .dashboard-reset-view-button {{
        width: 30px;
        height: 30px;
        display: flex;
        align-items: center;
        justify-content: center;
        background-color: white;
        color: #222222;
        font-family: Arial, sans-serif;
        font-size: 18px;
        font-weight: bold;
        text-decoration: none;
        cursor: pointer;
    }}

    /* Highlight custom navigation buttons when the cursor passes over them. */
    .dashboard-fullscreen-button:hover,
    .dashboard-reset-view-button:hover {{
        background-color: #f4f4f4;
    }}

    /* Use a restrained home symbol for the original-extent reset action. */
    .dashboard-reset-view-button {{
        font-size: 19px;
        line-height: 30px;
    }}

    /* Style the live mouse-coordinate display. */
    .dashboard-mouse-position {{
        min-width: 215px;
        padding: 5px 8px;
        background-color: rgba(255, 255, 255, 0.94);
        border: 1px solid #6b7280;
        border-radius: 6px;
        color: #222222;
        font-family: Arial, Helvetica, sans-serif;
        font-size: 11px;
        line-height: 1.3;
        text-align: center;
        box-shadow: 0 2px 7px rgba(0, 0, 0, 0.22);
    }}

    /* Make the Folium map fill the browser window while fullscreen mode is active. */
    .leaflet-container:fullscreen {{
        width: 100%;
        height: 100%;
    }}

    .leaflet-container:-webkit-full-screen {{
        width: 100%;
        height: 100%;
    }}

    .leaflet-container:-moz-full-screen {{
        width: 100%;
        height: 100%;
    }}

    .leaflet-container:-ms-fullscreen {{
        width: 100%;
        height: 100%;
    }}

</style>

{{% endmacro %}}


{{% macro script(this, kwargs) %}}

    // Reference the Leaflet map created by Folium.
    var dashboardMap = {dashboard_map_javascript_name};

    // Retrieve the HTML element containing the map.
    var dashboardMapContainer = (
        dashboardMap.getContainer()
    );


    // ------------------------------------------------
    // CREATE CUSTOM FULLSCREEN CONTROL
    // ------------------------------------------------

    // Create a Leaflet control positioned below the standard zoom controls.
    var DashboardFullscreenControl = L.Control.extend({{

        options: {{
            position: "topleft"
        }},

        onAdd: function(map) {{

            // Create the fullscreen control container.
            var fullscreenContainer = L.DomUtil.create(
                "div",
                "dashboard-fullscreen-control"
            );

            // Create the clickable fullscreen button.
            var fullscreenButton = L.DomUtil.create(
                "a",
                "dashboard-fullscreen-button",
                fullscreenContainer
            );

            // Display a simple expand symbol.
            fullscreenButton.innerHTML = "⛶";

            // Add accessible button information.
            fullscreenButton.href = "#";
            fullscreenButton.title = "Open Fullscreen";
            fullscreenButton.setAttribute(
                "aria-label",
                "Open fullscreen map"
            );

            // Prevent map movement when the control itself is clicked.
            L.DomEvent.disableClickPropagation(
                fullscreenContainer
            );

            L.DomEvent.disableScrollPropagation(
                fullscreenContainer
            );

            // Enter or exit browser fullscreen mode when the button is clicked.
            L.DomEvent.on(
                fullscreenButton,
                "click",
                function(event) {{

                    L.DomEvent.preventDefault(event);
                    L.DomEvent.stopPropagation(event);

                    // Determine whether the browser is currently displaying
                    // a fullscreen element.
                    var fullscreenElement = (
                        document.fullscreenElement
                        || document.webkitFullscreenElement
                        || document.mozFullScreenElement
                        || document.msFullscreenElement
                    );

                    if (!fullscreenElement) {{

                        // Request fullscreen mode using the method supported by the browser.
                        if (dashboardMapContainer.requestFullscreen) {{
                            dashboardMapContainer.requestFullscreen();
                        }} else if (dashboardMapContainer.webkitRequestFullscreen) {{
                            dashboardMapContainer.webkitRequestFullscreen();
                        }} else if (dashboardMapContainer.mozRequestFullScreen) {{
                            dashboardMapContainer.mozRequestFullScreen();
                        }} else if (dashboardMapContainer.msRequestFullscreen) {{
                            dashboardMapContainer.msRequestFullscreen();
                        }}

                    }} else {{

                        // Exit fullscreen mode using the method supported by the browser.
                        if (document.exitFullscreen) {{
                            document.exitFullscreen();
                        }} else if (document.webkitExitFullscreen) {{
                            document.webkitExitFullscreen();
                        }} else if (document.mozCancelFullScreen) {{
                            document.mozCancelFullScreen();
                        }} else if (document.msExitFullscreen) {{
                            document.msExitFullscreen();
                        }}
                    }}
                }}
            );

            return fullscreenContainer;
        }}
    }});

    // Add the completed fullscreen control to the map.
    dashboardMap.addControl(
        new DashboardFullscreenControl()
    );

    // ------------------------------------------------
    // CREATE RESET-VIEW CONTROL
    // ------------------------------------------------

    // Preserve the original fitted extent used when the dashboard first opens.
    var dashboardResetViewBounds = L.latLngBounds(
        {dashboard_reset_view_bounds_json}
    );

    // Create a Leaflet control positioned with the other navigation controls.
    var DashboardResetViewControl = L.Control.extend({{

        options: {{
            position: "topleft"
        }},

        onAdd: function(map) {{

            // Create the reset-view control container.
            var resetViewContainer = L.DomUtil.create(
                "div",
                "dashboard-reset-view-control"
            );

            // Create the clickable reset-view button.
            var resetViewButton = L.DomUtil.create(
                "a",
                "dashboard-reset-view-button",
                resetViewContainer
            );

            // Display a familiar home symbol for the original map extent.
            resetViewButton.innerHTML = "⌂";
            resetViewButton.href = "#";
            resetViewButton.title = "Reset Map View";
            resetViewButton.setAttribute(
                "aria-label",
                "Reset map to the three-county study area"
            );

            // Prevent the control from triggering map drag or zoom behavior.
            L.DomEvent.disableClickPropagation(
                resetViewContainer
            );

            L.DomEvent.disableScrollPropagation(
                resetViewContainer
            );

            // Restore the original three-county extent when the button is clicked.
            L.DomEvent.on(
                resetViewButton,
                "click",
                function(event) {{
                    L.DomEvent.preventDefault(event);
                    L.DomEvent.stopPropagation(event);

                    dashboardMap.fitBounds(
                        dashboardResetViewBounds,
                        {{
                            paddingTopLeft: [20, 20],
                            paddingBottomRight: [20, 20]
                        }}
                    );
                }}
            );

            return resetViewContainer;
        }}
    }});

    // Add the reset-view control beneath the existing upper-left controls.
    dashboardMap.addControl(
        new DashboardResetViewControl()
    );



    // ------------------------------------------------
    // UPDATE MAP AFTER FULLSCREEN CHANGES
    // ------------------------------------------------

    // Resize the Leaflet map after entering or leaving fullscreen mode so
    // tiles and overlays render correctly.
    function updateDashboardMapSize() {{
        window.setTimeout(
            function() {{
                dashboardMap.invalidateSize();
            }},
            200
        );
    }}

    // Listen for standard and browser-specific fullscreen change events.
    document.addEventListener("fullscreenchange", updateDashboardMapSize);
    document.addEventListener("webkitfullscreenchange", updateDashboardMapSize);
    document.addEventListener("mozfullscreenchange", updateDashboardMapSize);
    document.addEventListener("MSFullscreenChange", updateDashboardMapSize);


    // ------------------------------------------------
    // CREATE CUSTOM MOUSE POSITION CONTROL
    // ------------------------------------------------

    // Create a Leaflet control for displaying live latitude and longitude values.
    var DashboardMousePositionControl = L.Control.extend({{

        options: {{
            position: "bottomright"
        }},

        onAdd: function(map) {{
            this._container = L.DomUtil.create(
                "div",
                "dashboard-mouse-position"
            );
            this._container.innerHTML = (
                "Latitude: -- | Longitude: --"
            );
            L.DomEvent.disableClickPropagation(this._container);
            return this._container;
        }}
    }});

    // Create and add the mouse-position control object.
    var dashboardMousePositionControl = (
        new DashboardMousePositionControl()
    );

    dashboardMap.addControl(
        dashboardMousePositionControl
    );

    // Update the coordinate display whenever the cursor moves across the map.
    dashboardMap.on("mousemove", function(event) {{
        var latitude = event.latlng.lat.toFixed(5);
        var longitude = event.latlng.lng.toFixed(5);
        dashboardMousePositionControl._container.innerHTML = (
            "Latitude: " + latitude + "° | Longitude: " + longitude + "°"
        );
    }});

    // Restore placeholder text when the cursor leaves the map display.
    dashboardMap.on("mouseout", function() {{
        dashboardMousePositionControl._container.innerHTML = (
            "Latitude: -- | Longitude: --"
        );
    }});

{{% endmacro %}}
'''

# Use a stable child name so rerunning this cell replaces the previous
# custom-control template instead of attaching duplicate JavaScript.
custom_dashboard_controls_name = (
    'dashboard_custom_navigation_controls'
)

previous_custom_dashboard_controls_found = (
    custom_dashboard_controls_name
    in wui_dashboard_map._children
)

wui_dashboard_map._children.pop(
    custom_dashboard_controls_name,
    None,
)

print(
    f'-> Previous custom dashboard controls removed: '
    f'{previous_custom_dashboard_controls_found}'
)

custom_dashboard_controls = MacroElement()
custom_dashboard_controls._name = custom_dashboard_controls_name
custom_dashboard_controls._template = Template(
    custom_dashboard_controls_template
)

wui_dashboard_map.add_child(
    custom_dashboard_controls,
    name=custom_dashboard_controls_name,
)

custom_dashboard_controls_count = sum(
    child_name == custom_dashboard_controls_name
    for child_name
    in wui_dashboard_map._children
)

if custom_dashboard_controls_count != 1:
    raise ValueError(
        'The Folium map does not contain exactly one custom '
        'dashboard-control definition.'
    )

print(
    f'-> Custom dashboard control definitions: '
    f'{custom_dashboard_controls_count}'
)

print('\nNOTE:')

print(
    'Phase 13 will add one compact analytical legend and one collapsed '
    'TreeLayerControl after all map layers are available.'
)

print(
    'The fullscreen, reset-view, and mouse-position controls use custom Leaflet '
    'JavaScript and do not depend on the Folium plugins package.'
)

print(
    '\n=== WUI DASHBOARD NAVIGATION CONTROLS COMPLETE ==='
)


=== CONFIGURING WUI DASHBOARD NAVIGATION CONTROLS ===

Registered basemaps:
   • Esri World Topographic
   • Esri World Imagery

Registered WUI operational layers:
   • Salt Lake County Boundary
   • Utah County Boundary
   • Washington County Boundary
   • High-Risk WUI
   • ACS Block Group WUI Exposure
   • Hospitals/Medical Facilities
   • Fire Stations
   • Police/Sheriff Stations
   • Public Schools
   • Emergency Shelters

-> Final legend and TreeLayerControl deferred to Phase 13 after all selectable dashboard layers are assembled.
-> Legacy Phase 5 legend removed: False
=== ADDING CUSTOM FULLSCREEN, RESET VIEW, AND MOUSE POSITION CONTROLS ===
-> Previous custom dashboard controls removed: False
-> Custom dashboard control definitions: 1

NOTE:
Phase 13 will add one compact analytical legend and one collapsed TreeLayerControl after all map layers are available.
The fullscreen, reset-view, and mouse-position controls use custom Leaflet JavaScript and do not depend on the Folium pl

## Validate WUI Dashboard Assembly


In [52]:
# ----------------------------------------------------
# VALIDATE WUI DASHBOARD ASSEMBLY
# ----------------------------------------------------
# Validate the Phase 5 WUI exposure map without rendering, saving, or
# displaying the Folium object. The shared map remains unrendered so
# Phase 13 can add the Composite Fire Hazard layer, create the single
# final legend and TreeLayerControl, and export the dashboard exactly once.

print('=== VALIDATING WUI DASHBOARD ASSEMBLY ===')


# ----------------------------------------------------
# VALIDATE MAP OBJECT
# ----------------------------------------------------

if 'wui_dashboard_map' not in globals():
    raise NameError(
        'wui_dashboard_map is unavailable. '
        'Run the preceding Phase 5 map-assembly cells first.'
    )

if not isinstance(wui_dashboard_map, folium.Map):
    raise TypeError(
        'wui_dashboard_map is not a valid Folium Map.'
    )

print('-> WUI dashboard Folium map object: valid')


# ----------------------------------------------------
# VALIDATE REGISTERED MAP LAYERS
# ----------------------------------------------------

phase5_registered_layer_names = {
    getattr(child_object, 'layer_name', None)
    for child_object
    in wui_dashboard_map._children.values()
    if getattr(child_object, 'layer_name', None) is not None
}

expected_phase5_basemaps = {
    'Esri World Topographic',
    'Esri World Imagery',
}

expected_phase5_operational_layers = {
    'Salt Lake County Boundary',
    'Utah County Boundary',
    'Washington County Boundary',
    'High-Risk WUI',
    'ACS Block Group WUI Exposure',
    'Hospitals/Medical Facilities',
    'Fire Stations',
    'Police/Sheriff Stations',
    'Public Schools',
    'Emergency Shelters',
}

missing_phase5_basemaps = (
    expected_phase5_basemaps
    - phase5_registered_layer_names
)

missing_phase5_operational_layers = (
    expected_phase5_operational_layers
    - phase5_registered_layer_names
)

if missing_phase5_basemaps:
    raise ValueError(
        'The Phase 5 WUI dashboard is missing required basemaps:\n'
        f'{sorted(missing_phase5_basemaps)}'
    )

if missing_phase5_operational_layers:
    raise ValueError(
        'The Phase 5 WUI dashboard is missing required operational layers:\n'
        f'{sorted(missing_phase5_operational_layers)}'
    )

print(
    f'-> Required Phase 5 basemaps registered: '
    f'{len(expected_phase5_basemaps)}'
)

print(
    f'-> Required Phase 5 operational layers registered: '
    f'{len(expected_phase5_operational_layers)}'
)


# ----------------------------------------------------
# ENFORCE DEFERRED FINAL UI ELEMENTS
# ----------------------------------------------------

phase5_layer_control_keys = [
    child_key
    for child_key, child_object
    in wui_dashboard_map._children.items()
    if isinstance(
        child_object,
        folium.map.LayerControl,
    )
]

for child_key in phase5_layer_control_keys:
    del wui_dashboard_map._children[child_key]

phase5_layer_control_count = sum(
    isinstance(child_object, folium.map.LayerControl)
    for child_object
    in wui_dashboard_map._children.values()
)

if phase5_layer_control_count != 0:
    raise ValueError(
        'Phase 5 must not contain a LayerControl. '
        'The single final TreeLayerControl is created in Phase 13.'
    )

# Confirm that the cluttered legacy legend is absent before final integration.
dashboard_legend_name = 'dashboard_map_legend'

phase5_dashboard_legend_present = (
    dashboard_legend_name
    in wui_dashboard_map._children
)

if phase5_dashboard_legend_present:
    raise ValueError(
        'The legacy Phase 5 dashboard legend must be removed.'
    )

print('-> Phase 5 LayerControl count: 0')
print('-> Legacy Phase 5 legend present: False')


# ----------------------------------------------------
# VALIDATE CUSTOM NAVIGATION CONTROLS
# ----------------------------------------------------

custom_dashboard_controls_name = (
    'dashboard_custom_navigation_controls'
)

phase5_custom_controls_present = (
    custom_dashboard_controls_name
    in wui_dashboard_map._children
)

if not phase5_custom_controls_present:
    raise ValueError(
        'The custom fullscreen, reset-view, and mouse-position controls '
        'are missing from the Phase 5 map.'
    )

print('-> Fullscreen, reset-view, and mouse-position controls present: True')


# ----------------------------------------------------
# CONFIRM FIRE-HAZARD INTEGRATION HAS NOT OCCURRED YET
# ----------------------------------------------------

phase5_hazard_overlay_present = (
    'Composite Fire Hazard Classes'
    in phase5_registered_layer_names
)

if phase5_hazard_overlay_present:
    raise ValueError(
        'The Composite Fire Hazard overlay is already attached during '
        'Phase 5. It must be added later during Phase 13 integration.'
    )

print('-> Composite Fire Hazard overlay deferred to Phase 13: True')


# ----------------------------------------------------
# RECORD PHASE 5 MAP-ASSEMBLY STATUS
# ----------------------------------------------------

wui_dashboard_assembly_complete = True

print(
    '-> WUI dashboard assembly complete: '
    f'{wui_dashboard_assembly_complete}'
)

print('\nNOTE:')

print(
    'Phase 5 intentionally does not save, render, or display the '
    'shared Folium map.'
)

print(
    'Phase 13 will add the Composite Fire Hazard overlay, create one '
    'integrated legend, create one collapsed TreeLayerControl, validate '
    'the dashboard, save the HTML once, and display it once.'
)

print(
    '\n=== WUI DASHBOARD ASSEMBLY VALIDATION COMPLETE ==='
)


=== VALIDATING WUI DASHBOARD ASSEMBLY ===
-> WUI dashboard Folium map object: valid
-> Required Phase 5 basemaps registered: 2
-> Required Phase 5 operational layers registered: 10
-> Phase 5 LayerControl count: 0
-> Legacy Phase 5 legend present: False
-> Fullscreen, reset-view, and mouse-position controls present: True
-> Composite Fire Hazard overlay deferred to Phase 13: True
-> WUI dashboard assembly complete: True

NOTE:
Phase 5 intentionally does not save, render, or display the shared Folium map.
Phase 13 will add the Composite Fire Hazard overlay, create one integrated legend, create one collapsed TreeLayerControl, validate the dashboard, save the HTML once, and display it once.

=== WUI DASHBOARD ASSEMBLY VALIDATION COMPLETE ===


# CHRG Phase 6 – Composite Fire Hazard Configuration

## Purpose

Configure the Composite Fire Hazard model, component weights, hazard classes, output grid, source catalog, cache policy, directories, and shared acquisition utilities.


## Composite Fire Hazard Model Configuration


In [53]:
print('\n=== PHASE 6: Composite Fire Hazard Grid ===')

# Record source releases and analysis periods so the hazard model can be reproduced with
# the same
# environmental inputs.
fire_hazard_dataset_versions = {
    'landfire': 'LF2025',
    'hls': 'Version 2.0',
    'hls_collections': ['hls2-l30', 'hls2-s30'],
    'hls_analysis_period': 'July-September 2025',
    'tiger_roads': '2025',
    'annual_nlcd': 'CONUS Collection 1.2 (1985-2025)',
    'mtbs': '2026 Q3',
    'inform_fodr': 'Current through 2026',
}

print('=== DEFINING BASELINE COMPOSITE FIRE HAZARD MODEL FRAMEWORK ===')

# Define the model identity and version used in metadata, QA summaries, and exported
# fire-hazard products.
fire_hazard_model_name = 'Baseline Composite Fire Hazard Grid'
fire_hazard_model_version = '1.0'

# Define the study scope, Utah counties, common raster CRS, resolution, score ranges,
# NoData value, and
# output format used by every hazard component.
fire_hazard_model_purpose = (
    'Estimate the relative baseline wildfire hazard across the study area using '
    'environmental, terrain, vegetation, historical fire, and human ignition '
    'characteristics.'
)
fire_hazard_model_scope = 'Hazard Only'
fire_hazard_study_counties = ['Salt Lake County', 'Utah County', 'Washington County']
fire_hazard_target_crs = 'EPSG:26912'
fire_hazard_cell_size = 30
fire_hazard_cell_units = 'meters'
fire_hazard_normalized_range = (0.0, 1.0)
fire_hazard_index_range = (1, 10)
fire_hazard_nodata_value = -9999.0
fire_hazard_output_format = 'GeoTIFF'
fire_hazard_raster_dtype = 'float32'
print(f'-> Model Name: {fire_hazard_model_name}')
print(f'-> Model Version: {fire_hazard_model_version}')
print(f'-> Model Scope: {fire_hazard_model_scope}')
print(f'-> Study Counties: {len(fire_hazard_study_counties)}')
print(f'-> Target CRS: {fire_hazard_target_crs}')
print(f'-> Cell Size: {fire_hazard_cell_size} {fire_hazard_cell_units}')
print(
    (
        f'-> Normalized Range: {fire_hazard_normalized_range[0]} to '
        f'{fire_hazard_normalized_range[1]}'
    ),
)
print(
    (
        f'-> Fire Hazard Index: {fire_hazard_index_range[0]} to {fire_hazard_index_range[1]}'
    ),
)
print(f'-> Output Format: {fire_hazard_output_format}')
print(f'-> Raster Data Type: {fire_hazard_raster_dtype}')
print(f'-> NoData Value: {fire_hazard_nodata_value}')
print('\nNOTE:')
print(
    (
        'This cell defines only the identity and spatial framework of the Baseline '
        'Composite Fire Hazard Grid. The hazard components, weights, and model '
        'equation are defined in step Define Hazard Components and Weights.'
    ),
)

print('\n=== MODEL FRAMEWORK DEFINED ===')

# Store the STAC endpoint used to locate cloud-hosted environmental source data without
# hard-coding
# individual asset URLs.
fire_hazard_planetary_computer_stac_url = 'https://planetarycomputer.microsoft.com/api/stac/v1'




=== PHASE 6: Composite Fire Hazard Grid ===
=== DEFINING BASELINE COMPOSITE FIRE HAZARD MODEL FRAMEWORK ===
-> Model Name: Baseline Composite Fire Hazard Grid
-> Model Version: 1.0
-> Model Scope: Hazard Only
-> Study Counties: 3
-> Target CRS: EPSG:26912
-> Cell Size: 30 meters
-> Normalized Range: 0.0 to 1.0
-> Fire Hazard Index: 1 to 10
-> Output Format: GeoTIFF
-> Raster Data Type: float32
-> NoData Value: -9999.0

NOTE:
This cell defines only the identity and spatial framework of the Baseline Composite Fire Hazard Grid. The hazard components, weights, and model equation are defined in step Define Hazard Components and Weights.

=== MODEL FRAMEWORK DEFINED ===


### Defining Fire Hazard Components and Model Weights


In [54]:
print('=== DEFINING FIRE HAZARD COMPONENTS AND MODEL WEIGHTS ===')

# Define the five hazard components and their normalized weights for the final
# weighted-overlay calculation.
fire_hazard_components = {
    'fuel_hazard': {
        'component_name': 'Surface and Canopy Fuel Hazard',
        'weight': 0.3,
        'description': (
            'Represents the relative potential for wildfire spread and intensity '
            'based on surface-fuel and canopy-fuel conditions.'
        ),
        'expected_input_type': 'Categorical and continuous LANDFIRE fuel rasters',
        'normalization_direction': 'Higher values indicate greater hazard',
    },
    'fuel_dryness': {
        'component_name': 'Vegetation and Fuel Dryness',
        'weight': 0.2,
        'description': (
            'Represents relative vegetation dryness and fuel condition using remotely '
            'sensed vegetation and moisture indicators.'
        ),
        'expected_input_type': 'Continuous NDVI, NDMI, or related vegetation-condition rasters',
        'normalization_direction': 'Drier conditions indicate greater hazard',
    },
    'slope_hazard': {
        'component_name': 'Slope-Based Spread Potential',
        'weight': 0.15,
        'description': (
            'Represents the potential for faster uphill fire spread across '
            'increasingly steep terrain.'
        ),
        'expected_input_type': 'Continuous slope raster',
        'normalization_direction': 'Steeper slopes indicate greater hazard',
    },
    'aspect_hazard': {
        'component_name': 'Aspect and Solar Exposure',
        'weight': 0.1,
        'description': (
            'Represents terrain-orientation effects on solar exposure, vegetation '
            'moisture, and relative surface dryness.'
        ),
        'expected_input_type': 'Categorical or continuous aspect raster',
        'normalization_direction': 'Warmer and drier exposures indicate greater hazard',
    },
    'historical_fire_likelihood': {
        'component_name': 'Historical Fire Likelihood',
        'weight': 0.15,
        'description': (
            'Represents relative wildfire likelihood based on historical fire '
            'occurrence, ignition density, burn frequency, or comparable historical '
            'patterns.'
        ),
        'expected_input_type': (
            'Historical ignition points, burn frequency, or '
            'fire-likelihood raster'
        ),
        'normalization_direction': 'Greater historical likelihood indicates greater hazard',
    },
    'human_ignition_potential': {
        'component_name': 'Human Ignition Potential',
        'weight': 0.1,
        'description': (
            'Represents relative opportunity for human-caused ignition near roads, '
            'development, access corridors, or related infrastructure.'
        ),
        'expected_input_type': (
            'Distance, density, or proximity surfaces derived from roads and '
            'development'
        ),
        'normalization_direction': 'Greater human ignition opportunity indicates greater hazard',
    },
}
fire_hazard_weights = {
    component_key: component_settings['weight']
    for (component_key, component_settings) in fire_hazard_components.items()
}

# Document the composite equation and calculate the stored weight total for later
# validation against 1.0.
fire_hazard_model_equation = (
    'Composite Fire Hazard = (Fuel Hazard × 0.30) + (Fuel Dryness × 0.20) + (Slope '
    'Hazard × 0.15) + (Aspect Hazard × 0.10) + (Historical Fire Likelihood × 0.15) + '
    '(Human Ignition Potential × 0.10)'
)
fire_hazard_weight_total = sum(fire_hazard_weights.values())
print(f'-> Hazard components defined: {len(fire_hazard_components)}')
print(f'-> Combined model weight: {fire_hazard_weight_total:.2f}')
print('-> Weighted-additive model equation created.')
print('\n--- FIRE HAZARD COMPONENT WEIGHTS ---')

# Iterate through configured hazard components to validate metadata and model weights
# consistently.
for component_key, component_settings in fire_hazard_components.items():
    print(
        (
            f"  • {component_settings['component_name']}: "
            f"{component_settings['weight'] * 100:.0f}%"
        ),
    )

print('\n--- FIRE HAZARD MODEL EQUATION ---')
print(fire_hazard_model_equation)
print('\nNOTE:')
print(
    (
        'The component weights defined in this cell are initial modeling assumptions. '
        'Their validity will be checked in the Validate Hazard Model Configuration '
        'step and evaluated later through sensitivity testing.'
    ),
)
print(
    (
        'All hazard-component rasters must be normalized to a common 0-to-1 scale '
        'before the weighted overlay is calculated.'
    ),
)

print('\n=== FIRE HAZARD COMPONENTS AND WEIGHTS DEFINED ===')



=== DEFINING FIRE HAZARD COMPONENTS AND MODEL WEIGHTS ===
-> Hazard components defined: 6
-> Combined model weight: 1.00
-> Weighted-additive model equation created.

--- FIRE HAZARD COMPONENT WEIGHTS ---
  • Surface and Canopy Fuel Hazard: 30%
  • Vegetation and Fuel Dryness: 20%
  • Slope-Based Spread Potential: 15%
  • Aspect and Solar Exposure: 10%
  • Historical Fire Likelihood: 15%
  • Human Ignition Potential: 10%

--- FIRE HAZARD MODEL EQUATION ---
Composite Fire Hazard = (Fuel Hazard × 0.30) + (Fuel Dryness × 0.20) + (Slope Hazard × 0.15) + (Aspect Hazard × 0.10) + (Historical Fire Likelihood × 0.15) + (Human Ignition Potential × 0.10)

NOTE:
The component weights defined in this cell are initial modeling assumptions. Their validity will be checked in the Validate Hazard Model Configuration step and evaluated later through sensitivity testing.
All hazard-component rasters must be normalized to a common 0-to-1 scale before the weighted overlay is calculated.

=== FIRE HAZARD CO

### Defining Fire Hazard Index Classes and Nonburnable Policy


In [55]:
print('=== DEFINING FIRE HAZARD INDEX CLASSES AND NONBURNABLE POLICY ===')

# Set the continuous and integer index bounds used to convert normalized hazard scores
# into a 0–100 product.
fire_hazard_continuous_minimum = fire_hazard_normalized_range[0]
fire_hazard_continuous_maximum = fire_hazard_normalized_range[1]
fire_hazard_index_minimum = fire_hazard_index_range[0]
fire_hazard_index_maximum = fire_hazard_index_range[1]

# Define ordered hazard classes and numeric breaks used for raster classification,
# summaries, and map
# interpretation.
fire_hazard_index_classes = {
    'Low': {
        'minimum_index': 1,
        'maximum_index': 2,
        'class_code': 1,
        'description': (
            'Locations with comparatively limited baseline wildfire hazard under the '
            'selected model assumptions.'
        ),
    },
    'Moderate': {
        'minimum_index': 3,
        'maximum_index': 4,
        'class_code': 2,
        'description': (
            'Locations with moderate environmental and ignition-related wildfire '
            'hazard.'
        ),
    },
    'High': {
        'minimum_index': 5,
        'maximum_index': 6,
        'class_code': 3,
        'description': (
            'Locations where fuel, terrain, dryness, historical fire, or ignition '
            'conditions combine to produce elevated hazard.'
        ),
    },
    'Very High': {
        'minimum_index': 7,
        'maximum_index': 8,
        'class_code': 4,
        'description': (
            'Locations with strongly elevated baseline wildfire hazard across several '
            'model components.'
        ),
    },
    'Extreme': {
        'minimum_index': 9,
        'maximum_index': 10,
        'class_code': 5,
        'description': (
            'Locations representing the highest relative baseline wildfire hazard '
            'within the three-county study area.'
        ),
    },
}
fire_hazard_continuous_class_breaks = {
    'Low': {'minimum_score': 0.0, 'maximum_score': 0.2},
    'Moderate': {'minimum_score': 0.2, 'maximum_score': 0.4},
    'High': {'minimum_score': 0.4, 'maximum_score': 0.6},
    'Very High': {'minimum_score': 0.6, 'maximum_score': 0.8},
    'Extreme': {'minimum_score': 0.8, 'maximum_score': 1.0},
}
fire_hazard_class_lower_bound_inclusive = True
fire_hazard_class_upper_bound_inclusive = False
fire_hazard_include_maximum_score = True
fire_hazard_classification_method = 'Equal-width normalized intervals'

# Define how nonburnable cells and missing components are masked so excluded land does
# not receive a
# misleading hazard score.
fire_hazard_nonburnable_policy = 'NoData'
fire_hazard_nonburnable_value = fire_hazard_nodata_value
fire_hazard_nonburnable_classes = [
    'Open Water',
    'Permanent Snow or Ice',
    'Developed Nonburnable Surface',
    'Barren Nonburnable Surface',
    'Other Confirmed Nonburnable Land',
]
fire_hazard_apply_nonburnable_mask_before_overlay = True
fire_hazard_mask_continuous_output = True
fire_hazard_mask_index_output = True
fire_hazard_mask_class_output = True
fire_hazard_missing_component_policy = 'Assign NoData'
fire_hazard_require_all_components = True
print(
    (
        f'-> Continuous hazard range: {fire_hazard_continuous_minimum:.1f} to '
        f'{fire_hazard_continuous_maximum:.1f}'
    ),
)
print(
    (
        f'-> Integer hazard index: {fire_hazard_index_minimum} to {fire_hazard_index_maximum}'
    ),
)
print(f'-> Descriptive hazard classes: {len(fire_hazard_index_classes)}')
print(f'-> Classification method: {fire_hazard_classification_method}')
print(f'-> Nonburnable-cell policy: {fire_hazard_nonburnable_policy}')
print(f'-> Nonburnable output value: {fire_hazard_nonburnable_value}')
print(f'-> Require all hazard components: {fire_hazard_require_all_components}')
print('\n--- FIRE HAZARD INDEX CLASSES ---')

# Iterate through each hazard class to validate ordered ranges and build classification
# summary records.
for hazard_class_name, hazard_class_settings in fire_hazard_index_classes.items():
    print(
        (
            f"  • {hazard_class_name}: {hazard_class_settings['minimum_index']}–"
            f"{hazard_class_settings['maximum_index']}"
        ),
    )

print('\n--- NORMALIZED HAZARD CLASS BREAKS ---')

# Iterate through each hazard class to validate ordered ranges and build classification
# summary records.
for (hazard_class_name, hazard_break_settings) in (
    fire_hazard_continuous_class_breaks.items(
    )
):
    print(
        (
            f"  • {hazard_class_name}: {hazard_break_settings['minimum_score']:.2f}–"
            f"{hazard_break_settings['maximum_score']:.2f}"
        ),
    )

print('\nNOTE:')
print(
    (
        'Confirmed nonburnable cells will be represented as NoData rather than zero '
        'so they cannot be misinterpreted as valid low-hazard locations.'
    ),
)
print(
    (
        'The equal-width class breaks are the initial classification method. Their '
        'usefulness will be reviewed after the composite hazard-score distribution is '
        'available.'
    ),
)

print('\n=== FIRE HAZARD INDEX CLASSES AND NONBURNABLE POLICY DEFINED ===')



=== DEFINING FIRE HAZARD INDEX CLASSES AND NONBURNABLE POLICY ===
-> Continuous hazard range: 0.0 to 1.0
-> Integer hazard index: 1 to 10
-> Descriptive hazard classes: 5
-> Classification method: Equal-width normalized intervals
-> Nonburnable-cell policy: NoData
-> Nonburnable output value: -9999.0
-> Require all hazard components: True

--- FIRE HAZARD INDEX CLASSES ---
  • Low: 1–2
  • Moderate: 3–4
  • High: 5–6
  • Very High: 7–8
  • Extreme: 9–10

--- NORMALIZED HAZARD CLASS BREAKS ---
  • Low: 0.00–0.20
  • Moderate: 0.20–0.40
  • High: 0.40–0.60
  • Very High: 0.60–0.80
  • Extreme: 0.80–1.00

NOTE:
Confirmed nonburnable cells will be represented as NoData rather than zero so they cannot be misinterpreted as valid low-hazard locations.
The equal-width class breaks are the initial classification method. Their usefulness will be reviewed after the composite hazard-score distribution is available.

=== FIRE HAZARD INDEX CLASSES AND NONBURNABLE POLICY DEFINED ===


### Validating Fire Hazard Model Configuration


In [56]:
print('=== VALIDATING FIRE HAZARD MODEL CONFIGURATION ===')

# List the configuration objects that must exist before model validation can run.
required_fire_hazard_configuration_objects = [
    'fire_hazard_model_name',
    'fire_hazard_model_version',
    'fire_hazard_model_purpose',
    'fire_hazard_model_scope',
    'fire_hazard_study_counties',
    'fire_hazard_target_crs',
    'fire_hazard_cell_size',
    'fire_hazard_cell_units',
    'fire_hazard_normalized_range',
    'fire_hazard_index_range',
    'fire_hazard_nodata_value',
    'fire_hazard_output_format',
    'fire_hazard_raster_dtype',
    'fire_hazard_components',
    'fire_hazard_weights',
    'fire_hazard_model_equation',
    'fire_hazard_weight_total',
    'fire_hazard_index_classes',
    'fire_hazard_continuous_class_breaks',
    'fire_hazard_classification_method',
    'fire_hazard_nonburnable_policy',
    'fire_hazard_nonburnable_value',
    'fire_hazard_nonburnable_classes',
    'fire_hazard_missing_component_policy',
    'fire_hazard_require_all_components',
]
missing_fire_hazard_configuration_objects = [
    object_name
    for object_name in required_fire_hazard_configuration_objects
    if object_name not in globals()
]

# Stop if required model-configuration objects are missing so validation does not run on
# incomplete settings.
if missing_fire_hazard_configuration_objects:
    raise (
        NameError(
            (
                f'The following fire-hazard configuration objects are missing:\n'
                f'{missing_fire_hazard_configuration_objects}'
                f'\n\nRun the preceding model-configuration cells before executing this validation cell.'
            ),
        )
    )

# Validate the model identity fields before they are written into summaries and
# metadata.
fire_hazard_model_name_valid = (
    isinstance(fire_hazard_model_name, str)
    and bool(fire_hazard_model_name.strip())
)
fire_hazard_model_version_valid = (
    isinstance(fire_hazard_model_version, str)
    and bool(fire_hazard_model_version.strip())
)
fire_hazard_model_purpose_valid = (
    isinstance(fire_hazard_model_purpose, str)
    and bool(fire_hazard_model_purpose.strip())
)
fire_hazard_model_scope_valid = (
    isinstance(fire_hazard_model_scope, str)
    and bool(fire_hazard_model_scope.strip())
)

# Stop if the model name is blank or malformed because it is required for model metadata
# and output identification.
if not fire_hazard_model_name_valid:
    raise ValueError('fire_hazard_model_name must contain a non-empty text value.')

# Stop if the model version is invalid so generated products retain traceable version
# metadata.
if not fire_hazard_model_version_valid:
    raise ValueError('fire_hazard_model_version must contain a non-empty text value.')

# Stop if the model purpose is missing because the configuration must document what the
# hazard grid represents.
if not fire_hazard_model_purpose_valid:
    raise ValueError('fire_hazard_model_purpose must contain a non-empty text value.')

# Stop if the model scope is missing so the analysis extent and intended use remain
# explicitly documented.
if not fire_hazard_model_scope_valid:
    raise ValueError('fire_hazard_model_scope must contain a non-empty text value.')

# Validate that the configured study area contains exactly the three Utah counties used
# throughout the WUI
# project.
expected_fire_hazard_study_counties = [
    'Salt Lake County',
    'Utah County',
    'Washington County',
]
fire_hazard_study_counties_valid = (
    fire_hazard_study_counties
    == expected_fire_hazard_study_counties
)

# Stop if the configured study counties do not match the approved three-county analysis
# area.
if not fire_hazard_study_counties_valid:
    raise (
        ValueError(
            (
                f'The configured fire-hazard study counties do not match the expected '
                f'three-county study area.\nExpected: {expected_fire_hazard_study_counties}'
                f'\nConfigured: {fire_hazard_study_counties}'
            ),
        )
    )

# Validate the common raster CRS, cell size, and units required for cell-by-cell
# overlay.
fire_hazard_target_crs_valid = fire_hazard_target_crs == 'EPSG:26912'
fire_hazard_cell_size_valid = (
    isinstance(fire_hazard_cell_size, (int, float))
    and fire_hazard_cell_size > 0
)
fire_hazard_cell_units_valid = fire_hazard_cell_units == 'meters'

# Stop if the target CRS is invalid because all component rasters must share one
# projected spatial reference.
if not fire_hazard_target_crs_valid:
    raise ValueError('The fire-hazard target CRS must be EPSG:26912.')

# Stop if the raster cell size is invalid because component alignment depends on a
# positive common resolution.
if not fire_hazard_cell_size_valid:
    raise ValueError('The fire-hazard cell size must be a positive numeric value.')

# Stop if raster cell units are invalid so resolution metadata remains consistent with
# the projected CRS.
if not fire_hazard_cell_units_valid:
    raise ValueError("The fire-hazard cell units must be 'meters'.")

# Validate the normalized and index score ranges before class breaks and weighted scores
# depend on them.
fire_hazard_normalized_range_valid = (
    isinstance(fire_hazard_normalized_range, tuple)
    and len(fire_hazard_normalized_range) == 2
    and all((isinstance(value, (int, float)) for value in fire_hazard_normalized_range))
    and (fire_hazard_normalized_range[0] < fire_hazard_normalized_range[1])
)
fire_hazard_normalized_bounds_valid = fire_hazard_normalized_range == (0.0, 1.0)
fire_hazard_index_range_valid = (
    isinstance(fire_hazard_index_range, tuple)
    and len(fire_hazard_index_range) == 2
    and all((isinstance(value, int) for value in fire_hazard_index_range))
    and (fire_hazard_index_range[0] < fire_hazard_index_range[1])
)
fire_hazard_index_bounds_valid = fire_hazard_index_range == (1, 10)

# Stop if the normalized-score range is malformed because every hazard component must
# use a valid numeric interval.
if not fire_hazard_normalized_range_valid:
    raise (
        ValueError(
            'fire_hazard_normalized_range must contain two ordered numeric values.',
        )
    )

# Stop if normalized bounds differ from 0–1 because the weighted overlay assumes a
# common 0-to-1 scale.
if not fire_hazard_normalized_bounds_valid:
    raise ValueError('The normalized fire-hazard range must be (0.0, 1.0).')

# Stop if the integer hazard-index range is malformed before class breaks or outputs are
# derived from it.
if not fire_hazard_index_range_valid:
    raise ValueError('fire_hazard_index_range must contain two ordered integer values.')

# Stop if the hazard-index bounds differ from 1–10 because later classification rules
# assume that scale.
if not fire_hazard_index_bounds_valid:
    raise ValueError('The public-facing fire-hazard index must use the range (1, 10).')

# Validate NoData, raster format, and data type settings so exported hazard rasters can
# represent the full
# score range safely.
fire_hazard_nodata_value_valid = isinstance(fire_hazard_nodata_value, (int, float))
fire_hazard_nodata_outside_valid_range = (
    fire_hazard_nodata_value < fire_hazard_normalized_range[0]
    or fire_hazard_nodata_value > fire_hazard_normalized_range[1]
)
fire_hazard_output_format_valid = fire_hazard_output_format == 'GeoTIFF'
fire_hazard_raster_dtype_valid = fire_hazard_raster_dtype == 'float32'

# Stop if the NoData value is invalid so masked and unavailable raster cells can be
# represented consistently.
if not fire_hazard_nodata_value_valid:
    raise ValueError('fire_hazard_nodata_value must be numeric.')

# Stop if the NoData value overlaps the valid hazard range because valid scores and
# missing cells must remain distinct.
if not fire_hazard_nodata_outside_valid_range:
    raise (
        ValueError(
            (
                'The fire-hazard NoData value must fall outside the valid normalized '
                'score range.'
            ),
        )
    )

# Stop if the output raster format is unsupported so later exports use the approved
# geospatial file type.
if not fire_hazard_output_format_valid:
    raise ValueError('The configured fire-hazard output format must be GeoTIFF.')

# Stop if the raster data type is invalid because output precision and storage depend on
# the configured dtype.
if not fire_hazard_raster_dtype_valid:
    raise ValueError('The continuous fire-hazard raster data type must be float32.')

# Compare configured component keys with the required model design to catch missing or
# unexpected hazard
# layers.
expected_fire_hazard_component_keys = [
    'fuel_hazard',
    'fuel_dryness',
    'slope_hazard',
    'aspect_hazard',
    'historical_fire_likelihood',
    'human_ignition_potential',
]
configured_fire_hazard_component_keys = list(fire_hazard_components.keys())
fire_hazard_component_keys_valid = (
    configured_fire_hazard_component_keys
    == expected_fire_hazard_component_keys
)
missing_fire_hazard_components = [
    component_key
    for component_key in expected_fire_hazard_component_keys
    if component_key not in configured_fire_hazard_component_keys
]
unexpected_fire_hazard_components = [
    component_key
    for component_key in configured_fire_hazard_component_keys
    if component_key not in expected_fire_hazard_component_keys
]

# Stop if configured component keys differ from the approved hazard model so the
# weighted overlay cannot silently change.
if not fire_hazard_component_keys_valid:
    raise (
        ValueError(
            (
                f'The configured fire-hazard components do not match the approved model '
                f'structure.\nMissing components: {missing_fire_hazard_components}'
                f'\nUnexpected components: {unexpected_fire_hazard_components}'
            ),
        )
    )

# Verify that every component record contains the metadata needed for acquisition,
# normalization, weighting,
# and interpretation.
required_fire_hazard_component_fields = [
    'component_name',
    'weight',
    'description',
    'expected_input_type',
    'normalization_direction',
]
missing_fire_hazard_component_fields = {}

# Iterate through configured hazard components to validate metadata and model weights
# consistently.
for component_key, component_settings in fire_hazard_components.items():
    missing_component_fields = [
        field_name
        for field_name in required_fire_hazard_component_fields
        if field_name not in component_settings
    ]

    # Record missing component metadata fields so incomplete hazard-component
    # definitions can be rejected after inspection.
    if missing_component_fields:
        missing_fire_hazard_component_fields[component_key] = missing_component_fields

# Stop if any hazard component lacks required metadata because every weighted input must
# be fully documented.
if missing_fire_hazard_component_fields:
    raise (
        ValueError(
            (
                f'One or more fire-hazard components are missing required configuration fields:\n'
                f'{missing_fire_hazard_component_fields}'
            ),
        )
    )

# Validate component weight keys, numeric values, and the total weight so the weighted
# overlay remains
# mathematically consistent.
fire_hazard_weight_keys_valid = (
    list(fire_hazard_weights.keys())
    == expected_fire_hazard_component_keys
)
invalid_fire_hazard_weights = {
    component_key: component_weight
    for (component_key, component_weight) in fire_hazard_weights.items()
    if (
        not isinstance(component_weight, (int, float))
        or component_weight <= 0
        or component_weight > 1
    )
}
validated_fire_hazard_weight_total = sum(fire_hazard_weights.values())
fire_hazard_weight_tolerance = 1e-09
fire_hazard_weight_total_valid = (
    abs(validated_fire_hazard_weight_total - 1.0)
    <= fire_hazard_weight_tolerance
)
fire_hazard_stored_weight_total_valid = (
    abs(fire_hazard_weight_total - validated_fire_hazard_weight_total)
    <= fire_hazard_weight_tolerance
)

# Stop if weight keys do not match component keys so every hazard component has exactly
# one corresponding weight.
if not fire_hazard_weight_keys_valid:
    raise (
        ValueError(
            (
                'The fire_hazard_weights dictionary does not contain the approved '
                'component keys in the required order.'
            ),
        )
    )

# Stop if any component weight is nonnumeric or outside the approved range before the
# weighted overlay is calculated.
if invalid_fire_hazard_weights:
    raise (
        ValueError(
            (
                f'Every fire-hazard component weight must be numeric, greater than zero, '
                f'and no greater than one. Invalid weights:\n{invalid_fire_hazard_weights}'
            ),
        )
    )

# Stop if component weights do not sum to 1.0 because the composite score assumes a
# normalized weighted sum.
if not fire_hazard_weight_total_valid:
    raise (
        ValueError(
            (
                f'Fire-hazard component weights must sum to 1.0. Current total: '
                f'{validated_fire_hazard_weight_total:.6f}'
            ),
        )
    )

# Stop if the stored weight total disagrees with the recalculated total so model
# metadata cannot become inconsistent.
if not fire_hazard_stored_weight_total_valid:
    raise (
        ValueError(
            'The stored fire-hazard weight total does not match the recalculated total.',
        )
    )

# Validate that categorical hazard classes remain in the intended Low-to-Extreme order.
expected_fire_hazard_class_order = ['Low', 'Moderate', 'High', 'Very High', 'Extreme']
fire_hazard_index_class_order_valid = (
    list(fire_hazard_index_classes.keys())
    == expected_fire_hazard_class_order
)
fire_hazard_continuous_class_order_valid = (
    list(fire_hazard_continuous_class_breaks.keys())
    == expected_fire_hazard_class_order
)

# Stop if index classes are not ordered as configured because class labels must map
# predictably to increasing hazard.
if not fire_hazard_index_class_order_valid:
    raise (
        ValueError(
            'The fire-hazard integer classes are not in the required Low-to-Extreme order.',
        )
    )

# Stop if normalized class breaks are not ordered consistently with the descriptive
# hazard classes.
if not fire_hazard_continuous_class_order_valid:
    raise (
        ValueError(
            (
                'The normalized fire-hazard class breaks are not in the required '
                'Low-to-Extreme order.'
            ),
        )
    )

# Expand the configured integer class ranges to verify that every hazard index value
# from 0 through 100 is
# classified exactly once.
configured_fire_hazard_index_values = []

# Iterate through each hazard class to validate ordered ranges and build classification
# summary records.
for hazard_class_name, hazard_class_settings in fire_hazard_index_classes.items():
    required_index_class_fields = [
        'minimum_index',
        'maximum_index',
        'class_code',
        'description',
    ]
    missing_index_class_fields = [
        field_name
        for field_name in required_index_class_fields
        if field_name not in hazard_class_settings
    ]

    # Record missing index-class fields so incomplete class definitions can be rejected
    # before classification.
    if missing_index_class_fields:
        raise (
            ValueError(
                (
                    f'The {hazard_class_name} hazard class is missing required fields: '
                    f'{missing_index_class_fields}'
                ),
            )
        )
    minimum_index = hazard_class_settings['minimum_index']
    maximum_index = hazard_class_settings['maximum_index']

    # Reject an invalid configuration type before downstream processing depends on this
    # value.
    if (
        not isinstance(minimum_index, int)
        or not isinstance(maximum_index, int)
        or minimum_index > maximum_index
    ):
        raise (
            ValueError(
                (
                    f'The {hazard_class_name}'
                    f' hazard class contains an invalid integer index range.'
                ),
            )
        )
    configured_fire_hazard_index_values.extend(range(minimum_index, maximum_index + 1))

# Build the complete expected hazard-index sequence so class ranges can be checked for
# gaps, overlaps, and full 1–10 coverage.
expected_fire_hazard_index_values = (
    list(
        range(fire_hazard_index_range[0], fire_hazard_index_range[1] + 1),
    )
)
fire_hazard_index_coverage_valid = (
    configured_fire_hazard_index_values
    == expected_fire_hazard_index_values
)

# Stop if integer classes do not cover the full 1–10 index without gaps or overlaps.
if not fire_hazard_index_coverage_valid:
    raise (
        ValueError(
            (
                'The descriptive hazard classes must cover every integer index from 1 '
                'through 10 exactly once, without gaps or overlaps.'
            ),
        )
    )

# Compare adjacent continuous class boundaries to verify complete, gap-free coverage of
# the normalized 0–1
# score range.
continuous_class_minimums = []
continuous_class_maximums = []

# Iterate through each hazard class to validate ordered ranges and build classification
# summary records.
for hazard_class_name, break_settings in fire_hazard_continuous_class_breaks.items():
    required_break_fields = ['minimum_score', 'maximum_score']
    missing_break_fields = [
        field_name
        for field_name in required_break_fields
        if field_name not in break_settings
    ]

    # Record missing normalized-break fields so incomplete continuous classes cannot
    # pass configuration validation.
    if missing_break_fields:
        raise (
            ValueError(
                (
                    f'The {hazard_class_name} continuous class is missing required fields: '
                    f'{missing_break_fields}'
                ),
            )
        )
    minimum_score = break_settings['minimum_score']
    maximum_score = break_settings['maximum_score']

    # Reject an invalid configuration type before downstream processing depends on this
    # value.
    if (
        not isinstance(minimum_score, (int, float))
        or not isinstance(maximum_score, (int, float))
        or minimum_score < 0
        or maximum_score > 1
        or minimum_score >= maximum_score
    ):
        raise (
            ValueError(
                f'The {hazard_class_name} normalized class interval is invalid.',
            )
        )
    continuous_class_minimums.append(minimum_score)
    continuous_class_maximums.append(maximum_score)

# Compare normalized class boundaries with the configured 0–1 range and adjacent
# intervals to confirm continuous coverage.
continuous_class_start_valid = (
    continuous_class_minimums[0]
    == fire_hazard_normalized_range[0]
)
continuous_class_end_valid = (
    continuous_class_maximums[-1]
    == fire_hazard_normalized_range[1]
)
continuous_class_contiguity_valid = (
    all(
        (
            continuous_class_maximums[interval_index]
            == continuous_class_minimums[interval_index + 1]
            for interval_index in range(len(continuous_class_maximums) - 1)
        ),
    )
)

# Stop if normalized class breaks do not begin at 0.0 because the full valid score range
# must be classified.
if not continuous_class_start_valid:
    raise (
        ValueError(
            'The first continuous hazard class must begin at the normalized minimum of 0.0.',
        )
    )

# Stop if normalized class breaks do not end at 1.0 because the highest valid hazard
# scores must be classified.
if not continuous_class_end_valid:
    raise (
        ValueError(
            'The final continuous hazard class must end at the normalized maximum of 1.0.',
        )
    )

# Stop if normalized class intervals contain gaps because every valid composite score
# must map to one class.
if not continuous_class_contiguity_valid:
    raise (
        ValueError(
            (
                'The continuous hazard class breaks must be contiguous, with no gaps '
                'between intervals.'
            ),
        )
    )

# Validate nonburnable and missing-component policies before those rules are applied to
# raster cells.
valid_fire_hazard_nonburnable_policies = ['NoData']
fire_hazard_nonburnable_policy_valid = (
    fire_hazard_nonburnable_policy
    in valid_fire_hazard_nonburnable_policies
)
fire_hazard_nonburnable_value_valid = (
    fire_hazard_nonburnable_value
    == fire_hazard_nodata_value
)
fire_hazard_nonburnable_classes_valid = (
    isinstance(fire_hazard_nonburnable_classes, list)
    and len(fire_hazard_nonburnable_classes) > 0
    and all(
        (
            isinstance(class_name, str) and bool(class_name.strip())
            for class_name in fire_hazard_nonburnable_classes
        ),
    )
)
fire_hazard_missing_component_policy_valid = (
    isinstance(fire_hazard_missing_component_policy, str)
    and bool(fire_hazard_missing_component_policy.strip())
)
fire_hazard_require_all_components_valid = fire_hazard_require_all_components is True

# Stop if the nonburnable-cell policy is invalid so excluded land is not assigned a
# misleading hazard score.
if not fire_hazard_nonburnable_policy_valid:
    raise (
        ValueError(
            (
                "The current hazard-model implementation supports 'NoData' as the "
                'nonburnable policy.'
            ),
        )
    )

# Stop if the nonburnable output value conflicts with the configured NoData policy.
if not fire_hazard_nonburnable_value_valid:
    raise (
        ValueError(
            'The nonburnable-cell value must match the project fire-hazard NoData value.',
        )
    )

# Stop if the nonburnable class list is empty or malformed because masking depends on
# documented exclusion classes.
if not fire_hazard_nonburnable_classes_valid:
    raise (
        ValueError(
            (
                'fire_hazard_nonburnable_classes must contain at least one non-empty '
                'class name.'
            ),
        )
    )

# Stop if the missing-component policy is invalid so incomplete raster stacks are
# handled consistently.
if not fire_hazard_missing_component_policy_valid:
    raise (
        ValueError(
            'The missing-component policy must contain a non-empty text value.',
        )
    )

# Stop if the all-components requirement is not Boolean because overlay completeness
# must be unambiguous.
if not fire_hazard_require_all_components_valid:
    raise (
        ValueError(
            (
                'The baseline hazard model must require valid values from every hazard '
                'component.'
            ),
        )
    )

# Assemble all configuration QA results into one table for reader-facing review
# before source acquisition
# begins.
fire_hazard_configuration_validation_summary = (
    pd.DataFrame(
        [
            {
                'VALIDATION_CATEGORY': 'Model Identity',
                'VALID': (
                    fire_hazard_model_name_valid
                    and fire_hazard_model_version_valid
                    and fire_hazard_model_purpose_valid
                    and fire_hazard_model_scope_valid
                ),
            },
            {'VALIDATION_CATEGORY': 'Study Area', 'VALID': fire_hazard_study_counties_valid},
            {
                'VALIDATION_CATEGORY': 'Spatial Framework',
                'VALID': (
                    fire_hazard_target_crs_valid
                    and fire_hazard_cell_size_valid
                    and fire_hazard_cell_units_valid
                ),
            },
            {
                'VALIDATION_CATEGORY': 'Raster Output Settings',
                'VALID': (
                    fire_hazard_nodata_value_valid
                    and fire_hazard_nodata_outside_valid_range
                    and fire_hazard_output_format_valid
                    and fire_hazard_raster_dtype_valid
                ),
            },
            {
                'VALIDATION_CATEGORY': 'Hazard Components',
                'VALID': (
                    fire_hazard_component_keys_valid
                    and not missing_fire_hazard_component_fields
                ),
            },
            {
                'VALIDATION_CATEGORY': 'Model Weights',
                'VALID': (
                    fire_hazard_weight_keys_valid
                    and not invalid_fire_hazard_weights
                    and fire_hazard_weight_total_valid
                    and fire_hazard_stored_weight_total_valid
                ),
            },
            {
                'VALIDATION_CATEGORY': 'Integer Hazard Classes',
                'VALID': (
                    fire_hazard_index_class_order_valid
                    and fire_hazard_index_coverage_valid
                ),
            },
            {
                'VALIDATION_CATEGORY': 'Continuous Class Breaks',
                'VALID': (
                    fire_hazard_continuous_class_order_valid
                    and continuous_class_start_valid
                    and continuous_class_end_valid
                    and continuous_class_contiguity_valid
                ),
            },
            {
                'VALIDATION_CATEGORY': 'Nonburnable and Missing Data Policy',
                'VALID': (
                    fire_hazard_nonburnable_policy_valid
                    and fire_hazard_nonburnable_value_valid
                    and fire_hazard_nonburnable_classes_valid
                    and fire_hazard_missing_component_policy_valid
                    and fire_hazard_require_all_components_valid
                ),
            },
        ],
    )
)

# Collect failed QA checks so the workflow can stop with a concise list of configuration
# problems.
failed_fire_hazard_configuration_checks = fire_hazard_configuration_validation_summary[
    ~fire_hazard_configuration_validation_summary['VALID']
]
fire_hazard_configuration_valid = failed_fire_hazard_configuration_checks.empty

# Stop if any configuration QA check failed so invalid model settings are not propagated
# into source acquisition.
if not fire_hazard_configuration_valid:
    raise (
        ValueError(
            'One or more fire-hazard model configuration categories failed validation.',
        )
    )

print(
    (
        f'-> Required configuration objects found: '
        f'{len(required_fire_hazard_configuration_objects)}'
    ),
)
print(f'-> Hazard components validated: {len(fire_hazard_components)}')
print(f'-> Validated model weight total: {validated_fire_hazard_weight_total:.2f}')
print(f'-> Integer index coverage valid: {fire_hazard_index_coverage_valid}')
print(f'-> Continuous class breaks contiguous: {continuous_class_contiguity_valid}')
print(f'-> Nonburnable policy valid: {fire_hazard_nonburnable_policy_valid}')
print(f'-> Overall configuration valid: {fire_hazard_configuration_valid}')
print('\n--- FIRE HAZARD CONFIGURATION VALIDATION ---')
display(fire_hazard_configuration_validation_summary)
print('\nNOTE:')
print(
    (
        'The baseline fire-hazard model configuration passed identity, '
        'spatial-framework, component, weight, classification, and masking '
        'validation.'
    ),
)
print(
    (
        'This validation confirms that the model framework is internally consistent '
        'before source datasets are acquired or raster processing begins.'
    ),
)

print('\n=== FIRE HAZARD MODEL CONFIGURATION VALIDATION COMPLETE ===')

print('=== CREATING FIRE HAZARD MODEL SUMMARY ===')

# Confirm that validated configuration objects are available before building model
# summary tables.
required_fire_hazard_summary_objects = [
    'fire_hazard_model_name',
    'fire_hazard_model_version',
    'fire_hazard_model_purpose',
    'fire_hazard_model_scope',
    'fire_hazard_study_counties',
    'fire_hazard_target_crs',
    'fire_hazard_cell_size',
    'fire_hazard_cell_units',
    'fire_hazard_normalized_range',
    'fire_hazard_index_range',
    'fire_hazard_nodata_value',
    'fire_hazard_output_format',
    'fire_hazard_raster_dtype',
    'fire_hazard_components',
    'fire_hazard_weights',
    'fire_hazard_model_equation',
    'fire_hazard_weight_total',
    'fire_hazard_index_classes',
    'fire_hazard_continuous_class_breaks',
    'fire_hazard_classification_method',
    'fire_hazard_nonburnable_policy',
    'fire_hazard_nonburnable_value',
    'fire_hazard_nonburnable_classes',
    'fire_hazard_missing_component_policy',
    'fire_hazard_require_all_components',
    'fire_hazard_configuration_validation_summary',
    'fire_hazard_configuration_valid',
]
missing_fire_hazard_summary_objects = [
    object_name
    for object_name in required_fire_hazard_summary_objects
    if object_name not in globals()
]

# Stop if validated configuration objects are missing because the model summary tables
# depend on those approved settings.
if missing_fire_hazard_summary_objects:
    raise (
        NameError(
            (
                f'The following fire-hazard summary objects are missing:\n'
                f'{missing_fire_hazard_summary_objects}'
                f'\n\nRun the preceding model-configuration and validation cells before this summary cell.'
            ),
        )
    )

# Stop if any configuration QA check failed so invalid model settings are not propagated
# into source acquisition.
if not fire_hazard_configuration_valid:
    raise (
        ValueError(
            (
                'The fire-hazard model summary cannot be created because the '
                'configuration did not pass validation.'
            ),
        )
    )

# Build the model overview table used to document the spatial framework, scoring range,
# and output
# specifications.
fire_hazard_model_overview_summary = (
    pd.DataFrame(
        [
            {'SETTING': 'Model Name', 'VALUE': fire_hazard_model_name},
            {'SETTING': 'Model Version', 'VALUE': fire_hazard_model_version},
            {'SETTING': 'Model Scope', 'VALUE': fire_hazard_model_scope},
            {'SETTING': 'Study Counties', 'VALUE': ', '.join(fire_hazard_study_counties)},
            {'SETTING': 'Target CRS', 'VALUE': fire_hazard_target_crs},
            {
                'SETTING': 'Target Cell Size',
                'VALUE': f'{fire_hazard_cell_size} {fire_hazard_cell_units}',
            },
            {
                'SETTING': 'Normalized Score Range',
                'VALUE': (
                    f'{fire_hazard_normalized_range[0]} to '
                    f'{fire_hazard_normalized_range[1]}'
                ),
            },
            {
                'SETTING': 'Public Hazard Index',
                'VALUE': f'{fire_hazard_index_range[0]} to {fire_hazard_index_range[1]}',
            },
            {'SETTING': 'Output Format', 'VALUE': fire_hazard_output_format},
            {'SETTING': 'Raster Data Type', 'VALUE': fire_hazard_raster_dtype},
            {'SETTING': 'NoData Value', 'VALUE': fire_hazard_nodata_value},
            {'SETTING': 'Classification Method', 'VALUE': fire_hazard_classification_method},
            {'SETTING': 'Nonburnable Policy', 'VALUE': fire_hazard_nonburnable_policy},
            {
                'SETTING': 'Missing Component Policy',
                'VALUE': fire_hazard_missing_component_policy,
            },
            {
                'SETTING': 'Require All Components',
                'VALUE': fire_hazard_require_all_components,
            },
        ],
    )
)

# Convert component configuration records into a tabular summary of datasets, methods,
# and weights.
fire_hazard_component_summary_records = []

# Iterate through configured hazard components to validate metadata and model weights
# consistently.
for component_key, component_settings in fire_hazard_components.items():
    fire_hazard_component_summary_records.append(
        {
            'COMPONENT_KEY': component_key,
            'COMPONENT': component_settings['component_name'],
            'WEIGHT': component_settings['weight'],
            'WEIGHT_PERCENT': component_settings['weight'] * 100,
            'EXPECTED_INPUT': component_settings['expected_input_type'],
            'NORMALIZATION_DIRECTION': component_settings['normalization_direction'],
            'DESCRIPTION': component_settings['description'],
        },
    )

# Convert the accumulated component records into the model-component summary table used
# for review and documentation.
fire_hazard_component_summary = pd.DataFrame(fire_hazard_component_summary_records)

# Convert hazard-class definitions into a table that documents continuous and integer
# classification ranges.
fire_hazard_index_class_summary_records = []

# Iterate through each hazard class to validate ordered ranges and build classification
# summary records.
for hazard_class_name in fire_hazard_index_classes.keys():
    index_settings = fire_hazard_index_classes[hazard_class_name]
    continuous_settings = fire_hazard_continuous_class_breaks[hazard_class_name]
    fire_hazard_index_class_summary_records.append(
        {
            'HAZARD_CLASS': hazard_class_name,
            'CLASS_CODE': index_settings['class_code'],
            'INDEX_RANGE': (
                f"{index_settings['minimum_index']}–"
                f"{index_settings['maximum_index']}"
            ),
            'NORMALIZED_RANGE': (
                f"{continuous_settings['minimum_score']:.2f}–"
                f"{continuous_settings['maximum_score']:.2f}"
            ),
            'DESCRIPTION': index_settings['description'],
        },
    )

# Convert the accumulated class records into the final hazard-class summary table.
fire_hazard_index_class_summary = pd.DataFrame(fire_hazard_index_class_summary_records)

# Summarize masking and missing-data rules so excluded cells are documented alongside
# the model outputs.
fire_hazard_nonburnable_summary = (
    pd.DataFrame(
        [
            {
                'NONBURNABLE_CLASS': nonburnable_class,
                'OUTPUT_POLICY': fire_hazard_nonburnable_policy,
                'OUTPUT_VALUE': fire_hazard_nonburnable_value,
            }
            for nonburnable_class in fire_hazard_nonburnable_classes
        ],
    )
)

# Package the validated model configuration into a metadata object that can later be
# exported with the
# hazard products.
fire_hazard_model_metadata = {
    'model_name': fire_hazard_model_name,
    'model_version': fire_hazard_model_version,
    'model_purpose': fire_hazard_model_purpose,
    'model_scope': fire_hazard_model_scope,
    'study_counties': fire_hazard_study_counties,
    'target_crs': fire_hazard_target_crs,
    'cell_size': fire_hazard_cell_size,
    'cell_units': fire_hazard_cell_units,
    'normalized_range': list(fire_hazard_normalized_range),
    'index_range': list(fire_hazard_index_range),
    'nodata_value': fire_hazard_nodata_value,
    'output_format': fire_hazard_output_format,
    'raster_dtype': fire_hazard_raster_dtype,
    'component_weights': fire_hazard_weights,
    'model_equation': fire_hazard_model_equation,
    'classification_method': fire_hazard_classification_method,
    'nonburnable_policy': fire_hazard_nonburnable_policy,
    'nonburnable_classes': fire_hazard_nonburnable_classes,
    'missing_component_policy': fire_hazard_missing_component_policy,
    'require_all_components': fire_hazard_require_all_components,
    'configuration_valid': fire_hazard_configuration_valid,
}

# Verify that the summary tables contain the expected number of components, classes, and
# nonburnable
# records.
fire_hazard_component_summary_count_valid = (
    len(fire_hazard_component_summary)
    == len(fire_hazard_components)
)
fire_hazard_class_summary_count_valid = (
    len(fire_hazard_index_class_summary)
    == len(fire_hazard_index_classes)
)
fire_hazard_nonburnable_summary_count_valid = (
    len(fire_hazard_nonburnable_summary)
    == len(fire_hazard_nonburnable_classes)
)

# Stop if the component summary does not contain one record per configured hazard
# component.
if not fire_hazard_component_summary_count_valid:
    raise (
        ValueError(
            (
                'The hazard-component summary does not contain one record for every '
                'configured component.'
            ),
        )
    )

# Stop if the class summary does not contain one record per configured hazard class.
if not fire_hazard_class_summary_count_valid:
    raise (
        ValueError(
            (
                'The hazard-class summary does not contain one record for every '
                'configured class.'
            ),
        )
    )

# Stop if the nonburnable summary count differs from the configured exclusion-class
# count.
if not fire_hazard_nonburnable_summary_count_valid:
    raise (
        ValueError(
            (
                'The nonburnable-policy summary does not contain one record for every '
                'documented surface class.'
            ),
        )
    )

print(f'-> Model summary created: {fire_hazard_model_name}')
print(f'-> Model version: {fire_hazard_model_version}')
print(f'-> Hazard components summarized: {len(fire_hazard_component_summary)}')
print(f'-> Hazard classes summarized: {len(fire_hazard_index_class_summary)}')
print(f'-> Nonburnable classes documented: {len(fire_hazard_nonburnable_summary)}')
print(f'-> Model weight total: {fire_hazard_weight_total:.2f}')
print(f'-> Configuration validation passed: {fire_hazard_configuration_valid}')
print('\n--- FIRE HAZARD MODEL OVERVIEW ---')
display(fire_hazard_model_overview_summary)
print('\n--- FIRE HAZARD COMPONENTS AND WEIGHTS ---')
display(fire_hazard_component_summary)
print('\n--- FIRE HAZARD INDEX CLASSES ---')
display(fire_hazard_index_class_summary)
print('\n--- NONBURNABLE-CELL POLICY ---')
display(fire_hazard_nonburnable_summary)
print('\n--- MODEL CONFIGURATION VALIDATION ---')
display(fire_hazard_configuration_validation_summary)
print('\n--- FIRE HAZARD MODEL EQUATION ---')
print(fire_hazard_model_equation)
print('\nNOTE:')
print(
    (
        'The Baseline Composite Fire Hazard Grid framework is now fully defined and '
        'internally validated.'
    ),
)
print(
    (
        'The model summary and metadata objects created in this cell can be reused '
        'later when exporting the hazard-model documentation and project '
        'deliverables.'
    ),
)
print(
    (
        'No source rasters have been acquired or processed during step Define Model '
        'Identity and Spatial Framework.The next phase will identify and prepare the '
        'source datasets required for each hazard component.'
    ),
)

print('\n=== FIRE HAZARD MODEL SUMMARY COMPLETE ===')


=== VALIDATING FIRE HAZARD MODEL CONFIGURATION ===
-> Required configuration objects found: 25
-> Hazard components validated: 6
-> Validated model weight total: 1.00
-> Integer index coverage valid: True
-> Continuous class breaks contiguous: True
-> Nonburnable policy valid: True
-> Overall configuration valid: True

--- FIRE HAZARD CONFIGURATION VALIDATION ---


,VALIDATION_CATEGORY,VALID
0,Model Identity,True
1,Study Area,True
2,Spatial Framework,True
3,Raster Output Settings,True
4,Hazard Components,True
5,Model Weights,True
6,Integer Hazard Classes,True
7,Continuous Class Breaks,True
8,Nonburnable and Missing Data Policy,True



NOTE:
The baseline fire-hazard model configuration passed identity, spatial-framework, component, weight, classification, and masking validation.
This validation confirms that the model framework is internally consistent before source datasets are acquired or raster processing begins.

=== FIRE HAZARD MODEL CONFIGURATION VALIDATION COMPLETE ===
=== CREATING FIRE HAZARD MODEL SUMMARY ===
-> Model summary created: Baseline Composite Fire Hazard Grid
-> Model version: 1.0
-> Hazard components summarized: 6
-> Hazard classes summarized: 5
-> Nonburnable classes documented: 5
-> Model weight total: 1.00
-> Configuration validation passed: True

--- FIRE HAZARD MODEL OVERVIEW ---


,SETTING,VALUE
0,Model Name,Baseline Composite Fire Hazard Grid
1,Model Version,1.0
2,Model Scope,Hazard Only
3,Study Counties,"Salt Lake County, Utah County, Washington County"
4,Target CRS,EPSG:26912
5,Target Cell Size,30 meters
6,Normalized Score Range,0.0 to 1.0
7,Public Hazard Index,1 to 10
8,Output Format,GeoTIFF
9,Raster Data Type,float32



--- FIRE HAZARD COMPONENTS AND WEIGHTS ---


...


--- NONBURNABLE-CELL POLICY ---


,NONBURNABLE_CLASS,OUTPUT_POLICY,OUTPUT_VALUE
0,Open Water,NoData,-9999.0
1,Permanent Snow or Ice,NoData,-9999.0
2,Developed Nonburnable Surface,NoData,-9999.0
3,Barren Nonburnable Surface,NoData,-9999.0
4,Other Confirmed Nonburnable Land,NoData,-9999.0



--- MODEL CONFIGURATION VALIDATION ---


,VALIDATION_CATEGORY,VALID
0,Model Identity,True
1,Study Area,True
2,Spatial Framework,True
3,Raster Output Settings,True
4,Hazard Components,True
5,Model Weights,True
6,Integer Hazard Classes,True
7,Continuous Class Breaks,True
8,Nonburnable and Missing Data Policy,True



--- FIRE HAZARD MODEL EQUATION ---
Composite Fire Hazard = (Fuel Hazard × 0.30) + (Fuel Dryness × 0.20) + (Slope Hazard × 0.15) + (Aspect Hazard × 0.10) + (Historical Fire Likelihood × 0.15) + (Human Ignition Potential × 0.10)

NOTE:
The Baseline Composite Fire Hazard Grid framework is now fully defined and internally validated.
The model summary and metadata objects created in this cell can be reused later when exporting the hazard-model documentation and project deliverables.
No source rasters have been acquired or processed during step Define Model Identity and Spatial Framework.The next phase will identify and prepare the source datasets required for each hazard component.

=== FIRE HAZARD MODEL SUMMARY COMPLETE ===


## Project Paths, Cache Policy, and Shared Acquisition Utilities


In [57]:
print('=== DEFINING FIRE HAZARD SOURCE DATA REQUIREMENTS ===')

# List the validated model objects required before component-specific source
# requirements are defined.
required_hazard_requirement_inputs = [
    'fire_hazard_components',
    'fire_hazard_target_crs',
    'fire_hazard_cell_size',
    'fire_hazard_cell_units',
    'fire_hazard_normalized_range',
]
missing_hazard_requirement_inputs = [
    object_name
    for object_name in required_hazard_requirement_inputs
    if object_name not in globals()
]

# Stop if validated model inputs are missing because source-data requirements depend on
# the approved spatial framework.
if missing_hazard_requirement_inputs:
    raise (
        NameError(
            (
                f'The following fire-hazard model objects are missing:\n'
                f'{missing_hazard_requirement_inputs}'
                f'\n\nRun Step Define Model Identity and Spatial Framework before defining '
                f'the source data requirements.'
            ),
        )
    )

# Store study-wide spatial, format, coverage, metadata, and licensing requirements
# shared by every hazard source.
fire_hazard_common_data_requirements = {
    'study_area': 'Salt Lake, Utah, and Washington counties, Utah',
    'target_crs': fire_hazard_target_crs,
    'target_resolution': fire_hazard_cell_size,
    'resolution_units': fire_hazard_cell_units,
    'preferred_raster_format': 'GeoTIFF',
    'preferred_vector_format': 'GeoPackage or GeoJSON',
    'normalized_output_range': fire_hazard_normalized_range,
    'complete_study_area_coverage_required': True,
    'source_metadata_required': True,
    'source_license_documentation_required': True,
}

# Define source-specific acquisition and preprocessing requirements for each hazard
# component.
fire_hazard_data_requirements = {
    'fuel_hazard': {
        'component_name': 'Surface and Canopy Fuel Hazard',
        'required_data_role': (
            'Represent fuel types and structural conditions that influence potential '
            'fire spread and intensity.'
        ),
        'required_source_variables': [
            'Surface fire behavior fuel model',
            'Canopy cover',
            'Canopy bulk density',
        ],
        'optional_source_variables': [
            'Canopy base height',
            'Canopy height',
            'Existing vegetation type',
        ],
        'preferred_data_type': 'Raster',
        'measurement_structure': 'Categorical surface-fuel classes and continuous canopy variables',
        'preferred_native_resolution_meters': 30,
        'maximum_acceptable_resolution_meters': 30,
        'preferred_temporal_basis': 'Most recent authoritative landscape fuel release available',
        'preprocessing_required': [
            'Reproject',
            'Clip',
            'Align',
            'Resample where required',
            'Reclassify categorical fuel models',
            'Normalize continuous canopy variables',
            'Create nonburnable mask',
        ],
        'normalization_rule': (
            'Higher modeled spread or intensity potential receives a higher '
            'normalized hazard score.'
        ),
        'required_for_model': True,
    },
    'fuel_dryness': {
        'component_name': 'Vegetation and Fuel Dryness',
        'required_data_role': (
            'Represent relative vegetation moisture and dryness during the selected '
            'seasonal analysis period.'
        ),
        'required_source_variables': [
            'Near-infrared reflectance',
            'Red reflectance',
            'Shortwave-infrared reflectance',
        ],
        'optional_source_variables': [
            'Precomputed NDVI',
            'Precomputed NDMI',
            'Precomputed vegetation-condition product',
        ],
        'preferred_data_type': 'Raster',
        'measurement_structure': (
            'Continuous satellite-derived spectral bands or vegetation indices'
        ),
        'preferred_native_resolution_meters': 30,
        'maximum_acceptable_resolution_meters': 30,
        'preferred_temporal_basis': 'Cloud-filtered peak fire-season multi-date composite',
        'preprocessing_required': [
            'Cloud and shadow filtering',
            'Seasonal compositing',
            'Reproject',
            'Clip',
            'Align',
            'Calculate vegetation indices',
            'Invert moisture or greenness where needed',
            'Normalize',
        ],
        'normalization_rule': (
            'Drier vegetation and lower moisture conditions receive a higher '
            'normalized hazard score.'
        ),
        'required_for_model': True,
    },
    'slope_hazard': {
        'component_name': 'Slope-Based Spread Potential',
        'required_data_role': (
            'Represent terrain steepness associated with accelerated uphill fire '
            'spread.'
        ),
        'required_source_variables': ['Digital elevation model'],
        'optional_source_variables': ['Precomputed slope'],
        'preferred_data_type': 'Raster',
        'measurement_structure': 'Continuous elevation values from which slope will be derived',
        'preferred_native_resolution_meters': 30,
        'maximum_acceptable_resolution_meters': 30,
        'preferred_temporal_basis': 'Current authoritative bare-earth elevation product',
        'preprocessing_required': [
            'Mosaic source tiles where required',
            'Reproject',
            'Clip',
            'Align',
            'Derive slope',
            'Normalize',
        ],
        'normalization_rule': (
            'Steeper terrain receives a higher normalized spread-potential score.'
        ),
        'required_for_model': True,
    },
    'aspect_hazard': {
        'component_name': 'Aspect and Solar Exposure',
        'required_data_role': (
            'Represent terrain orientation associated with solar exposure and '
            'relative surface dryness.'
        ),
        'required_source_variables': ['Digital elevation model'],
        'optional_source_variables': ['Precomputed aspect', 'Solar radiation surface'],
        'preferred_data_type': 'Raster',
        'measurement_structure': 'Continuous elevation values from which aspect will be derived',
        'preferred_native_resolution_meters': 30,
        'maximum_acceptable_resolution_meters': 30,
        'preferred_temporal_basis': 'Same elevation product used for the slope component',
        'preprocessing_required': [
            'Use terrain-aligned elevation raster',
            'Derive aspect',
            'Handle flat terrain',
            'Reclassify exposure directions',
            'Normalize',
        ],
        'normalization_rule': (
            'Warmer and generally drier terrain orientations receive higher hazard '
            'scores than cooler orientations.'
        ),
        'required_for_model': True,
    },
    'historical_fire_likelihood': {
        'component_name': 'Historical Fire Likelihood',
        'required_data_role': (
            'Represent spatial patterns of documented wildfire occurrence across the '
            'study area.'
        ),
        'required_source_variables': [
            'Historical wildfire ignition locations',
            'Historical wildfire occurrence dates',
        ],
        'optional_source_variables': [
            'Historical fire perimeters',
            'Burn frequency',
            'Time since last fire',
            'Modeled annual burn probability',
        ],
        'preferred_data_type': 'Vector or Raster',
        'measurement_structure': (
            'Point-event records, polygon fire perimeters, or continuous modeled '
            'likelihood values'
        ),
        'preferred_native_resolution_meters': None,
        'maximum_acceptable_resolution_meters': None,
        'preferred_temporal_basis': (
            'Multi-year historical record with a documented and consistent '
            'observation period'
        ),
        'preprocessing_required': [
            'Filter to selected historical period',
            'Remove duplicate events',
            'Validate dates and geometries',
            'Reproject',
            'Clip',
            'Create density, frequency, or likelihood surface',
            'Align',
            'Normalize',
        ],
        'normalization_rule': (
            'Areas with greater historical ignition or modeled fire occurrence '
            'receive a higher normalized likelihood score.'
        ),
        'required_for_model': True,
    },
    'human_ignition_potential': {
        'component_name': 'Human Ignition Potential',
        'required_data_role': (
            'Represent access and development patterns associated with opportunities '
            'for human-caused wildfire ignition.'
        ),
        'required_source_variables': [
            'Road centerlines',
            'Developed land or built-area coverage',
        ],
        'optional_source_variables': [
            'Recreation access points',
            'Railroads',
            'Utility corridors',
            'Campgrounds',
            'Trailheads',
        ],
        'preferred_data_type': 'Vector and Raster',
        'measurement_structure': (
            'Linear transportation features, point access features, and categorical '
            'or continuous developed-land coverage'
        ),
        'preferred_native_resolution_meters': None,
        'maximum_acceptable_resolution_meters': None,
        'preferred_temporal_basis': (
            'Most recent authoritative transportation and developed-land datasets '
            'available'
        ),
        'preprocessing_required': [
            'Validate geometries',
            'Reproject',
            'Clip',
            'Calculate distance or density surfaces',
            'Combine access and development indicators',
            'Align',
            'Normalize',
        ],
        'normalization_rule': (
            'Locations with greater proximity to or density of human access and '
            'development receive a higher ignition-potential score.'
        ),
        'required_for_model': True,
    },
}

# Validate that every source-requirement record contains the fields needed for
# acquisition and downstream
# raster preparation.
required_fire_hazard_requirement_fields = [
    'component_name',
    'required_data_role',
    'required_source_variables',
    'optional_source_variables',
    'preferred_data_type',
    'measurement_structure',
    'preferred_native_resolution_meters',
    'maximum_acceptable_resolution_meters',
    'preferred_temporal_basis',
    'preprocessing_required',
    'normalization_rule',
    'required_for_model',
]
configured_fire_hazard_component_keys = list(fire_hazard_components.keys())
requirement_component_keys = list(fire_hazard_data_requirements.keys())
missing_requirement_components = [
    component_key
    for component_key in configured_fire_hazard_component_keys
    if component_key not in requirement_component_keys
]
unexpected_requirement_components = [
    component_key
    for component_key in requirement_component_keys
    if component_key not in configured_fire_hazard_component_keys
]
fire_hazard_requirement_order_valid = (
    requirement_component_keys
    == configured_fire_hazard_component_keys
)

# Stop if any hazard component lacks a source-data requirement record before acquisition
# planning continues.
if missing_requirement_components:
    raise (
        ValueError(
            (
                f'Source-data requirements are missing for the following hazard components:\n'
                f'{missing_requirement_components}'
            ),
        )
    )

# Reject source-requirement entries that do not correspond to configured hazard
# components.
if unexpected_requirement_components:
    raise (
        ValueError(
            (
                f'Unexpected source-data requirement entries were found:\n'
                f'{unexpected_requirement_components}'
            ),
        )
    )

# Stop if source requirements do not follow model-component order so later summaries
# remain aligned.
if not fire_hazard_requirement_order_valid:
    raise (
        ValueError(
            (
                'The source-data requirements must follow the same component order as the '
                'hazard model.'
            ),
        )
    )

# Initialize a component-level validation dictionary so missing source-requirement
# fields can be accumulated consistently.
missing_hazard_requirement_fields = {}

# Iterate through each hazard component to validate source requirements and build
# consistent
# requirement/preprocessing summaries.
for component_key, requirement_settings in fire_hazard_data_requirements.items():
    missing_component_fields = [
        field_name
        for field_name in required_fire_hazard_requirement_fields
        if field_name not in requirement_settings
    ]

    # Record missing component metadata fields so incomplete hazard-component
    # definitions can be rejected after inspection.
    if missing_component_fields:
        missing_hazard_requirement_fields[component_key] = missing_component_fields

# Stop if any source-requirement record lacks required fields because acquisition rules
# would be incomplete.
if missing_hazard_requirement_fields:
    raise (
        ValueError(
            (
                f'One or more hazard components have incomplete source-data requirements:\n'
                f'{missing_hazard_requirement_fields}'
            ),
        )
    )

# Identify components without required source variables because every modeled hazard
# input needs at least one defined variable.
components_without_required_variables = [
    component_key
    for (component_key, requirement_settings) in fire_hazard_data_requirements.items()
    if (
        not isinstance(requirement_settings['required_source_variables'], list)
        or len(requirement_settings['required_source_variables']) == 0
    )
]

# Stop if a hazard component has no required source variables because its input data
# cannot be defined.
if components_without_required_variables:
    raise (
        ValueError(
            (
                f'The following components do not define at least one required source variable:\n'
                f'{components_without_required_variables}'
            ),
        )
    )

# Initialize records used to convert component source requirements into a
# reader-readable summary table.
fire_hazard_data_requirement_records = []

# Iterate through each hazard component to validate source requirements and build
# consistent
# requirement/preprocessing summaries.
for component_key, requirement_settings in fire_hazard_data_requirements.items():
    preferred_resolution = (
        f"{requirement_settings['preferred_native_resolution_meters']} m"
        if requirement_settings['preferred_native_resolution_meters'] is not None
        else 'Source dependent'
    )
    maximum_resolution = (
        f"{requirement_settings['maximum_acceptable_resolution_meters']} m"
        if requirement_settings['maximum_acceptable_resolution_meters'] is not None
        else 'Source dependent'
    )
    fire_hazard_data_requirement_records.append(
        {
            'COMPONENT_KEY': component_key,
            'COMPONENT': requirement_settings['component_name'],
            'PREFERRED_DATA_TYPE': requirement_settings['preferred_data_type'],
            'REQUIRED_VARIABLES': (
                '; '.join(
                    requirement_settings['required_source_variables'],
                )
            ),
            'OPTIONAL_VARIABLES': (
                '; '.join(
                    requirement_settings['optional_source_variables'],
                )
            ),
            'PREFERRED_RESOLUTION': preferred_resolution,
            'MAXIMUM_RESOLUTION': maximum_resolution,
            'TEMPORAL_BASIS': requirement_settings['preferred_temporal_basis'],
            'REQUIRED_FOR_MODEL': requirement_settings['required_for_model'],
        },
    )

# Convert the validated source-requirement records into a tabular summary for review
# before acquisition.
fire_hazard_data_requirements_summary = (
    pd.DataFrame(
        fire_hazard_data_requirement_records,
    )
)
fire_hazard_preprocessing_requirement_records = []

# Iterate through each hazard component to validate source requirements and build
# consistent
# requirement/preprocessing summaries.
for component_key, requirement_settings in fire_hazard_data_requirements.items():
    fire_hazard_preprocessing_requirement_records.append(
        {
            'COMPONENT_KEY': component_key,
            'COMPONENT': requirement_settings['component_name'],
            'PREPROCESSING_STEPS': (
                ' → '.join(
                    requirement_settings['preprocessing_required'],
                )
            ),
            'NORMALIZATION_RULE': requirement_settings['normalization_rule'],
        },
    )

# Convert preprocessing records into a table that documents how each hazard source must
# be prepared for the common grid.
fire_hazard_preprocessing_requirements_summary = (
    pd.DataFrame(
        fire_hazard_preprocessing_requirement_records,
    )
)
fire_hazard_requirement_summary_count_valid = (
    len(fire_hazard_data_requirements_summary)
    == len(fire_hazard_components)
)
fire_hazard_preprocessing_summary_count_valid = (
    len(fire_hazard_preprocessing_requirements_summary)
    == len(fire_hazard_components)
)

# Stop if the source-requirement summary does not contain one record per hazard
# component.
if not fire_hazard_requirement_summary_count_valid:
    raise (
        ValueError(
            (
                'The source-data requirements summary does not contain one record per '
                'hazard component.'
            ),
        )
    )

# Stop if the preprocessing summary does not contain one record per hazard component.
if not fire_hazard_preprocessing_summary_count_valid:
    raise (
        ValueError(
            (
                'The preprocessing requirements summary does not contain one record per '
                'hazard component.'
            ),
        )
    )

print(f'-> Hazard components represented: {len(fire_hazard_data_requirements)}')
print(
    (
        f'-> Required metadata fields per component: '
        f'{len(required_fire_hazard_requirement_fields)}'
    ),
)
print(f'-> Missing requirement components: {missing_requirement_components}')
print(f'-> Unexpected requirement components: {unexpected_requirement_components}')
print(f'-> Requirement order valid: {fire_hazard_requirement_order_valid}')
print(f'-> Requirement summary complete: {fire_hazard_requirement_summary_count_valid}')
print('\n--- FIRE HAZARD SOURCE DATA REQUIREMENTS ---')
display(fire_hazard_data_requirements_summary)
print('\n--- FIRE HAZARD PREPROCESSING REQUIREMENTS ---')
display(fire_hazard_preprocessing_requirements_summary)
print('\nNOTE:')
print(
    (
        'This cell defines the source-data characteristics required by each hazard '
        'component. It does not yet select a specific dataset, agency, URL, or file.'
    ),
)
print(
    (
        'Specific source products and fallback datasets will be documented in step '
        'Define source catalog.'
    ),
)

print('\n=== FIRE HAZARD SOURCE DATA REQUIREMENTS DEFINED ===')



=== DEFINING FIRE HAZARD SOURCE DATA REQUIREMENTS ===
-> Hazard components represented: 6
-> Required metadata fields per component: 12
-> Missing requirement components: []
-> Unexpected requirement components: []
-> Requirement order valid: True
-> Requirement summary complete: True

--- FIRE HAZARD SOURCE DATA REQUIREMENTS ---


,COMPONENT_KEY,COMPONENT,PREFERRED_DATA_TYPE,REQUIRED_VARIABLES,OPTIONAL_VARIABLES,PREFERRED_RESOLUTION,MAXIMUM_RESOLUTION,TEMPORAL_BASIS,REQUIRED_FOR_MODEL
0,fuel_hazard,Surface and Canopy Fuel Hazard,Raster,Surface fire behavior fuel model; Canopy cover...,Canopy base height; Canopy height; Existing ve...,30 m,30 m,Most recent authoritative landscape fuel relea...,True
1,fuel_dryness,Vegetation and Fuel Dryness,Raster,Near-infrared reflectance; Red reflectance; Sh...,Precomputed NDVI; Precomputed NDMI; Precompute...,30 m,30 m,Cloud-filtered peak fire-season multi-date com...,True
2,slope_hazard,Slope-Based Spread Potential,Raster,Digital elevation model,Precomputed slope,30 m,30 m,Current authoritative bare-earth elevation pro...,True
3,aspect_hazard,Aspect and Solar Exposure,Raster,Digital elevation model,Precomputed aspect; Solar radiation surface,30 m,30 m,Same elevation product used for the slope comp...,True
4,historical_fire_likelihood,Historical Fire Likelihood,Vector or Raster,Historical wildfire ignition locations; Histor...,Historical fire perimeters; Burn frequency; Ti...,Source dependent,Source dependent,Multi-year historical record with a documented...,True
5,human_ignition_potential,Human Ignition Potential,Vector and Raster,Road centerlines; Developed land or built-area...,Recreation access points; Railroads; Utility c...,Source dependent,Source dependent,Most recent authoritative transportation and d...,True



--- FIRE HAZARD PREPROCESSING REQUIREMENTS ---


,COMPONENT_KEY,COMPONENT,PREPROCESSING_STEPS,NORMALIZATION_RULE
0,fuel_hazard,Surface and Canopy Fuel Hazard,Reproject → Clip → Align → Resample where requ...,Higher modeled spread or intensity potential r...
1,fuel_dryness,Vegetation and Fuel Dryness,Cloud and shadow filtering → Seasonal composit...,Drier vegetation and lower moisture conditions...
2,slope_hazard,Slope-Based Spread Potential,Mosaic source tiles where required → Reproject...,Steeper terrain receives a higher normalized s...
3,aspect_hazard,Aspect and Solar Exposure,Use terrain-aligned elevation raster → Derive ...,Warmer and generally drier terrain orientation...
4,historical_fire_likelihood,Historical Fire Likelihood,Filter to selected historical period → Remove ...,Areas with greater historical ignition or mode...
5,human_ignition_potential,Human Ignition Potential,Validate geometries → Reproject → Clip → Calcu...,Locations with greater proximity to or density...



NOTE:
This cell defines the source-data characteristics required by each hazard component. It does not yet select a specific dataset, agency, URL, or file.
Specific source products and fallback datasets will be documented in step Define source catalog.

=== FIRE HAZARD SOURCE DATA REQUIREMENTS DEFINED ===


### Defining Fire Hazard Source Data Catalog


In [58]:
print('=== DEFINING FIRE HAZARD SOURCE DATA CATALOG ===')

# List the validated model and source-requirement objects needed before authoritative
# datasets are entered in the source catalog.
required_fire_hazard_catalog_inputs = [
    'fire_hazard_components',
    'fire_hazard_data_requirements',
    'fire_hazard_target_crs',
    'fire_hazard_cell_size',
]
missing_fire_hazard_catalog_inputs = [
    object_name
    for object_name in required_fire_hazard_catalog_inputs
    if object_name not in globals()
]

# Stop if model or requirement objects are missing because the source catalog is built
# from those validated definitions.
if missing_fire_hazard_catalog_inputs:
    raise (
        NameError(
            (
                f'The following fire-hazard source-catalog inputs are missing:\n'
                f'{missing_fire_hazard_catalog_inputs}'
                f'\n\nRun step Define Data Requirements before defining the source dataset '
                f'catalog.'
            ),
        )
    )

# Assign source-catalog version and selection status metadata so the chosen dataset
# inventory can be tracked across revisions.
fire_hazard_source_catalog_version = '1.0'
fire_hazard_source_catalog_status = 'Selected'

# Build catalog records that connect each hazard component to its authoritative dataset,
# access method, and local filename.
# Define the authoritative and supporting datasets required by the fire-hazard model.
fire_hazard_source_catalog_records = [

    # Define LANDFIRE fuel-model data as the primary surface-fuel hazard input.
    {
        'SOURCE_ID': 'LANDFIRE_FBFM40',
        'COMPONENT_KEY': 'fuel_hazard',
        'COMPONENT': 'Surface and Canopy Fuel Hazard',
        'SOURCE_AGENCY': 'LANDFIRE',
        'SOURCE_NAME': 'Fire Behavior Fuel Model 40',
        'DATASET_NAME': (
            f"{fire_hazard_dataset_versions['landfire']}"
            f' Fire Behavior Fuel Model 40'
        ),
        'DATASET_VERSION': fire_hazard_dataset_versions['landfire'],
        'DATA_ROLE': 'Primary categorical surface-fuel hazard input.',
        'SOURCE_VARIABLE': 'Fire Behavior Fuel Model 40 class',
        'DATA_TYPE': 'Raster',
        'GEOMETRY_OR_STRUCTURE': 'Categorical single-band raster',
        'NATIVE_RESOLUTION_METERS': 30,
        'NATIVE_CRS': 'EPSG:5070',
        'TEMPORAL_COVERAGE': 'Disturbances incorporated through 2025',
        'ACCESS_METHOD': 'ArcGIS ImageServer exportImage',
        'ACCESS_URL': (
            'https://lfps.usgs.gov/arcgis/rest/services/Landfire_LF2025/LF2025_FBFM40'
            '_CONUS/ImageServer'
        ),
        'PREFERRED_FORMAT': 'GeoTIFF',
        'EXPECTED_ARCHIVE_FORMAT': 'Not applicable',
        'LOCAL_FILENAME': 'landfire_2025_fbfm40_study_area.tif',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'Public federal geospatial product; retain source metadata and citation.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define LANDFIRE canopy cover as the canopy-density input for fuel hazard.
    {
        'SOURCE_ID': 'LANDFIRE_CANOPY_COVER',
        'COMPONENT_KEY': 'fuel_hazard',
        'COMPONENT': 'Surface and Canopy Fuel Hazard',
        'SOURCE_AGENCY': 'LANDFIRE',
        'SOURCE_NAME': 'Forest Canopy Cover',
        'DATASET_NAME': (
            f"{fire_hazard_dataset_versions['landfire']} Forest Canopy Cover"
        ),
        'DATASET_VERSION': fire_hazard_dataset_versions['landfire'],
        'DATA_ROLE': 'Continuous canopy-density input used within the fuel-hazard component.',
        'SOURCE_VARIABLE': 'Canopy cover percentage',
        'DATA_TYPE': 'Raster',
        'GEOMETRY_OR_STRUCTURE': 'Continuous single-band raster',
        'NATIVE_RESOLUTION_METERS': 30,
        'NATIVE_CRS': 'EPSG:5070',
        'TEMPORAL_COVERAGE': 'Disturbances incorporated through 2025',
        'ACCESS_METHOD': 'ArcGIS ImageServer exportImage',
        'ACCESS_URL': (
            'https://lfps.usgs.gov/arcgis/rest/services/Landfire_LF2025/LF2025_CC_CON'
            'US/ImageServer'
        ),
        'PREFERRED_FORMAT': 'GeoTIFF',
        'EXPECTED_ARCHIVE_FORMAT': 'Not applicable',
        'LOCAL_FILENAME': 'landfire_2025_canopy_cover_study_area.tif',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'Public federal geospatial product; retain source metadata and citation.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define LANDFIRE canopy bulk density as the canopy-fuel concentration input.
    {
        'SOURCE_ID': 'LANDFIRE_CANOPY_BULK_DENSITY',
        'COMPONENT_KEY': 'fuel_hazard',
        'COMPONENT': 'Surface and Canopy Fuel Hazard',
        'SOURCE_AGENCY': 'LANDFIRE',
        'SOURCE_NAME': 'Forest Canopy Bulk Density',
        'DATASET_NAME': (
            f"{fire_hazard_dataset_versions['landfire']}"
            f' Forest Canopy Bulk Density'
        ),
        'DATASET_VERSION': fire_hazard_dataset_versions['landfire'],
        'DATA_ROLE': (
            'Continuous canopy-fuel concentration input used within the fuel-hazard '
            'component.'
        ),
        'SOURCE_VARIABLE': 'Canopy bulk density',
        'DATA_TYPE': 'Raster',
        'GEOMETRY_OR_STRUCTURE': 'Continuous single-band raster',
        'NATIVE_RESOLUTION_METERS': 30,
        'NATIVE_CRS': 'EPSG:5070',
        'TEMPORAL_COVERAGE': 'Disturbances incorporated through 2025',
        'ACCESS_METHOD': 'ArcGIS ImageServer exportImage',
        'ACCESS_URL': (
            'https://lfps.usgs.gov/arcgis/rest/services/Landfire_LF2025/LF2025_CBD_CO'
            'NUS/ImageServer'
        ),
        'PREFERRED_FORMAT': 'GeoTIFF',
        'EXPECTED_ARCHIVE_FORMAT': 'Not applicable',
        'LOCAL_FILENAME': 'landfire_2025_canopy_bulk_density_study_area.tif',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'Public federal geospatial product; retain source metadata and citation.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define HLS imagery as the seasonal vegetation and fuel-dryness input.
    {
        'SOURCE_ID': 'NASA_HLS_VI',
        'COMPONENT_KEY': 'fuel_dryness',
        'COMPONENT': 'Vegetation and Fuel Dryness',
        'SOURCE_AGENCY': 'NASA Harmonized Landsat Sentinel-2 Program',
        'SOURCE_NAME': 'Harmonized Landsat Sentinel-2 Surface Reflectance',
        'DATASET_NAME': 'HLS L30 and S30 Version 2.0',
        'DATASET_VERSION': fire_hazard_dataset_versions['hls'],
        'DATA_ROLE': (
            'Thirty-meter surface-reflectance and quality source used to calculate a '
            'seasonal NDVI and NDMI vegetation-dryness composite.'
        ),
        'SOURCE_VARIABLE': (
            'Red, near-infrared, shortwave-infrared 1, and Fmask assets'
        ),
        'DATA_TYPE': 'Raster',
        'GEOMETRY_OR_STRUCTURE': 'Tiled 30-meter cloud-optimized GeoTIFF assets',
        'NATIVE_RESOLUTION_METERS': 30,
        'NATIVE_CRS': 'Granule-based UTM/MGRS; verify each item before processing',
        'TEMPORAL_COVERAGE': fire_hazard_dataset_versions['hls_analysis_period'],
        'ACCESS_METHOD': 'Microsoft Planetary Computer STAC API',
        'ACCESS_URL': 'https://planetarycomputer.microsoft.com/api/stac/v1',
        'PREFERRED_FORMAT': 'Cloud Optimized GeoTIFF',
        'EXPECTED_ARCHIVE_FORMAT': 'Not applicable',
        'LOCAL_FILENAME': 'hls_planetary_computer_2025_fire_season',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'Public STAC discovery does not require an interactive login. Asset links '
            'are signed temporarily at runtime.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define 3DEP elevation as the terrain source used to calculate slope hazard.
    {
        'SOURCE_ID': 'USGS_3DEP_DEM',
        'COMPONENT_KEY': 'slope_hazard',
        'COMPONENT': 'Slope-Based Spread Potential',
        'SOURCE_AGENCY': 'U.S. Geological Survey',
        'SOURCE_NAME': '3D Elevation Program',
        'DATASET_NAME': '3DEP 1 Arc-Second Digital Elevation Model',
        'DATASET_VERSION': 'Current downloadable 1 arc-second DEM',
        'DATA_ROLE': 'Bare-earth elevation source used to derive the slope-hazard raster.',
        'SOURCE_VARIABLE': 'Elevation in meters',
        'DATA_TYPE': 'Raster',
        'GEOMETRY_OR_STRUCTURE': 'Continuous tiled elevation raster',
        'NATIVE_RESOLUTION_METERS': 30,
        'NATIVE_CRS': 'NAD83 geographic coordinates',
        'TEMPORAL_COVERAGE': 'Current authoritative 3DEP elevation coverage',
        'ACCESS_METHOD': 'The National Map download application or service',
        'ACCESS_URL': 'https://apps.nationalmap.gov/downloader/',
        'PREFERRED_FORMAT': 'Cloud Optimized GeoTIFF',
        'EXPECTED_ARCHIVE_FORMAT': 'ZIP or GeoTIFF',
        'LOCAL_FILENAME': 'usgs_3dep_1arcsec_dem_utah_tiles',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'USGS public-domain elevation data; retain tile metadata.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Reuse the 3DEP elevation source to derive aspect and solar-exposure hazard.
    {
        'SOURCE_ID': 'USGS_3DEP_DEM_ASPECT',
        'COMPONENT_KEY': 'aspect_hazard',
        'COMPONENT': 'Aspect and Solar Exposure',
        'SOURCE_AGENCY': 'U.S. Geological Survey',
        'SOURCE_NAME': '3D Elevation Program',
        'DATASET_NAME': '3DEP 1 Arc-Second Digital Elevation Model',
        'DATASET_VERSION': 'Current downloadable 1 arc-second DEM',
        'DATA_ROLE': (
            'Same terrain source used to derive the aspect and solar-exposure raster.'
        ),
        'SOURCE_VARIABLE': 'Elevation in meters',
        'DATA_TYPE': 'Raster',
        'GEOMETRY_OR_STRUCTURE': 'Continuous tiled elevation raster',
        'NATIVE_RESOLUTION_METERS': 30,
        'NATIVE_CRS': 'NAD83 geographic coordinates',
        'TEMPORAL_COVERAGE': 'Same elevation coverage used for slope derivation',
        'ACCESS_METHOD': 'Reuse the 3DEP files acquired for the slope component',
        'ACCESS_URL': 'https://apps.nationalmap.gov/downloader/',
        'PREFERRED_FORMAT': 'Cloud Optimized GeoTIFF',
        'EXPECTED_ARCHIVE_FORMAT': 'ZIP or GeoTIFF',
        'LOCAL_FILENAME': 'usgs_3dep_1arcsec_dem_utah_tiles',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'Shared terrain source; do not download a second duplicate DEM '
            'collection.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Shared Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define NIFC fire occurrences as the primary historical wildfire-likelihood input.
    {
        'SOURCE_ID': 'INFORM_FODR_FIRE_OCCURRENCES',
        'COMPONENT_KEY': 'historical_fire_likelihood',
        'COMPONENT': 'Historical Fire Likelihood',
        'SOURCE_AGENCY': 'National Interagency Fire Center',
        'SOURCE_NAME': 'InFORM Fire Occurrence Data Records',
        'DATASET_NAME': 'InFORM Fire Occurrence Data Records (FODR)',
        'DATASET_VERSION': fire_hazard_dataset_versions['inform_fodr'],
        'DATA_ROLE': (
            'Primary authoritative fire-occurrence dataset used to calculate '
            'historical wildfire occurrence density.'
        ),
        'SOURCE_VARIABLE': (
            'Fire location, discovery date, cause, final fire size, and certification '
            'status'
        ),
        'DATA_TYPE': 'Vector',
        'GEOMETRY_OR_STRUCTURE': 'Point features',
        'NATIVE_RESOLUTION_METERS': None,
        'NATIVE_CRS': 'Verify from service metadata',
        'TEMPORAL_COVERAGE': (
            'Historic records with ongoing certified records through 2026'
        ),
        'ACCESS_METHOD': 'NIFC ArcGIS Hub feature-service access',
        'ACCESS_URL': (
            'https://data-nifc.opendata.arcgis.com/datasets/nifc::inform-fire-occurre'
            'nce-data-records/about'
        ),
        'PREFERRED_FORMAT': 'GeoJSON',
        'EXPECTED_ARCHIVE_FORMAT': 'Not applicable',
        'LOCAL_FILENAME': (
            'inform_fodr_utah_fire_occurrences_through_2026.geojson'
        ),
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'Authoritative NIFC occurrence records. Record the acquisition date '
            'because the service is continuously maintained.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define MTBS perimeters as supporting burn-frequency and spatial QA data.
    {
        'SOURCE_ID': 'MTBS_FIRE_PERIMETERS',
        'COMPONENT_KEY': 'historical_fire_likelihood',
        'COMPONENT': 'Historical Fire Likelihood',
        'SOURCE_AGENCY': 'USDA Forest Service and U.S. Geological Survey',
        'SOURCE_NAME': 'Monitoring Trends in Burn Severity',
        'DATASET_NAME': 'MTBS National Burned Area Boundaries',
        'DATASET_VERSION': fire_hazard_dataset_versions['mtbs'],
        'DATA_ROLE': (
            'Supporting completed-fire perimeter dataset used for burn-frequency and '
            'spatial quality assurance.'
        ),
        'SOURCE_VARIABLE': (
            'Fire perimeter, fire year, fire type, and mapped fire identifier'
        ),
        'DATA_TYPE': 'Vector',
        'GEOMETRY_OR_STRUCTURE': 'Polygon features',
        'NATIVE_RESOLUTION_METERS': None,
        'NATIVE_CRS': 'Verify from downloaded metadata',
        'TEMPORAL_COVERAGE': (
            '1984 through the fires completed in the 2026 Q3 MTBS release'
        ),
        'ACCESS_METHOD': 'MTBS national direct-download archive',
        'ACCESS_URL': 'https://www.mtbs.gov/direct-download',
        'PREFERRED_FORMAT': 'Shapefile',
        'EXPECTED_ARCHIVE_FORMAT': 'ZIP',
        'LOCAL_FILENAME': 'mtbs_perimeter_data_2026_q3.zip',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'MTBS maps qualifying completed fires and is not a complete ignition '
            'inventory.'
        ),
        'REQUIRED_FOR_MODEL': False,
        'SOURCE_PRIORITY': 'Supporting',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define TIGER roads as the transportation component of human ignition potential.
    {
        'SOURCE_ID': 'TIGER_ROADS_UTAH',
        'COMPONENT_KEY': 'human_ignition_potential',
        'COMPONENT': 'Human Ignition Potential',
        'SOURCE_AGENCY': 'U.S. Census Bureau',
        'SOURCE_NAME': 'TIGER/Line Shapefiles',
        'DATASET_NAME': (
            f"{fire_hazard_dataset_versions['tiger_roads']} Utah All Roads"
        ),
        'DATASET_VERSION': (
            f"{fire_hazard_dataset_versions['tiger_roads']} TIGER/Line"
        ),
        'DATA_ROLE': (
            'Primary transportation-access input used to calculate road proximity or '
            'road-density surfaces.'
        ),
        'SOURCE_VARIABLE': 'Road centerline and MTFCC road class',
        'DATA_TYPE': 'Vector',
        'GEOMETRY_OR_STRUCTURE': 'Line features',
        'NATIVE_RESOLUTION_METERS': None,
        'NATIVE_CRS': 'Verify from downloaded metadata',
        'TEMPORAL_COVERAGE': '2025 geographic vintage',
        'ACCESS_METHOD': 'Census TIGER/Line FTP download',
        'ACCESS_URL': 'https://www2.census.gov/geo/tiger/TIGER2025/ROADS/',
        'PREFERRED_FORMAT': 'Shapefile',
        'EXPECTED_ARCHIVE_FORMAT': 'ZIP',
        'LOCAL_FILENAME': 'tl_2025_49_roads.zip',
        'REQUIRES_AUTHENTICATION': False,
        'LICENSE_OR_USE_NOTES': (
            'Public U.S. Census Bureau geographic data; retain TIGER vintage '
            'metadata.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },

    # Define Annual NLCD developed land as the built-environment ignition input.
    {
        'SOURCE_ID': 'ANNUAL_NLCD_2025_LAND_COVER',
        'COMPONENT_KEY': 'human_ignition_potential',
        'COMPONENT': 'Human Ignition Potential',
        'SOURCE_AGENCY': (
            'Multi-Resolution Land Characteristics Consortium and U.S. Geological '
            'Survey'
        ),
        'SOURCE_NAME': 'Annual National Land Cover Database',
        'DATASET_NAME': 'Annual NLCD 2025 Land Cover',
        'DATASET_VERSION': fire_hazard_dataset_versions['annual_nlcd'],
        'DATA_ROLE': (
            'Current developed-land input used to identify built surfaces associated '
            'with human ignition opportunity.'
        ),
        'SOURCE_VARIABLE': '2025 annual land-cover class',
        'DATA_TYPE': 'Raster',
        'GEOMETRY_OR_STRUCTURE': 'Categorical single-band raster',
        'NATIVE_RESOLUTION_METERS': 30,
        'NATIVE_CRS': 'Verify from downloaded metadata',
        'TEMPORAL_COVERAGE': 'Annual land cover from 1985 through 2025',
        'ACCESS_METHOD': 'MRLC Annual NLCD data download',
        'ACCESS_URL': 'https://www.mrlc.gov/data/project/annual-nlcd',
        'PREFERRED_FORMAT': 'GeoTIFF',
        'EXPECTED_ARCHIVE_FORMAT': 'ZIP or GeoTIFF',
        'LOCAL_FILENAME': 'annual_nlcd_c1_2_2025_land_cover_conus.tif',
        'REQUIRES_AUTHENTICATION': False,
        # Document source-use requirements and model status for the Annual NLCD input.
        'LICENSE_OR_USE_NOTES': (
            'Public MRLC/USGS land-cover product; retain annual product-version metadata.'
        ),
        'REQUIRED_FOR_MODEL': True,
        'SOURCE_PRIORITY': 'Primary',
        'CATALOG_STATUS': fire_hazard_source_catalog_status,
    },
]
# Convert catalog records into a DataFrame so required fields, IDs, component coverage,
# and source metadata can be validated consistently.
fire_hazard_source_catalog = pd.DataFrame(fire_hazard_source_catalog_records)
required_fire_hazard_catalog_fields = [
    'SOURCE_ID',
    'COMPONENT_KEY',
    'COMPONENT',
    'SOURCE_AGENCY',
    'SOURCE_NAME',
    'DATASET_NAME',
    'DATASET_VERSION',
    'DATA_ROLE',
    'SOURCE_VARIABLE',
    'DATA_TYPE',
    'GEOMETRY_OR_STRUCTURE',
    'NATIVE_RESOLUTION_METERS',
    'NATIVE_CRS',
    'TEMPORAL_COVERAGE',
    'ACCESS_METHOD',
    'ACCESS_URL',
    'PREFERRED_FORMAT',
    'EXPECTED_ARCHIVE_FORMAT',
    'LOCAL_FILENAME',
    'REQUIRES_AUTHENTICATION',
    'LICENSE_OR_USE_NOTES',
    'REQUIRED_FOR_MODEL',
    'SOURCE_PRIORITY',
    'CATALOG_STATUS',
]
missing_fire_hazard_catalog_fields = [
    field_name
    for field_name in required_fire_hazard_catalog_fields
    if field_name not in fire_hazard_source_catalog.columns
]

# Stop if source-catalog records omit required metadata fields before catalog QA
# continues.
if missing_fire_hazard_catalog_fields:
    raise (
        ValueError(
            (
                f'The fire-hazard source catalog is missing required fields:\n'
                f'{missing_fire_hazard_catalog_fields}'
            ),
        )
    )

# Count duplicate source identifiers because each catalog dataset must remain uniquely
# addressable during acquisition and QA.
duplicate_fire_hazard_source_id_count = (
    fire_hazard_source_catalog['SOURCE_ID'].duplicated().sum(
    )
)

# Reject duplicate source IDs so each dataset remains uniquely addressable in
# acquisition and QA tables.
if duplicate_fire_hazard_source_id_count > 0:
    raise (
        ValueError(
            (
                f'The fire-hazard source catalog contains {duplicate_fire_hazard_source_id_count}'
                f' duplicate SOURCE_ID values.'
            ),
        )
    )

# Compare model component keys with catalog component keys so every approved hazard
# input has catalog coverage and no extras are introduced.
approved_fire_hazard_component_keys = list(fire_hazard_components.keys())
catalog_fire_hazard_component_keys = (
    fire_hazard_source_catalog['COMPONENT_KEY'].drop_duplicates().tolist(
    )
)
missing_fire_hazard_catalog_components = [
    component_key
    for component_key in approved_fire_hazard_component_keys
    if component_key not in catalog_fire_hazard_component_keys
]
unexpected_fire_hazard_catalog_components = [
    component_key
    for component_key in catalog_fire_hazard_component_keys
    if component_key not in approved_fire_hazard_component_keys
]

# Stop if configured hazard components are missing from the source catalog because each
# model input requires a source.
if missing_fire_hazard_catalog_components:
    raise (
        ValueError(
            (
                f'The source catalog does not represent the following hazard components:\n'
                f'{missing_fire_hazard_catalog_components}'
            ),
        )
    )

# Reject catalog components not used by the approved hazard model so unplanned sources
# cannot enter processing.
if unexpected_fire_hazard_catalog_components:
    raise (
        ValueError(
            (
                f'The source catalog contains unexpected hazard components:\n'
                f'{unexpected_fire_hazard_catalog_components}'
            ),
        )
    )

# Isolate required catalog sources and count them by component so model-critical data
# coverage can be verified.
required_fire_hazard_source_records = fire_hazard_source_catalog[
    fire_hazard_source_catalog['REQUIRED_FOR_MODEL']
]
required_source_count_by_component = (
    required_fire_hazard_source_records.groupby('COMPONENT_KEY').size(
    )
)
components_without_required_sources = [
    component_key
    for component_key in approved_fire_hazard_component_keys
    if required_source_count_by_component.get(component_key, 0) == 0
]

# Stop if a hazard component lacks at least one required source because the model cannot
# be reproduced without it.
if components_without_required_sources:
    raise (
        ValueError(
            (
                f'The following hazard components do not have a required source dataset:\n'
                f'{components_without_required_sources}'
            ),
        )
    )

# Define catalog fields that must contain values so source provenance, access, and
# local-file metadata remain complete.
required_nonblank_catalog_fields = [
    'SOURCE_ID',
    'COMPONENT_KEY',
    'COMPONENT',
    'SOURCE_AGENCY',
    'SOURCE_NAME',
    'DATASET_NAME',
    'DATASET_VERSION',
    'DATA_ROLE',
    'SOURCE_VARIABLE',
    'DATA_TYPE',
    'TEMPORAL_COVERAGE',
    'ACCESS_METHOD',
    'ACCESS_URL',
    'PREFERRED_FORMAT',
    'LOCAL_FILENAME',
    'SOURCE_PRIORITY',
    'CATALOG_STATUS',
]
blank_fire_hazard_catalog_values = {}

# Iterate through source-catalog records so each dataset receives consistent metadata,
# paths, and
# acquisition checks.
for field_name in required_nonblank_catalog_fields:
    blank_value_count = (
        fire_hazard_source_catalog[field_name].astype(str).str.strip().eq('').sum(
        )
    )

    # Flag blank required catalog values so incomplete source metadata can be identified
    # before acquisition.
    if blank_value_count > 0:
        blank_fire_hazard_catalog_values[field_name] = int(blank_value_count)

# Stop if required source-catalog fields contain blanks because downloads and provenance
# checks need complete metadata.
if blank_fire_hazard_catalog_values:
    raise (
        ValueError(
            (
                f'Required source-catalog fields contain blank values:\n'
                f'{blank_fire_hazard_catalog_values}'
            ),
        )
    )

# Summarize total and required datasets by hazard component for source-catalog QA and
# reporting.
fire_hazard_source_count_summary = (
    fire_hazard_source_catalog.groupby(['COMPONENT_KEY', 'COMPONENT'], as_index=False).agg(
        TOTAL_SOURCES=('SOURCE_ID', 'count'),
        REQUIRED_SOURCES=('REQUIRED_FOR_MODEL', 'sum'),
    )
)
fire_hazard_source_catalog_summary = (
    fire_hazard_source_catalog[
        [
            'SOURCE_ID',
            'COMPONENT',
            'SOURCE_AGENCY',
            'DATASET_NAME',
            'DATASET_VERSION',
            'DATA_TYPE',
            'NATIVE_RESOLUTION_METERS',
            'TEMPORAL_COVERAGE',
            'SOURCE_PRIORITY',
            'REQUIRED_FOR_MODEL',
        ]
    ].copy(
    )
)
print(f'-> Source catalog version: {fire_hazard_source_catalog_version}')
print(f'-> Source records selected: {len(fire_hazard_source_catalog):,}')
print(f'-> Required source records: {len(required_fire_hazard_source_records):,}')
print(f'-> Hazard components represented: {len(catalog_fire_hazard_component_keys)}')
print(f'-> Duplicate source IDs: {duplicate_fire_hazard_source_id_count}')
print(f'-> Missing component sources: {missing_fire_hazard_catalog_components}')
print(f'-> Unexpected component sources: {unexpected_fire_hazard_catalog_components}')
print(f'-> Components without required sources: {components_without_required_sources}')
print('\n--- FIRE HAZARD SOURCE CATALOG ---')
display(fire_hazard_source_catalog_summary)
print('\n--- SOURCE COUNTS BY HAZARD COMPONENT ---')
display(fire_hazard_source_count_summary)
print('\nNOTE:')
print(
    (
        'This cell selects and documents the authoritative source products intended '
        'for the initial Baseline Composite Fire Hazard Grid.'
    ),
)
print(
    (
        'No data were downloaded in this step. Project directories and local source '
        'paths will be created during step Define Directories and Paths.'
    ),
)
print(
    (
        'Access URLs should be treated as catalog metadata. Download endpoints and '
        'service parameters will be validated immediately before acquisition.'
    ),
)

print('\n=== FIRE HAZARD SOURCE CATALOG DEFINED ===')


=== DEFINING FIRE HAZARD SOURCE DATA CATALOG ===
-> Source catalog version: 1.0
-> Source records selected: 10
-> Required source records: 9
-> Hazard components represented: 6
-> Duplicate source IDs: 0
-> Missing component sources: []
-> Unexpected component sources: []
-> Components without required sources: []

--- FIRE HAZARD SOURCE CATALOG ---


,SOURCE_ID,COMPONENT,SOURCE_AGENCY,DATASET_NAME,DATASET_VERSION,DATA_TYPE,NATIVE_RESOLUTION_METERS,TEMPORAL_COVERAGE,SOURCE_PRIORITY,REQUIRED_FOR_MODEL
0,LANDFIRE_FBFM40,Surface and Canopy Fuel Hazard,LANDFIRE,LF2025 Fire Behavior Fuel Model 40,LF2025,Raster,30.0,Disturbances incorporated through 2025,Primary,True
1,LANDFIRE_CANOPY_COVER,Surface and Canopy Fuel Hazard,LANDFIRE,LF2025 Forest Canopy Cover,LF2025,Raster,30.0,Disturbances incorporated through 2025,Primary,True
2,LANDFIRE_CANOPY_BULK_DENSITY,Surface and Canopy Fuel Hazard,LANDFIRE,LF2025 Forest Canopy Bulk Density,LF2025,Raster,30.0,Disturbances incorporated through 2025,Primary,True
3,NASA_HLS_VI,Vegetation and Fuel Dryness,NASA Harmonized Landsat Sentinel-2 Program,HLS L30 and S30 Version 2.0,Version 2.0,Raster,30.0,July-September 2025,Primary,True
4,USGS_3DEP_DEM,Slope-Based Spread Potential,U.S. Geological Survey,3DEP 1 Arc-Second Digital Elevation Model,Current downloadable 1 arc-second DEM,Raster,30.0,Current authoritative 3DEP elevation coverage,Primary,True
5,USGS_3DEP_DEM_ASPECT,Aspect and Solar Exposure,U.S. Geological Survey,3DEP 1 Arc-Second Digital Elevation Model,Current downloadable 1 arc-second DEM,Raster,30.0,Same elevation coverage used for slope derivation,Shared Primary,True
6,INFORM_FODR_FIRE_OCCURRENCES,Historical Fire Likelihood,National Interagency Fire Center,InFORM Fire Occurrence Data Records (FODR),Current through 2026,Vector,NaN,Historic records with ongoing certified record...,Primary,True
7,MTBS_FIRE_PERIMETERS,Historical Fire Likelihood,USDA Forest Service and U.S. Geological Survey,MTBS National Burned Area Boundaries,2026 Q3,Vector,NaN,1984 through the fires completed in the 2026 Q...,Supporting,False
8,TIGER_ROADS_UTAH,Human Ignition Potential,U.S. Census Bureau,2025 Utah All Roads,2025 TIGER/Line,Vector,NaN,2025 geographic vintage,Primary,True
9,ANNUAL_NLCD_2025_LAND_COVER,Human Ignition Potential,Multi-Resolution Land Characteristics Consorti...,Annual NLCD 2025 Land Cover,CONUS Collection 1.2 (1985-2025),Raster,30.0,Annual land cover from 1985 through 2025,Primary,True



--- SOURCE COUNTS BY HAZARD COMPONENT ---


,COMPONENT_KEY,COMPONENT,TOTAL_SOURCES,REQUIRED_SOURCES
0,aspect_hazard,Aspect and Solar Exposure,1,1
1,fuel_dryness,Vegetation and Fuel Dryness,1,1
2,fuel_hazard,Surface and Canopy Fuel Hazard,3,3
3,historical_fire_likelihood,Historical Fire Likelihood,2,1
4,human_ignition_potential,Human Ignition Potential,2,2
5,slope_hazard,Slope-Based Spread Potential,1,1



NOTE:
This cell selects and documents the authoritative source products intended for the initial Baseline Composite Fire Hazard Grid.
No data were downloaded in this step. Project directories and local source paths will be created during step Define Directories and Paths.
Access URLs should be treated as catalog metadata. Download endpoints and service parameters will be validated immediately before acquisition.

=== FIRE HAZARD SOURCE CATALOG DEFINED ===


### Defining Fire Hazard Directories and File Paths


In [59]:
print('=== DEFINING FIRE HAZARD DIRECTORIES AND FILE PATHS ===')

# List catalog and project objects required before local directory and file paths are
# created.
required_fire_hazard_path_inputs = [
    'fire_hazard_source_catalog',
    'fire_hazard_source_catalog_records',
    'fire_hazard_source_catalog_version',
]
missing_fire_hazard_path_inputs = [
    object_name
    for object_name in required_fire_hazard_path_inputs
    if object_name not in globals()
]

# Stop if catalog or project-path inputs are missing because deterministic local source
# paths cannot be constructed.
if missing_fire_hazard_path_inputs:
    raise (
        NameError(
            (
                f'The following fire-hazard source-catalog objects are missing:\n'
                f'{missing_fire_hazard_path_inputs}'
                f'\n\nRun the source-catalog cell before defining project directories and file paths.'
            ),
        )
    )

# Import path utilities needed to construct reproducible project directories and local
# source-file locations.
from pathlib import Path

# Anchor the fire-hazard file structure to the current project so raw, intermediate,
# processed, and output paths remain reproducible.
fire_hazard_project_root = Path.cwd()
fire_hazard_data_directory = fire_hazard_project_root / 'data'
fire_hazard_raw_directory = fire_hazard_data_directory / 'raw' / 'fire_hazard'
fire_hazard_interim_directory = fire_hazard_data_directory / 'interim' / 'fire_hazard'
fire_hazard_processed_directory = (
    (fire_hazard_data_directory / 'processed')
    / 'fire_hazard'
)
fire_hazard_raw_fuels_directory = fire_hazard_raw_directory / 'fuels'
fire_hazard_raw_vegetation_directory = fire_hazard_raw_directory / 'vegetation'
fire_hazard_raw_terrain_directory = fire_hazard_raw_directory / 'terrain'
fire_hazard_raw_historical_fire_directory = (
    fire_hazard_raw_directory
    / 'historical_fire'
)
fire_hazard_raw_human_ignition_directory = fire_hazard_raw_directory / 'human_ignition'
fire_hazard_interim_fuels_directory = fire_hazard_interim_directory / 'fuels'
fire_hazard_interim_vegetation_directory = fire_hazard_interim_directory / 'vegetation'
fire_hazard_interim_terrain_directory = fire_hazard_interim_directory / 'terrain'
fire_hazard_interim_historical_fire_directory = (
    fire_hazard_interim_directory
    / 'historical_fire'
)
fire_hazard_interim_human_ignition_directory = (
    fire_hazard_interim_directory
    / 'human_ignition'
)
fire_hazard_interim_reference_directory = fire_hazard_interim_directory / 'reference'
fire_hazard_processed_components_directory = (
    fire_hazard_processed_directory
    / 'components'
)
fire_hazard_processed_composite_directory = (
    fire_hazard_processed_directory
    / 'composite'
)
fire_hazard_processed_metadata_directory = fire_hazard_processed_directory / 'metadata'
fire_hazard_outputs_directory = fire_hazard_project_root / 'outputs'
fire_hazard_outputs_model_directory = fire_hazard_outputs_directory / 'fire_hazard'
fire_hazard_outputs_raster_directory = fire_hazard_outputs_model_directory / 'rasters'
fire_hazard_outputs_table_directory = fire_hazard_outputs_model_directory / 'tables'
fire_hazard_outputs_figure_directory = fire_hazard_outputs_model_directory / 'figures'
fire_hazard_outputs_metadata_directory = (
    fire_hazard_outputs_model_directory
    / 'metadata'
)

# Define the directory structure that separates raw sources, intermediate rasters,
# processed products,
# metadata, tables, and maps.
fire_hazard_directories = {
    'project_root': fire_hazard_project_root,
    'data': fire_hazard_data_directory,
    'raw': fire_hazard_raw_directory,
    'raw_fuels': fire_hazard_raw_fuels_directory,
    'raw_vegetation': fire_hazard_raw_vegetation_directory,
    'raw_terrain': fire_hazard_raw_terrain_directory,
    'raw_historical_fire': fire_hazard_raw_historical_fire_directory,
    'raw_human_ignition': fire_hazard_raw_human_ignition_directory,
    'interim': fire_hazard_interim_directory,
    'interim_fuels': fire_hazard_interim_fuels_directory,
    'interim_vegetation': fire_hazard_interim_vegetation_directory,
    'interim_terrain': fire_hazard_interim_terrain_directory,
    'interim_historical_fire': fire_hazard_interim_historical_fire_directory,
    'interim_human_ignition': fire_hazard_interim_human_ignition_directory,
    'interim_reference': fire_hazard_interim_reference_directory,
    'processed': fire_hazard_processed_directory,
    'processed_components': fire_hazard_processed_components_directory,
    'processed_composite': fire_hazard_processed_composite_directory,
    'processed_metadata': fire_hazard_processed_metadata_directory,
    'outputs': fire_hazard_outputs_directory,
    'outputs_model': fire_hazard_outputs_model_directory,
    'outputs_rasters': fire_hazard_outputs_raster_directory,
    'outputs_tables': fire_hazard_outputs_table_directory,
    'outputs_figures': fire_hazard_outputs_figure_directory,
    'outputs_metadata': fire_hazard_outputs_metadata_directory,
}

# Create every configured project directory before source files, intermediate rasters,
# or outputs are written.
for directory_path in fire_hazard_directories.values():
    # Create the required destination directory before files are downloaded, extracted,
    # or written.
    directory_path.mkdir(parents=True, exist_ok=True)

# Map each hazard component to its raw-data directory so catalog filenames resolve to
# deterministic local
# paths.
fire_hazard_component_raw_directories = {
    'fuel_hazard': fire_hazard_raw_fuels_directory,
    'fuel_dryness': fire_hazard_raw_vegetation_directory,
    'slope_hazard': fire_hazard_raw_terrain_directory,
    'aspect_hazard': fire_hazard_raw_terrain_directory,
    'historical_fire_likelihood': fire_hazard_raw_historical_fire_directory,
    'human_ignition_potential': fire_hazard_raw_human_ignition_directory,
}

# Map each hazard component to its interim-processing directory so derived rasters are
# routed consistently.
fire_hazard_component_interim_directories = {
    'fuel_hazard': fire_hazard_interim_fuels_directory,
    'fuel_dryness': fire_hazard_interim_vegetation_directory,
    'slope_hazard': fire_hazard_interim_terrain_directory,
    'aspect_hazard': fire_hazard_interim_terrain_directory,
    'historical_fire_likelihood': fire_hazard_interim_historical_fire_directory,
    'human_ignition_potential': fire_hazard_interim_human_ignition_directory,
}
fire_hazard_source_catalog_with_paths = fire_hazard_source_catalog.copy()

# Build deterministic local source paths from component keys so catalog records map to
# the correct raw-data
# directories.
def build_fire_hazard_source_path(component_key, local_filename):

    """
    Builds the local raw-data path for one selected
    fire-hazard source file.

    Parameters
    ----------
    component_key : str
        Hazard-model component associated with the
        source dataset.

    local_filename : str
        Filename or local source-directory name defined
        in the source catalog.

    Returns
    -------
    pathlib.Path
        Complete local project path assigned to the
        selected source dataset.
    """

    # Reject unknown component keys so source files cannot be routed into an incorrect
    # raw-data directory.
    if component_key not in fire_hazard_component_raw_directories:
        raise (
            KeyError(
                f'No raw source directory is configured for the component: {component_key}',
            )
        )
    return fire_hazard_component_raw_directories[component_key] / local_filename

# Apply the component path helper to every catalog record so each selected source
# receives a deterministic local destination.
fire_hazard_source_catalog_with_paths['LOCAL_PATH'] = [
    build_fire_hazard_source_path(component_key=component_key, local_filename=local_filename)
    for (component_key, local_filename) in zip(
        fire_hazard_source_catalog_with_paths['COMPONENT_KEY'],
        fire_hazard_source_catalog_with_paths['LOCAL_FILENAME'],
    )
]

# Promote the path-enriched working catalog as the active source catalog used by later
# acquisition and QA steps.
fire_hazard_source_catalog = fire_hazard_source_catalog_with_paths

# Build the local source path for each catalog record and attach those paths back to the
# source catalog.
fire_hazard_source_paths = {
    source_id: local_path
    for (source_id, local_path) in zip(
        fire_hazard_source_catalog['SOURCE_ID'],
        fire_hazard_source_catalog['LOCAL_PATH'],
    )
}
fire_hazard_reference_raster_path = (
    fire_hazard_interim_reference_directory
    / 'fire_hazard_reference_grid.tif'
)
fire_hazard_study_area_mask_path = (
    fire_hazard_interim_reference_directory
    / 'fire_hazard_study_area_mask.tif'
)
fire_hazard_burnable_mask_path = (
    fire_hazard_interim_reference_directory
    / 'fire_hazard_burnable_mask.tif'
)

# Define processed component-raster paths used later to assemble the aligned
# weighted-overlay inputs.
fire_hazard_fuel_component_path = (
    fire_hazard_processed_components_directory
    / 'fuel_hazard_normalized.tif'
)
fire_hazard_dryness_component_path = (
    fire_hazard_processed_components_directory
    / 'fuel_dryness_normalized.tif'
)
fire_hazard_slope_component_path = (
    fire_hazard_processed_components_directory
    / 'slope_hazard_normalized.tif'
)

# Continue defining processed component-raster paths for aspect, historical-fire, and
# human-ignition inputs.
fire_hazard_aspect_component_path = (
    fire_hazard_processed_components_directory
    / 'aspect_hazard_normalized.tif'
)
fire_hazard_history_component_path = (
    fire_hazard_processed_components_directory
    / 'historical_fire_likelihood_normalized.tif'
)
fire_hazard_human_ignition_component_path = (
    fire_hazard_processed_components_directory
    / 'human_ignition_potential_normalized.tif'
)
fire_hazard_processed_component_paths = {
    'fuel_hazard': fire_hazard_fuel_component_path,
    'fuel_dryness': fire_hazard_dryness_component_path,
    'slope_hazard': fire_hazard_slope_component_path,
    'aspect_hazard': fire_hazard_aspect_component_path,
    'historical_fire_likelihood': fire_hazard_history_component_path,
    'human_ignition_potential': fire_hazard_human_ignition_component_path,
}

# Define final composite output paths for the continuous score, 1–10 index, and
# descriptive hazard class rasters.
fire_hazard_continuous_output_path = (
    fire_hazard_processed_composite_directory
    / 'composite_fire_hazard_continuous.tif'
)
fire_hazard_index_output_path = (
    fire_hazard_processed_composite_directory
    / 'composite_fire_hazard_index_1_10.tif'
)
fire_hazard_class_output_path = (
    fire_hazard_processed_composite_directory
    / 'composite_fire_hazard_classes.tif'
)

# Define output paths for metadata, source catalogs, weights, statistics, and
# county-level hazard summaries.
fire_hazard_model_metadata_path = (
    fire_hazard_processed_metadata_directory
    / 'fire_hazard_model_metadata.json'
)
fire_hazard_source_catalog_path = (
    fire_hazard_processed_metadata_directory
    / 'fire_hazard_source_catalog.csv'
)
fire_hazard_weights_path = (
    fire_hazard_processed_metadata_directory
    / 'fire_hazard_weights.csv'
)
fire_hazard_component_statistics_path = (
    fire_hazard_outputs_table_directory
    / 'fire_hazard_component_statistics.csv'
)
fire_hazard_county_summary_path = (
    fire_hazard_outputs_table_directory
    / 'fire_hazard_class_area_by_county.csv'
)

# Confirm that every required project directory exists after directory creation.
missing_fire_hazard_directories = [
    directory_name
    for (directory_name, directory_path) in fire_hazard_directories.items()
    if not directory_path.exists()
]

# Stop if any required project directory was not created before files are downloaded or
# written.
if missing_fire_hazard_directories:
    raise (
        OSError(
            (
                f'The following fire-hazard project directories could not be created:\n'
                f'{missing_fire_hazard_directories}'
            ),
        )
    )

# Validate that catalog source paths are Path objects and that unintended duplicate
# local paths are not
# present.
fire_hazard_source_paths_valid = (
    all(
        (
            isinstance(local_path, Path)
            for local_path in fire_hazard_source_catalog['LOCAL_PATH']
        ),
    )
)
duplicate_fire_hazard_local_path_count = (
    fire_hazard_source_catalog['LOCAL_PATH'].astype(str).duplicated().sum(
    )
)

# Declare the intentional shared DEM source IDs before checking whether unrelated
# catalog records resolve to duplicate local paths.
expected_shared_source_ids = {'USGS_3DEP_DEM', 'USGS_3DEP_DEM_ASPECT'}
duplicate_path_records = fire_hazard_source_catalog[
    fire_hazard_source_catalog['LOCAL_PATH'].astype(str).duplicated(keep=False)
]
unexpected_duplicate_path_records = duplicate_path_records[
    ~duplicate_path_records['SOURCE_ID'].isin(expected_shared_source_ids)
]

# Stop if catalog local paths are invalid because acquisition functions require
# deterministic pathlib destinations.
if not fire_hazard_source_paths_valid:
    raise (
        TypeError(
            (
                'One or more fire-hazard catalog records do not contain a valid '
                'pathlib.Path LOCAL_PATH.'
            ),
        )
    )

# Stop if unrelated source records resolve to the same local path because one download
# could overwrite another dataset.
if not unexpected_duplicate_path_records.empty:
    raise (
        ValueError(
            (
                'Unexpected duplicate local source paths were found in the fire-hazard '
                'source catalog.'
            ),
        )
    )

# Build directory and source-path QA tables so the configured file structure can be
# reviewed before
# acquisition.
fire_hazard_directory_summary = (
    pd.DataFrame(
        [
            {
                'DIRECTORY_KEY': directory_name,
                'DIRECTORY_PATH': str(directory_path),
                'EXISTS': directory_path.exists(),
            }
            for (directory_name, directory_path) in fire_hazard_directories.items()
        ],
    )
)
fire_hazard_source_path_summary = (
    fire_hazard_source_catalog[
        ['SOURCE_ID', 'COMPONENT_KEY', 'DATASET_NAME', 'LOCAL_FILENAME', 'LOCAL_PATH']
    ].copy(
    )
)
fire_hazard_source_path_summary['LOCAL_PATH'] = (
    fire_hazard_source_path_summary['LOCAL_PATH'].astype(
        str,
    )
)
print(f'-> Project root: {fire_hazard_project_root}')
print(f'-> Directories configured: {len(fire_hazard_directories)}')
print(f'-> Missing directories: {missing_fire_hazard_directories}')
print(f'-> Source paths assigned: {len(fire_hazard_source_paths)}')
print(f'-> Duplicate local paths detected: {duplicate_fire_hazard_local_path_count}')
print(f'-> Source paths valid: {fire_hazard_source_paths_valid}')
print('\n--- FIRE HAZARD PROJECT DIRECTORIES ---')
display(fire_hazard_directory_summary)
print('\n--- FIRE HAZARD SOURCE FILE PATHS ---')
display(fire_hazard_source_path_summary)
print('\nNOTE:')
print(
    (
        'Raw source files will remain unchanged within the data/raw/fire_hazard '
        'directory structure.'
    ),
)
print(
    (
        'Intermediate reprojection, clipping, mosaicking, alignment, and derivation '
        'products will be stored under data/interim/fire_hazard.'
    ),
)
print(
    (
        'Final normalized components and composite hazard rasters will be stored '
        'under data/processed/fire_hazard.'
    ),
)
print(
    (
        'The shared 3DEP DEM source path is intentionally assigned to both the slope '
        'and aspect components to avoid duplicate downloads.'
    ),
)

print('\n=== FIRE HAZARD DIRECTORIES AND FILE PATHS DEFINED ===')


=== DEFINING FIRE HAZARD DIRECTORIES AND FILE PATHS ===
-> Project root: C:\Users\adamd\Projects\WUI
-> Directories configured: 25
-> Missing directories: []
-> Source paths assigned: 10
-> Duplicate local paths detected: 1
-> Source paths valid: True

--- FIRE HAZARD PROJECT DIRECTORIES ---


,DIRECTORY_KEY,DIRECTORY_PATH,EXISTS
0,project_root,C:\Users\adamd\Projects\WUI,True
1,data,C:\Users\adamd\Projects\WUI\data,True
2,raw,C:\Users\adamd\Projects\WUI\data\raw\fire_hazard,True
3,raw_fuels,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
4,raw_vegetation,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
5,raw_terrain,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
6,raw_historical_fire,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
7,raw_human_ignition,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
8,interim,C:\Users\adamd\Projects\WUI\data\interim\fire_...,True
9,interim_fuels,C:\Users\adamd\Projects\WUI\data\interim\fire_...,True



--- FIRE HAZARD SOURCE FILE PATHS ---


,SOURCE_ID,COMPONENT_KEY,DATASET_NAME,LOCAL_FILENAME,LOCAL_PATH
0,LANDFIRE_FBFM40,fuel_hazard,LF2025 Fire Behavior Fuel Model 40,landfire_2025_fbfm40_study_area.tif,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
1,LANDFIRE_CANOPY_COVER,fuel_hazard,LF2025 Forest Canopy Cover,landfire_2025_canopy_cover_study_area.tif,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
2,LANDFIRE_CANOPY_BULK_DENSITY,fuel_hazard,LF2025 Forest Canopy Bulk Density,landfire_2025_canopy_bulk_density_study_area.tif,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
3,NASA_HLS_VI,fuel_dryness,HLS L30 and S30 Version 2.0,hls_planetary_computer_2025_fire_season,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
4,USGS_3DEP_DEM,slope_hazard,3DEP 1 Arc-Second Digital Elevation Model,usgs_3dep_1arcsec_dem_utah_tiles,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
5,USGS_3DEP_DEM_ASPECT,aspect_hazard,3DEP 1 Arc-Second Digital Elevation Model,usgs_3dep_1arcsec_dem_utah_tiles,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
6,INFORM_FODR_FIRE_OCCURRENCES,historical_fire_likelihood,InFORM Fire Occurrence Data Records (FODR),inform_fodr_utah_fire_occurrences_through_2026...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
7,MTBS_FIRE_PERIMETERS,historical_fire_likelihood,MTBS National Burned Area Boundaries,mtbs_perimeter_data_2026_q3.zip,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
8,TIGER_ROADS_UTAH,human_ignition_potential,2025 Utah All Roads,tl_2025_49_roads.zip,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
9,ANNUAL_NLCD_2025_LAND_COVER,human_ignition_potential,Annual NLCD 2025 Land Cover,annual_nlcd_c1_2_2025_land_cover_conus.tif,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...



NOTE:
Raw source files will remain unchanged within the data/raw/fire_hazard directory structure.
Intermediate reprojection, clipping, mosaicking, alignment, and derivation products will be stored under data/interim/fire_hazard.
Final normalized components and composite hazard rasters will be stored under data/processed/fire_hazard.
The shared 3DEP DEM source path is intentionally assigned to both the slope and aspect components to avoid duplicate downloads.

=== FIRE HAZARD DIRECTORIES AND FILE PATHS DEFINED ===


### Defining Fire Hazard Snapshot and Refresh Policy


In [60]:
print('=== DEFINING FIRE HAZARD SNAPSHOT AND REFRESH POLICY ===')

# List source-catalog and path objects required before cache and refresh behavior is
# configured.
required_fire_hazard_cache_inputs = [
    'fire_hazard_source_catalog',
    'fire_hazard_source_paths',
    'fire_hazard_raw_directory',
    'fire_hazard_interim_directory',
    'fire_hazard_processed_directory',
]
missing_fire_hazard_cache_inputs = [
    object_name
    for object_name in required_fire_hazard_cache_inputs
    if object_name not in globals()
]

# Stop if source-catalog or path objects are missing because cache status cannot be
# evaluated reliably.
if missing_fire_hazard_cache_inputs:
    raise (
        NameError(
            (
                f'The following fire-hazard cache-policy inputs are missing:\n'
                f'{missing_fire_hazard_cache_inputs}'
                f'\n\nRun step not completed before defining the snapshot and refresh policy.'
            ),
        )
    )

# Select snapshot or refresh mode; this single setting controls source reuse, overwrite
# behavior, and
# extraction refresh decisions.
fire_hazard_data_mode = 'snapshot'
valid_fire_hazard_data_modes = ['snapshot', 'refresh']

# Reject unsupported acquisition modes before cache and overwrite flags are derived from
# them.
if fire_hazard_data_mode not in valid_fire_hazard_data_modes:
    raise ValueError("fire_hazard_data_mode must be either 'snapshot' or 'refresh'.")

# Derive cache reuse and overwrite flags from the selected data mode so acquisition
# behavior remains
# internally consistent.
refresh_fire_hazard_sources = fire_hazard_data_mode == 'refresh'
fire_hazard_use_existing_cache = fire_hazard_data_mode == 'snapshot'
fire_hazard_allow_source_overwrite = fire_hazard_data_mode == 'refresh'
fire_hazard_preserve_cached_sources = fire_hazard_data_mode == 'snapshot'
fire_hazard_allow_automatic_refresh_fallback = False

# Define how missing optional versus required sources are handled during acquisition
# validation.
fire_hazard_missing_optional_source_policy = 'Report and continue'
fire_hazard_missing_required_source_policy = 'Fail validation'

# Configure temporary-file, atomic-replacement, and failure-cleanup safeguards used
# during source downloads.
fire_hazard_temporary_download_suffix = '.part'
fire_hazard_minimum_download_size_bytes = 1
fire_hazard_validate_before_replace = True
fire_hazard_use_atomic_file_replacement = True
fire_hazard_remove_failed_temporary_files = True
fire_hazard_preserve_cache_on_refresh_failure = True

# Configure connection/read timeouts and retry limits for resilient access to remote
# geospatial data
# services.
fire_hazard_connection_timeout_seconds = 15
fire_hazard_read_timeout_seconds = 120
fire_hazard_request_timeout = (
    fire_hazard_connection_timeout_seconds,
    fire_hazard_read_timeout_seconds,
)
fire_hazard_retry_total = 5
fire_hazard_retry_connect = 5
fire_hazard_retry_read = 5
fire_hazard_retry_status = 5
fire_hazard_retry_backoff_factor = 1

# Specify transient HTTP status codes and methods that are eligible for automatic retry.
fire_hazard_retry_status_codes = [429, 500, 502, 503, 504]
fire_hazard_retry_allowed_methods = ['GET', 'HEAD']
fire_hazard_respect_retry_after_header = True

# Configure streamed chunked downloads so large rasters and archives are written without
# loading entire
# files into memory.
fire_hazard_stream_downloads = True
fire_hazard_download_chunk_size_bytes = 1024 * 1024
fire_hazard_download_chunk_size_mb = (
    fire_hazard_download_chunk_size_bytes
    / (1024 * 1024)
)

# Define post-download validation rules for file existence, size, extension, and
# response metadata.
fire_hazard_require_downloaded_file_exists = True
fire_hazard_require_nonempty_downloads = True
fire_hazard_validate_file_extensions = True
fire_hazard_record_content_type = True
fire_hazard_fail_on_unexpected_content_type = False

# Document integrity-validation behavior when an authoritative source checksum is
# available.
fire_hazard_checksum_policy = 'Validate when an authoritative checksum is available'

# Configure supported archive types, cache reuse, overwrite behavior, and path-safety
# requirements for
# compressed GIS sources.
fire_hazard_extract_archives = True
fire_hazard_supported_archive_extensions = ['.zip', '.tar', '.tar.gz', '.tgz']
fire_hazard_reuse_extracted_cache = fire_hazard_data_mode == 'snapshot'
fire_hazard_overwrite_extracted_files = fire_hazard_data_mode == 'refresh'
fire_hazard_require_safe_archive_extraction = True

# Identify authenticated sources that require manual acquisition while keeping
# credentials out of the
# notebook.
fire_hazard_manual_acquisition_source_ids = (
    fire_hazard_source_catalog.loc[
        fire_hazard_source_catalog['REQUIRES_AUTHENTICATION'],
        'SOURCE_ID',
    ].tolist(
    )
)
fire_hazard_manual_acquisition_policy = (
    'User downloads authenticated sources to the configured local source path'
)
fire_hazard_store_credentials_in_notebook = False

# Create a working catalog used to evaluate which configured source files already exist
# locally.
fire_hazard_cache_status_catalog = fire_hazard_source_catalog.copy()
fire_hazard_cache_status_catalog['CACHE_EXISTS'] = (
    fire_hazard_cache_status_catalog['LOCAL_PATH'].apply(
        lambda source_path: source_path.exists(),
    )
)
fire_hazard_cache_status_catalog['CACHE_SIZE_BYTES'] = (
    fire_hazard_cache_status_catalog['LOCAL_PATH'].apply(
        lambda source_path: (
            source_path.stat().st_size
            if source_path.exists() and source_path.is_file()
            else None
        ),
    )
)

# Determine whether each source should be reused, downloaded, or refreshed from the
# selected snapshot mode and
# local file state.
def determine_fire_hazard_cache_action(source_path):

    """
    Determines the intended acquisition action for one
    configured fire-hazard source path.

    Parameters
    ----------
    source_path : pathlib.Path
        Local project path assigned to the source.

    Returns
    -------
    str
        Intended cache or refresh action.
    """

    # Use refresh behavior when the project is configured to reacquire source data.
    if refresh_fire_hazard_sources:

        # Branch on local file availability so the cache action reflects whether a
        # usable source is already
        # present.
        if source_path.exists():
            return 'Refresh and replace after validation'
        return 'Download and create cache'

    # Branch on local file availability so the cache action reflects whether a usable
    # source is already
    # present.
    if source_path.exists():
        return 'Use cached source'
    return 'Missing from cache'

# Apply the cache-decision helper to each source so the catalog records whether data
# will be reused, refreshed, or reported missing.
fire_hazard_cache_status_catalog['ACQUISITION_ACTION'] = (
    fire_hazard_cache_status_catalog['LOCAL_PATH'].apply(
        determine_fire_hazard_cache_action,
    )
)

# Separate required source records and summarize missing cache entries before
# acquisition functions are
# used.
required_fire_hazard_cache_records = fire_hazard_cache_status_catalog[
    fire_hazard_cache_status_catalog['REQUIRED_FOR_MODEL']
]
available_required_fire_hazard_cache_count = (
    required_fire_hazard_cache_records['CACHE_EXISTS'].sum(
    )
)
missing_required_fire_hazard_cache_ids = (
    required_fire_hazard_cache_records.loc[
        ~required_fire_hazard_cache_records['CACHE_EXISTS'],
        'SOURCE_ID',
    ].tolist(
    )
)
missing_optional_fire_hazard_cache_ids = (
    fire_hazard_cache_status_catalog.loc[
        ~fire_hazard_cache_status_catalog['REQUIRED_FOR_MODEL']
        & ~fire_hazard_cache_status_catalog['CACHE_EXISTS'],
        'SOURCE_ID',
    ].tolist(
    )
)

# Validate timeout, retry, chunk-size, temporary-file, and archive settings before
# remote requests are
# allowed.
fire_hazard_timeout_configuration_valid = (
    fire_hazard_connection_timeout_seconds > 0
    and fire_hazard_read_timeout_seconds > 0
)
fire_hazard_retry_configuration_valid = (
    all(
        (
            isinstance(retry_value, int) and retry_value >= 0
            for retry_value in [
                fire_hazard_retry_total,
                fire_hazard_retry_connect,
                fire_hazard_retry_read,
                fire_hazard_retry_status,
            ]
        ),
    )
)
fire_hazard_chunk_configuration_valid = (
    isinstance(fire_hazard_download_chunk_size_bytes, int)
    and fire_hazard_download_chunk_size_bytes > 0
)
fire_hazard_temporary_suffix_valid = (
    isinstance(fire_hazard_temporary_download_suffix, str)
    and fire_hazard_temporary_download_suffix.startswith('.')
    and (len(fire_hazard_temporary_download_suffix) > 1)
)
fire_hazard_archive_policy_valid = (
    isinstance(fire_hazard_supported_archive_extensions, list)
    and len(fire_hazard_supported_archive_extensions) > 0
    and all(
        (
            isinstance(archive_extension, str) and archive_extension.startswith('.')
            for archive_extension in fire_hazard_supported_archive_extensions
        ),
    )
)

# Stop if connection or read timeouts are invalid, preventing predictable
# network-failure handling.
if not fire_hazard_timeout_configuration_valid:
    raise (
        ValueError(
            'Fire-hazard connection and read timeouts must both be greater than zero.',
        )
    )

# Stop if retry counts are invalid before the retry strategy is constructed.
if not fire_hazard_retry_configuration_valid:
    raise (
        ValueError(
            'Fire-hazard retry settings must contain nonnegative integer values.',
        )
    )

# Stop if the download chunk size is invalid before streamed file transfers begin.
if not fire_hazard_chunk_configuration_valid:
    raise ValueError('The fire-hazard download chunk size must be a positive integer.')

# Stop if the temporary-file suffix is invalid because atomic download replacement
# depends on a separate
# staging path.
if not fire_hazard_temporary_suffix_valid:
    raise (
        ValueError(
            (
                'The temporary download suffix must be a non-empty extension beginning '
                'with a period.'
            ),
        )
    )

# Stop if supported archive-format configuration is empty or malformed before extraction
# helpers are
# created.
if not fire_hazard_archive_policy_valid:
    raise (
        ValueError(
            'The supported archive-extension list is not configured correctly.',
        )
    )

# Build QA tables that document cache policy and current source availability.
# Summarize the active cache, download, retry, validation, and archive settings for QA review.
fire_hazard_cache_policy_summary = (
    pd.DataFrame(
        [
            {'POLICY_SETTING': 'Data Mode', 'VALUE': fire_hazard_data_mode},
            {
                'POLICY_SETTING': 'Use Existing Cache',
                'VALUE': fire_hazard_use_existing_cache,
            },
            {
                'POLICY_SETTING': 'Allow Source Overwrite',
                'VALUE': fire_hazard_allow_source_overwrite,
            },
            {
                'POLICY_SETTING': 'Automatic Refresh Fallback',
                'VALUE': fire_hazard_allow_automatic_refresh_fallback,
            },
            {
                'POLICY_SETTING': 'Connection Timeout',
                'VALUE': f'{fire_hazard_connection_timeout_seconds} seconds',
            },
            {
                'POLICY_SETTING': 'Read Timeout',
                'VALUE': f'{fire_hazard_read_timeout_seconds} seconds',
            },
            {'POLICY_SETTING': 'Total Retries', 'VALUE': fire_hazard_retry_total},
            {
                'POLICY_SETTING': 'Retry Backoff Factor',
                'VALUE': fire_hazard_retry_backoff_factor,
            },
            {
                'POLICY_SETTING': 'Streaming Downloads',
                'VALUE': fire_hazard_stream_downloads,
            },
            {
                'POLICY_SETTING': 'Download Chunk Size',
                'VALUE': f'{fire_hazard_download_chunk_size_mb:.0f} MB',
            },
            {
                'POLICY_SETTING': 'Temporary Download Suffix',
                'VALUE': fire_hazard_temporary_download_suffix,
            },
            {
                'POLICY_SETTING': 'Validate Before Replace',
                'VALUE': fire_hazard_validate_before_replace,
            },
            {
                'POLICY_SETTING': 'Preserve Cache on Refresh Failure',
                'VALUE': fire_hazard_preserve_cache_on_refresh_failure,
            },
            {
                'POLICY_SETTING': 'Checksum Policy',
                'VALUE': fire_hazard_checksum_policy,
            },
            {
                'POLICY_SETTING': 'Extract Archives',
                'VALUE': fire_hazard_extract_archives,
            },
            {
                'POLICY_SETTING': 'Store Credentials in Notebook',
                'VALUE': fire_hazard_store_credentials_in_notebook,
            },
        ],
    )
)

# Retain the source-level fields needed to review cache availability and planned
# acquisition actions.
fire_hazard_source_cache_summary = (
    fire_hazard_cache_status_catalog[
        [
            'SOURCE_ID',
            'COMPONENT_KEY',
            'DATASET_NAME',
            'REQUIRED_FOR_MODEL',
            'CACHE_EXISTS',
            'CACHE_SIZE_BYTES',
            'ACQUISITION_ACTION',
        ]
    ].copy()
)

# Report the active cache policy and availability of required and optional source datasets.
print(f'-> Fire-hazard data mode: {fire_hazard_data_mode}')
print(f'-> Refresh source files: {refresh_fire_hazard_sources}')
print(f'-> Use existing cache: {fire_hazard_use_existing_cache}')
print(
    (
        f'-> Required source records available: '
        f'{int(available_required_fire_hazard_cache_count)} of '
        f'{len(required_fire_hazard_cache_records)}'
    ),
)
print(f'-> Missing required cached sources: {missing_required_fire_hazard_cache_ids}')
print(f'-> Missing optional cached sources: {missing_optional_fire_hazard_cache_ids}')

# Identify sources that require authentication or manual acquisition outside the automated workflow.
print(
    (
        f'-> Authenticated or manual source IDs: '
        f'{fire_hazard_manual_acquisition_source_ids}'
    ),
)

# Report the request and retry settings that control automated source downloads.
print(f'-> Request timeout: {fire_hazard_request_timeout}')
print(f'-> Retry status codes: {fire_hazard_retry_status_codes}')

# Display the complete cache policy so configuration settings can be reviewed before acquisition.
print('\n--- FIRE HAZARD SNAPSHOT AND REFRESH POLICY ---')
display(fire_hazard_cache_policy_summary)

# Display source-level cache status to identify datasets that can be reused or must be acquired.
print('\n--- FIRE HAZARD SOURCE CACHE STATUS ---')
display(fire_hazard_source_cache_summary)

# Introduce the explanatory note that follows the cache-status summary.
print('\nNOTE:')

# Document snapshot-mode behavior so missing local files are surfaced instead of
# triggering unplanned downloads.
if fire_hazard_data_mode == 'snapshot':
    print(
        (
            'The acquisition workflow will reuse valid local source files and will '
            'not silently initiate live downloads for missing files.'
        ),
    )

    # Report required sources absent from the local cache so snapshot mode cannot be
    # mistaken for complete data availability.
    if missing_required_fire_hazard_cache_ids:
        print(
            (
                'One or more required files are not currently available in the local '
                'cache. They must be downloaded during the later acquisition steps '
                'before source-data QA can pass.'
            ),
        )

# Document refresh-mode behavior so new downloads replace cached files only after
# validation.
if fire_hazard_data_mode == 'refresh':
    print(
        (
            'The acquisition workflow will request fresh copies of downloadable '
            'source files. Existing cached files will be replaced only after the new '
            'temporary downloads pass validation.'
        ),
    )

print(
    (
        'Authenticated source credentials will not be stored inside the notebook. '
        'Manually acquired files must be placed at their configured local project '
        'paths.'
    ),
)

print('\n=== FIRE HAZARD SNAPSHOT AND REFRESH POLICY DEFINED ===')


=== DEFINING FIRE HAZARD SNAPSHOT AND REFRESH POLICY ===
-> Fire-hazard data mode: snapshot
-> Refresh source files: False
-> Use existing cache: True
-> Required source records available: 7 of 9
-> Missing required cached sources: ['USGS_3DEP_DEM', 'USGS_3DEP_DEM_ASPECT']
-> Missing optional cached sources: ['MTBS_FIRE_PERIMETERS']
-> Authenticated or manual source IDs: []
-> Request timeout: (15, 120)
-> Retry status codes: [429, 500, 502, 503, 504]

--- FIRE HAZARD SNAPSHOT AND REFRESH POLICY ---


,POLICY_SETTING,VALUE
0,Data Mode,snapshot
1,Use Existing Cache,True
2,Allow Source Overwrite,False
3,Automatic Refresh Fallback,False
4,Connection Timeout,15 seconds
5,Read Timeout,120 seconds
6,Total Retries,5
7,Retry Backoff Factor,1
8,Streaming Downloads,True
9,Download Chunk Size,1 MB



--- FIRE HAZARD SOURCE CACHE STATUS ---


,SOURCE_ID,COMPONENT_KEY,DATASET_NAME,REQUIRED_FOR_MODEL,CACHE_EXISTS,CACHE_SIZE_BYTES,ACQUISITION_ACTION
0,LANDFIRE_FBFM40,fuel_hazard,LF2025 Fire Behavior Fuel Model 40,True,True,41161421.0,Use cached source
1,LANDFIRE_CANOPY_COVER,fuel_hazard,LF2025 Forest Canopy Cover,True,True,18900707.0,Use cached source
2,LANDFIRE_CANOPY_BULK_DENSITY,fuel_hazard,LF2025 Forest Canopy Bulk Density,True,True,20415393.0,Use cached source
3,NASA_HLS_VI,fuel_dryness,HLS L30 and S30 Version 2.0,True,True,NaN,Use cached source
4,USGS_3DEP_DEM,slope_hazard,3DEP 1 Arc-Second Digital Elevation Model,True,False,NaN,Missing from cache
5,USGS_3DEP_DEM_ASPECT,aspect_hazard,3DEP 1 Arc-Second Digital Elevation Model,True,False,NaN,Missing from cache
6,INFORM_FODR_FIRE_OCCURRENCES,historical_fire_likelihood,InFORM Fire Occurrence Data Records (FODR),True,True,52218520.0,Use cached source
7,MTBS_FIRE_PERIMETERS,historical_fire_likelihood,MTBS National Burned Area Boundaries,False,False,NaN,Missing from cache
8,TIGER_ROADS_UTAH,human_ignition_potential,2025 Utah All Roads,True,True,15957754.0,Use cached source
9,ANNUAL_NLCD_2025_LAND_COVER,human_ignition_potential,Annual NLCD 2025 Land Cover,True,True,20224249.0,Use cached source



NOTE:
The acquisition workflow will reuse valid local source files and will not silently initiate live downloads for missing files.
One or more required files are not currently available in the local cache. They must be downloaded during the later acquisition steps before source-data QA can pass.
Authenticated source credentials will not be stored inside the notebook. Manually acquired files must be placed at their configured local project paths.

=== FIRE HAZARD SNAPSHOT AND REFRESH POLICY DEFINED ===


### Creating Retry-Enabled Fire Hazard Download Session


In [61]:
print('=== CREATING RETRY-ENABLED FIRE HAZARD DOWNLOAD SESSION ===')

# List network-policy objects required before constructing the reusable HTTP session.
required_fire_hazard_session_inputs = [
    'fire_hazard_retry_total',
    'fire_hazard_retry_connect',
    'fire_hazard_retry_read',
    'fire_hazard_retry_status',
    'fire_hazard_retry_backoff_factor',
    'fire_hazard_retry_status_codes',
    'fire_hazard_retry_allowed_methods',
    'fire_hazard_respect_retry_after_header',
    'fire_hazard_request_timeout',
]
missing_fire_hazard_session_inputs = [
    object_name
    for object_name in required_fire_hazard_session_inputs
    if object_name not in globals()
]

# Stop if network-policy settings are missing before constructing the reusable
# retry-enabled HTTP session.
if missing_fire_hazard_session_inputs:
    raise (
        NameError(
            (
                f'The following fire-hazard network configuration objects are missing:\n'
                f'{missing_fire_hazard_session_inputs}'
                f'\n\nRun step Define Snapshot and Refresh Policy before creating the '
                f'retry-enabled download session.'
            ),
        )
    )

# Import HTTP request and retry utilities used to build a reusable, fault-tolerant
# geospatial download session.
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Build the urllib3 retry strategy using the configured retry counts, status codes,
# methods, and backoff
# behavior.
fire_hazard_retry_strategy = (
    Retry(
        total=fire_hazard_retry_total,
        connect=fire_hazard_retry_connect,
        read=fire_hazard_retry_read,
        status=fire_hazard_retry_status,
        backoff_factor=fire_hazard_retry_backoff_factor,
        status_forcelist=fire_hazard_retry_status_codes,
        allowed_methods=frozenset(fire_hazard_retry_allowed_methods),
        respect_retry_after_header=fire_hazard_respect_retry_after_header,
        raise_on_status=True,
    )
)

# Create a reusable requests session and attach the retry-enabled adapter to both HTTPS
# and HTTP traffic.
fire_hazard_request_session = requests.Session()
fire_hazard_request_adapter = HTTPAdapter(max_retries=fire_hazard_retry_strategy)

# Mount the retry-enabled adapter so both HTTP and HTTPS source requests use the same
# recovery policy.
fire_hazard_request_session.mount('https://', fire_hazard_request_adapter)

# Mount the retry-enabled adapter so both HTTP and HTTPS source requests use the same
# recovery policy.
fire_hazard_request_session.mount('http://', fire_hazard_request_adapter)

# Apply a project-specific User-Agent and standard headers to every source request made
# through the session.
fire_hazard_request_headers = {
    'User-Agent': 'Utah-WUI-Fire-Hazard-Grid/1.0 (Geospatial research and portfolio project)',
    'Accept': '*/*',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
}

# Apply the project User-Agent to every request made through the reusable download
# session.
fire_hazard_request_session.headers.update(fire_hazard_request_headers)

# Disable persistent cookies and configure SSL/proxy behavior so the acquisition session
# remains predictable
# and stateless.
fire_hazard_session_use_persistent_cookies = False

# Clear session cookies so requests remain stateless and do not depend on prior
# authenticated browsing
# state.
if not fire_hazard_session_use_persistent_cookies:
    fire_hazard_request_session.cookies.clear()

# Set SSL verification and environment-proxy behavior explicitly so remote requests use
# predictable connection settings.
fire_hazard_verify_ssl_certificates = True
fire_hazard_use_environment_proxy_settings = True
fire_hazard_request_session.trust_env = fire_hazard_use_environment_proxy_settings

# Validate the session type, adapters, retry totals, and User-Agent before it is used
# for downloads.
fire_hazard_request_session_valid = (
    isinstance(
        fire_hazard_request_session,
        requests.Session,
    )
)
fire_hazard_https_adapter_valid = (
    'https://' in fire_hazard_request_session.adapters
    and isinstance(fire_hazard_request_session.adapters['https://'], HTTPAdapter)
)
fire_hazard_http_adapter_valid = (
    'http://' in fire_hazard_request_session.adapters
    and isinstance(fire_hazard_request_session.adapters['http://'], HTTPAdapter)
)
fire_hazard_https_retry_total = fire_hazard_request_session.adapters['https://'].max_retries.total
fire_hazard_http_retry_total = fire_hazard_request_session.adapters['http://'].max_retries.total
fire_hazard_adapter_retry_settings_valid = (
    fire_hazard_https_retry_total == fire_hazard_retry_total
    and fire_hazard_http_retry_total == fire_hazard_retry_total
)
fire_hazard_user_agent_valid = (
    fire_hazard_request_session.headers.get('User-Agent')
    == fire_hazard_request_headers['User-Agent']
)

# Stop if the reusable HTTP session was not created as a valid `requests.Session`
# object.
if not fire_hazard_request_session_valid:
    raise (
        TypeError(
            'fire_hazard_request_session is not a valid requests.Session object.',
        )
    )

# Stop if the retry-enabled adapter is missing from HTTPS requests, which would bypass
# the configured retry
# policy.
if not fire_hazard_https_adapter_valid:
    raise (
        ValueError(
            (
                'The retry-enabled HTTPS adapter was not attached to the fire-hazard '
                'request session.'
            ),
        )
    )

# Stop if the retry-enabled adapter is missing from HTTP requests, which would bypass
# the configured retry
# policy.
if not fire_hazard_http_adapter_valid:
    raise (
        ValueError(
            (
                'The retry-enabled HTTP adapter was not attached to the fire-hazard '
                'request session.'
            ),
        )
    )

# Stop if the mounted adapters do not carry the configured retry total.
if not fire_hazard_adapter_retry_settings_valid:
    raise (
        ValueError(
            'The request-session adapters do not contain the configured retry total.',
        )
    )

# Stop if the project User-Agent was not applied to the reusable request session.
if not fire_hazard_user_agent_valid:
    raise (
        ValueError(
            (
                'The project User-Agent header was not applied to the fire-hazard request '
                'session.'
            ),
        )
    )

# Build a QA table documenting the active retry-enabled download-session settings.
fire_hazard_request_session_summary = (
    pd.DataFrame(
        [
            {
                'SESSION_SETTING': 'Session Type',
                'VALUE': type(fire_hazard_request_session).__name__,
            },
            {
                'SESSION_SETTING': 'HTTPS Adapter Configured',
                'VALUE': fire_hazard_https_adapter_valid,
            },
            {
                'SESSION_SETTING': 'HTTP Adapter Configured',
                'VALUE': fire_hazard_http_adapter_valid,
            },
            {'SESSION_SETTING': 'Total Retries', 'VALUE': fire_hazard_retry_total},
            {'SESSION_SETTING': 'Connection Retries', 'VALUE': fire_hazard_retry_connect},
            {'SESSION_SETTING': 'Read Retries', 'VALUE': fire_hazard_retry_read},
            {'SESSION_SETTING': 'Status Retries', 'VALUE': fire_hazard_retry_status},
            {'SESSION_SETTING': 'Backoff Factor', 'VALUE': fire_hazard_retry_backoff_factor},
            {
                'SESSION_SETTING': 'Retry Status Codes',
                'VALUE': (
                    ', '.join(
                        (str(status_code) for status_code in fire_hazard_retry_status_codes),
                    )
                ),
            },
            {
                'SESSION_SETTING': 'Allowed Retry Methods',
                'VALUE': ', '.join(fire_hazard_retry_allowed_methods),
            },
            {
                'SESSION_SETTING': 'Request Timeout',
                'VALUE': str(fire_hazard_request_timeout),
            },
            {
                'SESSION_SETTING': 'SSL Verification',
                'VALUE': fire_hazard_verify_ssl_certificates,
            },
            {
                'SESSION_SETTING': 'Environment Proxy Settings',
                'VALUE': fire_hazard_use_environment_proxy_settings,
            },
            {
                'SESSION_SETTING': 'Persistent Cookies',
                'VALUE': fire_hazard_session_use_persistent_cookies,
            },
            {'SESSION_SETTING': 'User-Agent Applied', 'VALUE': fire_hazard_user_agent_valid},
        ],
    )
)
print(f'-> Requests session created: {fire_hazard_request_session_valid}')
print(f'-> HTTPS retry adapter configured: {fire_hazard_https_adapter_valid}')
print(f'-> HTTP retry adapter configured: {fire_hazard_http_adapter_valid}')
print(f'-> Retry attempts configured: {fire_hazard_retry_total}')
print(f'-> Retry status codes: {fire_hazard_retry_status_codes}')
print(f'-> Request timeout: {fire_hazard_request_timeout}')
print(f'-> SSL certificate verification: {fire_hazard_verify_ssl_certificates}')
print(f'-> User-Agent applied: {fire_hazard_user_agent_valid}')
print('\n--- RETRY-ENABLED DOWNLOAD SESSION ---')
display(fire_hazard_request_session_summary)
print('\nNOTE:')
print(
    (
        'The retry-enabled session is now available as fire_hazard_request_session '
        'and will be reused by later source-download functions.'
    ),
)
print(
    (
        'The session automatically retries HTTP 429, 500, 502, 503, and 504 responses '
        'using progressive backoff delays.'
    ),
)
print(
    (
        'No remote source request was submitted during this step. The next step will '
        'define the reusable file-download function.'
    ),
)

print('\n=== RETRY-ENABLED FIRE HAZARD DOWNLOAD SESSION READY ===')

print('=== CREATING FIRE HAZARD SOURCE FILE DOWNLOAD FUNCTION ===')

# List cache, network, validation, and file-replacement settings required by the
# reusable download function.
required_fire_hazard_download_inputs = [
    'fire_hazard_request_session',
    'fire_hazard_request_timeout',
    'fire_hazard_verify_ssl_certificates',
    'fire_hazard_data_mode',
    'fire_hazard_use_existing_cache',
    'fire_hazard_allow_source_overwrite',
    'fire_hazard_temporary_download_suffix',
    'fire_hazard_minimum_download_size_bytes',
    'fire_hazard_validate_before_replace',
    'fire_hazard_use_atomic_file_replacement',
    'fire_hazard_remove_failed_temporary_files',
    'fire_hazard_preserve_cache_on_refresh_failure',
    'fire_hazard_stream_downloads',
    'fire_hazard_download_chunk_size_bytes',
    'fire_hazard_require_downloaded_file_exists',
    'fire_hazard_require_nonempty_downloads',
    'fire_hazard_record_content_type',
    'fire_hazard_fail_on_unexpected_content_type',
]

missing_fire_hazard_download_inputs = [
    object_name
    for object_name in required_fire_hazard_download_inputs
    if object_name not in globals()
]

# Stop if cache, network, or validation settings are missing before the reusable
# download helper is defined.
if missing_fire_hazard_download_inputs:
    raise NameError(
        f'The following fire-hazard download configuration objects are missing:\n'
        f'{missing_fire_hazard_download_inputs}'
        f'\n\nRun Steps 16.2D and 16.2E before creating the source file download '
        f'function.'
    )

# Import filesystem, timestamp, MIME-detection, timing, and request-exception utilities
# used to validate, retry, and stage downloaded source files.
from pathlib import Path
from datetime import datetime, timezone
import mimetypes
import os
import time
from requests import exceptions as requests_exceptions

# Configure whole-file transfer retries. These retries operate above the urllib3/requests
# adapter retries and specifically protect long streamed downloads that fail after an HTTP
# connection has already been established.
fire_hazard_file_transfer_retry_attempts = 4
fire_hazard_file_transfer_retry_backoff_seconds = 5
fire_hazard_file_transfer_retry_backoff_max_seconds = 60

# Validate whole-file retry settings before they are used.
if (
    not isinstance(fire_hazard_file_transfer_retry_attempts, int)
    or fire_hazard_file_transfer_retry_attempts < 1
):
    raise ValueError('fire_hazard_file_transfer_retry_attempts must be a positive integer.')

if (
    not isinstance(fire_hazard_file_transfer_retry_backoff_seconds, (int, float))
    or fire_hazard_file_transfer_retry_backoff_seconds < 0
):
    raise ValueError(
        'fire_hazard_file_transfer_retry_backoff_seconds must be zero or greater.'
    )

if (
    not isinstance(fire_hazard_file_transfer_retry_backoff_max_seconds, (int, float))
    or fire_hazard_file_transfer_retry_backoff_max_seconds < 0
):
    raise ValueError(
        'fire_hazard_file_transfer_retry_backoff_max_seconds must be zero or greater.'
    )

# Define transient failures that justify restarting the entire streamed file transfer.
# Validation failures, malformed URLs, unexpected content types, and other deterministic
# errors are intentionally excluded so they fail immediately rather than being retried.
fire_hazard_transient_transfer_exceptions = (
    requests_exceptions.ChunkedEncodingError,
    requests_exceptions.ConnectionError,
    requests_exceptions.ReadTimeout,
    requests_exceptions.Timeout,
)

# Define common binary and geospatial response types used to detect accidental
# HTML/error responses during downloads.
fire_hazard_accepted_binary_content_types = {
    'application/octet-stream',
    'application/zip',
    'application/x-zip-compressed',
    'application/x-tar',
    'application/gzip',
    'application/x-gzip',
    'image/tiff',
    'image/geotiff',
    'application/geotiff',
    'application/vnd.geo+json',
    'application/geo+json',
    'application/json',
    'text/csv',
    'text/plain',
}


# Close an HTTP response without allowing cleanup failures to mask the original
# transfer exception.
def close_fire_hazard_response_safely(response):
    """
    Closes a requests.Response object without allowing a cleanup-time
    network exception to replace the original download error.
    """
    if response is None:
        return

    try:
        response.close()
    except Exception as close_error:
        print(f'-> WARNING: Response cleanup reported: {close_error}')


# Remove an incomplete temporary download when project policy allows it.
def remove_fire_hazard_temporary_download(temporary_download_path):
    """
    Removes an incomplete .part file so a failed transfer cannot be
    mistaken for a validated source file.
    """
    if (
        fire_hazard_remove_failed_temporary_files
        and temporary_download_path.exists()
    ):
        temporary_download_path.unlink()


# Download one hazard source through the retry-enabled session, validate it, and replace
# cached data atomically only after success.
def download_fire_hazard_source_file(
    source_url,
    destination_path,
    source_name,
    expected_content_types=None,
    request_params=None,
    request_headers=None,
    force_refresh=None,
    minimum_size_bytes=None,
):
    """
    Downloads one fire-hazard source file using the retry-enabled project
    HTTP session plus whole-file streamed-transfer retries.

    The function:

      1. validates the source URL and destination path;
      2. reuses an existing cached file when permitted;
      3. downloads to a temporary file;
      4. streams large files in memory-efficient chunks;
      5. restarts the whole transfer after transient stream failures;
      6. applies exponential retry backoff;
      7. validates file size and response metadata;
      8. safely replaces the final cached file;
      9. preserves an existing cache if refresh fails;
     10. returns a structured acquisition-status record.
    """

    # Require a nonempty source URL before attempting a remote request.
    if not isinstance(source_url, str) or not source_url.strip():
        raise ValueError('source_url must contain a non-empty download URL.')

    # Restrict automated acquisition to HTTP/HTTPS URLs supported by the requests session.
    if not source_url.lower().startswith(('http://', 'https://')):
        raise ValueError("source_url must begin with 'http://' or 'https://'.")

    # Require a readable source name so errors and acquisition records identify the
    # dataset clearly.
    if not isinstance(source_name, str) or not source_name.strip():
        raise ValueError('source_name must contain a non-empty text value.')

    # Normalize the configured destination to a Path object before cache checks and file
    # replacement.
    destination_path = Path(destination_path)

    # Require a filename in the destination path so the download cannot target a
    # directory accidentally.
    if not destination_path.name:
        raise ValueError('destination_path must include a final filename.')

    # Use the project minimum-size rule when the caller does not provide a
    # source-specific threshold.
    if minimum_size_bytes is None:
        minimum_size_bytes = fire_hazard_minimum_download_size_bytes

    # Reject invalid minimum-size thresholds before they are used to validate cached or
    # downloaded files.
    if not isinstance(minimum_size_bytes, int) or minimum_size_bytes < 1:
        raise ValueError('minimum_size_bytes must be a positive integer.')

    # Normalize optional query parameters and request headers.
    if request_params is None:
        request_params = {}

    if request_headers is None:
        request_headers = {}

    if not isinstance(request_params, dict):
        raise TypeError('request_params must be a dictionary or None.')

    if not isinstance(request_headers, dict):
        raise TypeError('request_headers must be a dictionary or None.')

    # Use the notebook snapshot mode when no per-source refresh override is provided.
    if force_refresh is None:
        source_refresh_requested = fire_hazard_data_mode == 'refresh'
    else:
        if not isinstance(force_refresh, bool):
            raise TypeError('force_refresh must be True, False, or None.')
        source_refresh_requested = force_refresh

    # Check whether a local source file already exists and can potentially satisfy the
    # request without downloading.
    existing_cache_available = (
        destination_path.exists()
        and destination_path.is_file()
    )

    existing_cache_size_bytes = (
        destination_path.stat().st_size
        if existing_cache_available
        else None
    )

    existing_cache_valid = (
        existing_cache_available
        and existing_cache_size_bytes >= minimum_size_bytes
    )

    # Return the validated local cache immediately when refresh is not requested.
    if not source_refresh_requested and existing_cache_valid:
        print(f'-> Using cached source: {source_name}')
        print(f'   Path: {destination_path}')
        print(f'   Size: {existing_cache_size_bytes / 1024 ** 2:,.2f} MB')

        return {
            'SOURCE_NAME': source_name,
            'SOURCE_URL': source_url,
            'DESTINATION_PATH': destination_path,
            'ACQUISITION_ACTION': 'Used cached source',
            'CACHE_USED': True,
            'REFRESH_REQUESTED': False,
            'HTTP_STATUS': None,
            'CONTENT_TYPE': None,
            'CONTENT_LENGTH_HEADER': None,
            'CONTENT_ENCODING': None,
            'TRANSFER_ENCODING': None,
            'CONTENT_LENGTH_COMPARISON_REQUIRED': False,
            'CONTENT_LENGTH_MATCHES': None,
            'FILE_SIZE_BYTES': existing_cache_size_bytes,
            'FILE_SIZE_MB': existing_cache_size_bytes / 1024 ** 2,
            'DOWNLOAD_ATTEMPTS_USED': 0,
            'DOWNLOADED_AT_UTC': None,
            'VALID': True,
        }

    # Fail clearly in snapshot mode when no valid local source is available.
    if not source_refresh_requested and not existing_cache_valid:
        cache_problem = (
            'does not exist'
            if not existing_cache_available
            else 'is smaller than the configured minimum file size'
        )

        raise FileNotFoundError(
            f'The cached source for {source_name} {cache_problem}:\n'
            f'{destination_path}'
            f"\n\nSet fire_hazard_data_mode = 'refresh' or call the function with "
            f'force_refresh=True to acquire the file.'
        )

    # Create the required destination directory before files are downloaded.
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    # Create a separate temporary path so incomplete downloads never overwrite the final
    # cached source.
    temporary_download_path = (
        destination_path.parent
        / (destination_path.name + fire_hazard_temporary_download_suffix)
    )

    # Remove a stale temporary download before starting a new transfer.
    if temporary_download_path.exists():
        temporary_download_path.unlink()
        print(f'-> Removed stale temporary file for {source_name}.')

    # Normalize accepted response content types.
    if expected_content_types is None:
        accepted_content_types = set(fire_hazard_accepted_binary_content_types)
    else:
        accepted_content_types = set(expected_content_types)

        if not all(
            isinstance(content_type, str) and bool(content_type.strip())
            for content_type in accepted_content_types
        ):
            raise ValueError(
                'expected_content_types must contain only non-empty strings.'
            )

    inferred_destination_content_type = mimetypes.guess_type(destination_path.name)[0]

    print(f'\nDownloading {source_name}...')
    print(f'-> Source URL: {source_url}')
    print(f'-> Temporary path: {temporary_download_path}')
    print(f'-> Final path: {destination_path}')
    print(
        f'-> Whole-file retry attempts available: '
        f'{fire_hazard_file_transfer_retry_attempts}'
    )

    # Track the last transient error so the final failure can preserve useful context.
    last_transfer_error = None

    # Restart the complete HTTP stream after transient mid-transfer failures.
    for transfer_attempt in range(1, fire_hazard_file_transfer_retry_attempts + 1):

        response = None

        # Ensure each attempt begins from a clean .part file.
        remove_fire_hazard_temporary_download(temporary_download_path)

        if transfer_attempt > 1:
            print(
                f'-> Starting whole-file retry attempt '
                f'{transfer_attempt} of {fire_hazard_file_transfer_retry_attempts}...'
            )

        try:
            # Submit a fresh HTTP request for this transfer attempt.
            response = fire_hazard_request_session.get(
                source_url,
                params=request_params,
                headers=request_headers,
                stream=fire_hazard_stream_downloads,
                timeout=fire_hazard_request_timeout,
                verify=fire_hazard_verify_ssl_certificates,
                allow_redirects=True,
            )

            response.raise_for_status()

            # Read and normalize response metadata.
            response_content_type = (
                response.headers.get('Content-Type', '')
                .split(';', 1)[0]
                .strip()
                .lower()
            )

            response_content_length = response.headers.get('Content-Length')

            response_content_encoding = (
                response.headers.get('Content-Encoding', '')
                .strip()
                .lower()
            )

            response_transfer_encoding = (
                response.headers.get('Transfer-Encoding', '')
                .strip()
                .lower()
            )

            try:
                response_content_length_bytes = (
                    int(response_content_length)
                    if response_content_length is not None
                    else None
                )
            except (TypeError, ValueError):
                response_content_length_bytes = None

            # Evaluate returned content type before accepting the payload.
            response_content_type_valid = (
                not response_content_type
                or response_content_type
                in {
                    content_type.lower()
                    for content_type in accepted_content_types
                }
            )

            if (
                fire_hazard_fail_on_unexpected_content_type
                and not response_content_type_valid
            ):
                raise ValueError(
                    f'{source_name} returned an unexpected Content-Type:\n'
                    f"{response_content_type or 'Not provided'}\n"
                    f'Expected one of: {sorted(accepted_content_types)}'
                )

            if response_content_type and not response_content_type_valid:
                print('-> WARNING: The server returned an unexpected Content-Type.')
                print(f'   Returned: {response_content_type}')
                print(f'   Inferred from filename: {inferred_destination_content_type}')

            # Track bytes streamed during this transfer attempt.
            bytes_written = 0

            # Stream the response in chunks so large geospatial files are not loaded
            # entirely into memory.
            with temporary_download_path.open('wb') as temporary_file:
                for response_chunk in response.iter_content(
                    chunk_size=fire_hazard_download_chunk_size_bytes
                ):
                    if not response_chunk:
                        continue

                    temporary_file.write(response_chunk)
                    bytes_written += len(response_chunk)

                temporary_file.flush()
                os.fsync(temporary_file.fileno())

            # Verify that streaming created the temporary file.
            temporary_file_exists = (
                temporary_download_path.exists()
                and temporary_download_path.is_file()
            )

            temporary_file_size_bytes = (
                temporary_download_path.stat().st_size
                if temporary_file_exists
                else 0
            )

            streamed_size_matches_file = (
                temporary_file_size_bytes == bytes_written
            )

            temporary_file_size_valid = (
                temporary_file_size_bytes >= minimum_size_bytes
            )

            content_length_comparison_required = (
                response_content_length_bytes is not None
                and response_content_encoding in {'', 'identity'}
                and 'chunked' not in response_transfer_encoding
            )

            content_length_matches = (
                not content_length_comparison_required
                or temporary_file_size_bytes == response_content_length_bytes
            )

            if not temporary_file_exists:
                raise IOError(
                    f'The temporary download for {source_name} was not created.'
                )

            if (
                fire_hazard_require_nonempty_downloads
                and not temporary_file_size_valid
            ):
                raise IOError(
                    f'The downloaded {source_name} file is smaller than the '
                    f'configured minimum size of {minimum_size_bytes:,} bytes.'
                )

            if not streamed_size_matches_file:
                raise IOError(
                    f'The streamed byte total for {source_name} does not match '
                    f'the temporary file size.'
                )

            if not content_length_matches:
                raise IOError(
                    f'The downloaded size for {source_name} does not match the '
                    f'applicable HTTP Content-Length header.'
                )

            if (
                response_content_length_bytes is not None
                and not content_length_comparison_required
            ):
                print(
                    '-> Content-Length strict comparison skipped for an encoded '
                    'or chunked transfer.'
                )

            # Promote the validated temporary file only after all checks pass.
            if fire_hazard_validate_before_replace:
                if fire_hazard_use_atomic_file_replacement:
                    temporary_download_path.replace(destination_path)
                else:
                    temporary_download_path.rename(destination_path)
            else:
                temporary_download_path.replace(destination_path)

            # Verify final cached source integrity.
            final_file_exists = (
                destination_path.exists()
                and destination_path.is_file()
            )

            final_file_size_bytes = (
                destination_path.stat().st_size
                if final_file_exists
                else 0
            )

            final_file_size_valid = (
                final_file_size_bytes >= minimum_size_bytes
            )

            if (
                fire_hazard_require_downloaded_file_exists
                and not final_file_exists
            ):
                raise IOError(
                    f'The completed {source_name} source file does not exist '
                    f'after download.'
                )

            if (
                fire_hazard_require_nonempty_downloads
                and not final_file_size_valid
            ):
                raise IOError(
                    f'The completed {source_name} source file does not pass '
                    f'minimum-size validation.'
                )

            downloaded_at_utc = datetime.now(timezone.utc).isoformat()

            print(f'-> Download complete: {source_name}')
            print(f'-> HTTP status: {response.status_code}')
            print(f'-> Transfer attempt used: {transfer_attempt}')

            if fire_hazard_record_content_type:
                print(
                    f"-> Content type: "
                    f"{response_content_type or 'Not provided'}"
                )

            print(
                f'-> File size: '
                f'{final_file_size_bytes / 1024 ** 2:,.2f} MB'
            )
            print(f'-> Cached path: {destination_path}')

            return {
                'SOURCE_NAME': source_name,
                'SOURCE_URL': source_url,
                'DESTINATION_PATH': destination_path,
                'ACQUISITION_ACTION': (
                    'Refreshed cached source'
                    if existing_cache_available
                    else 'Downloaded new source'
                ),
                'CACHE_USED': False,
                'REFRESH_REQUESTED': source_refresh_requested,
                'HTTP_STATUS': response.status_code,
                'CONTENT_TYPE': (
                    response_content_type
                    if fire_hazard_record_content_type
                    else None
                ),
                'CONTENT_LENGTH_HEADER': response_content_length_bytes,
                'CONTENT_ENCODING': response_content_encoding,
                'TRANSFER_ENCODING': response_transfer_encoding,
                'CONTENT_LENGTH_COMPARISON_REQUIRED':
                    content_length_comparison_required,
                'CONTENT_LENGTH_MATCHES': content_length_matches,
                'FILE_SIZE_BYTES': final_file_size_bytes,
                'FILE_SIZE_MB': final_file_size_bytes / 1024 ** 2,
                'DOWNLOAD_ATTEMPTS_USED': transfer_attempt,
                'DOWNLOADED_AT_UTC': downloaded_at_utc,
                'VALID': (
                    final_file_exists
                    and final_file_size_valid
                    and streamed_size_matches_file
                    and content_length_matches
                ),
            }

        # Retry only transient network/stream failures.
        except fire_hazard_transient_transfer_exceptions as transfer_error:
            last_transfer_error = transfer_error

            # Close the response without allowing cleanup failures to mask this error.
            close_fire_hazard_response_safely(response)
            response = None

            # Remove the incomplete .part file before retrying.
            remove_fire_hazard_temporary_download(temporary_download_path)

            print(
                f'-> Transient transfer failure on attempt '
                f'{transfer_attempt} of '
                f'{fire_hazard_file_transfer_retry_attempts}: '
                f'{transfer_error}'
            )

            # Stop after the configured final attempt.
            if transfer_attempt >= fire_hazard_file_transfer_retry_attempts:
                break

            # Apply bounded exponential backoff between whole-file retries.
            retry_delay_seconds = min(
                fire_hazard_file_transfer_retry_backoff_seconds
                * (2 ** (transfer_attempt - 1)),
                fire_hazard_file_transfer_retry_backoff_max_seconds,
            )

            print(
                f'-> Retrying complete file transfer after '
                f'{retry_delay_seconds:g} seconds...'
            )

            time.sleep(retry_delay_seconds)
            continue

        # Deterministic validation or HTTP errors should fail immediately rather than
        # repeatedly downloading the same invalid content.
        except (
            requests_exceptions.HTTPError,
            requests_exceptions.RequestException,
            OSError,
            ValueError,
        ) as download_error:
            close_fire_hazard_response_safely(response)
            response = None

            remove_fire_hazard_temporary_download(temporary_download_path)

            print(f'-> Download failed: {source_name}')
            print(f'-> Failure message: {download_error}')

            if (
                existing_cache_valid
                and fire_hazard_preserve_cache_on_refresh_failure
            ):
                print('-> The previously validated cached source was preserved.')

            raise

        finally:
            # Release the response after each attempt. The helper prevents cleanup errors
            # from masking the actual transfer outcome.
            close_fire_hazard_response_safely(response)

    # The loop reaches this point only after all whole-file transient retries fail.
    remove_fire_hazard_temporary_download(temporary_download_path)

    print(f'-> Download failed after all transfer attempts: {source_name}')
    print(f'-> Final failure message: {last_transfer_error}')

    if existing_cache_valid and fire_hazard_preserve_cache_on_refresh_failure:
        print('-> The previously validated cached source was preserved.')

    if last_transfer_error is not None:
        raise last_transfer_error

    raise RuntimeError(
        f'{source_name} failed without a captured transfer exception.'
    )


# Confirm that the reusable download function exists before later acquisition steps call it.
fire_hazard_download_function_created = callable(
    download_fire_hazard_source_file
)

# Stop if the reusable download helper was not created successfully.
if not fire_hazard_download_function_created:
    raise TypeError(
        'download_fire_hazard_source_file was not created successfully.'
    )

# Summarize the safety, caching, validation, and retry capabilities implemented by
# the download helper.
fire_hazard_download_function_summary = pd.DataFrame(
    [
        {
            'FUNCTION_CAPABILITY': 'Uses Retry-Enabled Session',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Supports Cached Files',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Supports Explicit Refresh',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Streams Large Downloads',
            'CONFIGURED': fire_hazard_stream_downloads,
        },
        {
            'FUNCTION_CAPABILITY': 'Whole-File Transfer Retries',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Whole-File Retry Attempts',
            'CONFIGURED': fire_hazard_file_transfer_retry_attempts,
        },
        {
            'FUNCTION_CAPABILITY': 'Exponential Transfer Backoff',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Uses Temporary Download File',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Validates Minimum File Size',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Validates Applicable Content-Length',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Handles Encoded HTTP Transfers',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Handles Chunked HTTP Transfers',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Safe Response Cleanup',
            'CONFIGURED': True,
        },
        {
            'FUNCTION_CAPABILITY': 'Records Content Type',
            'CONFIGURED': fire_hazard_record_content_type,
        },
        {
            'FUNCTION_CAPABILITY': 'Validates Before Replacement',
            'CONFIGURED': fire_hazard_validate_before_replace,
        },
        {
            'FUNCTION_CAPABILITY': 'Preserves Valid Cache on Failure',
            'CONFIGURED': fire_hazard_preserve_cache_on_refresh_failure,
        },
        {
            'FUNCTION_CAPABILITY': 'Removes Failed Temporary Files',
            'CONFIGURED': fire_hazard_remove_failed_temporary_files,
        },
    ]
)

print(f'-> Download function created: {fire_hazard_download_function_created}')
print('-> Function name: download_fire_hazard_source_file')
print(f'-> Current acquisition mode: {fire_hazard_data_mode}')
print(f'-> Temporary file suffix: {fire_hazard_temporary_download_suffix}')
print(
    f'-> Download chunk size: '
    f'{fire_hazard_download_chunk_size_bytes / 1024 ** 2:.0f} MB'
)
print(
    f'-> Whole-file transfer attempts: '
    f'{fire_hazard_file_transfer_retry_attempts}'
)
print(
    f'-> Whole-file initial backoff: '
    f'{fire_hazard_file_transfer_retry_backoff_seconds:g} seconds'
)
print(
    f'-> Whole-file maximum backoff: '
    f'{fire_hazard_file_transfer_retry_backoff_max_seconds:g} seconds'
)
print(
    f'-> Minimum file size: '
    f'{fire_hazard_minimum_download_size_bytes:,} bytes'
)

print('\n--- FIRE HAZARD DOWNLOAD FUNCTION ---')
display(fire_hazard_download_function_summary)

print('\nNOTE:')
print(
    'The reusable download function now combines the existing HTTP adapter retries '
    'with whole-file transfer retries for interrupted streamed downloads.'
)
print(
    'Transient ChunkedEncodingError, connection, read-timeout, and timeout failures '
    'restart the complete file transfer using bounded exponential backoff.'
)
print(
    'Each failed attempt removes its incomplete .part file. A completed source is '
    'promoted to the final cache path only after all configured validation checks pass.'
)
print(
    'Deterministic validation failures and exhausted HTTP errors are not blindly '
    'retried at the file-transfer layer.'
)
print(
    'An existing validated cached source remains protected when a refresh fails and '
    'the preserve-cache policy is enabled.'
)
print('No source file was downloaded during this step.')

print('\n=== FIRE HAZARD SOURCE FILE DOWNLOAD FUNCTION READY ===')

print('=== CREATING FIRE HAZARD ARCHIVE EXTRACTION FUNCTION ===')

# List archive-policy settings required before the reusable extraction helpers are
# created.
required_fire_hazard_extraction_inputs = [
    'fire_hazard_extract_archives',
    'fire_hazard_supported_archive_extensions',
    'fire_hazard_reuse_extracted_cache',
    'fire_hazard_overwrite_extracted_files',
    'fire_hazard_require_safe_archive_extraction',
]
missing_fire_hazard_extraction_inputs = [
    object_name
    for object_name in required_fire_hazard_extraction_inputs
    if object_name not in globals()
]

# Stop if archive-policy settings are missing before safe extraction helpers are
# created.
if missing_fire_hazard_extraction_inputs:
    raise (
        NameError(
            (
                f'The following fire-hazard archive configuration objects are missing:\n'
                f'{missing_fire_hazard_extraction_inputs}'
                f'\n\nRun step Define Snapshot and Refresh Policy before creating the archive '
                f'extraction function.'
            ),
        )
    )

# Import filesystem and archive utilities used to identify, validate, and safely extract
# compressed geospatial sources.
from pathlib import Path
import zipfile
import tarfile
import shutil
from datetime import datetime, timezone

# Identify the supported archive format before selecting the extraction method.
def identify_fire_hazard_archive_type(archive_path):

    """
    Identifies the supported archive type associated

    with a fire-hazard source file.

    Parameters
    ----------
    archive_path : str or pathlib.Path
        Path to the source archive.

    Returns
    -------
    str
        One of the following archive types:

        "zip"
        "tar"
        "tar.gz"
        "tgz"

    Raises
    ------
    ValueError
        Raised when the archive extension is not
        supported by the project configuration.
    """
    # Normalize the archive location to a Path object before filesystem and format
    # validation.
    archive_path = Path(archive_path)
    # Normalize the archive filename for case-insensitive format detection.
    archive_filename_lower = archive_path.name.lower()

    # Detect `.tar.gz` before the shorter `.gz`/`.tar` patterns so compressed TAR
    # archives are classified
    # correctly.
    if archive_filename_lower.endswith('.tar.gz'):
        return 'tar.gz'

    # Recognize the `.tgz` shorthand as a gzip-compressed TAR archive.
    if archive_filename_lower.endswith('.tgz'):
        return 'tgz'

    # Identify ZIP packages so they use ZIP-specific integrity checks and member
    # handling.
    if archive_filename_lower.endswith('.zip'):
        return 'zip'

    # Identify uncompressed TAR packages so they use the appropriate TAR reader mode.
    if archive_filename_lower.endswith('.tar'):
        return 'tar'
    raise (
        ValueError(
            (
                f'The archive format is not supported for extraction: {archive_path.name}'
                f'\nSupported formats: {fire_hazard_supported_archive_extensions}'
            ),
        )
    )

# Resolve and validate an archive member destination so path traversal cannot escape the
# approved extraction
# directory.
def build_safe_fire_hazard_extraction_path(extraction_directory, archive_member_name):

    """
    Builds and validates the destination path for one
    archive member.

    The function prevents archive members from writing
    outside the intended extraction directory.

    Parameters
    ----------
    extraction_directory : str or pathlib.Path
        Root directory assigned to the extracted
        archive contents.

    archive_member_name : str
        Relative path stored inside the archive.

    Returns
    -------
    pathlib.Path
        Validated extraction destination.

    Raises
    ------
    ValueError
        Raised when an archive member attempts path
        traversal or uses an absolute path.
    """
    # Resolve the approved extraction root before validating where archive members are
    # allowed to write.
    extraction_directory = Path(extraction_directory).resolve()

    # Require a nonempty relative archive-member path before any extraction destination
    # is constructed.
    if not isinstance(archive_member_name, str) or not archive_member_name.strip():
        raise ValueError('Archive members must contain a non-empty relative path.')
    # Normalize archive path separators so safety checks behave consistently across
    # operating systems.
    normalized_member_name = archive_member_name.replace('\\', '/')
    # Represent the archive member as a relative path before checking absolute paths and
    # parent traversal.
    member_relative_path = Path(normalized_member_name)

    # Reject absolute member paths because they could write outside the approved
    # extraction directory.
    if member_relative_path.is_absolute():
        raise (
            ValueError(
                f'Unsafe absolute archive-member path detected: {archive_member_name}',
            )
        )

    # Reject parent-directory traversal before resolving the archive member destination.
    if '..' in member_relative_path.parts:
        raise (
            ValueError(
                (
                    f'Unsafe parent-directory traversal was detected in archive member:\n'
                    f'{archive_member_name}'
                ),
            )
        )
    # Resolve the proposed member destination so it can be proven to remain inside the
    # extraction root.
    member_destination = (extraction_directory / member_relative_path).resolve()

    # Confirm the resolved archive-member destination remains inside the approved
    # extraction root.
    try:
        member_destination.relative_to(extraction_directory)
    # Reject archive members whose resolved destinations escape the approved extraction root.
    except ValueError as error:
        raise (
            ValueError(
                (
                    f'Archive member would be extracted outside the intended destination:\n'
                    f'{archive_member_name}'
                ),
            )
        ) from error
    return member_destination

# Extract a supported hazard archive with cache reuse, path-safety checks, and
# post-extraction validation.
def extract_fire_hazard_source_archive(
    archive_path,
    extraction_directory,
    source_name,
    expected_extensions=None,
    force_extract=None,
):
    """
    Safely extracts one fire-hazard source archive.

    The function:

      1. validates the source archive and destination;
      2. detects ZIP, TAR, TAR.GZ, or TGZ format;
      3. inventories archive members before extraction;
      4. prevents absolute paths and path traversal;
      5. rejects TAR symbolic and hard links;
      6. optionally reuses valid extracted contents;
      7. creates directories and writes files safely;
      8. validates the completed extraction;
      9. returns a structured extraction-status record.

    Parameters
    ----------
    archive_path : str or pathlib.Path
        Local path to the downloaded source archive.

    extraction_directory : str or pathlib.Path
        Directory where archive contents will be
        extracted.

    source_name : str
        Readable source name used in progress messages
        and the returned extraction record.

    expected_extensions : collection of str, optional
        Expected file extensions among extracted files.

        Examples:

        [".tif"]
        [".shp", ".dbf", ".shx", ".prj"]

        When omitted, extraction succeeds without a
        source-specific extension requirement.

    force_extract : bool, optional
        Overrides the project extraction-cache policy.

        True:
            Extract the archive and replace conflicting
            files when overwrite is allowed.

        False:
            Reuse existing extracted contents when
            available.

        None:
            Follow the project-level extraction policy.

    Returns
    -------
    dict
        Structured record describing the archive,
        extraction destination, member counts, file
        counts, extracted size, cache action, timestamp,
        and validation status.
    """
    # Normalize the archive location to a Path object before filesystem and format
    # validation.
    archive_path = Path(archive_path)
    # Resolve the approved extraction root before validating where archive members are
    # allowed to write.
    extraction_directory = Path(extraction_directory)

    # Require a readable source name so errors and acquisition records identify the
    # dataset clearly.
    if not isinstance(source_name, str) or not source_name.strip():
        raise ValueError('source_name must contain a non-empty text value.')

    # Honor the project archive policy by stopping extraction when compressed-source
    # handling is disabled.
    if not fire_hazard_extract_archives:
        raise (
            RuntimeError(
                'Archive extraction is disabled by the current fire-hazard project policy.',
            )
        )

    # Require the source archive to exist locally before format checks or extraction
    # begin.
    if not archive_path.exists():
        raise FileNotFoundError(f'The source archive does not exist:\n{archive_path}')

    # Reject directories or other nonfile paths before archive validation.
    if not archive_path.is_file():
        raise ValueError(f'archive_path must reference a file:\n{archive_path}')
    # Measure the source archive so empty packages are rejected before extraction
    # begins.
    archive_size_bytes = archive_path.stat().st_size

    # Reject empty archives so incomplete source packages do not enter the processing
    # workflow.
    if archive_size_bytes <= 0:
        raise ValueError(f'The source archive is empty:\n{archive_path}')
    # Detect the supported archive format so ZIP and TAR packages follow the correct
    # extraction path.
    archive_type = identify_fire_hazard_archive_type(archive_path)

    # Require a Boolean extraction override so cache/refresh behavior remains explicit.
    if force_extract is not None and (not isinstance(force_extract, bool)):
        raise TypeError('force_extract must be True, False, or None.')

    # Use the project overwrite policy when the caller does not provide an
    # extraction-specific refresh
    # override.
    if force_extract is None:
        # Resolve the per-call extraction override against the project-level overwrite
        # policy.
        extraction_refresh_requested = fire_hazard_overwrite_extracted_files
    else:
        # Resolve the per-call extraction override against the project-level overwrite
        # policy.
        extraction_refresh_requested = force_extract

    # Allow sources without extension requirements while keeping post-extraction
    # validation available when
    # needed.
    if expected_extensions is None:
        # Normalize required file extensions so post-extraction completeness checks are
        # consistent.
        normalized_expected_extensions = []
    else:

        # Require extension expectations to be a collection so each required GIS file
        # type can be validated
        # consistently.
        if not isinstance(expected_extensions, (list, tuple, set)):
            raise TypeError('expected_extensions must be a list, tuple, set, or None.')
        # Normalize required file extensions so post-extraction completeness checks are
        # consistent.
        normalized_expected_extensions = []

        # Normalize each expected file extension before using it to validate extracted
        # source contents.
        for expected_extension in expected_extensions:

            # Reject blank or nontext extension rules before normalizing the expected
            # source file types.
            if not isinstance(expected_extension, str) or not expected_extension.strip():
                raise ValueError('Every expected extension must be a non-empty string.')
            # Normalize each required extension to lowercase dot-prefixed form before
            # completeness checks.
            expected_extension = expected_extension.strip().lower()

            # Add a leading period so extension comparisons match pathlib suffix values.
            if not expected_extension.startswith('.'):
                # Normalize each required extension to lowercase dot-prefixed form
                # before completeness
                # checks.
                expected_extension = '.' + expected_extension
            normalized_expected_extensions.append(expected_extension)
    # Inventory previously extracted files so a complete cached extraction can be reused
    # when refresh is not
    # requested.
    existing_extracted_files = (
        [path for path in extraction_directory.rglob('*') if path.is_file()]
        if extraction_directory.exists()
        else []
    )
    # Record whether the extraction cache currently contains any files.
    existing_extraction_available = len(existing_extracted_files) > 0
    # Collect cached file extensions for source-specific completeness checks.
    existing_extensions = {
        file_path.suffix.lower()
        for file_path in existing_extracted_files
    }
    # Verify that cached extraction contents include every required source extension.
    existing_expected_extensions_valid = (
        not normalized_expected_extensions
        or all(
            (
                extension in existing_extensions
                for extension in normalized_expected_extensions
            ),
        )
    )
    # Combine cache availability and extension checks to determine whether extracted
    # contents are safe to
    # reuse.
    existing_extraction_valid = (
        existing_extraction_available
        and existing_expected_extensions_valid
    )

    # Reuse a complete validated extraction cache when refresh is not requested,
    # avoiding unnecessary
    # archive work.
    if (
        not extraction_refresh_requested
        and fire_hazard_reuse_extracted_cache
        and existing_extraction_valid
    ):
        existing_extracted_size_bytes = (
            sum(
                (file_path.stat().st_size for file_path in existing_extracted_files),
            )
        )
        print(f'-> Using cached extracted contents: {source_name}')
        print(f'   Directory: {extraction_directory}')
        print(f'   Files: {len(existing_extracted_files):,}')
        return {
            'SOURCE_NAME': source_name,
            'ARCHIVE_PATH': archive_path,
            'ARCHIVE_TYPE': archive_type,
            'ARCHIVE_SIZE_BYTES': archive_size_bytes,
            'EXTRACTION_DIRECTORY': extraction_directory,
            'EXTRACTION_ACTION': 'Used cached extraction',
            'CACHE_USED': True,
            'REFRESH_REQUESTED': False,
            'ARCHIVE_MEMBER_COUNT': None,
            'EXTRACTED_DIRECTORY_COUNT': None,
            'EXTRACTED_FILE_COUNT': len(existing_extracted_files),
            'EXTRACTED_SIZE_BYTES': existing_extracted_size_bytes,
            'EXTRACTED_SIZE_MB': existing_extracted_size_bytes / 1024 ** 2,
            'EXPECTED_EXTENSIONS': normalized_expected_extensions,
            'EXPECTED_EXTENSIONS_VALID': existing_expected_extensions_valid,
            'EXTRACTED_AT_UTC': None,
            'VALID': True,
        }
    # Create the required destination directory before files are downloaded, extracted,
    # or written.
    extraction_directory.mkdir(parents=True, exist_ok=True)
    # Initialize extraction counters used to report archive contents and QA totals.
    archive_member_count = 0
    extracted_file_count = 0
    extracted_directory_count = 0
    skipped_existing_file_count = 0
    extracted_file_paths = []

    # Write one already-validated archive member while honoring refresh and overwrite
    # rules.
    def write_archive_member(source_file_object, destination_path):

        """
        Writes one archive member to its validated
        extraction destination.
        """
        nonlocal extracted_file_count
        nonlocal skipped_existing_file_count
        # Create the required destination directory before files are downloaded,
        # extracted, or written.
        destination_path.parent.mkdir(parents=True, exist_ok=True)

        # Reuse an already extracted file when refresh is not requested instead of
        # overwriting it.
        if destination_path.exists() and (not extraction_refresh_requested):
            skipped_existing_file_count += 1
            extracted_file_paths.append(destination_path)
            return

        # Stop before overwriting an extracted file when refresh is requested but
        # project overwrite policy
        # forbids replacement.
        if (
            destination_path.exists()
            and extraction_refresh_requested
            and not fire_hazard_overwrite_extracted_files
        ):
            raise (
                FileExistsError(
                    (
                        f'An extracted file already exists and the overwrite policy is '
                        f'disabled:\n{destination_path}'
                    ),
                )
            )

        # Write the validated member as binary data after its destination has passed all
        # archive-safety
        # checks.
        with destination_path.open('wb') as destination_file:
            shutil.copyfileobj(source_file_object, destination_file)
        extracted_file_count += 1
        extracted_file_paths.append(destination_path)

    # Route ZIP sources through ZIP-specific integrity checks and safe member
    # extraction.
    if archive_type == 'zip':

        # Verify that the file contents form a valid ZIP archive rather than trusting
        # the filename extension
        # alone.
        if not zipfile.is_zipfile(archive_path):
            raise (
                zipfile.BadZipFile(
                    f'The source file does not contain a valid ZIP archive:\n{archive_path}',
                )
            )

        # Open the validated ZIP archive in read-only mode before inventorying and
        # extracting its members.
        with zipfile.ZipFile(archive_path, mode='r') as zip_archive:
            # Run the ZIP integrity test and capture the first corrupt member, if one
            # exists.
            corrupt_zip_member = zip_archive.testzip()

            # Stop extraction when the ZIP integrity test identifies a corrupt member.
            if corrupt_zip_member is not None:
                raise (
                    zipfile.BadZipFile(
                        f'The ZIP archive contains a corrupt member:\n{corrupt_zip_member}',
                    )
                )
            # Inventory ZIP members before extraction so every destination can be
            # validated first.
            zip_members = zip_archive.infolist()
            # Initialize extraction counters used to report archive contents and QA
            # totals.
            archive_member_count = len(zip_members)

            # Inspect each ZIP member, validate its destination, and extract only safe
            # directory or file
            # entries.
            for zip_member in zip_members:
                # Resolve and validate this member's destination before creating
                # directories or writing file
                # content.
                safe_destination = (
                    build_safe_fire_hazard_extraction_path(
                        extraction_directory=extraction_directory,
                        archive_member_name=zip_member.filename,
                    )
                )

                # Create validated directory members without attempting to read them as
                # files.
                if zip_member.is_dir():
                    # Create the required destination directory before files are
                    # downloaded, extracted, or
                    # written.
                    safe_destination.mkdir(parents=True, exist_ok=True)
                    extracted_directory_count += 1
                    continue

                # Close the archive member stream immediately after copying it to the
                # validated destination.
                with zip_archive.open(zip_member, mode='r') as source_file_object:
                    write_archive_member(
                        source_file_object=source_file_object,
                        destination_path=safe_destination,
                    )
    else:
        # Select the TAR reader mode based on whether the archive uses gzip compression.
        tar_read_mode = 'r:gz' if archive_type in {'tar.gz', 'tgz'} else 'r:'

        # Verify TAR structure before opening the source archive for member extraction.
        if not tarfile.is_tarfile(archive_path):
            raise (
                tarfile.ReadError(
                    f'The source file does not contain a valid TAR archive:\n{archive_path}',
                )
            )

        # Open the validated TAR archive with the compression mode detected from its
        # filename.
        with tarfile.open(archive_path, mode=tar_read_mode) as tar_archive:
            # Inventory TAR members before extraction so links and unsupported member
            # types can be screened.
            tar_members = tar_archive.getmembers()
            # Initialize extraction counters used to report archive contents and QA
            # totals.
            archive_member_count = len(tar_members)

            # Inspect each TAR member, reject unsafe links, and extract only validated
            # directories or
            # regular files.
            for tar_member in tar_members:
                # Resolve and validate this member's destination before creating
                # directories or writing file
                # content.
                safe_destination = (
                    build_safe_fire_hazard_extraction_path(
                        extraction_directory=extraction_directory,
                        archive_member_name=tar_member.name,
                    )
                )

                # Reject TAR symbolic and hard links because link targets can bypass
                # validated extraction
                # paths.
                if tar_member.issym() or tar_member.islnk():
                    raise (
                        ValueError(
                            f'Unsafe TAR link member detected:\n{tar_member.name}',
                        )
                    )

                # Create validated TAR directory members explicitly before continuing to
                # file entries.
                if tar_member.isdir():
                    # Create the required destination directory before files are
                    # downloaded, extracted, or
                    # written.
                    safe_destination.mkdir(parents=True, exist_ok=True)
                    extracted_directory_count += 1
                    continue

                # Skip unsupported TAR member types so only regular files are written
                # into the project
                # workspace.
                if not tar_member.isfile():
                    print(f'-> Skipping unsupported TAR member: {tar_member.name}')
                    continue
                # Open the TAR member as a readable stream only after the member type
                # and destination pass
                # validation.
                source_file_object = tar_archive.extractfile(tar_member)

                # Stop when a regular TAR member cannot be opened, preventing an
                # incomplete extraction from
                # being accepted.
                if source_file_object is None:
                    raise (
                        IOError(
                            (
                                f'The TAR member could not be opened for extraction:\n'
                                f'{tar_member.name}'
                            ),
                        )
                    )

                # Close the archive member stream immediately after copying it to the
                # validated destination.
                with source_file_object:
                    write_archive_member(
                        source_file_object=source_file_object,
                        destination_path=safe_destination,
                    )
    # Inventory all files present after extraction so the completed source package can
    # be validated.
    completed_extracted_files = [
        path
        for path in extraction_directory.rglob('*')
        if path.is_file()
    ]
    # Count completed extracted files for QA and the returned extraction record.
    completed_extracted_file_count = len(completed_extracted_files)
    # Measure total extracted size so empty or incomplete packages are rejected.
    completed_extracted_size_bytes = (
        sum(
            (file_path.stat().st_size for file_path in completed_extracted_files),
        )
    )
    # Collect final file extensions for source-specific completeness validation.
    completed_extracted_extensions = {
        file_path.suffix.lower()
        for file_path in completed_extracted_files
    }
    # Verify that the completed extraction contains every file type required by the
    # source.
    expected_extensions_valid = (
        not normalized_expected_extensions
        or all(
            (
                extension in completed_extracted_extensions
                for extension in normalized_expected_extensions
            ),
        )
    )
    # Confirm that archive extraction produced at least one usable file.
    extraction_contains_files = completed_extracted_file_count > 0
    # Confirm that extracted files contain nonzero data before downstream GIS
    # processing.
    extraction_size_valid = completed_extracted_size_bytes > 0

    # Reject an archive that produces no files because it cannot supply a usable hazard
    # dataset.
    if not extraction_contains_files:
        raise IOError(f'The {source_name} archive did not produce any extracted files.')

    # Reject extracted contents with zero total size as incomplete or invalid source
    # data.
    if not extraction_size_valid:
        raise IOError(f'The extracted {source_name} files contain no data.')

    # Stop when the extracted package lacks one or more source-specific GIS file types
    # required downstream.
    if not expected_extensions_valid:
        # Identify required file types missing from the extracted package for a clear
        # validation error.
        missing_expected_extensions = [
            extension
            for extension in normalized_expected_extensions
            if extension not in completed_extracted_extensions
        ]
        raise (
            ValueError(
                (
                    f'The extracted {source_name} archive is missing expected file extensions:\n'
                    f'{missing_expected_extensions}'
                ),
            )
        )
    # Record successful extraction time in UTC for reproducibility and cache metadata.
    extracted_at_utc = datetime.now(timezone.utc).isoformat()
    print(f'\n-> Archive extraction complete: {source_name}')
    print(f'-> Archive type: {archive_type}')
    print(f'-> Archive members inspected: {archive_member_count:,}')
    print(f'-> Files newly extracted: {extracted_file_count:,}')
    print(f'-> Existing files reused: {skipped_existing_file_count:,}')
    print(f'-> Total extracted files available: {completed_extracted_file_count:,}')
    print(f'-> Extracted size: {completed_extracted_size_bytes / 1024 ** 2:,.2f} MB')
    print(f'-> Extraction directory: {extraction_directory}')
    return {
        'SOURCE_NAME': source_name,
        'ARCHIVE_PATH': archive_path,
        'ARCHIVE_TYPE': archive_type,
        'ARCHIVE_SIZE_BYTES': archive_size_bytes,
        'EXTRACTION_DIRECTORY': extraction_directory,
        'EXTRACTION_ACTION': (
            'Refreshed extracted contents'
            if existing_extraction_available
            else 'Extracted new archive'
        ),
        'CACHE_USED': False,
        'REFRESH_REQUESTED': extraction_refresh_requested,
        'ARCHIVE_MEMBER_COUNT': archive_member_count,
        'EXTRACTED_DIRECTORY_COUNT': extracted_directory_count,
        'NEW_FILES_EXTRACTED': extracted_file_count,
        'EXISTING_FILES_REUSED': skipped_existing_file_count,
        'EXTRACTED_FILE_COUNT': completed_extracted_file_count,
        'EXTRACTED_SIZE_BYTES': completed_extracted_size_bytes,
        'EXTRACTED_SIZE_MB': completed_extracted_size_bytes / 1024 ** 2,
        'EXPECTED_EXTENSIONS': normalized_expected_extensions,
        'EXPECTED_EXTENSIONS_VALID': expected_extensions_valid,
        'EXTRACTED_AT_UTC': extracted_at_utc,
        'VALID': (
            extraction_contains_files
            and extraction_size_valid
            and expected_extensions_valid
        ),
    }

# Confirm that all archive helper functions were created successfully before they are
# used on source
# packages.
fire_hazard_archive_type_function_created = callable(identify_fire_hazard_archive_type)
fire_hazard_safe_path_function_created = (
    callable(
        build_safe_fire_hazard_extraction_path,
    )
)
fire_hazard_extraction_function_created = callable(extract_fire_hazard_source_archive)

# Stop if the archive-type helper was not created because extraction cannot safely
# choose ZIP or TAR handling.
if not fire_hazard_archive_type_function_created:
    raise TypeError('identify_fire_hazard_archive_type was not created successfully.')

# Stop if the safe-path helper was not created because archive members must be checked
# for path traversal.
if not fire_hazard_safe_path_function_created:
    raise (
        TypeError(
            'build_safe_fire_hazard_extraction_path was not created successfully.',
        )
    )

# Stop if the archive-extraction helper was not created because compressed source data
# cannot be unpacked safely.
if not fire_hazard_extraction_function_created:
    raise TypeError('extract_fire_hazard_source_archive was not created successfully.')

# Create a controlled path-safety test location used to verify that valid members are
# accepted and traversal
# attempts are rejected.
fire_hazard_safe_path_test_root = Path.cwd() / '_fire_hazard_safe_path_test'
safe_member_test_passed = False

# Exercise the safe-path helper with controlled test members to verify traversal
# protection before real
# extraction.
try:
    safe_member_test_path = (
        build_safe_fire_hazard_extraction_path(
            extraction_directory=fire_hazard_safe_path_test_root,
            archive_member_name='folder/example.tif',
        )
    )
    safe_member_test_passed = safe_member_test_path.name == 'example.tif'
# Record a failed valid-member test if the safety helper rejects an approved relative path.
except ValueError:
    safe_member_test_passed = False

# Initialize the unsafe-member test result separately so the archive path-safety helper
# can be verified against traversal attempts.
unsafe_member_test_passed = False

# Exercise the safe-path helper with controlled test members to verify traversal
# protection before real
# extraction.
try:
    build_safe_fire_hazard_extraction_path(
        extraction_directory=fire_hazard_safe_path_test_root,
        archive_member_name='../outside.tif',
    )
# Confirm traversal protection when the safety helper rejects the controlled unsafe path.
except ValueError:
    unsafe_member_test_passed = True

# Stop if the safe-path helper rejects a valid relative member, indicating the
# extraction guard is too
# restrictive.
if not safe_member_test_passed:
    raise (
        ValueError(
            'The archive safe-path function rejected a valid relative member path.',
        )
    )

# Stop if path traversal is not rejected, because archive extraction would be unsafe.
if not unsafe_member_test_passed:
    raise (
        ValueError(
            'The archive safe-path function did not reject a path-traversal test member.',
        )
    )

# Summarize supported archive formats, cache behavior, safety checks, and
# post-extraction validation.
fire_hazard_extraction_function_summary = (
    pd.DataFrame(
        [
            {'FUNCTION_CAPABILITY': 'ZIP Extraction', 'CONFIGURED': True},
            {'FUNCTION_CAPABILITY': 'TAR Extraction', 'CONFIGURED': True},
            {'FUNCTION_CAPABILITY': 'TAR.GZ Extraction', 'CONFIGURED': True},
            {'FUNCTION_CAPABILITY': 'TGZ Extraction', 'CONFIGURED': True},
            {'FUNCTION_CAPABILITY': 'Archive Integrity Check', 'CONFIGURED': True},
            {
                'FUNCTION_CAPABILITY': 'Path-Traversal Protection',
                'CONFIGURED': (
                    fire_hazard_require_safe_archive_extraction
                    and unsafe_member_test_passed
                ),
            },
            {'FUNCTION_CAPABILITY': 'Reject TAR Links', 'CONFIGURED': True},
            {
                'FUNCTION_CAPABILITY': 'Reuse Extracted Cache',
                'CONFIGURED': fire_hazard_reuse_extracted_cache,
            },
            {
                'FUNCTION_CAPABILITY': 'Refresh Extracted Files',
                'CONFIGURED': fire_hazard_overwrite_extracted_files,
            },
            {'FUNCTION_CAPABILITY': 'Expected Extension Validation', 'CONFIGURED': True},
            {'FUNCTION_CAPABILITY': 'Extracted File Count Validation', 'CONFIGURED': True},
            {'FUNCTION_CAPABILITY': 'Extracted File Size Validation', 'CONFIGURED': True},
        ],
    )
)
print(f'-> Archive-type function created: {fire_hazard_archive_type_function_created}')
print(f'-> Safe-path function created: {fire_hazard_safe_path_function_created}')
print(
    f'-> Archive extraction function created: {fire_hazard_extraction_function_created}',
)
print(f'-> Safe member test passed: {safe_member_test_passed}')
print(f'-> Unsafe member rejection passed: {unsafe_member_test_passed}')
print(f'-> Supported archive formats: {fire_hazard_supported_archive_extensions}')
print(f'-> Reuse extracted cache: {fire_hazard_reuse_extracted_cache}')
print(f'-> Overwrite extracted files: {fire_hazard_overwrite_extracted_files}')
print('\n--- FIRE HAZARD ARCHIVE EXTRACTION FUNCTION ---')
display(fire_hazard_extraction_function_summary)
print('\nNOTE:')
print(
    (
        'The reusable archive function is now available as '
        'extract_fire_hazard_source_archive().'
    ),
)
print(
    (
        'Archive members are validated before extraction to prevent absolute paths, '
        'parent-directory traversal, and unsafe TAR link targets.'
    ),
)
print(
    (
        'The function may reuse existing extracted files in snapshot mode or replace '
        'conflicting files when refresh mode and the overwrite policy permit it.'
    ),
)
print('No source archive was extracted during this step.')

print('\n=== FIRE HAZARD ARCHIVE EXTRACTION FUNCTION READY ===')


=== CREATING RETRY-ENABLED FIRE HAZARD DOWNLOAD SESSION ===
-> Requests session created: True
-> HTTPS retry adapter configured: True
-> HTTP retry adapter configured: True
-> Retry attempts configured: 5
-> Retry status codes: [429, 500, 502, 503, 504]
-> Request timeout: (15, 120)
-> SSL certificate verification: True
-> User-Agent applied: True

--- RETRY-ENABLED DOWNLOAD SESSION ---


,SESSION_SETTING,VALUE
0,Session Type,Session
1,HTTPS Adapter Configured,True
2,HTTP Adapter Configured,True
3,Total Retries,5
4,Connection Retries,5
5,Read Retries,5
6,Status Retries,5
7,Backoff Factor,1
8,Retry Status Codes,"429, 500, 502, 503, 504"
9,Allowed Retry Methods,"GET, HEAD"



NOTE:
The retry-enabled session is now available as fire_hazard_request_session and will be reused by later source-download functions.
The session automatically retries HTTP 429, 500, 502, 503, and 504 responses using progressive backoff delays.
No remote source request was submitted during this step. The next step will define the reusable file-download function.

=== RETRY-ENABLED FIRE HAZARD DOWNLOAD SESSION READY ===
=== CREATING FIRE HAZARD SOURCE FILE DOWNLOAD FUNCTION ===
-> Download function created: True
-> Function name: download_fire_hazard_source_file
-> Current acquisition mode: snapshot
-> Temporary file suffix: .part
-> Download chunk size: 1 MB
-> Whole-file transfer attempts: 4
-> Whole-file initial backoff: 5 seconds
-> Whole-file maximum backoff: 60 seconds
-> Minimum file size: 1 bytes

--- FIRE HAZARD DOWNLOAD FUNCTION ---


,FUNCTION_CAPABILITY,CONFIGURED
0,Uses Retry-Enabled Session,True
1,Supports Cached Files,True
2,Supports Explicit Refresh,True
3,Streams Large Downloads,True
4,Whole-File Transfer Retries,True
5,Whole-File Retry Attempts,4
6,Exponential Transfer Backoff,True
7,Uses Temporary Download File,True
8,Validates Minimum File Size,True
9,Validates Applicable Content-Length,True



NOTE:
The reusable download function now combines the existing HTTP adapter retries with whole-file transfer retries for interrupted streamed downloads.
Transient ChunkedEncodingError, connection, read-timeout, and timeout failures restart the complete file transfer using bounded exponential backoff.
Each failed attempt removes its incomplete .part file. A completed source is promoted to the final cache path only after all configured validation checks pass.
Deterministic validation failures and exhausted HTTP errors are not blindly retried at the file-transfer layer.
An existing validated cached source remains protected when a refresh fails and the preserve-cache policy is enabled.
No source file was downloaded during this step.

=== FIRE HAZARD SOURCE FILE DOWNLOAD FUNCTION READY ===
=== CREATING FIRE HAZARD ARCHIVE EXTRACTION FUNCTION ===
-> Archive-type function created: True
-> Safe-path function created: True
-> Archive extraction function created: True
-> Safe member test passed

,FUNCTION_CAPABILITY,CONFIGURED
0,ZIP Extraction,True
1,TAR Extraction,True
2,TAR.GZ Extraction,True
3,TGZ Extraction,True
4,Archive Integrity Check,True
5,Path-Traversal Protection,True
6,Reject TAR Links,True
7,Reuse Extracted Cache,True
8,Refresh Extracted Files,False
9,Expected Extension Validation,True



NOTE:
The reusable archive function is now available as extract_fire_hazard_source_archive().
Archive members are validated before extraction to prevent absolute paths, parent-directory traversal, and unsafe TAR link targets.
The function may reuse existing extracted files in snapshot mode or replace conflicting files when refresh mode and the overwrite policy permit it.
No source archive was extracted during this step.

=== FIRE HAZARD ARCHIVE EXTRACTION FUNCTION READY ===


# CHRG Phase 7 – Vegetation Dryness

## Purpose

Acquire and process HLS imagery, apply quality masks and reflectance scaling, calculate NDVI and NDMI, align scenes, create temporal composites, and build the vegetation-dryness hazard component.


## Harmonized Landsat and Sentinel-2 Planetary Computer Search and Asset Acquisition


### Configuring Planetary Computer HLS Acquisition


In [62]:
print('=== CONFIGURING PLANETARY COMPUTER HLS ACQUISITION ===')
# List the required HLS planetary computer prerequisites required before planetary computer
# HLS acquisition can run.
required_hls_planetary_computer_inputs = ['fire_hazard_source_catalog',
    'fire_hazard_raw_vegetation_directory', 'fire_hazard_data_mode',
    'fire_hazard_dataset_versions', 'gdf_county_boundaries_aligned']
# Identify unavailable HLS planetary computer so planetary computer HLS acquisition stops
# before using incomplete inputs.
missing_hls_planetary_computer_inputs = [object_name for object_name in \
    required_hls_planetary_computer_inputs if object_name not in globals()]

# Stop execution when required hls planetary computer inputs inputs are unavailable.
if missing_hls_planetary_computer_inputs:
    raise NameError(f'The following Planetary '
        f'Computer HLS configuration '
        f'objects are missing:\n'
        f'{missing_hls_planetary_computer_inputs}\n\n'
        f'Run the source-catalog, '
        f'directory, cache-policy, '
        f'dataset-version, and '
        f'county-boundary cells before '
        f'configuring HLS acquisition.')
import json
from pathlib import Path
from shapely.geometry import mapping
# Extract collections from the current record for planetary computer HLS acquisition.
fire_hazard_hls_collections = fire_hazard_dataset_versions.get('hls_collections',
    ['hls2-l30', 'hls2-s30'])
# Define expected hls collections used to configure this workflow stage.
expected_hls_collections = {'hls2-l30', 'hls2-s30'}

# Require The HLS collection configuration to contain hls2-l30 and hls2-s30.
if not isinstance(fire_hazard_hls_collections,
    list) or set(fire_hazard_hls_collections) != expected_hls_collections:
    raise ValueError('The HLS collection configuration must contain hls2-l30 and hls2-s30.')

# Decode HLS quality flags to remove clouds, cloud shadows,
# cirrus, snow, and other invalid observations.
# Define asset requirements to control the inputs and rules used by planetary computer HLS
# acquisition.
fire_hazard_hls_asset_requirements = {'hls2-l30': {'red': 'B04',
    'nir': 'B05', 'swir1': 'B06', 'quality': 'Fmask'},
    'hls2-s30': {'red': 'B04', 'nir': 'B8A', 'swir1': 'B11',
    'quality': 'Fmask'}}
# Set datetime range to control planetary computer HLS acquisition.
fire_hazard_hls_datetime_range = '2025-07-01/2025-09-30'
# Extract analysis period from the current record for planetary computer HLS acquisition.
fire_hazard_hls_analysis_period = fire_hazard_dataset_versions.get('hls_analysis_period',
    'July-September 2025')
# Set max cloud cover to control planetary computer HLS acquisition.
fire_hazard_hls_max_cloud_cover = 20.0

# Require The HLS cloud-cover threshold to be between 0 and 100 percent.
if not 0.0 <= fire_hazard_hls_max_cloud_cover <= 100.0:
    raise ValueError('The HLS cloud-cover threshold must be between 0 and 100 percent.')
# Reproject the study-area counties to WGS 84 so their geometry can drive the STAC search.
gdf_fire_hazard_counties_hls = gdf_county_boundaries_aligned.to_crs('EPSG:4326').copy()

# Stop execution if hLS study-area GeoDataFrame contains no records.
if gdf_fire_hazard_counties_hls.empty:
    raise ValueError('The HLS study-area GeoDataFrame contains no records.')

# Dissolve the county polygons with the current GeoPandas union API.
fire_hazard_hls_study_area_geometry = (
    gdf_fire_hazard_counties_hls.geometry.union_all()
)

# Stop execution if combined HLS study-area geometry is empty.
if fire_hazard_hls_study_area_geometry.is_empty:
    raise ValueError('The combined HLS study-area geometry is empty.')
# Prepare intersects geometry to define the spatial extent used by planetary computer HLS
# acquisition.
fire_hazard_hls_intersects_geometry = mapping(fire_hazard_hls_study_area_geometry)
# Build the raw directory location so planetary computer HLS acquisition uses the expected project
# file structure.
fire_hazard_hls_raw_directory = fire_hazard_raw_vegetation_directory / \
    'hls_planetary_computer_2025_fire_season'
# Build the STAC directory location so planetary computer HLS acquisition uses the expected project
# file structure.
fire_hazard_hls_stac_directory = fire_hazard_hls_raw_directory / 'stac_items'
# Build the manifest directory location so planetary computer HLS acquisition uses the expected
# project file structure.
fire_hazard_hls_manifest_directory = fire_hazard_hls_raw_directory / 'manifests'

# Process each directory path entry so planetary computer HLS acquisition is applied consistently
# across all records.
for directory_path in [fire_hazard_hls_raw_directory,
    fire_hazard_hls_stac_directory, fire_hazard_hls_manifest_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the STAC cache paths location so planetary computer HLS acquisition uses the expected
# project file structure.
fire_hazard_hls_stac_cache_paths = {collection_id: fire_hazard_hls_stac_directory / \
    f'{collection_id}_2025_fire_season_items.json' for collection_id in fire_hazard_hls_collections}
# Build the item manifest path location so planetary computer HLS acquisition uses the expected
# project file structure.
fire_hazard_hls_item_manifest_path = fire_hazard_hls_manifest_directory / \
    'hls_2025_fire_season_item_manifest.csv'
# Build the asset manifest path location so planetary computer HLS acquisition uses the expected
# project file structure.
fire_hazard_hls_asset_manifest_path = fire_hazard_hls_manifest_directory / \
    'hls_2025_fire_season_asset_manifest.csv'
print(f'-> STAC endpoint: {fire_hazard_planetary_computer_stac_url}')
print(f'-> HLS collections: {fire_hazard_hls_collections}')
print(f'-> Analysis period: {fire_hazard_hls_analysis_period}')
print(f'-> STAC date interval: {fire_hazard_hls_datetime_range}')
print(f'-> Maximum scene cloud cover: {fire_hazard_hls_max_cloud_cover}%')
print(f'-> Data mode: {fire_hazard_data_mode}')
print(f'-> Metadata cache directory: {fire_hazard_hls_raw_directory}')
print('\n=== PLANETARY COMPUTER HLS ACQUISITION CONFIGURED ===')


=== CONFIGURING PLANETARY COMPUTER HLS ACQUISITION ===
-> STAC endpoint: https://planetarycomputer.microsoft.com/api/stac/v1
-> HLS collections: ['hls2-l30', 'hls2-s30']
-> Analysis period: July-September 2025
-> STAC date interval: 2025-07-01/2025-09-30
-> Maximum scene cloud cover: 20.0%
-> Data mode: snapshot
-> Metadata cache directory: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season

=== PLANETARY COMPUTER HLS ACQUISITION CONFIGURED ===


### Verifying Planetary Computer Acquisition Prerequisites


In [63]:
print('=== VERIFYING PLANETARY COMPUTER ACQUISITION PREREQUISITES ===')
# List the required rest objects prerequisites required before planetary computer acquisition
# prerequisites can run.
required_rest_objects = ['fire_hazard_request_session',
    'fire_hazard_request_timeout', 'fire_hazard_verify_ssl_certificates',
    'fire_hazard_planetary_computer_stac_url']
# Identify unavailable rest objects so planetary computer acquisition prerequisites stops before
# using incomplete inputs.
missing_rest_objects = [object_name for object_name in required_rest_objects if object_name not \
    in globals()]

# Stop execution when required rest objects inputs are unavailable.
if missing_rest_objects:
    raise NameError(f'The following Planetary '
        f'Computer acquisition '
        f'prerequisites are missing:\n'
        f'{missing_rest_objects}\n\nRun '
        f'the project constants and step '
        f'Create Retry-Enabled Download '
        f'Session before continuing.')

# Stop execution if fire_hazard_request_session is not a valid requests.Session object.
if not isinstance(fire_hazard_request_session, requests.Session):
    raise TypeError('fire_hazard_request_session is not a valid requests.Session object.')

# Require fire_hazard_request_timeout to be a two-value tuple.
if not isinstance(fire_hazard_request_timeout, tuple) or len(fire_hazard_request_timeout) != 2:
    raise TypeError('fire_hazard_request_timeout must be a two-value tuple.')

# Require fire_hazard_verify_ssl_certificates to be a Boolean value.
if not isinstance(fire_hazard_verify_ssl_certificates, bool):
    raise TypeError('fire_hazard_verify_ssl_certificates must be a Boolean value.')

# Require fire_hazard_planetary_computer_stac_url to contain a valid HTTPS endpoint.
if not isinstance(fire_hazard_planetary_computer_stac_url,
    str) or not fire_hazard_planetary_computer_stac_url.startswith('https://'):
    raise ValueError('fire_hazard_planetary_computer_stac_url must contain a valid HTTPS endpoint.')
print('-> Retry-enabled HTTP session available: True')
print(f'-> Request timeout: {fire_hazard_request_timeout}')
print(f'-> SSL verification enabled: {fire_hazard_verify_ssl_certificates}')
print(f'-> Planetary Computer STAC endpoint: {fire_hazard_planetary_computer_stac_url}')
print('-> pystac-client required: False')
print('-> planetary-computer package required: False')
print('\n=== PLANETARY COMPUTER ACQUISITION PREREQUISITES AVAILABLE ===')



=== VERIFYING PLANETARY COMPUTER ACQUISITION PREREQUISITES ===
-> Retry-enabled HTTP session available: True
-> Request timeout: (15, 120)
-> SSL verification enabled: True
-> Planetary Computer STAC endpoint: https://planetarycomputer.microsoft.com/api/stac/v1
-> pystac-client required: False
-> planetary-computer package required: False

=== PLANETARY COMPUTER ACQUISITION PREREQUISITES AVAILABLE ===


### Connecting to Microsoft Planetary Computer


In [64]:
print('=== CONNECTING TO MICROSOFT PLANETARY COMPUTER ===')
# List the required HLS connection prerequisites required before microsoft planetary computer
# can run.
required_hls_connection_inputs = ['fire_hazard_planetary_computer_stac_url',
    'fire_hazard_hls_collections', 'fire_hazard_request_session',
    'fire_hazard_request_timeout', 'fire_hazard_verify_ssl_certificates']
# Identify unavailable HLS connection so microsoft planetary computer stops before using
# incomplete inputs.
missing_hls_connection_inputs = [object_name for object_name in required_hls_connection_inputs if \
    object_name not in globals()]

# Stop execution when required hls connection inputs inputs are unavailable.
if missing_hls_connection_inputs:
    raise NameError(f'The following Planetary '
        f'Computer connection objects '
        f'are missing:\n'
        f'{missing_hls_connection_inputs}\n\n'
        f'Run step Configure Planetary '
        f'Computer HLS Acquisition and '
        f'the preceding prerequisite '
        f'validation cell before '
        f'connecting.')

# Connect to Microsoft Planetary Computer so HLS scenes can be
# searched and signed for download.
# Encapsulate get planetary computer collection so repeated microsoft planetary computer steps use
# consistent logic.
def get_planetary_computer_collection(collection_id):

    """
    Retrieves metadata for one Planetary Computer
    STAC collection using the public REST API.
    """

    # Require collection_id to contain a non-empty STAC collection identifier.
    if not isinstance(collection_id, str) or not collection_id.strip():
        raise ValueError('collection_id must contain a non-empty STAC collection identifier.')
    # Define collection url used to access the configured source service.
    collection_url = f'{fire_hazard_planetary_computer_stac_url}/collections/{collection_id}'
    # Capture the response response so service status and returned content can be validated.
    response = fire_hazard_request_session.get(collection_url,
        timeout=fire_hazard_request_timeout, verify=fire_hazard_verify_ssl_certificates)

    # Stop execution if planetary Computer STAC API did not find the configured collection: the
    # reported value.
    if response.status_code == 404:
        # Close the resource after processing is complete.
        response.close()
        raise ValueError(f'The Planetary Computer STAC '
            f'API did not find the '
            f'configured collection:\n'
            f'{collection_id}')
    response.raise_for_status()

    # Protect microsoft planetary computer so expected source or file failures do not leave partial
    # outputs.
    try:
        # Parse collection metadata from returned content so microsoft planetary computer can use
        # structured metadata.
        collection_metadata = response.json()
    finally:
        # Close the resource after processing is complete.
        response.close()

    # Stop execution if planetary Computer STAC API returned an unexpected collection response.
    if not isinstance(collection_metadata, dict):
        raise TypeError('The Planetary Computer STAC '
            'API returned an unexpected '
            'collection response.')

    # Stop execution if returned STAC collection identifier does not match the requested collection.
    if collection_metadata.get('id') != collection_id:
        raise ValueError('The returned STAC collection '
            'identifier does not match the '
            'requested collection.')
    return collection_metadata
# Define collection records to control the inputs and rules used by microsoft planetary computer.
fire_hazard_hls_collection_records = []
# Prepare collection metadata to preserve and validate raster structure during microsoft planetary
# computer.
fire_hazard_hls_collection_metadata = {}

# Process each collection ID entry so microsoft planetary computer is applied consistently across
# all records.
for collection_id in fire_hazard_hls_collections:
    # Prepare collection metadata to preserve and validate raster structure during microsoft
    # planetary computer.
    collection_metadata = get_planetary_computer_collection(collection_id)
    # Prepare collection metadata[collection id] to preserve and validate raster structure during
    # microsoft planetary computer.
    fire_hazard_hls_collection_metadata[collection_id] = collection_metadata
    fire_hazard_hls_collection_records.append({'COLLECTION_ID': collection_metadata.get('id'),
        'TITLE': collection_metadata.get('title'), 'DESCRIPTION': \
            collection_metadata.get('description'),
        'LICENSE': collection_metadata.get('license'),
        'STAC_VERSION': collection_metadata.get('stac_version'),
        'VALID': collection_metadata.get('id') == collection_id})
# Store collection summary needed to carry out microsoft planetary computer.
fire_hazard_hls_collection_summary = pd.DataFrame(fire_hazard_hls_collection_records)
# Build failed hls collection records used to track the records included in this processing stage.
failed_hls_collection_records = \
    fire_hazard_hls_collection_summary[~fire_hazard_hls_collection_summary['VALID']]

# Stop execution if one or more Planetary Computer HLS collections failed validation.
if not failed_hls_collection_records.empty:
    raise ValueError('One or more Planetary Computer HLS collections failed validation.')
print('-> Planetary Computer STAC connection created.')
print(f'-> HLS collections validated: {len(fire_hazard_hls_collection_summary)}')
print('\n--- HLS STAC COLLECTIONS ---')
display(fire_hazard_hls_collection_summary)
print('\nNOTE:')
print("This workflow uses the public "
    "Planetary Computer REST API "
    "through the project's existing "
    "Requests session.")
print('No NASA Earthdata username, '
    'pystac-client, '
    'planetary-computer package, or '
    'interactive authentication is '
    'required.')
print('\n=== PLANETARY COMPUTER HLS COLLECTIONS VALIDATED ===')



=== CONNECTING TO MICROSOFT PLANETARY COMPUTER ===
-> Planetary Computer STAC connection created.
-> HLS collections validated: 2

--- HLS STAC COLLECTIONS ---


,COLLECTION_ID,TITLE,DESCRIPTION,LICENSE,STAC_VERSION,VALID
0,hls2-l30,Harmonized Landsat Sentinel-2 (HLS) Version 2....,Harmonized Landsat Sentinel-2 (HLS) Version 2....,proprietary,1.0.0,True
1,hls2-s30,Harmonized Landsat Sentinel-2 (HLS) Version 2....,Harmonized Landsat Sentinel-2 (HLS) Version 2....,proprietary,1.0.0,True



NOTE:
This workflow uses the public Planetary Computer REST API through the project's existing Requests session.
No NASA Earthdata username, pystac-client, planetary-computer package, or interactive authentication is required.

=== PLANETARY COMPUTER HLS COLLECTIONS VALIDATED ===


### Searching Planetary Computer for HLS Items


In [65]:
print('=== SEARCHING PLANETARY COMPUTER FOR HLS ITEMS ===')
# List the required HLS search prerequisites required before planetary computer for HLS items
# can run.
required_hls_search_inputs = ['fire_hazard_planetary_computer_stac_url',
    'fire_hazard_hls_collections', 'fire_hazard_hls_intersects_geometry',
    'fire_hazard_hls_datetime_range', 'fire_hazard_hls_max_cloud_cover',
    'fire_hazard_hls_asset_requirements', 'fire_hazard_hls_stac_cache_paths',
    'fire_hazard_hls_item_manifest_path', 'fire_hazard_hls_asset_manifest_path',
    'fire_hazard_request_session', 'fire_hazard_request_timeout',
    'fire_hazard_verify_ssl_certificates', 'fire_hazard_data_mode']
# Identify unavailable HLS search so planetary computer for HLS items stops before using
# incomplete inputs.
missing_hls_search_inputs = [object_name for object_name in required_hls_search_inputs if \
    object_name not in globals()]

# Stop execution when required hls search inputs inputs are unavailable.
if missing_hls_search_inputs:
    raise NameError(f'The following Planetary '
        f'Computer HLS search objects '
        f'are missing:\n'
        f'{missing_hls_search_inputs}\n\n'
        f'Run step Configure Planetary '
        f'Computer HLS Acquisition, step '
        f'Verify Planetary Computer '
        f'Acquisition Prerequisites, and '
        f'step Connect to Planetary '
        f'Computer and Validate HLS '
        f'Collections first.')
# Define fire hazard planetary computer search url used to access the configured source service.
fire_hazard_planetary_computer_search_url = f'{fire_hazard_planetary_computer_stac_url}/search'
# Define item collections to control the inputs and rules used by planetary computer for HLS items.
fire_hazard_hls_item_collections = {}
# Define item records to control the inputs and rules used by planetary computer for HLS items.
fire_hazard_hls_item_records = []
# Define asset records to control the inputs and rules used by planetary computer for HLS items.
fire_hazard_hls_asset_records = []

# Process each collection ID entry so planetary computer for HLS items is applied consistently
# across all records.
for collection_id in fire_hazard_hls_collections:
    # Build the STAC cache path location so planetary computer for HLS items uses the expected
    # project file structure.
    stac_cache_path = fire_hazard_hls_stac_cache_paths[collection_id]

    # Use the acquisition path selected by the configured fire-hazard data mode.
    if fire_hazard_data_mode == 'refresh':
        # Define hls search payload used to configure this workflow stage.
        hls_search_payload = {'collections': [collection_id],
            'intersects': fire_hazard_hls_intersects_geometry,
            'datetime': fire_hazard_hls_datetime_range, 'query': {'eo:cloud_cover': {'lt': \
                fire_hazard_hls_max_cloud_cover}},
            'limit': 1000}
        # Capture the search response response so service status and returned content can be
        # validated.
        search_response = \
            fire_hazard_request_session.post(fire_hazard_planetary_computer_search_url,
            json=hls_search_payload, timeout=fire_hazard_request_timeout,
            verify=fire_hazard_verify_ssl_certificates)
        search_response.raise_for_status()

        # Protect planetary computer for HLS items so expected source or file failures do not leave
        # partial outputs.
        try:
            # Parse item collection from returned content so planetary computer for HLS items can
            # use structured metadata.
            hls_item_collection = search_response.json()
        finally:
            # Close the resource after processing is complete.
            search_response.close()

        # Stop execution if planetary Computer STAC search returned an unexpected response
        # structure.
        if not isinstance(hls_item_collection, dict):
            raise TypeError('The Planetary Computer STAC '
                'search returned an unexpected '
                'response structure.')
        # Extract features from the current record for planetary computer for HLS items.
        hls_features = hls_item_collection.get('features', [])

        # Stop execution if no Planetary Computer HLS items were found for collection the reported
        # value.
        if not hls_features:
            raise ValueError(f'No Planetary Computer HLS '
                f'items were found for '
                f'collection {collection_id}.')

        # Open the file in a managed context so its handle closes reliably after planetary computer
        # for HLS items.
        with stac_cache_path.open('w', encoding='utf-8') as stac_cache_file:
            # Write the structured metadata needed to reproduce this processing stage.
            json.dump(hls_item_collection, stac_cache_file, indent=2)
    else:

        # Stop execution if cached HLS STAC metadata is missing: the reported value Change
        # fire_hazard_data_mode to 'refresh' and rerun this cell.
        if not stac_cache_path.exists():
            raise FileNotFoundError(f"Cached HLS STAC metadata is "
                f"missing:\n{stac_cache_path}\n\n"
                f"Change fire_hazard_data_mode "
                f"to 'refresh' and rerun this "
                f"cell.")

        # Open the file in a managed context so its handle closes reliably after planetary computer
        # for HLS items.
        with stac_cache_path.open('r', encoding='utf-8') as stac_cache_file:
            # Parse item collection from returned content so planetary computer for HLS items can
            # use structured metadata.
            hls_item_collection = json.load(stac_cache_file)
        # Extract features from the current record for planetary computer for HLS items.
        hls_features = hls_item_collection.get('features', [])

        # Stop execution if cached HLS STAC metadata contains no items for the reported value.
        if not hls_features:
            raise ValueError(f'The cached HLS STAC metadata contains no items for {collection_id}.')
    # Record item collections[collection id] needed to identify the current HLS asset in manifests
    # and requests.
    fire_hazard_hls_item_collections[collection_id] = hls_item_collection
    # List the required assets prerequisites required before planetary computer for HLS items can
    # run.
    required_assets = fire_hazard_hls_asset_requirements[collection_id]

    # Process each item entry so planetary computer for HLS items is applied consistently across all
    # records.
    for item in hls_features:
        # Extract item ID from the current record for planetary computer for HLS items.
        item_id = item.get('id')
        # Extract item properties from the current record for planetary computer for HLS items.
        item_properties = item.get('properties', {})
        # Extract item assets from the current record for planetary computer for HLS items.
        item_assets = item.get('assets', {})

        # Stop execution if a Planetary Computer HLS item is missing its STAC item identifier.
        if not item_id:
            raise ValueError('A Planetary Computer HLS item is missing its STAC item identifier.')
        # Record available asset keys needed to identify the current HLS asset in manifests and
        # requests.
        available_asset_keys = set(item_assets.keys())
        # Identify unavailable item assets so planetary computer for HLS items stops before
        # using incomplete inputs.
        missing_item_assets = [asset_key for asset_key in required_assets.values() if asset_key \
            not in available_asset_keys]

        # Stop execution when required item assets inputs are unavailable.
        if missing_item_assets:
            raise ValueError(f'HLS item {item_id} is missing '
                f'required assets: '
                f'{missing_item_assets}\n'
                f'Available assets: '
                f'{sorted(available_asset_keys)}')
        # Extract item datetime from the current record for planetary computer for HLS items.
        item_datetime = item_properties.get('datetime')
        fire_hazard_hls_item_records.append({'COLLECTION_ID': collection_id,
            'ITEM_ID': item_id, 'DATETIME': item_datetime,
            'PLATFORM': item_properties.get('platform'), 'CLOUD_COVER': \
                item_properties.get('eo:cloud_cover'),
            'MGRS_TILE': item_properties.get('s2:mgrs_tile') or item_properties.get('mgrs:tile') \
                or item_properties.get('mgrs:utm_zone'),
            'STAC_CACHE_PATH': str(stac_cache_path)})

        # Process each (asset role entry so planetary computer for HLS items is applied consistently
        # across all records.
        for asset_role, asset_key in required_assets.items():
            # Prepare asset metadata to preserve and validate raster structure during planetary
            # computer for HLS items.
            asset_metadata = item_assets[asset_key]
            # Extract unsigned href from the current record for planetary computer for HLS items.
            unsigned_href = asset_metadata.get('href')

            # Stop execution if hLS item the reported value, asset the reported value, does not
            # provide a valid source URL.
            if not isinstance(unsigned_href, str) or not unsigned_href.strip():
                raise ValueError(f'HLS item {item_id}, asset '
                    f'{asset_key}, does not provide '
                    f'a valid source URL.')
            # Store unsigned href so the current HLS asset can be signed or downloaded reliably.
            unsigned_href = unsigned_href.split('?', 1)[0]
            fire_hazard_hls_asset_records.append({'COLLECTION_ID': collection_id,
                'ITEM_ID': item_id, 'DATETIME': item_datetime,
                'ASSET_ROLE': asset_role, 'ASSET_KEY': asset_key,
                'UNSIGNED_HREF': unsigned_href, 'MEDIA_TYPE': asset_metadata.get('type')})
# Store item manifest needed to carry out planetary computer for HLS items.
fire_hazard_hls_item_manifest = \
    pd.DataFrame(fire_hazard_hls_item_records).sort_values(['COLLECTION_ID',
    'DATETIME', 'ITEM_ID']).reset_index(drop=True)
# Store asset manifest needed to carry out planetary computer for HLS items.
fire_hazard_hls_asset_manifest = \
    pd.DataFrame(fire_hazard_hls_asset_records).sort_values(['COLLECTION_ID',
    'DATETIME', 'ITEM_ID', 'ASSET_ROLE']).reset_index(drop=True)

# Stop execution if planetary Computer HLS item manifest contains no records.
if fire_hazard_hls_item_manifest.empty:
    raise ValueError('The Planetary Computer HLS item manifest contains no records.')

# Stop execution if planetary Computer HLS asset manifest contains no records.
if fire_hazard_hls_asset_manifest.empty:
    raise ValueError('The Planetary Computer HLS asset manifest contains no records.')
# Save the item manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_item_manifest.to_csv(fire_hazard_hls_item_manifest_path, index=False)
# Save the asset manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_asset_manifest.to_csv(fire_hazard_hls_asset_manifest_path, index=False)
print(f'-> HLS observations discovered: {len(fire_hazard_hls_item_manifest):,}')
print(f'-> Required assets inventoried: {len(fire_hazard_hls_asset_manifest):,}')
print(f'-> Item manifest saved: {fire_hazard_hls_item_manifest_path}')
print(f'-> Asset manifest saved: {fire_hazard_hls_asset_manifest_path}')
print('\n--- HLS ITEM INVENTORY ---')
display(fire_hazard_hls_item_manifest.head(5))
print('\n=== PLANETARY COMPUTER HLS SEARCH AND CACHE COMPLETE ===')


=== SEARCHING PLANETARY COMPUTER FOR HLS ITEMS ===
-> HLS observations discovered: 313
-> Required assets inventoried: 1,252
-> Item manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\manifests\hls_2025_fire_season_item_manifest.csv
-> Asset manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\manifests\hls_2025_fire_season_asset_manifest.csv

--- HLS ITEM INVENTORY ---


,COLLECTION_ID,ITEM_ID,DATETIME,PLATFORM,CLOUD_COVER,MGRS_TILE,STAC_CACHE_PATH
0,hls2-l30,HLS.L30.T11SQA.2025184T181456.v2.0,2025-07-03T18:14:56.471369Z,landsat-8,6.0,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
1,hls2-l30,HLS.L30.T12STF.2025184T181456.v2.0,2025-07-03T18:14:56.471369Z,landsat-8,9.0,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
2,hls2-l30,HLS.L30.T11SQA.2025185T180843.v2.0,2025-07-04T18:08:43.163500Z,landsat-9,1.0,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
3,hls2-l30,HLS.L30.T11SQB.2025185T180843.v2.0,2025-07-04T18:08:43.163500Z,landsat-9,3.0,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
4,hls2-l30,HLS.L30.T12STF.2025185T180843.v2.0,2025-07-04T18:08:43.163500Z,landsat-9,1.0,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...



=== PLANETARY COMPUTER HLS SEARCH AND CACHE COMPLETE ===


### Selecting Best HLS Fire-Season Scenes


In [66]:
print('=== SELECTING BEST HLS FIRE-SEASON SCENES ===')
# List the required HLS scene selection prerequisites required before best HLS fire-season
# scenes can run.
required_hls_scene_selection_inputs = ['fire_hazard_hls_item_manifest',
    'fire_hazard_hls_asset_manifest']
# Identify unavailable HLS scene selection so best HLS fire-season scenes stops before
# using incomplete inputs.
missing_hls_scene_selection_inputs = [object_name for object_name in \
    required_hls_scene_selection_inputs if object_name not in globals()]

# Stop execution when required hls scene selection inputs inputs are unavailable.
if missing_hls_scene_selection_inputs:
    raise NameError(f'The following HLS '
        f'scene-selection objects are '
        f'missing:\n'
        f'{missing_hls_scene_selection_inputs}\n\n'
        f'Run step Search and Cache HLS '
        f'STAC Items before selecting '
        f'the best observations.')

# Stop execution if hLS item manifest contains no scenes.
if fire_hazard_hls_item_manifest.empty:
    raise ValueError('The HLS item manifest contains no scenes.')

# Stop execution if hLS asset manifest contains no records.
if fire_hazard_hls_asset_manifest.empty:
    raise ValueError('The HLS asset manifest contains no records.')
# Set max scenes per tile month to control best HLS fire-season scenes.
fire_hazard_hls_max_scenes_per_tile_month = 2
# Create an isolated scene candidates working copy so best HLS fire-season scenes does not modify
# upstream data.
fire_hazard_hls_scene_candidates = fire_hazard_hls_item_manifest.copy()
# Store scene candidates['datetime parsed'] needed to carry out best HLS fire-season scenes.
fire_hazard_hls_scene_candidates['DATETIME_PARSED'] = \
    pd.to_datetime(fire_hazard_hls_scene_candidates['DATETIME'],
    utc=True, errors='coerce')

# Stop execution if one or more HLS scene timestamps could not be parsed.
if fire_hazard_hls_scene_candidates['DATETIME_PARSED'].isna().any():
    raise ValueError('One or more HLS scene timestamps could not be parsed.')
# Store parsed MGRS tiles needed to carry out best HLS fire-season scenes.
parsed_mgrs_tiles = \
    fire_hazard_hls_scene_candidates['ITEM_ID'].str.extract('\\.(T[0-9]{2}[A-Z]{3})\\.',
    expand=False)

# Handle the scene candidates case explicitly during best HLS fire-season scenes.
if 'MGRS_TILE' not in fire_hazard_hls_scene_candidates.columns:
    # Store scene candidates['MGRS tile'] needed to carry out best HLS fire-season scenes.
    fire_hazard_hls_scene_candidates['MGRS_TILE'] = parsed_mgrs_tiles
else:
    # Store scene candidates['MGRS tile'] needed to carry out best HLS fire-season scenes.
    fire_hazard_hls_scene_candidates['MGRS_TILE'] = \
        fire_hazard_hls_scene_candidates['MGRS_TILE'].fillna(parsed_mgrs_tiles)

# Stop execution if one or more HLS scenes do not have a resolvable MGRS tile identifier.
if fire_hazard_hls_scene_candidates['MGRS_TILE'].isna().any():
    raise ValueError('One or more HLS scenes do not have a resolvable MGRS tile identifier.')
# Convert UTC timestamps to timezone-naive UTC values before creating monthly grouping keys.
# Period values do not retain timezone metadata, so removing the timezone explicitly avoids an
# implicit-conversion warning while preserving the intended UTC calendar month.
fire_hazard_hls_scene_candidates['MONTH'] = (
    fire_hazard_hls_scene_candidates['DATETIME_PARSED']
    .dt.tz_convert(None)
    .dt.to_period('M')
    .astype('string')
)
# Store scene candidates['cloud cover'] needed to carry out best HLS fire-season scenes.
fire_hazard_hls_scene_candidates['CLOUD_COVER'] = \
    pd.to_numeric(fire_hazard_hls_scene_candidates['CLOUD_COVER'],
    errors='coerce')
# Store scene candidates['cloud cover rank'] needed to carry out best HLS fire-season scenes.
fire_hazard_hls_scene_candidates['CLOUD_COVER_RANK'] = \
    fire_hazard_hls_scene_candidates['CLOUD_COVER'].fillna(100.0)
# Store scene candidates needed to carry out best HLS fire-season scenes.
fire_hazard_hls_scene_candidates = fire_hazard_hls_scene_candidates.sort_values(['MGRS_TILE',
    'MONTH', 'CLOUD_COVER_RANK', 'DATETIME_PARSED',
    'COLLECTION_ID', 'ITEM_ID'], ascending=[True, True,
    True, True, True, True]).reset_index(drop=True)
# Store scene candidates['tile month rank'] needed to carry out best HLS fire-season scenes.
fire_hazard_hls_scene_candidates['TILE_MONTH_RANK'] = \
    fire_hazard_hls_scene_candidates.groupby(['MGRS_TILE',
    'MONTH']).cumcount() + 1
# Store selected items needed to carry out best HLS fire-season scenes.
fire_hazard_hls_selected_items = \
    fire_hazard_hls_scene_candidates[fire_hazard_hls_scene_candidates['TILE_MONTH_RANK'] <= \
    fire_hazard_hls_max_scenes_per_tile_month].copy().reset_index(drop=True)

# Stop execution if hLS scene-selection policy retained no observations.
if fire_hazard_hls_selected_items.empty:
    raise ValueError('The HLS scene-selection policy retained no observations.')
# Collect selected HLS item keys for membership and completeness checks during best HLS fire-season
# scenes.
selected_hls_item_keys = set(zip(fire_hazard_hls_selected_items['COLLECTION_ID'],
    fire_hazard_hls_selected_items['ITEM_ID']))
# Build the selected hls asset mask mask used to isolate records required for this analysis.
selected_hls_asset_mask = [(collection_id, item_id) in selected_hls_item_keys for collection_id,
    item_id in zip(fire_hazard_hls_asset_manifest['COLLECTION_ID'],
    fire_hazard_hls_asset_manifest['ITEM_ID'])]
# Create an isolated complete asset manifest working copy so best HLS fire-season scenes does not
# modify upstream data.
fire_hazard_hls_complete_asset_manifest = fire_hazard_hls_asset_manifest.copy()
# Store asset manifest needed to carry out best HLS fire-season scenes.
fire_hazard_hls_asset_manifest = \
    fire_hazard_hls_asset_manifest[selected_hls_asset_mask].copy().sort_values(['COLLECTION_ID',
    'DATETIME', 'ITEM_ID', 'ASSET_ROLE']).reset_index(drop=True)
# Store expected assets per scene needed to carry out best HLS fire-season scenes.
expected_assets_per_scene = fire_hazard_hls_asset_manifest.groupby(['COLLECTION_ID',
    'ITEM_ID']).size()
# Record invalid selected scene assets so best HLS fire-season scenes can preserve a clear
# processing and validation outcome.
invalid_selected_scene_assets = expected_assets_per_scene[expected_assets_per_scene != 4]

# Stop execution if one or more selected HLS scenes do not have exactly four required asset records.
if not invalid_selected_scene_assets.empty:
    raise ValueError('One or more selected HLS '
        'scenes do not have exactly '
        'four required asset records.')
# Store scene selection summary needed to carry out best HLS fire-season scenes.
fire_hazard_hls_scene_selection_summary = pd.DataFrame([{'ORIGINAL_SCENES': \
    len(fire_hazard_hls_item_manifest),
    'SELECTED_SCENES': len(fire_hazard_hls_selected_items),
    'ORIGINAL_ASSETS': len(fire_hazard_hls_complete_asset_manifest),
    'SELECTED_ASSETS': len(fire_hazard_hls_asset_manifest),
    'UNIQUE_TILES': fire_hazard_hls_selected_items['MGRS_TILE'].nunique(),
    'MONTHS': fire_hazard_hls_selected_items['MONTH'].nunique(),
    'MAX_SCENES_PER_TILE_MONTH': fire_hazard_hls_max_scenes_per_tile_month,
    'SCENE_REDUCTION_PERCENT': 100.0 * (1 - len(fire_hazard_hls_selected_items) / \
        len(fire_hazard_hls_item_manifest))}])
print(f'-> Original HLS scenes: {len(fire_hazard_hls_item_manifest):,}')
print(f'-> Selected HLS scenes: {len(fire_hazard_hls_selected_items):,}')
print(f'-> Original HLS assets: {len(fire_hazard_hls_complete_asset_manifest):,}')
print(f'-> Selected HLS assets: {len(fire_hazard_hls_asset_manifest):,}')
print(f'-> Maximum scenes per tile-month: {fire_hazard_hls_max_scenes_per_tile_month}')
print('\n--- HLS SCENE SELECTION SUMMARY ---')
display(fire_hazard_hls_scene_selection_summary)
print('\n--- SELECTED HLS SCENES ---')
display(fire_hazard_hls_selected_items[['COLLECTION_ID',
    'ITEM_ID', 'MGRS_TILE', 'MONTH', 'CLOUD_COVER',
    'TILE_MONTH_RANK']])
print('\n=== BEST HLS FIRE-SEASON SCENES SELECTED ===')



=== SELECTING BEST HLS FIRE-SEASON SCENES ===
-> Original HLS scenes: 313
-> Selected HLS scenes: 65
-> Original HLS assets: 1,252
-> Selected HLS assets: 260
-> Maximum scenes per tile-month: 2

--- HLS SCENE SELECTION SUMMARY ---


,ORIGINAL_SCENES,SELECTED_SCENES,ORIGINAL_ASSETS,SELECTED_ASSETS,UNIQUE_TILES,MONTHS,MAX_SCENES_PER_TILE_MONTH,SCENE_REDUCTION_PERCENT
0,313,65,1252,260,11,3,2,79.233227



--- SELECTED HLS SCENES ---


,COLLECTION_ID,ITEM_ID,MGRS_TILE,MONTH,CLOUD_COVER,TILE_MONTH_RANK
0,hls2-s30,HLS.S30.T11SQA.2025186T180941.v2.0,T11SQA,2025-07,0.0,1
1,hls2-s30,HLS.S30.T11SQA.2025188T181801.v2.0,T11SQA,2025-07,0.0,2
2,hls2-s30,HLS.S30.T11SQA.2025221T180919.v2.0,T11SQA,2025-08,0.0,1
3,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,T11SQA,2025-08,0.0,2
4,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,T11SQA,2025-09,0.0,1
...,...,...,...,...,...,...
60,hls2-l30,HLS.L30.T12TWK.2025194T180146.v2.0,T12TWK,2025-07,0.0,2
61,hls2-s30,HLS.S30.T12TWK.2025231T180919.v2.0,T12TWK,2025-08,2.0,1
62,hls2-l30,HLS.L30.T12TWK.2025242T180209.v2.0,T12TWK,2025-08,19.0,2
63,hls2-s30,HLS.S30.T12TWK.2025261T180919.v2.0,T12TWK,2025-09,2.0,1



=== BEST HLS FIRE-SEASON SCENES SELECTED ===


### Validating Planetary Computer HLS Acquisition


In [67]:
print('=== VALIDATING PLANETARY COMPUTER HLS ACQUISITION ===')
# List the required HLS validation prerequisites required before planetary computer HLS
# acquisition can run.
required_hls_validation_inputs = ['fire_hazard_hls_item_manifest',
    'fire_hazard_hls_asset_manifest', 'fire_hazard_hls_collections',
    'fire_hazard_hls_stac_cache_paths', 'fire_hazard_hls_raw_directory',
    'fire_hazard_hls_analysis_period', 'fire_hazard_hls_max_cloud_cover',
    'fire_hazard_planetary_computer_stac_url', 'fire_hazard_source_catalog',
    'fire_hazard_data_mode']
# Identify unavailable HLS validation so planetary computer HLS acquisition stops before
# using incomplete inputs.
missing_hls_validation_inputs = [object_name for object_name in required_hls_validation_inputs if \
    object_name not in globals()]

# Stop execution when required hls validation inputs inputs are unavailable.
if missing_hls_validation_inputs:
    raise NameError(f'The following Planetary '
        f'Computer HLS validation '
        f'objects are missing:\n'
        f'{missing_hls_validation_inputs}\n\n'
        f'Run step Search and Cache HLS '
        f'STAC Items before validating '
        f'the acquisition.')
# List the required HLS item manifest fields prerequisites required before planetary computer HLS
# acquisition can run.
required_hls_item_manifest_fields = ['COLLECTION_ID',
    'ITEM_ID', 'DATETIME', 'CLOUD_COVER', 'STAC_CACHE_PATH']
# List the required HLS asset manifest fields prerequisites required before planetary computer HLS
# acquisition can run.
required_hls_asset_manifest_fields = ['COLLECTION_ID',
    'ITEM_ID', 'ASSET_ROLE', 'ASSET_KEY', 'UNSIGNED_HREF']
# Identify unavailable HLS item manifest fields so planetary computer HLS acquisition stops
# before using incomplete inputs.
missing_hls_item_manifest_fields = [field_name for field_name in \
    required_hls_item_manifest_fields if field_name not in fire_hazard_hls_item_manifest.columns]
# Identify unavailable HLS asset manifest fields so planetary computer HLS acquisition stops
# before using incomplete inputs.
missing_hls_asset_manifest_fields = [field_name for field_name in \
    required_hls_asset_manifest_fields if field_name not in fire_hazard_hls_asset_manifest.columns]

# Stop execution when required hls item manifest fields inputs are unavailable.
if missing_hls_item_manifest_fields:
    raise ValueError(f'The HLS item manifest is '
        f'missing fields:\n'
        f'{missing_hls_item_manifest_fields}')

# Stop execution when required hls asset manifest fields inputs are unavailable.
if missing_hls_asset_manifest_fields:
    raise ValueError(f'The HLS asset manifest is '
        f'missing fields:\n'
        f'{missing_hls_asset_manifest_fields}')

# Stop execution if hLS item manifest contains no records.
if fire_hazard_hls_item_manifest.empty:
    raise ValueError('The HLS item manifest contains no records.')

# Stop execution if hLS asset manifest contains no records.
if fire_hazard_hls_asset_manifest.empty:
    raise ValueError('The HLS asset manifest contains no records.')
# Identify unavailable HLS collection results so planetary computer HLS acquisition stops before
# using incomplete inputs.
missing_hls_collection_results = [collection_id for collection_id in fire_hazard_hls_collections \
    if collection_id not in set(fire_hazard_hls_item_manifest['COLLECTION_ID'])]

# Stop execution when required hls collection results inputs are unavailable.
if missing_hls_collection_results:
    raise ValueError(f'No HLS items were retained for '
        f'these collections:\n'
        f'{missing_hls_collection_results}')
# Define expected hls asset roles used to configure this workflow stage.
expected_hls_asset_roles = {'red', 'nir', 'swir1', 'quality'}
# Record available HLS asset roles needed to identify the current HLS asset in manifests and
# requests.
available_hls_asset_roles = set(fire_hazard_hls_asset_manifest['ASSET_ROLE'])
# Identify unavailable HLS asset roles so planetary computer HLS acquisition stops before using
# incomplete inputs.
missing_hls_asset_roles = expected_hls_asset_roles - available_hls_asset_roles

# Stop execution when required hls asset roles inputs are unavailable.
if missing_hls_asset_roles:
    raise ValueError(f'The HLS asset manifest is '
        f'missing required roles: '
        f'{sorted(missing_hls_asset_roles)}')
# Identify unavailable HLS STAC cache files so planetary computer HLS acquisition stops before
# using incomplete inputs.
missing_hls_stac_cache_files = [str(cache_path) for cache_path in \
    fire_hazard_hls_stac_cache_paths.values() if not cache_path.exists()]

# Stop execution when required hls stac cache files inputs are unavailable.
if missing_hls_stac_cache_files:
    raise FileNotFoundError(f'One or more HLS STAC cache '
        f'files are missing:\n'
        f'{missing_hls_stac_cache_files}')
# Build the hls catalog mask mask used to isolate records required for this analysis.
hls_catalog_mask = fire_hazard_source_catalog['SOURCE_ID'] == 'NASA_HLS_VI'

# Require The source catalog to contain exactly one NASA_HLS_VI record.
if hls_catalog_mask.sum() != 1:
    raise ValueError('The source catalog must contain exactly one NASA_HLS_VI record.')
# Build the source catalog.loc[HLS catalog mask, 'source name'] mask to isolate pixels that satisfy
# the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'SOURCE_NAME'] = 'NASA Harmonized Landsat Sentinel-2 Version 2.0'
# Build the source catalog.loc[HLS catalog mask, 'dataset name'] mask to isolate pixels that satisfy
# the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'DATASET_NAME'] = 'HLS L30 and S30 Surface Reflectance'
# Build the source catalog.loc[HLS catalog mask, 'source variable'] mask to isolate pixels that
# satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask, 'SOURCE_VARIABLE'] = (
    'Red, near-infrared, '
        'shortwave-infrared 1, and '
        'Fmask assets'
)
# Build the source catalog.loc[HLS catalog mask, 'access method'] mask to isolate pixels that
# satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'ACCESS_METHOD'] = 'Microsoft Planetary Computer STAC API'
# Build the source catalog.loc[HLS catalog mask, 'access URL'] mask to isolate pixels that satisfy
# the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'ACCESS_URL'] = fire_hazard_planetary_computer_stac_url
# Build the source catalog.loc[HLS catalog mask, 'local filename'] mask to isolate pixels that
# satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'LOCAL_FILENAME'] = fire_hazard_hls_raw_directory.name
# Define loc used to configure this workflow stage.
fire_hazard_source_catalog.loc[hls_catalog_mask, 'LOCAL_PATH'] = [fire_hazard_hls_raw_directory]
# Build the source catalog.loc[HLS catalog mask, 'requires authentication'] mask to isolate pixels
# that satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask, 'REQUIRES_AUTHENTICATION'] = False
# Build the source catalog.loc[HLS catalog mask, 'license or use notes'] mask to isolate pixels that
# satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask, 'LICENSE_OR_USE_NOTES'] = (
    'Public STAC discovery does not '
        'require an interactive login. '
        'Asset URLs are signed '
        'temporarily at runtime.'
)
# Build the source catalog.loc[HLS catalog mask, 'cache exists'] mask to isolate pixels that satisfy
# the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask, 'CACHE_EXISTS'] = True
# Store cache size bytes needed to carry out planetary computer HLS acquisition.
fire_hazard_hls_cache_size_bytes = sum((path.stat().st_size for path in \
    fire_hazard_hls_raw_directory.rglob('*') if path.is_file()))
# Build the source catalog.loc[HLS catalog mask, 'cache size bytes'] mask to isolate pixels that
# satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'CACHE_SIZE_BYTES'] = fire_hazard_hls_cache_size_bytes
# Store acquisition summary needed to carry out planetary computer HLS acquisition.
fire_hazard_hls_acquisition_summary = pd.DataFrame([{'SOURCE_ID': 'NASA_HLS_VI',
    'ACCESS_PLATFORM': 'Microsoft Planetary Computer',
    'COLLECTIONS': ', '.join(fire_hazard_hls_collections),
    'ANALYSIS_PERIOD': fire_hazard_hls_analysis_period,
    'DATA_MODE': fire_hazard_data_mode, 'ITEMS_DISCOVERED': len(fire_hazard_hls_item_manifest),
    'ASSETS_INVENTORIED': len(fire_hazard_hls_asset_manifest),
    'MAX_CLOUD_COVER': fire_hazard_hls_max_cloud_cover,
    'LOCAL_CACHE_SIZE_MB': fire_hazard_hls_cache_size_bytes / 1024 ** 2,
    'VALID': True}])
print(f'-> HLS collections available: {len(fire_hazard_hls_collections)}')
print(f'-> HLS observations available: {len(fire_hazard_hls_item_manifest):,}')
print(f'-> HLS source assets available: {len(fire_hazard_hls_asset_manifest):,}')
print(f'-> Cached metadata size: {fire_hazard_hls_cache_size_bytes / 1024 ** 2:,.2f} MB')
print('-> Interactive authentication required: False')
print('\n--- PLANETARY COMPUTER HLS ACQUISITION SUMMARY ---')
display(fire_hazard_hls_acquisition_summary)
print('\nNOTE:')
print('The acquisition cache contains '
    'stable unsigned STAC metadata '
    'rather than temporary signed '
    'URLs.')

# Transform vegetation condition into a normalized dryness score
# where larger values represent greater fire potential.
print('Required HLS assets will be '
    'signed at runtime and read by '
    'spatial window during '
    'vegetation-dryness processing.')

# Use NDVI to represent live vegetation density and potential fuel availability.
print('NDVI and NDMI will be '
    'calculated from red, '
    'near-infrared, and '
    'shortwave-infrared assets '
    'after Fmask quality screening.')
print('\n=== PLANETARY COMPUTER HLS ACQUISITION VALIDATED ===')



=== VALIDATING PLANETARY COMPUTER HLS ACQUISITION ===
-> HLS collections available: 2
-> HLS observations available: 313
-> HLS source assets available: 260
-> Cached metadata size: 14,661.90 MB
-> Interactive authentication required: False

--- PLANETARY COMPUTER HLS ACQUISITION SUMMARY ---


,SOURCE_ID,ACCESS_PLATFORM,COLLECTIONS,ANALYSIS_PERIOD,DATA_MODE,ITEMS_DISCOVERED,ASSETS_INVENTORIED,MAX_CLOUD_COVER,LOCAL_CACHE_SIZE_MB,VALID
0,NASA_HLS_VI,Microsoft Planetary Computer,"hls2-l30, hls2-s30",July-September 2025,snapshot,313,260,20.0,14661.904335,True



NOTE:
The acquisition cache contains stable unsigned STAC metadata rather than temporary signed URLs.
Required HLS assets will be signed at runtime and read by spatial window during vegetation-dryness processing.
NDVI and NDMI will be calculated from red, near-infrared, and shortwave-infrared assets after Fmask quality screening.

=== PLANETARY COMPUTER HLS ACQUISITION VALIDATED ===


### Configuring Required HLS Asset Downloads


In [68]:
print('=== CONFIGURING REQUIRED HLS ASSET DOWNLOADS ===')
# List the required HLS asset download prerequisites required before required HLS asset
# downloads can run.
required_hls_asset_download_inputs = ['fire_hazard_hls_asset_manifest',
    'fire_hazard_hls_raw_directory', 'fire_hazard_request_session',
    'fire_hazard_request_timeout', 'fire_hazard_verify_ssl_certificates',
    'fire_hazard_data_mode']
# Identify unavailable HLS asset download so required HLS asset downloads stops before
# using incomplete inputs.
missing_hls_asset_download_inputs = [object_name for object_name in \
    required_hls_asset_download_inputs if object_name not in globals()]

# Stop execution when required hls asset download inputs inputs are unavailable.
if missing_hls_asset_download_inputs:
    raise NameError(f'The following HLS '
        f'asset-download objects are '
        f'missing:\n'
        f'{missing_hls_asset_download_inputs}\n\n'
        f'Run step Search and Cache HLS '
        f'STAC Items before configuring '
        f'HLS asset downloads.')
# List the required HLS asset manifest fields prerequisites required before required HLS asset
# downloads can run.
required_hls_asset_manifest_fields = ['COLLECTION_ID',
    'ITEM_ID', 'DATETIME', 'ASSET_ROLE', 'ASSET_KEY',
    'UNSIGNED_HREF']
# Identify unavailable HLS asset manifest fields so required HLS asset downloads stops before
# using incomplete inputs.
missing_hls_asset_manifest_fields = [field_name for field_name in \
    required_hls_asset_manifest_fields if field_name not in fire_hazard_hls_asset_manifest.columns]

# Stop execution when required hls asset manifest fields inputs are unavailable.
if missing_hls_asset_manifest_fields:
    raise ValueError(f'The HLS asset manifest is '
        f'missing required fields:\n'
        f'{missing_hls_asset_manifest_fields}')

# Stop execution if hLS asset manifest contains no records.
if fire_hazard_hls_asset_manifest.empty:
    raise ValueError('The HLS asset manifest contains no records.')
# Record invalid HLS asset URL records so required HLS asset downloads can preserve a clear
# processing and validation outcome.
invalid_hls_asset_url_records = \
    fire_hazard_hls_asset_manifest[fire_hazard_hls_asset_manifest['UNSIGNED_HREF'].isna() | \
    (fire_hazard_hls_asset_manifest['UNSIGNED_HREF'].astype(str).str.strip() == '')]

# Stop execution if one or more HLS asset-manifest records do not contain a valid unsigned source
# URL.
if not invalid_hls_asset_url_records.empty:
    raise ValueError('One or more HLS asset-manifest '
        'records do not contain a valid '
        'unsigned source URL.')
# Build duplicate hls asset records used to track the records included in this processing stage.
duplicate_hls_asset_records = fire_hazard_hls_asset_manifest.duplicated(subset=['COLLECTION_ID',
    'ITEM_ID', 'ASSET_KEY'], keep=False)

# Stop execution if hLS asset manifest contains the reported value duplicate collection-item-asset
# records.
if duplicate_hls_asset_records.any():
    # Identify duplicate duplicate record count so repeated records do not bias required HLS asset
    # downloads.
    duplicate_record_count = int(duplicate_hls_asset_records.sum())
    raise ValueError(f'The HLS asset manifest '
        f'contains '
        f'{duplicate_record_count} '
        f'duplicate '
        f'collection-item-asset records.')
# Define fire hazard planetary computer sign url used to access the configured source service.
fire_hazard_planetary_computer_sign_url = 'https://planetarycomputer.microsoft.com/api/sas/v1/sign'
# Build the asset download directory location so required HLS asset downloads uses the expected
# project file structure.
fire_hazard_hls_asset_download_directory = fire_hazard_hls_raw_directory / 'downloaded_assets'
# Build the download manifest directory location so required HLS asset downloads uses the expected
# project file structure.
fire_hazard_hls_download_manifest_directory = fire_hazard_hls_raw_directory / 'download_manifests'

# Process each directory path entry so required HLS asset downloads is applied consistently across
# all records.
for directory_path in [fire_hazard_hls_asset_download_directory,
    fire_hazard_hls_download_manifest_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the download manifest path location so required HLS asset downloads uses the expected
# project file structure.
fire_hazard_hls_download_manifest_path = fire_hazard_hls_download_manifest_directory / \
    'hls_asset_download_manifest.csv'
# Build the download validation path location so required HLS asset downloads uses the expected
# project file structure.
fire_hazard_hls_download_validation_path = fire_hazard_hls_download_manifest_directory / \
    'hls_asset_download_validation.csv'
# Build the download summary path location so required HLS asset downloads uses the expected project
# file structure.
fire_hazard_hls_download_summary_path = fire_hazard_hls_download_manifest_directory / \
    'hls_asset_download_summary.csv'
# Extract download chunk size bytes from the current record for required HLS asset downloads.
fire_hazard_hls_download_chunk_size_bytes = globals().get('fire_hazard_download_chunk_size_bytes',
    1024 * 1024)
# Extract temporary suffix from the current record for required HLS asset downloads.
fire_hazard_hls_temporary_suffix = globals().get('fire_hazard_temporary_download_suffix', '.part')
# Track minimum file size bytes across processed pixels for output-range QA.
fire_hazard_hls_minimum_file_size_bytes = 1024
# Define expected content types to control the inputs and rules used by required HLS asset
# downloads.
fire_hazard_hls_expected_content_types = {'image/tiff',
    'image/geotiff', 'application/geotiff', 'application/octet-stream',
    'binary/octet-stream'}
# Set stop on download error to control required HLS asset downloads.
fire_hazard_hls_stop_on_download_error = True

# Create deterministic local filenames for each HLS asset so
# cached downloads can be reused safely.
# Encapsulate build HLS asset local path so repeated required HLS asset downloads steps use
# consistent logic.
def build_hls_asset_local_path(collection_id, item_id, asset_key):

    """
    Creates the stable local path assigned to one HLS
    raster asset.

    Parameters
    ----------
    collection_id : str
        Planetary Computer HLS collection identifier.

    item_id : str
        STAC item identifier.

    asset_key : str
        STAC raster-asset key.

    Returns
    -------
    pathlib.Path
        Local GeoTIFF destination path.
    """
    # Sanitize safe collection ID so it is safe for filenames, identifiers, and manifest records.
    safe_collection_id = re.sub('[^A-Za-z0-9._-]+', '_', str(collection_id))
    # Sanitize safe item ID so it is safe for filenames, identifiers, and manifest records.
    safe_item_id = re.sub('[^A-Za-z0-9._-]+', '_', str(item_id))
    # Sanitize safe asset key so it is safe for filenames, identifiers, and manifest records.
    safe_asset_key = re.sub('[^A-Za-z0-9._-]+', '_', str(asset_key))
    # Build the item directory location so required HLS asset downloads uses the expected project
    # file structure.
    item_directory = fire_hazard_hls_asset_download_directory / safe_collection_id / safe_item_id
    # Create the output directory before writing workflow products.
    item_directory.mkdir(parents=True, exist_ok=True)
    return item_directory / f'{safe_asset_key}.tif'
# Store download plan needed to carry out required HLS asset downloads.
fire_hazard_hls_download_plan = fire_hazard_hls_asset_manifest.copy().reset_index(drop=True)
# Build the download plan['local path'] location so required HLS asset downloads uses the expected
# project file structure.
fire_hazard_hls_download_plan['LOCAL_PATH'] = \
    [str(build_hls_asset_local_path(collection_id=row['COLLECTION_ID'],
    item_id=row['ITEM_ID'], asset_key=row['ASSET_KEY'])) for _,
    row in fire_hazard_hls_download_plan.iterrows()]
# Store download plan['cache exists'] needed to carry out required HLS asset downloads.
fire_hazard_hls_download_plan['CACHE_EXISTS'] = [Path(local_path).exists() and \
    Path(local_path).is_file() and (Path(local_path).stat().st_size >= \
    fire_hazard_hls_minimum_file_size_bytes) for local_path in \
    fire_hazard_hls_download_plan['LOCAL_PATH']]
# Calculate existing HLS asset count to quantify completeness and support QA checks.
existing_hls_asset_count = int(fire_hazard_hls_download_plan['CACHE_EXISTS'].sum())
# Identify unavailable HLS asset count so required HLS asset downloads stops before using
# incomplete inputs.
missing_hls_asset_count = len(fire_hazard_hls_download_plan) - existing_hls_asset_count
print(f'-> HLS assets required: {len(fire_hazard_hls_download_plan):,}')
print(f'-> Valid cached assets available: {existing_hls_asset_count:,}')
print(f'-> Assets requiring download: {missing_hls_asset_count:,}')
print(f'-> Download mode: {fire_hazard_data_mode}')
print(f'-> Asset download directory: {fire_hazard_hls_asset_download_directory}')
print('\n--- HLS ASSET DOWNLOAD PLAN ---')
display(fire_hazard_hls_download_plan.head(5))
print('\n=== HLS ASSET DOWNLOAD CONFIGURATION COMPLETE ===')



=== CONFIGURING REQUIRED HLS ASSET DOWNLOADS ===
-> HLS assets required: 260
-> Valid cached assets available: 260
-> Assets requiring download: 0
-> Download mode: snapshot
-> Asset download directory: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\downloaded_assets

--- HLS ASSET DOWNLOAD PLAN ---


,COLLECTION_ID,ITEM_ID,DATETIME,ASSET_ROLE,ASSET_KEY,UNSIGNED_HREF,MEDIA_TYPE,LOCAL_PATH,CACHE_EXISTS
0,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,2025-07-12T18:07:35.908119Z,nir,B05,https://hls2euwest.blob.core.windows.net/hls2/...,image/tiff; application=geotiff; profile=cloud...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
1,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,2025-07-12T18:07:35.908119Z,quality,Fmask,https://hls2euwest.blob.core.windows.net/hls2/...,image/tiff; application=geotiff; profile=cloud...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
2,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,2025-07-12T18:07:35.908119Z,red,B04,https://hls2euwest.blob.core.windows.net/hls2/...,image/tiff; application=geotiff; profile=cloud...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
3,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,2025-07-12T18:07:35.908119Z,swir1,B06,https://hls2euwest.blob.core.windows.net/hls2/...,image/tiff; application=geotiff; profile=cloud...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
4,hls2-l30,HLS.L30.T12TVL.2025193T180735.v2.0,2025-07-12T18:07:35.908119Z,nir,B05,https://hls2euwest.blob.core.windows.net/hls2/...,image/tiff; application=geotiff; profile=cloud...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



=== HLS ASSET DOWNLOAD CONFIGURATION COMPLETE ===


### Defining HLS Asset Signing and Download Functions


In [69]:
print('=== DEFINING HLS ASSET SIGNING AND DOWNLOAD FUNCTIONS ===')
# List the required HLS download function prerequisites required before HLS asset signing and
# download functions can run.
required_hls_download_function_inputs = ['fire_hazard_planetary_computer_sign_url',
    'fire_hazard_request_session', 'fire_hazard_request_timeout',
    'fire_hazard_verify_ssl_certificates', 'fire_hazard_hls_download_chunk_size_bytes',
    'fire_hazard_hls_temporary_suffix', 'fire_hazard_hls_minimum_file_size_bytes',
    'fire_hazard_hls_expected_content_types']
# Identify unavailable HLS download function so HLS asset signing and download functions
# stops before using incomplete inputs.
missing_hls_download_function_inputs = [object_name for object_name in \
    required_hls_download_function_inputs if object_name not in globals()]

# Stop execution when required hls download function inputs inputs are unavailable.
if missing_hls_download_function_inputs:
    raise NameError(f'The following HLS '
        f'download-function objects are '
        f'missing:\n'
        f'{missing_hls_download_function_inputs}\n\n'
        f'Run step Configure Required '
        f'HLS Asset Downloads first.')

# Encapsulate sign planetary computer asset URL so repeated HLS asset signing and download functions
# steps use consistent logic.
def sign_planetary_computer_asset_url(unsigned_href):

    """
    Requests a temporary signed Planetary Computer
    asset URL through the public SAS REST API.

    Parameters
    ----------
    unsigned_href : str
        Stable unsigned Azure Blob asset URL obtained
        from Planetary Computer STAC metadata.

    Returns
    -------
    dict
        Signed URL and expiration metadata returned by
        the Planetary Computer SAS service.
    """

    # Require unsigned_href to contain a non-empty Planetary Computer asset URL.
    if not isinstance(unsigned_href, str) or not unsigned_href.strip():
        raise ValueError('unsigned_href must contain a non-empty Planetary Computer asset URL.')
    # Store stable unsigned href so the current HLS asset can be signed or downloaded reliably.
    stable_unsigned_href = unsigned_href.split('?', 1)[0]
    # Capture the signing response response so service status and returned content can be validated.
    signing_response = fire_hazard_request_session.get(fire_hazard_planetary_computer_sign_url,
        params={'href': stable_unsigned_href}, timeout=fire_hazard_request_timeout,
        verify=fire_hazard_verify_ssl_certificates)
    signing_response.raise_for_status()

    # Protect HLS asset signing and download functions so expected source or file failures do not
    # leave partial outputs.
    try:
        # Parse signing metadata from returned content so HLS asset signing and download functions
        # can use structured metadata.
        signing_metadata = signing_response.json()
    finally:
        # Close the resource after processing is complete.
        signing_response.close()

    # Stop execution if planetary Computer signing endpoint returned an unexpected response.
    if not isinstance(signing_metadata, dict):
        raise TypeError('The Planetary Computer signing endpoint returned an unexpected response.')
    # Extract signed href from the current record for HLS asset signing and download functions.
    signed_href = signing_metadata.get('href')

    # Stop execution if planetary Computer signing endpoint did not return a usable signed asset
    # URL.
    if not isinstance(signed_href, str) or not signed_href.strip():
        raise ValueError('The Planetary Computer signing '
            'endpoint did not return a '
            'usable signed asset URL.')
    return {'UNSIGNED_HREF': stable_unsigned_href,
        'SIGNED_HREF': signed_href, 'SIGNATURE_EXPIRY': signing_metadata.get('msft:expiry')}

# Sign and download each required HLS asset while recording
# acquisition status in the source inventory.
# Encapsulate download HLS asset so repeated HLS asset signing and download functions steps use
# consistent logic.
def download_hls_asset(signed_href, destination_path, minimum_size_bytes=1024):

    """
    Downloads one signed HLS raster asset to a local
    GeoTIFF using a temporary partial file.

    Parameters
    ----------
    signed_href : str
        Temporary Planetary Computer signed URL.

    destination_path : pathlib.Path
        Final local GeoTIFF destination.

    minimum_size_bytes : int, optional
        Minimum accepted downloaded file size.

    Returns
    -------
    dict
        HTTP, file-size, content-type, and validation
        metadata for the completed download.
    """
    # Build the destination path location so HLS asset signing and download functions uses the
    # expected project file structure.
    destination_path = Path(destination_path)
    # Create the output directory before writing workflow products.
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    # Build the temporary path location so HLS asset signing and download functions uses the
    # expected project file structure.
    temporary_path = destination_path.parent / (destination_path.name + \
        fire_hazard_hls_temporary_suffix)

    # Use the existing file only when it is present and valid for HLS asset signing and download
    # functions.
    if temporary_path.exists():
        temporary_path.unlink()
    # Capture the download response response so service status and returned content can be
    # validated.
    download_response = fire_hazard_request_session.get(signed_href,
        stream=True, allow_redirects=True, timeout=fire_hazard_request_timeout,
        verify=fire_hazard_verify_ssl_certificates)
    download_response.raise_for_status()
    # Store content type needed to carry out HLS asset signing and download functions.
    content_type = download_response.headers.get('Content-Type',
        '').split(';', 1)[0].strip().lower()
    # Extract content length header from the current record for HLS asset signing and download
    # functions.
    content_length_header = download_response.headers.get('Content-Length')
    # Store expected size bytes needed to carry out HLS asset signing and download functions.
    expected_size_bytes = int(content_length_header) if content_length_header and \
        str(content_length_header).isdigit() else None
    # Set bytes written to control HLS asset signing and download functions.
    bytes_written = 0

    # Protect HLS asset signing and download functions so expected source or file failures do not
    # leave partial outputs.
    try:

        # Open the file in a managed context so its handle closes reliably after HLS asset signing
        # and download functions.
        with temporary_path.open('wb') as destination_file:

            # Process each file chunk entry so HLS asset signing and download functions is applied
            # consistently across all records.
            for file_chunk in \
                download_response.iter_content(chunk_size= \
                fire_hazard_hls_download_chunk_size_bytes):

                # Write only nonempty streamed chunks while assembling the temporary download.
                if not file_chunk:
                    continue
                # Write the processed data to the configured output resource.
                destination_file.write(file_chunk)
                # Store bytes written needed to carry out HLS asset signing and download functions.
                bytes_written += len(file_chunk)
    finally:
        # Close the resource after processing is complete.
        download_response.close()

    # Stop execution if downloaded HLS asset is smaller than the minimum accepted size of the
    # reported value bytes.
    if bytes_written < minimum_size_bytes:

        # Use the existing file only when it is present and valid for HLS asset signing and download
        # functions.
        if temporary_path.exists():
            temporary_path.unlink()
        raise IOError(f'The downloaded HLS asset is '
            f'smaller than the minimum '
            f'accepted size of '
            f'{minimum_size_bytes:,} bytes.')
    # Store size matches header needed to carry out HLS asset signing and download functions.
    size_matches_header = expected_size_bytes is None or bytes_written == expected_size_bytes

    # Stop execution if downloaded HLS asset size does not match the server Content-Length value.
    if not size_matches_header:

        # Use the existing file only when it is present and valid for HLS asset signing and download
        # functions.
        if temporary_path.exists():
            temporary_path.unlink()
        raise IOError('The downloaded HLS asset size '
            'does not match the server '
            'Content-Length value.')
    temporary_path.replace(destination_path)
    # Record final file valid so HLS asset signing and download functions can preserve a clear
    # processing and validation outcome.
    final_file_valid = destination_path.exists() and destination_path.is_file() and \
        (destination_path.stat().st_size >= minimum_size_bytes)

    # Stop execution if final HLS asset failed local file validation after download.
    if not final_file_valid:
        raise IOError('The final HLS asset failed local file validation after download.')
    return {'HTTP_STATUS': download_response.status_code,
        'CONTENT_TYPE': content_type, 'EXPECTED_SIZE_BYTES': expected_size_bytes,
        'DOWNLOADED_SIZE_BYTES': destination_path.stat().st_size,
        'SIZE_MATCHES_HEADER': size_matches_header, 'VALID': final_file_valid}
print('-> Planetary Computer URL-signing function created.')
print('-> Streaming HLS asset-download function created.')
print('\n=== HLS ASSET SIGNING AND DOWNLOAD FUNCTIONS READY ===')


=== DEFINING HLS ASSET SIGNING AND DOWNLOAD FUNCTIONS ===
-> Planetary Computer URL-signing function created.
-> Streaming HLS asset-download function created.

=== HLS ASSET SIGNING AND DOWNLOAD FUNCTIONS READY ===


### Downloading Required HLS Raster Assets


In [70]:
print('=== DOWNLOADING REQUIRED HLS RASTER ASSETS ===')

# List the required HLS download execution prerequisites required before required HLS raster
# assets can run.
required_hls_download_execution_inputs = [
    'fire_hazard_hls_download_plan',
    'fire_hazard_data_mode',
    'fire_hazard_hls_minimum_file_size_bytes',
    'fire_hazard_hls_stop_on_download_error',
    'sign_planetary_computer_asset_url',
    'download_hls_asset'
]

# Identify unavailable HLS download execution so required HLS raster assets stops before
# using incomplete inputs.
missing_hls_download_execution_inputs = [
    object_name
    for object_name in required_hls_download_execution_inputs
    if object_name not in globals()
]

# Stop execution when required HLS download execution inputs are unavailable.
if missing_hls_download_execution_inputs:
    raise NameError(
        f'The following HLS '
        f'download-execution objects are '
        f'missing:\n'
        f'{missing_hls_download_execution_inputs}\n\n'
        f'Run the HLS '
        f'download-configuration and '
        f'download-function cells first.'
    )

# Identify unavailable cached HLS assets so required HLS raster assets stops before using
# incomplete inputs.
missing_cached_hls_assets = \
    fire_hazard_hls_download_plan[~fire_hazard_hls_download_plan['CACHE_EXISTS']]

# Stop execution if cached HLS asset acquisition was selected but required local files
# are unavailable.
if fire_hazard_data_mode == 'snapshot' and (not missing_cached_hls_assets.empty):
    raise FileNotFoundError(
        f"Cached HLS asset acquisition "
        f"was selected, but the "
        f"following number of required "
        f"local files are missing or "
        f"invalid:\n"
        f"{len(missing_cached_hls_assets):,}\n\n"
        f"Change fire_hazard_data_mode "
        f"to 'refresh', rerun the "
        f"cache-policy and HLS download "
        f"cells, then return the mode to "
        f"'snapshot'."
    )

# Define download records to control the inputs and rules used by required HLS raster assets.
fire_hazard_hls_download_records = []

# Store the total number of HLS assets for progress reporting.
total_hls_assets = len(fire_hazard_hls_download_plan)

# Process each HLS asset in the download plan.
for asset_number, (manifest_index, asset_record) in \
        enumerate(fire_hazard_hls_download_plan.iterrows(), start=1):

    # Record collection ID needed to identify the current HLS asset.
    collection_id = asset_record['COLLECTION_ID']

    # Record item ID needed to identify the current HLS asset.
    item_id = asset_record['ITEM_ID']

    # Record asset role needed to identify the current HLS asset.
    asset_role = asset_record['ASSET_ROLE']

    # Record asset key needed to identify the current HLS asset.
    asset_key = asset_record['ASSET_KEY']

    # Store unsigned HREF so the current HLS asset can be signed or downloaded.
    unsigned_href = asset_record['UNSIGNED_HREF']

    # Build the local destination path for the current HLS asset.
    destination_path = Path(asset_record['LOCAL_PATH'])

    # Determine whether a valid cached file already exists.
    valid_cached_file = (
        destination_path.exists()
        and destination_path.is_file()
        and destination_path.stat().st_size
        >= fire_hazard_hls_minimum_file_size_bytes
    )

    # Report progress every 25 HLS rasters and after the final raster.
    report_hls_progress = (
        asset_number % 25 == 0
        or asset_number == total_hls_assets
    )

    # Protect HLS raster acquisition so failures do not leave the workflow inconsistent.
    try:

        # Reuse a validated cached product when available.
        if fire_hazard_data_mode == 'snapshot' and valid_cached_file:

            acquisition_action = 'Reused cached asset'
            signature_expiry = None
            http_status = None
            content_type = None
            expected_size_bytes = None
            downloaded_size_bytes = destination_path.stat().st_size
            size_matches_header = None
            download_valid = True

        else:

            # Sign the Planetary Computer asset URL.
            signing_record = sign_planetary_computer_asset_url(unsigned_href)

            signature_expiry = signing_record['SIGNATURE_EXPIRY']

            # Remove an existing file before refreshing the asset.
            if destination_path.exists():
                destination_path.unlink()

            # Download and validate the HLS asset.
            download_record = download_hls_asset(
                signed_href=signing_record['SIGNED_HREF'],
                destination_path=destination_path,
                minimum_size_bytes=fire_hazard_hls_minimum_file_size_bytes
            )

            acquisition_action = 'Downloaded refreshed asset'
            http_status = download_record['HTTP_STATUS']
            content_type = download_record['CONTENT_TYPE']
            expected_size_bytes = download_record['EXPECTED_SIZE_BYTES']
            downloaded_size_bytes = download_record['DOWNLOADED_SIZE_BYTES']
            size_matches_header = download_record['SIZE_MATCHES_HEADER']
            download_valid = download_record['VALID']

        # Record the completed HLS acquisition result in the download manifest.
        fire_hazard_hls_download_records.append({
            'COLLECTION_ID': collection_id,
            'ITEM_ID': item_id,
            'DATETIME': asset_record['DATETIME'],
            'ASSET_ROLE': asset_role,
            'ASSET_KEY': asset_key,
            'UNSIGNED_HREF': unsigned_href,
            'SIGNATURE_EXPIRY': signature_expiry,
            'LOCAL_PATH': str(destination_path),
            'ACQUISITION_ACTION': acquisition_action,
            'HTTP_STATUS': http_status,
            'CONTENT_TYPE': content_type,
            'EXPECTED_SIZE_BYTES': expected_size_bytes,
            'DOWNLOADED_SIZE_BYTES': downloaded_size_bytes,
            'FILE_SIZE_MB': downloaded_size_bytes / 1024 ** 2,
            'SIZE_MATCHES_HEADER': size_matches_header,
            'ERROR_MESSAGE': None,
            'VALID': download_valid
        })

    # Handle download failures without leaving the workflow in an inconsistent state.
    except Exception as error:

        fire_hazard_hls_download_records.append({
            'COLLECTION_ID': collection_id,
            'ITEM_ID': item_id,
            'DATETIME': asset_record['DATETIME'],
            'ASSET_ROLE': asset_role,
            'ASSET_KEY': asset_key,
            'UNSIGNED_HREF': unsigned_href,
            'SIGNATURE_EXPIRY': None,
            'LOCAL_PATH': str(destination_path),
            'ACQUISITION_ACTION': 'Download failed',
            'HTTP_STATUS': None,
            'CONTENT_TYPE': None,
            'EXPECTED_SIZE_BYTES': None,
            'DOWNLOADED_SIZE_BYTES':
                destination_path.stat().st_size
                if destination_path.exists()
                else 0,
            'FILE_SIZE_MB':
                destination_path.stat().st_size / 1024 ** 2
                if destination_path.exists()
                else 0.0,
            'SIZE_MATCHES_HEADER': False,
            'ERROR_MESSAGE': str(error),
            'VALID': False
        })

        # Always report a failed HLS raster immediately.
        print(
            f'WARNING: HLS raster {asset_number:,} of '
            f'{total_hls_assets:,} failed to download.'
        )
        print(f'-> Error: {error}')

        # Stop execution when configured to stop after a download failure.
        if fire_hazard_hls_stop_on_download_error:

            fire_hazard_hls_download_manifest = pd.DataFrame(
                fire_hazard_hls_download_records
            )

            fire_hazard_hls_download_manifest.to_csv(
                fire_hazard_hls_download_manifest_path,
                index=False
            )

            raise

    # Print a clean progress update every 25 rasters and after the final raster.
    if report_hls_progress:
        print(
            f'HLS rasters processed: '
            f'{asset_number:,} of {total_hls_assets:,}'
        )

# Store the completed HLS download manifest.
fire_hazard_hls_download_manifest = pd.DataFrame(
    fire_hazard_hls_download_records
)

# Sort the download manifest for consistent QA and reuse.
fire_hazard_hls_download_manifest = (
    fire_hazard_hls_download_manifest
    .sort_values([
        'COLLECTION_ID',
        'DATETIME',
        'ITEM_ID',
        'ASSET_ROLE'
    ])
    .reset_index(drop=True)
)

# Save the download manifest CSV.
fire_hazard_hls_download_manifest.to_csv(
    fire_hazard_hls_download_manifest_path,
    index=False
)

# Calculate successful HLS asset count.
successful_hls_download_count = int(
    fire_hazard_hls_download_manifest['VALID'].sum()
)

# Calculate failed HLS asset count.
failed_hls_download_count = (
    len(fire_hazard_hls_download_manifest)
    - successful_hls_download_count
)

# Report the final HLS acquisition summary.
print(f'\nRequired HLS assets processed: {len(fire_hazard_hls_download_manifest):,}')
print(f'Successful or cached assets: {successful_hls_download_count:,}')
print(f'Failed assets: {failed_hls_download_count:,}')
print(f'Download manifest saved: {fire_hazard_hls_download_manifest_path}')

print('\n=== REQUIRED HLS ASSET DOWNLOADS COMPLETE ===')


=== DOWNLOADING REQUIRED HLS RASTER ASSETS ===
HLS rasters processed: 25 of 260
HLS rasters processed: 50 of 260
HLS rasters processed: 75 of 260
HLS rasters processed: 100 of 260
HLS rasters processed: 125 of 260
HLS rasters processed: 150 of 260
HLS rasters processed: 175 of 260
HLS rasters processed: 200 of 260
HLS rasters processed: 225 of 260
HLS rasters processed: 250 of 260
HLS rasters processed: 260 of 260

Required HLS assets processed: 260
Successful or cached assets: 260
Failed assets: 0
Download manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\download_manifests\hls_asset_download_manifest.csv

=== REQUIRED HLS ASSET DOWNLOADS COMPLETE ===


### Validating Downloaded HLS Raster Assets


In [71]:
print('=== VALIDATING DOWNLOADED HLS RASTER ASSETS ===')

# Handle the globals case explicitly during downloaded HLS raster assets.
if 'fire_hazard_hls_download_manifest' not in globals():

    # Use the existing file only when it is present and valid for downloaded HLS raster assets.
    if fire_hazard_hls_download_manifest_path.exists():
        # Store download manifest needed to carry out downloaded HLS raster assets.
        fire_hazard_hls_download_manifest = pd.read_csv(fire_hazard_hls_download_manifest_path)
    else:
        raise NameError('The HLS download manifest is '
            'unavailable. Run step Download '
            'Required HLS Assets before '
            'validating the local cache.')
# Record download validation records so downloaded HLS raster assets can preserve a clear processing
# and validation outcome.
fire_hazard_hls_download_validation_records = []

# Process each ( entry so downloaded HLS raster assets is applied consistently across all records.
for _, asset_record in fire_hazard_hls_download_manifest.iterrows():
    # Build the local path location so downloaded HLS raster assets uses the expected project file
    # structure.
    local_path = Path(asset_record['LOCAL_PATH'])
    # Store file exists needed to carry out downloaded HLS raster assets.
    file_exists = local_path.exists() and local_path.is_file()
    # Store file size bytes needed to carry out downloaded HLS raster assets.
    file_size_bytes = local_path.stat().st_size if file_exists else 0
    # Record file extension valid so downloaded HLS raster assets can preserve a clear processing
    # and validation outcome.
    file_extension_valid = local_path.suffix.lower() in {'.tif', '.tiff'}
    # Track minimum size valid across processed pixels for output-range QA.
    minimum_size_valid = file_size_bytes >= fire_hazard_hls_minimum_file_size_bytes
    # Record local file valid so downloaded HLS raster assets can preserve a clear processing and
    # validation outcome.
    local_file_valid = file_exists and file_extension_valid and minimum_size_valid
    fire_hazard_hls_download_validation_records.append({'COLLECTION_ID': \
        asset_record['COLLECTION_ID'],
        'ITEM_ID': asset_record['ITEM_ID'], 'ASSET_ROLE': asset_record['ASSET_ROLE'],
        'ASSET_KEY': asset_record['ASSET_KEY'], 'LOCAL_PATH': str(local_path),
        'FILE_EXISTS': file_exists, 'FILE_EXTENSION_VALID': file_extension_valid,
        'FILE_SIZE_BYTES': file_size_bytes, 'FILE_SIZE_MB': file_size_bytes / 1024 ** 2,
        'MINIMUM_SIZE_VALID': minimum_size_valid, 'VALID': local_file_valid})
# Record download validation so downloaded HLS raster assets can preserve a clear processing and
# validation outcome.
fire_hazard_hls_download_validation = pd.DataFrame(fire_hazard_hls_download_validation_records)
# Record failed HLS download validations so downloaded HLS raster assets can preserve a clear
# processing and validation outcome.
failed_hls_download_validations = \
    fire_hazard_hls_download_validation[~fire_hazard_hls_download_validation['VALID']]
# Calculate expected HLS asset count to quantify completeness and support QA checks.
expected_hls_asset_count = len(fire_hazard_hls_download_plan)
# Calculate validated HLS asset count to quantify completeness and support QA checks.
validated_hls_asset_count = len(fire_hazard_hls_download_validation)
# Calculate valid HLS asset count to quantify completeness and support QA checks.
valid_hls_asset_count = int(fire_hazard_hls_download_validation['VALID'].sum())
# Store downloads complete needed to carry out downloaded HLS raster assets.
fire_hazard_hls_downloads_complete = failed_hls_download_validations.empty and \
    validated_hls_asset_count == expected_hls_asset_count and (valid_hls_asset_count == \
    expected_hls_asset_count)

# Stop execution if one or more required HLS raster assets failed local download validation.
if not fire_hazard_hls_downloads_complete:
    raise ValueError('One or more required HLS raster assets failed local download validation.')
# Save the download validation CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_download_validation.to_csv(fire_hazard_hls_download_validation_path, index=False)
# Store total HLS download size bytes needed to carry out downloaded HLS raster assets.
total_hls_download_size_bytes = int(fire_hazard_hls_download_validation['FILE_SIZE_BYTES'].sum())
# Store download summary needed to carry out downloaded HLS raster assets.
fire_hazard_hls_download_summary = pd.DataFrame([{'DATA_SOURCE': \
    'NASA Harmonized Landsat Sentinel-2 Version 2.0',
    'ACCESS_PLATFORM': 'Microsoft Planetary Computer',
    'COLLECTIONS': ', '.join(fire_hazard_hls_collections),
    'ANALYSIS_PERIOD': fire_hazard_hls_analysis_period,
    'EXPECTED_ASSETS': expected_hls_asset_count, 'VALIDATED_ASSETS': validated_hls_asset_count,
    'VALID_ASSETS': valid_hls_asset_count, 'TOTAL_SIZE_BYTES': total_hls_download_size_bytes,
    'TOTAL_SIZE_GB': total_hls_download_size_bytes / 1024 ** 3,
    'DOWNLOAD_COMPLETE': fire_hazard_hls_downloads_complete}])
# Save the download summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_download_summary.to_csv(fire_hazard_hls_download_summary_path, index=False)
# Build the hls catalog mask mask used to isolate records required for this analysis.
hls_catalog_mask = fire_hazard_source_catalog['SOURCE_ID'] == 'NASA_HLS_VI'

# Require The fire-hazard source catalog to contain exactly one NASA_HLS_VI record.
if hls_catalog_mask.sum() != 1:
    raise ValueError('The fire-hazard source catalog must contain exactly one NASA_HLS_VI record.')
# Build the source catalog.loc[HLS catalog mask, 'local filename'] mask to isolate pixels that
# satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'LOCAL_FILENAME'] = fire_hazard_hls_asset_download_directory.name
# Define loc used to configure this workflow stage.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'LOCAL_PATH'] = [fire_hazard_hls_asset_download_directory]
# Build the source catalog.loc[HLS catalog mask, 'cache exists'] mask to isolate pixels that satisfy
# the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask,
    'CACHE_EXISTS'] = fire_hazard_hls_downloads_complete
# Build the source catalog.loc[HLS catalog mask, 'cache size bytes'] mask to isolate pixels that
# satisfy the current analysis criteria.
fire_hazard_source_catalog.loc[hls_catalog_mask, 'CACHE_SIZE_BYTES'] = total_hls_download_size_bytes
print(f'-> Expected HLS assets: {expected_hls_asset_count:,}')
print(f'-> Validated HLS assets: {validated_hls_asset_count:,}')
print(f'-> Valid local HLS assets: {valid_hls_asset_count:,}')
print(f'-> Total HLS cache size: {total_hls_download_size_bytes / 1024 ** 3:,.2f} GB')
print(f'-> HLS download cache complete: {fire_hazard_hls_downloads_complete}')
print('\n--- HLS DOWNLOAD SUMMARY ---')
display(fire_hazard_hls_download_summary)
print('\n--- HLS LOCAL FILE VALIDATION ---')
display(fire_hazard_hls_download_validation.head(5))
print('\nNOTE:')
print('The downloaded files are raw '
    'HLS source assets. They have '
    'not yet been scaled, quality '
    'masked, mosaicked, temporally '
    'composited, reprojected, or '
    'aligned to the fire-hazard '
    'reference grid.')
print('Those transformations will occur during the vegetation-dryness component-processing phase.')
print('\n=== DOWNLOADED HLS ASSET VALIDATION COMPLETE ===')



=== VALIDATING DOWNLOADED HLS RASTER ASSETS ===
-> Expected HLS assets: 260
-> Validated HLS assets: 260
-> Valid local HLS assets: 260
-> Total HLS cache size: 3.57 GB
-> HLS download cache complete: True

--- HLS DOWNLOAD SUMMARY ---


,DATA_SOURCE,ACCESS_PLATFORM,COLLECTIONS,ANALYSIS_PERIOD,EXPECTED_ASSETS,VALIDATED_ASSETS,VALID_ASSETS,TOTAL_SIZE_BYTES,TOTAL_SIZE_GB,DOWNLOAD_COMPLETE
0,NASA Harmonized Landsat Sentinel-2 Version 2.0,Microsoft Planetary Computer,"hls2-l30, hls2-s30",July-September 2025,260,260,260,3835336398,3.571935,True



--- HLS LOCAL FILE VALIDATION ---


,COLLECTION_ID,ITEM_ID,ASSET_ROLE,ASSET_KEY,LOCAL_PATH,FILE_EXISTS,FILE_EXTENSION_VALID,FILE_SIZE_BYTES,FILE_SIZE_MB,MINIMUM_SIZE_VALID,VALID
0,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,nir,B05,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,14236174,13.576674,True,True
1,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,quality,Fmask,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,937763,0.894320,True,True
2,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,red,B04,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,14353819,13.688869,True,True
3,hls2-l30,HLS.L30.T12TUL.2025193T180735.v2.0,swir1,B06,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,13823281,13.182908,True,True
4,hls2-l30,HLS.L30.T12TVL.2025193T180735.v2.0,nir,B05,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,27722171,26.437922,True,True



NOTE:
The downloaded files are raw HLS source assets. They have not yet been scaled, quality masked, mosaicked, temporally composited, reprojected, or aligned to the fire-hazard reference grid.
Those transformations will occur during the vegetation-dryness component-processing phase.

=== DOWNLOADED HLS ASSET VALIDATION COMPLETE ===


### Inspecting HLS Raster Metadata


In [72]:
print('=== INSPECTING HLS RASTER METADATA ===')
# List the required HLS metadata prerequisites required before HLS raster metadata can run.
required_hls_metadata_inputs = ['fire_hazard_hls_download_manifest',
    'fire_hazard_hls_download_manifest_directory',
    'fire_hazard_hls_minimum_file_size_bytes']
# Identify unavailable HLS metadata so HLS raster metadata stops before using incomplete
# inputs.
missing_hls_metadata_inputs = [object_name for object_name in required_hls_metadata_inputs if \
    object_name not in globals()]

# Stop execution when required hls metadata inputs inputs are unavailable.
if missing_hls_metadata_inputs:
    raise NameError(f'The following HLS '
        f'raster-metadata objects are '
        f'missing:\n'
        f'{missing_hls_metadata_inputs}\n\n'
        f'Run step Download Required HLS '
        f'Assets and step Validate '
        f'Downloaded HLS Assets before '
        f'inspecting raster metadata.')

# Protect HLS raster metadata so expected source or file failures do not leave partial outputs.
try:
    import rasterio
# Handle the expected failure without leaving the workflow in an inconsistent state.
except ImportError as error:
    raise ImportError('Rasterio is required to '
        'inspect the downloaded HLS '
        'GeoTIFF files.') from error
from rasterio.windows import Window
# List the required HLS metadata manifest fields prerequisites required before HLS raster metadata
# can run.
required_hls_metadata_manifest_fields = ['COLLECTION_ID',
    'ITEM_ID', 'ASSET_ROLE', 'ASSET_KEY', 'LOCAL_PATH']
# Identify unavailable HLS metadata manifest fields so HLS raster metadata stops before using
# incomplete inputs.
missing_hls_metadata_manifest_fields = [field_name for field_name in \
    required_hls_metadata_manifest_fields if field_name not in \
    fire_hazard_hls_download_manifest.columns]

# Stop execution when required hls metadata manifest fields inputs are unavailable.
if missing_hls_metadata_manifest_fields:
    raise ValueError(f'The HLS download manifest is '
        f'missing required '
        f'metadata-inspection fields:\n'
        f'{missing_hls_metadata_manifest_fields}')

# Stop execution if hLS download manifest contains no raster assets.
if fire_hazard_hls_download_manifest.empty:
    raise ValueError('The HLS download manifest contains no raster assets.')
# Build the raster metadata path location so HLS raster metadata uses the expected project file
# structure.
fire_hazard_hls_raster_metadata_path = fire_hazard_hls_download_manifest_directory / \
    'hls_raster_metadata_inventory.csv'
# Build the raster metadata failures path location so HLS raster metadata uses the expected project
# file structure.
fire_hazard_hls_raster_metadata_failures_path = fire_hazard_hls_download_manifest_directory / \
    'hls_raster_metadata_failures.csv'
# Build the raster metadata summary path location so HLS raster metadata uses the expected project
# file structure.
fire_hazard_hls_raster_metadata_summary_path = fire_hazard_hls_download_manifest_directory / \
    'hls_raster_metadata_summary.csv'
# Prepare metadata sample size to preserve and validate raster structure during HLS raster metadata.
fire_hazard_hls_metadata_sample_size = 32
# Prepare metadata progress interval to preserve and validate raster structure during HLS raster
# metadata.
fire_hazard_hls_metadata_progress_interval = 25
# Set expected driver to control HLS raster metadata.
fire_hazard_hls_expected_driver = 'GTiff'
# Calculate expected band count to quantify completeness and support QA checks.
fire_hazard_hls_expected_band_count = 1
# Prepare raster metadata records to preserve and validate raster structure during HLS raster
# metadata.
fire_hazard_hls_raster_metadata_records = []
# Prepare total HLS metadata assets to preserve and validate raster structure during HLS raster
# metadata.
total_hls_metadata_assets = len(fire_hazard_hls_download_manifest)

# Process each (asset number entry so HLS raster metadata is applied consistently across all
# records.
for asset_number, (_, asset_record) in enumerate(fire_hazard_hls_download_manifest.iterrows(),
    start=1):

    # Report progress at the configured interval without changing raster-processing results.
    if asset_number == 1 or asset_number % fire_hazard_hls_metadata_progress_interval == 0 or \
        asset_number == total_hls_metadata_assets:
        print(f'-> Inspecting HLS raster {asset_number:,} of {total_hls_metadata_assets:,}...')
    # Build the local path location so HLS raster metadata uses the expected project file structure.
    local_path = Path(asset_record['LOCAL_PATH'])
    # Record raster opened so HLS raster metadata can preserve a clear processing and validation
    # outcome.
    raster_opened = False
    # Record sample readable so HLS raster metadata can preserve a clear processing and validation
    # outcome.
    sample_readable = False
    # Set driver to control HLS raster metadata.
    driver = None
    # Store crs text so spatial operations use the required coordinate reference system.
    crs_text = None
    # Set epsg code to control HLS raster metadata.
    epsg_code = None
    # Set width to control HLS raster metadata.
    width = None
    # Set height to control HLS raster metadata.
    height = None
    # Calculate band count to quantify completeness and support QA checks.
    band_count = None
    # Set raster dtype to control HLS raster metadata.
    raster_dtype = None
    # Set NoData value to control HLS raster metadata.
    nodata_value = None
    # Set pixel width to control HLS raster metadata.
    pixel_width = None
    # Set pixel height to control HLS raster metadata.
    pixel_height = None
    # Set transform text to control HLS raster metadata.
    transform_text = None
    # Set bounds left to control HLS raster metadata.
    bounds_left = None
    # Set bounds bottom to control HLS raster metadata.
    bounds_bottom = None
    # Set bounds right to control HLS raster metadata.
    bounds_right = None
    # Set bounds top to control HLS raster metadata.
    bounds_top = None
    # Set scale factor to control HLS raster metadata.
    scale_factor = None
    # Set offset value to control HLS raster metadata.
    offset_value = None
    # Set block width to control HLS raster metadata.
    block_width = None
    # Set block height to control HLS raster metadata.
    block_height = None
    # Set tiled to control HLS raster metadata.
    tiled = None
    # Set compression to control HLS raster metadata.
    compression = None
    # Set interleave to control HLS raster metadata.
    interleave = None
    # Set band description to control HLS raster metadata.
    band_description = None
    # Set band units to control HLS raster metadata.
    band_units = None
    # Set raster tags JSON to control HLS raster metadata.
    raster_tags_json = None
    # Set band tags JSON to control HLS raster metadata.
    band_tags_json = None
    # Track sample minimum across processed pixels for output-range QA.
    sample_minimum = None
    # Track sample maximum across processed pixels for output-range QA.
    sample_maximum = None
    # Calculate sample valid pixel count to quantify completeness and support QA checks.
    sample_valid_pixel_count = None
    # Prepare metadata error to preserve and validate raster structure during HLS raster metadata.
    metadata_error = None
    # Store file exists needed to carry out HLS raster metadata.
    file_exists = local_path.exists() and local_path.is_file()
    # Store file size bytes needed to carry out HLS raster metadata.
    file_size_bytes = local_path.stat().st_size if file_exists else 0
    # Track minimum size valid across processed pixels for output-range QA.
    minimum_size_valid = file_size_bytes >= fire_hazard_hls_minimum_file_size_bytes

    # Protect HLS raster metadata so expected source or file failures do not leave partial outputs.
    try:

        # Stop execution if hLS raster does not exist: the reported value.
        if not file_exists:
            raise FileNotFoundError(f'HLS raster does not exist: {local_path}')

        # Stop execution if hLS raster is smaller than the minimum accepted file size.
        if not minimum_size_valid:
            raise ValueError('HLS raster is smaller than the minimum accepted file size.')

        # Open the raster in a managed context so its file handle closes reliably after HLS raster
        # metadata.
        with rasterio.open(local_path) as raster_source:
            # Record raster opened so HLS raster metadata can preserve a clear processing and
            # validation outcome.
            raster_opened = True
            # Store driver needed to carry out HLS raster metadata.
            driver = raster_source.driver
            # Store width needed to carry out HLS raster metadata.
            width = raster_source.width
            # Store height needed to carry out HLS raster metadata.
            height = raster_source.height
            # Calculate band count to quantify completeness and support QA checks.
            band_count = raster_source.count
            # Store raster dtype needed to carry out HLS raster metadata.
            raster_dtype = raster_source.dtypes[0] if raster_source.dtypes else None
            # Store NoData value needed to carry out HLS raster metadata.
            nodata_value = raster_source.nodata
            # Store crs text so spatial operations use the required coordinate reference system.
            crs_text = raster_source.crs.to_string() if raster_source.crs is not None else None
            # Store epsg code needed to carry out HLS raster metadata.
            epsg_code = raster_source.crs.to_epsg() if raster_source.crs is not None else None
            # Store pixel width needed to carry out HLS raster metadata.
            pixel_width = abs(raster_source.transform.a)
            # Store pixel height needed to carry out HLS raster metadata.
            pixel_height = abs(raster_source.transform.e)
            # Store transform text needed to carry out HLS raster metadata.
            transform_text = str(raster_source.transform)
            # Store bounds left needed to carry out HLS raster metadata.
            bounds_left = raster_source.bounds.left
            # Store bounds bottom needed to carry out HLS raster metadata.
            bounds_bottom = raster_source.bounds.bottom
            # Store bounds right needed to carry out HLS raster metadata.
            bounds_right = raster_source.bounds.right
            # Store bounds top needed to carry out HLS raster metadata.
            bounds_top = raster_source.bounds.top
            # Store scale factor needed to carry out HLS raster metadata.
            scale_factor = raster_source.scales[0] if raster_source.scales else None
            # Store offset value needed to carry out HLS raster metadata.
            offset_value = raster_source.offsets[0] if raster_source.offsets else None

            # Handle the raster source case explicitly during HLS raster metadata.
            if raster_source.block_shapes:
                # Store block height needed to carry out HLS raster metadata.
                block_height = raster_source.block_shapes[0][0]
                # Store block width needed to carry out HLS raster metadata.
                block_width = raster_source.block_shapes[0][1]
            # Extract tiled from the current record for HLS raster metadata.
            tiled = raster_source.profile.get('tiled')
            # Extract compression from the current record for HLS raster metadata.
            compression = raster_source.profile.get('compress')
            # Extract interleave from the current record for HLS raster metadata.
            interleave = raster_source.profile.get('interleave')
            # Store band description needed to carry out HLS raster metadata.
            band_description = raster_source.descriptions[0] if raster_source.descriptions else None
            # Store band units needed to carry out HLS raster metadata.
            band_units = raster_source.units[0] if raster_source.units else None
            # Store raster tags JSON needed to carry out HLS raster metadata.
            raster_tags_json = json.dumps(raster_source.tags(), sort_keys=True)
            # Store band tags JSON needed to carry out HLS raster metadata.
            band_tags_json = json.dumps(raster_source.tags(1), sort_keys=True)
            # Store sample width needed to carry out HLS raster metadata.
            sample_width = min(fire_hazard_hls_metadata_sample_size, raster_source.width)
            # Store sample height needed to carry out HLS raster metadata.
            sample_height = min(fire_hazard_hls_metadata_sample_size, raster_source.height)
            # Store sample window needed to carry out HLS raster metadata.
            sample_window = Window(col_off=0, row_off=0, width=sample_width, height=sample_height)
            # Prepare sample array used to process the current raster window.
            sample_array = raster_source.read(1, window=sample_window, masked=True)
            # Record sample readable so HLS raster metadata can preserve a clear processing and
            # validation outcome.
            sample_readable = sample_array.size > 0
            # Store compressed sample needed to carry out HLS raster metadata.
            compressed_sample = sample_array.compressed()
            # Calculate sample valid pixel count to quantify completeness and support QA checks.
            sample_valid_pixel_count = int(compressed_sample.size)

            # Handle the compressed sample case explicitly during HLS raster metadata.
            if compressed_sample.size > 0:
                # Track sample minimum across processed pixels for output-range QA.
                sample_minimum = float(compressed_sample.min())
                # Track sample maximum across processed pixels for output-range QA.
                sample_maximum = float(compressed_sample.max())
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception as error:
        # Prepare metadata error to preserve and validate raster structure during HLS raster
        # metadata.
        metadata_error = str(error)
    # Record driver valid so HLS raster metadata can preserve a clear processing and validation
    # outcome.
    driver_valid = driver == fire_hazard_hls_expected_driver
    # Record dimensions valid so HLS raster metadata can preserve a clear processing and validation
    # outcome.
    dimensions_valid = isinstance(width, int) and isinstance(height,
        int) and (width > 0) and (height > 0)
    # Calculate band count valid to quantify completeness and support QA checks.
    band_count_valid = band_count == fire_hazard_hls_expected_band_count
    # Store crs valid so spatial operations use the required coordinate reference system.
    crs_valid = crs_text is not None
    # Record resolution valid so HLS raster metadata can preserve a clear processing and validation
    # outcome.
    resolution_valid = pixel_width is not None and pixel_height is not None and (pixel_width > 0) \
        and (pixel_height > 0)
    # Record bounds valid so HLS raster metadata can preserve a clear processing and validation
    # outcome.
    bounds_valid = bounds_left is not None and bounds_bottom is not None and (bounds_right is not \
        None) and (bounds_top is not None) and (bounds_right > bounds_left) and (bounds_top > \
        bounds_bottom)
    # Prepare raster metadata valid to preserve and validate raster structure during HLS raster
    # metadata.
    raster_metadata_valid = all([file_exists, minimum_size_valid,
        raster_opened, sample_readable, driver_valid, dimensions_valid,
        band_count_valid, crs_valid, resolution_valid,
        bounds_valid, metadata_error is None])
    fire_hazard_hls_raster_metadata_records.append({'COLLECTION_ID': asset_record['COLLECTION_ID'],
        'ITEM_ID': asset_record['ITEM_ID'], 'ASSET_ROLE': asset_record['ASSET_ROLE'],
        'ASSET_KEY': asset_record['ASSET_KEY'], 'LOCAL_PATH': str(local_path),
        'FILE_EXISTS': file_exists, 'FILE_SIZE_BYTES': file_size_bytes,
        'FILE_SIZE_MB': file_size_bytes / 1024 ** 2, 'MINIMUM_SIZE_VALID': minimum_size_valid,
        'RASTER_OPENED': raster_opened, 'DRIVER': driver,
        'DRIVER_VALID': driver_valid, 'CRS': crs_text,
        'EPSG_CODE': epsg_code, 'CRS_VALID': crs_valid,
        'WIDTH': width, 'HEIGHT': height, 'DIMENSIONS_VALID': dimensions_valid,
        'BAND_COUNT': band_count, 'BAND_COUNT_VALID': band_count_valid,
        'DATA_TYPE': raster_dtype, 'NODATA_VALUE': nodata_value,
        'PIXEL_WIDTH': pixel_width, 'PIXEL_HEIGHT': pixel_height,
        'RESOLUTION_VALID': resolution_valid, 'TRANSFORM': transform_text,
        'BOUNDS_LEFT': bounds_left, 'BOUNDS_BOTTOM': bounds_bottom,
        'BOUNDS_RIGHT': bounds_right, 'BOUNDS_TOP': bounds_top,
        'BOUNDS_VALID': bounds_valid, 'SCALE_FACTOR': scale_factor,
        'OFFSET': offset_value, 'BLOCK_WIDTH': block_width,
        'BLOCK_HEIGHT': block_height, 'TILED': tiled, 'COMPRESSION': compression,
        'INTERLEAVE': interleave, 'BAND_DESCRIPTION': band_description,
        'BAND_UNITS': band_units, 'RASTER_TAGS': raster_tags_json,
        'BAND_TAGS': band_tags_json, 'SAMPLE_READABLE': sample_readable,
        'SAMPLE_VALID_PIXEL_COUNT': sample_valid_pixel_count,
        'SAMPLE_MINIMUM': sample_minimum, 'SAMPLE_MAXIMUM': sample_maximum,
        'METADATA_ERROR': metadata_error, 'VALID': raster_metadata_valid})
# Prepare raster metadata to preserve and validate raster structure during HLS raster metadata.
fire_hazard_hls_raster_metadata = \
    pd.DataFrame(fire_hazard_hls_raster_metadata_records).sort_values(['COLLECTION_ID',
    'ITEM_ID', 'ASSET_ROLE']).reset_index(drop=True)
# Prepare failed HLS raster metadata to preserve and validate raster structure during HLS raster
# metadata.
failed_hls_raster_metadata = \
    fire_hazard_hls_raster_metadata[~fire_hazard_hls_raster_metadata['VALID']].copy()
# Calculate valid HLS raster metadata count to quantify completeness and support QA checks.
valid_hls_raster_metadata_count = int(fire_hazard_hls_raster_metadata['VALID'].sum())
# Calculate failed HLS raster metadata count to quantify completeness and support QA checks.
failed_hls_raster_metadata_count = len(fire_hazard_hls_raster_metadata) - \
    valid_hls_raster_metadata_count
# Store unique hls crs values so spatial operations use the required coordinate reference system.
unique_hls_crs_values = \
    sorted(fire_hazard_hls_raster_metadata['CRS'].dropna().astype(str).unique().tolist())
# Store unique HLS resolutions needed to carry out HLS raster metadata.
unique_hls_resolutions = fire_hazard_hls_raster_metadata[['PIXEL_WIDTH',
    'PIXEL_HEIGHT']].drop_duplicates().sort_values(['PIXEL_WIDTH',
    'PIXEL_HEIGHT']).reset_index(drop=True)
# Prepare raster metadata summary to preserve and validate raster structure during HLS raster
# metadata.
fire_hazard_hls_raster_metadata_summary = pd.DataFrame([{'EXPECTED_RASTERS': \
    len(fire_hazard_hls_download_manifest),
    'INSPECTED_RASTERS': len(fire_hazard_hls_raster_metadata),
    'VALID_RASTERS': valid_hls_raster_metadata_count,
    'FAILED_RASTERS': failed_hls_raster_metadata_count,
    'UNIQUE_CRS_COUNT': len(unique_hls_crs_values),
    'UNIQUE_RESOLUTION_COUNT': len(unique_hls_resolutions),
    'ALL_RASTERS_VALID': failed_hls_raster_metadata_count == 0 and \
        valid_hls_raster_metadata_count == len(fire_hazard_hls_download_manifest)}])
# Save the raster metadata CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_raster_metadata.to_csv(fire_hazard_hls_raster_metadata_path, index=False)
# Save the raster metadata summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_raster_metadata_summary.to_csv(fire_hazard_hls_raster_metadata_summary_path,
    index=False)

# Handle missing or invalid failed HLS raster metadata explicitly during HLS raster metadata.
if not failed_hls_raster_metadata.empty:
    # Save the failed HLS raster metadata CSV so later steps can reuse the recorded workflow
    # results.
    failed_hls_raster_metadata.to_csv(fire_hazard_hls_raster_metadata_failures_path, index=False)
# Use the existing file only when it is present and valid for HLS raster metadata.
elif fire_hazard_hls_raster_metadata_failures_path.exists():
    fire_hazard_hls_raster_metadata_failures_path.unlink()
# Prepare raster metadata valid to preserve and validate raster structure during HLS raster
# metadata.
fire_hazard_hls_raster_metadata_valid = failed_hls_raster_metadata.empty and \
    len(fire_hazard_hls_raster_metadata) == len(fire_hazard_hls_download_manifest)

# Stop execution if one or more downloaded HLS raster assets failed metadata or readability
# validation.
if not fire_hazard_hls_raster_metadata_valid:
    raise ValueError(f'One or more downloaded HLS '
        f'raster assets failed metadata '
        f'or readability validation.\n\n'
        f'Review:\n'
        f'{fire_hazard_hls_raster_metadata_failures_path}')
print(f'-> Expected HLS rasters: {len(fire_hazard_hls_download_manifest):,}')
print(f'-> Inspected HLS rasters: {len(fire_hazard_hls_raster_metadata):,}')
print(f'-> Valid HLS rasters: {valid_hls_raster_metadata_count:,}')
print(f'-> Failed HLS rasters: {failed_hls_raster_metadata_count:,}')
print(f'-> Unique source CRS values: {len(unique_hls_crs_values):,}')
print(f'-> Unique source resolutions: {len(unique_hls_resolutions):,}')
print(f'-> Metadata inventory valid: {fire_hazard_hls_raster_metadata_valid}')
print(f'-> Metadata inventory saved: {fire_hazard_hls_raster_metadata_path}')
print('\n--- HLS RASTER METADATA SUMMARY ---')
display(fire_hazard_hls_raster_metadata_summary)
print('\n--- HLS SOURCE CRS VALUES ---')
display(pd.DataFrame({'CRS': unique_hls_crs_values}))
print('\n--- HLS SOURCE RESOLUTIONS ---')
display(unique_hls_resolutions)
print('\n--- HLS RASTER METADATA SAMPLE ---')
display(fire_hazard_hls_raster_metadata[['COLLECTION_ID',
    'ITEM_ID', 'ASSET_ROLE', 'ASSET_KEY', 'DRIVER',
    'CRS', 'WIDTH', 'HEIGHT', 'PIXEL_WIDTH', 'PIXEL_HEIGHT',
    'DATA_TYPE', 'NODATA_VALUE', 'SCALE_FACTOR', 'OFFSET',
    'VALID']].head(5))
print('\nNOTE:')
print('This inspection reads only a '
    'small sample from each raster. '
    'It does not load complete HLS '
    'tiles into memory.')
print('The recorded scale factors, '
    'offsets, NoData values, CRS '
    'values, and raster dimensions '
    'will be used during HLS '
    'reflectance scaling, Fmask '
    'processing, reprojection, and '
    'grid alignment.')
print('\n=== HLS RASTER METADATA INSPECTION COMPLETE ===')


=== INSPECTING HLS RASTER METADATA ===
-> Inspecting HLS raster 1 of 260...
-> Inspecting HLS raster 25 of 260...
-> Inspecting HLS raster 50 of 260...
-> Inspecting HLS raster 75 of 260...
-> Inspecting HLS raster 100 of 260...
-> Inspecting HLS raster 125 of 260...
-> Inspecting HLS raster 150 of 260...
-> Inspecting HLS raster 175 of 260...
-> Inspecting HLS raster 200 of 260...
-> Inspecting HLS raster 225 of 260...
-> Inspecting HLS raster 250 of 260...
-> Inspecting HLS raster 260 of 260...
-> Expected HLS rasters: 260
-> Inspected HLS rasters: 260
-> Valid HLS rasters: 260
-> Failed HLS rasters: 0
-> Unique source CRS values: 2
-> Unique source resolutions: 1
-> Metadata inventory valid: True
-> Metadata inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\download_manifests\hls_raster_metadata_inventory.csv

--- HLS RASTER METADATA SUMMARY ---


,EXPECTED_RASTERS,INSPECTED_RASTERS,VALID_RASTERS,FAILED_RASTERS,UNIQUE_CRS_COUNT,UNIQUE_RESOLUTION_COUNT,ALL_RASTERS_VALID
0,260,260,260,0,2,1,True



--- HLS SOURCE CRS VALUES ---


,CRS
0,EPSG:32611
1,EPSG:32612



--- HLS SOURCE RESOLUTIONS ---


,PIXEL_WIDTH,PIXEL_HEIGHT
0,30.0,30.0



--- HLS RASTER METADATA SAMPLE ---


,COLLECTION_ID,ITEM_ID,ASSET_ROLE,ASSET_KEY,DRIVER,CRS,WIDTH,HEIGHT,PIXEL_WIDTH,PIXEL_HEIGHT,DATA_TYPE,NODATA_VALUE,SCALE_FACTOR,OFFSET,VALID
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,nir,B05,GTiff,EPSG:32611,3660,3660,30.0,30.0,int16,-9999.0,0.0001,0.0,True
1,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,quality,Fmask,GTiff,EPSG:32611,3660,3660,30.0,30.0,uint8,255.0,1.0000,0.0,True
2,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,red,B04,GTiff,EPSG:32611,3660,3660,30.0,30.0,int16,-9999.0,0.0001,0.0,True
3,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,swir1,B06,GTiff,EPSG:32611,3660,3660,30.0,30.0,int16,-9999.0,0.0001,0.0,True
4,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,nir,B05,GTiff,EPSG:32611,3660,3660,30.0,30.0,int16,-9999.0,0.0001,0.0,True



NOTE:
This inspection reads only a small sample from each raster. It does not load complete HLS tiles into memory.
The recorded scale factors, offsets, NoData values, CRS values, and raster dimensions will be used during HLS reflectance scaling, Fmask processing, reprojection, and grid alignment.

=== HLS RASTER METADATA INSPECTION COMPLETE ===


### Quality-Mask HLS Clouds, Cloud Shadows, and Invalid Pixels


## Defining HLS Fmask Quality Policy


In [73]:
print('=== DEFINING HLS FMASK QUALITY POLICY ===')
# List the required HLS Fmask policy prerequisites required before HLS Fmask quality policy
# can run.
required_hls_fmask_policy_inputs = ['fire_hazard_hls_download_manifest',
    'fire_hazard_hls_raster_metadata', 'fire_hazard_hls_raw_directory']
# Identify unavailable HLS Fmask policy so HLS Fmask quality policy stops before using
# incomplete inputs.
missing_hls_fmask_policy_inputs = [object_name for object_name in \
    required_hls_fmask_policy_inputs if object_name not in globals()]

# Stop execution when required hls fmask policy inputs inputs are unavailable.
if missing_hls_fmask_policy_inputs:
    raise NameError(f'The following HLS Fmask policy '
        f'objects are missing:\n'
        f'{missing_hls_fmask_policy_inputs}\n\n'
        f'Run step Validate Downloaded '
        f'HLS Assets and step Inspect '
        f'HLS Raster Metadata before '
        f'defining the Fmask quality '
        f'policy.')

# Stop execution if downloaded HLS raster metadata has not passed complete validation.
if 'fire_hazard_hls_raster_metadata_valid' not in globals() or not \
    fire_hazard_hls_raster_metadata_valid:
    raise ValueError('The downloaded HLS raster metadata has not passed complete validation.')
# Build the Fmask bits mask to isolate pixels that satisfy the current analysis criteria.
fire_hazard_hls_fmask_bits = {'cirrus': 0, 'cloud': 1,
    'adjacent_cloud_shadow': 2, 'cloud_shadow': 3,
    'snow_ice': 4, 'water': 5}
# Set aerosol bit offset to control HLS Fmask quality policy.
fire_hazard_hls_aerosol_bit_offset = 6
# Build the fire hazard hls aerosol bit mask mask used to isolate records required for this
# analysis.
fire_hazard_hls_aerosol_bit_mask = 3
# Define aerosol categories to control the inputs and rules used by HLS Fmask quality policy.
fire_hazard_hls_aerosol_categories = {0: 'climatology', 1: 'low', 2: 'moderate', 3: 'high'}
# Build the invalid Fmask conditions mask to isolate pixels that satisfy the current analysis
# criteria.
fire_hazard_hls_invalid_fmask_conditions = {'cirrus': True,
    'cloud': True, 'adjacent_cloud_shadow': True, 'cloud_shadow': True,
    'snow_ice': True, 'water': True}
# Set exclude high aerosol to control HLS Fmask quality policy.
fire_hazard_hls_exclude_high_aerosol = True
# Build the fire hazard hls valid mask value mask used to isolate records required for this
# analysis.
fire_hazard_hls_valid_mask_value = 1
# Build the fire hazard hls invalid mask value mask used to isolate records required for this
# analysis.
fire_hazard_hls_invalid_mask_value = 0
# Build the fire hazard hls mask nodata value mask used to isolate records required for this
# analysis.
fire_hazard_hls_mask_nodata_value = 255
# Build the Fmask output directory location so HLS Fmask quality policy uses the expected project
# file structure.
fire_hazard_hls_fmask_output_directory = fire_hazard_hls_raw_directory / 'processed_fmask'
# Build the Fmask manifest directory location so HLS Fmask quality policy uses the expected project
# file structure.
fire_hazard_hls_fmask_manifest_directory = fire_hazard_hls_raw_directory / 'fmask_manifests'

# Process each directory path entry so HLS Fmask quality policy is applied consistently across all
# records.
for directory_path in [fire_hazard_hls_fmask_output_directory,
    fire_hazard_hls_fmask_manifest_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the Fmask manifest path location so HLS Fmask quality policy uses the expected project file
# structure.
fire_hazard_hls_fmask_manifest_path = fire_hazard_hls_fmask_manifest_directory / \
    'hls_fmask_processing_manifest.csv'
# Build the Fmask summary path location so HLS Fmask quality policy uses the expected project file
# structure.
fire_hazard_hls_fmask_summary_path = fire_hazard_hls_fmask_manifest_directory / \
    'hls_fmask_quality_summary.csv'
# Build the Fmask policy path location so HLS Fmask quality policy uses the expected project file
# structure.
fire_hazard_hls_fmask_policy_path = fire_hazard_hls_fmask_manifest_directory / \
    'hls_fmask_quality_policy.json'
# Build the Fmask policy mask to isolate pixels that satisfy the current analysis criteria.
fire_hazard_hls_fmask_policy = {'bit_positions': fire_hazard_hls_fmask_bits,
    'aerosol_bit_offset': fire_hazard_hls_aerosol_bit_offset,
    'aerosol_bit_mask': fire_hazard_hls_aerosol_bit_mask,
    'aerosol_categories': fire_hazard_hls_aerosol_categories,
    'invalid_conditions': fire_hazard_hls_invalid_fmask_conditions,
    'exclude_high_aerosol': fire_hazard_hls_exclude_high_aerosol,
    'valid_mask_value': fire_hazard_hls_valid_mask_value,
    'invalid_mask_value': fire_hazard_hls_invalid_mask_value,
    'nodata_value': fire_hazard_hls_mask_nodata_value}

# Open the file in a managed context so its handle closes reliably after HLS Fmask quality policy.
with fire_hazard_hls_fmask_policy_path.open('w', encoding='utf-8') as fmask_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_hls_fmask_policy, fmask_policy_file, indent=2)
# Build the Fmask policy summary mask to isolate pixels that satisfy the current analysis criteria.
fire_hazard_hls_fmask_policy_summary = pd.DataFrame([{'CONDITION': condition_name,
    'BIT_POSITION': fire_hazard_hls_fmask_bits[condition_name],
    'EXCLUDED': condition_excluded} for condition_name,
    condition_excluded in fire_hazard_hls_invalid_fmask_conditions.items()])
print(f'-> Fmask output directory: {fire_hazard_hls_fmask_output_directory}')
print(f'-> Exclude high aerosol: {fire_hazard_hls_exclude_high_aerosol}')
print(f'-> Valid mask value: {fire_hazard_hls_valid_mask_value}')
print(f'-> Invalid mask value: {fire_hazard_hls_invalid_mask_value}')
print(f'-> Mask NoData value: {fire_hazard_hls_mask_nodata_value}')
print(f'-> Fmask policy saved: {fire_hazard_hls_fmask_policy_path}')
print('\n--- HLS FMASK QUALITY POLICY ---')
display(fire_hazard_hls_fmask_policy_summary)
print('\nNOTE:')
print('Cloud, cirrus, adjacent cloud '
    'or shadow, cloud shadow, snow '
    'or ice, water, and '
    'high-aerosol pixels will be '
    'excluded from vegetation-index '
    'processing.')
print('\n=== HLS FMASK QUALITY POLICY DEFINED ===')



=== DEFINING HLS FMASK QUALITY POLICY ===
-> Fmask output directory: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\processed_fmask
-> Exclude high aerosol: True
-> Valid mask value: 1
-> Invalid mask value: 0
-> Mask NoData value: 255
-> Fmask policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\fmask_manifests\hls_fmask_quality_policy.json

--- HLS FMASK QUALITY POLICY ---


,CONDITION,BIT_POSITION,EXCLUDED
0,cirrus,0,True
1,cloud,1,True
2,adjacent_cloud_shadow,2,True
3,cloud_shadow,3,True
4,snow_ice,4,True
5,water,5,True



NOTE:
Cloud, cirrus, adjacent cloud or shadow, cloud shadow, snow or ice, water, and high-aerosol pixels will be excluded from vegetation-index processing.

=== HLS FMASK QUALITY POLICY DEFINED ===


### Decoding HLS Fmask Quality Rasters


In [74]:
print('=== DECODING HLS FMASK QUALITY RASTERS ===')
# List the required HLS Fmask processing prerequisites required before HLS Fmask quality
# rasters can run.
required_hls_fmask_processing_inputs = ['fire_hazard_hls_download_manifest',
    'fire_hazard_hls_fmask_bits', 'fire_hazard_hls_invalid_fmask_conditions',
    'fire_hazard_hls_aerosol_bit_offset', 'fire_hazard_hls_aerosol_bit_mask',
    'fire_hazard_hls_exclude_high_aerosol', 'fire_hazard_hls_valid_mask_value',
    'fire_hazard_hls_invalid_mask_value', 'fire_hazard_hls_mask_nodata_value',
    'fire_hazard_hls_fmask_output_directory', 'fire_hazard_hls_fmask_manifest_path',
    'fire_hazard_hls_fmask_summary_path']
# Identify unavailable HLS Fmask processing so HLS Fmask quality rasters stops before
# using incomplete inputs.
missing_hls_fmask_processing_inputs = [object_name for object_name in \
    required_hls_fmask_processing_inputs if object_name not in globals()]

# Stop execution when required hls fmask processing inputs inputs are unavailable.
if missing_hls_fmask_processing_inputs:
    raise NameError(f'The following HLS Fmask '
        f'processing objects are '
        f'missing:\n'
        f'{missing_hls_fmask_processing_inputs}\n\n'
        f'Run step Define HLS Fmask '
        f'Quality Policy before decoding '
        f'the Fmask rasters.')
# Build the Fmask assets mask to isolate pixels that satisfy the current analysis criteria.
fire_hazard_hls_fmask_assets = \
    fire_hazard_hls_download_manifest[fire_hazard_hls_download_manifest['ASSET_ROLE'].astype(str) \
    .str.lower() == 'quality'].copy().reset_index(drop=True)

# Stop execution if hLS download manifest contains no quality or Fmask asset records.
if fire_hazard_hls_fmask_assets.empty:
    raise ValueError('The HLS download manifest contains no quality or Fmask asset records.')
# Calculate Fmask counts to quantify completeness and support QA checks.
fire_hazard_hls_fmask_counts = fire_hazard_hls_fmask_assets.groupby(['COLLECTION_ID',
    'ITEM_ID']).size()
# Calculate invalid HLS Fmask counts to quantify completeness and support QA checks.
invalid_hls_fmask_counts = fire_hazard_hls_fmask_counts[fire_hazard_hls_fmask_counts != 1]

# Stop execution if one or more selected HLS scenes do not have exactly one Fmask raster.
if not invalid_hls_fmask_counts.empty:
    raise ValueError('One or more selected HLS scenes do not have exactly one Fmask raster.')

# Encapsulate build HLS valid mask path so repeated HLS Fmask quality rasters steps use consistent
# logic.
def build_hls_valid_mask_path(collection_id, item_id):

    """
    Builds the local output path for one decoded HLS
    valid-pixel mask.
    """
    # Sanitize safe collection ID so it is safe for filenames, identifiers, and manifest records.
    safe_collection_id = re.sub('[^A-Za-z0-9._-]+', '_', str(collection_id))
    # Sanitize safe item ID so it is safe for filenames, identifiers, and manifest records.
    safe_item_id = re.sub('[^A-Za-z0-9._-]+', '_', str(item_id))
    # Build the collection directory location so HLS Fmask quality rasters uses the expected project
    # file structure.
    collection_directory = fire_hazard_hls_fmask_output_directory / safe_collection_id
    # Create the output directory before writing workflow products.
    collection_directory.mkdir(parents=True, exist_ok=True)
    return collection_directory / f'{safe_item_id}_valid_mask.tif'
# Build the Fmask processing records mask to isolate pixels that satisfy the current analysis
# criteria.
fire_hazard_hls_fmask_processing_records = []
# Build the Fmask quality records mask to isolate pixels that satisfy the current analysis criteria.
fire_hazard_hls_fmask_quality_records = []
# Build the total HLS Fmask rasters mask to isolate pixels that satisfy the current analysis
# criteria.
total_hls_fmask_rasters = len(fire_hazard_hls_fmask_assets)
# Build the Fmask progress interval mask to isolate pixels that satisfy the current analysis
# criteria.
fire_hazard_hls_fmask_progress_interval = 10

# Process each (Fmask number entry so HLS Fmask quality rasters is applied consistently across all
# records.
for fmask_number, (_, fmask_record) in enumerate(fire_hazard_hls_fmask_assets.iterrows(), start=1):

    # Apply the configured Fmask exclusion rule when that quality condition is enabled.
    if fmask_number == 1 or fmask_number % fire_hazard_hls_fmask_progress_interval == 0 or \
        fmask_number == total_hls_fmask_rasters:
        print(f'-> Decoding Fmask raster {fmask_number:,} of {total_hls_fmask_rasters:,}...')
    # Record collection ID needed to identify the current HLS asset in manifests and requests.
    collection_id = fmask_record['COLLECTION_ID']
    # Record item ID needed to identify the current HLS asset in manifests and requests.
    item_id = fmask_record['ITEM_ID']
    # Build the source Fmask path location so HLS Fmask quality rasters uses the expected project
    # file structure.
    source_fmask_path = Path(fmask_record['LOCAL_PATH'])
    # Build the output valid mask path mask used to isolate records required for this analysis.
    output_valid_mask_path = build_hls_valid_mask_path(collection_id=collection_id, item_id=item_id)

    # Open the raster in a managed context so its file handle closes reliably after HLS Fmask
    # quality rasters.
    with rasterio.open(source_fmask_path) as fmask_source:
        # Build the Fmask array mask to isolate pixels that satisfy the current analysis criteria.
        fmask_array = fmask_source.read(1).astype(np.uint8)
        # Build the valid mask profile mask used to isolate records required for this analysis.
        valid_mask_profile = fmask_source.profile.copy()

        # Apply the configured Fmask exclusion rule when that quality condition is enabled.
        if fmask_source.nodata is not None:
            # Build the source nodata mask mask used to isolate records required for this analysis.
            source_nodata_mask = fmask_array == np.uint8(fmask_source.nodata)
        else:
            # Build the source nodata mask mask used to isolate records required for this analysis.
            source_nodata_mask = np.zeros(fmask_array.shape, dtype=bool)
    # Build the cirrus mask mask used to isolate records required for this analysis.
    cirrus_mask = (fmask_array >> fire_hazard_hls_fmask_bits['cirrus'] & 1).astype(bool)
    # Build the cloud mask mask used to isolate records required for this analysis.
    cloud_mask = (fmask_array >> fire_hazard_hls_fmask_bits['cloud'] & 1).astype(bool)
    # Build the adjacent cloud shadow mask mask used to isolate records required for this analysis.
    adjacent_cloud_shadow_mask = (fmask_array >> \
        fire_hazard_hls_fmask_bits['adjacent_cloud_shadow'] & 1).astype(bool)
    # Build the cloud shadow mask mask used to isolate records required for this analysis.
    cloud_shadow_mask = (fmask_array >> fire_hazard_hls_fmask_bits['cloud_shadow'] & 1).astype(bool)
    # Build the snow ice mask mask used to isolate records required for this analysis.
    snow_ice_mask = (fmask_array >> fire_hazard_hls_fmask_bits['snow_ice'] & 1).astype(bool)
    # Build the water mask mask used to isolate records required for this analysis.
    water_mask = (fmask_array >> fire_hazard_hls_fmask_bits['water'] & 1).astype(bool)
    # Store aerosol level needed to carry out HLS Fmask quality rasters.
    aerosol_level = fmask_array >> fire_hazard_hls_aerosol_bit_offset & \
        fire_hazard_hls_aerosol_bit_mask
    # Build the high aerosol mask mask used to isolate records required for this analysis.
    high_aerosol_mask = aerosol_level == 3
    # Build the invalid pixel mask mask used to isolate records required for this analysis.
    invalid_pixel_mask = source_nodata_mask.copy()

    # Apply the configured Fmask exclusion rule when that quality condition is enabled.
    if fire_hazard_hls_invalid_fmask_conditions['cirrus']:
        # Build the invalid pixel mask mask used to isolate records required for this analysis.
        invalid_pixel_mask |= cirrus_mask

    # Apply the configured Fmask exclusion rule when that quality condition is enabled.
    if fire_hazard_hls_invalid_fmask_conditions['cloud']:
        # Build the invalid pixel mask mask used to isolate records required for this analysis.
        invalid_pixel_mask |= cloud_mask

    # Apply the configured Fmask exclusion rule when that quality condition is enabled.
    if fire_hazard_hls_invalid_fmask_conditions['adjacent_cloud_shadow']:
        # Build the invalid pixel mask mask used to isolate records required for this analysis.
        invalid_pixel_mask |= adjacent_cloud_shadow_mask

    # Apply the configured Fmask exclusion rule when that quality condition is enabled.
    if fire_hazard_hls_invalid_fmask_conditions['cloud_shadow']:
        # Build the invalid pixel mask mask used to isolate records required for this analysis.
        invalid_pixel_mask |= cloud_shadow_mask

    # Apply the configured Fmask exclusion rule when that quality condition is enabled.
    if fire_hazard_hls_invalid_fmask_conditions['snow_ice']:
        # Build the invalid pixel mask mask used to isolate records required for this analysis.
        invalid_pixel_mask |= snow_ice_mask

    # Apply the configured Fmask exclusion rule when that quality condition is enabled.
    if fire_hazard_hls_invalid_fmask_conditions['water']:
        # Build the invalid pixel mask mask used to isolate records required for this analysis.
        invalid_pixel_mask |= water_mask

    # Handle the exclude high aerosol case explicitly during HLS Fmask quality rasters.
    if fire_hazard_hls_exclude_high_aerosol:
        # Build the invalid pixel mask mask used to isolate records required for this analysis.
        invalid_pixel_mask |= high_aerosol_mask
    # Build the valid pixel mask mask used to isolate records required for this analysis.
    valid_pixel_mask = np.full(fmask_array.shape, fire_hazard_hls_valid_mask_value, dtype=np.uint8)
    # Build the valid pixel mask mask used to isolate records required for this analysis.
    valid_pixel_mask[invalid_pixel_mask] = fire_hazard_hls_invalid_mask_value
    # Build the valid pixel mask mask used to isolate records required for this analysis.
    valid_pixel_mask[source_nodata_mask] = fire_hazard_hls_mask_nodata_value
    valid_mask_profile.update({'driver': 'GTiff',
        'dtype': 'uint8', 'count': 1, 'nodata': fire_hazard_hls_mask_nodata_value,
        'compress': 'deflate', 'predictor': 2})

    # Open the raster in a managed context so its file handle closes reliably after HLS Fmask
    # quality rasters.
    with rasterio.open(output_valid_mask_path, 'w', **valid_mask_profile) as valid_mask_destination:
        # Write the processed data to the configured output resource.
        valid_mask_destination.write(valid_pixel_mask, 1)
        valid_mask_destination.set_band_description(1, 'HLS vegetation-analysis valid-pixel mask')
        valid_mask_destination.update_tags(MASK_VALID_VALUE=fire_hazard_hls_valid_mask_value,
            MASK_INVALID_VALUE=fire_hazard_hls_invalid_mask_value,
            MASK_NODATA_VALUE=fire_hazard_hls_mask_nodata_value,
            HIGH_AEROSOL_EXCLUDED=fire_hazard_hls_exclude_high_aerosol)
    # Calculate total pixel count to quantify completeness and support QA checks.
    total_pixel_count = int(fmask_array.size)
    # Calculate source NoData count to quantify completeness and support QA checks.
    source_nodata_count = int(source_nodata_mask.sum())
    # Calculate valid pixel count to quantify completeness and support QA checks.
    valid_pixel_count = int((valid_pixel_mask == fire_hazard_hls_valid_mask_value).sum())
    # Calculate invalid pixel count to quantify completeness and support QA checks.
    invalid_pixel_count = int((valid_pixel_mask == fire_hazard_hls_invalid_mask_value).sum())
    # Calculate analyzable pixel count to quantify completeness and support QA checks.
    analyzable_pixel_count = total_pixel_count - source_nodata_count
    # Record valid pixel percent so HLS Fmask quality rasters can preserve a clear processing and
    # validation outcome.
    valid_pixel_percent = 100.0 * valid_pixel_count / analyzable_pixel_count if \
        analyzable_pixel_count > 0 else np.nan
    fire_hazard_hls_fmask_processing_records.append({'COLLECTION_ID': collection_id,
        'ITEM_ID': item_id, 'SOURCE_FMASK_PATH': str(source_fmask_path),
        'VALID_MASK_PATH': str(output_valid_mask_path),
        'TOTAL_PIXELS': total_pixel_count, 'SOURCE_NODATA_PIXELS': source_nodata_count,
        'VALID_PIXELS': valid_pixel_count, 'INVALID_PIXELS': invalid_pixel_count,
        'VALID_PIXEL_PERCENT': valid_pixel_percent, 'OUTPUT_EXISTS': \
            output_valid_mask_path.exists(),
        'VALID': output_valid_mask_path.exists() and valid_pixel_count > 0})
    fire_hazard_hls_fmask_quality_records.append({'COLLECTION_ID': collection_id,
        'ITEM_ID': item_id, 'CIRRUS_PIXELS': int(cirrus_mask.sum()),
        'CLOUD_PIXELS': int(cloud_mask.sum()), 'ADJACENT_CLOUD_SHADOW_PIXELS': \
            int(adjacent_cloud_shadow_mask.sum()),
        'CLOUD_SHADOW_PIXELS': int(cloud_shadow_mask.sum()),
        'SNOW_ICE_PIXELS': int(snow_ice_mask.sum()), 'WATER_PIXELS': int(water_mask.sum()),
        'HIGH_AEROSOL_PIXELS': int(high_aerosol_mask.sum()),
        'VALID_PIXELS': valid_pixel_count, 'VALID_PIXEL_PERCENT': valid_pixel_percent})
# Build the Fmask manifest mask to isolate pixels that satisfy the current analysis criteria.
fire_hazard_hls_fmask_manifest = \
    pd.DataFrame(fire_hazard_hls_fmask_processing_records).sort_values(['COLLECTION_ID',
    'ITEM_ID']).reset_index(drop=True)
# Build the Fmask quality summary mask to isolate pixels that satisfy the current analysis criteria.
fire_hazard_hls_fmask_quality_summary = \
    pd.DataFrame(fire_hazard_hls_fmask_quality_records).sort_values(['COLLECTION_ID',
    'ITEM_ID']).reset_index(drop=True)
# Build the failed HLS Fmask outputs mask to isolate pixels that satisfy the current analysis
# criteria.
failed_hls_fmask_outputs = fire_hazard_hls_fmask_manifest[~fire_hazard_hls_fmask_manifest['VALID']]
# Build the Fmask processing valid mask to isolate pixels that satisfy the current analysis
# criteria.
fire_hazard_hls_fmask_processing_valid = failed_hls_fmask_outputs.empty and \
    len(fire_hazard_hls_fmask_manifest) == len(fire_hazard_hls_fmask_assets)

# Stop execution if one or more HLS Fmask rasters failed quality mask creation.
if not fire_hazard_hls_fmask_processing_valid:
    raise ValueError('One or more HLS Fmask rasters failed quality mask creation.')
# Save the Fmask manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_fmask_manifest.to_csv(fire_hazard_hls_fmask_manifest_path, index=False)
# Save the Fmask quality summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_fmask_quality_summary.to_csv(fire_hazard_hls_fmask_summary_path, index=False)
print(f'-> Fmask rasters processed: {len(fire_hazard_hls_fmask_manifest):,}')
print(f"-> Valid masks created: {int(fire_hazard_hls_fmask_manifest['VALID'].sum()):,}")
print(f"-> Mean valid pixel "
    f"percentage: "
    f"{fire_hazard_hls_fmask_manifest['VALID_PIXEL_PERCENT'].mean():,.2f}%")
print(f'-> Fmask processing valid: {fire_hazard_hls_fmask_processing_valid}')
print(f'-> Fmask manifest saved: {fire_hazard_hls_fmask_manifest_path}')
print('\n--- HLS FMASK PROCESSING SUMMARY ---')
display(fire_hazard_hls_fmask_manifest.head(5))
print('\n--- HLS QUALITY CONDITION SUMMARY ---')
display(fire_hazard_hls_fmask_quality_summary.head(5))
print('\nNOTE:')
print("The generated masks remain in "
    "each HLS scene's native MGRS "
    "tile grid. Reprojection and "
    "alignment to the fire-hazard "
    "reference grid will occur in a "
    "later preprocessing step.")
print('\n=== HLS FMASK DECODING COMPLETE ===')



=== DECODING HLS FMASK QUALITY RASTERS ===
-> Decoding Fmask raster 1 of 65...
-> Decoding Fmask raster 10 of 65...
-> Decoding Fmask raster 20 of 65...
-> Decoding Fmask raster 30 of 65...
-> Decoding Fmask raster 40 of 65...
-> Decoding Fmask raster 50 of 65...
-> Decoding Fmask raster 60 of 65...
-> Decoding Fmask raster 65 of 65...
-> Fmask rasters processed: 65
-> Valid masks created: 65
-> Mean valid pixel percentage: 89.71%
-> Fmask processing valid: True
-> Fmask manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\fmask_manifests\hls_fmask_processing_manifest.csv

--- HLS FMASK PROCESSING SUMMARY ---


,COLLECTION_ID,ITEM_ID,SOURCE_FMASK_PATH,VALID_MASK_PATH,TOTAL_PIXELS,SOURCE_NODATA_PIXELS,VALID_PIXELS,INVALID_PIXELS,VALID_PIXEL_PERCENT,OUTPUT_EXISTS,VALID
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,13395600,869815,11874391,651394,94.799575,True,True
1,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,13395600,6112734,6284644,998222,86.293555,True,True
2,hls2-l30,HLS.L30.T11SQB.2025224T181513.v2.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,13395600,48,13092809,302743,97.739974,True,True
3,hls2-l30,HLS.L30.T11SQB.2025257T180910.v2.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,13395600,8582955,4367221,445424,90.744715,True,True
4,hls2-l30,HLS.L30.T12STF.2025249T180912.v2.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,13395600,1533989,10621298,1240313,89.543469,True,True



--- HLS QUALITY CONDITION SUMMARY ---


,COLLECTION_ID,ITEM_ID,CIRRUS_PIXELS,CLOUD_PIXELS,ADJACENT_CLOUD_SHADOW_PIXELS,CLOUD_SHADOW_PIXELS,SNOW_ICE_PIXELS,WATER_PIXELS,HIGH_AEROSOL_PIXELS,VALID_PIXELS,VALID_PIXEL_PERCENT
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,869815,872554,876325,872905,869815,1249490,1144767,11874391,94.799575
1,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,6112734,6119047,6130862,6118730,6112734,6193760,7033380,6284644,86.293555
2,hls2-l30,HLS.L30.T11SQB.2025224T181513.v2.0,48,49348,47246,20848,48,4522,193480,13092809,97.739974
3,hls2-l30,HLS.L30.T11SQB.2025257T180910.v2.0,8582955,8582955,8582955,8582955,8582955,8588810,9025864,4367221,90.744715
4,hls2-l30,HLS.L30.T12STF.2025249T180912.v2.0,1533989,1542017,1555437,1541563,1533989,1637662,2677875,10621298,89.543469



NOTE:
The generated masks remain in each HLS scene's native MGRS tile grid. Reprojection and alignment to the fire-hazard reference grid will occur in a later preprocessing step.

=== HLS FMASK DECODING COMPLETE ===


### Defining HLS Surface-Reflectance Scaling Policy


In [75]:
print('=== DEFINING HLS SURFACE-REFLECTANCE SCALING POLICY ===')
# List the required HLS reflectance policy prerequisites required before HLS
# surface-reflectance scaling policy can run.
required_hls_reflectance_policy_inputs = ['fire_hazard_hls_download_manifest',
    'fire_hazard_hls_raster_metadata', 'fire_hazard_hls_raw_directory']
# Identify unavailable HLS reflectance policy so HLS surface-reflectance scaling policy
# stops before using incomplete inputs.
missing_hls_reflectance_policy_inputs = [object_name for object_name in \
    required_hls_reflectance_policy_inputs if object_name not in globals()]

# Stop execution when required hls reflectance policy inputs inputs are unavailable.
if missing_hls_reflectance_policy_inputs:
    raise NameError(f'The following HLS '
        f'reflectance-scaling objects '
        f'are missing:\n'
        f'{missing_hls_reflectance_policy_inputs}\n\n'
        f'Run step Validate Downloaded '
        f'HLS Assets and step Inspect '
        f'HLS Raster Metadata before '
        f'defining the '
        f'reflectance-scaling policy.')

# Stop execution if downloaded HLS raster metadata has not passed complete validation.
if 'fire_hazard_hls_raster_metadata_valid' not in globals() or not \
    fire_hazard_hls_raster_metadata_valid:
    raise ValueError('The downloaded HLS raster metadata has not passed complete validation.')
# Record reflectance asset roles needed to identify the current HLS asset in manifests and requests.
fire_hazard_hls_reflectance_asset_roles = ['red', 'nir', 'swir1']
# Set reflectance scale factor to control HLS surface-reflectance scaling policy.
fire_hazard_hls_reflectance_scale_factor = 0.0001
# Set reflectance offset to control HLS surface-reflectance scaling policy.
fire_hazard_hls_reflectance_offset = 0.0
# Set reflectance saturation value to control HLS surface-reflectance scaling policy.
fire_hazard_hls_reflectance_saturation_value = 12000

# Convert integer HLS values to surface reflectance before
# calculating vegetation indices.
# Set scaled reflectance dtype to control HLS surface-reflectance scaling policy.
fire_hazard_hls_scaled_reflectance_dtype = 'float32'
# Extract scaled reflectance NoData from the current record for HLS surface-reflectance scaling
# policy.
fire_hazard_hls_scaled_reflectance_nodata = globals().get('fire_hazard_nodata_value', -9999.0)
# Set scaled reflectance compression to control HLS surface-reflectance scaling policy.
fire_hazard_hls_scaled_reflectance_compression = 'deflate'
# Build the scaled reflectance directory location so HLS surface-reflectance scaling policy uses the
# expected project file structure.
fire_hazard_hls_scaled_reflectance_directory = fire_hazard_hls_raw_directory / 'scaled_reflectance'
# Build the reflectance manifest directory location so HLS surface-reflectance scaling policy uses
# the expected project file structure.
fire_hazard_hls_reflectance_manifest_directory = fire_hazard_hls_raw_directory / \
    'reflectance_manifests'

# Process each directory path entry so HLS surface-reflectance scaling policy is applied
# consistently across all records.
for directory_path in [fire_hazard_hls_scaled_reflectance_directory,
    fire_hazard_hls_reflectance_manifest_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the scaled reflectance manifest path location so HLS surface-reflectance scaling policy uses
# the expected project file structure.
fire_hazard_hls_scaled_reflectance_manifest_path = fire_hazard_hls_reflectance_manifest_directory \
    / 'hls_scaled_reflectance_manifest.csv'
# Build the scaled reflectance summary path location so HLS surface-reflectance scaling policy uses
# the expected project file structure.
fire_hazard_hls_scaled_reflectance_summary_path = fire_hazard_hls_reflectance_manifest_directory \
    / 'hls_scaled_reflectance_summary.csv'
# Build the reflectance policy path location so HLS surface-reflectance scaling policy uses the
# expected project file structure.
fire_hazard_hls_reflectance_policy_path = fire_hazard_hls_reflectance_manifest_directory / \
    'hls_reflectance_scaling_policy.json'
# Define reflectance policy to control the inputs and rules used by HLS surface-reflectance scaling
# policy.
fire_hazard_hls_reflectance_policy = {'asset_roles': fire_hazard_hls_reflectance_asset_roles,
    'scale_factor': fire_hazard_hls_reflectance_scale_factor,
    'offset': fire_hazard_hls_reflectance_offset, 'saturation_value': \
        fire_hazard_hls_reflectance_saturation_value,
    'output_dtype': fire_hazard_hls_scaled_reflectance_dtype,
    'output_nodata': fire_hazard_hls_scaled_reflectance_nodata,
    'compression': fire_hazard_hls_scaled_reflectance_compression}

# Open the file in a managed context so its handle closes reliably after HLS surface-reflectance
# scaling policy.
with fire_hazard_hls_reflectance_policy_path.open('w', encoding='utf-8') as reflectance_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_hls_reflectance_policy, reflectance_policy_file, indent=2)
# Store reflectance policy summary needed to carry out HLS surface-reflectance scaling policy.
fire_hazard_hls_reflectance_policy_summary = pd.DataFrame([{'ASSET_ROLES': \
    ', '.join(fire_hazard_hls_reflectance_asset_roles),
    'SCALE_FACTOR': fire_hazard_hls_reflectance_scale_factor,
    'OFFSET': fire_hazard_hls_reflectance_offset, 'SATURATION_VALUE': \
        fire_hazard_hls_reflectance_saturation_value,
    'OUTPUT_DTYPE': fire_hazard_hls_scaled_reflectance_dtype,
    'OUTPUT_NODATA': fire_hazard_hls_scaled_reflectance_nodata,
    'COMPRESSION': fire_hazard_hls_scaled_reflectance_compression}])
print(f'-> Reflectance asset roles: {fire_hazard_hls_reflectance_asset_roles}')
print(f'-> HLS scale factor: {fire_hazard_hls_reflectance_scale_factor}')
print(f'-> HLS offset: {fire_hazard_hls_reflectance_offset}')
print(f'-> Saturation value: {fire_hazard_hls_reflectance_saturation_value}')
print(f'-> Output data type: {fire_hazard_hls_scaled_reflectance_dtype}')
print(f'-> Output NoData value: {fire_hazard_hls_scaled_reflectance_nodata}')
print(f'-> Scaling policy saved: {fire_hazard_hls_reflectance_policy_path}')
print('\n--- HLS REFLECTANCE SCALING POLICY ---')
display(fire_hazard_hls_reflectance_policy_summary)
print('\nNOTE:')
print('Source NoData and saturated pixels will remain NoData in the scaled reflectance outputs.')
print('The scene-level Fmask quality masks will be applied during vegetation-index calculation.')
print('\n=== HLS SURFACE-REFLECTANCE SCALING POLICY DEFINED ===')



=== DEFINING HLS SURFACE-REFLECTANCE SCALING POLICY ===
-> Reflectance asset roles: ['red', 'nir', 'swir1']
-> HLS scale factor: 0.0001
-> HLS offset: 0.0
-> Saturation value: 12000
-> Output data type: float32
-> Output NoData value: -9999.0
-> Scaling policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\reflectance_manifests\hls_reflectance_scaling_policy.json

--- HLS REFLECTANCE SCALING POLICY ---


,ASSET_ROLES,SCALE_FACTOR,OFFSET,SATURATION_VALUE,OUTPUT_DTYPE,OUTPUT_NODATA,COMPRESSION
0,"red, nir, swir1",0.0001,0.0,12000,float32,-9999.0,deflate



NOTE:
Source NoData and saturated pixels will remain NoData in the scaled reflectance outputs.
The scene-level Fmask quality masks will be applied during vegetation-index calculation.

=== HLS SURFACE-REFLECTANCE SCALING POLICY DEFINED ===


### Scaling HLS Surface-Reflectance Bands


In [76]:
print('=== SCALING HLS SURFACE-REFLECTANCE BANDS ===')
# List the required HLS reflectance processing prerequisites required before HLS
# surface-reflectance bands can run.
required_hls_reflectance_processing_inputs = ['fire_hazard_hls_download_manifest',
    'fire_hazard_hls_reflectance_asset_roles', 'fire_hazard_hls_reflectance_scale_factor',
    'fire_hazard_hls_reflectance_offset', 'fire_hazard_hls_reflectance_saturation_value',
    'fire_hazard_hls_scaled_reflectance_dtype', 'fire_hazard_hls_scaled_reflectance_nodata',
    'fire_hazard_hls_scaled_reflectance_compression',
    'fire_hazard_hls_scaled_reflectance_directory',
    'fire_hazard_hls_scaled_reflectance_manifest_path',
    'fire_hazard_hls_scaled_reflectance_summary_path']
# Identify unavailable HLS reflectance processing so HLS surface-reflectance bands stops
# before using incomplete inputs.
missing_hls_reflectance_processing_inputs = [object_name for object_name in \
    required_hls_reflectance_processing_inputs if object_name not in globals()]

# Stop execution when required hls reflectance processing inputs inputs are unavailable.
if missing_hls_reflectance_processing_inputs:
    raise NameError(f'The following HLS '
        f'reflectance-processing objects '
        f'are missing:\n'
        f'{missing_hls_reflectance_processing_inputs}\n\n'
        f'Run step Define HLS '
        f'Surface-Reflectance Scaling '
        f'Policy before scaling the '
        f'bands.')
# Store reflectance assets needed to carry out HLS surface-reflectance bands.
fire_hazard_hls_reflectance_assets = \
    fire_hazard_hls_download_manifest[fire_hazard_hls_download_manifest['ASSET_ROLE'].astype(str) \
    .str.lower().isin(fire_hazard_hls_reflectance_asset_roles)].copy().reset_index(drop=True)

# Stop execution if hLS download manifest contains no red, NIR, or SWIR1 asset records.
if fire_hazard_hls_reflectance_assets.empty:
    raise ValueError('The HLS download manifest contains no red, NIR, or SWIR1 asset records.')
# Calculate reflectance role counts to quantify completeness and support QA checks.
reflectance_role_counts = fire_hazard_hls_reflectance_assets.groupby(['COLLECTION_ID',
    'ITEM_ID'])['ASSET_ROLE'].nunique()
# Calculate invalid reflectance role counts to quantify completeness and support QA checks.
invalid_reflectance_role_counts = reflectance_role_counts[reflectance_role_counts != \
    len(fire_hazard_hls_reflectance_asset_roles)]

# Stop execution if one or more selected HLS scenes do not have all three required reflectance-band
# roles.
if not invalid_reflectance_role_counts.empty:
    raise ValueError('One or more selected HLS '
        'scenes do not have all three '
        'required reflectance-band '
        'roles.')

# Encapsulate build HLS scaled reflectance path so repeated HLS surface-reflectance bands steps use
# consistent logic.
def build_hls_scaled_reflectance_path(collection_id, item_id, asset_role):

    """
    Builds the local output path for one scaled HLS
    surface-reflectance raster.
    """
    # Sanitize safe collection ID so it is safe for filenames, identifiers, and manifest records.
    safe_collection_id = re.sub('[^A-Za-z0-9._-]+', '_', str(collection_id))
    # Sanitize safe item ID so it is safe for filenames, identifiers, and manifest records.
    safe_item_id = re.sub('[^A-Za-z0-9._-]+', '_', str(item_id))
    # Sanitize safe asset role so it is safe for filenames, identifiers, and manifest records.
    safe_asset_role = re.sub('[^A-Za-z0-9._-]+', '_', str(asset_role).lower())
    # Build the scene directory location so HLS surface-reflectance bands uses the expected project
    # file structure.
    scene_directory = fire_hazard_hls_scaled_reflectance_directory / safe_collection_id / \
        safe_item_id
    # Create the output directory before writing workflow products.
    scene_directory.mkdir(parents=True, exist_ok=True)
    return scene_directory / f'{safe_item_id}_{safe_asset_role}_reflectance.tif'
# Define scaled reflectance records to control the inputs and rules used by HLS surface-reflectance
# bands.
fire_hazard_hls_scaled_reflectance_records = []
# Store total HLS reflectance assets needed to carry out HLS surface-reflectance bands.
total_hls_reflectance_assets = len(fire_hazard_hls_reflectance_assets)
# Set reflectance progress interval to control HLS surface-reflectance bands.
fire_hazard_hls_reflectance_progress_interval = 20

# Process the raster in internal windows to control memory use
# while preserving the full-resolution output.
# Process each (reflectance number entry so HLS surface-reflectance bands is applied consistently
# across all records.
for reflectance_number, (_, reflectance_record) in \
    enumerate(fire_hazard_hls_reflectance_assets.iterrows(),
    start=1):

    # Handle the reflectance number case explicitly during HLS surface-reflectance bands.
    if reflectance_number == 1 or reflectance_number % \
        fire_hazard_hls_reflectance_progress_interval == 0 or reflectance_number == \
        total_hls_reflectance_assets:
        print(f'-> Scaling reflectance raster '
            f'{reflectance_number:,} of '
            f'{total_hls_reflectance_assets:,}...')
    # Record collection ID needed to identify the current HLS asset in manifests and requests.
    collection_id = reflectance_record['COLLECTION_ID']
    # Record item ID needed to identify the current HLS asset in manifests and requests.
    item_id = reflectance_record['ITEM_ID']
    # Record asset role needed to identify the current HLS asset in manifests and requests.
    asset_role = str(reflectance_record['ASSET_ROLE']).lower()
    # Record asset key needed to identify the current HLS asset in manifests and requests.
    asset_key = reflectance_record['ASSET_KEY']
    # Build the source reflectance path location so HLS surface-reflectance bands uses the expected
    # project file structure.
    source_reflectance_path = Path(reflectance_record['LOCAL_PATH'])
    # Build the output reflectance path location so HLS surface-reflectance bands uses the expected
    # project file structure.
    output_reflectance_path = build_hls_scaled_reflectance_path(collection_id=collection_id,
        item_id=item_id, asset_role=asset_role)

    # Open the raster in a managed context so its file handle closes reliably after HLS
    # surface-reflectance bands.
    with rasterio.open(source_reflectance_path) as reflectance_source:
        # Prepare source profile to preserve and validate raster structure during HLS
        # surface-reflectance bands.
        source_profile = reflectance_source.profile.copy()
        # Store source NoData value needed to carry out HLS surface-reflectance bands.
        source_nodata_value = reflectance_source.nodata
        # Store source width needed to carry out HLS surface-reflectance bands.
        source_width = reflectance_source.width
        # Store source height needed to carry out HLS surface-reflectance bands.
        source_height = reflectance_source.height
        # Store source crs so spatial operations use the required coordinate reference system.
        source_crs = reflectance_source.crs
        # Store source transform needed to carry out HLS surface-reflectance bands.
        source_transform = reflectance_source.transform
        # Prepare scaled profile to preserve and validate raster structure during HLS
        # surface-reflectance bands.
        scaled_profile = source_profile.copy()
        scaled_profile.update({'driver': 'GTiff',
            'dtype': fire_hazard_hls_scaled_reflectance_dtype,
            'count': 1, 'nodata': fire_hazard_hls_scaled_reflectance_nodata,
            'compress': fire_hazard_hls_scaled_reflectance_compression,
            'predictor': 3, 'BIGTIFF': 'IF_SAFER'})

        # Open the raster in a managed context so its file handle closes reliably after HLS
        # surface-reflectance bands.
        with rasterio.open(output_reflectance_path,
            'w', **scaled_profile) as reflectance_destination:
            # Calculate source valid pixel count to quantify completeness and support QA checks.
            source_valid_pixel_count = 0
            # Calculate source NoData pixel count to quantify completeness and support QA checks.
            source_nodata_pixel_count = 0
            # Calculate saturation pixel count to quantify completeness and support QA checks.
            saturation_pixel_count = 0
            # Calculate scaled valid pixel count to quantify completeness and support QA checks.
            scaled_valid_pixel_count = 0
            # Track scaled minimum across processed pixels for output-range QA.
            scaled_minimum = None
            # Track scaled maximum across processed pixels for output-range QA.
            scaled_maximum = None

            # Process each ( entry so HLS surface-reflectance bands is applied consistently across
            # all records.
            for _, source_window in reflectance_source.block_windows(1):
                # Store raw reflectance needed to carry out HLS surface-reflectance bands.
                raw_reflectance = reflectance_source.read(1, window=source_window)

                # Handle the source NoData value case explicitly during HLS surface-reflectance
                # bands.
                if source_nodata_value is not None:
                    # Build the source nodata mask mask used to isolate records required for this
                    # analysis.
                    source_nodata_mask = raw_reflectance == source_nodata_value
                else:
                    # Build the source nodata mask mask used to isolate records required for this
                    # analysis.
                    source_nodata_mask = np.zeros(raw_reflectance.shape, dtype=bool)
                # Build the saturation mask mask used to isolate records required for this analysis.
                saturation_mask = raw_reflectance >= fire_hazard_hls_reflectance_saturation_value
                # Build the invalid reflectance mask mask used to isolate records required for this
                # analysis.
                invalid_reflectance_mask = source_nodata_mask | saturation_mask
                # Store scaled reflectance needed to carry out HLS surface-reflectance bands.
                scaled_reflectance = raw_reflectance.astype(np.float32) * \
                    fire_hazard_hls_reflectance_scale_factor + fire_hazard_hls_reflectance_offset
                # Build the scaled reflectance[invalid reflectance mask] mask to isolate pixels that
                # satisfy the current analysis criteria.
                scaled_reflectance[invalid_reflectance_mask] = \
                    fire_hazard_hls_scaled_reflectance_nodata
                # Write the processed data to the configured output resource.
                reflectance_destination.write(scaled_reflectance, 1, window=source_window)
                # Calculate source NoData pixel count to quantify completeness and support QA
                # checks.
                source_nodata_pixel_count += int(source_nodata_mask.sum())
                # Calculate saturation pixel count to quantify completeness and support QA checks.
                saturation_pixel_count += int(saturation_mask.sum())
                # Build the valid block mask mask used to isolate records required for this
                # analysis.
                valid_block_mask = ~invalid_reflectance_mask
                # Prepare valid block values used to process the current raster window.
                valid_block_values = scaled_reflectance[valid_block_mask]
                # Calculate block valid count to quantify completeness and support QA checks.
                block_valid_count = int(valid_block_values.size)
                # Calculate source valid pixel count to quantify completeness and support QA checks.
                source_valid_pixel_count += block_valid_count
                # Calculate scaled valid pixel count to quantify completeness and support QA checks.
                scaled_valid_pixel_count += block_valid_count

                # Handle the block valid count case explicitly during HLS surface-reflectance bands.
                if block_valid_count > 0:
                    # Track block minimum across processed pixels for output-range QA.
                    block_minimum = float(valid_block_values.min())
                    # Track block maximum across processed pixels for output-range QA.
                    block_maximum = float(valid_block_values.max())
                    # Track scaled minimum across processed pixels for output-range QA.
                    scaled_minimum = block_minimum if scaled_minimum is None else \
                        min(scaled_minimum,
                        block_minimum)
                    # Track scaled maximum across processed pixels for output-range QA.
                    scaled_maximum = block_maximum if scaled_maximum is None else \
                        max(scaled_maximum,
                        block_maximum)
            reflectance_destination.set_band_description(1,
                f'HLS scaled {asset_role} surface reflectance')
            reflectance_destination.update_tags(COLLECTION_ID=collection_id,
                ITEM_ID=item_id, SOURCE_ASSET_KEY=asset_key, ASSET_ROLE=asset_role,
                SOURCE_PATH=str(source_reflectance_path), \
                    REFLECTANCE_SCALE_FACTOR=fire_hazard_hls_reflectance_scale_factor,
                REFLECTANCE_OFFSET=fire_hazard_hls_reflectance_offset,
                SOURCE_SATURATION_VALUE=fire_hazard_hls_reflectance_saturation_value)

    # Open the raster in a managed context so its file handle closes reliably after HLS
    # surface-reflectance bands.
    with rasterio.open(output_reflectance_path) as scaled_source:
        # Store output grid matches source needed to carry out HLS surface-reflectance bands.
        output_grid_matches_source = scaled_source.width == source_width and scaled_source.height \
            == source_height and (scaled_source.crs == source_crs) and (scaled_source.transform \
            == source_transform)
        # Record output structure valid so HLS surface-reflectance bands can preserve a clear
        # processing and validation outcome.
        output_structure_valid = scaled_source.count == 1 and scaled_source.dtypes[0] == \
            fire_hazard_hls_scaled_reflectance_dtype and (scaled_source.nodata == \
            fire_hazard_hls_scaled_reflectance_nodata)
    # Record output file valid so HLS surface-reflectance bands can preserve a clear processing and
    # validation outcome.
    output_file_valid = output_reflectance_path.exists() and output_reflectance_path.is_file() \
        and (output_reflectance_path.stat().st_size > 0) and output_grid_matches_source and \
        output_structure_valid and (scaled_valid_pixel_count > 0)
    fire_hazard_hls_scaled_reflectance_records.append({'COLLECTION_ID': collection_id,
        'ITEM_ID': item_id, 'ASSET_ROLE': asset_role, 'ASSET_KEY': asset_key,
        'SOURCE_PATH': str(source_reflectance_path), 'SCALED_PATH': str(output_reflectance_path),
        'SOURCE_NODATA_VALUE': source_nodata_value, 'SCALE_FACTOR': \
            fire_hazard_hls_reflectance_scale_factor,
        'OFFSET': fire_hazard_hls_reflectance_offset, 'SATURATION_VALUE': \
            fire_hazard_hls_reflectance_saturation_value,
        'SOURCE_NODATA_PIXELS': source_nodata_pixel_count,
        'SATURATION_PIXELS': saturation_pixel_count, 'VALID_PIXELS': scaled_valid_pixel_count,
        'SCALED_MINIMUM': scaled_minimum, 'SCALED_MAXIMUM': scaled_maximum,
        'OUTPUT_SIZE_BYTES': output_reflectance_path.stat().st_size if \
            output_reflectance_path.exists() else 0,
        'OUTPUT_SIZE_MB': output_reflectance_path.stat().st_size / 1024 ** 2 if \
            output_reflectance_path.exists() else 0.0,
        'GRID_MATCHES_SOURCE': output_grid_matches_source,
        'STRUCTURE_VALID': output_structure_valid, 'VALID': output_file_valid})
# Store scaled reflectance manifest needed to carry out HLS surface-reflectance bands.
fire_hazard_hls_scaled_reflectance_manifest = \
    pd.DataFrame(fire_hazard_hls_scaled_reflectance_records).sort_values(['COLLECTION_ID',
    'ITEM_ID', 'ASSET_ROLE']).reset_index(drop=True)
# Store failed HLS scaled reflectance needed to carry out HLS surface-reflectance bands.
failed_hls_scaled_reflectance = \
    fire_hazard_hls_scaled_reflectance_manifest[~fire_hazard_hls_scaled_reflectance_manifest \
    ['VALID']]
# Record scaled reflectance valid so HLS surface-reflectance bands can preserve a clear processing
# and validation outcome.
fire_hazard_hls_scaled_reflectance_valid = failed_hls_scaled_reflectance.empty and \
    len(fire_hazard_hls_scaled_reflectance_manifest) == len(fire_hazard_hls_reflectance_assets)

# Stop execution if one or more HLS surface-reflectance bands failed scaling or output validation.
if not fire_hazard_hls_scaled_reflectance_valid:
    raise ValueError('One or more HLS '
        'surface-reflectance bands '
        'failed scaling or output '
        'validation.')
# Store total scaled reflectance size bytes needed to carry out HLS surface-reflectance bands.
total_scaled_reflectance_size_bytes = \
    int(fire_hazard_hls_scaled_reflectance_manifest['OUTPUT_SIZE_BYTES'].sum())
# Store scaled reflectance summary needed to carry out HLS surface-reflectance bands.
fire_hazard_hls_scaled_reflectance_summary = pd.DataFrame([{'SOURCE_BANDS': \
    len(fire_hazard_hls_reflectance_assets),
    'SCALED_BANDS': len(fire_hazard_hls_scaled_reflectance_manifest),
    'VALID_SCALED_BANDS': int(fire_hazard_hls_scaled_reflectance_manifest['VALID'].sum()),
    'SCENES': fire_hazard_hls_scaled_reflectance_manifest['ITEM_ID'].nunique(),
    'SCALE_FACTOR': fire_hazard_hls_reflectance_scale_factor,
    'OUTPUT_DTYPE': fire_hazard_hls_scaled_reflectance_dtype,
    'TOTAL_OUTPUT_SIZE_GB': total_scaled_reflectance_size_bytes / 1024 ** 3,
    'ALL_OUTPUTS_VALID': fire_hazard_hls_scaled_reflectance_valid}])
# Save the scaled reflectance manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_scaled_reflectance_manifest.to_csv(fire_hazard_hls_scaled_reflectance_manifest_path,
    index=False)
# Save the scaled reflectance summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_scaled_reflectance_summary.to_csv(fire_hazard_hls_scaled_reflectance_summary_path,
    index=False)
print(f'-> Reflectance source bands processed: {len(fire_hazard_hls_reflectance_assets):,}')
print(f'-> Scaled reflectance bands created: {len(fire_hazard_hls_scaled_reflectance_manifest):,}')
print(f"-> HLS scenes processed: "
    f"{fire_hazard_hls_scaled_reflectance_manifest['ITEM_ID'].nunique():,}")
print(f'-> Total scaled-output size: {total_scaled_reflectance_size_bytes / 1024 ** 3:,.2f} GB')
print(f'-> Scaled reflectance valid: {fire_hazard_hls_scaled_reflectance_valid}')
print(f'-> Scaling manifest saved: {fire_hazard_hls_scaled_reflectance_manifest_path}')
print('\n--- HLS SCALED REFLECTANCE SUMMARY ---')
display(fire_hazard_hls_scaled_reflectance_summary)
print('\n--- HLS SCALED REFLECTANCE SAMPLE ---')
display(fire_hazard_hls_scaled_reflectance_manifest[['COLLECTION_ID',
    'ITEM_ID', 'ASSET_ROLE', 'ASSET_KEY', 'SCALE_FACTOR',
    'SOURCE_NODATA_VALUE', 'SATURATION_PIXELS', 'VALID_PIXELS',
    'SCALED_MINIMUM', 'SCALED_MAXIMUM', 'OUTPUT_SIZE_MB',
    'VALID']].head(5))
print('\nNOTE:')
print("The scaled reflectance rasters remain in each scene's native HLS tile grid.")

# Use NDVI to represent live vegetation density and potential fuel availability.
print('The scene-level Fmask outputs '
    'will be applied while '
    'calculating NDVI and NDMI in '
    'the next processing step.')
print('\n=== HLS SURFACE-REFLECTANCE SCALING COMPLETE ===')

# Use NDMI to represent vegetation and canopy moisture conditions.


=== SCALING HLS SURFACE-REFLECTANCE BANDS ===
-> Scaling reflectance raster 1 of 195...
-> Scaling reflectance raster 20 of 195...
-> Scaling reflectance raster 40 of 195...
-> Scaling reflectance raster 60 of 195...
-> Scaling reflectance raster 80 of 195...
-> Scaling reflectance raster 100 of 195...
-> Scaling reflectance raster 120 of 195...
-> Scaling reflectance raster 140 of 195...
-> Scaling reflectance raster 160 of 195...
-> Scaling reflectance raster 180 of 195...
-> Scaling reflectance raster 195 of 195...
-> Reflectance source bands processed: 195
-> Scaled reflectance bands created: 195
-> HLS scenes processed: 65
-> Total scaled-output size: 5.20 GB
-> Scaled reflectance valid: True
-> Scaling manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\reflectance_manifests\hls_scaled_reflectance_manifest.csv

--- HLS SCALED REFLECTANCE SUMMARY ---


,SOURCE_BANDS,SCALED_BANDS,VALID_SCALED_BANDS,SCENES,SCALE_FACTOR,OUTPUT_DTYPE,TOTAL_OUTPUT_SIZE_GB,ALL_OUTPUTS_VALID
0,195,195,195,65,0.0001,float32,5.203704,True



--- HLS SCALED REFLECTANCE SAMPLE ---


,COLLECTION_ID,ITEM_ID,ASSET_ROLE,ASSET_KEY,SCALE_FACTOR,SOURCE_NODATA_VALUE,SATURATION_PIXELS,VALID_PIXELS,SCALED_MINIMUM,SCALED_MAXIMUM,OUTPUT_SIZE_MB,VALID
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,nir,B05,0.0001,-9999.0,0,12525785,-0.0778,0.9418,34.842062,True
1,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,red,B04,0.0001,-9999.0,0,12525785,-0.0553,0.9060,35.244364,True
2,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,swir1,B06,0.0001,-9999.0,0,12525785,-0.0563,1.0651,35.133884,True
3,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,nir,B05,0.0001,-9999.0,0,7282866,-0.0548,0.8502,20.563116,True
4,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,red,B04,0.0001,-9999.0,0,7282866,-0.0300,0.8118,21.083252,True



NOTE:
The scaled reflectance rasters remain in each scene's native HLS tile grid.
The scene-level Fmask outputs will be applied while calculating NDVI and NDMI in the next processing step.

=== HLS SURFACE-REFLECTANCE SCALING COMPLETE ===


### Defining Scene-Level Ndvi and Ndmi Settings


In [77]:
print('=== DEFINING SCENE-LEVEL NDVI AND NDMI SETTINGS ===')
# List the required HLS index setting prerequisites required before scene-level NDVI and NDMI
# settings can run.
required_hls_index_setting_inputs = ['fire_hazard_hls_scaled_reflectance_manifest',
    'fire_hazard_hls_fmask_manifest', 'fire_hazard_hls_raw_directory',
    'fire_hazard_hls_scaled_reflectance_nodata', 'fire_hazard_hls_valid_mask_value']
# Identify unavailable HLS index setting so scene-level NDVI and NDMI settings stops
# before using incomplete inputs.
missing_hls_index_setting_inputs = [object_name for object_name in \
    required_hls_index_setting_inputs if object_name not in globals()]

# Stop execution when required hls index setting inputs inputs are unavailable.
if missing_hls_index_setting_inputs:
    raise NameError(f'The following HLS '
        f'vegetation-index objects are '
        f'missing:\n'
        f'{missing_hls_index_setting_inputs}\n\n'
        f'Run step Decode HLS Fmask and '
        f'Create Valid-Pixel Masks and '
        f'step Scale HLS '
        f'Surface-Reflectance Bands '
        f'before calculating NDVI and '
        f'NDMI.')

# Stop execution if hLS scaled-reflectance outputs have not passed complete validation.
if 'fire_hazard_hls_scaled_reflectance_valid' not in globals() or not \
    fire_hazard_hls_scaled_reflectance_valid:
    raise ValueError('The HLS scaled-reflectance outputs have not passed complete validation.')

# Stop execution if hLS Fmask outputs have not passed complete validation.
if 'fire_hazard_hls_fmask_processing_valid' not in globals() or not \
    fire_hazard_hls_fmask_processing_valid:
    raise ValueError('The HLS Fmask outputs have not passed complete validation.')
# Define index band roles to control the inputs and rules used by scene-level NDVI and NDMI
# settings.
fire_hazard_hls_index_band_roles = {'NDVI': {'numerator_positive': 'nir',
    'numerator_negative': 'red', 'denominator_first': 'nir',
    'denominator_second': 'red'}, 'NDMI': {'numerator_positive': 'nir',
    'numerator_negative': 'swir1', 'denominator_first': 'nir',
    'denominator_second': 'swir1'}}
# Set index denominator tolerance to control scene-level NDVI and NDMI settings.
fire_hazard_hls_index_denominator_tolerance = 1e-06
# Track index minimum across processed pixels for output-range QA.
fire_hazard_hls_index_minimum = -1.0
# Track index maximum across processed pixels for output-range QA.
fire_hazard_hls_index_maximum = 1.0
# Set index dtype to control scene-level NDVI and NDMI settings.
fire_hazard_hls_index_dtype = 'float32'
# Extract index NoData from the current record for scene-level NDVI and NDMI settings.
fire_hazard_hls_index_nodata = globals().get('fire_hazard_nodata_value', -9999.0)
# Set index compression to control scene-level NDVI and NDMI settings.
fire_hazard_hls_index_compression = 'deflate'
# Build the scene index directory location so scene-level NDVI and NDMI settings uses the expected
# project file structure.
fire_hazard_hls_scene_index_directory = fire_hazard_hls_raw_directory / 'scene_vegetation_indices'
# Build the index manifest directory location so scene-level NDVI and NDMI settings uses the
# expected project file structure.
fire_hazard_hls_index_manifest_directory = fire_hazard_hls_raw_directory / \
    'vegetation_index_manifests'

# Process each directory path entry so scene-level NDVI and NDMI settings is applied consistently
# across all records.
for directory_path in [fire_hazard_hls_scene_index_directory,
    fire_hazard_hls_index_manifest_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the scene index manifest path location so scene-level NDVI and NDMI settings uses the
# expected project file structure.
fire_hazard_hls_scene_index_manifest_path = fire_hazard_hls_index_manifest_directory / \
    'hls_scene_index_manifest.csv'
# Build the scene index summary path location so scene-level NDVI and NDMI settings uses the
# expected project file structure.
fire_hazard_hls_scene_index_summary_path = fire_hazard_hls_index_manifest_directory / \
    'hls_scene_index_summary.csv'
# Build the index policy path location so scene-level NDVI and NDMI settings uses the expected
# project file structure.
fire_hazard_hls_index_policy_path = fire_hazard_hls_index_manifest_directory / \
    'hls_scene_index_policy.json'
# Define index policy to control the inputs and rules used by scene-level NDVI and NDMI settings.
fire_hazard_hls_index_policy = {'indices': fire_hazard_hls_index_band_roles,
    'denominator_tolerance': fire_hazard_hls_index_denominator_tolerance,
    'valid_range': [fire_hazard_hls_index_minimum,
    fire_hazard_hls_index_maximum], 'output_dtype': fire_hazard_hls_index_dtype,
    'output_nodata': fire_hazard_hls_index_nodata,
    'compression': fire_hazard_hls_index_compression,
    'valid_fmask_value': fire_hazard_hls_valid_mask_value}

# Open the file in a managed context so its handle closes reliably after scene-level NDVI and NDMI
# settings.
with fire_hazard_hls_index_policy_path.open('w', encoding='utf-8') as index_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_hls_index_policy, index_policy_file, indent=2)
# Store index policy summary needed to carry out scene-level NDVI and NDMI settings.
fire_hazard_hls_index_policy_summary = pd.DataFrame([{'INDEX': 'NDVI',
    'FORMULA': '(NIR - Red) / (NIR + Red)', 'OUTPUT_DTYPE': fire_hazard_hls_index_dtype,
    'OUTPUT_NODATA': fire_hazard_hls_index_nodata,
    'VALID_MINIMUM': fire_hazard_hls_index_minimum,
    'VALID_MAXIMUM': fire_hazard_hls_index_maximum},
    {'INDEX': 'NDMI', 'FORMULA': '(NIR - SWIR1) / (NIR + SWIR1)',
    'OUTPUT_DTYPE': fire_hazard_hls_index_dtype, 'OUTPUT_NODATA': fire_hazard_hls_index_nodata,
    'VALID_MINIMUM': fire_hazard_hls_index_minimum,
    'VALID_MAXIMUM': fire_hazard_hls_index_maximum}])
print(f'-> Indices configured: {list(fire_hazard_hls_index_band_roles.keys())}')
print(f'-> Denominator tolerance: {fire_hazard_hls_index_denominator_tolerance}')
print(f'-> Valid index range: {fire_hazard_hls_index_minimum} to {fire_hazard_hls_index_maximum}')
print(f'-> Output data type: {fire_hazard_hls_index_dtype}')
print(f'-> Output NoData value: {fire_hazard_hls_index_nodata}')
print(f'-> Index policy saved: {fire_hazard_hls_index_policy_path}')
print('\n--- HLS VEGETATION-INDEX POLICY ---')
display(fire_hazard_hls_index_policy_summary)
print('\n=== SCENE-LEVEL NDVI AND NDMI SETTINGS DEFINED ===')



=== DEFINING SCENE-LEVEL NDVI AND NDMI SETTINGS ===
-> Indices configured: ['NDVI', 'NDMI']
-> Denominator tolerance: 1e-06
-> Valid index range: -1.0 to 1.0
-> Output data type: float32
-> Output NoData value: -9999.0
-> Index policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_index_manifests\hls_scene_index_policy.json

--- HLS VEGETATION-INDEX POLICY ---


,INDEX,FORMULA,OUTPUT_DTYPE,OUTPUT_NODATA,VALID_MINIMUM,VALID_MAXIMUM
0,NDVI,(NIR - Red) / (NIR + Red),float32,-9999.0,-1.0,1.0
1,NDMI,(NIR - SWIR1) / (NIR + SWIR1),float32,-9999.0,-1.0,1.0



=== SCENE-LEVEL NDVI AND NDMI SETTINGS DEFINED ===


### Calculating Scene-Level Ndvi and Ndmi


In [78]:
print('=== CALCULATING SCENE-LEVEL NDVI AND NDMI ===')
# List the required HLS index processing prerequisites required before scene-level NDVI and
# NDMI can run.
required_hls_index_processing_inputs = ['fire_hazard_hls_scaled_reflectance_manifest',
    'fire_hazard_hls_fmask_manifest', 'fire_hazard_hls_index_denominator_tolerance',
    'fire_hazard_hls_index_minimum', 'fire_hazard_hls_index_maximum',
    'fire_hazard_hls_index_dtype', 'fire_hazard_hls_index_nodata',
    'fire_hazard_hls_index_compression', 'fire_hazard_hls_scaled_reflectance_nodata',
    'fire_hazard_hls_valid_mask_value', 'fire_hazard_hls_scene_index_directory',
    'fire_hazard_hls_scene_index_manifest_path', 'fire_hazard_hls_scene_index_summary_path']
# Identify unavailable HLS index processing so scene-level NDVI and NDMI stops before
# using incomplete inputs.
missing_hls_index_processing_inputs = [object_name for object_name in \
    required_hls_index_processing_inputs if object_name not in globals()]

# Stop execution when required hls index processing inputs inputs are unavailable.
if missing_hls_index_processing_inputs:
    raise NameError(f'The following HLS '
        f'vegetation-index processing '
        f'objects are missing:\n'
        f'{missing_hls_index_processing_inputs}\n\n'
        f'Run step Define Scene-Level '
        f'Vegetation-Index Settings, '
        f'step Scale HLS '
        f'Surface-Reflectance Bands, and '
        f'step Decode HLS Fmask before '
        f'calculating NDVI and NDMI.')

# Stop execution if hLS scaled-reflectance outputs have not passed complete validation.
if 'fire_hazard_hls_scaled_reflectance_valid' not in globals() or not \
    fire_hazard_hls_scaled_reflectance_valid:
    raise ValueError('The HLS scaled-reflectance outputs have not passed complete validation.')

# Stop execution if hLS Fmask outputs have not passed complete validation.
if 'fire_hazard_hls_fmask_processing_valid' not in globals() or not \
    fire_hazard_hls_fmask_processing_valid:
    raise ValueError('The HLS Fmask outputs have not passed complete validation.')
# List the required scaled reflectance fields prerequisites required before scene-level NDVI and
# NDMI can run.
required_scaled_reflectance_fields = ['COLLECTION_ID',
    'ITEM_ID', 'ASSET_ROLE', 'SCALED_PATH', 'VALID']
# Identify unavailable scaled reflectance fields so scene-level NDVI and NDMI stops before using
# incomplete inputs.
missing_scaled_reflectance_fields = [field_name for field_name in \
    required_scaled_reflectance_fields if field_name not in \
    fire_hazard_hls_scaled_reflectance_manifest.columns]

# Stop execution when required scaled reflectance fields inputs are unavailable.
if missing_scaled_reflectance_fields:
    raise ValueError(f'The scaled-reflectance '
        f'manifest is missing required '
        f'fields:\n'
        f'{missing_scaled_reflectance_fields}')

# Stop execution if one or more scaled reflectance rasters are marked invalid.
if not fire_hazard_hls_scaled_reflectance_manifest['VALID'].all():
    raise ValueError('One or more scaled reflectance rasters are marked invalid.')
# List the required Fmask manifest fields prerequisites required before scene-level NDVI and NDMI
# can run.
required_fmask_manifest_fields = ['COLLECTION_ID', 'ITEM_ID', 'VALID_MASK_PATH', 'VALID']
# Identify unavailable Fmask manifest fields so scene-level NDVI and NDMI stops before using
# incomplete inputs.
missing_fmask_manifest_fields = [field_name for field_name in required_fmask_manifest_fields if \
    field_name not in fire_hazard_hls_fmask_manifest.columns]

# Stop execution when required fmask manifest fields inputs are unavailable.
if missing_fmask_manifest_fields:
    raise ValueError(f'The HLS Fmask manifest is '
        f'missing required fields:\n'
        f'{missing_fmask_manifest_fields}')

# Stop execution if one or more HLS valid-pixel masks are marked invalid.
if not fire_hazard_hls_fmask_manifest['VALID'].all():
    raise ValueError('One or more HLS valid-pixel masks are marked invalid.')
# Build the scene band paths location so scene-level NDVI and NDMI uses the expected project file
# structure.
fire_hazard_hls_scene_band_paths = \
    fire_hazard_hls_scaled_reflectance_manifest.assign(ASSET_ROLE= \
    fire_hazard_hls_scaled_reflectance_manifest['ASSET_ROLE'].astype(str).str.lower()) \
    .pivot_table(index=['COLLECTION_ID',
    'ITEM_ID'], columns='ASSET_ROLE', values='SCALED_PATH',
    aggfunc='first').reset_index()
# Set name to control scene-level NDVI and NDMI.
fire_hazard_hls_scene_band_paths.columns.name = None
# Build the scene band paths location so scene-level NDVI and NDMI uses the expected project file
# structure.
fire_hazard_hls_scene_band_paths = fire_hazard_hls_scene_band_paths.rename(columns={'red': \
    'RED_PATH',
    'nir': 'NIR_PATH', 'swir1': 'SWIR1_PATH'})
# List the required scene band path fields prerequisites required before scene-level NDVI and NDMI
# can run.
required_scene_band_path_fields = ['RED_PATH', 'NIR_PATH', 'SWIR1_PATH']
# Identify unavailable scene band path fields so scene-level NDVI and NDMI stops before using
# incomplete inputs.
missing_scene_band_path_fields = [field_name for field_name in required_scene_band_path_fields if \
    field_name not in fire_hazard_hls_scene_band_paths.columns]

# Stop execution when required scene band path fields inputs are unavailable.
if missing_scene_band_path_fields:
    raise ValueError(f'The scene-level reflectance '
        f'table is missing required '
        f'band-path fields:\n'
        f'{missing_scene_band_path_fields}')
# Build incomplete scene band records used to track the records included in this processing stage.
incomplete_scene_band_records = \
    fire_hazard_hls_scene_band_paths[required_scene_band_path_fields].isna().any(axis=1)

# Stop execution if one or more HLS scenes are missing a scaled red, NIR, or SWIR1 raster.
if incomplete_scene_band_records.any():
    raise ValueError('One or more HLS scenes are missing a scaled red, NIR, or SWIR1 raster.')
# Build the fire hazard hls scene mask paths mask used to isolate records required for this
# analysis.
fire_hazard_hls_scene_mask_paths = fire_hazard_hls_fmask_manifest[['COLLECTION_ID',
    'ITEM_ID', 'VALID_MASK_PATH']].copy()
# Build duplicate fmask scene records used to track the records included in this processing stage.
duplicate_fmask_scene_records = fire_hazard_hls_scene_mask_paths.duplicated(subset=['COLLECTION_ID',
    'ITEM_ID'], keep=False)

# Stop execution if fmask manifest contains duplicate valid-mask records for one or more scenes.
if duplicate_fmask_scene_records.any():
    raise ValueError('The Fmask manifest contains '
        'duplicate valid-mask records '
        'for one or more scenes.')
# Store scene index sources needed to carry out scene-level NDVI and NDMI.
fire_hazard_hls_scene_index_sources = \
    fire_hazard_hls_scene_band_paths.merge(fire_hazard_hls_scene_mask_paths,
    on=['COLLECTION_ID', 'ITEM_ID'], how='inner', validate='one_to_one')
# Calculate expected HLS index scene count to quantify completeness and support QA checks.
expected_hls_index_scene_count = fire_hazard_hls_scaled_reflectance_manifest[['COLLECTION_ID',
    'ITEM_ID']].drop_duplicates().shape[0]

# Stop execution if scaled-reflectance and Fmask manifests could not be paired for every HLS scene.
if len(fire_hazard_hls_scene_index_sources) != expected_hls_index_scene_count:
    raise ValueError('The scaled-reflectance and '
        'Fmask manifests could not be '
        'paired for every HLS scene.')
# Identify unavailable HLS index source files so scene-level NDVI and NDMI stops before using
# incomplete inputs.
missing_hls_index_source_files = []

# Process each ( entry so scene-level NDVI and NDMI is applied consistently across all records.
for _, scene_record in fire_hazard_hls_scene_index_sources.iterrows():

    # Process each path field entry so scene-level NDVI and NDMI is applied consistently across all
    # records.
    for path_field in ['RED_PATH', 'NIR_PATH', 'SWIR1_PATH', 'VALID_MASK_PATH']:
        # Build the source path location so scene-level NDVI and NDMI uses the expected project file
        # structure.
        source_path = Path(scene_record[path_field])

        # Use the existing file only when it is present and valid for scene-level NDVI and NDMI.
        if not source_path.exists() or not source_path.is_file():
            missing_hls_index_source_files.append(str(source_path))

# Stop execution when required hls index source files inputs are unavailable.
if missing_hls_index_source_files:
    raise FileNotFoundError(f'One or more scene-level source '
        f'rasters are missing.\n\nFirst '
        f'missing paths:\n'
        f'{missing_hls_index_source_files[:20]}')

# Encapsulate build HLS scene index path so repeated scene-level NDVI and NDMI steps use consistent
# logic.
def build_hls_scene_index_path(collection_id, item_id, index_name):

    """
    Builds the output path for one scene-level NDVI
    or NDMI raster.
    """
    # Sanitize safe collection ID so it is safe for filenames, identifiers, and manifest records.
    safe_collection_id = re.sub('[^A-Za-z0-9._-]+', '_', str(collection_id))
    # Sanitize safe item ID so it is safe for filenames, identifiers, and manifest records.
    safe_item_id = re.sub('[^A-Za-z0-9._-]+', '_', str(item_id))
    # Sanitize safe index name so it is safe for filenames, identifiers, and manifest records.
    safe_index_name = re.sub('[^A-Za-z0-9._-]+', '_', str(index_name).lower())
    # Build the scene directory location so scene-level NDVI and NDMI uses the expected project file
    # structure.
    scene_directory = fire_hazard_hls_scene_index_directory / safe_collection_id / safe_item_id
    # Create the output directory before writing workflow products.
    scene_directory.mkdir(parents=True, exist_ok=True)
    return scene_directory / f'{safe_item_id}_{safe_index_name}.tif'
# Define scene index records to control the inputs and rules used by scene-level NDVI and NDMI.
fire_hazard_hls_scene_index_records = []
# Store total HLS index scenes needed to carry out scene-level NDVI and NDMI.
total_hls_index_scenes = len(fire_hazard_hls_scene_index_sources)
# Set index progress interval to control scene-level NDVI and NDMI.
fire_hazard_hls_index_progress_interval = 5

# Process each (scene number entry so scene-level NDVI and NDMI is applied consistently across all
# records.
for scene_number, (_, scene_record) in enumerate(fire_hazard_hls_scene_index_sources.iterrows(),
    start=1):

    # Handle the scene number case explicitly during scene-level NDVI and NDMI.
    if scene_number == 1 or scene_number % fire_hazard_hls_index_progress_interval == 0 or \
        scene_number == total_hls_index_scenes:
        print(f'-> Calculating indices for scene {scene_number:,} of {total_hls_index_scenes:,}...')
    # Record collection ID needed to identify the current HLS asset in manifests and requests.
    collection_id = scene_record['COLLECTION_ID']
    # Record item ID needed to identify the current HLS asset in manifests and requests.
    item_id = scene_record['ITEM_ID']
    # Build the red path location so scene-level NDVI and NDMI uses the expected project file
    # structure.
    red_path = Path(scene_record['RED_PATH'])
    # Build the nir path location so scene-level NDVI and NDMI uses the expected project file
    # structure.
    nir_path = Path(scene_record['NIR_PATH'])
    # Build the swir1 path location so scene-level NDVI and NDMI uses the expected project file
    # structure.
    swir1_path = Path(scene_record['SWIR1_PATH'])
    # Build the valid mask path mask used to isolate records required for this analysis.
    valid_mask_path = Path(scene_record['VALID_MASK_PATH'])
    # Build the NDVI output path location so scene-level NDVI and NDMI uses the expected project
    # file structure.
    ndvi_output_path = build_hls_scene_index_path(collection_id=collection_id,
        item_id=item_id, index_name='ndvi')
    # Build the NDMI output path location so scene-level NDVI and NDMI uses the expected project
    # file structure.
    ndmi_output_path = build_hls_scene_index_path(collection_id=collection_id,
        item_id=item_id, index_name='ndmi')

    # Open the raster in a managed context so its file handle closes reliably after scene-level NDVI
    # and NDMI.
    with rasterio.open(red_path) as red_source:

        # Open the raster in a managed context so its file handle closes reliably after scene-level
        # NDVI and NDMI.
        with rasterio.open(nir_path) as nir_source:

            # Open the raster in a managed context so its file handle closes reliably after
            # scene-level NDVI and NDMI.
            with rasterio.open(swir1_path) as swir1_source:

                # Open the raster in a managed context so its file handle closes reliably after
                # scene-level NDVI and NDMI.
                with rasterio.open(valid_mask_path) as mask_source:
                    # Store scene grids match needed to carry out scene-level NDVI and NDMI.
                    scene_grids_match = all([nir_source.width == red_source.width,
                        nir_source.height == red_source.height, nir_source.crs == red_source.crs,
                        nir_source.transform == red_source.transform, swir1_source.width == \
                            red_source.width,
                        swir1_source.height == red_source.height, swir1_source.crs == \
                            red_source.crs,
                        swir1_source.transform == red_source.transform,
                        mask_source.width == red_source.width, mask_source.height == \
                            red_source.height,
                        mask_source.crs == red_source.crs, mask_source.transform == \
                            red_source.transform])

                    # Stop execution if red, NIR, SWIR1, and Fmask rasters do not share one grid for
                    # HLS scene the reported value.
                    if not scene_grids_match:
                        raise ValueError(f'The red, NIR, SWIR1, and Fmask '
                            f'rasters do not share one grid '
                            f'for HLS scene {item_id}.')
                    # Prepare index profile to preserve and validate raster structure during
                    # scene-level NDVI and NDMI.
                    index_profile = red_source.profile.copy()
                    index_profile.update({'driver': 'GTiff',
                        'dtype': fire_hazard_hls_index_dtype, 'count': 1,
                        'nodata': fire_hazard_hls_index_nodata, 'compress': \
                            fire_hazard_hls_index_compression,
                        'predictor': 3, 'BIGTIFF': 'IF_SAFER'})

                    # Open the raster in a managed context so its file handle closes reliably after
                    # scene-level NDVI and NDMI.
                    with rasterio.open(ndvi_output_path, 'w', **index_profile) as ndvi_destination:

                        # Open the raster in a managed context so its file handle closes reliably
                        # after scene-level NDVI and NDMI.
                        with rasterio.open(ndmi_output_path,
                            'w', **index_profile) as ndmi_destination:
                            # Calculate valid Fmask pixel count to quantify completeness and support
                            # QA checks.
                            valid_fmask_pixel_count = 0
                            # Calculate invalid Fmask pixel count to quantify completeness and
                            # support QA checks.
                            invalid_fmask_pixel_count = 0
                            # Calculate invalid reflectance pixel count to quantify completeness and
                            # support QA checks.
                            invalid_reflectance_pixel_count = 0
                            # Calculate NDVI valid pixel count to quantify completeness and support
                            # QA checks.
                            ndvi_valid_pixel_count = 0
                            # Calculate NDMI valid pixel count to quantify completeness and support
                            # QA checks.
                            ndmi_valid_pixel_count = 0
                            # Calculate NDVI denominator zero count to quantify completeness and
                            # support QA checks.
                            ndvi_denominator_zero_count = 0
                            # Calculate NDMI denominator zero count to quantify completeness and
                            # support QA checks.
                            ndmi_denominator_zero_count = 0
                            # Track NDVI minimum across processed pixels for output-range QA.
                            ndvi_minimum = None
                            # Track NDVI maximum across processed pixels for output-range QA.
                            ndvi_maximum = None
                            # Track NDMI minimum across processed pixels for output-range QA.
                            ndmi_minimum = None
                            # Track NDMI maximum across processed pixels for output-range QA.
                            ndmi_maximum = None

                            # Process each ( entry so scene-level NDVI and NDMI is applied
                            # consistently across all records.
                            for _, source_window in red_source.block_windows(1):
                                # Prepare red array used to process the current raster window.
                                red_array = red_source.read(1,
                                    window=source_window).astype(np.float32)
                                # Prepare nir array used to process the current raster window.
                                nir_array = nir_source.read(1,
                                    window=source_window).astype(np.float32)
                                # Prepare swir1 array used to process the current raster window.
                                swir1_array = swir1_source.read(1,
                                    window=source_window).astype(np.float32)
                                # Build the valid mask array mask used to isolate records required
                                # for this analysis.
                                valid_mask_array = mask_source.read(1, window=source_window)
                                # Build the Fmask valid mask to isolate pixels that satisfy the
                                # current analysis criteria.
                                fmask_valid = valid_mask_array == fire_hazard_hls_valid_mask_value
                                # Record reflectance invalid so scene-level NDVI and NDMI can
                                # preserve a clear processing and validation outcome.
                                reflectance_invalid = (red_array == \
                                    fire_hazard_hls_scaled_reflectance_nodata) | (nir_array == \
                                    fire_hazard_hls_scaled_reflectance_nodata) | (swir1_array == \
                                    fire_hazard_hls_scaled_reflectance_nodata) | \
                                    ~np.isfinite(red_array) | ~np.isfinite(nir_array) | \
                                    ~np.isfinite(swir1_array)
                                # Record base valid so scene-level NDVI and NDMI can preserve a
                                # clear processing and validation outcome.
                                base_valid = fmask_valid & ~reflectance_invalid
                                # Store NDVI denominator needed to carry out scene-level NDVI and
                                # NDMI.
                                ndvi_denominator = nir_array + red_array
                                # Record NDVI denominator valid so scene-level NDVI and NDMI can
                                # preserve a clear processing and validation outcome.
                                ndvi_denominator_valid = np.abs(ndvi_denominator) > \
                                    fire_hazard_hls_index_denominator_tolerance
                                # Record NDVI valid so scene-level NDVI and NDMI can preserve a
                                # clear processing and validation outcome.
                                ndvi_valid = base_valid & ndvi_denominator_valid
                                # Prepare NDVI array used to process the current raster window.
                                ndvi_array = np.full(red_array.shape,
                                    fire_hazard_hls_index_nodata, dtype=np.float32)
                                # Prepare NDVI array[NDVI valid] used to process the current raster
                                # window.
                                ndvi_array[ndvi_valid] = (nir_array[ndvi_valid] - \
                                    red_array[ndvi_valid]) / ndvi_denominator[ndvi_valid]
                                # Prepare NDVI array[NDVI valid] used to process the current raster
                                # window.
                                ndvi_array[ndvi_valid] = np.clip(ndvi_array[ndvi_valid],
                                    fire_hazard_hls_index_minimum, fire_hazard_hls_index_maximum)
                                # Store NDMI denominator needed to carry out scene-level NDVI and
                                # NDMI.
                                ndmi_denominator = nir_array + swir1_array
                                # Record NDMI denominator valid so scene-level NDVI and NDMI can
                                # preserve a clear processing and validation outcome.
                                ndmi_denominator_valid = np.abs(ndmi_denominator) > \
                                    fire_hazard_hls_index_denominator_tolerance
                                # Record NDMI valid so scene-level NDVI and NDMI can preserve a
                                # clear processing and validation outcome.
                                ndmi_valid = base_valid & ndmi_denominator_valid
                                # Prepare NDMI array used to process the current raster window.
                                ndmi_array = np.full(red_array.shape,
                                    fire_hazard_hls_index_nodata, dtype=np.float32)
                                # Prepare NDMI array[NDMI valid] used to process the current raster
                                # window.
                                ndmi_array[ndmi_valid] = (nir_array[ndmi_valid] - \
                                    swir1_array[ndmi_valid]) / ndmi_denominator[ndmi_valid]
                                # Prepare NDMI array[NDMI valid] used to process the current raster
                                # window.
                                ndmi_array[ndmi_valid] = np.clip(ndmi_array[ndmi_valid],
                                    fire_hazard_hls_index_minimum, fire_hazard_hls_index_maximum)
                                # Write the processed data to the configured output resource.
                                ndvi_destination.write(ndvi_array, 1, window=source_window)
                                # Write the processed data to the configured output resource.
                                ndmi_destination.write(ndmi_array, 1, window=source_window)
                                # Calculate valid Fmask pixel count to quantify completeness and
                                # support QA checks.
                                valid_fmask_pixel_count += int(fmask_valid.sum())
                                # Calculate invalid Fmask pixel count to quantify completeness and
                                # support QA checks.
                                invalid_fmask_pixel_count += int((~fmask_valid).sum())
                                # Calculate invalid reflectance pixel count to quantify completeness
                                # and support QA checks.
                                invalid_reflectance_pixel_count += int(reflectance_invalid.sum())
                                # Calculate NDVI denominator zero count to quantify completeness and
                                # support QA checks.
                                ndvi_denominator_zero_count += int((base_valid & \
                                    ~ndvi_denominator_valid).sum())
                                # Calculate NDMI denominator zero count to quantify completeness and
                                # support QA checks.
                                ndmi_denominator_zero_count += int((base_valid & \
                                    ~ndmi_denominator_valid).sum())
                                # Prepare NDVI block values used to process the current raster
                                # window.
                                ndvi_block_values = ndvi_array[ndvi_valid]
                                # Prepare NDMI block values used to process the current raster
                                # window.
                                ndmi_block_values = ndmi_array[ndmi_valid]
                                # Calculate NDVI valid pixel count to quantify completeness and
                                # support QA checks.
                                ndvi_valid_pixel_count += int(ndvi_block_values.size)
                                # Calculate NDMI valid pixel count to quantify completeness and
                                # support QA checks.
                                ndmi_valid_pixel_count += int(ndmi_block_values.size)

                                # Handle the NDVI block values case explicitly during scene-level
                                # NDVI and NDMI.
                                if ndvi_block_values.size > 0:
                                    # Track block NDVI minimum across processed pixels for
                                    # output-range QA.
                                    block_ndvi_minimum = float(ndvi_block_values.min())
                                    # Track block NDVI maximum across processed pixels for
                                    # output-range QA.
                                    block_ndvi_maximum = float(ndvi_block_values.max())
                                    # Track NDVI minimum across processed pixels for output-range
                                    # QA.
                                    ndvi_minimum = block_ndvi_minimum if ndvi_minimum is None \
                                        else min(ndvi_minimum,
                                        block_ndvi_minimum)
                                    # Track NDVI maximum across processed pixels for output-range
                                    # QA.
                                    ndvi_maximum = block_ndvi_maximum if ndvi_maximum is None \
                                        else max(ndvi_maximum,
                                        block_ndvi_maximum)

                                # Handle the NDMI block values case explicitly during scene-level
                                # NDVI and NDMI.
                                if ndmi_block_values.size > 0:
                                    # Track block NDMI minimum across processed pixels for
                                    # output-range QA.
                                    block_ndmi_minimum = float(ndmi_block_values.min())
                                    # Track block NDMI maximum across processed pixels for
                                    # output-range QA.
                                    block_ndmi_maximum = float(ndmi_block_values.max())
                                    # Track NDMI minimum across processed pixels for output-range
                                    # QA.
                                    ndmi_minimum = block_ndmi_minimum if ndmi_minimum is None \
                                        else min(ndmi_minimum,
                                        block_ndmi_minimum)
                                    # Track NDMI maximum across processed pixels for output-range
                                    # QA.
                                    ndmi_maximum = block_ndmi_maximum if ndmi_maximum is None \
                                        else max(ndmi_maximum,
                                        block_ndmi_maximum)
                            ndvi_destination.set_band_description(1, 'HLS scene-level NDVI')
                            ndvi_destination.update_tags(COLLECTION_ID=collection_id,
                                ITEM_ID=item_id, INDEX_NAME='NDVI', \
                                    INDEX_FORMULA='(NIR - Red) / (NIR + Red)',
                                FMASK_APPLIED=True)
                            ndmi_destination.set_band_description(1, 'HLS scene-level NDMI')
                            ndmi_destination.update_tags(COLLECTION_ID=collection_id,
                                ITEM_ID=item_id, INDEX_NAME='NDMI', \
                                    INDEX_FORMULA='(NIR - SWIR1) / (NIR + SWIR1)',
                                FMASK_APPLIED=True)

    # Open the raster in a managed context so its file handle closes reliably after scene-level NDVI
    # and NDMI.
    with rasterio.open(ndvi_output_path) as ndvi_source:

        # Open the raster in a managed context so its file handle closes reliably after scene-level
        # NDVI and NDMI.
        with rasterio.open(ndmi_output_path) as ndmi_source:
            # Record NDVI structure valid so scene-level NDVI and NDMI can preserve a clear
            # processing and validation outcome.
            ndvi_structure_valid = ndvi_source.count == 1 and ndvi_source.dtypes[0] == \
                fire_hazard_hls_index_dtype and (ndvi_source.nodata == fire_hazard_hls_index_nodata)
            # Record NDMI structure valid so scene-level NDVI and NDMI can preserve a clear
            # processing and validation outcome.
            ndmi_structure_valid = ndmi_source.count == 1 and ndmi_source.dtypes[0] == \
                fire_hazard_hls_index_dtype and (ndmi_source.nodata == fire_hazard_hls_index_nodata)
            # Store index grids match needed to carry out scene-level NDVI and NDMI.
            index_grids_match = ndvi_source.width == ndmi_source.width and ndvi_source.height == \
                ndmi_source.height and (ndvi_source.crs == ndmi_source.crs) and \
                (ndvi_source.transform == ndmi_source.transform)
    # Record NDVI output valid so scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    ndvi_output_valid = ndvi_output_path.exists() and ndvi_output_path.is_file() and \
        (ndvi_output_path.stat().st_size > 0) and ndvi_structure_valid and index_grids_match and \
        (ndvi_valid_pixel_count > 0)
    # Record NDMI output valid so scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    ndmi_output_valid = ndmi_output_path.exists() and ndmi_output_path.is_file() and \
        (ndmi_output_path.stat().st_size > 0) and ndmi_structure_valid and index_grids_match and \
        (ndmi_valid_pixel_count > 0)
    fire_hazard_hls_scene_index_records.append({'COLLECTION_ID': collection_id,
        'ITEM_ID': item_id, 'INDEX_NAME': 'NDVI', 'RED_PATH': str(red_path),
        'NIR_PATH': str(nir_path), 'SWIR1_PATH': None,
        'VALID_MASK_PATH': str(valid_mask_path), 'OUTPUT_PATH': str(ndvi_output_path),
        'VALID_FMASK_PIXELS': valid_fmask_pixel_count,
        'INVALID_FMASK_PIXELS': invalid_fmask_pixel_count,
        'INVALID_REFLECTANCE_PIXELS': invalid_reflectance_pixel_count,
        'ZERO_DENOMINATOR_PIXELS': ndvi_denominator_zero_count,
        'VALID_INDEX_PIXELS': ndvi_valid_pixel_count, 'INDEX_MINIMUM': ndvi_minimum,
        'INDEX_MAXIMUM': ndvi_maximum, 'OUTPUT_SIZE_BYTES': ndvi_output_path.stat().st_size if \
            ndvi_output_path.exists() else 0,
        'STRUCTURE_VALID': ndvi_structure_valid, 'GRID_MATCHES_PAIRED_INDEX': index_grids_match,
        'VALID': ndvi_output_valid})
    fire_hazard_hls_scene_index_records.append({'COLLECTION_ID': collection_id,
        'ITEM_ID': item_id, 'INDEX_NAME': 'NDMI', 'RED_PATH': None,
        'NIR_PATH': str(nir_path), 'SWIR1_PATH': str(swir1_path),
        'VALID_MASK_PATH': str(valid_mask_path), 'OUTPUT_PATH': str(ndmi_output_path),
        'VALID_FMASK_PIXELS': valid_fmask_pixel_count,
        'INVALID_FMASK_PIXELS': invalid_fmask_pixel_count,
        'INVALID_REFLECTANCE_PIXELS': invalid_reflectance_pixel_count,
        'ZERO_DENOMINATOR_PIXELS': ndmi_denominator_zero_count,
        'VALID_INDEX_PIXELS': ndmi_valid_pixel_count, 'INDEX_MINIMUM': ndmi_minimum,
        'INDEX_MAXIMUM': ndmi_maximum, 'OUTPUT_SIZE_BYTES': ndmi_output_path.stat().st_size if \
            ndmi_output_path.exists() else 0,
        'STRUCTURE_VALID': ndmi_structure_valid, 'GRID_MATCHES_PAIRED_INDEX': index_grids_match,
        'VALID': ndmi_output_valid})
# Store scene index manifest needed to carry out scene-level NDVI and NDMI.
fire_hazard_hls_scene_index_manifest = \
    pd.DataFrame(fire_hazard_hls_scene_index_records).sort_values(['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME']).reset_index(drop=True)
# Calculate scene index counts to quantify completeness and support QA checks.
scene_index_counts = fire_hazard_hls_scene_index_manifest.groupby(['COLLECTION_ID',
    'ITEM_ID'])['INDEX_NAME'].nunique()
# Calculate invalid scene index counts to quantify completeness and support QA checks.
invalid_scene_index_counts = scene_index_counts[scene_index_counts != 2]
# Store failed HLS scene indices needed to carry out scene-level NDVI and NDMI.
failed_hls_scene_indices = \
    fire_hazard_hls_scene_index_manifest[~fire_hazard_hls_scene_index_manifest['VALID']]
# Record scene indices valid so scene-level NDVI and NDMI can preserve a clear processing and
# validation outcome.
fire_hazard_hls_scene_indices_valid = invalid_scene_index_counts.empty and \
    failed_hls_scene_indices.empty and (len(fire_hazard_hls_scene_index_manifest) == \
    expected_hls_index_scene_count * 2)

# Stop execution if one or more HLS scenes failed NDVI or NDMI calculation or output validation.
if not fire_hazard_hls_scene_indices_valid:
    raise ValueError('One or more HLS scenes failed NDVI or NDMI calculation or output validation.')
# Store total scene index size bytes needed to carry out scene-level NDVI and NDMI.
total_scene_index_size_bytes = int(fire_hazard_hls_scene_index_manifest['OUTPUT_SIZE_BYTES'].sum())
# Store scene index summary needed to carry out scene-level NDVI and NDMI.
fire_hazard_hls_scene_index_summary = pd.DataFrame([{'SCENES_PROCESSED': \
    expected_hls_index_scene_count,
    'NDVI_RASTERS': int((fire_hazard_hls_scene_index_manifest['INDEX_NAME'] == 'NDVI').sum()),
    'NDMI_RASTERS': int((fire_hazard_hls_scene_index_manifest['INDEX_NAME'] == 'NDMI').sum()),
    'TOTAL_INDEX_RASTERS': len(fire_hazard_hls_scene_index_manifest),
    'VALID_INDEX_RASTERS': int(fire_hazard_hls_scene_index_manifest['VALID'].sum()),
    'TOTAL_OUTPUT_SIZE_GB': total_scene_index_size_bytes / 1024 ** 3,
    'ALL_OUTPUTS_VALID': fire_hazard_hls_scene_indices_valid}])
# Save the scene index manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_scene_index_manifest.to_csv(fire_hazard_hls_scene_index_manifest_path, index=False)
# Save the scene index summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_scene_index_summary.to_csv(fire_hazard_hls_scene_index_summary_path, index=False)
# Calculate NDVI raster count to quantify completeness and support QA checks.
ndvi_raster_count = int((fire_hazard_hls_scene_index_manifest['INDEX_NAME'] == 'NDVI').sum())
# Calculate NDMI raster count to quantify completeness and support QA checks.
ndmi_raster_count = int((fire_hazard_hls_scene_index_manifest['INDEX_NAME'] == 'NDMI').sum())
print(f'-> HLS scenes processed: {expected_hls_index_scene_count:,}')
print(f'-> NDVI rasters created: {ndvi_raster_count:,}')
print(f'-> NDMI rasters created: {ndmi_raster_count:,}')
print(f'-> Total scene-level index rasters: {len(fire_hazard_hls_scene_index_manifest):,}')
print(f'-> Total scene-level index size: {total_scene_index_size_bytes / 1024 ** 3:,.2f} GB')
print(f'-> Scene-level indices valid: {fire_hazard_hls_scene_indices_valid}')
print(f'-> Index manifest saved: {fire_hazard_hls_scene_index_manifest_path}')
print('\n--- HLS SCENE-LEVEL INDEX SUMMARY ---')
display(fire_hazard_hls_scene_index_summary)
print('\n--- HLS SCENE-LEVEL INDEX SAMPLE ---')
display(fire_hazard_hls_scene_index_manifest[['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'VALID_FMASK_PIXELS',
    'ZERO_DENOMINATOR_PIXELS', 'VALID_INDEX_PIXELS',
    'INDEX_MINIMUM', 'INDEX_MAXIMUM', 'OUTPUT_SIZE_BYTES',
    'VALID']].head(5))
print('\nNOTE:')
print('The scene-level NDVI and NDMI rasters remain in their native HLS MGRS tile grids.')
print('The next step will clip, '
    'reproject, and align these '
    'rasters to the common 30-meter '
    'fire-hazard analysis grid.')
print('\n=== SCENE-LEVEL NDVI AND NDMI CALCULATION COMPLETE ===')


=== CALCULATING SCENE-LEVEL NDVI AND NDMI ===
-> Calculating indices for scene 1 of 65...
-> Calculating indices for scene 5 of 65...
-> Calculating indices for scene 10 of 65...
-> Calculating indices for scene 15 of 65...
-> Calculating indices for scene 20 of 65...
-> Calculating indices for scene 25 of 65...
-> Calculating indices for scene 30 of 65...
-> Calculating indices for scene 35 of 65...
-> Calculating indices for scene 40 of 65...
-> Calculating indices for scene 45 of 65...
-> Calculating indices for scene 50 of 65...
-> Calculating indices for scene 55 of 65...
-> Calculating indices for scene 60 of 65...
-> Calculating indices for scene 65 of 65...
-> HLS scenes processed: 65
-> NDVI rasters created: 65
-> NDMI rasters created: 65
-> Total scene-level index rasters: 130
-> Total scene-level index size: 3.40 GB
-> Scene-level indices valid: True
-> Index manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\v

,SCENES_PROCESSED,NDVI_RASTERS,NDMI_RASTERS,TOTAL_INDEX_RASTERS,VALID_INDEX_RASTERS,TOTAL_OUTPUT_SIZE_GB,ALL_OUTPUTS_VALID
0,65,65,65,130,130,3.401912,True



--- HLS SCENE-LEVEL INDEX SAMPLE ---


,COLLECTION_ID,ITEM_ID,INDEX_NAME,VALID_FMASK_PIXELS,ZERO_DENOMINATOR_PIXELS,VALID_INDEX_PIXELS,INDEX_MINIMUM,INDEX_MAXIMUM,OUTPUT_SIZE_BYTES,VALID
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDMI,11874391,0,11874391,-1.000000,0.789287,37918494,True
1,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDVI,11874391,0,11874391,-1.000000,1.000000,36329356,True
2,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDMI,6284644,0,6284644,-0.526205,1.000000,21709012,True
3,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDVI,6284644,0,6284644,-0.337455,0.937775,20319688,True
4,hls2-l30,HLS.L30.T11SQB.2025224T181513.v2.0,NDMI,13092809,0,13092809,-1.000000,1.000000,42966892,True



NOTE:
The scene-level NDVI and NDMI rasters remain in their native HLS MGRS tile grids.
The next step will clip, reproject, and align these rasters to the common 30-meter fire-hazard analysis grid.

=== SCENE-LEVEL NDVI AND NDMI CALCULATION COMPLETE ===


## HLS Alignment and Scene-Level Validation


### Defining HLS Scene Alignment Parameters


In [79]:
print('=== DEFINING HLS SCENE ALIGNMENT PARAMETERS ===')
# List the required HLS alignment prerequisites required before HLS scene alignment
# parameters can run.
required_hls_alignment_inputs = ['gdf_county_boundaries_aligned',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_nodata_value', 'fire_hazard_raster_dtype',
    'fire_hazard_hls_scene_index_manifest', 'fire_hazard_hls_scene_index_directory',
    'fire_hazard_hls_raw_directory']
# Identify unavailable HLS alignment so HLS scene alignment parameters stops before using
# incomplete inputs.
missing_hls_alignment_inputs = [object_name for object_name in required_hls_alignment_inputs if \
    object_name not in globals()]

# Transform vegetation condition into a normalized dryness score
# where larger values represent greater fire potential.
# Stop execution when required hls alignment inputs inputs are unavailable.
if missing_hls_alignment_inputs:
    raise NameError(f'The following HLS alignment '
        f'objects are missing:\n'
        f'{missing_hls_alignment_inputs}\n\n'
        f'Complete the previous '
        f'vegetation-dryness processing '
        f'steps before defining the '
        f'common analysis grid.')
# Build the aligned directory location so HLS scene alignment parameters uses the expected project
# file structure.
fire_hazard_hls_aligned_directory = fire_hazard_hls_raw_directory / 'aligned_scene_indices'

# Use NDVI to represent live vegetation density and potential fuel availability.
# Build the aligned NDVI directory location so HLS scene alignment parameters uses the expected
# project file structure.
fire_hazard_hls_aligned_ndvi_directory = fire_hazard_hls_aligned_directory / 'NDVI'

# Use NDMI to represent vegetation and canopy moisture conditions.
# Build the aligned NDMI directory location so HLS scene alignment parameters uses the expected
# project file structure.
fire_hazard_hls_aligned_ndmi_directory = fire_hazard_hls_aligned_directory / 'NDMI'
# Build the alignment manifest directory location so HLS scene alignment parameters uses the
# expected project file structure.
fire_hazard_hls_alignment_manifest_directory = fire_hazard_hls_aligned_directory / 'metadata'

# Process each directory path entry so HLS scene alignment parameters is applied consistently across
# all records.
for directory_path in [fire_hazard_hls_aligned_directory,
    fire_hazard_hls_aligned_ndvi_directory, fire_hazard_hls_aligned_ndmi_directory,
    fire_hazard_hls_alignment_manifest_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the alignment manifest path location so HLS scene alignment parameters uses the expected
# project file structure.
fire_hazard_hls_alignment_manifest_path = fire_hazard_hls_alignment_manifest_directory / \
    'hls_alignment_manifest.csv'
# Build the alignment summary path location so HLS scene alignment parameters uses the expected
# project file structure.
fire_hazard_hls_alignment_summary_path = fire_hazard_hls_alignment_manifest_directory / \
    'hls_alignment_summary.csv'
# Build the alignment policy path location so HLS scene alignment parameters uses the expected
# project file structure.
fire_hazard_hls_alignment_policy_path = fire_hazard_hls_alignment_manifest_directory / \
    'hls_alignment_policy.json'
# Store alignment bounds needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_bounds = gdf_county_boundaries_aligned.total_bounds
# Store intermediate value needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_min_x, fire_hazard_alignment_min_y, fire_hazard_alignment_max_x, \
    fire_hazard_alignment_max_y = fire_hazard_alignment_bounds
# Store alignment min x needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_min_x = np.floor(fire_hazard_alignment_min_x / fire_hazard_cell_size) * \
    fire_hazard_cell_size
# Store alignment min y needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_min_y = np.floor(fire_hazard_alignment_min_y / fire_hazard_cell_size) * \
    fire_hazard_cell_size
# Store alignment max x needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_max_x = np.ceil(fire_hazard_alignment_max_x / fire_hazard_cell_size) * \
    fire_hazard_cell_size
# Store alignment max y needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_max_y = np.ceil(fire_hazard_alignment_max_y / fire_hazard_cell_size) * \
    fire_hazard_cell_size
# Store alignment width needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_width = int((fire_hazard_alignment_max_x - fire_hazard_alignment_min_x) / \
    fire_hazard_cell_size)
# Store alignment height needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_height = int((fire_hazard_alignment_max_y - fire_hazard_alignment_min_y) / \
    fire_hazard_cell_size)
# Store alignment transform needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_transform = rasterio.transform.from_origin(west=fire_hazard_alignment_min_x,
    north=fire_hazard_alignment_max_y, xsize=fire_hazard_cell_size,
    ysize=fire_hazard_cell_size)
# Prepare alignment profile to preserve and validate raster structure during HLS scene alignment
# parameters.
fire_hazard_alignment_profile = {'driver': 'GTiff',
    'dtype': fire_hazard_raster_dtype, 'nodata': fire_hazard_nodata_value,
    'count': 1, 'width': fire_hazard_alignment_width,
    'height': fire_hazard_alignment_height, 'crs': fire_hazard_target_crs,
    'transform': fire_hazard_alignment_transform, 'compress': 'deflate',
    'predictor': 3, 'BIGTIFF': 'IF_SAFER'}
# Store alignment resampling method needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_resampling_method = Resampling.bilinear
# Define alignment policy to control the inputs and rules used by HLS scene alignment parameters.
fire_hazard_alignment_policy = {'target_crs': fire_hazard_target_crs,
    'cell_size': fire_hazard_cell_size, 'bounds': [fire_hazard_alignment_min_x,
    fire_hazard_alignment_min_y, fire_hazard_alignment_max_x,
    fire_hazard_alignment_max_y], 'width': fire_hazard_alignment_width,
    'height': fire_hazard_alignment_height, 'resampling': 'bilinear',
    'compression': 'deflate', 'nodata': fire_hazard_nodata_value}

# Open the file in a managed context so its handle closes reliably after HLS scene alignment
# parameters.
with open(fire_hazard_hls_alignment_policy_path, 'w', encoding='utf-8') as file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_alignment_policy, file, indent=2)
# Store alignment summary needed to carry out HLS scene alignment parameters.
fire_hazard_alignment_summary = pd.DataFrame([{'TARGET_CRS': fire_hazard_target_crs,
    'CELL_SIZE': fire_hazard_cell_size, 'WIDTH': fire_hazard_alignment_width,
    'HEIGHT': fire_hazard_alignment_height, 'RESAMPLING': 'Bilinear',
    'OUTPUT_DTYPE': fire_hazard_raster_dtype, 'OUTPUT_NODATA': fire_hazard_nodata_value}])
print(f'-> Target CRS: {fire_hazard_target_crs}')
print(f'-> Cell size: {fire_hazard_cell_size} meters')
print(f'-> Raster size: {fire_hazard_alignment_width:,} x {fire_hazard_alignment_height:,}')
print(f'-> Resampling: Bilinear')
print(f'-> Alignment policy saved: {fire_hazard_hls_alignment_policy_path}')
print('\n--- HLS ALIGNMENT PARAMETERS ---')
display(fire_hazard_alignment_summary)
print('\nNOTE:')
print('All scene-level NDVI and NDMI '
    'rasters will be reprojected, '
    'clipped, and aligned to this '
    'common 30-meter fire-hazard '
    'analysis grid.')
print('\n=== HLS ALIGNMENT PARAMETERS DEFINED ===')


=== DEFINING HLS SCENE ALIGNMENT PARAMETERS ===
-> Target CRS: EPSG:26912
-> Cell size: 30 meters
-> Raster size: 9,454 x 14,467
-> Resampling: Bilinear
-> Alignment policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\aligned_scene_indices\metadata\hls_alignment_policy.json

--- HLS ALIGNMENT PARAMETERS ---


,TARGET_CRS,CELL_SIZE,WIDTH,HEIGHT,RESAMPLING,OUTPUT_DTYPE,OUTPUT_NODATA
0,EPSG:26912,30,9454,14467,Bilinear,float32,-9999.0



NOTE:
All scene-level NDVI and NDMI rasters will be reprojected, clipped, and aligned to this common 30-meter fire-hazard analysis grid.

=== HLS ALIGNMENT PARAMETERS DEFINED ===


### Preparing HLS Alignment Workspace


In [80]:
print('=== PREPARING HLS ALIGNMENT WORKSPACE ===')
# List the required HLS alignment workspace prerequisites required before HLS alignment
# workspace can run.
required_hls_alignment_workspace_inputs = ['fire_hazard_hls_scene_index_manifest',
    'gdf_county_boundaries_aligned', 'fire_hazard_target_crs',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_profile',
    'fire_hazard_alignment_resampling_method', 'fire_hazard_alignment_min_x',
    'fire_hazard_alignment_min_y', 'fire_hazard_alignment_max_x',
    'fire_hazard_alignment_max_y', 'fire_hazard_nodata_value',
    'fire_hazard_raster_dtype', 'fire_hazard_hls_aligned_ndvi_directory',
    'fire_hazard_hls_aligned_ndmi_directory', 'fire_hazard_hls_alignment_manifest_path',
    'fire_hazard_hls_alignment_summary_path']
# Identify unavailable HLS alignment workspace so HLS alignment workspace stops before
# using incomplete inputs.
missing_hls_alignment_workspace_inputs = [object_name for object_name in \
    required_hls_alignment_workspace_inputs if object_name not in globals()]

# Stop execution when required hls alignment workspace inputs inputs are unavailable.
if missing_hls_alignment_workspace_inputs:
    raise NameError(f'The following HLS '
        f'alignment-workspace objects '
        f'are missing:\n'
        f'{missing_hls_alignment_workspace_inputs}\n\n'
        f'Run step Calculate Scene-Level '
        f'NDVI and NDMI and step Define '
        f'HLS Alignment Parameters '
        f'before preparing the alignment '
        f'workspace.')
from rasterio.features import geometry_mask
from rasterio.warp import reproject
from shapely.geometry import mapping

# Stop execution if scene-level HLS NDVI and NDMI outputs have not passed complete validation.
if 'fire_hazard_hls_scene_indices_valid' not in globals() or not \
    fire_hazard_hls_scene_indices_valid:
    raise ValueError('The scene-level HLS NDVI and '
        'NDMI outputs have not passed '
        'complete validation.')
# List the required HLS scene index fields prerequisites required before HLS alignment workspace can
# run.
required_hls_scene_index_fields = ['COLLECTION_ID', 'ITEM_ID', 'INDEX_NAME', 'OUTPUT_PATH', 'VALID']
# Identify unavailable HLS scene index fields so HLS alignment workspace stops before using
# incomplete inputs.
missing_hls_scene_index_fields = [field_name for field_name in required_hls_scene_index_fields if \
    field_name not in fire_hazard_hls_scene_index_manifest.columns]

# Stop execution when required hls scene index fields inputs are unavailable.
if missing_hls_scene_index_fields:
    raise ValueError(f'The HLS scene-index manifest '
        f'is missing required alignment '
        f'fields:\n'
        f'{missing_hls_scene_index_fields}')

# Stop execution if hLS scene-index manifest contains no raster records.
if fire_hazard_hls_scene_index_manifest.empty:
    raise ValueError('The HLS scene-index manifest contains no raster records.')

# Stop execution if one or more scene-level NDVI or NDMI rasters are marked invalid.
if not fire_hazard_hls_scene_index_manifest['VALID'].all():
    raise ValueError('One or more scene-level NDVI or NDMI rasters are marked invalid.')
# Store alignment sources needed to carry out HLS alignment workspace.
fire_hazard_hls_alignment_sources = \
    fire_hazard_hls_scene_index_manifest.copy().reset_index(drop=True)
# Store alignment sources['index name'] needed to carry out HLS alignment workspace.
fire_hazard_hls_alignment_sources['INDEX_NAME'] = \
    fire_hazard_hls_alignment_sources['INDEX_NAME'].astype(str).str.strip().str.upper()
# Record valid HLS alignment index names so HLS alignment workspace can preserve a clear processing
# and validation outcome.
valid_hls_alignment_index_names = {'NDVI', 'NDMI'}
# Record invalid HLS alignment index names so HLS alignment workspace can preserve a clear
# processing and validation outcome.
invalid_hls_alignment_index_names = \
    sorted(set(fire_hazard_hls_alignment_sources['INDEX_NAME'].unique()) - \
    valid_hls_alignment_index_names)

# Stop execution if scene-index manifest contains unsupported index names: the reported value.
if invalid_hls_alignment_index_names:
    raise ValueError(f'The scene-index manifest '
        f'contains unsupported index '
        f'names:\n'
        f'{invalid_hls_alignment_index_names}')
# Calculate scene index counts to quantify completeness and support QA checks.
fire_hazard_hls_scene_index_counts = fire_hazard_hls_alignment_sources.groupby(['COLLECTION_ID',
    'ITEM_ID'])['INDEX_NAME'].nunique()
# Calculate invalid HLS scene index counts to quantify completeness and support QA checks.
invalid_hls_scene_index_counts = \
    fire_hazard_hls_scene_index_counts[fire_hazard_hls_scene_index_counts != 2]

# Stop execution if one or more HLS scenes do not contain exactly one NDVI and one NDMI raster.
if not invalid_hls_scene_index_counts.empty:
    raise ValueError('One or more HLS scenes do not contain exactly one NDVI and one NDMI raster.')
# Build duplicate hls alignment records used to track the records included in this processing stage.
duplicate_hls_alignment_records = \
    fire_hazard_hls_alignment_sources.duplicated(subset=['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME'], keep=False)

# Stop execution if scene-index manifest contains the reported value duplicate collection-item-index
# records.
if duplicate_hls_alignment_records.any():
    # Identify duplicate duplicate alignment record count so repeated records do not bias HLS
    # alignment workspace.
    duplicate_alignment_record_count = int(duplicate_hls_alignment_records.sum())
    raise ValueError(f'The scene-index manifest '
        f'contains '
        f'{duplicate_alignment_record_count} '
        f'duplicate '
        f'collection-item-index records.')
# Identify unavailable HLS alignment source paths so HLS alignment workspace stops before using
# incomplete inputs.
missing_hls_alignment_source_paths = []
# Record invalid HLS alignment source extensions so HLS alignment workspace can preserve a clear
# processing and validation outcome.
invalid_hls_alignment_source_extensions = []

# Process each ( entry so HLS alignment workspace is applied consistently across all records.
for _, source_record in fire_hazard_hls_alignment_sources.iterrows():
    # Build the source index path location so HLS alignment workspace uses the expected project file
    # structure.
    source_index_path = Path(source_record['OUTPUT_PATH'])

    # Use the existing file only when it is present and valid for HLS alignment workspace.
    if not source_index_path.exists() or not source_index_path.is_file():
        missing_hls_alignment_source_paths.append(str(source_index_path))

    # Handle the source index path case explicitly during HLS alignment workspace.
    if source_index_path.suffix.lower() not in {'.tif', '.tiff'}:
        invalid_hls_alignment_source_extensions.append(str(source_index_path))

# Stop execution when required hls alignment source paths inputs are unavailable.
if missing_hls_alignment_source_paths:
    raise FileNotFoundError(f'One or more scene-level NDVI '
        f'or NDMI source rasters are '
        f'missing.\n\nFirst missing '
        f'paths:\n'
        f'{missing_hls_alignment_source_paths[:20]}')

# Stop execution if one or more scene-level index files do not use a GeoTIFF extension: the reported
# value.
if invalid_hls_alignment_source_extensions:
    raise ValueError(f'One or more scene-level index '
        f'files do not use a GeoTIFF '
        f'extension:\n'
        f'{invalid_hls_alignment_source_extensions[:20]}')

# Stop execution if aligned county-boundary GeoDataFrame contains no study-area features.
if gdf_county_boundaries_aligned.empty:
    raise ValueError('The aligned county-boundary GeoDataFrame contains no study-area features.')

# Stop execution if aligned county-boundary GeoDataFrame contains no usable geometries.
if gdf_county_boundaries_aligned.geometry.isna().all():
    raise ValueError('The aligned county-boundary GeoDataFrame contains no usable geometries.')

# Stop execution if aligned county-boundary GeoDataFrame does not define a coordinate reference
# system.
if gdf_county_boundaries_aligned.crs is None:
    raise ValueError('The aligned county-boundary '
        'GeoDataFrame does not define a '
        'coordinate reference system.')

# Stop execution if county-boundary CRS does not match the fire-hazard target CRS.
if gdf_county_boundaries_aligned.crs.to_string() != fire_hazard_target_crs:
    raise ValueError(f'The county-boundary CRS does '
        f'not match the fire-hazard '
        f'target CRS.\nCounty CRS: '
        f'{gdf_county_boundaries_aligned.crs}\n'
        f'Target CRS: '
        f'{fire_hazard_target_crs}')
# Prepare alignment study area geometry to define the spatial extent used by HLS alignment
# workspace.
fire_hazard_alignment_study_area_geometry = (
    gdf_county_boundaries_aligned.geometry.union_all()
)

# Stop execution if combined fire-hazard study-area geometry is empty.
if fire_hazard_alignment_study_area_geometry is None or \
    fire_hazard_alignment_study_area_geometry.is_empty:
    raise ValueError('The combined fire-hazard study-area geometry is empty.')

# Handle missing or invalid alignment study area geometry explicitly during HLS alignment workspace.
if not fire_hazard_alignment_study_area_geometry.is_valid:
    # Prepare alignment study area geometry to define the spatial extent used by HLS alignment
    # workspace.
    fire_hazard_alignment_study_area_geometry = fire_hazard_alignment_study_area_geometry.buffer(0)

# Stop execution if combined study-area geometry could not be repaired for raster masking.
if fire_hazard_alignment_study_area_geometry.is_empty or not \
    fire_hazard_alignment_study_area_geometry.is_valid:
    raise ValueError('The combined study-area geometry could not be repaired for raster masking.')
# Define alignment study area shapes to control the inputs and rules used by HLS alignment
# workspace.
fire_hazard_alignment_study_area_shapes = [mapping(fire_hazard_alignment_study_area_geometry)]
# Build the fire hazard alignment study area mask mask used to isolate records required for this
# analysis.
fire_hazard_alignment_study_area_mask = \
    geometry_mask(geometries=fire_hazard_alignment_study_area_shapes,
    out_shape=(fire_hazard_alignment_height, fire_hazard_alignment_width),
    transform=fire_hazard_alignment_transform, invert=True,
    all_touched=False)
# Calculate alignment study area pixel count to quantify completeness and support QA checks.
fire_hazard_alignment_study_area_pixel_count = int(fire_hazard_alignment_study_area_mask.sum())
# Calculate alignment total pixel count to quantify completeness and support QA checks.
fire_hazard_alignment_total_pixel_count = int(fire_hazard_alignment_study_area_mask.size)
# Calculate alignment outside pixel count to quantify completeness and support QA checks.
fire_hazard_alignment_outside_pixel_count = fire_hazard_alignment_total_pixel_count - \
    fire_hazard_alignment_study_area_pixel_count

# Stop execution if target-grid study-area mask contains no included raster cells.
if fire_hazard_alignment_study_area_pixel_count == 0:
    raise ValueError('The target-grid study-area mask contains no included raster cells.')

# Encapsulate build HLS aligned index path so repeated HLS alignment workspace steps use consistent
# logic.
def build_hls_aligned_index_path(collection_id, item_id, index_name):

    """
    Creates the target path for one clipped,
    reprojected, and aligned NDVI or NDMI raster.
    """
    # Sanitize safe collection ID so it is safe for filenames, identifiers, and manifest records.
    safe_collection_id = re.sub('[^A-Za-z0-9._-]+', '_', str(collection_id))
    # Sanitize safe item ID so it is safe for filenames, identifiers, and manifest records.
    safe_item_id = re.sub('[^A-Za-z0-9._-]+', '_', str(item_id))
    # Sanitize safe index name so it is safe for filenames, identifiers, and manifest records.
    safe_index_name = re.sub('[^A-Za-z0-9._-]+', '_', str(index_name).lower())

    # Handle the safe index name case explicitly during HLS alignment workspace.
    if safe_index_name == 'ndvi':
        # Build the index directory location so HLS alignment workspace uses the expected project
        # file structure.
        index_directory = fire_hazard_hls_aligned_ndvi_directory
    # Handle the safe index name case explicitly during HLS alignment workspace.
    elif safe_index_name == 'ndmi':
        # Build the index directory location so HLS alignment workspace uses the expected project
        # file structure.
        index_directory = fire_hazard_hls_aligned_ndmi_directory
    else:
        raise ValueError(f'Unsupported HLS alignment index name: {index_name}')
    # Build the collection directory location so HLS alignment workspace uses the expected project
    # file structure.
    collection_directory = index_directory / safe_collection_id
    # Create the output directory before writing workflow products.
    collection_directory.mkdir(parents=True, exist_ok=True)
    return collection_directory / f'{safe_item_id}_{safe_index_name}_aligned.tif'
# Build the alignment sources['aligned path'] location so HLS alignment workspace uses the expected
# project file structure.
fire_hazard_hls_alignment_sources['ALIGNED_PATH'] = \
    [str(build_hls_aligned_index_path(collection_id=source_record['COLLECTION_ID'],
    item_id=source_record['ITEM_ID'], index_name=source_record['INDEX_NAME'])) for _,
    source_record in fire_hazard_hls_alignment_sources.iterrows()]
# Store alignment sources['aligned output exists'] needed to carry out HLS alignment workspace.
fire_hazard_hls_alignment_sources['ALIGNED_OUTPUT_EXISTS'] = [Path(aligned_path).exists() and \
    Path(aligned_path).is_file() and (Path(aligned_path).stat().st_size > 0) for aligned_path in \
    fire_hazard_hls_alignment_sources['ALIGNED_PATH']]
# Identify duplicate duplicate HLS aligned paths so repeated records do not bias HLS alignment
# workspace.
duplicate_hls_aligned_paths = fire_hazard_hls_alignment_sources.duplicated(subset=['ALIGNED_PATH'],
    keep=False)

# Stop execution if hLS alignment plan contains duplicate aligned output paths.
if duplicate_hls_aligned_paths.any():
    raise ValueError('The HLS alignment plan contains duplicate aligned output paths.')
# Set alignment progress interval to control HLS alignment workspace.
fire_hazard_hls_alignment_progress_interval = 5
# Store alignment overwrite needed to carry out HLS alignment workspace.
fire_hazard_hls_alignment_overwrite = globals().get('fire_hazard_data_mode', 'snapshot') == 'refresh'
# Store alignment reuse existing needed to carry out HLS alignment workspace.
fire_hazard_hls_alignment_reuse_existing = not fire_hazard_hls_alignment_overwrite
# Track alignment minimum valid pixels across processed pixels for output-range QA.
fire_hazard_hls_alignment_minimum_valid_pixels = 1
# Define alignment records to control the inputs and rules used by HLS alignment workspace.
fire_hazard_hls_alignment_records = []
# Define alignment error records to control the inputs and rules used by HLS alignment workspace.
fire_hazard_hls_alignment_error_records = []
# Calculate alignment source count to quantify completeness and support QA checks.
fire_hazard_hls_alignment_source_count = len(fire_hazard_hls_alignment_sources)
# Calculate alignment scene count to quantify completeness and support QA checks.
fire_hazard_hls_alignment_scene_count = fire_hazard_hls_alignment_sources[['COLLECTION_ID',
    'ITEM_ID']].drop_duplicates().shape[0]
# Calculate alignment NDVI count to quantify completeness and support QA checks.
fire_hazard_hls_alignment_ndvi_count = int((fire_hazard_hls_alignment_sources['INDEX_NAME'] == \
    'NDVI').sum())
# Calculate alignment NDMI count to quantify completeness and support QA checks.
fire_hazard_hls_alignment_ndmi_count = int((fire_hazard_hls_alignment_sources['INDEX_NAME'] == \
    'NDMI').sum())
# Calculate existing HLS aligned output count to quantify completeness and support QA checks.
existing_hls_aligned_output_count = \
    int(fire_hazard_hls_alignment_sources['ALIGNED_OUTPUT_EXISTS'].sum())
# Store alignment workspace summary needed to carry out HLS alignment workspace.
fire_hazard_hls_alignment_workspace_summary = pd.DataFrame([{'HLS_SCENES': \
    fire_hazard_hls_alignment_scene_count,
    'SOURCE_INDEX_RASTERS': fire_hazard_hls_alignment_source_count,
    'NDVI_RASTERS': fire_hazard_hls_alignment_ndvi_count,
    'NDMI_RASTERS': fire_hazard_hls_alignment_ndmi_count,
    'EXISTING_ALIGNED_OUTPUTS': existing_hls_aligned_output_count,
    'TARGET_CRS': fire_hazard_target_crs, 'TARGET_WIDTH': fire_hazard_alignment_width,
    'TARGET_HEIGHT': fire_hazard_alignment_height,
    'STUDY_AREA_PIXELS': fire_hazard_alignment_study_area_pixel_count,
    'OUTSIDE_STUDY_AREA_PIXELS': fire_hazard_alignment_outside_pixel_count,
    'REUSE_EXISTING': fire_hazard_hls_alignment_reuse_existing,
    'OVERWRITE_EXISTING': fire_hazard_hls_alignment_overwrite}])
print(f'-> HLS scenes prepared: {fire_hazard_hls_alignment_scene_count:,}')
print(f'-> Scene-level index rasters prepared: {fire_hazard_hls_alignment_source_count:,}')
print(f'-> NDVI rasters prepared: {fire_hazard_hls_alignment_ndvi_count:,}')
print(f'-> NDMI rasters prepared: {fire_hazard_hls_alignment_ndmi_count:,}')
print(f'-> Existing aligned outputs: {existing_hls_aligned_output_count:,}')
print(f'-> Target-grid study-area pixels: {fire_hazard_alignment_study_area_pixel_count:,}')
print(f'-> Reuse existing outputs: {fire_hazard_hls_alignment_reuse_existing}')
print(f'-> Overwrite existing outputs: {fire_hazard_hls_alignment_overwrite}')
print('\n--- HLS ALIGNMENT WORKSPACE SUMMARY ---')
display(fire_hazard_hls_alignment_workspace_summary)
print('\n--- HLS ALIGNMENT PROCESSING PLAN ---')
display(fire_hazard_hls_alignment_sources[['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'OUTPUT_PATH', 'ALIGNED_PATH',
    'ALIGNED_OUTPUT_EXISTS']].head(5))
print('\nNOTE:')
print('The study-area mask was '
    'created on the common 30-meter '
    'target grid. The next cell '
    'will reproject each native HLS '
    'index raster directly onto '
    'this grid and set all pixels '
    'outside the three counties to '
    'NoData.')
print('\n=== HLS ALIGNMENT WORKSPACE READY ===')



=== PREPARING HLS ALIGNMENT WORKSPACE ===
-> HLS scenes prepared: 65
-> Scene-level index rasters prepared: 130
-> NDVI rasters prepared: 65
-> NDMI rasters prepared: 65
-> Existing aligned outputs: 126
-> Target-grid study-area pixels: 15,475,232
-> Reuse existing outputs: True
-> Overwrite existing outputs: False

--- HLS ALIGNMENT WORKSPACE SUMMARY ---


,HLS_SCENES,SOURCE_INDEX_RASTERS,NDVI_RASTERS,NDMI_RASTERS,EXISTING_ALIGNED_OUTPUTS,TARGET_CRS,TARGET_WIDTH,TARGET_HEIGHT,STUDY_AREA_PIXELS,OUTSIDE_STUDY_AREA_PIXELS,REUSE_EXISTING,OVERWRITE_EXISTING
0,65,130,65,65,126,EPSG:26912,9454,14467,15475232,121295786,True,False



--- HLS ALIGNMENT PROCESSING PLAN ---


,COLLECTION_ID,ITEM_ID,INDEX_NAME,OUTPUT_PATH,ALIGNED_PATH,ALIGNED_OUTPUT_EXISTS
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
1,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDVI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
2,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
3,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDVI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
4,hls2-l30,HLS.L30.T11SQB.2025224T181513.v2.0,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



NOTE:
The study-area mask was created on the common 30-meter target grid. The next cell will reproject each native HLS index raster directly onto this grid and set all pixels outside the three counties to NoData.

=== HLS ALIGNMENT WORKSPACE READY ===


### Filtering HLS Sources by Valid Study-Area Coverage


In [81]:
print('=== FILTERING HLS SOURCES BY VALID STUDY-AREA COVERAGE ===')
# List the required HLS coverage filter prerequisites required before HLS sources by valid
# study-area coverage can run.
required_hls_coverage_filter_inputs = ['fire_hazard_hls_alignment_sources',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_target_crs', 'fire_hazard_hls_alignment_manifest_directory']
# Identify unavailable HLS coverage filter so HLS sources by valid study-area coverage
# stops before using incomplete inputs.
missing_hls_coverage_filter_inputs = [object_name for object_name in \
    required_hls_coverage_filter_inputs if object_name not in globals()]

# Stop execution when required hls coverage filter inputs inputs are unavailable.
if missing_hls_coverage_filter_inputs:
    raise NameError(f'The following HLS '
        f'coverage-filter objects are '
        f'missing:\n'
        f'{missing_hls_coverage_filter_inputs}\n\n'
        f'Run Prepare HLS Alignment '
        f'Workspace first.')
# Define alignment coverage records to control the inputs and rules used by HLS sources by valid
# study-area coverage.
fire_hazard_hls_alignment_coverage_records = []
# Store total coverage sources needed to carry out HLS sources by valid study-area coverage.
total_coverage_sources = len(fire_hazard_hls_alignment_sources)

# Reproject the raster onto the common 30-meter project grid so
# every hazard component aligns cell by cell.
# Process each (source number entry so HLS sources by valid study-area coverage is applied
# consistently across all records.
for source_number, (_, source_record) in enumerate(fire_hazard_hls_alignment_sources.iterrows(),
    start=1):

    # Handle the source number case explicitly during HLS sources by valid study-area coverage.
    if source_number == 1 or source_number % 10 == 0 or source_number == total_coverage_sources:
        print(f'-> Checking source coverage {source_number:,} of {total_coverage_sources:,}...')
    # Build the source path location so HLS sources by valid study-area coverage uses the expected
    # project file structure.
    source_path = Path(source_record['OUTPUT_PATH'])

    # Open the raster in a managed context so its file handle closes reliably after HLS sources by
    # valid study-area coverage.
    with rasterio.open(source_path) as source_raster:
        # Store source NoData needed to carry out HLS sources by valid study-area coverage.
        source_nodata = source_raster.nodata
        # Prepare source array used to process the current raster window.
        source_array = source_raster.read(1)
        # Build the source valid mask mask used to isolate records required for this analysis.
        source_valid_mask = np.isfinite(source_array) & (source_array != source_nodata)
        # Build the target valid mask mask used to isolate records required for this analysis.
        target_valid_mask = np.zeros((fire_hazard_alignment_height,
            fire_hazard_alignment_width), dtype=np.uint8)
        reproject(source=source_valid_mask.astype(np.uint8),
            destination=target_valid_mask, src_transform=source_raster.transform,
            src_crs=source_raster.crs, src_nodata=0, dst_transform=fire_hazard_alignment_transform,
            dst_crs=fire_hazard_target_crs, dst_nodata=0, resampling=Resampling.nearest,
            init_dest_nodata=True)
    # Record valid target pixels so HLS sources by valid study-area coverage can preserve a clear
    # processing and validation outcome.
    valid_target_pixels = int((target_valid_mask == 1).sum())
    # Record valid study area pixels so HLS sources by valid study-area coverage can preserve a
    # clear processing and validation outcome.
    valid_study_area_pixels = int(((target_valid_mask == 1) & \
        fire_hazard_alignment_study_area_mask).sum())
    # Store contributes to study area needed to carry out HLS sources by valid study-area coverage.
    contributes_to_study_area = valid_study_area_pixels > 0
    fire_hazard_hls_alignment_coverage_records.append({'COLLECTION_ID': \
        source_record['COLLECTION_ID'],
        'ITEM_ID': source_record['ITEM_ID'], 'INDEX_NAME': source_record['INDEX_NAME'],
        'OUTPUT_PATH': source_record['OUTPUT_PATH'], 'VALID_PIXELS_ON_TARGET_GRID': \
            valid_target_pixels,
        'VALID_PIXELS_INSIDE_STUDY_AREA': valid_study_area_pixels,
        'CONTRIBUTES_TO_STUDY_AREA': contributes_to_study_area})
# Store alignment coverage needed to carry out HLS sources by valid study-area coverage.
fire_hazard_hls_alignment_coverage = pd.DataFrame(fire_hazard_hls_alignment_coverage_records)
# Store alignment sources needed to carry out HLS sources by valid study-area coverage.
fire_hazard_hls_alignment_sources = \
    fire_hazard_hls_alignment_sources.merge(fire_hazard_hls_alignment_coverage[['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'VALID_PIXELS_ON_TARGET_GRID',
    'VALID_PIXELS_INSIDE_STUDY_AREA', 'CONTRIBUTES_TO_STUDY_AREA']],
    on=['COLLECTION_ID', 'ITEM_ID', 'INDEX_NAME'],
    how='left', validate='one_to_one')
# Store alignment exclusions needed to carry out HLS sources by valid study-area coverage.
fire_hazard_hls_alignment_exclusions = \
    fire_hazard_hls_alignment_sources[~fire_hazard_hls_alignment_sources \
    ['CONTRIBUTES_TO_STUDY_AREA'].fillna(False)].copy().reset_index(drop=True)
# Set alignment exclusions['exclusion reason'] to control HLS sources by valid study-area coverage.
fire_hazard_hls_alignment_exclusions['EXCLUSION_REASON'] = (
    'No valid NDVI or NDMI pixels '
        'fall inside the three-county '
        'study-area mask after '
        'reprojection.'
)
# Build the alignment exclusion path location so HLS sources by valid study-area coverage uses the
# expected project file structure.
fire_hazard_hls_alignment_exclusion_path = fire_hazard_hls_alignment_manifest_directory / \
    'hls_alignment_exclusions.csv'
# Save the alignment exclusions CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_alignment_exclusions.to_csv(fire_hazard_hls_alignment_exclusion_path, index=False)
# Store alignment sources needed to carry out HLS sources by valid study-area coverage.
fire_hazard_hls_alignment_sources = \
    fire_hazard_hls_alignment_sources[fire_hazard_hls_alignment_sources \
    ['CONTRIBUTES_TO_STUDY_AREA'].fillna(False)].copy().reset_index(drop=True)
print(f'-> Source rasters evaluated: {total_coverage_sources:,}')
print(f'-> Contributing rasters retained: {len(fire_hazard_hls_alignment_sources):,}')
print(f'-> Non-contributing rasters excluded: {len(fire_hazard_hls_alignment_exclusions):,}')
print(f'-> Exclusion manifest saved: {fire_hazard_hls_alignment_exclusion_path}')

# Handle missing or invalid alignment exclusions explicitly during HLS sources by valid study-area
# coverage.
if not fire_hazard_hls_alignment_exclusions.empty:
    print('\n--- HLS ALIGNMENT EXCLUSIONS ---')
    display(fire_hazard_hls_alignment_exclusions[['COLLECTION_ID',
        'ITEM_ID', 'INDEX_NAME', 'VALID_PIXELS_ON_TARGET_GRID',
        'VALID_PIXELS_INSIDE_STUDY_AREA', 'EXCLUSION_REASON']].head(5))
print('\n=== HLS SOURCE COVERAGE FILTER COMPLETE ===')



=== FILTERING HLS SOURCES BY VALID STUDY-AREA COVERAGE ===
-> Checking source coverage 1 of 130...
-> Checking source coverage 10 of 130...
-> Checking source coverage 20 of 130...
-> Checking source coverage 30 of 130...
-> Checking source coverage 40 of 130...
-> Checking source coverage 50 of 130...
-> Checking source coverage 60 of 130...
-> Checking source coverage 70 of 130...
-> Checking source coverage 80 of 130...
-> Checking source coverage 90 of 130...
-> Checking source coverage 100 of 130...
-> Checking source coverage 110 of 130...
-> Checking source coverage 120 of 130...
-> Checking source coverage 130 of 130...
-> Source rasters evaluated: 130
-> Contributing rasters retained: 126
-> Non-contributing rasters excluded: 4
-> Exclusion manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\aligned_scene_indices\metadata\hls_alignment_exclusions.csv

--- HLS ALIGNMENT EXCLUSIONS ---


,COLLECTION_ID,ITEM_ID,INDEX_NAME,VALID_PIXELS_ON_TARGET_GRID,VALID_PIXELS_INSIDE_STUDY_AREA,EXCLUSION_REASON
0,hls2-l30,HLS.L30.T12TUL.2025224T181402.v2.0,NDMI,3036660,0,No valid NDVI or NDMI pixels fall inside the t...
1,hls2-l30,HLS.L30.T12TUL.2025224T181402.v2.0,NDVI,3036660,0,No valid NDVI or NDMI pixels fall inside the t...
2,hls2-l30,HLS.L30.T12TUL.2025256T181412.v2.0,NDMI,2804562,0,No valid NDVI or NDMI pixels fall inside the t...
3,hls2-l30,HLS.L30.T12TUL.2025256T181412.v2.0,NDVI,2804562,0,No valid NDVI or NDMI pixels fall inside the t...



=== HLS SOURCE COVERAGE FILTER COMPLETE ===


### Clipping, Reprojecting, and Aligning Scene-Level Ndvi and Ndmi


In [82]:
print('=== CLIPPING, REPROJECTING, AND ALIGNING SCENE-LEVEL NDVI AND NDMI ===')
# List the required HLS alignment processing prerequisites required before scene-level NDVI
# and NDMI can run.
required_hls_alignment_processing_inputs = ['fire_hazard_hls_alignment_sources',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_target_crs', 'fire_hazard_nodata_value',
    'fire_hazard_raster_dtype', 'fire_hazard_alignment_resampling_method',
    'fire_hazard_alignment_profile', 'fire_hazard_hls_alignment_progress_interval',
    'fire_hazard_hls_alignment_reuse_existing', 'fire_hazard_hls_alignment_overwrite',
    'fire_hazard_hls_alignment_minimum_valid_pixels',
    'fire_hazard_hls_alignment_manifest_path']
# Identify unavailable HLS alignment processing so scene-level NDVI and NDMI stops before
# using incomplete inputs.
missing_hls_alignment_processing_inputs = [object_name for object_name in \
    required_hls_alignment_processing_inputs if object_name not in globals()]

# Stop execution when required hls alignment processing inputs inputs are unavailable.
if missing_hls_alignment_processing_inputs:
    raise NameError(f'The following HLS '
        f'alignment-processing objects '
        f'are missing:\n'
        f'{missing_hls_alignment_processing_inputs}\n\n'
        f'Run step Prepare HLS Alignment '
        f'Workspace before aligning the '
        f'scene-level rasters.')
from rasterio import band
from rasterio.warp import reproject
# List the required HLS alignment source fields prerequisites required before scene-level NDVI and
# NDMI can run.
required_hls_alignment_source_fields = ['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'OUTPUT_PATH', 'ALIGNED_PATH']
# Identify unavailable HLS alignment source fields so scene-level NDVI and NDMI stops before
# using incomplete inputs.
missing_hls_alignment_source_fields = [field_name for field_name in \
    required_hls_alignment_source_fields if field_name not in \
    fire_hazard_hls_alignment_sources.columns]

# Stop execution when required hls alignment source fields inputs are unavailable.
if missing_hls_alignment_source_fields:
    raise ValueError(f'The HLS alignment processing '
        f'plan is missing required '
        f'fields:\n'
        f'{missing_hls_alignment_source_fields}')

# Stop execution if hLS alignment processing plan contains no scene-level index rasters.
if fire_hazard_hls_alignment_sources.empty:
    raise ValueError('The HLS alignment processing plan contains no scene-level index rasters.')
# Set alignment stop on error to control scene-level NDVI and NDMI.
fire_hazard_hls_alignment_stop_on_error = False
# Set alignment temporary suffix to control scene-level NDVI and NDMI.
fire_hazard_hls_alignment_temporary_suffix = '.part.tif'
# Track alignment index minimum across processed pixels for output-range QA.
fire_hazard_hls_alignment_index_minimum = -1.0
# Track alignment index maximum across processed pixels for output-range QA.
fire_hazard_hls_alignment_index_maximum = 1.0
# Define alignment records to control the inputs and rules used by scene-level NDVI and NDMI.
fire_hazard_hls_alignment_records = []
# Define alignment error records to control the inputs and rules used by scene-level NDVI and NDMI.
fire_hazard_hls_alignment_error_records = []
# Store total HLS alignment rasters needed to carry out scene-level NDVI and NDMI.
total_hls_alignment_rasters = len(fire_hazard_hls_alignment_sources)

# Encapsulate validate existing HLS aligned raster so repeated scene-level NDVI and NDMI steps use
# consistent logic.
def validate_existing_hls_aligned_raster(aligned_path):

    """
    Checks whether an existing aligned raster matches
    the common fire-hazard analysis grid.

    Parameters
    ----------
    aligned_path : pathlib.Path
        Existing aligned raster path.

    Returns
    -------
    bool
        True when the raster exists and matches the
        expected target-grid structure.
    """
    # Build the aligned path location so scene-level NDVI and NDMI uses the expected project file
    # structure.
    aligned_path = Path(aligned_path)

    # Use the existing file only when it is present and valid for scene-level NDVI and NDMI.
    if not aligned_path.exists() or not aligned_path.is_file() or aligned_path.stat().st_size <= 0:
        return False

    # Protect scene-level NDVI and NDMI so expected source or file failures do not leave partial
    # outputs.
    try:

        # Open the raster in a managed context so its file handle closes reliably after scene-level
        # NDVI and NDMI.
        with rasterio.open(aligned_path) as aligned_source:
            return all([aligned_source.count == 1,
                aligned_source.width == fire_hazard_alignment_width,
                aligned_source.height == fire_hazard_alignment_height,
                aligned_source.crs is not None, aligned_source.crs.to_string() == \
                    fire_hazard_target_crs,
                aligned_source.transform == fire_hazard_alignment_transform,
                aligned_source.dtypes[0] == fire_hazard_raster_dtype,
                aligned_source.nodata == fire_hazard_nodata_value])
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception:
        return False

# Process the raster in internal windows to control memory use
# while preserving the full-resolution output.
# Process each (alignment number entry so scene-level NDVI and NDMI is applied consistently across
# all records.
for alignment_number, (_, alignment_source_record) in \
    enumerate(fire_hazard_hls_alignment_sources.iterrows(),
    start=1):

    # Handle the alignment number case explicitly during scene-level NDVI and NDMI.
    if alignment_number == 1 or alignment_number % fire_hazard_hls_alignment_progress_interval == \
        0 or alignment_number == total_hls_alignment_rasters:
        print(f'-> Aligning scene-level raster '
            f'{alignment_number:,} of '
            f'{total_hls_alignment_rasters:,}...')
    # Record collection ID needed to identify the current HLS asset in manifests and requests.
    collection_id = alignment_source_record['COLLECTION_ID']
    # Record item ID needed to identify the current HLS asset in manifests and requests.
    item_id = alignment_source_record['ITEM_ID']
    # Store index name needed to carry out scene-level NDVI and NDMI.
    index_name = str(alignment_source_record['INDEX_NAME']).upper()
    # Build the source index path location so scene-level NDVI and NDMI uses the expected project
    # file structure.
    source_index_path = Path(alignment_source_record['OUTPUT_PATH'])
    # Build the aligned index path location so scene-level NDVI and NDMI uses the expected project
    # file structure.
    aligned_index_path = Path(alignment_source_record['ALIGNED_PATH'])
    # Build the temporary aligned path location so scene-level NDVI and NDMI uses the expected
    # project file structure.
    temporary_aligned_path = aligned_index_path.parent / (aligned_index_path.stem + \
        fire_hazard_hls_alignment_temporary_suffix)
    # Record acquisition action so scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    acquisition_action = None
    # Store source crs text so spatial operations use the required coordinate reference system.
    source_crs_text = None
    # Set source width to control scene-level NDVI and NDMI.
    source_width = None
    # Set source height to control scene-level NDVI and NDMI.
    source_height = None
    # Set source pixel width to control scene-level NDVI and NDMI.
    source_pixel_width = None
    # Set source pixel height to control scene-level NDVI and NDMI.
    source_pixel_height = None
    # Calculate aligned valid pixel count to quantify completeness and support QA checks.
    aligned_valid_pixel_count = 0
    # Calculate aligned NoData pixel count to quantify completeness and support QA checks.
    aligned_nodata_pixel_count = 0
    # Track aligned minimum across processed pixels for output-range QA.
    aligned_minimum = None
    # Track aligned maximum across processed pixels for output-range QA.
    aligned_maximum = None
    # Set aligned output size bytes to control scene-level NDVI and NDMI.
    aligned_output_size_bytes = 0
    # Record aligned grid valid so scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    aligned_grid_valid = False
    # Record aligned structure valid so scene-level NDVI and NDMI can preserve a clear processing
    # and validation outcome.
    aligned_structure_valid = False
    # Record aligned output valid so scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    aligned_output_valid = False
    # Set alignment error to control scene-level NDVI and NDMI.
    alignment_error = None

    # Protect scene-level NDVI and NDMI so expected source or file failures do not leave partial
    # outputs.
    try:
        # Record existing output valid so scene-level NDVI and NDMI can preserve a clear processing
        # and validation outcome.
        existing_output_valid = validate_existing_hls_aligned_raster(aligned_index_path)

        # Handle the alignment reuse existing case explicitly during scene-level NDVI and NDMI.
        if fire_hazard_hls_alignment_reuse_existing and existing_output_valid:
            # Record acquisition action so scene-level NDVI and NDMI can preserve a clear processing
            # and validation outcome.
            acquisition_action = 'Reused existing aligned raster'

            # Open the raster in a managed context so its file handle closes reliably after
            # scene-level NDVI and NDMI.
            with rasterio.open(aligned_index_path) as aligned_source:
                # Store source crs text so spatial operations use the required coordinate reference
                # system.
                source_crs_text = aligned_source.crs.to_string() if aligned_source.crs is not \
                    None else None
                # Store source width needed to carry out scene-level NDVI and NDMI.
                source_width = aligned_source.width
                # Store source height needed to carry out scene-level NDVI and NDMI.
                source_height = aligned_source.height
                # Store source pixel width needed to carry out scene-level NDVI and NDMI.
                source_pixel_width = abs(aligned_source.transform.a)
                # Store source pixel height needed to carry out scene-level NDVI and NDMI.
                source_pixel_height = abs(aligned_source.transform.e)

                # Process each ( entry so scene-level NDVI and NDMI is applied consistently across
                # all records.
                for _, aligned_window in aligned_source.block_windows(1):
                    # Prepare aligned array used to process the current raster window.
                    aligned_array = aligned_source.read(1, window=aligned_window)
                    # Record valid aligned pixels so scene-level NDVI and NDMI can preserve a clear
                    # processing and validation outcome.
                    valid_aligned_pixels = np.isfinite(aligned_array) & (aligned_array != \
                        fire_hazard_nodata_value)
                    # Record valid values so scene-level NDVI and NDMI can preserve a clear
                    # processing and validation outcome.
                    valid_values = aligned_array[valid_aligned_pixels]
                    # Calculate aligned valid pixel count to quantify completeness and support QA
                    # checks.
                    aligned_valid_pixel_count += int(valid_values.size)
                    # Calculate aligned NoData pixel count to quantify completeness and support QA
                    # checks.
                    aligned_nodata_pixel_count += int((~valid_aligned_pixels).sum())

                    # Handle the valid values case explicitly during scene-level NDVI and NDMI.
                    if valid_values.size > 0:
                        # Track window minimum across processed pixels for output-range QA.
                        window_minimum = float(valid_values.min())
                        # Track window maximum across processed pixels for output-range QA.
                        window_maximum = float(valid_values.max())
                        # Track aligned minimum across processed pixels for output-range QA.
                        aligned_minimum = window_minimum if aligned_minimum is None else \
                            min(aligned_minimum,
                            window_minimum)
                        # Track aligned maximum across processed pixels for output-range QA.
                        aligned_maximum = window_maximum if aligned_maximum is None else \
                            max(aligned_maximum,
                            window_maximum)
            # Record aligned grid valid so scene-level NDVI and NDMI can preserve a clear processing
            # and validation outcome.
            aligned_grid_valid = existing_output_valid
            # Record aligned structure valid so scene-level NDVI and NDMI can preserve a clear
            # processing and validation outcome.
            aligned_structure_valid = existing_output_valid
            # Store aligned output size bytes needed to carry out scene-level NDVI and NDMI.
            aligned_output_size_bytes = aligned_index_path.stat().st_size
            # Record aligned output valid so scene-level NDVI and NDMI can preserve a clear
            # processing and validation outcome.
            aligned_output_valid = existing_output_valid and aligned_valid_pixel_count >= \
                fire_hazard_hls_alignment_minimum_valid_pixels
        else:
            # Record acquisition action so scene-level NDVI and NDMI can preserve a clear processing
            # and validation outcome.
            acquisition_action = 'Reprojected and aligned raster'

            # Use the existing file only when it is present and valid for scene-level NDVI and NDMI.
            if temporary_aligned_path.exists():
                temporary_aligned_path.unlink()

            # Protect existing outputs unless the configured overwrite policy permits replacement.
            if aligned_index_path.exists() and fire_hazard_hls_alignment_overwrite:
                aligned_index_path.unlink()

            # Stop execution if scene-level index raster does not exist: the reported value.
            if not source_index_path.exists() or not source_index_path.is_file():
                raise FileNotFoundError(f'The scene-level index raster '
                    f'does not exist:\n'
                    f'{source_index_path}')

            # Open the raster in a managed context so its file handle closes reliably after
            # scene-level NDVI and NDMI.
            with rasterio.open(source_index_path) as source_index:

                # Stop execution if scene-level index raster does not define a CRS: the reported
                # value.
                if source_index.crs is None:
                    raise ValueError(f'The scene-level index raster '
                        f'does not define a CRS:\n'
                        f'{source_index_path}')
                # Store source crs text so spatial operations use the required coordinate reference
                # system.
                source_crs_text = source_index.crs.to_string()
                # Store source width needed to carry out scene-level NDVI and NDMI.
                source_width = source_index.width
                # Store source height needed to carry out scene-level NDVI and NDMI.
                source_height = source_index.height
                # Store source pixel width needed to carry out scene-level NDVI and NDMI.
                source_pixel_width = abs(source_index.transform.a)
                # Store source pixel height needed to carry out scene-level NDVI and NDMI.
                source_pixel_height = abs(source_index.transform.e)
                # Store source NoData value needed to carry out scene-level NDVI and NDMI.
                source_nodata_value = source_index.nodata
                # Prepare aligned profile to preserve and validate raster structure during
                # scene-level NDVI and NDMI.
                aligned_profile = fire_hazard_alignment_profile.copy()
                aligned_profile.update({'driver': 'GTiff',
                    'dtype': fire_hazard_raster_dtype, 'count': 1,
                    'nodata': fire_hazard_nodata_value, 'width': fire_hazard_alignment_width,
                    'height': fire_hazard_alignment_height, 'crs': fire_hazard_target_crs,
                    'transform': fire_hazard_alignment_transform, 'compress': 'deflate',
                    'predictor': 3, 'tiled': True, 'BIGTIFF': 'IF_SAFER'})

                # Open the raster in a managed context so its file handle closes reliably after
                # scene-level NDVI and NDMI.
                with rasterio.open(temporary_aligned_path,
                    'w', **aligned_profile) as aligned_destination:
                    reproject(source=band(source_index,
                        1), destination=band(aligned_destination, 1), \
                            src_transform=source_index.transform,
                        src_crs=source_index.crs, src_nodata=source_nodata_value,
                        dst_transform=fire_hazard_alignment_transform,
                        dst_crs=fire_hazard_target_crs, dst_nodata=fire_hazard_nodata_value,
                        resampling=fire_hazard_alignment_resampling_method,
                        init_dest_nodata=True, num_threads=2)
                    aligned_destination.set_band_description(1,
                        f'Aligned HLS scene-level {index_name}')
                    aligned_destination.update_tags(COLLECTION_ID=collection_id,
                        ITEM_ID=item_id, INDEX_NAME=index_name, SOURCE_PATH=str(source_index_path),
                        SOURCE_CRS=source_crs_text, TARGET_CRS=fire_hazard_target_crs,
                        TARGET_CELL_SIZE=fire_hazard_cell_size, RESAMPLING_METHOD='bilinear',
                        STUDY_AREA_MASK_APPLIED=True)

            # Open the raster in a managed context so its file handle closes reliably after
            # scene-level NDVI and NDMI.
            with rasterio.open(temporary_aligned_path, 'r+') as aligned_destination:

                # Process each ( entry so scene-level NDVI and NDMI is applied consistently across
                # all records.
                for _, aligned_window in aligned_destination.block_windows(1):
                    # Prepare aligned array used to process the current raster window.
                    aligned_array = aligned_destination.read(1,
                        window=aligned_window).astype(np.float32)
                    # Calculate row start for the current raster processing window.
                    row_start = int(aligned_window.row_off)
                    # Store row end needed to carry out scene-level NDVI and NDMI.
                    row_end = int(aligned_window.row_off + aligned_window.height)
                    # Store column start needed to carry out scene-level NDVI and NDMI.
                    column_start = int(aligned_window.col_off)
                    # Store column end needed to carry out scene-level NDVI and NDMI.
                    column_end = int(aligned_window.col_off + aligned_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Record invalid aligned pixels so scene-level NDVI and NDMI can preserve a
                    # clear processing and validation outcome.
                    invalid_aligned_pixels = ~np.isfinite(aligned_array) | (aligned_array == \
                        fire_hazard_nodata_value)
                    # Record invalid aligned pixels so scene-level NDVI and NDMI can preserve a
                    # clear processing and validation outcome.
                    invalid_aligned_pixels |= ~study_area_window_mask
                    # Record valid aligned pixels so scene-level NDVI and NDMI can preserve a clear
                    # processing and validation outcome.
                    valid_aligned_pixels = ~invalid_aligned_pixels

                    # Handle the valid aligned pixels case explicitly during scene-level NDVI and
                    # NDMI.
                    if valid_aligned_pixels.any():
                        # Prepare aligned array[valid aligned pixels] used to process the current
                        # raster window.
                        aligned_array[valid_aligned_pixels] = \
                            np.clip(aligned_array[valid_aligned_pixels],
                            fire_hazard_hls_alignment_index_minimum, \
                                fire_hazard_hls_alignment_index_maximum)
                    # Prepare aligned array[invalid aligned pixels] used to process the current
                    # raster window.
                    aligned_array[invalid_aligned_pixels] = fire_hazard_nodata_value
                    # Write the processed data to the configured output resource.
                    aligned_destination.write(aligned_array, 1, window=aligned_window)
                    # Record valid values so scene-level NDVI and NDMI can preserve a clear
                    # processing and validation outcome.
                    valid_values = aligned_array[valid_aligned_pixels]
                    # Calculate aligned valid pixel count to quantify completeness and support QA
                    # checks.
                    aligned_valid_pixel_count += int(valid_values.size)
                    # Calculate aligned NoData pixel count to quantify completeness and support QA
                    # checks.
                    aligned_nodata_pixel_count += int(invalid_aligned_pixels.sum())

                    # Handle the valid values case explicitly during scene-level NDVI and NDMI.
                    if valid_values.size > 0:
                        # Track window minimum across processed pixels for output-range QA.
                        window_minimum = float(valid_values.min())
                        # Track window maximum across processed pixels for output-range QA.
                        window_maximum = float(valid_values.max())
                        # Track aligned minimum across processed pixels for output-range QA.
                        aligned_minimum = window_minimum if aligned_minimum is None else \
                            min(aligned_minimum,
                            window_minimum)
                        # Track aligned maximum across processed pixels for output-range QA.
                        aligned_maximum = window_maximum if aligned_maximum is None else \
                            max(aligned_maximum,
                            window_maximum)

            # Open the raster in a managed context so its file handle closes reliably after
            # scene-level NDVI and NDMI.
            with rasterio.open(temporary_aligned_path) as aligned_source:
                # Record aligned grid valid so scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                aligned_grid_valid = all([aligned_source.width == fire_hazard_alignment_width,
                    aligned_source.height == fire_hazard_alignment_height,
                    aligned_source.crs is not None, aligned_source.crs.to_string() == \
                        fire_hazard_target_crs,
                    aligned_source.transform == fire_hazard_alignment_transform])
                # Record aligned structure valid so scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                aligned_structure_valid = all([aligned_source.count == 1,
                    aligned_source.dtypes[0] == fire_hazard_raster_dtype,
                    aligned_source.nodata == fire_hazard_nodata_value])
            # Store aligned value range available needed to carry out scene-level NDVI and NDMI.
            aligned_value_range_available = aligned_minimum is not None and aligned_maximum is \
                not None
            # Record aligned value range valid so scene-level NDVI and NDMI can preserve a clear
            # processing and validation outcome.
            aligned_value_range_valid = aligned_value_range_available and aligned_minimum >= \
                fire_hazard_hls_alignment_index_minimum and (aligned_maximum <= \
                fire_hazard_hls_alignment_index_maximum)
            # Record aligned output valid so scene-level NDVI and NDMI can preserve a clear
            # processing and validation outcome.
            aligned_output_valid = all([temporary_aligned_path.exists(),
                temporary_aligned_path.is_file(), temporary_aligned_path.stat().st_size > 0,
                aligned_grid_valid, aligned_structure_valid, aligned_valid_pixel_count >= \
                    fire_hazard_hls_alignment_minimum_valid_pixels,
                aligned_value_range_available, aligned_value_range_valid])

            # Stop execution if aligned raster failed output validation.
            if not aligned_output_valid:
                raise ValueError(f'The aligned raster failed '
                    f'output validation.\n\nSource '
                    f'raster: {source_index_path}\n'
                    f'Temporary output: '
                    f'{temporary_aligned_path}\nGrid '
                    f'valid: {aligned_grid_valid}\n'
                    f'Structure valid: '
                    f'{aligned_structure_valid}\n'
                    f'Valid study-area pixels: '
                    f'{aligned_valid_pixel_count:,}\n'
                    f'Minimum required pixels: '
                    f'{fire_hazard_hls_alignment_minimum_valid_pixels:,}\n'
                    f'Aligned minimum: '
                    f'{aligned_minimum}\nAligned '
                    f'maximum: {aligned_maximum}\n'
                    f'Value range available: '
                    f'{aligned_value_range_available}\n'
                    f'Value range valid: '
                    f'{aligned_value_range_valid}')
            temporary_aligned_path.replace(aligned_index_path)
            # Store aligned output size bytes needed to carry out scene-level NDVI and NDMI.
            aligned_output_size_bytes = aligned_index_path.stat().st_size
        fire_hazard_hls_alignment_records.append({'COLLECTION_ID': collection_id,
            'ITEM_ID': item_id, 'INDEX_NAME': index_name, 'SOURCE_PATH': str(source_index_path),
            'ALIGNED_PATH': str(aligned_index_path), 'PROCESSING_ACTION': acquisition_action,
            'SOURCE_CRS': source_crs_text, 'SOURCE_WIDTH': source_width,
            'SOURCE_HEIGHT': source_height, 'SOURCE_PIXEL_WIDTH': source_pixel_width,
            'SOURCE_PIXEL_HEIGHT': source_pixel_height, 'TARGET_CRS': fire_hazard_target_crs,
            'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': \
                fire_hazard_alignment_height,
            'TARGET_PIXEL_SIZE': fire_hazard_cell_size, 'RESAMPLING_METHOD': 'bilinear',
            'VALID_PIXELS': aligned_valid_pixel_count, 'NODATA_PIXELS': aligned_nodata_pixel_count,
            'INDEX_MINIMUM': aligned_minimum, 'INDEX_MAXIMUM': aligned_maximum,
            'OUTPUT_SIZE_BYTES': aligned_output_size_bytes,
            'OUTPUT_SIZE_MB': aligned_output_size_bytes / 1024 ** 2,
            'GRID_VALID': aligned_grid_valid, 'STRUCTURE_VALID': aligned_structure_valid,
            'ERROR_MESSAGE': None, 'VALID': aligned_output_valid})
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception as error:
        # Store alignment error needed to carry out scene-level NDVI and NDMI.
        alignment_error = str(error)

        # Use the existing file only when it is present and valid for scene-level NDVI and NDMI.
        if temporary_aligned_path.exists():
            temporary_aligned_path.unlink()
        fire_hazard_hls_alignment_error_records.append({'COLLECTION_ID': collection_id,
            'ITEM_ID': item_id, 'INDEX_NAME': index_name, 'SOURCE_PATH': str(source_index_path),
            'ALIGNED_PATH': str(aligned_index_path), 'ERROR_MESSAGE': alignment_error})
        fire_hazard_hls_alignment_records.append({'COLLECTION_ID': collection_id,
            'ITEM_ID': item_id, 'INDEX_NAME': index_name, 'SOURCE_PATH': str(source_index_path),
            'ALIGNED_PATH': str(aligned_index_path), 'PROCESSING_ACTION': 'Alignment failed',
            'SOURCE_CRS': source_crs_text, 'SOURCE_WIDTH': source_width,
            'SOURCE_HEIGHT': source_height, 'SOURCE_PIXEL_WIDTH': source_pixel_width,
            'SOURCE_PIXEL_HEIGHT': source_pixel_height, 'TARGET_CRS': fire_hazard_target_crs,
            'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': \
                fire_hazard_alignment_height,
            'TARGET_PIXEL_SIZE': fire_hazard_cell_size, 'RESAMPLING_METHOD': 'bilinear',
            'VALID_PIXELS': aligned_valid_pixel_count, 'NODATA_PIXELS': aligned_nodata_pixel_count,
            'INDEX_MINIMUM': aligned_minimum, 'INDEX_MAXIMUM': aligned_maximum,
            'OUTPUT_SIZE_BYTES': 0, 'OUTPUT_SIZE_MB': 0.0,
            'GRID_VALID': False, 'STRUCTURE_VALID': False,
            'ERROR_MESSAGE': alignment_error, 'VALID': False})
        print(f'-> Alignment failed for {item_id} | {index_name}: {alignment_error}')
        # Store partial alignment manifest needed to carry out scene-level NDVI and NDMI.
        partial_alignment_manifest = pd.DataFrame(fire_hazard_hls_alignment_records)
        # Save the partial alignment manifest CSV so later steps can reuse the recorded workflow
        # results.
        partial_alignment_manifest.to_csv(fire_hazard_hls_alignment_manifest_path, index=False)

        # Handle the alignment stop on error case explicitly during scene-level NDVI and NDMI.
        if fire_hazard_hls_alignment_stop_on_error:
            raise
# Store alignment manifest needed to carry out scene-level NDVI and NDMI.
fire_hazard_hls_alignment_manifest = \
    pd.DataFrame(fire_hazard_hls_alignment_records).sort_values(['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME']).reset_index(drop=True)
# Save the alignment manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_alignment_manifest.to_csv(fire_hazard_hls_alignment_manifest_path, index=False)
# Calculate successful HLS alignment count to quantify completeness and support QA checks.
successful_hls_alignment_count = int(fire_hazard_hls_alignment_manifest['VALID'].sum())
# Calculate failed HLS alignment count to quantify completeness and support QA checks.
failed_hls_alignment_count = len(fire_hazard_hls_alignment_manifest) - \
    successful_hls_alignment_count
# Calculate reused HLS alignment count to quantify completeness and support QA checks.
reused_hls_alignment_count = int((fire_hazard_hls_alignment_manifest['PROCESSING_ACTION'] == \
    'Reused existing aligned raster').sum())
# Calculate new HLS alignment count to quantify completeness and support QA checks.
new_hls_alignment_count = int((fire_hazard_hls_alignment_manifest['PROCESSING_ACTION'] == \
    'Reprojected and aligned raster').sum())
print(f'\n-> Scene-level rasters processed: {len(fire_hazard_hls_alignment_manifest):,}')
print(f'-> Newly aligned rasters: {new_hls_alignment_count:,}')
print(f'-> Reused aligned rasters: {reused_hls_alignment_count:,}')
print(f'-> Valid aligned rasters: {successful_hls_alignment_count:,}')
print(f'-> Failed aligned rasters: {failed_hls_alignment_count:,}')
print(f'-> Alignment manifest saved: {fire_hazard_hls_alignment_manifest_path}')
print('\n--- HLS ALIGNMENT EXECUTION SAMPLE ---')
display(fire_hazard_hls_alignment_manifest[['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'PROCESSING_ACTION', 'SOURCE_CRS',
    'TARGET_CRS', 'VALID_PIXELS', 'INDEX_MINIMUM',
    'INDEX_MAXIMUM', 'OUTPUT_SIZE_MB', 'VALID']].head(5))
print('\nNOTE:')
print('Each aligned raster now uses the common 30-meter EPSG:26912 fire-hazard grid.')
print('Pixels outside Salt Lake, Utah,'
    ' and Washington counties were '
    'assigned the project NoData '
    'value.')
print("The next step will "
    "independently validate every "
    "aligned raster's grid, "
    "structure, value range, and "
    "study-area coverage.")
print('\n=== SCENE-LEVEL NDVI AND NDMI ALIGNMENT COMPLETE ===')


=== CLIPPING, REPROJECTING, AND ALIGNING SCENE-LEVEL NDVI AND NDMI ===
-> Aligning scene-level raster 1 of 126...
-> Aligning scene-level raster 5 of 126...
-> Aligning scene-level raster 10 of 126...
-> Aligning scene-level raster 15 of 126...
-> Aligning scene-level raster 20 of 126...
-> Aligning scene-level raster 25 of 126...
-> Aligning scene-level raster 30 of 126...
-> Aligning scene-level raster 35 of 126...
-> Aligning scene-level raster 40 of 126...
-> Aligning scene-level raster 45 of 126...
-> Aligning scene-level raster 50 of 126...
-> Aligning scene-level raster 55 of 126...
-> Aligning scene-level raster 60 of 126...
-> Aligning scene-level raster 65 of 126...
-> Aligning scene-level raster 70 of 126...
-> Aligning scene-level raster 75 of 126...
-> Aligning scene-level raster 80 of 126...
-> Aligning scene-level raster 85 of 126...
-> Aligning scene-level raster 90 of 126...
-> Aligning scene-level raster 95 of 126...
-> Aligning scene-level raster 100 of 126...
-> Ali

,COLLECTION_ID,ITEM_ID,INDEX_NAME,PROCESSING_ACTION,SOURCE_CRS,TARGET_CRS,VALID_PIXELS,INDEX_MINIMUM,INDEX_MAXIMUM,OUTPUT_SIZE_MB,VALID
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDMI,Reused existing aligned raster,EPSG:26912,EPSG:26912,19740,-0.243638,0.122167,2.745051,True
1,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDVI,Reused existing aligned raster,EPSG:26912,EPSG:26912,19740,0.072353,0.408986,2.742518,True
2,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDMI,Reused existing aligned raster,EPSG:26912,EPSG:26912,14951,-0.226472,0.220386,2.698915,True
3,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDVI,Reused existing aligned raster,EPSG:26912,EPSG:26912,14951,0.090153,0.406582,2.694722,True
4,hls2-l30,HLS.L30.T11SQB.2025224T181513.v2.0,NDMI,Reused existing aligned raster,EPSG:26912,EPSG:26912,3511372,-0.632768,1.000000,19.535708,True



NOTE:
Each aligned raster now uses the common 30-meter EPSG:26912 fire-hazard grid.
Pixels outside Salt Lake, Utah, and Washington counties were assigned the project NoData value.
The next step will independently validate every aligned raster's grid, structure, value range, and study-area coverage.

=== SCENE-LEVEL NDVI AND NDMI ALIGNMENT COMPLETE ===


### Validating Aligned Scene-Level Ndvi and Ndmi


In [83]:
print('=== VALIDATING ALIGNED SCENE-LEVEL NDVI AND NDMI ===')
# List the required HLS alignment validation prerequisites required before aligned
# scene-level NDVI and NDMI can run.
required_hls_alignment_validation_inputs = ['fire_hazard_hls_alignment_sources',
    'fire_hazard_hls_alignment_manifest', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_nodata_value',
    'fire_hazard_raster_dtype', 'fire_hazard_hls_alignment_manifest_directory',
    'fire_hazard_hls_alignment_minimum_valid_pixels']
# Identify unavailable HLS alignment validation so aligned scene-level NDVI and NDMI
# stops before using incomplete inputs.
missing_hls_alignment_validation_inputs = [object_name for object_name in \
    required_hls_alignment_validation_inputs if object_name not in globals()]

# Stop execution when required hls alignment validation inputs inputs are unavailable.
if missing_hls_alignment_validation_inputs:
    raise NameError(f'The following HLS '
        f'alignment-validation objects '
        f'are missing:\n'
        f'{missing_hls_alignment_validation_inputs}\n\n'
        f'Run step Prepare HLS Alignment '
        f'Workspace and step Clip, '
        f'Reproject, and Align '
        f'Scene-Level NDVI and NDMI '
        f'before validating the outputs.')
# Build the alignment validation path location so aligned scene-level NDVI and NDMI uses the
# expected project file structure.
fire_hazard_hls_alignment_validation_path = fire_hazard_hls_alignment_manifest_directory / \
    'hls_alignment_validation_manifest.csv'
# Build the alignment validation summary path location so aligned scene-level NDVI and NDMI uses the
# expected project file structure.
fire_hazard_hls_alignment_validation_summary_path = fire_hazard_hls_alignment_manifest_directory \
    / 'hls_alignment_validation_summary.csv'
# Build the alignment failed validation path location so aligned scene-level NDVI and NDMI uses the
# expected project file structure.
fire_hazard_hls_alignment_failed_validation_path = fire_hazard_hls_alignment_manifest_directory / \
    'hls_alignment_failed_validation.csv'
# Record alignment valid extensions so aligned scene-level NDVI and NDMI can preserve a clear
# processing and validation outcome.
fire_hazard_hls_alignment_valid_extensions = {'.tif', '.tiff'}
# Track alignment minimum file size bytes across processed pixels for output-range QA.
fire_hazard_hls_alignment_minimum_file_size_bytes = 1024
# Track alignment validation minimum across processed pixels for output-range QA.
fire_hazard_hls_alignment_validation_minimum = -1.0
# Track alignment validation maximum across processed pixels for output-range QA.
fire_hazard_hls_alignment_validation_maximum = 1.0
# Set alignment value tolerance to control aligned scene-level NDVI and NDMI.
fire_hazard_hls_alignment_value_tolerance = 1e-06
# Record alignment validation progress interval so aligned scene-level NDVI and NDMI can preserve a
# clear processing and validation outcome.
fire_hazard_hls_alignment_validation_progress_interval = 5
# List the required alignment source fields prerequisites required before aligned scene-level NDVI
# and NDMI can run.
required_alignment_source_fields = ['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'OUTPUT_PATH', 'ALIGNED_PATH']
# Identify unavailable alignment source fields so aligned scene-level NDVI and NDMI stops before
# using incomplete inputs.
missing_alignment_source_fields = [field_name for field_name in required_alignment_source_fields \
    if field_name not in fire_hazard_hls_alignment_sources.columns]

# Stop execution when required alignment source fields inputs are unavailable.
if missing_alignment_source_fields:
    raise ValueError(f'The HLS alignment source plan '
        f'is missing required fields:\n'
        f'{missing_alignment_source_fields}')

# Stop execution if hLS alignment source plan contains no expected NDVI or NDMI rasters.
if fire_hazard_hls_alignment_sources.empty:
    raise ValueError('The HLS alignment source plan contains no expected NDVI or NDMI rasters.')
# Store expected aligned outputs needed to carry out aligned scene-level NDVI and NDMI.
fire_hazard_hls_expected_aligned_outputs = \
    fire_hazard_hls_alignment_sources[required_alignment_source_fields].copy().reset_index(drop= \
    True)
# Store expected aligned outputs['index name'] needed to carry out aligned scene-level NDVI and
# NDMI.
fire_hazard_hls_expected_aligned_outputs['INDEX_NAME'] = \
    fire_hazard_hls_expected_aligned_outputs['INDEX_NAME'].astype(str).str.strip().str.upper()
# Identify unexpected unexpected alignment index names so only approved inputs reach aligned
# scene-level NDVI and NDMI.
unexpected_alignment_index_names = \
    sorted(set(fire_hazard_hls_expected_aligned_outputs['INDEX_NAME'].unique()) - {'NDVI',
    'NDMI'})

# Stop execution if alignment inventory contains unsupported index names: the reported value.
if unexpected_alignment_index_names:
    raise ValueError(f'The alignment inventory '
        f'contains unsupported index '
        f'names:\n'
        f'{unexpected_alignment_index_names}')
# Build duplicate expected alignment records used to track the records included in this processing
# stage.
duplicate_expected_alignment_records = \
    fire_hazard_hls_expected_aligned_outputs.duplicated(subset=['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME'], keep=False)

# Stop execution if expected HLS alignment inventory contains duplicate collection-item-index
# records.
if duplicate_expected_alignment_records.any():
    raise ValueError('The expected HLS alignment '
        'inventory contains duplicate '
        'collection-item-index records.')
# Identify duplicate duplicate expected alignment paths so repeated records do not bias aligned
# scene-level NDVI and NDMI.
duplicate_expected_alignment_paths = \
    fire_hazard_hls_expected_aligned_outputs.duplicated(subset=['ALIGNED_PATH'],
    keep=False)

# Stop execution if expected HLS alignment inventory contains duplicate aligned output paths.
if duplicate_expected_alignment_paths.any():
    raise ValueError('The expected HLS alignment '
        'inventory contains duplicate '
        'aligned output paths.')
# Calculate expected index counts by scene to quantify completeness and support QA checks.
expected_index_counts_by_scene = fire_hazard_hls_expected_aligned_outputs.groupby(['COLLECTION_ID',
    'ITEM_ID'])['INDEX_NAME'].nunique()
# Calculate invalid expected scene index counts to quantify completeness and support QA checks.
invalid_expected_scene_index_counts = \
    expected_index_counts_by_scene[expected_index_counts_by_scene != 2]

# Stop execution if one or more HLS scenes do not have exactly one expected NDVI and one expected
# NDMI aligned raster.
if not invalid_expected_scene_index_counts.empty:
    raise ValueError('One or more HLS scenes do not '
        'have exactly one expected NDVI '
        'and one expected NDMI aligned '
        'raster.')
# Build the expected study area mask shape mask used to isolate records required for this analysis.
expected_study_area_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if alignment study-area mask does not match the master raster dimensions.
if fire_hazard_alignment_study_area_mask.shape != expected_study_area_mask_shape:
    raise ValueError(f'The alignment study-area mask '
        f'does not match the master '
        f'raster dimensions.\nMask '
        f'shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_study_area_mask_shape}')
# Record alignment validation study area pixels so aligned scene-level NDVI and NDMI can preserve a
# clear processing and validation outcome.
fire_hazard_alignment_validation_study_area_pixels = \
    int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if alignment study-area mask contains no included raster cells.
if fire_hazard_alignment_validation_study_area_pixels == 0:
    raise ValueError('The alignment study-area mask contains no included raster cells.')
# Record alignment validation records so aligned scene-level NDVI and NDMI can preserve a clear
# processing and validation outcome.
fire_hazard_hls_alignment_validation_records = []
# Store total expected aligned rasters needed to carry out aligned scene-level NDVI and NDMI.
total_expected_aligned_rasters = len(fire_hazard_hls_expected_aligned_outputs)

# Process each (validation number entry so aligned scene-level NDVI and NDMI is applied consistently
# across all records.
for validation_number, (_, expected_record) in \
    enumerate(fire_hazard_hls_expected_aligned_outputs.iterrows(),
    start=1):

    # Handle the validation number case explicitly during aligned scene-level NDVI and NDMI.
    if validation_number == 1 or validation_number % \
        fire_hazard_hls_alignment_validation_progress_interval == 0 or validation_number == \
        total_expected_aligned_rasters:
        print(f'-> Validating aligned raster '
            f'{validation_number:,} of '
            f'{total_expected_aligned_rasters:,}...')
    # Record collection ID needed to identify the current HLS asset in manifests and requests.
    collection_id = expected_record['COLLECTION_ID']
    # Record item ID needed to identify the current HLS asset in manifests and requests.
    item_id = expected_record['ITEM_ID']
    # Store index name needed to carry out aligned scene-level NDVI and NDMI.
    index_name = str(expected_record['INDEX_NAME']).upper()
    # Build the source index path location so aligned scene-level NDVI and NDMI uses the expected
    # project file structure.
    source_index_path = Path(expected_record['OUTPUT_PATH'])
    # Build the aligned index path location so aligned scene-level NDVI and NDMI uses the expected
    # project file structure.
    aligned_index_path = Path(expected_record['ALIGNED_PATH'])
    # Store file exists needed to carry out aligned scene-level NDVI and NDMI.
    file_exists = aligned_index_path.exists() and aligned_index_path.is_file()
    # Record file extension valid so aligned scene-level NDVI and NDMI can preserve a clear
    # processing and validation outcome.
    file_extension_valid = aligned_index_path.suffix.lower() in \
        fire_hazard_hls_alignment_valid_extensions
    # Store file size bytes needed to carry out aligned scene-level NDVI and NDMI.
    file_size_bytes = aligned_index_path.stat().st_size if file_exists else 0
    # Record file size valid so aligned scene-level NDVI and NDMI can preserve a clear processing
    # and validation outcome.
    file_size_valid = file_size_bytes >= fire_hazard_hls_alignment_minimum_file_size_bytes
    # Record raster readable so aligned scene-level NDVI and NDMI can preserve a clear processing
    # and validation outcome.
    raster_readable = False
    # Calculate band count valid to quantify completeness and support QA checks.
    band_count_valid = False
    # Record width valid so aligned scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    width_valid = False
    # Record height valid so aligned scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    height_valid = False
    # Store crs valid so spatial operations use the required coordinate reference system.
    crs_valid = False
    # Record transform valid so aligned scene-level NDVI and NDMI can preserve a clear processing
    # and validation outcome.
    transform_valid = False
    # Record pixel size valid so aligned scene-level NDVI and NDMI can preserve a clear processing
    # and validation outcome.
    pixel_size_valid = False
    # Record dtype valid so aligned scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    dtype_valid = False
    # Record NoData valid so aligned scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    nodata_valid = False
    # Record grid valid so aligned scene-level NDVI and NDMI can preserve a clear processing and
    # validation outcome.
    grid_valid = False
    # Record structure valid so aligned scene-level NDVI and NDMI can preserve a clear processing
    # and validation outcome.
    structure_valid = False
    # Calculate valid pixel count to quantify completeness and support QA checks.
    valid_pixel_count = 0
    # Calculate NoData pixel count to quantify completeness and support QA checks.
    nodata_pixel_count = 0
    # Calculate valid inside study area count to quantify completeness and support QA checks.
    valid_inside_study_area_count = 0
    # Calculate valid outside study area count to quantify completeness and support QA checks.
    valid_outside_study_area_count = 0
    # Record outside study area NoData valid so aligned scene-level NDVI and NDMI can preserve a
    # clear processing and validation outcome.
    outside_study_area_nodata_valid = False
    # Track index minimum across processed pixels for output-range QA.
    index_minimum = None
    # Track index maximum across processed pixels for output-range QA.
    index_maximum = None
    # Set value range available to control aligned scene-level NDVI and NDMI.
    value_range_available = False
    # Record value range valid so aligned scene-level NDVI and NDMI can preserve a clear processing
    # and validation outcome.
    value_range_valid = False
    # Record raster validation error so aligned scene-level NDVI and NDMI can preserve a clear
    # processing and validation outcome.
    raster_validation_error = None

    # Handle the file exists case explicitly during aligned scene-level NDVI and NDMI.
    if file_exists and file_extension_valid and file_size_valid:

        # Protect aligned scene-level NDVI and NDMI so expected source or file failures do not leave
        # partial outputs.
        try:

            # Open the raster in a managed context so its file handle closes reliably after aligned
            # scene-level NDVI and NDMI.
            with rasterio.open(aligned_index_path) as aligned_source:
                # Record raster readable so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                raster_readable = True
                # Calculate band count valid to quantify completeness and support QA checks.
                band_count_valid = aligned_source.count == 1
                # Record width valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                width_valid = aligned_source.width == fire_hazard_alignment_width
                # Record height valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                height_valid = aligned_source.height == fire_hazard_alignment_height
                # Store crs valid so spatial operations use the required coordinate reference
                # system.
                crs_valid = aligned_source.crs is not None and aligned_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs)
                # Record transform valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                transform_valid = \
                    aligned_source.transform.almost_equals(fire_hazard_alignment_transform)
                # Record pixel size valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                pixel_size_valid = np.isclose(abs(aligned_source.transform.a),
                    fire_hazard_cell_size) and np.isclose(abs(aligned_source.transform.e),
                    fire_hazard_cell_size)
                # Record dtype valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                dtype_valid = aligned_source.dtypes[0] == fire_hazard_raster_dtype
                # Record NoData valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                nodata_valid = aligned_source.nodata == fire_hazard_nodata_value
                # Record grid valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                grid_valid = all([width_valid,
                    height_valid, crs_valid, transform_valid, pixel_size_valid])
                # Record structure valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                structure_valid = all([band_count_valid, dtype_valid, nodata_valid])

                # Process each ( entry so aligned scene-level NDVI and NDMI is applied consistently
                # across all records.
                for _, aligned_window in aligned_source.block_windows(1):
                    # Prepare aligned array used to process the current raster window.
                    aligned_array = aligned_source.read(1, window=aligned_window)
                    # Calculate row start for the current raster processing window.
                    row_start = int(aligned_window.row_off)
                    # Store row end needed to carry out aligned scene-level NDVI and NDMI.
                    row_end = int(aligned_window.row_off + aligned_window.height)
                    # Store column start needed to carry out aligned scene-level NDVI and NDMI.
                    column_start = int(aligned_window.col_off)
                    # Store column end needed to carry out aligned scene-level NDVI and NDMI.
                    column_end = int(aligned_window.col_off + aligned_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Build the valid pixel mask mask used to isolate records required for this
                    # analysis.
                    valid_pixel_mask = np.isfinite(aligned_array) & (aligned_array != \
                        fire_hazard_nodata_value)
                    # Build the valid inside mask mask used to isolate records required for this
                    # analysis.
                    valid_inside_mask = valid_pixel_mask & study_area_window_mask
                    # Build the valid outside mask mask used to isolate records required for this
                    # analysis.
                    valid_outside_mask = valid_pixel_mask & ~study_area_window_mask
                    # Record block valid values so aligned scene-level NDVI and NDMI can preserve a
                    # clear processing and validation outcome.
                    block_valid_values = aligned_array[valid_pixel_mask]
                    # Calculate valid pixel count to quantify completeness and support QA checks.
                    valid_pixel_count += int(valid_pixel_mask.sum())
                    # Calculate NoData pixel count to quantify completeness and support QA checks.
                    nodata_pixel_count += int((~valid_pixel_mask).sum())
                    # Calculate valid inside study area count to quantify completeness and support
                    # QA checks.
                    valid_inside_study_area_count += int(valid_inside_mask.sum())
                    # Calculate valid outside study area count to quantify completeness and support
                    # QA checks.
                    valid_outside_study_area_count += int(valid_outside_mask.sum())

                    # Handle the block valid values case explicitly during aligned scene-level NDVI
                    # and NDMI.
                    if block_valid_values.size > 0:
                        # Track block minimum across processed pixels for output-range QA.
                        block_minimum = float(block_valid_values.min())
                        # Track block maximum across processed pixels for output-range QA.
                        block_maximum = float(block_valid_values.max())
                        # Track index minimum across processed pixels for output-range QA.
                        index_minimum = block_minimum if index_minimum is None else \
                            min(index_minimum,
                            block_minimum)
                        # Track index maximum across processed pixels for output-range QA.
                        index_maximum = block_maximum if index_maximum is None else \
                            max(index_maximum,
                            block_maximum)
                # Record outside study area NoData valid so aligned scene-level NDVI and NDMI can
                # preserve a clear processing and validation outcome.
                outside_study_area_nodata_valid = valid_outside_study_area_count == 0
                # Store value range available needed to carry out aligned scene-level NDVI and NDMI.
                value_range_available = index_minimum is not None and index_maximum is not None
                # Record value range valid so aligned scene-level NDVI and NDMI can preserve a clear
                # processing and validation outcome.
                value_range_valid = value_range_available and index_minimum >= \
                    fire_hazard_hls_alignment_validation_minimum - \
                    fire_hazard_hls_alignment_value_tolerance and (index_maximum <= \
                    fire_hazard_hls_alignment_validation_maximum + \
                    fire_hazard_hls_alignment_value_tolerance)
        # Handle the expected failure without leaving the workflow in an inconsistent state.
        except Exception as error:
            # Record raster validation error so aligned scene-level NDVI and NDMI can preserve a
            # clear processing and validation outcome.
            raster_validation_error = str(error)
    # Record aligned raster valid so aligned scene-level NDVI and NDMI can preserve a clear
    # processing and validation outcome.
    aligned_raster_valid = all([file_exists, file_extension_valid,
        file_size_valid, raster_readable, grid_valid, structure_valid,
        valid_inside_study_area_count >= fire_hazard_hls_alignment_minimum_valid_pixels,
        outside_study_area_nodata_valid, value_range_available,
        value_range_valid, raster_validation_error is None])
    fire_hazard_hls_alignment_validation_records.append({'COLLECTION_ID': collection_id,
        'ITEM_ID': item_id, 'INDEX_NAME': index_name, 'SOURCE_PATH': str(source_index_path),
        'ALIGNED_PATH': str(aligned_index_path), 'FILE_EXISTS': file_exists,
        'FILE_EXTENSION_VALID': file_extension_valid, 'FILE_SIZE_BYTES': file_size_bytes,
        'FILE_SIZE_MB': file_size_bytes / 1024 ** 2, 'FILE_SIZE_VALID': file_size_valid,
        'RASTER_READABLE': raster_readable, 'BAND_COUNT_VALID': band_count_valid,
        'WIDTH_VALID': width_valid, 'HEIGHT_VALID': height_valid,
        'CRS_VALID': crs_valid, 'TRANSFORM_VALID': transform_valid,
        'PIXEL_SIZE_VALID': pixel_size_valid, 'DTYPE_VALID': dtype_valid,
        'NODATA_VALID': nodata_valid, 'GRID_VALID': grid_valid,
        'STRUCTURE_VALID': structure_valid, 'VALID_PIXELS': valid_pixel_count,
        'NODATA_PIXELS': nodata_pixel_count, 'VALID_INSIDE_STUDY_AREA': \
            valid_inside_study_area_count,
        'VALID_OUTSIDE_STUDY_AREA': valid_outside_study_area_count,
        'OUTSIDE_STUDY_AREA_NODATA_VALID': outside_study_area_nodata_valid,
        'INDEX_MINIMUM': index_minimum, 'INDEX_MAXIMUM': index_maximum,
        'VALUE_RANGE_AVAILABLE': value_range_available,
        'VALUE_RANGE_VALID': value_range_valid, 'ERROR_MESSAGE': raster_validation_error,
        'VALID': aligned_raster_valid})
# Record alignment validation so aligned scene-level NDVI and NDMI can preserve a clear processing
# and validation outcome.
fire_hazard_hls_alignment_validation = \
    pd.DataFrame(fire_hazard_hls_alignment_validation_records).sort_values(['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME']).reset_index(drop=True)
# Record failed HLS alignment validations so aligned scene-level NDVI and NDMI can preserve a clear
# processing and validation outcome.
failed_hls_alignment_validations = \
    fire_hazard_hls_alignment_validation[~fire_hazard_hls_alignment_validation['VALID']].copy() \
    .reset_index(drop=True)
# Calculate expected aligned raster count to quantify completeness and support QA checks.
expected_aligned_raster_count = len(fire_hazard_hls_expected_aligned_outputs)
# Calculate validated aligned raster count to quantify completeness and support QA checks.
validated_aligned_raster_count = len(fire_hazard_hls_alignment_validation)
# Calculate valid aligned raster count to quantify completeness and support QA checks.
valid_aligned_raster_count = int(fire_hazard_hls_alignment_validation['VALID'].sum())
# Calculate validated NDVI count to quantify completeness and support QA checks.
validated_ndvi_count = int((fire_hazard_hls_alignment_validation['INDEX_NAME'] == 'NDVI').sum())
# Calculate validated NDMI count to quantify completeness and support QA checks.
validated_ndmi_count = int((fire_hazard_hls_alignment_validation['INDEX_NAME'] == 'NDMI').sum())
# Calculate valid NDVI count to quantify completeness and support QA checks.
valid_ndvi_count = int(((fire_hazard_hls_alignment_validation['INDEX_NAME'] == 'NDVI') & \
    fire_hazard_hls_alignment_validation['VALID']).sum())
# Calculate valid NDMI count to quantify completeness and support QA checks.
valid_ndmi_count = int(((fire_hazard_hls_alignment_validation['INDEX_NAME'] == 'NDMI') & \
    fire_hazard_hls_alignment_validation['VALID']).sum())
# Calculate expected alignment scene count to quantify completeness and support QA checks.
expected_alignment_scene_count = fire_hazard_hls_expected_aligned_outputs[['COLLECTION_ID',
    'ITEM_ID']].drop_duplicates().shape[0]
# Calculate alignment validation record count complete to quantify completeness and support QA
# checks.
alignment_validation_record_count_complete = validated_aligned_raster_count == \
    expected_aligned_raster_count
# Record alignment validation complete so aligned scene-level NDVI and NDMI can preserve a clear
# processing and validation outcome.
fire_hazard_hls_alignment_validation_complete = alignment_validation_record_count_complete and \
    failed_hls_alignment_validations.empty and (valid_aligned_raster_count == \
    expected_aligned_raster_count)
# Record total hls alignment cache size bytes used to manage reproducible cached source data.
total_hls_alignment_cache_size_bytes = \
    int(fire_hazard_hls_alignment_validation['FILE_SIZE_BYTES'].sum())
# Record alignment validation summary so aligned scene-level NDVI and NDMI can preserve a clear
# processing and validation outcome.
fire_hazard_hls_alignment_validation_summary = pd.DataFrame([{'EXPECTED_SCENES': \
    expected_alignment_scene_count,
    'EXPECTED_ALIGNED_RASTERS': expected_aligned_raster_count,
    'VALIDATED_ALIGNED_RASTERS': validated_aligned_raster_count,
    'VALID_ALIGNED_RASTERS': valid_aligned_raster_count,
    'FAILED_ALIGNED_RASTERS': len(failed_hls_alignment_validations),
    'VALIDATED_NDVI_RASTERS': validated_ndvi_count,
    'VALID_NDVI_RASTERS': valid_ndvi_count, 'VALIDATED_NDMI_RASTERS': validated_ndmi_count,
    'VALID_NDMI_RASTERS': valid_ndmi_count, 'TARGET_CRS': fire_hazard_target_crs,
    'TARGET_CELL_SIZE': fire_hazard_cell_size, 'TARGET_WIDTH': fire_hazard_alignment_width,
    'TARGET_HEIGHT': fire_hazard_alignment_height,
    'TOTAL_CACHE_SIZE_BYTES': total_hls_alignment_cache_size_bytes,
    'TOTAL_CACHE_SIZE_GB': total_hls_alignment_cache_size_bytes / 1024 ** 3,
    'VALIDATION_COMPLETE': fire_hazard_hls_alignment_validation_complete}])
# Save the alignment validation CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_alignment_validation.to_csv(fire_hazard_hls_alignment_validation_path, index=False)
# Save the alignment validation summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_alignment_validation_summary.to_csv \
    (fire_hazard_hls_alignment_validation_summary_path,
    index=False)
# Save the failed HLS alignment validations CSV so later steps can reuse the recorded workflow
# results.
failed_hls_alignment_validations.to_csv(fire_hazard_hls_alignment_failed_validation_path,
    index=False)
print(f'-> Expected HLS scenes: {expected_alignment_scene_count:,}')
print(f'-> Expected aligned rasters: {expected_aligned_raster_count:,}')
print(f'-> Validated aligned rasters: {validated_aligned_raster_count:,}')
print(f'-> Valid aligned rasters: {valid_aligned_raster_count:,}')
print(f'-> Failed aligned rasters: {len(failed_hls_alignment_validations):,}')
print(f'-> Valid NDVI rasters: {valid_ndvi_count:,}')
print(f'-> Valid NDMI rasters: {valid_ndmi_count:,}')
print(f'-> Total alignment cache size: {total_hls_alignment_cache_size_bytes / 1024 ** 3:,.2f} GB')
print(f'-> Alignment validation complete: {fire_hazard_hls_alignment_validation_complete}')
print(f'-> Validation manifest saved: {fire_hazard_hls_alignment_validation_path}')
print('\n--- HLS ALIGNMENT VALIDATION SUMMARY ---')
display(fire_hazard_hls_alignment_validation_summary)
print('\n--- HLS ALIGNMENT VALIDATION SAMPLE ---')
display(fire_hazard_hls_alignment_validation[['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'FILE_EXISTS', 'GRID_VALID',
    'STRUCTURE_VALID', 'VALID_INSIDE_STUDY_AREA', 'VALID_OUTSIDE_STUDY_AREA',
    'INDEX_MINIMUM', 'INDEX_MAXIMUM', 'VALID']].head(5))

# Handle missing or invalid failed HLS alignment validations explicitly during aligned scene-level
# NDVI and NDMI.
if not failed_hls_alignment_validations.empty:
    print('\n--- FAILED HLS ALIGNMENT VALIDATION SAMPLE ---')
    display(failed_hls_alignment_validations[['COLLECTION_ID',
        'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH', 'FILE_EXISTS',
        'RASTER_READABLE', 'GRID_VALID', 'STRUCTURE_VALID',
        'VALID_INSIDE_STUDY_AREA', 'VALID_OUTSIDE_STUDY_AREA',
        'VALUE_RANGE_VALID', 'ERROR_MESSAGE']].head(5))

# Stop execution if one or more aligned HLS NDVI or NDMI rasters failed independent validation.
if not fire_hazard_hls_alignment_validation_complete:
    raise ValueError(f'One or more aligned HLS NDVI '
        f'or NDMI rasters failed '
        f'independent validation.\n\n'
        f'Expected rasters: '
        f'{expected_aligned_raster_count:,}\n'
        f'Valid rasters: '
        f'{valid_aligned_raster_count:,}\n'
        f'Failed rasters: '
        f'{len(failed_hls_alignment_validations):,}\n\n'
        f'Review the failed alignment '
        f'validation CSV:\n'
        f'{fire_hazard_hls_alignment_failed_validation_path}')
print('\nNOTE:')
print('Every validated NDVI and NDMI '
    'raster now uses the common '
    '30-meter EPSG:26912 analysis '
    'grid, contains no valid pixels '
    'outside the three study '
    'counties, and remains within '
    'the expected '
    'normalized-difference range.')
print('The next step is to create '
    'multi-date NDVI and NDMI '
    'composites from the validated '
    'aligned scene rasters.')
print('\n=== ALIGNED SCENE-LEVEL NDVI AND NDMI VALIDATION COMPLETE ===')


=== VALIDATING ALIGNED SCENE-LEVEL NDVI AND NDMI ===
-> Validating aligned raster 1 of 126...
-> Validating aligned raster 5 of 126...
-> Validating aligned raster 10 of 126...
-> Validating aligned raster 15 of 126...
-> Validating aligned raster 20 of 126...
-> Validating aligned raster 25 of 126...
-> Validating aligned raster 30 of 126...
-> Validating aligned raster 35 of 126...
-> Validating aligned raster 40 of 126...
-> Validating aligned raster 45 of 126...
-> Validating aligned raster 50 of 126...
-> Validating aligned raster 55 of 126...
-> Validating aligned raster 60 of 126...
-> Validating aligned raster 65 of 126...
-> Validating aligned raster 70 of 126...
-> Validating aligned raster 75 of 126...
-> Validating aligned raster 80 of 126...
-> Validating aligned raster 85 of 126...
-> Validating aligned raster 90 of 126...
-> Validating aligned raster 95 of 126...
-> Validating aligned raster 100 of 126...
-> Validating aligned raster 105 of 126...
-> Validating aligned r

,EXPECTED_SCENES,EXPECTED_ALIGNED_RASTERS,VALIDATED_ALIGNED_RASTERS,VALID_ALIGNED_RASTERS,FAILED_ALIGNED_RASTERS,VALIDATED_NDVI_RASTERS,VALID_NDVI_RASTERS,VALIDATED_NDMI_RASTERS,VALID_NDMI_RASTERS,TARGET_CRS,TARGET_CELL_SIZE,TARGET_WIDTH,TARGET_HEIGHT,TOTAL_CACHE_SIZE_BYTES,TOTAL_CACHE_SIZE_GB,VALIDATION_COMPLETE
0,63,126,126,126,0,63,63,63,63,EPSG:26912,30,9454,14467,2039622293,1.899546,True



--- HLS ALIGNMENT VALIDATION SAMPLE ---


,COLLECTION_ID,ITEM_ID,INDEX_NAME,FILE_EXISTS,GRID_VALID,STRUCTURE_VALID,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,INDEX_MINIMUM,INDEX_MAXIMUM,VALID
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDMI,True,True,True,19740,0,-0.243638,0.122167,True
1,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDVI,True,True,True,19740,0,0.072353,0.408986,True
2,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDMI,True,True,True,14951,0,-0.226472,0.220386,True
3,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDVI,True,True,True,14951,0,0.090153,0.406582,True
4,hls2-l30,HLS.L30.T11SQB.2025224T181513.v2.0,NDMI,True,True,True,3511372,0,-0.632768,1.000000,True



NOTE:
Every validated NDVI and NDMI raster now uses the common 30-meter EPSG:26912 analysis grid, contains no valid pixels outside the three study counties, and remains within the expected normalized-difference range.
The next step is to create multi-date NDVI and NDMI composites from the validated aligned scene rasters.

=== ALIGNED SCENE-LEVEL NDVI AND NDMI VALIDATION COMPLETE ===


## Temporal Composites and Vegetation-Dryness Hazard


### Configuring HLS Temporal Composite Workflow


In [84]:
print('=== CONFIGURING HLS TEMPORAL COMPOSITE WORKFLOW ===')
# List the required HLS composite configuration prerequisites required before HLS temporal
# composite workflow can run.
required_hls_composite_configuration_inputs = ['fire_hazard_hls_alignment_validation',
    'fire_hazard_hls_alignment_validation_complete',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_profile',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_nodata_value', 'fire_hazard_raster_dtype',
    'fire_hazard_hls_raw_directory']
# Identify unavailable HLS composite configuration so HLS temporal composite workflow
# stops before using incomplete inputs.
missing_hls_composite_configuration_inputs = [object_name for object_name in \
    required_hls_composite_configuration_inputs if object_name not in globals()]

# Use NDVI to represent live vegetation density and potential fuel availability.
# Stop execution when required hls composite configuration inputs inputs are unavailable.
if missing_hls_composite_configuration_inputs:
    raise NameError(f'The following HLS '
        f'temporal-composite '
        f'configuration objects are '
        f'missing:\n'
        f'{missing_hls_composite_configuration_inputs}\n\n'
        f'Run step Validate Aligned '
        f'Scene-Level NDVI and NDMI '
        f'before configuring the '
        f'temporal composite workflow.')

# Use NDMI to represent vegetation and canopy moisture conditions.
# Stop execution if aligned HLS NDVI and NDMI rasters have not passed complete independent
# validation.
if not fire_hazard_hls_alignment_validation_complete:
    raise ValueError('The aligned HLS NDVI and NDMI '
        'rasters have not passed '
        'complete independent '
        'validation.')

# Stop execution if hLS alignment validation table contains no aligned raster records.
if fire_hazard_hls_alignment_validation.empty:
    raise ValueError('The HLS alignment validation table contains no aligned raster records.')
# List the required HLS composite source fields prerequisites required before HLS temporal composite
# workflow can run.
required_hls_composite_source_fields = ['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH', 'VALID']
# Identify unavailable HLS composite source fields so HLS temporal composite workflow stops
# before using incomplete inputs.
missing_hls_composite_source_fields = [field_name for field_name in \
    required_hls_composite_source_fields if field_name not in \
    fire_hazard_hls_alignment_validation.columns]

# Stop execution when required hls composite source fields inputs are unavailable.
if missing_hls_composite_source_fields:
    raise ValueError(f'The HLS alignment validation '
        f'table is missing required '
        f'composite fields:\n'
        f'{missing_hls_composite_source_fields}')
# Store composite sources needed to carry out HLS temporal composite workflow.
fire_hazard_hls_composite_sources = \
    fire_hazard_hls_alignment_validation[fire_hazard_hls_alignment_validation['VALID']] \
    [required_hls_composite_source_fields].copy().reset_index(drop=True)
# Store composite sources['index name'] needed to carry out HLS temporal composite workflow.
fire_hazard_hls_composite_sources['INDEX_NAME'] = \
    fire_hazard_hls_composite_sources['INDEX_NAME'].astype(str).str.strip().str.upper()
# Identify unexpected unexpected HLS composite indices so only approved inputs reach HLS temporal
# composite workflow.
unexpected_hls_composite_indices = \
    sorted(set(fire_hazard_hls_composite_sources['INDEX_NAME'].unique()) - {'NDVI',
    'NDMI'})

# Stop execution if hLS composite inventory contains unsupported vegetation indices: the reported
# value.
if unexpected_hls_composite_indices:
    raise ValueError(f'The HLS composite inventory '
        f'contains unsupported '
        f'vegetation indices:\n'
        f'{unexpected_hls_composite_indices}')
# Identify duplicate duplicate HLS composite sources so repeated records do not bias HLS temporal
# composite workflow.
duplicate_hls_composite_sources = \
    fire_hazard_hls_composite_sources.duplicated(subset=['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME'], keep=False)

# Stop execution if hLS composite inventory contains duplicate collection-item-index records.
if duplicate_hls_composite_sources.any():
    raise ValueError('The HLS composite inventory '
        'contains duplicate '
        'collection-item-index records.')
# Identify duplicate duplicate HLS composite paths so repeated records do not bias HLS temporal
# composite workflow.
duplicate_hls_composite_paths = \
    fire_hazard_hls_composite_sources.duplicated(subset=['ALIGNED_PATH'],
    keep=False)

# Stop execution if hLS composite inventory contains duplicate aligned raster paths.
if duplicate_hls_composite_paths.any():
    raise ValueError('The HLS composite inventory contains duplicate aligned raster paths.')
# Identify unavailable HLS composite source paths so HLS temporal composite workflow stops
# before using incomplete inputs.
missing_hls_composite_source_paths = []

# Process each aligned path text entry so HLS temporal composite workflow is applied consistently
# across all records.
for aligned_path_text in fire_hazard_hls_composite_sources['ALIGNED_PATH']:
    # Build the aligned path location so HLS temporal composite workflow uses the expected project
    # file structure.
    aligned_path = Path(aligned_path_text)

    # Use the existing file only when it is present and valid for HLS temporal composite workflow.
    if not aligned_path.exists() or not aligned_path.is_file() or aligned_path.stat().st_size <= 0:
        missing_hls_composite_source_paths.append(str(aligned_path))

# Stop execution when required hls composite source paths inputs are unavailable.
if missing_hls_composite_source_paths:
    raise FileNotFoundError(f'One or more validated aligned '
        f'HLS rasters are unavailable.\n\n'
        f'First missing paths:\n'
        f'{missing_hls_composite_source_paths[:5]}')

# Combine valid scene observations with a temporal median to
# reduce cloud artifacts and scene-specific noise.
# Set composite method to control HLS temporal composite workflow.
fire_hazard_hls_composite_method = 'median'
# Define composite methods to control the inputs and rules used by HLS temporal composite workflow.
fire_hazard_hls_composite_methods = {'NDVI': 'median', 'NDMI': 'median'}

# Transform vegetation condition into a normalized dryness score
# where larger values represent greater fire potential.
# Define composite interpretation to control the inputs and rules used by HLS temporal composite
# workflow.
fire_hazard_hls_composite_interpretation = (
    {'NDVI': 'Lower values indicate reduced '
        'vegetation greenness or vigor '
        'and will correspond to higher '
        'vegetation-dryness hazard.', 'NDMI': 'Lower values indicate reduced ' \
            'vegetation or canopy moisture and will ' \
            'correspond to higher ' \
            'vegetation-dryness hazard.'}
)
# Track composite minimum observations across processed pixels for output-range QA.
fire_hazard_hls_composite_minimum_observations = 3
# Calculate composite count dtype to quantify completeness and support QA checks.
fire_hazard_hls_composite_count_dtype = 'uint16'
# Calculate composite count NoData to quantify completeness and support QA checks.
fire_hazard_hls_composite_count_nodata = 0
# Track composite minimum value across processed pixels for output-range QA.
fire_hazard_hls_composite_minimum_value = -1.0
# Track composite maximum value across processed pixels for output-range QA.
fire_hazard_hls_composite_maximum_value = 1.0
# Store composite dtype needed to carry out HLS temporal composite workflow.
fire_hazard_hls_composite_dtype = fire_hazard_raster_dtype
# Store composite NoData needed to carry out HLS temporal composite workflow.
fire_hazard_hls_composite_nodata = fire_hazard_nodata_value
# Set composite compression to control HLS temporal composite workflow.
fire_hazard_hls_composite_compression = 'deflate'
# Set composite process by window to control HLS temporal composite workflow.
fire_hazard_hls_composite_process_by_window = True
# Set composite window size to control HLS temporal composite workflow.
fire_hazard_hls_composite_window_size = 512
# Define composite index order to control the inputs and rules used by HLS temporal composite
# workflow.
fire_hazard_hls_composite_index_order = ['NDVI', 'NDMI']
# Set composite progress interval to control HLS temporal composite workflow.
fire_hazard_hls_composite_progress_interval = 25
# Build the composite directory location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_directory = fire_hazard_hls_raw_directory / 'temporal_composites'
# Build the composite raster directory location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_raster_directory = fire_hazard_hls_composite_directory / 'rasters'
# Build the composite count directory location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_count_directory = fire_hazard_hls_composite_directory / \
    'observation_counts'
# Build the composite metadata directory location so HLS temporal composite workflow uses the
# expected project file structure.
fire_hazard_hls_composite_metadata_directory = fire_hazard_hls_composite_directory / 'metadata'

# Process each directory path entry so HLS temporal composite workflow is applied consistently
# across all records.
for directory_path in [fire_hazard_hls_composite_directory,
    fire_hazard_hls_composite_raster_directory, fire_hazard_hls_composite_count_directory,
    fire_hazard_hls_composite_metadata_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the NDVI composite path location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_ndvi_composite_path = fire_hazard_hls_composite_raster_directory / \
    'hls_2025_fire_season_ndvi_median.tif'
# Build the NDMI composite path location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_ndmi_composite_path = fire_hazard_hls_composite_raster_directory / \
    'hls_2025_fire_season_ndmi_median.tif'
# Build the NDVI observation count path location so HLS temporal composite workflow uses the
# expected project file structure.
fire_hazard_hls_ndvi_observation_count_path = fire_hazard_hls_composite_count_directory / \
    'hls_2025_fire_season_ndvi_observation_count.tif'
# Build the NDMI observation count path location so HLS temporal composite workflow uses the
# expected project file structure.
fire_hazard_hls_ndmi_observation_count_path = fire_hazard_hls_composite_count_directory / \
    'hls_2025_fire_season_ndmi_observation_count.tif'
# Build the composite source manifest path location so HLS temporal composite workflow uses the
# expected project file structure.
fire_hazard_hls_composite_source_manifest_path = fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_source_manifest.csv'
# Build the composite manifest path location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_manifest_path = fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_manifest.csv'
# Build the composite policy path location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_policy_path = fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_policy.json'
# Build the composite configuration summary path location so HLS temporal composite workflow uses
# the expected project file structure.
fire_hazard_hls_composite_configuration_summary_path = \
    fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_configuration_summary.csv'
# Build the composite validation path location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_validation_path = fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_validation.csv'
# Build the composite validation summary path location so HLS temporal composite workflow uses the
# expected project file structure.
fire_hazard_hls_composite_validation_summary_path = fire_hazard_hls_composite_metadata_directory \
    / 'hls_temporal_composite_validation_summary.csv'
# Build the composite output paths location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_output_paths = {'NDVI': fire_hazard_hls_ndvi_composite_path,
    'NDMI': fire_hazard_hls_ndmi_composite_path}
# Build the composite count paths location so HLS temporal composite workflow uses the expected
# project file structure.
fire_hazard_hls_composite_count_paths = {'NDVI': fire_hazard_hls_ndvi_observation_count_path,
    'NDMI': fire_hazard_hls_ndmi_observation_count_path}
# Prepare composite profile to preserve and validate raster structure during HLS temporal composite
# workflow.
fire_hazard_hls_composite_profile = fire_hazard_alignment_profile.copy()
fire_hazard_hls_composite_profile.update({'driver': 'GTiff',
    'dtype': fire_hazard_hls_composite_dtype, 'count': 1,
    'nodata': fire_hazard_hls_composite_nodata, 'width': fire_hazard_alignment_width,
    'height': fire_hazard_alignment_height, 'crs': fire_hazard_target_crs,
    'transform': fire_hazard_alignment_transform, 'compress': fire_hazard_hls_composite_compression,
    'predictor': 3, 'tiled': True, 'BIGTIFF': 'IF_SAFER'})
# Calculate composite count profile to quantify completeness and support QA checks.
fire_hazard_hls_composite_count_profile = fire_hazard_alignment_profile.copy()
fire_hazard_hls_composite_count_profile.update({'driver': 'GTiff',
    'dtype': fire_hazard_hls_composite_count_dtype,
    'count': 1, 'nodata': fire_hazard_hls_composite_count_nodata,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': fire_hazard_hls_composite_compression,
    'predictor': 2, 'tiled': True, 'BIGTIFF': 'IF_SAFER'})
# Store composite reuse existing needed to carry out HLS temporal composite workflow.
fire_hazard_hls_composite_reuse_existing = globals().get('fire_hazard_data_mode',
    'snapshot') == 'snapshot'
# Store composite overwrite needed to carry out HLS temporal composite workflow.
fire_hazard_hls_composite_overwrite = globals().get('fire_hazard_data_mode', 'snapshot') == 'refresh'
# Calculate composite NDVI source count to quantify completeness and support QA checks.
fire_hazard_hls_composite_ndvi_source_count = \
    int((fire_hazard_hls_composite_sources['INDEX_NAME'] == 'NDVI').sum())
# Calculate composite NDMI source count to quantify completeness and support QA checks.
fire_hazard_hls_composite_ndmi_source_count = \
    int((fire_hazard_hls_composite_sources['INDEX_NAME'] == 'NDMI').sum())
# Calculate composite scene count to quantify completeness and support QA checks.
fire_hazard_hls_composite_scene_count = fire_hazard_hls_composite_sources[['COLLECTION_ID',
    'ITEM_ID']].drop_duplicates().shape[0]

# Stop execution if validated HLS inventory does not contain enough NDVI or NDMI rasters to support
# the configured temporal-composite minimum.
if fire_hazard_hls_composite_ndvi_source_count < fire_hazard_hls_composite_minimum_observations \
    or fire_hazard_hls_composite_ndmi_source_count < fire_hazard_hls_composite_minimum_observations:
    raise ValueError('The validated HLS inventory '
        'does not contain enough NDVI '
        'or NDMI rasters to support the '
        'configured temporal-composite '
        'minimum.')
# Collect NDVI scene keys for membership and completeness checks during HLS temporal composite
# workflow.
fire_hazard_hls_ndvi_scene_keys = \
    set(zip(fire_hazard_hls_composite_sources.loc[fire_hazard_hls_composite_sources['INDEX_NAME'] \
    == 'NDVI',
    'COLLECTION_ID'], \
        fire_hazard_hls_composite_sources.loc[fire_hazard_hls_composite_sources['INDEX_NAME'] == \
        'NDVI',
    'ITEM_ID']))
# Collect NDMI scene keys for membership and completeness checks during HLS temporal composite
# workflow.
fire_hazard_hls_ndmi_scene_keys = \
    set(zip(fire_hazard_hls_composite_sources.loc[fire_hazard_hls_composite_sources['INDEX_NAME'] \
    == 'NDMI',
    'COLLECTION_ID'], \
        fire_hazard_hls_composite_sources.loc[fire_hazard_hls_composite_sources['INDEX_NAME'] == \
        'NDMI',
    'ITEM_ID']))

# Stop execution if validated NDVI and NDMI source inventories do not contain matching HLS scene
# identifiers.
if fire_hazard_hls_ndvi_scene_keys != fire_hazard_hls_ndmi_scene_keys:
    raise ValueError('The validated NDVI and NDMI '
        'source inventories do not '
        'contain matching HLS scene '
        'identifiers.')
# Save the composite sources CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_composite_sources.to_csv(fire_hazard_hls_composite_source_manifest_path,
    index=False)
# Define composite policy to control the inputs and rules used by HLS temporal composite workflow.
fire_hazard_hls_composite_policy = {'data_source': 'NASA Harmonized Landsat Sentinel-2 Version 2.0',
    'access_platform': 'Microsoft Planetary Computer',
    'analysis_period': globals().get('fire_hazard_hls_analysis_period',
    '2025 fire season'), 'indices': ['NDVI', 'NDMI'],
    'composite_methods': fire_hazard_hls_composite_methods,
    'minimum_valid_observations': int(fire_hazard_hls_composite_minimum_observations),
    'valid_value_range': [float(fire_hazard_hls_composite_minimum_value),
    float(fire_hazard_hls_composite_maximum_value)],
    'target_crs': fire_hazard_target_crs, 'target_cell_size': float(fire_hazard_cell_size),
    'target_width': int(fire_hazard_alignment_width),
    'target_height': int(fire_hazard_alignment_height),
    'output_dtype': fire_hazard_hls_composite_dtype,
    'output_nodata': float(fire_hazard_hls_composite_nodata),
    'observation_count_dtype': fire_hazard_hls_composite_count_dtype,
    'observation_count_nodata': int(fire_hazard_hls_composite_count_nodata),
    'compression': fire_hazard_hls_composite_compression,
    'process_by_window': fire_hazard_hls_composite_process_by_window,
    'window_size': int(fire_hazard_hls_composite_window_size),
    'reuse_existing_outputs': fire_hazard_hls_composite_reuse_existing,
    'overwrite_existing_outputs': fire_hazard_hls_composite_overwrite,
    'interpretation': fire_hazard_hls_composite_interpretation}

# Open the file in a managed context so its handle closes reliably after HLS temporal composite
# workflow.
with fire_hazard_hls_composite_policy_path.open('w', encoding='utf-8') as composite_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_hls_composite_policy, composite_policy_file, indent=2)
# Store composite configuration summary needed to carry out HLS temporal composite workflow.
fire_hazard_hls_composite_configuration_summary = pd.DataFrame([{'CONTRIBUTING_HLS_SCENES': \
    fire_hazard_hls_composite_scene_count,
    'NDVI_SOURCE_RASTERS': fire_hazard_hls_composite_ndvi_source_count,
    'NDMI_SOURCE_RASTERS': fire_hazard_hls_composite_ndmi_source_count,
    'NDVI_COMPOSITE_METHOD': fire_hazard_hls_composite_methods['NDVI'],
    'NDMI_COMPOSITE_METHOD': fire_hazard_hls_composite_methods['NDMI'],
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'TARGET_CRS': fire_hazard_target_crs, 'TARGET_CELL_SIZE': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'PROCESSING_WINDOW_SIZE': fire_hazard_hls_composite_window_size,
    'REUSE_EXISTING_OUTPUTS': fire_hazard_hls_composite_reuse_existing,
    'OVERWRITE_EXISTING_OUTPUTS': fire_hazard_hls_composite_overwrite}])
# Save the composite configuration summary CSV so later steps can reuse the recorded workflow
# results.
fire_hazard_hls_composite_configuration_summary.to_csv \
    (fire_hazard_hls_composite_configuration_summary_path,
    index=False)
print(f'-> Contributing HLS scenes: {fire_hazard_hls_composite_scene_count:,}')
print(f'-> NDVI source rasters: {fire_hazard_hls_composite_ndvi_source_count:,}')
print(f'-> NDMI source rasters: {fire_hazard_hls_composite_ndmi_source_count:,}')
print(f"-> NDVI composite method: {fire_hazard_hls_composite_methods['NDVI']}")
print(f"-> NDMI composite method: {fire_hazard_hls_composite_methods['NDMI']}")
print(f'-> Minimum valid observations '
    f'per pixel: '
    f'{fire_hazard_hls_composite_minimum_observations:,}')
print(f'-> Processing window size: '
    f'{fire_hazard_hls_composite_window_size:,} '
    f'x '
    f'{fire_hazard_hls_composite_window_size:,}')
print(f'-> Reuse existing outputs: {fire_hazard_hls_composite_reuse_existing}')
print(f'-> Composite policy saved: {fire_hazard_hls_composite_policy_path}')
print(f'-> Source manifest saved: {fire_hazard_hls_composite_source_manifest_path}')
print('\n--- HLS TEMPORAL COMPOSITE CONFIGURATION ---')
display(fire_hazard_hls_composite_configuration_summary)
print('\n--- HLS TEMPORAL COMPOSITE SOURCE SAMPLE ---')
display(fire_hazard_hls_composite_sources[['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH', 'VALID']].head(5))
print('\nNOTE:')
print('The temporal-composite '
    'workflow is configured to '
    'calculate pixel-wise median '
    'NDVI and NDMI values from the '
    'validated aligned scene '
    'rasters.')
print(f'A composite pixel will be '
    f'retained only where at least '
    f'{fire_hazard_hls_composite_minimum_observations} '
    f'valid scene observations are '
    f'available.')
print('Observation-count rasters will '
    'be created with the final '
    'composites to document spatial '
    'and temporal data support.')
print('Lower composite NDVI and NDMI '
    'values will later be '
    'transformed into higher '
    'vegetation-dryness hazard '
    'scores.')
print('\n=== HLS TEMPORAL COMPOSITE WORKFLOW CONFIGURED ===')



=== CONFIGURING HLS TEMPORAL COMPOSITE WORKFLOW ===
-> Contributing HLS scenes: 63
-> NDVI source rasters: 63
-> NDMI source rasters: 63
-> NDVI composite method: median
-> NDMI composite method: median
-> Minimum valid observations per pixel: 3
-> Processing window size: 512 x 512
-> Reuse existing outputs: True
-> Composite policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_policy.json
-> Source manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_source_manifest.csv

--- HLS TEMPORAL COMPOSITE CONFIGURATION ---


,CONTRIBUTING_HLS_SCENES,NDVI_SOURCE_RASTERS,NDMI_SOURCE_RASTERS,NDVI_COMPOSITE_METHOD,NDMI_COMPOSITE_METHOD,MINIMUM_VALID_OBSERVATIONS,TARGET_CRS,TARGET_CELL_SIZE,TARGET_WIDTH,TARGET_HEIGHT,PROCESSING_WINDOW_SIZE,REUSE_EXISTING_OUTPUTS,OVERWRITE_EXISTING_OUTPUTS
0,63,63,63,median,median,3,EPSG:26912,30,9454,14467,512,True,False



--- HLS TEMPORAL COMPOSITE SOURCE SAMPLE ---


,COLLECTION_ID,ITEM_ID,INDEX_NAME,ALIGNED_PATH,VALID
0,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
1,hls2-l30,HLS.L30.T11SQA.2025224T181513.v2.0,NDVI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
2,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
3,hls2-l30,HLS.L30.T11SQA.2025249T180912.v2.0,NDVI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
4,hls2-l30,HLS.L30.T11SQB.2025224T181513.v2.0,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



NOTE:
The temporal-composite workflow is configured to calculate pixel-wise median NDVI and NDMI values from the validated aligned scene rasters.
A composite pixel will be retained only where at least 3 valid scene observations are available.
Observation-count rasters will be created with the final composites to document spatial and temporal data support.
Lower composite NDVI and NDMI values will later be transformed into higher vegetation-dryness hazard scores.

=== HLS TEMPORAL COMPOSITE WORKFLOW CONFIGURED ===


### Creating HLS Temporal Composite Scene Grouping Inventory


In [85]:
print('=== CREATING HLS TEMPORAL COMPOSITE SCENE GROUPING INVENTORY ===')
# List the required HLS scene grouping prerequisites required before HLS temporal composite
# scene grouping inventory can run.
required_hls_scene_grouping_inputs = ['fire_hazard_hls_composite_sources',
    'fire_hazard_hls_composite_index_order', 'fire_hazard_hls_composite_methods',
    'fire_hazard_hls_composite_output_paths', 'fire_hazard_hls_composite_count_paths',
    'fire_hazard_hls_composite_window_size', 'fire_hazard_hls_composite_minimum_observations',
    'fire_hazard_hls_composite_source_manifest_path',
    'fire_hazard_hls_composite_metadata_directory',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_target_crs', 'fire_hazard_cell_size']
# Identify unavailable HLS scene grouping so HLS temporal composite scene grouping
# inventory stops before using incomplete inputs.
missing_hls_scene_grouping_inputs = [object_name for object_name in \
    required_hls_scene_grouping_inputs if object_name not in globals()]

# Stop execution when required hls scene grouping inputs inputs are unavailable.
if missing_hls_scene_grouping_inputs:
    raise NameError(f'The following HLS '
        f'scene-grouping objects are '
        f'missing:\n'
        f'{missing_hls_scene_grouping_inputs}\n\n'
        f'Run step Configure HLS '
        f'Temporal Composite Workflow '
        f'before creating the scene '
        f'grouping inventory.')
# List the required HLS grouping source fields prerequisites required before HLS temporal composite
# scene grouping inventory can run.
required_hls_grouping_source_fields = ['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH', 'VALID']
# Identify unavailable HLS grouping source fields so HLS temporal composite scene grouping
# inventory stops before using incomplete inputs.
missing_hls_grouping_source_fields = [field_name for field_name in \
    required_hls_grouping_source_fields if field_name not in \
    fire_hazard_hls_composite_sources.columns]

# Stop execution when required hls grouping source fields inputs are unavailable.
if missing_hls_grouping_source_fields:
    raise ValueError(f'The HLS temporal-composite '
        f'source inventory is missing '
        f'required grouping fields:\n'
        f'{missing_hls_grouping_source_fields}')

# Stop execution if hLS temporal-composite source inventory contains no aligned raster records.
if fire_hazard_hls_composite_sources.empty:
    raise ValueError('The HLS temporal-composite '
        'source inventory contains no '
        'aligned raster records.')
# Store scene grouping inventory needed to carry out HLS temporal composite scene grouping
# inventory.
fire_hazard_hls_scene_grouping_inventory = \
    fire_hazard_hls_composite_sources[required_hls_grouping_source_fields].copy().reset_index \
    (drop=True)

# Process each field name entry so HLS temporal composite scene grouping inventory is applied
# consistently across all records.
for field_name in ['COLLECTION_ID', 'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH']:
    # Store scene grouping inventory[field name] needed to carry out HLS temporal composite scene
    # grouping inventory.
    fire_hazard_hls_scene_grouping_inventory[field_name] = \
        fire_hazard_hls_scene_grouping_inventory[field_name].astype(str).str.strip()
# Store scene grouping inventory['index name'] needed to carry out HLS temporal composite scene
# grouping inventory.
fire_hazard_hls_scene_grouping_inventory['INDEX_NAME'] = \
    fire_hazard_hls_scene_grouping_inventory['INDEX_NAME'].str.upper()

# Handle the object case explicitly during HLS temporal composite scene grouping inventory.
if fire_hazard_hls_scene_grouping_inventory['VALID'].dtype == object:
    # Record scene grouping inventory['valid'] so HLS temporal composite scene grouping inventory
    # can preserve a clear processing and validation outcome.
    fire_hazard_hls_scene_grouping_inventory['VALID'] = \
        fire_hazard_hls_scene_grouping_inventory['VALID'].astype(str).str.strip().str.lower().map \
        ({'true': True,
        'false': False})

# Stop execution if hLS scene grouping inventory contains missing or unrecognized VALID values.
if fire_hazard_hls_scene_grouping_inventory['VALID'].isna().any():
    raise ValueError('The HLS scene grouping '
        'inventory contains missing or '
        'unrecognized VALID values.')
# Store scene grouping inventory needed to carry out HLS temporal composite scene grouping
# inventory.
fire_hazard_hls_scene_grouping_inventory = \
    fire_hazard_hls_scene_grouping_inventory[fire_hazard_hls_scene_grouping_inventory['VALID']] \
    .copy().reset_index(drop=True)
# Collect configured HLS index names for membership and completeness checks during HLS temporal
# composite scene grouping inventory.
configured_hls_index_names = set(fire_hazard_hls_composite_index_order)
# Collect inventory HLS index names for membership and completeness checks during HLS temporal
# composite scene grouping inventory.
inventory_hls_index_names = set(fire_hazard_hls_scene_grouping_inventory['INDEX_NAME'].unique())
# Identify unexpected unexpected HLS grouping indices so only approved inputs reach HLS temporal
# composite scene grouping inventory.
unexpected_hls_grouping_indices = sorted(inventory_hls_index_names - configured_hls_index_names)
# Identify unavailable HLS grouping indices so HLS temporal composite scene grouping inventory
# stops before using incomplete inputs.
missing_hls_grouping_indices = sorted(configured_hls_index_names - inventory_hls_index_names)

# Stop execution if grouping inventory contains unsupported vegetation indices: the reported value.
if unexpected_hls_grouping_indices:
    raise ValueError(f'The grouping inventory '
        f'contains unsupported '
        f'vegetation indices:\n'
        f'{unexpected_hls_grouping_indices}')

# Stop execution when required hls grouping indices inputs are unavailable.
if missing_hls_grouping_indices:
    raise ValueError(f'The grouping inventory is '
        f'missing configured vegetation '
        f'indices:\n'
        f'{missing_hls_grouping_indices}')
# Build duplicate hls grouping records used to track the records included in this processing stage.
duplicate_hls_grouping_records = \
    fire_hazard_hls_scene_grouping_inventory.duplicated(subset=['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME'], keep=False)

# Stop execution if hLS scene grouping inventory contains duplicate collection-item-index records.
if duplicate_hls_grouping_records.any():
    raise ValueError('The HLS scene grouping '
        'inventory contains duplicate '
        'collection-item-index records.')
# Identify duplicate duplicate HLS grouping paths so repeated records do not bias HLS temporal
# composite scene grouping inventory.
duplicate_hls_grouping_paths = \
    fire_hazard_hls_scene_grouping_inventory.duplicated(subset=['ALIGNED_PATH'],
    keep=False)

# Stop execution if hLS scene grouping inventory contains duplicate aligned raster paths.
if duplicate_hls_grouping_paths.any():
    raise ValueError('The HLS scene grouping inventory contains duplicate aligned raster paths.')
# Store scene grouping inventory['sensor family'] needed to carry out HLS temporal composite scene
# grouping inventory.
fire_hazard_hls_scene_grouping_inventory['SENSOR_FAMILY'] = \
    fire_hazard_hls_scene_grouping_inventory['COLLECTION_ID'].str.lower().map({'hls2-l30': \
    'Landsat',
    'hls2-s30': 'Sentinel-2'})

# Stop execution if following HLS collection identifiers could not be assigned a sensor family: the
# reported value.
if fire_hazard_hls_scene_grouping_inventory['SENSOR_FAMILY'].isna().any():
    # Record invalid collection IDs needed to identify the current HLS asset in manifests and
    # requests.
    invalid_collection_ids = \
        sorted(fire_hazard_hls_scene_grouping_inventory.loc \
        [fire_hazard_hls_scene_grouping_inventory['SENSOR_FAMILY'].isna(),
        'COLLECTION_ID'].unique())
    raise ValueError(f'The following HLS collection '
        f'identifiers could not be '
        f'assigned a sensor family:\n'
        f'{invalid_collection_ids}')
# Store scene grouping inventory['MGRS tile'] needed to carry out HLS temporal composite scene
# grouping inventory.
fire_hazard_hls_scene_grouping_inventory['MGRS_TILE'] = \
    fire_hazard_hls_scene_grouping_inventory['ITEM_ID'].str.extract('\\.(T[0-9]{2}[A-Z]{3})\\.',
    expand=False)

# Stop execution if mGRS tile identifiers could not be extracted from one or more HLS item IDs.
if fire_hazard_hls_scene_grouping_inventory['MGRS_TILE'].isna().any():
    # Record invalid tile item IDs needed to identify the current HLS asset in manifests and
    # requests.
    invalid_tile_item_ids = \
        fire_hazard_hls_scene_grouping_inventory.loc[fire_hazard_hls_scene_grouping_inventory \
        ['MGRS_TILE'].isna(),
        'ITEM_ID'].drop_duplicates().tolist()
    raise ValueError(f'MGRS tile identifiers could '
        f'not be extracted from one or '
        f'more HLS item IDs.\n\nFirst '
        f'affected item IDs:\n'
        f'{invalid_tile_item_ids[:5]}')
# Store scene grouping inventory['acquisition token'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['ACQUISITION_TOKEN'] = \
    fire_hazard_hls_scene_grouping_inventory['ITEM_ID'].str.extract('\\.([0-9]{7}T[0-9]{6})\\.',
    expand=False)

# Stop execution if acquisition timestamps could not be extracted from one or more HLS item IDs.
if fire_hazard_hls_scene_grouping_inventory['ACQUISITION_TOKEN'].isna().any():
    # Record invalid date item IDs needed to identify the current HLS asset in manifests and
    # requests.
    invalid_date_item_ids = \
        fire_hazard_hls_scene_grouping_inventory.loc[fire_hazard_hls_scene_grouping_inventory \
        ['ACQUISITION_TOKEN'].isna(),
        'ITEM_ID'].drop_duplicates().tolist()
    raise ValueError(f'Acquisition timestamps could '
        f'not be extracted from one or '
        f'more HLS item IDs.\n\nFirst '
        f'affected item IDs:\n'
        f'{invalid_date_item_ids[:5]}')
# Store scene grouping inventory['acquisition datetime'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['ACQUISITION_DATETIME'] = \
    pd.to_datetime(fire_hazard_hls_scene_grouping_inventory['ACQUISITION_TOKEN'],
    format='%Y%jT%H%M%S', errors='coerce', utc=True)

# Stop execution if one or more extracted HLS acquisition timestamps could not be parsed.
if fire_hazard_hls_scene_grouping_inventory['ACQUISITION_DATETIME'].isna().any():
    raise ValueError('One or more extracted HLS acquisition timestamps could not be parsed.')
# Store scene grouping inventory['acquisition date'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['ACQUISITION_DATE'] = \
    fire_hazard_hls_scene_grouping_inventory['ACQUISITION_DATETIME'].dt.strftime('%Y-%m-%d')
# Store scene grouping inventory['acquisition year'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['ACQUISITION_YEAR'] = \
    fire_hazard_hls_scene_grouping_inventory['ACQUISITION_DATETIME'].dt.year.astype(int)
# Store scene grouping inventory['julian day'] needed to carry out HLS temporal composite scene
# grouping inventory.
fire_hazard_hls_scene_grouping_inventory['JULIAN_DAY'] = \
    fire_hazard_hls_scene_grouping_inventory['ACQUISITION_DATETIME'].dt.dayofyear.astype(int)
# Store scene grouping inventory['acquisition month'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['ACQUISITION_MONTH'] = \
    fire_hazard_hls_scene_grouping_inventory['ACQUISITION_DATETIME'].dt.month.astype(int)
# Store scene grouping inventory['file exists'] needed to carry out HLS temporal composite scene
# grouping inventory.
fire_hazard_hls_scene_grouping_inventory['FILE_EXISTS'] = [Path(aligned_path).exists() and \
    Path(aligned_path).is_file() for aligned_path in \
    fire_hazard_hls_scene_grouping_inventory['ALIGNED_PATH']]
# Store scene grouping inventory['file size bytes'] needed to carry out HLS temporal composite scene
# grouping inventory.
fire_hazard_hls_scene_grouping_inventory['FILE_SIZE_BYTES'] = [Path(aligned_path).stat().st_size \
    if Path(aligned_path).exists() and Path(aligned_path).is_file() else 0 for aligned_path in \
    fire_hazard_hls_scene_grouping_inventory['ALIGNED_PATH']]
# Record scene grouping inventory['source file valid'] so HLS temporal composite scene grouping
# inventory can preserve a clear processing and validation outcome.
fire_hazard_hls_scene_grouping_inventory['SOURCE_FILE_VALID'] = \
    fire_hazard_hls_scene_grouping_inventory['FILE_EXISTS'] & \
    (fire_hazard_hls_scene_grouping_inventory['FILE_SIZE_BYTES'] > 0)

# Stop execution if one or more grouped HLS aligned rasters are missing or empty.
if not fire_hazard_hls_scene_grouping_inventory['SOURCE_FILE_VALID'].all():
    # Identify unavailable grouped source files so HLS temporal composite scene grouping
    # inventory stops before using incomplete inputs.
    missing_grouped_source_files = \
        fire_hazard_hls_scene_grouping_inventory[~fire_hazard_hls_scene_grouping_inventory \
        ['SOURCE_FILE_VALID']][['COLLECTION_ID',
        'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH']]
    raise FileNotFoundError(f"One or more grouped HLS "
        f"aligned rasters are missing or "
        f"empty.\n\nFirst affected "
        f"records:\n"
        f"{missing_grouped_source_files.head(5).to_dict('records')}")
# Store scene grouping inventory['composite group id'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['COMPOSITE_GROUP_ID'] = 'HLS_2025_FIRE_SEASON_' + \
    fire_hazard_hls_scene_grouping_inventory['INDEX_NAME'] + '_MEDIAN'
# Store scene grouping inventory['composite method'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['COMPOSITE_METHOD'] = \
    fire_hazard_hls_scene_grouping_inventory['INDEX_NAME'].map(fire_hazard_hls_composite_methods)
# Build the scene grouping inventory['composite output path'] location so HLS temporal composite
# scene grouping inventory uses the expected project file structure.
fire_hazard_hls_scene_grouping_inventory['COMPOSITE_OUTPUT_PATH'] = \
    fire_hazard_hls_scene_grouping_inventory['INDEX_NAME'].map({index_name: str(output_path) for \
    index_name,
    output_path in fire_hazard_hls_composite_output_paths.items()})
# Build the scene grouping inventory['observation count output path'] location so HLS temporal
# composite scene grouping inventory uses the expected project file structure.
fire_hazard_hls_scene_grouping_inventory['OBSERVATION_COUNT_OUTPUT_PATH'] = \
    fire_hazard_hls_scene_grouping_inventory['INDEX_NAME'].map({index_name: str(output_path) for \
    index_name,
    output_path in fire_hazard_hls_composite_count_paths.items()})
# Store scene grouping inventory needed to carry out HLS temporal composite scene grouping
# inventory.
fire_hazard_hls_scene_grouping_inventory = \
    fire_hazard_hls_scene_grouping_inventory.sort_values(['INDEX_NAME',
    'ACQUISITION_DATETIME', 'COLLECTION_ID', 'ITEM_ID']).reset_index(drop=True)
# Store scene grouping inventory['group source number'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['GROUP_SOURCE_NUMBER'] = \
    fire_hazard_hls_scene_grouping_inventory.groupby('INDEX_NAME').cumcount() + 1
# Store scene grouping inventory['group source count'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_inventory['GROUP_SOURCE_COUNT'] = \
    fire_hazard_hls_scene_grouping_inventory.groupby('INDEX_NAME')['ITEM_ID'].transform('count')
# Store NDVI composite sources needed to carry out HLS temporal composite scene grouping inventory.
fire_hazard_hls_ndvi_composite_sources = \
    fire_hazard_hls_scene_grouping_inventory[fire_hazard_hls_scene_grouping_inventory \
    ['INDEX_NAME'] == 'NDVI'].copy().reset_index(drop=True)
# Store NDMI composite sources needed to carry out HLS temporal composite scene grouping inventory.
fire_hazard_hls_ndmi_composite_sources = \
    fire_hazard_hls_scene_grouping_inventory[fire_hazard_hls_scene_grouping_inventory \
    ['INDEX_NAME'] == 'NDMI'].copy().reset_index(drop=True)
# Collect NDVI group scene keys for membership and completeness checks during HLS temporal composite
# scene grouping inventory.
ndvi_group_scene_keys = set(zip(fire_hazard_hls_ndvi_composite_sources['COLLECTION_ID'],
    fire_hazard_hls_ndvi_composite_sources['ITEM_ID']))
# Collect NDMI group scene keys for membership and completeness checks during HLS temporal composite
# scene grouping inventory.
ndmi_group_scene_keys = set(zip(fire_hazard_hls_ndmi_composite_sources['COLLECTION_ID'],
    fire_hazard_hls_ndmi_composite_sources['ITEM_ID']))

# Stop execution if grouped NDVI and NDMI inventories do not contain identical contributing HLS
# scenes.
if ndvi_group_scene_keys != ndmi_group_scene_keys:
    # Store NDVI only scene keys needed to carry out HLS temporal composite scene grouping
    # inventory.
    ndvi_only_scene_keys = sorted(ndvi_group_scene_keys - ndmi_group_scene_keys)
    # Store NDMI only scene keys needed to carry out HLS temporal composite scene grouping
    # inventory.
    ndmi_only_scene_keys = sorted(ndmi_group_scene_keys - ndvi_group_scene_keys)
    raise ValueError(f'The grouped NDVI and NDMI '
        f'inventories do not contain '
        f'identical contributing HLS '
        f'scenes.\n\nNDVI-only scenes: '
        f'{ndvi_only_scene_keys[:5]}\n'
        f'NDMI-only scenes: '
        f'{ndmi_only_scene_keys[:5]}')

# Stop execution if one or both vegetation-index groups contain fewer source rasters than the
# configured minimum valid observation requirement.
if len(fire_hazard_hls_ndvi_composite_sources) < fire_hazard_hls_composite_minimum_observations \
    or len(fire_hazard_hls_ndmi_composite_sources) < fire_hazard_hls_composite_minimum_observations:
    raise ValueError('One or both vegetation-index '
        'groups contain fewer source '
        'rasters than the configured '
        'minimum valid observation '
        'requirement.')
# Define fire hazard hls composite window columns needed to keep the analysis schema consistent.
fire_hazard_hls_composite_window_columns = int(np.ceil(fire_hazard_alignment_width / \
    fire_hazard_hls_composite_window_size))
# Store composite window rows needed to carry out HLS temporal composite scene grouping inventory.
fire_hazard_hls_composite_window_rows = int(np.ceil(fire_hazard_alignment_height / \
    fire_hazard_hls_composite_window_size))
# Define composite window records to control the inputs and rules used by HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_composite_window_records = []
# Set window number to control HLS temporal composite scene grouping inventory.
window_number = 0

# Process each window row entry so HLS temporal composite scene grouping inventory is applied
# consistently across all records.
for window_row in range(fire_hazard_hls_composite_window_rows):
    # Calculate row offset for the current raster processing window.
    row_offset = window_row * fire_hazard_hls_composite_window_size
    # Store window height needed to carry out HLS temporal composite scene grouping inventory.
    window_height = min(fire_hazard_hls_composite_window_size,
        fire_hazard_alignment_height - row_offset)

    # Process each window column entry so HLS temporal composite scene grouping inventory is applied
    # consistently across all records.
    for window_column in range(fire_hazard_hls_composite_window_columns):
        # Calculate column offset for the current raster processing window.
        column_offset = window_column * fire_hazard_hls_composite_window_size
        # Store window width needed to carry out HLS temporal composite scene grouping inventory.
        window_width = min(fire_hazard_hls_composite_window_size,
            fire_hazard_alignment_width - column_offset)
        # Set window number to control HLS temporal composite scene grouping inventory.
        window_number += 1
        fire_hazard_hls_composite_window_records.append({'WINDOW_NUMBER': window_number,
            'WINDOW_ROW': window_row, 'WINDOW_COLUMN': window_column,
            'ROW_OFFSET': row_offset, 'COLUMN_OFFSET': column_offset,
            'WINDOW_HEIGHT': window_height, 'WINDOW_WIDTH': window_width,
            'PIXEL_COUNT': window_height * window_width})
# Store composite window inventory needed to carry out HLS temporal composite scene grouping
# inventory.
fire_hazard_hls_composite_window_inventory = pd.DataFrame(fire_hazard_hls_composite_window_records)
# Calculate planned composite pixel count to quantify completeness and support QA checks.
planned_composite_pixel_count = int(fire_hazard_hls_composite_window_inventory['PIXEL_COUNT'].sum())
# Calculate expected composite pixel count to quantify completeness and support QA checks.
expected_composite_pixel_count = int(fire_hazard_alignment_width * fire_hazard_alignment_height)
# Record composite window plan valid so HLS temporal composite scene grouping inventory can preserve
# a clear processing and validation outcome.
fire_hazard_hls_composite_window_plan_valid = planned_composite_pixel_count == \
    expected_composite_pixel_count

# Stop execution if temporal-composite processing windows do not cover the complete target raster
# grid.
if not fire_hazard_hls_composite_window_plan_valid:
    raise ValueError(f'The temporal-composite '
        f'processing windows do not '
        f'cover the complete target '
        f'raster grid.\nPlanned pixels: '
        f'{planned_composite_pixel_count:,}\n'
        f'Expected pixels: '
        f'{expected_composite_pixel_count:,}')
# Build the scene grouping inventory path location so HLS temporal composite scene grouping
# inventory uses the expected project file structure.
fire_hazard_hls_scene_grouping_inventory_path = fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_scene_groups.csv'
# Build the composite window inventory path location so HLS temporal composite scene grouping
# inventory uses the expected project file structure.
fire_hazard_hls_composite_window_inventory_path = fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_window_inventory.csv'
# Build the scene grouping summary path location so HLS temporal composite scene grouping inventory
# uses the expected project file structure.
fire_hazard_hls_scene_grouping_summary_path = fire_hazard_hls_composite_metadata_directory / \
    'hls_temporal_composite_scene_group_summary.csv'
# Store scene grouping summary needed to carry out HLS temporal composite scene grouping inventory.
fire_hazard_hls_scene_grouping_summary = \
    fire_hazard_hls_scene_grouping_inventory.groupby(['COMPOSITE_GROUP_ID',
    'INDEX_NAME', 'COMPOSITE_METHOD', 'COMPOSITE_OUTPUT_PATH',
    'OBSERVATION_COUNT_OUTPUT_PATH'], as_index=False).agg(SOURCE_RASTER_COUNT=('ITEM_ID',
    'count'), UNIQUE_SCENE_COUNT=('ITEM_ID', 'nunique'),
    UNIQUE_TILE_COUNT=('MGRS_TILE', 'nunique'), LANDSAT_SCENE_COUNT=('SENSOR_FAMILY',
    lambda values: int((values == 'Landsat').sum())),
    SENTINEL2_SCENE_COUNT=('SENSOR_FAMILY', lambda values: int((values == 'Sentinel-2').sum())),
    FIRST_ACQUISITION_DATE=('ACQUISITION_DATE', 'min'),
    LAST_ACQUISITION_DATE=('ACQUISITION_DATE', 'max'),
    TOTAL_SOURCE_SIZE_BYTES=('FILE_SIZE_BYTES', 'sum'))
# Record scene grouping summary['minimum valid observations'] so HLS temporal composite scene
# grouping inventory can preserve a clear processing and validation outcome.
fire_hazard_hls_scene_grouping_summary['MINIMUM_VALID_OBSERVATIONS'] = \
    fire_hazard_hls_composite_minimum_observations
# Store scene grouping summary['processing window count'] needed to carry out HLS temporal composite
# scene grouping inventory.
fire_hazard_hls_scene_grouping_summary['PROCESSING_WINDOW_COUNT'] = \
    len(fire_hazard_hls_composite_window_inventory)
# Save the scene grouping inventory CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_scene_grouping_inventory.to_csv(fire_hazard_hls_scene_grouping_inventory_path,
    index=False)
# Save the composite window inventory CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_composite_window_inventory.to_csv(fire_hazard_hls_composite_window_inventory_path,
    index=False)
# Save the scene grouping summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_scene_grouping_summary.to_csv(fire_hazard_hls_scene_grouping_summary_path,
    index=False)
print(f'-> Grouped source rasters: {len(fire_hazard_hls_scene_grouping_inventory):,}')
print(f'-> Contributing HLS scenes: {len(ndvi_group_scene_keys):,}')
print(f'-> NDVI source rasters: {len(fire_hazard_hls_ndvi_composite_sources):,}')
print(f'-> NDMI source rasters: {len(fire_hazard_hls_ndmi_composite_sources):,}')
print(f"-> Unique HLS tiles: {fire_hazard_hls_scene_grouping_inventory['MGRS_TILE'].nunique():,}")
print(f'-> Processing windows: {len(fire_hazard_hls_composite_window_inventory):,}')
print(f'-> Window plan valid: {fire_hazard_hls_composite_window_plan_valid}')
print(f'-> Scene grouping inventory saved: {fire_hazard_hls_scene_grouping_inventory_path}')
print(f'-> Processing window inventory saved: {fire_hazard_hls_composite_window_inventory_path}')
print('\n--- HLS TEMPORAL COMPOSITE SCENE GROUP SUMMARY ---')
display(fire_hazard_hls_scene_grouping_summary)
print('\n--- HLS SCENE GROUPING SAMPLE ---')
display(fire_hazard_hls_scene_grouping_inventory[['COMPOSITE_GROUP_ID',
    'GROUP_SOURCE_NUMBER', 'GROUP_SOURCE_COUNT', 'COLLECTION_ID',
    'SENSOR_FAMILY', 'ITEM_ID', 'MGRS_TILE', 'INDEX_NAME',
    'ACQUISITION_DATE', 'COMPOSITE_METHOD', 'ALIGNED_PATH']].head(5))
print('\n--- HLS COMPOSITE PROCESSING WINDOW SAMPLE ---')
display(fire_hazard_hls_composite_window_inventory.head(5))
print('\nNOTE:')
print('The validated aligned HLS '
    'rasters are now separated into '
    'ordered NDVI and NDMI source '
    'groups.')
print('Each group contains matching '
    'contributing HLS scenes '
    'arranged chronologically for '
    'reproducible pixel-wise '
    'temporal compositing.')
print('The common target grid has '
    'also been divided into '
    'memory-safe processing windows '
    'for use by the NDVI and NDMI '
    'composite calculations.')
print('\n=== HLS TEMPORAL COMPOSITE SCENE GROUPING COMPLETE ===')



=== CREATING HLS TEMPORAL COMPOSITE SCENE GROUPING INVENTORY ===
-> Grouped source rasters: 126
-> Contributing HLS scenes: 63
-> NDVI source rasters: 63
-> NDMI source rasters: 63
-> Unique HLS tiles: 11
-> Processing windows: 551
-> Window plan valid: True
-> Scene grouping inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_scene_groups.csv
-> Processing window inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_window_inventory.csv

--- HLS TEMPORAL COMPOSITE SCENE GROUP SUMMARY ---


,COMPOSITE_GROUP_ID,INDEX_NAME,COMPOSITE_METHOD,COMPOSITE_OUTPUT_PATH,OBSERVATION_COUNT_OUTPUT_PATH,SOURCE_RASTER_COUNT,UNIQUE_SCENE_COUNT,UNIQUE_TILE_COUNT,LANDSAT_SCENE_COUNT,SENTINEL2_SCENE_COUNT,FIRST_ACQUISITION_DATE,LAST_ACQUISITION_DATE,TOTAL_SOURCE_SIZE_BYTES,MINIMUM_VALID_OBSERVATIONS,PROCESSING_WINDOW_COUNT
0,HLS_2025_FIRE_SEASON_NDMI_MEDIAN,NDMI,median,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,63,63,11,20,43,2025-07-05,2025-09-25,1048019924,3,551
1,HLS_2025_FIRE_SEASON_NDVI_MEDIAN,NDVI,median,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,63,63,11,20,43,2025-07-05,2025-09-25,991602369,3,551



--- HLS SCENE GROUPING SAMPLE ---


,COMPOSITE_GROUP_ID,GROUP_SOURCE_NUMBER,GROUP_SOURCE_COUNT,COLLECTION_ID,SENSOR_FAMILY,ITEM_ID,MGRS_TILE,INDEX_NAME,ACQUISITION_DATE,COMPOSITE_METHOD,ALIGNED_PATH
0,HLS_2025_FIRE_SEASON_NDMI_MEDIAN,1,63,hls2-s30,Sentinel-2,HLS.S30.T11SQA.2025186T180941.v2.0,T11SQA,NDMI,2025-07-05,median,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
1,HLS_2025_FIRE_SEASON_NDMI_MEDIAN,2,63,hls2-s30,Sentinel-2,HLS.S30.T12STF.2025186T180941.v2.0,T12STF,NDMI,2025-07-05,median,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
2,HLS_2025_FIRE_SEASON_NDMI_MEDIAN,3,63,hls2-s30,Sentinel-2,HLS.S30.T12STG.2025186T180941.v2.0,T12STG,NDMI,2025-07-05,median,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
3,HLS_2025_FIRE_SEASON_NDMI_MEDIAN,4,63,hls2-s30,Sentinel-2,HLS.S30.T11SQA.2025188T181801.v2.0,T11SQA,NDMI,2025-07-07,median,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
4,HLS_2025_FIRE_SEASON_NDMI_MEDIAN,5,63,hls2-s30,Sentinel-2,HLS.S30.T12STF.2025188T181801.v2.0,T12STF,NDMI,2025-07-07,median,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...



--- HLS COMPOSITE PROCESSING WINDOW SAMPLE ---


,WINDOW_NUMBER,WINDOW_ROW,WINDOW_COLUMN,ROW_OFFSET,COLUMN_OFFSET,WINDOW_HEIGHT,WINDOW_WIDTH,PIXEL_COUNT
0,1,0,0,0,0,512,512,262144
1,2,0,1,0,512,512,512,262144
2,3,0,2,0,1024,512,512,262144
3,4,0,3,0,1536,512,512,262144
4,5,0,4,0,2048,512,512,262144



NOTE:
The validated aligned HLS rasters are now separated into ordered NDVI and NDMI source groups.
Each group contains matching contributing HLS scenes arranged chronologically for reproducible pixel-wise temporal compositing.
The common target grid has also been divided into memory-safe processing windows for use by the NDVI and NDMI composite calculations.

=== HLS TEMPORAL COMPOSITE SCENE GROUPING COMPLETE ===


### Building HLS Ndvi Temporal Median Composite


In [86]:
print('=== BUILDING HLS NDVI TEMPORAL MEDIAN COMPOSITE ===')
# List the required HLS NDVI composite prerequisites required before HLS NDVI temporal median
# composite can run.
required_hls_ndvi_composite_inputs = ['fire_hazard_hls_ndvi_composite_sources',
    'fire_hazard_hls_composite_window_inventory', 'fire_hazard_hls_ndvi_composite_path',
    'fire_hazard_hls_ndvi_observation_count_path',
    'fire_hazard_hls_composite_profile', 'fire_hazard_hls_composite_count_profile',
    'fire_hazard_hls_composite_minimum_observations',
    'fire_hazard_hls_composite_minimum_value', 'fire_hazard_hls_composite_maximum_value',
    'fire_hazard_hls_composite_nodata', 'fire_hazard_hls_composite_count_nodata',
    'fire_hazard_hls_composite_dtype', 'fire_hazard_hls_composite_count_dtype',
    'fire_hazard_hls_composite_method', 'fire_hazard_hls_composite_progress_interval',
    'fire_hazard_hls_composite_reuse_existing', 'fire_hazard_hls_composite_overwrite',
    'fire_hazard_hls_composite_manifest_path', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_nodata_value', 'fire_hazard_raster_dtype']
# Identify unavailable HLS NDVI composite so HLS NDVI temporal median composite stops
# before using incomplete inputs.
missing_hls_ndvi_composite_inputs = [object_name for object_name in \
    required_hls_ndvi_composite_inputs if object_name not in globals()]

# Stop execution when required hls ndvi composite inputs inputs are unavailable.
if missing_hls_ndvi_composite_inputs:
    raise NameError(f'The following HLS NDVI '
        f'composite objects are missing:\n'
        f'{missing_hls_ndvi_composite_inputs}\n\n'
        f'Run Configure HLS Temporal '
        f'Composite Workflow and Create '
        f'HLS Scene Grouping Inventory '
        f'before building the NDVI '
        f'composite.')
# List the required HLS NDVI source fields prerequisites required before HLS NDVI temporal median
# composite can run.
required_hls_ndvi_source_fields = ['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH', 'SOURCE_FILE_VALID']
# Identify unavailable HLS NDVI source fields so HLS NDVI temporal median composite stops before
# using incomplete inputs.
missing_hls_ndvi_source_fields = [field_name for field_name in required_hls_ndvi_source_fields if \
    field_name not in fire_hazard_hls_ndvi_composite_sources.columns]

# Stop execution when required hls ndvi source fields inputs are unavailable.
if missing_hls_ndvi_source_fields:
    raise ValueError(f'The NDVI composite source '
        f'inventory is missing required '
        f'fields:\n'
        f'{missing_hls_ndvi_source_fields}')

# Stop execution if nDVI composite source inventory contains no aligned rasters.
if fire_hazard_hls_ndvi_composite_sources.empty:
    raise ValueError('The NDVI composite source inventory contains no aligned rasters.')

# Stop execution if nDVI composite inventory contains one or more non-NDVI source records.
if not fire_hazard_hls_ndvi_composite_sources['INDEX_NAME'].astype(str).str.upper().eq('NDVI') \
    .all():
    raise ValueError('The NDVI composite inventory contains one or more non-NDVI source records.')

# Stop execution if one or more aligned NDVI source rasters are missing or invalid.
if not fire_hazard_hls_ndvi_composite_sources['SOURCE_FILE_VALID'].all():
    raise ValueError('One or more aligned NDVI source rasters are missing or invalid.')
# Calculate NDVI source count to quantify completeness and support QA checks.
fire_hazard_hls_ndvi_source_count = len(fire_hazard_hls_ndvi_composite_sources)

# Stop execution if nDVI source inventory contains fewer rasters than the configured minimum valid
# observation requirement.
if fire_hazard_hls_ndvi_source_count < fire_hazard_hls_composite_minimum_observations:
    raise ValueError('The NDVI source inventory '
        'contains fewer rasters than '
        'the configured minimum valid '
        'observation requirement.')
# Build the NDVI composite temporary path location so HLS NDVI temporal median composite uses the
# expected project file structure.
fire_hazard_hls_ndvi_composite_temporary_path = fire_hazard_hls_ndvi_composite_path.parent / \
    (fire_hazard_hls_ndvi_composite_path.stem + '.part.tif')
# Build the NDVI count temporary path location so HLS NDVI temporal median composite uses the
# expected project file structure.
fire_hazard_hls_ndvi_count_temporary_path = fire_hazard_hls_ndvi_observation_count_path.parent / \
    (fire_hazard_hls_ndvi_observation_count_path.stem + '.part.tif')

# Encapsulate validate existing HLS NDVI composite so repeated HLS NDVI temporal median composite
# steps use consistent logic.
def validate_existing_hls_ndvi_composite(composite_path, count_path):

    """
    Checks whether existing NDVI composite and
    observation-count rasters match the target grid.
    """
    # Build the composite path location so HLS NDVI temporal median composite uses the expected
    # project file structure.
    composite_path = Path(composite_path)
    # Build the count path location so HLS NDVI temporal median composite uses the expected project
    # file structure.
    count_path = Path(count_path)

    # Use the existing file only when it is present and valid for HLS NDVI temporal median
    # composite.
    if not composite_path.exists() or not composite_path.is_file() or \
        composite_path.stat().st_size <= 0 or (not count_path.exists()) or (not \
        count_path.is_file()) or (count_path.stat().st_size <= 0):
        return False

    # Protect HLS NDVI temporal median composite so expected source or file failures do not leave
    # partial outputs.
    try:

        # Open the raster in a managed context so its file handle closes reliably after HLS NDVI
        # temporal median composite.
        with rasterio.open(composite_path) as composite_source:
            # Record composite valid so HLS NDVI temporal median composite can preserve a clear
            # processing and validation outcome.
            composite_valid = all([composite_source.count == 1,
                composite_source.width == fire_hazard_alignment_width,
                composite_source.height == fire_hazard_alignment_height,
                composite_source.crs is not None, composite_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                composite_source.transform.almost_equals(fire_hazard_alignment_transform),
                composite_source.dtypes[0] == fire_hazard_hls_composite_dtype,
                composite_source.nodata == fire_hazard_hls_composite_nodata])

        # Open the raster in a managed context so its file handle closes reliably after HLS NDVI
        # temporal median composite.
        with rasterio.open(count_path) as count_source:
            # Calculate count valid to quantify completeness and support QA checks.
            count_valid = all([count_source.count == 1,
                count_source.width == fire_hazard_alignment_width,
                count_source.height == fire_hazard_alignment_height,
                count_source.crs is not None, count_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                count_source.transform.almost_equals(fire_hazard_alignment_transform),
                count_source.dtypes[0] == fire_hazard_hls_composite_count_dtype,
                count_source.nodata == fire_hazard_hls_composite_count_nodata])
        return composite_valid and count_valid
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception:
        return False
# Record NDVI existing outputs valid so HLS NDVI temporal median composite can preserve a clear
# processing and validation outcome.
fire_hazard_hls_ndvi_existing_outputs_valid = \
    validate_existing_hls_ndvi_composite(fire_hazard_hls_ndvi_composite_path,
    fire_hazard_hls_ndvi_observation_count_path)
# Store NDVI composite reused needed to carry out HLS NDVI temporal median composite.
fire_hazard_hls_ndvi_composite_reused = fire_hazard_hls_composite_reuse_existing and \
    fire_hazard_hls_ndvi_existing_outputs_valid
# Record NDVI valid composite pixels so HLS NDVI temporal median composite can preserve a clear
# processing and validation outcome.
fire_hazard_hls_ndvi_valid_composite_pixels = 0
# Set NDVI NoData composite pixels to control HLS NDVI temporal median composite.
fire_hazard_hls_ndvi_nodata_composite_pixels = 0
# Track NDVI minimum across processed pixels for output-range QA.
fire_hazard_hls_ndvi_minimum = None
# Track NDVI maximum across processed pixels for output-range QA.
fire_hazard_hls_ndvi_maximum = None
# Calculate NDVI minimum observation count to quantify completeness and support QA checks.
fire_hazard_hls_ndvi_minimum_observation_count = None
# Calculate NDVI maximum observation count to quantify completeness and support QA checks.
fire_hazard_hls_ndvi_maximum_observation_count = None
# Set NDVI total supported observations to control HLS NDVI temporal median composite.
fire_hazard_hls_ndvi_total_supported_observations = 0
# Record NDVI processing action so HLS NDVI temporal median composite can preserve a clear
# processing and validation outcome.
fire_hazard_hls_ndvi_processing_action = None

# Process the raster in internal windows to control memory use
# while preserving the full-resolution output.
# Handle the NDVI composite reused case explicitly during HLS NDVI temporal median composite.
if fire_hazard_hls_ndvi_composite_reused:
    # Record NDVI processing action so HLS NDVI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndvi_processing_action = 'Reused existing temporal composite'
    print('-> Valid cached NDVI composite and observation-count raster found.')

    # Open the raster in a managed context so its file handle closes reliably after HLS NDVI
    # temporal median composite.
    with rasterio.open(fire_hazard_hls_ndvi_composite_path) as composite_source:

        # Open the raster in a managed context so its file handle closes reliably after HLS NDVI
        # temporal median composite.
        with rasterio.open(fire_hazard_hls_ndvi_observation_count_path) as count_source:

            # Process each ( entry so HLS NDVI temporal median composite is applied consistently
            # across all records.
            for _, source_window in composite_source.block_windows(1):
                # Prepare composite array used to process the current raster window.
                composite_array = composite_source.read(1, window=source_window)
                # Calculate count array to quantify completeness and support QA checks.
                count_array = count_source.read(1, window=source_window)
                # Build the valid composite mask mask used to isolate records required for this
                # analysis.
                valid_composite_mask = np.isfinite(composite_array) & (composite_array != \
                    fire_hazard_hls_composite_nodata)
                # Record valid composite values so HLS NDVI temporal median composite can preserve a
                # clear processing and validation outcome.
                valid_composite_values = composite_array[valid_composite_mask]
                # Calculate valid count values to quantify completeness and support QA checks.
                valid_count_values = count_array[valid_composite_mask]
                # Record NDVI valid composite pixels so HLS NDVI temporal median composite can
                # preserve a clear processing and validation outcome.
                fire_hazard_hls_ndvi_valid_composite_pixels += int(valid_composite_values.size)
                # Store NDVI NoData composite pixels needed to carry out HLS NDVI temporal median
                # composite.
                fire_hazard_hls_ndvi_nodata_composite_pixels += int((~valid_composite_mask).sum())
                # Store NDVI total supported observations needed to carry out HLS NDVI temporal
                # median composite.
                fire_hazard_hls_ndvi_total_supported_observations += int(valid_count_values.sum())

                # Handle the valid composite values case explicitly during HLS NDVI temporal median
                # composite.
                if valid_composite_values.size > 0:
                    # Track window minimum across processed pixels for output-range QA.
                    window_minimum = float(valid_composite_values.min())
                    # Track window maximum across processed pixels for output-range QA.
                    window_maximum = float(valid_composite_values.max())
                    # Track NDVI minimum across processed pixels for output-range QA.
                    fire_hazard_hls_ndvi_minimum = window_minimum if fire_hazard_hls_ndvi_minimum \
                        is None else min(fire_hazard_hls_ndvi_minimum,
                        window_minimum)
                    # Track NDVI maximum across processed pixels for output-range QA.
                    fire_hazard_hls_ndvi_maximum = window_maximum if fire_hazard_hls_ndvi_maximum \
                        is None else max(fire_hazard_hls_ndvi_maximum,
                        window_maximum)

                # Handle the valid count values case explicitly during HLS NDVI temporal median
                # composite.
                if valid_count_values.size > 0:
                    # Calculate window count minimum to quantify completeness and support QA checks.
                    window_count_minimum = int(valid_count_values.min())
                    # Calculate window count maximum to quantify completeness and support QA checks.
                    window_count_maximum = int(valid_count_values.max())
                    # Calculate NDVI minimum observation count to quantify completeness and support
                    # QA checks.
                    fire_hazard_hls_ndvi_minimum_observation_count = window_count_minimum if \
                        fire_hazard_hls_ndvi_minimum_observation_count is None else \
                        min(fire_hazard_hls_ndvi_minimum_observation_count,
                        window_count_minimum)
                    # Calculate NDVI maximum observation count to quantify completeness and support
                    # QA checks.
                    fire_hazard_hls_ndvi_maximum_observation_count = window_count_maximum if \
                        fire_hazard_hls_ndvi_maximum_observation_count is None else \
                        max(fire_hazard_hls_ndvi_maximum_observation_count,
                        window_count_maximum)
else:
    # Record NDVI processing action so HLS NDVI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndvi_processing_action = 'Created temporal median composite'

    # Process each temporary path entry so HLS NDVI temporal median composite is applied
    # consistently across all records.
    for temporary_path in [fire_hazard_hls_ndvi_composite_temporary_path,
        fire_hazard_hls_ndvi_count_temporary_path]:

        # Use the existing file only when it is present and valid for HLS NDVI temporal median
        # composite.
        if temporary_path.exists():
            temporary_path.unlink()

    # Protect existing outputs unless the configured overwrite policy permits replacement.
    if fire_hazard_hls_composite_overwrite:

        # Process each final path entry so HLS NDVI temporal median composite is applied
        # consistently across all records.
        for final_path in [fire_hazard_hls_ndvi_composite_path,
            fire_hazard_hls_ndvi_observation_count_path]:

            # Use the existing file only when it is present and valid for HLS NDVI temporal median
            # composite.
            if final_path.exists():
                final_path.unlink()
    from contextlib import ExitStack
    from rasterio.windows import Window
    # Build the NDVI source paths location so HLS NDVI temporal median composite uses the expected
    # project file structure.
    ndvi_source_paths = [Path(aligned_path) for aligned_path in \
        fire_hazard_hls_ndvi_composite_sources['ALIGNED_PATH']]

    # Manage the resource context so it is released reliably after HLS NDVI temporal median
    # composite.
    with ExitStack() as source_stack:
        # Build ndvi source datasets from the records that satisfy this workflow's selection
        # criteria.
        ndvi_source_datasets = [source_stack.enter_context(rasterio.open(source_path)) for \
            source_path in ndvi_source_paths]
        # Record invalid NDVI source grids so HLS NDVI temporal median composite can preserve a
        # clear processing and validation outcome.
        invalid_ndvi_source_grids = []

        # Process each (source number entry so HLS NDVI temporal median composite is applied
        # consistently across all records.
        for source_number, source_dataset in enumerate(ndvi_source_datasets, start=1):
            # Record source grid valid so HLS NDVI temporal median composite can preserve a clear
            # processing and validation outcome.
            source_grid_valid = all([source_dataset.count == 1,
                source_dataset.width == fire_hazard_alignment_width,
                source_dataset.height == fire_hazard_alignment_height,
                source_dataset.crs is not None, source_dataset.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                source_dataset.transform.almost_equals(fire_hazard_alignment_transform),
                source_dataset.dtypes[0] == fire_hazard_raster_dtype,
                source_dataset.nodata == fire_hazard_nodata_value])

            # Handle missing or invalid source grid valid explicitly during HLS NDVI temporal median
            # composite.
            if not source_grid_valid:
                invalid_ndvi_source_grids.append(str(ndvi_source_paths[source_number - 1]))

        # Stop execution if one or more aligned NDVI source rasters do not match the common analysis
        # grid.
        if invalid_ndvi_source_grids:
            raise ValueError(f'One or more aligned NDVI '
                f'source rasters do not match '
                f'the common analysis grid.\n\n'
                f'First affected paths:\n'
                f'{invalid_ndvi_source_grids[:5]}')

        # Open the raster in a managed context so its file handle closes reliably after HLS NDVI
        # temporal median composite.
        with rasterio.open(fire_hazard_hls_ndvi_composite_temporary_path,
            'w', **fire_hazard_hls_composite_profile) as composite_destination:

            # Open the raster in a managed context so its file handle closes reliably after HLS NDVI
            # temporal median composite.
            with rasterio.open(fire_hazard_hls_ndvi_count_temporary_path,
                'w', **fire_hazard_hls_composite_count_profile) as count_destination:
                # Store total composite windows needed to carry out HLS NDVI temporal median
                # composite.
                total_composite_windows = len(fire_hazard_hls_composite_window_inventory)

                # Process each ( entry so HLS NDVI temporal median composite is applied consistently
                # across all records.
                for _, window_record in fire_hazard_hls_composite_window_inventory.iterrows():
                    # Store window number needed to carry out HLS NDVI temporal median composite.
                    window_number = int(window_record['WINDOW_NUMBER'])

                    # Report progress at the configured interval without changing raster-processing
                    # results.
                    if window_number == 1 or window_number % \
                        fire_hazard_hls_composite_progress_interval == 0 or window_number == \
                        total_composite_windows:
                        print(f'-> Processing NDVI window '
                            f'{window_number:,} of '
                            f'{total_composite_windows:,}...')
                    # Store processing window needed to carry out HLS NDVI temporal median
                    # composite.
                    processing_window = Window(col_off=int(window_record['COLUMN_OFFSET']),
                        row_off=int(window_record['ROW_OFFSET']), \
                            width=int(window_record['WINDOW_WIDTH']),
                        height=int(window_record['WINDOW_HEIGHT']))
                    # Store NDVI window stack needed to carry out HLS NDVI temporal median
                    # composite.
                    ndvi_window_stack = np.empty((fire_hazard_hls_ndvi_source_count,
                        int(processing_window.height), int(processing_window.width)),
                        dtype=np.float32)

                    # Process each (source index entry so HLS NDVI temporal median composite is
                    # applied consistently across all records.
                    for source_index, source_dataset in enumerate(ndvi_source_datasets):
                        # Prepare source array used to process the current raster window.
                        source_array = source_dataset.read(1,
                            window=processing_window).astype(np.float32)
                        # Build the source invalid mask mask used to isolate records required for
                        # this analysis.
                        source_invalid_mask = ~np.isfinite(source_array) | (source_array == \
                            fire_hazard_nodata_value)
                        # Build the source array[source invalid mask] mask to isolate pixels that
                        # satisfy the current analysis criteria.
                        source_array[source_invalid_mask] = np.nan
                        # Store NDVI window stack[source index] needed to carry out HLS NDVI
                        # temporal median composite.
                        ndvi_window_stack[source_index] = source_array
                    # Calculate valid observation count to quantify completeness and support QA
                    # checks.
                    valid_observation_count = \
                        np.isfinite(ndvi_window_stack).sum(axis=0).astype(np.uint16)
                    # Build the sufficient observation mask mask used to isolate records required
                    # for this analysis.
                    sufficient_observation_mask = valid_observation_count >= \
                        fire_hazard_hls_composite_minimum_observations
                    # Prepare NDVI composite array used to process the current raster window.
                    ndvi_composite_array = np.full((int(processing_window.height),
                        int(processing_window.width)), fire_hazard_hls_composite_nodata,
                        dtype=np.float32)

                    # Manage the resource context so it is released reliably after HLS NDVI temporal
                    # median composite.
                    with np.errstate(invalid='ignore'):
                        # Store NDVI window median needed to carry out HLS NDVI temporal median
                        # composite.
                        ndvi_window_median = np.nanmedian(ndvi_window_stack, axis=0)
                    # Build the retained composite mask mask used to isolate records required for
                    # this analysis.
                    retained_composite_mask = sufficient_observation_mask & \
                        np.isfinite(ndvi_window_median)
                    # Build the NDVI composite array[retained composite mask] mask to isolate pixels
                    # that satisfy the current analysis criteria.
                    ndvi_composite_array[retained_composite_mask] = \
                        ndvi_window_median[retained_composite_mask]

                    # Handle the retained composite mask case explicitly during HLS NDVI temporal
                    # median composite.
                    if retained_composite_mask.any():
                        # Build the NDVI composite array[retained composite mask] mask to isolate
                        # pixels that satisfy the current analysis criteria.
                        ndvi_composite_array[retained_composite_mask] = \
                            np.clip(ndvi_composite_array[retained_composite_mask],
                            fire_hazard_hls_composite_minimum_value, \
                                fire_hazard_hls_composite_maximum_value)
                    # Calculate NDVI count array to quantify completeness and support QA checks.
                    ndvi_count_array = valid_observation_count.astype(np.uint16)
                    # Write the processed data to the configured output resource.
                    composite_destination.write(ndvi_composite_array, 1, window=processing_window)
                    # Write the processed data to the configured output resource.
                    count_destination.write(ndvi_count_array, 1, window=processing_window)
                    # Record valid composite values so HLS NDVI temporal median composite can
                    # preserve a clear processing and validation outcome.
                    valid_composite_values = ndvi_composite_array[retained_composite_mask]
                    # Calculate retained count values to quantify completeness and support QA
                    # checks.
                    retained_count_values = ndvi_count_array[retained_composite_mask]
                    # Record NDVI valid composite pixels so HLS NDVI temporal median composite can
                    # preserve a clear processing and validation outcome.
                    fire_hazard_hls_ndvi_valid_composite_pixels += int(valid_composite_values.size)
                    # Store NDVI NoData composite pixels needed to carry out HLS NDVI temporal
                    # median composite.
                    fire_hazard_hls_ndvi_nodata_composite_pixels += \
                        int((~retained_composite_mask).sum())
                    # Store NDVI total supported observations needed to carry out HLS NDVI temporal
                    # median composite.
                    fire_hazard_hls_ndvi_total_supported_observations += \
                        int(retained_count_values.sum())

                    # Handle the valid composite values case explicitly during HLS NDVI temporal
                    # median composite.
                    if valid_composite_values.size > 0:
                        # Track window minimum across processed pixels for output-range QA.
                        window_minimum = float(valid_composite_values.min())
                        # Track window maximum across processed pixels for output-range QA.
                        window_maximum = float(valid_composite_values.max())
                        # Track NDVI minimum across processed pixels for output-range QA.
                        fire_hazard_hls_ndvi_minimum = window_minimum if \
                            fire_hazard_hls_ndvi_minimum is None else \
                            min(fire_hazard_hls_ndvi_minimum,
                            window_minimum)
                        # Track NDVI maximum across processed pixels for output-range QA.
                        fire_hazard_hls_ndvi_maximum = window_maximum if \
                            fire_hazard_hls_ndvi_maximum is None else \
                            max(fire_hazard_hls_ndvi_maximum,
                            window_maximum)

                    # Handle the retained count values case explicitly during HLS NDVI temporal
                    # median composite.
                    if retained_count_values.size > 0:
                        # Calculate window count minimum to quantify completeness and support QA
                        # checks.
                        window_count_minimum = int(retained_count_values.min())
                        # Calculate window count maximum to quantify completeness and support QA
                        # checks.
                        window_count_maximum = int(retained_count_values.max())
                        # Calculate NDVI minimum observation count to quantify completeness and
                        # support QA checks.
                        fire_hazard_hls_ndvi_minimum_observation_count = window_count_minimum if \
                            fire_hazard_hls_ndvi_minimum_observation_count is None else \
                            min(fire_hazard_hls_ndvi_minimum_observation_count,
                            window_count_minimum)
                        # Calculate NDVI maximum observation count to quantify completeness and
                        # support QA checks.
                        fire_hazard_hls_ndvi_maximum_observation_count = window_count_maximum if \
                            fire_hazard_hls_ndvi_maximum_observation_count is None else \
                            max(fire_hazard_hls_ndvi_maximum_observation_count,
                            window_count_maximum)
                    del ndvi_window_stack
                    del ndvi_window_median
                    del ndvi_composite_array
                    del ndvi_count_array
                    del valid_observation_count
                composite_destination.set_band_description(1, 'HLS 2025 fire-season NDVI median')
                composite_destination.update_tags(DATA_SOURCE='NASA HLS Version 2.0',
                    INDEX_NAME='NDVI', COMPOSITE_METHOD='median', \
                        SOURCE_RASTER_COUNT=fire_hazard_hls_ndvi_source_count,
                    MINIMUM_VALID_OBSERVATIONS=fire_hazard_hls_composite_minimum_observations,
                    TARGET_CRS=fire_hazard_target_crs, TARGET_CELL_SIZE=fire_hazard_cell_size)
                count_destination.set_band_description(1, 'Valid HLS NDVI observation count')
                count_destination.update_tags(DATA_SOURCE='NASA HLS Version 2.0',
                    INDEX_NAME='NDVI', RASTER_PURPOSE='Temporal composite observation count',
                    SOURCE_RASTER_COUNT=fire_hazard_hls_ndvi_source_count)
    # Record NDVI temporary outputs valid so HLS NDVI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndvi_temporary_outputs_valid = \
        validate_existing_hls_ndvi_composite(fire_hazard_hls_ndvi_composite_temporary_path,
        fire_hazard_hls_ndvi_count_temporary_path)
    # Record NDVI content valid so HLS NDVI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndvi_content_valid = fire_hazard_hls_ndvi_valid_composite_pixels > 0 and \
        fire_hazard_hls_ndvi_minimum is not None and (fire_hazard_hls_ndvi_maximum is not None) \
        and (fire_hazard_hls_ndvi_minimum >= fire_hazard_hls_composite_minimum_value) and \
        (fire_hazard_hls_ndvi_maximum <= fire_hazard_hls_composite_maximum_value) and \
        (fire_hazard_hls_ndvi_minimum_observation_count >= \
        fire_hazard_hls_composite_minimum_observations) and \
        (fire_hazard_hls_ndvi_maximum_observation_count <= fire_hazard_hls_ndvi_source_count)

    # Stop execution if temporary NDVI temporal-composite outputs failed validation.
    if not fire_hazard_hls_ndvi_temporary_outputs_valid or not fire_hazard_hls_ndvi_content_valid:
        raise ValueError(f'The temporary NDVI '
            f'temporal-composite outputs '
            f'failed validation.\n\nRaster '
            f'structure valid: '
            f'{fire_hazard_hls_ndvi_temporary_outputs_valid}\n'
            f'Valid composite pixels: '
            f'{fire_hazard_hls_ndvi_valid_composite_pixels:,}\n'
            f'NDVI minimum: '
            f'{fire_hazard_hls_ndvi_minimum}\n'
            f'NDVI maximum: '
            f'{fire_hazard_hls_ndvi_maximum}\n'
            f'Minimum observation count: '
            f'{fire_hazard_hls_ndvi_minimum_observation_count}\n'
            f'Maximum observation count: '
            f'{fire_hazard_hls_ndvi_maximum_observation_count}')
    fire_hazard_hls_ndvi_composite_temporary_path.replace(fire_hazard_hls_ndvi_composite_path)
    fire_hazard_hls_ndvi_count_temporary_path.replace(fire_hazard_hls_ndvi_observation_count_path)
# Record NDVI composite valid so HLS NDVI temporal median composite can preserve a clear processing
# and validation outcome.
fire_hazard_hls_ndvi_composite_valid = \
    validate_existing_hls_ndvi_composite(fire_hazard_hls_ndvi_composite_path,
    fire_hazard_hls_ndvi_observation_count_path) and fire_hazard_hls_ndvi_valid_composite_pixels \
        > 0 and (fire_hazard_hls_ndvi_minimum is not None) and (fire_hazard_hls_ndvi_maximum is \
        not None)

# Stop execution if final NDVI temporal composite or its observation-count raster failed validation.
if not fire_hazard_hls_ndvi_composite_valid:
    raise ValueError('The final NDVI temporal '
        'composite or its '
        'observation-count raster '
        'failed validation.')
# Store NDVI composite size bytes needed to carry out HLS NDVI temporal median composite.
fire_hazard_hls_ndvi_composite_size_bytes = fire_hazard_hls_ndvi_composite_path.stat().st_size
# Calculate NDVI count size bytes to quantify completeness and support QA checks.
fire_hazard_hls_ndvi_count_size_bytes = fire_hazard_hls_ndvi_observation_count_path.stat().st_size
# Define NDVI composite manifest records to control the inputs and rules used by HLS NDVI temporal
# median composite.
fire_hazard_hls_ndvi_composite_manifest_records = [{'INDEX_NAME': 'NDVI',
    'OUTPUT_TYPE': 'Temporal median composite', 'OUTPUT_PATH': \
        str(fire_hazard_hls_ndvi_composite_path),
    'PROCESSING_ACTION': fire_hazard_hls_ndvi_processing_action,
    'COMPOSITE_METHOD': 'median', 'SOURCE_RASTER_COUNT': fire_hazard_hls_ndvi_source_count,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'VALID_PIXELS': fire_hazard_hls_ndvi_valid_composite_pixels,
    'NODATA_PIXELS': fire_hazard_hls_ndvi_nodata_composite_pixels,
    'VALUE_MINIMUM': fire_hazard_hls_ndvi_minimum,
    'VALUE_MAXIMUM': fire_hazard_hls_ndvi_maximum,
    'MINIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndvi_minimum_observation_count,
    'MAXIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndvi_maximum_observation_count,
    'OUTPUT_SIZE_BYTES': fire_hazard_hls_ndvi_composite_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_hls_ndvi_composite_size_bytes / 1024 ** 2,
    'VALID': fire_hazard_hls_ndvi_composite_valid},
    {'INDEX_NAME': 'NDVI', 'OUTPUT_TYPE': 'Observation count',
    'OUTPUT_PATH': str(fire_hazard_hls_ndvi_observation_count_path),
    'PROCESSING_ACTION': fire_hazard_hls_ndvi_processing_action,
    'COMPOSITE_METHOD': 'count', 'SOURCE_RASTER_COUNT': fire_hazard_hls_ndvi_source_count,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'VALID_PIXELS': fire_hazard_hls_ndvi_valid_composite_pixels,
    'NODATA_PIXELS': fire_hazard_hls_ndvi_nodata_composite_pixels,
    'VALUE_MINIMUM': fire_hazard_hls_ndvi_minimum_observation_count,
    'VALUE_MAXIMUM': fire_hazard_hls_ndvi_maximum_observation_count,
    'MINIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndvi_minimum_observation_count,
    'MAXIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndvi_maximum_observation_count,
    'OUTPUT_SIZE_BYTES': fire_hazard_hls_ndvi_count_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_hls_ndvi_count_size_bytes / 1024 ** 2,
    'VALID': fire_hazard_hls_ndvi_composite_valid}]

# Use the existing file only when it is present and valid for HLS NDVI temporal median composite.
if fire_hazard_hls_composite_manifest_path.exists():
    # Store existing composite manifest needed to carry out HLS NDVI temporal median composite.
    existing_composite_manifest = pd.read_csv(fire_hazard_hls_composite_manifest_path)
else:
    # Store existing composite manifest needed to carry out HLS NDVI temporal median composite.
    existing_composite_manifest = pd.DataFrame()

# Handle missing or invalid existing composite manifest explicitly during HLS NDVI temporal median
# composite.
if not existing_composite_manifest.empty:
    # Create an isolated existing composite manifest working copy so HLS NDVI temporal median
    # composite does not modify upstream data.
    existing_composite_manifest = \
        existing_composite_manifest[existing_composite_manifest['INDEX_NAME'].astype(str).str \
        .upper() != 'NDVI'].copy()
# Store composite manifest needed to carry out HLS NDVI temporal median composite.
fire_hazard_hls_composite_manifest = pd.concat([existing_composite_manifest,
    pd.DataFrame(fire_hazard_hls_ndvi_composite_manifest_records)],
    ignore_index=True).sort_values(['INDEX_NAME', 'OUTPUT_TYPE']).reset_index(drop=True)
# Save the composite manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_composite_manifest.to_csv(fire_hazard_hls_composite_manifest_path, index=False)
# Store NDVI composite summary needed to carry out HLS NDVI temporal median composite.
fire_hazard_hls_ndvi_composite_summary = pd.DataFrame([{'INDEX_NAME': 'NDVI',
    'COMPOSITE_METHOD': 'Median', 'SOURCE_RASTERS': fire_hazard_hls_ndvi_source_count,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'VALID_COMPOSITE_PIXELS': fire_hazard_hls_ndvi_valid_composite_pixels,
    'NODATA_COMPOSITE_PIXELS': fire_hazard_hls_ndvi_nodata_composite_pixels,
    'NDVI_MINIMUM': fire_hazard_hls_ndvi_minimum, 'NDVI_MAXIMUM': fire_hazard_hls_ndvi_maximum,
    'MINIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndvi_minimum_observation_count,
    'MAXIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndvi_maximum_observation_count,
    'COMPOSITE_SIZE_MB': fire_hazard_hls_ndvi_composite_size_bytes / 1024 ** 2,
    'COUNT_RASTER_SIZE_MB': fire_hazard_hls_ndvi_count_size_bytes / 1024 ** 2,
    'PROCESSING_ACTION': fire_hazard_hls_ndvi_processing_action,
    'VALID': fire_hazard_hls_ndvi_composite_valid}])
print(f'-> NDVI source rasters: {fire_hazard_hls_ndvi_source_count:,}')
print(f'-> Processing action: {fire_hazard_hls_ndvi_processing_action}')
print(f'-> Valid composite pixels: {fire_hazard_hls_ndvi_valid_composite_pixels:,}')
print(f'-> Composite NDVI range: '
    f'{fire_hazard_hls_ndvi_minimum:.4f} '
    f'to '
    f'{fire_hazard_hls_ndvi_maximum:.4f}')
print(f'-> Observation-count range: '
    f'{fire_hazard_hls_ndvi_minimum_observation_count} '
    f'to '
    f'{fire_hazard_hls_ndvi_maximum_observation_count}')
print(f'-> NDVI composite saved: {fire_hazard_hls_ndvi_composite_path}')
print(f'-> Observation-count raster saved: {fire_hazard_hls_ndvi_observation_count_path}')
print(f'-> Composite manifest saved: {fire_hazard_hls_composite_manifest_path}')
print('\n--- HLS NDVI TEMPORAL COMPOSITE SUMMARY ---')
display(fire_hazard_hls_ndvi_composite_summary)
print('\n--- HLS TEMPORAL COMPOSITE MANIFEST SAMPLE ---')
display(fire_hazard_hls_composite_manifest.head(5))
import gc
gc.collect()
print('\nNOTE:')
print('The NDVI temporal composite '
    'represents the pixel-wise '
    'median vegetation greenness '
    'across the validated 2025 '
    'fire-season HLS scenes.')
print(f'Pixels with fewer than '
    f'{fire_hazard_hls_composite_minimum_observations} '
    f'valid observations remain '
    f'NoData in the composite.')
print('The companion '
    'observation-count raster '
    'records the number of valid '
    'NDVI scenes available at each '
    'target-grid pixel.')
print('The next step is to build the '
    'NDMI temporal median composite '
    'using the same source '
    'inventory and '
    'processing-window plan.')
print('\n=== HLS NDVI TEMPORAL MEDIAN COMPOSITE COMPLETE ===')



=== BUILDING HLS NDVI TEMPORAL MEDIAN COMPOSITE ===
-> Valid cached NDVI composite and observation-count raster found.
-> NDVI source rasters: 63
-> Processing action: Reused existing temporal composite
-> Valid composite pixels: 14,140,492
-> Composite NDVI range: -0.2462 to 1.0000
-> Observation-count range: 3 to 24
-> NDVI composite saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\rasters\hls_2025_fire_season_ndvi_median.tif
-> Observation-count raster saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\observation_counts\hls_2025_fire_season_ndvi_observation_count.tif
-> Composite manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_manifest.csv

--- HLS NDVI TEMPORAL COMPOSITE SUMMARY ---


,INDEX_NAME,COMPOSITE_METHOD,SOURCE_RASTERS,MINIMUM_VALID_OBSERVATIONS,VALID_COMPOSITE_PIXELS,NODATA_COMPOSITE_PIXELS,NDVI_MINIMUM,NDVI_MAXIMUM,MINIMUM_OBSERVATION_COUNT,MAXIMUM_OBSERVATION_COUNT,COMPOSITE_SIZE_MB,COUNT_RASTER_SIZE_MB,PROCESSING_ACTION,VALID
0,NDVI,Median,63,3,14140492,122630526,-0.246165,1.0,3,24,45.069147,1.691112,Reused existing temporal composite,True



--- HLS TEMPORAL COMPOSITE MANIFEST SAMPLE ---


,INDEX_NAME,OUTPUT_TYPE,OUTPUT_PATH,PROCESSING_ACTION,COMPOSITE_METHOD,SOURCE_RASTER_COUNT,MINIMUM_VALID_OBSERVATIONS,VALID_PIXELS,NODATA_PIXELS,VALUE_MINIMUM,VALUE_MAXIMUM,MINIMUM_OBSERVATION_COUNT,MAXIMUM_OBSERVATION_COUNT,OUTPUT_SIZE_BYTES,OUTPUT_SIZE_MB,VALID
0,NDMI,Observation count,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,count,63,3,14140492,122630526,3.000000,24.0,3,24,1773253,1.691106,True
1,NDMI,Temporal median composite,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,median,63,3,14140492,122630526,-0.560770,1.0,3,24,50371506,48.038012,True
2,NDVI,Observation count,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,count,63,3,14140492,122630526,3.000000,24.0,3,24,1773259,1.691112,True
3,NDVI,Temporal median composite,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,median,63,3,14140492,122630526,-0.246165,1.0,3,24,47258426,45.069147,True



NOTE:
The NDVI temporal composite represents the pixel-wise median vegetation greenness across the validated 2025 fire-season HLS scenes.
Pixels with fewer than 3 valid observations remain NoData in the composite.
The companion observation-count raster records the number of valid NDVI scenes available at each target-grid pixel.
The next step is to build the NDMI temporal median composite using the same source inventory and processing-window plan.

=== HLS NDVI TEMPORAL MEDIAN COMPOSITE COMPLETE ===


### Building HLS Ndmi Temporal Median Composite


In [87]:
print('=== BUILDING HLS NDMI TEMPORAL MEDIAN COMPOSITE ===')
# List the required HLS NDMI composite prerequisites required before HLS NDMI temporal median
# composite can run.
required_hls_ndmi_composite_inputs = ['fire_hazard_hls_ndmi_composite_sources',
    'fire_hazard_hls_composite_window_inventory', 'fire_hazard_hls_ndmi_composite_path',
    'fire_hazard_hls_ndmi_observation_count_path',
    'fire_hazard_hls_composite_profile', 'fire_hazard_hls_composite_count_profile',
    'fire_hazard_hls_composite_minimum_observations',
    'fire_hazard_hls_composite_minimum_value', 'fire_hazard_hls_composite_maximum_value',
    'fire_hazard_hls_composite_nodata', 'fire_hazard_hls_composite_count_nodata',
    'fire_hazard_hls_composite_dtype', 'fire_hazard_hls_composite_count_dtype',
    'fire_hazard_hls_composite_progress_interval',
    'fire_hazard_hls_composite_reuse_existing', 'fire_hazard_hls_composite_overwrite',
    'fire_hazard_hls_composite_manifest_path', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_nodata_value', 'fire_hazard_raster_dtype']
# Identify unavailable HLS NDMI composite so HLS NDMI temporal median composite stops
# before using incomplete inputs.
missing_hls_ndmi_composite_inputs = [object_name for object_name in \
    required_hls_ndmi_composite_inputs if object_name not in globals()]

# Stop execution when required hls ndmi composite inputs inputs are unavailable.
if missing_hls_ndmi_composite_inputs:
    raise NameError(f'The following HLS NDMI '
        f'composite objects are missing:\n'
        f'{missing_hls_ndmi_composite_inputs}\n\n'
        f'Run Configure HLS Temporal '
        f'Composite Workflow and Create '
        f'HLS Scene Grouping Inventory '
        f'before building the NDMI '
        f'composite.')
# List the required HLS NDMI source fields prerequisites required before HLS NDMI temporal median
# composite can run.
required_hls_ndmi_source_fields = ['COLLECTION_ID',
    'ITEM_ID', 'INDEX_NAME', 'ALIGNED_PATH', 'SOURCE_FILE_VALID']
# Identify unavailable HLS NDMI source fields so HLS NDMI temporal median composite stops before
# using incomplete inputs.
missing_hls_ndmi_source_fields = [field_name for field_name in required_hls_ndmi_source_fields if \
    field_name not in fire_hazard_hls_ndmi_composite_sources.columns]

# Stop execution when required hls ndmi source fields inputs are unavailable.
if missing_hls_ndmi_source_fields:
    raise ValueError(f'The NDMI composite source '
        f'inventory is missing required '
        f'fields:\n'
        f'{missing_hls_ndmi_source_fields}')

# Stop execution if nDMI composite source inventory contains no aligned rasters.
if fire_hazard_hls_ndmi_composite_sources.empty:
    raise ValueError('The NDMI composite source inventory contains no aligned rasters.')

# Stop execution if nDMI composite inventory contains one or more non-NDMI source records.
if not fire_hazard_hls_ndmi_composite_sources['INDEX_NAME'].astype(str).str.upper().eq('NDMI') \
    .all():
    raise ValueError('The NDMI composite inventory contains one or more non-NDMI source records.')

# Stop execution if one or more aligned NDMI source rasters are missing or invalid.
if not fire_hazard_hls_ndmi_composite_sources['SOURCE_FILE_VALID'].all():
    raise ValueError('One or more aligned NDMI source rasters are missing or invalid.')
# Calculate NDMI source count to quantify completeness and support QA checks.
fire_hazard_hls_ndmi_source_count = len(fire_hazard_hls_ndmi_composite_sources)

# Stop execution if nDMI source inventory contains fewer rasters than the configured minimum valid
# observation requirement.
if fire_hazard_hls_ndmi_source_count < fire_hazard_hls_composite_minimum_observations:
    raise ValueError('The NDMI source inventory '
        'contains fewer rasters than '
        'the configured minimum valid '
        'observation requirement.')
# Build the NDMI composite temporary path location so HLS NDMI temporal median composite uses the
# expected project file structure.
fire_hazard_hls_ndmi_composite_temporary_path = fire_hazard_hls_ndmi_composite_path.parent / \
    (fire_hazard_hls_ndmi_composite_path.stem + '.part.tif')
# Build the NDMI count temporary path location so HLS NDMI temporal median composite uses the
# expected project file structure.
fire_hazard_hls_ndmi_count_temporary_path = fire_hazard_hls_ndmi_observation_count_path.parent / \
    (fire_hazard_hls_ndmi_observation_count_path.stem + '.part.tif')

# Encapsulate validate existing HLS NDMI composite so repeated HLS NDMI temporal median composite
# steps use consistent logic.
def validate_existing_hls_ndmi_composite(composite_path, count_path):

    """
    Checks whether existing NDMI composite and
    observation-count rasters match the target grid.
    """
    # Build the composite path location so HLS NDMI temporal median composite uses the expected
    # project file structure.
    composite_path = Path(composite_path)
    # Build the count path location so HLS NDMI temporal median composite uses the expected project
    # file structure.
    count_path = Path(count_path)

    # Use the existing file only when it is present and valid for HLS NDMI temporal median
    # composite.
    if not composite_path.exists() or not composite_path.is_file() or \
        composite_path.stat().st_size <= 0 or (not count_path.exists()) or (not \
        count_path.is_file()) or (count_path.stat().st_size <= 0):
        return False

    # Protect HLS NDMI temporal median composite so expected source or file failures do not leave
    # partial outputs.
    try:

        # Open the raster in a managed context so its file handle closes reliably after HLS NDMI
        # temporal median composite.
        with rasterio.open(composite_path) as composite_source:
            # Record composite valid so HLS NDMI temporal median composite can preserve a clear
            # processing and validation outcome.
            composite_valid = all([composite_source.count == 1,
                composite_source.width == fire_hazard_alignment_width,
                composite_source.height == fire_hazard_alignment_height,
                composite_source.crs is not None, composite_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                composite_source.transform.almost_equals(fire_hazard_alignment_transform),
                composite_source.dtypes[0] == fire_hazard_hls_composite_dtype,
                composite_source.nodata == fire_hazard_hls_composite_nodata])

        # Open the raster in a managed context so its file handle closes reliably after HLS NDMI
        # temporal median composite.
        with rasterio.open(count_path) as count_source:
            # Calculate count valid to quantify completeness and support QA checks.
            count_valid = all([count_source.count == 1,
                count_source.width == fire_hazard_alignment_width,
                count_source.height == fire_hazard_alignment_height,
                count_source.crs is not None, count_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                count_source.transform.almost_equals(fire_hazard_alignment_transform),
                count_source.dtypes[0] == fire_hazard_hls_composite_count_dtype,
                count_source.nodata == fire_hazard_hls_composite_count_nodata])
        return composite_valid and count_valid
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception:
        return False
# Record NDMI existing outputs valid so HLS NDMI temporal median composite can preserve a clear
# processing and validation outcome.
fire_hazard_hls_ndmi_existing_outputs_valid = \
    validate_existing_hls_ndmi_composite(fire_hazard_hls_ndmi_composite_path,
    fire_hazard_hls_ndmi_observation_count_path)
# Store NDMI composite reused needed to carry out HLS NDMI temporal median composite.
fire_hazard_hls_ndmi_composite_reused = fire_hazard_hls_composite_reuse_existing and \
    fire_hazard_hls_ndmi_existing_outputs_valid
# Record NDMI valid composite pixels so HLS NDMI temporal median composite can preserve a clear
# processing and validation outcome.
fire_hazard_hls_ndmi_valid_composite_pixels = 0
# Set NDMI NoData composite pixels to control HLS NDMI temporal median composite.
fire_hazard_hls_ndmi_nodata_composite_pixels = 0
# Track NDMI minimum across processed pixels for output-range QA.
fire_hazard_hls_ndmi_minimum = None
# Track NDMI maximum across processed pixels for output-range QA.
fire_hazard_hls_ndmi_maximum = None
# Calculate NDMI minimum observation count to quantify completeness and support QA checks.
fire_hazard_hls_ndmi_minimum_observation_count = None
# Calculate NDMI maximum observation count to quantify completeness and support QA checks.
fire_hazard_hls_ndmi_maximum_observation_count = None
# Set NDMI total supported observations to control HLS NDMI temporal median composite.
fire_hazard_hls_ndmi_total_supported_observations = 0
# Record NDMI processing action so HLS NDMI temporal median composite can preserve a clear
# processing and validation outcome.
fire_hazard_hls_ndmi_processing_action = None

# Handle the NDMI composite reused case explicitly during HLS NDMI temporal median composite.
if fire_hazard_hls_ndmi_composite_reused:
    # Record NDMI processing action so HLS NDMI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndmi_processing_action = 'Reused existing temporal composite'
    print('-> Valid cached NDMI composite and observation-count raster found.')

    # Open the raster in a managed context so its file handle closes reliably after HLS NDMI
    # temporal median composite.
    with rasterio.open(fire_hazard_hls_ndmi_composite_path) as composite_source:

        # Open the raster in a managed context so its file handle closes reliably after HLS NDMI
        # temporal median composite.
        with rasterio.open(fire_hazard_hls_ndmi_observation_count_path) as count_source:

            # Process each ( entry so HLS NDMI temporal median composite is applied consistently
            # across all records.
            for _, source_window in composite_source.block_windows(1):
                # Prepare composite array used to process the current raster window.
                composite_array = composite_source.read(1, window=source_window)
                # Calculate count array to quantify completeness and support QA checks.
                count_array = count_source.read(1, window=source_window)
                # Build the valid composite mask mask used to isolate records required for this
                # analysis.
                valid_composite_mask = np.isfinite(composite_array) & (composite_array != \
                    fire_hazard_hls_composite_nodata)
                # Record valid composite values so HLS NDMI temporal median composite can preserve a
                # clear processing and validation outcome.
                valid_composite_values = composite_array[valid_composite_mask]
                # Calculate valid count values to quantify completeness and support QA checks.
                valid_count_values = count_array[valid_composite_mask]
                # Record NDMI valid composite pixels so HLS NDMI temporal median composite can
                # preserve a clear processing and validation outcome.
                fire_hazard_hls_ndmi_valid_composite_pixels += int(valid_composite_values.size)
                # Store NDMI NoData composite pixels needed to carry out HLS NDMI temporal median
                # composite.
                fire_hazard_hls_ndmi_nodata_composite_pixels += int((~valid_composite_mask).sum())
                # Store NDMI total supported observations needed to carry out HLS NDMI temporal
                # median composite.
                fire_hazard_hls_ndmi_total_supported_observations += int(valid_count_values.sum())

                # Handle the valid composite values case explicitly during HLS NDMI temporal median
                # composite.
                if valid_composite_values.size > 0:
                    # Track window minimum across processed pixels for output-range QA.
                    window_minimum = float(valid_composite_values.min())
                    # Track window maximum across processed pixels for output-range QA.
                    window_maximum = float(valid_composite_values.max())
                    # Track NDMI minimum across processed pixels for output-range QA.
                    fire_hazard_hls_ndmi_minimum = window_minimum if fire_hazard_hls_ndmi_minimum \
                        is None else min(fire_hazard_hls_ndmi_minimum,
                        window_minimum)
                    # Track NDMI maximum across processed pixels for output-range QA.
                    fire_hazard_hls_ndmi_maximum = window_maximum if fire_hazard_hls_ndmi_maximum \
                        is None else max(fire_hazard_hls_ndmi_maximum,
                        window_maximum)

                # Handle the valid count values case explicitly during HLS NDMI temporal median
                # composite.
                if valid_count_values.size > 0:
                    # Calculate window count minimum to quantify completeness and support QA checks.
                    window_count_minimum = int(valid_count_values.min())
                    # Calculate window count maximum to quantify completeness and support QA checks.
                    window_count_maximum = int(valid_count_values.max())
                    # Calculate NDMI minimum observation count to quantify completeness and support
                    # QA checks.
                    fire_hazard_hls_ndmi_minimum_observation_count = window_count_minimum if \
                        fire_hazard_hls_ndmi_minimum_observation_count is None else \
                        min(fire_hazard_hls_ndmi_minimum_observation_count,
                        window_count_minimum)
                    # Calculate NDMI maximum observation count to quantify completeness and support
                    # QA checks.
                    fire_hazard_hls_ndmi_maximum_observation_count = window_count_maximum if \
                        fire_hazard_hls_ndmi_maximum_observation_count is None else \
                        max(fire_hazard_hls_ndmi_maximum_observation_count,
                        window_count_maximum)
else:
    # Record NDMI processing action so HLS NDMI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndmi_processing_action = 'Created temporal median composite'

    # Process each temporary path entry so HLS NDMI temporal median composite is applied
    # consistently across all records.
    for temporary_path in [fire_hazard_hls_ndmi_composite_temporary_path,
        fire_hazard_hls_ndmi_count_temporary_path]:

        # Use the existing file only when it is present and valid for HLS NDMI temporal median
        # composite.
        if temporary_path.exists():
            temporary_path.unlink()

    # Protect existing outputs unless the configured overwrite policy permits replacement.
    if fire_hazard_hls_composite_overwrite:

        # Process each final path entry so HLS NDMI temporal median composite is applied
        # consistently across all records.
        for final_path in [fire_hazard_hls_ndmi_composite_path,
            fire_hazard_hls_ndmi_observation_count_path]:

            # Use the existing file only when it is present and valid for HLS NDMI temporal median
            # composite.
            if final_path.exists():
                final_path.unlink()
    from contextlib import ExitStack
    from rasterio.windows import Window
    # Build the NDMI source paths location so HLS NDMI temporal median composite uses the expected
    # project file structure.
    ndmi_source_paths = [Path(aligned_path) for aligned_path in \
        fire_hazard_hls_ndmi_composite_sources['ALIGNED_PATH']]

    # Manage the resource context so it is released reliably after HLS NDMI temporal median
    # composite.
    with ExitStack() as source_stack:
        # Build ndmi source datasets from the records that satisfy this workflow's selection
        # criteria.
        ndmi_source_datasets = [source_stack.enter_context(rasterio.open(source_path)) for \
            source_path in ndmi_source_paths]
        # Record invalid NDMI source grids so HLS NDMI temporal median composite can preserve a
        # clear processing and validation outcome.
        invalid_ndmi_source_grids = []

        # Process each (source number entry so HLS NDMI temporal median composite is applied
        # consistently across all records.
        for source_number, source_dataset in enumerate(ndmi_source_datasets, start=1):
            # Record source grid valid so HLS NDMI temporal median composite can preserve a clear
            # processing and validation outcome.
            source_grid_valid = all([source_dataset.count == 1,
                source_dataset.width == fire_hazard_alignment_width,
                source_dataset.height == fire_hazard_alignment_height,
                source_dataset.crs is not None, source_dataset.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                source_dataset.transform.almost_equals(fire_hazard_alignment_transform),
                source_dataset.dtypes[0] == fire_hazard_raster_dtype,
                source_dataset.nodata == fire_hazard_nodata_value])

            # Handle missing or invalid source grid valid explicitly during HLS NDMI temporal median
            # composite.
            if not source_grid_valid:
                invalid_ndmi_source_grids.append(str(ndmi_source_paths[source_number - 1]))

        # Stop execution if one or more aligned NDMI source rasters do not match the common analysis
        # grid.
        if invalid_ndmi_source_grids:
            raise ValueError(f'One or more aligned NDMI '
                f'source rasters do not match '
                f'the common analysis grid.\n\n'
                f'First affected paths:\n'
                f'{invalid_ndmi_source_grids[:5]}')

        # Open the raster in a managed context so its file handle closes reliably after HLS NDMI
        # temporal median composite.
        with rasterio.open(fire_hazard_hls_ndmi_composite_temporary_path,
            'w', **fire_hazard_hls_composite_profile) as composite_destination:

            # Open the raster in a managed context so its file handle closes reliably after HLS NDMI
            # temporal median composite.
            with rasterio.open(fire_hazard_hls_ndmi_count_temporary_path,
                'w', **fire_hazard_hls_composite_count_profile) as count_destination:
                # Store total composite windows needed to carry out HLS NDMI temporal median
                # composite.
                total_composite_windows = len(fire_hazard_hls_composite_window_inventory)

                # Process each ( entry so HLS NDMI temporal median composite is applied consistently
                # across all records.
                for _, window_record in fire_hazard_hls_composite_window_inventory.iterrows():
                    # Store window number needed to carry out HLS NDMI temporal median composite.
                    window_number = int(window_record['WINDOW_NUMBER'])

                    # Report progress at the configured interval without changing raster-processing
                    # results.
                    if window_number == 1 or window_number % \
                        fire_hazard_hls_composite_progress_interval == 0 or window_number == \
                        total_composite_windows:
                        print(f'-> Processing NDMI window '
                            f'{window_number:,} of '
                            f'{total_composite_windows:,}...')
                    # Store processing window needed to carry out HLS NDMI temporal median
                    # composite.
                    processing_window = Window(col_off=int(window_record['COLUMN_OFFSET']),
                        row_off=int(window_record['ROW_OFFSET']), \
                            width=int(window_record['WINDOW_WIDTH']),
                        height=int(window_record['WINDOW_HEIGHT']))
                    # Store NDMI window stack needed to carry out HLS NDMI temporal median
                    # composite.
                    ndmi_window_stack = np.empty((fire_hazard_hls_ndmi_source_count,
                        int(processing_window.height), int(processing_window.width)),
                        dtype=np.float32)

                    # Process each (source index entry so HLS NDMI temporal median composite is
                    # applied consistently across all records.
                    for source_index, source_dataset in enumerate(ndmi_source_datasets):
                        # Prepare source array used to process the current raster window.
                        source_array = source_dataset.read(1,
                            window=processing_window).astype(np.float32)
                        # Build the source invalid mask mask used to isolate records required for
                        # this analysis.
                        source_invalid_mask = ~np.isfinite(source_array) | (source_array == \
                            fire_hazard_nodata_value)
                        # Build the source array[source invalid mask] mask to isolate pixels that
                        # satisfy the current analysis criteria.
                        source_array[source_invalid_mask] = np.nan
                        # Store NDMI window stack[source index] needed to carry out HLS NDMI
                        # temporal median composite.
                        ndmi_window_stack[source_index] = source_array
                    # Calculate valid observation count to quantify completeness and support QA
                    # checks.
                    valid_observation_count = \
                        np.isfinite(ndmi_window_stack).sum(axis=0).astype(np.uint16)
                    # Build the sufficient observation mask mask used to isolate records required
                    # for this analysis.
                    sufficient_observation_mask = valid_observation_count >= \
                        fire_hazard_hls_composite_minimum_observations
                    # Prepare NDMI composite array used to process the current raster window.
                    ndmi_composite_array = np.full((int(processing_window.height),
                        int(processing_window.width)), fire_hazard_hls_composite_nodata,
                        dtype=np.float32)

                    # Manage the resource context so it is released reliably after HLS NDMI temporal
                    # median composite.
                    with np.errstate(invalid='ignore'):
                        # Store NDMI window median needed to carry out HLS NDMI temporal median
                        # composite.
                        ndmi_window_median = np.nanmedian(ndmi_window_stack, axis=0)
                    # Build the retained composite mask mask used to isolate records required for
                    # this analysis.
                    retained_composite_mask = sufficient_observation_mask & \
                        np.isfinite(ndmi_window_median)
                    # Build the NDMI composite array[retained composite mask] mask to isolate pixels
                    # that satisfy the current analysis criteria.
                    ndmi_composite_array[retained_composite_mask] = \
                        ndmi_window_median[retained_composite_mask]

                    # Handle the retained composite mask case explicitly during HLS NDMI temporal
                    # median composite.
                    if retained_composite_mask.any():
                        # Build the NDMI composite array[retained composite mask] mask to isolate
                        # pixels that satisfy the current analysis criteria.
                        ndmi_composite_array[retained_composite_mask] = \
                            np.clip(ndmi_composite_array[retained_composite_mask],
                            fire_hazard_hls_composite_minimum_value, \
                                fire_hazard_hls_composite_maximum_value)
                    # Calculate NDMI count array to quantify completeness and support QA checks.
                    ndmi_count_array = valid_observation_count.astype(np.uint16)
                    # Write the processed data to the configured output resource.
                    composite_destination.write(ndmi_composite_array, 1, window=processing_window)
                    # Write the processed data to the configured output resource.
                    count_destination.write(ndmi_count_array, 1, window=processing_window)
                    # Record valid composite values so HLS NDMI temporal median composite can
                    # preserve a clear processing and validation outcome.
                    valid_composite_values = ndmi_composite_array[retained_composite_mask]
                    # Calculate retained count values to quantify completeness and support QA
                    # checks.
                    retained_count_values = ndmi_count_array[retained_composite_mask]
                    # Record NDMI valid composite pixels so HLS NDMI temporal median composite can
                    # preserve a clear processing and validation outcome.
                    fire_hazard_hls_ndmi_valid_composite_pixels += int(valid_composite_values.size)
                    # Store NDMI NoData composite pixels needed to carry out HLS NDMI temporal
                    # median composite.
                    fire_hazard_hls_ndmi_nodata_composite_pixels += \
                        int((~retained_composite_mask).sum())
                    # Store NDMI total supported observations needed to carry out HLS NDMI temporal
                    # median composite.
                    fire_hazard_hls_ndmi_total_supported_observations += \
                        int(retained_count_values.sum())

                    # Handle the valid composite values case explicitly during HLS NDMI temporal
                    # median composite.
                    if valid_composite_values.size > 0:
                        # Track window minimum across processed pixels for output-range QA.
                        window_minimum = float(valid_composite_values.min())
                        # Track window maximum across processed pixels for output-range QA.
                        window_maximum = float(valid_composite_values.max())
                        # Track NDMI minimum across processed pixels for output-range QA.
                        fire_hazard_hls_ndmi_minimum = window_minimum if \
                            fire_hazard_hls_ndmi_minimum is None else \
                            min(fire_hazard_hls_ndmi_minimum,
                            window_minimum)
                        # Track NDMI maximum across processed pixels for output-range QA.
                        fire_hazard_hls_ndmi_maximum = window_maximum if \
                            fire_hazard_hls_ndmi_maximum is None else \
                            max(fire_hazard_hls_ndmi_maximum,
                            window_maximum)

                    # Handle the retained count values case explicitly during HLS NDMI temporal
                    # median composite.
                    if retained_count_values.size > 0:
                        # Calculate window count minimum to quantify completeness and support QA
                        # checks.
                        window_count_minimum = int(retained_count_values.min())
                        # Calculate window count maximum to quantify completeness and support QA
                        # checks.
                        window_count_maximum = int(retained_count_values.max())
                        # Calculate NDMI minimum observation count to quantify completeness and
                        # support QA checks.
                        fire_hazard_hls_ndmi_minimum_observation_count = window_count_minimum if \
                            fire_hazard_hls_ndmi_minimum_observation_count is None else \
                            min(fire_hazard_hls_ndmi_minimum_observation_count,
                            window_count_minimum)
                        # Calculate NDMI maximum observation count to quantify completeness and
                        # support QA checks.
                        fire_hazard_hls_ndmi_maximum_observation_count = window_count_maximum if \
                            fire_hazard_hls_ndmi_maximum_observation_count is None else \
                            max(fire_hazard_hls_ndmi_maximum_observation_count,
                            window_count_maximum)
                    del ndmi_window_stack
                    del ndmi_window_median
                    del ndmi_composite_array
                    del ndmi_count_array
                    del valid_observation_count
                composite_destination.set_band_description(1, 'HLS 2025 fire-season NDMI median')
                composite_destination.update_tags(DATA_SOURCE='NASA HLS Version 2.0',
                    INDEX_NAME='NDMI', COMPOSITE_METHOD='median', \
                        SOURCE_RASTER_COUNT=fire_hazard_hls_ndmi_source_count,
                    MINIMUM_VALID_OBSERVATIONS=fire_hazard_hls_composite_minimum_observations,
                    TARGET_CRS=fire_hazard_target_crs, TARGET_CELL_SIZE=fire_hazard_cell_size)
                count_destination.set_band_description(1, 'Valid HLS NDMI observation count')
                count_destination.update_tags(DATA_SOURCE='NASA HLS Version 2.0',
                    INDEX_NAME='NDMI', RASTER_PURPOSE='Temporal composite observation count',
                    SOURCE_RASTER_COUNT=fire_hazard_hls_ndmi_source_count)
    # Record NDMI temporary outputs valid so HLS NDMI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndmi_temporary_outputs_valid = \
        validate_existing_hls_ndmi_composite(fire_hazard_hls_ndmi_composite_temporary_path,
        fire_hazard_hls_ndmi_count_temporary_path)
    # Record NDMI content valid so HLS NDMI temporal median composite can preserve a clear
    # processing and validation outcome.
    fire_hazard_hls_ndmi_content_valid = fire_hazard_hls_ndmi_valid_composite_pixels > 0 and \
        fire_hazard_hls_ndmi_minimum is not None and (fire_hazard_hls_ndmi_maximum is not None) \
        and (fire_hazard_hls_ndmi_minimum >= fire_hazard_hls_composite_minimum_value) and \
        (fire_hazard_hls_ndmi_maximum <= fire_hazard_hls_composite_maximum_value) and \
        (fire_hazard_hls_ndmi_minimum_observation_count >= \
        fire_hazard_hls_composite_minimum_observations) and \
        (fire_hazard_hls_ndmi_maximum_observation_count <= fire_hazard_hls_ndmi_source_count)

    # Stop execution if temporary NDMI temporal-composite outputs failed validation.
    if not fire_hazard_hls_ndmi_temporary_outputs_valid or not fire_hazard_hls_ndmi_content_valid:
        raise ValueError(f'The temporary NDMI '
            f'temporal-composite outputs '
            f'failed validation.\n\nRaster '
            f'structure valid: '
            f'{fire_hazard_hls_ndmi_temporary_outputs_valid}\n'
            f'Valid composite pixels: '
            f'{fire_hazard_hls_ndmi_valid_composite_pixels:,}\n'
            f'NDMI minimum: '
            f'{fire_hazard_hls_ndmi_minimum}\n'
            f'NDMI maximum: '
            f'{fire_hazard_hls_ndmi_maximum}\n'
            f'Minimum observation count: '
            f'{fire_hazard_hls_ndmi_minimum_observation_count}\n'
            f'Maximum observation count: '
            f'{fire_hazard_hls_ndmi_maximum_observation_count}')
    fire_hazard_hls_ndmi_composite_temporary_path.replace(fire_hazard_hls_ndmi_composite_path)
    fire_hazard_hls_ndmi_count_temporary_path.replace(fire_hazard_hls_ndmi_observation_count_path)
# Record NDMI composite valid so HLS NDMI temporal median composite can preserve a clear processing
# and validation outcome.
fire_hazard_hls_ndmi_composite_valid = \
    validate_existing_hls_ndmi_composite(fire_hazard_hls_ndmi_composite_path,
    fire_hazard_hls_ndmi_observation_count_path) and fire_hazard_hls_ndmi_valid_composite_pixels \
        > 0 and (fire_hazard_hls_ndmi_minimum is not None) and (fire_hazard_hls_ndmi_maximum is \
        not None)

# Stop execution if final NDMI temporal composite or its observation-count raster failed validation.
if not fire_hazard_hls_ndmi_composite_valid:
    raise ValueError('The final NDMI temporal '
        'composite or its '
        'observation-count raster '
        'failed validation.')
# Store NDMI composite size bytes needed to carry out HLS NDMI temporal median composite.
fire_hazard_hls_ndmi_composite_size_bytes = fire_hazard_hls_ndmi_composite_path.stat().st_size
# Calculate NDMI count size bytes to quantify completeness and support QA checks.
fire_hazard_hls_ndmi_count_size_bytes = fire_hazard_hls_ndmi_observation_count_path.stat().st_size
# Define NDMI composite manifest records to control the inputs and rules used by HLS NDMI temporal
# median composite.
fire_hazard_hls_ndmi_composite_manifest_records = [{'INDEX_NAME': 'NDMI',
    'OUTPUT_TYPE': 'Temporal median composite', 'OUTPUT_PATH': \
        str(fire_hazard_hls_ndmi_composite_path),
    'PROCESSING_ACTION': fire_hazard_hls_ndmi_processing_action,
    'COMPOSITE_METHOD': 'median', 'SOURCE_RASTER_COUNT': fire_hazard_hls_ndmi_source_count,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'VALID_PIXELS': fire_hazard_hls_ndmi_valid_composite_pixels,
    'NODATA_PIXELS': fire_hazard_hls_ndmi_nodata_composite_pixels,
    'VALUE_MINIMUM': fire_hazard_hls_ndmi_minimum,
    'VALUE_MAXIMUM': fire_hazard_hls_ndmi_maximum,
    'MINIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndmi_minimum_observation_count,
    'MAXIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndmi_maximum_observation_count,
    'OUTPUT_SIZE_BYTES': fire_hazard_hls_ndmi_composite_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_hls_ndmi_composite_size_bytes / 1024 ** 2,
    'VALID': fire_hazard_hls_ndmi_composite_valid},
    {'INDEX_NAME': 'NDMI', 'OUTPUT_TYPE': 'Observation count',
    'OUTPUT_PATH': str(fire_hazard_hls_ndmi_observation_count_path),
    'PROCESSING_ACTION': fire_hazard_hls_ndmi_processing_action,
    'COMPOSITE_METHOD': 'count', 'SOURCE_RASTER_COUNT': fire_hazard_hls_ndmi_source_count,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'VALID_PIXELS': fire_hazard_hls_ndmi_valid_composite_pixels,
    'NODATA_PIXELS': fire_hazard_hls_ndmi_nodata_composite_pixels,
    'VALUE_MINIMUM': fire_hazard_hls_ndmi_minimum_observation_count,
    'VALUE_MAXIMUM': fire_hazard_hls_ndmi_maximum_observation_count,
    'MINIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndmi_minimum_observation_count,
    'MAXIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndmi_maximum_observation_count,
    'OUTPUT_SIZE_BYTES': fire_hazard_hls_ndmi_count_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_hls_ndmi_count_size_bytes / 1024 ** 2,
    'VALID': fire_hazard_hls_ndmi_composite_valid}]

# Use the existing file only when it is present and valid for HLS NDMI temporal median composite.
if fire_hazard_hls_composite_manifest_path.exists():
    # Store existing composite manifest needed to carry out HLS NDMI temporal median composite.
    existing_composite_manifest = pd.read_csv(fire_hazard_hls_composite_manifest_path)
else:
    # Store existing composite manifest needed to carry out HLS NDMI temporal median composite.
    existing_composite_manifest = pd.DataFrame()

# Handle missing or invalid existing composite manifest explicitly during HLS NDMI temporal median
# composite.
if not existing_composite_manifest.empty:
    # Create an isolated existing composite manifest working copy so HLS NDMI temporal median
    # composite does not modify upstream data.
    existing_composite_manifest = \
        existing_composite_manifest[existing_composite_manifest['INDEX_NAME'].astype(str).str \
        .upper() != 'NDMI'].copy()
# Store composite manifest needed to carry out HLS NDMI temporal median composite.
fire_hazard_hls_composite_manifest = pd.concat([existing_composite_manifest,
    pd.DataFrame(fire_hazard_hls_ndmi_composite_manifest_records)],
    ignore_index=True).sort_values(['INDEX_NAME', 'OUTPUT_TYPE']).reset_index(drop=True)
# Save the composite manifest CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_composite_manifest.to_csv(fire_hazard_hls_composite_manifest_path, index=False)
# Store NDMI composite summary needed to carry out HLS NDMI temporal median composite.
fire_hazard_hls_ndmi_composite_summary = pd.DataFrame([{'INDEX_NAME': 'NDMI',
    'COMPOSITE_METHOD': 'Median', 'SOURCE_RASTERS': fire_hazard_hls_ndmi_source_count,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'VALID_COMPOSITE_PIXELS': fire_hazard_hls_ndmi_valid_composite_pixels,
    'NODATA_COMPOSITE_PIXELS': fire_hazard_hls_ndmi_nodata_composite_pixels,
    'NDMI_MINIMUM': fire_hazard_hls_ndmi_minimum, 'NDMI_MAXIMUM': fire_hazard_hls_ndmi_maximum,
    'MINIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndmi_minimum_observation_count,
    'MAXIMUM_OBSERVATION_COUNT': fire_hazard_hls_ndmi_maximum_observation_count,
    'COMPOSITE_SIZE_MB': fire_hazard_hls_ndmi_composite_size_bytes / 1024 ** 2,
    'COUNT_RASTER_SIZE_MB': fire_hazard_hls_ndmi_count_size_bytes / 1024 ** 2,
    'PROCESSING_ACTION': fire_hazard_hls_ndmi_processing_action,
    'VALID': fire_hazard_hls_ndmi_composite_valid}])
print(f'-> NDMI source rasters: {fire_hazard_hls_ndmi_source_count:,}')
print(f'-> Processing action: {fire_hazard_hls_ndmi_processing_action}')
print(f'-> Valid composite pixels: {fire_hazard_hls_ndmi_valid_composite_pixels:,}')
print(f'-> Composite NDMI range: '
    f'{fire_hazard_hls_ndmi_minimum:.4f} '
    f'to '
    f'{fire_hazard_hls_ndmi_maximum:.4f}')
print(f'-> Observation-count range: '
    f'{fire_hazard_hls_ndmi_minimum_observation_count} '
    f'to '
    f'{fire_hazard_hls_ndmi_maximum_observation_count}')
print(f'-> NDMI composite saved: {fire_hazard_hls_ndmi_composite_path}')
print(f'-> Observation-count raster saved: {fire_hazard_hls_ndmi_observation_count_path}')
print(f'-> Composite manifest saved: {fire_hazard_hls_composite_manifest_path}')
print('\n--- HLS NDMI TEMPORAL COMPOSITE SUMMARY ---')
display(fire_hazard_hls_ndmi_composite_summary)
print('\n--- HLS TEMPORAL COMPOSITE MANIFEST SAMPLE ---')
display(fire_hazard_hls_composite_manifest.head(5))
import gc
gc.collect()
print('\nNOTE:')
print('The NDMI temporal composite '
    'represents the pixel-wise '
    'median vegetation and canopy '
    'moisture across the validated '
    '2025 fire-season HLS scenes.')
print(f'Pixels with fewer than '
    f'{fire_hazard_hls_composite_minimum_observations} '
    f'valid observations remain '
    f'NoData in the composite.')
print('The companion '
    'observation-count raster '
    'records the number of valid '
    'NDMI observations available at '
    'each target-grid pixel.')
print('The next step is to '
    'independently validate the '
    'NDVI and NDMI temporal '
    'composites and their '
    'observation-count rasters.')
print('\n=== HLS NDMI TEMPORAL MEDIAN COMPOSITE COMPLETE ===')


=== BUILDING HLS NDMI TEMPORAL MEDIAN COMPOSITE ===
-> Valid cached NDMI composite and observation-count raster found.
-> NDMI source rasters: 63
-> Processing action: Reused existing temporal composite
-> Valid composite pixels: 14,140,492
-> Composite NDMI range: -0.5608 to 1.0000
-> Observation-count range: 3 to 24
-> NDMI composite saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\rasters\hls_2025_fire_season_ndmi_median.tif
-> Observation-count raster saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\observation_counts\hls_2025_fire_season_ndmi_observation_count.tif
-> Composite manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_manifest.csv

--- HLS NDMI TEMPORAL COMPOSITE SUMMARY ---


,INDEX_NAME,COMPOSITE_METHOD,SOURCE_RASTERS,MINIMUM_VALID_OBSERVATIONS,VALID_COMPOSITE_PIXELS,NODATA_COMPOSITE_PIXELS,NDMI_MINIMUM,NDMI_MAXIMUM,MINIMUM_OBSERVATION_COUNT,MAXIMUM_OBSERVATION_COUNT,COMPOSITE_SIZE_MB,COUNT_RASTER_SIZE_MB,PROCESSING_ACTION,VALID
0,NDMI,Median,63,3,14140492,122630526,-0.56077,1.0,3,24,48.038012,1.691106,Reused existing temporal composite,True



--- HLS TEMPORAL COMPOSITE MANIFEST SAMPLE ---


,INDEX_NAME,OUTPUT_TYPE,OUTPUT_PATH,PROCESSING_ACTION,COMPOSITE_METHOD,SOURCE_RASTER_COUNT,MINIMUM_VALID_OBSERVATIONS,VALID_PIXELS,NODATA_PIXELS,VALUE_MINIMUM,VALUE_MAXIMUM,MINIMUM_OBSERVATION_COUNT,MAXIMUM_OBSERVATION_COUNT,OUTPUT_SIZE_BYTES,OUTPUT_SIZE_MB,VALID
0,NDMI,Observation count,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,count,63,3,14140492,122630526,3.000000,24.0,3,24,1773253,1.691106,True
1,NDMI,Temporal median composite,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,median,63,3,14140492,122630526,-0.560770,1.0,3,24,50371506,48.038012,True
2,NDVI,Observation count,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,count,63,3,14140492,122630526,3.000000,24.0,3,24,1773259,1.691112,True
3,NDVI,Temporal median composite,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing temporal composite,median,63,3,14140492,122630526,-0.246165,1.0,3,24,47258426,45.069147,True



NOTE:
The NDMI temporal composite represents the pixel-wise median vegetation and canopy moisture across the validated 2025 fire-season HLS scenes.
Pixels with fewer than 3 valid observations remain NoData in the composite.
The companion observation-count raster records the number of valid NDMI observations available at each target-grid pixel.
The next step is to independently validate the NDVI and NDMI temporal composites and their observation-count rasters.

=== HLS NDMI TEMPORAL MEDIAN COMPOSITE COMPLETE ===


### Validating HLS Temporal Composites


In [88]:
print('=== VALIDATING HLS TEMPORAL COMPOSITES ===')
# List the required HLS composite validation prerequisites required before HLS temporal
# composites can run.
required_hls_composite_validation_inputs = ['fire_hazard_hls_ndvi_composite_path',
    'fire_hazard_hls_ndvi_observation_count_path',
    'fire_hazard_hls_ndmi_composite_path', 'fire_hazard_hls_ndmi_observation_count_path',
    'fire_hazard_hls_composite_manifest_path', 'fire_hazard_hls_composite_validation_path',
    'fire_hazard_hls_composite_validation_summary_path',
    'fire_hazard_hls_composite_minimum_observations',
    'fire_hazard_hls_composite_minimum_value', 'fire_hazard_hls_composite_maximum_value',
    'fire_hazard_hls_composite_nodata', 'fire_hazard_hls_composite_count_nodata',
    'fire_hazard_hls_composite_dtype', 'fire_hazard_hls_composite_count_dtype',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_alignment_study_area_mask']
# Identify unavailable HLS composite validation so HLS temporal composites stops before
# using incomplete inputs.
missing_hls_composite_validation_inputs = [object_name for object_name in \
    required_hls_composite_validation_inputs if object_name not in globals()]

# Stop execution when required hls composite validation inputs inputs are unavailable.
if missing_hls_composite_validation_inputs:
    raise NameError(f'The following HLS '
        f'temporal-composite validation '
        f'objects are missing:\n'
        f'{missing_hls_composite_validation_inputs}\n\n'
        f'Run Build NDVI Temporal '
        f'Composite and Build NDMI '
        f'Temporal Composite before '
        f'validating the final products.')
# Store expected composite products needed to carry out HLS temporal composites.
fire_hazard_hls_expected_composite_products = pd.DataFrame([{'INDEX_NAME': 'NDVI',
    'OUTPUT_TYPE': 'Temporal median composite', 'OUTPUT_PATH': \
        str(fire_hazard_hls_ndvi_composite_path),
    'EXPECTED_DTYPE': fire_hazard_hls_composite_dtype,
    'EXPECTED_NODATA': fire_hazard_hls_composite_nodata},
    {'INDEX_NAME': 'NDVI', 'OUTPUT_TYPE': 'Observation count',
    'OUTPUT_PATH': str(fire_hazard_hls_ndvi_observation_count_path),
    'EXPECTED_DTYPE': fire_hazard_hls_composite_count_dtype,
    'EXPECTED_NODATA': fire_hazard_hls_composite_count_nodata},
    {'INDEX_NAME': 'NDMI', 'OUTPUT_TYPE': 'Temporal median composite',
    'OUTPUT_PATH': str(fire_hazard_hls_ndmi_composite_path),
    'EXPECTED_DTYPE': fire_hazard_hls_composite_dtype,
    'EXPECTED_NODATA': fire_hazard_hls_composite_nodata},
    {'INDEX_NAME': 'NDMI', 'OUTPUT_TYPE': 'Observation count',
    'OUTPUT_PATH': str(fire_hazard_hls_ndmi_observation_count_path),
    'EXPECTED_DTYPE': fire_hazard_hls_composite_count_dtype,
    'EXPECTED_NODATA': fire_hazard_hls_composite_count_nodata}])
# Record composite valid extensions so HLS temporal composites can preserve a clear processing and
# validation outcome.
fire_hazard_hls_composite_valid_extensions = {'.tif', '.tiff'}
# Track composite minimum file size bytes across processed pixels for output-range QA.
fire_hazard_hls_composite_minimum_file_size_bytes = 1024
# Set composite value tolerance to control HLS temporal composites.
fire_hazard_hls_composite_value_tolerance = 1e-06
# Build the expected hls composite mask shape mask used to isolate records required for this
# analysis.
expected_hls_composite_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if hLS composite study-area mask does not match the target raster grid.
if fire_hazard_alignment_study_area_mask.shape != expected_hls_composite_mask_shape:
    raise ValueError(f'The HLS composite study-area '
        f'mask does not match the target '
        f'raster grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_hls_composite_mask_shape}')
# Calculate composite study area pixel count to quantify completeness and support QA checks.
fire_hazard_hls_composite_study_area_pixel_count = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if temporal-composite study-area mask contains no included pixels.
if fire_hazard_hls_composite_study_area_pixel_count == 0:
    raise ValueError('The temporal-composite study-area mask contains no included pixels.')
# Record composite validation records so HLS temporal composites can preserve a clear processing and
# validation outcome.
fire_hazard_hls_composite_validation_records = []

# Process each ( entry so HLS temporal composites is applied consistently across all records.
for _, expected_product in fire_hazard_hls_expected_composite_products.iterrows():
    # Store index name needed to carry out HLS temporal composites.
    index_name = str(expected_product['INDEX_NAME']).upper()
    # Store output type needed to carry out HLS temporal composites.
    output_type = str(expected_product['OUTPUT_TYPE'])
    # Build the output path location so HLS temporal composites uses the expected project file
    # structure.
    output_path = Path(expected_product['OUTPUT_PATH'])
    # Store expected dtype needed to carry out HLS temporal composites.
    expected_dtype = str(expected_product['EXPECTED_DTYPE'])
    # Store expected NoData needed to carry out HLS temporal composites.
    expected_nodata = expected_product['EXPECTED_NODATA']
    # Store file exists needed to carry out HLS temporal composites.
    file_exists = output_path.exists() and output_path.is_file()
    # Record file extension valid so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    file_extension_valid = output_path.suffix.lower() in fire_hazard_hls_composite_valid_extensions
    # Store file size bytes needed to carry out HLS temporal composites.
    file_size_bytes = output_path.stat().st_size if file_exists else 0
    # Record file size valid so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    file_size_valid = file_size_bytes >= fire_hazard_hls_composite_minimum_file_size_bytes
    # Record raster readable so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    raster_readable = False
    # Calculate band count valid to quantify completeness and support QA checks.
    band_count_valid = False
    # Record width valid so HLS temporal composites can preserve a clear processing and validation
    # outcome.
    width_valid = False
    # Record height valid so HLS temporal composites can preserve a clear processing and validation
    # outcome.
    height_valid = False
    # Store crs valid so spatial operations use the required coordinate reference system.
    crs_valid = False
    # Record transform valid so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    transform_valid = False
    # Record pixel size valid so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    pixel_size_valid = False
    # Record dtype valid so HLS temporal composites can preserve a clear processing and validation
    # outcome.
    dtype_valid = False
    # Record NoData valid so HLS temporal composites can preserve a clear processing and validation
    # outcome.
    nodata_valid = False
    # Record grid valid so HLS temporal composites can preserve a clear processing and validation
    # outcome.
    grid_valid = False
    # Record structure valid so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    structure_valid = False
    # Calculate valid pixel count to quantify completeness and support QA checks.
    valid_pixel_count = 0
    # Calculate NoData pixel count to quantify completeness and support QA checks.
    nodata_pixel_count = 0
    # Calculate valid inside study area count to quantify completeness and support QA checks.
    valid_inside_study_area_count = 0
    # Calculate valid outside study area count to quantify completeness and support QA checks.
    valid_outside_study_area_count = 0
    # Track minimum value across processed pixels for output-range QA.
    minimum_value = None
    # Track maximum value across processed pixels for output-range QA.
    maximum_value = None
    # Set value range available to control HLS temporal composites.
    value_range_available = False
    # Record value range valid so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    value_range_valid = False
    # Record validation error so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    validation_error = None

    # Handle the file exists case explicitly during HLS temporal composites.
    if file_exists and file_extension_valid and file_size_valid:

        # Protect HLS temporal composites so expected source or file failures do not leave partial
        # outputs.
        try:

            # Open the raster in a managed context so its file handle closes reliably after HLS
            # temporal composites.
            with rasterio.open(output_path) as product_source:
                # Record raster readable so HLS temporal composites can preserve a clear processing
                # and validation outcome.
                raster_readable = True
                # Calculate band count valid to quantify completeness and support QA checks.
                band_count_valid = product_source.count == 1
                # Record width valid so HLS temporal composites can preserve a clear processing and
                # validation outcome.
                width_valid = product_source.width == fire_hazard_alignment_width
                # Record height valid so HLS temporal composites can preserve a clear processing and
                # validation outcome.
                height_valid = product_source.height == fire_hazard_alignment_height
                # Store crs valid so spatial operations use the required coordinate reference
                # system.
                crs_valid = product_source.crs is not None and product_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs)
                # Record transform valid so HLS temporal composites can preserve a clear processing
                # and validation outcome.
                transform_valid = \
                    product_source.transform.almost_equals(fire_hazard_alignment_transform)
                # Record pixel size valid so HLS temporal composites can preserve a clear processing
                # and validation outcome.
                pixel_size_valid = np.isclose(abs(product_source.transform.a),
                    fire_hazard_cell_size) and np.isclose(abs(product_source.transform.e),
                    fire_hazard_cell_size)
                # Record dtype valid so HLS temporal composites can preserve a clear processing and
                # validation outcome.
                dtype_valid = product_source.dtypes[0] == expected_dtype
                # Record NoData valid so HLS temporal composites can preserve a clear processing and
                # validation outcome.
                nodata_valid = product_source.nodata == expected_nodata
                # Record grid valid so HLS temporal composites can preserve a clear processing and
                # validation outcome.
                grid_valid = all([width_valid,
                    height_valid, crs_valid, transform_valid, pixel_size_valid])
                # Record structure valid so HLS temporal composites can preserve a clear processing
                # and validation outcome.
                structure_valid = all([band_count_valid, dtype_valid, nodata_valid])

                # Process each ( entry so HLS temporal composites is applied consistently across all
                # records.
                for _, product_window in product_source.block_windows(1):
                    # Prepare product array used to process the current raster window.
                    product_array = product_source.read(1, window=product_window)
                    # Calculate row start for the current raster processing window.
                    row_start = int(product_window.row_off)
                    # Store row end needed to carry out HLS temporal composites.
                    row_end = int(product_window.row_off + product_window.height)
                    # Store column start needed to carry out HLS temporal composites.
                    column_start = int(product_window.col_off)
                    # Store column end needed to carry out HLS temporal composites.
                    column_end = int(product_window.col_off + product_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]

                    # Handle the output type case explicitly during HLS temporal composites.
                    if output_type == 'Temporal median composite':
                        # Build the valid product mask mask used to isolate records required for
                        # this analysis.
                        valid_product_mask = np.isfinite(product_array) & (product_array != \
                            fire_hazard_hls_composite_nodata)
                    else:
                        # Build the valid product mask mask used to isolate records required for
                        # this analysis.
                        valid_product_mask = product_array > fire_hazard_hls_composite_count_nodata
                    # Build the valid inside mask mask used to isolate records required for this
                    # analysis.
                    valid_inside_mask = valid_product_mask & study_area_window_mask
                    # Build the valid outside mask mask used to isolate records required for this
                    # analysis.
                    valid_outside_mask = valid_product_mask & ~study_area_window_mask
                    # Record valid values so HLS temporal composites can preserve a clear processing
                    # and validation outcome.
                    valid_values = product_array[valid_product_mask]
                    # Calculate valid pixel count to quantify completeness and support QA checks.
                    valid_pixel_count += int(valid_product_mask.sum())
                    # Calculate NoData pixel count to quantify completeness and support QA checks.
                    nodata_pixel_count += int((~valid_product_mask).sum())
                    # Calculate valid inside study area count to quantify completeness and support
                    # QA checks.
                    valid_inside_study_area_count += int(valid_inside_mask.sum())
                    # Calculate valid outside study area count to quantify completeness and support
                    # QA checks.
                    valid_outside_study_area_count += int(valid_outside_mask.sum())

                    # Handle the valid values case explicitly during HLS temporal composites.
                    if valid_values.size > 0:
                        # Track block minimum across processed pixels for output-range QA.
                        block_minimum = float(valid_values.min())
                        # Track block maximum across processed pixels for output-range QA.
                        block_maximum = float(valid_values.max())
                        # Track minimum value across processed pixels for output-range QA.
                        minimum_value = block_minimum if minimum_value is None else \
                            min(minimum_value,
                            block_minimum)
                        # Track maximum value across processed pixels for output-range QA.
                        maximum_value = block_maximum if maximum_value is None else \
                            max(maximum_value,
                            block_maximum)
                # Store value range available needed to carry out HLS temporal composites.
                value_range_available = minimum_value is not None and maximum_value is not None

                # Handle the output type case explicitly during HLS temporal composites.
                if output_type == 'Temporal median composite':
                    # Record value range valid so HLS temporal composites can preserve a clear
                    # processing and validation outcome.
                    value_range_valid = value_range_available and minimum_value >= \
                        fire_hazard_hls_composite_minimum_value - \
                        fire_hazard_hls_composite_value_tolerance and (maximum_value <= \
                        fire_hazard_hls_composite_maximum_value + \
                        fire_hazard_hls_composite_value_tolerance)
                else:
                    # Calculate configured source count to quantify completeness and support QA
                    # checks.
                    configured_source_count = \
                        int((fire_hazard_hls_composite_sources['INDEX_NAME'].astype(str).str \
                        .upper() == index_name).sum())
                    # Record value range valid so HLS temporal composites can preserve a clear
                    # processing and validation outcome.
                    value_range_valid = value_range_available and minimum_value >= 1 and \
                        (maximum_value <= configured_source_count)
        # Handle the expected failure without leaving the workflow in an inconsistent state.
        except Exception as error:
            # Record validation error so HLS temporal composites can preserve a clear processing and
            # validation outcome.
            validation_error = str(error)
    # Record product valid so HLS temporal composites can preserve a clear processing and validation
    # outcome.
    product_valid = all([file_exists, file_extension_valid,
        file_size_valid, raster_readable, grid_valid, structure_valid,
        valid_inside_study_area_count > 0, valid_outside_study_area_count == 0,
        value_range_available, value_range_valid, validation_error is None])
    fire_hazard_hls_composite_validation_records.append({'INDEX_NAME': index_name,
        'OUTPUT_TYPE': output_type, 'OUTPUT_PATH': str(output_path),
        'FILE_EXISTS': file_exists, 'FILE_EXTENSION_VALID': file_extension_valid,
        'FILE_SIZE_BYTES': file_size_bytes, 'FILE_SIZE_MB': file_size_bytes / 1024 ** 2,
        'FILE_SIZE_VALID': file_size_valid, 'RASTER_READABLE': raster_readable,
        'BAND_COUNT_VALID': band_count_valid, 'WIDTH_VALID': width_valid,
        'HEIGHT_VALID': height_valid, 'CRS_VALID': crs_valid,
        'TRANSFORM_VALID': transform_valid, 'PIXEL_SIZE_VALID': pixel_size_valid,
        'DTYPE_VALID': dtype_valid, 'NODATA_VALID': nodata_valid,
        'GRID_VALID': grid_valid, 'STRUCTURE_VALID': structure_valid,
        'VALID_PIXELS': valid_pixel_count, 'NODATA_PIXELS': nodata_pixel_count,
        'VALID_INSIDE_STUDY_AREA': valid_inside_study_area_count,
        'VALID_OUTSIDE_STUDY_AREA': valid_outside_study_area_count,
        'VALUE_MINIMUM': minimum_value, 'VALUE_MAXIMUM': maximum_value,
        'VALUE_RANGE_AVAILABLE': value_range_available,
        'VALUE_RANGE_VALID': value_range_valid, 'ERROR_MESSAGE': validation_error,
        'VALID': product_valid})
# Record composite validation so HLS temporal composites can preserve a clear processing and
# validation outcome.
fire_hazard_hls_composite_validation = \
    pd.DataFrame(fire_hazard_hls_composite_validation_records).sort_values(['INDEX_NAME',
    'OUTPUT_TYPE']).reset_index(drop=True)
# Define composite pair records to control the inputs and rules used by HLS temporal composites.
fire_hazard_hls_composite_pair_records = []

# Process each index name entry so HLS temporal composites is applied consistently across all
# records.
for index_name in ['NDVI', 'NDMI']:

    # Handle the index name case explicitly during HLS temporal composites.
    if index_name == 'NDVI':
        # Build the composite path location so HLS temporal composites uses the expected project
        # file structure.
        composite_path = fire_hazard_hls_ndvi_composite_path
        # Build the count path location so HLS temporal composites uses the expected project file
        # structure.
        count_path = fire_hazard_hls_ndvi_observation_count_path
    else:
        # Build the composite path location so HLS temporal composites uses the expected project
        # file structure.
        composite_path = fire_hazard_hls_ndmi_composite_path
        # Build the count path location so HLS temporal composites uses the expected project file
        # structure.
        count_path = fire_hazard_hls_ndmi_observation_count_path
    # Calculate composite and count grids match to quantify completeness and support QA checks.
    composite_and_count_grids_match = False
    # Record composite support rule valid so HLS temporal composites can preserve a clear processing
    # and validation outcome.
    composite_support_rule_valid = True
    # Calculate unsupported composite pixel count to quantify completeness and support QA checks.
    unsupported_composite_pixel_count = 0
    # Calculate supported NoData pixel count to quantify completeness and support QA checks.
    supported_nodata_pixel_count = 0
    # Calculate matching valid pixel count to quantify completeness and support QA checks.
    matching_valid_pixel_count = 0
    # Record paired validation error so HLS temporal composites can preserve a clear processing and
    # validation outcome.
    paired_validation_error = None

    # Protect HLS temporal composites so expected source or file failures do not leave partial
    # outputs.
    try:

        # Open the raster in a managed context so its file handle closes reliably after HLS temporal
        # composites.
        with rasterio.open(composite_path) as composite_source:

            # Open the raster in a managed context so its file handle closes reliably after HLS
            # temporal composites.
            with rasterio.open(count_path) as count_source:
                # Calculate composite and count grids match to quantify completeness and support QA
                # checks.
                composite_and_count_grids_match = all([composite_source.width == count_source.width,
                    composite_source.height == count_source.height,
                    composite_source.crs == count_source.crs, \
                        composite_source.transform.almost_equals(count_source.transform)])

                # Stop execution if the reported value composite and observation-count rasters do
                # not use the same grid.
                if not composite_and_count_grids_match:
                    raise ValueError(f'The {index_name} composite and '
                        f'observation-count rasters do '
                        f'not use the same grid.')

                # Process each ( entry so HLS temporal composites is applied consistently across all
                # records.
                for _, paired_window in composite_source.block_windows(1):
                    # Prepare composite array used to process the current raster window.
                    composite_array = composite_source.read(1, window=paired_window)
                    # Calculate count array to quantify completeness and support QA checks.
                    count_array = count_source.read(1, window=paired_window)
                    # Build the composite valid mask mask used to isolate records required for this
                    # analysis.
                    composite_valid_mask = np.isfinite(composite_array) & (composite_array != \
                        fire_hazard_hls_composite_nodata)
                    # Build the sufficient count mask mask used to isolate records required for this
                    # analysis.
                    sufficient_count_mask = count_array >= \
                        fire_hazard_hls_composite_minimum_observations
                    # Build the unsupported composite mask mask used to isolate records required for
                    # this analysis.
                    unsupported_composite_mask = composite_valid_mask & ~sufficient_count_mask
                    # Build the supported nodata mask mask used to isolate records required for this
                    # analysis.
                    supported_nodata_mask = sufficient_count_mask & ~composite_valid_mask
                    # Build the matching valid mask mask used to isolate records required for this
                    # analysis.
                    matching_valid_mask = composite_valid_mask & sufficient_count_mask
                    # Calculate unsupported composite pixel count to quantify completeness and
                    # support QA checks.
                    unsupported_composite_pixel_count += int(unsupported_composite_mask.sum())
                    # Calculate supported NoData pixel count to quantify completeness and support QA
                    # checks.
                    supported_nodata_pixel_count += int(supported_nodata_mask.sum())
                    # Calculate matching valid pixel count to quantify completeness and support QA
                    # checks.
                    matching_valid_pixel_count += int(matching_valid_mask.sum())
                # Record composite support rule valid so HLS temporal composites can preserve a
                # clear processing and validation outcome.
                composite_support_rule_valid = unsupported_composite_pixel_count == 0 and \
                    supported_nodata_pixel_count == 0
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception as error:
        # Record paired validation error so HLS temporal composites can preserve a clear processing
        # and validation outcome.
        paired_validation_error = str(error)
        # Record composite support rule valid so HLS temporal composites can preserve a clear
        # processing and validation outcome.
        composite_support_rule_valid = False
    fire_hazard_hls_composite_pair_records.append({'INDEX_NAME': index_name,
        'COMPOSITE_PATH': str(composite_path), 'COUNT_PATH': str(count_path),
        'GRIDS_MATCH': composite_and_count_grids_match,
        'MINIMUM_REQUIRED_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
        'MATCHING_VALID_PIXELS': matching_valid_pixel_count,
        'UNSUPPORTED_COMPOSITE_PIXELS': unsupported_composite_pixel_count,
        'SUPPORTED_NODATA_PIXELS': supported_nodata_pixel_count,
        'SUPPORT_RULE_VALID': composite_support_rule_valid,
        'ERROR_MESSAGE': paired_validation_error, 'VALID': composite_and_count_grids_match and \
            composite_support_rule_valid and (paired_validation_error is None)})
# Record composite pair validation so HLS temporal composites can preserve a clear processing and
# validation outcome.
fire_hazard_hls_composite_pair_validation = pd.DataFrame(fire_hazard_hls_composite_pair_records)

# Open the raster in a managed context so its file handle closes reliably after HLS temporal
# composites.
with rasterio.open(fire_hazard_hls_ndvi_composite_path) as ndvi_source:

    # Open the raster in a managed context so its file handle closes reliably after HLS temporal
    # composites.
    with rasterio.open(fire_hazard_hls_ndmi_composite_path) as ndmi_source:
        # Store composite index grids match needed to carry out HLS temporal composites.
        fire_hazard_hls_composite_index_grids_match = all([ndvi_source.width == ndmi_source.width,
            ndvi_source.height == ndmi_source.height, ndvi_source.crs == ndmi_source.crs,
            ndvi_source.transform.almost_equals(ndmi_source.transform)])

# Stop execution if hLS temporal-composite processing manifest could not be found: the reported
# value.
if not fire_hazard_hls_composite_manifest_path.exists():
    raise FileNotFoundError(f'The HLS temporal-composite '
        f'processing manifest could not '
        f'be found:\n'
        f'{fire_hazard_hls_composite_manifest_path}')
# Store composite manifest needed to carry out HLS temporal composites.
fire_hazard_hls_composite_manifest = pd.read_csv(fire_hazard_hls_composite_manifest_path)
# List the required HLS composite manifest fields prerequisites required before HLS temporal
# composites can run.
required_hls_composite_manifest_fields = ['INDEX_NAME', 'OUTPUT_TYPE', 'OUTPUT_PATH', 'VALID']
# Identify unavailable HLS composite manifest fields so HLS temporal composites stops before
# using incomplete inputs.
missing_hls_composite_manifest_fields = [field_name for field_name in \
    required_hls_composite_manifest_fields if field_name not in \
    fire_hazard_hls_composite_manifest.columns]

# Stop execution when required hls composite manifest fields inputs are unavailable.
if missing_hls_composite_manifest_fields:
    raise ValueError(f'The HLS temporal-composite '
        f'manifest is missing required '
        f'fields:\n'
        f'{missing_hls_composite_manifest_fields}')
# Define expected hls composite manifest keys used to configure this workflow stage.
expected_hls_composite_manifest_keys = {('NDVI',
    'Temporal median composite'), ('NDVI', 'Observation count'),
    ('NDMI', 'Temporal median composite'), ('NDMI',
    'Observation count')}
# Collect actual HLS composite manifest keys for membership and completeness checks during HLS
# temporal composites.
actual_hls_composite_manifest_keys = \
    set(zip(fire_hazard_hls_composite_manifest['INDEX_NAME'].astype(str).str.upper(),
    fire_hazard_hls_composite_manifest['OUTPUT_TYPE'].astype(str)))
# Store composite manifest complete needed to carry out HLS temporal composites.
fire_hazard_hls_composite_manifest_complete = actual_hls_composite_manifest_keys == \
    expected_hls_composite_manifest_keys
# Calculate expected HLS composite product count to quantify completeness and support QA checks.
expected_hls_composite_product_count = len(fire_hazard_hls_expected_composite_products)
# Calculate validated HLS composite product count to quantify completeness and support QA checks.
validated_hls_composite_product_count = len(fire_hazard_hls_composite_validation)
# Calculate valid HLS composite product count to quantify completeness and support QA checks.
valid_hls_composite_product_count = int(fire_hazard_hls_composite_validation['VALID'].sum())
# Calculate valid HLS composite pair count to quantify completeness and support QA checks.
valid_hls_composite_pair_count = int(fire_hazard_hls_composite_pair_validation['VALID'].sum())
# Store failed HLS composite products needed to carry out HLS temporal composites.
failed_hls_composite_products = \
    fire_hazard_hls_composite_validation[~fire_hazard_hls_composite_validation['VALID']].copy() \
    .reset_index(drop=True)
# Store failed HLS composite pairs needed to carry out HLS temporal composites.
failed_hls_composite_pairs = \
    fire_hazard_hls_composite_pair_validation[~fire_hazard_hls_composite_pair_validation \
    ['VALID']].copy().reset_index(drop=True)
# Record composite validation complete so HLS temporal composites can preserve a clear processing
# and validation outcome.
fire_hazard_hls_composite_validation_complete = validated_hls_composite_product_count == \
    expected_hls_composite_product_count and valid_hls_composite_product_count == \
    expected_hls_composite_product_count and (valid_hls_composite_pair_count == 2) and \
    failed_hls_composite_products.empty and failed_hls_composite_pairs.empty and \
    fire_hazard_hls_composite_index_grids_match and fire_hazard_hls_composite_manifest_complete
# Store total HLS composite output size bytes needed to carry out HLS temporal composites.
total_hls_composite_output_size_bytes = \
    int(fire_hazard_hls_composite_validation['FILE_SIZE_BYTES'].sum())
# Record composite validation summary so HLS temporal composites can preserve a clear processing and
# validation outcome.
fire_hazard_hls_composite_validation_summary = pd.DataFrame([{'EXPECTED_PRODUCTS': \
    expected_hls_composite_product_count,
    'VALIDATED_PRODUCTS': validated_hls_composite_product_count,
    'VALID_PRODUCTS': valid_hls_composite_product_count,
    'FAILED_PRODUCTS': len(failed_hls_composite_products),
    'VALID_INDEX_PAIRS': valid_hls_composite_pair_count,
    'NDVI_NDMI_GRIDS_MATCH': fire_hazard_hls_composite_index_grids_match,
    'PROCESSING_MANIFEST_COMPLETE': fire_hazard_hls_composite_manifest_complete,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'TARGET_CRS': fire_hazard_target_crs, 'TARGET_CELL_SIZE': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'TOTAL_OUTPUT_SIZE_BYTES': total_hls_composite_output_size_bytes,
    'TOTAL_OUTPUT_SIZE_MB': total_hls_composite_output_size_bytes / 1024 ** 2,
    'VALIDATION_COMPLETE': fire_hazard_hls_composite_validation_complete}])
# Save the composite validation CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_composite_validation.to_csv(fire_hazard_hls_composite_validation_path, index=False)
# Save the composite validation summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_composite_validation_summary.to_csv \
    (fire_hazard_hls_composite_validation_summary_path,
    index=False)
# Build the composite pair validation path location so HLS temporal composites uses the expected
# project file structure.
fire_hazard_hls_composite_pair_validation_path = \
    Path(fire_hazard_hls_composite_validation_path).parent / \
    'hls_temporal_composite_pair_validation.csv'
# Save the composite pair validation CSV so later steps can reuse the recorded workflow results.
fire_hazard_hls_composite_pair_validation.to_csv(fire_hazard_hls_composite_pair_validation_path,
    index=False)
print(f'-> Expected temporal-composite products: {expected_hls_composite_product_count:,}')
print(f'-> Validated products: {validated_hls_composite_product_count:,}')
print(f'-> Valid products: {valid_hls_composite_product_count:,}')
print(f'-> Failed products: {len(failed_hls_composite_products):,}')
print(f'-> Valid composite/count pairs: {valid_hls_composite_pair_count:,} of 2')
print(f'-> NDVI and NDMI grids match: {fire_hazard_hls_composite_index_grids_match}')
print(f'-> Processing manifest complete: {fire_hazard_hls_composite_manifest_complete}')
print(f'-> Total composite output '
    f'size: '
    f'{total_hls_composite_output_size_bytes / 1024 ** 2:,.2f} '
    f'MB')
print(f'-> Temporal-composite validation complete: {fire_hazard_hls_composite_validation_complete}')
print(f'-> Validation table saved: {fire_hazard_hls_composite_validation_path}')
print(f'-> Validation summary saved: {fire_hazard_hls_composite_validation_summary_path}')
print('\n--- HLS TEMPORAL COMPOSITE VALIDATION SUMMARY ---')
display(fire_hazard_hls_composite_validation_summary)
print('\n--- HLS TEMPORAL COMPOSITE PRODUCT VALIDATION ---')
display(fire_hazard_hls_composite_validation[['INDEX_NAME',
    'OUTPUT_TYPE', 'FILE_EXISTS', 'GRID_VALID', 'STRUCTURE_VALID',
    'VALID_INSIDE_STUDY_AREA', 'VALID_OUTSIDE_STUDY_AREA',
    'VALUE_MINIMUM', 'VALUE_MAXIMUM', 'VALUE_RANGE_VALID',
    'VALID']])
print('\n--- HLS COMPOSITE SUPPORT-RULE VALIDATION ---')
display(fire_hazard_hls_composite_pair_validation[['INDEX_NAME',
    'GRIDS_MATCH', 'MATCHING_VALID_PIXELS', 'UNSUPPORTED_COMPOSITE_PIXELS',
    'SUPPORTED_NODATA_PIXELS', 'SUPPORT_RULE_VALID',
    'VALID']])

# Handle missing or invalid failed HLS composite products explicitly during HLS temporal composites.
if not failed_hls_composite_products.empty:
    print('\n--- FAILED HLS TEMPORAL COMPOSITE PRODUCTS ---')
    display(failed_hls_composite_products.head(5))

# Handle missing or invalid failed HLS composite pairs explicitly during HLS temporal composites.
if not failed_hls_composite_pairs.empty:
    print('\n--- FAILED HLS COMPOSITE SUPPORT-RULE CHECKS ---')
    display(failed_hls_composite_pairs.head(5))

# Stop execution if one or more HLS temporal-composite products failed independent validation.
if not fire_hazard_hls_composite_validation_complete:
    raise ValueError(f'One or more HLS '
        f'temporal-composite products '
        f'failed independent validation.\n\n'
        f'Expected products: '
        f'{expected_hls_composite_product_count:,}\n'
        f'Valid products: '
        f'{valid_hls_composite_product_count:,}\n'
        f'Failed products: '
        f'{len(failed_hls_composite_products):,}\n'
        f'Valid composite/count pairs: '
        f'{valid_hls_composite_pair_count:,} '
        f'of 2\nNDVI/NDMI grids match: '
        f'{fire_hazard_hls_composite_index_grids_match}\n'
        f'Processing manifest complete: '
        f'{fire_hazard_hls_composite_manifest_complete}\n\n'
        f'Review the saved '
        f'temporal-composite validation '
        f'files.')
import gc
gc.collect()
print('\nNOTE:')
print('The NDVI and NDMI temporal '
    'median composites and their '
    'observation-count rasters '
    'passed independent file, grid, '
    'structure, spatial-coverage, '
    'value-range, and support-rule '
    'checks.')
print(f'Each retained composite pixel '
    f'is supported by at least '
    f'{fire_hazard_hls_composite_minimum_observations} '
    f'valid HLS observations.')
print('The next step is to summarize '
    'and transform the validated '
    'NDVI and NDMI composites into '
    'the vegetation-dryness '
    'component used by the '
    'Composite Fire Hazard Grid.')
print('\n=== HLS TEMPORAL COMPOSITE VALIDATION COMPLETE ===')



=== VALIDATING HLS TEMPORAL COMPOSITES ===
-> Expected temporal-composite products: 4
-> Validated products: 4
-> Valid products: 4
-> Failed products: 0
-> Valid composite/count pairs: 2 of 2
-> NDVI and NDMI grids match: True
-> Processing manifest complete: True
-> Total composite output size: 96.49 MB
-> Temporal-composite validation complete: True
-> Validation table saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_validation.csv
-> Validation summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\temporal_composites\metadata\hls_temporal_composite_validation_summary.csv

--- HLS TEMPORAL COMPOSITE VALIDATION SUMMARY ---


,EXPECTED_PRODUCTS,VALIDATED_PRODUCTS,VALID_PRODUCTS,FAILED_PRODUCTS,VALID_INDEX_PAIRS,NDVI_NDMI_GRIDS_MATCH,PROCESSING_MANIFEST_COMPLETE,MINIMUM_VALID_OBSERVATIONS,TARGET_CRS,TARGET_CELL_SIZE,TARGET_WIDTH,TARGET_HEIGHT,TOTAL_OUTPUT_SIZE_BYTES,TOTAL_OUTPUT_SIZE_MB,VALIDATION_COMPLETE
0,4,4,4,0,2,True,True,3,EPSG:26912,30,9454,14467,101176444,96.489376,True



--- HLS TEMPORAL COMPOSITE PRODUCT VALIDATION ---


,INDEX_NAME,OUTPUT_TYPE,FILE_EXISTS,GRID_VALID,STRUCTURE_VALID,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,VALUE_MINIMUM,VALUE_MAXIMUM,VALUE_RANGE_VALID,VALID
0,NDMI,Observation count,True,True,True,14652069,0,1.000000,24.0,True,True
1,NDMI,Temporal median composite,True,True,True,14140492,0,-0.560770,1.0,True,True
2,NDVI,Observation count,True,True,True,14652069,0,1.000000,24.0,True,True
3,NDVI,Temporal median composite,True,True,True,14140492,0,-0.246165,1.0,True,True



--- HLS COMPOSITE SUPPORT-RULE VALIDATION ---


,INDEX_NAME,GRIDS_MATCH,MATCHING_VALID_PIXELS,UNSUPPORTED_COMPOSITE_PIXELS,SUPPORTED_NODATA_PIXELS,SUPPORT_RULE_VALID,VALID
0,NDVI,True,14140492,0,0,True,True
1,NDMI,True,14140492,0,0,True,True



NOTE:
The NDVI and NDMI temporal median composites and their observation-count rasters passed independent file, grid, structure, spatial-coverage, value-range, and support-rule checks.
Each retained composite pixel is supported by at least 3 valid HLS observations.
The next step is to summarize and transform the validated NDVI and NDMI composites into the vegetation-dryness component used by the Composite Fire Hazard Grid.

=== HLS TEMPORAL COMPOSITE VALIDATION COMPLETE ===


### Creating Vegetation-Dryness Component Summary


In [89]:
print('=== CREATING VEGETATION-DRYNESS COMPONENT SUMMARY ===')
# List the required vegetation dryness summary prerequisites required before
# vegetation-dryness component summary can run.
required_vegetation_dryness_summary_inputs = ['fire_hazard_hls_ndvi_composite_path',
    'fire_hazard_hls_ndmi_composite_path', 'fire_hazard_hls_ndvi_observation_count_path',
    'fire_hazard_hls_ndmi_observation_count_path',
    'fire_hazard_hls_composite_manifest', 'fire_hazard_hls_composite_validation',
    'fire_hazard_hls_composite_validation_summary',
    'fire_hazard_hls_composite_validation_complete',
    'fire_hazard_hls_composite_minimum_observations',
    'fire_hazard_hls_analysis_period', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_nodata_value',
    'fire_hazard_hls_raw_directory']
# Identify unavailable vegetation dryness summary so vegetation-dryness component summary
# stops before using incomplete inputs.
missing_vegetation_dryness_summary_inputs = [object_name for object_name in \
    required_vegetation_dryness_summary_inputs if object_name not in globals()]

# Stop execution when required vegetation dryness summary inputs inputs are unavailable.
if missing_vegetation_dryness_summary_inputs:
    raise NameError(f'The following '
        f'vegetation-dryness component '
        f'objects are missing:\n'
        f'{missing_vegetation_dryness_summary_inputs}\n\n'
        f'Run the HLS temporal-composite '
        f'processing and validation '
        f'steps before creating the '
        f'component summary.')

# Stop execution if hLS temporal-composite products have not passed complete independent validation.
if not fire_hazard_hls_composite_validation_complete:
    raise ValueError('The HLS temporal-composite '
        'products have not passed '
        'complete independent '
        'validation.')

# Stop execution if hLS temporal-composite validation table contains no product records.
if fire_hazard_hls_composite_validation.empty:
    raise ValueError('The HLS temporal-composite validation table contains no product records.')

# Stop execution if one or more HLS temporal-composite products remain marked invalid.
if not fire_hazard_hls_composite_validation['VALID'].fillna(False).all():
    raise ValueError('One or more HLS temporal-composite products remain marked invalid.')
# Build the vegetation dryness product paths location so vegetation-dryness component summary uses
# the expected project file structure.
fire_hazard_vegetation_dryness_product_paths = {'NDVI_COMPOSITE': \
    Path(fire_hazard_hls_ndvi_composite_path),
    'NDMI_COMPOSITE': Path(fire_hazard_hls_ndmi_composite_path),
    'NDVI_OBSERVATION_COUNT': Path(fire_hazard_hls_ndvi_observation_count_path),
    'NDMI_OBSERVATION_COUNT': Path(fire_hazard_hls_ndmi_observation_count_path)}
# Identify unavailable vegetation dryness products so vegetation-dryness component summary stops
# before using incomplete inputs.
missing_vegetation_dryness_products = [product_name for product_name,
    product_path in fire_hazard_vegetation_dryness_product_paths.items() if not \
        product_path.exists() or not product_path.is_file() or product_path.stat().st_size <= 0]

# Stop execution when required vegetation dryness products inputs are unavailable.
if missing_vegetation_dryness_products:
    raise FileNotFoundError(f'The following validated '
        f'vegetation-dryness products '
        f'are unavailable:\n'
        f'{missing_vegetation_dryness_products}')
# Build the vegetation dryness component directory location so vegetation-dryness component summary
# uses the expected project file structure.
fire_hazard_vegetation_dryness_component_directory = fire_hazard_hls_raw_directory / \
    'vegetation_dryness_component'
# Create the output directory before writing workflow products.
fire_hazard_vegetation_dryness_component_directory.mkdir(parents=True, exist_ok=True)
# Build the vegetation dryness summary path location so vegetation-dryness component summary uses
# the expected project file structure.
fire_hazard_vegetation_dryness_summary_path = fire_hazard_vegetation_dryness_component_directory \
    / 'vegetation_dryness_component_summary.csv'
# Build the vegetation dryness policy path location so vegetation-dryness component summary uses the
# expected project file structure.
fire_hazard_vegetation_dryness_policy_path = fire_hazard_vegetation_dryness_component_directory / \
    'vegetation_dryness_component_policy.json'
# Build the vegetation dryness product inventory path location so vegetation-dryness component
# summary uses the expected project file structure.
fire_hazard_vegetation_dryness_product_inventory_path = \
    fire_hazard_vegetation_dryness_component_directory / 'vegetation_dryness_product_inventory.csv'
# Record NDVI composite validation record so vegetation-dryness component summary can preserve a
# clear processing and validation outcome.
ndvi_composite_validation_record = \
    fire_hazard_hls_composite_validation[(fire_hazard_hls_composite_validation['INDEX_NAME'] == \
    'NDVI') & (fire_hazard_hls_composite_validation['OUTPUT_TYPE'] == 'Temporal median composite')]
# Record NDMI composite validation record so vegetation-dryness component summary can preserve a
# clear processing and validation outcome.
ndmi_composite_validation_record = \
    fire_hazard_hls_composite_validation[(fire_hazard_hls_composite_validation['INDEX_NAME'] == \
    'NDMI') & (fire_hazard_hls_composite_validation['OUTPUT_TYPE'] == 'Temporal median composite')]

# Stop execution if exactly one validated NDVI temporal-composite record is required.
if len(ndvi_composite_validation_record) != 1:
    raise ValueError('Exactly one validated NDVI temporal-composite record is required.')

# Stop execution if exactly one validated NDMI temporal-composite record is required.
if len(ndmi_composite_validation_record) != 1:
    raise ValueError('Exactly one validated NDMI temporal-composite record is required.')
# Track NDVI composite minimum across processed pixels for output-range QA.
ndvi_composite_minimum = float(ndvi_composite_validation_record.iloc[0]['VALUE_MINIMUM'])
# Track NDVI composite maximum across processed pixels for output-range QA.
ndvi_composite_maximum = float(ndvi_composite_validation_record.iloc[0]['VALUE_MAXIMUM'])
# Calculate NDVI valid pixel count to quantify completeness and support QA checks.
ndvi_valid_pixel_count = int(ndvi_composite_validation_record.iloc[0]['VALID_PIXELS'])
# Track NDMI composite minimum across processed pixels for output-range QA.
ndmi_composite_minimum = float(ndmi_composite_validation_record.iloc[0]['VALUE_MINIMUM'])
# Track NDMI composite maximum across processed pixels for output-range QA.
ndmi_composite_maximum = float(ndmi_composite_validation_record.iloc[0]['VALUE_MAXIMUM'])
# Calculate NDMI valid pixel count to quantify completeness and support QA checks.
ndmi_valid_pixel_count = int(ndmi_composite_validation_record.iloc[0]['VALID_PIXELS'])
# Store vegetation dryness total size bytes needed to carry out vegetation-dryness component
# summary.
fire_hazard_vegetation_dryness_total_size_bytes = sum((product_path.stat().st_size for \
    product_path in fire_hazard_vegetation_dryness_product_paths.values()))
# Store vegetation dryness total size mb needed to carry out vegetation-dryness component summary.
fire_hazard_vegetation_dryness_total_size_mb = fire_hazard_vegetation_dryness_total_size_bytes / \
    1024 ** 2
# Define vegetation dryness interpretation to control the inputs and rules used by
# vegetation-dryness component summary.
fire_hazard_vegetation_dryness_interpretation = (
    {'NDVI': 'Lower NDVI values indicate '
        'reduced vegetation greenness '
        'or vigor and will produce '
        'higher vegetation-dryness '
        'hazard scores.', 'NDMI': 'Lower NDMI values indicate reduced ' \
            'vegetation or canopy moisture and will ' \
            'produce higher vegetation-dryness ' \
            'hazard scores.'}
)
# Define vegetation dryness normalization direction to control the inputs and rules used by
# vegetation-dryness component summary.
fire_hazard_vegetation_dryness_normalization_direction = {'NDVI': 'Inverse', 'NDMI': 'Inverse'}
# Store vegetation dryness product inventory needed to carry out vegetation-dryness component
# summary.
fire_hazard_vegetation_dryness_product_inventory = pd.DataFrame([{'PRODUCT_ID': \
    'HLS_NDVI_COMPOSITE',
    'PRODUCT_TYPE': 'Temporal median composite', 'INDEX_NAME': 'NDVI',
    'LOCAL_PATH': str(fire_hazard_hls_ndvi_composite_path),
    'DATA_TYPE': 'float32', 'NODATA_VALUE': fire_hazard_nodata_value,
    'VALUE_MINIMUM': ndvi_composite_minimum, 'VALUE_MAXIMUM': ndvi_composite_maximum,
    'VALID_PIXELS': ndvi_valid_pixel_count, 'FILE_SIZE_BYTES': \
        fire_hazard_hls_ndvi_composite_path.stat().st_size,
    'VALID': True}, {'PRODUCT_ID': 'HLS_NDMI_COMPOSITE',
    'PRODUCT_TYPE': 'Temporal median composite', 'INDEX_NAME': 'NDMI',
    'LOCAL_PATH': str(fire_hazard_hls_ndmi_composite_path),
    'DATA_TYPE': 'float32', 'NODATA_VALUE': fire_hazard_nodata_value,
    'VALUE_MINIMUM': ndmi_composite_minimum, 'VALUE_MAXIMUM': ndmi_composite_maximum,
    'VALID_PIXELS': ndmi_valid_pixel_count, 'FILE_SIZE_BYTES': \
        fire_hazard_hls_ndmi_composite_path.stat().st_size,
    'VALID': True}, {'PRODUCT_ID': 'HLS_NDVI_OBSERVATION_COUNT',
    'PRODUCT_TYPE': 'Observation count', 'INDEX_NAME': 'NDVI',
    'LOCAL_PATH': str(fire_hazard_hls_ndvi_observation_count_path),
    'DATA_TYPE': 'uint16', 'NODATA_VALUE': 0, 'VALUE_MINIMUM': None,
    'VALUE_MAXIMUM': None, 'VALID_PIXELS': None, 'FILE_SIZE_BYTES': \
        fire_hazard_hls_ndvi_observation_count_path.stat().st_size,
    'VALID': True}, {'PRODUCT_ID': 'HLS_NDMI_OBSERVATION_COUNT',
    'PRODUCT_TYPE': 'Observation count', 'INDEX_NAME': 'NDMI',
    'LOCAL_PATH': str(fire_hazard_hls_ndmi_observation_count_path),
    'DATA_TYPE': 'uint16', 'NODATA_VALUE': 0, 'VALUE_MINIMUM': None,
    'VALUE_MAXIMUM': None, 'VALID_PIXELS': None, 'FILE_SIZE_BYTES': \
        fire_hazard_hls_ndmi_observation_count_path.stat().st_size,
    'VALID': True}])
# Store vegetation dryness summary needed to carry out vegetation-dryness component summary.
fire_hazard_vegetation_dryness_summary = pd.DataFrame([{'COMPONENT_ID': 'VEGETATION_DRYNESS',
    'COMPONENT_NAME': 'Vegetation Dryness', 'DATA_SOURCE': \
        'NASA Harmonized Landsat Sentinel-2 Version 2.0',
    'ACCESS_PLATFORM': 'Microsoft Planetary Computer',
    'ANALYSIS_PERIOD': fire_hazard_hls_analysis_period,
    'SOURCE_INDICES': 'NDVI, NDMI', 'TEMPORAL_COMPOSITE_METHOD': 'Pixel-wise median',
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_hls_composite_minimum_observations,
    'NDVI_MINIMUM': ndvi_composite_minimum, 'NDVI_MAXIMUM': ndvi_composite_maximum,
    'NDMI_MINIMUM': ndmi_composite_minimum, 'NDMI_MAXIMUM': ndmi_composite_maximum,
    'NDVI_VALID_PIXELS': ndvi_valid_pixel_count, 'NDMI_VALID_PIXELS': ndmi_valid_pixel_count,
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'FINAL_PRODUCT_COUNT': len(fire_hazard_vegetation_dryness_product_inventory),
    'TOTAL_PRODUCT_SIZE_MB': fire_hazard_vegetation_dryness_total_size_mb,
    'NORMALIZATION_DIRECTION': 'Inverse for NDVI and NDMI',
    'VALIDATION_COMPLETE': fire_hazard_hls_composite_validation_complete,
    'READY_FOR_HAZARD_INTEGRATION': True}])

# Decode HLS quality flags to remove clouds, cloud shadows,
# cirrus, snow, and other invalid observations.
# Define vegetation dryness policy to control the inputs and rules used by vegetation-dryness
# component summary.
fire_hazard_vegetation_dryness_policy = (
    {'component_id': 'VEGETATION_DRYNESS', 'component_name': 'Vegetation Dryness',
        'component_purpose': 'Represent seasonal vegetation '
        'greenness and moisture '
        'conditions for integration '
        'into the Composite Fire Hazard '
        'Grid.', 'data_source': {'dataset': 'NASA Harmonized Landsat Sentinel-2 Version 2.0',
            'access_platform': 'Microsoft Planetary Computer',
            'analysis_period': fire_hazard_hls_analysis_period}, 'input_products': \
                {'NDVI_composite': str(fire_hazard_hls_ndvi_composite_path),
            'NDMI_composite': str(fire_hazard_hls_ndmi_composite_path),
            'NDVI_observation_count': str(fire_hazard_hls_ndvi_observation_count_path),
            'NDMI_observation_count': str(fire_hazard_hls_ndmi_observation_count_path)}, \
                'completed_processing_steps': ['Downloaded required HLS raster assets',
            'Validated local HLS source files', 'Decoded Fmask quality information',
            'Created valid-pixel masks', 'Scaled surface-reflectance bands',
            'Calculated scene-level NDVI and NDMI', 'Reprojected and aligned scene-level indices',
            'Excluded non-contributing source scenes', \
                'Created pixel-wise temporal median composites',
            'Created observation-count rasters', 'Independently validated final products'], \
                'temporal_composite': {'method': 'median',
            'minimum_valid_observations': int(fire_hazard_hls_composite_minimum_observations),
            'ndvi_value_range': [ndvi_composite_minimum, ndvi_composite_maximum],
            'ndmi_value_range': [ndmi_composite_minimum, ndmi_composite_maximum]}, 'target_grid': \
                {'crs': fire_hazard_target_crs,
            'cell_size_meters': float(fire_hazard_cell_size),
            'nodata_value': float(fire_hazard_nodata_value)}, 'hazard_interpretation': \
                fire_hazard_vegetation_dryness_interpretation, 'normalization_direction': \
                fire_hazard_vegetation_dryness_normalization_direction, 'future_processing': \
                {'normalize_ndvi': True,
            'normalize_ndmi': True, 'combine_indices': True,
            'integration_target': 'Composite Fire Hazard Grid'}, 'validation': {'complete': \
                bool(fire_hazard_hls_composite_validation_complete),
            'validated_product_count': int(len(fire_hazard_hls_composite_validation)),
            'ready_for_hazard_integration': True}}
)
# Save the vegetation dryness product inventory CSV so later steps can reuse the recorded workflow
# results.
fire_hazard_vegetation_dryness_product_inventory.to_csv \
    (fire_hazard_vegetation_dryness_product_inventory_path,
    index=False)
# Save the vegetation dryness summary CSV so later steps can reuse the recorded workflow results.
fire_hazard_vegetation_dryness_summary.to_csv(fire_hazard_vegetation_dryness_summary_path,
    index=False)

# Open the file in a managed context so its handle closes reliably after vegetation-dryness
# component summary.
with fire_hazard_vegetation_dryness_policy_path.open('w',
    encoding='utf-8') as vegetation_dryness_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_vegetation_dryness_policy, vegetation_dryness_policy_file, indent=2)
# Store vegetation dryness component complete needed to carry out vegetation-dryness component
# summary.
fire_hazard_vegetation_dryness_component_complete = fire_hazard_hls_composite_validation_complete \
    and len(fire_hazard_vegetation_dryness_product_inventory) == 4 and \
    fire_hazard_vegetation_dryness_product_inventory['VALID'].all()

# Stop execution if vegetation-dryness component did not pass final completion checks.
if not fire_hazard_vegetation_dryness_component_complete:
    raise ValueError('The vegetation-dryness component did not pass final completion checks.')
print('-> Component: Vegetation Dryness')
print('-> Source indices: NDVI and NDMI')
print('-> Temporal composite method: Pixel-wise median')
print(f'-> Minimum valid observations: {fire_hazard_hls_composite_minimum_observations}')
print(f'-> NDVI composite range: {ndvi_composite_minimum:.4f} to {ndvi_composite_maximum:.4f}')
print(f'-> NDMI composite range: {ndmi_composite_minimum:.4f} to {ndmi_composite_maximum:.4f}')
print(f'-> Final product count: {len(fire_hazard_vegetation_dryness_product_inventory)}')
print(f'-> Total product size: {fire_hazard_vegetation_dryness_total_size_mb:,.2f} MB')
print(f'-> Component complete: {fire_hazard_vegetation_dryness_component_complete}')
print(f'-> Product inventory saved: {fire_hazard_vegetation_dryness_product_inventory_path}')
print(f'-> Component summary saved: {fire_hazard_vegetation_dryness_summary_path}')
print(f'-> Processing policy saved: {fire_hazard_vegetation_dryness_policy_path}')
print('\n--- VEGETATION-DRYNESS COMPONENT SUMMARY ---')
display(fire_hazard_vegetation_dryness_summary)
print('\n--- VEGETATION-DRYNESS PRODUCT INVENTORY ---')
display(fire_hazard_vegetation_dryness_product_inventory)
print('\nNOTE:')
print('The vegetation-dryness '
    'component is fully processed, '
    'independently validated, '
    'documented, and ready for '
    'integration into the Composite '
    'Fire Hazard Grid.')
print('The NDVI and NDMI temporal '
    'composites remain continuous '
    'physical-index products at '
    'this stage. They have not yet '
    'been converted into '
    'standardized fire-hazard '
    'scores.')
print('During the next processing '
    'phase, both indices will be '
    'inversely normalized so that '
    'lower greenness and moisture '
    'values produce higher '
    'vegetation-dryness hazard '
    'scores.')
print('\n=== VEGETATION-DRYNESS COMPONENT SUMMARY COMPLETE ===')



=== CREATING VEGETATION-DRYNESS COMPONENT SUMMARY ===
-> Component: Vegetation Dryness
-> Source indices: NDVI and NDMI
-> Temporal composite method: Pixel-wise median
-> Minimum valid observations: 3
-> NDVI composite range: -0.2462 to 1.0000
-> NDMI composite range: -0.5608 to 1.0000
-> Final product count: 4
-> Total product size: 96.49 MB
-> Component complete: True
-> Product inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\vegetation_dryness_product_inventory.csv
-> Component summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\vegetation_dryness_component_summary.csv
-> Processing policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\vegetation_dryness_component_policy.json

--- VEGETATION-DRYNESS COMPONENT

,COMPONENT_ID,COMPONENT_NAME,DATA_SOURCE,ACCESS_PLATFORM,ANALYSIS_PERIOD,SOURCE_INDICES,TEMPORAL_COMPOSITE_METHOD,MINIMUM_VALID_OBSERVATIONS,NDVI_MINIMUM,NDVI_MAXIMUM,...,NDMI_MAXIMUM,NDVI_VALID_PIXELS,NDMI_VALID_PIXELS,TARGET_CRS,CELL_SIZE_METERS,FINAL_PRODUCT_COUNT,TOTAL_PRODUCT_SIZE_MB,NORMALIZATION_DIRECTION,VALIDATION_COMPLETE,READY_FOR_HAZARD_INTEGRATION
0,VEGETATION_DRYNESS,Vegetation Dryness,NASA Harmonized Landsat Sentinel-2 Version 2.0,Microsoft Planetary Computer,July-September 2025,"NDVI, NDMI",Pixel-wise median,3,-0.246165,1.0,...,1.0,14140492,14140492,EPSG:26912,30,4,96.489376,Inverse for NDVI and NDMI,True,True



--- VEGETATION-DRYNESS PRODUCT INVENTORY ---


,PRODUCT_ID,PRODUCT_TYPE,INDEX_NAME,LOCAL_PATH,DATA_TYPE,NODATA_VALUE,VALUE_MINIMUM,VALUE_MAXIMUM,VALID_PIXELS,FILE_SIZE_BYTES,VALID
0,HLS_NDVI_COMPOSITE,Temporal median composite,NDVI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,float32,-9999.0,-0.246165,1.0,14140492.0,47258426,True
1,HLS_NDMI_COMPOSITE,Temporal median composite,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,float32,-9999.0,-0.560770,1.0,14140492.0,50371506,True
2,HLS_NDVI_OBSERVATION_COUNT,Observation count,NDVI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,uint16,0.0,NaN,NaN,NaN,1773259,True
3,HLS_NDMI_OBSERVATION_COUNT,Observation count,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,uint16,0.0,NaN,NaN,NaN,1773253,True



NOTE:
The vegetation-dryness component is fully processed, independently validated, documented, and ready for integration into the Composite Fire Hazard Grid.
The NDVI and NDMI temporal composites remain continuous physical-index products at this stage. They have not yet been converted into standardized fire-hazard scores.
During the next processing phase, both indices will be inversely normalized so that lower greenness and moisture values produce higher vegetation-dryness hazard scores.

=== VEGETATION-DRYNESS COMPONENT SUMMARY COMPLETE ===


### Configuring Vegetation-Dryness Normalization


In [90]:
print('=== CONFIGURING VEGETATION-DRYNESS NORMALIZATION ===')
# List the required vegetation dryness normalization prerequisites required before
# vegetation-dryness normalization can run.
required_vegetation_dryness_normalization_inputs = \
    ['fire_hazard_vegetation_dryness_component_complete',
    'fire_hazard_hls_ndvi_composite_path', 'fire_hazard_hls_ndmi_composite_path',
    'fire_hazard_hls_ndvi_observation_count_path',
    'fire_hazard_hls_ndmi_observation_count_path',
    'fire_hazard_hls_composite_minimum_observations',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_profile',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_nodata_value',
    'fire_hazard_raster_dtype', 'fire_hazard_hls_raw_directory']
# Identify unavailable vegetation dryness normalization so vegetation-dryness
# normalization stops before using incomplete inputs.
missing_vegetation_dryness_normalization_inputs = [object_name for object_name in \
    required_vegetation_dryness_normalization_inputs if object_name not in globals()]

# Stop execution when required vegetation dryness normalization inputs inputs are unavailable.
if missing_vegetation_dryness_normalization_inputs:
    raise NameError(f'The following '
        f'vegetation-dryness '
        f'normalization objects are '
        f'missing:\n'
        f'{missing_vegetation_dryness_normalization_inputs}\n\n'
        f'Run Create Vegetation-Dryness '
        f'Component Summary and '
        f'Processing Policy before '
        f'configuring normalization.')

# Stop execution if vegetation-dryness component has not passed final completion checks.
if not fire_hazard_vegetation_dryness_component_complete:
    raise ValueError('The vegetation-dryness component has not passed final completion checks.')
# Define vegetation dryness normalization inputs to control the inputs and rules used by
# vegetation-dryness normalization.
fire_hazard_vegetation_dryness_normalization_inputs = {'NDVI': \
    Path(fire_hazard_hls_ndvi_composite_path),
    'NDMI': Path(fire_hazard_hls_ndmi_composite_path),
    'NDVI_COUNT': Path(fire_hazard_hls_ndvi_observation_count_path),
    'NDMI_COUNT': Path(fire_hazard_hls_ndmi_observation_count_path)}
# Identify unavailable vegetation dryness normalization files so vegetation-dryness
# normalization stops before using incomplete inputs.
missing_vegetation_dryness_normalization_files = [input_name for input_name,
    input_path in fire_hazard_vegetation_dryness_normalization_inputs.items() if not \
        input_path.exists() or not input_path.is_file() or input_path.stat().st_size <= 0]

# Stop execution when required vegetation dryness normalization files inputs are unavailable.
if missing_vegetation_dryness_normalization_files:
    raise FileNotFoundError(f'The following '
        f'vegetation-dryness '
        f'normalization inputs are '
        f'unavailable:\n'
        f'{missing_vegetation_dryness_normalization_files}')
# Track vegetation dryness score minimum across processed pixels for output-range QA.
fire_hazard_vegetation_dryness_score_minimum = 0.0
# Track vegetation dryness score maximum across processed pixels for output-range QA.
fire_hazard_vegetation_dryness_score_maximum = 1.0
# Define vegetation dryness normalization direction to control the inputs and rules used by
# vegetation-dryness normalization.
fire_hazard_vegetation_dryness_normalization_direction = {'NDVI': 'inverse', 'NDMI': 'inverse'}
# Set vegetation dryness normalization method to control vegetation-dryness normalization.
fire_hazard_vegetation_dryness_normalization_method = 'inverse_min_max'
# Define vegetation dryness normalization bounds to control the inputs and rules used by
# vegetation-dryness normalization.
fire_hazard_vegetation_dryness_normalization_bounds = {'NDVI': {'lower_bound': 0.1,
    'upper_bound': 0.8}, 'NDMI': {'lower_bound': -0.2,
    'upper_bound': 0.5}}
# Set vegetation dryness clip to bounds to control vegetation-dryness normalization.
fire_hazard_vegetation_dryness_clip_to_bounds = True
# Record invalid vegetation dryness bounds so vegetation-dryness normalization can preserve a clear
# processing and validation outcome.
invalid_vegetation_dryness_bounds = [index_name for index_name,
    bound_settings in fire_hazard_vegetation_dryness_normalization_bounds.items() if \
        bound_settings['lower_bound'] >= bound_settings['upper_bound']]

# Stop execution if following vegetation-index normalization bounds are invalid: the reported value.
if invalid_vegetation_dryness_bounds:
    raise ValueError(f'The following vegetation-index '
        f'normalization bounds are '
        f'invalid:\n'
        f'{invalid_vegetation_dryness_bounds}')
# Define fire hazard vegetation dryness index weights used to combine component scores in the hazard
# calculation.
fire_hazard_vegetation_dryness_index_weights = {'NDVI': 0.4, 'NDMI': 0.6}
# Store vegetation dryness weight total needed to carry out vegetation-dryness normalization.
fire_hazard_vegetation_dryness_weight_total = \
    sum(fire_hazard_vegetation_dryness_index_weights.values())

# Require The NDVI and NDMI vegetation-dryness weights to sum to 1.0.
if not np.isclose(fire_hazard_vegetation_dryness_weight_total, 1.0):
    raise ValueError(f'The NDVI and NDMI '
        f'vegetation-dryness weights '
        f'must sum to 1.0.\nCurrent '
        f'total: '
        f'{fire_hazard_vegetation_dryness_weight_total}')
# Track vegetation dryness minimum observations across processed pixels for output-range QA.
fire_hazard_vegetation_dryness_minimum_observations = fire_hazard_hls_composite_minimum_observations
# Set vegetation dryness require both indices to control vegetation-dryness normalization.
fire_hazard_vegetation_dryness_require_both_indices = True
# Calculate vegetation dryness use observation counts to quantify completeness and support QA
# checks.
fire_hazard_vegetation_dryness_use_observation_counts = True
# Set vegetation dryness output dtype to control vegetation-dryness normalization.
fire_hazard_vegetation_dryness_output_dtype = 'float32'
# Store vegetation dryness output NoData needed to carry out vegetation-dryness normalization.
fire_hazard_vegetation_dryness_output_nodata = fire_hazard_nodata_value
# Set vegetation dryness output compression to control vegetation-dryness normalization.
fire_hazard_vegetation_dryness_output_compression = 'deflate'
# Set vegetation dryness process by window to control vegetation-dryness normalization.
fire_hazard_vegetation_dryness_process_by_window = True
# Extract vegetation dryness window size from the current record for vegetation-dryness
# normalization.
fire_hazard_vegetation_dryness_window_size = globals().get('fire_hazard_hls_composite_window_size',
    512)
# Set vegetation dryness progress interval to control vegetation-dryness normalization.
fire_hazard_vegetation_dryness_progress_interval = 25
# Build the vegetation dryness normalized directory location so vegetation-dryness normalization
# uses the expected project file structure.
fire_hazard_vegetation_dryness_normalized_directory = fire_hazard_hls_raw_directory / \
    'vegetation_dryness_component' / 'normalized_rasters'
# Build the vegetation dryness hazard directory location so vegetation-dryness normalization uses
# the expected project file structure.
fire_hazard_vegetation_dryness_hazard_directory = fire_hazard_hls_raw_directory / \
    'vegetation_dryness_component' / 'hazard_raster'
# Build the vegetation dryness metadata directory location so vegetation-dryness normalization uses
# the expected project file structure.
fire_hazard_vegetation_dryness_metadata_directory = fire_hazard_hls_raw_directory / \
    'vegetation_dryness_component' / 'metadata'

# Process each directory path entry so vegetation-dryness normalization is applied consistently
# across all records.
for directory_path in [fire_hazard_vegetation_dryness_normalized_directory,
    fire_hazard_vegetation_dryness_hazard_directory,
    fire_hazard_vegetation_dryness_metadata_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build the vegetation dryness NDVI score path location so vegetation-dryness normalization uses the
# expected project file structure.
fire_hazard_vegetation_dryness_ndvi_score_path = \
    fire_hazard_vegetation_dryness_normalized_directory / \
    'ndvi_inverse_normalized_dryness_score.tif'
# Build the vegetation dryness NDMI score path location so vegetation-dryness normalization uses the
# expected project file structure.
fire_hazard_vegetation_dryness_ndmi_score_path = \
    fire_hazard_vegetation_dryness_normalized_directory / \
    'ndmi_inverse_normalized_dryness_score.tif'
# Build the vegetation dryness combined score path location so vegetation-dryness normalization uses
# the expected project file structure.
fire_hazard_vegetation_dryness_combined_score_path = \
    fire_hazard_vegetation_dryness_hazard_directory / 'vegetation_dryness_hazard_score.tif'
# Build the vegetation dryness normalization policy path location so vegetation-dryness
# normalization uses the expected project file structure.
fire_hazard_vegetation_dryness_normalization_policy_path = \
    fire_hazard_vegetation_dryness_metadata_directory / \
    'vegetation_dryness_normalization_policy.json'
# Build the vegetation dryness normalization summary path location so vegetation-dryness
# normalization uses the expected project file structure.
fire_hazard_vegetation_dryness_normalization_summary_path = \
    fire_hazard_vegetation_dryness_metadata_directory / \
    'vegetation_dryness_normalization_summary.csv'
# Build the vegetation dryness normalization manifest path location so vegetation-dryness
# normalization uses the expected project file structure.
fire_hazard_vegetation_dryness_normalization_manifest_path = \
    fire_hazard_vegetation_dryness_metadata_directory / \
    'vegetation_dryness_normalization_manifest.csv'
# Build the vegetation dryness validation path location so vegetation-dryness normalization uses the
# expected project file structure.
fire_hazard_vegetation_dryness_validation_path = \
    fire_hazard_vegetation_dryness_metadata_directory / 'vegetation_dryness_validation.csv'
# Build the vegetation dryness validation summary path location so vegetation-dryness normalization
# uses the expected project file structure.
fire_hazard_vegetation_dryness_validation_summary_path = \
    fire_hazard_vegetation_dryness_metadata_directory / 'vegetation_dryness_validation_summary.csv'
# Build the vegetation dryness input paths location so vegetation-dryness normalization uses the
# expected project file structure.
fire_hazard_vegetation_dryness_input_paths = {'NDVI': Path(fire_hazard_hls_ndvi_composite_path),
    'NDMI': Path(fire_hazard_hls_ndmi_composite_path)}
# Build the vegetation dryness count paths location so vegetation-dryness normalization uses the
# expected project file structure.
fire_hazard_vegetation_dryness_count_paths = {'NDVI': \
    Path(fire_hazard_hls_ndvi_observation_count_path),
    'NDMI': Path(fire_hazard_hls_ndmi_observation_count_path)}
# Build the vegetation dryness output paths location so vegetation-dryness normalization uses the
# expected project file structure.
fire_hazard_vegetation_dryness_output_paths = {'NDVI': \
    fire_hazard_vegetation_dryness_ndvi_score_path,
    'NDMI': fire_hazard_vegetation_dryness_ndmi_score_path,
    'COMBINED': fire_hazard_vegetation_dryness_combined_score_path}
# Prepare vegetation dryness output profile to preserve and validate raster structure during
# vegetation-dryness normalization.
fire_hazard_vegetation_dryness_output_profile = fire_hazard_alignment_profile.copy()
fire_hazard_vegetation_dryness_output_profile.update({'driver': 'GTiff',
    'dtype': fire_hazard_vegetation_dryness_output_dtype,
    'count': 1, 'nodata': fire_hazard_vegetation_dryness_output_nodata,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': fire_hazard_vegetation_dryness_output_compression,
    'predictor': 3, 'tiled': True, 'BIGTIFF': 'IF_SAFER'})
# Store vegetation dryness reuse existing needed to carry out vegetation-dryness normalization.
fire_hazard_vegetation_dryness_reuse_existing = globals().get('fire_hazard_data_mode',
    'snapshot') == 'snapshot'
# Store vegetation dryness overwrite needed to carry out vegetation-dryness normalization.
fire_hazard_vegetation_dryness_overwrite = globals().get('fire_hazard_data_mode',
    'snapshot') == 'refresh'
# Define vegetation dryness normalization policy to control the inputs and rules used by
# vegetation-dryness normalization.
fire_hazard_vegetation_dryness_normalization_policy = {'component_id': 'VEGETATION_DRYNESS',
    'component_name': 'Vegetation Dryness', 'normalization_method': \
        fire_hazard_vegetation_dryness_normalization_method,
    'score_range': [fire_hazard_vegetation_dryness_score_minimum,
    fire_hazard_vegetation_dryness_score_maximum],
    'normalization_direction': fire_hazard_vegetation_dryness_normalization_direction,
    'normalization_bounds': fire_hazard_vegetation_dryness_normalization_bounds,
    'clip_to_bounds': fire_hazard_vegetation_dryness_clip_to_bounds,
    'weights': fire_hazard_vegetation_dryness_index_weights,
    'minimum_valid_observations': int(fire_hazard_vegetation_dryness_minimum_observations),
    'require_both_indices': fire_hazard_vegetation_dryness_require_both_indices,
    'use_observation_counts': fire_hazard_vegetation_dryness_use_observation_counts,
    'input_paths': {input_name: str(input_path) for input_name,
    input_path in fire_hazard_vegetation_dryness_input_paths.items()},
    'observation_count_paths': {index_name: str(count_path) for index_name,
    count_path in fire_hazard_vegetation_dryness_count_paths.items()},
    'output_paths': {output_name: str(output_path) for output_name,
    output_path in fire_hazard_vegetation_dryness_output_paths.items()},
    'target_grid': {'crs': fire_hazard_target_crs,
    'cell_size_meters': float(fire_hazard_cell_size),
    'width': int(fire_hazard_alignment_width), 'height': int(fire_hazard_alignment_height),
    'nodata_value': float(fire_hazard_vegetation_dryness_output_nodata)},
    'processing': {'process_by_window': fire_hazard_vegetation_dryness_process_by_window,
    'window_size': int(fire_hazard_vegetation_dryness_window_size),
    'progress_interval': int(fire_hazard_vegetation_dryness_progress_interval),
    'reuse_existing': fire_hazard_vegetation_dryness_reuse_existing,
    'overwrite_existing': fire_hazard_vegetation_dryness_overwrite}}

# Open the file in a managed context so its handle closes reliably after vegetation-dryness
# normalization.
with fire_hazard_vegetation_dryness_normalization_policy_path.open('w',
    encoding='utf-8') as normalization_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_vegetation_dryness_normalization_policy,
        normalization_policy_file, indent=2)
# Store vegetation dryness normalization summary needed to carry out vegetation-dryness
# normalization.
fire_hazard_vegetation_dryness_normalization_summary = pd.DataFrame([{'COMPONENT_ID': \
    'VEGETATION_DRYNESS',
    'NORMALIZATION_METHOD': fire_hazard_vegetation_dryness_normalization_method,
    'NDVI_LOWER_BOUND': fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['lower_bound'],
    'NDVI_UPPER_BOUND': fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['upper_bound'],
    'NDMI_LOWER_BOUND': fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['lower_bound'],
    'NDMI_UPPER_BOUND': fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['upper_bound'],
    'NDVI_WEIGHT': fire_hazard_vegetation_dryness_index_weights['NDVI'],
    'NDMI_WEIGHT': fire_hazard_vegetation_dryness_index_weights['NDMI'],
    'WEIGHT_TOTAL': fire_hazard_vegetation_dryness_weight_total,
    'MINIMUM_VALID_OBSERVATIONS': fire_hazard_vegetation_dryness_minimum_observations,
    'REQUIRE_BOTH_INDICES': fire_hazard_vegetation_dryness_require_both_indices,
    'OUTPUT_SCORE_MINIMUM': fire_hazard_vegetation_dryness_score_minimum,
    'OUTPUT_SCORE_MAXIMUM': fire_hazard_vegetation_dryness_score_maximum,
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'WINDOW_SIZE': fire_hazard_vegetation_dryness_window_size,
    'REUSE_EXISTING': fire_hazard_vegetation_dryness_reuse_existing,
    'OVERWRITE_EXISTING': fire_hazard_vegetation_dryness_overwrite}])
# Save the vegetation dryness normalization summary CSV so later steps can reuse the recorded
# workflow results.
fire_hazard_vegetation_dryness_normalization_summary.to_csv \
    (fire_hazard_vegetation_dryness_normalization_summary_path,
    index=False)
# Store vegetation dryness normalization configured needed to carry out vegetation-dryness
# normalization.
fire_hazard_vegetation_dryness_normalization_configured = \
    fire_hazard_vegetation_dryness_component_complete and \
    np.isclose(fire_hazard_vegetation_dryness_weight_total,
    1.0) and all((output_path.parent.exists() for output_path in \
        fire_hazard_vegetation_dryness_output_paths.values()))

# Stop execution if vegetation-dryness normalization configuration did not pass final checks.
if not fire_hazard_vegetation_dryness_normalization_configured:
    raise ValueError('The vegetation-dryness '
        'normalization configuration '
        'did not pass final checks.')
print(f'-> Normalization method: {fire_hazard_vegetation_dryness_normalization_method}')
print('-> Normalization direction: Inverse')
print(f"-> NDVI bounds: "
    f"{fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['lower_bound']} "
    f"to "
    f"{fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['upper_bound']}")
print(f"-> NDMI bounds: "
    f"{fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['lower_bound']} "
    f"to "
    f"{fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['upper_bound']}")
print(f"-> NDVI weight: {fire_hazard_vegetation_dryness_index_weights['NDVI']:.2f}")
print(f"-> NDMI weight: {fire_hazard_vegetation_dryness_index_weights['NDMI']:.2f}")
print(f'-> Minimum valid observations: {fire_hazard_vegetation_dryness_minimum_observations}')
print(f'-> Output score range: '
    f'{fire_hazard_vegetation_dryness_score_minimum:.1f} '
    f'to '
    f'{fire_hazard_vegetation_dryness_score_maximum:.1f}')
print(f'-> Processing window size: '
    f'{fire_hazard_vegetation_dryness_window_size} '
    f'x '
    f'{fire_hazard_vegetation_dryness_window_size}')
print(f'-> Reuse existing outputs: {fire_hazard_vegetation_dryness_reuse_existing}')
print(f'-> Normalization configured: {fire_hazard_vegetation_dryness_normalization_configured}')
print(f'-> Normalization policy saved: {fire_hazard_vegetation_dryness_normalization_policy_path}')
print(f'-> Configuration summary '
    f'saved: '
    f'{fire_hazard_vegetation_dryness_normalization_summary_path}')
print('\n--- VEGETATION-DRYNESS NORMALIZATION CONFIGURATION ---')
display(fire_hazard_vegetation_dryness_normalization_summary)
print('\nNOTE:')
print('NDVI and NDMI will be inversely normalized to a common zero-to-one hazard scale.')
print('Values at or below each '
    'configured lower bound will '
    'receive a maximum dryness '
    'score of 1.0, while values at '
    'or above the upper bound will '
    'receive a minimum dryness '
    'score of 0.0.')
print('The final vegetation-dryness '
    'hazard score will combine the '
    'normalized NDVI and NDMI '
    'layers using weights of 0.40 '
    'and 0.60, respectively.')
print('The next step is to normalize '
    'the NDVI temporal composite '
    'into an inverse '
    'vegetation-dryness hazard '
    'score.')
print('\n=== VEGETATION-DRYNESS NORMALIZATION CONFIGURED ===')



=== CONFIGURING VEGETATION-DRYNESS NORMALIZATION ===
-> Normalization method: inverse_min_max
-> Normalization direction: Inverse
-> NDVI bounds: 0.1 to 0.8
-> NDMI bounds: -0.2 to 0.5
-> NDVI weight: 0.40
-> NDMI weight: 0.60
-> Minimum valid observations: 3
-> Output score range: 0.0 to 1.0
-> Processing window size: 512 x 512
-> Reuse existing outputs: True
-> Normalization configured: True
-> Normalization policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_normalization_policy.json
-> Configuration summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_normalization_summary.csv

--- VEGETATION-DRYNESS NORMALIZATION CONFIGURATION ---


,COMPONENT_ID,NORMALIZATION_METHOD,NDVI_LOWER_BOUND,NDVI_UPPER_BOUND,NDMI_LOWER_BOUND,NDMI_UPPER_BOUND,NDVI_WEIGHT,NDMI_WEIGHT,WEIGHT_TOTAL,MINIMUM_VALID_OBSERVATIONS,REQUIRE_BOTH_INDICES,OUTPUT_SCORE_MINIMUM,OUTPUT_SCORE_MAXIMUM,TARGET_CRS,CELL_SIZE_METERS,WINDOW_SIZE,REUSE_EXISTING,OVERWRITE_EXISTING
0,VEGETATION_DRYNESS,inverse_min_max,0.1,0.8,-0.2,0.5,0.4,0.6,1.0,3,True,0.0,1.0,EPSG:26912,30,512,True,False



NOTE:
NDVI and NDMI will be inversely normalized to a common zero-to-one hazard scale.
Values at or below each configured lower bound will receive a maximum dryness score of 1.0, while values at or above the upper bound will receive a minimum dryness score of 0.0.
The final vegetation-dryness hazard score will combine the normalized NDVI and NDMI layers using weights of 0.40 and 0.60, respectively.
The next step is to normalize the NDVI temporal composite into an inverse vegetation-dryness hazard score.

=== VEGETATION-DRYNESS NORMALIZATION CONFIGURED ===


### Normalizing Ndvi to Vegetation-Dryness Hazard


In [91]:
print('=== NORMALIZING NDVI TO VEGETATION-DRYNESS HAZARD ===')
# List the required NDVI normalization prerequisites required before NDVI to
# vegetation-dryness hazard can run.
required_ndvi_normalization_inputs = ['fire_hazard_vegetation_dryness_normalization_configured',
    'fire_hazard_hls_ndvi_composite_path', 'fire_hazard_hls_ndvi_observation_count_path',
    'fire_hazard_vegetation_dryness_ndvi_score_path',
    'fire_hazard_vegetation_dryness_output_profile',
    'fire_hazard_vegetation_dryness_normalization_bounds',
    'fire_hazard_vegetation_dryness_score_minimum',
    'fire_hazard_vegetation_dryness_score_maximum',
    'fire_hazard_vegetation_dryness_minimum_observations',
    'fire_hazard_vegetation_dryness_output_nodata',
    'fire_hazard_vegetation_dryness_output_dtype',
    'fire_hazard_vegetation_dryness_window_size', \
        'fire_hazard_vegetation_dryness_progress_interval',
    'fire_hazard_vegetation_dryness_reuse_existing',
    'fire_hazard_vegetation_dryness_overwrite', \
        'fire_hazard_vegetation_dryness_normalization_manifest_path',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_nodata_value']
# Identify unavailable NDVI normalization so NDVI to vegetation-dryness hazard stops
# before using incomplete inputs.
missing_ndvi_normalization_inputs = [object_name for object_name in \
    required_ndvi_normalization_inputs if object_name not in globals()]

# Stop execution when required ndvi normalization inputs inputs are unavailable.
if missing_ndvi_normalization_inputs:
    raise NameError(f'The following NDVI '
        f'normalization objects are '
        f'missing:\n'
        f'{missing_ndvi_normalization_inputs}\n\n'
        f'Run Configure '
        f'Vegetation-Dryness '
        f'Normalization before '
        f'normalizing NDVI.')

# Stop execution if vegetation-dryness normalization configuration is not complete.
if not fire_hazard_vegetation_dryness_normalization_configured:
    raise ValueError('The vegetation-dryness normalization configuration is not complete.')
# Build the NDVI normalization input path location so NDVI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndvi_normalization_input_path = Path(fire_hazard_hls_ndvi_composite_path)
# Build the NDVI normalization count path location so NDVI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndvi_normalization_count_path = Path(fire_hazard_hls_ndvi_observation_count_path)
# Build the NDVI normalization output path location so NDVI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndvi_normalization_output_path = Path(fire_hazard_vegetation_dryness_ndvi_score_path)
# Build the NDVI normalization temporary path location so NDVI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndvi_normalization_temporary_path = fire_hazard_ndvi_normalization_output_path.parent \
    / (fire_hazard_ndvi_normalization_output_path.stem + '.part.tif')

# Process each (input name entry so NDVI to vegetation-dryness hazard is applied consistently across
# all records.
for input_name, input_path in {'NDVI composite': fire_hazard_ndvi_normalization_input_path,
    'NDVI observation count': fire_hazard_ndvi_normalization_count_path}.items():

    # Stop execution if the reported value raster is unavailable: the reported value.
    if not input_path.exists() or not input_path.is_file() or input_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The {input_name} raster is unavailable:\n{input_path}')
# Store NDVI lower bound needed to carry out NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_lower_bound = \
    float(fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['lower_bound'])
# Store NDVI upper bound needed to carry out NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_upper_bound = \
    float(fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['upper_bound'])
# Store NDVI normalization range needed to carry out NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_normalization_range = fire_hazard_ndvi_upper_bound - fire_hazard_ndvi_lower_bound

# Require The configured NDVI normalization range to be greater than zero.
if fire_hazard_ndvi_normalization_range <= 0:
    raise ValueError('The configured NDVI normalization range must be greater than zero.')

# Encapsulate validate existing NDVI dryness score so repeated NDVI to vegetation-dryness hazard
# steps use consistent logic.
def validate_existing_ndvi_dryness_score(output_path):

    """
    Confirms that an existing normalized NDVI raster
    matches the common fire-hazard grid.
    """
    # Build the output path location so NDVI to vegetation-dryness hazard uses the expected project
    # file structure.
    output_path = Path(output_path)

    # Use the existing file only when it is present and valid for NDVI to vegetation-dryness hazard.
    if not output_path.exists() or not output_path.is_file() or output_path.stat().st_size <= 0:
        return False

    # Protect NDVI to vegetation-dryness hazard so expected source or file failures do not leave
    # partial outputs.
    try:

        # Open the raster in a managed context so its file handle closes reliably after NDVI to
        # vegetation-dryness hazard.
        with rasterio.open(output_path) as output_source:
            return all([output_source.count == 1,
                output_source.width == fire_hazard_alignment_width,
                output_source.height == fire_hazard_alignment_height,
                output_source.crs is not None, output_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                output_source.transform.almost_equals(fire_hazard_alignment_transform),
                output_source.dtypes[0] == fire_hazard_vegetation_dryness_output_dtype,
                output_source.nodata == fire_hazard_vegetation_dryness_output_nodata])
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception:
        return False
# Record NDVI existing output valid so NDVI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndvi_existing_output_valid = \
    validate_existing_ndvi_dryness_score(fire_hazard_ndvi_normalization_output_path)
# Store NDVI normalization reused needed to carry out NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_normalization_reused = fire_hazard_vegetation_dryness_reuse_existing and \
    fire_hazard_ndvi_existing_output_valid
# Record NDVI dryness valid pixels so NDVI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndvi_dryness_valid_pixels = 0
# Set NDVI dryness NoData pixels to control NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_dryness_nodata_pixels = 0
# Track NDVI dryness minimum across processed pixels for output-range QA.
fire_hazard_ndvi_dryness_minimum = None
# Track NDVI dryness maximum across processed pixels for output-range QA.
fire_hazard_ndvi_dryness_maximum = None
# Set NDVI supported input pixels to control NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_supported_input_pixels = 0
# Set NDVI insufficient support pixels to control NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_insufficient_support_pixels = 0
# Record NDVI normalization action so NDVI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndvi_normalization_action = None

# Handle the NDVI normalization reused case explicitly during NDVI to vegetation-dryness hazard.
if fire_hazard_ndvi_normalization_reused:
    # Record NDVI normalization action so NDVI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    fire_hazard_ndvi_normalization_action = 'Reused existing normalized NDVI raster'
    print('-> Valid cached normalized NDVI raster found.')

    # Open the raster in a managed context so its file handle closes reliably after NDVI to
    # vegetation-dryness hazard.
    with rasterio.open(fire_hazard_ndvi_normalization_output_path) as normalized_source:

        # Process each ( entry so NDVI to vegetation-dryness hazard is applied consistently across
        # all records.
        for _, source_window in normalized_source.block_windows(1):
            # Prepare normalized array used to process the current raster window.
            normalized_array = normalized_source.read(1, window=source_window)
            # Build the valid normalized mask mask used to isolate records required for this
            # analysis.
            valid_normalized_mask = np.isfinite(normalized_array) & (normalized_array != \
                fire_hazard_vegetation_dryness_output_nodata)
            # Record valid normalized values so NDVI to vegetation-dryness hazard can preserve a
            # clear processing and validation outcome.
            valid_normalized_values = normalized_array[valid_normalized_mask]
            # Record NDVI dryness valid pixels so NDVI to vegetation-dryness hazard can preserve a
            # clear processing and validation outcome.
            fire_hazard_ndvi_dryness_valid_pixels += int(valid_normalized_values.size)
            # Store NDVI dryness NoData pixels needed to carry out NDVI to vegetation-dryness
            # hazard.
            fire_hazard_ndvi_dryness_nodata_pixels += int((~valid_normalized_mask).sum())

            # Handle the valid normalized values case explicitly during NDVI to vegetation-dryness
            # hazard.
            if valid_normalized_values.size > 0:
                # Track window minimum across processed pixels for output-range QA.
                window_minimum = float(valid_normalized_values.min())
                # Track window maximum across processed pixels for output-range QA.
                window_maximum = float(valid_normalized_values.max())
                # Track NDVI dryness minimum across processed pixels for output-range QA.
                fire_hazard_ndvi_dryness_minimum = window_minimum if \
                    fire_hazard_ndvi_dryness_minimum is None else \
                    min(fire_hazard_ndvi_dryness_minimum,
                    window_minimum)
                # Track NDVI dryness maximum across processed pixels for output-range QA.
                fire_hazard_ndvi_dryness_maximum = window_maximum if \
                    fire_hazard_ndvi_dryness_maximum is None else \
                    max(fire_hazard_ndvi_dryness_maximum,
                    window_maximum)
else:
    # Record NDVI normalization action so NDVI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    fire_hazard_ndvi_normalization_action = 'Created inverse-normalized NDVI raster'

    # Use the existing file only when it is present and valid for NDVI to vegetation-dryness hazard.
    if fire_hazard_ndvi_normalization_temporary_path.exists():
        fire_hazard_ndvi_normalization_temporary_path.unlink()

    # Protect existing outputs unless the configured overwrite policy permits replacement.
    if fire_hazard_ndvi_normalization_output_path.exists() and \
        fire_hazard_vegetation_dryness_overwrite:
        fire_hazard_ndvi_normalization_output_path.unlink()
    from rasterio.windows import Window

    # Open the raster in a managed context so its file handle closes reliably after NDVI to
    # vegetation-dryness hazard.
    with rasterio.open(fire_hazard_ndvi_normalization_input_path) as ndvi_source:

        # Open the raster in a managed context so its file handle closes reliably after NDVI to
        # vegetation-dryness hazard.
        with rasterio.open(fire_hazard_ndvi_normalization_count_path) as count_source:
            # Record NDVI source grid valid so NDVI to vegetation-dryness hazard can preserve a
            # clear processing and validation outcome.
            ndvi_source_grid_valid = all([ndvi_source.count == 1,
                count_source.count == 1, ndvi_source.width == fire_hazard_alignment_width,
                ndvi_source.height == fire_hazard_alignment_height,
                count_source.width == fire_hazard_alignment_width,
                count_source.height == fire_hazard_alignment_height,
                ndvi_source.crs is not None, count_source.crs is not None,
                ndvi_source.crs == count_source.crs, ndvi_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                ndvi_source.transform.almost_equals(fire_hazard_alignment_transform),
                count_source.transform.almost_equals(fire_hazard_alignment_transform)])

            # Stop execution if nDVI composite and observation-count rasters do not match the common
            # fire-hazard analysis grid.
            if not ndvi_source_grid_valid:
                raise ValueError('The NDVI composite and '
                    'observation-count rasters do '
                    'not match the common '
                    'fire-hazard analysis grid.')

            # Open the raster in a managed context so its file handle closes reliably after NDVI to
            # vegetation-dryness hazard.
            with rasterio.open(fire_hazard_ndvi_normalization_temporary_path,
                'w', **fire_hazard_vegetation_dryness_output_profile) as normalized_destination:
                # Calculate window column count to quantify completeness and support QA checks.
                window_column_count = int(np.ceil(fire_hazard_alignment_width / \
                    fire_hazard_vegetation_dryness_window_size))
                # Calculate window row count to quantify completeness and support QA checks.
                window_row_count = int(np.ceil(fire_hazard_alignment_height / \
                    fire_hazard_vegetation_dryness_window_size))
                # Store total processing windows needed to carry out NDVI to vegetation-dryness
                # hazard.
                total_processing_windows = window_column_count * window_row_count
                # Set current window number to control NDVI to vegetation-dryness hazard.
                current_window_number = 0

                # Process each window row entry so NDVI to vegetation-dryness hazard is applied
                # consistently across all records.
                for window_row in range(window_row_count):
                    # Calculate row offset for the current raster processing window.
                    row_offset = window_row * fire_hazard_vegetation_dryness_window_size
                    # Store window height needed to carry out NDVI to vegetation-dryness hazard.
                    window_height = min(fire_hazard_vegetation_dryness_window_size,
                        fire_hazard_alignment_height - row_offset)

                    # Process each window column entry so NDVI to vegetation-dryness hazard is
                    # applied consistently across all records.
                    for window_column in range(window_column_count):
                        # Calculate column offset for the current raster processing window.
                        column_offset = window_column * fire_hazard_vegetation_dryness_window_size
                        # Store window width needed to carry out NDVI to vegetation-dryness hazard.
                        window_width = min(fire_hazard_vegetation_dryness_window_size,
                            fire_hazard_alignment_width - column_offset)
                        # Set current window number to control NDVI to vegetation-dryness hazard.
                        current_window_number += 1

                        # Report progress at the configured interval without changing
                        # raster-processing results.
                        if current_window_number == 1 or current_window_number % \
                            fire_hazard_vegetation_dryness_progress_interval == 0 or \
                            current_window_number == total_processing_windows:
                            print(f'-> Normalizing NDVI window '
                                f'{current_window_number:,} of '
                                f'{total_processing_windows:,}...')
                        # Store processing window needed to carry out NDVI to vegetation-dryness
                        # hazard.
                        processing_window = Window(col_off=column_offset,
                            row_off=row_offset, width=window_width, height=window_height)
                        # Prepare NDVI array used to process the current raster window.
                        ndvi_array = ndvi_source.read(1,
                            window=processing_window).astype(np.float32)
                        # Calculate observation count array to quantify completeness and support QA
                        # checks.
                        observation_count_array = count_source.read(1, window=processing_window)
                        # Build the valid ndvi mask mask used to isolate records required for this
                        # analysis.
                        valid_ndvi_mask = np.isfinite(ndvi_array) & (ndvi_array != \
                            fire_hazard_nodata_value)
                        # Build the sufficient support mask mask used to isolate records required
                        # for this analysis.
                        sufficient_support_mask = observation_count_array >= \
                            fire_hazard_vegetation_dryness_minimum_observations
                        # Build the retained input mask mask used to isolate records required for
                        # this analysis.
                        retained_input_mask = valid_ndvi_mask & sufficient_support_mask
                        # Store NDVI supported input pixels needed to carry out NDVI to
                        # vegetation-dryness hazard.
                        fire_hazard_ndvi_supported_input_pixels += int(retained_input_mask.sum())
                        # Store NDVI insufficient support pixels needed to carry out NDVI to
                        # vegetation-dryness hazard.
                        fire_hazard_ndvi_insufficient_support_pixels += int((valid_ndvi_mask & \
                            ~sufficient_support_mask).sum())
                        # Prepare normalized NDVI array used to process the current raster window.
                        normalized_ndvi_array = np.full((int(processing_window.height),
                            int(processing_window.width)), \
                                fire_hazard_vegetation_dryness_output_nodata,
                            dtype=np.float32)

                        # Handle the retained input mask case explicitly during NDVI to
                        # vegetation-dryness hazard.
                        if retained_input_mask.any():
                            # Store clipped NDVI values needed to carry out NDVI to
                            # vegetation-dryness hazard.
                            clipped_ndvi_values = np.clip(ndvi_array[retained_input_mask],
                                fire_hazard_ndvi_lower_bound, fire_hazard_ndvi_upper_bound)
                            # Store normalized NDVI values needed to carry out NDVI to
                            # vegetation-dryness hazard.
                            normalized_ndvi_values = (fire_hazard_ndvi_upper_bound - \
                                clipped_ndvi_values) / fire_hazard_ndvi_normalization_range
                            # Store normalized NDVI values needed to carry out NDVI to
                            # vegetation-dryness hazard.
                            normalized_ndvi_values = np.clip(normalized_ndvi_values,
                                fire_hazard_vegetation_dryness_score_minimum, \
                                    fire_hazard_vegetation_dryness_score_maximum)
                            # Build the normalized NDVI array[retained input mask] mask to isolate
                            # pixels that satisfy the current analysis criteria.
                            normalized_ndvi_array[retained_input_mask] = \
                                normalized_ndvi_values.astype(np.float32)
                        # Write the processed data to the configured output resource.
                        normalized_destination.write(normalized_ndvi_array,
                            1, window=processing_window)
                        # Build the valid output mask mask used to isolate records required for this
                        # analysis.
                        valid_output_mask = np.isfinite(normalized_ndvi_array) & \
                            (normalized_ndvi_array != fire_hazard_vegetation_dryness_output_nodata)
                        # Record valid output values so NDVI to vegetation-dryness hazard can
                        # preserve a clear processing and validation outcome.
                        valid_output_values = normalized_ndvi_array[valid_output_mask]
                        # Record NDVI dryness valid pixels so NDVI to vegetation-dryness hazard can
                        # preserve a clear processing and validation outcome.
                        fire_hazard_ndvi_dryness_valid_pixels += int(valid_output_values.size)
                        # Store NDVI dryness NoData pixels needed to carry out NDVI to
                        # vegetation-dryness hazard.
                        fire_hazard_ndvi_dryness_nodata_pixels += int((~valid_output_mask).sum())

                        # Handle the valid output values case explicitly during NDVI to
                        # vegetation-dryness hazard.
                        if valid_output_values.size > 0:
                            # Track window minimum across processed pixels for output-range QA.
                            window_minimum = float(valid_output_values.min())
                            # Track window maximum across processed pixels for output-range QA.
                            window_maximum = float(valid_output_values.max())
                            # Track NDVI dryness minimum across processed pixels for output-range
                            # QA.
                            fire_hazard_ndvi_dryness_minimum = window_minimum if \
                                fire_hazard_ndvi_dryness_minimum is None else \
                                min(fire_hazard_ndvi_dryness_minimum,
                                window_minimum)
                            # Track NDVI dryness maximum across processed pixels for output-range
                            # QA.
                            fire_hazard_ndvi_dryness_maximum = window_maximum if \
                                fire_hazard_ndvi_dryness_maximum is None else \
                                max(fire_hazard_ndvi_dryness_maximum,
                                window_maximum)
                        del ndvi_array
                        del observation_count_array
                        del normalized_ndvi_array
                normalized_destination.set_band_description(1,
                    'Inverse-normalized NDVI vegetation-dryness score')
                normalized_destination.update_tags(COMPONENT_ID='VEGETATION_DRYNESS',
                    SOURCE_INDEX='NDVI', NORMALIZATION_METHOD='inverse_min_max',
                    LOWER_BOUND=fire_hazard_ndvi_lower_bound, \
                        UPPER_BOUND=fire_hazard_ndvi_upper_bound,
                    OUTPUT_MINIMUM=fire_hazard_vegetation_dryness_score_minimum,
                    OUTPUT_MAXIMUM=fire_hazard_vegetation_dryness_score_maximum,
                    MINIMUM_VALID_OBSERVATIONS=fire_hazard_vegetation_dryness_minimum_observations,
                    HAZARD_DIRECTION='Lower NDVI produces higher hazard',
                    TARGET_CRS=fire_hazard_target_crs, TARGET_CELL_SIZE=fire_hazard_cell_size)
    # Record temporary NDVI output valid so NDVI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    temporary_ndvi_output_valid = \
        validate_existing_ndvi_dryness_score(fire_hazard_ndvi_normalization_temporary_path)
    # Record NDVI normalized content valid so NDVI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    ndvi_normalized_content_valid = fire_hazard_ndvi_dryness_valid_pixels > 0 and \
        fire_hazard_ndvi_dryness_minimum is not None and (fire_hazard_ndvi_dryness_maximum is not \
        None) and (fire_hazard_ndvi_dryness_minimum >= \
        fire_hazard_vegetation_dryness_score_minimum) and (fire_hazard_ndvi_dryness_maximum <= \
        fire_hazard_vegetation_dryness_score_maximum)

    # Stop execution if temporary normalized NDVI dryness raster failed validation.
    if not temporary_ndvi_output_valid or not ndvi_normalized_content_valid:
        raise ValueError(f'The temporary normalized NDVI '
            f'dryness raster failed '
            f'validation.\n\nRaster '
            f'structure valid: '
            f'{temporary_ndvi_output_valid}\n'
            f'Valid pixels: '
            f'{fire_hazard_ndvi_dryness_valid_pixels:,}\n'
            f'Score minimum: '
            f'{fire_hazard_ndvi_dryness_minimum}\n'
            f'Score maximum: '
            f'{fire_hazard_ndvi_dryness_maximum}')
    fire_hazard_ndvi_normalization_temporary_path.replace \
        (fire_hazard_ndvi_normalization_output_path)
# Record NDVI dryness score valid so NDVI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndvi_dryness_score_valid = \
    validate_existing_ndvi_dryness_score(fire_hazard_ndvi_normalization_output_path) and \
    fire_hazard_ndvi_dryness_valid_pixels > 0 and (fire_hazard_ndvi_dryness_minimum is not None) \
    and (fire_hazard_ndvi_dryness_maximum is not None) and (fire_hazard_ndvi_dryness_minimum >= \
    fire_hazard_vegetation_dryness_score_minimum) and (fire_hazard_ndvi_dryness_maximum <= \
    fire_hazard_vegetation_dryness_score_maximum)

# Stop execution if final normalized NDVI vegetation-dryness raster failed validation.
if not fire_hazard_ndvi_dryness_score_valid:
    raise ValueError('The final normalized NDVI vegetation-dryness raster failed validation.')
# Define NDVI normalization manifest record to control the inputs and rules used by NDVI to
# vegetation-dryness hazard.
fire_hazard_ndvi_normalization_manifest_record = {'INDEX_NAME': 'NDVI',
    'OUTPUT_TYPE': 'Inverse-normalized dryness score',
    'INPUT_PATH': str(fire_hazard_ndvi_normalization_input_path),
    'OBSERVATION_COUNT_PATH': str(fire_hazard_ndvi_normalization_count_path),
    'OUTPUT_PATH': str(fire_hazard_ndvi_normalization_output_path),
    'PROCESSING_ACTION': fire_hazard_ndvi_normalization_action,
    'NORMALIZATION_METHOD': 'inverse_min_max', 'LOWER_BOUND': fire_hazard_ndvi_lower_bound,
    'UPPER_BOUND': fire_hazard_ndvi_upper_bound, 'MINIMUM_VALID_OBSERVATIONS': \
        fire_hazard_vegetation_dryness_minimum_observations,
    'VALID_PIXELS': fire_hazard_ndvi_dryness_valid_pixels,
    'NODATA_PIXELS': fire_hazard_ndvi_dryness_nodata_pixels,
    'INSUFFICIENT_SUPPORT_PIXELS': fire_hazard_ndvi_insufficient_support_pixels,
    'SCORE_MINIMUM': fire_hazard_ndvi_dryness_minimum,
    'SCORE_MAXIMUM': fire_hazard_ndvi_dryness_maximum,
    'OUTPUT_SIZE_BYTES': fire_hazard_ndvi_normalization_output_path.stat().st_size,
    'OUTPUT_SIZE_MB': fire_hazard_ndvi_normalization_output_path.stat().st_size / 1024 ** 2,
    'VALID': fire_hazard_ndvi_dryness_score_valid}

# Use the existing file only when it is present and valid for NDVI to vegetation-dryness hazard.
if fire_hazard_vegetation_dryness_normalization_manifest_path.exists():
    # Store existing normalization manifest needed to carry out NDVI to vegetation-dryness hazard.
    existing_normalization_manifest = \
        pd.read_csv(fire_hazard_vegetation_dryness_normalization_manifest_path)
else:
    # Store existing normalization manifest needed to carry out NDVI to vegetation-dryness hazard.
    existing_normalization_manifest = pd.DataFrame()

# Handle missing or invalid existing normalization manifest explicitly during NDVI to
# vegetation-dryness hazard.
if not existing_normalization_manifest.empty:
    # Create an isolated existing normalization manifest working copy so NDVI to vegetation-dryness
    # hazard does not modify upstream data.
    existing_normalization_manifest = \
        existing_normalization_manifest[~((existing_normalization_manifest['INDEX_NAME'].astype \
        (str).str.upper() == 'NDVI') & \
        (existing_normalization_manifest['OUTPUT_TYPE'].astype(str) == \
        'Inverse-normalized dryness score'))].copy()
# Store vegetation dryness normalization manifest needed to carry out NDVI to vegetation-dryness
# hazard.
fire_hazard_vegetation_dryness_normalization_manifest = pd.concat([existing_normalization_manifest,
    pd.DataFrame([fire_hazard_ndvi_normalization_manifest_record])],
    ignore_index=True).reset_index(drop=True)
# Save the vegetation dryness normalization manifest CSV so later steps can reuse the recorded
# workflow results.
fire_hazard_vegetation_dryness_normalization_manifest.to_csv \
    (fire_hazard_vegetation_dryness_normalization_manifest_path,
    index=False)
# Store NDVI normalization summary needed to carry out NDVI to vegetation-dryness hazard.
fire_hazard_ndvi_normalization_summary = pd.DataFrame([{'INDEX_NAME': 'NDVI',
    'NORMALIZATION_METHOD': 'Inverse min-max', 'LOWER_BOUND': fire_hazard_ndvi_lower_bound,
    'UPPER_BOUND': fire_hazard_ndvi_upper_bound, 'MINIMUM_VALID_OBSERVATIONS': \
        fire_hazard_vegetation_dryness_minimum_observations,
    'VALID_OUTPUT_PIXELS': fire_hazard_ndvi_dryness_valid_pixels,
    'NODATA_OUTPUT_PIXELS': fire_hazard_ndvi_dryness_nodata_pixels,
    'INSUFFICIENT_SUPPORT_PIXELS': fire_hazard_ndvi_insufficient_support_pixels,
    'SCORE_MINIMUM': fire_hazard_ndvi_dryness_minimum,
    'SCORE_MAXIMUM': fire_hazard_ndvi_dryness_maximum,
    'PROCESSING_ACTION': fire_hazard_ndvi_normalization_action,
    'OUTPUT_PATH': str(fire_hazard_ndvi_normalization_output_path),
    'VALID': fire_hazard_ndvi_dryness_score_valid}])
print(f'-> NDVI normalization bounds: '
    f'{fire_hazard_ndvi_lower_bound:.2f} '
    f'to '
    f'{fire_hazard_ndvi_upper_bound:.2f}')
print(f'-> Processing action: {fire_hazard_ndvi_normalization_action}')
print(f'-> Valid normalized pixels: {fire_hazard_ndvi_dryness_valid_pixels:,}')
print(f'-> Insufficient-support pixels: {fire_hazard_ndvi_insufficient_support_pixels:,}')
print(f'-> NDVI dryness-score range: '
    f'{fire_hazard_ndvi_dryness_minimum:.4f} '
    f'to '
    f'{fire_hazard_ndvi_dryness_maximum:.4f}')
print(f'-> Normalized NDVI raster saved: {fire_hazard_ndvi_normalization_output_path}')
print(f'-> Normalization manifest '
    f'saved: '
    f'{fire_hazard_vegetation_dryness_normalization_manifest_path}')
print('\n--- NDVI VEGETATION-DRYNESS NORMALIZATION SUMMARY ---')
display(fire_hazard_ndvi_normalization_summary)
print('\n--- VEGETATION-DRYNESS NORMALIZATION MANIFEST SAMPLE ---')
display(fire_hazard_vegetation_dryness_normalization_manifest.head(5))
import gc
gc.collect()
print('\nNOTE:')
print('The NDVI temporal composite '
    'has been converted to a '
    'zero-to-one vegetation-dryness '
    'hazard score.')
print('Lower NDVI values produce higher scores, while higher NDVI values produce lower scores.')
print('Only pixels supported by the '
    'configured minimum number of '
    'valid HLS observations were '
    'retained.')
print('The next step is to normalize '
    'the NDMI temporal composite '
    'using the same inverse '
    'hazard-scoring approach.')
print('\n=== NDVI VEGETATION-DRYNESS NORMALIZATION COMPLETE ===')



=== NORMALIZING NDVI TO VEGETATION-DRYNESS HAZARD ===
-> Valid cached normalized NDVI raster found.
-> NDVI normalization bounds: 0.10 to 0.80
-> Processing action: Reused existing normalized NDVI raster
-> Valid normalized pixels: 14,140,492
-> Insufficient-support pixels: 0
-> NDVI dryness-score range: 0.0000 to 1.0000
-> Normalized NDVI raster saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\normalized_rasters\ndvi_inverse_normalized_dryness_score.tif
-> Normalization manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_normalization_manifest.csv

--- NDVI VEGETATION-DRYNESS NORMALIZATION SUMMARY ---


,INDEX_NAME,NORMALIZATION_METHOD,LOWER_BOUND,UPPER_BOUND,MINIMUM_VALID_OBSERVATIONS,VALID_OUTPUT_PIXELS,NODATA_OUTPUT_PIXELS,INSUFFICIENT_SUPPORT_PIXELS,SCORE_MINIMUM,SCORE_MAXIMUM,PROCESSING_ACTION,OUTPUT_PATH,VALID
0,NDVI,Inverse min-max,0.1,0.8,3,14140492,122630526,0,0.0,1.0,Reused existing normalized NDVI raster,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



--- VEGETATION-DRYNESS NORMALIZATION MANIFEST SAMPLE ---


,INDEX_NAME,OUTPUT_TYPE,INPUT_PATH,OBSERVATION_COUNT_PATH,OUTPUT_PATH,PROCESSING_ACTION,NORMALIZATION_METHOD,LOWER_BOUND,UPPER_BOUND,MINIMUM_VALID_OBSERVATIONS,...,OUTPUT_SIZE_MB,VALID,NDVI_WEIGHT,NDMI_WEIGHT,REQUIRE_BOTH_INDICES,BOTH_INDICES_VALID_PIXELS,NDVI_ONLY_PIXELS,NDMI_ONLY_PIXELS,NEITHER_INDEX_PIXELS,VALID_OUTSIDE_STUDY_AREA
0,NDMI,Inverse-normalized dryness score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing normalized NDMI raster,inverse_min_max,-0.2,0.5,3.0,...,43.655200,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,COMBINED,Weighted vegetation-dryness hazard score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,NaN,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing combined dryness raster,weighted_linear_combination,0.0,1.0,NaN,...,43.922072,True,0.4,0.6,True,0.0,0.0,0.0,0.0,0.0
2,NDVI,Inverse-normalized dryness score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing normalized NDVI raster,inverse_min_max,0.1,0.8,3.0,...,44.696409,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



NOTE:
The NDVI temporal composite has been converted to a zero-to-one vegetation-dryness hazard score.
Lower NDVI values produce higher scores, while higher NDVI values produce lower scores.
Only pixels supported by the configured minimum number of valid HLS observations were retained.
The next step is to normalize the NDMI temporal composite using the same inverse hazard-scoring approach.

=== NDVI VEGETATION-DRYNESS NORMALIZATION COMPLETE ===


### Normalizing Ndmi to Vegetation-Dryness Hazard


In [92]:
print('=== NORMALIZING NDMI TO VEGETATION-DRYNESS HAZARD ===')
# List the required NDMI normalization prerequisites required before NDMI to
# vegetation-dryness hazard can run.
required_ndmi_normalization_inputs = ['fire_hazard_vegetation_dryness_normalization_configured',
    'fire_hazard_hls_ndmi_composite_path', 'fire_hazard_hls_ndmi_observation_count_path',
    'fire_hazard_vegetation_dryness_ndmi_score_path',
    'fire_hazard_vegetation_dryness_output_profile',
    'fire_hazard_vegetation_dryness_normalization_bounds',
    'fire_hazard_vegetation_dryness_score_minimum',
    'fire_hazard_vegetation_dryness_score_maximum',
    'fire_hazard_vegetation_dryness_minimum_observations',
    'fire_hazard_vegetation_dryness_output_nodata',
    'fire_hazard_vegetation_dryness_output_dtype',
    'fire_hazard_vegetation_dryness_window_size', \
        'fire_hazard_vegetation_dryness_progress_interval',
    'fire_hazard_vegetation_dryness_reuse_existing',
    'fire_hazard_vegetation_dryness_overwrite', \
        'fire_hazard_vegetation_dryness_normalization_manifest_path',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_nodata_value']
# Identify unavailable NDMI normalization so NDMI to vegetation-dryness hazard stops
# before using incomplete inputs.
missing_ndmi_normalization_inputs = [object_name for object_name in \
    required_ndmi_normalization_inputs if object_name not in globals()]

# Stop execution when required ndmi normalization inputs inputs are unavailable.
if missing_ndmi_normalization_inputs:
    raise NameError(f'The following NDMI '
        f'normalization objects are '
        f'missing:\n'
        f'{missing_ndmi_normalization_inputs}\n\n'
        f'Run Configure '
        f'Vegetation-Dryness '
        f'Normalization before '
        f'normalizing NDMI.')

# Stop execution if vegetation-dryness normalization configuration is not complete.
if not fire_hazard_vegetation_dryness_normalization_configured:
    raise ValueError('The vegetation-dryness normalization configuration is not complete.')
# Build the NDMI normalization input path location so NDMI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndmi_normalization_input_path = Path(fire_hazard_hls_ndmi_composite_path)
# Build the NDMI normalization count path location so NDMI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndmi_normalization_count_path = Path(fire_hazard_hls_ndmi_observation_count_path)
# Build the NDMI normalization output path location so NDMI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndmi_normalization_output_path = Path(fire_hazard_vegetation_dryness_ndmi_score_path)
# Build the NDMI normalization temporary path location so NDMI to vegetation-dryness hazard uses the
# expected project file structure.
fire_hazard_ndmi_normalization_temporary_path = fire_hazard_ndmi_normalization_output_path.parent \
    / (fire_hazard_ndmi_normalization_output_path.stem + '.part.tif')

# Process each (input name entry so NDMI to vegetation-dryness hazard is applied consistently across
# all records.
for input_name, input_path in {'NDMI composite': fire_hazard_ndmi_normalization_input_path,
    'NDMI observation count': fire_hazard_ndmi_normalization_count_path}.items():

    # Stop execution if the reported value raster is unavailable: the reported value.
    if not input_path.exists() or not input_path.is_file() or input_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The {input_name} raster is unavailable:\n{input_path}')
# Store NDMI lower bound needed to carry out NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_lower_bound = \
    float(fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['lower_bound'])
# Store NDMI upper bound needed to carry out NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_upper_bound = \
    float(fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['upper_bound'])
# Store NDMI normalization range needed to carry out NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_normalization_range = fire_hazard_ndmi_upper_bound - fire_hazard_ndmi_lower_bound

# Require The configured NDMI normalization range to be greater than zero.
if fire_hazard_ndmi_normalization_range <= 0:
    raise ValueError('The configured NDMI normalization range must be greater than zero.')

# Encapsulate validate existing NDMI dryness score so repeated NDMI to vegetation-dryness hazard
# steps use consistent logic.
def validate_existing_ndmi_dryness_score(output_path):

    """
    Confirms that an existing normalized NDMI raster
    matches the common fire-hazard analysis grid.
    """
    # Build the output path location so NDMI to vegetation-dryness hazard uses the expected project
    # file structure.
    output_path = Path(output_path)

    # Use the existing file only when it is present and valid for NDMI to vegetation-dryness hazard.
    if not output_path.exists() or not output_path.is_file() or output_path.stat().st_size <= 0:
        return False

    # Protect NDMI to vegetation-dryness hazard so expected source or file failures do not leave
    # partial outputs.
    try:

        # Open the raster in a managed context so its file handle closes reliably after NDMI to
        # vegetation-dryness hazard.
        with rasterio.open(output_path) as output_source:
            return all([output_source.count == 1,
                output_source.width == fire_hazard_alignment_width,
                output_source.height == fire_hazard_alignment_height,
                output_source.crs is not None, output_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                output_source.transform.almost_equals(fire_hazard_alignment_transform),
                output_source.dtypes[0] == fire_hazard_vegetation_dryness_output_dtype,
                output_source.nodata == fire_hazard_vegetation_dryness_output_nodata])
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception:
        return False
# Record NDMI existing output valid so NDMI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndmi_existing_output_valid = \
    validate_existing_ndmi_dryness_score(fire_hazard_ndmi_normalization_output_path)
# Store NDMI normalization reused needed to carry out NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_normalization_reused = fire_hazard_vegetation_dryness_reuse_existing and \
    fire_hazard_ndmi_existing_output_valid
# Record NDMI dryness valid pixels so NDMI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndmi_dryness_valid_pixels = 0
# Set NDMI dryness NoData pixels to control NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_dryness_nodata_pixels = 0
# Track NDMI dryness minimum across processed pixels for output-range QA.
fire_hazard_ndmi_dryness_minimum = None
# Track NDMI dryness maximum across processed pixels for output-range QA.
fire_hazard_ndmi_dryness_maximum = None
# Set NDMI supported input pixels to control NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_supported_input_pixels = 0
# Set NDMI insufficient support pixels to control NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_insufficient_support_pixels = 0
# Record NDMI normalization action so NDMI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndmi_normalization_action = None

# Handle the NDMI normalization reused case explicitly during NDMI to vegetation-dryness hazard.
if fire_hazard_ndmi_normalization_reused:
    # Record NDMI normalization action so NDMI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    fire_hazard_ndmi_normalization_action = 'Reused existing normalized NDMI raster'
    print('-> Valid cached normalized NDMI raster found.')

    # Open the raster in a managed context so its file handle closes reliably after NDMI to
    # vegetation-dryness hazard.
    with rasterio.open(fire_hazard_ndmi_normalization_output_path) as normalized_source:

        # Process each ( entry so NDMI to vegetation-dryness hazard is applied consistently across
        # all records.
        for _, source_window in normalized_source.block_windows(1):
            # Prepare normalized array used to process the current raster window.
            normalized_array = normalized_source.read(1, window=source_window)
            # Build the valid normalized mask mask used to isolate records required for this
            # analysis.
            valid_normalized_mask = np.isfinite(normalized_array) & (normalized_array != \
                fire_hazard_vegetation_dryness_output_nodata)
            # Record valid normalized values so NDMI to vegetation-dryness hazard can preserve a
            # clear processing and validation outcome.
            valid_normalized_values = normalized_array[valid_normalized_mask]
            # Record NDMI dryness valid pixels so NDMI to vegetation-dryness hazard can preserve a
            # clear processing and validation outcome.
            fire_hazard_ndmi_dryness_valid_pixels += int(valid_normalized_values.size)
            # Store NDMI dryness NoData pixels needed to carry out NDMI to vegetation-dryness
            # hazard.
            fire_hazard_ndmi_dryness_nodata_pixels += int((~valid_normalized_mask).sum())

            # Handle the valid normalized values case explicitly during NDMI to vegetation-dryness
            # hazard.
            if valid_normalized_values.size > 0:
                # Track window minimum across processed pixels for output-range QA.
                window_minimum = float(valid_normalized_values.min())
                # Track window maximum across processed pixels for output-range QA.
                window_maximum = float(valid_normalized_values.max())
                # Track NDMI dryness minimum across processed pixels for output-range QA.
                fire_hazard_ndmi_dryness_minimum = window_minimum if \
                    fire_hazard_ndmi_dryness_minimum is None else \
                    min(fire_hazard_ndmi_dryness_minimum,
                    window_minimum)
                # Track NDMI dryness maximum across processed pixels for output-range QA.
                fire_hazard_ndmi_dryness_maximum = window_maximum if \
                    fire_hazard_ndmi_dryness_maximum is None else \
                    max(fire_hazard_ndmi_dryness_maximum,
                    window_maximum)
else:
    # Record NDMI normalization action so NDMI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    fire_hazard_ndmi_normalization_action = 'Created inverse-normalized NDMI raster'

    # Use the existing file only when it is present and valid for NDMI to vegetation-dryness hazard.
    if fire_hazard_ndmi_normalization_temporary_path.exists():
        fire_hazard_ndmi_normalization_temporary_path.unlink()

    # Protect existing outputs unless the configured overwrite policy permits replacement.
    if fire_hazard_ndmi_normalization_output_path.exists() and \
        fire_hazard_vegetation_dryness_overwrite:
        fire_hazard_ndmi_normalization_output_path.unlink()
    from rasterio.windows import Window

    # Open the raster in a managed context so its file handle closes reliably after NDMI to
    # vegetation-dryness hazard.
    with rasterio.open(fire_hazard_ndmi_normalization_input_path) as ndmi_source:

        # Open the raster in a managed context so its file handle closes reliably after NDMI to
        # vegetation-dryness hazard.
        with rasterio.open(fire_hazard_ndmi_normalization_count_path) as count_source:
            # Record NDMI source grid valid so NDMI to vegetation-dryness hazard can preserve a
            # clear processing and validation outcome.
            ndmi_source_grid_valid = all([ndmi_source.count == 1,
                count_source.count == 1, ndmi_source.width == fire_hazard_alignment_width,
                ndmi_source.height == fire_hazard_alignment_height,
                count_source.width == fire_hazard_alignment_width,
                count_source.height == fire_hazard_alignment_height,
                ndmi_source.crs is not None, count_source.crs is not None,
                ndmi_source.crs == count_source.crs, ndmi_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                ndmi_source.transform.almost_equals(fire_hazard_alignment_transform),
                count_source.transform.almost_equals(fire_hazard_alignment_transform)])

            # Stop execution if nDMI composite and observation-count rasters do not match the common
            # fire-hazard analysis grid.
            if not ndmi_source_grid_valid:
                raise ValueError('The NDMI composite and '
                    'observation-count rasters do '
                    'not match the common '
                    'fire-hazard analysis grid.')

            # Open the raster in a managed context so its file handle closes reliably after NDMI to
            # vegetation-dryness hazard.
            with rasterio.open(fire_hazard_ndmi_normalization_temporary_path,
                'w', **fire_hazard_vegetation_dryness_output_profile) as normalized_destination:
                # Calculate window column count to quantify completeness and support QA checks.
                window_column_count = int(np.ceil(fire_hazard_alignment_width / \
                    fire_hazard_vegetation_dryness_window_size))
                # Calculate window row count to quantify completeness and support QA checks.
                window_row_count = int(np.ceil(fire_hazard_alignment_height / \
                    fire_hazard_vegetation_dryness_window_size))
                # Store total processing windows needed to carry out NDMI to vegetation-dryness
                # hazard.
                total_processing_windows = window_column_count * window_row_count
                # Set current window number to control NDMI to vegetation-dryness hazard.
                current_window_number = 0

                # Process each window row entry so NDMI to vegetation-dryness hazard is applied
                # consistently across all records.
                for window_row in range(window_row_count):
                    # Calculate row offset for the current raster processing window.
                    row_offset = window_row * fire_hazard_vegetation_dryness_window_size
                    # Store window height needed to carry out NDMI to vegetation-dryness hazard.
                    window_height = min(fire_hazard_vegetation_dryness_window_size,
                        fire_hazard_alignment_height - row_offset)

                    # Process each window column entry so NDMI to vegetation-dryness hazard is
                    # applied consistently across all records.
                    for window_column in range(window_column_count):
                        # Calculate column offset for the current raster processing window.
                        column_offset = window_column * fire_hazard_vegetation_dryness_window_size
                        # Store window width needed to carry out NDMI to vegetation-dryness hazard.
                        window_width = min(fire_hazard_vegetation_dryness_window_size,
                            fire_hazard_alignment_width - column_offset)
                        # Set current window number to control NDMI to vegetation-dryness hazard.
                        current_window_number += 1

                        # Report progress at the configured interval without changing
                        # raster-processing results.
                        if current_window_number == 1 or current_window_number % \
                            fire_hazard_vegetation_dryness_progress_interval == 0 or \
                            current_window_number == total_processing_windows:
                            print(f'-> Normalizing NDMI window '
                                f'{current_window_number:,} of '
                                f'{total_processing_windows:,}...')
                        # Store processing window needed to carry out NDMI to vegetation-dryness
                        # hazard.
                        processing_window = Window(col_off=column_offset,
                            row_off=row_offset, width=window_width, height=window_height)
                        # Prepare NDMI array used to process the current raster window.
                        ndmi_array = ndmi_source.read(1,
                            window=processing_window).astype(np.float32)
                        # Calculate observation count array to quantify completeness and support QA
                        # checks.
                        observation_count_array = count_source.read(1, window=processing_window)
                        # Build the valid ndmi mask mask used to isolate records required for this
                        # analysis.
                        valid_ndmi_mask = np.isfinite(ndmi_array) & (ndmi_array != \
                            fire_hazard_nodata_value)
                        # Build the sufficient support mask mask used to isolate records required
                        # for this analysis.
                        sufficient_support_mask = observation_count_array >= \
                            fire_hazard_vegetation_dryness_minimum_observations
                        # Build the retained input mask mask used to isolate records required for
                        # this analysis.
                        retained_input_mask = valid_ndmi_mask & sufficient_support_mask
                        # Store NDMI supported input pixels needed to carry out NDMI to
                        # vegetation-dryness hazard.
                        fire_hazard_ndmi_supported_input_pixels += int(retained_input_mask.sum())
                        # Store NDMI insufficient support pixels needed to carry out NDMI to
                        # vegetation-dryness hazard.
                        fire_hazard_ndmi_insufficient_support_pixels += int((valid_ndmi_mask & \
                            ~sufficient_support_mask).sum())
                        # Prepare normalized NDMI array used to process the current raster window.
                        normalized_ndmi_array = np.full((int(processing_window.height),
                            int(processing_window.width)), \
                                fire_hazard_vegetation_dryness_output_nodata,
                            dtype=np.float32)

                        # Handle the retained input mask case explicitly during NDMI to
                        # vegetation-dryness hazard.
                        if retained_input_mask.any():
                            # Store clipped NDMI values needed to carry out NDMI to
                            # vegetation-dryness hazard.
                            clipped_ndmi_values = np.clip(ndmi_array[retained_input_mask],
                                fire_hazard_ndmi_lower_bound, fire_hazard_ndmi_upper_bound)
                            # Store normalized NDMI values needed to carry out NDMI to
                            # vegetation-dryness hazard.
                            normalized_ndmi_values = (fire_hazard_ndmi_upper_bound - \
                                clipped_ndmi_values) / fire_hazard_ndmi_normalization_range
                            # Store normalized NDMI values needed to carry out NDMI to
                            # vegetation-dryness hazard.
                            normalized_ndmi_values = np.clip(normalized_ndmi_values,
                                fire_hazard_vegetation_dryness_score_minimum, \
                                    fire_hazard_vegetation_dryness_score_maximum)
                            # Build the normalized NDMI array[retained input mask] mask to isolate
                            # pixels that satisfy the current analysis criteria.
                            normalized_ndmi_array[retained_input_mask] = \
                                normalized_ndmi_values.astype(np.float32)
                        # Write the processed data to the configured output resource.
                        normalized_destination.write(normalized_ndmi_array,
                            1, window=processing_window)
                        # Build the valid output mask mask used to isolate records required for this
                        # analysis.
                        valid_output_mask = np.isfinite(normalized_ndmi_array) & \
                            (normalized_ndmi_array != fire_hazard_vegetation_dryness_output_nodata)
                        # Record valid output values so NDMI to vegetation-dryness hazard can
                        # preserve a clear processing and validation outcome.
                        valid_output_values = normalized_ndmi_array[valid_output_mask]
                        # Record NDMI dryness valid pixels so NDMI to vegetation-dryness hazard can
                        # preserve a clear processing and validation outcome.
                        fire_hazard_ndmi_dryness_valid_pixels += int(valid_output_values.size)
                        # Store NDMI dryness NoData pixels needed to carry out NDMI to
                        # vegetation-dryness hazard.
                        fire_hazard_ndmi_dryness_nodata_pixels += int((~valid_output_mask).sum())

                        # Handle the valid output values case explicitly during NDMI to
                        # vegetation-dryness hazard.
                        if valid_output_values.size > 0:
                            # Track window minimum across processed pixels for output-range QA.
                            window_minimum = float(valid_output_values.min())
                            # Track window maximum across processed pixels for output-range QA.
                            window_maximum = float(valid_output_values.max())
                            # Track NDMI dryness minimum across processed pixels for output-range
                            # QA.
                            fire_hazard_ndmi_dryness_minimum = window_minimum if \
                                fire_hazard_ndmi_dryness_minimum is None else \
                                min(fire_hazard_ndmi_dryness_minimum,
                                window_minimum)
                            # Track NDMI dryness maximum across processed pixels for output-range
                            # QA.
                            fire_hazard_ndmi_dryness_maximum = window_maximum if \
                                fire_hazard_ndmi_dryness_maximum is None else \
                                max(fire_hazard_ndmi_dryness_maximum,
                                window_maximum)
                        del ndmi_array
                        del observation_count_array
                        del normalized_ndmi_array
                normalized_destination.set_band_description(1,
                    'Inverse-normalized NDMI vegetation-dryness score')
                normalized_destination.update_tags(COMPONENT_ID='VEGETATION_DRYNESS',
                    SOURCE_INDEX='NDMI', NORMALIZATION_METHOD='inverse_min_max',
                    LOWER_BOUND=fire_hazard_ndmi_lower_bound, \
                        UPPER_BOUND=fire_hazard_ndmi_upper_bound,
                    OUTPUT_MINIMUM=fire_hazard_vegetation_dryness_score_minimum,
                    OUTPUT_MAXIMUM=fire_hazard_vegetation_dryness_score_maximum,
                    MINIMUM_VALID_OBSERVATIONS=fire_hazard_vegetation_dryness_minimum_observations,
                    HAZARD_DIRECTION='Lower NDMI produces higher hazard',
                    TARGET_CRS=fire_hazard_target_crs, TARGET_CELL_SIZE=fire_hazard_cell_size)
    # Record temporary NDMI output valid so NDMI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    temporary_ndmi_output_valid = \
        validate_existing_ndmi_dryness_score(fire_hazard_ndmi_normalization_temporary_path)
    # Record NDMI normalized content valid so NDMI to vegetation-dryness hazard can preserve a clear
    # processing and validation outcome.
    ndmi_normalized_content_valid = fire_hazard_ndmi_dryness_valid_pixels > 0 and \
        fire_hazard_ndmi_dryness_minimum is not None and (fire_hazard_ndmi_dryness_maximum is not \
        None) and (fire_hazard_ndmi_dryness_minimum >= \
        fire_hazard_vegetation_dryness_score_minimum) and (fire_hazard_ndmi_dryness_maximum <= \
        fire_hazard_vegetation_dryness_score_maximum)

    # Stop execution if temporary normalized NDMI dryness raster failed validation.
    if not temporary_ndmi_output_valid or not ndmi_normalized_content_valid:
        raise ValueError(f'The temporary normalized NDMI '
            f'dryness raster failed '
            f'validation.\n\nRaster '
            f'structure valid: '
            f'{temporary_ndmi_output_valid}\n'
            f'Valid pixels: '
            f'{fire_hazard_ndmi_dryness_valid_pixels:,}\n'
            f'Score minimum: '
            f'{fire_hazard_ndmi_dryness_minimum}\n'
            f'Score maximum: '
            f'{fire_hazard_ndmi_dryness_maximum}')
    fire_hazard_ndmi_normalization_temporary_path.replace \
        (fire_hazard_ndmi_normalization_output_path)
# Record NDMI dryness score valid so NDMI to vegetation-dryness hazard can preserve a clear
# processing and validation outcome.
fire_hazard_ndmi_dryness_score_valid = \
    validate_existing_ndmi_dryness_score(fire_hazard_ndmi_normalization_output_path) and \
    fire_hazard_ndmi_dryness_valid_pixels > 0 and (fire_hazard_ndmi_dryness_minimum is not None) \
    and (fire_hazard_ndmi_dryness_maximum is not None) and (fire_hazard_ndmi_dryness_minimum >= \
    fire_hazard_vegetation_dryness_score_minimum) and (fire_hazard_ndmi_dryness_maximum <= \
    fire_hazard_vegetation_dryness_score_maximum)

# Stop execution if final normalized NDMI vegetation-dryness raster failed validation.
if not fire_hazard_ndmi_dryness_score_valid:
    raise ValueError('The final normalized NDMI vegetation-dryness raster failed validation.')
# Define NDMI normalization manifest record to control the inputs and rules used by NDMI to
# vegetation-dryness hazard.
fire_hazard_ndmi_normalization_manifest_record = {'INDEX_NAME': 'NDMI',
    'OUTPUT_TYPE': 'Inverse-normalized dryness score',
    'INPUT_PATH': str(fire_hazard_ndmi_normalization_input_path),
    'OBSERVATION_COUNT_PATH': str(fire_hazard_ndmi_normalization_count_path),
    'OUTPUT_PATH': str(fire_hazard_ndmi_normalization_output_path),
    'PROCESSING_ACTION': fire_hazard_ndmi_normalization_action,
    'NORMALIZATION_METHOD': 'inverse_min_max', 'LOWER_BOUND': fire_hazard_ndmi_lower_bound,
    'UPPER_BOUND': fire_hazard_ndmi_upper_bound, 'MINIMUM_VALID_OBSERVATIONS': \
        fire_hazard_vegetation_dryness_minimum_observations,
    'VALID_PIXELS': fire_hazard_ndmi_dryness_valid_pixels,
    'NODATA_PIXELS': fire_hazard_ndmi_dryness_nodata_pixels,
    'INSUFFICIENT_SUPPORT_PIXELS': fire_hazard_ndmi_insufficient_support_pixels,
    'SCORE_MINIMUM': fire_hazard_ndmi_dryness_minimum,
    'SCORE_MAXIMUM': fire_hazard_ndmi_dryness_maximum,
    'OUTPUT_SIZE_BYTES': fire_hazard_ndmi_normalization_output_path.stat().st_size,
    'OUTPUT_SIZE_MB': fire_hazard_ndmi_normalization_output_path.stat().st_size / 1024 ** 2,
    'VALID': fire_hazard_ndmi_dryness_score_valid}

# Use the existing file only when it is present and valid for NDMI to vegetation-dryness hazard.
if fire_hazard_vegetation_dryness_normalization_manifest_path.exists():
    # Store existing normalization manifest needed to carry out NDMI to vegetation-dryness hazard.
    existing_normalization_manifest = \
        pd.read_csv(fire_hazard_vegetation_dryness_normalization_manifest_path)
else:
    # Store existing normalization manifest needed to carry out NDMI to vegetation-dryness hazard.
    existing_normalization_manifest = pd.DataFrame()

# Handle missing or invalid existing normalization manifest explicitly during NDMI to
# vegetation-dryness hazard.
if not existing_normalization_manifest.empty:
    # Create an isolated existing normalization manifest working copy so NDMI to vegetation-dryness
    # hazard does not modify upstream data.
    existing_normalization_manifest = \
        existing_normalization_manifest[~((existing_normalization_manifest['INDEX_NAME'].astype \
        (str).str.upper() == 'NDMI') & \
        (existing_normalization_manifest['OUTPUT_TYPE'].astype(str) == \
        'Inverse-normalized dryness score'))].copy()
# Store vegetation dryness normalization manifest needed to carry out NDMI to vegetation-dryness
# hazard.
fire_hazard_vegetation_dryness_normalization_manifest = pd.concat([existing_normalization_manifest,
    pd.DataFrame([fire_hazard_ndmi_normalization_manifest_record])],
    ignore_index=True).sort_values(['INDEX_NAME', 'OUTPUT_TYPE']).reset_index(drop=True)
# Save the vegetation dryness normalization manifest CSV so later steps can reuse the recorded
# workflow results.
fire_hazard_vegetation_dryness_normalization_manifest.to_csv \
    (fire_hazard_vegetation_dryness_normalization_manifest_path,
    index=False)
# Store normalized index manifest records needed to carry out NDMI to vegetation-dryness hazard.
normalized_index_manifest_records = \
    fire_hazard_vegetation_dryness_normalization_manifest \
    [fire_hazard_vegetation_dryness_normalization_manifest['OUTPUT_TYPE'].astype(str) == \
    'Inverse-normalized dryness score']
# Collect normalized index names for membership and completeness checks during NDMI to
# vegetation-dryness hazard.
normalized_index_names = \
    set(normalized_index_manifest_records['INDEX_NAME'].astype(str).str.upper())
# Store vegetation dryness normalized indices complete needed to carry out NDMI to
# vegetation-dryness hazard.
fire_hazard_vegetation_dryness_normalized_indices_complete = normalized_index_names == {'NDVI',
    'NDMI'} and normalized_index_manifest_records['VALID'].fillna(False).all()

# Stop execution if normalized vegetation-dryness index pair is incomplete or contains invalid
# manifest records.
if not fire_hazard_vegetation_dryness_normalized_indices_complete:
    raise ValueError('The normalized '
        'vegetation-dryness index pair '
        'is incomplete or contains '
        'invalid manifest records.')
# Store NDMI normalization summary needed to carry out NDMI to vegetation-dryness hazard.
fire_hazard_ndmi_normalization_summary = pd.DataFrame([{'INDEX_NAME': 'NDMI',
    'NORMALIZATION_METHOD': 'Inverse min-max', 'LOWER_BOUND': fire_hazard_ndmi_lower_bound,
    'UPPER_BOUND': fire_hazard_ndmi_upper_bound, 'MINIMUM_VALID_OBSERVATIONS': \
        fire_hazard_vegetation_dryness_minimum_observations,
    'VALID_OUTPUT_PIXELS': fire_hazard_ndmi_dryness_valid_pixels,
    'NODATA_OUTPUT_PIXELS': fire_hazard_ndmi_dryness_nodata_pixels,
    'INSUFFICIENT_SUPPORT_PIXELS': fire_hazard_ndmi_insufficient_support_pixels,
    'SCORE_MINIMUM': fire_hazard_ndmi_dryness_minimum,
    'SCORE_MAXIMUM': fire_hazard_ndmi_dryness_maximum,
    'PROCESSING_ACTION': fire_hazard_ndmi_normalization_action,
    'OUTPUT_PATH': str(fire_hazard_ndmi_normalization_output_path),
    'VALID': fire_hazard_ndmi_dryness_score_valid}])
print(f'-> NDMI normalization bounds: '
    f'{fire_hazard_ndmi_lower_bound:.2f} '
    f'to '
    f'{fire_hazard_ndmi_upper_bound:.2f}')
print(f'-> Processing action: {fire_hazard_ndmi_normalization_action}')
print(f'-> Valid normalized pixels: {fire_hazard_ndmi_dryness_valid_pixels:,}')
print(f'-> Insufficient-support pixels: {fire_hazard_ndmi_insufficient_support_pixels:,}')
print(f'-> NDMI dryness-score range: '
    f'{fire_hazard_ndmi_dryness_minimum:.4f} '
    f'to '
    f'{fire_hazard_ndmi_dryness_maximum:.4f}')
print(f'-> Normalized index pair '
    f'complete: '
    f'{fire_hazard_vegetation_dryness_normalized_indices_complete}')
print(f'-> Normalized NDMI raster saved: {fire_hazard_ndmi_normalization_output_path}')
print(f'-> Normalization manifest '
    f'saved: '
    f'{fire_hazard_vegetation_dryness_normalization_manifest_path}')
print('\n--- NDMI VEGETATION-DRYNESS NORMALIZATION SUMMARY ---')
display(fire_hazard_ndmi_normalization_summary)
print('\n--- VEGETATION-DRYNESS NORMALIZATION MANIFEST SAMPLE ---')
display(fire_hazard_vegetation_dryness_normalization_manifest.head(5))
import gc
gc.collect()
print('\nNOTE:')
print('The NDMI temporal composite '
    'has been converted to a '
    'zero-to-one vegetation-dryness '
    'hazard score.')
print('Lower NDMI values produce '
    'higher scores because they '
    'indicate reduced vegetation or '
    'canopy moisture.')
print('Only pixels supported by the '
    'configured minimum number of '
    'valid HLS observations were '
    'retained.')
print('The normalized NDVI and NDMI '
    'dryness rasters are now ready '
    'to be combined using the '
    'configured weights of 0.40 and '
    '0.60.')
print('\n=== NDMI VEGETATION-DRYNESS NORMALIZATION COMPLETE ===')



=== NORMALIZING NDMI TO VEGETATION-DRYNESS HAZARD ===
-> Valid cached normalized NDMI raster found.
-> NDMI normalization bounds: -0.20 to 0.50
-> Processing action: Reused existing normalized NDMI raster
-> Valid normalized pixels: 14,140,492
-> Insufficient-support pixels: 0
-> NDMI dryness-score range: 0.0000 to 1.0000
-> Normalized index pair complete: True
-> Normalized NDMI raster saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\normalized_rasters\ndmi_inverse_normalized_dryness_score.tif
-> Normalization manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_normalization_manifest.csv

--- NDMI VEGETATION-DRYNESS NORMALIZATION SUMMARY ---


,INDEX_NAME,NORMALIZATION_METHOD,LOWER_BOUND,UPPER_BOUND,MINIMUM_VALID_OBSERVATIONS,VALID_OUTPUT_PIXELS,NODATA_OUTPUT_PIXELS,INSUFFICIENT_SUPPORT_PIXELS,SCORE_MINIMUM,SCORE_MAXIMUM,PROCESSING_ACTION,OUTPUT_PATH,VALID
0,NDMI,Inverse min-max,-0.2,0.5,3,14140492,122630526,0,0.0,1.0,Reused existing normalized NDMI raster,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



--- VEGETATION-DRYNESS NORMALIZATION MANIFEST SAMPLE ---


,INDEX_NAME,OUTPUT_TYPE,INPUT_PATH,OBSERVATION_COUNT_PATH,OUTPUT_PATH,PROCESSING_ACTION,NORMALIZATION_METHOD,LOWER_BOUND,UPPER_BOUND,MINIMUM_VALID_OBSERVATIONS,...,OUTPUT_SIZE_MB,VALID,NDVI_WEIGHT,NDMI_WEIGHT,REQUIRE_BOTH_INDICES,BOTH_INDICES_VALID_PIXELS,NDVI_ONLY_PIXELS,NDMI_ONLY_PIXELS,NEITHER_INDEX_PIXELS,VALID_OUTSIDE_STUDY_AREA
0,COMBINED,Weighted vegetation-dryness hazard score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,NaN,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing combined dryness raster,weighted_linear_combination,0.0,1.0,NaN,...,43.922072,True,0.4,0.6,True,0.0,0.0,0.0,0.0,0.0
1,NDMI,Inverse-normalized dryness score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing normalized NDMI raster,inverse_min_max,-0.2,0.5,3.0,...,43.655200,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NDVI,Inverse-normalized dryness score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing normalized NDVI raster,inverse_min_max,0.1,0.8,3.0,...,44.696409,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



NOTE:
The NDMI temporal composite has been converted to a zero-to-one vegetation-dryness hazard score.
Lower NDMI values produce higher scores because they indicate reduced vegetation or canopy moisture.
Only pixels supported by the configured minimum number of valid HLS observations were retained.
The normalized NDVI and NDMI dryness rasters are now ready to be combined using the configured weights of 0.40 and 0.60.

=== NDMI VEGETATION-DRYNESS NORMALIZATION COMPLETE ===


### Combining Normalized Ndvi and Ndmi Dryness Scores


In [93]:
print('=== COMBINING NORMALIZED NDVI AND NDMI DRYNESS SCORES ===')
# List the required combined dryness prerequisites required before normalized NDVI and NDMI
# dryness scores can run.
required_combined_dryness_inputs = ['fire_hazard_vegetation_dryness_normalized_indices_complete',
    'fire_hazard_ndvi_dryness_score_valid', 'fire_hazard_ndmi_dryness_score_valid',
    'fire_hazard_vegetation_dryness_ndvi_score_path',
    'fire_hazard_vegetation_dryness_ndmi_score_path',
    'fire_hazard_vegetation_dryness_combined_score_path',
    'fire_hazard_vegetation_dryness_index_weights',
    'fire_hazard_vegetation_dryness_score_minimum',
    'fire_hazard_vegetation_dryness_score_maximum',
    'fire_hazard_vegetation_dryness_output_profile',
    'fire_hazard_vegetation_dryness_output_dtype',
    'fire_hazard_vegetation_dryness_output_nodata',
    'fire_hazard_vegetation_dryness_require_both_indices',
    'fire_hazard_vegetation_dryness_window_size', \
        'fire_hazard_vegetation_dryness_progress_interval',
    'fire_hazard_vegetation_dryness_reuse_existing',
    'fire_hazard_vegetation_dryness_overwrite', \
        'fire_hazard_vegetation_dryness_normalization_manifest_path',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size']
# Identify unavailable combined dryness so normalized NDVI and NDMI dryness scores stops
# before using incomplete inputs.
missing_combined_dryness_inputs = [object_name for object_name in \
    required_combined_dryness_inputs if object_name not in globals()]

# Stop execution when required combined dryness inputs inputs are unavailable.
if missing_combined_dryness_inputs:
    raise NameError(f'The following combined '
        f'vegetation-dryness objects are '
        f'missing:\n'
        f'{missing_combined_dryness_inputs}\n\n'
        f'Run Normalize NDVI to '
        f'Vegetation-Dryness Hazard and '
        f'Normalize NDMI to '
        f'Vegetation-Dryness Hazard '
        f'before combining the scores.')

# Stop execution if normalized NDVI and NDMI index pair is not complete.
if not fire_hazard_vegetation_dryness_normalized_indices_complete:
    raise ValueError('The normalized NDVI and NDMI index pair is not complete.')

# Stop execution if normalized NDVI dryness-score raster is not valid.
if not fire_hazard_ndvi_dryness_score_valid:
    raise ValueError('The normalized NDVI dryness-score raster is not valid.')

# Stop execution if normalized NDMI dryness-score raster is not valid.
if not fire_hazard_ndmi_dryness_score_valid:
    raise ValueError('The normalized NDMI dryness-score raster is not valid.')
# Build the combined dryness NDVI path location so normalized NDVI and NDMI dryness scores uses the
# expected project file structure.
fire_hazard_combined_dryness_ndvi_path = Path(fire_hazard_vegetation_dryness_ndvi_score_path)
# Build the combined dryness NDMI path location so normalized NDVI and NDMI dryness scores uses the
# expected project file structure.
fire_hazard_combined_dryness_ndmi_path = Path(fire_hazard_vegetation_dryness_ndmi_score_path)
# Build the combined dryness output path location so normalized NDVI and NDMI dryness scores uses
# the expected project file structure.
fire_hazard_combined_dryness_output_path = Path(fire_hazard_vegetation_dryness_combined_score_path)
# Build the combined dryness temporary path location so normalized NDVI and NDMI dryness scores uses
# the expected project file structure.
fire_hazard_combined_dryness_temporary_path = fire_hazard_combined_dryness_output_path.parent / \
    (fire_hazard_combined_dryness_output_path.stem + '.part.tif')

# Process each (input name entry so normalized NDVI and NDMI dryness scores is applied consistently
# across all records.
for input_name, input_path in {'Normalized NDVI dryness score': \
    fire_hazard_combined_dryness_ndvi_path,
    'Normalized NDMI dryness score': fire_hazard_combined_dryness_ndmi_path}.items():

    # Stop execution if the reported value raster is unavailable: the reported value.
    if not input_path.exists() or not input_path.is_file() or input_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The {input_name} raster is unavailable:\n{input_path}')
# Store combined dryness NDVI weight needed to carry out normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_ndvi_weight = \
    float(fire_hazard_vegetation_dryness_index_weights['NDVI'])
# Store combined dryness NDMI weight needed to carry out normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_ndmi_weight = \
    float(fire_hazard_vegetation_dryness_index_weights['NDMI'])
# Store combined dryness weight total needed to carry out normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_weight_total = fire_hazard_combined_dryness_ndvi_weight + \
    fire_hazard_combined_dryness_ndmi_weight

# Stop execution if vegetation-dryness index weights cannot be negative.
if fire_hazard_combined_dryness_ndvi_weight < 0 or fire_hazard_combined_dryness_ndmi_weight < 0:
    raise ValueError('Vegetation-dryness index weights cannot be negative.')

# Require The NDVI and NDMI combination weights to sum to 1.0.
if not np.isclose(fire_hazard_combined_dryness_weight_total, 1.0):
    raise ValueError(f'The NDVI and NDMI combination '
        f'weights must sum to 1.0.\n'
        f'Current total: '
        f'{fire_hazard_combined_dryness_weight_total}')

# Encapsulate validate existing combined dryness score so repeated normalized NDVI and NDMI dryness
# scores steps use consistent logic.
def validate_existing_combined_dryness_score(output_path):

    """
    Confirms that an existing combined vegetation-
    dryness raster matches the common analysis grid.
    """
    # Build the output path location so normalized NDVI and NDMI dryness scores uses the expected
    # project file structure.
    output_path = Path(output_path)

    # Use the existing file only when it is present and valid for normalized NDVI and NDMI dryness
    # scores.
    if not output_path.exists() or not output_path.is_file() or output_path.stat().st_size <= 0:
        return False

    # Protect normalized NDVI and NDMI dryness scores so expected source or file failures do not
    # leave partial outputs.
    try:

        # Open the raster in a managed context so its file handle closes reliably after normalized
        # NDVI and NDMI dryness scores.
        with rasterio.open(output_path) as output_source:
            return all([output_source.count == 1,
                output_source.width == fire_hazard_alignment_width,
                output_source.height == fire_hazard_alignment_height,
                output_source.crs is not None, output_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                output_source.transform.almost_equals(fire_hazard_alignment_transform),
                output_source.dtypes[0] == fire_hazard_vegetation_dryness_output_dtype,
                output_source.nodata == fire_hazard_vegetation_dryness_output_nodata])
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception:
        return False
# Record combined dryness existing output valid so normalized NDVI and NDMI dryness scores can
# preserve a clear processing and validation outcome.
fire_hazard_combined_dryness_existing_output_valid = \
    validate_existing_combined_dryness_score(fire_hazard_combined_dryness_output_path)
# Store combined dryness reused needed to carry out normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_reused = fire_hazard_vegetation_dryness_reuse_existing and \
    fire_hazard_combined_dryness_existing_output_valid
# Record combined dryness valid pixels so normalized NDVI and NDMI dryness scores can preserve a
# clear processing and validation outcome.
fire_hazard_combined_dryness_valid_pixels = 0
# Set combined dryness NoData pixels to control normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_nodata_pixels = 0
# Track combined dryness minimum across processed pixels for output-range QA.
fire_hazard_combined_dryness_minimum = None
# Track combined dryness maximum across processed pixels for output-range QA.
fire_hazard_combined_dryness_maximum = None
# Set combined dryness both indices pixels to control normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_both_indices_pixels = 0
# Set combined dryness NDVI only pixels to control normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_ndvi_only_pixels = 0
# Set combined dryness NDMI only pixels to control normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_ndmi_only_pixels = 0
# Set combined dryness neither index pixels to control normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_neither_index_pixels = 0
# Record combined dryness valid outside study area so normalized NDVI and NDMI dryness scores can
# preserve a clear processing and validation outcome.
fire_hazard_combined_dryness_valid_outside_study_area = 0
# Record combined dryness processing action so normalized NDVI and NDMI dryness scores can preserve
# a clear processing and validation outcome.
fire_hazard_combined_dryness_processing_action = None

# Handle the combined dryness reused case explicitly during normalized NDVI and NDMI dryness scores.
if fire_hazard_combined_dryness_reused:
    # Record combined dryness processing action so normalized NDVI and NDMI dryness scores can
    # preserve a clear processing and validation outcome.
    fire_hazard_combined_dryness_processing_action = 'Reused existing combined dryness raster'
    print('-> Valid cached combined vegetation-dryness raster found.')

    # Open the raster in a managed context so its file handle closes reliably after normalized NDVI
    # and NDMI dryness scores.
    with rasterio.open(fire_hazard_combined_dryness_output_path) as combined_source:

        # Process each ( entry so normalized NDVI and NDMI dryness scores is applied consistently
        # across all records.
        for _, source_window in combined_source.block_windows(1):
            # Prepare combined array used to process the current raster window.
            combined_array = combined_source.read(1, window=source_window)
            # Calculate row start for the current raster processing window.
            row_start = int(source_window.row_off)
            # Store row end needed to carry out normalized NDVI and NDMI dryness scores.
            row_end = int(source_window.row_off + source_window.height)
            # Store column start needed to carry out normalized NDVI and NDMI dryness scores.
            column_start = int(source_window.col_off)
            # Store column end needed to carry out normalized NDVI and NDMI dryness scores.
            column_end = int(source_window.col_off + source_window.width)
            # Build the study area window mask mask used to isolate records required for this
            # analysis.
            study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                column_start:column_end]
            # Build the valid combined mask mask used to isolate records required for this analysis.
            valid_combined_mask = np.isfinite(combined_array) & (combined_array != \
                fire_hazard_vegetation_dryness_output_nodata)
            # Record valid combined values so normalized NDVI and NDMI dryness scores can preserve a
            # clear processing and validation outcome.
            valid_combined_values = combined_array[valid_combined_mask]
            # Record combined dryness valid pixels so normalized NDVI and NDMI dryness scores can
            # preserve a clear processing and validation outcome.
            fire_hazard_combined_dryness_valid_pixels += int(valid_combined_values.size)
            # Store combined dryness NoData pixels needed to carry out normalized NDVI and NDMI
            # dryness scores.
            fire_hazard_combined_dryness_nodata_pixels += int((~valid_combined_mask).sum())
            # Record combined dryness valid outside study area so normalized NDVI and NDMI dryness
            # scores can preserve a clear processing and validation outcome.
            fire_hazard_combined_dryness_valid_outside_study_area += int((valid_combined_mask & \
                ~study_area_window_mask).sum())

            # Handle the valid combined values case explicitly during normalized NDVI and NDMI
            # dryness scores.
            if valid_combined_values.size > 0:
                # Track window minimum across processed pixels for output-range QA.
                window_minimum = float(valid_combined_values.min())
                # Track window maximum across processed pixels for output-range QA.
                window_maximum = float(valid_combined_values.max())
                # Track combined dryness minimum across processed pixels for output-range QA.
                fire_hazard_combined_dryness_minimum = window_minimum if \
                    fire_hazard_combined_dryness_minimum is None else \
                    min(fire_hazard_combined_dryness_minimum,
                    window_minimum)
                # Track combined dryness maximum across processed pixels for output-range QA.
                fire_hazard_combined_dryness_maximum = window_maximum if \
                    fire_hazard_combined_dryness_maximum is None else \
                    max(fire_hazard_combined_dryness_maximum,
                    window_maximum)
else:
    # Record combined dryness processing action so normalized NDVI and NDMI dryness scores can
    # preserve a clear processing and validation outcome.
    fire_hazard_combined_dryness_processing_action = 'Created weighted combined dryness raster'

    # Use the existing file only when it is present and valid for normalized NDVI and NDMI dryness
    # scores.
    if fire_hazard_combined_dryness_temporary_path.exists():
        fire_hazard_combined_dryness_temporary_path.unlink()

    # Protect existing outputs unless the configured overwrite policy permits replacement.
    if fire_hazard_combined_dryness_output_path.exists() and \
        fire_hazard_vegetation_dryness_overwrite:
        fire_hazard_combined_dryness_output_path.unlink()
    from rasterio.windows import Window

    # Open the raster in a managed context so its file handle closes reliably after normalized NDVI
    # and NDMI dryness scores.
    with rasterio.open(fire_hazard_combined_dryness_ndvi_path) as ndvi_source:

        # Open the raster in a managed context so its file handle closes reliably after normalized
        # NDVI and NDMI dryness scores.
        with rasterio.open(fire_hazard_combined_dryness_ndmi_path) as ndmi_source:
            # Record combined source grids valid so normalized NDVI and NDMI dryness scores can
            # preserve a clear processing and validation outcome.
            combined_source_grids_valid = all([ndvi_source.count == 1,
                ndmi_source.count == 1, ndvi_source.width == fire_hazard_alignment_width,
                ndvi_source.height == fire_hazard_alignment_height,
                ndmi_source.width == fire_hazard_alignment_width,
                ndmi_source.height == fire_hazard_alignment_height,
                ndvi_source.crs is not None, ndmi_source.crs is not None,
                ndvi_source.crs == ndmi_source.crs, ndvi_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                ndvi_source.transform.almost_equals(fire_hazard_alignment_transform),
                ndmi_source.transform.almost_equals(fire_hazard_alignment_transform),
                ndvi_source.dtypes[0] == fire_hazard_vegetation_dryness_output_dtype,
                ndmi_source.dtypes[0] == fire_hazard_vegetation_dryness_output_dtype,
                ndvi_source.nodata == fire_hazard_vegetation_dryness_output_nodata,
                ndmi_source.nodata == fire_hazard_vegetation_dryness_output_nodata])

            # Stop execution if normalized NDVI and NDMI rasters do not match the common fire-hazard
            # analysis grid.
            if not combined_source_grids_valid:
                raise ValueError('The normalized NDVI and NDMI '
                    'rasters do not match the '
                    'common fire-hazard analysis '
                    'grid.')

            # Open the raster in a managed context so its file handle closes reliably after
            # normalized NDVI and NDMI dryness scores.
            with rasterio.open(fire_hazard_combined_dryness_temporary_path,
                'w', **fire_hazard_vegetation_dryness_output_profile) as combined_destination:
                # Calculate window column count to quantify completeness and support QA checks.
                window_column_count = int(np.ceil(fire_hazard_alignment_width / \
                    fire_hazard_vegetation_dryness_window_size))
                # Calculate window row count to quantify completeness and support QA checks.
                window_row_count = int(np.ceil(fire_hazard_alignment_height / \
                    fire_hazard_vegetation_dryness_window_size))
                # Store total processing windows needed to carry out normalized NDVI and NDMI
                # dryness scores.
                total_processing_windows = window_column_count * window_row_count
                # Set current window number to control normalized NDVI and NDMI dryness scores.
                current_window_number = 0

                # Process each window row entry so normalized NDVI and NDMI dryness scores is
                # applied consistently across all records.
                for window_row in range(window_row_count):
                    # Calculate row offset for the current raster processing window.
                    row_offset = window_row * fire_hazard_vegetation_dryness_window_size
                    # Store window height needed to carry out normalized NDVI and NDMI dryness
                    # scores.
                    window_height = min(fire_hazard_vegetation_dryness_window_size,
                        fire_hazard_alignment_height - row_offset)

                    # Process each window column entry so normalized NDVI and NDMI dryness scores is
                    # applied consistently across all records.
                    for window_column in range(window_column_count):
                        # Calculate column offset for the current raster processing window.
                        column_offset = window_column * fire_hazard_vegetation_dryness_window_size
                        # Store window width needed to carry out normalized NDVI and NDMI dryness
                        # scores.
                        window_width = min(fire_hazard_vegetation_dryness_window_size,
                            fire_hazard_alignment_width - column_offset)
                        # Set current window number to control normalized NDVI and NDMI dryness
                        # scores.
                        current_window_number += 1

                        # Report progress at the configured interval without changing
                        # raster-processing results.
                        if current_window_number == 1 or current_window_number % \
                            fire_hazard_vegetation_dryness_progress_interval == 0 or \
                            current_window_number == total_processing_windows:
                            print(f'-> Combining '
                                f'vegetation-dryness window '
                                f'{current_window_number:,} of '
                                f'{total_processing_windows:,}...')
                        # Store processing window needed to carry out normalized NDVI and NDMI
                        # dryness scores.
                        processing_window = Window(col_off=column_offset,
                            row_off=row_offset, width=window_width, height=window_height)
                        # Prepare NDVI score array used to process the current raster window.
                        ndvi_score_array = ndvi_source.read(1,
                            window=processing_window).astype(np.float32)
                        # Prepare NDMI score array used to process the current raster window.
                        ndmi_score_array = ndmi_source.read(1,
                            window=processing_window).astype(np.float32)
                        # Build the study area window mask mask used to isolate records required for
                        # this analysis.
                        study_area_window_mask = \
                            fire_hazard_alignment_study_area_mask[row_offset:row_offset + \
                            window_height,
                            column_offset:column_offset + window_width]
                        # Build the valid ndvi score mask mask used to isolate records required for
                        # this analysis.
                        valid_ndvi_score_mask = np.isfinite(ndvi_score_array) & (ndvi_score_array \
                            != fire_hazard_vegetation_dryness_output_nodata) & \
                            study_area_window_mask
                        # Build the valid ndmi score mask mask used to isolate records required for
                        # this analysis.
                        valid_ndmi_score_mask = np.isfinite(ndmi_score_array) & (ndmi_score_array \
                            != fire_hazard_vegetation_dryness_output_nodata) & \
                            study_area_window_mask
                        # Build the both indices valid mask mask used to isolate records required
                        # for this analysis.
                        both_indices_valid_mask = valid_ndvi_score_mask & valid_ndmi_score_mask
                        # Build the ndvi only mask mask used to isolate records required for this
                        # analysis.
                        ndvi_only_mask = valid_ndvi_score_mask & ~valid_ndmi_score_mask
                        # Build the ndmi only mask mask used to isolate records required for this
                        # analysis.
                        ndmi_only_mask = valid_ndmi_score_mask & ~valid_ndvi_score_mask
                        # Build the neither index valid mask mask used to isolate records required
                        # for this analysis.
                        neither_index_valid_mask = study_area_window_mask & \
                            ~valid_ndvi_score_mask & ~valid_ndmi_score_mask
                        # Store combined dryness both indices pixels needed to carry out normalized
                        # NDVI and NDMI dryness scores.
                        fire_hazard_combined_dryness_both_indices_pixels += \
                            int(both_indices_valid_mask.sum())
                        # Store combined dryness NDVI only pixels needed to carry out normalized
                        # NDVI and NDMI dryness scores.
                        fire_hazard_combined_dryness_ndvi_only_pixels += int(ndvi_only_mask.sum())
                        # Store combined dryness NDMI only pixels needed to carry out normalized
                        # NDVI and NDMI dryness scores.
                        fire_hazard_combined_dryness_ndmi_only_pixels += int(ndmi_only_mask.sum())
                        # Store combined dryness neither index pixels needed to carry out normalized
                        # NDVI and NDMI dryness scores.
                        fire_hazard_combined_dryness_neither_index_pixels += \
                            int(neither_index_valid_mask.sum())
                        # Prepare combined dryness array used to process the current raster window.
                        combined_dryness_array = np.full((int(processing_window.height),
                            int(processing_window.width)), \
                                fire_hazard_vegetation_dryness_output_nodata,
                            dtype=np.float32)

                        # Handle the vegetation dryness require both indices case explicitly during
                        # normalized NDVI and NDMI dryness scores.
                        if fire_hazard_vegetation_dryness_require_both_indices:
                            # Build the retained combined mask mask used to isolate records required
                            # for this analysis.
                            retained_combined_mask = both_indices_valid_mask

                            # Handle the retained combined mask case explicitly during normalized
                            # NDVI and NDMI dryness scores.
                            if retained_combined_mask.any():
                                # Store combined values needed to carry out normalized NDVI and NDMI
                                # dryness scores.
                                combined_values = fire_hazard_combined_dryness_ndvi_weight * \
                                    ndvi_score_array[retained_combined_mask] + \
                                    fire_hazard_combined_dryness_ndmi_weight * \
                                    ndmi_score_array[retained_combined_mask]
                                # Build the combined dryness array[retained combined mask] mask to
                                # isolate pixels that satisfy the current analysis criteria.
                                combined_dryness_array[retained_combined_mask] = \
                                    np.clip(combined_values,
                                    fire_hazard_vegetation_dryness_score_minimum, \
                                        fire_hazard_vegetation_dryness_score_maximum).astype(np \
                                        .float32)
                        else:
                            # Build the any index valid mask mask used to isolate records required
                            # for this analysis.
                            any_index_valid_mask = valid_ndvi_score_mask | valid_ndmi_score_mask
                            # Build the retained combined mask mask used to isolate records required
                            # for this analysis.
                            retained_combined_mask = any_index_valid_mask
                            # Store local weight total needed to carry out normalized NDVI and NDMI
                            # dryness scores.
                            local_weight_total = valid_ndvi_score_mask.astype(np.float32) * \
                                fire_hazard_combined_dryness_ndvi_weight + \
                                valid_ndmi_score_mask.astype(np.float32) * \
                                fire_hazard_combined_dryness_ndmi_weight
                            # Store weighted score sum needed to carry out normalized NDVI and NDMI
                            # dryness scores.
                            weighted_score_sum = np.where(valid_ndvi_score_mask,
                                ndvi_score_array * fire_hazard_combined_dryness_ndvi_weight,
                                0.0) + np.where(valid_ndmi_score_mask, ndmi_score_array * \
                                    fire_hazard_combined_dryness_ndmi_weight,
                                0.0)
                            # Build the valid local weight mask mask used to isolate records
                            # required for this analysis.
                            valid_local_weight_mask = retained_combined_mask & \
                                (local_weight_total > 0)
                            # Build the combined dryness array[valid local weight mask] mask to
                            # isolate pixels that satisfy the current analysis criteria.
                            combined_dryness_array[valid_local_weight_mask] = \
                                np.clip(weighted_score_sum[valid_local_weight_mask] / \
                                local_weight_total[valid_local_weight_mask],
                                fire_hazard_vegetation_dryness_score_minimum, \
                                    fire_hazard_vegetation_dryness_score_maximum).astype(np.float32)
                        # Write the processed data to the configured output resource.
                        combined_destination.write(combined_dryness_array,
                            1, window=processing_window)
                        # Build the valid combined mask mask used to isolate records required for
                        # this analysis.
                        valid_combined_mask = np.isfinite(combined_dryness_array) & \
                            (combined_dryness_array != fire_hazard_vegetation_dryness_output_nodata)
                        # Record valid combined values so normalized NDVI and NDMI dryness scores
                        # can preserve a clear processing and validation outcome.
                        valid_combined_values = combined_dryness_array[valid_combined_mask]
                        # Record combined dryness valid pixels so normalized NDVI and NDMI dryness
                        # scores can preserve a clear processing and validation outcome.
                        fire_hazard_combined_dryness_valid_pixels += int(valid_combined_values.size)
                        # Store combined dryness NoData pixels needed to carry out normalized NDVI
                        # and NDMI dryness scores.
                        fire_hazard_combined_dryness_nodata_pixels += \
                            int((~valid_combined_mask).sum())
                        # Record combined dryness valid outside study area so normalized NDVI and
                        # NDMI dryness scores can preserve a clear processing and validation
                        # outcome.
                        fire_hazard_combined_dryness_valid_outside_study_area += \
                            int((valid_combined_mask & ~study_area_window_mask).sum())

                        # Handle the valid combined values case explicitly during normalized NDVI
                        # and NDMI dryness scores.
                        if valid_combined_values.size > 0:
                            # Track window minimum across processed pixels for output-range QA.
                            window_minimum = float(valid_combined_values.min())
                            # Track window maximum across processed pixels for output-range QA.
                            window_maximum = float(valid_combined_values.max())
                            # Track combined dryness minimum across processed pixels for
                            # output-range QA.
                            fire_hazard_combined_dryness_minimum = window_minimum if \
                                fire_hazard_combined_dryness_minimum is None else \
                                min(fire_hazard_combined_dryness_minimum,
                                window_minimum)
                            # Track combined dryness maximum across processed pixels for
                            # output-range QA.
                            fire_hazard_combined_dryness_maximum = window_maximum if \
                                fire_hazard_combined_dryness_maximum is None else \
                                max(fire_hazard_combined_dryness_maximum,
                                window_maximum)
                        del ndvi_score_array
                        del ndmi_score_array
                        del combined_dryness_array
                combined_destination.set_band_description(1,
                    'Combined vegetation-dryness hazard score')
                combined_destination.update_tags(COMPONENT_ID='VEGETATION_DRYNESS',
                    SOURCE_INDICES='NDVI,NDMI', COMBINATION_METHOD='weighted_linear_combination',
                    NDVI_WEIGHT=fire_hazard_combined_dryness_ndvi_weight,
                    NDMI_WEIGHT=fire_hazard_combined_dryness_ndmi_weight,
                    REQUIRE_BOTH_INDICES=str(fire_hazard_vegetation_dryness_require_both_indices),
                    OUTPUT_MINIMUM=fire_hazard_vegetation_dryness_score_minimum,
                    OUTPUT_MAXIMUM=fire_hazard_vegetation_dryness_score_maximum,
                    HAZARD_DIRECTION='Higher score represents greater vegetation dryness',
                    TARGET_CRS=fire_hazard_target_crs, TARGET_CELL_SIZE=fire_hazard_cell_size)
    # Record temporary combined output valid so normalized NDVI and NDMI dryness scores can preserve
    # a clear processing and validation outcome.
    temporary_combined_output_valid = \
        validate_existing_combined_dryness_score(fire_hazard_combined_dryness_temporary_path)
    # Record combined dryness content valid so normalized NDVI and NDMI dryness scores can preserve
    # a clear processing and validation outcome.
    combined_dryness_content_valid = fire_hazard_combined_dryness_valid_pixels > 0 and \
        fire_hazard_combined_dryness_minimum is not None and \
        (fire_hazard_combined_dryness_maximum is not None) and \
        (fire_hazard_combined_dryness_minimum >= fire_hazard_vegetation_dryness_score_minimum) \
        and (fire_hazard_combined_dryness_maximum <= \
        fire_hazard_vegetation_dryness_score_maximum) and \
        (fire_hazard_combined_dryness_valid_outside_study_area == 0)

    # Stop execution if temporary combined vegetation-dryness raster failed validation.
    if not temporary_combined_output_valid or not combined_dryness_content_valid:
        raise ValueError(f'The temporary combined '
            f'vegetation-dryness raster '
            f'failed validation.\n\nRaster '
            f'structure valid: '
            f'{temporary_combined_output_valid}\n'
            f'Valid pixels: '
            f'{fire_hazard_combined_dryness_valid_pixels:,}\n'
            f'Score minimum: '
            f'{fire_hazard_combined_dryness_minimum}\n'
            f'Score maximum: '
            f'{fire_hazard_combined_dryness_maximum}\n'
            f'Valid pixels outside study '
            f'area: '
            f'{fire_hazard_combined_dryness_valid_outside_study_area:,}')
    fire_hazard_combined_dryness_temporary_path.replace(fire_hazard_combined_dryness_output_path)
# Record combined dryness score valid so normalized NDVI and NDMI dryness scores can preserve a
# clear processing and validation outcome.
fire_hazard_combined_dryness_score_valid = \
    validate_existing_combined_dryness_score(fire_hazard_combined_dryness_output_path) and \
    fire_hazard_combined_dryness_valid_pixels > 0 and (fire_hazard_combined_dryness_minimum is \
    not None) and (fire_hazard_combined_dryness_maximum is not None) and \
    (fire_hazard_combined_dryness_minimum >= fire_hazard_vegetation_dryness_score_minimum) and \
    (fire_hazard_combined_dryness_maximum <= fire_hazard_vegetation_dryness_score_maximum) and \
    (fire_hazard_combined_dryness_valid_outside_study_area == 0)

# Stop execution if final combined vegetation-dryness hazard raster failed validation.
if not fire_hazard_combined_dryness_score_valid:
    raise ValueError('The final combined vegetation-dryness hazard raster failed validation.')
# Define combined dryness manifest record to control the inputs and rules used by normalized NDVI
# and NDMI dryness scores.
fire_hazard_combined_dryness_manifest_record = (
    {'INDEX_NAME': 'COMBINED', 'OUTPUT_TYPE': 'Weighted vegetation-dryness hazard score',
        'INPUT_PATH': f'{fire_hazard_combined_dryness_ndvi_path};'
        f' '
        f'{fire_hazard_combined_dryness_ndmi_path}', 'OBSERVATION_COUNT_PATH': None, \
            'OUTPUT_PATH': str(fire_hazard_combined_dryness_output_path), 'PROCESSING_ACTION': \
            fire_hazard_combined_dryness_processing_action, 'NORMALIZATION_METHOD': \
            'weighted_linear_combination', 'LOWER_BOUND': \
            fire_hazard_vegetation_dryness_score_minimum, 'UPPER_BOUND': \
            fire_hazard_vegetation_dryness_score_maximum, 'MINIMUM_VALID_OBSERVATIONS': None, \
            'VALID_PIXELS': fire_hazard_combined_dryness_valid_pixels, 'NODATA_PIXELS': \
            fire_hazard_combined_dryness_nodata_pixels, 'INSUFFICIENT_SUPPORT_PIXELS': \
            fire_hazard_combined_dryness_ndvi_only_pixels + \
            fire_hazard_combined_dryness_ndmi_only_pixels + \
            fire_hazard_combined_dryness_neither_index_pixels, 'SCORE_MINIMUM': \
            fire_hazard_combined_dryness_minimum, 'SCORE_MAXIMUM': \
            fire_hazard_combined_dryness_maximum, 'NDVI_WEIGHT': \
            fire_hazard_combined_dryness_ndvi_weight, 'NDMI_WEIGHT': \
            fire_hazard_combined_dryness_ndmi_weight, 'REQUIRE_BOTH_INDICES': \
            fire_hazard_vegetation_dryness_require_both_indices, 'BOTH_INDICES_VALID_PIXELS': \
            fire_hazard_combined_dryness_both_indices_pixels, 'NDVI_ONLY_PIXELS': \
            fire_hazard_combined_dryness_ndvi_only_pixels, 'NDMI_ONLY_PIXELS': \
            fire_hazard_combined_dryness_ndmi_only_pixels, 'NEITHER_INDEX_PIXELS': \
            fire_hazard_combined_dryness_neither_index_pixels, 'VALID_OUTSIDE_STUDY_AREA': \
            fire_hazard_combined_dryness_valid_outside_study_area, 'OUTPUT_SIZE_BYTES': \
            fire_hazard_combined_dryness_output_path.stat().st_size, 'OUTPUT_SIZE_MB': \
            fire_hazard_combined_dryness_output_path.stat().st_size / 1024 ** 2, 'VALID': \
            fire_hazard_combined_dryness_score_valid}
)

# Use the existing file only when it is present and valid for normalized NDVI and NDMI dryness
# scores.
if fire_hazard_vegetation_dryness_normalization_manifest_path.exists():
    # Store existing normalization manifest needed to carry out normalized NDVI and NDMI dryness
    # scores.
    existing_normalization_manifest = \
        pd.read_csv(fire_hazard_vegetation_dryness_normalization_manifest_path)
else:
    # Store existing normalization manifest needed to carry out normalized NDVI and NDMI dryness
    # scores.
    existing_normalization_manifest = pd.DataFrame()

# Handle missing or invalid existing normalization manifest explicitly during normalized NDVI and
# NDMI dryness scores.
if not existing_normalization_manifest.empty:
    # Create an isolated existing normalization manifest working copy so normalized NDVI and NDMI
    # dryness scores does not modify upstream data.
    existing_normalization_manifest = \
        existing_normalization_manifest[~(existing_normalization_manifest['INDEX_NAME'].astype \
        (str).str.upper() == 'COMBINED')].copy()
# Store vegetation dryness normalization manifest needed to carry out normalized NDVI and NDMI
# dryness scores.
fire_hazard_vegetation_dryness_normalization_manifest = pd.concat([existing_normalization_manifest,
    pd.DataFrame([fire_hazard_combined_dryness_manifest_record])],
    ignore_index=True, sort=False).reset_index(drop=True)
# Save the vegetation dryness normalization manifest CSV so later steps can reuse the recorded
# workflow results.
fire_hazard_vegetation_dryness_normalization_manifest.to_csv \
    (fire_hazard_vegetation_dryness_normalization_manifest_path,
    index=False)
# List the required combined manifest indices prerequisites required before normalized NDVI and NDMI
# dryness scores can run.
required_combined_manifest_indices = {'NDVI', 'NDMI', 'COMBINED'}
# Collect actual combined manifest indices for membership and completeness checks during normalized
# NDVI and NDMI dryness scores.
actual_combined_manifest_indices = \
    set(fire_hazard_vegetation_dryness_normalization_manifest['INDEX_NAME'].astype(str).str.upper())
# Store vegetation dryness combination complete needed to carry out normalized NDVI and NDMI dryness
# scores.
fire_hazard_vegetation_dryness_combination_complete = \
    required_combined_manifest_indices.issubset(actual_combined_manifest_indices) and \
    fire_hazard_combined_dryness_score_valid

# Stop execution if vegetation-dryness normalization and combination manifest is incomplete.
if not fire_hazard_vegetation_dryness_combination_complete:
    raise ValueError('The vegetation-dryness normalization and combination manifest is incomplete.')
# Store combined dryness summary needed to carry out normalized NDVI and NDMI dryness scores.
fire_hazard_combined_dryness_summary = pd.DataFrame([{'COMPONENT_ID': 'VEGETATION_DRYNESS',
    'COMBINATION_METHOD': 'Weighted linear combination',
    'NDVI_WEIGHT': fire_hazard_combined_dryness_ndvi_weight,
    'NDMI_WEIGHT': fire_hazard_combined_dryness_ndmi_weight,
    'REQUIRE_BOTH_INDICES': fire_hazard_vegetation_dryness_require_both_indices,
    'VALID_OUTPUT_PIXELS': fire_hazard_combined_dryness_valid_pixels,
    'NODATA_OUTPUT_PIXELS': fire_hazard_combined_dryness_nodata_pixels,
    'BOTH_INDICES_VALID_PIXELS': fire_hazard_combined_dryness_both_indices_pixels,
    'NDVI_ONLY_PIXELS': fire_hazard_combined_dryness_ndvi_only_pixels,
    'NDMI_ONLY_PIXELS': fire_hazard_combined_dryness_ndmi_only_pixels,
    'NEITHER_INDEX_PIXELS': fire_hazard_combined_dryness_neither_index_pixels,
    'SCORE_MINIMUM': fire_hazard_combined_dryness_minimum,
    'SCORE_MAXIMUM': fire_hazard_combined_dryness_maximum,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_combined_dryness_valid_outside_study_area,
    'PROCESSING_ACTION': fire_hazard_combined_dryness_processing_action,
    'OUTPUT_PATH': str(fire_hazard_combined_dryness_output_path),
    'VALID': fire_hazard_combined_dryness_score_valid}])
print(f'-> NDVI weight: {fire_hazard_combined_dryness_ndvi_weight:.2f}')
print(f'-> NDMI weight: {fire_hazard_combined_dryness_ndmi_weight:.2f}')
print(f'-> Require both indices: {fire_hazard_vegetation_dryness_require_both_indices}')
print(f'-> Processing action: {fire_hazard_combined_dryness_processing_action}')
print(f'-> Valid combined pixels: {fire_hazard_combined_dryness_valid_pixels:,}')
print(f'-> Pixels with both indices: {fire_hazard_combined_dryness_both_indices_pixels:,}')
print(f'-> NDVI-only pixels: {fire_hazard_combined_dryness_ndvi_only_pixels:,}')
print(f'-> NDMI-only pixels: {fire_hazard_combined_dryness_ndmi_only_pixels:,}')
print(f'-> Combined dryness-score '
    f'range: '
    f'{fire_hazard_combined_dryness_minimum:.4f} '
    f'to '
    f'{fire_hazard_combined_dryness_maximum:.4f}')
print(f'-> Valid pixels outside study '
    f'area: '
    f'{fire_hazard_combined_dryness_valid_outside_study_area:,}')
print(f'-> Combined raster saved: {fire_hazard_combined_dryness_output_path}')
print(f'-> Component combination complete: {fire_hazard_vegetation_dryness_combination_complete}')
print('\n--- COMBINED VEGETATION-DRYNESS HAZARD SUMMARY ---')
display(fire_hazard_combined_dryness_summary)
print('\n--- VEGETATION-DRYNESS NORMALIZATION MANIFEST SAMPLE ---')
display(fire_hazard_vegetation_dryness_normalization_manifest.head(5))
import gc
gc.collect()
print('\nNOTE:')
print('The normalized NDVI and NDMI '
    'dryness scores have been '
    'combined using the configured '
    'weights.')
print('The final raster uses a '
    'zero-to-one scale where higher '
    'values represent greater '
    'vegetation-dryness hazard.')
print('Under the current support '
    'policy, a final score is '
    'retained only where both '
    'normalized indices contain '
    'valid values.')
print('The next step is to '
    'independently validate the '
    'normalized NDVI, normalized '
    'NDMI, and combined '
    'vegetation-dryness hazard '
    'rasters.')
print('\n=== NORMALIZED NDVI AND NDMI COMBINATION COMPLETE ===')



=== COMBINING NORMALIZED NDVI AND NDMI DRYNESS SCORES ===
-> Valid cached combined vegetation-dryness raster found.
-> NDVI weight: 0.40
-> NDMI weight: 0.60
-> Require both indices: True
-> Processing action: Reused existing combined dryness raster
-> Valid combined pixels: 14,140,492
-> Pixels with both indices: 0
-> NDVI-only pixels: 0
-> NDMI-only pixels: 0
-> Combined dryness-score range: 0.0000 to 1.0000
-> Valid pixels outside study area: 0
-> Combined raster saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\hazard_raster\vegetation_dryness_hazard_score.tif
-> Component combination complete: True

--- COMBINED VEGETATION-DRYNESS HAZARD SUMMARY ---


,COMPONENT_ID,COMBINATION_METHOD,NDVI_WEIGHT,NDMI_WEIGHT,REQUIRE_BOTH_INDICES,VALID_OUTPUT_PIXELS,NODATA_OUTPUT_PIXELS,BOTH_INDICES_VALID_PIXELS,NDVI_ONLY_PIXELS,NDMI_ONLY_PIXELS,NEITHER_INDEX_PIXELS,SCORE_MINIMUM,SCORE_MAXIMUM,VALID_OUTSIDE_STUDY_AREA,PROCESSING_ACTION,OUTPUT_PATH,VALID
0,VEGETATION_DRYNESS,Weighted linear combination,0.4,0.6,True,14140492,122630526,0,0,0,0,0.0,1.0,0,Reused existing combined dryness raster,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



--- VEGETATION-DRYNESS NORMALIZATION MANIFEST SAMPLE ---


,INDEX_NAME,OUTPUT_TYPE,INPUT_PATH,OBSERVATION_COUNT_PATH,OUTPUT_PATH,PROCESSING_ACTION,NORMALIZATION_METHOD,LOWER_BOUND,UPPER_BOUND,MINIMUM_VALID_OBSERVATIONS,...,OUTPUT_SIZE_MB,VALID,NDVI_WEIGHT,NDMI_WEIGHT,REQUIRE_BOTH_INDICES,BOTH_INDICES_VALID_PIXELS,NDVI_ONLY_PIXELS,NDMI_ONLY_PIXELS,NEITHER_INDEX_PIXELS,VALID_OUTSIDE_STUDY_AREA
0,NDMI,Inverse-normalized dryness score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing normalized NDMI raster,inverse_min_max,-0.2,0.5,3.0,...,43.655200,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NDVI,Inverse-normalized dryness score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing normalized NDVI raster,inverse_min_max,0.1,0.8,3.0,...,44.696409,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,COMBINED,Weighted vegetation-dryness hazard score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing combined dryness raster,weighted_linear_combination,0.0,1.0,None,...,43.922072,True,0.4,0.6,True,0.0,0.0,0.0,0.0,0.0



NOTE:
The normalized NDVI and NDMI dryness scores have been combined using the configured weights.
The final raster uses a zero-to-one scale where higher values represent greater vegetation-dryness hazard.
Under the current support policy, a final score is retained only where both normalized indices contain valid values.
The next step is to independently validate the normalized NDVI, normalized NDMI, and combined vegetation-dryness hazard rasters.

=== NORMALIZED NDVI AND NDMI COMBINATION COMPLETE ===


### Validating Vegetation-Dryness Hazard Outputs


In [94]:
print('=== VALIDATING VEGETATION-DRYNESS HAZARD OUTPUTS ===')
# List the required vegetation dryness validation prerequisites required before
# vegetation-dryness hazard outputs can run.
required_vegetation_dryness_validation_inputs = ['fire_hazard_vegetation_dryness_ndvi_score_path',
    'fire_hazard_vegetation_dryness_ndmi_score_path',
    'fire_hazard_vegetation_dryness_combined_score_path',
    'fire_hazard_vegetation_dryness_normalization_manifest_path',
    'fire_hazard_vegetation_dryness_validation_path',
    'fire_hazard_vegetation_dryness_validation_summary_path',
    'fire_hazard_vegetation_dryness_index_weights',
    'fire_hazard_vegetation_dryness_require_both_indices',
    'fire_hazard_vegetation_dryness_score_minimum',
    'fire_hazard_vegetation_dryness_score_maximum',
    'fire_hazard_vegetation_dryness_output_nodata',
    'fire_hazard_vegetation_dryness_output_dtype',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size']
# Identify unavailable vegetation dryness validation so vegetation-dryness hazard outputs
# stops before using incomplete inputs.
missing_vegetation_dryness_validation_inputs = [object_name for object_name in \
    required_vegetation_dryness_validation_inputs if object_name not in globals()]

# Stop execution when required vegetation dryness validation inputs inputs are unavailable.
if missing_vegetation_dryness_validation_inputs:
    raise NameError(f'The following '
        f'vegetation-dryness validation '
        f'objects are missing:\n'
        f'{missing_vegetation_dryness_validation_inputs}\n\n'
        f'Run Normalize NDVI, Normalize '
        f'NDMI, and Combine Normalized '
        f'NDVI and NDMI before '
        f'validating the '
        f'vegetation-dryness outputs.')
# Store vegetation dryness expected products needed to carry out vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_expected_products = pd.DataFrame([{'PRODUCT_ID': \
    'NDVI_DRYNESS_SCORE',
    'PRODUCT_TYPE': 'Inverse-normalized index score',
    'INDEX_NAME': 'NDVI', 'OUTPUT_PATH': str(fire_hazard_vegetation_dryness_ndvi_score_path)},
    {'PRODUCT_ID': 'NDMI_DRYNESS_SCORE', 'PRODUCT_TYPE': 'Inverse-normalized index score',
    'INDEX_NAME': 'NDMI', 'OUTPUT_PATH': str(fire_hazard_vegetation_dryness_ndmi_score_path)},
    {'PRODUCT_ID': 'COMBINED_VEGETATION_DRYNESS', 'PRODUCT_TYPE': 'Weighted component hazard score',
    'INDEX_NAME': 'COMBINED', 'OUTPUT_PATH': \
        str(fire_hazard_vegetation_dryness_combined_score_path)}])
# Record vegetation dryness valid extensions so vegetation-dryness hazard outputs can preserve a
# clear processing and validation outcome.
fire_hazard_vegetation_dryness_valid_extensions = {'.tif', '.tiff'}
# Track vegetation dryness minimum file size bytes across processed pixels for output-range QA.
fire_hazard_vegetation_dryness_minimum_file_size_bytes = 1024
# Set vegetation dryness value tolerance to control vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_value_tolerance = 1e-06
# Set vegetation dryness formula tolerance to control vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_formula_tolerance = 1e-05
# Build the expected vegetation dryness mask shape mask used to isolate records required for this
# analysis.
expected_vegetation_dryness_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if vegetation-dryness study-area mask does not match the common raster grid.
if fire_hazard_alignment_study_area_mask.shape != expected_vegetation_dryness_mask_shape:
    raise ValueError(f'The vegetation-dryness '
        f'study-area mask does not match '
        f'the common raster grid.\nMask '
        f'shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_vegetation_dryness_mask_shape}')
# Store vegetation dryness study area pixels needed to carry out vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_study_area_pixels = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if vegetation-dryness study-area mask contains no included target-grid pixels.
if fire_hazard_vegetation_dryness_study_area_pixels == 0:
    raise ValueError('The vegetation-dryness '
        'study-area mask contains no '
        'included target-grid pixels.')
# Record vegetation dryness validation records so vegetation-dryness hazard outputs can preserve a
# clear processing and validation outcome.
fire_hazard_vegetation_dryness_validation_records = []

# Process each ( entry so vegetation-dryness hazard outputs is applied consistently across all
# records.
for _, expected_product in fire_hazard_vegetation_dryness_expected_products.iterrows():
    # Store product ID needed to carry out vegetation-dryness hazard outputs.
    product_id = str(expected_product['PRODUCT_ID'])
    # Store product type needed to carry out vegetation-dryness hazard outputs.
    product_type = str(expected_product['PRODUCT_TYPE'])
    # Store index name needed to carry out vegetation-dryness hazard outputs.
    index_name = str(expected_product['INDEX_NAME']).upper()
    # Build the output path location so vegetation-dryness hazard outputs uses the expected project
    # file structure.
    output_path = Path(expected_product['OUTPUT_PATH'])
    # Store file exists needed to carry out vegetation-dryness hazard outputs.
    file_exists = output_path.exists() and output_path.is_file()
    # Record file extension valid so vegetation-dryness hazard outputs can preserve a clear
    # processing and validation outcome.
    file_extension_valid = output_path.suffix.lower() in \
        fire_hazard_vegetation_dryness_valid_extensions
    # Store file size bytes needed to carry out vegetation-dryness hazard outputs.
    file_size_bytes = output_path.stat().st_size if file_exists else 0
    # Record file size valid so vegetation-dryness hazard outputs can preserve a clear processing
    # and validation outcome.
    file_size_valid = file_size_bytes >= fire_hazard_vegetation_dryness_minimum_file_size_bytes
    # Record raster readable so vegetation-dryness hazard outputs can preserve a clear processing
    # and validation outcome.
    raster_readable = False
    # Calculate band count valid to quantify completeness and support QA checks.
    band_count_valid = False
    # Record width valid so vegetation-dryness hazard outputs can preserve a clear processing and
    # validation outcome.
    width_valid = False
    # Record height valid so vegetation-dryness hazard outputs can preserve a clear processing and
    # validation outcome.
    height_valid = False
    # Store crs valid so spatial operations use the required coordinate reference system.
    crs_valid = False
    # Record transform valid so vegetation-dryness hazard outputs can preserve a clear processing
    # and validation outcome.
    transform_valid = False
    # Record pixel size valid so vegetation-dryness hazard outputs can preserve a clear processing
    # and validation outcome.
    pixel_size_valid = False
    # Record dtype valid so vegetation-dryness hazard outputs can preserve a clear processing and
    # validation outcome.
    dtype_valid = False
    # Record NoData valid so vegetation-dryness hazard outputs can preserve a clear processing and
    # validation outcome.
    nodata_valid = False
    # Record grid valid so vegetation-dryness hazard outputs can preserve a clear processing and
    # validation outcome.
    grid_valid = False
    # Record structure valid so vegetation-dryness hazard outputs can preserve a clear processing
    # and validation outcome.
    structure_valid = False
    # Calculate valid pixel count to quantify completeness and support QA checks.
    valid_pixel_count = 0
    # Calculate NoData pixel count to quantify completeness and support QA checks.
    nodata_pixel_count = 0
    # Calculate valid inside study area count to quantify completeness and support QA checks.
    valid_inside_study_area_count = 0
    # Calculate valid outside study area count to quantify completeness and support QA checks.
    valid_outside_study_area_count = 0
    # Track minimum value across processed pixels for output-range QA.
    minimum_value = None
    # Track maximum value across processed pixels for output-range QA.
    maximum_value = None
    # Set mean value to control vegetation-dryness hazard outputs.
    mean_value = None
    # Set value sum to control vegetation-dryness hazard outputs.
    value_sum = 0.0
    # Set value range available to control vegetation-dryness hazard outputs.
    value_range_available = False
    # Record value range valid so vegetation-dryness hazard outputs can preserve a clear processing
    # and validation outcome.
    value_range_valid = False
    # Record validation error so vegetation-dryness hazard outputs can preserve a clear processing
    # and validation outcome.
    validation_error = None

    # Handle the file exists case explicitly during vegetation-dryness hazard outputs.
    if file_exists and file_extension_valid and file_size_valid:

        # Protect vegetation-dryness hazard outputs so expected source or file failures do not leave
        # partial outputs.
        try:

            # Open the raster in a managed context so its file handle closes reliably after
            # vegetation-dryness hazard outputs.
            with rasterio.open(output_path) as product_source:
                # Record raster readable so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                raster_readable = True
                # Calculate band count valid to quantify completeness and support QA checks.
                band_count_valid = product_source.count == 1
                # Record width valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                width_valid = product_source.width == fire_hazard_alignment_width
                # Record height valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                height_valid = product_source.height == fire_hazard_alignment_height
                # Store crs valid so spatial operations use the required coordinate reference
                # system.
                crs_valid = product_source.crs is not None and product_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs)
                # Record transform valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                transform_valid = \
                    product_source.transform.almost_equals(fire_hazard_alignment_transform)
                # Record pixel size valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                pixel_size_valid = np.isclose(abs(product_source.transform.a),
                    fire_hazard_cell_size) and np.isclose(abs(product_source.transform.e),
                    fire_hazard_cell_size)
                # Record dtype valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                dtype_valid = product_source.dtypes[0] == \
                    fire_hazard_vegetation_dryness_output_dtype
                # Record NoData valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                nodata_valid = product_source.nodata == fire_hazard_vegetation_dryness_output_nodata
                # Record grid valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                grid_valid = all([width_valid,
                    height_valid, crs_valid, transform_valid, pixel_size_valid])
                # Record structure valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                structure_valid = all([band_count_valid, dtype_valid, nodata_valid])

                # Process each ( entry so vegetation-dryness hazard outputs is applied consistently
                # across all records.
                for _, product_window in product_source.block_windows(1):
                    # Prepare product array used to process the current raster window.
                    product_array = product_source.read(1, window=product_window)
                    # Calculate row start for the current raster processing window.
                    row_start = int(product_window.row_off)
                    # Store row end needed to carry out vegetation-dryness hazard outputs.
                    row_end = int(product_window.row_off + product_window.height)
                    # Store column start needed to carry out vegetation-dryness hazard outputs.
                    column_start = int(product_window.col_off)
                    # Store column end needed to carry out vegetation-dryness hazard outputs.
                    column_end = int(product_window.col_off + product_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Build the valid product mask mask used to isolate records required for this
                    # analysis.
                    valid_product_mask = np.isfinite(product_array) & (product_array != \
                        fire_hazard_vegetation_dryness_output_nodata)
                    # Build the valid inside mask mask used to isolate records required for this
                    # analysis.
                    valid_inside_mask = valid_product_mask & study_area_window_mask
                    # Build the valid outside mask mask used to isolate records required for this
                    # analysis.
                    valid_outside_mask = valid_product_mask & ~study_area_window_mask
                    # Record valid values so vegetation-dryness hazard outputs can preserve a clear
                    # processing and validation outcome.
                    valid_values = product_array[valid_product_mask]
                    # Calculate valid pixel count to quantify completeness and support QA checks.
                    valid_pixel_count += int(valid_product_mask.sum())
                    # Calculate NoData pixel count to quantify completeness and support QA checks.
                    nodata_pixel_count += int((~valid_product_mask).sum())
                    # Calculate valid inside study area count to quantify completeness and support
                    # QA checks.
                    valid_inside_study_area_count += int(valid_inside_mask.sum())
                    # Calculate valid outside study area count to quantify completeness and support
                    # QA checks.
                    valid_outside_study_area_count += int(valid_outside_mask.sum())

                    # Handle the valid values case explicitly during vegetation-dryness hazard
                    # outputs.
                    if valid_values.size > 0:
                        # Track block minimum across processed pixels for output-range QA.
                        block_minimum = float(valid_values.min())
                        # Track block maximum across processed pixels for output-range QA.
                        block_maximum = float(valid_values.max())
                        # Track minimum value across processed pixels for output-range QA.
                        minimum_value = block_minimum if minimum_value is None else \
                            min(minimum_value,
                            block_minimum)
                        # Track maximum value across processed pixels for output-range QA.
                        maximum_value = block_maximum if maximum_value is None else \
                            max(maximum_value,
                            block_maximum)
                        # Store value sum needed to carry out vegetation-dryness hazard outputs.
                        value_sum += float(valid_values.sum(dtype=np.float64))
                # Store value range available needed to carry out vegetation-dryness hazard outputs.
                value_range_available = minimum_value is not None and maximum_value is not None \
                    and (valid_pixel_count > 0)

                # Handle the value range available case explicitly during vegetation-dryness hazard
                # outputs.
                if value_range_available:
                    # Store mean value needed to carry out vegetation-dryness hazard outputs.
                    mean_value = value_sum / valid_pixel_count
                # Record value range valid so vegetation-dryness hazard outputs can preserve a clear
                # processing and validation outcome.
                value_range_valid = value_range_available and minimum_value >= \
                    fire_hazard_vegetation_dryness_score_minimum - \
                    fire_hazard_vegetation_dryness_value_tolerance and (maximum_value <= \
                    fire_hazard_vegetation_dryness_score_maximum + \
                    fire_hazard_vegetation_dryness_value_tolerance)
        # Handle the expected failure without leaving the workflow in an inconsistent state.
        except Exception as error:
            # Record validation error so vegetation-dryness hazard outputs can preserve a clear
            # processing and validation outcome.
            validation_error = str(error)
    # Record product valid so vegetation-dryness hazard outputs can preserve a clear processing and
    # validation outcome.
    product_valid = all([file_exists, file_extension_valid,
        file_size_valid, raster_readable, grid_valid, structure_valid,
        valid_inside_study_area_count > 0, valid_outside_study_area_count == 0,
        value_range_available, value_range_valid, validation_error is None])
    fire_hazard_vegetation_dryness_validation_records.append({'PRODUCT_ID': product_id,
        'PRODUCT_TYPE': product_type, 'INDEX_NAME': index_name,
        'OUTPUT_PATH': str(output_path), 'FILE_EXISTS': file_exists,
        'FILE_EXTENSION_VALID': file_extension_valid, 'FILE_SIZE_BYTES': file_size_bytes,
        'FILE_SIZE_MB': file_size_bytes / 1024 ** 2, 'FILE_SIZE_VALID': file_size_valid,
        'RASTER_READABLE': raster_readable, 'BAND_COUNT_VALID': band_count_valid,
        'WIDTH_VALID': width_valid, 'HEIGHT_VALID': height_valid,
        'CRS_VALID': crs_valid, 'TRANSFORM_VALID': transform_valid,
        'PIXEL_SIZE_VALID': pixel_size_valid, 'DTYPE_VALID': dtype_valid,
        'NODATA_VALID': nodata_valid, 'GRID_VALID': grid_valid,
        'STRUCTURE_VALID': structure_valid, 'VALID_PIXELS': valid_pixel_count,
        'NODATA_PIXELS': nodata_pixel_count, 'VALID_INSIDE_STUDY_AREA': \
            valid_inside_study_area_count,
        'VALID_OUTSIDE_STUDY_AREA': valid_outside_study_area_count,
        'VALUE_MINIMUM': minimum_value, 'VALUE_MAXIMUM': maximum_value,
        'VALUE_MEAN': mean_value, 'VALUE_RANGE_AVAILABLE': value_range_available,
        'VALUE_RANGE_VALID': value_range_valid, 'ERROR_MESSAGE': validation_error,
        'VALID': product_valid})
# Record vegetation dryness validation so vegetation-dryness hazard outputs can preserve a clear
# processing and validation outcome.
fire_hazard_vegetation_dryness_validation = \
    pd.DataFrame(fire_hazard_vegetation_dryness_validation_records).sort_values(['INDEX_NAME',
    'PRODUCT_TYPE']).reset_index(drop=True)

# Open the raster in a managed context so its file handle closes reliably after vegetation-dryness
# hazard outputs.
with rasterio.open(fire_hazard_vegetation_dryness_ndvi_score_path) as ndvi_source:

    # Open the raster in a managed context so its file handle closes reliably after
    # vegetation-dryness hazard outputs.
    with rasterio.open(fire_hazard_vegetation_dryness_ndmi_score_path) as ndmi_source:

        # Open the raster in a managed context so its file handle closes reliably after
        # vegetation-dryness hazard outputs.
        with rasterio.open(fire_hazard_vegetation_dryness_combined_score_path) as combined_source:
            # Store vegetation dryness grids match needed to carry out vegetation-dryness hazard
            # outputs.
            fire_hazard_vegetation_dryness_grids_match = all([ndvi_source.width == \
                ndmi_source.width == combined_source.width,
                ndvi_source.height == ndmi_source.height == combined_source.height,
                ndvi_source.crs == ndmi_source.crs == combined_source.crs,
                ndvi_source.transform.almost_equals(ndmi_source.transform),
                ndvi_source.transform.almost_equals(combined_source.transform)])
# Define vegetation dryness formula records to control the inputs and rules used by
# vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_formula_records = []
# Store vegetation dryness NDVI weight needed to carry out vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_ndvi_weight = \
    float(fire_hazard_vegetation_dryness_index_weights['NDVI'])
# Store vegetation dryness NDMI weight needed to carry out vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_ndmi_weight = \
    float(fire_hazard_vegetation_dryness_index_weights['NDMI'])
# Set formula checked pixels to control vegetation-dryness hazard outputs.
formula_checked_pixels = 0
# Set formula matching pixels to control vegetation-dryness hazard outputs.
formula_matching_pixels = 0
# Set formula mismatch pixels to control vegetation-dryness hazard outputs.
formula_mismatch_pixels = 0
# Track formula maximum absolute difference across processed pixels for output-range QA.
formula_maximum_absolute_difference = 0.0
# Set formula mean absolute difference to control vegetation-dryness hazard outputs.
formula_mean_absolute_difference = None
# Set formula absolute difference sum to control vegetation-dryness hazard outputs.
formula_absolute_difference_sum = 0.0
# Record combined valid without both so vegetation-dryness hazard outputs can preserve a
# clear processing and validation outcome.
combined_valid_without_both_inputs = 0
# Record both inputs valid without combined so vegetation-dryness hazard outputs can preserve a
# clear processing and validation outcome.
both_inputs_valid_without_combined = 0
# Record formula validation error so vegetation-dryness hazard outputs can preserve a clear
# processing and validation outcome.
formula_validation_error = None

# Protect vegetation-dryness hazard outputs so expected source or file failures do not leave partial
# outputs.
try:

    # Open the raster in a managed context so its file handle closes reliably after
    # vegetation-dryness hazard outputs.
    with rasterio.open(fire_hazard_vegetation_dryness_ndvi_score_path) as ndvi_source:

        # Open the raster in a managed context so its file handle closes reliably after
        # vegetation-dryness hazard outputs.
        with rasterio.open(fire_hazard_vegetation_dryness_ndmi_score_path) as ndmi_source:

            # Open the raster in a managed context so its file handle closes reliably after
            # vegetation-dryness hazard outputs.
            with rasterio.open(fire_hazard_vegetation_dryness_combined_score_path) as \
                combined_source:

                # Stop execution if nDVI, NDMI, and combined vegetation-dryness rasters do not use
                # the same grid.
                if not fire_hazard_vegetation_dryness_grids_match:
                    raise ValueError('The NDVI, NDMI, and combined '
                        'vegetation-dryness rasters do '
                        'not use the same grid.')

                # Process each ( entry so vegetation-dryness hazard outputs is applied consistently
                # across all records.
                for _, comparison_window in combined_source.block_windows(1):
                    # Prepare NDVI array used to process the current raster window.
                    ndvi_array = ndvi_source.read(1, window=comparison_window)
                    # Prepare NDMI array used to process the current raster window.
                    ndmi_array = ndmi_source.read(1, window=comparison_window)
                    # Prepare combined array used to process the current raster window.
                    combined_array = combined_source.read(1, window=comparison_window)
                    # Build the valid ndvi mask mask used to isolate records required for this
                    # analysis.
                    valid_ndvi_mask = np.isfinite(ndvi_array) & (ndvi_array != \
                        fire_hazard_vegetation_dryness_output_nodata)
                    # Build the valid ndmi mask mask used to isolate records required for this
                    # analysis.
                    valid_ndmi_mask = np.isfinite(ndmi_array) & (ndmi_array != \
                        fire_hazard_vegetation_dryness_output_nodata)
                    # Build the valid combined mask mask used to isolate records required for this
                    # analysis.
                    valid_combined_mask = np.isfinite(combined_array) & (combined_array != \
                        fire_hazard_vegetation_dryness_output_nodata)
                    # Build the both inputs valid mask mask used to isolate records required for
                    # this analysis.
                    both_inputs_valid_mask = valid_ndvi_mask & valid_ndmi_mask
                    # Record combined valid without both so vegetation-dryness hazard outputs
                    # can preserve a clear processing and validation outcome.
                    combined_valid_without_both_inputs += int((valid_combined_mask & \
                        ~both_inputs_valid_mask).sum())
                    # Record both inputs valid without combined so vegetation-dryness hazard outputs
                    # can preserve a clear processing and validation outcome.
                    both_inputs_valid_without_combined += int((both_inputs_valid_mask & \
                        ~valid_combined_mask).sum())
                    # Build the formula comparison mask mask used to isolate records required for
                    # this analysis.
                    formula_comparison_mask = both_inputs_valid_mask & valid_combined_mask

                    # Handle the formula comparison mask case explicitly during vegetation-dryness
                    # hazard outputs.
                    if formula_comparison_mask.any():
                        # Store expected combined values needed to carry out vegetation-dryness
                        # hazard outputs.
                        expected_combined_values = fire_hazard_vegetation_dryness_ndvi_weight * \
                            ndvi_array[formula_comparison_mask] + \
                            fire_hazard_vegetation_dryness_ndmi_weight * \
                            ndmi_array[formula_comparison_mask]
                        # Store expected combined values needed to carry out vegetation-dryness
                        # hazard outputs.
                        expected_combined_values = np.clip(expected_combined_values,
                            fire_hazard_vegetation_dryness_score_minimum, \
                                fire_hazard_vegetation_dryness_score_maximum)
                        # Store actual combined values needed to carry out vegetation-dryness hazard
                        # outputs.
                        actual_combined_values = combined_array[formula_comparison_mask]
                        # Store absolute differences needed to carry out vegetation-dryness hazard
                        # outputs.
                        absolute_differences = np.abs(actual_combined_values - \
                            expected_combined_values)
                        # Build the matching values mask mask used to isolate records required for
                        # this analysis.
                        matching_values_mask = absolute_differences <= \
                            fire_hazard_vegetation_dryness_formula_tolerance
                        # Store formula checked pixels needed to carry out vegetation-dryness hazard
                        # outputs.
                        formula_checked_pixels += int(absolute_differences.size)
                        # Store formula matching pixels needed to carry out vegetation-dryness
                        # hazard outputs.
                        formula_matching_pixels += int(matching_values_mask.sum())
                        # Store formula mismatch pixels needed to carry out vegetation-dryness
                        # hazard outputs.
                        formula_mismatch_pixels += int((~matching_values_mask).sum())
                        # Store formula absolute difference sum needed to carry out
                        # vegetation-dryness hazard outputs.
                        formula_absolute_difference_sum += \
                            float(absolute_differences.sum(dtype=np.float64))

                        # Handle the absolute differences case explicitly during vegetation-dryness
                        # hazard outputs.
                        if absolute_differences.size > 0:
                            # Track formula maximum absolute difference across processed pixels for
                            # output-range QA.
                            formula_maximum_absolute_difference = \
                                max(formula_maximum_absolute_difference,
                                float(absolute_differences.max()))

                # Handle the formula checked pixels case explicitly during vegetation-dryness hazard
                # outputs.
                if formula_checked_pixels > 0:
                    # Store formula mean absolute difference needed to carry out vegetation-dryness
                    # hazard outputs.
                    formula_mean_absolute_difference = formula_absolute_difference_sum / \
                        formula_checked_pixels
# Handle the expected failure without leaving the workflow in an inconsistent state.
except Exception as error:
    # Record formula validation error so vegetation-dryness hazard outputs can preserve a clear
    # processing and validation outcome.
    formula_validation_error = str(error)
# Record vegetation dryness formula valid so vegetation-dryness hazard outputs can preserve a clear
# processing and validation outcome.
fire_hazard_vegetation_dryness_formula_valid = formula_validation_error is None and \
    formula_checked_pixels > 0 and (formula_mismatch_pixels == 0) and \
    (combined_valid_without_both_inputs == 0) and (both_inputs_valid_without_combined == 0 if \
    fire_hazard_vegetation_dryness_require_both_indices else True)
# Record vegetation dryness formula validation so vegetation-dryness hazard outputs can preserve a
# clear processing and validation outcome.
fire_hazard_vegetation_dryness_formula_validation = pd.DataFrame([{'COMBINATION_METHOD': \
    'Weighted linear combination',
    'NDVI_WEIGHT': fire_hazard_vegetation_dryness_ndvi_weight,
    'NDMI_WEIGHT': fire_hazard_vegetation_dryness_ndmi_weight,
    'REQUIRE_BOTH_INDICES': fire_hazard_vegetation_dryness_require_both_indices,
    'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
    'FORMULA_MATCHING_PIXELS': formula_matching_pixels,
    'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
    'COMBINED_WITHOUT_BOTH_INPUTS': combined_valid_without_both_inputs,
    'BOTH_INPUTS_WITHOUT_COMBINED': both_inputs_valid_without_combined,
    'MAXIMUM_ABSOLUTE_DIFFERENCE': formula_maximum_absolute_difference,
    'MEAN_ABSOLUTE_DIFFERENCE': formula_mean_absolute_difference,
    'FORMULA_TOLERANCE': fire_hazard_vegetation_dryness_formula_tolerance,
    'ERROR_MESSAGE': formula_validation_error, 'VALID': \
        fire_hazard_vegetation_dryness_formula_valid}])

# Stop execution if vegetation-dryness normalization manifest could not be found: the reported
# value.
if not fire_hazard_vegetation_dryness_normalization_manifest_path.exists():
    raise FileNotFoundError(f'The vegetation-dryness '
        f'normalization manifest could '
        f'not be found:\n'
        f'{fire_hazard_vegetation_dryness_normalization_manifest_path}')
# Store vegetation dryness normalization manifest needed to carry out vegetation-dryness hazard
# outputs.
fire_hazard_vegetation_dryness_normalization_manifest = \
    pd.read_csv(fire_hazard_vegetation_dryness_normalization_manifest_path)
# List the required vegetation dryness manifest fields prerequisites required before
# vegetation-dryness hazard outputs can run.
required_vegetation_dryness_manifest_fields = ['INDEX_NAME', 'OUTPUT_TYPE', 'OUTPUT_PATH', 'VALID']
# Identify unavailable vegetation dryness manifest fields so vegetation-dryness hazard outputs
# stops before using incomplete inputs.
missing_vegetation_dryness_manifest_fields = [field_name for field_name in \
    required_vegetation_dryness_manifest_fields if field_name not in \
    fire_hazard_vegetation_dryness_normalization_manifest.columns]

# Stop execution when required vegetation dryness manifest fields inputs are unavailable.
if missing_vegetation_dryness_manifest_fields:
    raise ValueError(f'The vegetation-dryness '
        f'normalization manifest is '
        f'missing required fields:\n'
        f'{missing_vegetation_dryness_manifest_fields}')

# Handle the object case explicitly during vegetation-dryness hazard outputs.
if fire_hazard_vegetation_dryness_normalization_manifest['VALID'].dtype == object:
    # Record vegetation dryness normalization manifest['valid'] so vegetation-dryness hazard outputs
    # can preserve a clear processing and validation outcome.
    fire_hazard_vegetation_dryness_normalization_manifest['VALID'] = \
        fire_hazard_vegetation_dryness_normalization_manifest['VALID'].astype(str).str.strip() \
        .str.lower().map({'true': True,
        'false': False})
# List the required vegetation dryness manifest indices prerequisites required before
# vegetation-dryness hazard outputs can run.
required_vegetation_dryness_manifest_indices = {'NDVI', 'NDMI', 'COMBINED'}
# Collect actual vegetation dryness manifest indices for membership and completeness checks during
# vegetation-dryness hazard outputs.
actual_vegetation_dryness_manifest_indices = \
    set(fire_hazard_vegetation_dryness_normalization_manifest['INDEX_NAME'].astype(str).str.upper())
# Store vegetation dryness manifest complete needed to carry out vegetation-dryness hazard outputs.
fire_hazard_vegetation_dryness_manifest_complete = \
    required_vegetation_dryness_manifest_indices.issubset \
    (actual_vegetation_dryness_manifest_indices) and (not \
    fire_hazard_vegetation_dryness_normalization_manifest['VALID'].isna().any()) and \
    fire_hazard_vegetation_dryness_normalization_manifest['VALID'].all()
# Calculate expected vegetation dryness product count to quantify completeness and support QA
# checks.
expected_vegetation_dryness_product_count = len(fire_hazard_vegetation_dryness_expected_products)
# Calculate validated vegetation dryness product count to quantify completeness and support QA
# checks.
validated_vegetation_dryness_product_count = len(fire_hazard_vegetation_dryness_validation)
# Calculate valid vegetation dryness product count to quantify completeness and support QA checks.
valid_vegetation_dryness_product_count = \
    int(fire_hazard_vegetation_dryness_validation['VALID'].sum())
# Store failed vegetation dryness products needed to carry out vegetation-dryness hazard outputs.
failed_vegetation_dryness_products = \
    fire_hazard_vegetation_dryness_validation[~fire_hazard_vegetation_dryness_validation \
    ['VALID']].copy().reset_index(drop=True)
# Record vegetation dryness validation complete so vegetation-dryness hazard outputs can preserve a
# clear processing and validation outcome.
fire_hazard_vegetation_dryness_validation_complete = validated_vegetation_dryness_product_count \
    == expected_vegetation_dryness_product_count and valid_vegetation_dryness_product_count == \
    expected_vegetation_dryness_product_count and failed_vegetation_dryness_products.empty and \
    fire_hazard_vegetation_dryness_grids_match and fire_hazard_vegetation_dryness_formula_valid \
    and fire_hazard_vegetation_dryness_manifest_complete
# Store total vegetation dryness output size bytes needed to carry out vegetation-dryness hazard
# outputs.
total_vegetation_dryness_output_size_bytes = \
    int(fire_hazard_vegetation_dryness_validation['FILE_SIZE_BYTES'].sum())
# Record vegetation dryness validation summary so vegetation-dryness hazard outputs can preserve a
# clear processing and validation outcome.
fire_hazard_vegetation_dryness_validation_summary = pd.DataFrame([{'EXPECTED_PRODUCTS': \
    expected_vegetation_dryness_product_count,
    'VALIDATED_PRODUCTS': validated_vegetation_dryness_product_count,
    'VALID_PRODUCTS': valid_vegetation_dryness_product_count,
    'FAILED_PRODUCTS': len(failed_vegetation_dryness_products),
    'ALL_GRIDS_MATCH': fire_hazard_vegetation_dryness_grids_match,
    'WEIGHTED_FORMULA_VALID': fire_hazard_vegetation_dryness_formula_valid,
    'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
    'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
    'MAXIMUM_FORMULA_DIFFERENCE': formula_maximum_absolute_difference,
    'PROCESSING_MANIFEST_COMPLETE': fire_hazard_vegetation_dryness_manifest_complete,
    'SCORE_MINIMUM': fire_hazard_vegetation_dryness_score_minimum,
    'SCORE_MAXIMUM': fire_hazard_vegetation_dryness_score_maximum,
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TOTAL_OUTPUT_SIZE_BYTES': total_vegetation_dryness_output_size_bytes,
    'TOTAL_OUTPUT_SIZE_MB': total_vegetation_dryness_output_size_bytes / 1024 ** 2,
    'VALIDATION_COMPLETE': fire_hazard_vegetation_dryness_validation_complete}])
# Save the vegetation dryness validation CSV so later steps can reuse the recorded workflow results.
fire_hazard_vegetation_dryness_validation.to_csv(fire_hazard_vegetation_dryness_validation_path,
    index=False)
# Save the vegetation dryness validation summary CSV so later steps can reuse the recorded workflow
# results.
fire_hazard_vegetation_dryness_validation_summary.to_csv \
    (fire_hazard_vegetation_dryness_validation_summary_path,
    index=False)
# Build the vegetation dryness formula validation path location so vegetation-dryness hazard outputs
# uses the expected project file structure.
fire_hazard_vegetation_dryness_formula_validation_path = \
    Path(fire_hazard_vegetation_dryness_validation_path).parent / \
    'vegetation_dryness_formula_validation.csv'
# Save the vegetation dryness formula validation CSV so later steps can reuse the recorded workflow
# results.
fire_hazard_vegetation_dryness_formula_validation.to_csv \
    (fire_hazard_vegetation_dryness_formula_validation_path,
    index=False)
print(f'-> Expected vegetation-dryness products: {expected_vegetation_dryness_product_count}')
print(f'-> Validated products: {validated_vegetation_dryness_product_count}')
print(f'-> Valid products: {valid_vegetation_dryness_product_count}')
print(f'-> Failed products: {len(failed_vegetation_dryness_products)}')
print(f'-> All raster grids match: {fire_hazard_vegetation_dryness_grids_match}')
print(f'-> Formula pixels checked: {formula_checked_pixels:,}')
print(f'-> Formula mismatch pixels: {formula_mismatch_pixels:,}')
print(f'-> Maximum formula difference: {formula_maximum_absolute_difference:.8f}')
print(f'-> Weighted formula valid: {fire_hazard_vegetation_dryness_formula_valid}')
print(f'-> Processing manifest complete: {fire_hazard_vegetation_dryness_manifest_complete}')
print(f'-> Total output size: {total_vegetation_dryness_output_size_bytes / 1024 ** 2:,.2f} MB')
print(f'-> Vegetation-dryness '
    f'validation complete: '
    f'{fire_hazard_vegetation_dryness_validation_complete}')
print(f'-> Product validation saved: {fire_hazard_vegetation_dryness_validation_path}')
print(f'-> Formula validation saved: {fire_hazard_vegetation_dryness_formula_validation_path}')
print(f'-> Validation summary saved: {fire_hazard_vegetation_dryness_validation_summary_path}')
print('\n--- VEGETATION-DRYNESS VALIDATION SUMMARY ---')
display(fire_hazard_vegetation_dryness_validation_summary)
print('\n--- VEGETATION-DRYNESS PRODUCT VALIDATION ---')
display(fire_hazard_vegetation_dryness_validation[['PRODUCT_ID',
    'INDEX_NAME', 'FILE_EXISTS', 'GRID_VALID', 'STRUCTURE_VALID',
    'VALID_INSIDE_STUDY_AREA', 'VALID_OUTSIDE_STUDY_AREA',
    'VALUE_MINIMUM', 'VALUE_MAXIMUM', 'VALUE_MEAN',
    'VALUE_RANGE_VALID', 'VALID']])
print('\n--- VEGETATION-DRYNESS FORMULA VALIDATION ---')
display(fire_hazard_vegetation_dryness_formula_validation)

# Handle missing or invalid failed vegetation dryness products explicitly during vegetation-dryness
# hazard outputs.
if not failed_vegetation_dryness_products.empty:
    print('\n--- FAILED VEGETATION-DRYNESS PRODUCTS ---')
    display(failed_vegetation_dryness_products.head(5))

# Stop execution if one or more vegetation-dryness hazard outputs failed independent validation.
if not fire_hazard_vegetation_dryness_validation_complete:
    raise ValueError(f'One or more vegetation-dryness '
        f'hazard outputs failed '
        f'independent validation.\n\n'
        f'Expected products: '
        f'{expected_vegetation_dryness_product_count}\n'
        f'Valid products: '
        f'{valid_vegetation_dryness_product_count}\n'
        f'Failed products: '
        f'{len(failed_vegetation_dryness_products)}\n'
        f'All grids match: '
        f'{fire_hazard_vegetation_dryness_grids_match}\n'
        f'Weighted formula valid: '
        f'{fire_hazard_vegetation_dryness_formula_valid}\n'
        f'Formula mismatch pixels: '
        f'{formula_mismatch_pixels:,}\n'
        f'Processing manifest complete: '
        f'{fire_hazard_vegetation_dryness_manifest_complete}\n\n'
        f'Review the saved '
        f'vegetation-dryness validation '
        f'files.')
# Record vegetation dryness hazard valid so vegetation-dryness hazard outputs can preserve a clear
# processing and validation outcome.
fire_hazard_vegetation_dryness_hazard_valid = fire_hazard_vegetation_dryness_validation_complete
import gc
gc.collect()
print('\nNOTE:')
print('The normalized NDVI, '
    'normalized NDMI, and combined '
    'vegetation-dryness hazard '
    'rasters passed independent '
    'file, grid, structure, '
    'study-area coverage, and '
    'score-range checks.')
print('The combined raster was also '
    'independently recalculated '
    'from the NDVI and NDMI inputs '
    'to confirm that the configured '
    '0.40 and 0.60 weights were '
    'applied correctly.')
print('The validated combined '
    'vegetation-dryness raster is '
    'now ready to become an input '
    'component in the Composite '
    'Fire Hazard Grid.')
print('\n=== VEGETATION-DRYNESS HAZARD VALIDATION COMPLETE ===')



=== VALIDATING VEGETATION-DRYNESS HAZARD OUTPUTS ===
-> Expected vegetation-dryness products: 3
-> Validated products: 3
-> Valid products: 3
-> Failed products: 0
-> All raster grids match: True
-> Formula pixels checked: 14,140,492
-> Formula mismatch pixels: 0
-> Maximum formula difference: 0.00000000
-> Weighted formula valid: True
-> Processing manifest complete: True
-> Total output size: 132.27 MB
-> Vegetation-dryness validation complete: True
-> Product validation saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_validation.csv
-> Formula validation saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_formula_validation.csv
-> Validation summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\veget

,EXPECTED_PRODUCTS,VALIDATED_PRODUCTS,VALID_PRODUCTS,FAILED_PRODUCTS,ALL_GRIDS_MATCH,WEIGHTED_FORMULA_VALID,FORMULA_CHECKED_PIXELS,FORMULA_MISMATCH_PIXELS,MAXIMUM_FORMULA_DIFFERENCE,PROCESSING_MANIFEST_COMPLETE,SCORE_MINIMUM,SCORE_MAXIMUM,TARGET_CRS,CELL_SIZE_METERS,TOTAL_OUTPUT_SIZE_BYTES,TOTAL_OUTPUT_SIZE_MB,VALIDATION_COMPLETE
0,3,3,3,0,True,True,14140492,0,0.0,True,0.0,1.0,EPSG:26912,30,138699008,132.273682,True



--- VEGETATION-DRYNESS PRODUCT VALIDATION ---


,PRODUCT_ID,INDEX_NAME,FILE_EXISTS,GRID_VALID,STRUCTURE_VALID,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,VALUE_MINIMUM,VALUE_MAXIMUM,VALUE_MEAN,VALUE_RANGE_VALID,VALID
0,COMBINED_VEGETATION_DRYNESS,COMBINED,True,True,True,14140492,0,0.0,1.0,0.671171,True,True
1,NDMI_DRYNESS_SCORE,NDMI,True,True,True,14140492,0,0.0,1.0,0.715181,True,True
2,NDVI_DRYNESS_SCORE,NDVI,True,True,True,14140492,0,0.0,1.0,0.605157,True,True



--- VEGETATION-DRYNESS FORMULA VALIDATION ---


,COMBINATION_METHOD,NDVI_WEIGHT,NDMI_WEIGHT,REQUIRE_BOTH_INDICES,FORMULA_CHECKED_PIXELS,FORMULA_MATCHING_PIXELS,FORMULA_MISMATCH_PIXELS,COMBINED_WITHOUT_BOTH_INPUTS,BOTH_INPUTS_WITHOUT_COMBINED,MAXIMUM_ABSOLUTE_DIFFERENCE,MEAN_ABSOLUTE_DIFFERENCE,FORMULA_TOLERANCE,ERROR_MESSAGE,VALID
0,Weighted linear combination,0.4,0.6,True,14140492,14140492,0,0,0,0.0,0.0,0.00001,None,True



NOTE:
The normalized NDVI, normalized NDMI, and combined vegetation-dryness hazard rasters passed independent file, grid, structure, study-area coverage, and score-range checks.
The combined raster was also independently recalculated from the NDVI and NDMI inputs to confirm that the configured 0.40 and 0.60 weights were applied correctly.
The validated combined vegetation-dryness raster is now ready to become an input component in the Composite Fire Hazard Grid.

=== VEGETATION-DRYNESS HAZARD VALIDATION COMPLETE ===


### Finalizing Vegetation-Dryness Hazard Component


In [95]:
print('=== FINALIZING VEGETATION-DRYNESS HAZARD COMPONENT ===')
# List the required vegetation dryness finalization prerequisites required before
# vegetation-dryness hazard component can run.
required_vegetation_dryness_finalization_inputs = ['fire_hazard_vegetation_dryness_hazard_valid',
    'fire_hazard_vegetation_dryness_validation_complete',
    'fire_hazard_vegetation_dryness_validation', \
        'fire_hazard_vegetation_dryness_validation_summary',
    'fire_hazard_vegetation_dryness_formula_validation',
    'fire_hazard_vegetation_dryness_normalization_manifest',
    'fire_hazard_vegetation_dryness_ndvi_score_path',
    'fire_hazard_vegetation_dryness_ndmi_score_path',
    'fire_hazard_vegetation_dryness_combined_score_path',
    'fire_hazard_vegetation_dryness_index_weights',
    'fire_hazard_vegetation_dryness_normalization_bounds',
    'fire_hazard_vegetation_dryness_score_minimum',
    'fire_hazard_vegetation_dryness_score_maximum',
    'fire_hazard_vegetation_dryness_output_nodata',
    'fire_hazard_vegetation_dryness_minimum_observations',
    'fire_hazard_vegetation_dryness_metadata_directory',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform']
# Identify unavailable vegetation dryness finalization so vegetation-dryness hazard
# component stops before using incomplete inputs.
missing_vegetation_dryness_finalization_inputs = [object_name for object_name in \
    required_vegetation_dryness_finalization_inputs if object_name not in globals()]

# Stop execution when required vegetation dryness finalization inputs inputs are unavailable.
if missing_vegetation_dryness_finalization_inputs:
    raise NameError(f'The following '
        f'vegetation-dryness '
        f'finalization objects are '
        f'missing:\n'
        f'{missing_vegetation_dryness_finalization_inputs}\n\n'
        f'Run Validate '
        f'Vegetation-Dryness Hazard '
        f'Outputs before finalizing the '
        f'component.')

# Stop execution if vegetation-dryness hazard component has not passed final validation.
if not fire_hazard_vegetation_dryness_hazard_valid:
    raise ValueError('The vegetation-dryness hazard component has not passed final validation.')

# Stop execution if vegetation-dryness validation workflow is incomplete.
if not fire_hazard_vegetation_dryness_validation_complete:
    raise ValueError('The vegetation-dryness validation workflow is incomplete.')

# Stop execution if one or more vegetation-dryness products remain invalid.
if fire_hazard_vegetation_dryness_validation.empty or not \
    fire_hazard_vegetation_dryness_validation['VALID'].fillna(False).all():
    raise ValueError('One or more vegetation-dryness products remain invalid.')
# Define vegetation dryness final products to control the inputs and rules used by
# vegetation-dryness hazard component.
fire_hazard_vegetation_dryness_final_products = {'NDVI_DRYNESS_SCORE': \
    Path(fire_hazard_vegetation_dryness_ndvi_score_path),
    'NDMI_DRYNESS_SCORE': Path(fire_hazard_vegetation_dryness_ndmi_score_path),
    'VEGETATION_DRYNESS_HAZARD_SCORE': Path(fire_hazard_vegetation_dryness_combined_score_path)}
# Record invalid vegetation dryness final products so vegetation-dryness hazard component can
# preserve a clear processing and validation outcome.
invalid_vegetation_dryness_final_products = []

# Process each (product ID entry so vegetation-dryness hazard component is applied consistently
# across all records.
for product_id, product_path in fire_hazard_vegetation_dryness_final_products.items():

    # Use the existing file only when it is present and valid for vegetation-dryness hazard
    # component.
    if not product_path.exists() or not product_path.is_file() or product_path.stat().st_size <= 0:
        invalid_vegetation_dryness_final_products.append({'PRODUCT_ID': product_id,
            'PRODUCT_PATH': str(product_path)})

# Stop execution if one or more finalized vegetation-dryness products are unavailable: the reported
# value.
if invalid_vegetation_dryness_final_products:
    raise FileNotFoundError(f'One or more finalized '
        f'vegetation-dryness products '
        f'are unavailable:\n'
        f'{invalid_vegetation_dryness_final_products}')
# Build the vegetation dryness final registry path location so vegetation-dryness hazard component
# uses the expected project file structure.
fire_hazard_vegetation_dryness_final_registry_path = \
    fire_hazard_vegetation_dryness_metadata_directory / \
    'vegetation_dryness_final_product_registry.csv'
# Build the vegetation dryness final summary path location so vegetation-dryness hazard component
# uses the expected project file structure.
fire_hazard_vegetation_dryness_final_summary_path = \
    fire_hazard_vegetation_dryness_metadata_directory / \
    'vegetation_dryness_final_component_summary.csv'
# Build the vegetation dryness final policy path location so vegetation-dryness hazard component
# uses the expected project file structure.
fire_hazard_vegetation_dryness_final_policy_path = \
    fire_hazard_vegetation_dryness_metadata_directory / \
    'vegetation_dryness_final_component_policy.json'
# Build the vegetation dryness completion marker path location so vegetation-dryness hazard
# component uses the expected project file structure.
fire_hazard_vegetation_dryness_completion_marker_path = \
    fire_hazard_vegetation_dryness_metadata_directory / 'vegetation_dryness_component_complete.json'
# Record NDVI dryness validation record so vegetation-dryness hazard component can preserve a clear
# processing and validation outcome.
ndvi_dryness_validation_record = \
    fire_hazard_vegetation_dryness_validation[fire_hazard_vegetation_dryness_validation \
    ['INDEX_NAME'] == 'NDVI']
# Record NDMI dryness validation record so vegetation-dryness hazard component can preserve a clear
# processing and validation outcome.
ndmi_dryness_validation_record = \
    fire_hazard_vegetation_dryness_validation[fire_hazard_vegetation_dryness_validation \
    ['INDEX_NAME'] == 'NDMI']
# Record combined dryness validation record so vegetation-dryness hazard component can preserve a
# clear processing and validation outcome.
combined_dryness_validation_record = \
    fire_hazard_vegetation_dryness_validation[fire_hazard_vegetation_dryness_validation \
    ['INDEX_NAME'] == 'COMBINED']

# Stop execution if exactly one final NDVI dryness validation record is required.
if len(ndvi_dryness_validation_record) != 1:
    raise ValueError('Exactly one final NDVI dryness validation record is required.')

# Stop execution if exactly one final NDMI dryness validation record is required.
if len(ndmi_dryness_validation_record) != 1:
    raise ValueError('Exactly one final NDMI dryness validation record is required.')

# Stop execution if exactly one final combined dryness validation record is required.
if len(combined_dryness_validation_record) != 1:
    raise ValueError('Exactly one final combined dryness validation record is required.')
# Store vegetation dryness final registry needed to carry out vegetation-dryness hazard component.
fire_hazard_vegetation_dryness_final_registry = pd.DataFrame([{'COMPONENT_ID': 'VEGETATION_DRYNESS',
    'PRODUCT_ID': 'NDVI_DRYNESS_SCORE', 'PRODUCT_ROLE': 'Normalized supporting layer',
    'SOURCE_INDEX': 'NDVI', 'LOCAL_PATH': str(fire_hazard_vegetation_dryness_ndvi_score_path),
    'SCORE_MINIMUM': float(ndvi_dryness_validation_record.iloc[0]['VALUE_MINIMUM']),
    'SCORE_MAXIMUM': float(ndvi_dryness_validation_record.iloc[0]['VALUE_MAXIMUM']),
    'SCORE_MEAN': float(ndvi_dryness_validation_record.iloc[0]['VALUE_MEAN']),
    'VALID_PIXELS': int(ndvi_dryness_validation_record.iloc[0]['VALID_PIXELS']),
    'FILE_SIZE_BYTES': int(fire_hazard_vegetation_dryness_ndvi_score_path.stat().st_size),
    'USE_IN_FINAL_HAZARD_MODEL': False, 'VALID': True},
    {'COMPONENT_ID': 'VEGETATION_DRYNESS', 'PRODUCT_ID': 'NDMI_DRYNESS_SCORE',
    'PRODUCT_ROLE': 'Normalized supporting layer',
    'SOURCE_INDEX': 'NDMI', 'LOCAL_PATH': str(fire_hazard_vegetation_dryness_ndmi_score_path),
    'SCORE_MINIMUM': float(ndmi_dryness_validation_record.iloc[0]['VALUE_MINIMUM']),
    'SCORE_MAXIMUM': float(ndmi_dryness_validation_record.iloc[0]['VALUE_MAXIMUM']),
    'SCORE_MEAN': float(ndmi_dryness_validation_record.iloc[0]['VALUE_MEAN']),
    'VALID_PIXELS': int(ndmi_dryness_validation_record.iloc[0]['VALID_PIXELS']),
    'FILE_SIZE_BYTES': int(fire_hazard_vegetation_dryness_ndmi_score_path.stat().st_size),
    'USE_IN_FINAL_HAZARD_MODEL': False, 'VALID': True},
    {'COMPONENT_ID': 'VEGETATION_DRYNESS', 'PRODUCT_ID': 'VEGETATION_DRYNESS_HAZARD_SCORE',
    'PRODUCT_ROLE': 'Final component hazard input',
    'SOURCE_INDEX': 'NDVI and NDMI', 'LOCAL_PATH': \
        str(fire_hazard_vegetation_dryness_combined_score_path),
    'SCORE_MINIMUM': float(combined_dryness_validation_record.iloc[0]['VALUE_MINIMUM']),
    'SCORE_MAXIMUM': float(combined_dryness_validation_record.iloc[0]['VALUE_MAXIMUM']),
    'SCORE_MEAN': float(combined_dryness_validation_record.iloc[0]['VALUE_MEAN']),
    'VALID_PIXELS': int(combined_dryness_validation_record.iloc[0]['VALID_PIXELS']),
    'FILE_SIZE_BYTES': int(fire_hazard_vegetation_dryness_combined_score_path.stat().st_size),
    'USE_IN_FINAL_HAZARD_MODEL': True, 'VALID': True}])
# Build the final hazard input mask mask used to isolate records required for this analysis.
final_hazard_input_mask = fire_hazard_vegetation_dryness_final_registry['USE_IN_FINAL_HAZARD_MODEL']

# Require Exactly one vegetation-dryness product to be designated for use in the final fire-hazard
# model.
if int(final_hazard_input_mask.sum()) != 1:
    raise ValueError('Exactly one vegetation-dryness '
        'product must be designated for '
        'use in the final fire-hazard '
        'model.')
# Build the vegetation dryness hazard path location so vegetation-dryness hazard component uses the
# expected project file structure.
fire_hazard_vegetation_dryness_hazard_path = \
    Path(fire_hazard_vegetation_dryness_final_registry.loc[final_hazard_input_mask,
    'LOCAL_PATH'].iloc[0])
# Store vegetation dryness final summary needed to carry out vegetation-dryness hazard component.
fire_hazard_vegetation_dryness_final_summary = pd.DataFrame([{'COMPONENT_ID': 'VEGETATION_DRYNESS',
    'COMPONENT_NAME': 'Vegetation Dryness Hazard',
    'COMPONENT_STATUS': 'Complete', 'PRIMARY_OUTPUT': \
        str(fire_hazard_vegetation_dryness_hazard_path),
    'SOURCE_INDICES': 'NDVI and NDMI', 'NORMALIZATION_METHOD': 'Inverse min-max',
    'COMBINATION_METHOD': 'Weighted linear combination',
    'NDVI_WEIGHT': float(fire_hazard_vegetation_dryness_index_weights['NDVI']),
    'NDMI_WEIGHT': float(fire_hazard_vegetation_dryness_index_weights['NDMI']),
    'NDVI_LOWER_BOUND': \
        float(fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['lower_bound']),
    'NDVI_UPPER_BOUND': \
        float(fire_hazard_vegetation_dryness_normalization_bounds['NDVI']['upper_bound']),
    'NDMI_LOWER_BOUND': \
        float(fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['lower_bound']),
    'NDMI_UPPER_BOUND': \
        float(fire_hazard_vegetation_dryness_normalization_bounds['NDMI']['upper_bound']),
    'OUTPUT_SCORE_MINIMUM': float(fire_hazard_vegetation_dryness_score_minimum),
    'OUTPUT_SCORE_MAXIMUM': float(fire_hazard_vegetation_dryness_score_maximum),
    'ACTUAL_SCORE_MINIMUM': float(combined_dryness_validation_record.iloc[0]['VALUE_MINIMUM']),
    'ACTUAL_SCORE_MAXIMUM': float(combined_dryness_validation_record.iloc[0]['VALUE_MAXIMUM']),
    'ACTUAL_SCORE_MEAN': float(combined_dryness_validation_record.iloc[0]['VALUE_MEAN']),
    'VALID_PIXELS': int(combined_dryness_validation_record.iloc[0]['VALID_PIXELS']),
    'MINIMUM_VALID_OBSERVATIONS': int(fire_hazard_vegetation_dryness_minimum_observations),
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': float(fire_hazard_cell_size),
    'TARGET_WIDTH': int(fire_hazard_alignment_width),
    'TARGET_HEIGHT': int(fire_hazard_alignment_height),
    'FORMULA_VALID': bool(fire_hazard_vegetation_dryness_formula_validation['VALID'].iloc[0]),
    'VALIDATION_COMPLETE': bool(fire_hazard_vegetation_dryness_validation_complete),
    'READY_FOR_COMPOSITE_HAZARD_GRID': True}])
# Define vegetation dryness final policy to control the inputs and rules used by vegetation-dryness
# hazard component.
fire_hazard_vegetation_dryness_final_policy = (
    {'component_id': 'VEGETATION_DRYNESS', 'component_name': 'Vegetation Dryness Hazard',
        'status': 'complete', 'primary_model_input': \
            str(fire_hazard_vegetation_dryness_hazard_path),
        'supporting_products': {'normalized_ndvi': \
            str(fire_hazard_vegetation_dryness_ndvi_score_path),
        'normalized_ndmi': str(fire_hazard_vegetation_dryness_ndmi_score_path)},
        'normalization': {'method': 'inverse_min_max',
        'score_range': [float(fire_hazard_vegetation_dryness_score_minimum),
        float(fire_hazard_vegetation_dryness_score_maximum)],
        'bounds': fire_hazard_vegetation_dryness_normalization_bounds,
        'interpretation': 'Higher output scores represent greater vegetation-dryness hazard.'},
        'combination': {'method': 'weighted_linear_combination',
        'weights': {'NDVI': float(fire_hazard_vegetation_dryness_index_weights['NDVI']),
        'NDMI': float(fire_hazard_vegetation_dryness_index_weights['NDMI'])},
        'formula': 'Vegetation Dryness = 0.40 * '
        'NDVI Dryness Score + 0.60 * '
        'NDMI Dryness Score'}, 'support_policy': {'minimum_valid_observations': \
            int(fire_hazard_vegetation_dryness_minimum_observations),
            'require_both_normalized_indices': \
                bool(fire_hazard_vegetation_dryness_require_both_indices)}, 'target_grid': \
                {'crs': fire_hazard_target_crs,
            'cell_size_meters': float(fire_hazard_cell_size),
            'width': int(fire_hazard_alignment_width), 'height': int(fire_hazard_alignment_height),
            'transform': [float(transform_value) for transform_value in \
                fire_hazard_alignment_transform],
            'nodata_value': float(fire_hazard_vegetation_dryness_output_nodata)}, 'validation': \
                {'product_validation_complete': \
                bool(fire_hazard_vegetation_dryness_validation_complete),
            'weighted_formula_valid': \
                bool(fire_hazard_vegetation_dryness_formula_validation['VALID'].iloc[0]),
            'formula_mismatch_pixels': \
                int(fire_hazard_vegetation_dryness_formula_validation['FORMULA_MISMATCH_PIXELS'] \
                .iloc[0]),
            'ready_for_composite_hazard_grid': True}}
)
# Save the vegetation dryness final registry CSV so later steps can reuse the recorded workflow
# results.
fire_hazard_vegetation_dryness_final_registry.to_csv \
    (fire_hazard_vegetation_dryness_final_registry_path,
    index=False)
# Save the vegetation dryness final summary CSV so later steps can reuse the recorded workflow
# results.
fire_hazard_vegetation_dryness_final_summary.to_csv \
    (fire_hazard_vegetation_dryness_final_summary_path,
    index=False)

# Open the file in a managed context so its handle closes reliably after vegetation-dryness hazard
# component.
with fire_hazard_vegetation_dryness_final_policy_path.open('w',
    encoding='utf-8') as final_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_vegetation_dryness_final_policy, final_policy_file, indent=2)
# Define vegetation dryness completion marker to control the inputs and rules used by
# vegetation-dryness hazard component.
fire_hazard_vegetation_dryness_completion_marker = {'component_id': 'VEGETATION_DRYNESS',
    'status': 'complete', 'primary_output': str(fire_hazard_vegetation_dryness_hazard_path),
    'validation_complete': True, 'ready_for_composite_hazard_grid': True,
    'final_registry': str(fire_hazard_vegetation_dryness_final_registry_path),
    'final_summary': str(fire_hazard_vegetation_dryness_final_summary_path),
    'final_policy': str(fire_hazard_vegetation_dryness_final_policy_path)}

# Open the file in a managed context so its handle closes reliably after vegetation-dryness hazard
# component.
with fire_hazard_vegetation_dryness_completion_marker_path.open('w',
    encoding='utf-8') as completion_marker_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_vegetation_dryness_completion_marker, completion_marker_file, indent=2)
# Build the component registry path location so vegetation-dryness hazard component uses the
# expected project file structure.
fire_hazard_component_registry_path = fire_hazard_vegetation_dryness_metadata_directory.parent / \
    'fire_hazard_component_registry.csv'
# Define vegetation dryness registry record to control the inputs and rules used by
# vegetation-dryness hazard component.
fire_hazard_vegetation_dryness_registry_record = {'COMPONENT_ID': 'VEGETATION_DRYNESS',
    'COMPONENT_NAME': 'Vegetation Dryness Hazard',
    'PRIMARY_OUTPUT_PATH': str(fire_hazard_vegetation_dryness_hazard_path),
    'SCORE_MINIMUM': float(fire_hazard_vegetation_dryness_score_minimum),
    'SCORE_MAXIMUM': float(fire_hazard_vegetation_dryness_score_maximum),
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': float(fire_hazard_cell_size),
    'VALIDATION_COMPLETE': True, 'COMPONENT_COMPLETE': True,
    'READY_FOR_FINAL_MODEL': True}

# Use the existing file only when it is present and valid for vegetation-dryness hazard component.
if fire_hazard_component_registry_path.exists():
    # Store component registry needed to carry out vegetation-dryness hazard component.
    fire_hazard_component_registry = pd.read_csv(fire_hazard_component_registry_path)
else:
    # Store component registry needed to carry out vegetation-dryness hazard component.
    fire_hazard_component_registry = pd.DataFrame()

# Handle missing or invalid component registry explicitly during vegetation-dryness hazard
# component.
if not fire_hazard_component_registry.empty:
    # Create an isolated component registry working copy so vegetation-dryness hazard component does
    # not modify upstream data.
    fire_hazard_component_registry = \
        fire_hazard_component_registry[fire_hazard_component_registry['COMPONENT_ID'].astype(str) \
        .str.upper() != 'VEGETATION_DRYNESS'].copy()
# Store component registry needed to carry out vegetation-dryness hazard component.
fire_hazard_component_registry = pd.concat([fire_hazard_component_registry,
    pd.DataFrame([fire_hazard_vegetation_dryness_registry_record])],
    ignore_index=True, sort=False).reset_index(drop=True)
# Save the component registry CSV so later steps can reuse the recorded workflow results.
fire_hazard_component_registry.to_csv(fire_hazard_component_registry_path, index=False)
# Store vegetation dryness component finalized needed to carry out vegetation-dryness hazard
# component.
fire_hazard_vegetation_dryness_component_finalized = \
    all([fire_hazard_vegetation_dryness_validation_complete,
    fire_hazard_vegetation_dryness_hazard_path.exists(),
    fire_hazard_vegetation_dryness_final_registry_path.exists(),
    fire_hazard_vegetation_dryness_final_summary_path.exists(),
    fire_hazard_vegetation_dryness_final_policy_path.exists(),
    fire_hazard_vegetation_dryness_completion_marker_path.exists()])

# Stop execution if vegetation-dryness component did not pass final completion checks.
if not fire_hazard_vegetation_dryness_component_finalized:
    raise ValueError('The vegetation-dryness component did not pass final completion checks.')
print('-> Component ID: VEGETATION_DRYNESS')
print('-> Component status: Complete')
print(f'-> Primary hazard input: {fire_hazard_vegetation_dryness_hazard_path}')
print(f"-> NDVI weight: {fire_hazard_vegetation_dryness_index_weights['NDVI']:.2f}")
print(f"-> NDMI weight: {fire_hazard_vegetation_dryness_index_weights['NDMI']:.2f}")
print(f"-> Valid hazard pixels: "
    f"{int(combined_dryness_validation_record.iloc[0]['VALID_PIXELS']):,}")
print(f"-> Actual hazard-score range: "
    f"{float(combined_dryness_validation_record.iloc[0]['VALUE_MINIMUM']):.4f} "
    f"to "
    f"{float(combined_dryness_validation_record.iloc[0]['VALUE_MAXIMUM']):.4f}")
print(f"-> Weighted formula valid: "
    f"{bool(fire_hazard_vegetation_dryness_formula_validation['VALID'].iloc[0])}")
print(f'-> Final product registry saved: {fire_hazard_vegetation_dryness_final_registry_path}')
print(f'-> Final component summary saved: {fire_hazard_vegetation_dryness_final_summary_path}')
print(f'-> Final component policy saved: {fire_hazard_vegetation_dryness_final_policy_path}')
print(f'-> Completion marker saved: {fire_hazard_vegetation_dryness_completion_marker_path}')
print(f'-> Component registry saved: {fire_hazard_component_registry_path}')
print(f'-> Component finalized: {fire_hazard_vegetation_dryness_component_finalized}')
print('\n--- FINAL VEGETATION-DRYNESS COMPONENT SUMMARY ---')
display(fire_hazard_vegetation_dryness_final_summary)
print('\n--- FINAL VEGETATION-DRYNESS PRODUCT REGISTRY ---')
display(fire_hazard_vegetation_dryness_final_registry)
print('\n--- FIRE-HAZARD COMPONENT REGISTRY SAMPLE ---')
display(fire_hazard_component_registry.head(5))
import gc
gc.collect()
print('\nNOTE:')
print('The vegetation-dryness hazard '
    'component is now fully '
    'processed, independently '
    'validated, documented, '
    'registered, and finalized.')
print('The combined '
    'vegetation-dryness hazard '
    'raster is the only vegetation '
    'product that should receive a '
    'component-level weight in the '
    'final Composite Fire Hazard '
    'Grid.')
print('The normalized NDVI and NDMI '
    'rasters remain supporting '
    'products for auditing, '
    'interpretation, and '
    'sensitivity analysis.')
print('Future notebook sessions can '
    'restore the final '
    'vegetation-dryness component '
    'from the saved registry, '
    'summary, policy, and '
    'completion-marker files '
    'without repeating the HLS '
    'workflow.')
print('\n=== VEGETATION-DRYNESS HAZARD COMPONENT FINALIZED ===')


=== FINALIZING VEGETATION-DRYNESS HAZARD COMPONENT ===
-> Component ID: VEGETATION_DRYNESS
-> Component status: Complete
-> Primary hazard input: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\hazard_raster\vegetation_dryness_hazard_score.tif
-> NDVI weight: 0.40
-> NDMI weight: 0.60
-> Valid hazard pixels: 14,140,492
-> Actual hazard-score range: 0.0000 to 1.0000
-> Weighted formula valid: True
-> Final product registry saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_final_product_registry.csv
-> Final component summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\vegetation\hls_planetary_computer_2025_fire_season\vegetation_dryness_component\metadata\vegetation_dryness_final_component_summary.csv
-> Final component policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\v

,COMPONENT_ID,COMPONENT_NAME,COMPONENT_STATUS,PRIMARY_OUTPUT,SOURCE_INDICES,NORMALIZATION_METHOD,COMBINATION_METHOD,NDVI_WEIGHT,NDMI_WEIGHT,NDVI_LOWER_BOUND,...,ACTUAL_SCORE_MEAN,VALID_PIXELS,MINIMUM_VALID_OBSERVATIONS,TARGET_CRS,CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,FORMULA_VALID,VALIDATION_COMPLETE,READY_FOR_COMPOSITE_HAZARD_GRID
0,VEGETATION_DRYNESS,Vegetation Dryness Hazard,Complete,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,NDVI and NDMI,Inverse min-max,Weighted linear combination,0.4,0.6,0.1,...,0.671171,14140492,3,EPSG:26912,30.0,9454,14467,True,True,True



--- FINAL VEGETATION-DRYNESS PRODUCT REGISTRY ---


,COMPONENT_ID,PRODUCT_ID,PRODUCT_ROLE,SOURCE_INDEX,LOCAL_PATH,SCORE_MINIMUM,SCORE_MAXIMUM,SCORE_MEAN,VALID_PIXELS,FILE_SIZE_BYTES,USE_IN_FINAL_HAZARD_MODEL,VALID
0,VEGETATION_DRYNESS,NDVI_DRYNESS_SCORE,Normalized supporting layer,NDVI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.0,1.0,0.605157,14140492,46867582,False,True
1,VEGETATION_DRYNESS,NDMI_DRYNESS_SCORE,Normalized supporting layer,NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.0,1.0,0.715181,14140492,45775795,False,True
2,VEGETATION_DRYNESS,VEGETATION_DRYNESS_HAZARD_SCORE,Final component hazard input,NDVI and NDMI,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.0,1.0,0.671171,14140492,46055631,True,True



--- FIRE-HAZARD COMPONENT REGISTRY SAMPLE ---


,COMPONENT_ID,COMPONENT_NAME,PRIMARY_OUTPUT_PATH,SCORE_MINIMUM,SCORE_MAXIMUM,TARGET_CRS,CELL_SIZE_METERS,VALIDATION_COMPLETE,COMPONENT_COMPLETE,READY_FOR_FINAL_MODEL
0,VEGETATION_DRYNESS,Vegetation Dryness Hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.0,1.0,EPSG:26912,30.0,True,True,True



NOTE:
The vegetation-dryness hazard component is now fully processed, independently validated, documented, registered, and finalized.
The combined vegetation-dryness hazard raster is the only vegetation product that should receive a component-level weight in the final Composite Fire Hazard Grid.
The normalized NDVI and NDMI rasters remain supporting products for auditing, interpretation, and sensitivity analysis.
Future notebook sessions can restore the final vegetation-dryness component from the saved registry, summary, policy, and completion-marker files without repeating the HLS workflow.

=== VEGETATION-DRYNESS HAZARD COMPONENT FINALIZED ===


# CHRG Phase 8 – Fuel Hazard

## Purpose

Acquire and align LANDFIRE fuel layers, reclassify FBFM40, normalize canopy cover and canopy bulk density, and combine the inputs into the composite fuel-hazard component.


## LANDFIRE Fuel-Data Acquisition


### Acquiring LANDFIRE Fuel Source Data


In [96]:
print('=== ACQUIRING LANDFIRE FUEL SOURCE DATA ===')

# ----------------------------------------------------
# VALIDATE REQUIRED INPUTS
# ----------------------------------------------------
required_fire_hazard_fuel_acquisition_inputs = [
    'fire_hazard_source_catalog',
    'fire_hazard_raw_fuels_directory',
    'fire_hazard_data_mode',
    'fire_hazard_request_session',
    'fire_hazard_request_timeout',
    'fire_hazard_verify_ssl_certificates',
    'fire_hazard_dataset_versions',
    'download_fire_hazard_source_file',
    'gdf_county_boundaries_aligned',
]

missing_fire_hazard_fuel_acquisition_inputs = [
    object_name
    for object_name in required_fire_hazard_fuel_acquisition_inputs
    if object_name not in globals()
]

if missing_fire_hazard_fuel_acquisition_inputs:
    raise NameError(
        f'The following LANDFIRE fuel-acquisition objects are missing:\n'
        f'{missing_fire_hazard_fuel_acquisition_inputs}\n\n'
        f'Run the source-catalog, directory, cache-policy, retry-session, '
        f'download-function, and county-boundary preparation cells first.'
    )

if gdf_county_boundaries_aligned.empty:
    raise ValueError('The aligned county-boundary dataset contains no records.')

if gdf_county_boundaries_aligned.crs is None:
    raise ValueError(
        'The aligned county-boundary dataset does not have a coordinate reference system.'
    )

# ----------------------------------------------------
# DEFINE LANDFIRE FUEL IMAGE SERVICES
# ----------------------------------------------------
fire_hazard_fuel_image_services = {
    'LANDFIRE_FBFM40': {
        'service_url': (
            'https://lfps.usgs.gov/arcgis/rest/services/'
            'Landfire_LF2025/LF2025_FBFM40_CONUS/ImageServer'
        ),
        'output_filename': 'landfire_2025_fbfm40_study_area.tif',
        'source_name': (
            f"{fire_hazard_dataset_versions['landfire']} Fire Behavior Fuel Model 40"
        ),
        'expected_pixel_type': 'S16',
    },
    'LANDFIRE_CANOPY_COVER': {
        'service_url': (
            'https://lfps.usgs.gov/arcgis/rest/services/'
            'Landfire_LF2025/LF2025_CC_CONUS/ImageServer'
        ),
        'output_filename': 'landfire_2025_canopy_cover_study_area.tif',
        'source_name': (
            f"{fire_hazard_dataset_versions['landfire']} Forest Canopy Cover"
        ),
        'expected_pixel_type': 'S16',
    },
    'LANDFIRE_CANOPY_BULK_DENSITY': {
        'service_url': (
            'https://lfps.usgs.gov/arcgis/rest/services/'
            'Landfire_LF2025/LF2025_CBD_CONUS/ImageServer'
        ),
        'output_filename': 'landfire_2025_canopy_bulk_density_study_area.tif',
        'source_name': (
            f"{fire_hazard_dataset_versions['landfire']} Forest Canopy Bulk Density"
        ),
        'expected_pixel_type': 'S16',
    },
}

# ----------------------------------------------------
# DEFINE LANDFIRE NATIVE GRID
# ----------------------------------------------------
import contextlib
import io
import math
import shutil
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window

fire_hazard_landfire_native_crs = 'EPSG:5070'
fire_hazard_landfire_native_cell_size = 30
fire_hazard_landfire_extent_buffer_meters = fire_hazard_landfire_native_cell_size * 3
fire_hazard_landfire_preferred_tile_dimension = 4000

gdf_fire_hazard_counties_landfire = gdf_county_boundaries_aligned.to_crs(
    fire_hazard_landfire_native_crs
)

(
    fire_hazard_landfire_xmin,
    fire_hazard_landfire_ymin,
    fire_hazard_landfire_xmax,
    fire_hazard_landfire_ymax,
) = gdf_fire_hazard_counties_landfire.total_bounds

fire_hazard_landfire_xmin -= fire_hazard_landfire_extent_buffer_meters
fire_hazard_landfire_ymin -= fire_hazard_landfire_extent_buffer_meters
fire_hazard_landfire_xmax += fire_hazard_landfire_extent_buffer_meters
fire_hazard_landfire_ymax += fire_hazard_landfire_extent_buffer_meters

fire_hazard_landfire_image_width = math.ceil(
    (fire_hazard_landfire_xmax - fire_hazard_landfire_xmin)
    / fire_hazard_landfire_native_cell_size
)

fire_hazard_landfire_image_height = math.ceil(
    (fire_hazard_landfire_ymax - fire_hazard_landfire_ymin)
    / fire_hazard_landfire_native_cell_size
)

fire_hazard_landfire_grid_xmax = (
    fire_hazard_landfire_xmin
    + fire_hazard_landfire_image_width * fire_hazard_landfire_native_cell_size
)
fire_hazard_landfire_grid_ymax = fire_hazard_landfire_ymax
fire_hazard_landfire_grid_ymin = (
    fire_hazard_landfire_grid_ymax
    - fire_hazard_landfire_image_height * fire_hazard_landfire_native_cell_size
)

fire_hazard_landfire_transform = from_origin(
    fire_hazard_landfire_xmin,
    fire_hazard_landfire_grid_ymax,
    fire_hazard_landfire_native_cell_size,
    fire_hazard_landfire_native_cell_size,
)

# ----------------------------------------------------
# DEFINE LOCAL SOURCE AND TILE PATHS
# ----------------------------------------------------
fire_hazard_fuel_source_paths = {
    source_id: fire_hazard_raw_fuels_directory / source_settings['output_filename']
    for source_id, source_settings in fire_hazard_fuel_image_services.items()
}

fire_hazard_raw_fuels_directory.mkdir(parents=True, exist_ok=True)

fire_hazard_landfire_tile_root_directory = (
    fire_hazard_raw_fuels_directory / '_landfire_export_tiles'
)
fire_hazard_landfire_tile_root_directory.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------
# UPDATE SOURCE CATALOG
# ----------------------------------------------------
for source_id, source_settings in fire_hazard_fuel_image_services.items():
    source_catalog_mask = fire_hazard_source_catalog['SOURCE_ID'] == source_id

    if source_catalog_mask.sum() != 1:
        raise ValueError(
            f'The fire-hazard source catalog must contain exactly one record for:\n{source_id}'
        )

    fire_hazard_source_catalog.loc[source_catalog_mask, 'ACCESS_METHOD'] = (
        'ArcGIS ImageServer tiled exportImage direct image stream'
    )
    fire_hazard_source_catalog.loc[source_catalog_mask, 'ACCESS_URL'] = (
        source_settings['service_url']
    )
    fire_hazard_source_catalog.loc[source_catalog_mask, 'PREFERRED_FORMAT'] = 'GeoTIFF'
    fire_hazard_source_catalog.loc[source_catalog_mask, 'EXPECTED_ARCHIVE_FORMAT'] = (
        'Not applicable'
    )
    fire_hazard_source_catalog.loc[source_catalog_mask, 'LOCAL_FILENAME'] = (
        source_settings['output_filename']
    )
    fire_hazard_source_catalog.loc[source_catalog_mask, 'LOCAL_PATH'] = [
        fire_hazard_fuel_source_paths[source_id]
    ]

# ----------------------------------------------------
# VALIDATE SNAPSHOT MODE
# ----------------------------------------------------
missing_fire_hazard_fuel_source_ids = [
    source_id
    for source_id, source_path in fire_hazard_fuel_source_paths.items()
    if (
        not source_path.exists()
        or not source_path.is_file()
        or source_path.stat().st_size <= 0
    )
]

if fire_hazard_data_mode == 'snapshot' and missing_fire_hazard_fuel_source_ids:
    raise FileNotFoundError(
        f"Cached LANDFIRE fuel acquisition was selected, but the following required "
        f"fuel sources are missing:\n{missing_fire_hazard_fuel_source_ids}\n\n"
        f"Change fire_hazard_data_mode to 'refresh', rerun the cache-policy, "
        f"retry-session, download-function, and LANDFIRE acquisition cells."
    )

# ----------------------------------------------------
# SERVICE METADATA HELPER
# ----------------------------------------------------
def get_landfire_image_service_metadata(service_url, source_name):
    metadata_response = None
    try:
        metadata_response = fire_hazard_request_session.get(
            service_url,
            params={'f': 'pjson'},
            timeout=fire_hazard_request_timeout,
            verify=fire_hazard_verify_ssl_certificates,
        )
        metadata_response.raise_for_status()
        metadata = metadata_response.json()

        if 'error' in metadata:
            raise ValueError(
                f'LANDFIRE service metadata returned an error for {source_name}:\n'
                f"{metadata['error']}"
            )

        service_max_width = metadata.get('maxImageWidth')
        service_max_height = metadata.get('maxImageHeight')

        if not isinstance(service_max_width, int) or service_max_width < 1:
            raise ValueError(f'{source_name} did not report a valid maxImageWidth.')
        if not isinstance(service_max_height, int) or service_max_height < 1:
            raise ValueError(f'{source_name} did not report a valid maxImageHeight.')

        return {
            'MAX_IMAGE_WIDTH': service_max_width,
            'MAX_IMAGE_HEIGHT': service_max_height,
            'SUPPORTED_IMAGE_FORMAT_TYPES': metadata.get('supportedImageFormatTypes'),
            'PIXEL_TYPE': metadata.get('pixelType'),
        }
    finally:
        if metadata_response is not None:
            try:
                metadata_response.close()
            except Exception:
                pass

# ----------------------------------------------------
# TILE GRID HELPER
# ----------------------------------------------------
def build_landfire_export_tiles(tile_width_pixels, tile_height_pixels):
    tile_records = []
    tile_number = 0

    for row_offset in range(
        0,
        fire_hazard_landfire_image_height,
        tile_height_pixels,
    ):
        current_tile_height = min(
            tile_height_pixels,
            fire_hazard_landfire_image_height - row_offset,
        )

        for column_offset in range(
            0,
            fire_hazard_landfire_image_width,
            tile_width_pixels,
        ):
            current_tile_width = min(
                tile_width_pixels,
                fire_hazard_landfire_image_width - column_offset,
            )

            tile_number += 1

            tile_xmin = (
                fire_hazard_landfire_xmin
                + column_offset * fire_hazard_landfire_native_cell_size
            )
            tile_xmax = (
                tile_xmin
                + current_tile_width * fire_hazard_landfire_native_cell_size
            )
            tile_ymax = (
                fire_hazard_landfire_grid_ymax
                - row_offset * fire_hazard_landfire_native_cell_size
            )
            tile_ymin = (
                tile_ymax
                - current_tile_height * fire_hazard_landfire_native_cell_size
            )

            tile_records.append({
                'TILE_NUMBER': tile_number,
                'ROW_OFFSET': row_offset,
                'COLUMN_OFFSET': column_offset,
                'WIDTH': current_tile_width,
                'HEIGHT': current_tile_height,
                'XMIN': tile_xmin,
                'YMIN': tile_ymin,
                'XMAX': tile_xmax,
                'YMAX': tile_ymax,
                'BBOX': f'{tile_xmin},{tile_ymin},{tile_xmax},{tile_ymax}',
            })

    return tile_records

# ----------------------------------------------------
# QUIET TILE DOWNLOAD HELPER
# ----------------------------------------------------
def download_landfire_tile_quietly(**download_kwargs):
    """
    Suppresses normal per-file downloader chatter while preserving the complete
    download/retry behavior. If a tile fails, captured diagnostic output is printed.
    """
    suppressed_output = io.StringIO()

    try:
        with contextlib.redirect_stdout(suppressed_output):
            return download_fire_hazard_source_file(**download_kwargs)
    except Exception:
        captured_output = suppressed_output.getvalue().strip()
        if captured_output:
            print(captured_output)
        raise

# ----------------------------------------------------
# TILED LANDFIRE ACQUISITION FUNCTION
# ----------------------------------------------------
def request_landfire_fuel_export(source_id, source_settings):
    source_name = source_settings['source_name']
    service_url = source_settings['service_url']
    destination_path = fire_hazard_fuel_source_paths[source_id]

    if (
        fire_hazard_data_mode == 'snapshot'
        and destination_path.exists()
        and destination_path.is_file()
        and destination_path.stat().st_size > 0
    ):
        cached_size_bytes = destination_path.stat().st_size
        print(f'\nUsing cached LANDFIRE source: {source_name}')
        print(f'-> Cached source size: {cached_size_bytes / 1024 ** 2:,.2f} MB')

        return {
            'SOURCE_ID': source_id,
            'SOURCE_NAME': source_name,
            'IMAGE_SERVICE_URL': service_url,
            'ACCESS_METHOD': 'Cached tiled LANDFIRE mosaic',
            'ACQUISITION_ACTION': 'Used cached source',
            'CACHE_USED': True,
            'REFRESH_REQUESTED': False,
            'HTTP_STATUS': None,
            'CONTENT_TYPE': None,
            'FILE_SIZE_BYTES': cached_size_bytes,
            'FILE_SIZE_MB': cached_size_bytes / 1024 ** 2,
            'DOWNLOAD_ATTEMPTS_USED': 0,
            'SOURCE_CRS': fire_hazard_landfire_native_crs,
            'REQUESTED_CELL_SIZE_METERS': fire_hazard_landfire_native_cell_size,
            'REQUESTED_WIDTH': fire_hazard_landfire_image_width,
            'REQUESTED_HEIGHT': fire_hazard_landfire_image_height,
            'TILE_COUNT': None,
            'SERVICE_MAX_IMAGE_WIDTH': None,
            'SERVICE_MAX_IMAGE_HEIGHT': None,
            'TILE_MAX_WIDTH': None,
            'TILE_MAX_HEIGHT': None,
            'DESTINATION_PATH': destination_path,
            'VALID': True,
        }

    service_metadata = get_landfire_image_service_metadata(
        service_url=service_url,
        source_name=source_name,
    )

    service_max_width = service_metadata['MAX_IMAGE_WIDTH']
    service_max_height = service_metadata['MAX_IMAGE_HEIGHT']

    tile_width_pixels = min(
        service_max_width,
        fire_hazard_landfire_preferred_tile_dimension,
    )
    tile_height_pixels = min(
        service_max_height,
        fire_hazard_landfire_preferred_tile_dimension,
    )

    tile_records = build_landfire_export_tiles(
        tile_width_pixels=tile_width_pixels,
        tile_height_pixels=tile_height_pixels,
    )
    total_tiles = len(tile_records)

    # Clean, professional progress output.
    print(f'\nPreparing tiled LANDFIRE export: {source_name}')
    print(f'-> Tiles required: {total_tiles:,}')

    source_tile_directory = (
        fire_hazard_landfire_tile_root_directory / source_id.lower()
    )

    if source_tile_directory.exists():
        shutil.rmtree(source_tile_directory)

    source_tile_directory.mkdir(parents=True, exist_ok=True)

    tile_download_records = []
    tile_paths = []
    export_image_url = service_url + '/exportImage'

    for tile_record in tile_records:
        tile_number = tile_record['TILE_NUMBER']
        tile_path = (
            source_tile_directory
            / f'{source_id.lower()}_tile_{tile_number:03d}.tif'
        )

        export_image_parameters = {
            'bbox': tile_record['BBOX'],
            'bboxSR': '5070',
            'size': f"{tile_record['WIDTH']},{tile_record['HEIGHT']}",
            'imageSR': '5070',
            'format': 'tiff',
            'interpolation': 'RSP_NearestNeighbor',
            'pixelType': source_settings['expected_pixel_type'],
            'noDataInterpretation': 'esriNoDataMatchAny',
            'f': 'image',
        }

        try:
            tile_download_record = download_landfire_tile_quietly(
                source_url=export_image_url,
                destination_path=tile_path,
                source_name=f'{source_name} - tile {tile_number} of {total_tiles}',
                expected_content_types={
                    'image/tiff',
                    'image/geotiff',
                    'application/geotiff',
                    'application/octet-stream',
                },
                request_params=export_image_parameters,
                force_refresh=True,
                minimum_size_bytes=1024,
            )
        except Exception as tile_error:
            print(f'-> Tile {tile_number:,} of {total_tiles:,} failed')
            print(f'-> Failure message: {tile_error}')
            raise

        tile_download_record['TILE_NUMBER'] = tile_number
        tile_download_record['ROW_OFFSET'] = tile_record['ROW_OFFSET']
        tile_download_record['COLUMN_OFFSET'] = tile_record['COLUMN_OFFSET']
        tile_download_record['EXPECTED_WIDTH'] = tile_record['WIDTH']
        tile_download_record['EXPECTED_HEIGHT'] = tile_record['HEIGHT']

        with rasterio.open(tile_path) as tile_source:
            tile_structure_valid = (
                tile_source.count == 1
                and tile_source.width == tile_record['WIDTH']
                and tile_source.height == tile_record['HEIGHT']
                and tile_source.crs is not None
                and tile_source.crs.to_epsg() == 5070
            )

        if not tile_structure_valid:
            raise ValueError(
                f'LANDFIRE tile {tile_number} for {source_name} failed raster structure validation.'
            )

        tile_download_record['STRUCTURE_VALID'] = True
        tile_download_records.append(tile_download_record)
        tile_paths.append(tile_path)

        # Print every 3 tiles and always the final tile.
        if tile_number % 3 == 0 or tile_number == total_tiles:
            print(f'-> Tile {tile_number:,} of {total_tiles:,} complete')

    if len(tile_paths) != total_tiles:
        raise ValueError(
            f'Only {len(tile_paths):,} of {total_tiles:,} LANDFIRE tiles were acquired '
            f'for {source_name}.'
        )

    if destination_path.exists():
        destination_path.unlink()

    with rasterio.open(tile_paths[0]) as first_tile_source:
        output_dtype = first_tile_source.dtypes[0]
        output_nodata = first_tile_source.nodata
        output_profile = first_tile_source.profile.copy()

    output_profile.update({
        'driver': 'GTiff',
        'width': fire_hazard_landfire_image_width,
        'height': fire_hazard_landfire_image_height,
        'count': 1,
        'crs': fire_hazard_landfire_native_crs,
        'transform': fire_hazard_landfire_transform,
        'dtype': output_dtype,
        'nodata': output_nodata,
        'compress': 'LZW',
        'BIGTIFF': 'IF_SAFER',
        'tiled': True,
    })

    # Assemble individual tiles directly into their exact output windows.
    with rasterio.open(destination_path, 'w', **output_profile) as mosaic_destination:
        for tile_record, tile_path in zip(tile_records, tile_paths):
            with rasterio.open(tile_path) as tile_source:
                tile_array = tile_source.read(1)

            destination_window = Window(
                col_off=tile_record['COLUMN_OFFSET'],
                row_off=tile_record['ROW_OFFSET'],
                width=tile_record['WIDTH'],
                height=tile_record['HEIGHT'],
            )

            mosaic_destination.write(
                tile_array,
                1,
                window=destination_window,
            )

        mosaic_destination.set_band_description(1, source_name)
        mosaic_destination.update_tags(
            SOURCE='LANDFIRE',
            SOURCE_ID=source_id,
            ACCESS_METHOD='Tiled ArcGIS ImageServer exportImage f=image',
            SOURCE_CRS=fire_hazard_landfire_native_crs,
            CELL_SIZE_METERS=fire_hazard_landfire_native_cell_size,
            TILE_COUNT=total_tiles,
        )

    final_file_exists = destination_path.exists() and destination_path.is_file()
    final_file_size_bytes = destination_path.stat().st_size if final_file_exists else 0

    if not final_file_exists or final_file_size_bytes < 1024:
        raise IOError(
            f'The completed tiled LANDFIRE mosaic is missing or invalid:\n{destination_path}'
        )

    with rasterio.open(destination_path) as final_source:
        final_grid_valid = (
            final_source.count == 1
            and final_source.width == fire_hazard_landfire_image_width
            and final_source.height == fire_hazard_landfire_image_height
            and final_source.crs is not None
            and final_source.crs.to_epsg() == 5070
            and final_source.transform == fire_hazard_landfire_transform
        )

    if not final_grid_valid:
        raise ValueError(
            f'The completed LANDFIRE mosaic grid failed validation for {source_name}.'
        )

    total_download_attempts = sum(
        int(record.get('DOWNLOAD_ATTEMPTS_USED', 0) or 0)
        for record in tile_download_records
    )

    print(f'-> Mosaic complete: {final_file_size_bytes / 1024 ** 2:,.2f} MB')

    return {
        'SOURCE_ID': source_id,
        'SOURCE_NAME': source_name,
        'IMAGE_SERVICE_URL': service_url,
        'ACCESS_METHOD': 'Tiled ArcGIS ImageServer exportImage f=image',
        'ACQUISITION_ACTION': 'Downloaded and assembled tiled source',
        'CACHE_USED': False,
        'REFRESH_REQUESTED': True,
        'HTTP_STATUS': 200,
        'CONTENT_TYPE': 'image/tiff',
        'FILE_SIZE_BYTES': final_file_size_bytes,
        'FILE_SIZE_MB': final_file_size_bytes / 1024 ** 2,
        'DOWNLOAD_ATTEMPTS_USED': total_download_attempts,
        'SOURCE_CRS': fire_hazard_landfire_native_crs,
        'REQUESTED_CELL_SIZE_METERS': fire_hazard_landfire_native_cell_size,
        'REQUESTED_WIDTH': fire_hazard_landfire_image_width,
        'REQUESTED_HEIGHT': fire_hazard_landfire_image_height,
        'TILE_COUNT': total_tiles,
        'SERVICE_MAX_IMAGE_WIDTH': service_max_width,
        'SERVICE_MAX_IMAGE_HEIGHT': service_max_height,
        'TILE_MAX_WIDTH': tile_width_pixels,
        'TILE_MAX_HEIGHT': tile_height_pixels,
        'DESTINATION_PATH': destination_path,
        'VALID': True,
    }

# ----------------------------------------------------
# ACQUIRE REQUIRED LANDFIRE PRODUCTS
# ----------------------------------------------------
fire_hazard_fuel_acquisition_records = []

for source_id, source_settings in fire_hazard_fuel_image_services.items():
    fuel_acquisition_record = request_landfire_fuel_export(
        source_id=source_id,
        source_settings=source_settings,
    )
    fire_hazard_fuel_acquisition_records.append(fuel_acquisition_record)

# ----------------------------------------------------
# BUILD ACQUISITION SUMMARY
# ----------------------------------------------------
fire_hazard_fuel_acquisition_summary = pd.DataFrame(
    fire_hazard_fuel_acquisition_records
)

preferred_fire_hazard_fuel_summary_fields = [
    'SOURCE_ID',
    'SOURCE_NAME',
    'ACCESS_METHOD',
    'ACQUISITION_ACTION',
    'CACHE_USED',
    'REFRESH_REQUESTED',
    'HTTP_STATUS',
    'CONTENT_TYPE',
    'FILE_SIZE_MB',
    'DOWNLOAD_ATTEMPTS_USED',
    'SOURCE_CRS',
    'REQUESTED_CELL_SIZE_METERS',
    'REQUESTED_WIDTH',
    'REQUESTED_HEIGHT',
    'TILE_COUNT',
    'SERVICE_MAX_IMAGE_WIDTH',
    'SERVICE_MAX_IMAGE_HEIGHT',
    'TILE_MAX_WIDTH',
    'TILE_MAX_HEIGHT',
    'DESTINATION_PATH',
    'VALID',
]

available_fire_hazard_fuel_summary_fields = [
    field_name
    for field_name in preferred_fire_hazard_fuel_summary_fields
    if field_name in fire_hazard_fuel_acquisition_summary.columns
]

fire_hazard_fuel_acquisition_summary = fire_hazard_fuel_acquisition_summary[
    available_fire_hazard_fuel_summary_fields
]

# ----------------------------------------------------
# VALIDATE FINAL LANDFIRE FILES
# ----------------------------------------------------
fire_hazard_fuel_file_validation_records = []

for source_id, source_path in fire_hazard_fuel_source_paths.items():
    source_exists = source_path.exists() and source_path.is_file()
    source_size_bytes = source_path.stat().st_size if source_exists else 0
    source_valid = source_exists and source_size_bytes >= 1024

    fire_hazard_fuel_file_validation_records.append({
        'SOURCE_ID': source_id,
        'SOURCE_PATH': str(source_path),
        'FILE_EXISTS': source_exists,
        'FILE_SIZE_BYTES': source_size_bytes,
        'FILE_SIZE_MB': source_size_bytes / 1024 ** 2,
        'VALID': source_valid,
    })

fire_hazard_fuel_file_validation_summary = pd.DataFrame(
    fire_hazard_fuel_file_validation_records
)

failed_fire_hazard_fuel_files = fire_hazard_fuel_file_validation_summary[
    ~fire_hazard_fuel_file_validation_summary['VALID']
]

fire_hazard_fuel_data_acquired = (
    failed_fire_hazard_fuel_files.empty
    and len(fire_hazard_fuel_file_validation_summary)
    == len(fire_hazard_fuel_image_services)
)

if not fire_hazard_fuel_data_acquired:
    raise ValueError(
        'One or more required LANDFIRE fuel files failed acquisition validation.'
    )

# ----------------------------------------------------
# UPDATE SOURCE-CATALOG CACHE STATE
# ----------------------------------------------------
for source_id, source_path in fire_hazard_fuel_source_paths.items():
    source_catalog_mask = fire_hazard_source_catalog['SOURCE_ID'] == source_id

    fire_hazard_source_catalog.loc[source_catalog_mask, 'CACHE_EXISTS'] = (
        source_path.exists()
    )
    fire_hazard_source_catalog.loc[source_catalog_mask, 'CACHE_SIZE_BYTES'] = (
        source_path.stat().st_size if source_path.exists() else None
    )

# ----------------------------------------------------
# REPORT LANDFIRE ACQUISITION RESULTS
# ----------------------------------------------------
print(f'\n-> LANDFIRE acquisition mode: {fire_hazard_data_mode}')
print('-> LANDFIRE access method: service-aware tiled exportImage stream')
print(f'-> Fuel products required: {len(fire_hazard_fuel_image_services):,}')
print(
    f'-> Fuel products available: '
    f'{len(fire_hazard_fuel_file_validation_summary):,}'
)
print(f'-> LANDFIRE source CRS: {fire_hazard_landfire_native_crs}')
print(
    f'-> LANDFIRE source resolution: '
    f'{fire_hazard_landfire_native_cell_size} meters'
)
print(
    f'-> Final study-area image dimensions: '
    f'{fire_hazard_landfire_image_width:,} × '
    f'{fire_hazard_landfire_image_height:,}'
)
print(f'-> Fuel acquisition valid: {fire_hazard_fuel_data_acquired}')

print('\n--- LANDFIRE FUEL ACQUISITION SUMMARY ---')
display(fire_hazard_fuel_acquisition_summary)

print('\n--- LANDFIRE FUEL FILE VALIDATION ---')
display(fire_hazard_fuel_file_validation_summary)

print('\nNOTE:')
print(
    'Each LANDFIRE ImageServer is queried for its advertised maximum export '
    'width and height before acquisition.'
)
print(
    'The buffered study-area raster is divided into pixel-aligned export tiles '
    'that do not exceed the service limit or the project 4,000-pixel safety cap.'
)
print(
    'Each tile is downloaded independently with f=image and the reusable whole-file '
    'retry function, then written into its exact window in a local 30-meter EPSG:5070 '
    'GeoTIFF.'
)
print(
    'The final files remain raw study-area LANDFIRE sources. Exact county clipping, '
    'reprojection to the project CRS, reference-grid alignment, reclassification, and '
    'normalization occur in the downstream fuel-hazard component-processing phase.'
)

print('\n=== LANDFIRE FUEL SOURCE DATA ACQUISITION COMPLETE ===')


=== ACQUIRING LANDFIRE FUEL SOURCE DATA ===

Using cached LANDFIRE source: LF2025 Fire Behavior Fuel Model 40
-> Cached source size: 39.25 MB

Using cached LANDFIRE source: LF2025 Forest Canopy Cover
-> Cached source size: 18.03 MB

Using cached LANDFIRE source: LF2025 Forest Canopy Bulk Density
-> Cached source size: 19.47 MB

-> LANDFIRE acquisition mode: snapshot
-> LANDFIRE access method: service-aware tiled exportImage stream
-> Fuel products required: 3
-> Fuel products available: 3
-> LANDFIRE source CRS: EPSG:5070
-> LANDFIRE source resolution: 30 meters
-> Final study-area image dimensions: 10,934 × 13,996
-> Fuel acquisition valid: True

--- LANDFIRE FUEL ACQUISITION SUMMARY ---


,SOURCE_ID,SOURCE_NAME,ACCESS_METHOD,ACQUISITION_ACTION,CACHE_USED,REFRESH_REQUESTED,HTTP_STATUS,CONTENT_TYPE,FILE_SIZE_MB,DOWNLOAD_ATTEMPTS_USED,...,REQUESTED_CELL_SIZE_METERS,REQUESTED_WIDTH,REQUESTED_HEIGHT,TILE_COUNT,SERVICE_MAX_IMAGE_WIDTH,SERVICE_MAX_IMAGE_HEIGHT,TILE_MAX_WIDTH,TILE_MAX_HEIGHT,DESTINATION_PATH,VALID
0,LANDFIRE_FBFM40,LF2025 Fire Behavior Fuel Model 40,Cached tiled LANDFIRE mosaic,Used cached source,True,False,None,None,39.254590,0,...,30,10934,13996,None,None,None,None,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
1,LANDFIRE_CANOPY_COVER,LF2025 Forest Canopy Cover,Cached tiled LANDFIRE mosaic,Used cached source,True,False,None,None,18.025119,0,...,30,10934,13996,None,None,None,None,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
2,LANDFIRE_CANOPY_BULK_DENSITY,LF2025 Forest Canopy Bulk Density,Cached tiled LANDFIRE mosaic,Used cached source,True,False,None,None,19.469636,0,...,30,10934,13996,None,None,None,None,None,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



--- LANDFIRE FUEL FILE VALIDATION ---


,SOURCE_ID,SOURCE_PATH,FILE_EXISTS,FILE_SIZE_BYTES,FILE_SIZE_MB,VALID
0,LANDFIRE_FBFM40,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,41161421,39.254590,True
1,LANDFIRE_CANOPY_COVER,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,18900707,18.025119,True
2,LANDFIRE_CANOPY_BULK_DENSITY,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,20415393,19.469636,True



NOTE:
Each LANDFIRE ImageServer is queried for its advertised maximum export width and height before acquisition.
The buffered study-area raster is divided into pixel-aligned export tiles that do not exceed the service limit or the project 4,000-pixel safety cap.
Each tile is downloaded independently with f=image and the reusable whole-file retry function, then written into its exact window in a local 30-meter EPSG:5070 GeoTIFF.
The final files remain raw study-area LANDFIRE sources. Exact county clipping, reprojection to the project CRS, reference-grid alignment, reclassification, and normalization occur in the downstream fuel-hazard component-processing phase.

=== LANDFIRE FUEL SOURCE DATA ACQUISITION COMPLETE ===


## LANDFIRE Fuel Alignment and FBFM40 Reclassification


### Configuring Fuel-Hazard Processing


In [97]:
print('=== CONFIGURING FUEL-HAZARD PROCESSING ===')
# Collect required fuel hazard configuration inputs in one configuration object so downstream steps
# use the same processing rules.
required_fuel_hazard_configuration_inputs = ['fire_hazard_fuel_data_acquired',
    'fire_hazard_fuel_source_paths', 'fire_hazard_fuel_file_validation_summary',
    'fire_hazard_fuel_image_services', 'fire_hazard_source_catalog',
    'fire_hazard_dataset_versions', 'fire_hazard_data_mode',
    'fire_hazard_raw_fuels_directory', 'gdf_county_boundaries_aligned',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_nodata_value', 'fire_hazard_raster_dtype',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_profile',
    'fire_hazard_alignment_study_area_mask']
# Identify missing fuel hazard configuration inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fuel_hazard_configuration_inputs = [object_name for object_name in \
    required_fuel_hazard_configuration_inputs if object_name not in globals()]

# Stop execution if missing fuel hazard configuration inputs remain unresolved before this workflow
# stage begins.
if missing_fuel_hazard_configuration_inputs:
    raise NameError(f'The following fuel-hazard '
        f'configuration objects are '
        f'missing:\n'
        f'{missing_fuel_hazard_configuration_inputs}\n\n'
        f'Run the LANDFIRE fuel '
        f'acquisition and common '
        f'fire-hazard grid preparation '
        f'steps before configuring '
        f'fuel-hazard processing.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_data_acquired:
    raise ValueError('The required LANDFIRE fuel products have not passed acquisition validation.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_fuel_file_validation_summary.empty:
    raise ValueError('The LANDFIRE fuel file-validation table contains no records.')

# Stop execution if the raster data type does not match the output specification required by this
# stage.
if fire_hazard_fuel_file_validation_summary['VALID'].dtype == object:
    # Create fire hazard fuel file validation summary so the workflow can report and validate the
    # resulting metrics.
    fire_hazard_fuel_file_validation_summary['VALID'] = \
        fire_hazard_fuel_file_validation_summary['VALID'].astype(str).str.strip().str.lower().map \
        ({'true': True,
        'false': False})

# Stop execution if required validation fields contain null values that would make the QA result
# unreliable.
if fire_hazard_fuel_file_validation_summary['VALID'].isna().any() or not \
    fire_hazard_fuel_file_validation_summary['VALID'].all():
    raise ValueError('One or more LANDFIRE fuel source files are missing or invalid.')

# Reclassify LANDFIRE FBFM40 fuel models into normalized hazard
# scores based on expected fire behavior.
# Define the fire hazard fuel source ids inputs required by this workflow stage.
fire_hazard_required_fuel_source_ids = ['LANDFIRE_FBFM40',
    'LANDFIRE_CANOPY_COVER', 'LANDFIRE_CANOPY_BULK_DENSITY']
# Identify missing required fuel source IDs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_required_fuel_source_ids = [source_id for source_id in \
    fire_hazard_required_fuel_source_ids if source_id not in fire_hazard_fuel_source_paths]

# Stop execution when required required fuel source ids inputs are unavailable.
if missing_required_fuel_source_ids:
    raise KeyError(f'The fuel source-path '
        f'dictionary is missing required '
        f'LANDFIRE products:\n'
        f'{missing_required_fuel_source_ids}')
# Prepare unexpected fuel source IDs for the downstream processing or validation performed in this
# workflow stage.
unexpected_fuel_source_ids = sorted(set(fire_hazard_fuel_source_paths.keys()) - \
    set(fire_hazard_required_fuel_source_ids))

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if unexpected_fuel_source_ids:
    raise ValueError(f'The fuel source-path '
        f'dictionary contains unexpected '
        f'products:\n'
        f'{unexpected_fuel_source_ids}')
# Initialize fire hazard fuel source inventory records to collect consistent records for the stage
# summary and QA checks.
fire_hazard_fuel_source_inventory_records = []

# Iterate through fire hazard required fuel source IDs so each required item receives the same
# processing and QA checks.
for source_id in fire_hazard_required_fuel_source_ids:
    # Build source path used to read, cache, or save this workflow product.
    source_path = Path(fire_hazard_fuel_source_paths[source_id])
    # Prepare source exists for the downstream processing or validation performed in this workflow
    # stage.
    source_exists = source_path.exists() and source_path.is_file()
    # Calculate source size bytes for completeness, file-integrity, or processing QA.
    source_size_bytes = source_path.stat().st_size if source_exists else 0

    # Stop execution if this validation condition is not satisfied before dependent processing
    # continues.
    if not source_exists or source_size_bytes <= 0:
        raise FileNotFoundError(f'A required raw LANDFIRE source '
            f'is unavailable:\n{source_id}\n'
            f'{source_path}')
    # Prepare source catalog record for the downstream processing or validation performed in this
    # workflow stage.
    source_catalog_record = fire_hazard_source_catalog[fire_hazard_source_catalog['SOURCE_ID'] == \
        source_id]

    # Require the expected number of records before continuing so source configuration remains
    # unambiguous.
    if len(source_catalog_record) != 1:
        raise ValueError(f'The fire-hazard source catalog '
            f'must contain exactly one '
            f'record for:\n{source_id}')
    # Prepare source catalog record for the downstream processing or validation performed in this
    # workflow stage.
    source_catalog_record = source_catalog_record.iloc[0]
    # Add the current record to fire hazard fuel source inventory records so the stage summary
    # captures this processing result.
    fire_hazard_fuel_source_inventory_records.append({'SOURCE_ID': source_id,
        'SOURCE_NAME': source_catalog_record['SOURCE_NAME'],
        'DATASET_VERSION': source_catalog_record['DATASET_VERSION'],
        'SOURCE_ROLE': source_catalog_record['DATA_ROLE'],
        'RAW_PATH': str(source_path), 'RAW_FILE_SIZE_BYTES': source_size_bytes,
        'RAW_FILE_SIZE_MB': source_size_bytes / 1024 ** 2,
        'RAW_FILE_EXISTS': source_exists, 'VALID': True})
# Assemble fire hazard fuel source inventory into a table for QA review and downstream validation.
fire_hazard_fuel_source_inventory = pd.DataFrame(fire_hazard_fuel_source_inventory_records)
# Initialize fire hazard fuel raster metadata records to collect consistent records for the stage
# summary and QA checks.
fire_hazard_fuel_raster_metadata_records = []

# Iterate through fire hazard fuel source inventory.iterrows so each required item receives the same
# processing and QA checks.
for _, source_record in fire_hazard_fuel_source_inventory.iterrows():
    # Build source path used to read, cache, or save this workflow product.
    source_path = Path(source_record['RAW_PATH'])

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(source_path) as source_raster:
            # Store source crs so spatial operations use the required coordinate reference system.
            source_crs = source_raster.crs.to_string() if source_raster.crs is not None else None
            # Capture source bounds so source coverage can be compared with the required analysis
            # extent.
            source_bounds = source_raster.bounds
            # Evaluate source metadata valid so invalid inputs or outputs can be rejected before
            # continuing.
            source_metadata_valid = all([source_raster.count == 1,
                source_raster.width > 0, source_raster.height > 0,
                source_raster.crs is not None, source_raster.transform is not None])
            # Add the current record to fire hazard fuel raster metadata records so the stage
            # summary captures this processing result.
            fire_hazard_fuel_raster_metadata_records.append({'SOURCE_ID': \
                source_record['SOURCE_ID'],
                'RAW_PATH': str(source_path), 'SOURCE_CRS': source_crs,
                'SOURCE_DTYPE': source_raster.dtypes[0], 'SOURCE_NODATA': source_raster.nodata,
                'SOURCE_WIDTH': source_raster.width, 'SOURCE_HEIGHT': source_raster.height,
                'SOURCE_PIXEL_WIDTH': abs(source_raster.transform.a),
                'SOURCE_PIXEL_HEIGHT': abs(source_raster.transform.e),
                'SOURCE_MIN_X': source_bounds.left, 'SOURCE_MIN_Y': source_bounds.bottom,
                'SOURCE_MAX_X': source_bounds.right, 'SOURCE_MAX_Y': source_bounds.top,
                'BAND_COUNT': source_raster.count, 'METADATA_VALID': source_metadata_valid,
                'ERROR_MESSAGE': None})
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Add the current record to fire hazard fuel raster metadata records so the stage summary
        # captures this processing result.
        fire_hazard_fuel_raster_metadata_records.append({'SOURCE_ID': source_record['SOURCE_ID'],
            'RAW_PATH': str(source_path), 'SOURCE_CRS': None,
            'SOURCE_DTYPE': None, 'SOURCE_NODATA': None, 'SOURCE_WIDTH': None,
            'SOURCE_HEIGHT': None, 'SOURCE_PIXEL_WIDTH': None,
            'SOURCE_PIXEL_HEIGHT': None, 'SOURCE_MIN_X': None,
            'SOURCE_MIN_Y': None, 'SOURCE_MAX_X': None, 'SOURCE_MAX_Y': None,
            'BAND_COUNT': None, 'METADATA_VALID': False, 'ERROR_MESSAGE': str(error)})
# Prepare fire hazard fuel raster metadata so raster properties can be checked consistently before
# downstream processing.
fire_hazard_fuel_raster_metadata = pd.DataFrame(fire_hazard_fuel_raster_metadata_records)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_raster_metadata['METADATA_VALID'].all():
    # Evaluate invalid fuel raster metadata so invalid inputs or outputs can be rejected before
    # continuing.
    invalid_fuel_raster_metadata = \
        fire_hazard_fuel_raster_metadata[~fire_hazard_fuel_raster_metadata['METADATA_VALID']]
    raise ValueError(f"One or more raw LANDFIRE "
        f"rasters failed metadata "
        f"validation.\n\n"
        f"{invalid_fuel_raster_metadata.head(5).to_dict('records')}")
# Build the expected fuel study area mask shape mask used to isolate records required for this
# analysis.
expected_fuel_study_area_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_fuel_study_area_mask_shape:
    raise ValueError(f'The fire-hazard study-area '
        f'mask does not match the common '
        f'target grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_fuel_study_area_mask_shape}')
# Calculate fire hazard fuel study area pixel count for completeness, file-integrity, or processing
# QA.
fire_hazard_fuel_study_area_pixel_count = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fuel_study_area_pixel_count == 0:
    raise ValueError('The fire-hazard study-area mask contains no included target-grid pixels.')

# Normalize canopy cover to represent the contribution of
# continuous forest fuels to fire behavior.
# Collect fire hazard fuel product configuration in one configuration object so downstream steps use
# the same processing rules.
fire_hazard_fuel_product_configuration = {'LANDFIRE_FBFM40': {'short_name': 'FBFM40',
    'source_type': 'categorical', 'processing_role': 'Surface fuel-model hazard',
    'resampling_method': 'nearest', 'aligned_dtype': 'int16',
    'aligned_nodata': -9999}, 'LANDFIRE_CANOPY_COVER': {'short_name': 'CANOPY_COVER',
    'source_type': 'continuous', 'processing_role': 'Canopy cover hazard',
    'resampling_method': 'bilinear', 'aligned_dtype': 'float32',
    'aligned_nodata': fire_hazard_nodata_value}, 'LANDFIRE_CANOPY_BULK_DENSITY': {'short_name': \
        'CANOPY_BULK_DENSITY',
    'source_type': 'continuous', 'processing_role': 'Canopy bulk-density hazard',
    'resampling_method': 'bilinear', 'aligned_dtype': 'float32',
    'aligned_nodata': fire_hazard_nodata_value}}
# Assign fire hazard fuel categorical resampling so raster alignment uses the interpolation method
# appropriate for the source data.
fire_hazard_fuel_categorical_resampling = Resampling.nearest
# Assign fire hazard fuel continuous resampling so raster alignment uses the interpolation method
# appropriate for the source data.
fire_hazard_fuel_continuous_resampling = Resampling.bilinear

# Normalize canopy bulk density to represent potential crown-fire
# propagation through vertically connected fuels.
# Assign fire hazard fuel resampling methods so raster alignment uses the interpolation method
# appropriate for the source data.
fire_hazard_fuel_resampling_methods = {'LANDFIRE_FBFM40': fire_hazard_fuel_categorical_resampling,
    'LANDFIRE_CANOPY_COVER': fire_hazard_fuel_continuous_resampling,
    'LANDFIRE_CANOPY_BULK_DENSITY': fire_hazard_fuel_continuous_resampling}
# Build fire hazard fuel processing directory used to read, cache, or save this workflow product.
fire_hazard_fuel_processing_directory = fire_hazard_raw_fuels_directory / 'fuel_hazard_component'
# Build fire hazard fuel aligned directory used to read, cache, or save this workflow product.
fire_hazard_fuel_aligned_directory = fire_hazard_fuel_processing_directory / 'aligned_sources'
# Build fire hazard fuel normalized directory used to read, cache, or save this workflow product.
fire_hazard_fuel_normalized_directory = fire_hazard_fuel_processing_directory / 'normalized_sources'

# Combine surface-fuel and canopy-fuel indicators using the
# configured internal weights.
# Build fire hazard fuel component output directory used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_output_directory = fire_hazard_fuel_processing_directory / \
    'component_output'
# Build fire hazard fuel metadata directory used to read, cache, or save this workflow product.
fire_hazard_fuel_metadata_directory = fire_hazard_fuel_processing_directory / 'metadata'

# Iterate through the required records so the same processing and validation logic is applied
# consistently.
for directory_path in [fire_hazard_fuel_processing_directory,
    fire_hazard_fuel_aligned_directory, fire_hazard_fuel_normalized_directory,
    fire_hazard_fuel_component_output_directory, fire_hazard_fuel_metadata_directory]:
    # Create the output directory before writing workflow products.
    directory_path.mkdir(parents=True, exist_ok=True)
# Build fire hazard fuel aligned paths used to read, cache, or save this workflow product.
fire_hazard_fuel_aligned_paths = {'LANDFIRE_FBFM40': fire_hazard_fuel_aligned_directory / \
    'landfire_fbfm40_aligned.tif',
    'LANDFIRE_CANOPY_COVER': fire_hazard_fuel_aligned_directory / \
        'landfire_canopy_cover_aligned.tif',
    'LANDFIRE_CANOPY_BULK_DENSITY': fire_hazard_fuel_aligned_directory / \
        'landfire_canopy_bulk_density_aligned.tif'}
# Build fire hazard fbfm40 hazard score path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_hazard_score_path = fire_hazard_fuel_normalized_directory / \
    'fbfm40_fuel_hazard_score.tif'
# Build fire hazard canopy cover hazard score path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_cover_hazard_score_path = fire_hazard_fuel_normalized_directory / \
    'canopy_cover_hazard_score.tif'
# Build fire hazard canopy bulk density hazard score path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_bulk_density_hazard_score_path = fire_hazard_fuel_normalized_directory / \
    'canopy_bulk_density_hazard_score.tif'
# Build fire hazard fuel hazard score path used to read, cache, or save this workflow product.
fire_hazard_fuel_hazard_score_path = fire_hazard_fuel_component_output_directory / \
    'composite_fuel_hazard_score.tif'
# Build fire hazard fuel source inventory path used to read, cache, or save this workflow product.
fire_hazard_fuel_source_inventory_path = fire_hazard_fuel_metadata_directory / \
    'fuel_source_inventory.csv'
# Build fire hazard fuel raster metadata path used to read, cache, or save this workflow product.
fire_hazard_fuel_raster_metadata_path = fire_hazard_fuel_metadata_directory / \
    'fuel_raw_raster_metadata.csv'
# Build fire hazard fuel alignment manifest path used to read, cache, or save this workflow product.
fire_hazard_fuel_alignment_manifest_path = fire_hazard_fuel_metadata_directory / \
    'fuel_alignment_manifest.csv'
# Build fire hazard fuel alignment validation path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_alignment_validation_path = fire_hazard_fuel_metadata_directory / \
    'fuel_alignment_validation.csv'
# Build fire hazard fuel normalization manifest path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_normalization_manifest_path = fire_hazard_fuel_metadata_directory / \
    'fuel_normalization_manifest.csv'
# Build fire hazard fuel component manifest path used to read, cache, or save this workflow product.
fire_hazard_fuel_component_manifest_path = fire_hazard_fuel_metadata_directory / \
    'fuel_component_manifest.csv'
# Build fire hazard fuel validation path used to read, cache, or save this workflow product.
fire_hazard_fuel_validation_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_validation.csv'
# Build fire hazard fuel validation summary path used to read, cache, or save this workflow product.
fire_hazard_fuel_validation_summary_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_validation_summary.csv'
# Build fire hazard fuel processing policy path used to read, cache, or save this workflow product.
fire_hazard_fuel_processing_policy_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_processing_policy.json'
# Build fire hazard fuel configuration summary path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_configuration_summary_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_configuration_summary.csv'
# Prepare fire hazard FBFM40 aligned profile so raster outputs inherit the required grid, CRS, data
# type, and NoData metadata.
fire_hazard_fbfm40_aligned_profile = fire_hazard_alignment_profile.copy()
# Update fire hazard FBFM40 aligned profile with the values produced by the current processing step.
fire_hazard_fbfm40_aligned_profile.update({'driver': 'GTiff',
    'dtype': 'int16', 'count': 1, 'nodata': -9999,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': 'deflate', 'predictor': 2, 'tiled': True,
    'BIGTIFF': 'IF_SAFER'})
# Prepare fire hazard canopy aligned profile so raster outputs inherit the required grid, CRS, data
# type, and NoData metadata.
fire_hazard_canopy_aligned_profile = fire_hazard_alignment_profile.copy()
# Update fire hazard canopy aligned profile with the values produced by the current processing step.
fire_hazard_canopy_aligned_profile.update({'driver': 'GTiff',
    'dtype': 'float32', 'count': 1, 'nodata': fire_hazard_nodata_value,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': 'deflate', 'predictor': 3, 'tiled': True,
    'BIGTIFF': 'IF_SAFER'})
# Prepare fire hazard fuel score profile so raster outputs inherit the required grid, CRS, data
# type, and NoData metadata.
fire_hazard_fuel_score_profile = fire_hazard_alignment_profile.copy()
# Update fire hazard fuel score profile with the values produced by the current processing step.
fire_hazard_fuel_score_profile.update({'driver': 'GTiff',
    'dtype': 'float32', 'count': 1, 'nodata': fire_hazard_nodata_value,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': 'deflate', 'predictor': 3, 'tiled': True,
    'BIGTIFF': 'IF_SAFER'})
# Set fire hazard fuel score minimum as an explicit model or validation parameter used consistently
# in downstream calculations.
fire_hazard_fuel_score_minimum = 0.0
# Set fire hazard fuel score maximum as an explicit model or validation parameter used consistently
# in downstream calculations.
fire_hazard_fuel_score_maximum = 1.0
# Set fire hazard fuel score NoData as an explicit model or validation parameter used consistently
# in downstream calculations.
fire_hazard_fuel_score_nodata = fire_hazard_nodata_value
# Calculate fire hazard fuel process by window so raster processing covers the analysis grid in
# controlled blocks.
fire_hazard_fuel_process_by_window = True
# Calculate fire hazard fuel processing window size so raster processing covers the analysis grid in
# controlled blocks.
fire_hazard_fuel_processing_window_size = 512
# Prepare fire hazard fuel progress interval for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_progress_interval = 25
# Prepare fire hazard fuel reuse existing for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_fuel_reuse_existing = fire_hazard_data_mode == 'snapshot'
# Prepare fire hazard fuel overwrite for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_fuel_overwrite = fire_hazard_data_mode == 'refresh'
# Collect fire hazard fuel processing policy in one configuration object so downstream steps use the
# same processing rules.
fire_hazard_fuel_processing_policy = {'component_id': 'FUEL_HAZARD',
    'component_name': 'Surface and Canopy Fuel Hazard',
    'dataset_version': fire_hazard_dataset_versions['landfire'],
    'source_products': {source_id: {'raw_path': str(fire_hazard_fuel_source_paths[source_id]),
    'aligned_path': str(fire_hazard_fuel_aligned_paths[source_id]),
    'source_type': fire_hazard_fuel_product_configuration[source_id]['source_type'],
    'processing_role': fire_hazard_fuel_product_configuration[source_id]['processing_role'],
    'resampling_method': fire_hazard_fuel_product_configuration[source_id]['resampling_method']} \
        for source_id in fire_hazard_required_fuel_source_ids},
    'target_grid': {'crs': fire_hazard_target_crs,
    'cell_size_meters': float(fire_hazard_cell_size),
    'width': int(fire_hazard_alignment_width), 'height': int(fire_hazard_alignment_height),
    'transform': [float(transform_value) for transform_value in fire_hazard_alignment_transform],
    'continuous_nodata': float(fire_hazard_nodata_value),
    'categorical_nodata': -9999}, 'future_processing': {'clip_to_exact_study_area': True,
    'reproject_and_align': True, 'reclassify_fbfm40': True,
    'normalize_canopy_cover': True, 'normalize_canopy_bulk_density': True,
    'combine_component_scores': True, 'independently_validate_outputs': True},
    'score_range': [float(fire_hazard_fuel_score_minimum),
    float(fire_hazard_fuel_score_maximum)], 'processing': {'process_by_window': \
        fire_hazard_fuel_process_by_window,
    'window_size': int(fire_hazard_fuel_processing_window_size),
    'progress_interval': int(fire_hazard_fuel_progress_interval),
    'reuse_existing': fire_hazard_fuel_reuse_existing,
    'overwrite_existing': fire_hazard_fuel_overwrite}}

# Prepare with fire hazard fuel processing policy path.open so this workflow stage has the values
# required for downstream spatial processing and QA.
with fire_hazard_fuel_processing_policy_path.open('w', encoding='utf-8') as fuel_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fuel_processing_policy, fuel_policy_file, indent=2)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_source_inventory.to_csv(fire_hazard_fuel_source_inventory_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_raster_metadata.to_csv(fire_hazard_fuel_raster_metadata_path, index=False)
# Assemble fire hazard fuel configuration summary into a table for QA review and downstream
# validation.
fire_hazard_fuel_configuration_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'COMPONENT_NAME': 'Surface and Canopy Fuel Hazard',
    'LANDFIRE_VERSION': fire_hazard_dataset_versions['landfire'],
    'RAW_SOURCE_COUNT': len(fire_hazard_fuel_source_inventory),
    'FBFM40_RESAMPLING': 'Nearest neighbor', 'CANOPY_COVER_RESAMPLING': 'Bilinear',
    'CANOPY_BULK_DENSITY_RESAMPLING': 'Bilinear', 'TARGET_CRS': fire_hazard_target_crs,
    'CELL_SIZE_METERS': fire_hazard_cell_size, 'TARGET_WIDTH': fire_hazard_alignment_width,
    'TARGET_HEIGHT': fire_hazard_alignment_height,
    'STUDY_AREA_PIXELS': fire_hazard_fuel_study_area_pixel_count,
    'OUTPUT_SCORE_MINIMUM': fire_hazard_fuel_score_minimum,
    'OUTPUT_SCORE_MAXIMUM': fire_hazard_fuel_score_maximum,
    'PROCESSING_WINDOW_SIZE': fire_hazard_fuel_processing_window_size,
    'REUSE_EXISTING': fire_hazard_fuel_reuse_existing,
    'OVERWRITE_EXISTING': fire_hazard_fuel_overwrite}])
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_configuration_summary.to_csv(fire_hazard_fuel_configuration_summary_path,
    index=False)
# Record whether fire hazard fuel processing configured satisfies the checks required before the
# workflow advances.
fire_hazard_fuel_processing_configured = all([fire_hazard_fuel_data_acquired,
    len(fire_hazard_fuel_source_inventory) == 3, fire_hazard_fuel_source_inventory['VALID'].all(),
    fire_hazard_fuel_raster_metadata['METADATA_VALID'].all(),
    fire_hazard_fuel_processing_policy_path.exists(),
    fire_hazard_fuel_source_inventory_path.exists(),
    fire_hazard_fuel_raster_metadata_path.exists(),
    fire_hazard_fuel_configuration_summary_path.exists()])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_processing_configured:
    raise ValueError('The fuel-hazard processing '
        'configuration did not pass '
        'final completion checks.')
print(f"-> LANDFIRE version: {fire_hazard_dataset_versions['landfire']}")
print(f'-> Raw fuel sources configured: {len(fire_hazard_fuel_source_inventory)}')
print(f'-> Target CRS: {fire_hazard_target_crs}')
print(f'-> Target cell size: {fire_hazard_cell_size} meters')
print(f'-> Target raster size: {fire_hazard_alignment_width:,} × {fire_hazard_alignment_height:,}')
print('-> FBFM40 resampling: Nearest neighbor')
print('-> Canopy resampling: Bilinear')
print(f'-> Output score range: '
    f'{fire_hazard_fuel_score_minimum:.1f} '
    f'to '
    f'{fire_hazard_fuel_score_maximum:.1f}')
print(f'-> Processing window size: '
    f'{fire_hazard_fuel_processing_window_size} '
    f'× '
    f'{fire_hazard_fuel_processing_window_size}')
print(f'-> Reuse existing products: {fire_hazard_fuel_reuse_existing}')
print(f'-> Fuel processing configured: {fire_hazard_fuel_processing_configured}')
print(f'-> Processing policy saved: {fire_hazard_fuel_processing_policy_path}')
print(f'-> Configuration summary saved: {fire_hazard_fuel_configuration_summary_path}')
print('\n--- FUEL-HAZARD PROCESSING CONFIGURATION ---')
display(fire_hazard_fuel_configuration_summary)
print('\n--- LANDFIRE FUEL SOURCE INVENTORY ---')
display(fire_hazard_fuel_source_inventory)
print('\n--- LANDFIRE RAW RASTER METADATA ---')
display(fire_hazard_fuel_raster_metadata)
print('\nNOTE:')
print('The three raw LANDFIRE fuel products are now registered for component processing.')
print('FBFM40 will be treated as categorical data and aligned using nearest-neighbor resampling.')
print('Canopy cover and canopy bulk '
    'density will be treated as '
    'continuous data and aligned '
    'using bilinear resampling.')
print('No fuel values have been reclassified, normalized, or combined in this configuration step.')
print('The next step is to prepare '
    'the fuel alignment workspace '
    'before clipping, reprojecting, '
    'and aligning the three '
    'LANDFIRE rasters.')
print('\n=== FUEL-HAZARD PROCESSING CONFIGURED ===')



=== CONFIGURING FUEL-HAZARD PROCESSING ===
-> LANDFIRE version: LF2025
-> Raw fuel sources configured: 3
-> Target CRS: EPSG:26912
-> Target cell size: 30 meters
-> Target raster size: 9,454 × 14,467
-> FBFM40 resampling: Nearest neighbor
-> Canopy resampling: Bilinear
-> Output score range: 0.0 to 1.0
-> Processing window size: 512 × 512
-> Reuse existing products: True
-> Fuel processing configured: True
-> Processing policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_hazard_processing_policy.json
-> Configuration summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_hazard_configuration_summary.csv

--- FUEL-HAZARD PROCESSING CONFIGURATION ---


,COMPONENT_ID,COMPONENT_NAME,LANDFIRE_VERSION,RAW_SOURCE_COUNT,FBFM40_RESAMPLING,CANOPY_COVER_RESAMPLING,CANOPY_BULK_DENSITY_RESAMPLING,TARGET_CRS,CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,STUDY_AREA_PIXELS,OUTPUT_SCORE_MINIMUM,OUTPUT_SCORE_MAXIMUM,PROCESSING_WINDOW_SIZE,REUSE_EXISTING,OVERWRITE_EXISTING
0,FUEL_HAZARD,Surface and Canopy Fuel Hazard,LF2025,3,Nearest neighbor,Bilinear,Bilinear,EPSG:26912,30,9454,14467,15475232,0.0,1.0,512,True,False



--- LANDFIRE FUEL SOURCE INVENTORY ---


,SOURCE_ID,SOURCE_NAME,DATASET_VERSION,SOURCE_ROLE,RAW_PATH,RAW_FILE_SIZE_BYTES,RAW_FILE_SIZE_MB,RAW_FILE_EXISTS,VALID
0,LANDFIRE_FBFM40,Fire Behavior Fuel Model 40,LF2025,Primary categorical surface-fuel hazard input.,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,41161421,39.254590,True,True
1,LANDFIRE_CANOPY_COVER,Forest Canopy Cover,LF2025,Continuous canopy-density input used within th...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,18900707,18.025119,True,True
2,LANDFIRE_CANOPY_BULK_DENSITY,Forest Canopy Bulk Density,LF2025,Continuous canopy-fuel concentration input use...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,20415393,19.469636,True,True



--- LANDFIRE RAW RASTER METADATA ---


,SOURCE_ID,RAW_PATH,SOURCE_CRS,SOURCE_DTYPE,SOURCE_NODATA,SOURCE_WIDTH,SOURCE_HEIGHT,SOURCE_PIXEL_WIDTH,SOURCE_PIXEL_HEIGHT,SOURCE_MIN_X,SOURCE_MIN_Y,SOURCE_MAX_X,SOURCE_MAX_Y,BAND_COUNT,METADATA_VALID,ERROR_MESSAGE
0,LANDFIRE_FBFM40,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:5070,int16,None,10934,13996,30.0,30.0,-1.581839e+06,1.682792e+06,-1.253819e+06,2.102672e+06,1,True,None
1,LANDFIRE_CANOPY_COVER,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:5070,int16,None,10934,13996,30.0,30.0,-1.581839e+06,1.682792e+06,-1.253819e+06,2.102672e+06,1,True,None
2,LANDFIRE_CANOPY_BULK_DENSITY,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:5070,int16,None,10934,13996,30.0,30.0,-1.581839e+06,1.682792e+06,-1.253819e+06,2.102672e+06,1,True,None



NOTE:
The three raw LANDFIRE fuel products are now registered for component processing.
FBFM40 will be treated as categorical data and aligned using nearest-neighbor resampling.
Canopy cover and canopy bulk density will be treated as continuous data and aligned using bilinear resampling.
No fuel values have been reclassified, normalized, or combined in this configuration step.
The next step is to prepare the fuel alignment workspace before clipping, reprojecting, and aligning the three LANDFIRE rasters.

=== FUEL-HAZARD PROCESSING CONFIGURED ===


### Preparing Fuel Alignment Workspace


In [98]:
print('=== PREPARING FUEL ALIGNMENT WORKSPACE ===')
# Prepare required fuel alignment workspace inputs for the downstream processing or validation
# performed in this workflow stage.
required_fuel_alignment_workspace_inputs = ['fire_hazard_fuel_processing_configured',
    'fire_hazard_fuel_source_inventory', 'fire_hazard_fuel_raster_metadata',
    'fire_hazard_fuel_product_configuration', 'fire_hazard_fuel_aligned_paths',
    'fire_hazard_fuel_resampling_methods', 'fire_hazard_fbfm40_aligned_profile',
    'fire_hazard_canopy_aligned_profile', 'fire_hazard_fuel_alignment_manifest_path',
    'fire_hazard_fuel_alignment_validation_path', 'fire_hazard_fuel_metadata_directory',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_nodata_value', 'fire_hazard_fuel_reuse_existing',
    'fire_hazard_fuel_overwrite', 'fire_hazard_fuel_processing_window_size',
    'fire_hazard_fuel_progress_interval']
# Identify missing fuel alignment workspace inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fuel_alignment_workspace_inputs = [object_name for object_name in \
    required_fuel_alignment_workspace_inputs if object_name not in globals()]

# Stop execution if missing fuel alignment workspace inputs remain unresolved before this workflow
# stage begins.
if missing_fuel_alignment_workspace_inputs:
    raise NameError(f'The following fuel-alignment '
        f'workspace objects are missing:\n'
        f'{missing_fuel_alignment_workspace_inputs}\n\n'
        f'Run Configure Fuel-Hazard '
        f'Processing before preparing '
        f'the fuel alignment workspace.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_processing_configured:
    raise ValueError('The fuel-hazard processing configuration is not complete.')
# Define the fuel source inventory fields inputs required by this workflow stage.
required_fuel_source_inventory_fields = ['SOURCE_ID',
    'SOURCE_NAME', 'DATASET_VERSION', 'RAW_PATH', 'RAW_FILE_EXISTS',
    'VALID']
# Identify missing fuel source inventory fields so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fuel_source_inventory_fields = [field_name for field_name in \
    required_fuel_source_inventory_fields if field_name not in \
    fire_hazard_fuel_source_inventory.columns]

# Stop execution when required fuel source inventory fields inputs are unavailable.
if missing_fuel_source_inventory_fields:
    raise ValueError(f'The fuel source inventory is '
        f'missing required fields:\n'
        f'{missing_fuel_source_inventory_fields}')

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(fire_hazard_fuel_source_inventory) != 3:
    raise ValueError('The fuel source inventory must '
        'contain exactly three LANDFIRE '
        'source records.')

# Iterate through  so each required item receives the same processing and QA checks.
for boolean_field in ['RAW_FILE_EXISTS', 'VALID']:

    # Stop execution if the raster data type does not match the output specification required by
    # this stage.
    if fire_hazard_fuel_source_inventory[boolean_field].dtype == object:
        # Prepare fire hazard fuel source inventory so this workflow stage has the values required
        # for downstream spatial processing and QA.
        fire_hazard_fuel_source_inventory[boolean_field] = \
            fire_hazard_fuel_source_inventory[boolean_field].astype(str).str.strip().str.lower() \
            .map({'true': True,
            'false': False})

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fuel_source_inventory[['RAW_FILE_EXISTS',
    'VALID']].isna().any().any() or not \
        fire_hazard_fuel_source_inventory['RAW_FILE_EXISTS'].all() or (not \
        fire_hazard_fuel_source_inventory['VALID'].all()):
    raise ValueError('One or more raw LANDFIRE source records are missing or invalid.')
# Define the fuel raster metadata fields inputs required by this workflow stage.
required_fuel_raster_metadata_fields = ['SOURCE_ID',
    'RAW_PATH', 'SOURCE_CRS', 'SOURCE_DTYPE', 'SOURCE_NODATA',
    'SOURCE_WIDTH', 'SOURCE_HEIGHT', 'METADATA_VALID']
# Identify missing fuel raster metadata fields so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fuel_raster_metadata_fields = [field_name for field_name in \
    required_fuel_raster_metadata_fields if field_name not in \
    fire_hazard_fuel_raster_metadata.columns]

# Stop execution when required fuel raster metadata fields inputs are unavailable.
if missing_fuel_raster_metadata_fields:
    raise ValueError(f'The raw fuel-raster metadata '
        f'table is missing required '
        f'fields:\n'
        f'{missing_fuel_raster_metadata_fields}')

# Stop execution if the raster data type does not match the output specification required by this
# stage.
if fire_hazard_fuel_raster_metadata['METADATA_VALID'].dtype == object:
    # Prepare fire hazard fuel raster metadata so this workflow stage has the values required for
    # downstream spatial processing and QA.
    fire_hazard_fuel_raster_metadata['METADATA_VALID'] = \
        fire_hazard_fuel_raster_metadata['METADATA_VALID'].astype(str).str.strip().str.lower() \
        .map({'true': True,
        'false': False})

# Stop execution if required validation fields contain null values that would make the QA result
# unreliable.
if fire_hazard_fuel_raster_metadata['METADATA_VALID'].isna().any() or not \
    fire_hazard_fuel_raster_metadata['METADATA_VALID'].all():
    raise ValueError('One or more raw LANDFIRE raster metadata records remain invalid.')
# Build fuel inventory source ids used to track the records included in this processing stage.
fuel_inventory_source_ids = set(fire_hazard_fuel_source_inventory['SOURCE_ID'].astype(str))
# Prepare fuel metadata source IDs for the downstream processing or validation performed in this
# workflow stage.
fuel_metadata_source_ids = set(fire_hazard_fuel_raster_metadata['SOURCE_ID'].astype(str))
# Build fuel output path source ids used to read, cache, or save this workflow product.
fuel_output_path_source_ids = set(fire_hazard_fuel_aligned_paths.keys())
# Collect fuel configuration source IDs in one configuration object so downstream steps use the same
# processing rules.
fuel_configuration_source_ids = set(fire_hazard_fuel_product_configuration.keys())
# Assign fuel resampling source IDs so raster alignment uses the interpolation method appropriate
# for the source data.
fuel_resampling_source_ids = set(fire_hazard_fuel_resampling_methods.keys())
# Record whether fire hazard fuel alignment source IDs match satisfies the checks required before
# the workflow advances.
fire_hazard_fuel_alignment_source_ids_match = fuel_inventory_source_ids == \
    fuel_metadata_source_ids == fuel_output_path_source_ids == fuel_configuration_source_ids == \
    fuel_resampling_source_ids

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_alignment_source_ids_match:
    raise ValueError('The fuel inventory, metadata, '
        'output paths, product '
        'configuration, and resampling '
        'lookups do not contain '
        'matching source IDs.')
# Build the expected fuel alignment mask shape mask used to isolate records required for this
# analysis.
expected_fuel_alignment_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_fuel_alignment_mask_shape:
    raise ValueError(f'The fuel-alignment study-area '
        f'mask does not match the common '
        f'target grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_fuel_alignment_mask_shape}')
# Calculate fire hazard fuel alignment study area pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_alignment_study_area_pixels = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fuel_alignment_study_area_pixels == 0:
    raise ValueError('The fuel-alignment study-area mask contains no included target-grid pixels.')
# Build fire hazard fuel alignment temporary directory used to read, cache, or save this workflow
# product.
fire_hazard_fuel_alignment_temporary_directory = fire_hazard_fuel_metadata_directory.parent / \
    'temporary_alignment'
# Create the output directory before writing workflow products.
fire_hazard_fuel_alignment_temporary_directory.mkdir(parents=True, exist_ok=True)
# Build fire hazard fuel alignment policy path used to read, cache, or save this workflow product.
fire_hazard_fuel_alignment_policy_path = fire_hazard_fuel_metadata_directory / \
    'fuel_alignment_policy.json'
# Build fire hazard fuel alignment workspace summary path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_alignment_workspace_summary_path = fire_hazard_fuel_metadata_directory / \
    'fuel_alignment_workspace_summary.csv'
# Initialize fire hazard fuel alignment inventory records to collect consistent records for the
# stage summary and QA checks.
fire_hazard_fuel_alignment_inventory_records = []

# Iterate through enumerate so each required item receives the same processing and QA checks.
for source_order, source_id in enumerate(sorted(fuel_inventory_source_ids), start=1):
    # Build source inventory record used to track the records included in this processing stage.
    source_inventory_record = \
        fire_hazard_fuel_source_inventory[fire_hazard_fuel_source_inventory['SOURCE_ID'] == \
        source_id].iloc[0]
    # Prepare source metadata record for the downstream processing or validation performed in this
    # workflow stage.
    source_metadata_record = \
        fire_hazard_fuel_raster_metadata[fire_hazard_fuel_raster_metadata['SOURCE_ID'] == \
        source_id].iloc[0]
    # Collect source configuration in one configuration object so downstream steps use the same
    # processing rules.
    source_configuration = fire_hazard_fuel_product_configuration[source_id]
    # Build raw path used to read, cache, or save this workflow product.
    raw_path = Path(source_inventory_record['RAW_PATH'])
    # Build aligned path used to read, cache, or save this workflow product.
    aligned_path = Path(fire_hazard_fuel_aligned_paths[source_id])
    # Build temporary path used to read, cache, or save this workflow product.
    temporary_path = fire_hazard_fuel_alignment_temporary_directory / (aligned_path.stem + \
        '.part.tif')
    # Add the current record to fire hazard fuel alignment inventory records so the stage summary
    # captures this processing result.
    fire_hazard_fuel_alignment_inventory_records.append({'PROCESSING_ORDER': source_order,
        'SOURCE_ID': source_id, 'SOURCE_NAME': source_inventory_record['SOURCE_NAME'],
        'DATASET_VERSION': source_inventory_record['DATASET_VERSION'],
        'SOURCE_TYPE': source_configuration['source_type'],
        'PROCESSING_ROLE': source_configuration['processing_role'],
        'RAW_PATH': str(raw_path), 'SOURCE_CRS': source_metadata_record['SOURCE_CRS'],
        'SOURCE_DTYPE': source_metadata_record['SOURCE_DTYPE'],
        'SOURCE_NODATA': source_metadata_record['SOURCE_NODATA'],
        'SOURCE_WIDTH': int(source_metadata_record['SOURCE_WIDTH']),
        'SOURCE_HEIGHT': int(source_metadata_record['SOURCE_HEIGHT']),
        'RESAMPLING_METHOD': source_configuration['resampling_method'],
        'ALIGNED_DTYPE': source_configuration['aligned_dtype'],
        'ALIGNED_NODATA': source_configuration['aligned_nodata'],
        'TEMPORARY_PATH': str(temporary_path), 'ALIGNED_PATH': str(aligned_path),
        'REUSE_EXISTING': fire_hazard_fuel_reuse_existing,
        'OVERWRITE_EXISTING': fire_hazard_fuel_overwrite,
        'READY_FOR_ALIGNMENT': True})
# Prepare fire hazard fuel alignment inventory for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_alignment_inventory = \
    pd.DataFrame(fire_hazard_fuel_alignment_inventory_records).sort_values('PROCESSING_ORDER') \
    .reset_index(drop=True)
# Define alignment output directories used to configure this workflow stage.
alignment_output_directories = {'temporary': fire_hazard_fuel_alignment_temporary_directory,
    'aligned': next(iter(fire_hazard_fuel_aligned_paths.values())).parent,
    'metadata': fire_hazard_fuel_metadata_directory}
# Identify missing fuel alignment directories so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fuel_alignment_directories = [directory_name for directory_name,
    directory_path in alignment_output_directories.items() if not Path(directory_path).exists()]

# Stop execution when required fuel alignment directories inputs are unavailable.
if missing_fuel_alignment_directories:
    raise FileNotFoundError(f'The following fuel-alignment '
        f'directories could not be '
        f'prepared:\n'
        f'{missing_fuel_alignment_directories}')
# Prepare fire hazard fuel alignment profiles so raster outputs inherit the required grid, CRS, data
# type, and NoData metadata.
fire_hazard_fuel_alignment_profiles = {'LANDFIRE_FBFM40': fire_hazard_fbfm40_aligned_profile,
    'LANDFIRE_CANOPY_COVER': fire_hazard_canopy_aligned_profile,
    'LANDFIRE_CANOPY_BULK_DENSITY': fire_hazard_canopy_aligned_profile}

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if set(fire_hazard_fuel_alignment_profiles.keys()) != fuel_inventory_source_ids:
    raise ValueError('The fuel alignment-profile '
        'lookup does not match the '
        'configured LANDFIRE source '
        'IDs.')
# Evaluate fire hazard fuel alignment valid extensions so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_alignment_valid_extensions = {'.tif', '.tiff'}
# Calculate fire hazard fuel alignment minimum file size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_alignment_minimum_file_size_bytes = 1024
# Evaluate fire hazard fuel alignment minimum valid pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_alignment_minimum_valid_pixels = 1
# Prepare fire hazard fuel alignment require study area clip for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_fuel_alignment_require_study_area_clip = True
# Set fire hazard fuel alignment numeric tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_alignment_numeric_tolerance = 1e-06
# Define fire hazard fuel alignment window columns needed to keep the analysis schema consistent.
fire_hazard_fuel_alignment_window_columns = int(np.ceil(fire_hazard_alignment_width / \
    fire_hazard_fuel_processing_window_size))
# Calculate fire hazard fuel alignment window rows so raster processing covers the analysis grid in
# controlled blocks.
fire_hazard_fuel_alignment_window_rows = int(np.ceil(fire_hazard_alignment_height / \
    fire_hazard_fuel_processing_window_size))
# Calculate fire hazard fuel alignment total windows so raster processing covers the analysis grid
# in controlled blocks.
fire_hazard_fuel_alignment_total_windows = fire_hazard_fuel_alignment_window_columns * \
    fire_hazard_fuel_alignment_window_rows
# Initialize fire hazard fuel alignment window records to collect consistent records for the stage
# summary and QA checks.
fire_hazard_fuel_alignment_window_records = []
# Calculate window number so raster processing covers the analysis grid in controlled blocks.
window_number = 0

# Iterate through range so each required item receives the same processing and QA checks.
for window_row in range(fire_hazard_fuel_alignment_window_rows):
    # Calculate row offset so raster processing covers the analysis grid in controlled blocks.
    row_offset = window_row * fire_hazard_fuel_processing_window_size
    # Calculate window height so raster processing covers the analysis grid in controlled blocks.
    window_height = min(fire_hazard_fuel_processing_window_size,
        fire_hazard_alignment_height - row_offset)

    # Iterate through range so each required item receives the same processing and QA checks.
    for window_column in range(fire_hazard_fuel_alignment_window_columns):
        # Calculate column offset so raster processing covers the analysis grid in controlled
        # blocks.
        column_offset = window_column * fire_hazard_fuel_processing_window_size
        # Calculate window width so raster processing covers the analysis grid in controlled blocks.
        window_width = min(fire_hazard_fuel_processing_window_size,
            fire_hazard_alignment_width - column_offset)
        # Calculate window number so raster processing covers the analysis grid in controlled
        # blocks.
        window_number += 1
        # Add the current record to fire hazard fuel alignment window records so the stage summary
        # captures this processing result.
        fire_hazard_fuel_alignment_window_records.append({'WINDOW_NUMBER': window_number,
            'WINDOW_ROW': window_row, 'WINDOW_COLUMN': window_column,
            'ROW_OFFSET': row_offset, 'COLUMN_OFFSET': column_offset,
            'WINDOW_HEIGHT': window_height, 'WINDOW_WIDTH': window_width,
            'PIXEL_COUNT': window_height * window_width})
# Calculate fire hazard fuel alignment window inventory so raster processing covers the analysis
# grid in controlled blocks.
fire_hazard_fuel_alignment_window_inventory = \
    pd.DataFrame(fire_hazard_fuel_alignment_window_records)
# Calculate planned fuel alignment pixels for completeness, file-integrity, or processing QA.
planned_fuel_alignment_pixels = \
    int(fire_hazard_fuel_alignment_window_inventory['PIXEL_COUNT'].sum())
# Calculate expected fuel alignment pixels for completeness, file-integrity, or processing QA.
expected_fuel_alignment_pixels = int(fire_hazard_alignment_width * fire_hazard_alignment_height)
# Evaluate fire hazard fuel alignment window plan valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_alignment_window_plan_valid = planned_fuel_alignment_pixels == \
    expected_fuel_alignment_pixels

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_alignment_window_plan_valid:
    raise ValueError(f'The fuel-alignment processing '
        f'windows do not cover the '
        f'complete target raster grid.\n'
        f'Planned pixels: '
        f'{planned_fuel_alignment_pixels:,}\n'
        f'Expected pixels: '
        f'{expected_fuel_alignment_pixels:,}')
# Build fire hazard fuel alignment window inventory path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_alignment_window_inventory_path = fire_hazard_fuel_metadata_directory / \
    'fuel_alignment_window_inventory.csv'
# Build fire hazard fuel alignment inventory path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_alignment_inventory_path = fire_hazard_fuel_metadata_directory / \
    'fuel_alignment_inventory.csv'
# Collect fire hazard fuel alignment policy in one configuration object so downstream steps use the
# same processing rules.
fire_hazard_fuel_alignment_policy = {'component_id': 'FUEL_HAZARD',
    'workflow_step': 'clip_reproject_align', 'source_count': \
        int(len(fire_hazard_fuel_alignment_inventory)),
    'sources': {source_id: {'raw_path': \
        str(fire_hazard_fuel_source_inventory.loc[fire_hazard_fuel_source_inventory['SOURCE_ID'] \
        == source_id,
    'RAW_PATH'].iloc[0]), 'aligned_path': str(fire_hazard_fuel_aligned_paths[source_id]),
    'source_type': fire_hazard_fuel_product_configuration[source_id]['source_type'],
    'resampling_method': fire_hazard_fuel_product_configuration[source_id]['resampling_method'],
    'aligned_dtype': fire_hazard_fuel_product_configuration[source_id]['aligned_dtype'],
    'aligned_nodata': fire_hazard_fuel_product_configuration[source_id]['aligned_nodata']} for \
        source_id in sorted(fuel_inventory_source_ids)},
    'target_grid': {'crs': fire_hazard_target_crs,
    'cell_size_meters': float(fire_hazard_cell_size),
    'width': int(fire_hazard_alignment_width), 'height': int(fire_hazard_alignment_height),
    'transform': [float(transform_value) for transform_value in fire_hazard_alignment_transform],
    'study_area_pixels': int(fire_hazard_fuel_alignment_study_area_pixels)},
    'processing': {'window_size': int(fire_hazard_fuel_processing_window_size),
    'window_count': int(fire_hazard_fuel_alignment_total_windows),
    'progress_interval': int(fire_hazard_fuel_progress_interval),
    'reuse_existing': bool(fire_hazard_fuel_reuse_existing),
    'overwrite_existing': bool(fire_hazard_fuel_overwrite),
    'temporary_directory': str(fire_hazard_fuel_alignment_temporary_directory)},
    'validation': {'minimum_file_size_bytes': \
        int(fire_hazard_fuel_alignment_minimum_file_size_bytes),
    'minimum_valid_pixels': int(fire_hazard_fuel_alignment_minimum_valid_pixels),
    'require_exact_study_area_clip': bool(fire_hazard_fuel_alignment_require_study_area_clip),
    'numeric_tolerance': float(fire_hazard_fuel_alignment_numeric_tolerance)}}
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_inventory.to_csv(fire_hazard_fuel_alignment_inventory_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_window_inventory.to_csv(fire_hazard_fuel_alignment_window_inventory_path,
    index=False)

# Document this operation so its role in the current fuel-hazard workflow is clear before processing
# continues.
with fire_hazard_fuel_alignment_policy_path.open('w',
    encoding='utf-8') as fuel_alignment_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fuel_alignment_policy, fuel_alignment_policy_file, indent=2)
# Assemble fire hazard fuel alignment workspace summary into a table for QA review and downstream
# validation.
fire_hazard_fuel_alignment_workspace_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_COUNT': len(fire_hazard_fuel_alignment_inventory),
    'CATEGORICAL_SOURCE_COUNT': int((fire_hazard_fuel_alignment_inventory['SOURCE_TYPE'] == \
        'categorical').sum()),
    'CONTINUOUS_SOURCE_COUNT': int((fire_hazard_fuel_alignment_inventory['SOURCE_TYPE'] == \
        'continuous').sum()),
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'STUDY_AREA_PIXELS': fire_hazard_fuel_alignment_study_area_pixels,
    'PROCESSING_WINDOW_SIZE': fire_hazard_fuel_processing_window_size,
    'PROCESSING_WINDOW_COUNT': fire_hazard_fuel_alignment_total_windows,
    'WINDOW_PLAN_VALID': fire_hazard_fuel_alignment_window_plan_valid,
    'REUSE_EXISTING': fire_hazard_fuel_reuse_existing,
    'OVERWRITE_EXISTING': fire_hazard_fuel_overwrite,
    'READY_FOR_ALIGNMENT': True}])
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_workspace_summary.to_csv \
    (fire_hazard_fuel_alignment_workspace_summary_path,
    index=False)
# Record whether fire hazard fuel alignment workspace ready satisfies the checks required before the
# workflow advances.
fire_hazard_fuel_alignment_workspace_ready = all([fire_hazard_fuel_processing_configured,
    fire_hazard_fuel_alignment_source_ids_match, len(fire_hazard_fuel_alignment_inventory) == 3,
    fire_hazard_fuel_alignment_inventory['READY_FOR_ALIGNMENT'].all(),
    fire_hazard_fuel_alignment_window_plan_valid, \
        fire_hazard_fuel_alignment_inventory_path.exists(),
    fire_hazard_fuel_alignment_window_inventory_path.exists(),
    fire_hazard_fuel_alignment_policy_path.exists(),
    fire_hazard_fuel_alignment_workspace_summary_path.exists()])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_alignment_workspace_ready:
    raise ValueError('The fuel-alignment workspace did not pass final preparation checks.')
print(f'-> Fuel sources prepared: {len(fire_hazard_fuel_alignment_inventory)}')
print(f"-> Categorical sources: "
    f"{int((fire_hazard_fuel_alignment_inventory['SOURCE_TYPE'] == 'categorical').sum())}")
print(f"-> Continuous sources: "
    f"{int((fire_hazard_fuel_alignment_inventory['SOURCE_TYPE'] == 'continuous').sum())}")
print(f'-> Target grid: {fire_hazard_alignment_width:,} × {fire_hazard_alignment_height:,}')
print(f'-> Processing windows: {fire_hazard_fuel_alignment_total_windows:,}')
print(f'-> Window plan valid: {fire_hazard_fuel_alignment_window_plan_valid}')
print(f'-> Temporary directory: {fire_hazard_fuel_alignment_temporary_directory}')
print(f'-> Alignment inventory saved: {fire_hazard_fuel_alignment_inventory_path}')
print(f'-> Window inventory saved: {fire_hazard_fuel_alignment_window_inventory_path}')
print(f'-> Alignment policy saved: {fire_hazard_fuel_alignment_policy_path}')
print(f'-> Fuel alignment workspace ready: {fire_hazard_fuel_alignment_workspace_ready}')
print('\n--- FUEL ALIGNMENT WORKSPACE SUMMARY ---')
display(fire_hazard_fuel_alignment_workspace_summary)
print('\n--- FUEL ALIGNMENT INVENTORY ---')
display(fire_hazard_fuel_alignment_inventory)
print('\n--- FUEL ALIGNMENT WINDOW SAMPLE ---')
display(fire_hazard_fuel_alignment_window_inventory.head(5))
print('\nNOTE:')
print('The fuel-alignment workspace is fully prepared.')
print('No LANDFIRE pixel values were transformed in this step.')
print('The next step will clip, '
    'reproject, align, and mask '
    'FBFM40, canopy cover, and '
    'canopy bulk density to the '
    'common 30-meter fire-hazard '
    'grid.')
print('\n=== FUEL ALIGNMENT WORKSPACE READY ===')



=== PREPARING FUEL ALIGNMENT WORKSPACE ===
-> Fuel sources prepared: 3
-> Categorical sources: 1
-> Continuous sources: 2
-> Target grid: 9,454 × 14,467
-> Processing windows: 551
-> Window plan valid: True
-> Temporary directory: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\temporary_alignment
-> Alignment inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_alignment_inventory.csv
-> Window inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_alignment_window_inventory.csv
-> Alignment policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_alignment_policy.json
-> Fuel alignment workspace ready: True

--- FUEL ALIGNMENT WORKSPACE SUMMARY ---


,COMPONENT_ID,SOURCE_COUNT,CATEGORICAL_SOURCE_COUNT,CONTINUOUS_SOURCE_COUNT,TARGET_CRS,CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,STUDY_AREA_PIXELS,PROCESSING_WINDOW_SIZE,PROCESSING_WINDOW_COUNT,WINDOW_PLAN_VALID,REUSE_EXISTING,OVERWRITE_EXISTING,READY_FOR_ALIGNMENT
0,FUEL_HAZARD,3,1,2,EPSG:26912,30,9454,14467,15475232,512,551,True,True,False,True



--- FUEL ALIGNMENT INVENTORY ---


,PROCESSING_ORDER,SOURCE_ID,SOURCE_NAME,DATASET_VERSION,SOURCE_TYPE,PROCESSING_ROLE,RAW_PATH,SOURCE_CRS,SOURCE_DTYPE,SOURCE_NODATA,SOURCE_WIDTH,SOURCE_HEIGHT,RESAMPLING_METHOD,ALIGNED_DTYPE,ALIGNED_NODATA,TEMPORARY_PATH,ALIGNED_PATH,REUSE_EXISTING,OVERWRITE_EXISTING,READY_FOR_ALIGNMENT
0,1,LANDFIRE_CANOPY_BULK_DENSITY,Forest Canopy Bulk Density,LF2025,continuous,Canopy bulk-density hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:5070,int16,None,10934,13996,bilinear,float32,-9999.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,False,True
1,2,LANDFIRE_CANOPY_COVER,Forest Canopy Cover,LF2025,continuous,Canopy cover hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:5070,int16,None,10934,13996,bilinear,float32,-9999.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,False,True
2,3,LANDFIRE_FBFM40,Fire Behavior Fuel Model 40,LF2025,categorical,Surface fuel-model hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:5070,int16,None,10934,13996,nearest,int16,-9999.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,False,True



--- FUEL ALIGNMENT WINDOW SAMPLE ---


,WINDOW_NUMBER,WINDOW_ROW,WINDOW_COLUMN,ROW_OFFSET,COLUMN_OFFSET,WINDOW_HEIGHT,WINDOW_WIDTH,PIXEL_COUNT
0,1,0,0,0,0,512,512,262144
1,2,0,1,0,512,512,512,262144
2,3,0,2,0,1024,512,512,262144
3,4,0,3,0,1536,512,512,262144
4,5,0,4,0,2048,512,512,262144



NOTE:
The fuel-alignment workspace is fully prepared.
No LANDFIRE pixel values were transformed in this step.
The next step will clip, reproject, align, and mask FBFM40, canopy cover, and canopy bulk density to the common 30-meter fire-hazard grid.

=== FUEL ALIGNMENT WORKSPACE READY ===


### Clipping, Reprojecting, and Aligning LANDFIRE Fuel Layers


In [99]:
print('=== CLIPPING, REPROJECTING, AND ALIGNING LANDFIRE FUEL LAYERS ===')
# Prepare required fuel alignment inputs for the downstream processing or validation performed in
# this workflow stage.
required_fuel_alignment_inputs = ['fire_hazard_fuel_alignment_workspace_ready',
    'fire_hazard_fuel_alignment_inventory', 'fire_hazard_fuel_alignment_window_inventory',
    'fire_hazard_fuel_alignment_profiles', 'fire_hazard_fuel_resampling_methods',
    'fire_hazard_fuel_product_configuration', 'fire_hazard_fuel_alignment_manifest_path',
    'fire_hazard_fuel_alignment_study_area_pixels',
    'fire_hazard_fuel_alignment_minimum_file_size_bytes',
    'fire_hazard_fuel_alignment_minimum_valid_pixels',
    'fire_hazard_fuel_progress_interval', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_fuel_reuse_existing',
    'fire_hazard_fuel_overwrite']
# Identify missing fuel alignment inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fuel_alignment_inputs = [object_name for object_name in required_fuel_alignment_inputs if \
    object_name not in globals()]

# Stop execution if missing fuel alignment inputs remain unresolved before this workflow stage
# begins.
if missing_fuel_alignment_inputs:
    raise NameError(f'The following fuel-alignment '
        f'objects are missing:\n'
        f'{missing_fuel_alignment_inputs}\n\n'
        f'Run Prepare Fuel Alignment '
        f'Workspace before aligning the '
        f'LANDFIRE fuel layers.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_alignment_workspace_ready:
    raise ValueError('The fuel-alignment workspace is not ready.')
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window

# Define reusable validate existing aligned fuel raster logic for this phase of the workflow.
def validate_existing_aligned_fuel_raster(output_path, expected_dtype, expected_nodata):

    """
    Confirm that an existing aligned fuel raster
    matches the common project grid and structure.
    """
    # Build output path used to read, cache, or save this workflow product.
    output_path = Path(output_path)

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not output_path.exists() or not output_path.is_file() or output_path.stat().st_size < \
        fire_hazard_fuel_alignment_minimum_file_size_bytes:
        return False

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(output_path) as output_source:
            return all([output_source.count == 1,
                output_source.width == fire_hazard_alignment_width,
                output_source.height == fire_hazard_alignment_height,
                output_source.crs is not None, output_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                output_source.transform.almost_equals(fire_hazard_alignment_transform),
                output_source.dtypes[0] == expected_dtype, output_source.nodata == expected_nodata])
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception:
        return False
# Initialize fire hazard fuel alignment manifest records to collect consistent records for the stage
# summary and QA checks.
fire_hazard_fuel_alignment_manifest_records = []
# Prepare total fuel sources for the downstream processing or validation performed in this workflow
# stage.
total_fuel_sources = len(fire_hazard_fuel_alignment_inventory)

# Process the raster in internal windows to control memory use
# while preserving the full-resolution output.
# Iterate through fire hazard fuel alignment inventory.iterrows so each required item receives the
# same processing and QA checks.
for source_position, source_record in fire_hazard_fuel_alignment_inventory.iterrows():
    # Prepare source number for the downstream processing or validation performed in this workflow
    # stage.
    source_number = source_position + 1
    # Prepare source ID for the downstream processing or validation performed in this workflow
    # stage.
    source_id = str(source_record['SOURCE_ID'])
    # Prepare source name for the downstream processing or validation performed in this workflow
    # stage.
    source_name = str(source_record['SOURCE_NAME'])
    # Prepare source type for the downstream processing or validation performed in this workflow
    # stage.
    source_type = str(source_record['SOURCE_TYPE'])
    # Build raw path used to read, cache, or save this workflow product.
    raw_path = Path(source_record['RAW_PATH'])
    # Build temporary path used to read, cache, or save this workflow product.
    temporary_path = Path(source_record['TEMPORARY_PATH'])
    # Build aligned path used to read, cache, or save this workflow product.
    aligned_path = Path(source_record['ALIGNED_PATH'])
    # Prepare expected data type for the downstream processing or validation performed in this
    # workflow stage.
    expected_dtype = str(source_record['ALIGNED_DTYPE'])
    # Set expected NoData as an explicit model or validation parameter used consistently in
    # downstream calculations.
    expected_nodata = source_record['ALIGNED_NODATA']
    # Prepare output profile so raster outputs inherit the required grid, CRS, data type, and NoData
    # metadata.
    output_profile = fire_hazard_fuel_alignment_profiles[source_id].copy()
    # Assign resampling method so raster alignment uses the interpolation method appropriate for the
    # source data.
    resampling_method = fire_hazard_fuel_resampling_methods[source_id]
    print(f'\n[{source_number}/{total_fuel_sources}] {source_id}')
    print(f'-> Source: {source_name}')

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not raw_path.exists() or not raw_path.is_file() or raw_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The raw LANDFIRE source raster is unavailable:\n{raw_path}')
    # Evaluate existing output valid so invalid inputs or outputs can be rejected before continuing.
    existing_output_valid = validate_existing_aligned_fuel_raster(aligned_path,
        expected_dtype, expected_nodata)
    # Prepare reuse existing output for the downstream processing or validation performed in this
    # workflow stage.
    reuse_existing_output = fire_hazard_fuel_reuse_existing and existing_output_valid
    # Prepare processing action for the downstream processing or validation performed in this
    # workflow stage.
    processing_action = None
    # Store source crs so spatial operations use the required coordinate reference system.
    source_crs = None
    # Capture source data type for source-raster compatibility and alignment QA.
    source_dtype = None
    # Set source NoData as an explicit model or validation parameter used consistently in downstream
    # calculations.
    source_nodata = None
    # Capture source width for source-raster compatibility and alignment QA.
    source_width = None
    # Capture source height for source-raster compatibility and alignment QA.
    source_height = None
    # Capture source pixel width for source-raster compatibility and alignment QA.
    source_pixel_width = None
    # Capture source pixel height for source-raster compatibility and alignment QA.
    source_pixel_height = None
    # Evaluate aligned valid pixels so invalid inputs or outputs can be rejected before continuing.
    aligned_valid_pixels = 0
    # Calculate aligned NoData pixels for completeness, file-integrity, or processing QA.
    aligned_nodata_pixels = 0
    # Evaluate valid inside study area so invalid inputs or outputs can be rejected before
    # continuing.
    valid_inside_study_area = 0
    # Evaluate valid outside study area so invalid inputs or outputs can be rejected before
    # continuing.
    valid_outside_study_area = 0
    # Set aligned minimum as an explicit model or validation parameter used consistently in
    # downstream calculations.
    aligned_minimum = None
    # Set aligned maximum as an explicit model or validation parameter used consistently in
    # downstream calculations.
    aligned_maximum = None
    # Evaluate grid valid so invalid inputs or outputs can be rejected before continuing.
    grid_valid = False
    # Evaluate structure valid so invalid inputs or outputs can be rejected before continuing.
    structure_valid = False
    # Evaluate aligned output valid so invalid inputs or outputs can be rejected before continuing.
    aligned_output_valid = False
    # Prepare processing error for the downstream processing or validation performed in this
    # workflow stage.
    processing_error = None

    # Stop execution if this validation condition is not satisfied before dependent processing
    # continues.
    if reuse_existing_output:
        # Prepare processing action for the downstream processing or validation performed in this
        # workflow stage.
        processing_action = 'Reused existing aligned raster'
        print('-> Valid cached aligned raster found.')

        # Document this operation so its role in the current fuel-hazard workflow is clear before
        # processing continues.
        try:

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(aligned_path) as aligned_source:
                # Store source crs so spatial operations use the required coordinate reference
                # system.
                source_crs = aligned_source.crs.to_string() if aligned_source.crs is not None \
                    else None
                # Capture source data type for source-raster compatibility and alignment QA.
                source_dtype = aligned_source.dtypes[0]
                # Set source NoData as an explicit model or validation parameter used consistently
                # in downstream calculations.
                source_nodata = aligned_source.nodata
                # Capture source width for source-raster compatibility and alignment QA.
                source_width = aligned_source.width
                # Capture source height for source-raster compatibility and alignment QA.
                source_height = aligned_source.height
                # Capture source pixel width for source-raster compatibility and alignment QA.
                source_pixel_width = abs(aligned_source.transform.a)
                # Capture source pixel height for source-raster compatibility and alignment QA.
                source_pixel_height = abs(aligned_source.transform.e)
                # Evaluate grid valid so invalid inputs or outputs can be rejected before
                # continuing.
                grid_valid = all([aligned_source.width == fire_hazard_alignment_width,
                    aligned_source.height == fire_hazard_alignment_height,
                    aligned_source.crs == rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                    aligned_source.transform.almost_equals(fire_hazard_alignment_transform)])
                # Evaluate structure valid so invalid inputs or outputs can be rejected before
                # continuing.
                structure_valid = all([aligned_source.count == 1,
                    aligned_source.dtypes[0] == expected_dtype, aligned_source.nodata == \
                        expected_nodata])

                # Iterate through aligned source.block windows so each required item receives the
                # same processing and QA checks.
                for _, raster_window in aligned_source.block_windows(1):
                    # Prepare aligned array for the raster calculation or validation performed in
                    # this processing block.
                    aligned_array = aligned_source.read(1, window=raster_window)
                    # Prepare row start for the downstream processing or validation performed in
                    # this workflow stage.
                    row_start = int(raster_window.row_off)
                    # Prepare row end for the downstream processing or validation performed in this
                    # workflow stage.
                    row_end = int(raster_window.row_off + raster_window.height)
                    # Prepare column start for the downstream processing or validation performed in
                    # this workflow stage.
                    column_start = int(raster_window.col_off)
                    # Prepare column end for the downstream processing or validation performed in
                    # this workflow stage.
                    column_end = int(raster_window.col_off + raster_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Build the valid mask mask used to isolate records required for this analysis.
                    valid_mask = np.isfinite(aligned_array) & (aligned_array != expected_nodata)
                    # Evaluate valid values so invalid inputs or outputs can be rejected before
                    # continuing.
                    valid_values = aligned_array[valid_mask]
                    # Evaluate aligned valid pixels so invalid inputs or outputs can be rejected
                    # before continuing.
                    aligned_valid_pixels += int(valid_mask.sum())
                    # Calculate aligned NoData pixels for completeness, file-integrity, or
                    # processing QA.
                    aligned_nodata_pixels += int((~valid_mask).sum())
                    # Evaluate valid inside study area so invalid inputs or outputs can be rejected
                    # before continuing.
                    valid_inside_study_area += int((valid_mask & study_area_window_mask).sum())
                    # Evaluate valid outside study area so invalid inputs or outputs can be rejected
                    # before continuing.
                    valid_outside_study_area += int((valid_mask & ~study_area_window_mask).sum())

                    # Stop execution if the prerequisite validation has not passed before this
                    # workflow stage continues.
                    if valid_values.size > 0:
                        # Calculate window minimum so raster processing covers the analysis grid in
                        # controlled blocks.
                        window_minimum = float(valid_values.min())
                        # Calculate window maximum so raster processing covers the analysis grid in
                        # controlled blocks.
                        window_maximum = float(valid_values.max())
                        # Set aligned minimum as an explicit model or validation parameter used
                        # consistently in downstream calculations.
                        aligned_minimum = window_minimum if aligned_minimum is None else \
                            min(aligned_minimum,
                            window_minimum)
                        # Set aligned maximum as an explicit model or validation parameter used
                        # consistently in downstream calculations.
                        aligned_maximum = window_maximum if aligned_maximum is None else \
                            max(aligned_maximum,
                            window_maximum)
                # Evaluate aligned output valid so invalid inputs or outputs can be rejected before
                # continuing.
                aligned_output_valid = all([grid_valid,
                    structure_valid, aligned_valid_pixels >= \
                        fire_hazard_fuel_alignment_minimum_valid_pixels,
                    valid_inside_study_area > 0, valid_outside_study_area == 0,
                    aligned_minimum is not None, aligned_maximum is not None])
        # Handle the expected failure explicitly so the workflow can report or clean up the affected
        # operation.
        except Exception as error:
            # Prepare processing error for the downstream processing or validation performed in this
            # workflow stage.
            processing_error = str(error)
            # Evaluate aligned output valid so invalid inputs or outputs can be rejected before
            # continuing.
            aligned_output_valid = False
    else:
        # Prepare processing action for the downstream processing or validation performed in this
        # workflow stage.
        processing_action = 'Created clipped and aligned raster'

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if temporary_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            temporary_path.unlink()

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if aligned_path.exists() and fire_hazard_fuel_overwrite:
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            aligned_path.unlink()

        # Document this operation so its role in the current fuel-hazard workflow is clear before
        # processing continues.
        try:

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(raw_path) as source_raster:
                # Store source crs so spatial operations use the required coordinate reference
                # system.
                source_crs = source_raster.crs.to_string() if source_raster.crs is not None else \
                    None
                # Capture source data type for source-raster compatibility and alignment QA.
                source_dtype = source_raster.dtypes[0]
                # Set source NoData as an explicit model or validation parameter used consistently
                # in downstream calculations.
                source_nodata = source_raster.nodata
                # Capture source width for source-raster compatibility and alignment QA.
                source_width = source_raster.width
                # Capture source height for source-raster compatibility and alignment QA.
                source_height = source_raster.height
                # Capture source pixel width for source-raster compatibility and alignment QA.
                source_pixel_width = abs(source_raster.transform.a)
                # Capture source pixel height for source-raster compatibility and alignment QA.
                source_pixel_height = abs(source_raster.transform.e)

                # Stop execution if the raster CRS does not match the coordinate system required for
                # spatial alignment.
                if source_raster.crs is None:
                    raise ValueError('The raw LANDFIRE raster does '
                        'not define a coordinate '
                        'reference system.')

                # Prepare with warpedvrt so this workflow stage has the values required for
                # downstream spatial processing and QA.
                with WarpedVRT(source_raster, crs=fire_hazard_target_crs,
                    transform=fire_hazard_alignment_transform, width=fire_hazard_alignment_width,
                    height=fire_hazard_alignment_height, resampling=resampling_method,
                    src_nodata=source_nodata, nodata=expected_nodata,
                    dtype=expected_dtype) as aligned_vrt:

                    # Open the file in a managed context so required content is processed and the
                    # resource closes cleanly.
                    with rasterio.open(temporary_path,
                        'w', **output_profile) as aligned_destination:
                        # Calculate total processing windows so raster processing covers the
                        # analysis grid in controlled blocks.
                        total_processing_windows = len(fire_hazard_fuel_alignment_window_inventory)

                        # Iterate through the required records so the same processing and validation
                        # logic is applied consistently.
                        for _, window_record in \
                            fire_hazard_fuel_alignment_window_inventory.iterrows():
                            # Calculate window number so raster processing covers the analysis grid
                            # in controlled blocks.
                            window_number = int(window_record['WINDOW_NUMBER'])

                            # Stop execution if this validation condition is not satisfied before
                            # dependent processing continues.
                            if window_number == 1 or window_number % \
                                fire_hazard_fuel_progress_interval == 0 or window_number == \
                                total_processing_windows:
                                print(f'-> Aligning window '
                                    f'{window_number:,} of '
                                    f'{total_processing_windows:,}...')
                            # Calculate processing window so raster processing covers the analysis
                            # grid in controlled blocks.
                            processing_window = Window(col_off=int(window_record['COLUMN_OFFSET']),
                                row_off=int(window_record['ROW_OFFSET']), \
                                    width=int(window_record['WINDOW_WIDTH']),
                                height=int(window_record['WINDOW_HEIGHT']))
                            # Prepare aligned array for the raster calculation or validation
                            # performed in this processing block.
                            aligned_array = aligned_vrt.read(1,
                                window=processing_window, out_dtype=expected_dtype)
                            # Prepare row start for the downstream processing or validation
                            # performed in this workflow stage.
                            row_start = int(processing_window.row_off)
                            # Prepare row end for the downstream processing or validation performed
                            # in this workflow stage.
                            row_end = int(processing_window.row_off + processing_window.height)
                            # Prepare column start for the downstream processing or validation
                            # performed in this workflow stage.
                            column_start = int(processing_window.col_off)
                            # Prepare column end for the downstream processing or validation
                            # performed in this workflow stage.
                            column_end = int(processing_window.col_off + processing_window.width)
                            # Build the study area window mask mask used to isolate records required
                            # for this analysis.
                            study_area_window_mask = \
                                fire_hazard_alignment_study_area_mask[row_start:row_end,
                                column_start:column_end]
                            # Build the valid source mask mask used to isolate records required for
                            # this analysis.
                            valid_source_mask = np.isfinite(aligned_array) & (aligned_array != \
                                expected_nodata)
                            # Prepare aligned array so this workflow stage has the values required
                            # for downstream spatial processing and QA.
                            aligned_array[~study_area_window_mask] = expected_nodata
                            # Build the valid output mask mask used to isolate records required for
                            # this analysis.
                            valid_output_mask = np.isfinite(aligned_array) & (aligned_array != \
                                expected_nodata)
                            # Evaluate valid output values so invalid inputs or outputs can be
                            # rejected before continuing.
                            valid_output_values = aligned_array[valid_output_mask]
                            # Write the processed data to the configured output resource.
                            aligned_destination.write(aligned_array, 1, window=processing_window)
                            # Evaluate aligned valid pixels so invalid inputs or outputs can be
                            # rejected before continuing.
                            aligned_valid_pixels += int(valid_output_mask.sum())
                            # Calculate aligned NoData pixels for completeness, file-integrity, or
                            # processing QA.
                            aligned_nodata_pixels += int((~valid_output_mask).sum())
                            # Evaluate valid inside study area so invalid inputs or outputs can be
                            # rejected before continuing.
                            valid_inside_study_area += int((valid_output_mask & \
                                study_area_window_mask).sum())
                            # Evaluate valid outside study area so invalid inputs or outputs can be
                            # rejected before continuing.
                            valid_outside_study_area += int((valid_output_mask & \
                                ~study_area_window_mask).sum())

                            # Stop execution if the prerequisite validation has not passed before
                            # this workflow stage continues.
                            if valid_output_values.size > 0:
                                # Calculate window minimum so raster processing covers the analysis
                                # grid in controlled blocks.
                                window_minimum = float(valid_output_values.min())
                                # Calculate window maximum so raster processing covers the analysis
                                # grid in controlled blocks.
                                window_maximum = float(valid_output_values.max())
                                # Set aligned minimum as an explicit model or validation parameter
                                # used consistently in downstream calculations.
                                aligned_minimum = window_minimum if aligned_minimum is None else \
                                    min(aligned_minimum,
                                    window_minimum)
                                # Set aligned maximum as an explicit model or validation parameter
                                # used consistently in downstream calculations.
                                aligned_maximum = window_maximum if aligned_maximum is None else \
                                    max(aligned_maximum,
                                    window_maximum)
                            del aligned_array
                            del valid_source_mask
                            del valid_output_mask
                        # Assign a descriptive band label so the exported raster documents the
                        # meaning of its values.
                        aligned_destination.set_band_description(1, f'Aligned {source_name}')
                        # Write processing metadata to the raster so the final product retains its
                        # model and provenance context.
                        aligned_destination.update_tags(COMPONENT_ID='FUEL_HAZARD',
                            SOURCE_ID=source_id, SOURCE_TYPE=source_type, \
                                PROCESSING_ROLE=source_record['PROCESSING_ROLE'],
                            RESAMPLING_METHOD=source_record['RESAMPLING_METHOD'],
                            TARGET_CRS=fire_hazard_target_crs, \
                                TARGET_CELL_SIZE=fire_hazard_cell_size,
                            STUDY_AREA_CLIPPED='True')
            # Evaluate temporary structure valid so invalid inputs or outputs can be rejected before
            # continuing.
            temporary_structure_valid = validate_existing_aligned_fuel_raster(temporary_path,
                expected_dtype, expected_nodata)
            # Evaluate temporary content valid so invalid inputs or outputs can be rejected before
            # continuing.
            temporary_content_valid = all([aligned_valid_pixels >= \
                fire_hazard_fuel_alignment_minimum_valid_pixels,
                valid_inside_study_area > 0, valid_outside_study_area == 0,
                aligned_minimum is not None, aligned_maximum is not None])

            # Stop execution if the prerequisite validation has not passed before this workflow
            # stage continues.
            if not temporary_structure_valid or not temporary_content_valid:
                raise ValueError(f'The temporary aligned fuel '
                    f'raster failed validation.\n\n'
                    f'Source ID: {source_id}\n'
                    f'Structure valid: '
                    f'{temporary_structure_valid}\n'
                    f'Valid pixels: '
                    f'{aligned_valid_pixels:,}\n'
                    f'Valid inside study area: '
                    f'{valid_inside_study_area:,}\n'
                    f'Valid outside study area: '
                    f'{valid_outside_study_area:,}\n'
                    f'Minimum value: '
                    f'{aligned_minimum}\nMaximum '
                    f'value: {aligned_maximum}')
            # Promote the validated temporary raster to the final output path only after processing
            # succeeds.
            temporary_path.replace(aligned_path)

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(aligned_path) as final_source:
                # Evaluate grid valid so invalid inputs or outputs can be rejected before
                # continuing.
                grid_valid = all([final_source.width == fire_hazard_alignment_width,
                    final_source.height == fire_hazard_alignment_height,
                    final_source.crs == rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                    final_source.transform.almost_equals(fire_hazard_alignment_transform)])
                # Evaluate structure valid so invalid inputs or outputs can be rejected before
                # continuing.
                structure_valid = all([final_source.count == 1,
                    final_source.dtypes[0] == expected_dtype, final_source.nodata == \
                        expected_nodata])
            # Evaluate aligned output valid so invalid inputs or outputs can be rejected before
            # continuing.
            aligned_output_valid = all([validate_existing_aligned_fuel_raster(aligned_path,
                expected_dtype, expected_nodata), grid_valid, structure_valid,
                aligned_valid_pixels >= fire_hazard_fuel_alignment_minimum_valid_pixels,
                valid_inside_study_area > 0, valid_outside_study_area == 0,
                aligned_minimum is not None, aligned_maximum is not None])
        # Handle the expected failure explicitly so the workflow can report or clean up the affected
        # operation.
        except Exception as error:
            # Prepare processing error for the downstream processing or validation performed in this
            # workflow stage.
            processing_error = str(error)
            # Evaluate aligned output valid so invalid inputs or outputs can be rejected before
            # continuing.
            aligned_output_valid = False

            # Stop execution if a required raster file is missing or empty before spatial processing
            # begins.
            if temporary_path.exists():
                # Remove the temporary or replaceable raster so the next write starts from a clean
                # output path.
                temporary_path.unlink()

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if not aligned_output_valid:
        raise ValueError(f'The LANDFIRE fuel layer failed '
            f'alignment.\n\nSource ID: '
            f'{source_id}\nRaw path: '
            f'{raw_path}\nAligned path: '
            f'{aligned_path}\nError: '
            f'{processing_error}')
    # Calculate output size bytes for completeness, file-integrity, or processing QA.
    output_size_bytes = aligned_path.stat().st_size
    # Add the current record to fire hazard fuel alignment manifest records so the stage summary
    # captures this processing result.
    fire_hazard_fuel_alignment_manifest_records.append({'PROCESSING_ORDER': \
        int(source_record['PROCESSING_ORDER']),
        'SOURCE_ID': source_id, 'SOURCE_NAME': source_name,
        'SOURCE_TYPE': source_type, 'PROCESSING_ROLE': source_record['PROCESSING_ROLE'],
        'RAW_PATH': str(raw_path), 'ALIGNED_PATH': str(aligned_path),
        'PROCESSING_ACTION': processing_action, 'SOURCE_CRS': source_crs,
        'SOURCE_DTYPE': source_dtype, 'SOURCE_NODATA': source_nodata,
        'SOURCE_WIDTH': source_width, 'SOURCE_HEIGHT': source_height,
        'SOURCE_PIXEL_WIDTH': source_pixel_width, 'SOURCE_PIXEL_HEIGHT': source_pixel_height,
        'TARGET_CRS': fire_hazard_target_crs, 'TARGET_CELL_SIZE': fire_hazard_cell_size,
        'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
        'RESAMPLING_METHOD': source_record['RESAMPLING_METHOD'],
        'ALIGNED_DTYPE': expected_dtype, 'ALIGNED_NODATA': expected_nodata,
        'VALID_PIXELS': aligned_valid_pixels, 'NODATA_PIXELS': aligned_nodata_pixels,
        'VALID_INSIDE_STUDY_AREA': valid_inside_study_area,
        'VALID_OUTSIDE_STUDY_AREA': valid_outside_study_area,
        'VALUE_MINIMUM': aligned_minimum, 'VALUE_MAXIMUM': aligned_maximum,
        'OUTPUT_SIZE_BYTES': output_size_bytes, 'OUTPUT_SIZE_MB': output_size_bytes / 1024 ** 2,
        'GRID_VALID': grid_valid, 'STRUCTURE_VALID': structure_valid,
        'ERROR_MESSAGE': processing_error, 'VALID': aligned_output_valid})
    print(f'-> Processing action: {processing_action}')
    print(f'-> Valid aligned pixels: {aligned_valid_pixels:,}')
    print(f'-> Aligned value range: {aligned_minimum} to {aligned_maximum}')
    print(f'-> Valid pixels outside study area: {valid_outside_study_area:,}')
    print(f'-> Aligned raster saved: {aligned_path}')
# Prepare fire hazard fuel alignment manifest for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_alignment_manifest = \
    pd.DataFrame(fire_hazard_fuel_alignment_manifest_records).sort_values('PROCESSING_ORDER') \
    .reset_index(drop=True)
# Calculate expected aligned fuel count for completeness, file-integrity, or processing QA.
expected_aligned_fuel_count = len(fire_hazard_fuel_alignment_inventory)
# Calculate completed aligned fuel count for completeness, file-integrity, or processing QA.
completed_aligned_fuel_count = len(fire_hazard_fuel_alignment_manifest)
# Calculate valid aligned fuel count for completeness, file-integrity, or processing QA.
valid_aligned_fuel_count = int(fire_hazard_fuel_alignment_manifest['VALID'].sum())
# Build failed fuel alignment records used to track the records included in this processing stage.
failed_fuel_alignment_records = \
    fire_hazard_fuel_alignment_manifest[~fire_hazard_fuel_alignment_manifest['VALID']].copy() \
    .reset_index(drop=True)
# Prepare fire hazard fuel alignment complete for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_alignment_complete = all([completed_aligned_fuel_count == \
    expected_aligned_fuel_count,
    valid_aligned_fuel_count == expected_aligned_fuel_count,
    failed_fuel_alignment_records.empty, fire_hazard_fuel_alignment_manifest['GRID_VALID'].all(),
    fire_hazard_fuel_alignment_manifest['STRUCTURE_VALID'].all(),
    (fire_hazard_fuel_alignment_manifest['VALID_OUTSIDE_STUDY_AREA'] == 0).all()])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_alignment_complete:
    raise ValueError('The complete LANDFIRE fuel-alignment set failed final checks.')
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_manifest.to_csv(fire_hazard_fuel_alignment_manifest_path, index=False)
# Calculate total aligned fuel size bytes for completeness, file-integrity, or processing QA.
total_aligned_fuel_size_bytes = int(fire_hazard_fuel_alignment_manifest['OUTPUT_SIZE_BYTES'].sum())
# Assemble fire hazard fuel alignment summary into a table for QA review and downstream validation.
fire_hazard_fuel_alignment_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'EXPECTED_RASTERS': expected_aligned_fuel_count,
    'COMPLETED_RASTERS': completed_aligned_fuel_count,
    'VALID_RASTERS': valid_aligned_fuel_count, 'FAILED_RASTERS': len(failed_fuel_alignment_records),
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'STUDY_AREA_PIXELS': fire_hazard_fuel_alignment_study_area_pixels,
    'TOTAL_OUTPUT_SIZE_BYTES': total_aligned_fuel_size_bytes,
    'TOTAL_OUTPUT_SIZE_MB': total_aligned_fuel_size_bytes / 1024 ** 2,
    'ALIGNMENT_COMPLETE': fire_hazard_fuel_alignment_complete}])
# Build fire hazard fuel alignment summary path used to read, cache, or save this workflow product.
fire_hazard_fuel_alignment_summary_path = Path(fire_hazard_fuel_alignment_manifest_path).parent / \
    'fuel_alignment_summary.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_summary.to_csv(fire_hazard_fuel_alignment_summary_path, index=False)
print('\n=== LANDFIRE FUEL ALIGNMENT RESULTS ===')
print(f'-> Expected aligned rasters: {expected_aligned_fuel_count}')
print(f'-> Completed aligned rasters: {completed_aligned_fuel_count}')
print(f'-> Valid aligned rasters: {valid_aligned_fuel_count}')
print(f'-> Failed aligned rasters: {len(failed_fuel_alignment_records)}')
print(f'-> Total aligned output size: {total_aligned_fuel_size_bytes / 1024 ** 2:,.2f} MB')
print(f'-> Alignment complete: {fire_hazard_fuel_alignment_complete}')
print(f'-> Alignment manifest saved: {fire_hazard_fuel_alignment_manifest_path}')
print(f'-> Alignment summary saved: {fire_hazard_fuel_alignment_summary_path}')
print('\n--- FUEL ALIGNMENT SUMMARY ---')
display(fire_hazard_fuel_alignment_summary)
print('\n--- FUEL ALIGNMENT MANIFEST ---')
display(fire_hazard_fuel_alignment_manifest)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('FBFM40 was aligned using '
    'nearest-neighbor resampling to '
    'preserve its categorical '
    'fuel-model classes.')
print('Canopy cover and canopy bulk '
    'density were aligned using '
    'bilinear resampling because '
    'they represent continuous '
    'measurements.')
print('All three aligned rasters use '
    'the same 30-meter target grid '
    'and contain NoData outside the '
    'three-county study area.')
print('The next step is to independently validate the aligned LANDFIRE fuel products.')
print('\n=== LANDFIRE FUEL LAYERS ALIGNED SUCCESSFULLY ===')



=== CLIPPING, REPROJECTING, AND ALIGNING LANDFIRE FUEL LAYERS ===

[1/3] LANDFIRE_CANOPY_BULK_DENSITY
-> Source: Forest Canopy Bulk Density
-> Valid cached aligned raster found.
-> Processing action: Reused existing aligned raster
-> Valid aligned pixels: 15,475,232
-> Aligned value range: 0.0 to 38.0
-> Valid pixels outside study area: 0
-> Aligned raster saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\aligned_sources\landfire_canopy_bulk_density_aligned.tif

[2/3] LANDFIRE_CANOPY_COVER
-> Source: Forest Canopy Cover
-> Valid cached aligned raster found.
-> Processing action: Reused existing aligned raster
-> Valid aligned pixels: 15,475,232
-> Aligned value range: 0.0 to 85.0
-> Valid pixels outside study area: 0
-> Aligned raster saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\aligned_sources\landfire_canopy_cover_aligned.tif

[3/3] LANDFIRE_FBFM40
-> Source: Fire Behavior Fuel Model 40
-> Valid cached aligned r

,COMPONENT_ID,EXPECTED_RASTERS,COMPLETED_RASTERS,VALID_RASTERS,FAILED_RASTERS,TARGET_CRS,CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,STUDY_AREA_PIXELS,TOTAL_OUTPUT_SIZE_BYTES,TOTAL_OUTPUT_SIZE_MB,ALIGNMENT_COMPLETE
0,FUEL_HAZARD,3,3,3,0,EPSG:26912,30,9454,14467,15475232,65706472,62.662575,True



--- FUEL ALIGNMENT MANIFEST ---


,PROCESSING_ORDER,SOURCE_ID,SOURCE_NAME,SOURCE_TYPE,PROCESSING_ROLE,RAW_PATH,ALIGNED_PATH,PROCESSING_ACTION,SOURCE_CRS,SOURCE_DTYPE,...,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,VALUE_MINIMUM,VALUE_MAXIMUM,OUTPUT_SIZE_BYTES,OUTPUT_SIZE_MB,GRID_VALID,STRUCTURE_VALID,ERROR_MESSAGE,VALID
0,1,LANDFIRE_CANOPY_BULK_DENSITY,Forest Canopy Bulk Density,continuous,Canopy bulk-density hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing aligned raster,EPSG:26912,float32,...,15475232,0,0.0,38.0,29399214,28.037275,True,True,None,True
1,2,LANDFIRE_CANOPY_COVER,Forest Canopy Cover,continuous,Canopy cover hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing aligned raster,EPSG:26912,float32,...,15475232,0,0.0,85.0,29720123,28.343318,True,True,None,True
2,3,LANDFIRE_FBFM40,Fire Behavior Fuel Model 40,categorical,Surface fuel-model hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing aligned raster,EPSG:26912,int16,...,15475232,0,91.0,189.0,6587135,6.281981,True,True,None,True



NOTE:
FBFM40 was aligned using nearest-neighbor resampling to preserve its categorical fuel-model classes.
Canopy cover and canopy bulk density were aligned using bilinear resampling because they represent continuous measurements.
All three aligned rasters use the same 30-meter target grid and contain NoData outside the three-county study area.
The next step is to independently validate the aligned LANDFIRE fuel products.

=== LANDFIRE FUEL LAYERS ALIGNED SUCCESSFULLY ===


### Validating Aligned LANDFIRE Fuel Layers


In [100]:
print('=== VALIDATING ALIGNED LANDFIRE FUEL LAYERS ===')
# Record whether required fuel alignment validation inputs satisfies the checks required before the
# workflow advances.
required_fuel_alignment_validation_inputs = ['fire_hazard_fuel_alignment_complete',
    'fire_hazard_fuel_alignment_manifest', 'fire_hazard_fuel_alignment_inventory',
    'fire_hazard_fuel_alignment_validation_path', 'fire_hazard_fuel_alignment_manifest_path',
    'fire_hazard_fuel_product_configuration', 'fire_hazard_fuel_aligned_paths',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_fuel_alignment_minimum_file_size_bytes',
    'fire_hazard_fuel_alignment_minimum_valid_pixels',
    'fire_hazard_fuel_alignment_numeric_tolerance']
# Identify missing fuel alignment validation inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fuel_alignment_validation_inputs = [object_name for object_name in \
    required_fuel_alignment_validation_inputs if object_name not in globals()]

# Stop execution if missing fuel alignment validation inputs remain unresolved before this workflow
# stage begins.
if missing_fuel_alignment_validation_inputs:
    raise NameError(f'The following fuel-alignment '
        f'validation objects are '
        f'missing:\n'
        f'{missing_fuel_alignment_validation_inputs}\n\n'
        f'Run Prepare Fuel Alignment '
        f'Workspace and Clip, Reproject, '
        f'and Align LANDFIRE Fuel Layers '
        f'before validating the outputs.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_alignment_complete:
    raise ValueError('The LANDFIRE fuel-alignment workflow has not completed successfully.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fuel_alignment_manifest.empty:
    raise ValueError('The fuel-alignment manifest contains no records.')

# Iterate through  so each required item receives the same processing and QA checks.
for boolean_field in ['GRID_VALID', 'STRUCTURE_VALID', 'VALID']:

    # Stop execution if this validation condition is not satisfied before dependent processing
    # continues.
    if boolean_field in fire_hazard_fuel_alignment_manifest.columns and \
        fire_hazard_fuel_alignment_manifest[boolean_field].dtype == object:
        # Create fire hazard fuel alignment manifest to track the records and outputs produced by
        # this workflow stage.
        fire_hazard_fuel_alignment_manifest[boolean_field] = \
            fire_hazard_fuel_alignment_manifest[boolean_field].astype(str).str.strip().str.lower \
            ().map({'true': True,
            'false': False})
# Prepare fire hazard expected aligned fuel products for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_expected_aligned_fuel_products = pd.DataFrame([{'SOURCE_ID': 'LANDFIRE_FBFM40',
    'PRODUCT_NAME': 'Fire Behavior Fuel Model 40',
    'SOURCE_TYPE': 'categorical', 'ALIGNED_PATH': \
        str(fire_hazard_fuel_aligned_paths['LANDFIRE_FBFM40']),
    'EXPECTED_DTYPE': fire_hazard_fuel_product_configuration['LANDFIRE_FBFM40']['aligned_dtype'],
    'EXPECTED_NODATA': fire_hazard_fuel_product_configuration['LANDFIRE_FBFM40']['aligned_nodata']},
    {'SOURCE_ID': 'LANDFIRE_CANOPY_COVER', 'PRODUCT_NAME': 'Forest Canopy Cover',
    'SOURCE_TYPE': 'continuous', 'ALIGNED_PATH': \
        str(fire_hazard_fuel_aligned_paths['LANDFIRE_CANOPY_COVER']),
    'EXPECTED_DTYPE': \
        fire_hazard_fuel_product_configuration['LANDFIRE_CANOPY_COVER']['aligned_dtype'],
    'EXPECTED_NODATA': \
        fire_hazard_fuel_product_configuration['LANDFIRE_CANOPY_COVER']['aligned_nodata']},
    {'SOURCE_ID': 'LANDFIRE_CANOPY_BULK_DENSITY', 'PRODUCT_NAME': 'Forest Canopy Bulk Density',
    'SOURCE_TYPE': 'continuous', 'ALIGNED_PATH': \
        str(fire_hazard_fuel_aligned_paths['LANDFIRE_CANOPY_BULK_DENSITY']),
    'EXPECTED_DTYPE': \
        fire_hazard_fuel_product_configuration['LANDFIRE_CANOPY_BULK_DENSITY']['aligned_dtype'],
    'EXPECTED_NODATA': \
        fire_hazard_fuel_product_configuration['LANDFIRE_CANOPY_BULK_DENSITY']['aligned_nodata']}])
# Prepare expected fuel source IDs for the downstream processing or validation performed in this
# workflow stage.
expected_fuel_source_ids = set(fire_hazard_expected_aligned_fuel_products['SOURCE_ID'])
# Prepare manifest fuel source IDs for the downstream processing or validation performed in this
# workflow stage.
manifest_fuel_source_ids = set(fire_hazard_fuel_alignment_manifest['SOURCE_ID'].astype(str))

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if manifest_fuel_source_ids != expected_fuel_source_ids:
    raise ValueError(f'The aligned-fuel manifest does '
        f'not contain the expected three '
        f'LANDFIRE source IDs.\n'
        f'Expected: '
        f'{sorted(expected_fuel_source_ids)}\n'
        f'Found: '
        f'{sorted(manifest_fuel_source_ids)}')
# Build the expected fuel mask shape mask used to isolate records required for this analysis.
expected_fuel_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_fuel_mask_shape:
    raise ValueError(f'The fuel-validation study-area '
        f'mask does not match the common '
        f'target grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_fuel_mask_shape}')
# Evaluate fire hazard fuel validation study area pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_validation_study_area_pixels = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_fuel_validation_study_area_pixels == 0:
    raise ValueError('The fuel-validation study-area mask contains no included pixels.')
# Evaluate fire hazard fuel alignment valid extensions so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_alignment_valid_extensions = {'.tif', '.tiff'}
# Set fire hazard FBFM40 integer tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_integer_tolerance = 1e-06
# Prepare fire hazard fuel require finite values for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_require_finite_values = True
# Evaluate fire hazard fuel alignment validation records so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_alignment_validation_records = []

# Iterate through fire hazard expected aligned fuel products.iterrows so each required item receives
# the same processing and QA checks.
for _, expected_product in fire_hazard_expected_aligned_fuel_products.iterrows():
    # Prepare source ID for the downstream processing or validation performed in this workflow
    # stage.
    source_id = str(expected_product['SOURCE_ID'])
    # Prepare product name for the downstream processing or validation performed in this workflow
    # stage.
    product_name = str(expected_product['PRODUCT_NAME'])
    # Prepare source type for the downstream processing or validation performed in this workflow
    # stage.
    source_type = str(expected_product['SOURCE_TYPE'])
    # Build aligned path used to read, cache, or save this workflow product.
    aligned_path = Path(expected_product['ALIGNED_PATH'])
    # Prepare expected data type for the downstream processing or validation performed in this
    # workflow stage.
    expected_dtype = str(expected_product['EXPECTED_DTYPE'])
    # Set expected NoData as an explicit model or validation parameter used consistently in
    # downstream calculations.
    expected_nodata = expected_product['EXPECTED_NODATA']
    # Prepare file exists for the downstream processing or validation performed in this workflow
    # stage.
    file_exists = aligned_path.exists() and aligned_path.is_file()
    # Evaluate file extension valid so invalid inputs or outputs can be rejected before continuing.
    file_extension_valid = aligned_path.suffix.lower() in \
        fire_hazard_fuel_alignment_valid_extensions
    # Calculate file size bytes for completeness, file-integrity, or processing QA.
    file_size_bytes = aligned_path.stat().st_size if file_exists else 0
    # Evaluate file size valid so invalid inputs or outputs can be rejected before continuing.
    file_size_valid = file_size_bytes >= fire_hazard_fuel_alignment_minimum_file_size_bytes
    # Prepare raster readable for the downstream processing or validation performed in this workflow
    # stage.
    raster_readable = False
    # Evaluate band count valid so invalid inputs or outputs can be rejected before continuing.
    band_count_valid = False
    # Evaluate width valid so invalid inputs or outputs can be rejected before continuing.
    width_valid = False
    # Evaluate height valid so invalid inputs or outputs can be rejected before continuing.
    height_valid = False
    # Store crs valid so spatial operations use the required coordinate reference system.
    crs_valid = False
    # Evaluate transform valid so invalid inputs or outputs can be rejected before continuing.
    transform_valid = False
    # Evaluate pixel width valid so invalid inputs or outputs can be rejected before continuing.
    pixel_width_valid = False
    # Evaluate pixel height valid so invalid inputs or outputs can be rejected before continuing.
    pixel_height_valid = False
    # Evaluate dtype valid so invalid inputs or outputs can be rejected before continuing.
    dtype_valid = False
    # Evaluate nodata valid so invalid inputs or outputs can be rejected before continuing.
    nodata_valid = False
    # Evaluate grid valid so invalid inputs or outputs can be rejected before continuing.
    grid_valid = False
    # Evaluate structure valid so invalid inputs or outputs can be rejected before continuing.
    structure_valid = False
    # Calculate valid pixel count for completeness, file-integrity, or processing QA.
    valid_pixel_count = 0
    # Calculate NoData pixel count for completeness, file-integrity, or processing QA.
    nodata_pixel_count = 0
    # Evaluate valid inside study area so invalid inputs or outputs can be rejected before
    # continuing.
    valid_inside_study_area = 0
    # Evaluate valid outside study area so invalid inputs or outputs can be rejected before
    # continuing.
    valid_outside_study_area = 0
    # Calculate invalid nonfinite pixel count for completeness, file-integrity, or processing QA.
    invalid_nonfinite_pixel_count = 0
    # Calculate noninteger categorical pixel count for completeness, file-integrity, or processing
    # QA.
    noninteger_categorical_pixel_count = 0
    # Set minimum value as an explicit model or validation parameter used consistently in downstream
    # calculations.
    minimum_value = None
    # Set maximum value as an explicit model or validation parameter used consistently in downstream
    # calculations.
    maximum_value = None
    # Prepare mean value for the downstream processing or validation performed in this workflow
    # stage.
    mean_value = None
    # Prepare value sum for the downstream processing or validation performed in this workflow
    # stage.
    value_sum = 0.0
    # Prepare value range available for the downstream processing or validation performed in this
    # workflow stage.
    value_range_available = False
    # Evaluate content valid so invalid inputs or outputs can be rejected before continuing.
    content_valid = False
    # Evaluate validation error so invalid inputs or outputs can be rejected before continuing.
    validation_error = None

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if file_exists and file_extension_valid and file_size_valid:

        # Document this operation so its role in the current fuel-hazard workflow is clear before
        # processing continues.
        try:

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(aligned_path) as aligned_source:
                # Prepare raster readable for the downstream processing or validation performed in
                # this workflow stage.
                raster_readable = True
                # Evaluate band count valid so invalid inputs or outputs can be rejected before
                # continuing.
                band_count_valid = aligned_source.count == 1
                # Evaluate width valid so invalid inputs or outputs can be rejected before
                # continuing.
                width_valid = aligned_source.width == fire_hazard_alignment_width
                # Evaluate height valid so invalid inputs or outputs can be rejected before
                # continuing.
                height_valid = aligned_source.height == fire_hazard_alignment_height
                # Store crs valid so spatial operations use the required coordinate reference
                # system.
                crs_valid = aligned_source.crs is not None and aligned_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs)
                # Evaluate transform valid so invalid inputs or outputs can be rejected before
                # continuing.
                transform_valid = \
                    aligned_source.transform.almost_equals(fire_hazard_alignment_transform)
                # Evaluate pixel width valid so invalid inputs or outputs can be rejected before
                # continuing.
                pixel_width_valid = np.isclose(abs(aligned_source.transform.a),
                    fire_hazard_cell_size, atol=fire_hazard_fuel_alignment_numeric_tolerance)
                # Evaluate pixel height valid so invalid inputs or outputs can be rejected before
                # continuing.
                pixel_height_valid = np.isclose(abs(aligned_source.transform.e),
                    fire_hazard_cell_size, atol=fire_hazard_fuel_alignment_numeric_tolerance)
                # Evaluate grid valid so invalid inputs or outputs can be rejected before
                # continuing.
                grid_valid = all([width_valid,
                    height_valid, crs_valid, transform_valid, pixel_width_valid,
                    pixel_height_valid])
                # Evaluate dtype valid so invalid inputs or outputs can be rejected before
                # continuing.
                dtype_valid = aligned_source.dtypes[0] == expected_dtype
                # Evaluate nodata valid so invalid inputs or outputs can be rejected before
                # continuing.
                nodata_valid = aligned_source.nodata == expected_nodata
                # Evaluate structure valid so invalid inputs or outputs can be rejected before
                # continuing.
                structure_valid = all([band_count_valid, dtype_valid, nodata_valid])

                # Iterate through aligned source.block windows so each required item receives the
                # same processing and QA checks.
                for _, raster_window in aligned_source.block_windows(1):
                    # Prepare aligned array for the raster calculation or validation performed in
                    # this processing block.
                    aligned_array = aligned_source.read(1, window=raster_window)
                    # Prepare row start for the downstream processing or validation performed in
                    # this workflow stage.
                    row_start = int(raster_window.row_off)
                    # Prepare row end for the downstream processing or validation performed in this
                    # workflow stage.
                    row_end = int(raster_window.row_off + raster_window.height)
                    # Prepare column start for the downstream processing or validation performed in
                    # this workflow stage.
                    column_start = int(raster_window.col_off)
                    # Prepare column end for the downstream processing or validation performed in
                    # this workflow stage.
                    column_end = int(raster_window.col_off + raster_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Build the finite mask mask used to isolate records required for this analysis.
                    finite_mask = np.isfinite(aligned_array)
                    # Build the valid value mask mask used to isolate records required for this
                    # analysis.
                    valid_value_mask = finite_mask & (aligned_array != expected_nodata)
                    # Build the invalid nonfinite mask mask used to isolate records required for
                    # this analysis.
                    invalid_nonfinite_mask = ~finite_mask & (aligned_array != expected_nodata)
                    # Calculate invalid nonfinite pixel count for completeness, file-integrity, or
                    # processing QA.
                    invalid_nonfinite_pixel_count += int(invalid_nonfinite_mask.sum())
                    # Build the valid inside mask mask used to isolate records required for this
                    # analysis.
                    valid_inside_mask = valid_value_mask & study_area_window_mask
                    # Build the valid outside mask mask used to isolate records required for this
                    # analysis.
                    valid_outside_mask = valid_value_mask & ~study_area_window_mask
                    # Evaluate valid inside study area so invalid inputs or outputs can be rejected
                    # before continuing.
                    valid_inside_study_area += int(valid_inside_mask.sum())
                    # Evaluate valid outside study area so invalid inputs or outputs can be rejected
                    # before continuing.
                    valid_outside_study_area += int(valid_outside_mask.sum())
                    # Calculate valid pixel count for completeness, file-integrity, or processing
                    # QA.
                    valid_pixel_count += int(valid_value_mask.sum())
                    # Calculate NoData pixel count for completeness, file-integrity, or processing
                    # QA.
                    nodata_pixel_count += int((~valid_value_mask).sum())

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if source_type == 'categorical':
                        # Prepare categorical values for the downstream processing or validation
                        # performed in this workflow stage.
                        categorical_values = aligned_array[valid_value_mask].astype(np.float64)

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if categorical_values.size > 0:
                            # Build the noninteger mask mask used to isolate records required for
                            # this analysis.
                            noninteger_mask = np.abs(categorical_values - \
                                np.rint(categorical_values)) > fire_hazard_fbfm40_integer_tolerance
                            # Calculate noninteger categorical pixel count for completeness,
                            # file-integrity, or processing QA.
                            noninteger_categorical_pixel_count += int(noninteger_mask.sum())
                    # Evaluate valid values so invalid inputs or outputs can be rejected before
                    # continuing.
                    valid_values = aligned_array[valid_value_mask]

                    # Stop execution if the prerequisite validation has not passed before this
                    # workflow stage continues.
                    if valid_values.size > 0:
                        # Set block minimum as an explicit model or validation parameter used
                        # consistently in downstream calculations.
                        block_minimum = float(valid_values.min())
                        # Set block maximum as an explicit model or validation parameter used
                        # consistently in downstream calculations.
                        block_maximum = float(valid_values.max())

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if minimum_value is None:
                            # Set minimum value as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            minimum_value = block_minimum
                        else:
                            # Set minimum value as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            minimum_value = min(minimum_value, block_minimum)

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if maximum_value is None:
                            # Set maximum value as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            maximum_value = block_maximum
                        else:
                            # Set maximum value as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            maximum_value = max(maximum_value, block_maximum)
                        # Prepare value sum for the downstream processing or validation performed in
                        # this workflow stage.
                        value_sum += float(valid_values.sum(dtype=np.float64))
                    del aligned_array
                    del finite_mask
                    del valid_value_mask
                    del invalid_nonfinite_mask
                    del valid_inside_mask
                    del valid_outside_mask
                # Prepare value range available for the downstream processing or validation
                # performed in this workflow stage.
                value_range_available = valid_pixel_count > 0 and minimum_value is not None and \
                    (maximum_value is not None)

                # Stop execution if the prerequisite validation has not passed before this workflow
                # stage continues.
                if valid_pixel_count > 0:
                    # Prepare mean value for the downstream processing or validation performed in
                    # this workflow stage.
                    mean_value = value_sum / valid_pixel_count
                # Evaluate content valid so invalid inputs or outputs can be rejected before
                # continuing.
                content_valid = all([valid_pixel_count >= \
                    fire_hazard_fuel_alignment_minimum_valid_pixels,
                    valid_inside_study_area > 0, valid_outside_study_area == 0,
                    invalid_nonfinite_pixel_count == 0, noninteger_categorical_pixel_count == 0,
                    value_range_available])
        # Handle the expected failure explicitly so the workflow can report or clean up the affected
        # operation.
        except Exception as error:
            # Evaluate validation error so invalid inputs or outputs can be rejected before
            # continuing.
            validation_error = str(error)
    # Evaluate product valid so invalid inputs or outputs can be rejected before continuing.
    product_valid = all([file_exists, file_extension_valid,
        file_size_valid, raster_readable, grid_valid, structure_valid,
        content_valid, validation_error is None])
    # Add the current record to fire hazard fuel alignment validation records so the stage summary
    # captures this processing result.
    fire_hazard_fuel_alignment_validation_records.append({'SOURCE_ID': source_id,
        'PRODUCT_NAME': product_name, 'SOURCE_TYPE': source_type,
        'ALIGNED_PATH': str(aligned_path), 'FILE_EXISTS': file_exists,
        'FILE_EXTENSION_VALID': file_extension_valid, 'FILE_SIZE_BYTES': file_size_bytes,
        'FILE_SIZE_MB': file_size_bytes / 1024 ** 2, 'FILE_SIZE_VALID': file_size_valid,
        'RASTER_READABLE': raster_readable, 'BAND_COUNT_VALID': band_count_valid,
        'WIDTH_VALID': width_valid, 'HEIGHT_VALID': height_valid,
        'CRS_VALID': crs_valid, 'TRANSFORM_VALID': transform_valid,
        'PIXEL_WIDTH_VALID': pixel_width_valid, 'PIXEL_HEIGHT_VALID': pixel_height_valid,
        'DTYPE_VALID': dtype_valid, 'NODATA_VALID': nodata_valid,
        'GRID_VALID': grid_valid, 'STRUCTURE_VALID': structure_valid,
        'VALID_PIXELS': valid_pixel_count, 'NODATA_PIXELS': nodata_pixel_count,
        'VALID_INSIDE_STUDY_AREA': valid_inside_study_area,
        'VALID_OUTSIDE_STUDY_AREA': valid_outside_study_area,
        'INVALID_NONFINITE_PIXELS': invalid_nonfinite_pixel_count,
        'NONINTEGER_CATEGORICAL_PIXELS': noninteger_categorical_pixel_count,
        'VALUE_MINIMUM': minimum_value, 'VALUE_MAXIMUM': maximum_value,
        'VALUE_MEAN': mean_value, 'VALUE_RANGE_AVAILABLE': value_range_available,
        'CONTENT_VALID': content_valid, 'ERROR_MESSAGE': validation_error,
        'VALID': product_valid})
# Evaluate fire hazard fuel alignment validation so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fuel_alignment_validation = \
    pd.DataFrame(fire_hazard_fuel_alignment_validation_records).sort_values('SOURCE_ID') \
    .reset_index(drop=True)
# Define fuel grid signatures used to configure this workflow stage.
fuel_grid_signatures = []

# Iterate through fire hazard fuel aligned paths.items so each required item receives the same
# processing and QA checks.
for source_id, aligned_path in fire_hazard_fuel_aligned_paths.items():

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(aligned_path) as aligned_source:
        # Add the current record to fuel grid signatures so the stage summary captures this
        # processing result.
        fuel_grid_signatures.append({'SOURCE_ID': source_id,
            'WIDTH': aligned_source.width, 'HEIGHT': aligned_source.height,
            'CRS': aligned_source.crs.to_string() if aligned_source.crs is not None else None,
            'TRANSFORM': tuple(aligned_source.transform)})
# Prepare reference fuel grid for the downstream processing or validation performed in this workflow
# stage.
reference_fuel_grid = fuel_grid_signatures[0]
# Record whether fire hazard fuel aligned grids match satisfies the checks required before the
# workflow advances.
fire_hazard_fuel_aligned_grids_match = all((grid_signature['WIDTH'] == \
    reference_fuel_grid['WIDTH'] and grid_signature['HEIGHT'] == reference_fuel_grid['HEIGHT'] \
    and (grid_signature['CRS'] == reference_fuel_grid['CRS']) and \
    rasterio.Affine(*grid_signature['TRANSFORM']).almost_equals(rasterio.Affine \
    (*reference_fuel_grid['TRANSFORM'])) for grid_signature in fuel_grid_signatures))
# Prepare fuel manifest comparison fields for the downstream processing or validation performed in
# this workflow stage.
fuel_manifest_comparison_fields = fire_hazard_fuel_alignment_manifest[['SOURCE_ID',
    'ALIGNED_PATH', 'GRID_VALID', 'STRUCTURE_VALID',
    'VALID']].copy()
# Prepare fuel manifest comparison fields for the downstream processing or validation performed in
# this workflow stage.
fuel_manifest_comparison_fields = fuel_manifest_comparison_fields.rename(columns={'ALIGNED_PATH': \
    'MANIFEST_ALIGNED_PATH',
    'GRID_VALID': 'MANIFEST_GRID_VALID', 'STRUCTURE_VALID': 'MANIFEST_STRUCTURE_VALID',
    'VALID': 'MANIFEST_VALID'})
# Prepare fire hazard fuel alignment manifest comparison for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_fuel_alignment_manifest_comparison = fire_hazard_fuel_alignment_validation[['SOURCE_ID',
    'ALIGNED_PATH', 'GRID_VALID', 'STRUCTURE_VALID',
    'VALID']].merge(fuel_manifest_comparison_fields,
    on='SOURCE_ID', how='left', validate='one_to_one')
# Record whether fire hazard fuel alignment manifest consistent satisfies the checks required before
# the workflow advances.
fire_hazard_fuel_alignment_manifest_consistent = \
    (fire_hazard_fuel_alignment_manifest_comparison['ALIGNED_PATH'] == \
    fire_hazard_fuel_alignment_manifest_comparison['MANIFEST_ALIGNED_PATH']).all() and \
    (fire_hazard_fuel_alignment_manifest_comparison['GRID_VALID'] == \
    fire_hazard_fuel_alignment_manifest_comparison['MANIFEST_GRID_VALID']).all() and \
    (fire_hazard_fuel_alignment_manifest_comparison['STRUCTURE_VALID'] == \
    fire_hazard_fuel_alignment_manifest_comparison['MANIFEST_STRUCTURE_VALID']).all() and \
    (fire_hazard_fuel_alignment_manifest_comparison['VALID'] == \
    fire_hazard_fuel_alignment_manifest_comparison['MANIFEST_VALID']).all()
# Evaluate failed fuel alignment validations so invalid inputs or outputs can be rejected before
# continuing.
failed_fuel_alignment_validations = \
    fire_hazard_fuel_alignment_validation[~fire_hazard_fuel_alignment_validation['VALID']].copy() \
    .reset_index(drop=True)
# Record whether expected fuel validation count satisfies the checks required before the workflow
# advances.
expected_fuel_validation_count = len(fire_hazard_expected_aligned_fuel_products)
# Calculate validated fuel product count for completeness, file-integrity, or processing QA.
validated_fuel_product_count = len(fire_hazard_fuel_alignment_validation)
# Calculate valid fuel product count for completeness, file-integrity, or processing QA.
valid_fuel_product_count = int(fire_hazard_fuel_alignment_validation['VALID'].sum())
# Calculate failed fuel product count for completeness, file-integrity, or processing QA.
failed_fuel_product_count = len(failed_fuel_alignment_validations)
# Evaluate fire hazard fuel alignment validation complete so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_alignment_validation_complete = all([validated_fuel_product_count == \
    expected_fuel_validation_count,
    valid_fuel_product_count == expected_fuel_validation_count,
    failed_fuel_product_count == 0, fire_hazard_fuel_aligned_grids_match,
    fire_hazard_fuel_alignment_manifest_consistent,
    fire_hazard_fuel_alignment_validation['GRID_VALID'].all(),
    fire_hazard_fuel_alignment_validation['STRUCTURE_VALID'].all(),
    fire_hazard_fuel_alignment_validation['CONTENT_VALID'].all(),
    (fire_hazard_fuel_alignment_validation['VALID_OUTSIDE_STUDY_AREA'] == 0).all()])
# Evaluate total validated fuel alignment size bytes so invalid inputs or outputs can be rejected
# before continuing.
total_validated_fuel_alignment_size_bytes = \
    int(fire_hazard_fuel_alignment_validation['FILE_SIZE_BYTES'].sum())
# Assemble fire hazard fuel alignment validation summary into a table for QA review and downstream
# validation.
fire_hazard_fuel_alignment_validation_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'EXPECTED_PRODUCTS': expected_fuel_validation_count,
    'VALIDATED_PRODUCTS': validated_fuel_product_count,
    'VALID_PRODUCTS': valid_fuel_product_count, 'FAILED_PRODUCTS': failed_fuel_product_count,
    'ALL_GRIDS_MATCH': fire_hazard_fuel_aligned_grids_match,
    'PROCESSING_MANIFEST_CONSISTENT': fire_hazard_fuel_alignment_manifest_consistent,
    'FBFM40_INTEGER_CLASSES_VALID': \
        bool((fire_hazard_fuel_alignment_validation.loc[fire_hazard_fuel_alignment_validation \
        ['SOURCE_ID'] == 'LANDFIRE_FBFM40',
    'NONINTEGER_CATEGORICAL_PIXELS'] == 0).all()),
    'VALID_OUTSIDE_STUDY_AREA': \
        int(fire_hazard_fuel_alignment_validation['VALID_OUTSIDE_STUDY_AREA'].sum()),
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'TOTAL_OUTPUT_SIZE_BYTES': total_validated_fuel_alignment_size_bytes,
    'TOTAL_OUTPUT_SIZE_MB': total_validated_fuel_alignment_size_bytes / 1024 ** 2,
    'VALIDATION_COMPLETE': fire_hazard_fuel_alignment_validation_complete}])
# Build fire hazard fuel alignment validation summary path used to read, cache, or save this
# workflow product.
fire_hazard_fuel_alignment_validation_summary_path = \
    Path(fire_hazard_fuel_alignment_validation_path).parent / \
    'fuel_alignment_validation_summary.csv'
# Build fire hazard fuel alignment failed validation path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_alignment_failed_validation_path = \
    Path(fire_hazard_fuel_alignment_validation_path).parent / 'fuel_alignment_failed_validation.csv'
# Build fire hazard fuel alignment manifest comparison path used to read, cache, or save this
# workflow product.
fire_hazard_fuel_alignment_manifest_comparison_path = \
    Path(fire_hazard_fuel_alignment_validation_path).parent / \
    'fuel_alignment_manifest_comparison.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_validation.to_csv(fire_hazard_fuel_alignment_validation_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_validation_summary.to_csv \
    (fire_hazard_fuel_alignment_validation_summary_path,
    index=False)
# Export the table so this workflow result is available to later phases.
failed_fuel_alignment_validations.to_csv(fire_hazard_fuel_alignment_failed_validation_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_alignment_manifest_comparison.to_csv \
    (fire_hazard_fuel_alignment_manifest_comparison_path,
    index=False)
print(f'-> Expected aligned fuel products: {expected_fuel_validation_count}')
print(f'-> Independently validated products: {validated_fuel_product_count}')
print(f'-> Valid aligned products: {valid_fuel_product_count}')
print(f'-> Failed aligned products: {failed_fuel_product_count}')
print(f'-> All aligned grids match: {fire_hazard_fuel_aligned_grids_match}')
print(f'-> Processing manifest consistent: {fire_hazard_fuel_alignment_manifest_consistent}')
print(f"-> Valid pixels outside study "
    f"area: "
    f"{int(fire_hazard_fuel_alignment_validation['VALID_OUTSIDE_STUDY_AREA'].sum()):,}")
print(f'-> Total aligned output size: '
    f'{total_validated_fuel_alignment_size_bytes / 1024 ** 2:,.2f} '
    f'MB')
print(f'-> Alignment validation complete: {fire_hazard_fuel_alignment_validation_complete}')
print(f'-> Validation table saved: {fire_hazard_fuel_alignment_validation_path}')
print(f'-> Validation summary saved: {fire_hazard_fuel_alignment_validation_summary_path}')
print(f'-> Failed-record table saved: {fire_hazard_fuel_alignment_failed_validation_path}')
print('\n--- ALIGNED FUEL VALIDATION SUMMARY ---')
display(fire_hazard_fuel_alignment_validation_summary)
print('\n--- ALIGNED FUEL PRODUCT VALIDATION ---')
display(fire_hazard_fuel_alignment_validation[['SOURCE_ID',
    'SOURCE_TYPE', 'FILE_EXISTS', 'GRID_VALID', 'STRUCTURE_VALID',
    'VALID_PIXELS', 'VALID_INSIDE_STUDY_AREA', 'VALID_OUTSIDE_STUDY_AREA',
    'NONINTEGER_CATEGORICAL_PIXELS', 'VALUE_MINIMUM',
    'VALUE_MAXIMUM', 'VALUE_MEAN', 'CONTENT_VALID',
    'VALID']])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not failed_fuel_alignment_validations.empty:
    print('\n--- FAILED ALIGNED FUEL PRODUCTS ---')
    display(failed_fuel_alignment_validations.head(5))

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_alignment_validation_complete:
    raise ValueError(f'One or more aligned LANDFIRE '
        f'fuel products failed '
        f'independent validation.\n\n'
        f'Expected products: '
        f'{expected_fuel_validation_count}\n'
        f'Valid products: '
        f'{valid_fuel_product_count}\n'
        f'Failed products: '
        f'{failed_fuel_product_count}\n'
        f'All grids match: '
        f'{fire_hazard_fuel_aligned_grids_match}\n'
        f'Processing manifest '
        f'consistent: '
        f'{fire_hazard_fuel_alignment_manifest_consistent}\n\n'
        f'Review the saved '
        f'fuel-alignment validation '
        f'files.')
# Evaluate fire hazard aligned fuel layers valid so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_aligned_fuel_layers_valid = fire_hazard_fuel_alignment_validation_complete
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The aligned FBFM40, '
    'canopy-cover, and '
    'canopy-bulk-density rasters '
    'passed independent file, grid, '
    'structure, content, and '
    'study-area coverage checks.')
print('FBFM40 retained integer '
    'categorical class codes, '
    'confirming that '
    'nearest-neighbor resampling '
    'did not create fractional '
    'fuel-model values.')
print('The two canopy products remain '
    'continuous aligned '
    'measurements and have not yet '
    'been normalized to fuel-hazard '
    'scores.')
print('The next step is to configure the FBFM40 fuel-model reclassification policy.')
print('\n=== ALIGNED LANDFIRE FUEL LAYER VALIDATION COMPLETE ===')



=== VALIDATING ALIGNED LANDFIRE FUEL LAYERS ===
-> Expected aligned fuel products: 3
-> Independently validated products: 3
-> Valid aligned products: 3
-> Failed aligned products: 0
-> All aligned grids match: True
-> Processing manifest consistent: True
-> Valid pixels outside study area: 0
-> Total aligned output size: 62.66 MB
-> Alignment validation complete: True
-> Validation table saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_alignment_validation.csv
-> Validation summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_alignment_validation_summary.csv
-> Failed-record table saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_alignment_failed_validation.csv

--- ALIGNED FUEL VALIDATION SUMMARY ---


,COMPONENT_ID,EXPECTED_PRODUCTS,VALIDATED_PRODUCTS,VALID_PRODUCTS,FAILED_PRODUCTS,ALL_GRIDS_MATCH,PROCESSING_MANIFEST_CONSISTENT,FBFM40_INTEGER_CLASSES_VALID,VALID_OUTSIDE_STUDY_AREA,TARGET_CRS,CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,TOTAL_OUTPUT_SIZE_BYTES,TOTAL_OUTPUT_SIZE_MB,VALIDATION_COMPLETE
0,FUEL_HAZARD,3,3,3,0,True,True,True,0,EPSG:26912,30,9454,14467,65706472,62.662575,True



--- ALIGNED FUEL PRODUCT VALIDATION ---


,SOURCE_ID,SOURCE_TYPE,FILE_EXISTS,GRID_VALID,STRUCTURE_VALID,VALID_PIXELS,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,NONINTEGER_CATEGORICAL_PIXELS,VALUE_MINIMUM,VALUE_MAXIMUM,VALUE_MEAN,CONTENT_VALID,VALID
0,LANDFIRE_CANOPY_BULK_DENSITY,continuous,True,True,True,15475232,15475232,0,0,0.0,38.0,3.983394,True,True
1,LANDFIRE_CANOPY_COVER,continuous,True,True,True,15475232,15475232,0,0,0.0,85.0,10.402769,True,True
2,LANDFIRE_FBFM40,categorical,True,True,True,15475232,15475232,0,0,91.0,189.0,124.921194,True,True



NOTE:
The aligned FBFM40, canopy-cover, and canopy-bulk-density rasters passed independent file, grid, structure, content, and study-area coverage checks.
FBFM40 retained integer categorical class codes, confirming that nearest-neighbor resampling did not create fractional fuel-model values.
The two canopy products remain continuous aligned measurements and have not yet been normalized to fuel-hazard scores.
The next step is to configure the FBFM40 fuel-model reclassification policy.

=== ALIGNED LANDFIRE FUEL LAYER VALIDATION COMPLETE ===


### Configuring FBFM40 Fuel-Model Reclassification


In [101]:
print('=== CONFIGURING FBFM40 FUEL-MODEL RECLASSIFICATION ===')
# Prepare required FBFM40 reclassification inputs for the downstream processing or validation
# performed in this workflow stage.
required_fbfm40_reclassification_inputs = ['fire_hazard_aligned_fuel_layers_valid',
    'fire_hazard_fuel_alignment_validation_complete',
    'fire_hazard_fuel_alignment_validation', 'fire_hazard_fuel_aligned_paths',
    'fire_hazard_fbfm40_hazard_score_path', 'fire_hazard_fuel_score_profile',
    'fire_hazard_fuel_score_minimum', 'fire_hazard_fuel_score_maximum',
    'fire_hazard_fuel_score_nodata', 'fire_hazard_fuel_normalization_manifest_path',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_fuel_reuse_existing',
    'fire_hazard_fuel_overwrite', 'fire_hazard_fuel_processing_window_size',
    'fire_hazard_fuel_progress_interval', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size']
# Identify missing FBFM40 reclassification inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fbfm40_reclassification_inputs = [object_name for object_name in \
    required_fbfm40_reclassification_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 reclassification inputs remain unresolved before this workflow
# stage begins.
if missing_fbfm40_reclassification_inputs:
    raise NameError(f'The following FBFM40 '
        f'reclassification objects are '
        f'missing:\n'
        f'{missing_fbfm40_reclassification_inputs}\n\n'
        f'Run Validate Aligned Fuel '
        f'Layers before configuring '
        f'fuel-model reclassification.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_aligned_fuel_layers_valid:
    raise ValueError('The aligned LANDFIRE fuel layers have not passed final validation.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_alignment_validation_complete:
    raise ValueError('The aligned-fuel validation workflow is incomplete.')
# Evaluate fbfm40 alignment validation record so invalid inputs or outputs can be rejected before
# continuing.
fbfm40_alignment_validation_record = \
    fire_hazard_fuel_alignment_validation[fire_hazard_fuel_alignment_validation['SOURCE_ID'] \
    .astype(str).str.upper() == 'LANDFIRE_FBFM40']

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(fbfm40_alignment_validation_record) != 1:
    raise ValueError('Exactly one aligned FBFM40 validation record is required.')
# Evaluate fbfm40 alignment valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_alignment_valid = str(fbfm40_alignment_validation_record.iloc[0]['VALID']).strip().lower() \
    == 'true'

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fbfm40_alignment_valid:
    raise ValueError('The aligned FBFM40 raster is not valid.')
# Build fire hazard fbfm40 reclassification input path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_reclassification_input_path = \
    Path(fire_hazard_fuel_aligned_paths['LANDFIRE_FBFM40'])
# Build fire hazard fbfm40 reclassification output path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_reclassification_output_path = Path(fire_hazard_fbfm40_hazard_score_path)
# Build fire hazard fbfm40 reclassification temporary path used to read, cache, or save this
# workflow product.
fire_hazard_fbfm40_reclassification_temporary_path = \
    fire_hazard_fbfm40_reclassification_output_path.parent / \
    (fire_hazard_fbfm40_reclassification_output_path.stem + '.part.tif')

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if not fire_hazard_fbfm40_reclassification_input_path.exists() or not \
    fire_hazard_fbfm40_reclassification_input_path.is_file() or \
    fire_hazard_fbfm40_reclassification_input_path.stat().st_size <= 0:
    raise FileNotFoundError(f'The aligned FBFM40 raster is '
        f'unavailable:\n'
        f'{fire_hazard_fbfm40_reclassification_input_path}')
# Prepare fire hazard FBFM40 reclassification method for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fbfm40_reclassification_method = 'expert_informed_ordinal_lookup'
# Set fire hazard FBFM40 score minimum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_score_minimum = fire_hazard_fuel_score_minimum
# Set fire hazard FBFM40 score maximum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_score_maximum = fire_hazard_fuel_score_maximum
# Set fire hazard FBFM40 score NoData as an explicit model or validation parameter used consistently
# in downstream calculations.
fire_hazard_fbfm40_score_nodata = fire_hazard_fuel_score_nodata
# Prepare fire hazard FBFM40 reclassification table for the fuel-hazard calculation and subsequent
# QA checks.
# Define the FBFM40 reclassification rules used to convert fuel-model classes to hazard scores.
fire_hazard_fbfm40_reclassification_table = pd.DataFrame(
    [
        # Preserve undefined background pixels as NoData in the fuel-hazard output.
        {
            'FBFM40_CODE': 0,
            'FBFM40_MODEL': 'BACKGROUND',
            'FUEL_GROUP': 'Background',
            'BURNABLE': False,
            'HAZARD_SCORE': None,
            'OUTPUT_DISPOSITION': 'NoData',
            'RATIONALE': 'Background or undefined source pixels remain NoData.',
        },

        # Assign zero hazard to nonburnable FBFM40 classes.
        {
            'FBFM40_CODE': 91,
            'FBFM40_MODEL': 'NB1',
            'FUEL_GROUP': 'Nonburnable',
            'BURNABLE': False,
            'HAZARD_SCORE': 0.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Urban or developed nonburnable class.',
        },
        {
            'FBFM40_CODE': 92,
            'FBFM40_MODEL': 'NB2',
            'FUEL_GROUP': 'Nonburnable',
            'BURNABLE': False,
            'HAZARD_SCORE': 0.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Snow or ice nonburnable class.',
        },
        {
            'FBFM40_CODE': 93,
            'FBFM40_MODEL': 'NB3',
            'FUEL_GROUP': 'Nonburnable',
            'BURNABLE': False,
            'HAZARD_SCORE': 0.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Agricultural nonburnable class.',
        },
        {
            'FBFM40_CODE': 98,
            'FBFM40_MODEL': 'NB8',
            'FUEL_GROUP': 'Nonburnable',
            'BURNABLE': False,
            'HAZARD_SCORE': 0.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Open-water nonburnable class.',
        },
        {
            'FBFM40_CODE': 99,
            'FBFM40_MODEL': 'NB9',
            'FUEL_GROUP': 'Nonburnable',
            'BURNABLE': False,
            'HAZARD_SCORE': 0.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Barren nonburnable class.',
        },

        # Assign increasing hazard scores across the grass fuel models.
        {
            'FBFM40_CODE': 101,
            'FBFM40_MODEL': 'GR1',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.2,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low relative grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 102,
            'FBFM40_MODEL': 'GR2',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.35,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low-to-moderate grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 103,
            'FBFM40_MODEL': 'GR3',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.5,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 104,
            'FBFM40_MODEL': 'GR4',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.65,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate-to-high grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 105,
            'FBFM40_MODEL': 'GR5',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.75,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'High grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 106,
            'FBFM40_MODEL': 'GR6',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.85,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 107,
            'FBFM40_MODEL': 'GR7',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.9,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 108,
            'FBFM40_MODEL': 'GR8',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.95,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Extreme grass-fuel hazard.',
        },
        {
            'FBFM40_CODE': 109,
            'FBFM40_MODEL': 'GR9',
            'FUEL_GROUP': 'Grass',
            'BURNABLE': True,
            'HAZARD_SCORE': 1.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Maximum relative grass-fuel hazard.',
        },

        # Assign relative hazard scores across the grass-shrub fuel models.
        {
            'FBFM40_CODE': 121,
            'FBFM40_MODEL': 'GS1',
            'FUEL_GROUP': 'Grass-Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.35,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low-to-moderate grass-shrub hazard.',
        },
        {
            'FBFM40_CODE': 122,
            'FBFM40_MODEL': 'GS2',
            'FUEL_GROUP': 'Grass-Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.5,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate grass-shrub hazard.',
        },
        {
            'FBFM40_CODE': 123,
            'FBFM40_MODEL': 'GS3',
            'FUEL_GROUP': 'Grass-Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.7,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'High grass-shrub hazard.',
        },
        {
            'FBFM40_CODE': 124,
            'FBFM40_MODEL': 'GS4',
            'FUEL_GROUP': 'Grass-Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.85,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high grass-shrub hazard.',
        },

        # Assign relative hazard scores across the shrub fuel models.
        {
            'FBFM40_CODE': 141,
            'FBFM40_MODEL': 'SH1',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.3,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low-to-moderate shrub hazard.',
        },
        {
            'FBFM40_CODE': 142,
            'FBFM40_MODEL': 'SH2',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.4,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate shrub hazard.',
        },
        {
            'FBFM40_CODE': 143,
            'FBFM40_MODEL': 'SH3',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.55,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate shrub hazard.',
        },
        {
            'FBFM40_CODE': 144,
            'FBFM40_MODEL': 'SH4',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.65,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate-to-high shrub hazard.',
        },
        {
            'FBFM40_CODE': 145,
            'FBFM40_MODEL': 'SH5',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.75,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'High shrub hazard.',
        },
        {
            'FBFM40_CODE': 146,
            'FBFM40_MODEL': 'SH6',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.85,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high shrub hazard.',
        },
        {
            'FBFM40_CODE': 147,
            'FBFM40_MODEL': 'SH7',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.9,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high shrub hazard.',
        },
        {
            'FBFM40_CODE': 148,
            'FBFM40_MODEL': 'SH8',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.95,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Extreme shrub hazard.',
        },
        {
            'FBFM40_CODE': 149,
            'FBFM40_MODEL': 'SH9',
            'FUEL_GROUP': 'Shrub',
            'BURNABLE': True,
            'HAZARD_SCORE': 1.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Maximum relative shrub hazard.',
        },

        # Assign relative hazard scores across the timber-understory fuel models.
        {
            'FBFM40_CODE': 161,
            'FBFM40_MODEL': 'TU1',
            'FUEL_GROUP': 'Timber-Understory',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.3,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low-to-moderate timber-understory hazard.',
        },
        {
            'FBFM40_CODE': 162,
            'FBFM40_MODEL': 'TU2',
            'FUEL_GROUP': 'Timber-Understory',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.5,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate timber-understory hazard.',
        },
        {
            'FBFM40_CODE': 163,
            'FBFM40_MODEL': 'TU3',
            'FUEL_GROUP': 'Timber-Understory',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.7,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'High timber-understory hazard.',
        },
        {
            'FBFM40_CODE': 164,
            'FBFM40_MODEL': 'TU4',
            'FUEL_GROUP': 'Timber-Understory',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.85,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high timber-understory hazard.',
        },
        {
            'FBFM40_CODE': 165,
            'FBFM40_MODEL': 'TU5',
            'FUEL_GROUP': 'Timber-Understory',
            'BURNABLE': True,
            'HAZARD_SCORE': 1.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Maximum relative timber-understory hazard.',
        },

        # Assign relative hazard scores across the timber-litter fuel models.
        {
            'FBFM40_CODE': 181,
            'FBFM40_MODEL': 'TL1',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.15,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 182,
            'FBFM40_MODEL': 'TL2',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.2,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 183,
            'FBFM40_MODEL': 'TL3',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.3,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Low-to-moderate timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 184,
            'FBFM40_MODEL': 'TL4',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.4,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 185,
            'FBFM40_MODEL': 'TL5',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.5,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 186,
            'FBFM40_MODEL': 'TL6',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.6,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate-to-high timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 187,
            'FBFM40_MODEL': 'TL7',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.75,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'High timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 188,
            'FBFM40_MODEL': 'TL8',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.85,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high timber-litter hazard.',
        },
        {
            'FBFM40_CODE': 189,
            'FBFM40_MODEL': 'TL9',
            'FUEL_GROUP': 'Timber-Litter',
            'BURNABLE': True,
            'HAZARD_SCORE': 1.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Maximum relative timber-litter hazard.',
        },

        # Assign relative hazard scores across the slash-blowdown fuel models.
        {
            'FBFM40_CODE': 201,
            'FBFM40_MODEL': 'SB1',
            'FUEL_GROUP': 'Slash-Blowdown',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.45,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate slash-blowdown hazard.',
        },
        {
            'FBFM40_CODE': 202,
            'FBFM40_MODEL': 'SB2',
            'FUEL_GROUP': 'Slash-Blowdown',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.65,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Moderate-to-high slash-blowdown hazard.',
        },
        {
            'FBFM40_CODE': 203,
            'FBFM40_MODEL': 'SB3',
            'FUEL_GROUP': 'Slash-Blowdown',
            'BURNABLE': True,
            'HAZARD_SCORE': 0.85,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Very high slash-blowdown hazard.',
        },
        {
            'FBFM40_CODE': 204,
            'FBFM40_MODEL': 'SB4',
            'FUEL_GROUP': 'Slash-Blowdown',
            'BURNABLE': True,
            'HAZARD_SCORE': 1.0,
            'OUTPUT_DISPOSITION': 'Score',
            'RATIONALE': 'Maximum relative slash-blowdown hazard.',
        },
    ]
)

# Sort the lookup table by FBFM40 code and assign a clean sequential index.
fire_hazard_fbfm40_reclassification_table = (
    fire_hazard_fbfm40_reclassification_table
    .sort_values('FBFM40_CODE')
    .reset_index(drop=True)
)
# Prepare duplicate FBFM40 codes for the downstream processing or validation performed in this
# workflow stage.
duplicate_fbfm40_codes = \
    fire_hazard_fbfm40_reclassification_table[fire_hazard_fbfm40_reclassification_table \
    ['FBFM40_CODE'].duplicated(keep=False)]

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not duplicate_fbfm40_codes.empty:
    raise ValueError(f"The FBFM40 reclassification "
        f"table contains duplicate class "
        f"codes:\n"
        f"{duplicate_fbfm40_codes['FBFM40_CODE'].tolist()}")
# Prepare scored FBFM40 records for the fuel-hazard calculation and subsequent QA checks.
scored_fbfm40_records = \
    fire_hazard_fbfm40_reclassification_table[fire_hazard_fbfm40_reclassification_table \
    ['OUTPUT_DISPOSITION'] == 'Score']
# Evaluate invalid fbfm40 scores so invalid inputs or outputs can be rejected before continuing.
invalid_fbfm40_scores = scored_fbfm40_records[scored_fbfm40_records['HAZARD_SCORE'].isna() | \
    (scored_fbfm40_records['HAZARD_SCORE'] < fire_hazard_fbfm40_score_minimum) | \
    (scored_fbfm40_records['HAZARD_SCORE'] > fire_hazard_fbfm40_score_maximum)]

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not invalid_fbfm40_scores.empty:
    raise ValueError('One or more FBFM40 hazard '
        'scores fall outside the '
        'configured zero-to-one range.')
# Prepare fire hazard FBFM40 score lookup for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_fbfm40_score_lookup = {int(reclassification_record['FBFM40_CODE']): \
    float(reclassification_record['HAZARD_SCORE']) for _,
    reclassification_record in scored_fbfm40_records.iterrows()}
# Set fire hazard FBFM40 NoData codes as an explicit model or validation parameter used consistently
# in downstream calculations.
fire_hazard_fbfm40_nodata_codes = \
    set(fire_hazard_fbfm40_reclassification_table.loc[fire_hazard_fbfm40_reclassification_table \
    ['OUTPUT_DISPOSITION'] == 'NoData',
    'FBFM40_CODE'].astype(int))
# Prepare fire hazard FBFM40 recognized codes for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_recognized_codes = \
    set(fire_hazard_fbfm40_reclassification_table['FBFM40_CODE'].astype(int))
# Prepare observed FBFM40 codes for the downstream processing or validation performed in this
# workflow stage.
observed_fbfm40_codes = set()
# Record whether observed FBFM40 valid pixel count satisfies the checks required before the workflow
# advances.
observed_fbfm40_valid_pixel_count = 0
# Calculate observed FBFM40 NoData pixel count for completeness, file-integrity, or processing QA.
observed_fbfm40_nodata_pixel_count = 0

# Open the file in a managed context so required content is processed and the resource closes
# cleanly.
with rasterio.open(fire_hazard_fbfm40_reclassification_input_path) as fbfm40_source:
    # Evaluate fbfm40 source grid valid so invalid inputs or outputs can be rejected before
    # continuing.
    fbfm40_source_grid_valid = all([fbfm40_source.count == 1,
        fbfm40_source.width == fire_hazard_alignment_width,
        fbfm40_source.height == fire_hazard_alignment_height,
        fbfm40_source.crs is not None, fbfm40_source.crs == \
            rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
        fbfm40_source.transform.almost_equals(fire_hazard_alignment_transform)])

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if not fbfm40_source_grid_valid:
        raise ValueError('The aligned FBFM40 raster no longer matches the common fire-hazard grid.')

    # Iterate through FBFM40 source.block windows so each required item receives the same processing
    # and QA checks.
    for _, raster_window in fbfm40_source.block_windows(1):
        # Prepare FBFM40 array for the raster calculation or validation performed in this processing
        # block.
        fbfm40_array = fbfm40_source.read(1, window=raster_window)
        # Build the valid fbfm40 mask mask used to isolate records required for this analysis.
        valid_fbfm40_mask = np.isfinite(fbfm40_array) & (fbfm40_array != fbfm40_source.nodata)
        # Record whether observed FBFM40 valid pixel count satisfies the checks required before the
        # workflow advances.
        observed_fbfm40_valid_pixel_count += int(valid_fbfm40_mask.sum())
        # Calculate observed FBFM40 NoData pixel count for completeness, file-integrity, or
        # processing QA.
        observed_fbfm40_nodata_pixel_count += int((~valid_fbfm40_mask).sum())

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if valid_fbfm40_mask.any():
            # Calculate window unique codes so raster processing covers the analysis grid in
            # controlled blocks.
            window_unique_codes = np.unique(fbfm40_array[valid_fbfm40_mask])
            # Update observed FBFM40 codes with the values produced by the current processing step.
            observed_fbfm40_codes.update((int(class_code) for class_code in window_unique_codes))
        del fbfm40_array
        del valid_fbfm40_mask
# Prepare unmapped observed FBFM40 codes for the downstream processing or validation performed in
# this workflow stage.
unmapped_observed_fbfm40_codes = sorted(observed_fbfm40_codes - fire_hazard_fbfm40_recognized_codes)

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if unmapped_observed_fbfm40_codes:
    raise ValueError(f'The aligned FBFM40 raster '
        f'contains class codes that are '
        f'not included in the '
        f'reclassification policy:\n'
        f'{unmapped_observed_fbfm40_codes}')
# Collect unobserved policy FBFM40 codes in one configuration object so downstream steps use the
# same processing rules.
unobserved_policy_fbfm40_codes = sorted(fire_hazard_fbfm40_recognized_codes - observed_fbfm40_codes)
# Prepare fire hazard FBFM40 observed classes covered for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fbfm40_observed_classes_covered = len(unmapped_observed_fbfm40_codes) == 0
# Prepare fire hazard FBFM40 observed class inventory for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fbfm40_observed_class_inventory = fire_hazard_fbfm40_reclassification_table.copy()
# Prepare fire hazard FBFM40 observed class inventory so this workflow stage has the values required
# for downstream spatial processing and QA.
fire_hazard_fbfm40_observed_class_inventory['OBSERVED_IN_STUDY_AREA'] = \
    fire_hazard_fbfm40_observed_class_inventory['FBFM40_CODE'].isin(observed_fbfm40_codes)
# Build fire hazard fbfm40 reclassification table path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_reclassification_table_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_reclassification_table.csv'
# Build fire hazard fbfm40 observed class inventory path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_observed_class_inventory_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_observed_class_inventory.csv'
# Build fire hazard fbfm40 reclassification policy path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_reclassification_policy_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_reclassification_policy.json'
# Build fire hazard fbfm40 reclassification summary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_reclassification_summary_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_reclassification_summary.csv'
# Build fire hazard fbfm40 reclassification manifest path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_reclassification_manifest_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_reclassification_manifest.csv'
# Build fire hazard fbfm40 validation path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_validation_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_hazard_validation.csv'
# Prepare fire hazard FBFM40 reuse existing for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_reuse_existing = fire_hazard_fuel_reuse_existing
# Prepare fire hazard FBFM40 overwrite for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_fbfm40_overwrite = fire_hazard_fuel_overwrite
# Collect fire hazard FBFM40 reclassification policy in one configuration object so downstream steps
# use the same processing rules.
# Define the FBFM40 reclassification policy and record the inputs, outputs, and scoring method.
fire_hazard_fbfm40_reclassification_policy = {
    'component_id': 'FUEL_HAZARD',
    'source_product': 'LANDFIRE_FBFM40',
    'source_description': 'Scott and Burgan 40 Fire Behavior Fuel Models',
    'method': fire_hazard_fbfm40_reclassification_method,

    # Document that the hazard scores are project-specific rather than official LANDFIRE ratings.
    'modeling_note': (
        'Scores are project-specific '
        'ordinal relative hazard values '
        'and are not official LANDFIRE '
        'fire-behavior ratings.'
    ),

    # Record the source raster and the reclassified fuel-hazard output.
    'input_path': str(
        fire_hazard_fbfm40_reclassification_input_path
    ),
    'output_path': str(
        fire_hazard_fbfm40_reclassification_output_path
    ),

    # Record the valid hazard-score range and output NoData value.
    'score_range': [
        float(fire_hazard_fbfm40_score_minimum),
        float(fire_hazard_fbfm40_score_maximum),
    ],
    'output_nodata': float(
        fire_hazard_fbfm40_score_nodata
    ),

    # Record recognized and observed FBFM40 classes for QA comparison.
    'recognized_source_codes': sorted(
        fire_hazard_fbfm40_recognized_codes
    ),
    'observed_source_codes': sorted(
        observed_fbfm40_codes
    ),

    # Record any mismatches between observed classes and the reclassification policy.
    'unmapped_observed_codes': (
        unmapped_observed_fbfm40_codes
    ),
    'unobserved_policy_codes': (
        unobserved_policy_fbfm40_codes
    ),

    # Identify source classes that should remain NoData in the hazard output.
    'nodata_source_codes': sorted(
        fire_hazard_fbfm40_nodata_codes
    ),

    # Store the FBFM40 class-to-hazard-score lookup used during reclassification.
    'score_lookup': {
        str(class_code): float(hazard_score)
        for class_code, hazard_score
        in fire_hazard_fbfm40_score_lookup.items()
    },

    # Record the target raster grid used to align the reclassified fuel-hazard output.
    'target_grid': {
        'crs': fire_hazard_target_crs,
        'cell_size_meters': float(
            fire_hazard_cell_size
        ),
        'width': int(
            fire_hazard_alignment_width
        ),
        'height': int(
            fire_hazard_alignment_height
        ),
    },

    # Record the windowed processing and overwrite settings used during reclassification.
    'processing': {
        'process_by_window': True,
        'window_size': int(
            fire_hazard_fuel_processing_window_size
        ),
        'progress_interval': int(
            fire_hazard_fuel_progress_interval
        ),
        'reuse_existing': bool(
            fire_hazard_fbfm40_reuse_existing
        ),
        'overwrite_existing': bool(
            fire_hazard_fbfm40_overwrite
        ),
        'unexpected_class_action': 'raise_error',
    },
}
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_reclassification_table.to_csv(fire_hazard_fbfm40_reclassification_table_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_observed_class_inventory.to_csv(fire_hazard_fbfm40_observed_class_inventory_path,
    index=False)

# Document this operation so its role in the current fuel-hazard workflow is clear before processing
# continues.
with fire_hazard_fbfm40_reclassification_policy_path.open('w',
    encoding='utf-8') as fbfm40_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fbfm40_reclassification_policy, fbfm40_policy_file, indent=2)
# Assemble fire hazard FBFM40 reclassification summary into a table for QA review and downstream
# validation.
# Summarize the FBFM40 reclassification results for QA review and workflow documentation.
fire_hazard_fbfm40_reclassification_summary = pd.DataFrame(
    [
        {
            # Identify the hazard component and source dataset used for reclassification.
            'COMPONENT_ID': 'FUEL_HAZARD',
            'SOURCE_PRODUCT': 'LANDFIRE_FBFM40',
            'RECLASSIFICATION_METHOD': (
                fire_hazard_fbfm40_reclassification_method
            ),

            # Report how many policy classes were defined, scored, and observed.
            'POLICY_CLASS_COUNT': len(
                fire_hazard_fbfm40_reclassification_table
            ),
            'SCORED_CLASS_COUNT': len(
                scored_fbfm40_records
            ),
            'OBSERVED_CLASS_COUNT': len(
                observed_fbfm40_codes
            ),

            # Report any differences between observed source classes and the policy table.
            'UNMAPPED_OBSERVED_CLASS_COUNT': len(
                unmapped_observed_fbfm40_codes
            ),
            'UNOBSERVED_POLICY_CLASS_COUNT': len(
                unobserved_policy_fbfm40_codes
            ),

            # Record valid and NoData source-pixel totals used during reclassification.
            'VALID_SOURCE_PIXELS': (
                observed_fbfm40_valid_pixel_count
            ),
            'SOURCE_NODATA_PIXELS': (
                observed_fbfm40_nodata_pixel_count
            ),

            # Record the configured hazard-score range and output NoData value.
            'OUTPUT_SCORE_MINIMUM': (
                fire_hazard_fbfm40_score_minimum
            ),
            'OUTPUT_SCORE_MAXIMUM': (
                fire_hazard_fbfm40_score_maximum
            ),
            'OUTPUT_NODATA': (
                fire_hazard_fbfm40_score_nodata
            ),

            # Confirm source-class coverage and the output reuse/overwrite settings.
            'OBSERVED_CLASSES_COVERED': (
                fire_hazard_fbfm40_observed_classes_covered
            ),
            'REUSE_EXISTING': (
                fire_hazard_fbfm40_reuse_existing
            ),
            'OVERWRITE_EXISTING': (
                fire_hazard_fbfm40_overwrite
            ),
        }
    ]
)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_reclassification_summary.to_csv(fire_hazard_fbfm40_reclassification_summary_path,
    index=False)
# Record whether fire hazard FBFM40 reclassification configured satisfies the checks required before
# the workflow advances.
fire_hazard_fbfm40_reclassification_configured = all([fire_hazard_aligned_fuel_layers_valid,
    fire_hazard_fbfm40_observed_classes_covered, len(observed_fbfm40_codes) > 0,
    len(fire_hazard_fbfm40_score_lookup) > 0, \
        fire_hazard_fbfm40_reclassification_table_path.exists(),
    fire_hazard_fbfm40_observed_class_inventory_path.exists(),
    fire_hazard_fbfm40_reclassification_policy_path.exists(),
    fire_hazard_fbfm40_reclassification_summary_path.exists()])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_reclassification_configured:
    raise ValueError('The FBFM40 fuel-model '
        'reclassification configuration '
        'did not pass final checks.')
print(f'-> Reclassification method: {fire_hazard_fbfm40_reclassification_method}')
print(f'-> Policy classes defined: {len(fire_hazard_fbfm40_reclassification_table)}')
print(f'-> Numerical score classes: {len(scored_fbfm40_records)}')
print(f'-> Classes observed in study area: {len(observed_fbfm40_codes)}')
print(f'-> Unmapped observed classes: {len(unmapped_observed_fbfm40_codes)}')
print(f'-> Valid source pixels inspected: {observed_fbfm40_valid_pixel_count:,}')
print(f'-> Output hazard-score range: '
    f'{fire_hazard_fbfm40_score_minimum:.1f} '
    f'to '
    f'{fire_hazard_fbfm40_score_maximum:.1f}')
print(f'-> Observed classes covered: {fire_hazard_fbfm40_observed_classes_covered}')
print(f'-> Reclassification configured: {fire_hazard_fbfm40_reclassification_configured}')
print(f'-> Reclassification table saved: {fire_hazard_fbfm40_reclassification_table_path}')
print(f'-> Observed-class inventory saved: {fire_hazard_fbfm40_observed_class_inventory_path}')
print(f'-> Reclassification policy saved: {fire_hazard_fbfm40_reclassification_policy_path}')
print('\n--- FBFM40 RECLASSIFICATION CONFIGURATION SUMMARY ---')
display(fire_hazard_fbfm40_reclassification_summary)
print('\n--- OBSERVED FBFM40 CLASS SAMPLE ---')
display(fire_hazard_fbfm40_observed_class_inventory[fire_hazard_fbfm40_observed_class_inventory \
    ['OBSERVED_IN_STUDY_AREA']].head(5))
print('\nNOTE:')
print('The FBFM40 reclassification '
    'policy assigns '
    'project-specific relative '
    'surface-fuel hazard scores '
    'from zero to one.')
print('Nonburnable classes receive a '
    'score of zero, while undefined '
    'background pixels remain '
    'NoData.')
print('Every FBFM40 class observed in '
    'the aligned three-county '
    'raster is represented in the '
    'saved reclassification table.')
print('The next step is to apply this '
    'lookup to the aligned FBFM40 '
    'raster and create the '
    'categorical surface-fuel '
    'hazard-score raster.')
print('\n=== FBFM40 FUEL-MODEL RECLASSIFICATION CONFIGURED ===')


=== CONFIGURING FBFM40 FUEL-MODEL RECLASSIFICATION ===
-> Reclassification method: expert_informed_ordinal_lookup
-> Policy classes defined: 46
-> Numerical score classes: 45
-> Classes observed in study area: 30
-> Unmapped observed classes: 0
-> Valid source pixels inspected: 15,475,232
-> Output hazard-score range: 0.0 to 1.0
-> Observed classes covered: True
-> Reclassification configured: True
-> Reclassification table saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_reclassification_table.csv
-> Observed-class inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_observed_class_inventory.csv
-> Reclassification policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_reclassification_policy.json

--- FBFM40 RECLASSIFICATION CONFIGURATION SUMMARY ---


,COMPONENT_ID,SOURCE_PRODUCT,RECLASSIFICATION_METHOD,POLICY_CLASS_COUNT,SCORED_CLASS_COUNT,OBSERVED_CLASS_COUNT,UNMAPPED_OBSERVED_CLASS_COUNT,UNOBSERVED_POLICY_CLASS_COUNT,VALID_SOURCE_PIXELS,SOURCE_NODATA_PIXELS,OUTPUT_SCORE_MINIMUM,OUTPUT_SCORE_MAXIMUM,OUTPUT_NODATA,OBSERVED_CLASSES_COVERED,REUSE_EXISTING,OVERWRITE_EXISTING
0,FUEL_HAZARD,LANDFIRE_FBFM40,expert_informed_ordinal_lookup,46,45,30,0,16,15475232,121295786,0.0,1.0,-9999.0,True,True,False



--- OBSERVED FBFM40 CLASS SAMPLE ---


,FBFM40_CODE,FBFM40_MODEL,FUEL_GROUP,BURNABLE,HAZARD_SCORE,OUTPUT_DISPOSITION,RATIONALE,OBSERVED_IN_STUDY_AREA
1,91,NB1,Nonburnable,False,0.0,Score,Urban or developed nonburnable class.,True
3,93,NB3,Nonburnable,False,0.0,Score,Agricultural nonburnable class.,True
4,98,NB8,Nonburnable,False,0.0,Score,Open-water nonburnable class.,True
5,99,NB9,Nonburnable,False,0.0,Score,Barren nonburnable class.,True
6,101,GR1,Grass,True,0.2,Score,Low relative grass-fuel hazard.,True



NOTE:
The FBFM40 reclassification policy assigns project-specific relative surface-fuel hazard scores from zero to one.
Nonburnable classes receive a score of zero, while undefined background pixels remain NoData.
Every FBFM40 class observed in the aligned three-county raster is represented in the saved reclassification table.
The next step is to apply this lookup to the aligned FBFM40 raster and create the categorical surface-fuel hazard-score raster.

=== FBFM40 FUEL-MODEL RECLASSIFICATION CONFIGURED ===


### Initializing FBFM40 Fuel-Model Reclassification


In [102]:
print('=== INITIALIZING FBFM40 FUEL-MODEL RECLASSIFICATION ===')
# Prepare required FBFM40 initialization inputs for the downstream processing or validation
# performed in this workflow stage.
required_fbfm40_initialization_inputs = ['fire_hazard_fbfm40_reclassification_configured',
    'fire_hazard_fbfm40_reclassification_input_path',
    'fire_hazard_fbfm40_reclassification_output_path',
    'fire_hazard_fbfm40_reclassification_temporary_path',
    'fire_hazard_fbfm40_reclassification_method', 'fire_hazard_fbfm40_reclassification_table',
    'fire_hazard_fbfm40_score_lookup', 'fire_hazard_fbfm40_nodata_codes',
    'fire_hazard_fbfm40_recognized_codes', 'fire_hazard_fbfm40_score_minimum',
    'fire_hazard_fbfm40_score_maximum', 'fire_hazard_fbfm40_score_nodata',
    'fire_hazard_fbfm40_reclassification_manifest_path',
    'fire_hazard_fbfm40_validation_path', 'fire_hazard_fbfm40_reuse_existing',
    'fire_hazard_fbfm40_overwrite', 'fire_hazard_fuel_score_profile',
    'fire_hazard_fuel_processing_window_size', 'fire_hazard_fuel_progress_interval',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size']
# Identify missing FBFM40 initialization inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fbfm40_initialization_inputs = [object_name for object_name in \
    required_fbfm40_initialization_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 initialization inputs remain unresolved before this workflow
# stage begins.
if missing_fbfm40_initialization_inputs:
    raise NameError(f'The following FBFM40 '
        f'initialization objects are '
        f'missing:\n'
        f'{missing_fbfm40_initialization_inputs}\n\n'
        f'Run Configure Fuel-Model '
        f'Reclassification before '
        f'initializing raster '
        f'processing.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_reclassification_configured:
    raise ValueError('The FBFM40 reclassification configuration has not passed final checks.')
# Build fire hazard fbfm40 processing input path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_processing_input_path = Path(fire_hazard_fbfm40_reclassification_input_path)
# Build fire hazard fbfm40 processing output path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_output_path = Path(fire_hazard_fbfm40_reclassification_output_path)
# Build fire hazard fbfm40 processing temporary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_temporary_path = \
    Path(fire_hazard_fbfm40_reclassification_temporary_path)

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if not fire_hazard_fbfm40_processing_input_path.exists() or not \
    fire_hazard_fbfm40_processing_input_path.is_file() or \
    fire_hazard_fbfm40_processing_input_path.stat().st_size <= 0:
    raise FileNotFoundError(f'The aligned FBFM40 input '
        f'raster is unavailable:\n'
        f'{fire_hazard_fbfm40_processing_input_path}')

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(fire_hazard_fbfm40_score_lookup) == 0:
    raise ValueError('The FBFM40 score lookup contains no class-to-score mappings.')
# Prepare unexpected lookup codes for the downstream processing or validation performed in this
# workflow stage.
unexpected_lookup_codes = sorted(set(fire_hazard_fbfm40_score_lookup.keys()) - \
    set(fire_hazard_fbfm40_recognized_codes))

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if unexpected_lookup_codes:
    raise ValueError(f'The FBFM40 score lookup '
        f'contains codes that are not '
        f'recognized by the policy:\n'
        f'{unexpected_lookup_codes}')
# Set unexpected NoData codes as an explicit model or validation parameter used consistently in
# downstream calculations.
unexpected_nodata_codes = sorted(set(fire_hazard_fbfm40_nodata_codes) - \
    set(fire_hazard_fbfm40_recognized_codes))

# Stop execution if the NoData configuration is incompatible with the raster product being
# validated.
if unexpected_nodata_codes:
    raise ValueError(f'The FBFM40 NoData-code set '
        f'contains unrecognized values:\n'
        f'{unexpected_nodata_codes}')
# Evaluate invalid lookup scores so invalid inputs or outputs can be rejected before continuing.
invalid_lookup_scores = {class_code: hazard_score for class_code,
    hazard_score in fire_hazard_fbfm40_score_lookup.items() if not np.isfinite(hazard_score) or \
        hazard_score < fire_hazard_fbfm40_score_minimum or hazard_score > \
        fire_hazard_fbfm40_score_maximum}

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if invalid_lookup_scores:
    raise ValueError(f'The FBFM40 score lookup '
        f'contains invalid hazard '
        f'values:\n'
        f'{invalid_lookup_scores}')
# Set fire hazard FBFM40 maximum recognized code as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_maximum_recognized_code = max(fire_hazard_fbfm40_recognized_codes)
# Prepare fire hazard FBFM40 score lookup array for the raster calculation or validation performed
# in this processing block.
fire_hazard_fbfm40_score_lookup_array = np.full(fire_hazard_fbfm40_maximum_recognized_code + 1,
    fire_hazard_fbfm40_score_nodata, dtype=np.float32)
# Build the fire hazard fbfm40 recognized code mask mask used to isolate records required for this
# analysis.
fire_hazard_fbfm40_recognized_code_mask = np.zeros(fire_hazard_fbfm40_maximum_recognized_code + 1,
    dtype=bool)

# Iterate through fire hazard FBFM40 recognized codes so each required item receives the same
# processing and QA checks.
for recognized_code in fire_hazard_fbfm40_recognized_codes:
    # Build the fire hazard fbfm40 recognized code mask mask used to isolate records required for
    # this analysis.
    fire_hazard_fbfm40_recognized_code_mask[int(recognized_code)] = True

# Iterate through fire hazard FBFM40 score lookup.items so each required item receives the same
# processing and QA checks.
for class_code, hazard_score in fire_hazard_fbfm40_score_lookup.items():
    # Prepare fire hazard FBFM40 score lookup array so this workflow stage has the values required
    # for downstream spatial processing and QA.
    fire_hazard_fbfm40_score_lookup_array[int(class_code)] = np.float32(hazard_score)

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(fire_hazard_fbfm40_score_lookup_array) != fire_hazard_fbfm40_maximum_recognized_code + 1:
    raise ValueError('The optimized FBFM40 score lookup array has an unexpected length.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not all((fire_hazard_fbfm40_recognized_code_mask[int(class_code)] for class_code in \
    fire_hazard_fbfm40_recognized_codes)):
    raise ValueError('One or more recognized FBFM40 '
        'class codes were not '
        'registered in the optimized '
        'lookup.')
# Prepare fire hazard FBFM40 output profile so raster outputs inherit the required grid, CRS, data
# type, and NoData metadata.
fire_hazard_fbfm40_output_profile = fire_hazard_fuel_score_profile.copy()
# Update fire hazard FBFM40 output profile with the values produced by the current processing step.
fire_hazard_fbfm40_output_profile.update({'driver': 'GTiff',
    'dtype': 'float32', 'count': 1, 'nodata': fire_hazard_fbfm40_score_nodata,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': 'deflate', 'predictor': 3, 'tiled': True,
    'BIGTIFF': 'IF_SAFER'})
# Build the expected fbfm40 mask shape mask used to isolate records required for this analysis.
expected_fbfm40_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_fbfm40_mask_shape:
    raise ValueError(f'The FBFM40 study-area mask '
        f'does not match the common '
        f'target grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_fbfm40_mask_shape}')
# Calculate fire hazard FBFM40 study area pixel count for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_study_area_pixel_count = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_study_area_pixel_count == 0:
    raise ValueError('The FBFM40 study-area mask contains no included target-grid pixels.')

# Open the file in a managed context so required content is processed and the resource closes
# cleanly.
with rasterio.open(fire_hazard_fbfm40_processing_input_path) as fbfm40_source:
    # Store fire hazard fbfm40 source crs so spatial operations use the required coordinate
    # reference system.
    fire_hazard_fbfm40_source_crs = fbfm40_source.crs.to_string() if fbfm40_source.crs is not \
        None else None
    # Capture fire hazard FBFM40 source data type for source-raster compatibility and alignment QA.
    fire_hazard_fbfm40_source_dtype = fbfm40_source.dtypes[0]
    # Set fire hazard FBFM40 source NoData as an explicit model or validation parameter used
    # consistently in downstream calculations.
    fire_hazard_fbfm40_source_nodata = fbfm40_source.nodata
    # Capture fire hazard FBFM40 source width for source-raster compatibility and alignment QA.
    fire_hazard_fbfm40_source_width = fbfm40_source.width
    # Capture fire hazard FBFM40 source height for source-raster compatibility and alignment QA.
    fire_hazard_fbfm40_source_height = fbfm40_source.height
    # Prepare fire hazard FBFM40 source transform for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_source_transform = fbfm40_source.transform
    # Evaluate fire hazard fbfm40 source grid valid so invalid inputs or outputs can be rejected
    # before continuing.
    fire_hazard_fbfm40_source_grid_valid = all([fbfm40_source.count == 1,
        fbfm40_source.width == fire_hazard_alignment_width,
        fbfm40_source.height == fire_hazard_alignment_height,
        fbfm40_source.crs is not None, fbfm40_source.crs == \
            rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
        fbfm40_source.transform.almost_equals(fire_hazard_alignment_transform)])
    # Evaluate fire hazard fbfm40 source structure valid so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_source_structure_valid = all([fbfm40_source.count == 1,
        np.issubdtype(np.dtype(fbfm40_source.dtypes[0]),
        np.integer), fbfm40_source.nodata is not None])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_source_grid_valid:
    raise ValueError('The aligned FBFM40 source raster does not match the common project grid.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_source_structure_valid:
    raise ValueError('The aligned FBFM40 source '
        'raster does not have the '
        'expected categorical '
        'structure.')
# Calculate fire hazard FBFM40 window column count so raster processing covers the analysis grid in
# controlled blocks.
fire_hazard_fbfm40_window_column_count = int(np.ceil(fire_hazard_alignment_width / \
    fire_hazard_fuel_processing_window_size))
# Calculate fire hazard FBFM40 window row count so raster processing covers the analysis grid in
# controlled blocks.
fire_hazard_fbfm40_window_row_count = int(np.ceil(fire_hazard_alignment_height / \
    fire_hazard_fuel_processing_window_size))
# Calculate fire hazard FBFM40 total processing windows so raster processing covers the analysis
# grid in controlled blocks.
fire_hazard_fbfm40_total_processing_windows = fire_hazard_fbfm40_window_column_count * \
    fire_hazard_fbfm40_window_row_count
# Initialize fire hazard FBFM40 window records to collect consistent records for the stage summary
# and QA checks.
fire_hazard_fbfm40_window_records = []
# Calculate window number so raster processing covers the analysis grid in controlled blocks.
window_number = 0

# Iterate through range so each required item receives the same processing and QA checks.
for window_row in range(fire_hazard_fbfm40_window_row_count):
    # Calculate row offset so raster processing covers the analysis grid in controlled blocks.
    row_offset = window_row * fire_hazard_fuel_processing_window_size
    # Calculate window height so raster processing covers the analysis grid in controlled blocks.
    window_height = min(fire_hazard_fuel_processing_window_size,
        fire_hazard_alignment_height - row_offset)

    # Iterate through range so each required item receives the same processing and QA checks.
    for window_column in range(fire_hazard_fbfm40_window_column_count):
        # Calculate column offset so raster processing covers the analysis grid in controlled
        # blocks.
        column_offset = window_column * fire_hazard_fuel_processing_window_size
        # Calculate window width so raster processing covers the analysis grid in controlled blocks.
        window_width = min(fire_hazard_fuel_processing_window_size,
            fire_hazard_alignment_width - column_offset)
        # Calculate window number so raster processing covers the analysis grid in controlled
        # blocks.
        window_number += 1
        # Add the current record to fire hazard FBFM40 window records so the stage summary captures
        # this processing result.
        fire_hazard_fbfm40_window_records.append({'WINDOW_NUMBER': window_number,
            'WINDOW_ROW': window_row, 'WINDOW_COLUMN': window_column,
            'ROW_OFFSET': row_offset, 'COLUMN_OFFSET': column_offset,
            'WINDOW_HEIGHT': window_height, 'WINDOW_WIDTH': window_width,
            'PIXEL_COUNT': window_height * window_width})
# Assemble fire hazard FBFM40 window inventory into a table for QA review and downstream validation.
fire_hazard_fbfm40_window_inventory = pd.DataFrame(fire_hazard_fbfm40_window_records)
# Calculate planned FBFM40 processing pixels for completeness, file-integrity, or processing QA.
planned_fbfm40_processing_pixels = int(fire_hazard_fbfm40_window_inventory['PIXEL_COUNT'].sum())
# Calculate expected FBFM40 processing pixels for completeness, file-integrity, or processing QA.
expected_fbfm40_processing_pixels = int(fire_hazard_alignment_width * fire_hazard_alignment_height)
# Evaluate fire hazard fbfm40 window plan valid so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fbfm40_window_plan_valid = planned_fbfm40_processing_pixels == \
    expected_fbfm40_processing_pixels

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_window_plan_valid:
    raise ValueError(f'The FBFM40 processing-window '
        f'plan does not cover the '
        f'complete target grid.\nPlanned '
        f'pixels: '
        f'{planned_fbfm40_processing_pixels:,}\n'
        f'Expected pixels: '
        f'{expected_fbfm40_processing_pixels:,}')
# Build fire hazard fbfm40 window inventory path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_window_inventory_path = \
    Path(fire_hazard_fbfm40_reclassification_manifest_path).parent / \
    'fbfm40_reclassification_window_inventory.csv'

# Define reusable validate existing fbfm40 hazard raster logic for this phase of the workflow.
def validate_existing_fbfm40_hazard_raster(output_path):

    """
    Confirm that an existing FBFM40 hazard-score
    raster matches the common grid and output
    structure.

    This structural check is used only to determine
    whether a cached output may be reused. Detailed
    content and formula validation occur later.
    """
    # Build output path used to read, cache, or save this workflow product.
    output_path = Path(output_path)

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not output_path.exists() or not output_path.is_file() or output_path.stat().st_size <= 0:
        return False

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(output_path) as output_source:
            return all([output_source.count == 1,
                output_source.width == fire_hazard_alignment_width,
                output_source.height == fire_hazard_alignment_height,
                output_source.crs is not None, output_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                output_source.transform.almost_equals(fire_hazard_alignment_transform),
                output_source.dtypes[0] == 'float32', output_source.nodata == \
                    fire_hazard_fbfm40_score_nodata])
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception:
        return False
# Evaluate fire hazard fbfm40 existing output valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_existing_output_valid = \
    validate_existing_fbfm40_hazard_raster(fire_hazard_fbfm40_processing_output_path)
# Prepare fire hazard FBFM40 reclassification reuse output for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_fbfm40_reclassification_reuse_output = fire_hazard_fbfm40_reuse_existing and \
    fire_hazard_fbfm40_existing_output_valid

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_fbfm40_processing_temporary_path.exists():
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_fbfm40_processing_temporary_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_fbfm40_processing_output_path.exists() and fire_hazard_fbfm40_overwrite:
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_fbfm40_processing_output_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_fbfm40_processing_output_path.exists() and (not \
    fire_hazard_fbfm40_reclassification_reuse_output) and (not fire_hazard_fbfm40_overwrite):
    raise FileExistsError(f"An existing FBFM40 hazard "
        f"raster is present, but it is "
        f"not valid for cached reuse and "
        f"overwrite mode is disabled:\n"
        f"{fire_hazard_fbfm40_processing_output_path}\n\n"
        f"Set fire_hazard_data_mode to "
        f"'refresh' or remove the "
        f"invalid output before "
        f"continuing.")
# Evaluate fire hazard fbfm40 source valid pixels so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_source_valid_pixels = 0
# Calculate fire hazard FBFM40 source NoData pixels for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_source_nodata_pixels = 0
# Calculate fire hazard FBFM40 scored pixels for completeness, file-integrity, or processing QA.
fire_hazard_fbfm40_scored_pixels = 0
# Calculate fire hazard FBFM40 output NoData pixels for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_output_nodata_pixels = 0
# Calculate fire hazard FBFM40 nonburnable scored pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_nonburnable_scored_pixels = 0
# Calculate fire hazard FBFM40 background NoData pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_background_nodata_pixels = 0
# Calculate fire hazard FBFM40 unrecognized pixels for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_unrecognized_pixels = 0
# Evaluate fire hazard fbfm40 valid inside study area so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_valid_inside_study_area = 0
# Evaluate fire hazard fbfm40 valid outside study area so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_valid_outside_study_area = 0
# Set fire hazard FBFM40 output minimum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_output_minimum = None
# Set fire hazard FBFM40 output maximum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_output_maximum = None
# Prepare fire hazard FBFM40 output value sum for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_output_value_sum = 0.0
# Prepare fire hazard FBFM40 output mean for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_fbfm40_output_mean = None
# Calculate fire hazard FBFM40 class pixel counts for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_class_pixel_counts = {int(class_code): 0 for class_code in \
    sorted(fire_hazard_fbfm40_recognized_codes)}
# Calculate fire hazard FBFM40 unrecognized codes encountered for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_unrecognized_codes_encountered = set()
# Prepare fire hazard FBFM40 processing action for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_processing_action = 'Reused existing FBFM40 hazard raster' if \
    fire_hazard_fbfm40_reclassification_reuse_output else 'Create reclassified FBFM40 hazard raster'
# Calculate fire hazard FBFM40 window processing complete so raster processing covers the analysis
# grid in controlled blocks.
fire_hazard_fbfm40_window_processing_complete = False
# Evaluate fire hazard fbfm40 reclassification output valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fbfm40_reclassification_output_valid = False
# Calculate fire hazard FBFM40 last completed window so raster processing covers the analysis grid
# in controlled blocks.
fire_hazard_fbfm40_last_completed_window = 0
# Prepare fire hazard FBFM40 processing error for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_processing_error = None
# Assemble fire hazard FBFM40 initialization manifest into a table for QA review and downstream
# validation.
fire_hazard_fbfm40_initialization_manifest = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_PRODUCT': 'LANDFIRE_FBFM40', 'PROCESSING_STAGE': 'Initialized',
    'INPUT_PATH': str(fire_hazard_fbfm40_processing_input_path),
    'TEMPORARY_PATH': str(fire_hazard_fbfm40_processing_temporary_path),
    'OUTPUT_PATH': str(fire_hazard_fbfm40_processing_output_path),
    'RECLASSIFICATION_METHOD': fire_hazard_fbfm40_reclassification_method,
    'RECOGNIZED_CLASS_COUNT': len(fire_hazard_fbfm40_recognized_codes),
    'SCORED_CLASS_COUNT': len(fire_hazard_fbfm40_score_lookup),
    'NODATA_CLASS_COUNT': len(fire_hazard_fbfm40_nodata_codes),
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'PROCESSING_WINDOW_SIZE': fire_hazard_fuel_processing_window_size,
    'PROCESSING_WINDOW_COUNT': fire_hazard_fbfm40_total_processing_windows,
    'REUSE_EXISTING': fire_hazard_fbfm40_reuse_existing,
    'OVERWRITE_EXISTING': fire_hazard_fbfm40_overwrite,
    'EXISTING_OUTPUT_VALID': fire_hazard_fbfm40_existing_output_valid,
    'PROCESSING_ACTION': fire_hazard_fbfm40_processing_action,
    'INITIALIZATION_COMPLETE': True}])
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_window_inventory.to_csv(fire_hazard_fbfm40_window_inventory_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_initialization_manifest.to_csv(fire_hazard_fbfm40_reclassification_manifest_path,
    index=False)
# Prepare fire hazard FBFM40 reclassification initialized for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_fbfm40_reclassification_initialized = \
    all([fire_hazard_fbfm40_reclassification_configured,
    fire_hazard_fbfm40_source_grid_valid, fire_hazard_fbfm40_source_structure_valid,
    fire_hazard_fbfm40_window_plan_valid, len(fire_hazard_fbfm40_score_lookup) > 0,
    len(fire_hazard_fbfm40_recognized_codes) > 0, fire_hazard_fbfm40_window_inventory_path.exists(),
    fire_hazard_fbfm40_reclassification_manifest_path.exists()])

# Stop execution if the FBFM40 reclassification setup did not pass final validation.
if not fire_hazard_fbfm40_reclassification_initialized:
    raise ValueError(
        'The FBFM40 fuel-model '
        'reclassification '
        'initialization did not pass '
        'final checks.'
    )

# Report the input, output, and reclassification method configured for processing.
print(f'-> Input raster: {fire_hazard_fbfm40_processing_input_path}')
print(f'-> Output raster: {fire_hazard_fbfm40_processing_output_path}')
print(f'-> Reclassification method: {fire_hazard_fbfm40_reclassification_method}')

# Report the FBFM40 source classes and hazard-score configuration.
print(f'-> Recognized source classes: {len(fire_hazard_fbfm40_recognized_codes)}')
print(f'-> Numerical score classes: {len(fire_hazard_fbfm40_score_lookup)}')
print(f'-> NoData source classes: {len(fire_hazard_fbfm40_nodata_codes)}')
print(
    f'-> Output score range: '
    f'{fire_hazard_fbfm40_score_minimum:.1f} '
    f'to '
    f'{fire_hazard_fbfm40_score_maximum:.1f}'
)

# Report the target raster dimensions and windowed-processing configuration.
print(
    f'-> Target raster size: '
    f'{fire_hazard_alignment_width:,} × '
    f'{fire_hazard_alignment_height:,}'
)
print(
    f'-> Processing window size: '
    f'{fire_hazard_fuel_processing_window_size} '
    f'× '
    f'{fire_hazard_fuel_processing_window_size}'
)
print(f'-> Processing windows: {fire_hazard_fbfm40_total_processing_windows:,}')

# Report whether an existing output can be reused or new processing is required.
print(
    f'-> Existing output structurally valid: '
    f'{fire_hazard_fbfm40_existing_output_valid}'
)
print(
    f'-> Reuse existing output: '
    f'{fire_hazard_fbfm40_reclassification_reuse_output}'
)
print(f'-> Processing action: {fire_hazard_fbfm40_processing_action}')

# Report the files created to document the initialized reclassification workflow.
print(f'-> Window inventory saved: {fire_hazard_fbfm40_window_inventory_path}')
print(
    f'-> Initialization manifest saved: '
    f'{fire_hazard_fbfm40_reclassification_manifest_path}'
)
print(
    f'-> Reclassification initialized: '
    f'{fire_hazard_fbfm40_reclassification_initialized}'
)

# Display the initialization settings recorded for the FBFM40 workflow.
print('\n--- FBFM40 RECLASSIFICATION INITIALIZATION MANIFEST ---')
display(fire_hazard_fbfm40_initialization_manifest)

# Display a sample of the processing windows that will be used for raster reclassification.
print('\n--- FBFM40 PROCESSING WINDOW SAMPLE ---')
display(fire_hazard_fbfm40_window_inventory.head(5))

# Explain what was prepared during initialization and what processing occurs next.
print('\nNOTE:')
print(
    'The FBFM40 class-to-score '
    'lookup has been converted into '
    'optimized NumPy arrays for '
    'memory-efficient windowed '
    'raster processing.'
)
print(
    'The source raster has not yet been reclassified in this initialization step.'
)
print(
    'The next step will read each '
    'aligned FBFM40 window, '
    'validate its class codes, '
    'apply the hazard-score lookup, '
    'enforce the study-area mask, '
    'and write the temporary output '
    'raster.'
)
print('\n=== FBFM40 FUEL-MODEL RECLASSIFICATION INITIALIZED ===')


=== INITIALIZING FBFM40 FUEL-MODEL RECLASSIFICATION ===
-> Input raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\aligned_sources\landfire_fbfm40_aligned.tif
-> Output raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\normalized_sources\fbfm40_fuel_hazard_score.tif
-> Reclassification method: expert_informed_ordinal_lookup
-> Recognized source classes: 46
-> Numerical score classes: 45
-> NoData source classes: 1
-> Output score range: 0.0 to 1.0
-> Target raster size: 9,454 × 14,467
-> Processing window size: 512 × 512
-> Processing windows: 551
-> Existing output structurally valid: True
-> Reuse existing output: True
-> Processing action: Reused existing FBFM40 hazard raster
-> Window inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_reclassification_window_inventory.csv
-> Initialization manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\f

,COMPONENT_ID,SOURCE_PRODUCT,PROCESSING_STAGE,INPUT_PATH,TEMPORARY_PATH,OUTPUT_PATH,RECLASSIFICATION_METHOD,RECOGNIZED_CLASS_COUNT,SCORED_CLASS_COUNT,NODATA_CLASS_COUNT,...,CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,PROCESSING_WINDOW_SIZE,PROCESSING_WINDOW_COUNT,REUSE_EXISTING,OVERWRITE_EXISTING,EXISTING_OUTPUT_VALID,PROCESSING_ACTION,INITIALIZATION_COMPLETE
0,FUEL_HAZARD,LANDFIRE_FBFM40,Initialized,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,expert_informed_ordinal_lookup,46,45,1,...,30,9454,14467,512,551,True,False,True,Reused existing FBFM40 hazard raster,True



--- FBFM40 PROCESSING WINDOW SAMPLE ---


,WINDOW_NUMBER,WINDOW_ROW,WINDOW_COLUMN,ROW_OFFSET,COLUMN_OFFSET,WINDOW_HEIGHT,WINDOW_WIDTH,PIXEL_COUNT
0,1,0,0,0,0,512,512,262144
1,2,0,1,0,512,512,512,262144
2,3,0,2,0,1024,512,512,262144
3,4,0,3,0,1536,512,512,262144
4,5,0,4,0,2048,512,512,262144



NOTE:
The FBFM40 class-to-score lookup has been converted into optimized NumPy arrays for memory-efficient windowed raster processing.
The source raster has not yet been reclassified in this initialization step.
The next step will read each aligned FBFM40 window, validate its class codes, apply the hazard-score lookup, enforce the study-area mask, and write the temporary output raster.

=== FBFM40 FUEL-MODEL RECLASSIFICATION INITIALIZED ===


### Opening FBFM40 Source and Creating Temporary Output Raster


In [103]:
print('=== OPENING FBFM40 SOURCE AND CREATING TEMPORARY OUTPUT RASTER ===')
# Prepare required FBFM40 output creation inputs for the downstream processing or validation
# performed in this workflow stage.
required_fbfm40_output_creation_inputs = ['fire_hazard_fbfm40_reclassification_initialized',
    'fire_hazard_fbfm40_processing_input_path', 'fire_hazard_fbfm40_processing_output_path',
    'fire_hazard_fbfm40_processing_temporary_path',
    'fire_hazard_fbfm40_output_profile', 'fire_hazard_fbfm40_window_inventory',
    'fire_hazard_fbfm40_total_processing_windows',
    'fire_hazard_fbfm40_reclassification_reuse_output',
    'fire_hazard_fbfm40_existing_output_valid', 'fire_hazard_fbfm40_score_nodata',
    'fire_hazard_fuel_progress_interval', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_target_crs', 'fire_hazard_cell_size']
# Identify missing FBFM40 output creation inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fbfm40_output_creation_inputs = [object_name for object_name in \
    required_fbfm40_output_creation_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 output creation inputs remain unresolved before this workflow
# stage begins.
if missing_fbfm40_output_creation_inputs:
    raise NameError(f'The following FBFM40 '
        f'output-creation objects are '
        f'missing:\n'
        f'{missing_fbfm40_output_creation_inputs}\n\n'
        f'Run Initialize Fuel-Model '
        f'Reclassification before '
        f'creating the temporary output '
        f'raster.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_reclassification_initialized:
    raise ValueError('The FBFM40 reclassification workflow has not been initialized successfully.')
# Build fire hazard fbfm40 processing input path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_processing_input_path = Path(fire_hazard_fbfm40_processing_input_path)
# Build fire hazard fbfm40 processing output path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_output_path = Path(fire_hazard_fbfm40_processing_output_path)
# Build fire hazard fbfm40 processing temporary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_temporary_path = Path(fire_hazard_fbfm40_processing_temporary_path)

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if not fire_hazard_fbfm40_processing_input_path.exists() or not \
    fire_hazard_fbfm40_processing_input_path.is_file() or \
    fire_hazard_fbfm40_processing_input_path.stat().st_size <= 0:
    raise FileNotFoundError(f'The aligned FBFM40 source '
        f'raster is unavailable:\n'
        f'{fire_hazard_fbfm40_processing_input_path}')
# Define the fbfm40 window fields inputs required by this workflow stage.
required_fbfm40_window_fields = ['WINDOW_NUMBER',
    'ROW_OFFSET', 'COLUMN_OFFSET', 'WINDOW_HEIGHT',
    'WINDOW_WIDTH', 'PIXEL_COUNT']
# Identify missing FBFM40 window fields so unavailable prerequisites are caught before this workflow
# stage runs.
missing_fbfm40_window_fields = [field_name for field_name in required_fbfm40_window_fields if \
    field_name not in fire_hazard_fbfm40_window_inventory.columns]

# Stop execution when required fbfm40 window fields inputs are unavailable.
if missing_fbfm40_window_fields:
    raise ValueError(f'The FBFM40 processing-window '
        f'inventory is missing required '
        f'fields:\n'
        f'{missing_fbfm40_window_fields}')

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(fire_hazard_fbfm40_window_inventory) != fire_hazard_fbfm40_total_processing_windows:
    raise ValueError(f'The FBFM40 processing-window '
        f'inventory does not contain the '
        f'expected number of windows.\n'
        f'Expected: '
        f'{fire_hazard_fbfm40_total_processing_windows:,}\n'
        f'Found: '
        f'{len(fire_hazard_fbfm40_window_inventory):,}')
# Calculate planned FBFM40 output pixels for completeness, file-integrity, or processing QA.
planned_fbfm40_output_pixels = int(fire_hazard_fbfm40_window_inventory['PIXEL_COUNT'].sum())
# Calculate expected FBFM40 output pixels for completeness, file-integrity, or processing QA.
expected_fbfm40_output_pixels = int(fire_hazard_alignment_width * fire_hazard_alignment_height)

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if planned_fbfm40_output_pixels != expected_fbfm40_output_pixels:
    raise ValueError(f'The FBFM40 window inventory '
        f'does not cover the complete '
        f'output raster.\nPlanned '
        f'pixels: '
        f'{planned_fbfm40_output_pixels:,}\n'
        f'Expected pixels: '
        f'{expected_fbfm40_output_pixels:,}')
# Define the fbfm40 profile values inputs required by this workflow stage.
required_fbfm40_profile_values = {'driver': 'GTiff',
    'dtype': 'float32', 'count': 1, 'width': fire_hazard_alignment_width,
    'height': fire_hazard_alignment_height, 'nodata': fire_hazard_fbfm40_score_nodata}
# Evaluate invalid fbfm40 profile values so invalid inputs or outputs can be rejected before
# continuing.
invalid_fbfm40_profile_values = {profile_key: {'expected': expected_value,
    'configured': fire_hazard_fbfm40_output_profile.get(profile_key)} for profile_key,
    expected_value in required_fbfm40_profile_values.items() if \
        fire_hazard_fbfm40_output_profile.get(profile_key) != expected_value}

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if invalid_fbfm40_profile_values:
    raise ValueError(f'The FBFM40 output profile '
        f'contains invalid settings:\n'
        f'{invalid_fbfm40_profile_values}')
# Store configured fbfm40 output crs so spatial operations use the required coordinate reference
# system.
configured_fbfm40_output_crs = \
    rasterio.crs.CRS.from_user_input(fire_hazard_fbfm40_output_profile['crs'])
# Store expected fbfm40 output crs so spatial operations use the required coordinate reference
# system.
expected_fbfm40_output_crs = rasterio.crs.CRS.from_user_input(fire_hazard_target_crs)

# Stop execution if the raster CRS does not match the coordinate system required for spatial
# alignment.
if configured_fbfm40_output_crs != expected_fbfm40_output_crs:
    raise ValueError('The FBFM40 output profile CRS '
        'does not match the common '
        'fire-hazard target CRS.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_output_profile['transform'].almost_equals \
    (fire_hazard_alignment_transform):
    raise ValueError('The FBFM40 output profile '
        'transform does not match the '
        'common fire-hazard target '
        'grid.')
from rasterio.windows import Window
# Prepare fire hazard FBFM40 temporary output created for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fbfm40_temporary_output_created = False
# Calculate fire hazard FBFM40 initialized window count so raster processing covers the analysis
# grid in controlled blocks.
fire_hazard_fbfm40_initialized_window_count = 0
# Calculate fire hazard FBFM40 initialized pixel count for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_initialized_pixel_count = 0
# Evaluate fire hazard fbfm40 temporary grid valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_temporary_grid_valid = False
# Evaluate fire hazard fbfm40 temporary structure valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_temporary_structure_valid = False
# Prepare fire hazard FBFM40 output creation error for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_output_creation_error = None

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_reclassification_reuse_output:
    print('-> A structurally valid cached FBFM40 hazard raster will be reused.')
    print('-> Temporary output creation was skipped.')

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if fire_hazard_fbfm40_processing_temporary_path.exists():
        # Remove the temporary or replaceable raster so the next write starts from a clean output
        # path.
        fire_hazard_fbfm40_processing_temporary_path.unlink()
    # Record whether fire hazard FBFM40 output workspace ready satisfies the checks required before
    # the workflow advances.
    fire_hazard_fbfm40_output_workspace_ready = fire_hazard_fbfm40_existing_output_valid
else:
    print('-> Creating temporary FBFM40 hazard raster...')
    # Create the output directory before writing workflow products.
    fire_hazard_fbfm40_processing_temporary_path.parent.mkdir(parents=True, exist_ok=True)

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if fire_hazard_fbfm40_processing_temporary_path.exists():
        # Remove the temporary or replaceable raster so the next write starts from a clean output
        # path.
        fire_hazard_fbfm40_processing_temporary_path.unlink()

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fbfm40_processing_input_path) as fbfm40_source:
            # Evaluate source dimensions valid so invalid inputs or outputs can be rejected before
            # continuing.
            source_dimensions_valid = all([fbfm40_source.width == fire_hazard_alignment_width,
                fbfm40_source.height == fire_hazard_alignment_height])
            # Evaluate source spatial grid valid so invalid inputs or outputs can be rejected before
            # continuing.
            source_spatial_grid_valid = all([fbfm40_source.crs is not None,
                fbfm40_source.crs == expected_fbfm40_output_crs,
                fbfm40_source.transform.almost_equals(fire_hazard_alignment_transform)])
            # Evaluate source structure valid so invalid inputs or outputs can be rejected before
            # continuing.
            source_structure_valid = all([fbfm40_source.count == 1,
                np.issubdtype(np.dtype(fbfm40_source.dtypes[0]),
                np.integer), fbfm40_source.nodata is not None])

            # Stop execution if the prerequisite validation has not passed before this workflow
            # stage continues.
            if not all([source_dimensions_valid,
                source_spatial_grid_valid, source_structure_valid]):
                raise ValueError(f'The aligned FBFM40 source '
                    f'raster failed pre-processing '
                    f'checks.\nDimensions valid: '
                    f'{source_dimensions_valid}\n'
                    f'Spatial grid valid: '
                    f'{source_spatial_grid_valid}\n'
                    f'Structure valid: '
                    f'{source_structure_valid}')

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fbfm40_processing_temporary_path,
            'w', **fire_hazard_fbfm40_output_profile) as fbfm40_destination:

            # Iterate through fire hazard FBFM40 window inventory.iterrows so each required item
            # receives the same processing and QA checks.
            for _, window_record in fire_hazard_fbfm40_window_inventory.iterrows():
                # Calculate current window number so raster processing covers the analysis grid in
                # controlled blocks.
                current_window_number = int(window_record['WINDOW_NUMBER'])

                # Stop execution if this validation condition is not satisfied before dependent
                # processing continues.
                if current_window_number == 1 or current_window_number % \
                    fire_hazard_fuel_progress_interval == 0 or current_window_number == \
                    fire_hazard_fbfm40_total_processing_windows:
                    print(f'-> Initializing output window '
                        f'{current_window_number:,} of '
                        f'{fire_hazard_fbfm40_total_processing_windows:,}...')
                # Calculate output window so raster processing covers the analysis grid in
                # controlled blocks.
                output_window = Window(col_off=int(window_record['COLUMN_OFFSET']),
                    row_off=int(window_record['ROW_OFFSET']), \
                        width=int(window_record['WINDOW_WIDTH']),
                    height=int(window_record['WINDOW_HEIGHT']))
                # Set NoData output array as an explicit model or validation parameter used
                # consistently in downstream calculations.
                nodata_output_array = np.full((int(output_window.height),
                    int(output_window.width)), fire_hazard_fbfm40_score_nodata,
                    dtype=np.float32)
                # Write the processed data to the configured output resource.
                fbfm40_destination.write(nodata_output_array, 1, window=output_window)
                # Calculate fire hazard FBFM40 initialized window count so raster processing covers
                # the analysis grid in controlled blocks.
                fire_hazard_fbfm40_initialized_window_count += 1
                # Calculate fire hazard FBFM40 initialized pixel count for completeness,
                # file-integrity, or processing QA.
                fire_hazard_fbfm40_initialized_pixel_count += int(nodata_output_array.size)
                del nodata_output_array
            # Assign a descriptive band label so the exported raster documents the meaning of its
            # values.
            fbfm40_destination.set_band_description(1, 'FBFM40 relative surface-fuel hazard score')
            # Write processing metadata to the raster so the final product retains its model and
            # provenance context.
            fbfm40_destination.update_tags(COMPONENT_ID='FUEL_HAZARD',
                SOURCE_PRODUCT='LANDFIRE_FBFM40', PROCESSING_STAGE='Temporary output initialized',
                RECLASSIFICATION_METHOD=fire_hazard_fbfm40_reclassification_method,
                SCORE_MINIMUM=fire_hazard_fbfm40_score_minimum,
                SCORE_MAXIMUM=fire_hazard_fbfm40_score_maximum,
                OUTPUT_NODATA=fire_hazard_fbfm40_score_nodata,
                SOURCE_PATH=str(fire_hazard_fbfm40_processing_input_path),
                TARGET_CRS=fire_hazard_target_crs, TARGET_CELL_SIZE_METERS=fire_hazard_cell_size,
                TARGET_WIDTH=fire_hazard_alignment_width, \
                    TARGET_HEIGHT=fire_hazard_alignment_height,
                WINDOW_COUNT=fire_hazard_fbfm40_total_processing_windows)
        # Prepare fire hazard FBFM40 temporary output created for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fbfm40_temporary_output_created = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard FBFM40 output creation error for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fbfm40_output_creation_error = str(error)
        # Prepare fire hazard FBFM40 temporary output created for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fbfm40_temporary_output_created = False

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_fbfm40_processing_temporary_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_fbfm40_processing_temporary_path.unlink()

    # Stop execution if this validation condition is not satisfied before dependent processing
    # continues.
    if not fire_hazard_fbfm40_temporary_output_created:
        raise ValueError(f'The temporary FBFM40 hazard '
            f'raster could not be created.\n\n'
            f'Input path: '
            f'{fire_hazard_fbfm40_processing_input_path}\n'
            f'Temporary path: '
            f'{fire_hazard_fbfm40_processing_temporary_path}\n'
            f'Error: '
            f'{fire_hazard_fbfm40_output_creation_error}')
    # Evaluate initialized window count valid so invalid inputs or outputs can be rejected before
    # continuing.
    initialized_window_count_valid = fire_hazard_fbfm40_initialized_window_count == \
        fire_hazard_fbfm40_total_processing_windows
    # Evaluate initialized pixel count valid so invalid inputs or outputs can be rejected before
    # continuing.
    initialized_pixel_count_valid = fire_hazard_fbfm40_initialized_pixel_count == \
        expected_fbfm40_output_pixels

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if not all([initialized_window_count_valid, initialized_pixel_count_valid]):

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_fbfm40_processing_temporary_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_fbfm40_processing_temporary_path.unlink()
        raise ValueError(f'The temporary FBFM40 raster '
            f'was not initialized across the '
            f'complete target grid.\n'
            f'Initialized windows: '
            f'{fire_hazard_fbfm40_initialized_window_count:,}\n'
            f'Expected windows: '
            f'{fire_hazard_fbfm40_total_processing_windows:,}\n'
            f'Initialized pixels: '
            f'{fire_hazard_fbfm40_initialized_pixel_count:,}\n'
            f'Expected pixels: '
            f'{expected_fbfm40_output_pixels:,}')

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(fire_hazard_fbfm40_processing_temporary_path) as temporary_source:
        # Evaluate fire hazard fbfm40 temporary grid valid so invalid inputs or outputs can be
        # rejected before continuing.
        fire_hazard_fbfm40_temporary_grid_valid = all([temporary_source.width == \
            fire_hazard_alignment_width,
            temporary_source.height == fire_hazard_alignment_height,
            temporary_source.crs is not None, temporary_source.crs == expected_fbfm40_output_crs,
            temporary_source.transform.almost_equals(fire_hazard_alignment_transform)])
        # Evaluate fire hazard fbfm40 temporary structure valid so invalid inputs or outputs can be
        # rejected before continuing.
        fire_hazard_fbfm40_temporary_structure_valid = all([temporary_source.count == 1,
            temporary_source.dtypes[0] == 'float32', temporary_source.nodata == \
                fire_hazard_fbfm40_score_nodata])

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if not all([fire_hazard_fbfm40_temporary_grid_valid,
        fire_hazard_fbfm40_temporary_structure_valid]):

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_fbfm40_processing_temporary_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_fbfm40_processing_temporary_path.unlink()
        raise ValueError(f'The initialized temporary '
            f'FBFM40 raster failed '
            f'structural validation.\nGrid '
            f'valid: '
            f'{fire_hazard_fbfm40_temporary_grid_valid}\n'
            f'Structure valid: '
            f'{fire_hazard_fbfm40_temporary_structure_valid}')
    # Record whether fire hazard FBFM40 output workspace ready satisfies the checks required before
    # the workflow advances.
    fire_hazard_fbfm40_output_workspace_ready = all([fire_hazard_fbfm40_temporary_output_created,
        fire_hazard_fbfm40_processing_temporary_path.exists(),
        fire_hazard_fbfm40_temporary_grid_valid, fire_hazard_fbfm40_temporary_structure_valid])
# Configure fire hazard fbfm40 processing session policy used for reliable requests to the source
# service.
fire_hazard_fbfm40_processing_session_policy = (
    {'source_open_mode': 'read', 'temporary_output_open_mode': 'r+',
        'keep_dataset_handles_between_cells': False, 'reason': 'Close files at the end of each '
        'notebook cell to reduce '
        'corruption risk and improve '
        'restart safety.'}
)
# Assemble fire hazard FBFM40 output creation summary into a table for QA review and downstream
# validation.
fire_hazard_fbfm40_output_creation_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_PRODUCT': 'LANDFIRE_FBFM40', 'INPUT_PATH': \
        str(fire_hazard_fbfm40_processing_input_path),
    'TEMPORARY_OUTPUT_PATH': str(fire_hazard_fbfm40_processing_temporary_path),
    'PERMANENT_OUTPUT_PATH': str(fire_hazard_fbfm40_processing_output_path),
    'REUSE_CACHED_OUTPUT': fire_hazard_fbfm40_reclassification_reuse_output,
    'TEMPORARY_OUTPUT_CREATED': fire_hazard_fbfm40_temporary_output_created,
    'INITIALIZED_WINDOWS': fire_hazard_fbfm40_initialized_window_count,
    'INITIALIZED_PIXELS': fire_hazard_fbfm40_initialized_pixel_count,
    'TARGET_WINDOWS': fire_hazard_fbfm40_total_processing_windows,
    'TARGET_PIXELS': expected_fbfm40_output_pixels,
    'TEMPORARY_GRID_VALID': fire_hazard_fbfm40_temporary_grid_valid,
    'TEMPORARY_STRUCTURE_VALID': fire_hazard_fbfm40_temporary_structure_valid,
    'OUTPUT_WORKSPACE_READY': fire_hazard_fbfm40_output_workspace_ready,
    'ERROR_MESSAGE': fire_hazard_fbfm40_output_creation_error}])
# Build fire hazard fbfm40 output creation summary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_output_creation_summary_path = \
    Path(fire_hazard_fbfm40_reclassification_manifest_path).parent / \
    'fbfm40_output_creation_summary.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_output_creation_summary.to_csv(fire_hazard_fbfm40_output_creation_summary_path,
    index=False)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_output_workspace_ready:
    raise ValueError('The FBFM40 output workspace is not ready for window processing.')
print(f'-> Source raster: {fire_hazard_fbfm40_processing_input_path}')
print(f'-> Permanent output: {fire_hazard_fbfm40_processing_output_path}')
print(f'-> Reuse cached output: {fire_hazard_fbfm40_reclassification_reuse_output}')
print(f'-> Temporary output created: {fire_hazard_fbfm40_temporary_output_created}')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_temporary_output_created:
    print(f'-> Temporary output: {fire_hazard_fbfm40_processing_temporary_path}')
    print(f'-> Initialized windows: {fire_hazard_fbfm40_initialized_window_count:,}')
    print(f'-> Initialized pixels: {fire_hazard_fbfm40_initialized_pixel_count:,}')
    print(f'-> Temporary grid valid: {fire_hazard_fbfm40_temporary_grid_valid}')
    print(f'-> Temporary structure valid: {fire_hazard_fbfm40_temporary_structure_valid}')
print(f'-> Output workspace ready: {fire_hazard_fbfm40_output_workspace_ready}')
print(f'-> Output-creation summary saved: {fire_hazard_fbfm40_output_creation_summary_path}')
print('\n--- FBFM40 OUTPUT-CREATION SUMMARY ---')
display(fire_hazard_fbfm40_output_creation_summary)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_reclassification_reuse_output:
    print('A valid cached FBFM40 hazard '
        'raster is ready for '
        'statistical inspection and '
        'final validation.')
else:
    print('The temporary FBFM40 output '
        'raster has been created on the '
        'common project grid and '
        'initialized entirely with '
        'NoData.')
    print('No fuel-model hazard scores have been written yet.')
    print('The source and destination raster files were closed safely at the end of this cell.')
    print('The next step will reopen the '
        'source in read mode and the '
        'temporary output in update '
        'mode, then reclassify every '
        'processing window.')
print('\n=== FBFM40 SOURCE AND OUTPUT WORKSPACE READY ===')



=== OPENING FBFM40 SOURCE AND CREATING TEMPORARY OUTPUT RASTER ===
-> A structurally valid cached FBFM40 hazard raster will be reused.
-> Temporary output creation was skipped.
-> Source raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\aligned_sources\landfire_fbfm40_aligned.tif
-> Permanent output: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\normalized_sources\fbfm40_fuel_hazard_score.tif
-> Reuse cached output: True
-> Temporary output created: False
-> Output workspace ready: True
-> Output-creation summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_output_creation_summary.csv

--- FBFM40 OUTPUT-CREATION SUMMARY ---


,COMPONENT_ID,SOURCE_PRODUCT,INPUT_PATH,TEMPORARY_OUTPUT_PATH,PERMANENT_OUTPUT_PATH,REUSE_CACHED_OUTPUT,TEMPORARY_OUTPUT_CREATED,INITIALIZED_WINDOWS,INITIALIZED_PIXELS,TARGET_WINDOWS,TARGET_PIXELS,TEMPORARY_GRID_VALID,TEMPORARY_STRUCTURE_VALID,OUTPUT_WORKSPACE_READY,ERROR_MESSAGE
0,FUEL_HAZARD,LANDFIRE_FBFM40,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,False,0,0,551,136771018,False,False,True,None



NOTE:
A valid cached FBFM40 hazard raster is ready for statistical inspection and final validation.

=== FBFM40 SOURCE AND OUTPUT WORKSPACE READY ===


### Processing FBFM40 Reclassification Windows


In [104]:
print('=== PROCESSING FBFM40 RECLASSIFICATION WINDOWS ===')
# Calculate required FBFM40 window processing inputs so raster processing covers the analysis grid
# in controlled blocks.
required_fbfm40_window_processing_inputs = ['fire_hazard_fbfm40_reclassification_initialized',
    'fire_hazard_fbfm40_output_workspace_ready', 'fire_hazard_fbfm40_processing_input_path',
    'fire_hazard_fbfm40_processing_output_path', 'fire_hazard_fbfm40_processing_temporary_path',
    'fire_hazard_fbfm40_reclassification_reuse_output',
    'fire_hazard_fbfm40_window_inventory', 'fire_hazard_fbfm40_total_processing_windows',
    'fire_hazard_fbfm40_score_lookup_array', 'fire_hazard_fbfm40_recognized_code_mask',
    'fire_hazard_fbfm40_recognized_codes', 'fire_hazard_fbfm40_nodata_codes',
    'fire_hazard_fbfm40_class_pixel_counts', 'fire_hazard_fbfm40_maximum_recognized_code',
    'fire_hazard_fbfm40_score_minimum', 'fire_hazard_fbfm40_score_maximum',
    'fire_hazard_fbfm40_score_nodata', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_fuel_progress_interval']
# Identify missing FBFM40 window processing inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fbfm40_window_processing_inputs = [object_name for object_name in \
    required_fbfm40_window_processing_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 window processing inputs remain unresolved before this workflow
# stage begins.
if missing_fbfm40_window_processing_inputs:
    raise NameError(f'The following FBFM40 '
        f'window-processing objects are '
        f'missing:\n'
        f'{missing_fbfm40_window_processing_inputs}\n\n'
        f'Run Initialize Fuel-Model '
        f'Reclassification and Open '
        f'Source and Create Output '
        f'Raster before processing '
        f'raster windows.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_reclassification_initialized:
    raise ValueError('The FBFM40 reclassification workflow has not been initialized successfully.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_output_workspace_ready:
    raise ValueError('The FBFM40 output workspace is not ready.')
# Build fire hazard fbfm40 processing input path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_processing_input_path = Path(fire_hazard_fbfm40_processing_input_path)
# Build fire hazard fbfm40 processing output path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_output_path = Path(fire_hazard_fbfm40_processing_output_path)
# Build fire hazard fbfm40 processing temporary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_temporary_path = Path(fire_hazard_fbfm40_processing_temporary_path)
from rasterio.windows import Window
# Evaluate fire hazard fbfm40 source valid pixels so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_source_valid_pixels = 0
# Calculate fire hazard FBFM40 source NoData pixels for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_source_nodata_pixels = 0
# Calculate fire hazard FBFM40 scored pixels for completeness, file-integrity, or processing QA.
fire_hazard_fbfm40_scored_pixels = 0
# Calculate fire hazard FBFM40 output NoData pixels for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_output_nodata_pixels = 0
# Calculate fire hazard FBFM40 nonburnable scored pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_nonburnable_scored_pixels = 0
# Calculate fire hazard FBFM40 background NoData pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_background_nodata_pixels = 0
# Calculate fire hazard FBFM40 unrecognized pixels for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_unrecognized_pixels = 0
# Evaluate fire hazard fbfm40 valid inside study area so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_valid_inside_study_area = 0
# Evaluate fire hazard fbfm40 valid outside study area so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_valid_outside_study_area = 0
# Set fire hazard FBFM40 output minimum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_output_minimum = None
# Set fire hazard FBFM40 output maximum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_output_maximum = None
# Prepare fire hazard FBFM40 output value sum for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_output_value_sum = 0.0
# Prepare fire hazard FBFM40 output mean for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_fbfm40_output_mean = None
# Calculate fire hazard FBFM40 last completed window so raster processing covers the analysis grid
# in controlled blocks.
fire_hazard_fbfm40_last_completed_window = 0
# Prepare fire hazard FBFM40 processing error for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_processing_error = None
# Calculate fire hazard FBFM40 unrecognized codes encountered for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_unrecognized_codes_encountered = set()
# Calculate fire hazard FBFM40 class pixel counts for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_class_pixel_counts = {int(class_code): 0 for class_code in \
    sorted(fire_hazard_fbfm40_recognized_codes)}

# Define reusable update fbfm40 output statistics logic for this phase of the workflow.
def update_fbfm40_output_statistics(output_array, study_area_mask):

    """
    Update raster-wide FBFM40 output statistics from
    one processing window.
    """
    global fire_hazard_fbfm40_scored_pixels
    global fire_hazard_fbfm40_output_nodata_pixels
    global fire_hazard_fbfm40_valid_inside_study_area
    global fire_hazard_fbfm40_valid_outside_study_area
    global fire_hazard_fbfm40_output_minimum
    global fire_hazard_fbfm40_output_maximum
    global fire_hazard_fbfm40_output_value_sum
    # Build the valid output mask mask used to isolate records required for this analysis.
    valid_output_mask = np.isfinite(output_array) & (output_array != \
        fire_hazard_fbfm40_score_nodata)
    # Evaluate valid output values so invalid inputs or outputs can be rejected before continuing.
    valid_output_values = output_array[valid_output_mask]
    # Calculate fire hazard FBFM40 scored pixels for completeness, file-integrity, or processing QA.
    fire_hazard_fbfm40_scored_pixels += int(valid_output_mask.sum())
    # Calculate fire hazard FBFM40 output NoData pixels for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_output_nodata_pixels += int((~valid_output_mask).sum())
    # Evaluate fire hazard fbfm40 valid inside study area so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_valid_inside_study_area += int((valid_output_mask & study_area_mask).sum())
    # Evaluate fire hazard fbfm40 valid outside study area so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_valid_outside_study_area += int((valid_output_mask & ~study_area_mask).sum())

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if valid_output_values.size > 0:
        # Calculate window minimum so raster processing covers the analysis grid in controlled
        # blocks.
        window_minimum = float(valid_output_values.min())
        # Calculate window maximum so raster processing covers the analysis grid in controlled
        # blocks.
        window_maximum = float(valid_output_values.max())

        # Stop execution if this validation condition is not satisfied before dependent processing
        # continues.
        if fire_hazard_fbfm40_output_minimum is None:
            # Set fire hazard FBFM40 output minimum as an explicit model or validation parameter
            # used consistently in downstream calculations.
            fire_hazard_fbfm40_output_minimum = window_minimum
        else:
            # Set fire hazard FBFM40 output minimum as an explicit model or validation parameter
            # used consistently in downstream calculations.
            fire_hazard_fbfm40_output_minimum = min(fire_hazard_fbfm40_output_minimum,
                window_minimum)

        # Stop execution if this validation condition is not satisfied before dependent processing
        # continues.
        if fire_hazard_fbfm40_output_maximum is None:
            # Set fire hazard FBFM40 output maximum as an explicit model or validation parameter
            # used consistently in downstream calculations.
            fire_hazard_fbfm40_output_maximum = window_maximum
        else:
            # Set fire hazard FBFM40 output maximum as an explicit model or validation parameter
            # used consistently in downstream calculations.
            fire_hazard_fbfm40_output_maximum = max(fire_hazard_fbfm40_output_maximum,
                window_maximum)
        # Prepare fire hazard FBFM40 output value sum for the fuel-hazard calculation and subsequent
        # QA checks.
        fire_hazard_fbfm40_output_value_sum += float(valid_output_values.sum(dtype=np.float64))

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_reclassification_reuse_output:
    print('-> Reusing cached FBFM40 hazard raster.')
    # Prepare fire hazard FBFM40 processing action for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_processing_action = 'Reused existing FBFM40 hazard raster'

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fbfm40_processing_output_path) as cached_source:

            # Iterate through cached source.block windows so each required item receives the same
            # processing and QA checks.
            for _, raster_window in cached_source.block_windows(1):
                # Prepare output array for the raster calculation or validation performed in this
                # processing block.
                output_array = cached_source.read(1, window=raster_window)
                # Prepare row start for the downstream processing or validation performed in this
                # workflow stage.
                row_start = int(raster_window.row_off)
                # Prepare row end for the downstream processing or validation performed in this
                # workflow stage.
                row_end = int(raster_window.row_off + raster_window.height)
                # Prepare column start for the downstream processing or validation performed in this
                # workflow stage.
                column_start = int(raster_window.col_off)
                # Prepare column end for the downstream processing or validation performed in this
                # workflow stage.
                column_end = int(raster_window.col_off + raster_window.width)
                # Build the study area window mask mask used to isolate records required for this
                # analysis.
                study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                    column_start:column_end]
                # Update running raster statistics so final QA reflects all processed windows.
                update_fbfm40_output_statistics(output_array, study_area_window_mask)
                del output_array
        # Calculate fire hazard FBFM40 window processing complete so raster processing covers the
        # analysis grid in controlled blocks.
        fire_hazard_fbfm40_window_processing_complete = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard FBFM40 processing error for the fuel-hazard calculation and subsequent
        # QA checks.
        fire_hazard_fbfm40_processing_error = str(error)
        # Calculate fire hazard FBFM40 window processing complete so raster processing covers the
        # analysis grid in controlled blocks.
        fire_hazard_fbfm40_window_processing_complete = False
else:
    print('-> Reclassifying aligned FBFM40 raster...')
    # Prepare fire hazard FBFM40 processing action for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_processing_action = 'Created reclassified FBFM40 hazard raster'

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not fire_hazard_fbfm40_processing_temporary_path.exists() or not \
        fire_hazard_fbfm40_processing_temporary_path.is_file():
        raise FileNotFoundError(f'The initialized temporary '
            f'FBFM40 output raster is '
            f'unavailable:\n'
            f'{fire_hazard_fbfm40_processing_temporary_path}')

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fbfm40_processing_input_path) as fbfm40_source:

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(fire_hazard_fbfm40_processing_temporary_path,
                'r+') as fbfm40_destination:
                # Set source NoData as an explicit model or validation parameter used consistently
                # in downstream calculations.
                source_nodata = fbfm40_source.nodata
                # Calculate total windows so raster processing covers the analysis grid in
                # controlled blocks.
                total_windows = len(fire_hazard_fbfm40_window_inventory)

                # Iterate through fire hazard FBFM40 window inventory.iterrows so each required item
                # receives the same processing and QA checks.
                for _, window_record in fire_hazard_fbfm40_window_inventory.iterrows():
                    # Calculate window number so raster processing covers the analysis grid in
                    # controlled blocks.
                    window_number = int(window_record['WINDOW_NUMBER'])

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if window_number == 1 or window_number % fire_hazard_fuel_progress_interval \
                        == 0 or window_number == total_windows:
                        print(f'-> Processing FBFM40 window '
                            f'{window_number:,} of '
                            f'{total_windows:,}...')
                    # Calculate processing window so raster processing covers the analysis grid in
                    # controlled blocks.
                    processing_window = Window(col_off=int(window_record['COLUMN_OFFSET']),
                        row_off=int(window_record['ROW_OFFSET']), \
                            width=int(window_record['WINDOW_WIDTH']),
                        height=int(window_record['WINDOW_HEIGHT']))
                    # Prepare source array for the raster calculation or validation performed in
                    # this processing block.
                    source_array = fbfm40_source.read(1, window=processing_window)
                    # Prepare row start for the downstream processing or validation performed in
                    # this workflow stage.
                    row_start = int(processing_window.row_off)
                    # Prepare row end for the downstream processing or validation performed in this
                    # workflow stage.
                    row_end = int(processing_window.row_off + processing_window.height)
                    # Prepare column start for the downstream processing or validation performed in
                    # this workflow stage.
                    column_start = int(processing_window.col_off)
                    # Prepare column end for the downstream processing or validation performed in
                    # this workflow stage.
                    column_end = int(processing_window.col_off + processing_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Build the finite source mask mask used to isolate records required for this
                    # analysis.
                    finite_source_mask = np.isfinite(source_array)

                    # Stop execution if the NoData configuration is incompatible with the raster
                    # product being validated.
                    if source_nodata is None:
                        # Build the source data mask mask used to isolate records required for this
                        # analysis.
                        source_data_mask = finite_source_mask
                    else:
                        # Build the source data mask mask used to isolate records required for this
                        # analysis.
                        source_data_mask = finite_source_mask & (source_array != source_nodata)
                    # Build the source inside study area mask mask used to isolate records required
                    # for this analysis.
                    source_inside_study_area_mask = source_data_mask & study_area_window_mask
                    # Evaluate fire hazard fbfm40 source valid pixels so invalid inputs or outputs
                    # can be rejected before continuing.
                    fire_hazard_fbfm40_source_valid_pixels += \
                        int(source_inside_study_area_mask.sum())
                    # Calculate fire hazard FBFM40 source NoData pixels for completeness,
                    # file-integrity, or processing QA.
                    fire_hazard_fbfm40_source_nodata_pixels += int((~source_data_mask).sum())
                    # Prepare output array for the raster calculation or validation performed in
                    # this processing block.
                    output_array = np.full(source_array.shape,
                        fire_hazard_fbfm40_score_nodata, dtype=np.float32)

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if source_inside_study_area_mask.any():
                        # Prepare source values for the downstream processing or validation
                        # performed in this workflow stage.
                        source_values = source_array[source_inside_study_area_mask]
                        # Prepare rounded source values for the downstream processing or validation
                        # performed in this workflow stage.
                        rounded_source_values = np.rint(source_values).astype(np.int32)
                        # Build the fractional source mask mask used to isolate records required for
                        # this analysis.
                        fractional_source_mask = np.abs(source_values.astype(np.float64) - \
                            rounded_source_values.astype(np.float64)) > 1e-06

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if fractional_source_mask.any():
                            raise ValueError(f'Fractional FBFM40 class values '
                                f'were encountered during window '
                                f'{window_number:,}.')
                        # Prepare code within lookup range for the downstream processing or
                        # validation performed in this workflow stage.
                        code_within_lookup_range = (rounded_source_values >= 0) & \
                            (rounded_source_values <= fire_hazard_fbfm40_maximum_recognized_code)
                        # Build the recognized source mask mask used to isolate records required for
                        # this analysis.
                        recognized_source_mask = np.zeros(rounded_source_values.shape, dtype=bool)
                        # Build the recognized source mask mask used to isolate records required for
                        # this analysis.
                        recognized_source_mask[code_within_lookup_range] = \
                            fire_hazard_fbfm40_recognized_code_mask[rounded_source_values \
                            [code_within_lookup_range]]
                        # Build the unrecognized source mask mask used to isolate records required
                        # for this analysis.
                        unrecognized_source_mask = ~recognized_source_mask

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if unrecognized_source_mask.any():
                            # Prepare unknown codes for the downstream processing or validation
                            # performed in this workflow stage.
                            unknown_codes = \
                                np.unique(rounded_source_values[unrecognized_source_mask])
                            # Update fire hazard FBFM40 unrecognized codes encountered with the
                            # values produced by the current processing step.
                            fire_hazard_fbfm40_unrecognized_codes_encountered.update((int(code) \
                                for code in unknown_codes))
                            # Calculate fire hazard FBFM40 unrecognized pixels for completeness,
                            # file-integrity, or processing QA.
                            fire_hazard_fbfm40_unrecognized_pixels += \
                                int(unrecognized_source_mask.sum())
                            raise ValueError(f'Unrecognized FBFM40 class '
                                f'codes were encountered:\n'
                                f'{sorted(fire_hazard_fbfm40_unrecognized_codes_encountered)}')
                        # Prepare unique codes and unique counts so this workflow stage has the
                        # values required for downstream spatial processing and QA.
                        unique_codes, unique_counts = np.unique(rounded_source_values,
                            return_counts=True)

                        # Iterate through zip so each required item receives the same processing and
                        # QA checks.
                        for class_code, class_count in zip(unique_codes, unique_counts):
                            # Prepare fire hazard FBFM40 class pixel counts so this workflow stage
                            # has the values required for downstream spatial processing and QA.
                            fire_hazard_fbfm40_class_pixel_counts[int(class_code)] += \
                                int(class_count)
                        # Prepare reclassified values for the downstream processing or validation
                        # performed in this workflow stage.
                        reclassified_values = \
                            fire_hazard_fbfm40_score_lookup_array[rounded_source_values]
                        # Prepare output array so this workflow stage has the values required for
                        # downstream spatial processing and QA.
                        output_array[source_inside_study_area_mask] = reclassified_values
                        # Build the background mask mask used to isolate records required for this
                        # analysis.
                        background_mask = np.isin(rounded_source_values,
                            list(fire_hazard_fbfm40_nodata_codes))
                        # Calculate fire hazard FBFM40 background NoData pixels for completeness,
                        # file-integrity, or processing QA.
                        fire_hazard_fbfm40_background_nodata_pixels += int(background_mask.sum())
                        # Prepare scored nonburnable codes for the fuel-hazard calculation and
                        # subsequent QA checks.
                        scored_nonburnable_codes = [class_code for class_code,
                            score in fire_hazard_fbfm40_score_lookup.items() if np.isclose(score,
                            0.0)]
                        # Build the nonburnable mask mask used to isolate records required for this
                        # analysis.
                        nonburnable_mask = np.isin(rounded_source_values, scored_nonburnable_codes)
                        # Calculate fire hazard FBFM40 nonburnable scored pixels for completeness,
                        # file-integrity, or processing QA.
                        fire_hazard_fbfm40_nonburnable_scored_pixels += int(nonburnable_mask.sum())
                    # Prepare output array so this workflow stage has the values required for
                    # downstream spatial processing and QA.
                    output_array[~study_area_window_mask] = fire_hazard_fbfm40_score_nodata
                    # Build the valid window mask mask used to isolate records required for this
                    # analysis.
                    valid_window_mask = np.isfinite(output_array) & (output_array != \
                        fire_hazard_fbfm40_score_nodata)
                    # Evaluate valid window values so invalid inputs or outputs can be rejected
                    # before continuing.
                    valid_window_values = output_array[valid_window_mask]

                    # Stop execution if the prerequisite validation has not passed before this
                    # workflow stage continues.
                    if valid_window_values.size > 0:

                        # Stop execution if the prerequisite validation has not passed before this
                        # workflow stage continues.
                        if valid_window_values.min() < fire_hazard_fbfm40_score_minimum or \
                            valid_window_values.max() > fire_hazard_fbfm40_score_maximum:
                            raise ValueError('An FBFM40 hazard score fell '
                                'outside the configured '
                                'zero-to-one range.')
                    # Write the processed data to the configured output resource.
                    fbfm40_destination.write(output_array, 1, window=processing_window)
                    # Update running raster statistics so final QA reflects all processed windows.
                    update_fbfm40_output_statistics(output_array, study_area_window_mask)
                    # Calculate fire hazard FBFM40 last completed window so raster processing covers
                    # the analysis grid in controlled blocks.
                    fire_hazard_fbfm40_last_completed_window = window_number
                    del source_array
                    del finite_source_mask
                    del source_data_mask
                    del source_inside_study_area_mask
                    del output_array
                # Write processing metadata to the raster so the final product retains its model and
                # provenance context.
                fbfm40_destination.update_tags(PROCESSING_STAGE='Window reclassification complete',
                    PROCESSING_ACTION=fire_hazard_fbfm40_processing_action,
                    COMPLETED_WINDOWS=fire_hazard_fbfm40_last_completed_window,
                    VALID_SOURCE_PIXELS=fire_hazard_fbfm40_source_valid_pixels,
                    SCORED_OUTPUT_PIXELS=fire_hazard_fbfm40_scored_pixels,
                    UNRECOGNIZED_PIXELS=fire_hazard_fbfm40_unrecognized_pixels)
        # Calculate fire hazard FBFM40 window processing complete so raster processing covers the
        # analysis grid in controlled blocks.
        fire_hazard_fbfm40_window_processing_complete = fire_hazard_fbfm40_last_completed_window \
            == fire_hazard_fbfm40_total_processing_windows and \
            fire_hazard_fbfm40_unrecognized_pixels == 0
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard FBFM40 processing error for the fuel-hazard calculation and subsequent
        # QA checks.
        fire_hazard_fbfm40_processing_error = str(error)
        # Calculate fire hazard FBFM40 window processing complete so raster processing covers the
        # analysis grid in controlled blocks.
        fire_hazard_fbfm40_window_processing_complete = False

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_scored_pixels > 0:
    # Prepare fire hazard FBFM40 output mean for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_output_mean = fire_hazard_fbfm40_output_value_sum / \
        fire_hazard_fbfm40_scored_pixels

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_window_processing_complete:
    raise ValueError(f'FBFM40 raster-window '
        f'processing did not complete '
        f'successfully.\n\nLast '
        f'completed window: '
        f'{fire_hazard_fbfm40_last_completed_window:,}\n'
        f'Expected windows: '
        f'{fire_hazard_fbfm40_total_processing_windows:,}\n'
        f'Unrecognized pixels: '
        f'{fire_hazard_fbfm40_unrecognized_pixels:,}\n'
        f'Unrecognized codes: '
        f'{sorted(fire_hazard_fbfm40_unrecognized_codes_encountered)}\n'
        f'Error: '
        f'{fire_hazard_fbfm40_processing_error}')
# Calculate fire hazard FBFM40 class count inventory for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_class_count_inventory = fire_hazard_fbfm40_reclassification_table[['FBFM40_CODE',
    'FBFM40_MODEL', 'FUEL_GROUP', 'BURNABLE', 'HAZARD_SCORE',
    'OUTPUT_DISPOSITION']].copy()
# Prepare fire hazard FBFM40 class count inventory so this workflow stage has the values required
# for downstream spatial processing and QA.
fire_hazard_fbfm40_class_count_inventory['PIXEL_COUNT'] = \
    fire_hazard_fbfm40_class_count_inventory['FBFM40_CODE'].map \
    (fire_hazard_fbfm40_class_pixel_counts).fillna(0).astype(np.int64)
# Prepare fire hazard FBFM40 class count inventory so this workflow stage has the values required
# for downstream spatial processing and QA.
fire_hazard_fbfm40_class_count_inventory['OBSERVED'] = \
    fire_hazard_fbfm40_class_count_inventory['PIXEL_COUNT'] > 0
# Calculate fire hazard FBFM40 class count inventory path for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_class_count_inventory_path = \
    Path(fire_hazard_fbfm40_reclassification_manifest_path).parent / 'fbfm40_class_pixel_counts.csv'
# Build fire hazard fbfm40 window processing summary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_window_processing_summary_path = \
    Path(fire_hazard_fbfm40_reclassification_manifest_path).parent / \
    'fbfm40_window_processing_summary.csv'
# Assemble fire hazard FBFM40 window processing summary into a table for QA review and downstream
# validation.
fire_hazard_fbfm40_window_processing_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_PRODUCT': 'LANDFIRE_FBFM40', 'PROCESSING_ACTION': fire_hazard_fbfm40_processing_action,
    'REUSED_CACHED_OUTPUT': fire_hazard_fbfm40_reclassification_reuse_output,
    'TOTAL_WINDOWS': fire_hazard_fbfm40_total_processing_windows,
    'COMPLETED_WINDOWS': fire_hazard_fbfm40_last_completed_window,
    'SOURCE_VALID_PIXELS': fire_hazard_fbfm40_source_valid_pixels,
    'SOURCE_NODATA_PIXELS': fire_hazard_fbfm40_source_nodata_pixels,
    'SCORED_OUTPUT_PIXELS': fire_hazard_fbfm40_scored_pixels,
    'OUTPUT_NODATA_PIXELS': fire_hazard_fbfm40_output_nodata_pixels,
    'NONBURNABLE_SCORED_PIXELS': fire_hazard_fbfm40_nonburnable_scored_pixels,
    'BACKGROUND_NODATA_PIXELS': fire_hazard_fbfm40_background_nodata_pixels,
    'UNRECOGNIZED_PIXELS': fire_hazard_fbfm40_unrecognized_pixels,
    'UNRECOGNIZED_CODES': ', '.join((str(code) for code in \
        sorted(fire_hazard_fbfm40_unrecognized_codes_encountered))),
    'VALID_INSIDE_STUDY_AREA': fire_hazard_fbfm40_valid_inside_study_area,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_fbfm40_valid_outside_study_area,
    'OUTPUT_MINIMUM': fire_hazard_fbfm40_output_minimum,
    'OUTPUT_MAXIMUM': fire_hazard_fbfm40_output_maximum,
    'OUTPUT_MEAN': fire_hazard_fbfm40_output_mean,
    'PROCESSING_COMPLETE': fire_hazard_fbfm40_window_processing_complete,
    'ERROR_MESSAGE': fire_hazard_fbfm40_processing_error}])
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_class_count_inventory.to_csv(fire_hazard_fbfm40_class_count_inventory_path,
    index=False)
# Export the table so this workflow result is available to later phases.
# Save the window-processing summary for QA review and workflow documentation.
fire_hazard_fbfm40_window_processing_summary.to_csv(
    fire_hazard_fbfm40_window_processing_summary_path,
    index=False,
)

# Report the processing action and completion status for the FBFM40 windows.
print(f'-> Processing action: {fire_hazard_fbfm40_processing_action}')
print(
    f'-> Completed windows: '
    f'{fire_hazard_fbfm40_last_completed_window:,} '
    f'of '
    f'{fire_hazard_fbfm40_total_processing_windows:,}'
)

# Report source and output pixel counts from the completed reclassification.
print(f'-> Valid source pixels: {fire_hazard_fbfm40_source_valid_pixels:,}')
print(f'-> Scored output pixels: {fire_hazard_fbfm40_scored_pixels:,}')
print(
    f'-> Nonburnable scored pixels: '
    f'{fire_hazard_fbfm40_nonburnable_scored_pixels:,}'
)
print(
    f'-> Background NoData pixels: '
    f'{fire_hazard_fbfm40_background_nodata_pixels:,}'
)
print(f'-> Unrecognized pixels: {fire_hazard_fbfm40_unrecognized_pixels:,}')
print(
    f'-> Valid pixels outside study area: '
    f'{fire_hazard_fbfm40_valid_outside_study_area:,}'
)

# Report the range and mean of the resulting fuel-hazard scores.
print(
    f'-> Output score range: '
    f'{fire_hazard_fbfm40_output_minimum:.4f} '
    f'to '
    f'{fire_hazard_fbfm40_output_maximum:.4f}'
)
print(f'-> Output mean score: {fire_hazard_fbfm40_output_mean:.4f}')

# Confirm that window processing completed and report the saved QA files.
print(
    f'-> Window processing complete: '
    f'{fire_hazard_fbfm40_window_processing_complete}'
)
print(
    f'-> Processing summary saved: '
    f'{fire_hazard_fbfm40_window_processing_summary_path}'
)
print(
    f'-> Class-count inventory saved: '
    f'{fire_hazard_fbfm40_class_count_inventory_path}'
)

# Display the complete window-processing summary for review.
print('\n--- FBFM40 WINDOW PROCESSING SUMMARY ---')
display(fire_hazard_fbfm40_window_processing_summary)

# Display the five most common FBFM40 classes observed in the source raster.
print('\n--- OBSERVED FBFM40 CLASS COUNTS ---')
display(
    fire_hazard_fbfm40_class_count_inventory[
        fire_hazard_fbfm40_class_count_inventory['OBSERVED']
    ]
    .sort_values(
        'PIXEL_COUNT',
        ascending=False,
    )
    .head(5)
)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_reclassification_reuse_output:
    print('The existing cached FBFM40 '
        'hazard raster was inspected '
        'and its output statistics were '
        'restored.')
else:
    print('Every aligned FBFM40 '
        'processing window was '
        'validated, reclassified, '
        'masked to the three-county '
        'study area, and written to the '
        'temporary hazard raster.')
print('Recognized nonburnable classes '
    'received a hazard score of '
    'zero, while background and '
    'unsupported pixels remain '
    'NoData.')
print('The next step will validate '
    'the reclassified raster, '
    'confirm the class-to-score '
    'mapping, and promote the '
    'temporary output to its '
    'permanent location.')
print('\n=== FBFM40 RASTER-WINDOW PROCESSING COMPLETE ===')


=== PROCESSING FBFM40 RECLASSIFICATION WINDOWS ===
-> Reusing cached FBFM40 hazard raster.
-> Processing action: Reused existing FBFM40 hazard raster
-> Completed windows: 0 of 551
-> Valid source pixels: 0
-> Scored output pixels: 15,475,232
-> Nonburnable scored pixels: 0
-> Background NoData pixels: 0
-> Unrecognized pixels: 0
-> Valid pixels outside study area: 0
-> Output score range: 0.0000 to 1.0000
-> Output mean score: 0.4357
-> Window processing complete: True
-> Processing summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_window_processing_summary.csv
-> Class-count inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_class_pixel_counts.csv

--- FBFM40 WINDOW PROCESSING SUMMARY ---


,COMPONENT_ID,SOURCE_PRODUCT,PROCESSING_ACTION,REUSED_CACHED_OUTPUT,TOTAL_WINDOWS,COMPLETED_WINDOWS,SOURCE_VALID_PIXELS,SOURCE_NODATA_PIXELS,SCORED_OUTPUT_PIXELS,OUTPUT_NODATA_PIXELS,...,BACKGROUND_NODATA_PIXELS,UNRECOGNIZED_PIXELS,UNRECOGNIZED_CODES,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,PROCESSING_COMPLETE,ERROR_MESSAGE
0,FUEL_HAZARD,LANDFIRE_FBFM40,Reused existing FBFM40 hazard raster,True,551,0,0,0,15475232,121295786,...,0,0,,15475232,0,0.0,1.0,0.435666,True,None



--- OBSERVED FBFM40 CLASS COUNTS ---


,FBFM40_CODE,FBFM40_MODEL,FUEL_GROUP,BURNABLE,HAZARD_SCORE,OUTPUT_DISPOSITION,PIXEL_COUNT,OBSERVED



NOTE:
The existing cached FBFM40 hazard raster was inspected and its output statistics were restored.
Recognized nonburnable classes received a hazard score of zero, while background and unsupported pixels remain NoData.
The next step will validate the reclassified raster, confirm the class-to-score mapping, and promote the temporary output to its permanent location.

=== FBFM40 RASTER-WINDOW PROCESSING COMPLETE ===


### Updating FBFM40 Statistics and Class Counts


In [105]:
print('=== UPDATING FBFM40 STATISTICS AND CLASS COUNTS ===')
# Prepare required FBFM40 statistics inputs for the downstream processing or validation performed in
# this workflow stage.
required_fbfm40_statistics_inputs = ['fire_hazard_fbfm40_window_processing_complete',
    'fire_hazard_fbfm40_processing_input_path', 'fire_hazard_fbfm40_processing_output_path',
    'fire_hazard_fbfm40_processing_temporary_path',
    'fire_hazard_fbfm40_reclassification_reuse_output',
    'fire_hazard_fbfm40_reclassification_table', 'fire_hazard_fbfm40_score_lookup',
    'fire_hazard_fbfm40_nodata_codes', 'fire_hazard_fbfm40_recognized_codes',
    'fire_hazard_fbfm40_score_minimum', 'fire_hazard_fbfm40_score_maximum',
    'fire_hazard_fbfm40_score_nodata', 'fire_hazard_fbfm40_class_pixel_counts',
    'fire_hazard_fbfm40_scored_pixels', 'fire_hazard_fbfm40_output_nodata_pixels',
    'fire_hazard_fbfm40_valid_inside_study_area', 'fire_hazard_fbfm40_valid_outside_study_area',
    'fire_hazard_fbfm40_output_minimum', 'fire_hazard_fbfm40_output_maximum',
    'fire_hazard_fbfm40_output_mean', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_fuel_metadata_directory']
# Identify missing FBFM40 statistics inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fbfm40_statistics_inputs = [object_name for object_name in \
    required_fbfm40_statistics_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 statistics inputs remain unresolved before this workflow stage
# begins.
if missing_fbfm40_statistics_inputs:
    raise NameError(f'The following FBFM40 '
        f'statistics objects are '
        f'missing:\n'
        f'{missing_fbfm40_statistics_inputs}\n\n'
        f'Run Process FBFM40 Raster '
        f'Windows before updating '
        f'statistics and class counts.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_window_processing_complete:
    raise ValueError('FBFM40 window processing is incomplete.')
# Build fire hazard fbfm40 statistics source path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_statistics_source_path = Path(fire_hazard_fbfm40_processing_input_path)

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_reclassification_reuse_output:
    # Build fire hazard fbfm40 statistics output path used to read, cache, or save this workflow
    # product.
    fire_hazard_fbfm40_statistics_output_path = Path(fire_hazard_fbfm40_processing_output_path)
else:
    # Build fire hazard fbfm40 statistics output path used to read, cache, or save this workflow
    # product.
    fire_hazard_fbfm40_statistics_output_path = Path(fire_hazard_fbfm40_processing_temporary_path)

# Iterate through  so each required item receives the same processing and QA checks.
for raster_name, raster_path in {'FBFM40 source': fire_hazard_fbfm40_statistics_source_path,
    'FBFM40 hazard output': fire_hazard_fbfm40_statistics_output_path}.items():

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not raster_path.exists() or not raster_path.is_file() or raster_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The {raster_name} raster is unavailable:\n{raster_path}')
# Evaluate verified source valid pixels so invalid inputs or outputs can be rejected before
# continuing.
verified_source_valid_pixels = 0
# Calculate verified source NoData pixels for completeness, file-integrity, or processing QA.
verified_source_nodata_pixels = 0
# Evaluate verified output valid pixels so invalid inputs or outputs can be rejected before
# continuing.
verified_output_valid_pixels = 0
# Calculate verified output NoData pixels for completeness, file-integrity, or processing QA.
verified_output_nodata_pixels = 0
# Evaluate verified valid inside study area so invalid inputs or outputs can be rejected before
# continuing.
verified_valid_inside_study_area = 0
# Evaluate verified valid outside study area so invalid inputs or outputs can be rejected before
# continuing.
verified_valid_outside_study_area = 0
# Calculate verified background NoData pixels for completeness, file-integrity, or processing QA.
verified_background_nodata_pixels = 0
# Calculate verified nonburnable pixels for completeness, file-integrity, or processing QA.
verified_nonburnable_pixels = 0
# Calculate verified unrecognized pixels for completeness, file-integrity, or processing QA.
verified_unrecognized_pixels = 0
# Prepare verified unrecognized codes for the downstream processing or validation performed in this
# workflow stage.
verified_unrecognized_codes = set()
# Set verified output minimum as an explicit model or validation parameter used consistently in
# downstream calculations.
verified_output_minimum = None
# Set verified output maximum as an explicit model or validation parameter used consistently in
# downstream calculations.
verified_output_maximum = None
# Prepare verified output sum for the downstream processing or validation performed in this workflow
# stage.
verified_output_sum = 0.0
# Prepare verified output mean for the downstream processing or validation performed in this
# workflow stage.
verified_output_mean = None
# Build verified class pixel counts from the records that satisfy this workflow's selection
# criteria.
verified_class_pixel_counts = {int(class_code): 0 for class_code in \
    sorted(fire_hazard_fbfm40_recognized_codes)}
# Build verified nonburnable codes from the records that satisfy this workflow's selection criteria.
verified_nonburnable_codes = {int(class_code) for class_code,
    hazard_score in fire_hazard_fbfm40_score_lookup.items() if np.isclose(hazard_score,
    0.0)}

# Open the file in a managed context so required content is processed and the resource closes
# cleanly.
with rasterio.open(fire_hazard_fbfm40_statistics_source_path) as source_raster:

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(fire_hazard_fbfm40_statistics_output_path) as output_raster:
        # Store expected fbfm40 crs so spatial operations use the required coordinate reference
        # system.
        expected_fbfm40_crs = rasterio.crs.CRS.from_user_input(fire_hazard_target_crs)
        # Evaluate source output grids valid so invalid inputs or outputs can be rejected before
        # continuing.
        source_output_grids_valid = all([source_raster.count == 1,
            output_raster.count == 1, source_raster.width == output_raster.width == \
                fire_hazard_alignment_width,
            source_raster.height == output_raster.height == fire_hazard_alignment_height,
            source_raster.crs is not None, output_raster.crs is not None,
            source_raster.crs == output_raster.crs == expected_fbfm40_crs,
            source_raster.transform.almost_equals(fire_hazard_alignment_transform),
            output_raster.transform.almost_equals(fire_hazard_alignment_transform),
            output_raster.dtypes[0] == 'float32', output_raster.nodata == \
                fire_hazard_fbfm40_score_nodata])

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if not source_output_grids_valid:
            raise ValueError('The FBFM40 source and '
                'hazard-output rasters do not '
                'match the common project grid.')

        # Iterate through output raster.block windows so each required item receives the same
        # processing and QA checks.
        for _, raster_window in output_raster.block_windows(1):
            # Prepare source array for the raster calculation or validation performed in this
            # processing block.
            source_array = source_raster.read(1, window=raster_window)
            # Prepare output array for the raster calculation or validation performed in this
            # processing block.
            output_array = output_raster.read(1, window=raster_window)
            # Prepare row start for the downstream processing or validation performed in this
            # workflow stage.
            row_start = int(raster_window.row_off)
            # Prepare row end for the downstream processing or validation performed in this workflow
            # stage.
            row_end = int(raster_window.row_off + raster_window.height)
            # Prepare column start for the downstream processing or validation performed in this
            # workflow stage.
            column_start = int(raster_window.col_off)
            # Prepare column end for the downstream processing or validation performed in this
            # workflow stage.
            column_end = int(raster_window.col_off + raster_window.width)
            # Build the study area window mask mask used to isolate records required for this
            # analysis.
            study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                column_start:column_end]
            # Build the source finite mask mask used to isolate records required for this analysis.
            source_finite_mask = np.isfinite(source_array)

            # Stop execution if the NoData configuration is incompatible with the raster product
            # being validated.
            if source_raster.nodata is None:
                # Build the source data mask mask used to isolate records required for this
                # analysis.
                source_data_mask = source_finite_mask
            else:
                # Build the source data mask mask used to isolate records required for this
                # analysis.
                source_data_mask = source_finite_mask & (source_array != source_raster.nodata)
            # Build the source valid mask mask used to isolate records required for this analysis.
            source_valid_mask = source_data_mask & study_area_window_mask
            # Evaluate verified source valid pixels so invalid inputs or outputs can be rejected
            # before continuing.
            verified_source_valid_pixels += int(source_valid_mask.sum())
            # Calculate verified source NoData pixels for completeness, file-integrity, or
            # processing QA.
            verified_source_nodata_pixels += int((~source_data_mask).sum())

            # Stop execution if the prerequisite validation has not passed before this workflow
            # stage continues.
            if source_valid_mask.any():
                # Prepare source codes for the downstream processing or validation performed in this
                # workflow stage.
                source_codes = np.rint(source_array[source_valid_mask]).astype(np.int32)
                # Prepare unique codes and unique counts so this workflow stage has the values
                # required for downstream spatial processing and QA.
                unique_codes, unique_counts = np.unique(source_codes, return_counts=True)

                # Iterate through zip so each required item receives the same processing and QA
                # checks.
                for class_code, class_count in zip(unique_codes, unique_counts):
                    # Prepare class code for the downstream processing or validation performed in
                    # this workflow stage.
                    class_code = int(class_code)
                    # Calculate class count for completeness, file-integrity, or processing QA.
                    class_count = int(class_count)

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if class_code in verified_class_pixel_counts:
                        # Prepare verified class pixel counts so this workflow stage has the values
                        # required for downstream spatial processing and QA.
                        verified_class_pixel_counts[class_code] += class_count
                    else:
                        # Record the observed value so final QA can compare recognized and
                        # unexpected classes.
                        verified_unrecognized_codes.add(class_code)
                        # Calculate verified unrecognized pixels for completeness, file-integrity,
                        # or processing QA.
                        verified_unrecognized_pixels += class_count
                # Calculate verified background NoData pixels for completeness, file-integrity, or
                # processing QA.
                verified_background_nodata_pixels += int(np.isin(source_codes,
                    list(fire_hazard_fbfm40_nodata_codes)).sum())
                # Calculate verified nonburnable pixels for completeness, file-integrity, or
                # processing QA.
                verified_nonburnable_pixels += int(np.isin(source_codes,
                    list(verified_nonburnable_codes)).sum())
            # Build the output valid mask mask used to isolate records required for this analysis.
            output_valid_mask = np.isfinite(output_array) & (output_array != \
                fire_hazard_fbfm40_score_nodata)
            # Evaluate output valid values so invalid inputs or outputs can be rejected before
            # continuing.
            output_valid_values = output_array[output_valid_mask]
            # Evaluate verified output valid pixels so invalid inputs or outputs can be rejected
            # before continuing.
            verified_output_valid_pixels += int(output_valid_mask.sum())
            # Calculate verified output NoData pixels for completeness, file-integrity, or
            # processing QA.
            verified_output_nodata_pixels += int((~output_valid_mask).sum())
            # Evaluate verified valid inside study area so invalid inputs or outputs can be rejected
            # before continuing.
            verified_valid_inside_study_area += int((output_valid_mask & \
                study_area_window_mask).sum())
            # Evaluate verified valid outside study area so invalid inputs or outputs can be
            # rejected before continuing.
            verified_valid_outside_study_area += int((output_valid_mask & \
                ~study_area_window_mask).sum())

            # Stop execution if the prerequisite validation has not passed before this workflow
            # stage continues.
            if output_valid_values.size > 0:
                # Set block minimum as an explicit model or validation parameter used consistently
                # in downstream calculations.
                block_minimum = float(output_valid_values.min())
                # Set block maximum as an explicit model or validation parameter used consistently
                # in downstream calculations.
                block_maximum = float(output_valid_values.max())

                # Stop execution if this validation condition is not satisfied before dependent
                # processing continues.
                if verified_output_minimum is None:
                    # Set verified output minimum as an explicit model or validation parameter used
                    # consistently in downstream calculations.
                    verified_output_minimum = block_minimum
                else:
                    # Set verified output minimum as an explicit model or validation parameter used
                    # consistently in downstream calculations.
                    verified_output_minimum = min(verified_output_minimum, block_minimum)

                # Stop execution if this validation condition is not satisfied before dependent
                # processing continues.
                if verified_output_maximum is None:
                    # Set verified output maximum as an explicit model or validation parameter used
                    # consistently in downstream calculations.
                    verified_output_maximum = block_maximum
                else:
                    # Set verified output maximum as an explicit model or validation parameter used
                    # consistently in downstream calculations.
                    verified_output_maximum = max(verified_output_maximum, block_maximum)
                # Prepare verified output sum for the downstream processing or validation performed
                # in this workflow stage.
                verified_output_sum += float(output_valid_values.sum(dtype=np.float64))
            del source_array
            del output_array

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if verified_output_valid_pixels > 0:
    # Prepare verified output mean for the downstream processing or validation performed in this
    # workflow stage.
    verified_output_mean = verified_output_sum / verified_output_valid_pixels
# Evaluate verified score range valid so invalid inputs or outputs can be rejected before
# continuing.
verified_score_range_valid = all([verified_output_minimum is not None,
    verified_output_maximum is not None, verified_output_minimum >= \
        fire_hazard_fbfm40_score_minimum,
    verified_output_maximum <= fire_hazard_fbfm40_score_maximum])
# Record whether verified pixel statistics match satisfies the checks required before the workflow
# advances.
verified_pixel_statistics_match = all([verified_output_valid_pixels == \
    fire_hazard_fbfm40_scored_pixels,
    verified_output_nodata_pixels == fire_hazard_fbfm40_output_nodata_pixels,
    verified_valid_inside_study_area == fire_hazard_fbfm40_valid_inside_study_area,
    verified_valid_outside_study_area == fire_hazard_fbfm40_valid_outside_study_area])
# Record whether verified value statistics match satisfies the checks required before the workflow
# advances.
verified_value_statistics_match = all([verified_output_minimum is not None,
    fire_hazard_fbfm40_output_minimum is not None,
    verified_output_maximum is not None, fire_hazard_fbfm40_output_maximum is not None,
    verified_output_mean is not None, fire_hazard_fbfm40_output_mean is not None,
    np.isclose(verified_output_minimum, fire_hazard_fbfm40_output_minimum,
    atol=1e-06), np.isclose(verified_output_maximum,
    fire_hazard_fbfm40_output_maximum, atol=1e-06),
    np.isclose(verified_output_mean, fire_hazard_fbfm40_output_mean,
    atol=1e-06)])
# Build q2b2 class pixel counts from the records that satisfy this workflow's selection criteria.
q2b2_class_pixel_counts = {int(class_code): \
    int(fire_hazard_fbfm40_class_pixel_counts.get(int(class_code),
    0)) for class_code in sorted(fire_hazard_fbfm40_recognized_codes)}
# Calculate q2b2 class count total for completeness, file-integrity, or processing QA.
q2b2_class_count_total = int(sum(q2b2_class_pixel_counts.values()))
# Calculate verified class count total for completeness, file-integrity, or processing QA.
verified_class_count_total = int(sum(verified_class_pixel_counts.values()))
# Build class count difference records used to track the records included in this processing stage.
class_count_difference_records = []

# Iterate through sorted so each required item receives the same processing and QA checks.
for class_code in sorted(verified_class_pixel_counts):
    # Calculate q2b2 count for completeness, file-integrity, or processing QA.
    q2b2_count = int(q2b2_class_pixel_counts.get(class_code, 0))
    # Calculate verified count for completeness, file-integrity, or processing QA.
    verified_count = int(verified_class_pixel_counts.get(class_code, 0))

    # Stop execution if this validation condition is not satisfied before dependent processing
    # continues.
    if q2b2_count != verified_count:
        # Add the current record to class count difference records so the stage summary captures
        # this processing result.
        class_count_difference_records.append({'FBFM40_CODE': class_code,
            'Q2B2_COUNT': q2b2_count, 'VERIFIED_COUNT': verified_count,
            'DIFFERENCE': verified_count - q2b2_count})
# Calculate class count differences for completeness, file-integrity, or processing QA.
class_count_differences = pd.DataFrame(class_count_difference_records,
    columns=['FBFM40_CODE', 'Q2B2_COUNT', 'VERIFIED_COUNT',
    'DIFFERENCE'])
# Calculate fire hazard FBFM40 class counts restored for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_class_counts_restored = False
# Calculate fire hazard FBFM40 class count restoration reason for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_class_count_restoration_reason = None
# Calculate q2b2 class counts uninitialized for completeness, file-integrity, or processing QA.
q2b2_class_counts_uninitialized = all([q2b2_class_count_total == 0, verified_class_count_total > 0])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if q2b2_class_counts_uninitialized:
    # Calculate fire hazard FBFM40 class pixel counts for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_class_pixel_counts = verified_class_pixel_counts.copy()
    # Calculate fire hazard FBFM40 class counts restored for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_class_counts_restored = True
    # Calculate fire hazard FBFM40 class count restoration reason for completeness, file-integrity,
    # or processing QA.
    fire_hazard_fbfm40_class_count_restoration_reason = (
        'Q2B2 class counts were '
            'uninitialized during '
            'cached-output reuse; counts '
            'were restored by independent '
            'source-raster verification.'
    )
    # Record whether verified class counts match satisfies the checks required before the workflow
    # advances.
    verified_class_counts_match = True
else:
    # Record whether verified class counts match satisfies the checks required before the workflow
    # advances.
    verified_class_counts_match = all((verified_class_pixel_counts[class_code] == \
        q2b2_class_pixel_counts[class_code] for class_code in verified_class_pixel_counts))
# Calculate fire hazard FBFM40 class count status for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_class_count_status = 'Restored from independent verification' if \
    fire_hazard_fbfm40_class_counts_restored else 'Matched Q2B2 running counts' if \
    verified_class_counts_match else 'Mismatch requires investigation'
# Prepare fire hazard FBFM40 statistics verified for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_statistics_verified = all([source_output_grids_valid,
    verified_output_valid_pixels > 0, verified_valid_inside_study_area > 0,
    verified_valid_outside_study_area == 0, verified_unrecognized_pixels == 0,
    verified_score_range_valid, verified_pixel_statistics_match,
    verified_value_statistics_match, verified_class_counts_match])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_statistics_verified:
    # Evaluate fire hazard fbfm40 source valid pixels so invalid inputs or outputs can be rejected
    # before continuing.
    fire_hazard_fbfm40_source_valid_pixels = verified_source_valid_pixels
    # Calculate fire hazard FBFM40 source NoData pixels for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_source_nodata_pixels = verified_source_nodata_pixels
    # Calculate fire hazard FBFM40 scored pixels for completeness, file-integrity, or processing QA.
    fire_hazard_fbfm40_scored_pixels = verified_output_valid_pixels
    # Calculate fire hazard FBFM40 output NoData pixels for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_output_nodata_pixels = verified_output_nodata_pixels
    # Calculate fire hazard FBFM40 nonburnable scored pixels for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_nonburnable_scored_pixels = verified_nonburnable_pixels
    # Calculate fire hazard FBFM40 background NoData pixels for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_background_nodata_pixels = verified_background_nodata_pixels
    # Calculate fire hazard FBFM40 unrecognized pixels for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_unrecognized_pixels = verified_unrecognized_pixels
    # Evaluate fire hazard fbfm40 valid inside study area so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_valid_inside_study_area = verified_valid_inside_study_area
    # Evaluate fire hazard fbfm40 valid outside study area so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_valid_outside_study_area = verified_valid_outside_study_area
    # Set fire hazard FBFM40 output minimum as an explicit model or validation parameter used
    # consistently in downstream calculations.
    fire_hazard_fbfm40_output_minimum = verified_output_minimum
    # Set fire hazard FBFM40 output maximum as an explicit model or validation parameter used
    # consistently in downstream calculations.
    fire_hazard_fbfm40_output_maximum = verified_output_maximum
    # Prepare fire hazard FBFM40 output mean for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_output_mean = verified_output_mean
    # Calculate fire hazard FBFM40 class pixel counts for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fbfm40_class_pixel_counts = verified_class_pixel_counts.copy()
# Calculate fire hazard FBFM40 verified class counts for completeness, file-integrity, or processing
# QA.
fire_hazard_fbfm40_verified_class_counts = fire_hazard_fbfm40_reclassification_table[['FBFM40_CODE',
    'FBFM40_MODEL', 'FUEL_GROUP', 'BURNABLE', 'HAZARD_SCORE',
    'OUTPUT_DISPOSITION']].copy()
# Prepare fire hazard FBFM40 verified class counts so this workflow stage has the values required
# for downstream spatial processing and QA.
fire_hazard_fbfm40_verified_class_counts['PIXEL_COUNT'] = \
    fire_hazard_fbfm40_verified_class_counts['FBFM40_CODE'].map(verified_class_pixel_counts) \
    .fillna(0).astype(np.int64)
# Prepare fire hazard FBFM40 verified class counts so this workflow stage has the values required
# for downstream spatial processing and QA.
fire_hazard_fbfm40_verified_class_counts['OBSERVED'] = \
    fire_hazard_fbfm40_verified_class_counts['PIXEL_COUNT'] > 0
# Assemble fire hazard FBFM40 statistics summary into a table for QA review and downstream
# validation.
fire_hazard_fbfm40_statistics_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_PRODUCT': 'LANDFIRE_FBFM40', 'RASTER_INSPECTED': \
        str(fire_hazard_fbfm40_statistics_output_path),
    'REUSED_CACHED_OUTPUT': fire_hazard_fbfm40_reclassification_reuse_output,
    'SOURCE_VALID_PIXELS': verified_source_valid_pixels,
    'SOURCE_NODATA_PIXELS': verified_source_nodata_pixels,
    'OUTPUT_VALID_PIXELS': verified_output_valid_pixels,
    'OUTPUT_NODATA_PIXELS': verified_output_nodata_pixels,
    'VALID_INSIDE_STUDY_AREA': verified_valid_inside_study_area,
    'VALID_OUTSIDE_STUDY_AREA': verified_valid_outside_study_area,
    'BACKGROUND_NODATA_PIXELS': verified_background_nodata_pixels,
    'NONBURNABLE_SCORED_PIXELS': verified_nonburnable_pixels,
    'UNRECOGNIZED_PIXELS': verified_unrecognized_pixels,
    'UNRECOGNIZED_CODES': ', '.join((str(code) for code in sorted(verified_unrecognized_codes))),
    'OUTPUT_MINIMUM': verified_output_minimum, 'OUTPUT_MAXIMUM': verified_output_maximum,
    'OUTPUT_MEAN': verified_output_mean, 'SCORE_RANGE_VALID': verified_score_range_valid,
    'PIXEL_STATISTICS_MATCH': verified_pixel_statistics_match,
    'VALUE_STATISTICS_MATCH': verified_value_statistics_match,
    'Q2B2_CLASS_COUNT_TOTAL': q2b2_class_count_total,
    'VERIFIED_CLASS_COUNT_TOTAL': verified_class_count_total,
    'CLASS_COUNTS_UNINITIALIZED': q2b2_class_counts_uninitialized,
    'CLASS_COUNTS_RESTORED': fire_hazard_fbfm40_class_counts_restored,
    'CLASS_COUNT_STATUS': fire_hazard_fbfm40_class_count_status,
    'CLASS_COUNT_RESTORATION_REASON': fire_hazard_fbfm40_class_count_restoration_reason,
    'CLASS_COUNTS_MATCH': verified_class_counts_match,
    'STATISTICS_VERIFIED': fire_hazard_fbfm40_statistics_verified}])
# Calculate fire hazard FBFM40 verified class counts path for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_verified_class_counts_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_verified_class_pixel_counts.csv'
# Build fire hazard fbfm40 statistics summary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_statistics_summary_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_statistics_verification_summary.csv'
# Calculate fire hazard FBFM40 class count differences path for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_class_count_differences_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_class_count_differences.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_verified_class_counts.to_csv(fire_hazard_fbfm40_verified_class_counts_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_statistics_summary.to_csv(fire_hazard_fbfm40_statistics_summary_path,
    index=False)
# Export the table so this workflow result is available to later phases.
class_count_differences.to_csv(fire_hazard_fbfm40_class_count_differences_path, index=False)

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_statistics_verified:
    raise ValueError(f'The independently recalculated '
        f'FBFM40 statistics did not pass '
        f'verification.\n\nPixel '
        f'statistics match: '
        f'{verified_pixel_statistics_match}\n'
        f'Value statistics match: '
        f'{verified_value_statistics_match}\n'
        f'Q2B2 class-count total: '
        f'{q2b2_class_count_total:,}\n'
        f'Verified class-count total: '
        f'{verified_class_count_total:,}\n'
        f'Class counts restored: '
        f'{fire_hazard_fbfm40_class_counts_restored}\n'
        f'Class counts match: '
        f'{verified_class_counts_match}\n'
        f'Score range valid: '
        f'{verified_score_range_valid}\n'
        f'Valid outside study area: '
        f'{verified_valid_outside_study_area:,}\n'
        f'Unrecognized pixels: '
        f'{verified_unrecognized_pixels:,}\n\n'
        f'Review the saved class-count '
        f'differences:\n'
        f'{fire_hazard_fbfm40_class_count_differences_path}')
# Report the output raster and pixel totals used for the final verification.
print(f'-> Output raster inspected: {fire_hazard_fbfm40_statistics_output_path}')
print(f'-> Verified source pixels: {verified_source_valid_pixels:,}')
print(f'-> Verified output pixels: {verified_output_valid_pixels:,}')

# Report pixels excluded by the study-area mask or unrecognized by the reclassification policy.
print(
    f'-> Valid pixels outside study area: '
    f'{verified_valid_outside_study_area:,}'
)
print(
    f'-> Unrecognized source pixels: '
    f'{verified_unrecognized_pixels:,}'
)

# Report the verified range and mean of the FBFM40 hazard scores.
print(
    f'-> Verified score range: '
    f'{verified_output_minimum:.4f} '
    f'to '
    f'{verified_output_maximum:.4f}'
)
print(f'-> Verified mean score: {verified_output_mean:.4f}')

# Confirm that the verified pixel and value statistics match the processing results.
print(f'-> Pixel statistics match: {verified_pixel_statistics_match}')
print(f'-> Value statistics match: {verified_value_statistics_match}')

# Compare the original and verified FBFM40 class-count totals.
print(f'-> Q2B2 class-count total: {q2b2_class_count_total:,}')
print(f'-> Verified class-count total: {verified_class_count_total:,}')

# Report whether the FBFM40 class counts were restored and validated successfully.
print(
    f'-> Class counts restored: '
    f'{fire_hazard_fbfm40_class_counts_restored}'
)
print(f'-> Class-count status: {fire_hazard_fbfm40_class_count_status}')
print(f'-> Class counts valid: {verified_class_counts_match}')

# Confirm that the final FBFM40 output statistics passed verification.
print(
    f'-> Statistics verified: '
    f'{fire_hazard_fbfm40_statistics_verified}'
)
print('\n--- FBFM40 STATISTICS VERIFICATION SUMMARY ---')
display(fire_hazard_fbfm40_statistics_summary)

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_class_counts_restored:
    print('\n--- RESTORED FBFM40 CLASS COUNTS ---')
    print('Q2B2 contained no accumulated '
        'class counts. The '
        'independently verified '
        'source-raster counts are now '
        'the authoritative inventory.')
print('\n--- MOST COMMON OBSERVED FBFM40 CLASSES ---')
display(fire_hazard_fbfm40_verified_class_counts[fire_hazard_fbfm40_verified_class_counts \
    ['OBSERVED']].sort_values('PIXEL_COUNT',
    ascending=False).head(5))
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_class_counts_restored:
    print('The Q2B2 class-count '
        'dictionary was empty because '
        'cached-output reuse restored '
        'raster statistics without '
        'rebuilding source-class '
        'counts.')
    print('The class counts were '
        'independently recalculated '
        'from the aligned FBFM40 source '
        'raster and restored as the '
        'authoritative inventory.')
else:
    print('The independently recalculated class counts match the Q2B2 running class inventory.')
print('The output pixel counts, score '
    'statistics, grid, study-area '
    'mask, recognized classes, and '
    'score range all passed '
    'verification.')
print('The next step can finalize the FBFM40 window-processing session.')
print('\n=== FBFM40 STATISTICS AND CLASS COUNTS VERIFIED ===')


=== UPDATING FBFM40 STATISTICS AND CLASS COUNTS ===
-> Output raster inspected: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\normalized_sources\fbfm40_fuel_hazard_score.tif
-> Verified source pixels: 15,475,232
-> Verified output pixels: 15,475,232
-> Valid pixels outside study area: 0
-> Unrecognized source pixels: 0
-> Verified score range: 0.0000 to 1.0000
-> Verified mean score: 0.4357
-> Pixel statistics match: True
-> Value statistics match: True
-> Q2B2 class-count total: 0
-> Verified class-count total: 15,475,232
-> Class counts restored: True
-> Class-count status: Restored from independent verification
-> Class counts valid: True
-> Statistics verified: True

--- FBFM40 STATISTICS VERIFICATION SUMMARY ---


,COMPONENT_ID,SOURCE_PRODUCT,RASTER_INSPECTED,REUSED_CACHED_OUTPUT,SOURCE_VALID_PIXELS,SOURCE_NODATA_PIXELS,OUTPUT_VALID_PIXELS,OUTPUT_NODATA_PIXELS,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,...,PIXEL_STATISTICS_MATCH,VALUE_STATISTICS_MATCH,Q2B2_CLASS_COUNT_TOTAL,VERIFIED_CLASS_COUNT_TOTAL,CLASS_COUNTS_UNINITIALIZED,CLASS_COUNTS_RESTORED,CLASS_COUNT_STATUS,CLASS_COUNT_RESTORATION_REASON,CLASS_COUNTS_MATCH,STATISTICS_VERIFIED
0,FUEL_HAZARD,LANDFIRE_FBFM40,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,15475232,121295786,15475232,121295786,15475232,0,...,True,True,0,15475232,True,True,Restored from independent verification,Q2B2 class counts were uninitialized during ca...,True,True



--- RESTORED FBFM40 CLASS COUNTS ---
Q2B2 contained no accumulated class counts. The independently verified source-raster counts are now the authoritative inventory.

--- MOST COMMON OBSERVED FBFM40 CLASSES ---


,FBFM40_CODE,FBFM40_MODEL,FUEL_GROUP,BURNABLE,HAZARD_SCORE,OUTPUT_DISPOSITION,PIXEL_COUNT,OBSERVED
16,122,GS2,Grass-Shrub,True,0.50,Score,3496338,True
23,145,SH5,Shrub,True,0.75,Score,2033787,True
7,102,GR2,Grass,True,0.35,Score,1815794,True
1,91,NB1,Nonburnable,False,0.00,Score,1393298,True
15,121,GS1,Grass-Shrub,True,0.35,Score,1354419,True



NOTE:
The Q2B2 class-count dictionary was empty because cached-output reuse restored raster statistics without rebuilding source-class counts.
The class counts were independently recalculated from the aligned FBFM40 source raster and restored as the authoritative inventory.
The output pixel counts, score statistics, grid, study-area mask, recognized classes, and score range all passed verification.
The next step can finalize the FBFM40 window-processing session.

=== FBFM40 STATISTICS AND CLASS COUNTS VERIFIED ===


### Finalizing FBFM40 Window Processing


In [106]:
print('=== FINALIZING FBFM40 WINDOW PROCESSING ===')
# Prepare required FBFM40 finalization inputs for the downstream processing or validation performed
# in this workflow stage.
# Define the variables that must exist before the FBFM40 workflow can be finalized.
required_fbfm40_finalization_inputs = [

    # Require successful completion and verification of the FBFM40 processing workflow.
    'fire_hazard_fbfm40_window_processing_complete',
    'fire_hazard_fbfm40_statistics_verified',
    'fire_hazard_fbfm40_processing_action',

    # Require the source, output, and temporary raster paths used during processing.
    'fire_hazard_fbfm40_processing_input_path',
    'fire_hazard_fbfm40_processing_output_path',
    'fire_hazard_fbfm40_processing_temporary_path',

    # Require the reclassification settings and manifest used to document the workflow.
    'fire_hazard_fbfm40_reclassification_reuse_output',
    'fire_hazard_fbfm40_reclassification_method',
    'fire_hazard_fbfm40_reclassification_manifest_path',

    # Require the configured hazard-score range and NoData value.
    'fire_hazard_fbfm40_score_minimum',
    'fire_hazard_fbfm40_score_maximum',
    'fire_hazard_fbfm40_score_nodata',

    # Require source and output pixel counts for final QA.
    'fire_hazard_fbfm40_source_valid_pixels',
    'fire_hazard_fbfm40_source_nodata_pixels',
    'fire_hazard_fbfm40_scored_pixels',
    'fire_hazard_fbfm40_output_nodata_pixels',

    # Require detailed pixel counts for nonburnable, background, and unrecognized classes.
    'fire_hazard_fbfm40_nonburnable_scored_pixels',
    'fire_hazard_fbfm40_background_nodata_pixels',
    'fire_hazard_fbfm40_unrecognized_pixels',

    # Require study-area pixel counts to verify the spatial mask.
    'fire_hazard_fbfm40_valid_inside_study_area',
    'fire_hazard_fbfm40_valid_outside_study_area',

    # Require final output statistics for the reclassified fuel-hazard raster.
    'fire_hazard_fbfm40_output_minimum',
    'fire_hazard_fbfm40_output_maximum',
    'fire_hazard_fbfm40_output_mean',

    # Require window-processing totals to confirm that all raster windows were completed.
    'fire_hazard_fbfm40_total_processing_windows',
    'fire_hazard_fbfm40_last_completed_window',

    # Require verified class counts for final reclassification QA.
    'fire_hazard_fbfm40_verified_class_counts',

    # Require the metadata directory used to save final workflow documentation.
    'fire_hazard_fuel_metadata_directory',

    # Require the target-grid properties used to verify the finalized raster.
    'fire_hazard_alignment_width',
    'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform',
    'fire_hazard_target_crs',
    'fire_hazard_cell_size',
]
# Identify missing FBFM40 finalization inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fbfm40_finalization_inputs = [object_name for object_name in \
    required_fbfm40_finalization_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 finalization inputs remain unresolved before this workflow stage
# begins.
if missing_fbfm40_finalization_inputs:
    raise NameError(f'The following FBFM40 '
        f'finalization objects are '
        f'missing:\n'
        f'{missing_fbfm40_finalization_inputs}\n\n'
        f'Run Process FBFM40 Raster '
        f'Windows and Update Statistics '
        f'and Class Counts before '
        f'finalizing window processing.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_window_processing_complete:
    raise ValueError('FBFM40 window processing has not completed.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_statistics_verified:
    raise ValueError('FBFM40 processing statistics have not passed independent verification.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_last_completed_window != fire_hazard_fbfm40_total_processing_windows and \
    (not fire_hazard_fbfm40_reclassification_reuse_output):
    raise ValueError('The number of completed FBFM40 '
        'windows does not match the '
        'expected processing total.')
# Build fire hazard fbfm40 processing input path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_processing_input_path = Path(fire_hazard_fbfm40_processing_input_path)
# Build fire hazard fbfm40 processing output path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_output_path = Path(fire_hazard_fbfm40_processing_output_path)
# Build fire hazard fbfm40 processing temporary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_processing_temporary_path = Path(fire_hazard_fbfm40_processing_temporary_path)

# Define reusable validate finalized fbfm40 structure logic for this phase of the workflow.
def validate_finalized_fbfm40_structure(raster_path):

    """
    Confirm that an FBFM40 hazard raster matches the
    common fire-hazard grid and output structure.
    """
    # Build raster path used to read, cache, or save this workflow product.
    raster_path = Path(raster_path)

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not raster_path.exists() or not raster_path.is_file() or raster_path.stat().st_size <= 0:
        return False

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(raster_path) as raster_source:
            return all([raster_source.count == 1,
                raster_source.width == fire_hazard_alignment_width,
                raster_source.height == fire_hazard_alignment_height,
                raster_source.crs is not None, raster_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                raster_source.transform.almost_equals(fire_hazard_alignment_transform),
                raster_source.dtypes[0] == 'float32', raster_source.nodata == \
                    fire_hazard_fbfm40_score_nodata])
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception:
        return False
# Evaluate fire hazard fbfm40 processing content valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_processing_content_valid = all([fire_hazard_fbfm40_scored_pixels > 0,
    fire_hazard_fbfm40_valid_inside_study_area > 0,
    fire_hazard_fbfm40_valid_outside_study_area == 0,
    fire_hazard_fbfm40_unrecognized_pixels == 0, fire_hazard_fbfm40_output_minimum is not None,
    fire_hazard_fbfm40_output_maximum is not None,
    fire_hazard_fbfm40_output_mean is not None, fire_hazard_fbfm40_output_minimum >= \
        fire_hazard_fbfm40_score_minimum,
    fire_hazard_fbfm40_output_maximum <= fire_hazard_fbfm40_score_maximum])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_processing_content_valid:
    raise ValueError(f'The processed FBFM40 hazard '
        f'raster failed content checks.\n\n'
        f'Scored pixels: '
        f'{fire_hazard_fbfm40_scored_pixels:,}\n'
        f'Valid outside study area: '
        f'{fire_hazard_fbfm40_valid_outside_study_area:,}\n'
        f'Unrecognized pixels: '
        f'{fire_hazard_fbfm40_unrecognized_pixels:,}\n'
        f'Minimum score: '
        f'{fire_hazard_fbfm40_output_minimum}\n'
        f'Maximum score: '
        f'{fire_hazard_fbfm40_output_maximum}')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fbfm40_reclassification_reuse_output:
    # Prepare fire hazard FBFM40 finalization action for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fbfm40_finalization_action = 'Retained validated cached FBFM40 hazard raster'
    # Evaluate fire hazard fbfm40 pre promotion structure valid so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_pre_promotion_structure_valid = \
        validate_finalized_fbfm40_structure(fire_hazard_fbfm40_processing_output_path)

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if not fire_hazard_fbfm40_pre_promotion_structure_valid:
        raise ValueError('The cached permanent FBFM40 hazard raster failed structural validation.')

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if fire_hazard_fbfm40_processing_temporary_path.exists():
        # Remove the temporary or replaceable raster so the next write starts from a clean output
        # path.
        fire_hazard_fbfm40_processing_temporary_path.unlink()
else:
    # Prepare fire hazard FBFM40 finalization action for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fbfm40_finalization_action = 'Promoted temporary FBFM40 hazard raster'

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not fire_hazard_fbfm40_processing_temporary_path.exists() or not \
        fire_hazard_fbfm40_processing_temporary_path.is_file() or \
        fire_hazard_fbfm40_processing_temporary_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The processed temporary FBFM40 '
            f'hazard raster is unavailable:\n'
            f'{fire_hazard_fbfm40_processing_temporary_path}')
    # Evaluate fire hazard fbfm40 pre promotion structure valid so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_pre_promotion_structure_valid = \
        validate_finalized_fbfm40_structure(fire_hazard_fbfm40_processing_temporary_path)

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if not fire_hazard_fbfm40_pre_promotion_structure_valid:
        raise ValueError('The temporary FBFM40 hazard '
            'raster failed structural '
            'validation before promotion.')

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(fire_hazard_fbfm40_processing_temporary_path, 'r+') as temporary_raster:
        # Assign a descriptive band label so the exported raster documents the meaning of its
        # values.
        temporary_raster.set_band_description(1, 'FBFM40 relative surface-fuel hazard score')
        # Write processing metadata to the raster so the final product retains its model and
        # provenance context.
# Add processing and source metadata to the finalized fuel-hazard raster.
        temporary_raster.update_tags(
            COMPONENT_ID='FUEL_HAZARD',
            SOURCE_PRODUCT='LANDFIRE_FBFM40',
            PROCESSING_STAGE='Window processing finalized',
            PROCESSING_ACTION=fire_hazard_fbfm40_processing_action,
            RECLASSIFICATION_METHOD=fire_hazard_fbfm40_reclassification_method,

            # Record the configured and observed fuel-hazard score ranges.
            SCORE_MINIMUM=fire_hazard_fbfm40_score_minimum,
            SCORE_MAXIMUM=fire_hazard_fbfm40_score_maximum,
            ACTUAL_SCORE_MINIMUM=fire_hazard_fbfm40_output_minimum,
            ACTUAL_SCORE_MAXIMUM=fire_hazard_fbfm40_output_maximum,
            ACTUAL_SCORE_MEAN=fire_hazard_fbfm40_output_mean,

            # Record pixel counts used to document and validate the reclassification output.
            SCORED_PIXELS=fire_hazard_fbfm40_scored_pixels,
            OUTPUT_NODATA_PIXELS=fire_hazard_fbfm40_output_nodata_pixels,
            NONBURNABLE_SCORED_PIXELS=fire_hazard_fbfm40_nonburnable_scored_pixels,
            BACKGROUND_NODATA_PIXELS=fire_hazard_fbfm40_background_nodata_pixels,
            UNRECOGNIZED_PIXELS=fire_hazard_fbfm40_unrecognized_pixels,
            VALID_OUTSIDE_STUDY_AREA=fire_hazard_fbfm40_valid_outside_study_area,

            # Record processing completion and target-grid properties.
            COMPLETED_WINDOWS=fire_hazard_fbfm40_last_completed_window,
            TARGET_CRS=fire_hazard_target_crs,
            TARGET_CELL_SIZE_METERS=fire_hazard_cell_size,
        )

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if fire_hazard_fbfm40_processing_output_path.exists():
        # Remove the temporary or replaceable raster so the next write starts from a clean output
        # path.
        fire_hazard_fbfm40_processing_output_path.unlink()
    # Promote the validated temporary raster to the final output path only after processing
    # succeeds.
    fire_hazard_fbfm40_processing_temporary_path.replace(fire_hazard_fbfm40_processing_output_path)
# Evaluate fire hazard fbfm40 permanent structure valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_permanent_structure_valid = \
    validate_finalized_fbfm40_structure(fire_hazard_fbfm40_processing_output_path)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_permanent_structure_valid:
    raise ValueError('The permanent FBFM40 hazard '
        'raster failed '
        'post-finalization structural '
        'validation.')

# Open the file in a managed context so required content is processed and the resource closes
# cleanly.
with rasterio.open(fire_hazard_fbfm40_processing_output_path) as finalized_raster:
    # Store fire hazard fbfm40 final crs so spatial operations use the required coordinate reference
    # system.
    fire_hazard_fbfm40_final_crs = finalized_raster.crs.to_string() if finalized_raster.crs is \
        not None else None
    # Prepare fire hazard FBFM40 final data type for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_final_dtype = finalized_raster.dtypes[0]
    # Set fire hazard FBFM40 final NoData as an explicit model or validation parameter used
    # consistently in downstream calculations.
    fire_hazard_fbfm40_final_nodata = finalized_raster.nodata
    # Calculate fire hazard FBFM40 final width so raster processing covers the analysis grid in
    # controlled blocks.
    fire_hazard_fbfm40_final_width = finalized_raster.width
    # Calculate fire hazard FBFM40 final height so raster processing covers the analysis grid in
    # controlled blocks.
    fire_hazard_fbfm40_final_height = finalized_raster.height
    # Prepare fire hazard FBFM40 final transform for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_final_transform = finalized_raster.transform
    # Prepare fire hazard FBFM40 final tags for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fbfm40_final_tags = finalized_raster.tags()
# Calculate fire hazard FBFM40 final output size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_final_output_size_bytes = \
    int(fire_hazard_fbfm40_processing_output_path.stat().st_size)
# Prepare fire hazard FBFM40 final output size mb for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_final_output_size_mb = fire_hazard_fbfm40_final_output_size_bytes / 1024 ** 2
# Assemble fire hazard FBFM40 processing manifest into a table for QA review and downstream
# validation.
fire_hazard_fbfm40_processing_manifest = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_PRODUCT': 'LANDFIRE_FBFM40', 'PROCESSING_STAGE': 'Window processing finalized',
    'INPUT_PATH': str(fire_hazard_fbfm40_processing_input_path),
    'OUTPUT_PATH': str(fire_hazard_fbfm40_processing_output_path),
    'PROCESSING_ACTION': fire_hazard_fbfm40_processing_action,
    'FINALIZATION_ACTION': fire_hazard_fbfm40_finalization_action,
    'RECLASSIFICATION_METHOD': fire_hazard_fbfm40_reclassification_method,
    'REUSED_CACHED_OUTPUT': fire_hazard_fbfm40_reclassification_reuse_output,
    'TOTAL_WINDOWS': fire_hazard_fbfm40_total_processing_windows,
    'COMPLETED_WINDOWS': fire_hazard_fbfm40_last_completed_window,
    'SOURCE_VALID_PIXELS': fire_hazard_fbfm40_source_valid_pixels,
    'SOURCE_NODATA_PIXELS': fire_hazard_fbfm40_source_nodata_pixels,
    'SCORED_OUTPUT_PIXELS': fire_hazard_fbfm40_scored_pixels,
    'OUTPUT_NODATA_PIXELS': fire_hazard_fbfm40_output_nodata_pixels,
    'NONBURNABLE_SCORED_PIXELS': fire_hazard_fbfm40_nonburnable_scored_pixels,
    'BACKGROUND_NODATA_PIXELS': fire_hazard_fbfm40_background_nodata_pixels,
    'UNRECOGNIZED_PIXELS': fire_hazard_fbfm40_unrecognized_pixels,
    'VALID_INSIDE_STUDY_AREA': fire_hazard_fbfm40_valid_inside_study_area,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_fbfm40_valid_outside_study_area,
    'OUTPUT_MINIMUM': fire_hazard_fbfm40_output_minimum,
    'OUTPUT_MAXIMUM': fire_hazard_fbfm40_output_maximum,
    'OUTPUT_MEAN': fire_hazard_fbfm40_output_mean,
    'TARGET_CRS': fire_hazard_fbfm40_final_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_fbfm40_final_width,
    'TARGET_HEIGHT': fire_hazard_fbfm40_final_height,
    'OUTPUT_DTYPE': fire_hazard_fbfm40_final_dtype,
    'OUTPUT_NODATA': fire_hazard_fbfm40_final_nodata,
    'OUTPUT_SIZE_BYTES': fire_hazard_fbfm40_final_output_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_fbfm40_final_output_size_mb,
    'CONTENT_VALID': fire_hazard_fbfm40_processing_content_valid,
    'STRUCTURE_VALID': fire_hazard_fbfm40_permanent_structure_valid,
    'STATISTICS_VERIFIED': fire_hazard_fbfm40_statistics_verified,
    'WINDOW_PROCESSING_FINALIZED': True}])
# Assemble fire hazard FBFM40 finalization summary into a table for QA review and downstream
# validation.
fire_hazard_fbfm40_finalization_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_PRODUCT': 'LANDFIRE_FBFM40', 'FINAL_OUTPUT_PATH': \
        str(fire_hazard_fbfm40_processing_output_path),
    'REUSED_CACHED_OUTPUT': fire_hazard_fbfm40_reclassification_reuse_output,
    'FINALIZATION_ACTION': fire_hazard_fbfm40_finalization_action,
    'SCORED_PIXELS': fire_hazard_fbfm40_scored_pixels,
    'NODATA_PIXELS': fire_hazard_fbfm40_output_nodata_pixels,
    'OUTPUT_MINIMUM': fire_hazard_fbfm40_output_minimum,
    'OUTPUT_MAXIMUM': fire_hazard_fbfm40_output_maximum,
    'OUTPUT_MEAN': fire_hazard_fbfm40_output_mean,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_fbfm40_valid_outside_study_area,
    'OUTPUT_SIZE_MB': fire_hazard_fbfm40_final_output_size_mb,
    'STRUCTURE_VALID': fire_hazard_fbfm40_permanent_structure_valid,
    'CONTENT_VALID': fire_hazard_fbfm40_processing_content_valid,
    'STATISTICS_VERIFIED': fire_hazard_fbfm40_statistics_verified,
    'FINALIZATION_COMPLETE': True}])
# Build fire hazard fbfm40 finalization summary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_finalization_summary_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_window_processing_finalization_summary.csv'
# Calculate fire hazard FBFM40 final class counts path for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_final_class_counts_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_final_class_pixel_counts.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_processing_manifest.to_csv(fire_hazard_fbfm40_reclassification_manifest_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_finalization_summary.to_csv(fire_hazard_fbfm40_finalization_summary_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_verified_class_counts.to_csv(fire_hazard_fbfm40_final_class_counts_path,
    index=False)
# Calculate fire hazard FBFM40 window processing finalized so raster processing covers the analysis
# grid in controlled blocks.
fire_hazard_fbfm40_window_processing_finalized = all([fire_hazard_fbfm40_window_processing_complete,
    fire_hazard_fbfm40_statistics_verified, fire_hazard_fbfm40_processing_content_valid,
    fire_hazard_fbfm40_permanent_structure_valid, \
        fire_hazard_fbfm40_processing_output_path.exists(),
    not fire_hazard_fbfm40_processing_temporary_path.exists(),
    fire_hazard_fbfm40_reclassification_manifest_path.exists(),
    fire_hazard_fbfm40_finalization_summary_path.exists(),
    fire_hazard_fbfm40_final_class_counts_path.exists()])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_window_processing_finalized:
    raise ValueError('FBFM40 window processing did not pass final completion checks.')
print(f'-> Finalization action: {fire_hazard_fbfm40_finalization_action}')
print(f'-> Permanent output: {fire_hazard_fbfm40_processing_output_path}')
print(f'-> Scored pixels: {fire_hazard_fbfm40_scored_pixels:,}')
print(f'-> Output NoData pixels: {fire_hazard_fbfm40_output_nodata_pixels:,}')
print(f'-> Final score range: '
    f'{fire_hazard_fbfm40_output_minimum:.4f} '
    f'to '
    f'{fire_hazard_fbfm40_output_maximum:.4f}')
print(f'-> Final mean score: {fire_hazard_fbfm40_output_mean:.4f}')
print(f'-> Valid pixels outside study area: {fire_hazard_fbfm40_valid_outside_study_area:,}')
print(f'-> Final output size: {fire_hazard_fbfm40_final_output_size_mb:,.2f} MB')
print(f'-> Permanent structure valid: {fire_hazard_fbfm40_permanent_structure_valid}')
print(f'-> Processing content valid: {fire_hazard_fbfm40_processing_content_valid}')
print(f'-> Statistics verified: {fire_hazard_fbfm40_statistics_verified}')
print(f'-> Window processing finalized: {fire_hazard_fbfm40_window_processing_finalized}')
print(f'-> Final processing manifest saved: {fire_hazard_fbfm40_reclassification_manifest_path}')
print(f'-> Finalization summary saved: {fire_hazard_fbfm40_finalization_summary_path}')
print('\n--- FBFM40 WINDOW-PROCESSING FINALIZATION SUMMARY ---')
display(fire_hazard_fbfm40_finalization_summary)
print('\n--- FINAL FBFM40 PROCESSING MANIFEST ---')
display(fire_hazard_fbfm40_processing_manifest)
print('\n--- MOST COMMON FINAL FBFM40 CLASSES ---')
display(fire_hazard_fbfm40_verified_class_counts[fire_hazard_fbfm40_verified_class_counts \
    ['OBSERVED']].sort_values('PIXEL_COUNT',
    ascending=False).head(5))
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The processed FBFM40 hazard raster is now stored at its permanent project location.')
print('Temporary processing files '
    'were removed only after the '
    'raster structure, score range, '
    'study-area coverage, and class '
    'statistics passed current '
    'checks.')
print('This step finalizes window '
    'processing but does not '
    'replace the independent output '
    'validation required in the '
    'next phase.')
print('\n=== FBFM40 WINDOW PROCESSING FINALIZED ===')


=== FINALIZING FBFM40 WINDOW PROCESSING ===
-> Finalization action: Retained validated cached FBFM40 hazard raster
-> Permanent output: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\normalized_sources\fbfm40_fuel_hazard_score.tif
-> Scored pixels: 15,475,232
-> Output NoData pixels: 121,295,786
-> Final score range: 0.0000 to 1.0000
-> Final mean score: 0.4357
-> Valid pixels outside study area: 0
-> Final output size: 14.34 MB
-> Permanent structure valid: True
-> Processing content valid: True
-> Statistics verified: True
-> Window processing finalized: True
-> Final processing manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_reclassification_manifest.csv
-> Finalization summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_window_processing_finalization_summary.csv

--- FBFM40 WINDOW-PROCESSING FINALIZATION SUMMARY ---


,COMPONENT_ID,SOURCE_PRODUCT,FINAL_OUTPUT_PATH,REUSED_CACHED_OUTPUT,FINALIZATION_ACTION,SCORED_PIXELS,NODATA_PIXELS,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,VALID_OUTSIDE_STUDY_AREA,OUTPUT_SIZE_MB,STRUCTURE_VALID,CONTENT_VALID,STATISTICS_VERIFIED,FINALIZATION_COMPLETE
0,FUEL_HAZARD,LANDFIRE_FBFM40,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,Retained validated cached FBFM40 hazard raster,15475232,121295786,0.0,1.0,0.435666,0,14.335672,True,True,True,True



--- FINAL FBFM40 PROCESSING MANIFEST ---


,COMPONENT_ID,SOURCE_PRODUCT,PROCESSING_STAGE,INPUT_PATH,OUTPUT_PATH,PROCESSING_ACTION,FINALIZATION_ACTION,RECLASSIFICATION_METHOD,REUSED_CACHED_OUTPUT,TOTAL_WINDOWS,...,TARGET_WIDTH,TARGET_HEIGHT,OUTPUT_DTYPE,OUTPUT_NODATA,OUTPUT_SIZE_BYTES,OUTPUT_SIZE_MB,CONTENT_VALID,STRUCTURE_VALID,STATISTICS_VERIFIED,WINDOW_PROCESSING_FINALIZED
0,FUEL_HAZARD,LANDFIRE_FBFM40,Window processing finalized,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing FBFM40 hazard raster,Retained validated cached FBFM40 hazard raster,expert_informed_ordinal_lookup,True,551,...,9454,14467,float32,-9999.0,15032042,14.335672,True,True,True,True



--- MOST COMMON FINAL FBFM40 CLASSES ---


,FBFM40_CODE,FBFM40_MODEL,FUEL_GROUP,BURNABLE,HAZARD_SCORE,OUTPUT_DISPOSITION,PIXEL_COUNT,OBSERVED
16,122,GS2,Grass-Shrub,True,0.50,Score,3496338,True
23,145,SH5,Shrub,True,0.75,Score,2033787,True
7,102,GR2,Grass,True,0.35,Score,1815794,True
1,91,NB1,Nonburnable,False,0.00,Score,1393298,True
15,121,GS1,Grass-Shrub,True,0.35,Score,1354419,True



NOTE:
The processed FBFM40 hazard raster is now stored at its permanent project location.
Temporary processing files were removed only after the raster structure, score range, study-area coverage, and class statistics passed current checks.
This step finalizes window processing but does not replace the independent output validation required in the next phase.

=== FBFM40 WINDOW PROCESSING FINALIZED ===


### Validating Reclassified FBFM40 Fuel-Hazard Raster


In [107]:
print('=== VALIDATING RECLASSIFIED FBFM40 FUEL-HAZARD RASTER ===')
# Record whether required FBFM40 validation inputs satisfies the checks required before the workflow
# advances.
required_fbfm40_validation_inputs = ['fire_hazard_fbfm40_window_processing_finalized',
    'fire_hazard_fbfm40_processing_input_path', 'fire_hazard_fbfm40_processing_output_path',
    'fire_hazard_fbfm40_reclassification_method', 'fire_hazard_fbfm40_score_lookup',
    'fire_hazard_fbfm40_nodata_codes', 'fire_hazard_fbfm40_recognized_codes',
    'fire_hazard_fbfm40_score_minimum', 'fire_hazard_fbfm40_score_maximum',
    'fire_hazard_fbfm40_score_nodata', 'fire_hazard_fbfm40_validation_path',
    'fire_hazard_fbfm40_reclassification_manifest_path',
    'fire_hazard_fbfm40_scored_pixels', 'fire_hazard_fbfm40_output_nodata_pixels',
    'fire_hazard_fbfm40_output_minimum', 'fire_hazard_fbfm40_output_maximum',
    'fire_hazard_fbfm40_output_mean', 'fire_hazard_fbfm40_valid_inside_study_area',
    'fire_hazard_fbfm40_valid_outside_study_area',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_fuel_metadata_directory']
# Identify missing FBFM40 validation inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fbfm40_validation_inputs = [object_name for object_name in \
    required_fbfm40_validation_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 validation inputs remain unresolved before this workflow stage
# begins.
if missing_fbfm40_validation_inputs:
    raise NameError(f'The following FBFM40 '
        f'validation objects are '
        f'missing:\n'
        f'{missing_fbfm40_validation_inputs}\n\n'
        f'Run Finalize FBFM40 Window '
        f'Processing before validating '
        f'the reclassified fuel-hazard '
        f'raster.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_window_processing_finalized:
    raise ValueError('The FBFM40 raster has not passed window-processing finalization.')
# Build fire hazard fbfm40 validation source path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_validation_source_path = Path(fire_hazard_fbfm40_processing_input_path)
# Build fire hazard fbfm40 validation output path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_validation_output_path = Path(fire_hazard_fbfm40_processing_output_path)

# Iterate through  so each required item receives the same processing and QA checks.
for raster_name, raster_path in {'Aligned FBFM40 source': fire_hazard_fbfm40_validation_source_path,
    'Reclassified FBFM40 hazard': fire_hazard_fbfm40_validation_output_path}.items():

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not raster_path.exists() or not raster_path.is_file() or raster_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The {raster_name} raster is unavailable:\n{raster_path}')
# Evaluate fire hazard fbfm40 valid extensions so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fbfm40_valid_extensions = {'.tif', '.tiff'}
# Calculate fire hazard FBFM40 minimum file size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_fbfm40_minimum_file_size_bytes = 1024
# Set fire hazard FBFM40 formula tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_formula_tolerance = 1e-06
# Set fire hazard FBFM40 statistics tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fbfm40_statistics_tolerance = 1e-06
# Build the expected fbfm40 mask shape mask used to isolate records required for this analysis.
expected_fbfm40_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_fbfm40_mask_shape:
    raise ValueError(f'The FBFM40 validation '
        f'study-area mask does not match '
        f'the common raster grid.\nMask '
        f'shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_fbfm40_mask_shape}')
# Evaluate fire hazard fbfm40 validation study area pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fbfm40_validation_study_area_pixels = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_fbfm40_validation_study_area_pixels == 0:
    raise ValueError('The FBFM40 validation study-area mask contains no included pixels.')
# Prepare FBFM40 output file exists for the downstream processing or validation performed in this
# workflow stage.
fbfm40_output_file_exists = fire_hazard_fbfm40_validation_output_path.exists() and \
    fire_hazard_fbfm40_validation_output_path.is_file()
# Evaluate fbfm40 output extension valid so invalid inputs or outputs can be rejected before
# continuing.
fbfm40_output_extension_valid = fire_hazard_fbfm40_validation_output_path.suffix.lower() in \
    fire_hazard_fbfm40_valid_extensions
# Calculate FBFM40 output file size bytes for completeness, file-integrity, or processing QA.
fbfm40_output_file_size_bytes = fire_hazard_fbfm40_validation_output_path.stat().st_size if \
    fbfm40_output_file_exists else 0
# Evaluate fbfm40 output file size valid so invalid inputs or outputs can be rejected before
# continuing.
fbfm40_output_file_size_valid = fbfm40_output_file_size_bytes >= \
    fire_hazard_fbfm40_minimum_file_size_bytes
# Prepare FBFM40 output readable for the downstream processing or validation performed in this
# workflow stage.
fbfm40_output_readable = False
# Evaluate fbfm40 band count valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_band_count_valid = False
# Evaluate fbfm40 width valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_width_valid = False
# Evaluate fbfm40 height valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_height_valid = False
# Store fbfm40 crs valid so spatial operations use the required coordinate reference system.
fbfm40_crs_valid = False
# Evaluate fbfm40 transform valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_transform_valid = False
# Evaluate fbfm40 pixel size valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_pixel_size_valid = False
# Evaluate fbfm40 dtype valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_dtype_valid = False
# Evaluate fbfm40 nodata valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_nodata_valid = False
# Evaluate fbfm40 grid valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_grid_valid = False
# Evaluate fbfm40 structure valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_structure_valid = False
# Evaluate fbfm40 validation error so invalid inputs or outputs can be rejected before continuing.
fbfm40_validation_error = None
# Evaluate validated source pixels so invalid inputs or outputs can be rejected before continuing.
validated_source_pixels = 0
# Evaluate validated source nodata pixels so invalid inputs or outputs can be rejected before
# continuing.
validated_source_nodata_pixels = 0
# Evaluate validated output pixels so invalid inputs or outputs can be rejected before continuing.
validated_output_pixels = 0
# Evaluate validated output nodata pixels so invalid inputs or outputs can be rejected before
# continuing.
validated_output_nodata_pixels = 0
# Evaluate validated inside study area so invalid inputs or outputs can be rejected before
# continuing.
validated_inside_study_area = 0
# Evaluate validated outside study area so invalid inputs or outputs can be rejected before
# continuing.
validated_outside_study_area = 0
# Evaluate validated output minimum so invalid inputs or outputs can be rejected before continuing.
validated_output_minimum = None
# Evaluate validated output maximum so invalid inputs or outputs can be rejected before continuing.
validated_output_maximum = None
# Evaluate validated output sum so invalid inputs or outputs can be rejected before continuing.
validated_output_sum = 0.0
# Evaluate validated output mean so invalid inputs or outputs can be rejected before continuing.
validated_output_mean = None
# Calculate formula checked pixels for completeness, file-integrity, or processing QA.
formula_checked_pixels = 0
# Record whether formula matching pixels satisfies the checks required before the workflow advances.
formula_matching_pixels = 0
# Calculate formula mismatch pixels for completeness, file-integrity, or processing QA.
formula_mismatch_pixels = 0
# Set maximum formula difference as an explicit model or validation parameter used consistently in
# downstream calculations.
maximum_formula_difference = 0.0
# Prepare formula difference sum for the downstream processing or validation performed in this
# workflow stage.
formula_difference_sum = 0.0
# Prepare mean formula difference for the downstream processing or validation performed in this
# workflow stage.
mean_formula_difference = None
# Calculate recognized source pixels for completeness, file-integrity, or processing QA.
recognized_source_pixels = 0
# Calculate unrecognized source pixels for completeness, file-integrity, or processing QA.
unrecognized_source_pixels = 0
# Prepare unrecognized source codes for the downstream processing or validation performed in this
# workflow stage.
unrecognized_source_codes = set()
# Evaluate source nodata with valid output so invalid inputs or outputs can be rejected before
# continuing.
source_nodata_with_valid_output = 0
# Evaluate background code with valid output so invalid inputs or outputs can be rejected before
# continuing.
background_code_with_valid_output = 0
# Prepare scored code without output for the fuel-hazard calculation and subsequent QA checks.
scored_code_without_output = 0
# Evaluate output valid outside study area so invalid inputs or outputs can be rejected before
# continuing.
output_valid_outside_study_area = 0

# Document this operation so its role in the current fuel-hazard workflow is clear before processing
# continues.
try:

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(fire_hazard_fbfm40_validation_source_path) as source_raster:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fbfm40_validation_output_path) as output_raster:
            # Prepare FBFM40 output readable for the downstream processing or validation performed
            # in this workflow stage.
            fbfm40_output_readable = True
            # Evaluate fbfm40 band count valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_band_count_valid = output_raster.count == 1
            # Evaluate fbfm40 width valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_width_valid = output_raster.width == fire_hazard_alignment_width
            # Evaluate fbfm40 height valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_height_valid = output_raster.height == fire_hazard_alignment_height
            # Store fbfm40 crs valid so spatial operations use the required coordinate reference
            # system.
            fbfm40_crs_valid = output_raster.crs is not None and output_raster.crs == \
                rasterio.crs.CRS.from_user_input(fire_hazard_target_crs)
            # Evaluate fbfm40 transform valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_transform_valid = \
                output_raster.transform.almost_equals(fire_hazard_alignment_transform)
            # Evaluate fbfm40 pixel size valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_pixel_size_valid = all([np.isclose(abs(output_raster.transform.a),
                fire_hazard_cell_size), np.isclose(abs(output_raster.transform.e),
                fire_hazard_cell_size)])
            # Evaluate fbfm40 grid valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_grid_valid = all([fbfm40_width_valid,
                fbfm40_height_valid, fbfm40_crs_valid, fbfm40_transform_valid,
                fbfm40_pixel_size_valid])
            # Evaluate fbfm40 dtype valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_dtype_valid = output_raster.dtypes[0] == 'float32'
            # Evaluate fbfm40 nodata valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_nodata_valid = output_raster.nodata == fire_hazard_fbfm40_score_nodata
            # Evaluate fbfm40 structure valid so invalid inputs or outputs can be rejected before
            # continuing.
            fbfm40_structure_valid = all([fbfm40_band_count_valid,
                fbfm40_dtype_valid, fbfm40_nodata_valid])
            # Record whether source output grid match satisfies the checks required before the
            # workflow advances.
            source_output_grid_match = all([source_raster.count == 1,
                source_raster.width == output_raster.width, source_raster.height == \
                    output_raster.height,
                source_raster.crs == output_raster.crs, \
                    source_raster.transform.almost_equals(output_raster.transform)])

            # Stop execution if this validation condition is not satisfied before dependent
            # processing continues.
            if not source_output_grid_match:
                raise ValueError('The aligned FBFM40 source and '
                    'reclassified hazard raster do '
                    'not use the same grid.')

            # Iterate through output raster.block windows so each required item receives the same
            # processing and QA checks.
            for _, raster_window in output_raster.block_windows(1):
                # Prepare source array for the raster calculation or validation performed in this
                # processing block.
                source_array = source_raster.read(1, window=raster_window)
                # Prepare output array for the raster calculation or validation performed in this
                # processing block.
                output_array = output_raster.read(1, window=raster_window)
                # Prepare row start for the downstream processing or validation performed in this
                # workflow stage.
                row_start = int(raster_window.row_off)
                # Prepare row end for the downstream processing or validation performed in this
                # workflow stage.
                row_end = int(raster_window.row_off + raster_window.height)
                # Prepare column start for the downstream processing or validation performed in this
                # workflow stage.
                column_start = int(raster_window.col_off)
                # Prepare column end for the downstream processing or validation performed in this
                # workflow stage.
                column_end = int(raster_window.col_off + raster_window.width)
                # Build the study area window mask mask used to isolate records required for this
                # analysis.
                study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                    column_start:column_end]
                # Build the source finite mask mask used to isolate records required for this
                # analysis.
                source_finite_mask = np.isfinite(source_array)

                # Stop execution if the NoData configuration is incompatible with the raster product
                # being validated.
                if source_raster.nodata is None:
                    # Build the source data mask mask used to isolate records required for this
                    # analysis.
                    source_data_mask = source_finite_mask
                else:
                    # Build the source data mask mask used to isolate records required for this
                    # analysis.
                    source_data_mask = source_finite_mask & (source_array != source_raster.nodata)
                # Build the source inside mask mask used to isolate records required for this
                # analysis.
                source_inside_mask = source_data_mask & study_area_window_mask
                # Evaluate validated source pixels so invalid inputs or outputs can be rejected
                # before continuing.
                validated_source_pixels += int(source_inside_mask.sum())
                # Evaluate validated source nodata pixels so invalid inputs or outputs can be
                # rejected before continuing.
                validated_source_nodata_pixels += int((~source_data_mask).sum())
                # Build the output valid mask mask used to isolate records required for this
                # analysis.
                output_valid_mask = np.isfinite(output_array) & (output_array != \
                    fire_hazard_fbfm40_score_nodata)
                # Prepare output values for the downstream processing or validation performed in
                # this workflow stage.
                output_values = output_array[output_valid_mask]
                # Evaluate validated output pixels so invalid inputs or outputs can be rejected
                # before continuing.
                validated_output_pixels += int(output_valid_mask.sum())
                # Evaluate validated output nodata pixels so invalid inputs or outputs can be
                # rejected before continuing.
                validated_output_nodata_pixels += int((~output_valid_mask).sum())
                # Evaluate validated inside study area so invalid inputs or outputs can be rejected
                # before continuing.
                validated_inside_study_area += int((output_valid_mask & \
                    study_area_window_mask).sum())
                # Evaluate validated outside study area so invalid inputs or outputs can be rejected
                # before continuing.
                validated_outside_study_area += int((output_valid_mask & \
                    ~study_area_window_mask).sum())
                # Evaluate output valid outside study area so invalid inputs or outputs can be
                # rejected before continuing.
                output_valid_outside_study_area += int((output_valid_mask & \
                    ~study_area_window_mask).sum())

                # Stop execution if this validation condition is not satisfied before dependent
                # processing continues.
                if output_values.size > 0:
                    # Set block minimum as an explicit model or validation parameter used
                    # consistently in downstream calculations.
                    block_minimum = float(output_values.min())
                    # Set block maximum as an explicit model or validation parameter used
                    # consistently in downstream calculations.
                    block_maximum = float(output_values.max())

                    # Stop execution if the prerequisite validation has not passed before this
                    # workflow stage continues.
                    if validated_output_minimum is None:
                        # Evaluate validated output minimum so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_output_minimum = block_minimum
                    else:
                        # Evaluate validated output minimum so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_output_minimum = min(validated_output_minimum, block_minimum)

                    # Stop execution if the prerequisite validation has not passed before this
                    # workflow stage continues.
                    if validated_output_maximum is None:
                        # Evaluate validated output maximum so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_output_maximum = block_maximum
                    else:
                        # Evaluate validated output maximum so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_output_maximum = max(validated_output_maximum, block_maximum)
                    # Evaluate validated output sum so invalid inputs or outputs can be rejected
                    # before continuing.
                    validated_output_sum += float(output_values.sum(dtype=np.float64))

                # Stop execution if this validation condition is not satisfied before dependent
                # processing continues.
                if source_inside_mask.any():
                    # Prepare source codes for the downstream processing or validation performed in
                    # this workflow stage.
                    source_codes = np.rint(source_array[source_inside_mask]).astype(np.int32)
                    # Prepare source output values for the downstream processing or validation
                    # performed in this workflow stage.
                    source_output_values = output_array[source_inside_mask]
                    # Build the recognized mask mask used to isolate records required for this
                    # analysis.
                    recognized_mask = np.isin(source_codes,
                        list(fire_hazard_fbfm40_recognized_codes))
                    # Calculate recognized source pixels for completeness, file-integrity, or
                    # processing QA.
                    recognized_source_pixels += int(recognized_mask.sum())
                    # Build the unrecognized mask mask used to isolate records required for this
                    # analysis.
                    unrecognized_mask = ~recognized_mask

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if unrecognized_mask.any():
                        # Prepare unknown codes for the downstream processing or validation
                        # performed in this workflow stage.
                        unknown_codes = np.unique(source_codes[unrecognized_mask])
                        # Update unrecognized source codes with the values produced by the current
                        # processing step.
                        unrecognized_source_codes.update((int(code) for code in unknown_codes))
                        # Calculate unrecognized source pixels for completeness, file-integrity, or
                        # processing QA.
                        unrecognized_source_pixels += int(unrecognized_mask.sum())
                    # Prepare expected output values for the downstream processing or validation
                    # performed in this workflow stage.
                    expected_output_values = np.full(source_codes.shape,
                        fire_hazard_fbfm40_score_nodata, dtype=np.float32)

                    # Iterate through fire hazard FBFM40 score lookup.items so each required item
                    # receives the same processing and QA checks.
                    for class_code, hazard_score in fire_hazard_fbfm40_score_lookup.items():
                        # Build the class mask mask used to isolate records required for this
                        # analysis.
                        class_mask = source_codes == int(class_code)
                        # Prepare expected output values so this workflow stage has the values
                        # required for downstream spatial processing and QA.
                        expected_output_values[class_mask] = np.float32(hazard_score)
                    # Build the background mask mask used to isolate records required for this
                    # analysis.
                    background_mask = np.isin(source_codes, list(fire_hazard_fbfm40_nodata_codes))
                    # Build the background output valid mask mask used to isolate records required
                    # for this analysis.
                    background_output_valid_mask = background_mask & \
                        np.isfinite(source_output_values) & (source_output_values != \
                        fire_hazard_fbfm40_score_nodata)
                    # Evaluate background code with valid output so invalid inputs or outputs can be
                    # rejected before continuing.
                    background_code_with_valid_output += int(background_output_valid_mask.sum())
                    # Build the expected score mask mask used to isolate records required for this
                    # analysis.
                    expected_score_mask = np.isfinite(expected_output_values) & \
                        (expected_output_values != fire_hazard_fbfm40_score_nodata)
                    # Build the actual score mask mask used to isolate records required for this
                    # analysis.
                    actual_score_mask = np.isfinite(source_output_values) & (source_output_values \
                        != fire_hazard_fbfm40_score_nodata)
                    # Prepare scored code without output for the fuel-hazard calculation and
                    # subsequent QA checks.
                    scored_code_without_output += int((expected_score_mask & \
                        ~actual_score_mask).sum())
                    # Build the formula comparison mask mask used to isolate records required for
                    # this analysis.
                    formula_comparison_mask = expected_score_mask & actual_score_mask

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if formula_comparison_mask.any():
                        # Prepare expected scores for the fuel-hazard calculation and subsequent QA
                        # checks.
                        expected_scores = expected_output_values[formula_comparison_mask]
                        # Prepare actual scores for the fuel-hazard calculation and subsequent QA
                        # checks.
                        actual_scores = source_output_values[formula_comparison_mask]
                        # Prepare absolute differences for the downstream processing or validation
                        # performed in this workflow stage.
                        absolute_differences = np.abs(actual_scores - expected_scores)
                        # Build the matching mask mask used to isolate records required for this
                        # analysis.
                        matching_mask = absolute_differences <= fire_hazard_fbfm40_formula_tolerance
                        # Calculate formula checked pixels for completeness, file-integrity, or
                        # processing QA.
                        formula_checked_pixels += int(absolute_differences.size)
                        # Record whether formula matching pixels satisfies the checks required
                        # before the workflow advances.
                        formula_matching_pixels += int(matching_mask.sum())
                        # Calculate formula mismatch pixels for completeness, file-integrity, or
                        # processing QA.
                        formula_mismatch_pixels += int((~matching_mask).sum())
                        # Prepare formula difference sum for the downstream processing or validation
                        # performed in this workflow stage.
                        formula_difference_sum += float(absolute_differences.sum(dtype=np.float64))

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if absolute_differences.size > 0:
                            # Set maximum formula difference as an explicit model or validation
                            # parameter used consistently in downstream calculations.
                            maximum_formula_difference = max(maximum_formula_difference,
                                float(absolute_differences.max()))
                # Evaluate source nodata with valid output so invalid inputs or outputs can be
                # rejected before continuing.
                source_nodata_with_valid_output += int((~source_data_mask & \
                    output_valid_mask).sum())
                del source_array
                del output_array
                del source_finite_mask
                del source_data_mask
                del output_valid_mask
# Handle the expected failure explicitly so the workflow can report or clean up the affected
# operation.
except Exception as error:
    # Evaluate fbfm40 validation error so invalid inputs or outputs can be rejected before
    # continuing.
    fbfm40_validation_error = str(error)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if validated_output_pixels > 0:
    # Evaluate validated output mean so invalid inputs or outputs can be rejected before continuing.
    validated_output_mean = validated_output_sum / validated_output_pixels

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if formula_checked_pixels > 0:
    # Prepare mean formula difference for the downstream processing or validation performed in this
    # workflow stage.
    mean_formula_difference = formula_difference_sum / formula_checked_pixels
# Prepare FBFM40 value range available for the downstream processing or validation performed in this
# workflow stage.
fbfm40_value_range_available = validated_output_minimum is not None and validated_output_maximum \
    is not None and (validated_output_mean is not None)
# Evaluate fbfm40 value range valid so invalid inputs or outputs can be rejected before continuing.
fbfm40_value_range_valid = fbfm40_value_range_available and validated_output_minimum >= \
    fire_hazard_fbfm40_score_minimum and (validated_output_maximum <= \
    fire_hazard_fbfm40_score_maximum)
# Evaluate fire hazard fbfm40 formula valid so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fbfm40_formula_valid = all([formula_checked_pixels > 0,
    formula_mismatch_pixels == 0, maximum_formula_difference <= \
        fire_hazard_fbfm40_formula_tolerance,
    unrecognized_source_pixels == 0, background_code_with_valid_output == 0,
    scored_code_without_output == 0, source_nodata_with_valid_output == 0])
# Record whether FBFM40 pixel statistics match satisfies the checks required before the workflow
# advances.
fbfm40_pixel_statistics_match = all([validated_output_pixels == fire_hazard_fbfm40_scored_pixels,
    validated_output_nodata_pixels == fire_hazard_fbfm40_output_nodata_pixels,
    validated_inside_study_area == fire_hazard_fbfm40_valid_inside_study_area,
    validated_outside_study_area == fire_hazard_fbfm40_valid_outside_study_area])
# Record whether FBFM40 value statistics match satisfies the checks required before the workflow
# advances.
fbfm40_value_statistics_match = all([fbfm40_value_range_available,
    np.isclose(validated_output_minimum, fire_hazard_fbfm40_output_minimum,
    atol=fire_hazard_fbfm40_statistics_tolerance),
    np.isclose(validated_output_maximum, fire_hazard_fbfm40_output_maximum,
    atol=fire_hazard_fbfm40_statistics_tolerance),
    np.isclose(validated_output_mean, fire_hazard_fbfm40_output_mean,
    atol=fire_hazard_fbfm40_statistics_tolerance)])
# Prepare FBFM40 processing manifest exists for the downstream processing or validation performed in
# this workflow stage.
fbfm40_processing_manifest_exists = Path(fire_hazard_fbfm40_reclassification_manifest_path).exists()
# Evaluate fbfm40 processing manifest valid so invalid inputs or outputs can be rejected before
# continuing.
fbfm40_processing_manifest_valid = False

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fbfm40_processing_manifest_exists:
    # Prepare FBFM40 saved processing manifest for the downstream processing or validation performed
    # in this workflow stage.
    fbfm40_saved_processing_manifest = \
        pd.read_csv(fire_hazard_fbfm40_reclassification_manifest_path)
    # Define the manifest fields inputs required by this workflow stage.
    required_manifest_fields = ['SOURCE_PRODUCT',
        'OUTPUT_PATH', 'CONTENT_VALID', 'STRUCTURE_VALID',
        'STATISTICS_VERIFIED', 'WINDOW_PROCESSING_FINALIZED']
    # Identify missing manifest fields so unavailable prerequisites are caught before this workflow
    # stage runs.
    missing_manifest_fields = [field_name for field_name in required_manifest_fields if \
        field_name not in fbfm40_saved_processing_manifest.columns]

    # Stop execution when required manifest fields inputs are unavailable.
    if not missing_manifest_fields:
        # Evaluate manifest record count valid so invalid inputs or outputs can be rejected before
        # continuing.
        manifest_record_count_valid = len(fbfm40_saved_processing_manifest) == 1

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if manifest_record_count_valid:
            # Prepare manifest record for the downstream processing or validation performed in this
            # workflow stage.
            manifest_record = fbfm40_saved_processing_manifest.iloc[0]
            # Build manifest boolean values from the records that satisfy this workflow's selection
            # criteria.
            manifest_boolean_values = {field_name: \
                str(manifest_record[field_name]).strip().lower() == 'true' for field_name in \
                ['CONTENT_VALID',
                'STRUCTURE_VALID', 'STATISTICS_VERIFIED', 'WINDOW_PROCESSING_FINALIZED']}
            # Record whether FBFM40 processing manifest valid satisfies the checks required before
            # the workflow advances.
            fbfm40_processing_manifest_valid = all([str(manifest_record['SOURCE_PRODUCT']) == \
                'LANDFIRE_FBFM40',
                Path(str(manifest_record['OUTPUT_PATH'])) == \
                    fire_hazard_fbfm40_validation_output_path,
                all(manifest_boolean_values.values())])
# Evaluate fire hazard fbfm40 hazard raster valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fbfm40_hazard_raster_valid = all([fbfm40_output_file_exists,
    fbfm40_output_extension_valid, fbfm40_output_file_size_valid,
    fbfm40_output_readable, fbfm40_grid_valid, fbfm40_structure_valid,
    validated_output_pixels > 0, validated_inside_study_area > 0,
    validated_outside_study_area == 0, fbfm40_value_range_valid,
    fire_hazard_fbfm40_formula_valid, fbfm40_pixel_statistics_match,
    fbfm40_value_statistics_match, fbfm40_processing_manifest_valid,
    fbfm40_validation_error is None])
# Evaluate fire hazard fbfm40 validation so invalid inputs or outputs can be rejected before
# continuing.
# Create a validation summary for the completed FBFM40 fuel-hazard raster.
fire_hazard_fbfm40_validation = pd.DataFrame(
    [
        {
            # Identify the hazard component and the source product being validated.
            'COMPONENT_ID': 'FUEL_HAZARD',
            'SOURCE_PRODUCT': 'LANDFIRE_FBFM40',

            # Record the source and output raster paths used for validation.
            'INPUT_PATH': str(
                fire_hazard_fbfm40_validation_source_path
            ),
            'OUTPUT_PATH': str(
                fire_hazard_fbfm40_validation_output_path
            ),

            # Record file-level checks for existence, format, and file size.
            'FILE_EXISTS': fbfm40_output_file_exists,
            'FILE_EXTENSION_VALID': fbfm40_output_extension_valid,
            'FILE_SIZE_BYTES': fbfm40_output_file_size_bytes,
            'FILE_SIZE_MB': (
                fbfm40_output_file_size_bytes / 1024 ** 2
            ),
            'FILE_SIZE_VALID': fbfm40_output_file_size_valid,

            # Record raster structure and target-grid validation results.
            'RASTER_READABLE': fbfm40_output_readable,
            'BAND_COUNT_VALID': fbfm40_band_count_valid,
            'WIDTH_VALID': fbfm40_width_valid,
            'HEIGHT_VALID': fbfm40_height_valid,
            'CRS_VALID': fbfm40_crs_valid,
            'TRANSFORM_VALID': fbfm40_transform_valid,
            'PIXEL_SIZE_VALID': fbfm40_pixel_size_valid,
            'DTYPE_VALID': fbfm40_dtype_valid,
            'NODATA_VALID': fbfm40_nodata_valid,
            'GRID_VALID': fbfm40_grid_valid,
            'STRUCTURE_VALID': fbfm40_structure_valid,

            # Record source and output pixel totals used for QA comparison.
            'SOURCE_VALID_PIXELS': validated_source_pixels,
            'SOURCE_NODATA_PIXELS': validated_source_nodata_pixels,
            'OUTPUT_VALID_PIXELS': validated_output_pixels,
            'OUTPUT_NODATA_PIXELS': validated_output_nodata_pixels,

            # Record valid pixels inside and outside the study-area mask.
            'VALID_INSIDE_STUDY_AREA': validated_inside_study_area,
            'VALID_OUTSIDE_STUDY_AREA': validated_outside_study_area,

            # Record the validated range and mean of the fuel-hazard scores.
            'OUTPUT_MINIMUM': validated_output_minimum,
            'OUTPUT_MAXIMUM': validated_output_maximum,
            'OUTPUT_MEAN': validated_output_mean,
            'VALUE_RANGE_AVAILABLE': fbfm40_value_range_available,
            'VALUE_RANGE_VALID': fbfm40_value_range_valid,

            # Record pixel-level checks of the FBFM40 reclassification formula.
            'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
            'FORMULA_MATCHING_PIXELS': formula_matching_pixels,
            'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
            'MAXIMUM_FORMULA_DIFFERENCE': maximum_formula_difference,
            'MEAN_FORMULA_DIFFERENCE': mean_formula_difference,

            # Record any source classes that were not recognized by the reclassification policy.
            'UNRECOGNIZED_SOURCE_PIXELS': unrecognized_source_pixels,
            'UNRECOGNIZED_SOURCE_CODES': ', '.join(
                str(code)
                for code in sorted(unrecognized_source_codes)
            ),

            # Record unexpected source-to-output pixel relationships.
            'SOURCE_NODATA_WITH_VALID_OUTPUT': (
                source_nodata_with_valid_output
            ),
            'BACKGROUND_WITH_VALID_OUTPUT': (
                background_code_with_valid_output
            ),
            'SCORED_CODE_WITHOUT_OUTPUT': (
                scored_code_without_output
            ),

            # Record the final formula and statistical validation results.
            'FORMULA_VALID': fire_hazard_fbfm40_formula_valid,
            'PIXEL_STATISTICS_MATCH': fbfm40_pixel_statistics_match,
            'VALUE_STATISTICS_MATCH': fbfm40_value_statistics_match,

            # Confirm that the processing manifest remains valid for the completed output.
            'PROCESSING_MANIFEST_VALID': (
                fbfm40_processing_manifest_valid
            ),

            # Record any validation error and the final raster validation status.
            'ERROR_MESSAGE': fbfm40_validation_error,
            'VALID': fire_hazard_fbfm40_hazard_raster_valid,
        }
    ]
)
# Assemble fire hazard FBFM40 validation summary into a table for QA review and downstream
# validation.
# Create a concise QA summary for the completed FBFM40 hazard-score raster.
fire_hazard_fbfm40_validation_summary = pd.DataFrame(
    [
        {
            # Identify the hazard component and validated output product.
            'COMPONENT_ID': 'FUEL_HAZARD',
            'PRODUCT': 'Reclassified FBFM40 Hazard Score',

            # Record the target-grid, raster-structure, and score-range checks.
            'GRID_VALID': fbfm40_grid_valid,
            'STRUCTURE_VALID': fbfm40_structure_valid,
            'SCORE_RANGE_VALID': fbfm40_value_range_valid,

            # Record the pixel-level verification of the reclassification formula.
            'FORMULA_VALID': fire_hazard_fbfm40_formula_valid,
            'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
            'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
            'MAXIMUM_FORMULA_DIFFERENCE': maximum_formula_difference,

            # Confirm that output statistics match the completed processing results.
            'PIXEL_STATISTICS_MATCH': fbfm40_pixel_statistics_match,
            'VALUE_STATISTICS_MATCH': fbfm40_value_statistics_match,

            # Confirm that the processing manifest remains valid for the output raster.
            'PROCESSING_MANIFEST_VALID': fbfm40_processing_manifest_valid,

            # Record spatial-mask and source-class issues detected during validation.
            'VALID_OUTSIDE_STUDY_AREA': validated_outside_study_area,
            'UNRECOGNIZED_SOURCE_PIXELS': unrecognized_source_pixels,

            # Record the validated range and mean of the final hazard scores.
            'OUTPUT_MINIMUM': validated_output_minimum,
            'OUTPUT_MAXIMUM': validated_output_maximum,
            'OUTPUT_MEAN': validated_output_mean,

            # Record the final validation status for the FBFM40 hazard raster.
            'VALIDATION_COMPLETE': fire_hazard_fbfm40_hazard_raster_valid,
        }
    ]
)
# Build fire hazard fbfm40 validation summary path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_validation_summary_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_hazard_validation_summary.csv'
# Build fire hazard fbfm40 formula validation path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_formula_validation_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_formula_validation.csv'
# Evaluate fire hazard fbfm40 formula validation so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fbfm40_formula_validation = pd.DataFrame([{'RECLASSIFICATION_METHOD': \
    fire_hazard_fbfm40_reclassification_method,
    'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
    'FORMULA_MATCHING_PIXELS': formula_matching_pixels,
    'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
    'MAXIMUM_ABSOLUTE_DIFFERENCE': maximum_formula_difference,
    'MEAN_ABSOLUTE_DIFFERENCE': mean_formula_difference,
    'FORMULA_TOLERANCE': fire_hazard_fbfm40_formula_tolerance,
    'UNRECOGNIZED_SOURCE_PIXELS': unrecognized_source_pixels,
    'BACKGROUND_WITH_VALID_OUTPUT': background_code_with_valid_output,
    'SCORED_CODE_WITHOUT_OUTPUT': scored_code_without_output,
    'SOURCE_NODATA_WITH_VALID_OUTPUT': source_nodata_with_valid_output,
    'VALID': fire_hazard_fbfm40_formula_valid}])
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_validation.to_csv(fire_hazard_fbfm40_validation_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_validation_summary.to_csv(fire_hazard_fbfm40_validation_summary_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_formula_validation.to_csv(fire_hazard_fbfm40_formula_validation_path,
    index=False)
print(f'-> Output grid valid: {fbfm40_grid_valid}')
print(f'-> Output structure valid: {fbfm40_structure_valid}')
print(f'-> Valid output pixels: {validated_output_pixels:,}')
print(f'-> Valid pixels outside study area: {validated_outside_study_area:,}')
print(f'-> Output score range: {validated_output_minimum:.4f} to {validated_output_maximum:.4f}')
print(f'-> Output mean score: {validated_output_mean:.4f}')
print(f'-> Formula pixels checked: {formula_checked_pixels:,}')
print(f'-> Formula mismatch pixels: {formula_mismatch_pixels:,}')
print(f'-> Maximum formula difference: {maximum_formula_difference:.8f}')
print(f'-> Formula valid: {fire_hazard_fbfm40_formula_valid}')
print(f'-> Pixel statistics match: {fbfm40_pixel_statistics_match}')
print(f'-> Value statistics match: {fbfm40_value_statistics_match}')
print(f'-> Processing manifest valid: {fbfm40_processing_manifest_valid}')
print(f'-> Reclassified FBFM40 raster valid: {fire_hazard_fbfm40_hazard_raster_valid}')
print(f'-> Detailed validation saved: {fire_hazard_fbfm40_validation_path}')
print(f'-> Formula validation saved: {fire_hazard_fbfm40_formula_validation_path}')
print(f'-> Validation summary saved: {fire_hazard_fbfm40_validation_summary_path}')
print('\n--- FBFM40 HAZARD VALIDATION SUMMARY ---')
display(fire_hazard_fbfm40_validation_summary)
print('\n--- FBFM40 FORMULA VALIDATION ---')
display(fire_hazard_fbfm40_formula_validation)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_hazard_raster_valid:
    raise ValueError(f'The reclassified FBFM40 '
        f'fuel-hazard raster failed '
        f'independent validation.\n\n'
        f'Grid valid: '
        f'{fbfm40_grid_valid}\nStructure '
        f'valid: '
        f'{fbfm40_structure_valid}\n'
        f'Score range valid: '
        f'{fbfm40_value_range_valid}\n'
        f'Formula valid: '
        f'{fire_hazard_fbfm40_formula_valid}\n'
        f'Formula mismatch pixels: '
        f'{formula_mismatch_pixels:,}\n'
        f'Pixel statistics match: '
        f'{fbfm40_pixel_statistics_match}\n'
        f'Value statistics match: '
        f'{fbfm40_value_statistics_match}\n'
        f'Processing manifest valid: '
        f'{fbfm40_processing_manifest_valid}\n'
        f'Validation error: '
        f'{fbfm40_validation_error}')
# Evaluate fire hazard fbfm40 hazard valid so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fbfm40_hazard_valid = fire_hazard_fbfm40_hazard_raster_valid
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The finalized FBFM40 hazard '
    'raster passed independent file,'
    ' grid, structure, value-range, '
    'study-area, '
    'processing-manifest, and '
    'class-to-score formula checks.')
print('Every scored source class was '
    'compared with the saved '
    'reclassification lookup, and '
    'no formula mismatches or '
    'unrecognized source classes '
    'were found.')
print('The validated FBFM40 raster is '
    'now ready to serve as the '
    'categorical surface-fuel input '
    'to the composite fuel-hazard '
    'component.')
print('\n=== RECLASSIFIED FBFM40 FUEL-HAZARD VALIDATION COMPLETE ===')


=== VALIDATING RECLASSIFIED FBFM40 FUEL-HAZARD RASTER ===
-> Output grid valid: True
-> Output structure valid: True
-> Valid output pixels: 15,475,232
-> Valid pixels outside study area: 0
-> Output score range: 0.0000 to 1.0000
-> Output mean score: 0.4357
-> Formula pixels checked: 15,475,232
-> Formula mismatch pixels: 0
-> Maximum formula difference: 0.00000000
-> Formula valid: True
-> Pixel statistics match: True
-> Value statistics match: True
-> Processing manifest valid: True
-> Reclassified FBFM40 raster valid: True
-> Detailed validation saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_hazard_validation.csv
-> Formula validation saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_formula_validation.csv
-> Validation summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_hazard_validation_summary.csv

--- FBFM40 HAZARD VALIDA

,COMPONENT_ID,PRODUCT,GRID_VALID,STRUCTURE_VALID,SCORE_RANGE_VALID,FORMULA_VALID,FORMULA_CHECKED_PIXELS,FORMULA_MISMATCH_PIXELS,MAXIMUM_FORMULA_DIFFERENCE,PIXEL_STATISTICS_MATCH,VALUE_STATISTICS_MATCH,PROCESSING_MANIFEST_VALID,VALID_OUTSIDE_STUDY_AREA,UNRECOGNIZED_SOURCE_PIXELS,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,VALIDATION_COMPLETE
0,FUEL_HAZARD,Reclassified FBFM40 Hazard Score,True,True,True,True,15475232,0,0.0,True,True,True,0,0,0.0,1.0,0.435666,True



--- FBFM40 FORMULA VALIDATION ---


,RECLASSIFICATION_METHOD,FORMULA_CHECKED_PIXELS,FORMULA_MATCHING_PIXELS,FORMULA_MISMATCH_PIXELS,MAXIMUM_ABSOLUTE_DIFFERENCE,MEAN_ABSOLUTE_DIFFERENCE,FORMULA_TOLERANCE,UNRECOGNIZED_SOURCE_PIXELS,BACKGROUND_WITH_VALID_OUTPUT,SCORED_CODE_WITHOUT_OUTPUT,SOURCE_NODATA_WITH_VALID_OUTPUT,VALID
0,expert_informed_ordinal_lookup,15475232,15475232,0,0.0,0.0,0.000001,0,0,0,0,True



NOTE:
The finalized FBFM40 hazard raster passed independent file, grid, structure, value-range, study-area, processing-manifest, and class-to-score formula checks.
Every scored source class was compared with the saved reclassification lookup, and no formula mismatches or unrecognized source classes were found.
The validated FBFM40 raster is now ready to serve as the categorical surface-fuel input to the composite fuel-hazard component.

=== RECLASSIFIED FBFM40 FUEL-HAZARD VALIDATION COMPLETE ===


### Finalizing FBFM40 Fuel-Model Hazard Product


In [108]:
print('=== FINALIZING FBFM40 FUEL-MODEL HAZARD PRODUCT ===')
# Prepare required FBFM40 product finalization inputs for the downstream processing or validation
# performed in this workflow stage.
required_fbfm40_product_finalization_inputs = ['fire_hazard_fbfm40_hazard_valid',
    'fire_hazard_fbfm40_hazard_raster_valid', 'fire_hazard_fbfm40_validation',
    'fire_hazard_fbfm40_validation_summary', 'fire_hazard_fbfm40_formula_validation',
    'fire_hazard_fbfm40_processing_manifest', 'fire_hazard_fbfm40_processing_output_path',
    'fire_hazard_fbfm40_reclassification_method', 'fire_hazard_fbfm40_reclassification_table_path',
    'fire_hazard_fbfm40_reclassification_policy_path',
    'fire_hazard_fbfm40_validation_path', 'fire_hazard_fbfm40_validation_summary_path',
    'fire_hazard_fbfm40_formula_validation_path', 'fire_hazard_fbfm40_score_minimum',
    'fire_hazard_fbfm40_score_maximum', 'fire_hazard_fbfm40_score_nodata',
    'fire_hazard_fbfm40_scored_pixels', 'fire_hazard_fbfm40_output_nodata_pixels',
    'fire_hazard_fbfm40_output_minimum', 'fire_hazard_fbfm40_output_maximum',
    'fire_hazard_fbfm40_output_mean', 'fire_hazard_fbfm40_valid_outside_study_area',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_target_crs',
    'fire_hazard_cell_size', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height']
# Identify missing FBFM40 product finalization inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fbfm40_product_finalization_inputs = [object_name for object_name in \
    required_fbfm40_product_finalization_inputs if object_name not in globals()]

# Stop execution if missing FBFM40 product finalization inputs remain unresolved before this
# workflow stage begins.
if missing_fbfm40_product_finalization_inputs:
    raise NameError(f'The following FBFM40 '
        f'product-finalization objects '
        f'are missing:\n'
        f'{missing_fbfm40_product_finalization_inputs}\n\n'
        f'Run Validate Reclassified Fuel '
        f'Hazard Raster before '
        f'finalizing the fuel-model '
        f'product.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_hazard_valid:
    raise ValueError('The FBFM40 hazard raster has not passed final validation.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_hazard_raster_valid:
    raise ValueError('The FBFM40 hazard-raster validation status is incomplete.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_fbfm40_validation.empty or not \
    fire_hazard_fbfm40_validation['VALID'].fillna(False).all():
    raise ValueError('The saved FBFM40 validation '
        'table does not contain a valid '
        'finalized product.')
# Build fire hazard fbfm40 hazard path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_hazard_path = Path(fire_hazard_fbfm40_processing_output_path)

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if not fire_hazard_fbfm40_hazard_path.exists() or not fire_hazard_fbfm40_hazard_path.is_file() or \
    fire_hazard_fbfm40_hazard_path.stat().st_size <= 0:
    raise FileNotFoundError(f'The finalized FBFM40 hazard '
        f'raster is unavailable:\n'
        f'{fire_hazard_fbfm40_hazard_path}')

# Open the file in a managed context so required content is processed and the resource closes
# cleanly.
with rasterio.open(fire_hazard_fbfm40_hazard_path) as final_fbfm40_source:
    # Evaluate fire hazard fbfm40 final product structure valid so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fbfm40_final_product_structure_valid = all([final_fbfm40_source.count == 1,
        final_fbfm40_source.width == fire_hazard_alignment_width,
        final_fbfm40_source.height == fire_hazard_alignment_height,
        final_fbfm40_source.crs is not None, final_fbfm40_source.crs == \
            rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
        final_fbfm40_source.dtypes[0] == 'float32', final_fbfm40_source.nodata == \
            fire_hazard_fbfm40_score_nodata])
    # Prepare fire hazard FBFM40 final product tags for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fbfm40_final_product_tags = final_fbfm40_source.tags()

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_final_product_structure_valid:
    raise ValueError('The finalized FBFM40 hazard raster failed its final structural check.')
# Build fire hazard fbfm40 final registry path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_final_registry_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_final_product_registry.csv'
# Build fire hazard fbfm40 final summary path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_final_summary_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_final_product_summary.csv'
# Build fire hazard fbfm40 final policy path used to read, cache, or save this workflow product.
fire_hazard_fbfm40_final_policy_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_final_product_policy.json'
# Build fire hazard fbfm40 completion marker path used to read, cache, or save this workflow
# product.
fire_hazard_fbfm40_completion_marker_path = fire_hazard_fuel_metadata_directory / \
    'fbfm40_product_complete.json'
# Prepare fire hazard FBFM40 final registry for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_final_registry = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'PRODUCT_ID': 'FBFM40_HAZARD_SCORE', 'PRODUCT_NAME': 'FBFM40 Surface-Fuel Hazard Score',
    'PRODUCT_ROLE': 'Categorical surface-fuel hazard input',
    'SOURCE_PRODUCT': 'LANDFIRE_FBFM40', 'LOCAL_PATH': str(fire_hazard_fbfm40_hazard_path),
    'RECLASSIFICATION_METHOD': fire_hazard_fbfm40_reclassification_method,
    'SCORE_MINIMUM': fire_hazard_fbfm40_output_minimum,
    'SCORE_MAXIMUM': fire_hazard_fbfm40_output_maximum,
    'SCORE_MEAN': fire_hazard_fbfm40_output_mean, 'VALID_PIXELS': fire_hazard_fbfm40_scored_pixels,
    'NODATA_PIXELS': fire_hazard_fbfm40_output_nodata_pixels,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_fbfm40_valid_outside_study_area,
    'FILE_SIZE_BYTES': fire_hazard_fbfm40_hazard_path.stat().st_size,
    'FILE_SIZE_MB': fire_hazard_fbfm40_hazard_path.stat().st_size / 1024 ** 2,
    'USE_IN_COMPOSITE_FUEL_HAZARD': True, 'VALID': True}])
# Assemble fire hazard FBFM40 final summary into a table for QA review and downstream validation.
fire_hazard_fbfm40_final_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'PRODUCT_ID': 'FBFM40_HAZARD_SCORE', 'STATUS': 'Complete',
    'PRIMARY_OUTPUT': str(fire_hazard_fbfm40_hazard_path),
    'SOURCE_DATA': 'LANDFIRE FBFM40', 'PROCESSING_METHOD': \
        fire_hazard_fbfm40_reclassification_method,
    'OUTPUT_SCORE_MINIMUM': fire_hazard_fbfm40_score_minimum,
    'OUTPUT_SCORE_MAXIMUM': fire_hazard_fbfm40_score_maximum,
    'ACTUAL_SCORE_MINIMUM': fire_hazard_fbfm40_output_minimum,
    'ACTUAL_SCORE_MAXIMUM': fire_hazard_fbfm40_output_maximum,
    'ACTUAL_SCORE_MEAN': fire_hazard_fbfm40_output_mean,
    'VALID_PIXELS': fire_hazard_fbfm40_scored_pixels,
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'FORMULA_VALID': bool(fire_hazard_fbfm40_formula_validation['VALID'].iloc[0]),
    'VALIDATION_COMPLETE': fire_hazard_fbfm40_hazard_valid,
    'READY_FOR_CANOPY_INTEGRATION': True}])
# Collect fire hazard FBFM40 final policy in one configuration object so downstream steps use the
# same processing rules.
# Define the final policy record for the completed FBFM40 fuel-hazard product.
fire_hazard_fbfm40_final_policy = {
    # Identify the component, output product, source, and completion status.
    'component_id': 'FUEL_HAZARD',
    'product_id': 'FBFM40_HAZARD_SCORE',
    'status': 'complete',
    'source_product': 'LANDFIRE_FBFM40',
    'primary_output': str(
        fire_hazard_fbfm40_hazard_path
    ),

    # Record the reclassification method and interpretation of the hazard scores.
    'processing': {
        'method': fire_hazard_fbfm40_reclassification_method,
        'score_range': [
            float(fire_hazard_fbfm40_score_minimum),
            float(fire_hazard_fbfm40_score_maximum),
        ],
        'output_nodata': float(
            fire_hazard_fbfm40_score_nodata
        ),
        'interpretation': (
            'Higher values represent greater relative surface-fuel hazard.'
        ),
        'modeling_note': (
            'Scores are project-specific '
            'relative hazard values rather '
            'than official LANDFIRE hazard '
            'ratings.'
        ),
    },

    # Record the target-grid properties of the completed hazard raster.
    'target_grid': {
        'crs': fire_hazard_target_crs,
        'cell_size_meters': float(
            fire_hazard_cell_size
        ),
        'width': int(
            fire_hazard_alignment_width
        ),
        'height': int(
            fire_hazard_alignment_height
        ),
    },

    # Record final pixel counts and hazard-score statistics.
    'statistics': {
        'valid_pixels': int(
            fire_hazard_fbfm40_scored_pixels
        ),
        'nodata_pixels': int(
            fire_hazard_fbfm40_output_nodata_pixels
        ),
        'minimum': float(
            fire_hazard_fbfm40_output_minimum
        ),
        'maximum': float(
            fire_hazard_fbfm40_output_maximum
        ),
        'mean': float(
            fire_hazard_fbfm40_output_mean
        ),
        'valid_outside_study_area': int(
            fire_hazard_fbfm40_valid_outside_study_area
        ),
    },

    # Record the QA and policy files that document the completed product.
    'validation': {
        'product_validation': str(
            fire_hazard_fbfm40_validation_path
        ),
        'validation_summary': str(
            fire_hazard_fbfm40_validation_summary_path
        ),
        'formula_validation': str(
            fire_hazard_fbfm40_formula_validation_path
        ),
        'reclassification_table': str(
            fire_hazard_fbfm40_reclassification_table_path
        ),
        'processing_policy': str(
            fire_hazard_fbfm40_reclassification_policy_path
        ),
        'valid': True,
    },

    # Define how the completed FBFM40 product will be used in the composite model.
    'model_role': {
        'use_in_composite_fuel_hazard': True,
        'role': 'Categorical surface-fuel hazard input',
    },
}
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_final_registry.to_csv(fire_hazard_fbfm40_final_registry_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fbfm40_final_summary.to_csv(fire_hazard_fbfm40_final_summary_path, index=False)

# Prepare with fire hazard FBFM40 final policy path.open so this workflow stage has the values
# required for downstream spatial processing and QA.
with fire_hazard_fbfm40_final_policy_path.open('w', encoding='utf-8') as final_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fbfm40_final_policy, final_policy_file, indent=2)
# Prepare fire hazard FBFM40 completion marker for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_completion_marker = {'component_id': 'FUEL_HAZARD',
    'product_id': 'FBFM40_HAZARD_SCORE', 'status': 'complete',
    'primary_output': str(fire_hazard_fbfm40_hazard_path),
    'validation_complete': True, 'ready_for_canopy_integration': True,
    'final_registry': str(fire_hazard_fbfm40_final_registry_path),
    'final_summary': str(fire_hazard_fbfm40_final_summary_path),
    'final_policy': str(fire_hazard_fbfm40_final_policy_path)}

# Document this operation so its role in the current fuel-hazard workflow is clear before processing
# continues.
with fire_hazard_fbfm40_completion_marker_path.open('w',
    encoding='utf-8') as completion_marker_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fbfm40_completion_marker, completion_marker_file, indent=2)
# Build fire hazard fuel product registry path used to read, cache, or save this workflow product.
fire_hazard_fuel_product_registry_path = fire_hazard_fuel_metadata_directory / \
    'fuel_product_registry.csv'

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_fuel_product_registry_path.exists():
    # Prepare fire hazard fuel product registry for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fuel_product_registry = pd.read_csv(fire_hazard_fuel_product_registry_path)
else:
    # Prepare fire hazard fuel product registry for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fuel_product_registry = pd.DataFrame()

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_product_registry.empty:
    # Prepare fire hazard fuel product registry for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fuel_product_registry = \
        fire_hazard_fuel_product_registry[fire_hazard_fuel_product_registry['PRODUCT_ID'].astype \
        (str).str.upper() != 'FBFM40_HAZARD_SCORE'].copy()
# Prepare fire hazard FBFM40 registry record for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_registry_record = {'COMPONENT_ID': 'FUEL_HAZARD',
    'PRODUCT_ID': 'FBFM40_HAZARD_SCORE', 'PRODUCT_NAME': 'FBFM40 Surface-Fuel Hazard Score',
    'PRIMARY_OUTPUT_PATH': str(fire_hazard_fbfm40_hazard_path),
    'PRODUCT_ROLE': 'Categorical surface-fuel hazard input',
    'SCORE_MINIMUM': fire_hazard_fbfm40_score_minimum,
    'SCORE_MAXIMUM': fire_hazard_fbfm40_score_maximum,
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'VALIDATION_COMPLETE': True, 'PRODUCT_COMPLETE': True,
    'READY_FOR_COMPOSITE_FUEL_HAZARD': True}
# Prepare fire hazard fuel product registry for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_product_registry = pd.concat([fire_hazard_fuel_product_registry,
    pd.DataFrame([fire_hazard_fbfm40_registry_record])],
    ignore_index=True, sort=False).reset_index(drop=True)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_product_registry.to_csv(fire_hazard_fuel_product_registry_path, index=False)
# Prepare fire hazard FBFM40 product finalized for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fbfm40_product_finalized = all([fire_hazard_fbfm40_hazard_valid,
    fire_hazard_fbfm40_final_product_structure_valid,
    fire_hazard_fbfm40_hazard_path.exists(), fire_hazard_fbfm40_final_registry_path.exists(),
    fire_hazard_fbfm40_final_summary_path.exists(),
    fire_hazard_fbfm40_final_policy_path.exists(),
    fire_hazard_fbfm40_completion_marker_path.exists(),
    fire_hazard_fuel_product_registry_path.exists()])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_product_finalized:
    raise ValueError('The FBFM40 fuel-model hazard product did not pass final completion checks.')
print('-> Product ID: FBFM40_HAZARD_SCORE')
print('-> Product status: Complete')
print(f'-> Final raster: {fire_hazard_fbfm40_hazard_path}')
print(f'-> Valid pixels: {fire_hazard_fbfm40_scored_pixels:,}')
print(f'-> Score range: '
    f'{fire_hazard_fbfm40_output_minimum:.4f} '
    f'to '
    f'{fire_hazard_fbfm40_output_maximum:.4f}')
print(f'-> Mean score: {fire_hazard_fbfm40_output_mean:.4f}')
print(f"-> Formula valid: {bool(fire_hazard_fbfm40_formula_validation['VALID'].iloc[0])}")
print(f'-> Product finalized: {fire_hazard_fbfm40_product_finalized}')
print(f'-> Final registry saved: {fire_hazard_fbfm40_final_registry_path}')
print(f'-> Final summary saved: {fire_hazard_fbfm40_final_summary_path}')
print(f'-> Final policy saved: {fire_hazard_fbfm40_final_policy_path}')
print(f'-> Completion marker saved: {fire_hazard_fbfm40_completion_marker_path}')
print('\n--- FINAL FBFM40 PRODUCT SUMMARY ---')
display(fire_hazard_fbfm40_final_summary)
print('\n--- FBFM40 FINAL PRODUCT REGISTRY ---')
display(fire_hazard_fbfm40_final_registry)
print('\n--- FUEL PRODUCT REGISTRY SAMPLE ---')
display(fire_hazard_fuel_product_registry.head(5))
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The reclassified FBFM40 '
    'surface-fuel hazard product is '
    'fully processed, validated, '
    'documented, and registered.')
print('This raster will later be '
    'combined with the normalized '
    'canopy-cover and '
    'canopy-bulk-density hazard '
    'layers.')
print('The next workflow phase configures normalization for the two continuous canopy variables.')
print('\n=== FBFM40 FUEL-MODEL HAZARD PRODUCT FINALIZED ===')


=== FINALIZING FBFM40 FUEL-MODEL HAZARD PRODUCT ===
-> Product ID: FBFM40_HAZARD_SCORE
-> Product status: Complete
-> Final raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\normalized_sources\fbfm40_fuel_hazard_score.tif
-> Valid pixels: 15,475,232
-> Score range: 0.0000 to 1.0000
-> Mean score: 0.4357
-> Formula valid: True
-> Product finalized: True
-> Final registry saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_final_product_registry.csv
-> Final summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_final_product_summary.csv
-> Final policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_final_product_policy.json
-> Completion marker saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fbfm40_product_complete.json

--- FINAL FBFM40 PRODUCT SUMMARY ---

,COMPONENT_ID,PRODUCT_ID,STATUS,PRIMARY_OUTPUT,SOURCE_DATA,PROCESSING_METHOD,OUTPUT_SCORE_MINIMUM,OUTPUT_SCORE_MAXIMUM,ACTUAL_SCORE_MINIMUM,ACTUAL_SCORE_MAXIMUM,ACTUAL_SCORE_MEAN,VALID_PIXELS,TARGET_CRS,CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,FORMULA_VALID,VALIDATION_COMPLETE,READY_FOR_CANOPY_INTEGRATION
0,FUEL_HAZARD,FBFM40_HAZARD_SCORE,Complete,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,LANDFIRE FBFM40,expert_informed_ordinal_lookup,0.0,1.0,0.0,1.0,0.435666,15475232,EPSG:26912,30,9454,14467,True,True,True



--- FBFM40 FINAL PRODUCT REGISTRY ---


,COMPONENT_ID,PRODUCT_ID,PRODUCT_NAME,PRODUCT_ROLE,SOURCE_PRODUCT,LOCAL_PATH,RECLASSIFICATION_METHOD,SCORE_MINIMUM,SCORE_MAXIMUM,SCORE_MEAN,VALID_PIXELS,NODATA_PIXELS,VALID_OUTSIDE_STUDY_AREA,FILE_SIZE_BYTES,FILE_SIZE_MB,USE_IN_COMPOSITE_FUEL_HAZARD,VALID
0,FUEL_HAZARD,FBFM40_HAZARD_SCORE,FBFM40 Surface-Fuel Hazard Score,Categorical surface-fuel hazard input,LANDFIRE_FBFM40,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,expert_informed_ordinal_lookup,0.0,1.0,0.435666,15475232,121295786,0,15032042,14.335672,True,True



--- FUEL PRODUCT REGISTRY SAMPLE ---


,COMPONENT_ID,PRODUCT_ID,PRODUCT_NAME,PRIMARY_OUTPUT_PATH,PRODUCT_ROLE,SCORE_MINIMUM,SCORE_MAXIMUM,TARGET_CRS,CELL_SIZE_METERS,VALIDATION_COMPLETE,PRODUCT_COMPLETE,READY_FOR_COMPOSITE_FUEL_HAZARD
0,FUEL_HAZARD,FBFM40_HAZARD_SCORE,FBFM40 Surface-Fuel Hazard Score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Categorical surface-fuel hazard input,0.0,1.0,EPSG:26912,30,True,True,True



NOTE:
The reclassified FBFM40 surface-fuel hazard product is fully processed, validated, documented, and registered.
This raster will later be combined with the normalized canopy-cover and canopy-bulk-density hazard layers.
The next workflow phase configures normalization for the two continuous canopy variables.

=== FBFM40 FUEL-MODEL HAZARD PRODUCT FINALIZED ===


## Canopy Normalization and Composite Fuel Hazard


### Configuring Canopy Variable Normalization


In [109]:
print('=== CONFIGURING CANOPY VARIABLE NORMALIZATION ===')

# Reclassify LANDFIRE FBFM40 fuel models into normalized hazard
# scores based on expected fire behavior.
# Prepare required canopy normalization inputs for the downstream processing or validation performed
# in this workflow stage.
required_canopy_normalization_inputs = ['fire_hazard_fbfm40_product_finalized',
    'fire_hazard_fbfm40_hazard_path', 'fire_hazard_aligned_fuel_layers_valid',
    'fire_hazard_fuel_alignment_validation', 'fire_hazard_fuel_aligned_paths',
    'fire_hazard_canopy_cover_hazard_score_path', \
        'fire_hazard_canopy_bulk_density_hazard_score_path',
    'fire_hazard_fuel_score_profile', 'fire_hazard_fuel_score_minimum',
    'fire_hazard_fuel_score_maximum', 'fire_hazard_fuel_score_nodata',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_fuel_normalization_manifest_path',
    'fire_hazard_fuel_reuse_existing', 'fire_hazard_fuel_overwrite',
    'fire_hazard_fuel_processing_window_size', 'fire_hazard_fuel_progress_interval',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size']
# Identify missing canopy normalization inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_canopy_normalization_inputs = [object_name for object_name in \
    required_canopy_normalization_inputs if object_name not in globals()]

# Stop execution if missing canopy normalization inputs remain unresolved before this workflow stage
# begins.
if missing_canopy_normalization_inputs:
    raise NameError(f'The following '
        f'canopy-normalization objects '
        f'are missing:\n'
        f'{missing_canopy_normalization_inputs}\n\n'
        f'Run the aligned-fuel '
        f'validation and finalize the '
        f'FBFM40 hazard product before '
        f'configuring canopy '
        f'normalization.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_product_finalized:
    raise ValueError('The FBFM40 fuel-model hazard product has not been finalized.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_aligned_fuel_layers_valid:
    raise ValueError('The aligned LANDFIRE fuel layers have not passed independent validation.')

# Normalize canopy cover to represent the contribution of
# continuous forest fuels to fire behavior.
# Define the fire hazard canopy source ids inputs required by this workflow stage.
fire_hazard_required_canopy_source_ids = ['LANDFIRE_CANOPY_COVER', 'LANDFIRE_CANOPY_BULK_DENSITY']
# Identify missing canopy source IDs so unavailable prerequisites are caught before this workflow
# stage runs.
missing_canopy_source_ids = [source_id for source_id in fire_hazard_required_canopy_source_ids if \
    source_id not in fire_hazard_fuel_aligned_paths]

# Stop execution when required canopy source ids inputs are unavailable.
if missing_canopy_source_ids:
    raise KeyError(f'The aligned-fuel path '
        f'dictionary is missing required '
        f'canopy products:\n'
        f'{missing_canopy_source_ids}')
# Evaluate fire hazard canopy alignment validation so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_canopy_alignment_validation = \
    fire_hazard_fuel_alignment_validation[fire_hazard_fuel_alignment_validation['SOURCE_ID'].isin \
    (fire_hazard_required_canopy_source_ids)].copy().reset_index(drop=True)

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(fire_hazard_canopy_alignment_validation) != 2:
    raise ValueError('Exactly two aligned canopy validation records are required.')

# Stop execution if the raster data type does not match the output specification required by this
# stage.
if fire_hazard_canopy_alignment_validation['VALID'].dtype == object:
    # Evaluate fire hazard canopy alignment validation so invalid inputs or outputs can be rejected
    # before continuing.
    fire_hazard_canopy_alignment_validation['VALID'] = \
        fire_hazard_canopy_alignment_validation['VALID'].astype(str).str.strip().str.lower().map( \
        {'true': True,
        'false': False})

# Stop execution if required validation fields contain null values that would make the QA result
# unreliable.
if fire_hazard_canopy_alignment_validation['VALID'].isna().any() or not \
    fire_hazard_canopy_alignment_validation['VALID'].all():
    raise ValueError('One or more aligned canopy '
        'products failed the preceding '
        'independent validation.')
# Build fire hazard canopy cover input path used to read, cache, or save this workflow product.
fire_hazard_canopy_cover_input_path = Path(fire_hazard_fuel_aligned_paths['LANDFIRE_CANOPY_COVER'])

# Normalize canopy bulk density to represent potential crown-fire
# propagation through vertically connected fuels.
# Build fire hazard canopy bulk density input path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_bulk_density_input_path = \
    Path(fire_hazard_fuel_aligned_paths['LANDFIRE_CANOPY_BULK_DENSITY'])

# Iterate through  so each required item receives the same processing and QA checks.
for source_name, source_path in {'Canopy cover': fire_hazard_canopy_cover_input_path,
    'Canopy bulk density': fire_hazard_canopy_bulk_density_input_path}.items():

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not source_path.exists() or not source_path.is_file() or source_path.stat().st_size <= 0:
        raise FileNotFoundError(f'The aligned {source_name} raster is unavailable:\n{source_path}')
# Build fire hazard canopy cover output path used to read, cache, or save this workflow product.
fire_hazard_canopy_cover_output_path = Path(fire_hazard_canopy_cover_hazard_score_path)
# Build fire hazard canopy bulk density output path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_bulk_density_output_path = \
    Path(fire_hazard_canopy_bulk_density_hazard_score_path)
# Build fire hazard canopy cover temporary path used to read, cache, or save this workflow product.
fire_hazard_canopy_cover_temporary_path = fire_hazard_canopy_cover_output_path.parent / \
    (fire_hazard_canopy_cover_output_path.stem + '.part.tif')
# Build fire hazard canopy bulk density temporary path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_bulk_density_temporary_path = \
    fire_hazard_canopy_bulk_density_output_path.parent / \
    (fire_hazard_canopy_bulk_density_output_path.stem + '.part.tif')
# Evaluate canopy cover validation record so invalid inputs or outputs can be rejected before
# continuing.
canopy_cover_validation_record = \
    fire_hazard_canopy_alignment_validation[fire_hazard_canopy_alignment_validation['SOURCE_ID'] \
    == 'LANDFIRE_CANOPY_COVER']
# Evaluate canopy bulk density validation record so invalid inputs or outputs can be rejected before
# continuing.
canopy_bulk_density_validation_record = \
    fire_hazard_canopy_alignment_validation[fire_hazard_canopy_alignment_validation['SOURCE_ID'] \
    == 'LANDFIRE_CANOPY_BULK_DENSITY']

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(canopy_cover_validation_record) != 1:
    raise ValueError('Exactly one canopy-cover validation record is required.')

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if len(canopy_bulk_density_validation_record) != 1:
    raise ValueError('Exactly one canopy-bulk-density validation record is required.')
# Set canopy cover source minimum as an explicit model or validation parameter used consistently in
# downstream calculations.
canopy_cover_source_minimum = float(canopy_cover_validation_record.iloc[0]['VALUE_MINIMUM'])
# Set canopy cover source maximum as an explicit model or validation parameter used consistently in
# downstream calculations.
canopy_cover_source_maximum = float(canopy_cover_validation_record.iloc[0]['VALUE_MAXIMUM'])
# Set canopy bulk density source minimum as an explicit model or validation parameter used
# consistently in downstream calculations.
canopy_bulk_density_source_minimum = \
    float(canopy_bulk_density_validation_record.iloc[0]['VALUE_MINIMUM'])
# Set canopy bulk density source maximum as an explicit model or validation parameter used
# consistently in downstream calculations.
canopy_bulk_density_source_maximum = \
    float(canopy_bulk_density_validation_record.iloc[0]['VALUE_MAXIMUM'])
# Evaluate canopy cover range valid so invalid inputs or outputs can be rejected before continuing.
canopy_cover_range_valid = all([np.isfinite(canopy_cover_source_minimum),
    np.isfinite(canopy_cover_source_maximum), canopy_cover_source_maximum > \
        canopy_cover_source_minimum])
# Evaluate canopy bulk density range valid so invalid inputs or outputs can be rejected before
# continuing.
canopy_bulk_density_range_valid = all([np.isfinite(canopy_bulk_density_source_minimum),
    np.isfinite(canopy_bulk_density_source_maximum),
    canopy_bulk_density_source_maximum > canopy_bulk_density_source_minimum])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not canopy_cover_range_valid:
    raise ValueError('The aligned canopy-cover '
        'raster does not contain a '
        'valid continuous value range.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not canopy_bulk_density_range_valid:
    raise ValueError('The aligned '
        'canopy-bulk-density raster '
        'does not contain a valid '
        'continuous value range.')
# Prepare fire hazard canopy normalization method for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_normalization_method = 'direct_min_max'
# Set fire hazard canopy score minimum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_canopy_score_minimum = fire_hazard_fuel_score_minimum
# Set fire hazard canopy score maximum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_canopy_score_maximum = fire_hazard_fuel_score_maximum
# Set fire hazard canopy score NoData as an explicit model or validation parameter used consistently
# in downstream calculations.
fire_hazard_canopy_score_nodata = fire_hazard_fuel_score_nodata
# Capture fire hazard canopy normalization bounds so source coverage can be compared with the
# required analysis extent.
fire_hazard_canopy_normalization_bounds = {'LANDFIRE_CANOPY_COVER': {'lower_bound': 0.0,
    'upper_bound': canopy_cover_source_maximum, 'observed_minimum': canopy_cover_source_minimum,
    'observed_maximum': canopy_cover_source_maximum},
    'LANDFIRE_CANOPY_BULK_DENSITY': {'lower_bound': 0.0,
    'upper_bound': canopy_bulk_density_source_maximum,
    'observed_minimum': canopy_bulk_density_source_minimum,
    'observed_maximum': canopy_bulk_density_source_maximum}}

# Iterate through fire hazard canopy normalization bounds.items so each required item receives the
# same processing and QA checks.
for source_id, normalization_bounds in fire_hazard_canopy_normalization_bounds.items():
    # Prepare lower bound for the downstream processing or validation performed in this workflow
    # stage.
    lower_bound = float(normalization_bounds['lower_bound'])
    # Prepare upper bound for the downstream processing or validation performed in this workflow
    # stage.
    upper_bound = float(normalization_bounds['upper_bound'])

    # Stop execution if this validation condition is not satisfied before dependent processing
    # continues.
    if not np.isfinite(lower_bound) or not np.isfinite(upper_bound) or upper_bound <= lower_bound:
        raise ValueError(f'Invalid canopy normalization '
            f'bounds for {source_id}:\nLower '
            f'bound: {lower_bound}\nUpper '
            f'bound: {upper_bound}')
# Prepare fire hazard canopy normalization formula for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_normalization_formula = '(value - lower_bound) / (upper_bound - lower_bound)'
# Prepare fire hazard canopy clip normalized scores for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_canopy_clip_normalized_scores = True
# Collect fire hazard canopy normalization configuration in one configuration object so downstream
# steps use the same processing rules.
fire_hazard_canopy_normalization_configuration = {'LANDFIRE_CANOPY_COVER': {'product_name': \
    'Forest Canopy Cover',
    'input_path': fire_hazard_canopy_cover_input_path,
    'temporary_path': fire_hazard_canopy_cover_temporary_path,
    'output_path': fire_hazard_canopy_cover_output_path,
    'lower_bound': fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_COVER']['lower_bound'],
    'upper_bound': fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_COVER']['upper_bound']},
    'LANDFIRE_CANOPY_BULK_DENSITY': {'product_name': 'Forest Canopy Bulk Density',
    'input_path': fire_hazard_canopy_bulk_density_input_path,
    'temporary_path': fire_hazard_canopy_bulk_density_temporary_path,
    'output_path': fire_hazard_canopy_bulk_density_output_path,
    'lower_bound': \
        fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_BULK_DENSITY']['lower_bound'],
    'upper_bound': \
        fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_BULK_DENSITY']['upper_bound']}}
# Prepare fire hazard canopy normalized profile so raster outputs inherit the required grid, CRS,
# data type, and NoData metadata.
fire_hazard_canopy_normalized_profile = fire_hazard_fuel_score_profile.copy()
# Update fire hazard canopy normalized profile with the values produced by the current processing
# step.
fire_hazard_canopy_normalized_profile.update({'driver': 'GTiff',
    'dtype': 'float32', 'count': 1, 'nodata': fire_hazard_canopy_score_nodata,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': 'deflate', 'predictor': 3, 'tiled': True,
    'BIGTIFF': 'IF_SAFER'})
# Calculate fire hazard canopy process by window so raster processing covers the analysis grid in
# controlled blocks.
fire_hazard_canopy_process_by_window = True
# Calculate fire hazard canopy processing window size so raster processing covers the analysis grid
# in controlled blocks.
fire_hazard_canopy_processing_window_size = fire_hazard_fuel_processing_window_size
# Prepare fire hazard canopy progress interval for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_progress_interval = fire_hazard_fuel_progress_interval
# Prepare fire hazard canopy reuse existing for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_reuse_existing = fire_hazard_fuel_reuse_existing
# Prepare fire hazard canopy overwrite for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_canopy_overwrite = fire_hazard_fuel_overwrite
# Evaluate fire hazard canopy minimum valid pixels so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_canopy_minimum_valid_pixels = 1
# Set fire hazard canopy value tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_canopy_value_tolerance = 1e-06
# Set fire hazard canopy formula tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_canopy_formula_tolerance = 1e-05
# Prepare fire hazard canopy require study area clip for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_canopy_require_study_area_clip = True
# Build fire hazard canopy normalization policy path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_normalization_policy_path = fire_hazard_fuel_metadata_directory / \
    'canopy_normalization_policy.json'
# Build fire hazard canopy normalization summary path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_normalization_summary_path = fire_hazard_fuel_metadata_directory / \
    'canopy_normalization_configuration_summary.csv'
# Build fire hazard canopy normalization validation path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_normalization_validation_path = fire_hazard_fuel_metadata_directory / \
    'canopy_normalization_validation.csv'
# Build fire hazard canopy normalization validation summary path used to read, cache, or save this
# workflow product.
fire_hazard_canopy_normalization_validation_summary_path = fire_hazard_fuel_metadata_directory / \
    'canopy_normalization_validation_summary.csv'
# Collect fire hazard canopy normalization policy in one configuration object so downstream steps
# use the same processing rules.
fire_hazard_canopy_normalization_policy = (
    {'component_id': 'FUEL_HAZARD', 'workflow_phase': 'CANOPY_NORMALIZATION',
        'method': fire_hazard_canopy_normalization_method,
        'formula': fire_hazard_canopy_normalization_formula,
        'interpretation': 'Higher normalized scores '
        'represent greater relative '
        'canopy-fuel hazard.', 'score_range': [float(fire_hazard_canopy_score_minimum),
            float(fire_hazard_canopy_score_maximum)], 'output_nodata': \
                float(fire_hazard_canopy_score_nodata), 'clip_scores': \
                bool(fire_hazard_canopy_clip_normalized_scores), 'products': {source_id: \
                {'product_name': configuration['product_name'],
            'input_path': str(configuration['input_path']),
            'output_path': str(configuration['output_path']),
            'lower_bound': float(configuration['lower_bound']),
            'upper_bound': float(configuration['upper_bound']),
            'observed_minimum': \
                float(fire_hazard_canopy_normalization_bounds[source_id]['observed_minimum']),
            'observed_maximum': \
                float(fire_hazard_canopy_normalization_bounds[source_id]['observed_maximum'])} \
                for source_id,
            configuration in fire_hazard_canopy_normalization_configuration.items()}, \
                'target_grid': {'crs': fire_hazard_target_crs,
            'cell_size_meters': float(fire_hazard_cell_size),
            'width': int(fire_hazard_alignment_width), 'height': \
                int(fire_hazard_alignment_height)}, 'processing': {'process_by_window': \
                fire_hazard_canopy_process_by_window,
            'window_size': int(fire_hazard_canopy_processing_window_size),
            'progress_interval': int(fire_hazard_canopy_progress_interval),
            'reuse_existing': bool(fire_hazard_canopy_reuse_existing),
            'overwrite_existing': bool(fire_hazard_canopy_overwrite)}, 'validation': \
                {'minimum_valid_pixels': int(fire_hazard_canopy_minimum_valid_pixels),
            'value_tolerance': float(fire_hazard_canopy_value_tolerance),
            'formula_tolerance': float(fire_hazard_canopy_formula_tolerance),
            'require_study_area_clip': bool(fire_hazard_canopy_require_study_area_clip)}}
)

# Prepare with fire hazard canopy normalization policy path.open so this workflow stage has the
# values required for downstream spatial processing and QA.
with fire_hazard_canopy_normalization_policy_path.open('w', encoding='utf-8') as canopy_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_canopy_normalization_policy, canopy_policy_file, indent=2)
# Assemble fire hazard canopy normalization summary into a table for QA review and downstream
# validation.
fire_hazard_canopy_normalization_summary = pd.DataFrame([{'SOURCE_ID': source_id,
    'PRODUCT_NAME': configuration['product_name'],
    'INPUT_PATH': str(configuration['input_path']),
    'OUTPUT_PATH': str(configuration['output_path']),
    'METHOD': fire_hazard_canopy_normalization_method,
    'LOWER_BOUND': configuration['lower_bound'], 'UPPER_BOUND': configuration['upper_bound'],
    'OBSERVED_MINIMUM': fire_hazard_canopy_normalization_bounds[source_id]['observed_minimum'],
    'OBSERVED_MAXIMUM': fire_hazard_canopy_normalization_bounds[source_id]['observed_maximum'],
    'OUTPUT_SCORE_MINIMUM': fire_hazard_canopy_score_minimum,
    'OUTPUT_SCORE_MAXIMUM': fire_hazard_canopy_score_maximum,
    'OUTPUT_NODATA': fire_hazard_canopy_score_nodata,
    'READY_FOR_NORMALIZATION': True} for source_id,
    configuration in fire_hazard_canopy_normalization_configuration.items()])
# Export the table so this workflow result is available to later phases.
fire_hazard_canopy_normalization_summary.to_csv(fire_hazard_canopy_normalization_summary_path,
    index=False)
# Record whether fire hazard canopy normalization configured satisfies the checks required before
# the workflow advances.
fire_hazard_canopy_normalization_configured = all([fire_hazard_fbfm40_product_finalized,
    fire_hazard_aligned_fuel_layers_valid, len(fire_hazard_canopy_normalization_configuration) == 2,
    canopy_cover_range_valid, canopy_bulk_density_range_valid,
    fire_hazard_canopy_normalization_policy_path.exists(),
    fire_hazard_canopy_normalization_summary_path.exists(),
    fire_hazard_canopy_normalization_summary['READY_FOR_NORMALIZATION'].all()])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_canopy_normalization_configured:
    raise ValueError('The canopy-variable normalization configuration did not pass final checks.')
print(f'-> Normalization method: {fire_hazard_canopy_normalization_method}')
print(f'-> Canopy products configured: {len(fire_hazard_canopy_normalization_configuration)}')
print(f"-> Canopy-cover bounds: "
    f"{fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_COVER']['lower_bound']:.4f} "
    f"to "
    f"{fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_COVER']['upper_bound']:.4f}")
print(f"-> Canopy-bulk-density bounds: "
    f"{fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_BULK_DENSITY']['lower_bound']:.4f} "
    f"to "
    f"{fire_hazard_canopy_normalization_bounds['LANDFIRE_CANOPY_BULK_DENSITY']['upper_bound']:.4f}")
print(f'-> Output score range: '
    f'{fire_hazard_canopy_score_minimum:.1f} '
    f'to '
    f'{fire_hazard_canopy_score_maximum:.1f}')
print(f'-> Reuse existing outputs: {fire_hazard_canopy_reuse_existing}')
print(f'-> Overwrite existing outputs: {fire_hazard_canopy_overwrite}')
print(f'-> Canopy normalization configured: {fire_hazard_canopy_normalization_configured}')
print(f'-> Policy saved: {fire_hazard_canopy_normalization_policy_path}')
print(f'-> Summary saved: {fire_hazard_canopy_normalization_summary_path}')
print('\n--- CANOPY NORMALIZATION CONFIGURATION SUMMARY ---')
display(fire_hazard_canopy_normalization_summary)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('Canopy cover and canopy bulk density are configured for direct min-max normalization.')
print('Higher source values will produce higher relative canopy-fuel hazard scores.')
print('Values below the configured '
    'lower bound will be clipped to '
    'zero, and values above the '
    'upper bound will be clipped to '
    'one.')
print('No canopy raster values were transformed in this configuration step.')
print('\n=== CANOPY VARIABLE NORMALIZATION CONFIGURED ===')


=== CONFIGURING CANOPY VARIABLE NORMALIZATION ===
-> Normalization method: direct_min_max
-> Canopy products configured: 2
-> Canopy-cover bounds: 0.0000 to 85.0000
-> Canopy-bulk-density bounds: 0.0000 to 38.0000
-> Output score range: 0.0 to 1.0
-> Reuse existing outputs: True
-> Overwrite existing outputs: False
-> Canopy normalization configured: True
-> Policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\canopy_normalization_policy.json
-> Summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\canopy_normalization_configuration_summary.csv

--- CANOPY NORMALIZATION CONFIGURATION SUMMARY ---


,SOURCE_ID,PRODUCT_NAME,INPUT_PATH,OUTPUT_PATH,METHOD,LOWER_BOUND,UPPER_BOUND,OBSERVED_MINIMUM,OBSERVED_MAXIMUM,OUTPUT_SCORE_MINIMUM,OUTPUT_SCORE_MAXIMUM,OUTPUT_NODATA,READY_FOR_NORMALIZATION
0,LANDFIRE_CANOPY_COVER,Forest Canopy Cover,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,direct_min_max,0.0,85.0,0.0,85.0,0.0,1.0,-9999.0,True
1,LANDFIRE_CANOPY_BULK_DENSITY,Forest Canopy Bulk Density,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,direct_min_max,0.0,38.0,0.0,38.0,0.0,1.0,-9999.0,True



NOTE:
Canopy cover and canopy bulk density are configured for direct min-max normalization.
Higher source values will produce higher relative canopy-fuel hazard scores.
Values below the configured lower bound will be clipped to zero, and values above the upper bound will be clipped to one.
No canopy raster values were transformed in this configuration step.

=== CANOPY VARIABLE NORMALIZATION CONFIGURED ===


### Normalizing Forest Canopy Cover


In [110]:
print('=== NORMALIZING FOREST CANOPY COVER ===')
# Prepare required canopy cover normalization inputs for the downstream processing or validation
# performed in this workflow stage.
required_canopy_cover_normalization_inputs = ['fire_hazard_canopy_normalization_configured',
    'fire_hazard_canopy_normalization_configuration',
    'fire_hazard_canopy_normalized_profile', 'fire_hazard_canopy_score_minimum',
    'fire_hazard_canopy_score_maximum', 'fire_hazard_canopy_score_nodata',
    'fire_hazard_canopy_clip_normalized_scores', 'fire_hazard_canopy_minimum_valid_pixels',
    'fire_hazard_canopy_value_tolerance', 'fire_hazard_canopy_reuse_existing',
    'fire_hazard_canopy_overwrite', 'fire_hazard_canopy_processing_window_size',
    'fire_hazard_canopy_progress_interval', 'fire_hazard_fuel_normalization_manifest_path',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size']
# Identify missing canopy cover normalization inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_canopy_cover_normalization_inputs = [object_name for object_name in \
    required_canopy_cover_normalization_inputs if object_name not in globals()]

# Stop execution if missing canopy cover normalization inputs remain unresolved before this workflow
# stage begins.
if missing_canopy_cover_normalization_inputs:
    raise NameError(f'The following canopy-cover '
        f'normalization objects are '
        f'missing:\n'
        f'{missing_canopy_cover_normalization_inputs}\n\n'
        f'Run Configure Canopy Variable '
        f'Normalization before '
        f'normalizing forest canopy '
        f'cover.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_canopy_normalization_configured:
    raise ValueError('Canopy-variable normalization has not been configured successfully.')
# Collect canopy cover configuration in one configuration object so downstream steps use the same
# processing rules.
canopy_cover_configuration = fire_hazard_canopy_normalization_configuration['LANDFIRE_CANOPY_COVER']
# Build fire hazard canopy cover normalization input path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_cover_normalization_input_path = Path(canopy_cover_configuration['input_path'])
# Build fire hazard canopy cover normalization temporary path used to read, cache, or save this
# workflow product.
fire_hazard_canopy_cover_normalization_temporary_path = \
    Path(canopy_cover_configuration['temporary_path'])
# Build fire hazard canopy cover normalization output path used to read, cache, or save this
# workflow product.
fire_hazard_canopy_cover_normalization_output_path = Path(canopy_cover_configuration['output_path'])
# Prepare fire hazard canopy cover lower bound for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_cover_lower_bound = float(canopy_cover_configuration['lower_bound'])
# Prepare fire hazard canopy cover upper bound for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_cover_upper_bound = float(canopy_cover_configuration['upper_bound'])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not np.isfinite(fire_hazard_canopy_cover_lower_bound) or not \
    np.isfinite(fire_hazard_canopy_cover_upper_bound) or fire_hazard_canopy_cover_upper_bound <= \
    fire_hazard_canopy_cover_lower_bound:
    raise ValueError(f'The canopy-cover normalization '
        f'bounds are invalid.\nLower '
        f'bound: '
        f'{fire_hazard_canopy_cover_lower_bound}\n'
        f'Upper bound: '
        f'{fire_hazard_canopy_cover_upper_bound}')
# Prepare fire hazard canopy cover normalization range for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_cover_normalization_range = fire_hazard_canopy_cover_upper_bound - \
    fire_hazard_canopy_cover_lower_bound

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if not fire_hazard_canopy_cover_normalization_input_path.exists() or not \
    fire_hazard_canopy_cover_normalization_input_path.is_file() or \
    fire_hazard_canopy_cover_normalization_input_path.stat().st_size <= 0:
    raise FileNotFoundError(f'The aligned canopy-cover '
        f'raster is unavailable:\n'
        f'{fire_hazard_canopy_cover_normalization_input_path}')
# Build the expected canopy cover mask shape mask used to isolate records required for this
# analysis.
expected_canopy_cover_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_canopy_cover_mask_shape:
    raise ValueError(f'The canopy-cover study-area '
        f'mask does not match the common '
        f'project grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_canopy_cover_mask_shape}')

# Define reusable validate existing canopy cover score logic for this phase of the workflow.
def validate_existing_canopy_cover_score(raster_path):

    """
    Confirm that an existing canopy-cover score raster
    matches the common project grid and output
    structure.

    This function performs structural validation for
    cache reuse. Independent formula validation occurs
    later in R4.
    """
    # Build raster path used to read, cache, or save this workflow product.
    raster_path = Path(raster_path)

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not raster_path.exists() or not raster_path.is_file() or raster_path.stat().st_size <= 0:
        return False

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(raster_path) as raster_source:
            return all([raster_source.count == 1,
                raster_source.width == fire_hazard_alignment_width,
                raster_source.height == fire_hazard_alignment_height,
                raster_source.crs is not None, raster_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                raster_source.transform.almost_equals(fire_hazard_alignment_transform),
                raster_source.dtypes[0] == 'float32', raster_source.nodata == \
                    fire_hazard_canopy_score_nodata])
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception:
        return False
# Evaluate fire hazard canopy cover existing output valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_cover_existing_output_valid = \
    validate_existing_canopy_cover_score(fire_hazard_canopy_cover_normalization_output_path)
# Prepare fire hazard canopy cover reuse output for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_cover_reuse_output = fire_hazard_canopy_reuse_existing and \
    fire_hazard_canopy_cover_existing_output_valid

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_canopy_cover_normalization_temporary_path.exists():
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_canopy_cover_normalization_temporary_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_canopy_cover_normalization_output_path.exists() and fire_hazard_canopy_overwrite:
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_canopy_cover_normalization_output_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_canopy_cover_normalization_output_path.exists() and (not \
    fire_hazard_canopy_cover_reuse_output) and (not fire_hazard_canopy_overwrite):
    raise FileExistsError(f'An existing canopy-cover '
        f'hazard raster is present, but '
        f'it is not valid for cached '
        f'reuse and overwrite mode is '
        f'disabled:\n'
        f'{fire_hazard_canopy_cover_normalization_output_path}\n\n'
        f'Use refresh mode or remove the '
        f'invalid file before '
        f'continuing.')
# Evaluate fire hazard canopy cover source valid pixels so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_canopy_cover_source_valid_pixels = 0
# Calculate fire hazard canopy cover source NoData pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_cover_source_nodata_pixels = 0
# Evaluate fire hazard canopy cover output valid pixels so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_canopy_cover_output_valid_pixels = 0
# Calculate fire hazard canopy cover output NoData pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_cover_output_nodata_pixels = 0
# Calculate fire hazard canopy cover clipped low pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_cover_clipped_low_pixels = 0
# Calculate fire hazard canopy cover clipped high pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_cover_clipped_high_pixels = 0
# Evaluate fire hazard canopy cover valid inside study area so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_cover_valid_inside_study_area = 0
# Evaluate fire hazard canopy cover valid outside study area so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_cover_valid_outside_study_area = 0
# Set fire hazard canopy cover output minimum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_canopy_cover_output_minimum = None
# Set fire hazard canopy cover output maximum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_canopy_cover_output_maximum = None
# Prepare fire hazard canopy cover output sum for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_cover_output_sum = 0.0
# Prepare fire hazard canopy cover output mean for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_canopy_cover_output_mean = None
# Prepare fire hazard canopy cover processing error for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_canopy_cover_processing_error = None
# Prepare fire hazard canopy cover processing complete for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_cover_processing_complete = False

# Process the raster in internal windows to control memory use
# while preserving the full-resolution output.
# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_canopy_cover_reuse_output:
    # Prepare fire hazard canopy cover processing action for the fuel-hazard calculation and
    # subsequent QA checks.
    fire_hazard_canopy_cover_processing_action = 'Reused existing canopy-cover hazard raster'
    print('-> Reusing cached canopy-cover hazard raster.')

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_canopy_cover_normalization_output_path) as output_source:

            # Iterate through output source.block windows so each required item receives the same
            # processing and QA checks.
            for _, raster_window in output_source.block_windows(1):
                # Prepare output array for the raster calculation or validation performed in this
                # processing block.
                output_array = output_source.read(1, window=raster_window)
                # Prepare row start for the downstream processing or validation performed in this
                # workflow stage.
                row_start = int(raster_window.row_off)
                # Prepare row end for the downstream processing or validation performed in this
                # workflow stage.
                row_end = int(raster_window.row_off + raster_window.height)
                # Prepare column start for the downstream processing or validation performed in this
                # workflow stage.
                column_start = int(raster_window.col_off)
                # Prepare column end for the downstream processing or validation performed in this
                # workflow stage.
                column_end = int(raster_window.col_off + raster_window.width)
                # Build the study area window mask mask used to isolate records required for this
                # analysis.
                study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                    column_start:column_end]
                # Build the output valid mask mask used to isolate records required for this
                # analysis.
                output_valid_mask = np.isfinite(output_array) & (output_array != \
                    fire_hazard_canopy_score_nodata)
                # Prepare output values for the downstream processing or validation performed in
                # this workflow stage.
                output_values = output_array[output_valid_mask]
                # Evaluate fire hazard canopy cover output valid pixels so invalid inputs or outputs
                # can be rejected before continuing.
                fire_hazard_canopy_cover_output_valid_pixels += int(output_valid_mask.sum())
                # Calculate fire hazard canopy cover output NoData pixels for completeness,
                # file-integrity, or processing QA.
                fire_hazard_canopy_cover_output_nodata_pixels += int((~output_valid_mask).sum())
                # Evaluate fire hazard canopy cover valid inside study area so invalid inputs or
                # outputs can be rejected before continuing.
                fire_hazard_canopy_cover_valid_inside_study_area += int((output_valid_mask & \
                    study_area_window_mask).sum())
                # Evaluate fire hazard canopy cover valid outside study area so invalid inputs or
                # outputs can be rejected before continuing.
                fire_hazard_canopy_cover_valid_outside_study_area += int((output_valid_mask & \
                    ~study_area_window_mask).sum())

                # Stop execution if this validation condition is not satisfied before dependent
                # processing continues.
                if output_values.size > 0:
                    # Calculate window minimum so raster processing covers the analysis grid in
                    # controlled blocks.
                    window_minimum = float(output_values.min())
                    # Calculate window maximum so raster processing covers the analysis grid in
                    # controlled blocks.
                    window_maximum = float(output_values.max())
                    # Set fire hazard canopy cover output minimum as an explicit model or validation
                    # parameter used consistently in downstream calculations.
                    fire_hazard_canopy_cover_output_minimum = window_minimum if \
                        fire_hazard_canopy_cover_output_minimum is None else \
                        min(fire_hazard_canopy_cover_output_minimum,
                        window_minimum)
                    # Set fire hazard canopy cover output maximum as an explicit model or validation
                    # parameter used consistently in downstream calculations.
                    fire_hazard_canopy_cover_output_maximum = window_maximum if \
                        fire_hazard_canopy_cover_output_maximum is None else \
                        max(fire_hazard_canopy_cover_output_maximum,
                        window_maximum)
                    # Prepare fire hazard canopy cover output sum for the fuel-hazard calculation
                    # and subsequent QA checks.
                    fire_hazard_canopy_cover_output_sum += \
                        float(output_values.sum(dtype=np.float64))
                del output_array
        # Prepare fire hazard canopy cover processing complete for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_canopy_cover_processing_complete = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard canopy cover processing error for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_canopy_cover_processing_error = str(error)
else:
    # Prepare fire hazard canopy cover processing action for the fuel-hazard calculation and
    # subsequent QA checks.
    fire_hazard_canopy_cover_processing_action = 'Created normalized canopy-cover hazard raster'
    print('-> Normalizing canopy-cover raster...')
    # Create the output directory before writing workflow products.
    fire_hazard_canopy_cover_normalization_temporary_path.parent.mkdir(parents=True, exist_ok=True)

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_canopy_cover_normalization_input_path) as source_raster:
            # Evaluate source grid valid so invalid inputs or outputs can be rejected before
            # continuing.
            source_grid_valid = all([source_raster.count == 1,
                source_raster.width == fire_hazard_alignment_width,
                source_raster.height == fire_hazard_alignment_height,
                source_raster.crs is not None, source_raster.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                source_raster.transform.almost_equals(fire_hazard_alignment_transform)])

            # Stop execution if the prerequisite validation has not passed before this workflow
            # stage continues.
            if not source_grid_valid:
                raise ValueError('The aligned canopy-cover '
                    'raster does not match the '
                    'common project grid.')
            # Prepare output profile so raster outputs inherit the required grid, CRS, data type,
            # and NoData metadata.
            output_profile = fire_hazard_canopy_normalized_profile.copy()

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(fire_hazard_canopy_cover_normalization_temporary_path,
                'w', **output_profile) as output_raster:
                # Prepare source windows for the downstream processing or validation performed in
                # this workflow stage.
                source_windows = list(source_raster.block_windows(1))
                # Calculate total windows so raster processing covers the analysis grid in
                # controlled blocks.
                total_windows = len(source_windows)

                # Iterate through enumerate so each required item receives the same processing and
                # QA checks.
                for window_index, (_, raster_window) in enumerate(source_windows, start=1):

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if window_index == 1 or window_index % fire_hazard_canopy_progress_interval \
                        == 0 or window_index == total_windows:
                        print(f'-> Normalizing window {window_index:,} of {total_windows:,}...')
                    # Prepare source array for the raster calculation or validation performed in
                    # this processing block.
                    source_array = source_raster.read(1, window=raster_window, out_dtype='float32')
                    # Prepare row start for the downstream processing or validation performed in
                    # this workflow stage.
                    row_start = int(raster_window.row_off)
                    # Prepare row end for the downstream processing or validation performed in this
                    # workflow stage.
                    row_end = int(raster_window.row_off + raster_window.height)
                    # Prepare column start for the downstream processing or validation performed in
                    # this workflow stage.
                    column_start = int(raster_window.col_off)
                    # Prepare column end for the downstream processing or validation performed in
                    # this workflow stage.
                    column_end = int(raster_window.col_off + raster_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Build the source valid mask mask used to isolate records required for this
                    # analysis.
                    source_valid_mask = np.isfinite(source_array)

                    # Stop execution if the NoData configuration is incompatible with the raster
                    # product being validated.
                    if source_raster.nodata is not None:
                        # Build the source valid mask mask used to isolate records required for this
                        # analysis.
                        source_valid_mask &= source_array != source_raster.nodata
                    # Build the source valid mask mask used to isolate records required for this
                    # analysis.
                    source_valid_mask &= study_area_window_mask
                    # Evaluate fire hazard canopy cover source valid pixels so invalid inputs or
                    # outputs can be rejected before continuing.
                    fire_hazard_canopy_cover_source_valid_pixels += int(source_valid_mask.sum())
                    # Calculate fire hazard canopy cover source NoData pixels for completeness,
                    # file-integrity, or processing QA.
                    fire_hazard_canopy_cover_source_nodata_pixels += int((~source_valid_mask).sum())
                    # Prepare output array for the raster calculation or validation performed in
                    # this processing block.
                    output_array = np.full(source_array.shape,
                        fire_hazard_canopy_score_nodata, dtype=np.float32)

                    # Stop execution if the prerequisite validation has not passed before this
                    # workflow stage continues.
                    if source_valid_mask.any():
                        # Prepare source values for the downstream processing or validation
                        # performed in this workflow stage.
                        source_values = source_array[source_valid_mask].astype(np.float32)
                        # Calculate fire hazard canopy cover clipped low pixels for completeness,
                        # file-integrity, or processing QA.
                        fire_hazard_canopy_cover_clipped_low_pixels += int((source_values < \
                            fire_hazard_canopy_cover_lower_bound).sum())
                        # Calculate fire hazard canopy cover clipped high pixels for completeness,
                        # file-integrity, or processing QA.
                        fire_hazard_canopy_cover_clipped_high_pixels += int((source_values > \
                            fire_hazard_canopy_cover_upper_bound).sum())
                        # Prepare normalized values for the downstream processing or validation
                        # performed in this workflow stage.
                        normalized_values = (source_values - \
                            fire_hazard_canopy_cover_lower_bound) / \
                            fire_hazard_canopy_cover_normalization_range

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if fire_hazard_canopy_clip_normalized_scores:
                            # Prepare normalized values for the downstream processing or validation
                            # performed in this workflow stage.
                            normalized_values = np.clip(normalized_values,
                                fire_hazard_canopy_score_minimum, fire_hazard_canopy_score_maximum)
                        # Prepare output array so this workflow stage has the values required for
                        # downstream spatial processing and QA.
                        output_array[source_valid_mask] = normalized_values.astype(np.float32)
                    # Prepare output array so this workflow stage has the values required for
                    # downstream spatial processing and QA.
                    output_array[~study_area_window_mask] = fire_hazard_canopy_score_nodata
                    # Build the output valid mask mask used to isolate records required for this
                    # analysis.
                    output_valid_mask = np.isfinite(output_array) & (output_array != \
                        fire_hazard_canopy_score_nodata)
                    # Prepare output values for the downstream processing or validation performed in
                    # this workflow stage.
                    output_values = output_array[output_valid_mask]

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if output_values.size > 0:

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if output_values.min() < fire_hazard_canopy_score_minimum - \
                            fire_hazard_canopy_value_tolerance or output_values.max() > \
                            fire_hazard_canopy_score_maximum + fire_hazard_canopy_value_tolerance:
                            raise ValueError('A normalized canopy-cover '
                                'score fell outside the '
                                'configured zero-to-one range.')
                    # Write the processed data to the configured output resource.
                    output_raster.write(output_array, 1, window=raster_window)
                    # Evaluate fire hazard canopy cover output valid pixels so invalid inputs or
                    # outputs can be rejected before continuing.
                    fire_hazard_canopy_cover_output_valid_pixels += int(output_valid_mask.sum())
                    # Calculate fire hazard canopy cover output NoData pixels for completeness,
                    # file-integrity, or processing QA.
                    fire_hazard_canopy_cover_output_nodata_pixels += int((~output_valid_mask).sum())
                    # Evaluate fire hazard canopy cover valid inside study area so invalid inputs or
                    # outputs can be rejected before continuing.
                    fire_hazard_canopy_cover_valid_inside_study_area += int((output_valid_mask & \
                        study_area_window_mask).sum())
                    # Evaluate fire hazard canopy cover valid outside study area so invalid inputs
                    # or outputs can be rejected before continuing.
                    fire_hazard_canopy_cover_valid_outside_study_area += int((output_valid_mask & \
                        ~study_area_window_mask).sum())

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if output_values.size > 0:
                        # Calculate window minimum so raster processing covers the analysis grid in
                        # controlled blocks.
                        window_minimum = float(output_values.min())
                        # Calculate window maximum so raster processing covers the analysis grid in
                        # controlled blocks.
                        window_maximum = float(output_values.max())
                        # Set fire hazard canopy cover output minimum as an explicit model or
                        # validation parameter used consistently in downstream calculations.
                        fire_hazard_canopy_cover_output_minimum = window_minimum if \
                            fire_hazard_canopy_cover_output_minimum is None else \
                            min(fire_hazard_canopy_cover_output_minimum,
                            window_minimum)
                        # Set fire hazard canopy cover output maximum as an explicit model or
                        # validation parameter used consistently in downstream calculations.
                        fire_hazard_canopy_cover_output_maximum = window_maximum if \
                            fire_hazard_canopy_cover_output_maximum is None else \
                            max(fire_hazard_canopy_cover_output_maximum,
                            window_maximum)
                        # Prepare fire hazard canopy cover output sum for the fuel-hazard
                        # calculation and subsequent QA checks.
                        fire_hazard_canopy_cover_output_sum += \
                            float(output_values.sum(dtype=np.float64))
                    del source_array
                    del output_array
                    del source_valid_mask
                    del output_valid_mask
                # Assign a descriptive band label so the exported raster documents the meaning of
                # its values.
                output_raster.set_band_description(1, 'Normalized forest canopy-cover hazard score')
                # Write processing metadata to the raster so the final product retains its model and
                # provenance context.
                output_raster.update_tags(COMPONENT_ID='FUEL_HAZARD',
                    SOURCE_PRODUCT='LANDFIRE_CANOPY_COVER', \
                        NORMALIZATION_METHOD=fire_hazard_canopy_normalization_method,
                    NORMALIZATION_LOWER_BOUND=fire_hazard_canopy_cover_lower_bound,
                    NORMALIZATION_UPPER_BOUND=fire_hazard_canopy_cover_upper_bound,
                    SCORE_MINIMUM=fire_hazard_canopy_score_minimum,
                    SCORE_MAXIMUM=fire_hazard_canopy_score_maximum,
                    TARGET_CRS=fire_hazard_target_crs, \
                        TARGET_CELL_SIZE_METERS=fire_hazard_cell_size)
        # Evaluate temporary output valid so invalid inputs or outputs can be rejected before
        # continuing.
        temporary_output_valid = \
            validate_existing_canopy_cover_score \
            (fire_hazard_canopy_cover_normalization_temporary_path)

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if not temporary_output_valid:
            raise ValueError('The temporary normalized '
                'canopy-cover raster failed '
                'structural validation.')
        # Evaluate basic content valid so invalid inputs or outputs can be rejected before
        # continuing.
        basic_content_valid = all([fire_hazard_canopy_cover_output_valid_pixels >= \
            fire_hazard_canopy_minimum_valid_pixels,
            fire_hazard_canopy_cover_valid_inside_study_area > 0,
            fire_hazard_canopy_cover_valid_outside_study_area == 0,
            fire_hazard_canopy_cover_output_minimum is not None,
            fire_hazard_canopy_cover_output_maximum is not None])

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if not basic_content_valid:
            raise ValueError('The normalized canopy-cover raster failed basic content checks.')

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_canopy_cover_normalization_output_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_canopy_cover_normalization_output_path.unlink()
        # Promote the validated canopy-cover raster to its final output path after successful QA.
        fire_hazard_canopy_cover_normalization_temporary_path.replace \
            (fire_hazard_canopy_cover_normalization_output_path)
        # Prepare fire hazard canopy cover processing complete for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_canopy_cover_processing_complete = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard canopy cover processing error for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_canopy_cover_processing_error = str(error)
        # Prepare fire hazard canopy cover processing complete for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_canopy_cover_processing_complete = False

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_canopy_cover_normalization_temporary_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_canopy_cover_normalization_temporary_path.unlink()

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_canopy_cover_output_valid_pixels > 0:
    # Prepare fire hazard canopy cover output mean for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_canopy_cover_output_mean = fire_hazard_canopy_cover_output_sum / \
        fire_hazard_canopy_cover_output_valid_pixels
# Evaluate fire hazard canopy cover output structure valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_cover_output_structure_valid = \
    validate_existing_canopy_cover_score(fire_hazard_canopy_cover_normalization_output_path)
# Evaluate fire hazard canopy cover output range valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_canopy_cover_output_range_valid = all([fire_hazard_canopy_cover_output_minimum is not \
    None,
    fire_hazard_canopy_cover_output_maximum is not None,
    fire_hazard_canopy_cover_output_minimum >= fire_hazard_canopy_score_minimum - \
        fire_hazard_canopy_value_tolerance,
    fire_hazard_canopy_cover_output_maximum <= fire_hazard_canopy_score_maximum + \
        fire_hazard_canopy_value_tolerance])
# Prepare fire hazard canopy cover normalization complete for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_cover_normalization_complete = all([fire_hazard_canopy_cover_processing_complete,
    fire_hazard_canopy_cover_output_structure_valid,
    fire_hazard_canopy_cover_output_range_valid, fire_hazard_canopy_cover_output_valid_pixels >= \
        fire_hazard_canopy_minimum_valid_pixels,
    fire_hazard_canopy_cover_valid_inside_study_area > 0,
    fire_hazard_canopy_cover_valid_outside_study_area == 0,
    fire_hazard_canopy_cover_normalization_output_path.exists(),
    fire_hazard_canopy_cover_processing_error is None])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_canopy_cover_normalization_complete:
    raise ValueError(f'Forest canopy-cover '
        f'normalization did not complete '
        f'successfully.\n\nProcessing '
        f'complete: '
        f'{fire_hazard_canopy_cover_processing_complete}\n'
        f'Structure valid: '
        f'{fire_hazard_canopy_cover_output_structure_valid}\n'
        f'Score range valid: '
        f'{fire_hazard_canopy_cover_output_range_valid}\n'
        f'Valid output pixels: '
        f'{fire_hazard_canopy_cover_output_valid_pixels:,}\n'
        f'Valid outside study area: '
        f'{fire_hazard_canopy_cover_valid_outside_study_area:,}\n'
        f'Error: '
        f'{fire_hazard_canopy_cover_processing_error}')
# Calculate fire hazard canopy cover output size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_cover_output_size_bytes = \
    int(fire_hazard_canopy_cover_normalization_output_path.stat().st_size)
# Prepare fire hazard canopy cover normalization record for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_cover_normalization_record = {'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_ID': 'LANDFIRE_CANOPY_COVER', 'PRODUCT_NAME': 'Forest Canopy Cover Hazard Score',
    'INPUT_PATH': str(fire_hazard_canopy_cover_normalization_input_path),
    'OUTPUT_PATH': str(fire_hazard_canopy_cover_normalization_output_path),
    'PROCESSING_ACTION': fire_hazard_canopy_cover_processing_action,
    'NORMALIZATION_METHOD': fire_hazard_canopy_normalization_method,
    'LOWER_BOUND': fire_hazard_canopy_cover_lower_bound,
    'UPPER_BOUND': fire_hazard_canopy_cover_upper_bound,
    'SOURCE_VALID_PIXELS': fire_hazard_canopy_cover_source_valid_pixels,
    'SOURCE_NODATA_PIXELS': fire_hazard_canopy_cover_source_nodata_pixels,
    'OUTPUT_VALID_PIXELS': fire_hazard_canopy_cover_output_valid_pixels,
    'OUTPUT_NODATA_PIXELS': fire_hazard_canopy_cover_output_nodata_pixels,
    'CLIPPED_LOW_PIXELS': fire_hazard_canopy_cover_clipped_low_pixels,
    'CLIPPED_HIGH_PIXELS': fire_hazard_canopy_cover_clipped_high_pixels,
    'VALID_INSIDE_STUDY_AREA': fire_hazard_canopy_cover_valid_inside_study_area,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_canopy_cover_valid_outside_study_area,
    'OUTPUT_MINIMUM': fire_hazard_canopy_cover_output_minimum,
    'OUTPUT_MAXIMUM': fire_hazard_canopy_cover_output_maximum,
    'OUTPUT_MEAN': fire_hazard_canopy_cover_output_mean,
    'OUTPUT_SIZE_BYTES': fire_hazard_canopy_cover_output_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_canopy_cover_output_size_bytes / 1024 ** 2,
    'STRUCTURE_VALID': fire_hazard_canopy_cover_output_structure_valid,
    'VALUE_RANGE_VALID': fire_hazard_canopy_cover_output_range_valid,
    'VALID': fire_hazard_canopy_cover_normalization_complete}

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if Path(fire_hazard_fuel_normalization_manifest_path).exists():
    # Prepare fire hazard fuel normalization manifest for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fuel_normalization_manifest = \
        pd.read_csv(fire_hazard_fuel_normalization_manifest_path)
else:
    # Assemble fire hazard fuel normalization manifest into a table for QA review and downstream
    # validation.
    fire_hazard_fuel_normalization_manifest = pd.DataFrame()

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_normalization_manifest.empty and 'SOURCE_ID' in \
    fire_hazard_fuel_normalization_manifest.columns:
    # Prepare fire hazard fuel normalization manifest for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fuel_normalization_manifest = \
        fire_hazard_fuel_normalization_manifest[fire_hazard_fuel_normalization_manifest \
        ['SOURCE_ID'].astype(str).str.upper() != 'LANDFIRE_CANOPY_COVER'].copy()
# Assemble fire hazard fuel normalization manifest into a table for QA review and downstream
# validation.
fire_hazard_fuel_normalization_manifest = pd.concat([fire_hazard_fuel_normalization_manifest,
    pd.DataFrame([fire_hazard_canopy_cover_normalization_record])],
    ignore_index=True, sort=False).reset_index(drop=True)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_normalization_manifest.to_csv(fire_hazard_fuel_normalization_manifest_path,
    index=False)
# Prepare fire hazard canopy cover normalization summary for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_cover_normalization_summary = \
    pd.DataFrame([fire_hazard_canopy_cover_normalization_record])
# Build fire hazard canopy cover normalization summary path used to read, cache, or save this
# workflow product.
fire_hazard_canopy_cover_normalization_summary_path = fire_hazard_fuel_metadata_directory / \
    'canopy_cover_normalization_summary.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_canopy_cover_normalization_summary.to_csv \
    (fire_hazard_canopy_cover_normalization_summary_path,
    index=False)
print(f'-> Processing action: {fire_hazard_canopy_cover_processing_action}')
print(f'-> Input raster: {fire_hazard_canopy_cover_normalization_input_path}')
print(f'-> Output raster: {fire_hazard_canopy_cover_normalization_output_path}')
print(f'-> Normalization bounds: '
    f'{fire_hazard_canopy_cover_lower_bound:.4f} '
    f'to '
    f'{fire_hazard_canopy_cover_upper_bound:.4f}')
print(f'-> Valid output pixels: {fire_hazard_canopy_cover_output_valid_pixels:,}')
print(f'-> Values clipped below lower bound: {fire_hazard_canopy_cover_clipped_low_pixels:,}')
print(f'-> Values clipped above upper bound: {fire_hazard_canopy_cover_clipped_high_pixels:,}')
print(f'-> Output score range: '
    f'{fire_hazard_canopy_cover_output_minimum:.4f} '
    f'to '
    f'{fire_hazard_canopy_cover_output_maximum:.4f}')
print(f'-> Output mean score: {fire_hazard_canopy_cover_output_mean:.4f}')
print(f'-> Valid pixels outside study area: {fire_hazard_canopy_cover_valid_outside_study_area:,}')
print(f'-> Normalization complete: {fire_hazard_canopy_cover_normalization_complete}')
print(f'-> Normalization manifest saved: {fire_hazard_fuel_normalization_manifest_path}')
print(f'-> Normalization summary saved: {fire_hazard_canopy_cover_normalization_summary_path}')
print('\n--- CANOPY-COVER NORMALIZATION SUMMARY ---')
display(fire_hazard_canopy_cover_normalization_summary)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('Forest canopy cover has been transformed to the common zero-to-one fuel-hazard scale.')
print('Higher canopy-cover values '
    'produce higher hazard scores '
    'under the configured direct '
    'min-max normalization policy.')
print('This step performs processing '
    'and basic output checks. '
    'Independent formula validation '
    'will be completed after both '
    'canopy products have been '
    'normalized.')
print('\n=== FOREST CANOPY COVER NORMALIZATION COMPLETE ===')


=== NORMALIZING FOREST CANOPY COVER ===
-> Reusing cached canopy-cover hazard raster.
-> Processing action: Reused existing canopy-cover hazard raster
-> Input raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\aligned_sources\landfire_canopy_cover_aligned.tif
-> Output raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\normalized_sources\canopy_cover_hazard_score.tif
-> Normalization bounds: 0.0000 to 85.0000
-> Valid output pixels: 15,475,232
-> Values clipped below lower bound: 0
-> Values clipped above upper bound: 0
-> Output score range: 0.0000 to 1.0000
-> Output mean score: 0.1224
-> Valid pixels outside study area: 0
-> Normalization complete: True
-> Normalization manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_normalization_manifest.csv
-> Normalization summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metada

,COMPONENT_ID,SOURCE_ID,PRODUCT_NAME,INPUT_PATH,OUTPUT_PATH,PROCESSING_ACTION,NORMALIZATION_METHOD,LOWER_BOUND,UPPER_BOUND,SOURCE_VALID_PIXELS,...,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,OUTPUT_SIZE_BYTES,OUTPUT_SIZE_MB,STRUCTURE_VALID,VALUE_RANGE_VALID,VALID
0,FUEL_HAZARD,LANDFIRE_CANOPY_COVER,Forest Canopy Cover Hazard Score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing canopy-cover hazard raster,direct_min_max,0.0,85.0,0,...,15475232,0,0.0,1.0,0.122386,29542030,28.173475,True,True,True



NOTE:
Forest canopy cover has been transformed to the common zero-to-one fuel-hazard scale.
Higher canopy-cover values produce higher hazard scores under the configured direct min-max normalization policy.
This step performs processing and basic output checks. Independent formula validation will be completed after both canopy products have been normalized.

=== FOREST CANOPY COVER NORMALIZATION COMPLETE ===


### Normalizing Forest Canopy Bulk Density


In [111]:
print('=== NORMALIZING FOREST CANOPY BULK DENSITY ===')
# Prepare required canopy bulk density inputs for the downstream processing or validation performed
# in this workflow stage.
required_canopy_bulk_density_inputs = ['fire_hazard_canopy_normalization_configured',
    'fire_hazard_canopy_cover_normalization_complete',
    'fire_hazard_canopy_normalization_configuration',
    'fire_hazard_canopy_normalized_profile', 'fire_hazard_canopy_normalization_method',
    'fire_hazard_canopy_score_minimum', 'fire_hazard_canopy_score_maximum',
    'fire_hazard_canopy_score_nodata', 'fire_hazard_canopy_clip_normalized_scores',
    'fire_hazard_canopy_minimum_valid_pixels', 'fire_hazard_canopy_value_tolerance',
    'fire_hazard_canopy_reuse_existing', 'fire_hazard_canopy_overwrite',
    'fire_hazard_canopy_progress_interval', 'fire_hazard_fuel_normalization_manifest_path',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size']
# Identify missing canopy bulk density inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_canopy_bulk_density_inputs = [object_name for object_name in \
    required_canopy_bulk_density_inputs if object_name not in globals()]

# Stop execution if missing canopy bulk density inputs remain unresolved before this workflow stage
# begins.
if missing_canopy_bulk_density_inputs:
    raise NameError(f'The following '
        f'canopy-bulk-density '
        f'normalization objects are '
        f'missing:\n'
        f'{missing_canopy_bulk_density_inputs}\n\n'
        f'Run Configure Canopy Variable '
        f'Normalization and Normalize '
        f'Forest Canopy Cover before '
        f'normalizing canopy bulk '
        f'density.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_canopy_normalization_configured:
    raise ValueError('Canopy-variable normalization has not been configured successfully.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_canopy_cover_normalization_complete:
    raise ValueError('Forest canopy-cover normalization has not completed successfully.')
# Collect canopy bulk density configuration in one configuration object so downstream steps use the
# same processing rules.
canopy_bulk_density_configuration = \
    fire_hazard_canopy_normalization_configuration['LANDFIRE_CANOPY_BULK_DENSITY']
# Build fire hazard canopy bulk density normalization input path used to read, cache, or save this
# workflow product.
fire_hazard_canopy_bulk_density_normalization_input_path = \
    Path(canopy_bulk_density_configuration['input_path'])
# Build fire hazard canopy bulk density normalization temporary path used to read, cache, or save
# this workflow product.
fire_hazard_canopy_bulk_density_normalization_temporary_path = \
    Path(canopy_bulk_density_configuration['temporary_path'])
# Build fire hazard canopy bulk density normalization output path used to read, cache, or save this
# workflow product.
fire_hazard_canopy_bulk_density_normalization_output_path = \
    Path(canopy_bulk_density_configuration['output_path'])
# Prepare fire hazard canopy bulk density lower bound for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_canopy_bulk_density_lower_bound = \
    float(canopy_bulk_density_configuration['lower_bound'])
# Prepare fire hazard canopy bulk density upper bound for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_canopy_bulk_density_upper_bound = \
    float(canopy_bulk_density_configuration['upper_bound'])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not np.isfinite(fire_hazard_canopy_bulk_density_lower_bound) or not \
    np.isfinite(fire_hazard_canopy_bulk_density_upper_bound) or \
    fire_hazard_canopy_bulk_density_upper_bound <= fire_hazard_canopy_bulk_density_lower_bound:
    raise ValueError(f'The canopy-bulk-density '
        f'normalization bounds are '
        f'invalid.\nLower bound: '
        f'{fire_hazard_canopy_bulk_density_lower_bound}\n'
        f'Upper bound: '
        f'{fire_hazard_canopy_bulk_density_upper_bound}')
# Prepare fire hazard canopy bulk density normalization range for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_bulk_density_normalization_range = fire_hazard_canopy_bulk_density_upper_bound \
    - fire_hazard_canopy_bulk_density_lower_bound

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if not fire_hazard_canopy_bulk_density_normalization_input_path.exists() or not \
    fire_hazard_canopy_bulk_density_normalization_input_path.is_file() or \
    fire_hazard_canopy_bulk_density_normalization_input_path.stat().st_size <= 0:
    raise FileNotFoundError(f'The aligned '
        f'canopy-bulk-density raster is '
        f'unavailable:\n'
        f'{fire_hazard_canopy_bulk_density_normalization_input_path}')
# Build the expected canopy bulk density mask shape mask used to isolate records required for this
# analysis.
expected_canopy_bulk_density_mask_shape = (fire_hazard_alignment_height,
    fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_canopy_bulk_density_mask_shape:
    raise ValueError(f'The canopy-bulk-density '
        f'study-area mask does not match '
        f'the common project grid.\nMask '
        f'shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_canopy_bulk_density_mask_shape}')

# Define reusable validate existing canopy bulk density score logic for this phase of the workflow.
def validate_existing_canopy_bulk_density_score(raster_path):

    """
    Confirm that an existing canopy-bulk-density
    hazard raster matches the common project grid and
    expected Float32 output structure.
    """
    # Build raster path used to read, cache, or save this workflow product.
    raster_path = Path(raster_path)

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not raster_path.exists() or not raster_path.is_file() or raster_path.stat().st_size <= 0:
        return False

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(raster_path) as raster_source:
            return all([raster_source.count == 1,
                raster_source.width == fire_hazard_alignment_width,
                raster_source.height == fire_hazard_alignment_height,
                raster_source.crs is not None, raster_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                raster_source.transform.almost_equals(fire_hazard_alignment_transform),
                raster_source.dtypes[0] == 'float32', raster_source.nodata == \
                    fire_hazard_canopy_score_nodata])
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception:
        return False
# Evaluate fire hazard canopy bulk density existing output valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_bulk_density_existing_output_valid = \
    validate_existing_canopy_bulk_density_score \
    (fire_hazard_canopy_bulk_density_normalization_output_path)
# Prepare fire hazard canopy bulk density reuse output for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_bulk_density_reuse_output = fire_hazard_canopy_reuse_existing and \
    fire_hazard_canopy_bulk_density_existing_output_valid

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_canopy_bulk_density_normalization_temporary_path.exists():
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_canopy_bulk_density_normalization_temporary_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_canopy_bulk_density_normalization_output_path.exists() and \
    fire_hazard_canopy_overwrite:
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_canopy_bulk_density_normalization_output_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_canopy_bulk_density_normalization_output_path.exists() and (not \
    fire_hazard_canopy_bulk_density_reuse_output) and (not fire_hazard_canopy_overwrite):
    raise FileExistsError(f'An existing '
        f'canopy-bulk-density hazard '
        f'raster is present, but it is '
        f'not valid for cached reuse and '
        f'overwrite mode is disabled:\n'
        f'{fire_hazard_canopy_bulk_density_normalization_output_path}\n\n'
        f'Use refresh mode or remove the '
        f'invalid file before '
        f'continuing.')
# Evaluate fire hazard canopy bulk density source valid pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_bulk_density_source_valid_pixels = 0
# Calculate fire hazard canopy bulk density source NoData pixels for completeness, file-integrity,
# or processing QA.
fire_hazard_canopy_bulk_density_source_nodata_pixels = 0
# Evaluate fire hazard canopy bulk density output valid pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_bulk_density_output_valid_pixels = 0
# Calculate fire hazard canopy bulk density output NoData pixels for completeness, file-integrity,
# or processing QA.
fire_hazard_canopy_bulk_density_output_nodata_pixels = 0
# Calculate fire hazard canopy bulk density clipped low pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_bulk_density_clipped_low_pixels = 0
# Calculate fire hazard canopy bulk density clipped high pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_bulk_density_clipped_high_pixels = 0
# Evaluate fire hazard canopy bulk density valid inside study area so invalid inputs or outputs can
# be rejected before continuing.
fire_hazard_canopy_bulk_density_valid_inside_study_area = 0
# Evaluate fire hazard canopy bulk density valid outside study area so invalid inputs or outputs can
# be rejected before continuing.
fire_hazard_canopy_bulk_density_valid_outside_study_area = 0
# Set fire hazard canopy bulk density output minimum as an explicit model or validation parameter
# used consistently in downstream calculations.
fire_hazard_canopy_bulk_density_output_minimum = None
# Set fire hazard canopy bulk density output maximum as an explicit model or validation parameter
# used consistently in downstream calculations.
fire_hazard_canopy_bulk_density_output_maximum = None
# Prepare fire hazard canopy bulk density output sum for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_canopy_bulk_density_output_sum = 0.0
# Prepare fire hazard canopy bulk density output mean for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_canopy_bulk_density_output_mean = None
# Prepare fire hazard canopy bulk density processing error for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_bulk_density_processing_error = None
# Prepare fire hazard canopy bulk density processing complete for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_bulk_density_processing_complete = False

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_canopy_bulk_density_reuse_output:
    # Prepare fire hazard canopy bulk density processing action for the fuel-hazard calculation and
    # subsequent QA checks.
    fire_hazard_canopy_bulk_density_processing_action = \
        'Reused existing canopy-bulk-density hazard raster'
    print('-> Reusing cached canopy-bulk-density hazard raster.')

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_canopy_bulk_density_normalization_output_path) as \
            output_source:

            # Iterate through output source.block windows so each required item receives the same
            # processing and QA checks.
            for _, raster_window in output_source.block_windows(1):
                # Prepare output array for the raster calculation or validation performed in this
                # processing block.
                output_array = output_source.read(1, window=raster_window)
                # Prepare row start for the downstream processing or validation performed in this
                # workflow stage.
                row_start = int(raster_window.row_off)
                # Prepare row end for the downstream processing or validation performed in this
                # workflow stage.
                row_end = int(raster_window.row_off + raster_window.height)
                # Prepare column start for the downstream processing or validation performed in this
                # workflow stage.
                column_start = int(raster_window.col_off)
                # Prepare column end for the downstream processing or validation performed in this
                # workflow stage.
                column_end = int(raster_window.col_off + raster_window.width)
                # Build the study area window mask mask used to isolate records required for this
                # analysis.
                study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                    column_start:column_end]
                # Build the output valid mask mask used to isolate records required for this
                # analysis.
                output_valid_mask = np.isfinite(output_array) & (output_array != \
                    fire_hazard_canopy_score_nodata)
                # Prepare output values for the downstream processing or validation performed in
                # this workflow stage.
                output_values = output_array[output_valid_mask]
                # Evaluate fire hazard canopy bulk density output valid pixels so invalid inputs or
                # outputs can be rejected before continuing.
                fire_hazard_canopy_bulk_density_output_valid_pixels += int(output_valid_mask.sum())
                # Calculate fire hazard canopy bulk density output NoData pixels for completeness,
                # file-integrity, or processing QA.
                fire_hazard_canopy_bulk_density_output_nodata_pixels += \
                    int((~output_valid_mask).sum())
                # Evaluate fire hazard canopy bulk density valid inside study area so invalid inputs
                # or outputs can be rejected before continuing.
                fire_hazard_canopy_bulk_density_valid_inside_study_area += int((output_valid_mask \
                    & study_area_window_mask).sum())
                # Evaluate fire hazard canopy bulk density valid outside study area so invalid
                # inputs or outputs can be rejected before continuing.
                fire_hazard_canopy_bulk_density_valid_outside_study_area += \
                    int((output_valid_mask & ~study_area_window_mask).sum())

                # Stop execution if this validation condition is not satisfied before dependent
                # processing continues.
                if output_values.size > 0:
                    # Calculate window minimum so raster processing covers the analysis grid in
                    # controlled blocks.
                    window_minimum = float(output_values.min())
                    # Calculate window maximum so raster processing covers the analysis grid in
                    # controlled blocks.
                    window_maximum = float(output_values.max())

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if fire_hazard_canopy_bulk_density_output_minimum is None:
                        # Set fire hazard canopy bulk density output minimum as an explicit model or
                        # validation parameter used consistently in downstream calculations.
                        fire_hazard_canopy_bulk_density_output_minimum = window_minimum
                    else:
                        # Set fire hazard canopy bulk density output minimum as an explicit model or
                        # validation parameter used consistently in downstream calculations.
                        fire_hazard_canopy_bulk_density_output_minimum = \
                            min(fire_hazard_canopy_bulk_density_output_minimum,
                            window_minimum)

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if fire_hazard_canopy_bulk_density_output_maximum is None:
                        # Set fire hazard canopy bulk density output maximum as an explicit model or
                        # validation parameter used consistently in downstream calculations.
                        fire_hazard_canopy_bulk_density_output_maximum = window_maximum
                    else:
                        # Set fire hazard canopy bulk density output maximum as an explicit model or
                        # validation parameter used consistently in downstream calculations.
                        fire_hazard_canopy_bulk_density_output_maximum = \
                            max(fire_hazard_canopy_bulk_density_output_maximum,
                            window_maximum)
                    # Prepare fire hazard canopy bulk density output sum for the fuel-hazard
                    # calculation and subsequent QA checks.
                    fire_hazard_canopy_bulk_density_output_sum += \
                        float(output_values.sum(dtype=np.float64))
                del output_array
        # Prepare fire hazard canopy bulk density processing complete for the fuel-hazard
        # calculation and subsequent QA checks.
        fire_hazard_canopy_bulk_density_processing_complete = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard canopy bulk density processing error for the fuel-hazard calculation
        # and subsequent QA checks.
        fire_hazard_canopy_bulk_density_processing_error = str(error)
else:
    # Prepare fire hazard canopy bulk density processing action for the fuel-hazard calculation and
    # subsequent QA checks.
    fire_hazard_canopy_bulk_density_processing_action = \
        'Created normalized canopy-bulk-density hazard raster'
    print('-> Normalizing canopy-bulk-density raster...')
    # Create the output directory before writing workflow products.
    fire_hazard_canopy_bulk_density_normalization_temporary_path.parent.mkdir(parents=True,
        exist_ok=True)

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_canopy_bulk_density_normalization_input_path) as \
            source_raster:
            # Evaluate source grid valid so invalid inputs or outputs can be rejected before
            # continuing.
            source_grid_valid = all([source_raster.count == 1,
                source_raster.width == fire_hazard_alignment_width,
                source_raster.height == fire_hazard_alignment_height,
                source_raster.crs is not None, source_raster.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                source_raster.transform.almost_equals(fire_hazard_alignment_transform)])

            # Stop execution if the prerequisite validation has not passed before this workflow
            # stage continues.
            if not source_grid_valid:
                raise ValueError('The aligned '
                    'canopy-bulk-density raster '
                    'does not match the common '
                    'project grid.')
            # Prepare output profile so raster outputs inherit the required grid, CRS, data type,
            # and NoData metadata.
            output_profile = fire_hazard_canopy_normalized_profile.copy()

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(fire_hazard_canopy_bulk_density_normalization_temporary_path,
                'w', **output_profile) as output_raster:
                # Prepare source windows for the downstream processing or validation performed in
                # this workflow stage.
                source_windows = list(source_raster.block_windows(1))
                # Calculate total windows so raster processing covers the analysis grid in
                # controlled blocks.
                total_windows = len(source_windows)

                # Iterate through enumerate so each required item receives the same processing and
                # QA checks.
                for window_index, (_, raster_window) in enumerate(source_windows, start=1):

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if window_index == 1 or window_index % fire_hazard_canopy_progress_interval \
                        == 0 or window_index == total_windows:
                        print(f'-> Normalizing window {window_index:,} of {total_windows:,}...')
                    # Prepare source array for the raster calculation or validation performed in
                    # this processing block.
                    source_array = source_raster.read(1, window=raster_window, out_dtype='float32')
                    # Prepare row start for the downstream processing or validation performed in
                    # this workflow stage.
                    row_start = int(raster_window.row_off)
                    # Prepare row end for the downstream processing or validation performed in this
                    # workflow stage.
                    row_end = int(raster_window.row_off + raster_window.height)
                    # Prepare column start for the downstream processing or validation performed in
                    # this workflow stage.
                    column_start = int(raster_window.col_off)
                    # Prepare column end for the downstream processing or validation performed in
                    # this workflow stage.
                    column_end = int(raster_window.col_off + raster_window.width)
                    # Build the study area window mask mask used to isolate records required for
                    # this analysis.
                    study_area_window_mask = \
                        fire_hazard_alignment_study_area_mask[row_start:row_end,
                        column_start:column_end]
                    # Build the source valid mask mask used to isolate records required for this
                    # analysis.
                    source_valid_mask = np.isfinite(source_array)

                    # Stop execution if the NoData configuration is incompatible with the raster
                    # product being validated.
                    if source_raster.nodata is not None:
                        # Build the source valid mask mask used to isolate records required for this
                        # analysis.
                        source_valid_mask &= source_array != source_raster.nodata
                    # Build the source valid mask mask used to isolate records required for this
                    # analysis.
                    source_valid_mask &= study_area_window_mask
                    # Evaluate fire hazard canopy bulk density source valid pixels so invalid inputs
                    # or outputs can be rejected before continuing.
                    fire_hazard_canopy_bulk_density_source_valid_pixels += \
                        int(source_valid_mask.sum())
                    # Calculate fire hazard canopy bulk density source NoData pixels for
                    # completeness, file-integrity, or processing QA.
                    fire_hazard_canopy_bulk_density_source_nodata_pixels += \
                        int((~source_valid_mask).sum())
                    # Prepare output array for the raster calculation or validation performed in
                    # this processing block.
                    output_array = np.full(source_array.shape,
                        fire_hazard_canopy_score_nodata, dtype=np.float32)

                    # Stop execution if the prerequisite validation has not passed before this
                    # workflow stage continues.
                    if source_valid_mask.any():
                        # Prepare source values for the downstream processing or validation
                        # performed in this workflow stage.
                        source_values = source_array[source_valid_mask].astype(np.float32)
                        # Calculate fire hazard canopy bulk density clipped low pixels for
                        # completeness, file-integrity, or processing QA.
                        fire_hazard_canopy_bulk_density_clipped_low_pixels += int((source_values \
                            < fire_hazard_canopy_bulk_density_lower_bound).sum())
                        # Calculate fire hazard canopy bulk density clipped high pixels for
                        # completeness, file-integrity, or processing QA.
                        fire_hazard_canopy_bulk_density_clipped_high_pixels += int((source_values \
                            > fire_hazard_canopy_bulk_density_upper_bound).sum())
                        # Prepare normalized values for the downstream processing or validation
                        # performed in this workflow stage.
                        normalized_values = (source_values - \
                            fire_hazard_canopy_bulk_density_lower_bound) / \
                            fire_hazard_canopy_bulk_density_normalization_range

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if fire_hazard_canopy_clip_normalized_scores:
                            # Prepare normalized values for the downstream processing or validation
                            # performed in this workflow stage.
                            normalized_values = np.clip(normalized_values,
                                fire_hazard_canopy_score_minimum, fire_hazard_canopy_score_maximum)
                        # Prepare output array so this workflow stage has the values required for
                        # downstream spatial processing and QA.
                        output_array[source_valid_mask] = normalized_values.astype(np.float32)
                    # Prepare output array so this workflow stage has the values required for
                    # downstream spatial processing and QA.
                    output_array[~study_area_window_mask] = fire_hazard_canopy_score_nodata
                    # Build the output valid mask mask used to isolate records required for this
                    # analysis.
                    output_valid_mask = np.isfinite(output_array) & (output_array != \
                        fire_hazard_canopy_score_nodata)
                    # Prepare output values for the downstream processing or validation performed in
                    # this workflow stage.
                    output_values = output_array[output_valid_mask]

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if output_values.size > 0:

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if output_values.min() < fire_hazard_canopy_score_minimum - \
                            fire_hazard_canopy_value_tolerance or output_values.max() > \
                            fire_hazard_canopy_score_maximum + fire_hazard_canopy_value_tolerance:
                            raise ValueError('A normalized '
                                'canopy-bulk-density score fell '
                                'outside the configured '
                                'zero-to-one range.')
                    # Write the processed data to the configured output resource.
                    output_raster.write(output_array, 1, window=raster_window)
                    # Evaluate fire hazard canopy bulk density output valid pixels so invalid inputs
                    # or outputs can be rejected before continuing.
                    fire_hazard_canopy_bulk_density_output_valid_pixels += \
                        int(output_valid_mask.sum())
                    # Calculate fire hazard canopy bulk density output NoData pixels for
                    # completeness, file-integrity, or processing QA.
                    fire_hazard_canopy_bulk_density_output_nodata_pixels += \
                        int((~output_valid_mask).sum())
                    # Evaluate fire hazard canopy bulk density valid inside study area so invalid
                    # inputs or outputs can be rejected before continuing.
                    fire_hazard_canopy_bulk_density_valid_inside_study_area += \
                        int((output_valid_mask & study_area_window_mask).sum())
                    # Evaluate fire hazard canopy bulk density valid outside study area so invalid
                    # inputs or outputs can be rejected before continuing.
                    fire_hazard_canopy_bulk_density_valid_outside_study_area += \
                        int((output_valid_mask & ~study_area_window_mask).sum())

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if output_values.size > 0:
                        # Calculate window minimum so raster processing covers the analysis grid in
                        # controlled blocks.
                        window_minimum = float(output_values.min())
                        # Calculate window maximum so raster processing covers the analysis grid in
                        # controlled blocks.
                        window_maximum = float(output_values.max())

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if fire_hazard_canopy_bulk_density_output_minimum is None:
                            # Set fire hazard canopy bulk density output minimum as an explicit
                            # model or validation parameter used consistently in downstream
                            # calculations.
                            fire_hazard_canopy_bulk_density_output_minimum = window_minimum
                        else:
                            # Set fire hazard canopy bulk density output minimum as an explicit
                            # model or validation parameter used consistently in downstream
                            # calculations.
                            fire_hazard_canopy_bulk_density_output_minimum = \
                                min(fire_hazard_canopy_bulk_density_output_minimum,
                                window_minimum)

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if fire_hazard_canopy_bulk_density_output_maximum is None:
                            # Set fire hazard canopy bulk density output maximum as an explicit
                            # model or validation parameter used consistently in downstream
                            # calculations.
                            fire_hazard_canopy_bulk_density_output_maximum = window_maximum
                        else:
                            # Set fire hazard canopy bulk density output maximum as an explicit
                            # model or validation parameter used consistently in downstream
                            # calculations.
                            fire_hazard_canopy_bulk_density_output_maximum = \
                                max(fire_hazard_canopy_bulk_density_output_maximum,
                                window_maximum)
                        # Prepare fire hazard canopy bulk density output sum for the fuel-hazard
                        # calculation and subsequent QA checks.
                        fire_hazard_canopy_bulk_density_output_sum += \
                            float(output_values.sum(dtype=np.float64))
                    del source_array
                    del output_array
                    del source_valid_mask
                    del output_valid_mask
                # Assign a descriptive band label so the exported raster documents the meaning of
                # its values.
                output_raster.set_band_description(1,
                    'Normalized forest canopy bulk-density hazard score')
                # Write processing metadata to the raster so the final product retains its model and
                # provenance context.
                output_raster.update_tags(COMPONENT_ID='FUEL_HAZARD',
                    SOURCE_PRODUCT='LANDFIRE_CANOPY_BULK_DENSITY',
                    NORMALIZATION_METHOD=fire_hazard_canopy_normalization_method,
                    NORMALIZATION_LOWER_BOUND=fire_hazard_canopy_bulk_density_lower_bound,
                    NORMALIZATION_UPPER_BOUND=fire_hazard_canopy_bulk_density_upper_bound,
                    SCORE_MINIMUM=fire_hazard_canopy_score_minimum,
                    SCORE_MAXIMUM=fire_hazard_canopy_score_maximum,
                    TARGET_CRS=fire_hazard_target_crs, \
                        TARGET_CELL_SIZE_METERS=fire_hazard_cell_size)
        # Evaluate temporary output valid so invalid inputs or outputs can be rejected before
        # continuing.
        temporary_output_valid = \
            validate_existing_canopy_bulk_density_score \
            (fire_hazard_canopy_bulk_density_normalization_temporary_path)

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if not temporary_output_valid:
            raise ValueError('The temporary normalized '
                'canopy-bulk-density raster '
                'failed structural validation.')
        # Evaluate basic content valid so invalid inputs or outputs can be rejected before
        # continuing.
        basic_content_valid = all([fire_hazard_canopy_bulk_density_output_valid_pixels >= \
            fire_hazard_canopy_minimum_valid_pixels,
            fire_hazard_canopy_bulk_density_valid_inside_study_area > 0,
            fire_hazard_canopy_bulk_density_valid_outside_study_area == 0,
            fire_hazard_canopy_bulk_density_output_minimum is not None,
            fire_hazard_canopy_bulk_density_output_maximum is not None])

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if not basic_content_valid:
            raise ValueError('The normalized '
                'canopy-bulk-density raster '
                'failed basic content checks.')

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_canopy_bulk_density_normalization_output_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_canopy_bulk_density_normalization_output_path.unlink()
        # Promote the validated canopy-density raster to its final output path after successful QA.
        fire_hazard_canopy_bulk_density_normalization_temporary_path.replace \
            (fire_hazard_canopy_bulk_density_normalization_output_path)
        # Prepare fire hazard canopy bulk density processing complete for the fuel-hazard
        # calculation and subsequent QA checks.
        fire_hazard_canopy_bulk_density_processing_complete = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard canopy bulk density processing error for the fuel-hazard calculation
        # and subsequent QA checks.
        fire_hazard_canopy_bulk_density_processing_error = str(error)
        # Prepare fire hazard canopy bulk density processing complete for the fuel-hazard
        # calculation and subsequent QA checks.
        fire_hazard_canopy_bulk_density_processing_complete = False

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_canopy_bulk_density_normalization_temporary_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_canopy_bulk_density_normalization_temporary_path.unlink()

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_canopy_bulk_density_output_valid_pixels > 0:
    # Prepare fire hazard canopy bulk density output mean for the fuel-hazard calculation and
    # subsequent QA checks.
    fire_hazard_canopy_bulk_density_output_mean = fire_hazard_canopy_bulk_density_output_sum / \
        fire_hazard_canopy_bulk_density_output_valid_pixels
# Evaluate fire hazard canopy bulk density output structure valid so invalid inputs or outputs can
# be rejected before continuing.
fire_hazard_canopy_bulk_density_output_structure_valid = \
    validate_existing_canopy_bulk_density_score \
    (fire_hazard_canopy_bulk_density_normalization_output_path)
# Evaluate fire hazard canopy bulk density output range valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_bulk_density_output_range_valid = \
    all([fire_hazard_canopy_bulk_density_output_minimum is not None,
    fire_hazard_canopy_bulk_density_output_maximum is not None,
    fire_hazard_canopy_bulk_density_output_minimum >= fire_hazard_canopy_score_minimum - \
        fire_hazard_canopy_value_tolerance,
    fire_hazard_canopy_bulk_density_output_maximum <= fire_hazard_canopy_score_maximum + \
        fire_hazard_canopy_value_tolerance])
# Prepare fire hazard canopy bulk density normalization complete for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_bulk_density_normalization_complete = \
    all([fire_hazard_canopy_bulk_density_processing_complete,
    fire_hazard_canopy_bulk_density_output_structure_valid,
    fire_hazard_canopy_bulk_density_output_range_valid,
    fire_hazard_canopy_bulk_density_output_valid_pixels >= fire_hazard_canopy_minimum_valid_pixels,
    fire_hazard_canopy_bulk_density_valid_inside_study_area > 0,
    fire_hazard_canopy_bulk_density_valid_outside_study_area == 0,
    fire_hazard_canopy_bulk_density_normalization_output_path.exists(),
    fire_hazard_canopy_bulk_density_processing_error is None])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_canopy_bulk_density_normalization_complete:
    raise ValueError(f'Forest canopy-bulk-density '
        f'normalization did not complete '
        f'successfully.\n\nProcessing '
        f'complete: '
        f'{fire_hazard_canopy_bulk_density_processing_complete}\n'
        f'Structure valid: '
        f'{fire_hazard_canopy_bulk_density_output_structure_valid}\n'
        f'Score range valid: '
        f'{fire_hazard_canopy_bulk_density_output_range_valid}\n'
        f'Valid output pixels: '
        f'{fire_hazard_canopy_bulk_density_output_valid_pixels:,}\n'
        f'Valid outside study area: '
        f'{fire_hazard_canopy_bulk_density_valid_outside_study_area:,}\n'
        f'Error: '
        f'{fire_hazard_canopy_bulk_density_processing_error}')
# Calculate fire hazard canopy bulk density output size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_bulk_density_output_size_bytes = \
    int(fire_hazard_canopy_bulk_density_normalization_output_path.stat().st_size)
# Prepare fire hazard canopy bulk density normalization record for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_bulk_density_normalization_record = {'COMPONENT_ID': 'FUEL_HAZARD',
    'SOURCE_ID': 'LANDFIRE_CANOPY_BULK_DENSITY', 'PRODUCT_NAME': \
        'Forest Canopy Bulk Density Hazard Score',
    'INPUT_PATH': str(fire_hazard_canopy_bulk_density_normalization_input_path),
    'OUTPUT_PATH': str(fire_hazard_canopy_bulk_density_normalization_output_path),
    'PROCESSING_ACTION': fire_hazard_canopy_bulk_density_processing_action,
    'NORMALIZATION_METHOD': fire_hazard_canopy_normalization_method,
    'LOWER_BOUND': fire_hazard_canopy_bulk_density_lower_bound,
    'UPPER_BOUND': fire_hazard_canopy_bulk_density_upper_bound,
    'SOURCE_VALID_PIXELS': fire_hazard_canopy_bulk_density_source_valid_pixels,
    'SOURCE_NODATA_PIXELS': fire_hazard_canopy_bulk_density_source_nodata_pixels,
    'OUTPUT_VALID_PIXELS': fire_hazard_canopy_bulk_density_output_valid_pixels,
    'OUTPUT_NODATA_PIXELS': fire_hazard_canopy_bulk_density_output_nodata_pixels,
    'CLIPPED_LOW_PIXELS': fire_hazard_canopy_bulk_density_clipped_low_pixels,
    'CLIPPED_HIGH_PIXELS': fire_hazard_canopy_bulk_density_clipped_high_pixels,
    'VALID_INSIDE_STUDY_AREA': fire_hazard_canopy_bulk_density_valid_inside_study_area,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_canopy_bulk_density_valid_outside_study_area,
    'OUTPUT_MINIMUM': fire_hazard_canopy_bulk_density_output_minimum,
    'OUTPUT_MAXIMUM': fire_hazard_canopy_bulk_density_output_maximum,
    'OUTPUT_MEAN': fire_hazard_canopy_bulk_density_output_mean,
    'OUTPUT_SIZE_BYTES': fire_hazard_canopy_bulk_density_output_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_canopy_bulk_density_output_size_bytes / 1024 ** 2,
    'STRUCTURE_VALID': fire_hazard_canopy_bulk_density_output_structure_valid,
    'VALUE_RANGE_VALID': fire_hazard_canopy_bulk_density_output_range_valid,
    'VALID': fire_hazard_canopy_bulk_density_normalization_complete}

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if Path(fire_hazard_fuel_normalization_manifest_path).exists():
    # Prepare fire hazard fuel normalization manifest for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fuel_normalization_manifest = \
        pd.read_csv(fire_hazard_fuel_normalization_manifest_path)
else:
    # Assemble fire hazard fuel normalization manifest into a table for QA review and downstream
    # validation.
    fire_hazard_fuel_normalization_manifest = pd.DataFrame()

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_normalization_manifest.empty and 'SOURCE_ID' in \
    fire_hazard_fuel_normalization_manifest.columns:
    # Prepare fire hazard fuel normalization manifest for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fuel_normalization_manifest = \
        fire_hazard_fuel_normalization_manifest[fire_hazard_fuel_normalization_manifest \
        ['SOURCE_ID'].astype(str).str.upper() != 'LANDFIRE_CANOPY_BULK_DENSITY'].copy()
# Assemble fire hazard fuel normalization manifest into a table for QA review and downstream
# validation.
fire_hazard_fuel_normalization_manifest = pd.concat([fire_hazard_fuel_normalization_manifest,
    pd.DataFrame([fire_hazard_canopy_bulk_density_normalization_record])],
    ignore_index=True, sort=False).reset_index(drop=True)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_normalization_manifest.to_csv(fire_hazard_fuel_normalization_manifest_path,
    index=False)
# Prepare fire hazard canopy bulk density normalization summary for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_canopy_bulk_density_normalization_summary = \
    pd.DataFrame([fire_hazard_canopy_bulk_density_normalization_record])
# Build fire hazard canopy bulk density normalization summary path used to read, cache, or save this
# workflow product.
fire_hazard_canopy_bulk_density_normalization_summary_path = fire_hazard_fuel_metadata_directory \
    / 'canopy_bulk_density_normalization_summary.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_canopy_bulk_density_normalization_summary.to_csv \
    (fire_hazard_canopy_bulk_density_normalization_summary_path,
    index=False)
print(f'-> Processing action: {fire_hazard_canopy_bulk_density_processing_action}')
print(f'-> Input raster: {fire_hazard_canopy_bulk_density_normalization_input_path}')
print(f'-> Output raster: {fire_hazard_canopy_bulk_density_normalization_output_path}')
print(f'-> Normalization bounds: '
    f'{fire_hazard_canopy_bulk_density_lower_bound:.4f} '
    f'to '
    f'{fire_hazard_canopy_bulk_density_upper_bound:.4f}')
print(f'-> Valid output pixels: {fire_hazard_canopy_bulk_density_output_valid_pixels:,}')
print(f'-> Values clipped below lower '
    f'bound: '
    f'{fire_hazard_canopy_bulk_density_clipped_low_pixels:,}')
print(f'-> Values clipped above upper '
    f'bound: '
    f'{fire_hazard_canopy_bulk_density_clipped_high_pixels:,}')
print(f'-> Output score range: '
    f'{fire_hazard_canopy_bulk_density_output_minimum:.4f} '
    f'to '
    f'{fire_hazard_canopy_bulk_density_output_maximum:.4f}')
print(f'-> Output mean score: {fire_hazard_canopy_bulk_density_output_mean:.4f}')
print(f'-> Valid pixels outside study '
    f'area: '
    f'{fire_hazard_canopy_bulk_density_valid_outside_study_area:,}')
print(f'-> Normalization complete: {fire_hazard_canopy_bulk_density_normalization_complete}')
print(f'-> Normalization manifest saved: {fire_hazard_fuel_normalization_manifest_path}')
print(f'-> Normalization summary '
    f'saved: '
    f'{fire_hazard_canopy_bulk_density_normalization_summary_path}')
print('\n--- CANOPY-BULK-DENSITY NORMALIZATION SUMMARY ---')
display(fire_hazard_canopy_bulk_density_normalization_summary)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('Forest canopy bulk density has '
    'been transformed to the common '
    'zero-to-one fuel-hazard scale.')
print('Higher canopy-bulk-density '
    'values produce higher hazard '
    'scores under the direct '
    'min-max policy.')
print('Both continuous canopy '
    'products are now normalized '
    'and ready for independent '
    'formula, grid, and content '
    'validation.')
print('\n=== FOREST CANOPY BULK DENSITY NORMALIZATION COMPLETE ===')


=== NORMALIZING FOREST CANOPY BULK DENSITY ===
-> Reusing cached canopy-bulk-density hazard raster.
-> Processing action: Reused existing canopy-bulk-density hazard raster
-> Input raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\aligned_sources\landfire_canopy_bulk_density_aligned.tif
-> Output raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\normalized_sources\canopy_bulk_density_hazard_score.tif
-> Normalization bounds: 0.0000 to 38.0000
-> Valid output pixels: 15,475,232
-> Values clipped below lower bound: 0
-> Values clipped above upper bound: 0
-> Output score range: 0.0000 to 1.0000
-> Output mean score: 0.1048
-> Valid pixels outside study area: 0
-> Normalization complete: True
-> Normalization manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_normalization_manifest.csv
-> Normalization summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard

,COMPONENT_ID,SOURCE_ID,PRODUCT_NAME,INPUT_PATH,OUTPUT_PATH,PROCESSING_ACTION,NORMALIZATION_METHOD,LOWER_BOUND,UPPER_BOUND,SOURCE_VALID_PIXELS,...,VALID_INSIDE_STUDY_AREA,VALID_OUTSIDE_STUDY_AREA,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,OUTPUT_SIZE_BYTES,OUTPUT_SIZE_MB,STRUCTURE_VALID,VALUE_RANGE_VALID,VALID
0,FUEL_HAZARD,LANDFIRE_CANOPY_BULK_DENSITY,Forest Canopy Bulk Density Hazard Score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused existing canopy-bulk-density hazard raster,direct_min_max,0.0,38.0,0,...,15475232,0,0.0,1.0,0.104826,29496670,28.130217,True,True,True



NOTE:
Forest canopy bulk density has been transformed to the common zero-to-one fuel-hazard scale.
Higher canopy-bulk-density values produce higher hazard scores under the direct min-max policy.
Both continuous canopy products are now normalized and ready for independent formula, grid, and content validation.

=== FOREST CANOPY BULK DENSITY NORMALIZATION COMPLETE ===


### Validating Normalized Canopy Fuel Layers


In [112]:
print('=== VALIDATING NORMALIZED CANOPY FUEL LAYERS ===')
# Record whether required canopy validation inputs satisfies the checks required before the workflow
# advances.
required_canopy_validation_inputs = ['fire_hazard_canopy_cover_normalization_complete',
    'fire_hazard_canopy_bulk_density_normalization_complete',
    'fire_hazard_canopy_cover_normalization_input_path',
    'fire_hazard_canopy_cover_normalization_output_path',
    'fire_hazard_canopy_bulk_density_normalization_input_path',
    'fire_hazard_canopy_bulk_density_normalization_output_path',
    'fire_hazard_canopy_cover_lower_bound', 'fire_hazard_canopy_cover_upper_bound',
    'fire_hazard_canopy_bulk_density_lower_bound',
    'fire_hazard_canopy_bulk_density_upper_bound',
    'fire_hazard_canopy_score_minimum', 'fire_hazard_canopy_score_maximum',
    'fire_hazard_canopy_score_nodata', 'fire_hazard_canopy_clip_normalized_scores',
    'fire_hazard_canopy_formula_tolerance', 'fire_hazard_canopy_value_tolerance',
    'fire_hazard_canopy_minimum_valid_pixels', 'fire_hazard_canopy_normalization_validation_path',
    'fire_hazard_canopy_normalization_validation_summary_path',
    'fire_hazard_fuel_normalization_manifest_path',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size']
# Identify missing canopy validation inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_canopy_validation_inputs = [object_name for object_name in \
    required_canopy_validation_inputs if object_name not in globals()]

# Stop execution if missing canopy validation inputs remain unresolved before this workflow stage
# begins.
if missing_canopy_validation_inputs:
    raise NameError(f'The following '
        f'canopy-validation objects are '
        f'missing:\n'
        f'{missing_canopy_validation_inputs}\n\n'
        f'Run Normalize Forest Canopy '
        f'Cover and Normalize Forest '
        f'Canopy Bulk Density before '
        f'validating the normalized '
        f'canopy products.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_canopy_cover_normalization_complete:
    raise ValueError('Forest canopy-cover normalization has not completed successfully.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_canopy_bulk_density_normalization_complete:
    raise ValueError('Forest canopy-bulk-density normalization has not completed successfully.')
# Evaluate fire hazard canopy validation inventory so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_canopy_validation_inventory = {'LANDFIRE_CANOPY_COVER': {'product_name': \
    'Forest Canopy Cover Hazard Score',
    'input_path': Path(fire_hazard_canopy_cover_normalization_input_path),
    'output_path': Path(fire_hazard_canopy_cover_normalization_output_path),
    'lower_bound': float(fire_hazard_canopy_cover_lower_bound),
    'upper_bound': float(fire_hazard_canopy_cover_upper_bound)},
    'LANDFIRE_CANOPY_BULK_DENSITY': {'product_name': 'Forest Canopy Bulk Density Hazard Score',
    'input_path': Path(fire_hazard_canopy_bulk_density_normalization_input_path),
    'output_path': Path(fire_hazard_canopy_bulk_density_normalization_output_path),
    'lower_bound': float(fire_hazard_canopy_bulk_density_lower_bound),
    'upper_bound': float(fire_hazard_canopy_bulk_density_upper_bound)}}
# Build the expected canopy validation mask shape mask used to isolate records required for this
# analysis.
expected_canopy_validation_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_canopy_validation_mask_shape:
    raise ValueError(f'The canopy-validation '
        f'study-area mask does not match '
        f'the common project grid.\nMask '
        f'shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_canopy_validation_mask_shape}')
# Evaluate fire hazard canopy validation study area pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_validation_study_area_pixels = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_canopy_validation_study_area_pixels == 0:
    raise ValueError('The canopy-validation study-area mask contains no included pixels.')
# Evaluate fire hazard canopy valid extensions so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_canopy_valid_extensions = {'.tif', '.tiff'}
# Calculate fire hazard canopy minimum file size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_canopy_minimum_file_size_bytes = 1024
# Evaluate fire hazard canopy validation records so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_canopy_validation_records = []

# Iterate through fire hazard canopy validation inventory.items so each required item receives the
# same processing and QA checks.
for source_id, product_configuration in fire_hazard_canopy_validation_inventory.items():
    # Prepare product name for the downstream processing or validation performed in this workflow
    # stage.
    product_name = product_configuration['product_name']
    # Build input path used to read, cache, or save this workflow product.
    input_path = Path(product_configuration['input_path'])
    # Build output path used to read, cache, or save this workflow product.
    output_path = Path(product_configuration['output_path'])
    # Prepare lower bound for the downstream processing or validation performed in this workflow
    # stage.
    lower_bound = float(product_configuration['lower_bound'])
    # Prepare upper bound for the downstream processing or validation performed in this workflow
    # stage.
    upper_bound = float(product_configuration['upper_bound'])
    # Prepare normalization range for the downstream processing or validation performed in this
    # workflow stage.
    normalization_range = upper_bound - lower_bound
    # Prepare input exists for the downstream processing or validation performed in this workflow
    # stage.
    input_exists = input_path.exists() and input_path.is_file()
    # Prepare output exists for the downstream processing or validation performed in this workflow
    # stage.
    output_exists = output_path.exists() and output_path.is_file()
    # Evaluate output extension valid so invalid inputs or outputs can be rejected before
    # continuing.
    output_extension_valid = output_path.suffix.lower() in fire_hazard_canopy_valid_extensions
    # Calculate output size bytes for completeness, file-integrity, or processing QA.
    output_size_bytes = output_path.stat().st_size if output_exists else 0
    # Evaluate output size valid so invalid inputs or outputs can be rejected before continuing.
    output_size_valid = output_size_bytes >= fire_hazard_canopy_minimum_file_size_bytes
    # Prepare input readable for the downstream processing or validation performed in this workflow
    # stage.
    input_readable = False
    # Prepare output readable for the downstream processing or validation performed in this workflow
    # stage.
    output_readable = False
    # Record whether source output grid match satisfies the checks required before the workflow
    # advances.
    source_output_grid_match = False
    # Evaluate output grid valid so invalid inputs or outputs can be rejected before continuing.
    output_grid_valid = False
    # Evaluate output structure valid so invalid inputs or outputs can be rejected before
    # continuing.
    output_structure_valid = False
    # Evaluate source valid pixels so invalid inputs or outputs can be rejected before continuing.
    source_valid_pixels = 0
    # Calculate source NoData pixels for completeness, file-integrity, or processing QA.
    source_nodata_pixels = 0
    # Evaluate output valid pixels so invalid inputs or outputs can be rejected before continuing.
    output_valid_pixels = 0
    # Calculate output NoData pixels for completeness, file-integrity, or processing QA.
    output_nodata_pixels = 0
    # Evaluate valid inside study area so invalid inputs or outputs can be rejected before
    # continuing.
    valid_inside_study_area = 0
    # Evaluate valid outside study area so invalid inputs or outputs can be rejected before
    # continuing.
    valid_outside_study_area = 0
    # Set output minimum as an explicit model or validation parameter used consistently in
    # downstream calculations.
    output_minimum = None
    # Set output maximum as an explicit model or validation parameter used consistently in
    # downstream calculations.
    output_maximum = None
    # Prepare output sum for the downstream processing or validation performed in this workflow
    # stage.
    output_sum = 0.0
    # Prepare output mean for the downstream processing or validation performed in this workflow
    # stage.
    output_mean = None
    # Calculate formula checked pixels for completeness, file-integrity, or processing QA.
    formula_checked_pixels = 0
    # Record whether formula matching pixels satisfies the checks required before the workflow
    # advances.
    formula_matching_pixels = 0
    # Calculate formula mismatch pixels for completeness, file-integrity, or processing QA.
    formula_mismatch_pixels = 0
    # Set maximum formula difference as an explicit model or validation parameter used consistently
    # in downstream calculations.
    maximum_formula_difference = 0.0
    # Prepare formula difference sum for the downstream processing or validation performed in this
    # workflow stage.
    formula_difference_sum = 0.0
    # Prepare mean formula difference for the downstream processing or validation performed in this
    # workflow stage.
    mean_formula_difference = None
    # Evaluate source nodata with valid output so invalid inputs or outputs can be rejected before
    # continuing.
    source_nodata_with_valid_output = 0
    # Evaluate valid source without output so invalid inputs or outputs can be rejected before
    # continuing.
    valid_source_without_output = 0
    # Calculate clipped low pixels for completeness, file-integrity, or processing QA.
    clipped_low_pixels = 0
    # Calculate clipped high pixels for completeness, file-integrity, or processing QA.
    clipped_high_pixels = 0
    # Evaluate validation error so invalid inputs or outputs can be rejected before continuing.
    validation_error = None

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if input_exists and output_exists and output_extension_valid and output_size_valid:

        # Document this operation so its role in the current fuel-hazard workflow is clear before
        # processing continues.
        try:

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(input_path) as source_raster:

                # Open the file in a managed context so required content is processed and the
                # resource closes cleanly.
                with rasterio.open(output_path) as output_raster:
                    # Prepare input readable for the downstream processing or validation performed
                    # in this workflow stage.
                    input_readable = True
                    # Prepare output readable for the downstream processing or validation performed
                    # in this workflow stage.
                    output_readable = True
                    # Evaluate output grid valid so invalid inputs or outputs can be rejected before
                    # continuing.
                    output_grid_valid = all([output_raster.width == fire_hazard_alignment_width,
                        output_raster.height == fire_hazard_alignment_height,
                        output_raster.crs is not None, output_raster.crs == \
                            rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                        output_raster.transform.almost_equals(fire_hazard_alignment_transform),
                        np.isclose(abs(output_raster.transform.a), fire_hazard_cell_size,
                        atol=fire_hazard_canopy_value_tolerance), \
                            np.isclose(abs(output_raster.transform.e),
                        fire_hazard_cell_size, atol=fire_hazard_canopy_value_tolerance)])
                    # Evaluate output structure valid so invalid inputs or outputs can be rejected
                    # before continuing.
                    output_structure_valid = all([output_raster.count == 1,
                        output_raster.dtypes[0] == 'float32', output_raster.nodata == \
                            fire_hazard_canopy_score_nodata])
                    # Record whether source output grid match satisfies the checks required before
                    # the workflow advances.
                    source_output_grid_match = all([source_raster.count == 1,
                        source_raster.width == output_raster.width, source_raster.height == \
                            output_raster.height,
                        source_raster.crs == output_raster.crs, \
                            source_raster.transform.almost_equals(output_raster.transform)])

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if not source_output_grid_match:
                        raise ValueError(f'{product_name} does not match '
                            f'its aligned source raster '
                            f'grid.')

                    # Iterate through output raster.block windows so each required item receives the
                    # same processing and QA checks.
                    for _, raster_window in output_raster.block_windows(1):
                        # Prepare source array for the raster calculation or validation performed in
                        # this processing block.
                        source_array = source_raster.read(1,
                            window=raster_window, out_dtype='float32')
                        # Prepare output array for the raster calculation or validation performed in
                        # this processing block.
                        output_array = output_raster.read(1, window=raster_window)
                        # Prepare row start for the downstream processing or validation performed in
                        # this workflow stage.
                        row_start = int(raster_window.row_off)
                        # Prepare row end for the downstream processing or validation performed in
                        # this workflow stage.
                        row_end = int(raster_window.row_off + raster_window.height)
                        # Prepare column start for the downstream processing or validation performed
                        # in this workflow stage.
                        column_start = int(raster_window.col_off)
                        # Prepare column end for the downstream processing or validation performed
                        # in this workflow stage.
                        column_end = int(raster_window.col_off + raster_window.width)
                        # Build the study area window mask mask used to isolate records required for
                        # this analysis.
                        study_area_window_mask = \
                            fire_hazard_alignment_study_area_mask[row_start:row_end,
                            column_start:column_end]
                        # Build the source valid mask mask used to isolate records required for this
                        # analysis.
                        source_valid_mask = np.isfinite(source_array)

                        # Stop execution if the NoData configuration is incompatible with the raster
                        # product being validated.
                        if source_raster.nodata is not None:
                            # Build the source valid mask mask used to isolate records required for
                            # this analysis.
                            source_valid_mask &= source_array != source_raster.nodata
                        # Build the source valid inside mask mask used to isolate records required
                        # for this analysis.
                        source_valid_inside_mask = source_valid_mask & study_area_window_mask
                        # Evaluate source valid pixels so invalid inputs or outputs can be rejected
                        # before continuing.
                        source_valid_pixels += int(source_valid_inside_mask.sum())
                        # Calculate source NoData pixels for completeness, file-integrity, or
                        # processing QA.
                        source_nodata_pixels += int((~source_valid_mask).sum())
                        # Build the output valid mask mask used to isolate records required for this
                        # analysis.
                        output_valid_mask = np.isfinite(output_array) & (output_array != \
                            fire_hazard_canopy_score_nodata)
                        # Prepare output values for the downstream processing or validation
                        # performed in this workflow stage.
                        output_values = output_array[output_valid_mask]
                        # Evaluate output valid pixels so invalid inputs or outputs can be rejected
                        # before continuing.
                        output_valid_pixels += int(output_valid_mask.sum())
                        # Calculate output NoData pixels for completeness, file-integrity, or
                        # processing QA.
                        output_nodata_pixels += int((~output_valid_mask).sum())
                        # Evaluate valid inside study area so invalid inputs or outputs can be
                        # rejected before continuing.
                        valid_inside_study_area += int((output_valid_mask & \
                            study_area_window_mask).sum())
                        # Evaluate valid outside study area so invalid inputs or outputs can be
                        # rejected before continuing.
                        valid_outside_study_area += int((output_valid_mask & \
                            ~study_area_window_mask).sum())

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if output_values.size > 0:
                            # Set block minimum as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            block_minimum = float(output_values.min())
                            # Set block maximum as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            block_maximum = float(output_values.max())

                            # Stop execution if this validation condition is not satisfied before
                            # dependent processing continues.
                            if output_minimum is None:
                                # Set output minimum as an explicit model or validation parameter
                                # used consistently in downstream calculations.
                                output_minimum = block_minimum
                            else:
                                # Set output minimum as an explicit model or validation parameter
                                # used consistently in downstream calculations.
                                output_minimum = min(output_minimum, block_minimum)

                            # Stop execution if this validation condition is not satisfied before
                            # dependent processing continues.
                            if output_maximum is None:
                                # Set output maximum as an explicit model or validation parameter
                                # used consistently in downstream calculations.
                                output_maximum = block_maximum
                            else:
                                # Set output maximum as an explicit model or validation parameter
                                # used consistently in downstream calculations.
                                output_maximum = max(output_maximum, block_maximum)
                            # Prepare output sum for the downstream processing or validation
                            # performed in this workflow stage.
                            output_sum += float(output_values.sum(dtype=np.float64))

                        # Stop execution if the prerequisite validation has not passed before this
                        # workflow stage continues.
                        if source_valid_inside_mask.any():
                            # Prepare source values for the downstream processing or validation
                            # performed in this workflow stage.
                            source_values = \
                                source_array[source_valid_inside_mask].astype(np.float32)
                            # Prepare actual scores for the fuel-hazard calculation and subsequent
                            # QA checks.
                            actual_scores = output_array[source_valid_inside_mask]
                            # Prepare expected scores for the fuel-hazard calculation and subsequent
                            # QA checks.
                            expected_scores = (source_values - lower_bound) / normalization_range

                            # Stop execution if this validation condition is not satisfied before
                            # dependent processing continues.
                            if fire_hazard_canopy_clip_normalized_scores:
                                # Prepare expected scores for the fuel-hazard calculation and
                                # subsequent QA checks.
                                expected_scores = np.clip(expected_scores,
                                    fire_hazard_canopy_score_minimum, \
                                        fire_hazard_canopy_score_maximum)
                            # Prepare expected scores for the fuel-hazard calculation and subsequent
                            # QA checks.
                            expected_scores = expected_scores.astype(np.float32)
                            # Calculate clipped low pixels for completeness, file-integrity, or
                            # processing QA.
                            clipped_low_pixels += int((source_values < lower_bound).sum())
                            # Calculate clipped high pixels for completeness, file-integrity, or
                            # processing QA.
                            clipped_high_pixels += int((source_values > upper_bound).sum())
                            # Build the actual score valid mask mask used to isolate records
                            # required for this analysis.
                            actual_score_valid_mask = np.isfinite(actual_scores) & (actual_scores \
                                != fire_hazard_canopy_score_nodata)
                            # Evaluate valid source without output so invalid inputs or outputs can
                            # be rejected before continuing.
                            valid_source_without_output += int((~actual_score_valid_mask).sum())

                            # Stop execution if the prerequisite validation has not passed before
                            # this workflow stage continues.
                            if actual_score_valid_mask.any():
                                # Evaluate expected valid scores so invalid inputs or outputs can be
                                # rejected before continuing.
                                expected_valid_scores = expected_scores[actual_score_valid_mask]
                                # Evaluate actual valid scores so invalid inputs or outputs can be
                                # rejected before continuing.
                                actual_valid_scores = actual_scores[actual_score_valid_mask]
                                # Prepare absolute differences for the downstream processing or
                                # validation performed in this workflow stage.
                                absolute_differences = np.abs(actual_valid_scores - \
                                    expected_valid_scores)
                                # Build the formula match mask mask used to isolate records required
                                # for this analysis.
                                formula_match_mask = absolute_differences <= \
                                    fire_hazard_canopy_formula_tolerance
                                # Calculate formula checked pixels for completeness, file-integrity,
                                # or processing QA.
                                formula_checked_pixels += int(absolute_differences.size)
                                # Record whether formula matching pixels satisfies the checks
                                # required before the workflow advances.
                                formula_matching_pixels += int(formula_match_mask.sum())
                                # Calculate formula mismatch pixels for completeness,
                                # file-integrity, or processing QA.
                                formula_mismatch_pixels += int((~formula_match_mask).sum())
                                # Prepare formula difference sum for the downstream processing or
                                # validation performed in this workflow stage.
                                formula_difference_sum += \
                                    float(absolute_differences.sum(dtype=np.float64))

                                # Stop execution if this validation condition is not satisfied
                                # before dependent processing continues.
                                if absolute_differences.size > 0:
                                    # Set maximum formula difference as an explicit model or
                                    # validation parameter used consistently in downstream
                                    # calculations.
                                    maximum_formula_difference = max(maximum_formula_difference,
                                        float(absolute_differences.max()))
                        # Evaluate source nodata with valid output so invalid inputs or outputs can
                        # be rejected before continuing.
                        source_nodata_with_valid_output += int((~source_valid_mask & \
                            output_valid_mask).sum())
                        del source_array
                        del output_array
        # Handle the expected failure explicitly so the workflow can report or clean up the affected
        # operation.
        except Exception as error:
            # Evaluate validation error so invalid inputs or outputs can be rejected before
            # continuing.
            validation_error = str(error)

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if output_valid_pixels > 0:
        # Prepare output mean for the downstream processing or validation performed in this workflow
        # stage.
        output_mean = output_sum / output_valid_pixels

    # Stop execution if this validation condition is not satisfied before dependent processing
    # continues.
    if formula_checked_pixels > 0:
        # Prepare mean formula difference for the downstream processing or validation performed in
        # this workflow stage.
        mean_formula_difference = formula_difference_sum / formula_checked_pixels
    # Prepare value range available for the downstream processing or validation performed in this
    # workflow stage.
    value_range_available = output_minimum is not None and output_maximum is not None and \
        (output_mean is not None)
    # Evaluate value range valid so invalid inputs or outputs can be rejected before continuing.
    value_range_valid = value_range_available and output_minimum >= \
        fire_hazard_canopy_score_minimum - fire_hazard_canopy_value_tolerance and (output_maximum \
        <= fire_hazard_canopy_score_maximum + fire_hazard_canopy_value_tolerance)
    # Evaluate formula valid so invalid inputs or outputs can be rejected before continuing.
    formula_valid = all([formula_checked_pixels >= fire_hazard_canopy_minimum_valid_pixels,
        formula_mismatch_pixels == 0, maximum_formula_difference <= \
            fire_hazard_canopy_formula_tolerance,
        source_nodata_with_valid_output == 0, valid_source_without_output == 0])
    # Evaluate product valid so invalid inputs or outputs can be rejected before continuing.
    product_valid = all([input_exists, output_exists,
        output_extension_valid, output_size_valid, input_readable,
        output_readable, source_output_grid_match, output_grid_valid,
        output_structure_valid, output_valid_pixels >= fire_hazard_canopy_minimum_valid_pixels,
        valid_inside_study_area > 0, valid_outside_study_area == 0,
        value_range_valid, formula_valid, validation_error is None])
    # Add the current record to fire hazard canopy validation records so the stage summary captures
    # this processing result.
    fire_hazard_canopy_validation_records.append({'COMPONENT_ID': 'FUEL_HAZARD',
        'SOURCE_ID': source_id, 'PRODUCT_NAME': product_name,
        'INPUT_PATH': str(input_path), 'OUTPUT_PATH': str(output_path),
        'FILE_EXISTS': output_exists, 'FILE_EXTENSION_VALID': output_extension_valid,
        'FILE_SIZE_BYTES': output_size_bytes, 'FILE_SIZE_MB': output_size_bytes / 1024 ** 2,
        'FILE_SIZE_VALID': output_size_valid, 'INPUT_READABLE': input_readable,
        'OUTPUT_READABLE': output_readable, 'SOURCE_OUTPUT_GRID_MATCH': source_output_grid_match,
        'OUTPUT_GRID_VALID': output_grid_valid, 'OUTPUT_STRUCTURE_VALID': output_structure_valid,
        'LOWER_BOUND': lower_bound, 'UPPER_BOUND': upper_bound,
        'SOURCE_VALID_PIXELS': source_valid_pixels, 'SOURCE_NODATA_PIXELS': source_nodata_pixels,
        'OUTPUT_VALID_PIXELS': output_valid_pixels, 'OUTPUT_NODATA_PIXELS': output_nodata_pixels,
        'VALID_INSIDE_STUDY_AREA': valid_inside_study_area,
        'VALID_OUTSIDE_STUDY_AREA': valid_outside_study_area,
        'CLIPPED_LOW_PIXELS': clipped_low_pixels, 'CLIPPED_HIGH_PIXELS': clipped_high_pixels,
        'OUTPUT_MINIMUM': output_minimum, 'OUTPUT_MAXIMUM': output_maximum,
        'OUTPUT_MEAN': output_mean, 'VALUE_RANGE_AVAILABLE': value_range_available,
        'VALUE_RANGE_VALID': value_range_valid, 'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
        'FORMULA_MATCHING_PIXELS': formula_matching_pixels,
        'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
        'MAXIMUM_FORMULA_DIFFERENCE': maximum_formula_difference,
        'MEAN_FORMULA_DIFFERENCE': mean_formula_difference,
        'SOURCE_NODATA_WITH_VALID_OUTPUT': source_nodata_with_valid_output,
        'VALID_SOURCE_WITHOUT_OUTPUT': valid_source_without_output,
        'FORMULA_VALID': formula_valid, 'ERROR_MESSAGE': validation_error,
        'VALID': product_valid})
# Evaluate fire hazard canopy normalization validation so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_canopy_normalization_validation = \
    pd.DataFrame(fire_hazard_canopy_validation_records).sort_values('SOURCE_ID').reset_index(drop= \
    True)
# Define canopy output grid signatures used to configure this workflow stage.
canopy_output_grid_signatures = []

# Iterate through fire hazard canopy validation inventory.items so each required item receives the
# same processing and QA checks.
for source_id, product_configuration in fire_hazard_canopy_validation_inventory.items():
    # Build output path used to read, cache, or save this workflow product.
    output_path = Path(product_configuration['output_path'])

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(output_path) as output_raster:
        # Add the current record to canopy output grid signatures so the stage summary captures this
        # processing result.
        canopy_output_grid_signatures.append({'SOURCE_ID': source_id,
            'WIDTH': output_raster.width, 'HEIGHT': output_raster.height,
            'CRS': output_raster.crs.to_string() if output_raster.crs is not None else None,
            'TRANSFORM': tuple(output_raster.transform)})
# Prepare reference canopy grid for the downstream processing or validation performed in this
# workflow stage.
reference_canopy_grid = canopy_output_grid_signatures[0]
# Record whether fire hazard normalized canopy grids match satisfies the checks required before the
# workflow advances.
fire_hazard_normalized_canopy_grids_match = all((grid_signature['WIDTH'] == \
    reference_canopy_grid['WIDTH'] and grid_signature['HEIGHT'] == \
    reference_canopy_grid['HEIGHT'] and (grid_signature['CRS'] == reference_canopy_grid['CRS']) \
    and rasterio.Affine(*grid_signature['TRANSFORM']).almost_equals(rasterio.Affine \
    (*reference_canopy_grid['TRANSFORM'])) for grid_signature in canopy_output_grid_signatures))
# Prepare canopy normalization manifest exists for the downstream processing or validation performed
# in this workflow stage.
canopy_normalization_manifest_exists = Path(fire_hazard_fuel_normalization_manifest_path).exists()
# Evaluate canopy normalization manifest valid so invalid inputs or outputs can be rejected before
# continuing.
canopy_normalization_manifest_valid = False

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if canopy_normalization_manifest_exists:
    # Assemble saved canopy normalization manifest into a table for QA review and downstream
    # validation.
    saved_canopy_normalization_manifest = pd.read_csv(fire_hazard_fuel_normalization_manifest_path)
    # Define the manifest source ids inputs required by this workflow stage.
    required_manifest_source_ids = {'LANDFIRE_CANOPY_COVER', 'LANDFIRE_CANOPY_BULK_DENSITY'}

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if 'SOURCE_ID' in saved_canopy_normalization_manifest.columns and 'VALID' in \
        saved_canopy_normalization_manifest.columns:
        # Prepare canopy manifest records for the downstream processing or validation performed in
        # this workflow stage.
        canopy_manifest_records = \
            saved_canopy_normalization_manifest[saved_canopy_normalization_manifest['SOURCE_ID'] \
            .isin(required_manifest_source_ids)].copy()
        # Record whether canopy manifest valid values satisfies the checks required before the
        # workflow advances.
        canopy_manifest_valid_values = \
            canopy_manifest_records['VALID'].astype(str).str.strip().str.lower().map({'true': True,
            'false': False})
        # Record whether canopy normalization manifest valid satisfies the checks required before
        # the workflow advances.
        canopy_normalization_manifest_valid = all([len(canopy_manifest_records) == 2,
            set(canopy_manifest_records['SOURCE_ID']) == required_manifest_source_ids,
            canopy_manifest_valid_values.notna().all(), canopy_manifest_valid_values.all()])
# Evaluate failed canopy normalization validations so invalid inputs or outputs can be rejected
# before continuing.
failed_canopy_normalization_validations = \
    fire_hazard_canopy_normalization_validation[~fire_hazard_canopy_normalization_validation \
    ['VALID']].copy().reset_index(drop=True)
# Calculate expected canopy product count for completeness, file-integrity, or processing QA.
expected_canopy_product_count = 2
# Calculate validated canopy product count for completeness, file-integrity, or processing QA.
validated_canopy_product_count = len(fire_hazard_canopy_normalization_validation)
# Calculate valid canopy product count for completeness, file-integrity, or processing QA.
valid_canopy_product_count = int(fire_hazard_canopy_normalization_validation['VALID'].sum())
# Calculate failed canopy product count for completeness, file-integrity, or processing QA.
failed_canopy_product_count = len(failed_canopy_normalization_validations)
# Evaluate fire hazard canopy normalization validation complete so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_canopy_normalization_validation_complete = all([validated_canopy_product_count == \
    expected_canopy_product_count,
    valid_canopy_product_count == expected_canopy_product_count,
    failed_canopy_product_count == 0, fire_hazard_normalized_canopy_grids_match,
    canopy_normalization_manifest_valid, \
        fire_hazard_canopy_normalization_validation['OUTPUT_GRID_VALID'].all(),
    fire_hazard_canopy_normalization_validation['OUTPUT_STRUCTURE_VALID'].all(),
    fire_hazard_canopy_normalization_validation['VALUE_RANGE_VALID'].all(),
    fire_hazard_canopy_normalization_validation['FORMULA_VALID'].all(),
    (fire_hazard_canopy_normalization_validation['VALID_OUTSIDE_STUDY_AREA'] == 0).all()])
# Calculate total canopy output size bytes for completeness, file-integrity, or processing QA.
total_canopy_output_size_bytes = \
    int(fire_hazard_canopy_normalization_validation['FILE_SIZE_BYTES'].sum())
# Assemble fire hazard canopy normalization validation summary into a table for QA review and
# downstream validation.
fire_hazard_canopy_normalization_validation_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'EXPECTED_PRODUCTS': expected_canopy_product_count,
    'VALIDATED_PRODUCTS': validated_canopy_product_count,
    'VALID_PRODUCTS': valid_canopy_product_count, 'FAILED_PRODUCTS': failed_canopy_product_count,
    'ALL_GRIDS_MATCH': fire_hazard_normalized_canopy_grids_match,
    'NORMALIZATION_MANIFEST_VALID': canopy_normalization_manifest_valid,
    'ALL_VALUE_RANGES_VALID': \
        bool(fire_hazard_canopy_normalization_validation['VALUE_RANGE_VALID'].all()),
    'ALL_FORMULAS_VALID': bool(fire_hazard_canopy_normalization_validation['FORMULA_VALID'].all()),
    'TOTAL_FORMULA_MISMATCH_PIXELS': \
        int(fire_hazard_canopy_normalization_validation['FORMULA_MISMATCH_PIXELS'].sum()),
    'TOTAL_VALID_OUTSIDE_STUDY_AREA': \
        int(fire_hazard_canopy_normalization_validation['VALID_OUTSIDE_STUDY_AREA'].sum()),
    'TOTAL_OUTPUT_SIZE_BYTES': total_canopy_output_size_bytes,
    'TOTAL_OUTPUT_SIZE_MB': total_canopy_output_size_bytes / 1024 ** 2,
    'VALIDATION_COMPLETE': fire_hazard_canopy_normalization_validation_complete}])
# Build fire hazard canopy failed validation path used to read, cache, or save this workflow
# product.
fire_hazard_canopy_failed_validation_path = fire_hazard_fuel_metadata_directory / \
    'canopy_normalization_failed_validation.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_canopy_normalization_validation.to_csv(fire_hazard_canopy_normalization_validation_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_canopy_normalization_validation_summary.to_csv \
    (fire_hazard_canopy_normalization_validation_summary_path,
    index=False)
# Export the table so this workflow result is available to later phases.
failed_canopy_normalization_validations.to_csv(fire_hazard_canopy_failed_validation_path,
    index=False)
print(f'-> Expected canopy products: {expected_canopy_product_count}')
print(f'-> Validated canopy products: {validated_canopy_product_count}')
print(f'-> Valid canopy products: {valid_canopy_product_count}')
print(f'-> Failed canopy products: {failed_canopy_product_count}')
print(f'-> Normalized canopy grids match: {fire_hazard_normalized_canopy_grids_match}')
print(f'-> Normalization manifest valid: {canopy_normalization_manifest_valid}')
print(f"-> Formula mismatch pixels: "
    f"{int(fire_hazard_canopy_normalization_validation['FORMULA_MISMATCH_PIXELS'].sum()):,}")
print(f"-> Valid pixels outside study "
    f"area: "
    f"{int(fire_hazard_canopy_normalization_validation['VALID_OUTSIDE_STUDY_AREA'].sum()):,}")
print(f'-> Canopy validation complete: {fire_hazard_canopy_normalization_validation_complete}')
print(f'-> Detailed validation saved: {fire_hazard_canopy_normalization_validation_path}')
print(f'-> Validation summary saved: {fire_hazard_canopy_normalization_validation_summary_path}')
print('\n--- NORMALIZED CANOPY VALIDATION SUMMARY ---')
display(fire_hazard_canopy_normalization_validation_summary)
print('\n--- NORMALIZED CANOPY PRODUCT VALIDATION ---')
display(fire_hazard_canopy_normalization_validation[['SOURCE_ID',
    'PRODUCT_NAME', 'OUTPUT_GRID_VALID', 'OUTPUT_STRUCTURE_VALID',
    'OUTPUT_VALID_PIXELS', 'VALID_OUTSIDE_STUDY_AREA',
    'OUTPUT_MINIMUM', 'OUTPUT_MAXIMUM', 'OUTPUT_MEAN',
    'FORMULA_CHECKED_PIXELS', 'FORMULA_MISMATCH_PIXELS',
    'MAXIMUM_FORMULA_DIFFERENCE', 'VALUE_RANGE_VALID',
    'FORMULA_VALID', 'VALID']])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not failed_canopy_normalization_validations.empty:
    print('\n--- FAILED CANOPY VALIDATION RECORDS ---')
    display(failed_canopy_normalization_validations.head(5))

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_canopy_normalization_validation_complete:
    raise ValueError(f"One or more normalized canopy "
        f"fuel layers failed independent "
        f"validation.\n\nExpected "
        f"products: "
        f"{expected_canopy_product_count}\n"
        f"Valid products: "
        f"{valid_canopy_product_count}\n"
        f"Failed products: "
        f"{failed_canopy_product_count}\n"
        f"All grids match: "
        f"{fire_hazard_normalized_canopy_grids_match}\n"
        f"Manifest valid: "
        f"{canopy_normalization_manifest_valid}\n"
        f"Formula mismatch pixels: "
        f"{int(fire_hazard_canopy_normalization_validation['FORMULA_MISMATCH_PIXELS'].sum()):,}\n"
        f"Valid outside study area: "
        f"{int(fire_hazard_canopy_normalization_validation['VALID_OUTSIDE_STUDY_AREA'].sum()):,}\n\n"
        f"Review the saved canopy "
        f"normalization validation "
        f"files.")
# Evaluate fire hazard normalized canopy layers valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_normalized_canopy_layers_valid = fire_hazard_canopy_normalization_validation_complete
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The normalized canopy-cover '
    'and canopy-bulk-density '
    'rasters passed independent '
    'file, grid, structure, '
    'study-area, score-range, and '
    'min-max formula validation.')
print('Both products now use the '
    'common zero-to-one fuel-hazard '
    'scale and are ready for '
    'inclusion in the composite '
    'fuel-hazard component.')
print('\n=== NORMALIZED CANOPY FUEL-LAYER VALIDATION COMPLETE ===')


=== VALIDATING NORMALIZED CANOPY FUEL LAYERS ===
-> Expected canopy products: 2
-> Validated canopy products: 2
-> Valid canopy products: 2
-> Failed canopy products: 0
-> Normalized canopy grids match: True
-> Normalization manifest valid: True
-> Formula mismatch pixels: 0
-> Valid pixels outside study area: 0
-> Canopy validation complete: True
-> Detailed validation saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\canopy_normalization_validation.csv
-> Validation summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\canopy_normalization_validation_summary.csv

--- NORMALIZED CANOPY VALIDATION SUMMARY ---


,COMPONENT_ID,EXPECTED_PRODUCTS,VALIDATED_PRODUCTS,VALID_PRODUCTS,FAILED_PRODUCTS,ALL_GRIDS_MATCH,NORMALIZATION_MANIFEST_VALID,ALL_VALUE_RANGES_VALID,ALL_FORMULAS_VALID,TOTAL_FORMULA_MISMATCH_PIXELS,TOTAL_VALID_OUTSIDE_STUDY_AREA,TOTAL_OUTPUT_SIZE_BYTES,TOTAL_OUTPUT_SIZE_MB,VALIDATION_COMPLETE
0,FUEL_HAZARD,2,2,2,0,True,True,True,True,0,0,59038700,56.303692,True



--- NORMALIZED CANOPY PRODUCT VALIDATION ---


,SOURCE_ID,PRODUCT_NAME,OUTPUT_GRID_VALID,OUTPUT_STRUCTURE_VALID,OUTPUT_VALID_PIXELS,VALID_OUTSIDE_STUDY_AREA,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,FORMULA_CHECKED_PIXELS,FORMULA_MISMATCH_PIXELS,MAXIMUM_FORMULA_DIFFERENCE,VALUE_RANGE_VALID,FORMULA_VALID,VALID
0,LANDFIRE_CANOPY_BULK_DENSITY,Forest Canopy Bulk Density Hazard Score,True,True,15475232,0,0.0,1.0,0.104826,15475232,0,0.0,True,True,True
1,LANDFIRE_CANOPY_COVER,Forest Canopy Cover Hazard Score,True,True,15475232,0,0.0,1.0,0.122386,15475232,0,0.0,True,True,True



NOTE:
The normalized canopy-cover and canopy-bulk-density rasters passed independent file, grid, structure, study-area, score-range, and min-max formula validation.
Both products now use the common zero-to-one fuel-hazard scale and are ready for inclusion in the composite fuel-hazard component.

=== NORMALIZED CANOPY FUEL-LAYER VALIDATION COMPLETE ===


### Configuring Fuel-Hazard Component Weights


In [113]:
print('=== CONFIGURING FUEL-HAZARD COMPONENT WEIGHTS ===')

# Combine surface-fuel and canopy-fuel indicators using the
# configured internal weights.
# Set required fuel weight inputs as an explicit model or validation parameter used consistently in
# downstream calculations.
required_fuel_weight_inputs = ['fire_hazard_fbfm40_product_finalized',
    'fire_hazard_fbfm40_hazard_valid', 'fire_hazard_fbfm40_hazard_path',
    'fire_hazard_normalized_canopy_layers_valid', \
        'fire_hazard_canopy_normalization_validation_complete',
    'fire_hazard_canopy_cover_normalization_output_path',
    'fire_hazard_canopy_bulk_density_normalization_output_path',
    'fire_hazard_fuel_hazard_score_path', 'fire_hazard_fuel_score_profile',
    'fire_hazard_fuel_score_minimum', 'fire_hazard_fuel_score_maximum',
    'fire_hazard_fuel_score_nodata', 'fire_hazard_fuel_component_manifest_path',
    'fire_hazard_fuel_validation_path', 'fire_hazard_fuel_validation_summary_path',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_fuel_reuse_existing',
    'fire_hazard_fuel_overwrite', 'fire_hazard_fuel_processing_window_size',
    'fire_hazard_fuel_progress_interval', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size']
# Identify missing fuel weight inputs so unavailable prerequisites are caught before this workflow
# stage runs.
missing_fuel_weight_inputs = [object_name for object_name in required_fuel_weight_inputs if \
    object_name not in globals()]

# Stop execution if missing fuel weight inputs remain unresolved before this workflow stage begins.
if missing_fuel_weight_inputs:
    raise NameError(f'The following fuel-weight '
        f'configuration objects are '
        f'missing:\n'
        f'{missing_fuel_weight_inputs}\n\n'
        f'Finalize the FBFM40 hazard '
        f'product and validate both '
        f'normalized canopy layers '
        f'before configuring component '
        f'weights.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fbfm40_product_finalized:
    raise ValueError('The FBFM40 fuel-model hazard product has not been finalized.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fbfm40_hazard_valid:
    raise ValueError('The FBFM40 hazard raster has not passed independent validation.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_normalized_canopy_layers_valid:
    raise ValueError('The normalized canopy fuel layers have not passed independent validation.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_canopy_normalization_validation_complete:
    raise ValueError('Canopy normalization validation is incomplete.')
# Build fire hazard fuel component input paths used to read, cache, or save this workflow product.
fire_hazard_fuel_component_input_paths = {'FBFM40_HAZARD': Path(fire_hazard_fbfm40_hazard_path),
    'CANOPY_COVER_HAZARD': Path(fire_hazard_canopy_cover_normalization_output_path),
    'CANOPY_BULK_DENSITY_HAZARD': Path(fire_hazard_canopy_bulk_density_normalization_output_path)}

# Iterate through fire hazard fuel component input paths.items so each required item receives the
# same processing and QA checks.
for component_id, component_path in fire_hazard_fuel_component_input_paths.items():

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not component_path.exists() or not component_path.is_file() or \
        component_path.stat().st_size <= 0:
        raise FileNotFoundError(f'A required fuel-hazard '
            f'component raster is '
            f'unavailable:\nComponent: '
            f'{component_id}\nPath: '
            f'{component_path}')
# Define fire hazard fuel component weights used to combine component scores in the hazard
# calculation.
fire_hazard_fuel_component_weights = {'FBFM40_HAZARD': 0.5,
    'CANOPY_COVER_HAZARD': 0.25, 'CANOPY_BULK_DENSITY_HAZARD': 0.25}
# Prepare fuel input component IDs for the downstream processing or validation performed in this
# workflow stage.
fuel_input_component_ids = set(fire_hazard_fuel_component_input_paths.keys())
# Set fuel weight component IDs as an explicit model or validation parameter used consistently in
# downstream calculations.
fuel_weight_component_ids = set(fire_hazard_fuel_component_weights.keys())

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fuel_input_component_ids != fuel_weight_component_ids:
    raise ValueError(f'The fuel-component input paths '
        f'and weight configuration do '
        f'not contain matching component '
        f'identifiers.\nInput '
        f'components: '
        f'{sorted(fuel_input_component_ids)}\n'
        f'Weighted components: '
        f'{sorted(fuel_weight_component_ids)}')
# Evaluate invalid fuel component weights so invalid inputs or outputs can be rejected before
# continuing.
invalid_fuel_component_weights = {component_id: component_weight for component_id,
    component_weight in fire_hazard_fuel_component_weights.items() if not \
        np.isfinite(component_weight) or component_weight <= 0.0 or component_weight > 1.0}

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if invalid_fuel_component_weights:
    raise ValueError(f'One or more fuel-component '
        f'weights are invalid:\n'
        f'{invalid_fuel_component_weights}')
# Set fire hazard fuel component weight sum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_component_weight_sum = float(sum(fire_hazard_fuel_component_weights.values()))
# Set fire hazard fuel component weight tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_component_weight_tolerance = 1e-09
# Evaluate fire hazard fuel component weights valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_component_weights_valid = np.isclose(fire_hazard_fuel_component_weight_sum,
    1.0, atol=fire_hazard_fuel_component_weight_tolerance)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_component_weights_valid:
    raise ValueError(f'The fuel-component weights '
        f'must sum to 1.0.\nCurrent '
        f'weight sum: '
        f'{fire_hazard_fuel_component_weight_sum:.12f}')
# Prepare fire hazard fuel component formula for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_formula = (
    '(FBFM40_HAZARD * 0.50) + '
        '(CANOPY_COVER_HAZARD * 0.25) + '
        '(CANOPY_BULK_DENSITY_HAZARD * '
        '0.25)'
)
# Prepare fire hazard fuel component method for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_method = 'weighted_linear_combination'
# Prepare fire hazard fuel require all components for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_require_all_components = True
# Prepare fire hazard fuel missing component action for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fuel_missing_component_action = 'assign_nodata'
# Prepare fire hazard fuel clip composite scores for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_clip_composite_scores = True
# Build fire hazard fuel component output path used to read, cache, or save this workflow product.
fire_hazard_fuel_component_output_path = Path(fire_hazard_fuel_hazard_score_path)
# Build fire hazard fuel component temporary path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_temporary_path = fire_hazard_fuel_component_output_path.parent / \
    (fire_hazard_fuel_component_output_path.stem + '.part.tif')
# Build fire hazard fuel component weight table path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_weight_table_path = fire_hazard_fuel_metadata_directory / \
    'fuel_component_weights.csv'
# Build fire hazard fuel component policy path used to read, cache, or save this workflow product.
fire_hazard_fuel_component_policy_path = fire_hazard_fuel_metadata_directory / \
    'fuel_component_weighting_policy.json'
# Build fire hazard fuel component configuration summary path used to read, cache, or save this
# workflow product.
fire_hazard_fuel_component_configuration_summary_path = fire_hazard_fuel_metadata_directory / \
    'fuel_component_weighting_summary.csv'
# Prepare fire hazard fuel component output profile so raster outputs inherit the required grid,
# CRS, data type, and NoData metadata.
fire_hazard_fuel_component_output_profile = fire_hazard_fuel_score_profile.copy()
# Update fire hazard fuel component output profile with the values produced by the current
# processing step.
fire_hazard_fuel_component_output_profile.update({'driver': 'GTiff',
    'dtype': 'float32', 'count': 1, 'nodata': fire_hazard_fuel_score_nodata,
    'width': fire_hazard_alignment_width, 'height': fire_hazard_alignment_height,
    'crs': fire_hazard_target_crs, 'transform': fire_hazard_alignment_transform,
    'compress': 'deflate', 'predictor': 3, 'tiled': True,
    'BIGTIFF': 'IF_SAFER'})
# Initialize fire hazard fuel component grid records to collect consistent records for the stage
# summary and QA checks.
fire_hazard_fuel_component_grid_records = []

# Iterate through fire hazard fuel component input paths.items so each required item receives the
# same processing and QA checks.
for component_id, component_path in fire_hazard_fuel_component_input_paths.items():

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(component_path) as component_source:
        # Evaluate component grid valid so invalid inputs or outputs can be rejected before
        # continuing.
        component_grid_valid = all([component_source.count == 1,
            component_source.width == fire_hazard_alignment_width,
            component_source.height == fire_hazard_alignment_height,
            component_source.crs is not None, component_source.crs == \
                rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
            component_source.transform.almost_equals(fire_hazard_alignment_transform),
            component_source.dtypes[0] == 'float32', component_source.nodata == \
                fire_hazard_fuel_score_nodata])
        # Add the current record to fire hazard fuel component grid records so the stage summary
        # captures this processing result.
        fire_hazard_fuel_component_grid_records.append({'COMPONENT_ID': component_id,
            'INPUT_PATH': str(component_path), 'WIDTH': component_source.width,
            'HEIGHT': component_source.height, 'CRS': component_source.crs.to_string() if \
                component_source.crs is not None else None,
            'DTYPE': component_source.dtypes[0], 'NODATA': component_source.nodata,
            'GRID_VALID': component_grid_valid})
# Evaluate fire hazard fuel component grid validation so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_component_grid_validation = pd.DataFrame(fire_hazard_fuel_component_grid_records)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_component_grid_validation['GRID_VALID'].all():
    raise ValueError('One or more fuel-component '
        'input rasters do not match the '
        'common project grid.')
# Build the expected fuel component mask shape mask used to isolate records required for this
# analysis.
expected_fuel_component_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_fuel_component_mask_shape:
    raise ValueError(f'The fuel-component study-area '
        f'mask does not match the common '
        f'project grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_fuel_component_mask_shape}')
# Calculate fire hazard fuel component study area pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_component_study_area_pixels = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fuel_component_study_area_pixels == 0:
    raise ValueError('The fuel-component study-area mask contains no included target-grid pixels.')
# Calculate fire hazard fuel component process by window so raster processing covers the analysis
# grid in controlled blocks.
fire_hazard_fuel_component_process_by_window = True
# Calculate fire hazard fuel component processing window size so raster processing covers the
# analysis grid in controlled blocks.
fire_hazard_fuel_component_processing_window_size = fire_hazard_fuel_processing_window_size
# Prepare fire hazard fuel component progress interval for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_fuel_component_progress_interval = fire_hazard_fuel_progress_interval
# Prepare fire hazard fuel component reuse existing for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fuel_component_reuse_existing = fire_hazard_fuel_reuse_existing
# Prepare fire hazard fuel component overwrite for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_overwrite = fire_hazard_fuel_overwrite
# Set fire hazard fuel component value tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_component_value_tolerance = 1e-06
# Set fire hazard fuel component formula tolerance as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_component_formula_tolerance = 1e-05
# Evaluate fire hazard fuel component minimum valid pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_minimum_valid_pixels = 1
# Set fire hazard fuel component weight table as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_component_weight_table = pd.DataFrame([{'COMPONENT_ID': 'FBFM40_HAZARD',
    'COMPONENT_NAME': 'FBFM40 Surface-Fuel Hazard',
    'INPUT_PATH': str(fire_hazard_fuel_component_input_paths['FBFM40_HAZARD']),
    'WEIGHT': fire_hazard_fuel_component_weights['FBFM40_HAZARD'],
    'MODEL_ROLE': 'Primary surface-fuel behavior'},
    {'COMPONENT_ID': 'CANOPY_COVER_HAZARD', 'COMPONENT_NAME': 'Canopy-Cover Hazard',
    'INPUT_PATH': str(fire_hazard_fuel_component_input_paths['CANOPY_COVER_HAZARD']),
    'WEIGHT': fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'],
    'MODEL_ROLE': 'Horizontal canopy continuity'},
    {'COMPONENT_ID': 'CANOPY_BULK_DENSITY_HAZARD',
    'COMPONENT_NAME': 'Canopy Bulk-Density Hazard',
    'INPUT_PATH': str(fire_hazard_fuel_component_input_paths['CANOPY_BULK_DENSITY_HAZARD']),
    'WEIGHT': fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD'],
    'MODEL_ROLE': 'Available canopy fuel mass'}])
# Collect fire hazard fuel component policy in one configuration object so downstream steps use the
# same processing rules.
fire_hazard_fuel_component_policy = (
    {'component_id': 'FUEL_HAZARD', 'component_name': 'Composite Surface and Canopy Fuel Hazard',
        'method': fire_hazard_fuel_component_method, 'formula': fire_hazard_fuel_component_formula,
        'weights': {component_id: float(component_weight) for component_id,
        component_weight in fire_hazard_fuel_component_weights.items()},
        'weight_sum': float(fire_hazard_fuel_component_weight_sum),
        'modeling_note': 'Weights are project-specific '
        'and represent the relative '
        'contribution of surface-fuel '
        'behavior, canopy continuity, '
        'and canopy fuel mass to the '
        'composite fuel-hazard score.', 'valid_data_policy': {'require_all_components': \
            fire_hazard_fuel_require_all_components,
            'missing_component_action': fire_hazard_fuel_missing_component_action}, \
                'score_policy': {'minimum': float(fire_hazard_fuel_score_minimum),
            'maximum': float(fire_hazard_fuel_score_maximum),
            'nodata': float(fire_hazard_fuel_score_nodata),
            'clip_scores': fire_hazard_fuel_clip_composite_scores}, 'input_paths': {component_id: \
                str(component_path) for component_id,
            component_path in fire_hazard_fuel_component_input_paths.items()}, 'output_path': \
                str(fire_hazard_fuel_component_output_path), 'target_grid': {'crs': \
                fire_hazard_target_crs,
            'cell_size_meters': float(fire_hazard_cell_size),
            'width': int(fire_hazard_alignment_width), 'height': int(fire_hazard_alignment_height),
            'study_area_pixels': int(fire_hazard_fuel_component_study_area_pixels)}, \
                'processing': {'process_by_window': fire_hazard_fuel_component_process_by_window,
            'window_size': int(fire_hazard_fuel_component_processing_window_size),
            'progress_interval': int(fire_hazard_fuel_component_progress_interval),
            'reuse_existing': fire_hazard_fuel_component_reuse_existing,
            'overwrite_existing': fire_hazard_fuel_component_overwrite}}
)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_component_weight_table.to_csv(fire_hazard_fuel_component_weight_table_path,
    index=False)

# Document this operation so its role in the current fuel-hazard workflow is clear before processing
# continues.
with fire_hazard_fuel_component_policy_path.open('w',
    encoding='utf-8') as fuel_component_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fuel_component_policy, fuel_component_policy_file, indent=2)
# Assemble fire hazard fuel component configuration summary into a table for QA review and
# downstream validation.
fire_hazard_fuel_component_configuration_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'METHOD': fire_hazard_fuel_component_method, 'FBFM40_WEIGHT': \
        fire_hazard_fuel_component_weights['FBFM40_HAZARD'],
    'CANOPY_COVER_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'],
    'CANOPY_BULK_DENSITY_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD'],
    'WEIGHT_SUM': fire_hazard_fuel_component_weight_sum,
    'WEIGHTS_VALID': fire_hazard_fuel_component_weights_valid,
    'REQUIRE_ALL_COMPONENTS': fire_hazard_fuel_require_all_components,
    'MISSING_COMPONENT_ACTION': fire_hazard_fuel_missing_component_action,
    'OUTPUT_SCORE_MINIMUM': fire_hazard_fuel_score_minimum,
    'OUTPUT_SCORE_MAXIMUM': fire_hazard_fuel_score_maximum,
    'OUTPUT_NODATA': fire_hazard_fuel_score_nodata,
    'OUTPUT_PATH': str(fire_hazard_fuel_component_output_path),
    'INPUT_GRIDS_VALID': bool(fire_hazard_fuel_component_grid_validation['GRID_VALID'].all()),
    'READY_TO_BUILD': True}])
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_component_configuration_summary.to_csv \
    (fire_hazard_fuel_component_configuration_summary_path,
    index=False)
# Define fire hazard fuel component weights configured used to combine component scores in the
# hazard calculation.
fire_hazard_fuel_component_weights_configured = all([fire_hazard_fbfm40_product_finalized,
    fire_hazard_normalized_canopy_layers_valid, fire_hazard_fuel_component_weights_valid,
    len(fire_hazard_fuel_component_weights) == 3, \
        fire_hazard_fuel_component_grid_validation['GRID_VALID'].all(),
    fire_hazard_fuel_component_weight_table_path.exists(),
    fire_hazard_fuel_component_policy_path.exists(),
    fire_hazard_fuel_component_configuration_summary_path.exists()])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_component_weights_configured:
    raise ValueError('The composite fuel-hazard weighting configuration did not pass final checks.')
print(f'-> Combination method: {fire_hazard_fuel_component_method}')
print(f"-> FBFM40 weight: {fire_hazard_fuel_component_weights['FBFM40_HAZARD']:.2f}")
print(f"-> Canopy-cover weight: {fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD']:.2f}")
print(f"-> Canopy-bulk-density weight: "
    f"{fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD']:.2f}")
print(f'-> Weight sum: {fire_hazard_fuel_component_weight_sum:.2f}')
print(f'-> Require all three components: {fire_hazard_fuel_require_all_components}')
print(f"-> Input grids valid: "
    f"{bool(fire_hazard_fuel_component_grid_validation['GRID_VALID'].all())}")
print(f'-> Output raster: {fire_hazard_fuel_component_output_path}')
print(f'-> Fuel-component weights configured: {fire_hazard_fuel_component_weights_configured}')
print(f'-> Weight table saved: {fire_hazard_fuel_component_weight_table_path}')
print(f'-> Weighting policy saved: {fire_hazard_fuel_component_policy_path}')
print(f'-> Configuration summary saved: {fire_hazard_fuel_component_configuration_summary_path}')
print('\n--- FUEL-HAZARD COMPONENT WEIGHTS ---')
display(fire_hazard_fuel_component_weight_table)
print('\n--- FUEL-HAZARD WEIGHTING SUMMARY ---')
display(fire_hazard_fuel_component_configuration_summary)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The composite fuel-hazard '
    'model is configured as a '
    'weighted linear combination of '
    'FBFM40, canopy cover, and '
    'canopy bulk density.')
print('FBFM40 contributes 50 percent '
    'of the final fuel score, while '
    'each canopy variable '
    'contributes 25 percent.')
print('A composite score will be '
    'calculated only where all '
    'three standardized component '
    'rasters contain valid data.')
print('The next step will apply these weights and build the composite fuel-hazard raster.')
print('\n=== FUEL-HAZARD COMPONENT WEIGHTS CONFIGURED ===')


=== CONFIGURING FUEL-HAZARD COMPONENT WEIGHTS ===
-> Combination method: weighted_linear_combination
-> FBFM40 weight: 0.50
-> Canopy-cover weight: 0.25
-> Canopy-bulk-density weight: 0.25
-> Weight sum: 1.00
-> Require all three components: True
-> Input grids valid: True
-> Output raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\component_output\composite_fuel_hazard_score.tif
-> Fuel-component weights configured: True
-> Weight table saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_component_weights.csv
-> Weighting policy saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_component_weighting_policy.json
-> Configuration summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_component_weighting_summary.csv

--- FUEL-HAZARD COMPONENT WEIGHTS ---


,COMPONENT_ID,COMPONENT_NAME,INPUT_PATH,WEIGHT,MODEL_ROLE
0,FBFM40_HAZARD,FBFM40 Surface-Fuel Hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.50,Primary surface-fuel behavior
1,CANOPY_COVER_HAZARD,Canopy-Cover Hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.25,Horizontal canopy continuity
2,CANOPY_BULK_DENSITY_HAZARD,Canopy Bulk-Density Hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.25,Available canopy fuel mass



--- FUEL-HAZARD WEIGHTING SUMMARY ---


,COMPONENT_ID,METHOD,FBFM40_WEIGHT,CANOPY_COVER_WEIGHT,CANOPY_BULK_DENSITY_WEIGHT,WEIGHT_SUM,WEIGHTS_VALID,REQUIRE_ALL_COMPONENTS,MISSING_COMPONENT_ACTION,OUTPUT_SCORE_MINIMUM,OUTPUT_SCORE_MAXIMUM,OUTPUT_NODATA,OUTPUT_PATH,INPUT_GRIDS_VALID,READY_TO_BUILD
0,FUEL_HAZARD,weighted_linear_combination,0.5,0.25,0.25,1.0,True,True,assign_nodata,0.0,1.0,-9999.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True



NOTE:
The composite fuel-hazard model is configured as a weighted linear combination of FBFM40, canopy cover, and canopy bulk density.
FBFM40 contributes 50 percent of the final fuel score, while each canopy variable contributes 25 percent.
A composite score will be calculated only where all three standardized component rasters contain valid data.
The next step will apply these weights and build the composite fuel-hazard raster.

=== FUEL-HAZARD COMPONENT WEIGHTS CONFIGURED ===


### Building Composite Fuel-Hazard Raster


In [114]:
print('=== BUILDING COMPOSITE FUEL-HAZARD RASTER ===')
# Prepare required fuel component build inputs for the downstream processing or validation performed
# in this workflow stage.
required_fuel_component_build_inputs = ['fire_hazard_fuel_component_weights_configured',
    'fire_hazard_fuel_component_input_paths', 'fire_hazard_fuel_component_weights',
    'fire_hazard_fuel_component_method', 'fire_hazard_fuel_component_formula',
    'fire_hazard_fuel_require_all_components', 'fire_hazard_fuel_clip_composite_scores',
    'fire_hazard_fuel_component_output_path', 'fire_hazard_fuel_component_temporary_path',
    'fire_hazard_fuel_component_output_profile', 'fire_hazard_fuel_component_reuse_existing',
    'fire_hazard_fuel_component_overwrite', 'fire_hazard_fuel_component_progress_interval',
    'fire_hazard_fuel_component_minimum_valid_pixels',
    'fire_hazard_fuel_component_value_tolerance', 'fire_hazard_fuel_score_minimum',
    'fire_hazard_fuel_score_maximum', 'fire_hazard_fuel_score_nodata',
    'fire_hazard_fuel_component_manifest_path', 'fire_hazard_fuel_metadata_directory',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size']
# Identify missing fuel component build inputs so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fuel_component_build_inputs = [object_name for object_name in \
    required_fuel_component_build_inputs if object_name not in globals()]

# Stop execution if missing fuel component build inputs remain unresolved before this workflow stage
# begins.
if missing_fuel_component_build_inputs:
    raise NameError(f'The following composite '
        f'fuel-hazard objects are '
        f'missing:\n'
        f'{missing_fuel_component_build_inputs}\n\n'
        f'Run Configure Fuel-Hazard '
        f'Component Weights before '
        f'building the composite raster.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_component_weights_configured:
    raise ValueError('The fuel-hazard component weights have not been configured successfully.')
# Build fire hazard fuel component input paths used to read, cache, or save this workflow product.
fire_hazard_fuel_component_input_paths = {component_id: Path(component_path) for component_id,
    component_path in fire_hazard_fuel_component_input_paths.items()}
# Build fire hazard fuel component output path used to read, cache, or save this workflow product.
fire_hazard_fuel_component_output_path = Path(fire_hazard_fuel_component_output_path)
# Build fire hazard fuel component temporary path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_temporary_path = Path(fire_hazard_fuel_component_temporary_path)

# Iterate through fire hazard fuel component input paths.items so each required item receives the
# same processing and QA checks.
for component_id, component_path in fire_hazard_fuel_component_input_paths.items():

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not component_path.exists() or not component_path.is_file() or \
        component_path.stat().st_size <= 0:
        raise FileNotFoundError(f'A required fuel-hazard input '
            f'raster is unavailable:\n'
            f'Component: {component_id}\n'
            f'Path: {component_path}')

# Define reusable validate existing composite fuel hazard logic for this phase of the workflow.
def validate_existing_composite_fuel_hazard(raster_path):

    """
    Confirm that a composite fuel-hazard raster
    matches the common project grid and expected
    Float32 output structure.
    """
    # Build raster path used to read, cache, or save this workflow product.
    raster_path = Path(raster_path)

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not raster_path.exists() or not raster_path.is_file() or raster_path.stat().st_size <= 0:
        return False

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(raster_path) as raster_source:
            return all([raster_source.count == 1,
                raster_source.width == fire_hazard_alignment_width,
                raster_source.height == fire_hazard_alignment_height,
                raster_source.crs is not None, raster_source.crs == \
                    rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                raster_source.transform.almost_equals(fire_hazard_alignment_transform),
                raster_source.dtypes[0] == 'float32', raster_source.nodata == \
                    fire_hazard_fuel_score_nodata])
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception:
        return False
# Evaluate fire hazard fuel component existing output valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_existing_output_valid = \
    validate_existing_composite_fuel_hazard(fire_hazard_fuel_component_output_path)
# Prepare fire hazard fuel component reuse output for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_reuse_output = fire_hazard_fuel_component_reuse_existing and \
    fire_hazard_fuel_component_existing_output_valid

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_fuel_component_temporary_path.exists():
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_fuel_component_temporary_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_fuel_component_output_path.exists() and fire_hazard_fuel_component_overwrite:
    # Remove the temporary or replaceable raster so the next write starts from a clean output path.
    fire_hazard_fuel_component_output_path.unlink()

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_fuel_component_output_path.exists() and (not \
    fire_hazard_fuel_component_reuse_output) and (not fire_hazard_fuel_component_overwrite):
    raise FileExistsError(f'An existing composite '
        f'fuel-hazard raster is present, '
        f'but it cannot be reused and '
        f'overwrite mode is disabled:\n'
        f'{fire_hazard_fuel_component_output_path}\n\n'
        f'Use refresh mode or remove the '
        f'invalid file.')
# Calculate fire hazard fuel component total pixels for completeness, file-integrity, or processing
# QA.
fire_hazard_fuel_component_total_pixels = 0
# Calculate fire hazard fuel component complete case pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_component_complete_case_pixels = 0
# Evaluate fire hazard fuel component output valid pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_output_valid_pixels = 0
# Calculate fire hazard fuel component output NoData pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_component_output_nodata_pixels = 0
# Evaluate fire hazard fuel component valid inside study area so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_valid_inside_study_area = 0
# Evaluate fire hazard fuel component valid outside study area so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_valid_outside_study_area = 0
# Calculate fire hazard fuel component missing FBFM40 pixels for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_component_missing_fbfm40_pixels = 0
# Calculate fire hazard fuel component missing canopy cover pixels for completeness, file-integrity,
# or processing QA.
fire_hazard_fuel_component_missing_canopy_cover_pixels = 0
# Calculate fire hazard fuel component missing canopy bulk density pixels for completeness,
# file-integrity, or processing QA.
fire_hazard_fuel_component_missing_canopy_bulk_density_pixels = 0
# Set fire hazard fuel component output minimum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_component_output_minimum = None
# Set fire hazard fuel component output maximum as an explicit model or validation parameter used
# consistently in downstream calculations.
fire_hazard_fuel_component_output_maximum = None
# Prepare fire hazard fuel component output sum for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_output_sum = 0.0
# Prepare fire hazard fuel component output mean for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_output_mean = None
# Prepare fire hazard fuel component processing error for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fuel_component_processing_error = None
# Prepare fire hazard fuel component processing complete for the fuel-hazard calculation and
# subsequent QA checks.
fire_hazard_fuel_component_processing_complete = False

# Define reusable update composite fuel statistics logic for this phase of the workflow.
def update_composite_fuel_statistics(output_array, study_area_mask):

    """
    Update raster-wide statistics from one composite
    fuel-hazard processing window.
    """
    global fire_hazard_fuel_component_output_valid_pixels
    global fire_hazard_fuel_component_output_nodata_pixels
    global fire_hazard_fuel_component_valid_inside_study_area
    global fire_hazard_fuel_component_valid_outside_study_area
    global fire_hazard_fuel_component_output_minimum
    global fire_hazard_fuel_component_output_maximum
    global fire_hazard_fuel_component_output_sum
    # Build the valid output mask mask used to isolate records required for this analysis.
    valid_output_mask = np.isfinite(output_array) & (output_array != fire_hazard_fuel_score_nodata)
    # Evaluate valid output values so invalid inputs or outputs can be rejected before continuing.
    valid_output_values = output_array[valid_output_mask]
    # Evaluate fire hazard fuel component output valid pixels so invalid inputs or outputs can be
    # rejected before continuing.
    fire_hazard_fuel_component_output_valid_pixels += int(valid_output_mask.sum())
    # Calculate fire hazard fuel component output NoData pixels for completeness, file-integrity, or
    # processing QA.
    fire_hazard_fuel_component_output_nodata_pixels += int((~valid_output_mask).sum())
    # Evaluate fire hazard fuel component valid inside study area so invalid inputs or outputs can
    # be rejected before continuing.
    fire_hazard_fuel_component_valid_inside_study_area += int((valid_output_mask & \
        study_area_mask).sum())
    # Evaluate fire hazard fuel component valid outside study area so invalid inputs or outputs can
    # be rejected before continuing.
    fire_hazard_fuel_component_valid_outside_study_area += int((valid_output_mask & \
        ~study_area_mask).sum())

    # Stop execution if the prerequisite validation has not passed before this workflow stage
    # continues.
    if valid_output_values.size > 0:
        # Calculate window minimum so raster processing covers the analysis grid in controlled
        # blocks.
        window_minimum = float(valid_output_values.min())
        # Calculate window maximum so raster processing covers the analysis grid in controlled
        # blocks.
        window_maximum = float(valid_output_values.max())

        # Stop execution if this validation condition is not satisfied before dependent processing
        # continues.
        if fire_hazard_fuel_component_output_minimum is None:
            # Set fire hazard fuel component output minimum as an explicit model or validation
            # parameter used consistently in downstream calculations.
            fire_hazard_fuel_component_output_minimum = window_minimum
        else:
            # Set fire hazard fuel component output minimum as an explicit model or validation
            # parameter used consistently in downstream calculations.
            fire_hazard_fuel_component_output_minimum = \
                min(fire_hazard_fuel_component_output_minimum,
                window_minimum)

        # Stop execution if this validation condition is not satisfied before dependent processing
        # continues.
        if fire_hazard_fuel_component_output_maximum is None:
            # Set fire hazard fuel component output maximum as an explicit model or validation
            # parameter used consistently in downstream calculations.
            fire_hazard_fuel_component_output_maximum = window_maximum
        else:
            # Set fire hazard fuel component output maximum as an explicit model or validation
            # parameter used consistently in downstream calculations.
            fire_hazard_fuel_component_output_maximum = \
                max(fire_hazard_fuel_component_output_maximum,
                window_maximum)
        # Prepare fire hazard fuel component output sum for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fuel_component_output_sum += float(valid_output_values.sum(dtype=np.float64))

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fire_hazard_fuel_component_reuse_output:
    # Prepare fire hazard fuel component processing action for the fuel-hazard calculation and
    # subsequent QA checks.
    fire_hazard_fuel_component_processing_action = 'Reused existing composite fuel-hazard raster'
    print('-> Reusing cached composite fuel-hazard raster.')

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fuel_component_output_path) as output_source:

            # Iterate through output source.block windows so each required item receives the same
            # processing and QA checks.
            for _, raster_window in output_source.block_windows(1):
                # Prepare output array for the raster calculation or validation performed in this
                # processing block.
                output_array = output_source.read(1, window=raster_window)
                # Prepare row start for the downstream processing or validation performed in this
                # workflow stage.
                row_start = int(raster_window.row_off)
                # Prepare row end for the downstream processing or validation performed in this
                # workflow stage.
                row_end = int(raster_window.row_off + raster_window.height)
                # Prepare column start for the downstream processing or validation performed in this
                # workflow stage.
                column_start = int(raster_window.col_off)
                # Prepare column end for the downstream processing or validation performed in this
                # workflow stage.
                column_end = int(raster_window.col_off + raster_window.width)
                # Build the study area window mask mask used to isolate records required for this
                # analysis.
                study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                    column_start:column_end]
                # Calculate fire hazard fuel component total pixels for completeness,
                # file-integrity, or processing QA.
                fire_hazard_fuel_component_total_pixels += int(output_array.size)
                # Update running raster statistics so final QA reflects all processed windows.
                update_composite_fuel_statistics(output_array, study_area_window_mask)
                del output_array
        # Calculate fire hazard fuel component complete case pixels for completeness,
        # file-integrity, or processing QA.
        fire_hazard_fuel_component_complete_case_pixels = \
            fire_hazard_fuel_component_output_valid_pixels
        # Prepare fire hazard fuel component processing complete for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fuel_component_processing_complete = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard fuel component processing error for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fuel_component_processing_error = str(error)
else:
    # Prepare fire hazard fuel component processing action for the fuel-hazard calculation and
    # subsequent QA checks.
    fire_hazard_fuel_component_processing_action = 'Created composite fuel-hazard raster'
    print('-> Combining FBFM40, canopy cover, and canopy bulk density...')
    # Create the output directory before writing workflow products.
    fire_hazard_fuel_component_temporary_path.parent.mkdir(parents=True, exist_ok=True)

    # Document this operation so its role in the current fuel-hazard workflow is clear before
    # processing continues.
    try:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fuel_component_input_paths['FBFM40_HAZARD']) as \
            fbfm40_source:

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(fire_hazard_fuel_component_input_paths['CANOPY_COVER_HAZARD']) as \
                canopy_cover_source:

                # Open the file in a managed context so required content is processed and the
                # resource closes cleanly.
                with rasterio.open(fire_hazard_fuel_component_input_paths \
                    ['CANOPY_BULK_DENSITY_HAZARD']) as canopy_bulk_density_source:

                    # Open the file in a managed context so required content is processed and the
                    # resource closes cleanly.
                    with rasterio.open(fire_hazard_fuel_component_temporary_path,
                        'w', **fire_hazard_fuel_component_output_profile) as output_raster:
                        # Calculate processing windows so raster processing covers the analysis grid
                        # in controlled blocks.
                        processing_windows = list(fbfm40_source.block_windows(1))
                        # Calculate total windows so raster processing covers the analysis grid in
                        # controlled blocks.
                        total_windows = len(processing_windows)

                        # Iterate through the required records so the same processing and validation
                        # logic is applied consistently.
                        for window_index, (_, raster_window) in enumerate(processing_windows,
                            start=1):

                            # Stop execution if this validation condition is not satisfied before
                            # dependent processing continues.
                            if window_index == 1 or window_index % \
                                fire_hazard_fuel_component_progress_interval == 0 or window_index \
                                == total_windows:
                                print(f'-> Combining window '
                                    f'{window_index:,} of '
                                    f'{total_windows:,}...')
                            # Prepare FBFM40 array for the raster calculation or validation
                            # performed in this processing block.
                            fbfm40_array = fbfm40_source.read(1, window=raster_window)
                            # Prepare canopy cover array for the raster calculation or validation
                            # performed in this processing block.
                            canopy_cover_array = canopy_cover_source.read(1, window=raster_window)
                            # Prepare canopy bulk density array for the raster calculation or
                            # validation performed in this processing block.
                            canopy_bulk_density_array = canopy_bulk_density_source.read(1,
                                window=raster_window)
                            # Prepare row start for the downstream processing or validation
                            # performed in this workflow stage.
                            row_start = int(raster_window.row_off)
                            # Prepare row end for the downstream processing or validation performed
                            # in this workflow stage.
                            row_end = int(raster_window.row_off + raster_window.height)
                            # Prepare column start for the downstream processing or validation
                            # performed in this workflow stage.
                            column_start = int(raster_window.col_off)
                            # Prepare column end for the downstream processing or validation
                            # performed in this workflow stage.
                            column_end = int(raster_window.col_off + raster_window.width)
                            # Build the study area window mask mask used to isolate records required
                            # for this analysis.
                            study_area_window_mask = \
                                fire_hazard_alignment_study_area_mask[row_start:row_end,
                                column_start:column_end]
                            # Calculate fire hazard fuel component total pixels for completeness,
                            # file-integrity, or processing QA.
                            fire_hazard_fuel_component_total_pixels += int(fbfm40_array.size)
                            # Build the fbfm40 valid mask mask used to isolate records required for
                            # this analysis.
                            fbfm40_valid_mask = np.isfinite(fbfm40_array) & (fbfm40_array != \
                                fbfm40_source.nodata)
                            # Build the canopy cover valid mask mask used to isolate records
                            # required for this analysis.
                            canopy_cover_valid_mask = np.isfinite(canopy_cover_array) & \
                                (canopy_cover_array != canopy_cover_source.nodata)
                            # Build the canopy bulk density valid mask mask used to isolate records
                            # required for this analysis.
                            canopy_bulk_density_valid_mask = \
                                np.isfinite(canopy_bulk_density_array) & \
                                (canopy_bulk_density_array != canopy_bulk_density_source.nodata)
                            # Calculate fire hazard fuel component missing FBFM40 pixels for
                            # completeness, file-integrity, or processing QA.
                            fire_hazard_fuel_component_missing_fbfm40_pixels += \
                                int((study_area_window_mask & ~fbfm40_valid_mask).sum())
                            # Calculate fire hazard fuel component missing canopy cover pixels for
                            # completeness, file-integrity, or processing QA.
                            fire_hazard_fuel_component_missing_canopy_cover_pixels += \
                                int((study_area_window_mask & ~canopy_cover_valid_mask).sum())
                            # Calculate fire hazard fuel component missing canopy bulk density
                            # pixels for completeness, file-integrity, or processing QA.
                            fire_hazard_fuel_component_missing_canopy_bulk_density_pixels += \
                                int((study_area_window_mask & \
                                ~canopy_bulk_density_valid_mask).sum())
                            # Build the complete case mask mask used to isolate records required for
                            # this analysis.
                            complete_case_mask = study_area_window_mask & fbfm40_valid_mask & \
                                canopy_cover_valid_mask & canopy_bulk_density_valid_mask
                            # Calculate fire hazard fuel component complete case pixels for
                            # completeness, file-integrity, or processing QA.
                            fire_hazard_fuel_component_complete_case_pixels += \
                                int(complete_case_mask.sum())
                            # Prepare output array for the raster calculation or validation
                            # performed in this processing block.
                            output_array = np.full(fbfm40_array.shape,
                                fire_hazard_fuel_score_nodata, dtype=np.float32)

                            # Stop execution if this validation condition is not satisfied before
                            # dependent processing continues.
                            if complete_case_mask.any():
                                # Prepare composite values for the downstream processing or
                                # validation performed in this workflow stage.
                                composite_values = fbfm40_array[complete_case_mask] * \
                                    fire_hazard_fuel_component_weights['FBFM40_HAZARD'] + \
                                    canopy_cover_array[complete_case_mask] * \
                                    fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'] + \
                                    canopy_bulk_density_array[complete_case_mask] * \
                                    fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD']

                                # Stop execution if this validation condition is not satisfied
                                # before dependent processing continues.
                                if fire_hazard_fuel_clip_composite_scores:
                                    # Prepare composite values for the downstream processing or
                                    # validation performed in this workflow stage.
                                    composite_values = np.clip(composite_values,
                                        fire_hazard_fuel_score_minimum, \
                                            fire_hazard_fuel_score_maximum)
                                # Prepare output array so this workflow stage has the values
                                # required for downstream spatial processing and QA.
                                output_array[complete_case_mask] = \
                                    composite_values.astype(np.float32)
                            # Prepare output array so this workflow stage has the values required
                            # for downstream spatial processing and QA.
                            output_array[~study_area_window_mask] = fire_hazard_fuel_score_nodata
                            # Build the valid window mask mask used to isolate records required for
                            # this analysis.
                            valid_window_mask = np.isfinite(output_array) & (output_array != \
                                fire_hazard_fuel_score_nodata)
                            # Evaluate valid window values so invalid inputs or outputs can be
                            # rejected before continuing.
                            valid_window_values = output_array[valid_window_mask]

                            # Stop execution if the prerequisite validation has not passed before
                            # this workflow stage continues.
                            if valid_window_values.size > 0:

                                # Stop execution if the prerequisite validation has not passed
                                # before this workflow stage continues.
                                if valid_window_values.min() < fire_hazard_fuel_score_minimum - \
                                    fire_hazard_fuel_component_value_tolerance or \
                                    valid_window_values.max() > fire_hazard_fuel_score_maximum + \
                                    fire_hazard_fuel_component_value_tolerance:
                                    raise ValueError('A composite fuel-hazard score '
                                        'fell outside the configured '
                                        'zero-to-one range.')
                            # Write the processed data to the configured output resource.
                            output_raster.write(output_array, 1, window=raster_window)
                            # Update running raster statistics so final QA reflects all processed
                            # windows.
                            update_composite_fuel_statistics(output_array, study_area_window_mask)
                            del fbfm40_array
                            del canopy_cover_array
                            del canopy_bulk_density_array
                            del output_array
                            del fbfm40_valid_mask
                            del canopy_cover_valid_mask
                            del canopy_bulk_density_valid_mask
                            del complete_case_mask
                        # Assign a descriptive band label so the exported raster documents the
                        # meaning of its values.
                        output_raster.set_band_description(1,
                            'Composite surface and canopy fuel-hazard score')
                        # Write processing metadata to the raster so the final product retains its
                        # model and provenance context.
                        output_raster.update_tags(COMPONENT_ID='FUEL_HAZARD',
                            COMBINATION_METHOD=fire_hazard_fuel_component_method,
                            FORMULA=fire_hazard_fuel_component_formula, \
                                FBFM40_WEIGHT=fire_hazard_fuel_component_weights['FBFM40_HAZARD'],
                            CANOPY_COVER_WEIGHT=fire_hazard_fuel_component_weights \
                                ['CANOPY_COVER_HAZARD'],
                            CANOPY_BULK_DENSITY_WEIGHT=fire_hazard_fuel_component_weights \
                                ['CANOPY_BULK_DENSITY_HAZARD'],
                            REQUIRE_ALL_COMPONENTS=str(fire_hazard_fuel_require_all_components),
                            SCORE_MINIMUM=fire_hazard_fuel_score_minimum, \
                                SCORE_MAXIMUM=fire_hazard_fuel_score_maximum,
                            TARGET_CRS=fire_hazard_target_crs, \
                                TARGET_CELL_SIZE_METERS=fire_hazard_cell_size)
        # Evaluate temporary structure valid so invalid inputs or outputs can be rejected before
        # continuing.
        temporary_structure_valid = \
            validate_existing_composite_fuel_hazard(fire_hazard_fuel_component_temporary_path)
        # Evaluate temporary content valid so invalid inputs or outputs can be rejected before
        # continuing.
        temporary_content_valid = all([fire_hazard_fuel_component_output_valid_pixels >= \
            fire_hazard_fuel_component_minimum_valid_pixels,
            fire_hazard_fuel_component_valid_inside_study_area > 0,
            fire_hazard_fuel_component_valid_outside_study_area == 0,
            fire_hazard_fuel_component_output_minimum is not None,
            fire_hazard_fuel_component_output_maximum is not None])

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if not temporary_structure_valid:
            raise ValueError('The temporary composite '
                'fuel-hazard raster failed '
                'structural validation.')

        # Stop execution if the prerequisite validation has not passed before this workflow stage
        # continues.
        if not temporary_content_valid:
            raise ValueError('The temporary composite '
                'fuel-hazard raster failed '
                'basic content checks.')

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_fuel_component_output_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_fuel_component_output_path.unlink()
        # Promote the validated temporary raster to the final output path only after processing
        # succeeds.
        fire_hazard_fuel_component_temporary_path.replace(fire_hazard_fuel_component_output_path)
        # Prepare fire hazard fuel component processing complete for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fuel_component_processing_complete = True
    # Handle the expected failure explicitly so the workflow can report or clean up the affected
    # operation.
    except Exception as error:
        # Prepare fire hazard fuel component processing error for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fuel_component_processing_error = str(error)
        # Prepare fire hazard fuel component processing complete for the fuel-hazard calculation and
        # subsequent QA checks.
        fire_hazard_fuel_component_processing_complete = False

        # Stop execution if a required raster file is missing or empty before spatial processing
        # begins.
        if fire_hazard_fuel_component_temporary_path.exists():
            # Remove the temporary or replaceable raster so the next write starts from a clean
            # output path.
            fire_hazard_fuel_component_temporary_path.unlink()

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_fuel_component_output_valid_pixels > 0:
    # Prepare fire hazard fuel component output mean for the fuel-hazard calculation and subsequent
    # QA checks.
    fire_hazard_fuel_component_output_mean = fire_hazard_fuel_component_output_sum / \
        fire_hazard_fuel_component_output_valid_pixels
# Evaluate fire hazard fuel component output structure valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_output_structure_valid = \
    validate_existing_composite_fuel_hazard(fire_hazard_fuel_component_output_path)
# Evaluate fire hazard fuel component output range valid so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_output_range_valid = all([fire_hazard_fuel_component_output_minimum is \
    not None,
    fire_hazard_fuel_component_output_maximum is not None,
    fire_hazard_fuel_component_output_minimum >= fire_hazard_fuel_score_minimum - \
        fire_hazard_fuel_component_value_tolerance,
    fire_hazard_fuel_component_output_maximum <= fire_hazard_fuel_score_maximum + \
        fire_hazard_fuel_component_value_tolerance])
# Prepare fire hazard fuel component build complete for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fuel_component_build_complete = all([fire_hazard_fuel_component_processing_complete,
    fire_hazard_fuel_component_output_structure_valid,
    fire_hazard_fuel_component_output_range_valid,
    fire_hazard_fuel_component_output_valid_pixels >= \
        fire_hazard_fuel_component_minimum_valid_pixels,
    fire_hazard_fuel_component_valid_inside_study_area > 0,
    fire_hazard_fuel_component_valid_outside_study_area == 0,
    fire_hazard_fuel_component_output_path.exists(),
    fire_hazard_fuel_component_processing_error is None])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_component_build_complete:
    raise ValueError(f'The composite fuel-hazard '
        f'raster did not complete '
        f'successfully.\n\nProcessing '
        f'complete: '
        f'{fire_hazard_fuel_component_processing_complete}\n'
        f'Structure valid: '
        f'{fire_hazard_fuel_component_output_structure_valid}\n'
        f'Value range valid: '
        f'{fire_hazard_fuel_component_output_range_valid}\n'
        f'Valid output pixels: '
        f'{fire_hazard_fuel_component_output_valid_pixels:,}\n'
        f'Valid outside study area: '
        f'{fire_hazard_fuel_component_valid_outside_study_area:,}\n'
        f'Error: '
        f'{fire_hazard_fuel_component_processing_error}')
# Calculate fire hazard fuel component output size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_component_output_size_bytes = \
    int(fire_hazard_fuel_component_output_path.stat().st_size)
# Assemble fire hazard fuel component manifest into a table for QA review and downstream validation.
# Create the manifest for the completed composite surface and canopy fuel-hazard product.
fire_hazard_fuel_component_manifest = pd.DataFrame(
    [
        {
            # Identify the fuel-hazard component and output product.
            'COMPONENT_ID': 'FUEL_HAZARD',
            'PRODUCT_ID': 'COMPOSITE_FUEL_HAZARD',
            'PRODUCT_NAME': 'Composite Surface and Canopy Fuel Hazard',

            # Record the processing action and method used to combine the fuel inputs.
            'PROCESSING_ACTION': fire_hazard_fuel_component_processing_action,
            'COMBINATION_METHOD': fire_hazard_fuel_component_method,
            'FORMULA': fire_hazard_fuel_component_formula,

            # Record the weights assigned to each fuel-hazard input.
            'FBFM40_WEIGHT': (
                fire_hazard_fuel_component_weights['FBFM40_HAZARD']
            ),
            'CANOPY_COVER_WEIGHT': (
                fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD']
            ),
            'CANOPY_BULK_DENSITY_WEIGHT': (
                fire_hazard_fuel_component_weights[
                    'CANOPY_BULK_DENSITY_HAZARD'
                ]
            ),

            # Record whether all three fuel inputs are required for a valid output pixel.
            'REQUIRE_ALL_COMPONENTS': fire_hazard_fuel_require_all_components,

            # Record total, complete-case, valid, and NoData pixel counts.
            'TOTAL_PIXELS': fire_hazard_fuel_component_total_pixels,
            'COMPLETE_CASE_PIXELS': (
                fire_hazard_fuel_component_complete_case_pixels
            ),
            'OUTPUT_VALID_PIXELS': (
                fire_hazard_fuel_component_output_valid_pixels
            ),
            'OUTPUT_NODATA_PIXELS': (
                fire_hazard_fuel_component_output_nodata_pixels
            ),

            # Record missing-data counts for each fuel-hazard input.
            'MISSING_FBFM40_PIXELS': (
                fire_hazard_fuel_component_missing_fbfm40_pixels
            ),
            'MISSING_CANOPY_COVER_PIXELS': (
                fire_hazard_fuel_component_missing_canopy_cover_pixels
            ),
            'MISSING_CANOPY_BULK_DENSITY_PIXELS': (
                fire_hazard_fuel_component_missing_canopy_bulk_density_pixels
            ),

            # Record valid output pixels inside and outside the study area.
            'VALID_INSIDE_STUDY_AREA': (
                fire_hazard_fuel_component_valid_inside_study_area
            ),
            'VALID_OUTSIDE_STUDY_AREA': (
                fire_hazard_fuel_component_valid_outside_study_area
            ),

            # Record the range and mean of the composite fuel-hazard scores.
            'OUTPUT_MINIMUM': fire_hazard_fuel_component_output_minimum,
            'OUTPUT_MAXIMUM': fire_hazard_fuel_component_output_maximum,
            'OUTPUT_MEAN': fire_hazard_fuel_component_output_mean,

            # Record the output location and file size.
            'OUTPUT_PATH': str(
                fire_hazard_fuel_component_output_path
            ),
            'OUTPUT_SIZE_BYTES': (
                fire_hazard_fuel_component_output_size_bytes
            ),
            'OUTPUT_SIZE_MB': (
                fire_hazard_fuel_component_output_size_bytes / 1024 ** 2
            ),

            # Record the structural and value-range validation results.
            'STRUCTURE_VALID': (
                fire_hazard_fuel_component_output_structure_valid
            ),
            'VALUE_RANGE_VALID': (
                fire_hazard_fuel_component_output_range_valid
            ),

            # Record the final build and validation status.
            'BUILD_COMPLETE': fire_hazard_fuel_component_build_complete,
            'VALID': fire_hazard_fuel_component_build_complete,
        }
    ]
)
# Prepare fire hazard fuel component build summary for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_build_summary = fire_hazard_fuel_component_manifest[['COMPONENT_ID',
    'PRODUCT_ID', 'PROCESSING_ACTION', 'COMBINATION_METHOD',
    'FBFM40_WEIGHT', 'CANOPY_COVER_WEIGHT', 'CANOPY_BULK_DENSITY_WEIGHT',
    'COMPLETE_CASE_PIXELS', 'OUTPUT_VALID_PIXELS',
    'OUTPUT_NODATA_PIXELS', 'VALID_OUTSIDE_STUDY_AREA',
    'OUTPUT_MINIMUM', 'OUTPUT_MAXIMUM', 'OUTPUT_MEAN',
    'OUTPUT_SIZE_MB', 'BUILD_COMPLETE']].copy()
# Build fire hazard fuel component build summary path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_build_summary_path = fire_hazard_fuel_metadata_directory / \
    'fuel_component_build_summary.csv'
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_component_manifest.to_csv(fire_hazard_fuel_component_manifest_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_component_build_summary.to_csv(fire_hazard_fuel_component_build_summary_path,
    index=False)
print(f'-> Processing action: {fire_hazard_fuel_component_processing_action}')
print(f'-> Composite output: {fire_hazard_fuel_component_output_path}')
print(f'-> Complete-case pixels: {fire_hazard_fuel_component_complete_case_pixels:,}')
print(f'-> Valid output pixels: {fire_hazard_fuel_component_output_valid_pixels:,}')
print(f'-> Output NoData pixels: {fire_hazard_fuel_component_output_nodata_pixels:,}')
print(f'-> Missing FBFM40 pixels: {fire_hazard_fuel_component_missing_fbfm40_pixels:,}')
print(f'-> Missing canopy-cover pixels: {fire_hazard_fuel_component_missing_canopy_cover_pixels:,}')
print(f'-> Missing canopy-bulk-density '
    f'pixels: '
    f'{fire_hazard_fuel_component_missing_canopy_bulk_density_pixels:,}')
print(f'-> Valid pixels outside study '
    f'area: '
    f'{fire_hazard_fuel_component_valid_outside_study_area:,}')
print(f'-> Composite score range: '
    f'{fire_hazard_fuel_component_output_minimum:.4f} '
    f'to '
    f'{fire_hazard_fuel_component_output_maximum:.4f}')
print(f'-> Composite mean score: {fire_hazard_fuel_component_output_mean:.4f}')
print(f'-> Output size: {fire_hazard_fuel_component_output_size_bytes / 1024 ** 2:,.2f} MB')
print(f'-> Composite build complete: {fire_hazard_fuel_component_build_complete}')
print(f'-> Component manifest saved: {fire_hazard_fuel_component_manifest_path}')
print(f'-> Build summary saved: {fire_hazard_fuel_component_build_summary_path}')
print('\n--- COMPOSITE FUEL-HAZARD BUILD SUMMARY ---')
display(fire_hazard_fuel_component_build_summary)
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The composite fuel-hazard '
    'raster was calculated from the '
    'validated FBFM40, canopy-cover,'
    ' and canopy-bulk-density '
    'hazard layers.')
print('The calculation used weights of 0.50, 0.25, and 0.25, respectively.')
print('Only cells with valid values '
    'in all three inputs received a '
    'composite score; incomplete '
    'cells remain NoData.')
print('The next step will '
    'independently validate the '
    'weighted formula, grid, score '
    'range, and study-area masking '
    'of the composite fuel-hazard '
    'product.')
print('\n=== COMPOSITE FUEL-HAZARD RASTER COMPLETE ===')


=== BUILDING COMPOSITE FUEL-HAZARD RASTER ===
-> Reusing cached composite fuel-hazard raster.
-> Processing action: Reused existing composite fuel-hazard raster
-> Composite output: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\component_output\composite_fuel_hazard_score.tif
-> Complete-case pixels: 15,475,232
-> Valid output pixels: 15,475,232
-> Output NoData pixels: 121,295,786
-> Missing FBFM40 pixels: 0
-> Missing canopy-cover pixels: 0
-> Missing canopy-bulk-density pixels: 0
-> Valid pixels outside study area: 0
-> Composite score range: 0.0000 to 0.9274
-> Composite mean score: 0.2746
-> Output size: 31.36 MB
-> Composite build complete: True
-> Component manifest saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_component_manifest.csv
-> Build summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_component_build_summary.csv

--- COMPOSITE FUEL-HAZ

,COMPONENT_ID,PRODUCT_ID,PROCESSING_ACTION,COMBINATION_METHOD,FBFM40_WEIGHT,CANOPY_COVER_WEIGHT,CANOPY_BULK_DENSITY_WEIGHT,COMPLETE_CASE_PIXELS,OUTPUT_VALID_PIXELS,OUTPUT_NODATA_PIXELS,VALID_OUTSIDE_STUDY_AREA,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,OUTPUT_SIZE_MB,BUILD_COMPLETE
0,FUEL_HAZARD,COMPOSITE_FUEL_HAZARD,Reused existing composite fuel-hazard raster,weighted_linear_combination,0.5,0.25,0.25,15475232,15475232,121295786,0,0.0,0.927376,0.274636,31.360385,True



NOTE:
The composite fuel-hazard raster was calculated from the validated FBFM40, canopy-cover, and canopy-bulk-density hazard layers.
The calculation used weights of 0.50, 0.25, and 0.25, respectively.
Only cells with valid values in all three inputs received a composite score; incomplete cells remain NoData.
The next step will independently validate the weighted formula, grid, score range, and study-area masking of the composite fuel-hazard product.

=== COMPOSITE FUEL-HAZARD RASTER COMPLETE ===


### Validating Composite Fuel-Hazard Raster


In [115]:
print('=== VALIDATING COMPOSITE FUEL-HAZARD RASTER ===')
# Record whether required fuel component validation inputs satisfies the checks required before the
# workflow advances.
required_fuel_component_validation_inputs = ['fire_hazard_fuel_component_build_complete',
    'fire_hazard_fuel_component_output_path', 'fire_hazard_fuel_component_input_paths',
    'fire_hazard_fuel_component_weights', 'fire_hazard_fuel_component_method',
    'fire_hazard_fuel_component_formula', 'fire_hazard_fuel_require_all_components',
    'fire_hazard_fuel_clip_composite_scores', 'fire_hazard_fuel_component_manifest',
    'fire_hazard_fuel_component_manifest_path', 'fire_hazard_fuel_validation_path',
    'fire_hazard_fuel_validation_summary_path', 'fire_hazard_fuel_score_minimum',
    'fire_hazard_fuel_score_maximum', 'fire_hazard_fuel_score_nodata',
    'fire_hazard_fuel_component_minimum_valid_pixels',
    'fire_hazard_fuel_component_value_tolerance', 'fire_hazard_fuel_component_formula_tolerance',
    'fire_hazard_fuel_component_output_valid_pixels',
    'fire_hazard_fuel_component_output_nodata_pixels',
    'fire_hazard_fuel_component_valid_inside_study_area',
    'fire_hazard_fuel_component_valid_outside_study_area',
    'fire_hazard_fuel_component_output_minimum', 'fire_hazard_fuel_component_output_maximum',
    'fire_hazard_fuel_component_output_mean', 'fire_hazard_fuel_component_complete_case_pixels',
    'fire_hazard_fuel_metadata_directory', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_alignment_study_area_mask', 'fire_hazard_target_crs',
    'fire_hazard_cell_size']
# Identify missing fuel component validation inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fuel_component_validation_inputs = [object_name for object_name in \
    required_fuel_component_validation_inputs if object_name not in globals()]

# Stop execution if missing fuel component validation inputs remain unresolved before this workflow
# stage begins.
if missing_fuel_component_validation_inputs:
    raise NameError(f'The following composite '
        f'fuel-hazard validation objects '
        f'are missing:\n'
        f'{missing_fuel_component_validation_inputs}\n\n'
        f'Run Build Composite '
        f'Fuel-Hazard Raster before '
        f'validating the final '
        f'fuel-hazard product.')

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_component_build_complete:
    raise ValueError('The composite fuel-hazard raster has not completed successfully.')
# Build fire hazard fuel component validation output path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_validation_output_path = Path(fire_hazard_fuel_component_output_path)
# Build fire hazard fuel component validation input paths used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_validation_input_paths = {component_id: Path(component_path) for \
    component_id,
    component_path in fire_hazard_fuel_component_input_paths.items()}
# Build validation file paths used to read, cache, or save this workflow product.
validation_file_paths = {**fire_hazard_fuel_component_validation_input_paths,
    'COMPOSITE_FUEL_HAZARD': fire_hazard_fuel_component_validation_output_path}

# Iterate through validation file paths.items so each required item receives the same processing and
# QA checks.
for product_id, product_path in validation_file_paths.items():

    # Stop execution if a required raster file is missing or empty before spatial processing begins.
    if not product_path.exists() or not product_path.is_file() or product_path.stat().st_size <= 0:
        raise FileNotFoundError(f'A required fuel-hazard '
            f'validation raster is '
            f'unavailable:\nProduct: '
            f'{product_id}\nPath: '
            f'{product_path}')
# Build the expected fuel validation mask shape mask used to isolate records required for this
# analysis.
expected_fuel_validation_mask_shape = (fire_hazard_alignment_height, fire_hazard_alignment_width)

# Stop execution if the raster or mask dimensions do not match the analysis grid required for
# cell-by-cell processing.
if fire_hazard_alignment_study_area_mask.shape != expected_fuel_validation_mask_shape:
    raise ValueError(f'The fuel-hazard validation '
        f'mask does not match the common '
        f'project grid.\nMask shape: '
        f'{fire_hazard_alignment_study_area_mask.shape}\n'
        f'Expected shape: '
        f'{expected_fuel_validation_mask_shape}')
# Evaluate fire hazard fuel validation study area pixels so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_validation_study_area_pixels = int(fire_hazard_alignment_study_area_mask.sum())

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if fire_hazard_fuel_validation_study_area_pixels == 0:
    raise ValueError('The fuel-hazard validation study-area mask contains no included pixels.')
# Evaluate fire hazard fuel component valid extensions so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_component_valid_extensions = {'.tif', '.tiff'}
# Calculate fire hazard fuel component minimum file size bytes for completeness, file-integrity, or
# processing QA.
fire_hazard_fuel_component_minimum_file_size_bytes = 1024
# Prepare fuel output file exists for the downstream processing or validation performed in this
# workflow stage.
fuel_output_file_exists = fire_hazard_fuel_component_validation_output_path.exists() and \
    fire_hazard_fuel_component_validation_output_path.is_file()
# Evaluate fuel output extension valid so invalid inputs or outputs can be rejected before
# continuing.
fuel_output_extension_valid = fire_hazard_fuel_component_validation_output_path.suffix.lower() in \
    fire_hazard_fuel_component_valid_extensions
# Calculate fuel output file size bytes for completeness, file-integrity, or processing QA.
fuel_output_file_size_bytes = fire_hazard_fuel_component_validation_output_path.stat().st_size if \
    fuel_output_file_exists else 0
# Evaluate fuel output file size valid so invalid inputs or outputs can be rejected before
# continuing.
fuel_output_file_size_valid = fuel_output_file_size_bytes >= \
    fire_hazard_fuel_component_minimum_file_size_bytes
# Prepare fuel output readable for the downstream processing or validation performed in this
# workflow stage.
fuel_output_readable = False
# Evaluate fuel output grid valid so invalid inputs or outputs can be rejected before continuing.
fuel_output_grid_valid = False
# Evaluate fuel output structure valid so invalid inputs or outputs can be rejected before
# continuing.
fuel_output_structure_valid = False
# Record whether fuel input grids match satisfies the checks required before the workflow advances.
fuel_input_grids_match = False
# Evaluate fuel validation error so invalid inputs or outputs can be rejected before continuing.
fuel_validation_error = None
# Evaluate validated total pixels so invalid inputs or outputs can be rejected before continuing.
validated_total_pixels = 0
# Evaluate validated complete case pixels so invalid inputs or outputs can be rejected before
# continuing.
validated_complete_case_pixels = 0
# Evaluate validated output valid pixels so invalid inputs or outputs can be rejected before
# continuing.
validated_output_valid_pixels = 0
# Evaluate validated output nodata pixels so invalid inputs or outputs can be rejected before
# continuing.
validated_output_nodata_pixels = 0
# Evaluate validated inside study area so invalid inputs or outputs can be rejected before
# continuing.
validated_inside_study_area = 0
# Evaluate validated outside study area so invalid inputs or outputs can be rejected before
# continuing.
validated_outside_study_area = 0
# Calculate validated missing FBFM40 pixels for completeness, file-integrity, or processing QA.
validated_missing_fbfm40_pixels = 0
# Calculate validated missing canopy cover pixels for completeness, file-integrity, or processing
# QA.
validated_missing_canopy_cover_pixels = 0
# Calculate validated missing canopy bulk density pixels for completeness, file-integrity, or
# processing QA.
validated_missing_canopy_bulk_density_pixels = 0
# Evaluate validated output minimum so invalid inputs or outputs can be rejected before continuing.
validated_output_minimum = None
# Evaluate validated output maximum so invalid inputs or outputs can be rejected before continuing.
validated_output_maximum = None
# Evaluate validated output sum so invalid inputs or outputs can be rejected before continuing.
validated_output_sum = 0.0
# Evaluate validated output mean so invalid inputs or outputs can be rejected before continuing.
validated_output_mean = None
# Calculate formula checked pixels for completeness, file-integrity, or processing QA.
formula_checked_pixels = 0
# Record whether formula matching pixels satisfies the checks required before the workflow advances.
formula_matching_pixels = 0
# Calculate formula mismatch pixels for completeness, file-integrity, or processing QA.
formula_mismatch_pixels = 0
# Set maximum formula difference as an explicit model or validation parameter used consistently in
# downstream calculations.
maximum_formula_difference = 0.0
# Prepare formula difference sum for the downstream processing or validation performed in this
# workflow stage.
formula_difference_sum = 0.0
# Prepare mean formula difference for the downstream processing or validation performed in this
# workflow stage.
mean_formula_difference = None
# Prepare complete case without output for the downstream processing or validation performed in this
# workflow stage.
complete_case_without_output = 0
# Prepare incomplete case with output for the downstream processing or validation performed in this
# workflow stage.
incomplete_case_with_output = 0
# Evaluate output valid outside study area so invalid inputs or outputs can be rejected before
# continuing.
output_valid_outside_study_area = 0

# Document this operation so its role in the current fuel-hazard workflow is clear before processing
# continues.
try:

    # Open the file in a managed context so required content is processed and the resource closes
    # cleanly.
    with rasterio.open(fire_hazard_fuel_component_validation_input_paths['FBFM40_HAZARD']) as \
        fbfm40_source:

        # Open the file in a managed context so required content is processed and the resource
        # closes cleanly.
        with rasterio.open(fire_hazard_fuel_component_validation_input_paths \
            ['CANOPY_COVER_HAZARD']) as canopy_cover_source:

            # Open the file in a managed context so required content is processed and the resource
            # closes cleanly.
            with rasterio.open(fire_hazard_fuel_component_validation_input_paths \
                ['CANOPY_BULK_DENSITY_HAZARD']) as canopy_bulk_density_source:

                # Open the file in a managed context so required content is processed and the
                # resource closes cleanly.
                with rasterio.open(fire_hazard_fuel_component_validation_output_path) as \
                    output_source:
                    # Prepare fuel output readable for the downstream processing or validation
                    # performed in this workflow stage.
                    fuel_output_readable = True
                    # Evaluate fuel output grid valid so invalid inputs or outputs can be rejected
                    # before continuing.
                    fuel_output_grid_valid = all([output_source.width == \
                        fire_hazard_alignment_width,
                        output_source.height == fire_hazard_alignment_height,
                        output_source.crs is not None, output_source.crs == \
                            rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
                        output_source.transform.almost_equals(fire_hazard_alignment_transform),
                        np.isclose(abs(output_source.transform.a), fire_hazard_cell_size,
                        atol=fire_hazard_fuel_component_value_tolerance),
                        np.isclose(abs(output_source.transform.e), fire_hazard_cell_size,
                        atol=fire_hazard_fuel_component_value_tolerance)])
                    # Evaluate fuel output structure valid so invalid inputs or outputs can be
                    # rejected before continuing.
                    fuel_output_structure_valid = all([output_source.count == 1,
                        output_source.dtypes[0] == 'float32', output_source.nodata == \
                            fire_hazard_fuel_score_nodata])
                    # Define input sources used to configure this workflow stage.
                    input_sources = [fbfm40_source, canopy_cover_source, canopy_bulk_density_source]
                    # Record whether fuel input grids match satisfies the checks required before the
                    # workflow advances.
                    fuel_input_grids_match = all((source.count == 1 and source.width == \
                        output_source.width and (source.height == output_source.height) and \
                        (source.crs == output_source.crs) and \
                        source.transform.almost_equals(output_source.transform) for source in \
                        input_sources))

                    # Stop execution if this validation condition is not satisfied before dependent
                    # processing continues.
                    if not fuel_input_grids_match:
                        raise ValueError('One or more fuel-component '
                            'input rasters do not match the '
                            'composite output grid.')

                    # Iterate through output source.block windows so each required item receives the
                    # same processing and QA checks.
                    for _, raster_window in output_source.block_windows(1):
                        # Prepare FBFM40 array for the raster calculation or validation performed in
                        # this processing block.
                        fbfm40_array = fbfm40_source.read(1, window=raster_window)
                        # Prepare canopy cover array for the raster calculation or validation
                        # performed in this processing block.
                        canopy_cover_array = canopy_cover_source.read(1, window=raster_window)
                        # Prepare canopy bulk density array for the raster calculation or validation
                        # performed in this processing block.
                        canopy_bulk_density_array = canopy_bulk_density_source.read(1,
                            window=raster_window)
                        # Prepare output array for the raster calculation or validation performed in
                        # this processing block.
                        output_array = output_source.read(1, window=raster_window)
                        # Prepare row start for the downstream processing or validation performed in
                        # this workflow stage.
                        row_start = int(raster_window.row_off)
                        # Prepare row end for the downstream processing or validation performed in
                        # this workflow stage.
                        row_end = int(raster_window.row_off + raster_window.height)
                        # Prepare column start for the downstream processing or validation performed
                        # in this workflow stage.
                        column_start = int(raster_window.col_off)
                        # Prepare column end for the downstream processing or validation performed
                        # in this workflow stage.
                        column_end = int(raster_window.col_off + raster_window.width)
                        # Build the study area window mask mask used to isolate records required for
                        # this analysis.
                        study_area_window_mask = \
                            fire_hazard_alignment_study_area_mask[row_start:row_end,
                            column_start:column_end]
                        # Evaluate validated total pixels so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_total_pixels += int(output_array.size)
                        # Build the fbfm40 valid mask mask used to isolate records required for this
                        # analysis.
                        fbfm40_valid_mask = np.isfinite(fbfm40_array) & (fbfm40_array != \
                            fbfm40_source.nodata)
                        # Build the canopy cover valid mask mask used to isolate records required
                        # for this analysis.
                        canopy_cover_valid_mask = np.isfinite(canopy_cover_array) & \
                            (canopy_cover_array != canopy_cover_source.nodata)
                        # Build the canopy bulk density valid mask mask used to isolate records
                        # required for this analysis.
                        canopy_bulk_density_valid_mask = np.isfinite(canopy_bulk_density_array) & \
                            (canopy_bulk_density_array != canopy_bulk_density_source.nodata)
                        # Calculate validated missing FBFM40 pixels for completeness,
                        # file-integrity, or processing QA.
                        validated_missing_fbfm40_pixels += int((study_area_window_mask & \
                            ~fbfm40_valid_mask).sum())
                        # Calculate validated missing canopy cover pixels for completeness,
                        # file-integrity, or processing QA.
                        validated_missing_canopy_cover_pixels += int((study_area_window_mask & \
                            ~canopy_cover_valid_mask).sum())
                        # Calculate validated missing canopy bulk density pixels for completeness,
                        # file-integrity, or processing QA.
                        validated_missing_canopy_bulk_density_pixels += \
                            int((study_area_window_mask & ~canopy_bulk_density_valid_mask).sum())
                        # Build the complete case mask mask used to isolate records required for
                        # this analysis.
                        complete_case_mask = study_area_window_mask & fbfm40_valid_mask & \
                            canopy_cover_valid_mask & canopy_bulk_density_valid_mask
                        # Evaluate validated complete case pixels so invalid inputs or outputs can
                        # be rejected before continuing.
                        validated_complete_case_pixels += int(complete_case_mask.sum())
                        # Build the output valid mask mask used to isolate records required for this
                        # analysis.
                        output_valid_mask = np.isfinite(output_array) & (output_array != \
                            fire_hazard_fuel_score_nodata)
                        # Prepare output values for the downstream processing or validation
                        # performed in this workflow stage.
                        output_values = output_array[output_valid_mask]
                        # Evaluate validated output valid pixels so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_output_valid_pixels += int(output_valid_mask.sum())
                        # Evaluate validated output nodata pixels so invalid inputs or outputs can
                        # be rejected before continuing.
                        validated_output_nodata_pixels += int((~output_valid_mask).sum())
                        # Evaluate validated inside study area so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_inside_study_area += int((output_valid_mask & \
                            study_area_window_mask).sum())
                        # Evaluate validated outside study area so invalid inputs or outputs can be
                        # rejected before continuing.
                        validated_outside_study_area += int((output_valid_mask & \
                            ~study_area_window_mask).sum())
                        # Evaluate output valid outside study area so invalid inputs or outputs can
                        # be rejected before continuing.
                        output_valid_outside_study_area += int((output_valid_mask & \
                            ~study_area_window_mask).sum())

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if output_values.size > 0:
                            # Set block minimum as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            block_minimum = float(output_values.min())
                            # Set block maximum as an explicit model or validation parameter used
                            # consistently in downstream calculations.
                            block_maximum = float(output_values.max())

                            # Stop execution if the prerequisite validation has not passed before
                            # this workflow stage continues.
                            if validated_output_minimum is None:
                                # Evaluate validated output minimum so invalid inputs or outputs can
                                # be rejected before continuing.
                                validated_output_minimum = block_minimum
                            else:
                                # Evaluate validated output minimum so invalid inputs or outputs can
                                # be rejected before continuing.
                                validated_output_minimum = min(validated_output_minimum,
                                    block_minimum)

                            # Stop execution if the prerequisite validation has not passed before
                            # this workflow stage continues.
                            if validated_output_maximum is None:
                                # Evaluate validated output maximum so invalid inputs or outputs can
                                # be rejected before continuing.
                                validated_output_maximum = block_maximum
                            else:
                                # Evaluate validated output maximum so invalid inputs or outputs can
                                # be rejected before continuing.
                                validated_output_maximum = max(validated_output_maximum,
                                    block_maximum)
                            # Evaluate validated output sum so invalid inputs or outputs can be
                            # rejected before continuing.
                            validated_output_sum += float(output_values.sum(dtype=np.float64))
                        # Prepare complete case without output for the downstream processing or
                        # validation performed in this workflow stage.
                        complete_case_without_output += int((complete_case_mask & \
                            ~output_valid_mask).sum())
                        # Prepare incomplete case with output for the downstream processing or
                        # validation performed in this workflow stage.
                        incomplete_case_with_output += int((study_area_window_mask & \
                            ~complete_case_mask & output_valid_mask).sum())

                        # Stop execution if this validation condition is not satisfied before
                        # dependent processing continues.
                        if complete_case_mask.any():
                            # Prepare expected values for the downstream processing or validation
                            # performed in this workflow stage.
                            expected_values = fbfm40_array[complete_case_mask] * \
                                fire_hazard_fuel_component_weights['FBFM40_HAZARD'] + \
                                canopy_cover_array[complete_case_mask] * \
                                fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'] + \
                                canopy_bulk_density_array[complete_case_mask] * \
                                fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD']

                            # Stop execution if this validation condition is not satisfied before
                            # dependent processing continues.
                            if fire_hazard_fuel_clip_composite_scores:
                                # Prepare expected values for the downstream processing or
                                # validation performed in this workflow stage.
                                expected_values = np.clip(expected_values,
                                    fire_hazard_fuel_score_minimum, fire_hazard_fuel_score_maximum)
                            # Prepare expected values for the downstream processing or validation
                            # performed in this workflow stage.
                            expected_values = expected_values.astype(np.float32)
                            # Prepare actual values for the downstream processing or validation
                            # performed in this workflow stage.
                            actual_values = output_array[complete_case_mask]
                            # Evaluate actual values valid so invalid inputs or outputs can be
                            # rejected before continuing.
                            actual_values_valid = np.isfinite(actual_values) & (actual_values != \
                                fire_hazard_fuel_score_nodata)

                            # Stop execution if the prerequisite validation has not passed before
                            # this workflow stage continues.
                            if actual_values_valid.any():
                                # Evaluate expected valid values so invalid inputs or outputs can be
                                # rejected before continuing.
                                expected_valid_values = expected_values[actual_values_valid]
                                # Evaluate actual valid values so invalid inputs or outputs can be
                                # rejected before continuing.
                                actual_valid_values = actual_values[actual_values_valid]
                                # Prepare absolute differences for the downstream processing or
                                # validation performed in this workflow stage.
                                absolute_differences = np.abs(actual_valid_values - \
                                    expected_valid_values)
                                # Build the matching mask mask used to isolate records required for
                                # this analysis.
                                matching_mask = absolute_differences <= \
                                    fire_hazard_fuel_component_formula_tolerance
                                # Calculate formula checked pixels for completeness, file-integrity,
                                # or processing QA.
                                formula_checked_pixels += int(absolute_differences.size)
                                # Record whether formula matching pixels satisfies the checks
                                # required before the workflow advances.
                                formula_matching_pixels += int(matching_mask.sum())
                                # Calculate formula mismatch pixels for completeness,
                                # file-integrity, or processing QA.
                                formula_mismatch_pixels += int((~matching_mask).sum())
                                # Prepare formula difference sum for the downstream processing or
                                # validation performed in this workflow stage.
                                formula_difference_sum += \
                                    float(absolute_differences.sum(dtype=np.float64))

                                # Stop execution if this validation condition is not satisfied
                                # before dependent processing continues.
                                if absolute_differences.size > 0:
                                    # Set maximum formula difference as an explicit model or
                                    # validation parameter used consistently in downstream
                                    # calculations.
                                    maximum_formula_difference = max(maximum_formula_difference,
                                        float(absolute_differences.max()))
                        del fbfm40_array
                        del canopy_cover_array
                        del canopy_bulk_density_array
                        del output_array
                        del fbfm40_valid_mask
                        del canopy_cover_valid_mask
                        del canopy_bulk_density_valid_mask
                        del complete_case_mask
                        del output_valid_mask
# Handle the expected failure explicitly so the workflow can report or clean up the affected
# operation.
except Exception as error:
    # Evaluate fuel validation error so invalid inputs or outputs can be rejected before continuing.
    fuel_validation_error = str(error)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if validated_output_valid_pixels > 0:
    # Evaluate validated output mean so invalid inputs or outputs can be rejected before continuing.
    validated_output_mean = validated_output_sum / validated_output_valid_pixels

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if formula_checked_pixels > 0:
    # Prepare mean formula difference for the downstream processing or validation performed in this
    # workflow stage.
    mean_formula_difference = formula_difference_sum / formula_checked_pixels
# Prepare fuel value range available for the downstream processing or validation performed in this
# workflow stage.
fuel_value_range_available = all([validated_output_minimum is not None,
    validated_output_maximum is not None, validated_output_mean is not None])
# Evaluate fuel value range valid so invalid inputs or outputs can be rejected before continuing.
fuel_value_range_valid = fuel_value_range_available and validated_output_minimum >= \
    fire_hazard_fuel_score_minimum - fire_hazard_fuel_component_value_tolerance and \
    (validated_output_maximum <= fire_hazard_fuel_score_maximum + \
    fire_hazard_fuel_component_value_tolerance)
# Evaluate fire hazard fuel component formula valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_component_formula_valid = all([formula_checked_pixels >= \
    fire_hazard_fuel_component_minimum_valid_pixels,
    formula_mismatch_pixels == 0, maximum_formula_difference <= \
        fire_hazard_fuel_component_formula_tolerance,
    complete_case_without_output == 0, incomplete_case_with_output == 0,
    output_valid_outside_study_area == 0])
# Record whether fuel pixel statistics match satisfies the checks required before the workflow
# advances.
fuel_pixel_statistics_match = all([validated_complete_case_pixels == \
    fire_hazard_fuel_component_complete_case_pixels,
    validated_output_valid_pixels == fire_hazard_fuel_component_output_valid_pixels,
    validated_output_nodata_pixels == fire_hazard_fuel_component_output_nodata_pixels,
    validated_inside_study_area == fire_hazard_fuel_component_valid_inside_study_area,
    validated_outside_study_area == fire_hazard_fuel_component_valid_outside_study_area])
# Record whether fuel value statistics match satisfies the checks required before the workflow
# advances.
fuel_value_statistics_match = all([fuel_value_range_available,
    np.isclose(validated_output_minimum, fire_hazard_fuel_component_output_minimum,
    atol=fire_hazard_fuel_component_value_tolerance),
    np.isclose(validated_output_maximum, fire_hazard_fuel_component_output_maximum,
    atol=fire_hazard_fuel_component_value_tolerance),
    np.isclose(validated_output_mean, fire_hazard_fuel_component_output_mean,
    atol=fire_hazard_fuel_component_value_tolerance)])
# Prepare fuel component manifest exists for the downstream processing or validation performed in
# this workflow stage.
fuel_component_manifest_exists = Path(fire_hazard_fuel_component_manifest_path).exists()
# Evaluate fuel component manifest valid so invalid inputs or outputs can be rejected before
# continuing.
fuel_component_manifest_valid = False

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if fuel_component_manifest_exists:
    # Assemble saved fuel component manifest into a table for QA review and downstream validation.
    saved_fuel_component_manifest = pd.read_csv(fire_hazard_fuel_component_manifest_path)
    # Define the fuel manifest fields inputs required by this workflow stage.
    required_fuel_manifest_fields = ['PRODUCT_ID',
        'OUTPUT_PATH', 'FBFM40_WEIGHT', 'CANOPY_COVER_WEIGHT',
        'CANOPY_BULK_DENSITY_WEIGHT', 'BUILD_COMPLETE',
        'VALID']
    # Identify missing fuel manifest fields so unavailable prerequisites are caught before this
    # workflow stage runs.
    missing_fuel_manifest_fields = [field_name for field_name in required_fuel_manifest_fields if \
        field_name not in saved_fuel_component_manifest.columns]

    # Stop execution when required fuel manifest fields inputs are unavailable.
    if not missing_fuel_manifest_fields and len(saved_fuel_component_manifest) == 1:
        # Prepare manifest record for the downstream processing or validation performed in this
        # workflow stage.
        manifest_record = saved_fuel_component_manifest.iloc[0]
        # Prepare manifest build complete for the downstream processing or validation performed in
        # this workflow stage.
        manifest_build_complete = str(manifest_record['BUILD_COMPLETE']).strip().lower() == 'true'
        # Evaluate manifest valid so invalid inputs or outputs can be rejected before continuing.
        manifest_valid = str(manifest_record['VALID']).strip().lower() == 'true'
        # Record whether fuel component manifest valid satisfies the checks required before the
        # workflow advances.
        fuel_component_manifest_valid = all([str(manifest_record['PRODUCT_ID']) == \
            'COMPOSITE_FUEL_HAZARD',
            Path(str(manifest_record['OUTPUT_PATH'])) == \
                fire_hazard_fuel_component_validation_output_path,
            np.isclose(float(manifest_record['FBFM40_WEIGHT']),
            fire_hazard_fuel_component_weights['FBFM40_HAZARD']),
            np.isclose(float(manifest_record['CANOPY_COVER_WEIGHT']),
            fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD']),
            np.isclose(float(manifest_record['CANOPY_BULK_DENSITY_WEIGHT']),
            fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD']),
            manifest_build_complete, manifest_valid])
# Evaluate fire hazard fuel component validation complete so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_validation_complete = all([fuel_output_file_exists,
    fuel_output_extension_valid, fuel_output_file_size_valid,
    fuel_output_readable, fuel_output_grid_valid, fuel_output_structure_valid,
    fuel_input_grids_match, validated_output_valid_pixels >= \
        fire_hazard_fuel_component_minimum_valid_pixels,
    validated_inside_study_area > 0, validated_outside_study_area == 0,
    fuel_value_range_valid, fire_hazard_fuel_component_formula_valid,
    fuel_pixel_statistics_match, fuel_value_statistics_match,
    fuel_component_manifest_valid, fuel_validation_error is None])
# Evaluate fire hazard fuel component validation so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fuel_component_validation = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'PRODUCT_ID': 'COMPOSITE_FUEL_HAZARD', 'PRODUCT_NAME': \
        'Composite Surface and Canopy Fuel Hazard',
    'OUTPUT_PATH': str(fire_hazard_fuel_component_validation_output_path),
    'FILE_EXISTS': fuel_output_file_exists, 'FILE_EXTENSION_VALID': fuel_output_extension_valid,
    'FILE_SIZE_BYTES': fuel_output_file_size_bytes,
    'FILE_SIZE_MB': fuel_output_file_size_bytes / 1024 ** 2,
    'FILE_SIZE_VALID': fuel_output_file_size_valid,
    'RASTER_READABLE': fuel_output_readable, 'OUTPUT_GRID_VALID': fuel_output_grid_valid,
    'OUTPUT_STRUCTURE_VALID': fuel_output_structure_valid,
    'INPUT_GRIDS_MATCH': fuel_input_grids_match, 'TOTAL_PIXELS': validated_total_pixels,
    'COMPLETE_CASE_PIXELS': validated_complete_case_pixels,
    'OUTPUT_VALID_PIXELS': validated_output_valid_pixels,
    'OUTPUT_NODATA_PIXELS': validated_output_nodata_pixels,
    'MISSING_FBFM40_PIXELS': validated_missing_fbfm40_pixels,
    'MISSING_CANOPY_COVER_PIXELS': validated_missing_canopy_cover_pixels,
    'MISSING_CANOPY_BULK_DENSITY_PIXELS': validated_missing_canopy_bulk_density_pixels,
    'VALID_INSIDE_STUDY_AREA': validated_inside_study_area,
    'VALID_OUTSIDE_STUDY_AREA': validated_outside_study_area,
    'OUTPUT_MINIMUM': validated_output_minimum, 'OUTPUT_MAXIMUM': validated_output_maximum,
    'OUTPUT_MEAN': validated_output_mean, 'VALUE_RANGE_AVAILABLE': fuel_value_range_available,
    'VALUE_RANGE_VALID': fuel_value_range_valid, 'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
    'FORMULA_MATCHING_PIXELS': formula_matching_pixels,
    'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
    'MAXIMUM_FORMULA_DIFFERENCE': maximum_formula_difference,
    'MEAN_FORMULA_DIFFERENCE': mean_formula_difference,
    'COMPLETE_CASE_WITHOUT_OUTPUT': complete_case_without_output,
    'INCOMPLETE_CASE_WITH_OUTPUT': incomplete_case_with_output,
    'OUTPUT_VALID_OUTSIDE_STUDY_AREA': output_valid_outside_study_area,
    'FORMULA_VALID': fire_hazard_fuel_component_formula_valid,
    'PIXEL_STATISTICS_MATCH': fuel_pixel_statistics_match,
    'VALUE_STATISTICS_MATCH': fuel_value_statistics_match,
    'PROCESSING_MANIFEST_VALID': fuel_component_manifest_valid,
    'ERROR_MESSAGE': fuel_validation_error, 'VALID': \
        fire_hazard_fuel_component_validation_complete}])
# Assemble fire hazard fuel component validation summary into a table for QA review and downstream
# validation.
fire_hazard_fuel_component_validation_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'PRODUCT_ID': 'COMPOSITE_FUEL_HAZARD', 'GRID_VALID': fuel_output_grid_valid,
    'STRUCTURE_VALID': fuel_output_structure_valid,
    'INPUT_GRIDS_MATCH': fuel_input_grids_match, 'SCORE_RANGE_VALID': fuel_value_range_valid,
    'FORMULA_VALID': fire_hazard_fuel_component_formula_valid,
    'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
    'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
    'MAXIMUM_FORMULA_DIFFERENCE': maximum_formula_difference,
    'COMPLETE_CASE_WITHOUT_OUTPUT': complete_case_without_output,
    'INCOMPLETE_CASE_WITH_OUTPUT': incomplete_case_with_output,
    'PIXEL_STATISTICS_MATCH': fuel_pixel_statistics_match,
    'VALUE_STATISTICS_MATCH': fuel_value_statistics_match,
    'PROCESSING_MANIFEST_VALID': fuel_component_manifest_valid,
    'VALID_OUTSIDE_STUDY_AREA': validated_outside_study_area,
    'OUTPUT_MINIMUM': validated_output_minimum, 'OUTPUT_MAXIMUM': validated_output_maximum,
    'OUTPUT_MEAN': validated_output_mean, 'VALIDATION_COMPLETE': \
        fire_hazard_fuel_component_validation_complete}])
# Evaluate fire hazard fuel component formula validation so invalid inputs or outputs can be
# rejected before continuing.
fire_hazard_fuel_component_formula_validation = pd.DataFrame([{'COMBINATION_METHOD': \
    fire_hazard_fuel_component_method,
    'FORMULA': fire_hazard_fuel_component_formula,
    'FBFM40_WEIGHT': fire_hazard_fuel_component_weights['FBFM40_HAZARD'],
    'CANOPY_COVER_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'],
    'CANOPY_BULK_DENSITY_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD'],
    'FORMULA_CHECKED_PIXELS': formula_checked_pixels,
    'FORMULA_MATCHING_PIXELS': formula_matching_pixels,
    'FORMULA_MISMATCH_PIXELS': formula_mismatch_pixels,
    'MAXIMUM_ABSOLUTE_DIFFERENCE': maximum_formula_difference,
    'MEAN_ABSOLUTE_DIFFERENCE': mean_formula_difference,
    'FORMULA_TOLERANCE': fire_hazard_fuel_component_formula_tolerance,
    'COMPLETE_CASE_WITHOUT_OUTPUT': complete_case_without_output,
    'INCOMPLETE_CASE_WITH_OUTPUT': incomplete_case_with_output,
    'VALID': fire_hazard_fuel_component_formula_valid}])
# Build fire hazard fuel component formula validation path used to read, cache, or save this
# workflow product.
fire_hazard_fuel_component_formula_validation_path = fire_hazard_fuel_metadata_directory / \
    'fuel_component_formula_validation.csv'
# Build fire hazard fuel component failed validation path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_component_failed_validation_path = fire_hazard_fuel_metadata_directory / \
    'fuel_component_failed_validation.csv'
# Evaluate failed fuel component validation so invalid inputs or outputs can be rejected before
# continuing.
failed_fuel_component_validation = \
    fire_hazard_fuel_component_validation[~fire_hazard_fuel_component_validation['VALID']].copy() \
    .reset_index(drop=True)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_component_validation.to_csv(fire_hazard_fuel_validation_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_component_validation_summary.to_csv(fire_hazard_fuel_validation_summary_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_component_formula_validation.to_csv \
    (fire_hazard_fuel_component_formula_validation_path,
    index=False)
# Export the table so this workflow result is available to later phases.
failed_fuel_component_validation.to_csv(fire_hazard_fuel_component_failed_validation_path,
    index=False)
print(f'-> Output grid valid: {fuel_output_grid_valid}')
print(f'-> Output structure valid: {fuel_output_structure_valid}')
print(f'-> Input grids match: {fuel_input_grids_match}')
print(f'-> Complete-case pixels: {validated_complete_case_pixels:,}')
print(f'-> Valid output pixels: {validated_output_valid_pixels:,}')
print(f'-> Valid pixels outside study area: {validated_outside_study_area:,}')
print(f'-> Output score range: {validated_output_minimum:.4f} to {validated_output_maximum:.4f}')
print(f'-> Output mean score: {validated_output_mean:.4f}')
print(f'-> Formula pixels checked: {formula_checked_pixels:,}')
print(f'-> Formula mismatch pixels: {formula_mismatch_pixels:,}')
print(f'-> Maximum formula difference: {maximum_formula_difference:.8f}')
print(f'-> Complete cases without output: {complete_case_without_output:,}')
print(f'-> Incomplete cases with output: {incomplete_case_with_output:,}')
print(f'-> Formula valid: {fire_hazard_fuel_component_formula_valid}')
print(f'-> Pixel statistics match: {fuel_pixel_statistics_match}')
print(f'-> Value statistics match: {fuel_value_statistics_match}')
print(f'-> Processing manifest valid: {fuel_component_manifest_valid}')
print(f'-> Composite fuel validation complete: {fire_hazard_fuel_component_validation_complete}')
print(f'-> Detailed validation saved: {fire_hazard_fuel_validation_path}')
print(f'-> Validation summary saved: {fire_hazard_fuel_validation_summary_path}')
print(f'-> Formula validation saved: {fire_hazard_fuel_component_formula_validation_path}')
print('\n--- COMPOSITE FUEL-HAZARD VALIDATION SUMMARY ---')
display(fire_hazard_fuel_component_validation_summary)
print('\n--- COMPOSITE FUEL-HAZARD FORMULA VALIDATION ---')
display(fire_hazard_fuel_component_formula_validation)

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not failed_fuel_component_validation.empty:
    print('\n--- FAILED COMPOSITE FUEL VALIDATION ---')
    display(failed_fuel_component_validation.head(5))

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_component_validation_complete:
    raise ValueError(f'The composite fuel-hazard '
        f'raster failed independent '
        f'validation.\n\nGrid valid: '
        f'{fuel_output_grid_valid}\n'
        f'Structure valid: '
        f'{fuel_output_structure_valid}\n'
        f'Input grids match: '
        f'{fuel_input_grids_match}\n'
        f'Score range valid: '
        f'{fuel_value_range_valid}\n'
        f'Formula valid: '
        f'{fire_hazard_fuel_component_formula_valid}\n'
        f'Formula mismatch pixels: '
        f'{formula_mismatch_pixels:,}\n'
        f'Complete cases without output: '
        f'{complete_case_without_output:,}\n'
        f'Incomplete cases with output: '
        f'{incomplete_case_with_output:,}\n'
        f'Pixel statistics match: '
        f'{fuel_pixel_statistics_match}\n'
        f'Value statistics match: '
        f'{fuel_value_statistics_match}\n'
        f'Manifest valid: '
        f'{fuel_component_manifest_valid}\n'
        f'Validation error: '
        f'{fuel_validation_error}')
# Evaluate fire hazard fuel component valid so invalid inputs or outputs can be rejected before
# continuing.
fire_hazard_fuel_component_valid = fire_hazard_fuel_component_validation_complete
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The composite fuel-hazard '
    'raster passed independent file,'
    ' grid, structure, score-range, '
    'complete-case, '
    'processing-manifest, and '
    'weighted-formula validation.')
print('Every valid composite pixel '
    'was independently recalculated '
    'from the FBFM40, canopy-cover, '
    'and canopy-bulk-density source '
    'scores.')
print('The validated composite '
    'fuel-hazard raster is ready to '
    'be finalized and registered as '
    'a major input to the Composite '
    'Fire Hazard Grid.')
print('\n=== COMPOSITE FUEL-HAZARD VALIDATION COMPLETE ===')


=== VALIDATING COMPOSITE FUEL-HAZARD RASTER ===
-> Output grid valid: True
-> Output structure valid: True
-> Input grids match: True
-> Complete-case pixels: 15,475,232
-> Valid output pixels: 15,475,232
-> Valid pixels outside study area: 0
-> Output score range: 0.0000 to 0.9274
-> Output mean score: 0.2746
-> Formula pixels checked: 15,475,232
-> Formula mismatch pixels: 0
-> Maximum formula difference: 0.00000000
-> Complete cases without output: 0
-> Incomplete cases with output: 0
-> Formula valid: True
-> Pixel statistics match: True
-> Value statistics match: True
-> Processing manifest valid: True
-> Composite fuel validation complete: True
-> Detailed validation saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_hazard_validation.csv
-> Validation summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_hazard_validation_summary.csv
-> Formula validation saved: C:\Users\adamd\Proj

,COMPONENT_ID,PRODUCT_ID,GRID_VALID,STRUCTURE_VALID,INPUT_GRIDS_MATCH,SCORE_RANGE_VALID,FORMULA_VALID,FORMULA_CHECKED_PIXELS,FORMULA_MISMATCH_PIXELS,MAXIMUM_FORMULA_DIFFERENCE,COMPLETE_CASE_WITHOUT_OUTPUT,INCOMPLETE_CASE_WITH_OUTPUT,PIXEL_STATISTICS_MATCH,VALUE_STATISTICS_MATCH,PROCESSING_MANIFEST_VALID,VALID_OUTSIDE_STUDY_AREA,OUTPUT_MINIMUM,OUTPUT_MAXIMUM,OUTPUT_MEAN,VALIDATION_COMPLETE
0,FUEL_HAZARD,COMPOSITE_FUEL_HAZARD,True,True,True,True,True,15475232,0,0.0,0,0,True,True,True,0,0.0,0.927376,0.274636,True



--- COMPOSITE FUEL-HAZARD FORMULA VALIDATION ---


,COMBINATION_METHOD,FORMULA,FBFM40_WEIGHT,CANOPY_COVER_WEIGHT,CANOPY_BULK_DENSITY_WEIGHT,FORMULA_CHECKED_PIXELS,FORMULA_MATCHING_PIXELS,FORMULA_MISMATCH_PIXELS,MAXIMUM_ABSOLUTE_DIFFERENCE,MEAN_ABSOLUTE_DIFFERENCE,FORMULA_TOLERANCE,COMPLETE_CASE_WITHOUT_OUTPUT,INCOMPLETE_CASE_WITH_OUTPUT,VALID
0,weighted_linear_combination,(FBFM40_HAZARD * 0.50) + (CANOPY_COVER_HAZARD ...,0.5,0.25,0.25,15475232,15475232,0,0.0,0.0,0.00001,0,0,True



NOTE:
The composite fuel-hazard raster passed independent file, grid, structure, score-range, complete-case, processing-manifest, and weighted-formula validation.
Every valid composite pixel was independently recalculated from the FBFM40, canopy-cover, and canopy-bulk-density source scores.
The validated composite fuel-hazard raster is ready to be finalized and registered as a major input to the Composite Fire Hazard Grid.

=== COMPOSITE FUEL-HAZARD VALIDATION COMPLETE ===


### Finalizing the Fuel-Hazard Component


In [116]:
print('=== FINALIZING THE FUEL-HAZARD COMPONENT ===')
# Prepare required fuel component finalization inputs for the downstream processing or validation
# performed in this workflow stage.
required_fuel_component_finalization_inputs = ['fire_hazard_fuel_component_valid',
    'fire_hazard_fuel_component_validation_complete',
    'fire_hazard_fuel_component_validation', 'fire_hazard_fuel_component_validation_summary',
    'fire_hazard_fuel_component_formula_validation',
    'fire_hazard_fuel_component_output_path', 'fire_hazard_fuel_component_manifest',
    'fire_hazard_fuel_component_manifest_path', 'fire_hazard_fuel_component_policy_path',
    'fire_hazard_fuel_component_weight_table_path',
    'fire_hazard_fuel_validation_path', 'fire_hazard_fuel_validation_summary_path',
    'fire_hazard_fuel_component_formula_validation_path',
    'fire_hazard_fuel_component_weights', 'fire_hazard_fuel_component_method',
    'fire_hazard_fuel_component_formula', 'fire_hazard_fuel_component_output_valid_pixels',
    'fire_hazard_fuel_component_output_nodata_pixels',
    'fire_hazard_fuel_component_valid_inside_study_area',
    'fire_hazard_fuel_component_valid_outside_study_area',
    'fire_hazard_fuel_component_output_minimum', 'fire_hazard_fuel_component_output_maximum',
    'fire_hazard_fuel_component_output_mean', 'fire_hazard_fuel_component_complete_case_pixels',
    'fire_hazard_fuel_score_minimum', 'fire_hazard_fuel_score_maximum',
    'fire_hazard_fuel_score_nodata', 'fire_hazard_fuel_metadata_directory',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform']
# Identify missing fuel component finalization inputs so unavailable prerequisites are caught before
# this workflow stage runs.
missing_fuel_component_finalization_inputs = [object_name for object_name in \
    required_fuel_component_finalization_inputs if object_name not in globals()]

# Stop execution if missing fuel component finalization inputs remain unresolved before this
# workflow stage begins.
if missing_fuel_component_finalization_inputs:
    raise NameError(f'The following fuel-component '
        f'finalization objects are '
        f'missing:\n'
        f'{missing_fuel_component_finalization_inputs}\n\n'
        f'Run Validate Composite '
        f'Fuel-Hazard Raster before '
        f'finalizing the fuel-hazard '
        f'component.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_component_valid:
    raise ValueError('The composite fuel-hazard raster has not passed independent validation.')

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_component_validation_complete:
    raise ValueError('The composite fuel-hazard validation workflow is incomplete.')

# Require the expected number of records before continuing so source configuration remains
# unambiguous.
if fire_hazard_fuel_component_validation.empty or len(fire_hazard_fuel_component_validation) != 1 \
    or (not fire_hazard_fuel_component_validation['VALID'].fillna(False).all()):
    raise ValueError('The composite fuel-hazard '
        'validation table does not '
        'contain one valid product '
        'record.')
# Build fire hazard fuel hazard path used to read, cache, or save this workflow product.
fire_hazard_fuel_hazard_path = Path(fire_hazard_fuel_component_output_path)

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if not fire_hazard_fuel_hazard_path.exists() or not fire_hazard_fuel_hazard_path.is_file() or \
    fire_hazard_fuel_hazard_path.stat().st_size <= 0:
    raise FileNotFoundError(f'The validated composite '
        f'fuel-hazard raster is '
        f'unavailable:\n'
        f'{fire_hazard_fuel_hazard_path}')

# Open the file in a managed context so required content is processed and the resource closes
# cleanly.
with rasterio.open(fire_hazard_fuel_hazard_path) as final_fuel_source:
    # Evaluate fire hazard fuel final structure valid so invalid inputs or outputs can be rejected
    # before continuing.
    fire_hazard_fuel_final_structure_valid = all([final_fuel_source.count == 1,
        final_fuel_source.width == fire_hazard_alignment_width,
        final_fuel_source.height == fire_hazard_alignment_height,
        final_fuel_source.crs is not None, final_fuel_source.crs == \
            rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
        final_fuel_source.transform.almost_equals(fire_hazard_alignment_transform),
        final_fuel_source.dtypes[0] == 'float32', final_fuel_source.nodata == \
            fire_hazard_fuel_score_nodata])
    # Store fire hazard fuel final crs so spatial operations use the required coordinate reference
    # system.
    fire_hazard_fuel_final_crs = final_fuel_source.crs.to_string() if final_fuel_source.crs is \
        not None else None
    # Calculate fire hazard fuel final width so raster processing covers the analysis grid in
    # controlled blocks.
    fire_hazard_fuel_final_width = final_fuel_source.width
    # Calculate fire hazard fuel final height so raster processing covers the analysis grid in
    # controlled blocks.
    fire_hazard_fuel_final_height = final_fuel_source.height
    # Prepare fire hazard fuel final data type for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_fuel_final_dtype = final_fuel_source.dtypes[0]
    # Set fire hazard fuel final NoData as an explicit model or validation parameter used
    # consistently in downstream calculations.
    fire_hazard_fuel_final_nodata = final_fuel_source.nodata
    # Prepare fire hazard fuel final tags for the fuel-hazard calculation and subsequent QA checks.
    fire_hazard_fuel_final_tags = final_fuel_source.tags()

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_final_structure_valid:
    raise ValueError('The finalized composite '
        'fuel-hazard raster failed its '
        'final structural check.')
# Define the fuel component artifact paths inputs required by this workflow stage.
required_fuel_component_artifact_paths = {'Processing manifest': \
    Path(fire_hazard_fuel_component_manifest_path),
    'Weighting policy': Path(fire_hazard_fuel_component_policy_path),
    'Weight table': Path(fire_hazard_fuel_component_weight_table_path),
    'Detailed validation': Path(fire_hazard_fuel_validation_path),
    'Validation summary': Path(fire_hazard_fuel_validation_summary_path),
    'Formula validation': Path(fire_hazard_fuel_component_formula_validation_path)}
# Identify missing fuel component artifacts so unavailable prerequisites are caught before this
# workflow stage runs.
missing_fuel_component_artifacts = [artifact_name for artifact_name,
    artifact_path in required_fuel_component_artifact_paths.items() if not artifact_path.exists() \
        or not artifact_path.is_file() or artifact_path.stat().st_size <= 0]

# Stop execution when required fuel component artifacts inputs are unavailable.
if missing_fuel_component_artifacts:
    raise FileNotFoundError(f'The following required '
        f'fuel-component artifacts are '
        f'missing or empty:\n'
        f'{missing_fuel_component_artifacts}')
# Build fire hazard fuel final registry path used to read, cache, or save this workflow product.
fire_hazard_fuel_final_registry_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_final_product_registry.csv'
# Build fire hazard fuel final summary path used to read, cache, or save this workflow product.
fire_hazard_fuel_final_summary_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_final_component_summary.csv'
# Build fire hazard fuel final policy path used to read, cache, or save this workflow product.
fire_hazard_fuel_final_policy_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_final_component_policy.json'
# Build fire hazard fuel completion marker path used to read, cache, or save this workflow product.
fire_hazard_fuel_completion_marker_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_component_complete.json'
# Build fire hazard fuel final input inventory path used to read, cache, or save this workflow
# product.
fire_hazard_fuel_final_input_inventory_path = fire_hazard_fuel_metadata_directory / \
    'fuel_hazard_final_input_inventory.csv'
# Assemble fire hazard fuel final input inventory into a table for QA review and downstream
# validation.
fire_hazard_fuel_final_input_inventory = pd.DataFrame([{'INPUT_ID': 'FBFM40_HAZARD',
    'INPUT_NAME': 'FBFM40 Surface-Fuel Hazard Score',
    'INPUT_PATH': str(fire_hazard_fuel_component_input_paths['FBFM40_HAZARD']),
    'WEIGHT': fire_hazard_fuel_component_weights['FBFM40_HAZARD'],
    'INPUT_ROLE': 'Primary surface-fuel behavior',
    'VALIDATED': True}, {'INPUT_ID': 'CANOPY_COVER_HAZARD',
    'INPUT_NAME': 'Forest Canopy-Cover Hazard Score',
    'INPUT_PATH': str(fire_hazard_fuel_component_input_paths['CANOPY_COVER_HAZARD']),
    'WEIGHT': fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'],
    'INPUT_ROLE': 'Horizontal canopy continuity', 'VALIDATED': True},
    {'INPUT_ID': 'CANOPY_BULK_DENSITY_HAZARD', 'INPUT_NAME': \
        'Forest Canopy Bulk-Density Hazard Score',
    'INPUT_PATH': str(fire_hazard_fuel_component_input_paths['CANOPY_BULK_DENSITY_HAZARD']),
    'WEIGHT': fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD'],
    'INPUT_ROLE': 'Available canopy fuel mass', 'VALIDATED': True}])
# Define expected final fuel input ids used to configure this workflow stage.
expected_final_fuel_input_ids = {'FBFM40_HAZARD',
    'CANOPY_COVER_HAZARD', 'CANOPY_BULK_DENSITY_HAZARD'}
# Prepare final fuel input IDs for the downstream processing or validation performed in this
# workflow stage.
final_fuel_input_ids = set(fire_hazard_fuel_final_input_inventory['INPUT_ID'])
# Set final fuel input weight sum as an explicit model or validation parameter used consistently in
# downstream calculations.
final_fuel_input_weight_sum = float(fire_hazard_fuel_final_input_inventory['WEIGHT'].sum())
# Evaluate fire hazard fuel final input inventory valid so invalid inputs or outputs can be rejected
# before continuing.
fire_hazard_fuel_final_input_inventory_valid = all([len(fire_hazard_fuel_final_input_inventory) \
    == 3,
    final_fuel_input_ids == expected_final_fuel_input_ids,
    np.isclose(final_fuel_input_weight_sum, 1.0, atol=1e-09),
    fire_hazard_fuel_final_input_inventory['VALIDATED'].all()])

# Stop execution if the prerequisite validation has not passed before this workflow stage continues.
if not fire_hazard_fuel_final_input_inventory_valid:
    raise ValueError('The final fuel-component input inventory failed validation.')
# Calculate fire hazard fuel final output size bytes for completeness, file-integrity, or processing
# QA.
fire_hazard_fuel_final_output_size_bytes = int(fire_hazard_fuel_hazard_path.stat().st_size)
# Prepare fire hazard fuel final output size mb for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_final_output_size_mb = fire_hazard_fuel_final_output_size_bytes / 1024 ** 2

# Combine the finalized component rasters using the configured weights to create the Composite Fire
# Hazard Grid.
# Prepare fire hazard fuel final registry for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_fuel_final_registry = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'PRODUCT_ID': 'COMPOSITE_FUEL_HAZARD', 'PRODUCT_NAME': \
        'Composite Surface and Canopy Fuel Hazard',
    'PRODUCT_ROLE': 'Fuel component for Composite Fire Hazard Grid',
    'PRIMARY_OUTPUT_PATH': str(fire_hazard_fuel_hazard_path),
    'COMBINATION_METHOD': fire_hazard_fuel_component_method,
    'FORMULA': fire_hazard_fuel_component_formula,
    'FBFM40_WEIGHT': fire_hazard_fuel_component_weights['FBFM40_HAZARD'],
    'CANOPY_COVER_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'],
    'CANOPY_BULK_DENSITY_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD'],
    'VALID_PIXELS': fire_hazard_fuel_component_output_valid_pixels,
    'NODATA_PIXELS': fire_hazard_fuel_component_output_nodata_pixels,
    'COMPLETE_CASE_PIXELS': fire_hazard_fuel_component_complete_case_pixels,
    'VALID_INSIDE_STUDY_AREA': fire_hazard_fuel_component_valid_inside_study_area,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_fuel_component_valid_outside_study_area,
    'ACTUAL_SCORE_MINIMUM': fire_hazard_fuel_component_output_minimum,
    'ACTUAL_SCORE_MAXIMUM': fire_hazard_fuel_component_output_maximum,
    'ACTUAL_SCORE_MEAN': fire_hazard_fuel_component_output_mean,
    'TARGET_CRS': fire_hazard_fuel_final_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_fuel_final_width, 'TARGET_HEIGHT': fire_hazard_fuel_final_height,
    'OUTPUT_DTYPE': fire_hazard_fuel_final_dtype, 'OUTPUT_NODATA': fire_hazard_fuel_final_nodata,
    'OUTPUT_SIZE_BYTES': fire_hazard_fuel_final_output_size_bytes,
    'OUTPUT_SIZE_MB': fire_hazard_fuel_final_output_size_mb,
    'VALIDATION_COMPLETE': True, 'READY_FOR_COMPOSITE_FIRE_HAZARD': True,
    'PRODUCT_COMPLETE': True}])
# Assemble fire hazard fuel final summary into a table for QA review and downstream validation.
fire_hazard_fuel_final_summary = pd.DataFrame([{'COMPONENT_ID': 'FUEL_HAZARD',
    'STATUS': 'Complete', 'PRIMARY_OUTPUT': str(fire_hazard_fuel_hazard_path),
    'INPUT_PRODUCT_COUNT': 3, 'COMBINATION_METHOD': fire_hazard_fuel_component_method,
    'FBFM40_WEIGHT': fire_hazard_fuel_component_weights['FBFM40_HAZARD'],
    'CANOPY_COVER_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_COVER_HAZARD'],
    'CANOPY_BULK_DENSITY_WEIGHT': fire_hazard_fuel_component_weights['CANOPY_BULK_DENSITY_HAZARD'],
    'VALID_PIXELS': fire_hazard_fuel_component_output_valid_pixels,
    'NODATA_PIXELS': fire_hazard_fuel_component_output_nodata_pixels,
    'ACTUAL_SCORE_MINIMUM': fire_hazard_fuel_component_output_minimum,
    'ACTUAL_SCORE_MAXIMUM': fire_hazard_fuel_component_output_maximum,
    'ACTUAL_SCORE_MEAN': fire_hazard_fuel_component_output_mean,
    'VALID_OUTSIDE_STUDY_AREA': fire_hazard_fuel_component_valid_outside_study_area,
    'GRID_VALID': fire_hazard_fuel_final_structure_valid,
    'FORMULA_VALID': bool(fire_hazard_fuel_component_formula_validation['VALID'].iloc[0]),
    'VALIDATION_COMPLETE': fire_hazard_fuel_component_validation_complete,
    'READY_FOR_COMPOSITE_FIRE_HAZARD': True}])
# Collect fire hazard fuel final policy in one configuration object so downstream steps use the same
# processing rules.
# Define the final policy for the completed composite fuel-hazard product.
fire_hazard_fuel_final_policy = {
    # Identify the completed product and its role in the fire-hazard model.
    'component_id': 'FUEL_HAZARD',
    'product_id': 'COMPOSITE_FUEL_HAZARD',
    'status': 'complete',
    'primary_output': str(
        fire_hazard_fuel_hazard_path
    ),
    'model_role': (
        'Fuel-hazard input to the Composite Fire Hazard Grid.'
    ),

    # Record the method and formula used to combine the fuel-hazard inputs.
    'method': fire_hazard_fuel_component_method,
    'formula': fire_hazard_fuel_component_formula,

    # Record the weight assigned to each fuel-hazard component.
    'weights': {
        component_id: float(component_weight)
        for component_id, component_weight
        in fire_hazard_fuel_component_weights.items()
    },

    # Define the valid score range, NoData value, and interpretation.
    'score_policy': {
        'minimum': float(
            fire_hazard_fuel_score_minimum
        ),
        'maximum': float(
            fire_hazard_fuel_score_maximum
        ),
        'nodata': float(
            fire_hazard_fuel_score_nodata
        ),
        'interpretation': (
            'Higher scores represent '
            'greater relative surface and '
            'canopy fuel hazard.'
        ),
    },

    # Require complete input data before assigning a composite fuel-hazard score.
    'valid_data_policy': {
        'require_all_components': True,
        'incomplete_pixel_action': 'assign_nodata',
    },

    # Record the target-grid properties of the completed fuel-hazard raster.
    'target_grid': {
        'crs': fire_hazard_target_crs,
        'cell_size_meters': float(
            fire_hazard_cell_size
        ),
        'width': int(
            fire_hazard_alignment_width
        ),
        'height': int(
            fire_hazard_alignment_height
        ),
    },

    # Record final pixel counts and composite fuel-hazard statistics.
    'statistics': {
        'valid_pixels': int(
            fire_hazard_fuel_component_output_valid_pixels
        ),
        'nodata_pixels': int(
            fire_hazard_fuel_component_output_nodata_pixels
        ),
        'complete_case_pixels': int(
            fire_hazard_fuel_component_complete_case_pixels
        ),
        'minimum': float(
            fire_hazard_fuel_component_output_minimum
        ),
        'maximum': float(
            fire_hazard_fuel_component_output_maximum
        ),
        'mean': float(
            fire_hazard_fuel_component_output_mean
        ),
        'valid_outside_study_area': int(
            fire_hazard_fuel_component_valid_outside_study_area
        ),
    },

    # Record the processing, validation, and finalization files for the product.
    'artifacts': {
        'processing_manifest': str(
            fire_hazard_fuel_component_manifest_path
        ),
        'weighting_policy': str(
            fire_hazard_fuel_component_policy_path
        ),
        'weight_table': str(
            fire_hazard_fuel_component_weight_table_path
        ),
        'detailed_validation': str(
            fire_hazard_fuel_validation_path
        ),
        'validation_summary': str(
            fire_hazard_fuel_validation_summary_path
        ),
        'formula_validation': str(
            fire_hazard_fuel_component_formula_validation_path
        ),
        'final_input_inventory': str(
            fire_hazard_fuel_final_input_inventory_path
        ),
        'final_registry': str(
            fire_hazard_fuel_final_registry_path
        ),
        'final_summary': str(
            fire_hazard_fuel_final_summary_path
        ),
    },

    # Record the final grid, formula, and component validation results.
    'validation': {
        'grid_valid': bool(
            fire_hazard_fuel_final_structure_valid
        ),
        'formula_valid': bool(
            fire_hazard_fuel_component_formula_validation[
                'VALID'
            ].iloc[0]
        ),
        'component_valid': bool(
            fire_hazard_fuel_component_valid
        ),
    },

    # Confirm that the fuel-hazard product can be used in the final composite grid.
    'ready_for_composite_fire_hazard': True,
}
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_final_input_inventory.to_csv(fire_hazard_fuel_final_input_inventory_path,
    index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_final_registry.to_csv(fire_hazard_fuel_final_registry_path, index=False)
# Export the table so this workflow result is available to later phases.
fire_hazard_fuel_final_summary.to_csv(fire_hazard_fuel_final_summary_path, index=False)

# Prepare with fire hazard fuel final policy path.open so this workflow stage has the values
# required for downstream spatial processing and QA.
with fire_hazard_fuel_final_policy_path.open('w', encoding='utf-8') as fuel_final_policy_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fuel_final_policy, fuel_final_policy_file, indent=2)
# Prepare fire hazard fuel completion marker for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_completion_marker = {'component_id': 'FUEL_HAZARD',
    'product_id': 'COMPOSITE_FUEL_HAZARD', 'status': 'complete',
    'primary_output': str(fire_hazard_fuel_hazard_path),
    'validation_complete': True, 'ready_for_composite_fire_hazard': True,
    'final_input_inventory': str(fire_hazard_fuel_final_input_inventory_path),
    'final_registry': str(fire_hazard_fuel_final_registry_path),
    'final_summary': str(fire_hazard_fuel_final_summary_path),
    'final_policy': str(fire_hazard_fuel_final_policy_path)}

# Document this operation so its role in the current fuel-hazard workflow is clear before processing
# continues.
with fire_hazard_fuel_completion_marker_path.open('w',
    encoding='utf-8') as fuel_completion_marker_file:
    # Write the structured metadata needed to reproduce this processing stage.
    json.dump(fire_hazard_fuel_completion_marker, fuel_completion_marker_file, indent=2)
# Build fire hazard component registry path used to read, cache, or save this workflow product.
fire_hazard_component_registry_path = fire_hazard_fuel_metadata_directory.parent / \
    'fire_hazard_component_registry.csv'

# Stop execution if a required raster file is missing or empty before spatial processing begins.
if fire_hazard_component_registry_path.exists():
    # Prepare fire hazard component registry for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_component_registry = pd.read_csv(fire_hazard_component_registry_path)
else:
    # Prepare fire hazard component registry for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_component_registry = pd.DataFrame()

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_component_registry.empty and 'COMPONENT_ID' in \
    fire_hazard_component_registry.columns:
    # Prepare fire hazard component registry for the fuel-hazard calculation and subsequent QA
    # checks.
    fire_hazard_component_registry = \
        fire_hazard_component_registry[fire_hazard_component_registry['COMPONENT_ID'].astype(str) \
        .str.upper() != 'FUEL_HAZARD'].copy()
# Prepare fire hazard fuel component registry record for the fuel-hazard calculation and subsequent
# QA checks.
fire_hazard_fuel_component_registry_record = {'COMPONENT_ID': 'FUEL_HAZARD',
    'COMPONENT_NAME': 'Composite Surface and Canopy Fuel Hazard',
    'PRIMARY_OUTPUT_PATH': str(fire_hazard_fuel_hazard_path),
    'SCORE_MINIMUM': fire_hazard_fuel_score_minimum,
    'SCORE_MAXIMUM': fire_hazard_fuel_score_maximum,
    'ACTUAL_MINIMUM': fire_hazard_fuel_component_output_minimum,
    'ACTUAL_MAXIMUM': fire_hazard_fuel_component_output_maximum,
    'ACTUAL_MEAN': fire_hazard_fuel_component_output_mean,
    'TARGET_CRS': fire_hazard_target_crs, 'CELL_SIZE_METERS': fire_hazard_cell_size,
    'VALIDATION_COMPLETE': True, 'COMPONENT_COMPLETE': True,
    'READY_FOR_COMPOSITE_FIRE_HAZARD': True}
# Prepare fire hazard component registry for the fuel-hazard calculation and subsequent QA checks.
fire_hazard_component_registry = pd.concat([fire_hazard_component_registry,
    pd.DataFrame([fire_hazard_fuel_component_registry_record])],
    ignore_index=True, sort=False).reset_index(drop=True)
# Export the table so this workflow result is available to later phases.
fire_hazard_component_registry.to_csv(fire_hazard_component_registry_path, index=False)
# Prepare fire hazard fuel component finalized for the fuel-hazard calculation and subsequent QA
# checks.
fire_hazard_fuel_component_finalized = all([fire_hazard_fuel_component_valid,
    fire_hazard_fuel_component_validation_complete,
    fire_hazard_fuel_final_structure_valid, fire_hazard_fuel_final_input_inventory_valid,
    fire_hazard_fuel_hazard_path.exists(), fire_hazard_fuel_final_input_inventory_path.exists(),
    fire_hazard_fuel_final_registry_path.exists(),
    fire_hazard_fuel_final_summary_path.exists(), fire_hazard_fuel_final_policy_path.exists(),
    fire_hazard_fuel_completion_marker_path.exists(),
    fire_hazard_component_registry_path.exists()])

# Stop execution if this validation condition is not satisfied before dependent processing
# continues.
if not fire_hazard_fuel_component_finalized:
    raise ValueError('The fuel-hazard component did not pass final completion checks.')
print('-> Component ID: FUEL_HAZARD')
print('-> Component status: Complete')
print(f'-> Final raster: {fire_hazard_fuel_hazard_path}')
print(f'-> Combination method: {fire_hazard_fuel_component_method}')
print(f'-> Valid pixels: {fire_hazard_fuel_component_output_valid_pixels:,}')
print(f'-> NoData pixels: {fire_hazard_fuel_component_output_nodata_pixels:,}')
print(f'-> Final score range: '
    f'{fire_hazard_fuel_component_output_minimum:.4f} '
    f'to '
    f'{fire_hazard_fuel_component_output_maximum:.4f}')
print(f'-> Final mean score: {fire_hazard_fuel_component_output_mean:.4f}')
print(f'-> Valid pixels outside study '
    f'area: '
    f'{fire_hazard_fuel_component_valid_outside_study_area:,}')
print(f"-> Formula valid: {bool(fire_hazard_fuel_component_formula_validation['VALID'].iloc[0])}")
print(f'-> Output size: {fire_hazard_fuel_final_output_size_mb:,.2f} MB')
print(f'-> Component finalized: {fire_hazard_fuel_component_finalized}')
print(f'-> Final input inventory saved: {fire_hazard_fuel_final_input_inventory_path}')
print(f'-> Final registry saved: {fire_hazard_fuel_final_registry_path}')
print(f'-> Final summary saved: {fire_hazard_fuel_final_summary_path}')
print(f'-> Final policy saved: {fire_hazard_fuel_final_policy_path}')
print(f'-> Completion marker saved: {fire_hazard_fuel_completion_marker_path}')
print(f'-> Project component registry saved: {fire_hazard_component_registry_path}')
print('\n--- FINAL FUEL-HAZARD COMPONENT SUMMARY ---')
display(fire_hazard_fuel_final_summary)
print('\n--- FINAL FUEL-HAZARD INPUT INVENTORY ---')
display(fire_hazard_fuel_final_input_inventory)
print('\n--- FIRE-HAZARD COMPONENT REGISTRY ---')
display(fire_hazard_component_registry.head(5))
import gc
# Release unneeded Python objects before the next raster-intensive operation to limit memory
# pressure.
gc.collect()
print('\nNOTE:')
print('The fuel-hazard component is '
    'fully processed, independently '
    'validated, documented, and '
    'registered.')
print('The final raster combines the '
    'FBFM40 surface-fuel hazard, '
    'normalized canopy cover, and '
    'normalized canopy bulk density '
    'on the common zero-to-one '
    'hazard scale.')

# Transform vegetation condition into a normalized dryness score
# where larger values represent greater fire potential.
print('This component is now ready to '
    'be combined with the '
    'vegetation-dryness, terrain, '
    'historical-fire, and '
    'human-ignition components.')
print('\n=== FUEL-HAZARD COMPONENT FINALIZED ===')


=== FINALIZING THE FUEL-HAZARD COMPONENT ===
-> Component ID: FUEL_HAZARD
-> Component status: Complete
-> Final raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\component_output\composite_fuel_hazard_score.tif
-> Combination method: weighted_linear_combination
-> Valid pixels: 15,475,232
-> NoData pixels: 121,295,786
-> Final score range: 0.0000 to 0.9274
-> Final mean score: 0.2746
-> Valid pixels outside study area: 0
-> Formula valid: True
-> Output size: 31.36 MB
-> Component finalized: True
-> Final input inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_hazard_final_input_inventory.csv
-> Final registry saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_hazard_final_product_registry.csv
-> Final summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\fuel_hazard_final_component_summary.csv
-> Fin

,COMPONENT_ID,STATUS,PRIMARY_OUTPUT,INPUT_PRODUCT_COUNT,COMBINATION_METHOD,FBFM40_WEIGHT,CANOPY_COVER_WEIGHT,CANOPY_BULK_DENSITY_WEIGHT,VALID_PIXELS,NODATA_PIXELS,ACTUAL_SCORE_MINIMUM,ACTUAL_SCORE_MAXIMUM,ACTUAL_SCORE_MEAN,VALID_OUTSIDE_STUDY_AREA,GRID_VALID,FORMULA_VALID,VALIDATION_COMPLETE,READY_FOR_COMPOSITE_FIRE_HAZARD
0,FUEL_HAZARD,Complete,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,3,weighted_linear_combination,0.5,0.25,0.25,15475232,121295786,0.0,0.927376,0.274636,0,True,True,True,True



--- FINAL FUEL-HAZARD INPUT INVENTORY ---


,INPUT_ID,INPUT_NAME,INPUT_PATH,WEIGHT,INPUT_ROLE,VALIDATED
0,FBFM40_HAZARD,FBFM40 Surface-Fuel Hazard Score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.50,Primary surface-fuel behavior,True
1,CANOPY_COVER_HAZARD,Forest Canopy-Cover Hazard Score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.25,Horizontal canopy continuity,True
2,CANOPY_BULK_DENSITY_HAZARD,Forest Canopy Bulk-Density Hazard Score,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.25,Available canopy fuel mass,True



--- FIRE-HAZARD COMPONENT REGISTRY ---


,COMPONENT_ID,COMPONENT_NAME,PRIMARY_OUTPUT_PATH,SCORE_MINIMUM,SCORE_MAXIMUM,ACTUAL_MINIMUM,ACTUAL_MAXIMUM,ACTUAL_MEAN,TARGET_CRS,CELL_SIZE_METERS,VALIDATION_COMPLETE,COMPONENT_COMPLETE,READY_FOR_COMPOSITE_FIRE_HAZARD
0,FUEL_HAZARD,Composite Surface and Canopy Fuel Hazard,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.0,1.0,0.0,0.927376,0.274636,EPSG:26912,30,True,True,True



NOTE:
The fuel-hazard component is fully processed, independently validated, documented, and registered.
The final raster combines the FBFM40 surface-fuel hazard, normalized canopy cover, and normalized canopy bulk density on the common zero-to-one hazard scale.
This component is now ready to be combined with the vegetation-dryness, terrain, historical-fire, and human-ignition components.

=== FUEL-HAZARD COMPONENT FINALIZED ===


# CHRG Phase 9 – Terrain

## Purpose

Acquire USGS 3DEP elevation tiles, build an aligned project-grid DEM mosaic, derive slope and aspect, normalize both terrain variables, and create the terrain-hazard component.


## Restore Project-Level Fire-Hazard Directories


In [117]:
print('=== RESTORING PROJECT-LEVEL FIRE-HAZARD DIRECTORIES ===')
# Import Path support for validating and rebuilding project workspaces.
from pathlib import Path

# Select the processing branch required by the current cache, validation, or data state.
if 'fire_hazard_fuel_metadata_directory' in globals():
    # Route fire hazard fuel metadata files to the workspace reserved for this phase.
    fire_hazard_fuel_metadata_directory = Path(fire_hazard_fuel_metadata_directory)
    # Route fire hazard output files to the workspace reserved for this phase.
    fire_hazard_output_directory = fire_hazard_fuel_metadata_directory.parent
else:
    raise NameError("The project-level fire-hazard "
        "output directory cannot be "
        "restored because "
        "'fire_hazard_fuel_metadata_directory' "
        "is missing.\n\nRerun the "
        "fuel-hazard configuration cell "
        "that defines the fuel metadata "
        "workspace.")
# Route fire hazard metadata files to the workspace reserved for this phase.
fire_hazard_metadata_directory = fire_hazard_output_directory / 'metadata'

# Prepare each required workspace before any phase outputs are written.
for directory in [fire_hazard_output_directory, fire_hazard_metadata_directory]:
    # Create the required workspace before writing cached sources, metadata, or raster products.
    directory.mkdir(parents=True, exist_ok=True)

    # Stop execution if the required file or directory is unavailable for this processing step.
    if not directory.exists():
        raise FileNotFoundError(f'''The required fire-hazard directory could not be created:
{directory}''')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Fire-hazard output directory: {fire_hazard_output_directory}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Fire-hazard metadata directory: {fire_hazard_metadata_directory}')
# Report workflow status so notebook execution can be verified interactively.
print('\n=== PROJECT-LEVEL FIRE-HAZARD DIRECTORIES RESTORED ===')

# Report workflow status so notebook execution can be verified interactively.


=== RESTORING PROJECT-LEVEL FIRE-HAZARD DIRECTORIES ===
-> Fire-hazard output directory: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component
-> Fire-hazard metadata directory: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata

=== PROJECT-LEVEL FIRE-HAZARD DIRECTORIES RESTORED ===


## Configure Remaining Fire-Hazard Workflow


In [118]:
print('=== CONFIGURING REMAINING FIRE-HAZARD WORKFLOW ===')
# Import file, raster, resampling, and memory-management tools used by the remaining hazard phases.
from pathlib import Path                   # Work with file and directory paths.
from contextlib import ExitStack           # Manage multiple context managers safely.
from rasterio.enums import Resampling      # Specify raster resampling methods.
from rasterio.features import rasterize    # Convert vector features to raster cells.
from rasterio.warp import reproject        # Reproject and align raster data.
from rasterio.windows import Window        # Define raster processing windows.

# Model human-ignition potential from proximity to roads,
# developed areas, and other likely ignition sources.
from scipy.ndimage import distance_transform_edt
import gc

# Transform vegetation condition into a normalized dryness score
# where larger values represent greater fire potential.
# List the prerequisite objects required before this workflow stage can run.
required_remaining_workflow_objects = ['fire_hazard_fuel_component_finalized',
    'fire_hazard_fuel_hazard_path', 'fire_hazard_vegetation_dryness_component_finalized',
    'fire_hazard_vegetation_dryness_hazard_path', 'fire_hazard_source_paths',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_cell_size',
    'fire_hazard_nodata_value', 'fire_hazard_components',
    'fire_hazard_output_directory', 'fire_hazard_metadata_directory']
# Identify missing prerequisites so execution can stop before incomplete processing begins.
missing_remaining_workflow_objects = [object_name for object_name in \
    required_remaining_workflow_objects if object_name not in globals()]

# Stop execution if required upstream objects are missing from the sequential workflow.
if missing_remaining_workflow_objects:
    raise NameError(f'''The following required objects are missing:
{missing_remaining_workflow_objects}

Run the completed vegetation-dryness and fuel-hazard workflow cells before continuing.''')

# Combine surface-fuel and canopy-fuel indicators using the
# configured internal weights.
# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_fuel_component_finalized:
    raise ValueError('The fuel-hazard component is not finalized.')

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_vegetation_dryness_component_finalized:
    raise ValueError('The vegetation-dryness component is not finalized.')
# Use the existing fire-hazard output root as the parent workspace for all remaining components.
fire_hazard_remaining_root = Path(fire_hazard_output_directory)
# Use the existing metadata workspace to store QA records for the remaining hazard components.
fire_hazard_remaining_metadata = Path(fire_hazard_metadata_directory)
# Route fire hazard terrain files to the workspace reserved for this phase.
fire_hazard_terrain_directory = fire_hazard_remaining_root / 'terrain_hazard'

# Convert historical fire occurrence and proximity into a
# normalized fire-likelihood component.
# Route fire hazard historical files to the workspace reserved for this phase.
fire_hazard_historical_directory = fire_hazard_remaining_root / 'historical_fire_hazard'
# Route fire hazard human files to the workspace reserved for this phase.
fire_hazard_human_directory = fire_hazard_remaining_root / 'human_ignition_hazard'

# Combine the finalized component rasters using the configured
# weights to create the Composite Fire Hazard Grid.
# Route fire hazard composite files to the workspace reserved for this phase.
fire_hazard_composite_directory = fire_hazard_remaining_root / 'composite_fire_hazard'
# Group the terrain, historical-fire, human-ignition, composite, and metadata workspaces for
# creation and validation.
remaining_fire_hazard_directories = [fire_hazard_terrain_directory,
    fire_hazard_historical_directory, fire_hazard_human_directory,
    fire_hazard_composite_directory, fire_hazard_remaining_metadata]

# Prepare each required workspace before any phase outputs are written.
for directory in remaining_fire_hazard_directories:
    # Create the required workspace before writing cached sources, metadata, or raster products.
    directory.mkdir(parents=True, exist_ok=True)

    # Stop execution if the required file or directory is unavailable for this processing step.
    if not directory.exists():
        raise FileNotFoundError(f'''A required fire-hazard workspace could not be created:
{directory}''')
# Define the raster profile that keeps outputs aligned with the common project grid.
fire_hazard_component_profile = {'driver': 'GTiff',
    'dtype': 'float32', 'count': 1, 'width': fire_hazard_alignment_width,
    'height': fire_hazard_alignment_height, 'crs': fire_hazard_target_crs,
    'transform': fire_hazard_alignment_transform, 'nodata': fire_hazard_nodata_value,
    'compress': 'deflate', 'predictor': 3, 'tiled': True,
    'BIGTIFF': 'IF_SAFER'}

# Resolve a required cached source and stop if it is missing.
def require_fire_hazard_source(source_id):

    """
    Resolve one source path from the fire-hazard
    catalog and confirm that the cached source exists.

    Parameters
    ----------
    source_id : str
        Source identifier stored in
        fire_hazard_source_paths.

    Returns
    -------
    pathlib.Path
        Validated local path to the cached source.
    """

    # Handle the missing or unavailable case before downstream processing depends on the value.
    if source_id not in fire_hazard_source_paths:
        raise KeyError(f'Source ID is absent from the fire-hazard source catalog:\n{source_id}')
    # Route the source product to its designated project output or cache location.
    source_path = Path(fire_hazard_source_paths[source_id])

    # Stop execution if the required file or directory is unavailable for this processing step.
    if not source_path.exists():
        raise FileNotFoundError(f'''Required source is not available in the local cache:
Source ID: {source_id}
Expected path: {source_path}''')
    return source_path

# Write an aligned hazard raster through a temporary file before promotion.
def write_fire_hazard_array(array, output_path, description, tags=None):

    """
    Write a single-band Float32 array to the common
    fire-hazard grid using a temporary-file workflow.

    The temporary raster is promoted only after the
    write operation completes successfully.
    """
    # Route the output product to its designated project output or cache location.
    output_path = Path(output_path)
    # Route the temporary product to its designated project output or cache location.
    temporary_path = output_path.with_name(output_path.stem + '.part.tif')
    # Create the required workspace before writing cached sources, metadata, or raster products.
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Reuse the existing cached product when it is already available locally.
    if temporary_path.exists():
        temporary_path.unlink()
    # Define the raster profile that keeps outputs aligned with the common project grid.
    output_profile = fire_hazard_component_profile.copy()

    # Open the raster with a context manager so the dataset closes after this processing block.
    with rasterio.open(temporary_path, 'w', **output_profile) as destination:
        # Write the processed data to the configured output resource.
        destination.write(array.astype('float32'), 1)
        destination.set_band_description(1, description)

        # Select the processing branch required by the current cache, validation, or data state.
        if tags:
            destination.update_tags(**{key: str(value) for key, value in tags.items()})

    # Reuse the existing cached product when it is already available locally.
    if output_path.exists():
        output_path.unlink()
    temporary_path.replace(output_path)
    return output_path

# Process the raster in internal windows to control memory use
# while preserving the full-resolution output.
# Validate grid alignment, study-area coverage, and 0–1 component values.
def validate_normalized_component(path, minimum_valid_pixels=1):

    """
    Independently validate a normalized fire-hazard
    component raster.

    Validation includes:
    - file availability,
    - grid consistency,
    - raster structure,
    - valid-pixel coverage,
    - study-area masking,
    - and the expected zero-to-one score range.
    """
    # Normalize the component output to a Path object before file and raster validation.
    path = Path(path)
    # Initialize the component QA record with file, grid, coverage, and value-range checks.
    validation_result = {'PATH': str(path), 'FILE_EXISTS': path.exists(),
        'GRID_VALID': False, 'STRUCTURE_VALID': False,
        'VALID_PIXELS': 0, 'VALID_OUTSIDE_STUDY_AREA': 0,
        'MINIMUM': None, 'MAXIMUM': None, 'MEAN': None,
        'VALID': False}

    # Stop execution if the required file or directory is unavailable for this processing step.
    if not path.exists() or not path.is_file() or path.stat().st_size <= 0:
        return validation_result

    # Open the raster with a context manager so the dataset closes after this processing block.
    with rasterio.open(path) as source:
        # Update only the cells or records required by the current processing mask.
        validation_result['GRID_VALID'] = all([source.width == fire_hazard_alignment_width,
            source.height == fire_hazard_alignment_height,
            source.crs == rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
            source.transform.almost_equals(fire_hazard_alignment_transform)])
        # Update only the cells or records required by the current processing mask.
        validation_result['STRUCTURE_VALID'] = all([source.count == 1,
            source.dtypes[0] == 'float32', source.nodata == fire_hazard_nodata_value])
        # Track the valid pixel count metric used for QA and completeness checks.
        valid_pixel_count = 0
        # Track the valid outside study area count metric used for QA and completeness checks.
        valid_outside_study_area_count = 0
        # Track the valid value sum statistic used to validate raster values and report coverage.
        valid_value_sum = 0.0
        # Track the valid value minimum statistic used to validate raster values and report
        # coverage.
        valid_value_minimum = None
        # Track the valid value maximum statistic used to validate raster values and report
        # coverage.
        valid_value_maximum = None

        # Process raster windows sequentially to limit memory use on the full-resolution grid.
        for _, raster_window in source.block_windows(1):
            # Isolate raster values used to update coverage counts and normalized-value QA
            # statistics.
            values = source.read(1, window=raster_window)
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            row_start = int(raster_window.row_off)
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            row_end = row_start + int(raster_window.height)
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            column_start = int(raster_window.col_off)
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            column_end = column_start + int(raster_window.width)
            # Create the study area window mask used to restrict calculations to valid study-area
            # cells.
            study_area_window_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                column_start:column_end]
            # Create the valid value mask used to restrict calculations to valid study-area cells.
            valid_value_mask = np.isfinite(values) & (values != fire_hazard_nodata_value)
            # Isolate raster values used to update coverage counts and normalized-value QA
            # statistics.
            valid_values = values[valid_value_mask]
            # Track the valid pixel count metric used for QA and completeness checks.
            valid_pixel_count += int(valid_value_mask.sum())
            # Track the valid outside study area count metric used for QA and completeness checks.
            valid_outside_study_area_count += int((valid_value_mask & \
                ~study_area_window_mask).sum())

            # Select the processing branch required by the current cache, validation, or data state.
            if valid_values.size > 0:
                # Track the block minimum statistic used to validate raster values and report
                # coverage.
                block_minimum = float(valid_values.min())
                # Track the block maximum statistic used to validate raster values and report
                # coverage.
                block_maximum = float(valid_values.max())

                # Handle the missing or unavailable case before downstream processing depends on the
                # value.
                if valid_value_minimum is None:
                    # Track the valid value minimum statistic used to validate raster values and
                    # report coverage.
                    valid_value_minimum = block_minimum
                else:
                    # Track the valid value minimum statistic used to validate raster values and
                    # report coverage.
                    valid_value_minimum = min(valid_value_minimum, block_minimum)

                # Handle the missing or unavailable case before downstream processing depends on the
                # value.
                if valid_value_maximum is None:
                    # Track the valid value maximum statistic used to validate raster values and
                    # report coverage.
                    valid_value_maximum = block_maximum
                else:
                    # Track the valid value maximum statistic used to validate raster values and
                    # report coverage.
                    valid_value_maximum = max(valid_value_maximum, block_maximum)
                # Track the valid value sum statistic used to validate raster values and report
                # coverage.
                valid_value_sum += float(valid_values.sum(dtype=np.float64))
            del values
        validation_result.update({'VALID_PIXELS': valid_pixel_count,
            'VALID_OUTSIDE_STUDY_AREA': valid_outside_study_area_count,
            'MINIMUM': valid_value_minimum, 'MAXIMUM': valid_value_maximum,
            'MEAN': valid_value_sum / valid_pixel_count if valid_pixel_count > 0 else None})
        # Update only the cells or records required by the current processing mask.
        validation_result['VALID'] = all([validation_result['GRID_VALID'],
            validation_result['STRUCTURE_VALID'], valid_pixel_count >= minimum_valid_pixels,
            valid_outside_study_area_count == 0, valid_value_minimum is not None,
            valid_value_maximum is not None, valid_value_minimum >= -1e-06,
            valid_value_maximum <= 1.0 + 1e-06])
    return validation_result
# Record whether fire hazard remaining workflow has satisfied its completion requirement.
fire_hazard_remaining_workflow_configured = all([fire_hazard_fuel_component_finalized,
    fire_hazard_vegetation_dryness_component_finalized,
    all((directory.exists() for directory in remaining_fire_hazard_directories)),
    callable(require_fire_hazard_source), callable(write_fire_hazard_array),
    callable(validate_normalized_component)])

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_remaining_workflow_configured:
    raise ValueError('The remaining fire-hazard workflow did not pass final configuration checks.')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Terrain workspace: {fire_hazard_terrain_directory}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Historical-fire workspace: {fire_hazard_historical_directory}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Human-ignition workspace: {fire_hazard_human_directory}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Composite-fire-hazard workspace: {fire_hazard_composite_directory}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Shared metadata workspace: {fire_hazard_remaining_metadata}')
# Report workflow status so notebook execution can be verified interactively.
print('-> Shared normalized-component raster profile created.')
# Report workflow status so notebook execution can be verified interactively.
print('-> Cached-source resolver defined.')
# Report workflow status so notebook execution can be verified interactively.
print('-> Array-to-raster writer defined.')
# Report workflow status so notebook execution can be verified interactively.
print('-> Normalized-component validator defined.')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Remaining workflow configured: {fire_hazard_remaining_workflow_configured}')
gc.collect()
# Report workflow status so notebook execution can be verified interactively.
print('\nNOTE:')
# Report workflow status so notebook execution can be verified interactively.
print('The remaining terrain, '
    'historical-fire, '
    'human-ignition, and final '
    'composite workspaces are now '
    'configured.')
# Report workflow status so notebook execution can be verified interactively.
print('All subsequent normalized '
    'component rasters will use the '
    'same CRS, transform, '
    'dimensions, NoData value, and '
    'zero-to-one score scale.')
# Report workflow status so notebook execution can be verified interactively.
print('The shared helper functions '
    'will enforce cached-source '
    'availability, temporary-file '
    'promotion, and independent '
    'raster validation.')
# Report workflow status so notebook execution can be verified interactively.
print('\n=== REMAINING FIRE-HAZARD WORKFLOW CONFIGURED ===')


=== CONFIGURING REMAINING FIRE-HAZARD WORKFLOW ===
-> Terrain workspace: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\terrain_hazard
-> Historical-fire workspace: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\historical_fire_hazard
-> Human-ignition workspace: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\human_ignition_hazard
-> Composite-fire-hazard workspace: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\composite_fire_hazard
-> Shared metadata workspace: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata
-> Shared normalized-component raster profile created.
-> Cached-source resolver defined.
-> Array-to-raster writer defined.
-> Normalized-component validator defined.
-> Remaining workflow configured: True

NOTE:
The remaining terrain, historical-fire, human-ignition, and final composite workspaces are now configured.
All s

## Configure and Query USGS 3DEP DEM


In [119]:
print('=== CONFIGURING AND QUERYING USGS 3DEP DEM ===')
# Import request, timing, and raster-bound tools required to query the USGS elevation catalog.
from pathlib import Path
import requests
import time
from rasterio.warp import transform_bounds

# Acquire and align USGS 3DEP elevation data for deriving slope
# and aspect on the common project grid.
# List the prerequisite objects required before this workflow stage can run.
required_usgs_dem_query_objects = ['fire_hazard_remaining_workflow_configured',
    'fire_hazard_terrain_directory', 'fire_hazard_remaining_metadata',
    'fire_hazard_source_paths', 'fire_hazard_alignment_width',
    'fire_hazard_alignment_height', 'fire_hazard_alignment_transform',
    'fire_hazard_target_crs', 'fire_hazard_request_session',
    'fire_hazard_verify_ssl_certificates']
# Identify missing prerequisites so execution can stop before incomplete processing begins.
missing_usgs_dem_query_objects = [object_name for object_name in required_usgs_dem_query_objects \
    if object_name not in globals()]

# Stop execution if required upstream objects are missing from the sequential workflow.
if missing_usgs_dem_query_objects:
    raise NameError(f'''The following USGS DEM query objects are missing:
{missing_usgs_dem_query_objects}

Run Configure Remaining Fire-Hazard Workflow before querying the DEM catalog.''')

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_remaining_workflow_configured:
    raise ValueError('The remaining fire-hazard workflow has not been configured successfully.')
# Store fire hazard USGS DEM source ID so the acquired DEM can be registered consistently in the
# source catalog.
fire_hazard_usgs_dem_source_id = 'USGS_3DEP_DEM'
# Resolve fire hazard USGS DEM product name for readable source inventory and local cache naming.
fire_hazard_usgs_dem_product_name = 'USGS 3DEP 1 Arc-Second DEM'
# Define fire hazard usgs tnm products url used to access the configured source service.
fire_hazard_usgs_tnm_products_url = 'https://tnmaccess.nationalmap.gov/api/v1/products'
# List fire hazard USGS DEM dataset candidates in preferred order for selecting a compatible 3DEP
# elevation product.
fire_hazard_usgs_dem_dataset_candidates = [
    '1 arc-second DEM',
    'Digital Elevation Model (DEM) 1 arc-second',
    'National Elevation Dataset (NED) 1 arc-second',
    '3DEP 1 arc-second DEM'
]
# Restrict the source search to the fire hazard USGS DEM requested format accepted by the
# raster-processing workflow.
fire_hazard_usgs_dem_requested_format = 'GeoTIFF'
# Configure fire hazard USGS DEM query page size to control catalog paging or source-file
# validation.
fire_hazard_usgs_dem_query_page_size = 100
# Configure fire hazard USGS DEM query timeout seconds to control request reliability and retry
# behavior.
fire_hazard_usgs_dem_query_timeout_seconds = 120
# Limit catalog retries so temporary service failures are retried without creating an endless
# request loop.
fire_hazard_usgs_dem_query_max_attempts = 4
# Configure fire hazard USGS DEM query retry seconds to control request reliability and retry
# behavior.
fire_hazard_usgs_dem_query_retry_seconds = 5
# Cap exponential backoff so transient TNM service failures are retried without creating
# excessively long pauses.
fire_hazard_usgs_dem_query_retry_max_seconds = 40
# Route fire hazard USGS DEM tile files to the workspace reserved for this phase.
fire_hazard_usgs_dem_tile_directory = Path(fire_hazard_terrain_directory) / \
    'usgs_3dep_1arcsec_dem_tiles'
# Route fire hazard USGS DEM metadata files to the workspace reserved for this phase.
fire_hazard_usgs_dem_metadata_directory = Path(fire_hazard_terrain_directory) / 'metadata'
# Route the fire hazard USGS DEM aligned mosaic product to its designated project output or cache
# location.
fire_hazard_usgs_dem_aligned_mosaic_path = Path(fire_hazard_terrain_directory) / \
    'usgs_3dep_dem_aligned_30m.tif'
# Persist fire hazard USGS DEM query inventory metadata at the project location used for QA and
# reproducibility.
fire_hazard_usgs_dem_query_inventory_path = fire_hazard_usgs_dem_metadata_directory / \
    'usgs_3dep_dem_query_inventory.csv'
# Persist fire hazard USGS DEM query summary metadata at the project location used for QA and
# reproducibility.
fire_hazard_usgs_dem_query_summary_path = fire_hazard_usgs_dem_metadata_directory / \
    'usgs_3dep_dem_query_summary.csv'
# Persist fire hazard USGS DEM query response metadata at the project location used for QA and
# reproducibility.
fire_hazard_usgs_dem_query_response_path = fire_hazard_usgs_dem_metadata_directory / \
    'usgs_3dep_dem_query_response.json'

# Prepare each required workspace before any phase outputs are written.
for directory in [fire_hazard_usgs_dem_tile_directory, fire_hazard_usgs_dem_metadata_directory]:
    # Create the required workspace before writing cached sources, metadata, or raster products.
    directory.mkdir(parents=True, exist_ok=True)
# Calculate a project-grid edge coordinate used to construct the DEM search extent.
fire_hazard_grid_left = float(fire_hazard_alignment_transform.c)
# Calculate a project-grid edge coordinate used to construct the DEM search extent.
fire_hazard_grid_top = float(fire_hazard_alignment_transform.f)
# Calculate a project-grid edge coordinate used to construct the DEM search extent.
fire_hazard_grid_right = float(fire_hazard_grid_left + fire_hazard_alignment_width * \
    fire_hazard_alignment_transform.a)
# Calculate a project-grid edge coordinate used to construct the DEM search extent.
fire_hazard_grid_bottom = float(fire_hazard_grid_top + fire_hazard_alignment_height * \
    fire_hazard_alignment_transform.e)
# Store the fire hazard project extent used to constrain the elevation search or validation.
fire_hazard_project_bounds = (min(fire_hazard_grid_left,
    fire_hazard_grid_right), min(fire_hazard_grid_bottom,
    fire_hazard_grid_top), max(fire_hazard_grid_left,
    fire_hazard_grid_right), max(fire_hazard_grid_bottom,
    fire_hazard_grid_top))
# Store the fire hazard USGS DEM query extent used to constrain the elevation search or validation.
fire_hazard_usgs_dem_query_bounds = transform_bounds(fire_hazard_target_crs,
    'EPSG:4326', *fire_hazard_project_bounds, densify_pts=21)
# Unpack the transformed project extent into the west, south, east, and north values used by the
# API.
fire_hazard_usgs_dem_west, fire_hazard_usgs_dem_south, fire_hazard_usgs_dem_east, \
    fire_hazard_usgs_dem_north = fire_hazard_usgs_dem_query_bounds
# Format the transformed project extent as the bounding-box string required by the USGS product
# query.
fire_hazard_usgs_dem_bbox_string = (
    f'{fire_hazard_usgs_dem_west:.8f},'
        f'{fire_hazard_usgs_dem_south:.8f},'
        f'{fire_hazard_usgs_dem_east:.8f},'
        f'{fire_hazard_usgs_dem_north:.8f}'
)

# Encapsulate the reusable extract USGS DEM items workflow logic.
def extract_usgs_dem_items(response_json):

    """
    Extract the TNM product list while supporting
    common TNMAccess response structures.
    """

    # Inspect each source record so unsuitable or non-overlapping elevation products can be
    # excluded.
    for key_name in ['items', 'products', 'results']:
        # Extract candidate items from the service response for product selection and paging.
        candidate_items = response_json.get(key_name)

        # Select the processing branch required by the current cache, validation, or data state.
        if isinstance(candidate_items, list):
            return candidate_items
    return []

# Encapsulate the reusable extract USGS DEM total workflow logic.
def extract_usgs_dem_total(response_json, fallback_total):

    """
    Extract the reported product total while falling
    back to the current result length.
    """

    # Process each collection item under the same acquisition, validation, or mosaic criteria.
    for key_name in ['total', 'totalItems', 'totalProducts', 'count']:
        # Track candidate total so paging can stop after all available records are retrieved.
        candidate_total = response_json.get(key_name)

        # Select the processing branch required by the current cache, validation, or data state.
        if candidate_total is not None:

            # Attempt the operation so expected data, file, or service failures can be handled
            # explicitly.
            try:
                return int(candidate_total)
            # Handle the expected failure without leaving the workflow in an inconsistent state.
            except (TypeError, ValueError):
                pass
    return int(fallback_total)

# Encapsulate the reusable extract USGS DEM download URL workflow logic.
def extract_usgs_dem_download_url(product_record):

    """
    Resolve the downloadable GeoTIFF URL from a TNM
    product record.
    """

    # Process each collection item under the same acquisition, validation, or mosaic criteria.
    for key_name in ['downloadURL', 'downloadUrl', 'download_url', 'url']:
        # Define candidate url used to access the configured source service.
        candidate_url = product_record.get(key_name)

        # Select the processing branch required by the current cache, validation, or data state.
        if isinstance(candidate_url, str) and candidate_url.strip():
            return candidate_url.strip()
    return None
# Track the first preferred 3DEP dataset that returns usable downloadable elevation records.
fire_hazard_usgs_dem_selected_dataset = None
# Collect fire hazard USGS DEM query records so the selected source set can be validated and
# documented.
fire_hazard_usgs_dem_query_records = []
# Initialize or capture the fire hazard USGS DEM query error message used to diagnose a failed
# request or file operation.
fire_hazard_usgs_dem_query_error = None
# Preserve the fire hazard USGS DEM last response so the source query can be audited and reproduced.
fire_hazard_usgs_dem_last_response = None

# Process each collection item under the same acquisition, validation, or mosaic criteria.
for dataset_name in fire_hazard_usgs_dem_dataset_candidates:
    # Collect current dataset records so the selected source set can be validated and documented.
    current_dataset_records = []
    # Track current offset so the next catalog request starts at the correct page position.
    current_offset = 0
    # Track current total so paging can stop after all available records are retrieved.
    current_total = None
    # Report workflow status so notebook execution can be verified interactively.
    print(f'-> Querying TNM dataset: {dataset_name}')

    # Repeat the operation while the workflow condition remains active.
    while current_total is None or current_offset < current_total:
        # Define query parameters used to configure this workflow stage.
        query_parameters = {'datasets': dataset_name,
            'bbox': fire_hazard_usgs_dem_bbox_string, 'prodFormats': \
                fire_hazard_usgs_dem_requested_format,
            'max': fire_hazard_usgs_dem_query_page_size, 'offset': current_offset}
        # Track request success so retry logic can distinguish a completed page from a failed
        # attempt.
        query_successful = False

        # Process each catalog page using both the shared retry-enabled HTTP session and a
        # bounded outer retry loop. The outer loop protects against transient gateway/service
        # failures that remain after the session-level retry policy is exhausted.
        for attempt_number in range(1, fire_hazard_usgs_dem_query_max_attempts + 1):

            query_response = None

            try:
                # Query TNMAccess through the reusable project session so catalog requests use the
                # same connection pooling, retry, timeout, and SSL policy as other live sources.
                query_response = fire_hazard_request_session.get(
                    fire_hazard_usgs_tnm_products_url,
                    params=query_parameters,
                    timeout=fire_hazard_usgs_dem_query_timeout_seconds,
                    verify=fire_hazard_verify_ssl_certificates
                )

                query_response.raise_for_status()

                # Preserve the response JSON so the source query can be audited and reproduced.
                response_json = query_response.json()

                # Treat API-level error objects as failed catalog requests even when HTTP status is
                # successful.
                if 'error' in response_json:
                    raise RuntimeError(
                        f"TNMAccess returned an API error:\n"
                        f"{response_json['error']}"
                    )

                # Preserve the last successful response for provenance.
                fire_hazard_usgs_dem_last_response = response_json

                query_successful = True
                break

            except Exception as error:
                fire_hazard_usgs_dem_query_error = str(error)

                if attempt_number < fire_hazard_usgs_dem_query_max_attempts:
                    # Apply bounded exponential backoff between complete catalog-query attempts.
                    retry_delay_seconds = min(
                        fire_hazard_usgs_dem_query_retry_seconds
                        * (2 ** (attempt_number - 1)),
                        fire_hazard_usgs_dem_query_retry_max_seconds
                    )

                    print(
                        f'   Query attempt {attempt_number} failed; '
                        f'retrying in {retry_delay_seconds:g} seconds...'
                    )

                    time.sleep(retry_delay_seconds)

            finally:
                # Release the response after every attempt so persistent sessions do not retain
                # unnecessary network resources.
                if query_response is not None:
                    try:
                        query_response.close()
                    except Exception:
                        pass

        # Select the processing branch required by the current cache, validation, or data state.
        if not query_successful:
            break
        # Extract page items from the service response for product selection and paging.
        page_items = extract_usgs_dem_items(response_json)
        # Track current total so paging can stop after all available records are retrieved.
        current_total = extract_usgs_dem_total(response_json, len(page_items))
        current_dataset_records.extend(page_items)

        # Select the processing branch required by the current cache, validation, or data state.
        if len(page_items) == 0:
            break
        # Track current offset so the next catalog request starts at the correct page position.
        current_offset += len(page_items)

        # Select the processing branch required by the current cache, validation, or data state.
        if len(page_items) < fire_hazard_usgs_dem_query_page_size:
            break
    # Collect downloadable records so the selected source set can be validated and documented.
    downloadable_records = [record for record in current_dataset_records if \
        extract_usgs_dem_download_url(record) is not None]

    # Select the processing branch required by the current cache, validation, or data state.
    if downloadable_records:
        # Track the first preferred 3DEP dataset that returns usable downloadable elevation records.
        fire_hazard_usgs_dem_selected_dataset = dataset_name
        # Collect fire hazard USGS DEM query records so the selected source set can be validated and
        # documented.
        fire_hazard_usgs_dem_query_records = downloadable_records
        break

# Select the processing branch required by the current cache, validation, or data state.
if not fire_hazard_usgs_dem_query_records:
    raise ValueError(f'''The TNMAccess query returned no downloadable 1 arc-second DEM products.

Query bounds: {fire_hazard_usgs_dem_bbox_string}
Datasets attempted: {fire_hazard_usgs_dem_dataset_candidates}
Last error: {fire_hazard_usgs_dem_query_error}

The query used the retry-enabled project HTTP session plus bounded exponential
backoff. If TNMAccess is temporarily unavailable, rerun this query cell later
without changing the project bounds.''')
# Assemble fire hazard USGS DEM inventory records metadata for QA reporting and later workflow
# phases.
fire_hazard_usgs_dem_inventory_records = []
# Define seen usgs dem download urls used to access the configured source service.
seen_usgs_dem_download_urls = set()

# Inspect each source record so unsuitable or non-overlapping elevation products can be excluded.
for product_number, product_record in enumerate(fire_hazard_usgs_dem_query_records, start=1):
    # Define download url used to access the configured source service.
    download_url = extract_usgs_dem_download_url(product_record)

    # Select the processing branch required by the current cache, validation, or data state.
    if download_url in seen_usgs_dem_download_urls:
        continue
    seen_usgs_dem_download_urls.add(download_url)
    # Resolve product title for readable source inventory and local cache naming.
    product_title = product_record.get('title') or product_record.get('name') or \
        f'USGS_3DEP_DEM_{product_number:03d}'
    # Resolve product filename for readable source inventory and local cache naming.
    product_filename = product_record.get('filename') or Path(download_url.split('?')[0]).name

    # Select the processing branch required by the current cache, validation, or data state.
    if not product_filename:
        # Resolve product filename for readable source inventory and local cache naming.
        product_filename = f'usgs_3dep_dem_{product_number:03d}.tif'
    # Route the local tile product to its designated project output or cache location.
    local_tile_path = fire_hazard_usgs_dem_tile_directory / product_filename
    fire_hazard_usgs_dem_inventory_records.append({'PRODUCT_NUMBER': product_number,
        'DATASET': fire_hazard_usgs_dem_selected_dataset,
        'TITLE': product_title, 'PUBLICATION_DATE': product_record.get('publicationDate') or \
            product_record.get('publication_date'),
        'FORMAT': product_record.get('format') or product_record.get('prodFormat') or \
            fire_hazard_usgs_dem_requested_format,
        'DOWNLOAD_URL': download_url, 'LOCAL_TILE_PATH': str(local_tile_path),
        'FILE_SIZE_BYTES_REPORTED': product_record.get('sizeInBytes') or \
            product_record.get('nbytes')})
# Assemble fire hazard USGS DEM query inventory metadata for QA reporting and later workflow phases.
fire_hazard_usgs_dem_query_inventory = \
    pd.DataFrame(fire_hazard_usgs_dem_inventory_records).drop_duplicates(subset=['DOWNLOAD_URL']) \
    .reset_index(drop=True)
# Track the fire hazard USGS DEM expected tile count metric used for QA and completeness checks.
fire_hazard_usgs_dem_expected_tile_count = len(fire_hazard_usgs_dem_query_inventory)
# Record whether fire hazard USGS DEM query has satisfied its completion requirement.
fire_hazard_usgs_dem_query_complete = all([fire_hazard_usgs_dem_selected_dataset is not None,
    fire_hazard_usgs_dem_expected_tile_count > 0, \
        fire_hazard_usgs_dem_query_inventory['DOWNLOAD_URL'].notna().all(),
    fire_hazard_usgs_dem_query_inventory['LOCAL_TILE_PATH'].notna().all()])

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_usgs_dem_query_complete:
    raise ValueError('The USGS 3DEP DEM query inventory failed validation.')
# Assemble fire hazard USGS DEM query summary metadata for QA reporting and later workflow phases.
fire_hazard_usgs_dem_query_summary = pd.DataFrame([{'SOURCE_ID': fire_hazard_usgs_dem_source_id,
    'PRODUCT_NAME': fire_hazard_usgs_dem_product_name,
    'SELECTED_DATASET': fire_hazard_usgs_dem_selected_dataset,
    'QUERY_BBOX': fire_hazard_usgs_dem_bbox_string,
    'WEST': fire_hazard_usgs_dem_west, 'SOUTH': fire_hazard_usgs_dem_south,
    'EAST': fire_hazard_usgs_dem_east, 'NORTH': fire_hazard_usgs_dem_north,
    'EXPECTED_TILE_COUNT': fire_hazard_usgs_dem_expected_tile_count,
    'TILE_DIRECTORY': str(fire_hazard_usgs_dem_tile_directory),
    'ALIGNED_MOSAIC_PATH': str(fire_hazard_usgs_dem_aligned_mosaic_path),
    'QUERY_COMPLETE': fire_hazard_usgs_dem_query_complete}])
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_usgs_dem_query_inventory.to_csv(fire_hazard_usgs_dem_query_inventory_path, index=False)
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_usgs_dem_query_summary.to_csv(fire_hazard_usgs_dem_query_summary_path, index=False)

# Select the processing branch required by the current cache, validation, or data state.
if fire_hazard_usgs_dem_last_response is not None:

    # Open the metadata file with a context manager so it closes cleanly after writing.
    with fire_hazard_usgs_dem_query_response_path.open('w', encoding='utf-8') as response_file:
        # Write the completion metadata used to document and verify the cached DEM source.
        json.dump(fire_hazard_usgs_dem_last_response, response_file, indent=2)
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Selected TNM dataset: {fire_hazard_usgs_dem_selected_dataset}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Geographic query bounds: {fire_hazard_usgs_dem_bbox_string}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> DEM products found: {fire_hazard_usgs_dem_expected_tile_count:,}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Tile cache directory: {fire_hazard_usgs_dem_tile_directory}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Query inventory saved: {fire_hazard_usgs_dem_query_inventory_path}')
# Report workflow status so notebook execution can be verified interactively.
print('\n--- USGS 3DEP DEM QUERY SUMMARY ---')
display(fire_hazard_usgs_dem_query_summary)
# Report workflow status so notebook execution can be verified interactively.
print('\n--- USGS 3DEP DEM TILE INVENTORY SAMPLE ---')
display(fire_hazard_usgs_dem_query_inventory.head(5))
# Report workflow status so notebook execution can be verified interactively.
print('\nNOTE:')
# Report workflow status so notebook execution can be verified interactively.
print('The TNMAccess catalog was queried using the geographic extent of the common project grid.')
print('Catalog requests use the project retry-enabled HTTP session with bounded exponential backoff.')
# Report workflow status so notebook execution can be verified interactively.
print('No DEM files were downloaded in this query step.')
# Report workflow status so notebook execution can be verified interactively.
print('\n=== USGS 3DEP DEM QUERY COMPLETE ===')


=== CONFIGURING AND QUERYING USGS 3DEP DEM ===
-> Querying TNM dataset: 1 arc-second DEM
-> Querying TNM dataset: Digital Elevation Model (DEM) 1 arc-second
-> Querying TNM dataset: National Elevation Dataset (NED) 1 arc-second
-> Selected TNM dataset: National Elevation Dataset (NED) 1 arc-second
-> Geographic query bounds: -114.22140159,36.97630148,-110.85521951,40.92652211
-> DEM products found: 148
-> Tile cache directory: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\terrain_hazard\usgs_3dep_1arcsec_dem_tiles
-> Query inventory saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\terrain_hazard\metadata\usgs_3dep_dem_query_inventory.csv

--- USGS 3DEP DEM QUERY SUMMARY ---


,SOURCE_ID,PRODUCT_NAME,SELECTED_DATASET,QUERY_BBOX,WEST,SOUTH,EAST,NORTH,EXPECTED_TILE_COUNT,TILE_DIRECTORY,ALIGNED_MOSAIC_PATH,QUERY_COMPLETE
0,USGS_3DEP_DEM,USGS 3DEP 1 Arc-Second DEM,National Elevation Dataset (NED) 1 arc-second,"-114.22140159,36.97630148,-110.85521951,40.926...",-114.221402,36.976301,-110.85522,40.926522,148,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True



--- USGS 3DEP DEM TILE INVENTORY SAMPLE ---


,PRODUCT_NUMBER,DATASET,TITLE,PUBLICATION_DATE,FORMAT,DOWNLOAD_URL,LOCAL_TILE_PATH,FILE_SIZE_BYTES_REPORTED
0,1,National Elevation Dataset (NED) 1 arc-second,USGS 1 Arc Second n37w111 20210611,2021-11-16,GeoTIFF,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,51467116
1,2,National Elevation Dataset (NED) 1 arc-second,USGS 1 Arc Second n37w111 20211215,2021-12-15,GeoTIFF,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,55835154
2,3,National Elevation Dataset (NED) 1 arc-second,USGS 1 Arc Second n37w111 20240606,2024-06-06,GeoTIFF,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,51420837
3,4,National Elevation Dataset (NED) 1 arc-second,USGS 1 Arc Second n37w111 20241031,2024-10-31,GeoTIFF,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,51421242
4,5,National Elevation Dataset (NED) 1 arc-second,USGS 1 Arc Second n37w111 20260514,2026-05-14,GeoTIFF,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,51887305



NOTE:
The TNMAccess catalog was queried using the geographic extent of the common project grid.
Catalog requests use the project retry-enabled HTTP session with bounded exponential backoff.
No DEM files were downloaded in this query step.

=== USGS 3DEP DEM QUERY COMPLETE ===


## Download and Validate USGS 3DEP DEM Tiles


In [120]:
print('=== DOWNLOADING AND VALIDATING USGS 3DEP DEM TILES ===')
# List the prerequisite objects required before this workflow stage can run.
required_usgs_dem_download_objects = ['fire_hazard_usgs_dem_query_complete',
    'fire_hazard_usgs_dem_query_inventory', 'fire_hazard_usgs_dem_expected_tile_count',
    'fire_hazard_usgs_dem_tile_directory', 'fire_hazard_usgs_dem_metadata_directory']
# Identify missing prerequisites so execution can stop before incomplete processing begins.
missing_usgs_dem_download_objects = [object_name for object_name in \
    required_usgs_dem_download_objects if object_name not in globals()]

# Stop execution if required upstream objects are missing from the sequential workflow.
if missing_usgs_dem_download_objects:
    raise NameError(f'''The following USGS DEM download objects are missing:
{missing_usgs_dem_download_objects}

Run Configure and Query USGS 3DEP DEM before downloading the source tiles.''')

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_usgs_dem_query_complete:
    raise ValueError('The USGS 3DEP DEM query did not complete successfully.')
# Configure fire hazard USGS DEM download timeout seconds to control request reliability and retry
# behavior.
fire_hazard_usgs_dem_download_timeout_seconds = 300
# Configure fire hazard USGS DEM download chunk bytes to support efficient streaming or minimum-file
# validation.
fire_hazard_usgs_dem_download_chunk_bytes = 1024 * 1024
# Limit tile-download retries so transient network failures can recover without looping
# indefinitely.
fire_hazard_usgs_dem_download_max_attempts = 3
# Configure fire hazard USGS DEM download retry seconds to control request reliability and retry
# behavior.
fire_hazard_usgs_dem_download_retry_seconds = 5
# Configure fire hazard USGS DEM minimum tile size bytes to support efficient streaming or
# minimum-file validation.
fire_hazard_usgs_dem_minimum_tile_size_bytes = 1024
# Set the byte interval used to report progress during streamed DEM tile downloads.
fire_hazard_usgs_dem_progress_interval = 25
# Collect fire hazard USGS DEM download records so the selected source set can be validated and
# documented.
fire_hazard_usgs_dem_download_records = []

# Inspect each source record so unsuitable or non-overlapping elevation products can be excluded.
for inventory_index, inventory_record in fire_hazard_usgs_dem_query_inventory.iterrows():
    # Convert the inventory index to a one-based tile number for readable progress and QA reporting.
    tile_number = inventory_index + 1
    # Define download url used to access the configured source service.
    download_url = str(inventory_record['DOWNLOAD_URL'])
    # Route the local tile product to its designated project output or cache location.
    local_tile_path = Path(inventory_record['LOCAL_TILE_PATH'])
    # Route the temporary tile product to its designated project output or cache location.
    temporary_tile_path = local_tile_path.with_name(local_tile_path.name + '.part')

    # Select the processing branch required by the current cache, validation, or data state.
    if tile_number == 1 or tile_number % fire_hazard_usgs_dem_progress_interval == 0 or \
        tile_number == fire_hazard_usgs_dem_expected_tile_count:
        # Report workflow status so notebook execution can be verified interactively.
        print(f'-> Processing DEM tile '
            f'{tile_number:,} of '
            f'{fire_hazard_usgs_dem_expected_tile_count:,}...')
    # Create the required workspace before writing cached sources, metadata, or raster products.
    local_tile_path.parent.mkdir(parents=True, exist_ok=True)
    # Record whether the existing tile satisfies the workflow QA requirements.
    existing_tile_valid = False

    # Reuse the existing cached product when it is already available locally.
    if local_tile_path.exists() and local_tile_path.is_file() and (local_tile_path.stat().st_size \
        >= fire_hazard_usgs_dem_minimum_tile_size_bytes):

        # Attempt the operation so expected data, file, or service failures can be handled
        # explicitly.
        try:

            # Open the raster with a context manager so the dataset closes after this processing
            # block.
            with rasterio.open(local_tile_path) as existing_tile_source:
                # Record whether the existing tile satisfies the workflow QA requirements.
                existing_tile_valid = all([existing_tile_source.count >= 1,
                    existing_tile_source.width > 0, existing_tile_source.height > 0,
                    existing_tile_source.crs is not None])
        # Handle the expected failure without leaving the workflow in an inconsistent state.
        except Exception:
            # Record whether the existing tile satisfies the workflow QA requirements.
            existing_tile_valid = False

    # Select the processing branch required by the current cache, validation, or data state.
    if existing_tile_valid:
        # Record download action so the download summary distinguishes cached and newly retrieved
        # tiles.
        download_action = 'Reused cached tile'
        # Initialize or capture the download error message used to diagnose a failed request or file
        # operation.
        download_error = None
    else:
        # Record download action so the download summary distinguishes cached and newly retrieved
        # tiles.
        download_action = 'Downloaded tile'
        # Initialize or capture the download error message used to diagnose a failed request or file
        # operation.
        download_error = None

        # Reuse the existing cached product when it is already available locally.
        if temporary_tile_path.exists():
            temporary_tile_path.unlink()
        # Record whether download has satisfied its completion requirement.
        download_complete = False

        # Process each collection item under the same acquisition, validation, or mosaic criteria.
        for attempt_number in range(1, fire_hazard_usgs_dem_download_max_attempts + 1):

            # Attempt the operation so expected data, file, or service failures can be handled
            # explicitly.
            try:

                # Manage temporary resources with a context manager so they are released after
                # processing.
                with requests.get(download_url,
                    stream=True, timeout=fire_hazard_usgs_dem_download_timeout_seconds) as \
                        download_response:
                    download_response.raise_for_status()

                    # Open the metadata file with a context manager so it closes cleanly after
                    # writing.
                    with temporary_tile_path.open('wb') as tile_file:

                        # Process each collection item under the same acquisition, validation, or
                        # mosaic criteria.
                        for data_chunk in \
                            download_response.iter_content(chunk_size= \
                            fire_hazard_usgs_dem_download_chunk_bytes):

                            # Select the processing branch required by the current cache,
                            # validation, or data state.
                            if data_chunk:
                                # Write the processed data to the configured output resource.
                                tile_file.write(data_chunk)

                # Stop execution if the required file or directory is unavailable for this
                # processing step.
                if not temporary_tile_path.exists() or temporary_tile_path.stat().st_size < \
                    fire_hazard_usgs_dem_minimum_tile_size_bytes:
                    raise ValueError('The downloaded tile is missing '
                        'or smaller than the minimum '
                        'expected file size.')

                # Open the raster with a context manager so the dataset closes after this processing
                # block.
                with rasterio.open(temporary_tile_path) as downloaded_tile_source:
                    # Record whether the downloaded tile satisfies the workflow QA requirements.
                    downloaded_tile_valid = all([downloaded_tile_source.count >= 1,
                        downloaded_tile_source.width > 0, downloaded_tile_source.height > 0,
                        downloaded_tile_source.crs is not None])

                # Stop execution if the preceding QA requirement has not been satisfied.
                if not downloaded_tile_valid:
                    raise ValueError('The downloaded DEM tile failed basic raster validation.')

                # Reuse the existing cached product when it is already available locally.
                if local_tile_path.exists():
                    local_tile_path.unlink()
                temporary_tile_path.replace(local_tile_path)
                # Record whether download has satisfied its completion requirement.
                download_complete = True
                break
            # Handle the expected failure without leaving the workflow in an inconsistent state.
            except Exception as error:
                # Initialize or capture the download error message used to diagnose a failed request
                # or file operation.
                download_error = str(error)

                # Reuse the existing cached product when it is already available locally.
                if temporary_tile_path.exists():
                    temporary_tile_path.unlink()

                # Select the processing branch required by the current cache, validation, or data
                # state.
                if attempt_number < fire_hazard_usgs_dem_download_max_attempts:
                    # Report workflow status so notebook execution can be verified interactively.
                    print(f'   Tile {tile_number:,} '
                        f'download attempt '
                        f'{attempt_number} failed; '
                        f'retrying...')
                    # Pause before retrying so temporary service failures do not trigger immediate
                    # repeated requests.
                    time.sleep(fire_hazard_usgs_dem_download_retry_seconds)

        # Stop execution if the preceding QA requirement has not been satisfied.
        if not download_complete:
            raise RuntimeError(f'''A required USGS 3DEP DEM tile could not be downloaded.

Tile number: {tile_number:,}
Tile: {local_tile_path.name}
URL: {download_url}
Error: {download_error}''')

    # Open the raster with a context manager so the dataset closes after this processing block.
    with rasterio.open(local_tile_path) as tile_source:
        # Define tile record used to configure this workflow stage.
        tile_record = {'PRODUCT_NUMBER': tile_number,
            'TITLE': inventory_record['TITLE'], 'DOWNLOAD_URL': download_url,
            'LOCAL_TILE_PATH': str(local_tile_path), 'DOWNLOAD_ACTION': download_action,
            'FILE_SIZE_BYTES': int(local_tile_path.stat().st_size),
            'CRS': tile_source.crs.to_string() if tile_source.crs is not None else None,
            'WIDTH': tile_source.width, 'HEIGHT': tile_source.height,
            'BAND_COUNT': tile_source.count, 'DTYPE': tile_source.dtypes[0],
            'NODATA': tile_source.nodata, 'LEFT': tile_source.bounds.left,
            'BOTTOM': tile_source.bounds.bottom, 'RIGHT': tile_source.bounds.right,
            'TOP': tile_source.bounds.top, 'READABLE': True,
            'VALID': True, 'ERROR_MESSAGE': download_error}
    fire_hazard_usgs_dem_download_records.append(tile_record)
# Record whether the fire hazard USGS DEM download satisfies the workflow QA requirements.
fire_hazard_usgs_dem_download_validation = \
    pd.DataFrame(fire_hazard_usgs_dem_download_records).sort_values('PRODUCT_NUMBER').reset_index \
    (drop=True)
# Track the fire hazard USGS DEM downloaded tile count metric used for QA and completeness checks.
fire_hazard_usgs_dem_downloaded_tile_count = len(fire_hazard_usgs_dem_download_validation)
# Track the fire hazard USGS DEM valid tile count metric used for QA and completeness checks.
fire_hazard_usgs_dem_valid_tile_count = int(fire_hazard_usgs_dem_download_validation['VALID'].sum())
# Record whether fire hazard USGS DEM tile download has satisfied its completion requirement.
fire_hazard_usgs_dem_tile_download_complete = all([fire_hazard_usgs_dem_downloaded_tile_count == \
    fire_hazard_usgs_dem_expected_tile_count,
    fire_hazard_usgs_dem_valid_tile_count == fire_hazard_usgs_dem_expected_tile_count,
    fire_hazard_usgs_dem_download_validation['LOCAL_TILE_PATH'].map(lambda path_value: \
        Path(path_value).exists()).all()])
# Route the fire hazard USGS DEM download validation product to its designated project output or
# cache location.
fire_hazard_usgs_dem_download_validation_path = fire_hazard_usgs_dem_metadata_directory / \
    'usgs_3dep_dem_download_validation.csv'
# Persist fire hazard USGS DEM download summary metadata at the project location used for QA and
# reproducibility.
fire_hazard_usgs_dem_download_summary_path = fire_hazard_usgs_dem_metadata_directory / \
    'usgs_3dep_dem_download_summary.csv'
# Assemble fire hazard USGS DEM download summary metadata for QA reporting and later workflow
# phases.
fire_hazard_usgs_dem_download_summary = pd.DataFrame([{'EXPECTED_TILES': \
    fire_hazard_usgs_dem_expected_tile_count,
    'DOWNLOADED_OR_REUSED_TILES': fire_hazard_usgs_dem_downloaded_tile_count,
    'VALID_TILES': fire_hazard_usgs_dem_valid_tile_count,
    'TOTAL_SIZE_BYTES': int(fire_hazard_usgs_dem_download_validation['FILE_SIZE_BYTES'].sum()),
    'TOTAL_SIZE_MB': fire_hazard_usgs_dem_download_validation['FILE_SIZE_BYTES'].sum() / 1024 ** 2,
    'DOWNLOAD_COMPLETE': fire_hazard_usgs_dem_tile_download_complete}])
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_usgs_dem_download_validation.to_csv(fire_hazard_usgs_dem_download_validation_path,
    index=False)
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_usgs_dem_download_summary.to_csv(fire_hazard_usgs_dem_download_summary_path,
    index=False)

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_usgs_dem_tile_download_complete:
    raise ValueError(f'''The USGS 3DEP DEM tile cache is incomplete.
Expected tiles: {fire_hazard_usgs_dem_expected_tile_count:,}
Valid tiles: {fire_hazard_usgs_dem_valid_tile_count:,}''')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Expected tiles: {fire_hazard_usgs_dem_expected_tile_count:,}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Valid cached tiles: {fire_hazard_usgs_dem_valid_tile_count:,}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Tile cache complete: {fire_hazard_usgs_dem_tile_download_complete}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Download validation saved: {fire_hazard_usgs_dem_download_validation_path}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Download summary saved: {fire_hazard_usgs_dem_download_summary_path}')
# Report workflow status so notebook execution can be verified interactively.
print('\n--- USGS 3DEP DEM DOWNLOAD SUMMARY ---')
display(fire_hazard_usgs_dem_download_summary)
# Report workflow status so notebook execution can be verified interactively.
print('\n--- USGS 3DEP DEM TILE VALIDATION SAMPLE ---')
display(fire_hazard_usgs_dem_download_validation.head(5))
gc.collect()
# Report workflow status so notebook execution can be verified interactively.
print('\nNOTE:')
# Report workflow status so notebook execution can be verified interactively.
print('Every expected USGS 3DEP DEM '
    'tile was either downloaded or '
    'reused from the local cache '
    'and then validated as a '
    'readable georeferenced raster.')
# Report workflow status so notebook execution can be verified interactively.
print('Progress messages were limited '
    'to the first tile, every 25th '
    'tile, and the final tile to '
    'reduce notebook output.')
# Report workflow status so notebook execution can be verified interactively.
print('The validated tile cache is ready for creation of the aligned project-grid DEM mosaic.')
# Report workflow status so notebook execution can be verified interactively.
print('\n=== USGS 3DEP DEM TILE DOWNLOAD AND VALIDATION COMPLETE ===')

# Report workflow status so notebook execution can be verified interactively.


=== DOWNLOADING AND VALIDATING USGS 3DEP DEM TILES ===
-> Processing DEM tile 1 of 148...
-> Processing DEM tile 25 of 148...
-> Processing DEM tile 50 of 148...
-> Processing DEM tile 75 of 148...
-> Processing DEM tile 100 of 148...
-> Processing DEM tile 125 of 148...
-> Processing DEM tile 148 of 148...
-> Expected tiles: 148
-> Valid cached tiles: 148
-> Tile cache complete: True
-> Download validation saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\terrain_hazard\metadata\usgs_3dep_dem_download_validation.csv
-> Download summary saved: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\terrain_hazard\metadata\usgs_3dep_dem_download_summary.csv

--- USGS 3DEP DEM DOWNLOAD SUMMARY ---


,EXPECTED_TILES,DOWNLOADED_OR_REUSED_TILES,VALID_TILES,TOTAL_SIZE_BYTES,TOTAL_SIZE_MB,DOWNLOAD_COMPLETE
0,148,148,148,7425009114,7081.040491,True



--- USGS 3DEP DEM TILE VALIDATION SAMPLE ---


,PRODUCT_NUMBER,TITLE,DOWNLOAD_URL,LOCAL_TILE_PATH,DOWNLOAD_ACTION,FILE_SIZE_BYTES,CRS,WIDTH,HEIGHT,BAND_COUNT,DTYPE,NODATA,LEFT,BOTTOM,RIGHT,TOP,READABLE,VALID,ERROR_MESSAGE
0,1,USGS 1 Arc Second n37w111 20210611,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused cached tile,51555085,EPSG:4269,3612,3612,1,float32,-999999.0,-111.001667,35.998333,-109.998333,37.001667,True,True,None
1,2,USGS 1 Arc Second n37w111 20211215,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused cached tile,51318661,EPSG:4269,3612,3612,1,float32,-999999.0,-111.001667,35.998333,-109.998333,37.001667,True,True,None
2,3,USGS 1 Arc Second n37w111 20240606,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused cached tile,51420837,EPSG:4269,3612,3612,1,float32,-999999.0,-111.001667,35.998333,-109.998333,37.001667,True,True,None
3,4,USGS 1 Arc Second n37w111 20241031,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused cached tile,51421242,EPSG:4269,3612,3612,1,float32,-999999.0,-111.001667,35.998333,-109.998333,37.001667,True,True,None
4,5,USGS 1 Arc Second n37w111 20260514,https://prd-tnm.s3.amazonaws.com/StagedProduct...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,Reused cached tile,51887305,EPSG:4269,3612,3612,1,float32,-999999.0,-111.001667,35.998333,-109.998333,37.001667,True,True,None



NOTE:
Every expected USGS 3DEP DEM tile was either downloaded or reused from the local cache and then validated as a readable georeferenced raster.
Progress messages were limited to the first tile, every 25th tile, and the final tile to reduce notebook output.
The validated tile cache is ready for creation of the aligned project-grid DEM mosaic.

=== USGS 3DEP DEM TILE DOWNLOAD AND VALIDATION COMPLETE ===


## Prepare and Validate DEM Mosaic Inputs


In [121]:
print('=== PREPARING AND VALIDATING DEM MOSAIC INPUTS ===')
# List the prerequisite objects required before this workflow stage can run.
required_dem_mosaic_objects = ['fire_hazard_usgs_dem_tile_download_complete',
    'fire_hazard_usgs_dem_download_validation', 'fire_hazard_usgs_dem_metadata_directory',
    'fire_hazard_project_bounds', 'fire_hazard_target_crs']
# Identify missing prerequisites so execution can stop before incomplete processing begins.
missing_dem_mosaic_objects = [name for name in required_dem_mosaic_objects if name not in globals()]

# Stop execution if required upstream objects are missing from the sequential workflow.
if missing_dem_mosaic_objects:
    raise NameError(f'Required DEM mosaic objects are missing:\n{missing_dem_mosaic_objects}')

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_usgs_dem_tile_download_complete:
    raise ValueError('The USGS 3DEP DEM tile cache is incomplete.')
# Collect fire hazard USGS DEM mosaic input records so the selected source set can be validated and
# documented.
fire_hazard_usgs_dem_mosaic_input_records = []

# Inspect each source record so unsuitable or non-overlapping elevation products can be excluded.
for _, record in fire_hazard_usgs_dem_download_validation.sort_values('PRODUCT_NUMBER').iterrows():
    # Route the tile product to its designated project output or cache location.
    tile_path = Path(record['LOCAL_TILE_PATH'])
    # Initialize or capture the tile error message used to diagnose a failed request or file
    # operation.
    tile_error = None
    # Record whether the tile satisfies the workflow QA requirements.
    tile_valid = False
    # Store the target extent used to constrain the elevation search or validation.
    target_bounds = None

    # Attempt the operation so expected data, file, or service failures can be handled explicitly.
    try:

        # Open the raster with a context manager so the dataset closes after this processing block.
        with rasterio.open(tile_path) as source:
            # Record whether the source satisfies the workflow QA requirements.
            source_valid = all([source.count == 1,
                source.width > 0, source.height > 0, source.crs is not None])

            # Select the processing branch required by the current cache, validation, or data state.
            if source_valid:
                # Store the target extent used to constrain the elevation search or validation.
                target_bounds = transform_bounds(source.crs,
                    fire_hazard_target_crs, *source.bounds, densify_pts=21)
                # Unpack the transformed tile extent for overlap testing against the project bounds.
                tile_left, tile_bottom, tile_right, tile_top = target_bounds
                # Unpack the project extent so tile coverage can be tested with explicit edge
                # comparisons.
                project_left, project_bottom, project_right, project_top = \
                    fire_hazard_project_bounds
                # Test the tile footprint against the project extent before accepting it as a mosaic
                # input.
                overlaps_project = all([tile_right > project_left,
                    tile_left < project_right, tile_top > project_bottom,
                    tile_bottom < project_top])
                # Record whether the tile satisfies the workflow QA requirements.
                tile_valid = source_valid and overlaps_project
            else:
                # Test the tile footprint against the project extent before accepting it as a mosaic
                # input.
                overlaps_project = False
            fire_hazard_usgs_dem_mosaic_input_records.append({'PRODUCT_NUMBER': \
                int(record['PRODUCT_NUMBER']),
                'LOCAL_TILE_PATH': str(tile_path), 'CRS': source.crs.to_string() if source.crs is \
                    not None else None,
                'DTYPE': source.dtypes[0], 'NODATA': source.nodata,
                'WIDTH': source.width, 'HEIGHT': source.height,
                'PIXEL_WIDTH': abs(source.transform.a), 'PIXEL_HEIGHT': abs(source.transform.e),
                'TARGET_LEFT': target_bounds[0] if target_bounds is not None else None,
                'TARGET_BOTTOM': target_bounds[1] if target_bounds is not None else None,
                'TARGET_RIGHT': target_bounds[2] if target_bounds is not None else None,
                'TARGET_TOP': target_bounds[3] if target_bounds is not None else None,
                'OVERLAPS_PROJECT_GRID': overlaps_project, 'ERROR_MESSAGE': tile_error,
                'VALID_FOR_MOSAIC': tile_valid})
    # Handle the expected failure without leaving the workflow in an inconsistent state.
    except Exception as error:
        # Initialize or capture the tile error message used to diagnose a failed request or file
        # operation.
        tile_error = str(error)
        fire_hazard_usgs_dem_mosaic_input_records.append({'PRODUCT_NUMBER': \
            int(record['PRODUCT_NUMBER']),
            'LOCAL_TILE_PATH': str(tile_path), 'CRS': None,
            'DTYPE': None, 'NODATA': None, 'WIDTH': None, 'HEIGHT': None,
            'PIXEL_WIDTH': None, 'PIXEL_HEIGHT': None, 'TARGET_LEFT': None,
            'TARGET_BOTTOM': None, 'TARGET_RIGHT': None, 'TARGET_TOP': None,
            'OVERLAPS_PROJECT_GRID': False, 'ERROR_MESSAGE': tile_error,
            'VALID_FOR_MOSAIC': False})
# Record whether the fire hazard USGS DEM mosaic input satisfies the workflow QA requirements.
fire_hazard_usgs_dem_mosaic_input_validation = \
    pd.DataFrame(fire_hazard_usgs_dem_mosaic_input_records)
# Retain DEM tiles that passed raster validation and overlap the project extent for mosaicking.
fire_hazard_usgs_dem_selected_mosaic_inputs = \
    fire_hazard_usgs_dem_mosaic_input_validation[fire_hazard_usgs_dem_mosaic_input_validation \
    ['VALID_FOR_MOSAIC']].drop_duplicates(subset='LOCAL_TILE_PATH').reset_index(drop=True)
# Retain rejected DEM tiles separately so their exclusion reasons remain visible in QA metadata.
fire_hazard_usgs_dem_rejected_mosaic_inputs = \
    fire_hazard_usgs_dem_mosaic_input_validation[~fire_hazard_usgs_dem_mosaic_input_validation \
    ['VALID_FOR_MOSAIC']].reset_index(drop=True)
# Collect the validated local DEM paths that will be opened during mosaic construction.
fire_hazard_usgs_dem_local_tile_paths = [Path(path_value) for path_value in \
    fire_hazard_usgs_dem_selected_mosaic_inputs['LOCAL_TILE_PATH'].tolist()]
# Track the fire hazard USGS DEM mosaic input count metric used for QA and completeness checks.
fire_hazard_usgs_dem_mosaic_input_count = len(fire_hazard_usgs_dem_local_tile_paths)

# Select the processing branch required by the current cache, validation, or data state.
if fire_hazard_usgs_dem_mosaic_input_count == 0:
    raise ValueError('No validated DEM tiles overlap the project grid.')
# Route the fire hazard USGS DEM mosaic input validation product to its designated project output or
# cache location.
fire_hazard_usgs_dem_mosaic_input_validation_path = Path(fire_hazard_usgs_dem_metadata_directory) \
    / 'usgs_3dep_dem_mosaic_input_validation.csv'
# Route the fire hazard USGS DEM selected mosaic inputs product to its designated project output or
# cache location.
fire_hazard_usgs_dem_selected_mosaic_inputs_path = Path(fire_hazard_usgs_dem_metadata_directory) \
    / 'usgs_3dep_dem_selected_mosaic_inputs.csv'
# Save the full mosaic-input QA table so accepted and rejected DEM tiles remain documented.
fire_hazard_usgs_dem_mosaic_input_validation.to_csv \
    (fire_hazard_usgs_dem_mosaic_input_validation_path,
    index=False)
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_usgs_dem_selected_mosaic_inputs.to_csv(fire_hazard_usgs_dem_selected_mosaic_inputs_path,
    index=False)
# Mark mosaic preparation complete only when at least one validated local DEM tile is available.
fire_hazard_usgs_dem_mosaic_inputs_prepared = all([fire_hazard_usgs_dem_mosaic_input_count > 0,
    fire_hazard_usgs_dem_mosaic_input_validation_path.exists(),
    fire_hazard_usgs_dem_selected_mosaic_inputs_path.exists()])
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Valid project-overlapping tiles: {fire_hazard_usgs_dem_mosaic_input_count:,}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Rejected tile records: {len(fire_hazard_usgs_dem_rejected_mosaic_inputs):,}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Mosaic inputs prepared: {fire_hazard_usgs_dem_mosaic_inputs_prepared}')
display(fire_hazard_usgs_dem_selected_mosaic_inputs.head(5))
# Report workflow status so notebook execution can be verified interactively.
print('\n=== DEM MOSAIC INPUT PREPARATION COMPLETE ===')

# Report workflow status so notebook execution can be verified interactively.


=== PREPARING AND VALIDATING DEM MOSAIC INPUTS ===
-> Valid project-overlapping tiles: 135
-> Rejected tile records: 13
-> Mosaic inputs prepared: True


,PRODUCT_NUMBER,LOCAL_TILE_PATH,CRS,DTYPE,NODATA,WIDTH,HEIGHT,PIXEL_WIDTH,PIXEL_HEIGHT,TARGET_LEFT,TARGET_BOTTOM,TARGET_RIGHT,TARGET_TOP,OVERLAPS_PROJECT_GRID,ERROR_MESSAGE,VALID_FOR_MOSAIC
0,12,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:4269,float32,-999999.0,3612,3612,0.000278,0.000278,319579.302433,3.984224e+06,411172.914384,4.096930e+06,True,None,True
1,13,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:4269,float32,-999999.0,3612,3612,0.000278,0.000278,319579.302433,3.984224e+06,411172.914384,4.096930e+06,True,None,True
2,14,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:4269,float32,-999999.0,3612,3612,0.000278,0.000278,319579.302433,3.984224e+06,411172.914384,4.096930e+06,True,None,True
3,15,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:4269,float32,-999999.0,3612,3612,0.000278,0.000278,319579.302433,3.984224e+06,411172.914384,4.096930e+06,True,None,True
4,16,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,EPSG:4269,float32,-999999.0,3612,3612,0.000278,0.000278,319579.302433,3.984224e+06,411172.914384,4.096930e+06,True,None,True



=== DEM MOSAIC INPUT PREPARATION COMPLETE ===


## Build Aligned USGS 3DEP DEM Mosaic


In [122]:
print('=== BUILDING ALIGNED USGS 3DEP DEM MOSAIC WITH RASTERIO ===')
# List the prerequisite objects required before this workflow stage can run.
required_aligned_dem_objects = ['fire_hazard_usgs_dem_mosaic_inputs_prepared',
    'fire_hazard_usgs_dem_selected_mosaic_inputs',
    'fire_hazard_terrain_directory', 'fire_hazard_usgs_dem_metadata_directory',
    'fire_hazard_alignment_width', 'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform', 'fire_hazard_alignment_study_area_mask',
    'fire_hazard_target_crs', 'fire_hazard_nodata_value']
# Identify missing prerequisites so execution can stop before incomplete processing begins.
missing_aligned_dem_objects = [name for name in required_aligned_dem_objects if name not in \
    globals()]

# Stop execution if required upstream objects are missing from the sequential workflow.
if missing_aligned_dem_objects:
    raise NameError(f'Required aligned-DEM objects are missing:\n{missing_aligned_dem_objects}')

# Select the processing branch required by the current cache, validation, or data state.
if not fire_hazard_usgs_dem_mosaic_inputs_prepared:
    raise ValueError('The DEM mosaic inputs are not prepared.')
from rasterio.windows import bounds as window_bounds
from rasterio.windows import transform as window_transform
from contextlib import ExitStack
# Route the fire hazard USGS DEM aligned mosaic product to its designated project output or cache
# location.
fire_hazard_usgs_dem_aligned_mosaic_path = Path(fire_hazard_terrain_directory) / \
    'usgs_3dep_dem_aligned_30m.tif'
# Route the fire hazard USGS DEM aligned temporary product to its designated project output or cache
# location.
fire_hazard_usgs_dem_aligned_temporary_path = \
    fire_hazard_usgs_dem_aligned_mosaic_path.with_name(fire_hazard_usgs_dem_aligned_mosaic_path \
    .stem + '.part.tif')
# Define aligned dem profile used to configure this workflow stage.
aligned_dem_profile = {'driver': 'GTiff', 'dtype': 'float32',
    'count': 1, 'width': fire_hazard_alignment_width,
    'height': fire_hazard_alignment_height, 'crs': fire_hazard_target_crs,
    'transform': fire_hazard_alignment_transform, 'nodata': fire_hazard_nodata_value,
    'compress': 'deflate', 'predictor': 3, 'tiled': True,
    'blockxsize': 512, 'blockysize': 512, 'BIGTIFF': 'IF_SAFER'}

# Reuse the existing cached product when it is already available locally.
if fire_hazard_usgs_dem_aligned_temporary_path.exists():
    fire_hazard_usgs_dem_aligned_temporary_path.unlink()
# Track the mosaic valid pixels metric used for QA and completeness checks.
mosaic_valid_pixels = 0
# Track the mosaic outside pixels metric used for QA and completeness checks.
mosaic_outside_pixels = 0
# Track the mosaic minimum statistic used to validate raster values and report coverage.
mosaic_minimum = None
# Track the mosaic maximum statistic used to validate raster values and report coverage.
mosaic_maximum = None
# Track the mosaic sum statistic used to validate raster values and report coverage.
mosaic_sum = 0.0

# Reproject the raster onto the common 30-meter project grid so
# every hazard component aligns cell by cell.
# Manage temporary resources with a context manager so they are released after processing.
with ExitStack() as stack:
    # Track source files used to construct the aligned DEM and preserve provenance.
    source_records = []

    # Inspect each source record so unsuitable or non-overlapping elevation products can be
    # excluded.
    for _, record in fire_hazard_usgs_dem_selected_mosaic_inputs.iterrows():
        # Route the source product to its designated project output or cache location.
        source_path = Path(record['LOCAL_TILE_PATH'])
        # Keep the source raster open within the shared context while mosaic windows are processed.
        source = stack.enter_context(rasterio.open(source_path))
        source_records.append({'source': source,
            'left': float(record['TARGET_LEFT']), 'bottom': float(record['TARGET_BOTTOM']),
            'right': float(record['TARGET_RIGHT']), 'top': float(record['TARGET_TOP'])})

    # Process the raster in internal windows to control memory use
    # while preserving the full-resolution output.
    # Open the raster with a context manager so the dataset closes after this processing block.
    with rasterio.open(fire_hazard_usgs_dem_aligned_temporary_path,
        'w', **aligned_dem_profile) as destination:
        # Count output raster windows so mosaic progress can be reported during windowed processing.
        total_windows = sum((1 for _ in destination.block_windows(1)))

        # Process raster windows sequentially to limit memory use on the full-resolution grid.
        for window_number, (_, window) in enumerate(destination.block_windows(1), start=1):

            # Select the processing branch required by the current cache, validation, or data state.
            if window_number == 1 or window_number % 250 == 0 or window_number == total_windows:
                # Report workflow status so notebook execution can be verified interactively.
                print(f'-> Processing target window {window_number:,} of {total_windows:,}...')
            # Calculate the current output-window bounds so non-overlapping source tiles can be
            # skipped.
            win_left, win_bottom, win_right, win_top = window_bounds(window,
                fire_hazard_alignment_transform)
            # Allocate the current mosaic window with NoData before valid source pixels are merged
            # into it.
            destination_array = np.full((int(window.height),
                int(window.width)), fire_hazard_nodata_value, dtype=np.float32)

            # Inspect each source record so unsuitable or non-overlapping elevation products can be
            # excluded.
            for source_record in source_records:
                # Test source bounds against the current output window to skip tiles that cannot
                # contribute data.
                overlaps_window = all([source_record['right'] > win_left,
                    source_record['left'] < win_right, source_record['top'] > win_bottom,
                    source_record['bottom'] < win_top])

                # Select the processing branch required by the current cache, validation, or data
                # state.
                if not overlaps_window:
                    continue
                # Keep the source raster open within the shared context while mosaic windows are
                # processed.
                source = source_record['source']
                # Allocate a temporary tile array matching the destination window for reprojection.
                tile_array = np.full(destination_array.shape,
                    fire_hazard_nodata_value, dtype=np.float32)
                reproject(source=rasterio.band(source,
                    1), destination=tile_array, src_transform=source.transform,
                    src_crs=source.crs, src_nodata=source.nodata, \
                        dst_transform=window_transform(window,
                    fire_hazard_alignment_transform), dst_crs=fire_hazard_target_crs,
                    dst_nodata=fire_hazard_nodata_value, resampling=Resampling.bilinear,
                    init_dest_nodata=True)
                # Record whether the tile satisfies the workflow QA requirements.
                tile_valid = np.isfinite(tile_array) & (tile_array != fire_hazard_nodata_value)
                # Update only the cells or records required by the current processing mask.
                destination_array[tile_valid] = tile_array[tile_valid]
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            row_start = int(window.row_off)
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            row_end = row_start + int(window.height)
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            column_start = int(window.col_off)
            # Convert the raster-window extent to array indices for the matching study-area mask
            # slice.
            column_end = column_start + int(window.width)
            # Create the study mask used to restrict calculations to valid study-area cells.
            study_mask = fire_hazard_alignment_study_area_mask[row_start:row_end,
                column_start:column_end]
            # Update only the cells or records required by the current processing mask.
            destination_array[~study_mask] = fire_hazard_nodata_value
            # Create the valid mask used to restrict calculations to valid study-area cells.
            valid_mask = np.isfinite(destination_array) & (destination_array != \
                fire_hazard_nodata_value)
            # Isolate raster values used to update coverage counts and normalized-value QA
            # statistics.
            valid_values = destination_array[valid_mask]
            # Track the mosaic valid pixels metric used for QA and completeness checks.
            mosaic_valid_pixels += int(valid_mask.sum())
            # Track the mosaic outside pixels metric used for QA and completeness checks.
            mosaic_outside_pixels += int((valid_mask & ~study_mask).sum())

            # Select the processing branch required by the current cache, validation, or data state.
            if valid_values.size > 0:
                # Track the block minimum statistic used to validate raster values and report
                # coverage.
                block_minimum = float(valid_values.min())
                # Track the block maximum statistic used to validate raster values and report
                # coverage.
                block_maximum = float(valid_values.max())
                # Track the mosaic minimum statistic used to validate raster values and report
                # coverage.
                mosaic_minimum = block_minimum if mosaic_minimum is None else min(mosaic_minimum,
                    block_minimum)
                # Track the mosaic maximum statistic used to validate raster values and report
                # coverage.
                mosaic_maximum = block_maximum if mosaic_maximum is None else max(mosaic_maximum,
                    block_maximum)
                # Track the mosaic sum statistic used to validate raster values and report coverage.
                mosaic_sum += float(valid_values.sum(dtype=np.float64))
            # Write the processed data to the configured output resource.
            destination.write(destination_array, 1, window=window)
        destination.set_band_description(1, 'Aligned USGS 3DEP elevation')
        destination.update_tags(SOURCE_ID='USGS_3DEP_DEM',
            PROCESSING_METHOD='Rasterio windowed reprojection mosaic',
            TARGET_CRS=fire_hazard_target_crs, TARGET_CELL_SIZE_METERS=fire_hazard_cell_size)
# Track the mosaic mean statistic used to validate raster values and report coverage.
mosaic_mean = mosaic_sum / mosaic_valid_pixels if mosaic_valid_pixels > 0 else None

# Open the raster with a context manager so the dataset closes after this processing block.
with rasterio.open(fire_hazard_usgs_dem_aligned_temporary_path) as aligned_source:
    # Record whether the aligned DEM structure satisfies the workflow QA requirements.
    aligned_dem_structure_valid = all([aligned_source.count == 1,
        aligned_source.width == fire_hazard_alignment_width,
        aligned_source.height == fire_hazard_alignment_height,
        aligned_source.crs == rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
        aligned_source.transform.almost_equals(fire_hazard_alignment_transform),
        aligned_source.dtypes[0] == 'float32', aligned_source.nodata == fire_hazard_nodata_value])
# Record whether the fire hazard USGS DEM aligned mosaic satisfies the workflow QA requirements.
fire_hazard_usgs_dem_aligned_mosaic_valid = all([aligned_dem_structure_valid,
    mosaic_valid_pixels > 0, mosaic_outside_pixels == 0,
    mosaic_minimum is not None, mosaic_maximum is not None])

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_usgs_dem_aligned_mosaic_valid:
    raise ValueError('The aligned DEM mosaic failed validation.')

# Reuse the existing cached product when it is already available locally.
if fire_hazard_usgs_dem_aligned_mosaic_path.exists():
    fire_hazard_usgs_dem_aligned_mosaic_path.unlink()
fire_hazard_usgs_dem_aligned_temporary_path.replace(fire_hazard_usgs_dem_aligned_mosaic_path)
# Assemble fire hazard USGS DEM aligned mosaic summary metadata for QA reporting and later workflow
# phases.
fire_hazard_usgs_dem_aligned_mosaic_summary = pd.DataFrame([{'SOURCE_ID': 'USGS_3DEP_DEM',
    'INPUT_TILE_COUNT': len(source_records), 'OUTPUT_PATH': \
        str(fire_hazard_usgs_dem_aligned_mosaic_path),
    'VALID_PIXELS': mosaic_valid_pixels, 'VALID_OUTSIDE_STUDY_AREA': mosaic_outside_pixels,
    'MINIMUM_ELEVATION': mosaic_minimum, 'MAXIMUM_ELEVATION': mosaic_maximum,
    'MEAN_ELEVATION': mosaic_mean, 'GRID_VALID': aligned_dem_structure_valid,
    'VALID': fire_hazard_usgs_dem_aligned_mosaic_valid}])
# Persist fire hazard USGS DEM aligned mosaic summary metadata at the project location used for QA
# and reproducibility.
fire_hazard_usgs_dem_aligned_mosaic_summary_path = Path(fire_hazard_usgs_dem_metadata_directory) \
    / 'usgs_3dep_dem_aligned_mosaic_summary.csv'
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_usgs_dem_aligned_mosaic_summary.to_csv(fire_hazard_usgs_dem_aligned_mosaic_summary_path,
    index=False)
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Aligned DEM: {fire_hazard_usgs_dem_aligned_mosaic_path}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Valid elevation pixels: {mosaic_valid_pixels:,}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Elevation range: {mosaic_minimum:.2f} to {mosaic_maximum:.2f}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Aligned mosaic valid: {fire_hazard_usgs_dem_aligned_mosaic_valid}')
display(fire_hazard_usgs_dem_aligned_mosaic_summary)
gc.collect()
# Report workflow status so notebook execution can be verified interactively.
print('\n=== ALIGNED USGS 3DEP DEM MOSAIC COMPLETE ===')

# Report workflow status so notebook execution can be verified interactively.


=== BUILDING ALIGNED USGS 3DEP DEM MOSAIC WITH RASTERIO ===
-> Processing target window 1 of 551...
-> Processing target window 250 of 551...
-> Processing target window 500 of 551...
-> Processing target window 551 of 551...
-> Aligned DEM: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\terrain_hazard\usgs_3dep_dem_aligned_30m.tif
-> Valid elevation pixels: 15,475,232
-> Elevation range: 664.34 to 3609.10
-> Aligned mosaic valid: True


,SOURCE_ID,INPUT_TILE_COUNT,OUTPUT_PATH,VALID_PIXELS,VALID_OUTSIDE_STUDY_AREA,MINIMUM_ELEVATION,MAXIMUM_ELEVATION,MEAN_ELEVATION,GRID_VALID,VALID
0,USGS_3DEP_DEM,135,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,15475232,0,664.33783,3609.096191,1715.982485,True,True



=== ALIGNED USGS 3DEP DEM MOSAIC COMPLETE ===


## Register Aligned USGS 3DEP DEM Source


In [123]:
print('=== REGISTERING ALIGNED USGS 3DEP DEM SOURCE ===')
# List the prerequisite objects required before this workflow stage can run.
required_dem_registration_objects = ['fire_hazard_usgs_dem_aligned_mosaic_valid',
    'fire_hazard_usgs_dem_aligned_mosaic_path', 'fire_hazard_usgs_dem_aligned_mosaic_summary',
    'fire_hazard_usgs_dem_selected_dataset', 'fire_hazard_usgs_dem_tile_directory',
    'fire_hazard_usgs_dem_metadata_directory', 'fire_hazard_source_paths']
# Identify missing prerequisites so execution can stop before incomplete processing begins.
missing_dem_registration_objects = [name for name in required_dem_registration_objects if name \
    not in globals()]

# Stop execution if required upstream objects are missing from the sequential workflow.
if missing_dem_registration_objects:
    raise NameError(f'''Required DEM registration objects are missing:
{missing_dem_registration_objects}''')

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_usgs_dem_aligned_mosaic_valid:
    raise ValueError('The aligned DEM mosaic has not passed validation.')
# Calculate normalized hazard scores only for valid cells that participate in the model.
fire_hazard_source_paths['USGS_3DEP_DEM'] = Path(fire_hazard_usgs_dem_aligned_mosaic_path)
# Route the terrain DEM source product to its designated project output or cache location.
terrain_dem_source_path = Path(fire_hazard_source_paths['USGS_3DEP_DEM'])
# Assemble fire hazard USGS DEM acquisition manifest metadata for QA reporting and later workflow
# phases.
fire_hazard_usgs_dem_acquisition_manifest = pd.DataFrame([{'SOURCE_ID': 'USGS_3DEP_DEM',
    'PRODUCT_NAME': 'USGS 3DEP 1 Arc-Second DEM', 'DATASET': fire_hazard_usgs_dem_selected_dataset,
    'PROCESSING_METHOD': 'Rasterio windowed reprojection mosaic',
    'RAW_TILE_COUNT': len(fire_hazard_usgs_dem_selected_mosaic_inputs),
    'RAW_TILE_DIRECTORY': str(fire_hazard_usgs_dem_tile_directory),
    'PRIMARY_SOURCE_PATH': str(terrain_dem_source_path),
    'SOURCE_ALREADY_ALIGNED': True, 'TARGET_CRS': fire_hazard_target_crs,
    'TARGET_CELL_SIZE_METERS': fire_hazard_cell_size,
    'TARGET_WIDTH': fire_hazard_alignment_width, 'TARGET_HEIGHT': fire_hazard_alignment_height,
    'VALID': True}])
# Persist fire hazard USGS DEM acquisition manifest metadata at the project location used for QA and
# reproducibility.
fire_hazard_usgs_dem_acquisition_manifest_path = Path(fire_hazard_usgs_dem_metadata_directory) / \
    'usgs_3dep_dem_acquisition_manifest.csv'
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_usgs_dem_acquisition_manifest.to_csv(fire_hazard_usgs_dem_acquisition_manifest_path,
    index=False)
# Record the aligned DEM source and processing method in the completion marker used by later phases.
fire_hazard_usgs_dem_completion_marker = {'source_id': 'USGS_3DEP_DEM',
    'status': 'complete', 'processing_method': 'Rasterio windowed reprojection mosaic',
    'primary_source': str(terrain_dem_source_path),
    'source_already_aligned': True, 'valid': True}
# Persist fire hazard USGS DEM completion marker metadata at the project location used for QA and
# reproducibility.
fire_hazard_usgs_dem_completion_marker_path = Path(fire_hazard_usgs_dem_metadata_directory) / \
    'usgs_3dep_dem_cache_complete.json'

# Open the metadata file with a context manager so it closes cleanly after writing.
with fire_hazard_usgs_dem_completion_marker_path.open('w', encoding='utf-8') as completion_file:
    # Write the completion metadata used to document and verify the cached DEM source.
    json.dump(fire_hazard_usgs_dem_completion_marker, completion_file, indent=2)
# Record whether fire hazard USGS DEM acquisition has satisfied its completion requirement.
fire_hazard_usgs_dem_acquisition_complete = all([terrain_dem_source_path.exists(),
    fire_hazard_usgs_dem_acquisition_manifest_path.exists(),
    fire_hazard_usgs_dem_completion_marker_path.exists()])

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_usgs_dem_acquisition_complete:
    raise ValueError('DEM acquisition registration did not complete.')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Registered DEM source: {terrain_dem_source_path}')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Source already aligned: True')
# Report workflow status so notebook execution can be verified interactively.
print(f'-> Acquisition complete: {fire_hazard_usgs_dem_acquisition_complete}')
display(fire_hazard_usgs_dem_acquisition_manifest)
# Report workflow status so notebook execution can be verified interactively.
print('\n=== ALIGNED USGS 3DEP DEM SOURCE REGISTERED ===')


=== REGISTERING ALIGNED USGS 3DEP DEM SOURCE ===
-> Registered DEM source: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\terrain_hazard\usgs_3dep_dem_aligned_30m.tif
-> Source already aligned: True
-> Acquisition complete: True


,SOURCE_ID,PRODUCT_NAME,DATASET,PROCESSING_METHOD,RAW_TILE_COUNT,RAW_TILE_DIRECTORY,PRIMARY_SOURCE_PATH,SOURCE_ALREADY_ALIGNED,TARGET_CRS,TARGET_CELL_SIZE_METERS,TARGET_WIDTH,TARGET_HEIGHT,VALID
0,USGS_3DEP_DEM,USGS 3DEP 1 Arc-Second DEM,National Elevation Dataset (NED) 1 arc-second,Rasterio windowed reprojection mosaic,135,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,EPSG:26912,30,9454,14467,True



=== ALIGNED USGS 3DEP DEM SOURCE REGISTERED ===


## Build Terrain-Hazard Component


In [124]:
# Report workflow status so notebook execution can be verified interactively.
print('=== BUILDING TERRAIN-HAZARD COMPONENT ===')

# Acquire and align USGS 3DEP elevation data for deriving slope
# and aspect on the common project grid.
# Route the terrain DEM source product to its designated project output or cache location.
terrain_dem_source_path = require_fire_hazard_source('USGS_3DEP_DEM')

# Convert terrain slope to a normalized score representing its
# influence on potential fire spread.
# Route the terrain slope product to its designated project output or cache location.
terrain_slope_path = fire_hazard_terrain_directory / 'slope_degrees.tif'

# Convert terrain aspect to a warm-and-dry exposure score, with
# southwest-facing slopes receiving greater modeled hazard.
# Route the terrain aspect product to its designated project output or cache location.
terrain_aspect_path = fire_hazard_terrain_directory / 'aspect_degrees.tif'
# Route the terrain slope hazard product to its designated project output or cache location.
terrain_slope_hazard_path = fire_hazard_terrain_directory / 'slope_hazard.tif'
# Route the terrain aspect hazard product to its designated project output or cache location.
terrain_aspect_hazard_path = fire_hazard_terrain_directory / 'aspect_hazard.tif'
# Route the fire hazard terrain hazard product to its designated project output or cache location.
fire_hazard_terrain_hazard_path = fire_hazard_terrain_directory / 'terrain_hazard.tif'

# Open the raster with a context manager so the dataset closes after this processing block.
with rasterio.open(terrain_dem_source_path) as dem_source:
    # Record whether the DEM grid satisfies the workflow QA requirements.
    dem_grid_valid = all([dem_source.width == fire_hazard_alignment_width,
        dem_source.height == fire_hazard_alignment_height,
        dem_source.crs == rasterio.crs.CRS.from_user_input(fire_hazard_target_crs),
        dem_source.transform.almost_equals(fire_hazard_alignment_transform)])

    # Stop execution if the preceding QA requirement has not been satisfied.
    if not dem_grid_valid:
        raise ValueError('The registered DEM does not match the common project grid.')
    # Read the aligned elevation band as Float32 so slope and aspect calculations use a consistent
    # numeric type.
    aligned_dem = dem_source.read(1, out_dtype='float32')
# Identify DEM cells with finite elevation values before terrain derivatives are calculated.
valid_dem = np.isfinite(aligned_dem) & (aligned_dem != fire_hazard_nodata_value)
# Convert invalid DEM cells to NaN so gradient calculations do not treat NoData as elevation.
dem_work = np.where(valid_dem, aligned_dem, np.nan)
# Calculate elevation gradients in both grid directions using the project cell size.
dz_dy, dz_dx = np.gradient(dem_work, fire_hazard_cell_size, fire_hazard_cell_size)
# Calculate slope in degrees from the DEM gradients for terrain-hazard modeling.
slope_degrees = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype(np.float32)
# Calculate compass aspect from the DEM gradients while ignoring expected NoData warnings.
with np.errstate(invalid='ignore'):
    aspect_degrees = (
        (90.0 - np.degrees(np.arctan2(-dz_dy, dz_dx))) % 360.0
    ).astype(np.float32)
# Restore the project NoData value outside valid DEM cells before raster export.
slope_degrees[~valid_dem] = fire_hazard_nodata_value
# Restore the project NoData value outside valid DEM cells before raster export.
aspect_degrees[~valid_dem] = fire_hazard_nodata_value
# Write the derived terrain raster on the common project grid.
write_fire_hazard_array(slope_degrees, terrain_slope_path, 'Terrain slope in degrees')
# Write the derived terrain raster on the common project grid.
write_fire_hazard_array(aspect_degrees, terrain_aspect_path, 'Terrain aspect in degrees')
# Allocate the normalized slope-hazard raster while preserving the project NoData value.
slope_hazard = np.full(slope_degrees.shape, fire_hazard_nodata_value, dtype=np.float32)
# Calculate normalized hazard scores only for valid cells that participate in the model.
slope_hazard[valid_dem] = np.clip(slope_degrees[valid_dem] / 45.0, 0.0, 1.0)
# Allocate the normalized aspect-hazard raster while preserving the project NoData value.
aspect_hazard = np.full(aspect_degrees.shape, fire_hazard_nodata_value, dtype=np.float32)
# Rotate aspect around 225 degrees so southwest-facing terrain receives the highest score.
aspect_radians = np.radians(aspect_degrees[valid_dem] - 225.0)
# Calculate normalized hazard scores only for valid cells that participate in the model.
aspect_hazard[valid_dem] = ((np.cos(aspect_radians) + 1.0) / 2.0).astype(np.float32)
# Write the derived terrain raster on the common project grid.
write_fire_hazard_array(slope_hazard, terrain_slope_hazard_path, 'Normalized slope hazard')
# Write the derived terrain raster on the common project grid.
write_fire_hazard_array(aspect_hazard, terrain_aspect_hazard_path, 'Normalized aspect hazard')
# Store the slope model weight used to combine terrain indicators consistently.
slope_model_weight = float(fire_hazard_components['slope_hazard']['weight'])
# Store the aspect model weight used to combine terrain indicators consistently.
aspect_model_weight = float(fire_hazard_components['aspect_hazard']['weight'])
# Track the terrain weight sum statistic used to validate raster values and report coverage.
terrain_weight_sum = slope_model_weight + aspect_model_weight
# Store the terrain slope internal weight used to combine terrain indicators consistently.
terrain_slope_internal_weight = slope_model_weight / terrain_weight_sum
# Store the terrain aspect internal weight used to combine terrain indicators consistently.
terrain_aspect_internal_weight = aspect_model_weight / terrain_weight_sum
# Record whether the terrain satisfies the workflow QA requirements.
terrain_valid = valid_dem & fire_hazard_alignment_study_area_mask
# Allocate the composite terrain raster before applying the internal slope and aspect weights.
terrain_hazard = np.full(slope_hazard.shape, fire_hazard_nodata_value, dtype=np.float32)
# Calculate normalized hazard scores only for valid cells that participate in the model.
terrain_hazard[terrain_valid] = slope_hazard[terrain_valid] * terrain_slope_internal_weight + \
    aspect_hazard[terrain_valid] * terrain_aspect_internal_weight
# Write the derived terrain raster on the common project grid.
write_fire_hazard_array(terrain_hazard, fire_hazard_terrain_hazard_path,
    'Composite terrain hazard', {'SLOPE_INTERNAL_WEIGHT': terrain_slope_internal_weight,
    'ASPECT_INTERNAL_WEIGHT': terrain_aspect_internal_weight})
# Record whether the terrain satisfies the workflow QA requirements.
terrain_validation = validate_normalized_component(fire_hazard_terrain_hazard_path)
# Mark the terrain component complete only when its normalized-output validation passes.
fire_hazard_terrain_component_finalized = bool(terrain_validation['VALID'])
# Assemble fire hazard terrain summary metadata for QA reporting and later workflow phases.
fire_hazard_terrain_summary = pd.DataFrame([{'COMPONENT_ID': 'TERRAIN_HAZARD',
    'PRIMARY_OUTPUT': str(fire_hazard_terrain_hazard_path),
    'SLOPE_WEIGHT': terrain_slope_internal_weight,
    'ASPECT_WEIGHT': terrain_aspect_internal_weight,
    **terrain_validation}])
# Persist fire hazard terrain summary metadata at the project location used for QA and
# reproducibility.
fire_hazard_terrain_summary_path = fire_hazard_remaining_metadata / 'terrain_hazard_summary.csv'
# Save the QA or provenance table so later phases can verify this processing stage.
fire_hazard_terrain_summary.to_csv(fire_hazard_terrain_summary_path, index=False)

# Stop execution if the preceding QA requirement has not been satisfied.
if not fire_hazard_terrain_component_finalized:
    raise ValueError('The terrain-hazard component failed validation.')
display(fire_hazard_terrain_summary)
del aligned_dem
del dem_work
del dz_dy
del dz_dx
del slope_degrees
del aspect_degrees
del slope_hazard
del aspect_hazard
del terrain_hazard
gc.collect()
# Report workflow status so notebook execution can be verified interactively.
print('\n=== TERRAIN-HAZARD COMPONENT COMPLETE ===')


=== BUILDING TERRAIN-HAZARD COMPONENT ===


,COMPONENT_ID,PRIMARY_OUTPUT,SLOPE_WEIGHT,ASPECT_WEIGHT,PATH,FILE_EXISTS,GRID_VALID,STRUCTURE_VALID,VALID_PIXELS,VALID_OUTSIDE_STUDY_AREA,MINIMUM,MAXIMUM,MEAN,VALID
0,TERRAIN_HAZARD,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,0.6,0.4,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,True,15446582,0,0.000002,1.0,0.366972,True



=== TERRAIN-HAZARD COMPONENT COMPLETE ===


# CHRG Phase 10 – Historical Fire

## Purpose

Convert historical fire occurrence and proximity information into a normalized historical-fire-likelihood component.


## Acquire and Cache Historical Fire Sources

Acquire the required InFORM fire-occurrence records for the three-county study area and document optional MTBS availability before historical-fire processing.


In [125]:
print('=== ACQUIRING HISTORICAL FIRE SOURCES ===')

# List the source, network, cache, and study-area objects required for historical-fire acquisition.
required_historical_fire_acquisition_inputs = [
    'fire_hazard_source_paths',
    'fire_hazard_request_session',
    'fire_hazard_request_timeout',
    'fire_hazard_verify_ssl_certificates',
    'refresh_fire_hazard_sources',
    'fire_hazard_temporary_download_suffix',
    'gdf_county_boundaries_aligned',
]

# Identify missing prerequisites before any source request or cache update is attempted.
missing_historical_fire_acquisition_inputs = [
    object_name
    for object_name in required_historical_fire_acquisition_inputs
    if object_name not in globals()
]

# Stop execution if historical-fire acquisition cannot use the validated project configuration.
if missing_historical_fire_acquisition_inputs:
    raise NameError(
        'The following historical-fire acquisition inputs are missing:\n'
        f'{missing_historical_fire_acquisition_inputs}'
    )

# Resolve the configured cache path for the required InFORM fire-occurrence GeoJSON.
historical_occurrence_cache_path = fire_hazard_source_paths[
    'INFORM_FODR_FIRE_OCCURRENCES'
]

# Resolve the configured cache path for the optional MTBS burned-area boundary archive.
historical_perimeter_cache_path = fire_hazard_source_paths[
    'MTBS_FIRE_PERIMETERS'
]

# Use the authoritative InFORM public feature layer rather than the catalog landing page.
inform_occurrence_query_url = (
    'https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/'
    'InFORM_FireOccurrence_Public/FeatureServer/0/query'
)

# Convert the validated study-area counties to geographic coordinates for the service query.
historical_fire_query_counties = gdf_county_boundaries_aligned.to_crs('EPSG:4326')

# Use the three-county bounding extent to limit the national InFORM service response.
historical_fire_query_bounds = historical_fire_query_counties.total_bounds

# Format the study-area envelope for the ArcGIS REST spatial filter.
historical_fire_query_geometry = ','.join(
    (
        f'{historical_fire_query_bounds[0]:.8f}',
        f'{historical_fire_query_bounds[1]:.8f}',
        f'{historical_fire_query_bounds[2]:.8f}',
        f'{historical_fire_query_bounds[3]:.8f}',
    )
)

# Reuse the cached occurrence file unless a refresh is requested or the cache is missing.
historical_occurrence_refresh_requested = (
    refresh_fire_hazard_sources
    or not historical_occurrence_cache_path.exists()
)

# Acquire the required occurrence source explicitly when the configured cache is unavailable.
if historical_occurrence_refresh_requested:
    # Create the destination directory before writing the paged GeoJSON response.
    historical_occurrence_cache_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Use a separate temporary file so an interrupted request cannot replace valid cached data.
    historical_occurrence_temporary_path = (
        historical_occurrence_cache_path.parent
        / (
            historical_occurrence_cache_path.name
            + fire_hazard_temporary_download_suffix
        )
    )

    # Remove a stale partial response before starting a new InFORM acquisition.
    if historical_occurrence_temporary_path.exists():
        historical_occurrence_temporary_path.unlink()

    # Match the service record limit so each request returns the largest supported page.
    inform_page_size = 2000

    # Begin pagination at the first InFORM record that intersects the study-area envelope.
    inform_result_offset = 0

    # Accumulate GeoJSON features from every service page before creating the local cache.
    inform_features = []

    # Request pages until the service returns fewer records than the configured page size.
    while True:
        # Build the ArcGIS REST query for one spatially filtered GeoJSON response page.
        inform_query_parameters = {
            'where': '1=1',
            'outFields': '*',
            'returnGeometry': 'true',
            'geometry': historical_fire_query_geometry,
            'geometryType': 'esriGeometryEnvelope',
            'inSR': '4326',
            'spatialRel': 'esriSpatialRelIntersects',
            'outSR': '4326',
            'orderByFields': 'OBJECTID ASC',
            'resultOffset': inform_result_offset,
            'resultRecordCount': inform_page_size,
            'f': 'geojson',
        }

        # Submit one retry-enabled InFORM query and retain the response for validation.
        inform_response = None
        try:
            inform_response = fire_hazard_request_session.get(
                inform_occurrence_query_url,
                params=inform_query_parameters,
                timeout=fire_hazard_request_timeout,
                verify=fire_hazard_verify_ssl_certificates,
            )
            inform_response.raise_for_status()

            # Parse the service response only after the HTTP request completes successfully.
            inform_page = inform_response.json()

        # Convert request or JSON failures into a source-specific acquisition error.
        except (requests.RequestException, ValueError) as error:
            raise RuntimeError(
                'The InFORM fire-occurrence service request failed during Phase 10.'
            ) from error

        # Release the HTTP connection after each page so long pagination runs remain stable.
        finally:
            if inform_response is not None:
                inform_response.close()

        # Stop if ArcGIS reports a service-level error inside an otherwise valid JSON response.
        if isinstance(inform_page, dict) and 'error' in inform_page:
            raise RuntimeError(
                'The InFORM fire-occurrence service returned an ArcGIS error:\n'
                f"{inform_page['error']}"
            )

        # Require a GeoJSON feature collection before records are added to the local cache.
        if (
            not isinstance(inform_page, dict)
            or not isinstance(inform_page.get('features'), list)
        ):
            raise ValueError(
                'The InFORM response did not contain a valid GeoJSON feature collection.'
            )

        # Retain the current page of historical fire occurrences for the combined cache file.
        inform_page_features = inform_page['features']
        inform_features.extend(inform_page_features)

        # End pagination when the current response contains the final partial page.
        if len(inform_page_features) < inform_page_size:
            break

        # Advance by the number of records actually returned to avoid skipping service records.
        inform_result_offset += len(inform_page_features)

    # Stop if the authoritative service returned no records for the three-county study extent.
    if not inform_features:
        raise ValueError(
            'The InFORM service returned no fire-occurrence records for the study area.'
        )

    # Build one GeoJSON feature collection containing every paged InFORM response feature.
    inform_geojson = {
        'type': 'FeatureCollection',
        'features': inform_features,
    }

    # Write the completed response to a temporary file before replacing the final cache.
    with historical_occurrence_temporary_path.open(
        'w',
        encoding='utf-8',
    ) as temporary_file:
        json.dump(inform_geojson, temporary_file)

    # Reject an empty temporary GeoJSON before it can become the required cached source.
    if historical_occurrence_temporary_path.stat().st_size <= 0:
        raise IOError(
            'The temporary InFORM GeoJSON contains no data and cannot replace the cache.'
        )

    # Promote the completed GeoJSON atomically so Phase 10 sees only a complete source file.
    historical_occurrence_temporary_path.replace(
        historical_occurrence_cache_path
    )

    # Report the completed source acquisition and record count for notebook-level QA.
    print(
        f'-> InFORM occurrence cache created with '
        f'{len(inform_features):,} study-area records.'
    )
    print(f'-> InFORM cache path: {historical_occurrence_cache_path}')

# Report cache reuse when the required InFORM source was already available locally.
else:
    print('-> Reusing cached InFORM fire-occurrence source.')
    print(f'-> InFORM cache path: {historical_occurrence_cache_path}')

# Record whether the supporting MTBS archive is available without making it model-required.
historical_perimeter_cache_available = (
    historical_perimeter_cache_path.exists()
    and historical_perimeter_cache_path.is_file()
    and historical_perimeter_cache_path.stat().st_size > 0
)

# Explain optional MTBS handling so its absence is not mistaken for a required-source failure.
if historical_perimeter_cache_available:
    print('-> Optional MTBS perimeter cache is available and will be used.')
else:
    print('-> Optional MTBS perimeter cache is not available.')
    print('-> Phase 10 will continue with the required InFORM occurrence source.')

print('=== HISTORICAL FIRE SOURCE ACQUISITION COMPLETE ===')


=== ACQUIRING HISTORICAL FIRE SOURCES ===
-> Reusing cached InFORM fire-occurrence source.
-> InFORM cache path: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\historical_fire\inform_fodr_utah_fire_occurrences_through_2026.geojson
-> Optional MTBS perimeter cache is not available.
-> Phase 10 will continue with the required InFORM occurrence source.
=== HISTORICAL FIRE SOURCE ACQUISITION COMPLETE ===


## Load and Validate Historical Fire Sources

In [126]:
print('=== BUILDING HISTORICAL-FIRE-LIKELIHOOD COMPONENT ===')

# Resolve the required InFORM source only after the Phase 10 acquisition step has populated cache.
historical_occurrence_path = require_fire_hazard_source(
    'INFORM_FODR_FIRE_OCCURRENCES'
)

# Resolve the configured MTBS path without requiring the supporting source to exist.
historical_perimeter_path = fire_hazard_source_paths[
    'MTBS_FIRE_PERIMETERS'
]

# Define the raster output path for the normalized historical-fire-likelihood component.
fire_hazard_historical_fire_path = (
    fire_hazard_historical_directory / 'historical_fire_likelihood.tif'
)

# Load historical fire occurrences for the required point-based occurrence analysis.
occurrences = gpd.read_file(historical_occurrence_path)

# Stop execution if the required InFORM source contains no records for historical analysis.
if occurrences.empty:
    raise ValueError(
        'The required InFORM historical-fire source contains no records.'
    )

# Use the optional MTBS perimeter source when a valid local archive is available.
if (
    historical_perimeter_path.exists()
    and historical_perimeter_path.is_file()
    and historical_perimeter_path.stat().st_size > 0
):
    # Load supporting historical fire perimeters for burn-frequency analysis.
    perimeters = gpd.read_file(historical_perimeter_path)

# Preserve the Phase 10 workflow when optional MTBS data are not available locally.
else:
    # Create an empty spatial layer with the occurrence CRS so later CRS handling remains valid.
    perimeters = gpd.GeoDataFrame(
        geometry=[],
        crs=occurrences.crs,
    )

# Verify that every available historical-fire source has a CRS before spatial alignment.
for name, layer in {
    'occurrences': occurrences,
    'perimeters': perimeters,
}.items():
    # Stop execution if an available historical-fire layer cannot be spatially aligned.
    if layer.crs is None:
        raise ValueError(
            f'The {name} historical-fire layer has no CRS.'
        )


=== BUILDING HISTORICAL-FIRE-LIKELIHOOD COMPONENT ===


## Align and Clip Historical Fire Features

In [127]:
# Reproject occurrences to the common fire-hazard analysis CRS.
occurrences = occurrences.to_crs(fire_hazard_target_crs)
# Reproject perimeters to the common fire-hazard analysis CRS.
perimeters = perimeters.to_crs(fire_hazard_target_crs)

# Use aligned county bounds to limit source features to the study-area extent.
study_bounds = gdf_county_boundaries_aligned.total_bounds
# Retain only occurrence points within the study-area bounding extent.
occurrences = occurrences.cx[
    study_bounds[0]:study_bounds[2],
    study_bounds[1]:study_bounds[3]
].copy()
# Retain only fire perimeters within the study-area bounding extent.
perimeters = perimeters.cx[
    study_bounds[0]:study_bounds[2],
    study_bounds[1]:study_bounds[3]
].copy()


## Rasterize Historical Fire Occurrence and Burn Frequency

In [128]:
# Initialize the occurrence-count raster on the common analysis grid.
occurrence_count = np.zeros(
    (fire_hazard_alignment_height, fire_hazard_alignment_width),
    dtype=np.float32
)

# Rasterize occurrence points only when source records remain after clipping.
if not occurrences.empty:
    # Retain valid point geometries so empty features do not enter rasterization.
    point_shapes = [
        (geom, 1.0)
        for geom in occurrences.geometry
        if geom is not None and (not geom.is_empty)
    ]
    # Accumulate occurrence counts on the aligned raster grid.
    occurrence_count = rasterize(
        point_shapes,
        out_shape=occurrence_count.shape,
        transform=fire_hazard_alignment_transform,
        fill=0.0,
        all_touched=True,
        merge_alg=rasterio.enums.MergeAlg.add,
        dtype='float32'
    )

# Initialize burn-frequency values on the same aligned grid as occurrence counts.
burn_frequency = np.zeros_like(occurrence_count)

# Rasterize historical perimeters only when source records remain after clipping.
if not perimeters.empty:
    # Retain valid perimeter geometries so empty features do not enter rasterization.
    perimeter_shapes = [
        (geom, 1.0)
        for geom in perimeters.geometry
        if geom is not None and (not geom.is_empty)
    ]
    # Accumulate overlapping historical burns to represent repeat-fire frequency.
    burn_frequency = rasterize(
        perimeter_shapes,
        out_shape=burn_frequency.shape,
        transform=fire_hazard_alignment_transform,
        fill=0.0,
        all_touched=True,
        merge_alg=rasterio.enums.MergeAlg.add,
        dtype='float32'
    )


## Smooth Occurrence Density and Define Normalization Logic

In [129]:
# Smooth point counts to represent broader historical occurrence influence where SciPy is available.
try:
    from scipy.ndimage import gaussian_filter

    # Convert the 5-km influence distance to an approximate Gaussian sigma in raster cells.
    sigma_cells = max(1.0, 5000.0 / fire_hazard_cell_size / 3.0)
    # Smooth occurrence counts into a continuous historical occurrence-density surface.
    occurrence_density = gaussian_filter(occurrence_count, sigma=sigma_cells)
# Fall back to unsmoothed counts if Gaussian filtering cannot be performed.
except Exception:
    # Preserve occurrence information so the component can still be normalized.
    occurrence_density = occurrence_count.copy()

# Normalize valid study-area values against a high percentile while preserving nodata cells.
def percentile_scale(values, mask, percentile=99.0):
    # Initialize the normalized result with the workflow nodata value.
    output = np.full(values.shape, fire_hazard_nodata_value, dtype=np.float32)
    # Isolate finite study-area values used to calculate the normalization threshold.
    sample = values[mask & np.isfinite(values)]

    # Return nodata output when no valid sample exists for percentile scaling.
    if sample.size == 0:
        return output

    # Use the configured percentile as the upper scaling threshold.
    upper = float(np.percentile(sample, percentile))

    # Assign zero hazard when the valid sample contains no positive values.
    if upper <= 0:
        output[mask] = 0.0
    else:
        # Scale study-area values to 0-1 and cap values above the percentile threshold.
        output[mask] = np.clip(values[mask] / upper, 0.0, 1.0)

    return output


## Combine and Export Historical Fire Likelihood

In [130]:
# Use the aligned study-area mask to restrict normalization and combination to valid cells.
study_mask = fire_hazard_alignment_study_area_mask
# Normalize smoothed occurrence density to the component's 0-1 scale.
occurrence_hazard = percentile_scale(occurrence_density, study_mask)
# Normalize burn frequency to the component's 0-1 scale.
burn_hazard = percentile_scale(burn_frequency, study_mask)

# Initialize the combined historical-fire raster with nodata outside the study area.
historical_hazard = np.full(
    occurrence_hazard.shape,
    fire_hazard_nodata_value,
    dtype=np.float32
)
# Combine occurrence density and burn frequency using the configured 60/40 weighting.
historical_hazard[study_mask] = (
    occurrence_hazard[study_mask] * 0.6
    + burn_hazard[study_mask] * 0.4
)

# Export the weighted historical-fire-likelihood raster with its component metadata.
write_fire_hazard_array(
    historical_hazard,
    fire_hazard_historical_fire_path,
    'Historical fire likelihood',
    {
        'OCCURRENCE_WEIGHT': 0.6,
        'BURN_FREQUENCY_WEIGHT': 0.4
    }
)


WindowsPath('C:/Users/adamd/Projects/WUI/data/raw/fire_hazard/fuels/fuel_hazard_component/historical_fire_hazard/historical_fire_likelihood.tif')

## Validate, Summarize, and Finalize the Component

In [131]:
# Validate that the exported component satisfies normalized-raster requirements.
historical_validation = validate_normalized_component(
    fire_hazard_historical_fire_path
)
# Record whether the historical-fire component passed all validation checks.
fire_hazard_historical_fire_component_finalized = bool(
    historical_validation['VALID']
)

# Summarize source counts, output location, and validation results for workflow QA.
fire_hazard_historical_summary = pd.DataFrame(
    [
        {
            'COMPONENT_ID': 'HISTORICAL_FIRE_LIKELIHOOD',
            'OCCURRENCE_RECORDS': len(occurrences),
            'PERIMETER_RECORDS': len(perimeters),
            'PRIMARY_OUTPUT': str(fire_hazard_historical_fire_path),
            **historical_validation
        }
    ]
)
# Define the metadata-table path used by later workflow review and validation.
fire_hazard_historical_summary_path = (
    fire_hazard_remaining_metadata
    / 'historical_fire_likelihood_summary.csv'
)
# Export the historical-fire QA summary for downstream phases.
fire_hazard_historical_summary.to_csv(
    fire_hazard_historical_summary_path,
    index=False
)

# Stop execution if the exported historical-fire component failed validation.
if not fire_hazard_historical_fire_component_finalized:
    raise ValueError('The historical-fire component failed validation.')

# Display the final component summary for notebook-level QA review.
display(fire_hazard_historical_summary)
# Announce successful completion of the historical-fire-likelihood workflow.
print('=== HISTORICAL-FIRE-LIKELIHOOD COMPONENT COMPLETE ===')

# Release large intermediate arrays after the component has been finalized.
del occurrence_count, burn_frequency, occurrence_density
del occurrence_hazard, burn_hazard, historical_hazard
# Request garbage collection after removing large raster intermediates.
gc.collect()


,COMPONENT_ID,OCCURRENCE_RECORDS,PERIMETER_RECORDS,PRIMARY_OUTPUT,PATH,FILE_EXISTS,GRID_VALID,STRUCTURE_VALID,VALID_PIXELS,VALID_OUTSIDE_STUDY_AREA,MINIMUM,MAXIMUM,MEAN,VALID
0,HISTORICAL_FIRE_LIKELIHOOD,21123,0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,True,15475232,0,0.0,0.6,0.129745,True


=== HISTORICAL-FIRE-LIKELIHOOD COMPONENT COMPLETE ===


0

# CHRG Phase 11 – Human Ignition

## Purpose

Model human-ignition potential from proximity to roads, developed areas, and other likely human-caused ignition sources.


## Acquire and Cache Human-Ignition Sources

Acquire and assemble the three county-level 2025 TIGER road packages and resolve the Annual NLCD 2025 land-cover raster before human-ignition processing.


In [132]:
print('=== ACQUIRING HUMAN-IGNITION SOURCES ===')

# List the source, cache, grid, and download objects required for Phase 11 acquisition.
required_human_ignition_acquisition_inputs = [
    'fire_hazard_source_paths',
    'fire_hazard_raw_human_ignition_directory',
    'fire_hazard_interim_human_ignition_directory',
    'download_fire_hazard_source_file',
    'fire_hazard_request_session',
    'fire_hazard_request_timeout',
    'fire_hazard_verify_ssl_certificates',
    'fire_hazard_data_mode',
    'refresh_fire_hazard_sources',
    'fire_hazard_target_crs',
    'fire_hazard_alignment_width',
    'fire_hazard_alignment_height',
    'fire_hazard_alignment_transform',
    'fire_hazard_nodata_value',
]

# Identify missing prerequisites before road or land-cover acquisition begins.
missing_human_ignition_acquisition_inputs = [
    object_name
    for object_name in required_human_ignition_acquisition_inputs
    if object_name not in globals()
]

# Stop execution if Phase 11 cannot use the required project configuration.
if missing_human_ignition_acquisition_inputs:
    raise NameError(
        'The following human-ignition acquisition inputs are missing:\n'
        f'{missing_human_ignition_acquisition_inputs}'
    )

# Import archive and temporary-directory utilities used by Phase 11 acquisition.
import gc
import hashlib
import json
import shutil
import tempfile
import warnings
import zipfile
from pathlib import Path

# Import raster reprojection tools used to align Annual NLCD to the project grid.
from rasterio.warp import reproject
from rasterio.enums import Resampling


print(f'-> Fire-hazard data mode: {fire_hazard_data_mode}')
print(
    '-> Cold-start behavior: authoritative reacquisition'
    if refresh_fire_hazard_sources
    else '-> Cached behavior: validated local reuse only'
)

# ----------------------------------------------------
# ACQUIRE TIGER ROAD SOURCES
# ----------------------------------------------------

# Resolve the combined TIGER road cache used by downstream road-proximity analysis.
roads_cache_path = fire_hazard_source_paths[
    'TIGER_ROADS_UTAH'
]

# Define the three 2025 Census county road archives covering the project study area.
tiger_road_downloads = {
    'Salt Lake County': (
        '49035',
        (
            'https://www2.census.gov/geo/tiger/TIGER2025/ROADS/'
            'tl_2025_49035_roads.zip'
        ),
    ),
    'Utah County': (
        '49049',
        (
            'https://www2.census.gov/geo/tiger/TIGER2025/ROADS/'
            'tl_2025_49049_roads.zip'
        ),
    ),
    'Washington County': (
        '49053',
        (
            'https://www2.census.gov/geo/tiger/TIGER2025/ROADS/'
            'tl_2025_49053_roads.zip'
        ),
    ),
}

# Enforce the notebook-wide fire-hazard acquisition contract.
# Refresh mode must reconstruct the source from the authoritative Census endpoints even
# when a local cache exists. Snapshot mode may reuse only an already validated local cache.
roads_refresh_requested = (
    refresh_fire_hazard_sources
    or not roads_cache_path.exists()
)

# Snapshot mode is intentionally offline: a missing combined source is reported rather than
# silently downloaded. A true cold start therefore uses fire_hazard_data_mode='refresh'.
if fire_hazard_data_mode == 'snapshot' and not roads_cache_path.exists():
    raise FileNotFoundError(
        'The combined TIGER road cache is missing while fire_hazard_data_mode is set '
        "to 'snapshot'.\n\nSet fire_hazard_data_mode = 'refresh' and rerun the fire-hazard "
        'configuration and acquisition cells to reconstruct the source from the '
        'authoritative Census TIGER endpoints.'
    )

# Acquire and merge county road archives only when the combined cache must be created.
if roads_refresh_requested:

    # Retain each county road layer so the three sources can be merged consistently.
    tiger_road_layers = []

    # Download each county road package from the authoritative Census TIGER directory.
    for county_name, county_settings in tiger_road_downloads.items():

        # Separate the county FIPS and direct archive URL for the current source.
        county_fips, county_url = county_settings

        # Preserve each original county archive for reproducibility and source QA.
        county_archive_path = (
            fire_hazard_raw_human_ignition_directory
            / f'tl_2025_{county_fips}_roads.zip'
        )

        # Reuse the county archive or acquire it when the local cache is unavailable.
        county_download_record = download_fire_hazard_source_file(
            source_url=county_url,
            destination_path=county_archive_path,
            source_name=f'2025 TIGER roads - {county_name}',
            expected_content_types={
                'application/zip',
                'application/x-zip-compressed',
                'application/octet-stream',
            },
            # This branch runs only during refresh/cold-start reconstruction. Require a
            # fresh authoritative county archive so refresh mode never substitutes old bytes.
            force_refresh=True,
            minimum_size_bytes=1024,
        )

        # Stop if the completed county acquisition fails download validation.
        if not county_download_record['VALID']:
            raise ValueError(
                f'The TIGER road acquisition failed validation for '
                f'{county_name}.'
            )

        # Stop if the downloaded county source is not structurally a ZIP archive.
        if not zipfile.is_zipfile(county_archive_path):
            raise zipfile.BadZipFile(
                f'The TIGER road archive is invalid: '
                f'{county_archive_path.name}'
            )

        # Load the current county road archive for combined study-area preparation.
        county_roads = gpd.read_file(
            f'zip://{county_archive_path}'
        )

        # Stop if the county roads lack a CRS required for spatial alignment.
        if county_roads.crs is None:
            raise ValueError(
                f'The TIGER road source for {county_name} has no CRS.'
            )

        # Align later county layers to the first source CRS before concatenation.
        if (
            tiger_road_layers
            and county_roads.crs != tiger_road_layers[0].crs
        ):
            county_roads = county_roads.to_crs(
                tiger_road_layers[0].crs
            )

        # Retain the validated county layer for the combined road source.
        tiger_road_layers.append(
            county_roads
        )

    # Combine the three county road layers into the single source expected by Phase 11.
    combined_roads = gpd.GeoDataFrame(
        pd.concat(
            tiger_road_layers,
            ignore_index=True,
        ),
        crs=tiger_road_layers[0].crs,
    )

    # Stop if county acquisition produces no usable road features.
    if combined_roads.empty:
        raise ValueError(
            'The combined 2025 TIGER road source contains no features.'
        )

    # Create the road-cache directory before writing the merged source.
    roads_cache_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Use a temporary workspace so partial shapefile components cannot enter the cache.
    with tempfile.TemporaryDirectory(
        dir=str(fire_hazard_interim_human_ignition_directory)
    ) as temporary_road_directory:

        # Convert the temporary workspace to a Path for predictable file handling.
        temporary_road_directory = Path(
            temporary_road_directory
        )

        # Define the temporary merged shapefile written before ZIP creation.
        combined_road_shapefile = (
            temporary_road_directory
            / 'tl_2025_49_roads.shp'
        )

        # Export the three combined county road layers as one shapefile package.
        combined_roads.to_file(
            combined_road_shapefile,
            driver='ESRI Shapefile',
        )

        # Define a temporary ZIP so incomplete output cannot replace the final cache.
        temporary_roads_archive = (
            roads_cache_path.parent
            / (roads_cache_path.name + '.part')
        )

        # Remove a stale temporary road archive left by an interrupted prior run.
        if temporary_roads_archive.exists():
            temporary_roads_archive.unlink()

        # Package every shapefile component required by the combined vector source.
        with zipfile.ZipFile(
            temporary_roads_archive,
            mode='w',
            compression=zipfile.ZIP_DEFLATED,
        ) as roads_archive:

            # Add every merged shapefile sidecar to the completed ZIP package.
            for shapefile_component in temporary_road_directory.glob(
                'tl_2025_49_roads.*'
            ):
                roads_archive.write(
                    shapefile_component,
                    arcname=shapefile_component.name,
                )

        # Reject an empty merged archive before it replaces the project cache.
        if temporary_roads_archive.stat().st_size <= 0:
            raise IOError(
                'The combined TIGER road archive contains no data.'
            )

        # Stop if the completed temporary archive is not structurally valid.
        if not zipfile.is_zipfile(temporary_roads_archive):
            raise zipfile.BadZipFile(
                'The completed combined TIGER road archive is invalid.'
            )

        # Promote the completed road package to its deterministic cache path.
        temporary_roads_archive.replace(
            roads_cache_path
        )

    # Report the completed road source for notebook-level acquisition QA.
    print(
        f'-> Combined TIGER road cache created with '
        f'{len(combined_roads):,} features.'
    )
    print(f'-> TIGER road cache path: {roads_cache_path}')

# Report cache reuse when the combined road archive is already available.
else:
    print('-> Reusing cached combined TIGER road source.')
    print(f'-> TIGER road cache path: {roads_cache_path}')


# ----------------------------------------------------
# ACQUIRE ANNUAL NLCD 2025 LAND COVER
# ----------------------------------------------------

# Resolve the aligned Annual NLCD cache consumed by the developed-land scoring workflow.
nlcd_cache_path = fire_hazard_source_paths[
    'ANNUAL_NLCD_2025_LAND_COVER'
]

# Pin the exact public Annual NLCD release used by this portfolio. The versioned MRLC
# direct-download URL is anonymous HTTPS and therefore requires no account, token, or
# requester-pays cloud configuration.
target_nlcd_year = 2025
target_nlcd_collection = 'C1V2'
target_nlcd_archive_name = (
    'Annual_NLCD_LndCov_2025_CU_C1V2.zip'
)
target_nlcd_direct_url = (
    'https://www.mrlc.gov/downloads/sciweb1/shared/mrlc/data-bundles/'
    'Annual_NLCD_LndCov_2025_CU_C1V2.zip'
)
target_nlcd_doi = '10.5066/P94UXNTS'

# Keep the original version-pinned archive and extracted CONUS source separate from the
# aligned project-grid cache so acquisition, provenance, and derived processing remain
# independently auditable.
nlcd_raw_directory = (
    fire_hazard_raw_human_ignition_directory
    / 'annual_nlcd_2025'
)
nlcd_raw_directory.mkdir(
    parents=True,
    exist_ok=True,
)

nlcd_2025_archive_path = (
    nlcd_raw_directory
    / target_nlcd_archive_name
)

nlcd_2025_source_path = (
    nlcd_raw_directory
    / 'Annual_NLCD_LndCov_2025_CU_C1V2.tif'
)

# Define a compact provenance manifest stored beside the authoritative source files.
nlcd_manifest_path = (
    nlcd_raw_directory
    / 'Annual_NLCD_LndCov_2025_CU_C1V2_manifest.json'
)

# Define the valid categorical class codes published for Annual NLCD Land Cover.
annual_nlcd_valid_classes = {
    11, 12,
    21, 22, 23, 24,
    31,
    41, 42, 43,
    52,
    71,
    81, 82,
    90, 95,
}

# Calculate SHA-256 in chunks so even the large source archive can be verified without
# loading the file into memory.
def calculate_human_ignition_sha256(file_path, chunk_size=8 * 1024 * 1024):

    file_hash = hashlib.sha256()

    with Path(file_path).open('rb') as source_file:
        while True:
            source_chunk = source_file.read(chunk_size)
            if not source_chunk:
                break
            file_hash.update(source_chunk)

    return file_hash.hexdigest()


# Validate an existing aligned NLCD cache before deciding whether any source processing
# is required. Validation is used for snapshot mode; refresh mode deliberately reconstructs
# the source from the pinned authoritative release even when a valid local cache exists.
def validate_aligned_annual_nlcd_cache(candidate_path):

    candidate_path = Path(candidate_path)

    if not candidate_path.exists() or not candidate_path.is_file():
        return False, 'cache file is missing'

    if candidate_path.stat().st_size < 1024:
        return False, 'cache file is empty or incomplete'

    try:
        with rasterio.open(candidate_path) as candidate_raster:

            if candidate_raster.count != 1:
                return False, 'cache does not contain exactly one raster band'

            if candidate_raster.width != fire_hazard_alignment_width:
                return False, 'cache width does not match the common project grid'

            if candidate_raster.height != fire_hazard_alignment_height:
                return False, 'cache height does not match the common project grid'

            if candidate_raster.crs != rasterio.crs.CRS.from_user_input(
                fire_hazard_target_crs
            ):
                return False, 'cache CRS does not match the common project grid'

            if not candidate_raster.transform.almost_equals(
                fire_hazard_alignment_transform
            ):
                return False, 'cache transform does not match the common project grid'

            # Read the categorical cache under the notebook-wide, exact-message
            # Rasterio/NumPy 2.5 compatibility policy configured during project setup.
            candidate_array = candidate_raster.read(1)
            candidate_nodata = candidate_raster.nodata

    except Exception as cache_validation_error:
        return False, f'cache could not be opened: {cache_validation_error}'

    # Build the valid-data mask using the raster NoData value when one is published.
    if candidate_nodata is None:
        candidate_valid_mask = np.ones(
            candidate_array.shape,
            dtype=bool,
        )
    else:
        candidate_valid_mask = (
            candidate_array != candidate_nodata
        )

    if not candidate_valid_mask.any():
        return False, 'cache contains no valid land-cover pixels'

    candidate_classes = set(
        np.unique(
            candidate_array[candidate_valid_mask]
        ).astype(int).tolist()
    )

    unexpected_classes = (
        candidate_classes
        - annual_nlcd_valid_classes
    )

    if unexpected_classes:
        return (
            False,
            f'cache contains unexpected Annual NLCD classes: '
            f'{sorted(unexpected_classes)}',
        )

    return True, 'validated'


# Validate the aligned cache independently from acquisition mode so cached execution can
# be strictly offline while refresh execution can deliberately rebuild from source.
nlcd_cache_valid, nlcd_cache_validation_message = (
    validate_aligned_annual_nlcd_cache(
        nlcd_cache_path
    )
)

if fire_hazard_data_mode == 'snapshot':
    if not nlcd_cache_valid:
        raise FileNotFoundError(
            'The Annual NLCD project-grid cache is unavailable or invalid while '
            "fire_hazard_data_mode is set to 'snapshot'.\n\n"
            f'Validation result: {nlcd_cache_validation_message}\n\n'
            "Set fire_hazard_data_mode = 'refresh' and rerun the fire-hazard "
            'configuration and acquisition cells to reconstruct the source from the '
            'pinned MRLC release.'
        )

    print(
        '-> Reusing validated Annual NLCD 2025 C1V2 project-grid cache.'
    )
    print(f'-> Annual NLCD cache path: {nlcd_cache_path}')

else:
    print(
        '-> Annual NLCD aligned cache requires creation: '
        f'{nlcd_cache_validation_message}'
    )
    print(
        '-> Resolving Annual NLCD Collection 1.2 Land Cover through '
        'the public MRLC direct-download endpoint.'
    )
    print(f'-> Annual NLCD release: {target_nlcd_year} {target_nlcd_collection}')
    print(f'-> Annual NLCD DOI: {target_nlcd_doi}')
    print(f'-> Annual NLCD direct URL: {target_nlcd_direct_url}')

    # Validate any existing version-pinned source archive before using it. ZIP structure
    # is a stronger gate than HTTP headers because it rejects HTML/error payloads even
    # when a server reports a misleading content type.
    nlcd_archive_valid = (
        nlcd_2025_archive_path.exists()
        and nlcd_2025_archive_path.is_file()
        and nlcd_2025_archive_path.stat().st_size >= 1024
        and zipfile.is_zipfile(nlcd_2025_archive_path)
    )

    # Refresh mode always requests a fresh copy of the exact pinned MRLC release. The
    # common downloader writes through a temporary file and replaces the cache only after
    # validation, preserving the previous valid archive if the network request fails.
    nlcd_download_record = download_fire_hazard_source_file(
            source_url=target_nlcd_direct_url,
            destination_path=nlcd_2025_archive_path,
            source_name=(
                'Annual NLCD 2025 Collection 1.2 Land Cover - MRLC direct download'
            ),
            expected_content_types={
                'application/zip',
                'application/x-zip-compressed',
                'application/octet-stream',
            },
            force_refresh=True,
            minimum_size_bytes=100 * 1024 * 1024,
        )

    # Stop immediately if the common project download helper rejected the transfer.
    if not nlcd_download_record['VALID']:
        raise ValueError(
            'The public MRLC Annual NLCD 2025 source failed download validation.'
        )

    # Require a real ZIP archive so redirects, maintenance pages, or HTML responses
    # cannot enter the reproducible source cache.
    if not zipfile.is_zipfile(nlcd_2025_archive_path):

        with nlcd_2025_archive_path.open('rb') as invalid_nlcd_source:
            invalid_nlcd_sample = invalid_nlcd_source.read(160)

        raise RuntimeError(
            'The MRLC direct-download endpoint did not return a valid '
            'Annual NLCD ZIP archive.\n\n'
            f'URL: {target_nlcd_direct_url}\n'
            f'First response bytes: {invalid_nlcd_sample!r}'
        )

    print(
        f'-> Annual NLCD source ZIP size: '
        f'{nlcd_2025_archive_path.stat().st_size / 1024 ** 3:,.2f} GB'
    )

    # Open the version-pinned archive and identify the primary 2025 Land Cover raster.
    with zipfile.ZipFile(
        nlcd_2025_archive_path,
        mode='r',
    ) as nlcd_2025_archive:

        # Run ZIP CRC validation before the authoritative source is extracted.
        corrupt_nlcd_member = nlcd_2025_archive.testzip()

        if corrupt_nlcd_member is not None:
            raise zipfile.BadZipFile(
                f'The Annual NLCD source ZIP contains a corrupt member: '
                f'{corrupt_nlcd_member}'
            )

        nlcd_archive_members = nlcd_2025_archive.namelist()

        nlcd_2025_raster_members = [
            member_name
            for member_name in nlcd_archive_members
            if (
                member_name.lower().endswith(('.tif', '.tiff'))
                and '2025' in member_name.lower()
                and (
                    'lndcov' in member_name.lower()
                    or 'landcover' in member_name.lower()
                    or 'land_cover' in member_name.lower()
                    or 'land cover' in member_name.lower()
                )
                and '.aux.tif' not in member_name.lower()
            )
        ]

        if not nlcd_2025_raster_members:
            raise FileNotFoundError(
                'The Annual NLCD 2025 C1V2 archive does not contain a '
                'recognizable 2025 Land Cover GeoTIFF.'
            )

        # Prefer the shortest matching path so an unexpected nested auxiliary copy does
        # not supersede the primary published raster.
        nlcd_2025_raster_member = sorted(
            nlcd_2025_raster_members,
            key=len,
        )[0]

        print(
            f'-> Annual NLCD raster selected from ZIP: '
            f'{nlcd_2025_raster_member}'
        )

        # Refresh/cold-start reconstruction always extracts the source GeoTIFF from the
        # newly downloaded authoritative archive so stale extracted bytes cannot survive a
        # refresh operation.
        temporary_nlcd_extraction_path = (
                nlcd_2025_source_path.parent
                / (nlcd_2025_source_path.name + '.part')
            )

        if temporary_nlcd_extraction_path.exists():
            temporary_nlcd_extraction_path.unlink()

        # Stream only the selected GeoTIFF member to disk; no archive member is loaded
        # wholly into memory.
        with nlcd_2025_archive.open(
                nlcd_2025_raster_member,
                mode='r',
        ) as archived_nlcd_raster:

            with temporary_nlcd_extraction_path.open('wb') as extracted_nlcd_file:
                shutil.copyfileobj(
                    archived_nlcd_raster,
                    extracted_nlcd_file,
                    length=8 * 1024 * 1024,
                )

        if temporary_nlcd_extraction_path.stat().st_size < 1024:
            temporary_nlcd_extraction_path.unlink(missing_ok=True)
            raise IOError(
                'The extracted Annual NLCD 2025 source is empty or incomplete.'
            )

        # Validate the raster before promoting it into the raw-data cache.
        with rasterio.open(temporary_nlcd_extraction_path) as nlcd_source_validation:
            if nlcd_source_validation.count < 1:
                raise ValueError(
                    'The extracted Annual NLCD source contains no raster bands.'
                )
            if nlcd_source_validation.crs is None:
                raise ValueError(
                    'The extracted Annual NLCD source does not contain a CRS.'
                )

        temporary_nlcd_extraction_path.replace(
            nlcd_2025_source_path
        )

        print(
            f'-> Extracted Annual NLCD source size: '
            f'{nlcd_2025_source_path.stat().st_size / 1024 ** 3:,.2f} GB'
        )

    # Calculate strong source hashes after successful acquisition/extraction. These
    # hashes are recorded as provenance so future runs can demonstrate exactly which
    # immutable source bytes generated the project-grid raster.
    print('-> Calculating Annual NLCD source SHA-256 checksums...')
    nlcd_archive_sha256 = calculate_human_ignition_sha256(
        nlcd_2025_archive_path
    )
    nlcd_source_sha256 = calculate_human_ignition_sha256(
        nlcd_2025_source_path
    )

    # Open the authoritative categorical source and align it directly onto the common
    # fire-hazard grid. Rasterio/GDAL reads only the source windows required by the
    # destination transform instead of loading the CONUS raster into a NumPy array.
    with rasterio.open(
        nlcd_2025_source_path
    ) as nlcd_source_raster:

        if nlcd_source_raster.crs is None:
            raise ValueError(
                'The Annual NLCD 2025 source raster does not contain a CRS.'
            )

        if nlcd_source_raster.count < 1:
            raise ValueError(
                'The Annual NLCD 2025 source raster contains no bands.'
            )

        nlcd_source_nodata = nlcd_source_raster.nodata

        aligned_nlcd_2025 = np.full(
            (
                fire_hazard_alignment_height,
                fire_hazard_alignment_width,
            ),
            int(fire_hazard_nodata_value),
            dtype=np.int16,
        )

        # Nearest-neighbor resampling is mandatory for the categorical NLCD class codes.
        reproject(
            source=rasterio.band(
                nlcd_source_raster,
                1,
            ),
            destination=aligned_nlcd_2025,
            src_transform=nlcd_source_raster.transform,
            src_crs=nlcd_source_raster.crs,
            src_nodata=nlcd_source_nodata,
            dst_transform=fire_hazard_alignment_transform,
            dst_crs=fire_hazard_target_crs,
            dst_nodata=int(fire_hazard_nodata_value),
            resampling=Resampling.nearest,
        )

    # Identify valid categorical cells after project-grid reprojection.
    nlcd_aligned_valid = (
        aligned_nlcd_2025
        != int(fire_hazard_nodata_value)
    )

    nlcd_project_valid_pixel_count = int(
        nlcd_aligned_valid.sum()
    )

    if nlcd_project_valid_pixel_count == 0:
        raise ValueError(
            'Annual NLCD 2025 produced no valid cells on the project grid.'
        )

    # Confirm the reprojection preserved only documented Annual NLCD class values.
    nlcd_project_classes = set(
        np.unique(
            aligned_nlcd_2025[nlcd_aligned_valid]
        ).astype(int).tolist()
    )

    nlcd_unexpected_project_classes = (
        nlcd_project_classes
        - annual_nlcd_valid_classes
    )

    if nlcd_unexpected_project_classes:
        raise ValueError(
            'Annual NLCD reprojection produced unexpected categorical values: '
            f'{sorted(nlcd_unexpected_project_classes)}'
        )

    # Create the final cache directory before writing the aligned source.
    nlcd_cache_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    nlcd_output_profile = {
        'driver': 'GTiff',
        'height': fire_hazard_alignment_height,
        'width': fire_hazard_alignment_width,
        'count': 1,
        'dtype': 'int16',
        'crs': fire_hazard_target_crs,
        'transform': fire_hazard_alignment_transform,
        'nodata': int(fire_hazard_nodata_value),
        'compress': 'lzw',
    }

    # Write through a temporary GeoTIFF so an interrupted process cannot leave a partial
    # raster at the catalog path used by downstream human-ignition calculations.
    temporary_nlcd_cache_path = (
        nlcd_cache_path.parent
        / (nlcd_cache_path.name + '.part.tif')
    )

    if temporary_nlcd_cache_path.exists():
        temporary_nlcd_cache_path.unlink()

    with rasterio.open(
        temporary_nlcd_cache_path,
        mode='w',
        **nlcd_output_profile,
    ) as nlcd_output_raster:

        nlcd_output_raster.write(
            aligned_nlcd_2025,
            1,
        )

        nlcd_output_raster.update_tags(
            SOURCE='USGS Annual NLCD Collection 1.2 via MRLC Direct Download',
            SOURCE_YEAR=str(target_nlcd_year),
            SOURCE_COLLECTION=target_nlcd_collection,
            SOURCE_DOI=target_nlcd_doi,
            SOURCE_URL=target_nlcd_direct_url,
            SOURCE_ARCHIVE=target_nlcd_archive_name,
            SOURCE_ARCHIVE_SHA256=nlcd_archive_sha256,
            SOURCE_RASTER_SHA256=nlcd_source_sha256,
            RESAMPLING_METHOD='nearest',
            PROJECT_GRID_CRS=str(fire_hazard_target_crs),
        )

    temporary_nlcd_cache_path.replace(
        nlcd_cache_path
    )

    del aligned_nlcd_2025

    # Validate the completed cache using the same deterministic QA gate used at startup.
    nlcd_cache_valid, nlcd_cache_validation_message = (
        validate_aligned_annual_nlcd_cache(
            nlcd_cache_path
        )
    )

    if not nlcd_cache_valid:
        raise ValueError(
            'The aligned Annual NLCD 2025 cache failed final validation: '
            f'{nlcd_cache_validation_message}'
        )

    # Create a machine-readable provenance manifest without making it a prerequisite for
    # later runs. The source hashes are retained so another analyst can independently
    # verify the exact version-pinned input bytes.
    nlcd_manifest = {
        'source_agency': 'U.S. Geological Survey / MRLC',
        'dataset': 'Annual NLCD Land Cover',
        'collection': target_nlcd_collection,
        'year': target_nlcd_year,
        'doi': target_nlcd_doi,
        'access_method': 'Anonymous MRLC HTTPS direct download',
        'source_url': target_nlcd_direct_url,
        'archive_filename': target_nlcd_archive_name,
        'archive_sha256': nlcd_archive_sha256,
        'source_raster_filename': nlcd_2025_source_path.name,
        'source_raster_sha256': nlcd_source_sha256,
        'aligned_cache_path': str(nlcd_cache_path),
        'project_crs': str(fire_hazard_target_crs),
        'resampling': 'nearest',
        'valid_land_cover_classes': sorted(annual_nlcd_valid_classes),
    }

    with nlcd_manifest_path.open(
        'w',
        encoding='utf-8',
    ) as nlcd_manifest_file:
        json.dump(
            nlcd_manifest,
            nlcd_manifest_file,
            indent=2,
        )

    print(
        f'-> Annual NLCD valid project-grid pixels: '
        f'{nlcd_project_valid_pixel_count:,}'
    )
    print(
        f'-> Annual NLCD project classes: '
        f'{sorted(nlcd_project_classes)}'
    )
    print(f'-> Annual NLCD cache path: {nlcd_cache_path}')
    print(f'-> Annual NLCD provenance manifest: {nlcd_manifest_path}')
    print('-> Annual NLCD 2025 C1V2 project-grid validation passed.')

# Release unused temporary objects before downstream raster processing begins.
gc.collect()

# Confirm that both required Phase 11 source caches now exist locally.
human_ignition_source_cache_complete = all(
    [
        roads_cache_path.exists(),
        nlcd_cache_path.exists(),
    ]
)

# Stop if either required Human-Ignition source is unavailable after acquisition.
if not human_ignition_source_cache_complete:
    raise FileNotFoundError(
        'One or more required human-ignition source caches are unavailable '
        'after Phase 11 acquisition.'
    )

# Report successful completion of the Phase 11 source acquisition stage.
print('\n=== HUMAN-IGNITION SOURCE ACQUISITION COMPLETE ===')

=== ACQUIRING HUMAN-IGNITION SOURCES ===
-> Fire-hazard data mode: snapshot
-> Cached behavior: validated local reuse only
-> Reusing cached combined TIGER road source.
-> TIGER road cache path: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\human_ignition\tl_2025_49_roads.zip
-> Reusing validated Annual NLCD 2025 C1V2 project-grid cache.
-> Annual NLCD cache path: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\human_ignition\annual_nlcd_c1_2_2025_land_cover_conus.tif

=== HUMAN-IGNITION SOURCE ACQUISITION COMPLETE ===


## Human-Ignition-Potential Component


### Configure Human-Ignition Sources and Output


In [133]:
print('=== BUILDING HUMAN-IGNITION-POTENTIAL COMPONENT ===')

# Resolve the required roads source used to model ignition potential near transportation corridors.
roads_source_path = require_fire_hazard_source('TIGER_ROADS_UTAH')
# Resolve the required NLCD source used to represent development-related ignition potential.
nlcd_source_path = require_fire_hazard_source('ANNUAL_NLCD_2025_LAND_COVER')

# Define the raster output path for the completed human-ignition-potential component.
fire_hazard_human_ignition_path = (
    fire_hazard_human_directory / 'human_ignition_potential.tif'
)


=== BUILDING HUMAN-IGNITION-POTENTIAL COMPONENT ===


### Derive Road-Proximity Ignition Potential


In [134]:
# Load Utah roads for spatial preparation on the common fire-hazard analysis grid.
roads = gpd.read_file(roads_source_path)

# Stop execution if the roads layer lacks the CRS required for reliable reprojection.
if roads.crs is None:
    raise ValueError('The TIGER roads source has no CRS.')

# Reproject roads to the common project CRS before rasterization and distance calculations.
roads = roads.to_crs(fire_hazard_target_crs)

# Use the aligned county extent to limit road processing to the project study area.
study_bounds = gdf_county_boundaries_aligned.total_bounds
# Subset roads to the study-area bounds to reduce unnecessary rasterization work.
roads = roads.cx[
    study_bounds[0]:study_bounds[2],
    study_bounds[1]:study_bounds[3],
].copy()

# Rasterize road locations on the common grid so cell distances can be calculated consistently.
road_presence = rasterize(
    [
        (geom, 1)
        # Evaluate each road geometry so all valid transportation features contribute.
        for geom in roads.geometry
        # Exclude missing or empty geometries that cannot represent road presence.
        if geom is not None and (not geom.is_empty)
    ],
    out_shape=(fire_hazard_alignment_height, fire_hazard_alignment_width),
    transform=fire_hazard_alignment_transform,
    fill=0,
    all_touched=True,
    dtype='uint8',
)

# Calculate each grid cell's distance to the nearest road in project distance units.
road_distance_m = (
    distance_transform_edt(road_presence == 0) * fire_hazard_cell_size
)
# Convert road distance to a 0-1 ignition score that declines to zero at 5 kilometers.
road_hazard = np.clip(
    1.0 - road_distance_m / 5000.0,
    0.0,
    1.0,
).astype(np.float32)


### Align NLCD and Score Developed Land


In [135]:
# Initialize the aligned NLCD grid with nodata values before reprojection.
aligned_nlcd = np.full(
    (fire_hazard_alignment_height, fire_hazard_alignment_width),
    -9999,
    dtype=np.int16,
)

# Keep the NLCD dataset open only while projecting its land-cover classes to the project grid.
with rasterio.open(nlcd_source_path) as nlcd_source:
    # Preserve categorical NLCD classes while aligning them to the common 30-meter grid.
    reproject(
        source=rasterio.band(nlcd_source, 1),
        destination=aligned_nlcd,
        src_transform=nlcd_source.transform,
        src_crs=nlcd_source.crs,
        src_nodata=nlcd_source.nodata,
        dst_transform=fire_hazard_alignment_transform,
        dst_crs=fire_hazard_target_crs,
        dst_nodata=-9999,
        resampling=Resampling.nearest,
    )

# Assign increasing ignition scores to NLCD developed classes based on development intensity.
developed_lookup = {21: 0.25, 22: 0.5, 23: 0.75, 24: 1.0}
# Initialize development-related ignition scores for all aligned cells.
developed_hazard = np.zeros(aligned_nlcd.shape, dtype=np.float32)

# Apply each development score to the corresponding NLCD class on the aligned grid.
for class_code, score in developed_lookup.items():
    # Populate development-related ignition potential for cells in the current NLCD class.
    developed_hazard[aligned_nlcd == class_code] = score


### Combine Human-Ignition Factors and Write Raster


In [136]:
# Initialize the combined component with project nodata values outside the study area.
human_hazard = np.full(
    road_hazard.shape,
    fire_hazard_nodata_value,
    dtype=np.float32,
)
# Reuse the aligned study-area mask so scoring is restricted to valid project cells.
study_mask = fire_hazard_alignment_study_area_mask

# Combine road proximity and developed land using the configured 65/35 model weights.
human_hazard[study_mask] = (
    road_hazard[study_mask] * 0.65
    + developed_hazard[study_mask] * 0.35
)

# Write the normalized human-ignition component with its weighting metadata for later phases.
write_fire_hazard_array(
    human_hazard,
    fire_hazard_human_ignition_path,
    'Human ignition potential',
    {
        'ROAD_PROXIMITY_WEIGHT': 0.65,
        'DEVELOPED_LAND_WEIGHT': 0.35,
    },
)


WindowsPath('C:/Users/adamd/Projects/WUI/data/raw/fire_hazard/fuels/fuel_hazard_component/human_ignition_hazard/human_ignition_potential.tif')

### Validate Component and Export Summary


In [137]:
# Validate the output raster before marking the human-ignition component as complete.
human_validation = validate_normalized_component(
    fire_hazard_human_ignition_path
)
# Record whether the component satisfies all normalized-raster validation requirements.
fire_hazard_human_ignition_component_finalized = bool(
    human_validation['VALID']
)

# Summarize the road input count, output location, and validation metrics for QA records.
fire_hazard_human_summary = pd.DataFrame(
    [
        {
            'COMPONENT_ID': 'HUMAN_IGNITION_POTENTIAL',
            'ROAD_FEATURES': len(roads),
            'PRIMARY_OUTPUT': str(fire_hazard_human_ignition_path),
            **human_validation,
        }
    ]
)
# Define the metadata-table path used to document the completed component.
fire_hazard_human_summary_path = (
    fire_hazard_remaining_metadata / 'human_ignition_potential_summary.csv'
)
# Export the QA summary so later phases can inspect the component's validation results.
fire_hazard_human_summary.to_csv(
    fire_hazard_human_summary_path,
    index=False,
)


### Enforce Final QA and Release Temporary Arrays


In [138]:
# Stop execution if validation shows the human-ignition component is not analysis-ready.
if not fire_hazard_human_ignition_component_finalized:
    raise ValueError('The human-ignition component failed validation.')

# Display the component summary for an immediate notebook-level QA review.
display(fire_hazard_human_summary)
print('=== HUMAN-IGNITION-POTENTIAL COMPONENT COMPLETE ===')

# Release road-proximity arrays after their values have been written and validated.
del road_presence, road_distance_m, road_hazard
# Release aligned land-cover and combined hazard arrays after component completion.
del aligned_nlcd, developed_hazard, human_hazard
# Request garbage collection to recover memory before later notebook phases run.
gc.collect()


,COMPONENT_ID,ROAD_FEATURES,PRIMARY_OUTPUT,PATH,FILE_EXISTS,GRID_VALID,STRUCTURE_VALID,VALID_PIXELS,VALID_OUTSIDE_STUDY_AREA,MINIMUM,MAXIMUM,MEAN,VALID
0,HUMAN_IGNITION_POTENTIAL,87291,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True,True,True,15475232,0,0.0,1.0,0.592958,True


=== HUMAN-IGNITION-POTENTIAL COMPONENT COMPLETE ===


0

# CHRG Phase 12 – Composite Hazard Grid

## Purpose

Validate the finalized component rasters, apply the configured weights, create the Composite Fire Hazard Grid, classify the continuous hazard scores, and export final summaries.


## Final Composite Fire Hazard Grid

### Configure and Validate Final Fire-Hazard Components


In [139]:
print('=== CONFIGURING FINAL COMPOSITE FIRE-HAZARD GRID ===')

# Map each finalized hazard component to the raster used in the composite calculation.
fire_hazard_final_component_paths = {
    'fuel_hazard': Path(fire_hazard_fuel_hazard_path),
    'fuel_dryness': Path(fire_hazard_vegetation_dryness_hazard_path),
    'slope_hazard': Path(terrain_slope_hazard_path),
    'aspect_hazard': Path(terrain_aspect_hazard_path),
    'historical_fire_likelihood': Path(fire_hazard_historical_fire_path),
    'human_ignition_potential': Path(fire_hazard_human_ignition_path),
}

# Retrieve the configured weights so every component contributes its intended model influence.
fire_hazard_final_component_weights = {
    key: float(fire_hazard_components[key]['weight'])
    for key in fire_hazard_final_component_paths
}

# Sum component weights to verify that the composite model is fully allocated.
final_weight_sum = sum(fire_hazard_final_component_weights.values())

# Stop execution if the final component weights do not sum to the required total of 1.0.
if not np.isclose(final_weight_sum, 1.0, atol=1e-09):
    raise ValueError(
        f'Final fire-hazard weights sum to {final_weight_sum}, not 1.0.'
    )

# Validate every component raster before combining them into the final hazard surface.
for component_id, component_path in fire_hazard_final_component_paths.items():
    # Confirm the component satisfies the normalized-raster requirements used by the model.
    validation = validate_normalized_component(component_path)

    # Stop execution if any component fails validation required for reliable compositing.
    if not validation['VALID']:
        raise ValueError(
            f'Final component is not valid: {component_id}\n{validation}'
        )


=== CONFIGURING FINAL COMPOSITE FIRE-HAZARD GRID ===


### Prepare Composite-Grid Outputs and Weight Metadata

In [140]:
# Define final raster outputs for normalized hazard, 1-10 index, and hazard class.
fire_hazard_final_normalized_path = (
    fire_hazard_composite_directory / 'composite_fire_hazard_normalized.tif'
)
fire_hazard_final_index_path = (
    fire_hazard_composite_directory / 'composite_fire_hazard_index_1_10.tif'
)
fire_hazard_final_class_path = (
    fire_hazard_composite_directory / 'composite_fire_hazard_class.tif'
)

# Record that the final component configuration has passed its setup requirements.
fire_hazard_final_configuration_complete = True

# Create a component-weight table that documents the exact inputs used by the final model.
fire_hazard_final_weight_table = pd.DataFrame(
    [
        {
            'COMPONENT_ID': key,
            'COMPONENT_NAME': fire_hazard_components[key]['component_name'],
            'WEIGHT': fire_hazard_final_component_weights[key],
            'INPUT_PATH': str(fire_hazard_final_component_paths[key]),
        }
        for key in fire_hazard_final_component_paths
    ]
)

# Define the metadata destination for the reproducible component-weight table.
fire_hazard_final_weight_table_path = (
    fire_hazard_remaining_metadata / 'composite_fire_hazard_weights.csv'
)

# Export the component weights so later phases can verify how the composite was constructed.
fire_hazard_final_weight_table.to_csv(
    fire_hazard_final_weight_table_path,
    index=False,
)

# Display the final component weights for notebook-level QA.
display(fire_hazard_final_weight_table)

print('=== FINAL COMPOSITE FIRE-HAZARD GRID CONFIGURED ===')


,COMPONENT_ID,COMPONENT_NAME,WEIGHT,INPUT_PATH
0,fuel_hazard,Surface and Canopy Fuel Hazard,0.30,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
1,fuel_dryness,Vegetation and Fuel Dryness,0.20,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
2,slope_hazard,Slope-Based Spread Potential,0.15,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
3,aspect_hazard,Aspect and Solar Exposure,0.10,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
4,historical_fire_likelihood,Historical Fire Likelihood,0.15,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...
5,human_ignition_potential,Human Ignition Potential,0.10,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...


=== FINAL COMPOSITE FIRE-HAZARD GRID CONFIGURED ===


### Build the Normalized Composite Fire-Hazard Raster

In [141]:
print('=== BUILDING FINAL COMPOSITE FIRE-HAZARD GRID ===')

# Use a temporary raster so an incomplete build cannot overwrite the finalized output.
temporary_normalized_path = fire_hazard_final_normalized_path.with_name(
    fire_hazard_final_normalized_path.stem + '.part.tif'
)

# Remove a stale temporary raster before starting a new composite build.
if temporary_normalized_path.exists():
    temporary_normalized_path.unlink()

# Open all component rasters together so aligned windows can be combined consistently.
with ExitStack() as stack:
    # Keep each component raster open for repeated windowed reads during compositing.
    component_sources = {
        key: stack.enter_context(rasterio.open(path))
        for key, path in fire_hazard_final_component_paths.items()
    }

    # Create the temporary normalized output using the established component raster profile.
    destination = stack.enter_context(
        rasterio.open(
            temporary_normalized_path,
            'w',
            **fire_hazard_component_profile,
        )
    )

    # Use fuel hazard as the reference grid because all components are aligned to this raster.
    reference_source = component_sources['fuel_hazard']

    # Count raster blocks so progress can be reported during memory-controlled processing.
    total_windows = sum(
        1
        for _ in reference_source.block_windows(1)
    )

    # Process one aligned raster window at a time to limit memory use at full resolution.
    for window_number, (_, window) in enumerate(
        reference_source.block_windows(1),
        start=1,
    ):
        # Convert the window offsets to array indices for the matching study-area mask subset.
        row0, col0 = (
            int(window.row_off),
            int(window.col_off),
        )
        row1, col1 = (
            row0 + int(window.height),
            col0 + int(window.width),
        )

        # Restrict calculations to cells inside the configured study area.
        study_mask = fire_hazard_alignment_study_area_mask[
            row0:row1,
            col0:col1,
        ]

        # Read matching windows from every component so cell-by-cell weighting stays aligned.
        arrays = {
            key: source.read(1, window=window)
            for key, source in component_sources.items()
        }

        # Begin with the study-area mask before excluding invalid component cells.
        valid = study_mask.copy()

        # Require valid data from every component before a cell contributes to the composite.
        for key, values in arrays.items():
            # Retrieve component-specific nodata metadata for the current validity test.
            source = component_sources[key]

            # Exclude nonfinite and nodata cells so weighted scores use complete component data.
            valid &= np.isfinite(values) & (values != source.nodata)

        # Initialize the output window with nodata outside valid composite cells.
        output = np.full(
            arrays['fuel_hazard'].shape,
            fire_hazard_nodata_value,
            dtype=np.float32,
        )

        # Calculate weighted hazard only where every required component is valid.
        if valid.any():
            # Accumulate weighted values in float64 before writing the final float32 score.
            combined = np.zeros(
                int(valid.sum()),
                dtype=np.float64,
            )

            # Add each normalized component according to its configured final model weight.
            for key, values in arrays.items():
                combined += (
                    values[valid] * fire_hazard_final_component_weights[key]
                )

            # Constrain weighted scores to the normalized 0-1 model range.
            output[valid] = np.clip(
                combined,
                0.0,
                1.0,
            ).astype(np.float32)

        # Write the completed window to its matching location in the temporary raster.
        destination.write(
            output,
            1,
            window=window,
        )

        # Report progress at the first, every hundredth, and final processing window.
        if (
            window_number == 1
            or window_number % 100 == 0
            or window_number == total_windows
        ):
            print(
                f'-> Processed window {window_number:,} of {total_windows:,}'
            )

    # Label the raster band so the normalized composite is identifiable in downstream use.
    destination.set_band_description(
        1,
        'Normalized Composite Fire Hazard',
    )

    # Store model metadata with the normalized raster for reproducibility and QA.
    destination.update_tags(
        MODEL_NAME=fire_hazard_model_name,
        MODEL_VERSION=fire_hazard_model_version,
        WEIGHT_SUM=final_weight_sum,
        SCORE_MINIMUM=0.0,
        SCORE_MAXIMUM=1.0,
    )

# Remove an older finalized raster before promoting the successfully completed temporary output.
if fire_hazard_final_normalized_path.exists():
    fire_hazard_final_normalized_path.unlink()

# Promote the completed temporary raster to the finalized normalized-hazard path.
temporary_normalized_path.replace(fire_hazard_final_normalized_path)


=== BUILDING FINAL COMPOSITE FIRE-HAZARD GRID ===
-> Processed window 1 of 2,109
-> Processed window 100 of 2,109
-> Processed window 200 of 2,109
-> Processed window 300 of 2,109
-> Processed window 400 of 2,109
-> Processed window 500 of 2,109
-> Processed window 600 of 2,109
-> Processed window 700 of 2,109
-> Processed window 800 of 2,109
-> Processed window 900 of 2,109
-> Processed window 1,000 of 2,109
-> Processed window 1,100 of 2,109
-> Processed window 1,200 of 2,109
-> Processed window 1,300 of 2,109
-> Processed window 1,400 of 2,109
-> Processed window 1,500 of 2,109
-> Processed window 1,600 of 2,109
-> Processed window 1,700 of 2,109
-> Processed window 1,800 of 2,109
-> Processed window 1,900 of 2,109
-> Processed window 2,000 of 2,109
-> Processed window 2,100 of 2,109
-> Processed window 2,109 of 2,109


WindowsPath('C:/Users/adamd/Projects/WUI/data/raw/fire_hazard/fuels/fuel_hazard_component/composite_fire_hazard/composite_fire_hazard_normalized.tif')

### Convert Normalized Hazard to Index and Class Rasters

In [142]:
# Read the finalized normalized raster to derive the 1-10 index and categorical classes.
with rasterio.open(fire_hazard_final_normalized_path) as normalized_source:
    # Load normalized hazard values for the final scoring transformations.
    normalized = normalized_source.read(1)

    # Identify finite, non-nodata cells eligible for index and class assignment.
    valid = (
        np.isfinite(normalized)
        & (normalized != fire_hazard_nodata_value)
    )

# Initialize the continuous hazard index with nodata outside valid analysis cells.
hazard_index = np.full(
    normalized.shape,
    fire_hazard_nodata_value,
    dtype=np.float32,
)

# Scale normalized hazard from 0-1 to the configured continuous 1-10 index.
hazard_index[valid] = 1.0 + normalized[valid] * 9.0

# Export the continuous 1-10 Composite Fire Hazard Index.
write_fire_hazard_array(
    hazard_index,
    fire_hazard_final_index_path,
    'Composite Fire Hazard Index (1-10)',
)

# Initialize categorical hazard classes with nodata outside valid analysis cells.
hazard_class = np.full(
    normalized.shape,
    fire_hazard_nodata_value,
    dtype=np.float32,
)

# Classify normalized hazard into the five configured fire-hazard severity categories.
hazard_class[valid] = np.digitize(
    normalized[valid],
    bins=[0.2, 0.4, 0.6, 0.8],
    right=True,
).astype(np.float32) + 1.0

# Export the classified hazard raster with labels for each severity category.
write_fire_hazard_array(
    hazard_class,
    fire_hazard_final_class_path,
    'Composite Fire Hazard Class',
    {
        '1': 'Very Low',
        '2': 'Low',
        '3': 'Moderate',
        '4': 'High',
        '5': 'Very High',
    },
)

# Record that all three final composite hazard rasters have been created.
fire_hazard_final_build_complete = True

print('=== FINAL COMPOSITE FIRE-HAZARD GRID BUILT ===')

# Release full-size arrays before final validation to reduce notebook memory use.
del normalized, valid, hazard_index, hazard_class
gc.collect()


=== FINAL COMPOSITE FIRE-HAZARD GRID BUILT ===


0

### Validate Final Raster Ranges and Observed Hazard Classes

In [143]:
print('=== VALIDATING AND FINALIZING COMPOSITE FIRE-HAZARD GRID ===')

# Validate the normalized composite against the established normalized-component requirements.
normalized_validation = validate_normalized_component(
    fire_hazard_final_normalized_path
)

# Run the shared raster validation on the index before its specific 1-10 range check.
index_validation = validate_normalized_component(
    fire_hazard_final_index_path
)

# Inspect the continuous index to verify its populated cells remain within the 1-10 scale.
with rasterio.open(fire_hazard_final_index_path) as index_source:
    # Load final index values for range validation.
    index_values = index_source.read(1)

    # Isolate finite, non-nodata index cells for summary statistics.
    index_valid_mask = (
        np.isfinite(index_values)
        & (index_values != fire_hazard_nodata_value)
    )

    # Calculate the observed minimum only when the index contains valid cells.
    index_minimum = (
        float(index_values[index_valid_mask].min())
        if index_valid_mask.any()
        else None
    )

    # Calculate the observed maximum only when the index contains valid cells.
    index_maximum = (
        float(index_values[index_valid_mask].max())
        if index_valid_mask.any()
        else None
    )

    # Require populated index values to stay within the intended 1-10 model bounds.
    index_range_valid = (
        index_minimum is not None
        and index_maximum is not None
        and (index_minimum >= 1.0 - 1e-06)
        and (index_maximum <= 10.0 + 1e-06)
    )

# Inspect categorical output to verify only the five configured classes are present.
with rasterio.open(fire_hazard_final_class_path) as class_source:
    # Load final class values for categorical QA.
    class_values = class_source.read(1)

    # Isolate finite, non-nodata cells before evaluating observed categories.
    class_valid_mask = (
        np.isfinite(class_values)
        & (class_values != fire_hazard_nodata_value)
    )

    # Collect unique observed classes to confirm the categorical output is valid.
    observed_classes = sorted(
        np.unique(
            class_values[class_valid_mask]
        ).astype(int).tolist()
    )

    # Require at least one populated cell and limit classes to the configured values 1-5.
    class_range_valid = (
        set(observed_classes).issubset({1, 2, 3, 4, 5})
        and len(observed_classes) > 0
    )

# Combine raster, range, class, and file-existence checks into the final validation status.
fire_hazard_final_validation_complete = all(
    [
        normalized_validation['VALID'],
        index_range_valid,
        class_range_valid,
        fire_hazard_final_normalized_path.exists(),
        fire_hazard_final_index_path.exists(),
        fire_hazard_final_class_path.exists(),
    ]
)


=== VALIDATING AND FINALIZING COMPOSITE FIRE-HAZARD GRID ===


### Export Final Validation Metadata and Complete Composite-Grid QA

In [144]:
# Summarize final raster statistics and validation status for reproducibility and QA.
fire_hazard_final_validation_summary = pd.DataFrame(
    [
        {
            'MODEL_NAME': fire_hazard_model_name,
            'MODEL_VERSION': fire_hazard_model_version,
            'NORMALIZED_OUTPUT': str(fire_hazard_final_normalized_path),
            'INDEX_OUTPUT': str(fire_hazard_final_index_path),
            'CLASS_OUTPUT': str(fire_hazard_final_class_path),
            'NORMALIZED_VALID_PIXELS': normalized_validation['VALID_PIXELS'],
            'NORMALIZED_MINIMUM': normalized_validation['MINIMUM'],
            'NORMALIZED_MAXIMUM': normalized_validation['MAXIMUM'],
            'NORMALIZED_MEAN': normalized_validation['MEAN'],
            'INDEX_MINIMUM': index_minimum,
            'INDEX_MAXIMUM': index_maximum,
            'OBSERVED_CLASSES': ', '.join(map(str, observed_classes)),
            'VALID_OUTSIDE_STUDY_AREA': (
                normalized_validation['VALID_OUTSIDE_STUDY_AREA']
            ),
            'VALIDATION_COMPLETE': fire_hazard_final_validation_complete,
        }
    ]
)

# Define the metadata destination for the final validation summary.
fire_hazard_final_validation_summary_path = (
    fire_hazard_remaining_metadata
    / 'composite_fire_hazard_validation_summary.csv'
)

# Export validation metrics so later phases can confirm the final grid passed QA.
fire_hazard_final_validation_summary.to_csv(
    fire_hazard_final_validation_summary_path,
    index=False,
)

# Record the finalized model scope, weights, outputs, formulas, classes, and validation state.
fire_hazard_final_policy = {
    'model_name': fire_hazard_model_name,
    'model_version': fire_hazard_model_version,
    'scope': fire_hazard_model_scope,
    'target_crs': fire_hazard_target_crs,
    'cell_size_meters': fire_hazard_cell_size,
    'component_weights': fire_hazard_final_component_weights,
    'normalized_output': str(fire_hazard_final_normalized_path),
    'index_output': str(fire_hazard_final_index_path),
    'class_output': str(fire_hazard_final_class_path),
    'index_formula': '1 + normalized_hazard * 9',
    'classes': {
        '1': 'Very Low (0.00-0.20)',
        '2': 'Low (0.20-0.40)',
        '3': 'Moderate (0.40-0.60)',
        '4': 'High (0.60-0.80)',
        '5': 'Very High (0.80-1.00)',
    },
    'validation_complete': bool(fire_hazard_final_validation_complete),
}

# Define the JSON destination for the final composite-hazard policy record.
fire_hazard_final_policy_path = (
    fire_hazard_remaining_metadata
    / 'composite_fire_hazard_final_policy.json'
)

# Write the final policy so the model configuration can be reproduced outside the notebook.
with fire_hazard_final_policy_path.open(
    'w',
    encoding='utf-8',
) as policy_file:
    json.dump(
        fire_hazard_final_policy,
        policy_file,
        indent=2,
    )

# Build a registry linking every final model component to its weight and validated input raster.
fire_hazard_final_component_registry = pd.DataFrame(
    [
        {
            'COMPONENT_ID': key,
            'WEIGHT': fire_hazard_final_component_weights[key],
            'INPUT_PATH': str(fire_hazard_final_component_paths[key]),
            'VALID': True,
        }
        for key in fire_hazard_final_component_paths
    ]
)

# Define the metadata destination for the final component registry.
fire_hazard_final_component_registry_path = (
    fire_hazard_remaining_metadata
    / 'composite_fire_hazard_component_registry.csv'
)

# Export the registry so downstream phases can trace every component used in the final grid.
fire_hazard_final_component_registry.to_csv(
    fire_hazard_final_component_registry_path,
    index=False,
)

# Propagate the final validation status as the completion state for the composite hazard grid.
fire_hazard_composite_grid_finalized = fire_hazard_final_validation_complete

# Stop execution if any final validation requirement prevents the grid from being finalized.
if not fire_hazard_composite_grid_finalized:
    raise ValueError(
        'The completed Composite Fire Hazard Grid failed final validation.'
    )

# Display final QA tables so the completed model can be reviewed directly in the notebook.
display(fire_hazard_final_validation_summary)
display(fire_hazard_final_component_registry)

# Report the finalized raster and policy locations for downstream workflow use.
print(f'-> Normalized hazard grid: {fire_hazard_final_normalized_path}')
print(f'-> Fire Hazard Index grid: {fire_hazard_final_index_path}')
print(f'-> Fire Hazard Class grid: {fire_hazard_final_class_path}')
print(f'-> Final policy: {fire_hazard_final_policy_path}')
print('=== COMPOSITE FIRE-HAZARD GRID COMPLETE ===')

# Release any unreferenced objects after the final composite-grid outputs are complete.
gc.collect()


,MODEL_NAME,MODEL_VERSION,NORMALIZED_OUTPUT,INDEX_OUTPUT,CLASS_OUTPUT,NORMALIZED_VALID_PIXELS,NORMALIZED_MINIMUM,NORMALIZED_MAXIMUM,NORMALIZED_MEAN,INDEX_MINIMUM,INDEX_MAXIMUM,OBSERVED_CLASSES,VALID_OUTSIDE_STUDY_AREA,VALIDATION_COMPLETE
0,Baseline Composite Fire Hazard Grid,1.0,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,14112977,0.019249,0.760326,0.397139,1.173243,7.842934,"1, 2, 3, 4",0,True


,COMPONENT_ID,WEIGHT,INPUT_PATH,VALID
0,fuel_hazard,0.30,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
1,fuel_dryness,0.20,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
2,slope_hazard,0.15,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
3,aspect_hazard,0.10,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
4,historical_fire_likelihood,0.15,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True
5,human_ignition_potential,0.10,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,True


-> Normalized hazard grid: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\composite_fire_hazard\composite_fire_hazard_normalized.tif
-> Fire Hazard Index grid: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\composite_fire_hazard\composite_fire_hazard_index_1_10.tif
-> Fire Hazard Class grid: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\composite_fire_hazard\composite_fire_hazard_class.tif
-> Final policy: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\metadata\composite_fire_hazard_final_policy.json
=== COMPOSITE FIRE-HAZARD GRID COMPLETE ===


33

# CHRG Phase 13 – Integrated Wildfire Decision-Support Dashboard

## Purpose

Integrate the completed Composite Fire Hazard Grid with the existing Wildfire Exposure
Assessment Folium dashboard.


## Prepare the Hazard Raster


In [145]:
print('=== PREPARING COMPOSITE FIRE HAZARD FOR DASHBOARD INTEGRATION ===')

# Define the existing dashboard and final hazard outputs required by Phase 13.
required_integrated_dashboard_inputs = [
    'wui_dashboard_map',
    'fire_hazard_final_class_path',
    'fire_hazard_final_validation_complete',
]

# Identify missing upstream objects before the integrated dashboard is modified.
missing_integrated_dashboard_inputs = [
    object_name
    for object_name in required_integrated_dashboard_inputs
    if object_name not in globals()
]

# Stop execution if the exposure dashboard or validated hazard output is unavailable.
if missing_integrated_dashboard_inputs:
    raise NameError(
        'The following integrated-dashboard inputs are missing:\n'
        f'{missing_integrated_dashboard_inputs}'
    )

# Stop execution if the Composite Fire Hazard Grid did not pass final validation.
if not fire_hazard_final_validation_complete:
    raise ValueError(
        'The Composite Fire Hazard Grid has not passed final validation.'
    )

# Require the existing exposure dashboard to remain a valid Folium map.
if not isinstance(wui_dashboard_map, folium.Map):
    raise TypeError(
        'wui_dashboard_map is not a valid Folium Map.'
    )

# Normalize the classified hazard path before raster file validation.
fire_hazard_dashboard_source_path = Path(
    fire_hazard_final_class_path
)

# Stop execution if the classified hazard raster is unavailable for visualization.
if (
    not fire_hazard_dashboard_source_path.exists()
    or not fire_hazard_dashboard_source_path.is_file()
    or fire_hazard_dashboard_source_path.stat().st_size <= 0
):
    raise FileNotFoundError(
        'The final Composite Fire Hazard Class raster is unavailable:\n'
        f'{fire_hazard_dashboard_source_path}'
    )

# Import categorical color and raster-reprojection tools used by the Folium
# hazard overlay.
from matplotlib.colors import ListedColormap
from rasterio.enums import Resampling
from rasterio.transform import array_bounds
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    transform_bounds,
)

# Limit web-map raster dimensions so the interactive dashboard remains responsive.
fire_hazard_dashboard_max_dimension = 1800

# Set partial transparency so exposure polygons and reference features remain visible.
fire_hazard_dashboard_opacity = 0.58

# Define the five class colors used to interpret the final categorical hazard surface.
fire_hazard_dashboard_class_colors = [
    '#2c7bb6',
    '#abd9e9',
    '#ffffbf',
    '#fdae61',
    '#d7191c',
]

# Define labels that match the five classes stored in the final hazard-class raster.
fire_hazard_dashboard_class_labels = {
    1: 'Very Low',
    2: 'Low',
    3: 'Moderate',
    4: 'High',
    5: 'Very High',
}

# Create a categorical colormap used only for the browser-display representation.
fire_hazard_dashboard_colormap = ListedColormap(
    fire_hazard_dashboard_class_colors
)

# Define the Web Mercator CRS used internally by Leaflet and Folium.
fire_hazard_dashboard_crs = 'EPSG:3857'

# Open the authoritative class raster and create a reprojected browser-display copy.
with rasterio.open(
    fire_hazard_dashboard_source_path
) as fire_hazard_dashboard_source:

    # Stop execution if the categorical source does not contain exactly one raster band.
    if fire_hazard_dashboard_source.count != 1:
        raise ValueError(
            'The Composite Fire Hazard Class raster must contain exactly one band.'
        )

    # Stop execution if the raster lacks a CRS required for reprojection.
    if fire_hazard_dashboard_source.crs is None:
        raise ValueError(
            'The Composite Fire Hazard Class raster does not contain a CRS.'
        )

    # Calculate the native Web Mercator transform and dimensions used by Leaflet.
    (
        fire_hazard_dashboard_transform,
        fire_hazard_dashboard_width,
        fire_hazard_dashboard_height,
    ) = calculate_default_transform(
        fire_hazard_dashboard_source.crs,
        fire_hazard_dashboard_crs,
        fire_hazard_dashboard_source.width,
        fire_hazard_dashboard_source.height,
        *fire_hazard_dashboard_source.bounds,
    )

    # Preserve aspect ratio while limiting the longest browser-display dimension.
    fire_hazard_dashboard_scale = min(
        1.0,
        (
            fire_hazard_dashboard_max_dimension
            / max(
                fire_hazard_dashboard_width,
                fire_hazard_dashboard_height,
            )
        ),
    )

    # Calculate the reduced display width for the reprojected visualization raster.
    fire_hazard_dashboard_width = max(
        1,
        int(
            round(
                fire_hazard_dashboard_width
                * fire_hazard_dashboard_scale
            )
        ),
    )

    # Calculate the reduced display height for the reprojected visualization raster.
    fire_hazard_dashboard_height = max(
        1,
        int(
            round(
                fire_hazard_dashboard_height
                * fire_hazard_dashboard_scale
            )
        ),
    )

    # Recalculate the Web Mercator transform for the reduced display dimensions.
    (
        fire_hazard_dashboard_transform,
        fire_hazard_dashboard_width,
        fire_hazard_dashboard_height,
    ) = calculate_default_transform(
        fire_hazard_dashboard_source.crs,
        fire_hazard_dashboard_crs,
        fire_hazard_dashboard_source.width,
        fire_hazard_dashboard_source.height,
        *fire_hazard_dashboard_source.bounds,
        dst_width=fire_hazard_dashboard_width,
        dst_height=fire_hazard_dashboard_height,
    )

    # Use zero as display NoData because the validated hazard classes are one through five.
    fire_hazard_dashboard_nodata = 0

    # Create the destination array for the Web Mercator display raster.
    fire_hazard_dashboard_classes = np.full(
        (
            fire_hazard_dashboard_height,
            fire_hazard_dashboard_width,
        ),
        fire_hazard_dashboard_nodata,
        dtype=np.uint8,
    )

    # Reproject categorical pixels into Leaflet's CRS without interpolating class values.
    reproject(
        source=rasterio.band(
            fire_hazard_dashboard_source,
            1,
        ),
        destination=fire_hazard_dashboard_classes,
        src_transform=fire_hazard_dashboard_source.transform,
        src_crs=fire_hazard_dashboard_source.crs,
        src_nodata=fire_hazard_dashboard_source.nodata,
        dst_transform=fire_hazard_dashboard_transform,
        dst_crs=fire_hazard_dashboard_crs,
        dst_nodata=fire_hazard_dashboard_nodata,
        resampling=Resampling.nearest,
    )

    # Restrict visible cells to the five validated hazard classes used by the model.
    fire_hazard_dashboard_valid = np.isin(
        fire_hazard_dashboard_classes,
        [1, 2, 3, 4, 5],
    )

    # Stop execution if reprojection removes every valid class from the display raster.
    if not fire_hazard_dashboard_valid.any():
        raise ValueError(
            'The dashboard hazard raster contains no valid fire-hazard classes.'
        )

    # Convert one-based hazard classes to zero-based categorical color indices.
    fire_hazard_dashboard_color_index = np.clip(
        fire_hazard_dashboard_classes.astype(np.int16) - 1,
        0,
        4,
    )

    # Convert class colors to an RGBA image that Folium can embed in the dashboard.
    fire_hazard_dashboard_rgba = (
        fire_hazard_dashboard_colormap(
            fire_hazard_dashboard_color_index / 4.0
        )
        * 255
    ).astype(np.uint8)

    # Make NoData and excluded cells transparent so the basemap remains visible.
    fire_hazard_dashboard_rgba[..., 3] = np.where(
        fire_hazard_dashboard_valid,
        255,
        0,
    ).astype(np.uint8)

    # Calculate the exact Web Mercator bounds of the reprojected display array.
    fire_hazard_dashboard_bounds_3857 = array_bounds(
        fire_hazard_dashboard_height,
        fire_hazard_dashboard_width,
        fire_hazard_dashboard_transform,
    )

    # Convert display bounds to geographic coordinates required by ImageOverlay.
    fire_hazard_dashboard_bounds_wgs84 = transform_bounds(
        fire_hazard_dashboard_crs,
        'EPSG:4326',
        *fire_hazard_dashboard_bounds_3857,
        densify_pts=21,
    )

# Convert geographic bounds to Folium's southwest and northeast coordinate order.
fire_hazard_dashboard_bounds = [
    [
        fire_hazard_dashboard_bounds_wgs84[1],
        fire_hazard_dashboard_bounds_wgs84[0],
    ],
    [
        fire_hazard_dashboard_bounds_wgs84[3],
        fire_hazard_dashboard_bounds_wgs84[2],
    ],
]

# Record successful preparation so the next Phase 13 cell can require this output explicitly.
fire_hazard_dashboard_preparation_complete = True

# Report the browser-display raster dimensions for dashboard QA.
print(
    f'-> Hazard display size: '
    f'{fire_hazard_dashboard_width:,} x '
    f'{fire_hazard_dashboard_height:,} pixels'
)

# Report the classified source retained as the authoritative analytical product.
print(
    f'-> Hazard source raster: '
    f'{fire_hazard_dashboard_source_path}'
)

print('\n=== COMPOSITE FIRE HAZARD DASHBOARD PREPARATION COMPLETE ===')

=== PREPARING COMPOSITE FIRE HAZARD FOR DASHBOARD INTEGRATION ===
-> Hazard display size: 1,192 x 1,800 pixels
-> Hazard source raster: C:\Users\adamd\Projects\WUI\data\raw\fire_hazard\fuels\fuel_hazard_component\composite_fire_hazard\composite_fire_hazard_class.tif

=== COMPOSITE FIRE HAZARD DASHBOARD PREPARATION COMPLETE ===


## Assemble the Integrated Dashboard


In [146]:
print(
    '=== ASSEMBLING INTEGRATED INTEGRATED WILDFIRE DASHBOARD ==='
)


# ----------------------------------------------------
# REQUIRE PHASE 13 INPUTS
# ----------------------------------------------------

if not globals().get(
    'fire_hazard_dashboard_preparation_complete',
    False,
):
    raise RuntimeError(
        'Run the Phase 13 hazard-dashboard preparation cell first.'
    )

if not globals().get(
    'wui_dashboard_assembly_complete',
    False,
):
    raise RuntimeError(
        'The Phase 5 WUI dashboard assembly has not been validated. '
        'Run the Phase 5 dashboard-assembly cells first.'
    )

if not isinstance(wui_dashboard_map, folium.Map):
    raise TypeError(
        'wui_dashboard_map is not a valid Folium Map.'
    )

from branca.element import MacroElement, Template
from folium.template import Template as FoliumTemplate
from folium.plugins import TreeLayerControl

print('-> Phase 5 WUI dashboard assembly: validated')
print('-> Phase 13 hazard display preparation: validated')


# ----------------------------------------------------
# REMOVE PREEXISTING FINAL UI ELEMENTS
# ----------------------------------------------------
# Remove final controls and legends from an earlier execution so rerunning this
# cell cannot create duplicate interface elements.

preexisting_layer_control_keys = [
    child_key
    for child_key, child_object
    in wui_dashboard_map._children.items()
    if isinstance(
        child_object,
        (
            folium.map.LayerControl,
            TreeLayerControl,
        ),
    )
]

for child_key in preexisting_layer_control_keys:
    del wui_dashboard_map._children[child_key]

legacy_dashboard_legend_name = 'dashboard_map_legend'
portfolio_legend_name = 'phase13_portfolio_legend'
portfolio_title_name = 'phase13_portfolio_title'

wui_dashboard_map._children.pop(
    legacy_dashboard_legend_name,
    None,
)

wui_dashboard_map._children.pop(
    portfolio_legend_name,
    None,
)

wui_dashboard_map._children.pop(
    portfolio_title_name,
    None,
)

print(
    f'-> Preexisting layer controls removed: '
    f'{len(preexisting_layer_control_keys)}'
)


# ----------------------------------------------------
# REMOVE PRIOR PHASE 13 HAZARD OVERLAY
# ----------------------------------------------------

fire_hazard_dashboard_layer_name = (
    'Composite Fire Hazard'
)

existing_fire_hazard_overlay_keys = [
    child_key
    for child_key, child_object
    in wui_dashboard_map._children.items()
    if getattr(child_object, 'layer_name', None)
    in {
        'Composite Fire Hazard',
        'Composite Fire Hazard Classes',
    }
]

for child_key in existing_fire_hazard_overlay_keys:
    del wui_dashboard_map._children[child_key]

print(
    f'-> Previous hazard overlays removed: '
    f'{len(existing_fire_hazard_overlay_keys)}'
)


# ----------------------------------------------------
# ADD COMPOSITE FIRE HAZARD OVERLAY
# ----------------------------------------------------

fire_hazard_dashboard_overlay = (
    folium.raster_layers.ImageOverlay(
        image=fire_hazard_dashboard_rgba,
        bounds=fire_hazard_dashboard_bounds,
        name=fire_hazard_dashboard_layer_name,
        opacity=fire_hazard_dashboard_opacity,
        interactive=False,
        cross_origin=False,
        zindex=1,
        show=True,
    )
)

fire_hazard_dashboard_overlay.add_to(
    wui_dashboard_map
)

print('-> Composite Fire Hazard overlay added to shared dashboard.')


# ----------------------------------------------------
# ADD COMPACT PORTFOLIO MAP TITLE
# ----------------------------------------------------

portfolio_title_html = """
{% macro html(this, kwargs) %}

<div id="phase13-portfolio-title" class="portfolio-map-title">
    <div class="portfolio-map-title-main">
        Utah Wildland–Urban Interface Exposure
    </div>
    <div class="portfolio-map-title-subtitle">
        Population, housing, critical facilities, and modeled wildfire hazard
    </div>
</div>

{% endmacro %}
"""

portfolio_title = MacroElement()
portfolio_title._name = 'Phase13PortfolioTitle'
portfolio_title._template = Template(portfolio_title_html)

wui_dashboard_map.add_child(
    portfolio_title,
    name=portfolio_title_name,
)


# ----------------------------------------------------
# ADD ONE INTEGRATED ANALYTICAL LEGEND
# ----------------------------------------------------
# Keep the five hazard classes visible, identify the two polygon overlays, and
# reveal facility symbols only while at least one facility layer is active.

portfolio_legend_html = f"""
{{% macro header(this, kwargs) %}}

<style>

    /* Apply one restrained visual system to titles, legends, and controls. */
    .portfolio-map-title,
    .portfolio-map-legend,
    .leaflet-control-layers {{
        box-sizing: border-box;
        background: rgba(255, 255, 255, 0.96) !important;
        border: 1px solid #6b7280 !important;
        border-radius: 6px !important;
        color: #1f2937;
        font-family: Arial, Helvetica, sans-serif !important;
        box-shadow: 0 2px 7px rgba(0, 0, 0, 0.22) !important;
    }}

    /* Keep the title clear of the standard upper-left navigation controls. */
    .portfolio-map-title {{
        position: fixed;
        top: 9px;
        left: 72px;
        max-width: 470px;
        padding: 8px 11px;
        z-index: 9998;
        pointer-events: none;
    }}

    .portfolio-map-title-main {{
        font-size: 15px;
        font-weight: 700;
        line-height: 1.2;
    }}

    .portfolio-map-title-subtitle {{
        margin-top: 2px;
        color: #4b5563;
        font-size: 11px;
        line-height: 1.25;
    }}

    /* Match the layer control to the analytical legend without overemphasizing it. */
    .leaflet-control-layers {{
        font-size: 12px !important;
        line-height: 1.35 !important;
    }}

    .leaflet-control-layers-expanded {{
        max-height: 70vh;
        min-width: 235px;
        padding: 9px 11px !important;
        overflow-y: auto;
    }}

    .leaflet-control-layers-toggle {{
        width: 34px !important;
        height: 34px !important;
        background-size: 20px 20px !important;
    }}

    .leaflet-control-layers label {{
        margin-bottom: 3px;
    }}

    /* Keep lower map controls clear of the dashboard frame edge. */
    .leaflet-control-scale {{
        margin-bottom: 10px !important;
    }}

    .leaflet-control-attribution {{
        margin-right: 8px !important;
        margin-bottom: 8px !important;
    }}

    /* Align the analytical legend with the standard upper-left map controls. */
    .portfolio-map-legend {{
        position: fixed;
        left: 10px;
        bottom: 58px;
        width: 224px;
        padding: 10px 11px;
        z-index: 9998;
        font-size: 12px;
        line-height: 1.3;
    }}

    .portfolio-legend-title {{
        margin-bottom: 6px;
        font-size: 14px;
        font-weight: 700;
    }}

    .portfolio-legend-subtitle {{
        margin: 8px 0 4px 0;
        padding-top: 6px;
        border-top: 1px solid #d1d5db;
        font-size: 12px;
        font-weight: 700;
    }}

    .portfolio-legend-row {{
        display: flex;
        align-items: center;
        min-height: 18px;
    }}

    .portfolio-hazard-swatch {{
        display: inline-block;
        width: 15px;
        height: 13px;
        margin-right: 7px;
        flex: 0 0 auto;
    }}

    .portfolio-outline-swatch {{
        display: inline-block;
        width: 18px;
        height: 12px;
        margin-right: 7px;
        border: 2px solid #6d28d9;
        background: rgba(139, 92, 246, 0.14);
        flex: 0 0 auto;
    }}

    .portfolio-exposure-ramp {{
        display: flex;
        width: 102px;
        height: 10px;
        margin: 3px 0 2px 0;
        border: 1px solid #9ca3af;
    }}

    .portfolio-exposure-ramp span {{
        flex: 1;
    }}

    .portfolio-exposure-labels {{
        display: flex;
        justify-content: space-between;
        width: 104px;
        color: #4b5563;
        font-size: 9px;
    }}

    .portfolio-facility-dot {{
        display: inline-block;
        width: 9px;
        height: 9px;
        margin-right: 7px;
        border-radius: 50%;
        flex: 0 0 auto;
    }}

    .portfolio-legend-note {{
        margin-top: 7px;
        color: #4b5563;
        font-size: 10px;
        line-height: 1.25;
    }}

    /* Preserve room for the data while adapting the title on narrow displays. */
    @media (max-width: 850px) {{
        .portfolio-map-title-subtitle {{
            display: none;
        }}

        .portfolio-map-title {{
            max-width: 310px;
        }}
    }}

</style>

{{% endmacro %}}


{{% macro html(this, kwargs) %}}

<div id="phase13-portfolio-legend" class="portfolio-map-legend">

    <div class="portfolio-legend-title">Composite Fire Hazard</div>

    <div class="portfolio-legend-row">
        <span class="portfolio-hazard-swatch" style="background:#2c7bb6;"></span>
        Very Low
    </div>
    <div class="portfolio-legend-row">
        <span class="portfolio-hazard-swatch" style="background:#abd9e9;"></span>
        Low
    </div>
    <div class="portfolio-legend-row">
        <span class="portfolio-hazard-swatch" style="background:#ffffbf;"></span>
        Moderate
    </div>
    <div class="portfolio-legend-row">
        <span class="portfolio-hazard-swatch" style="background:#fdae61;"></span>
        High
    </div>
    <div class="portfolio-legend-row">
        <span class="portfolio-hazard-swatch" style="background:#d7191c;"></span>
        Very High
    </div>

    <div class="portfolio-legend-subtitle">Analysis overlays</div>

    <div class="portfolio-legend-row">
        <span class="portfolio-outline-swatch"></span>
        High-Risk WUI
    </div>

    <div>Block Group WUI Exposure</div>
    <div class="portfolio-exposure-ramp" aria-hidden="true">
        <span style="background:#f7fbff;"></span>
        <span style="background:#fee8c8;"></span>
        <span style="background:#fdbb84;"></span>
        <span style="background:#fc8d59;"></span>
        <span style="background:#d7301f;"></span>
        <span style="background:#7f0000;"></span>
    </div>
    <div class="portfolio-exposure-labels">
        <span>0%</span><span>10</span><span>25</span><span>50</span><span>75</span><span>100%</span>
    </div>

    <div id="phase13-facility-key" style="display:none;">
        <div class="portfolio-legend-subtitle">Critical facilities</div>
        <div class="portfolio-legend-row">
            <span class="portfolio-facility-dot" style="background:red;"></span>
            Hospitals / Medical Facilities
        </div>
        <div class="portfolio-legend-row">
            <span class="portfolio-facility-dot" style="background:orange;"></span>
            Fire Stations
        </div>
        <div class="portfolio-legend-row">
            <span class="portfolio-facility-dot" style="background:blue;"></span>
            Police / Sheriff Stations
        </div>
        <div class="portfolio-legend-row">
            <span class="portfolio-facility-dot" style="background:green;"></span>
            Public Schools
        </div>
        <div class="portfolio-legend-row">
            <span class="portfolio-facility-dot" style="background:purple;"></span>
            Emergency Shelters
        </div>
    </div>

    <div class="portfolio-legend-note">
        Relative modeled hazard for screening and decision support; not a fire forecast.
    </div>

</div>

{{% endmacro %}}


{{% macro script(this, kwargs) %}}

    // Reference the current map and all five facility FeatureGroups.
    var portfolioMap = {wui_dashboard_map.get_name()};
    var portfolioFacilityLayers = [
        {facility_feature_groups['Hospitals and Medical Facilities'].get_name()},
        {facility_feature_groups['Fire Stations'].get_name()},
        {facility_feature_groups['Police and Sheriff Stations'].get_name()},
        {facility_feature_groups['Public Schools'].get_name()},
        {facility_feature_groups['Emergency Shelters'].get_name()}
    ];

    // Display the facility key only when at least one facility layer is visible.
    function updatePortfolioFacilityKey() {{
        var facilityKey = document.getElementById('phase13-facility-key');
        var facilityLayerVisible = portfolioFacilityLayers.some(
            function(layer) {{
                return portfolioMap.hasLayer(layer);
            }}
        );

        if (facilityKey) {{
            facilityKey.style.display = facilityLayerVisible ? 'block' : 'none';
        }}
    }}

    portfolioMap.on('overlayadd overlayremove', updatePortfolioFacilityKey);
    updatePortfolioFacilityKey();

{{% endmacro %}}
"""

portfolio_legend = MacroElement()
portfolio_legend._name = 'Phase13PortfolioLegend'
portfolio_legend._template = Template(
    portfolio_legend_html
)

wui_dashboard_map.add_child(
    portfolio_legend,
    name=portfolio_legend_name,
)

print('-> One integrated analytical legend added in the lower-left corner.')
print('-> County-boundary legend omitted as redundant.')
print('-> Critical-facility key configured to appear contextually.')


# ----------------------------------------------------
# CREATE THE SINGLE FINAL TREE LAYER CONTROL
# ----------------------------------------------------
# The complete control begins collapsed so the analytical map, rather than the
# interface, receives visual priority when the portfolio dashboard opens.

print('=== CREATING FINAL TREE LAYER CONTROL ===')

final_tree_required_layers = {
    'Esri World Topographic': esri_topographic_basemap,
    'Esri World Imagery': esri_imagery_basemap,
    'Salt Lake County': county_boundary_groups['SALT LAKE'],
    'Utah County': county_boundary_groups['UTAH'],
    'Washington County': county_boundary_groups['WASHINGTON'],
    'Hospitals / Medical Facilities': facility_feature_groups[
        'Hospitals and Medical Facilities'
    ],
    'Fire Stations': facility_feature_groups['Fire Stations'],
    'Police / Sheriff Stations': facility_feature_groups[
        'Police and Sheriff Stations'
    ],
    'Public Schools': facility_feature_groups['Public Schools'],
    'Emergency Shelters': facility_feature_groups['Emergency Shelters'],
    'High-Risk WUI': wui_feature_group,
    'Block Group WUI Exposure': acs_exposure_feature_group,
    'Composite Fire Hazard': fire_hazard_dashboard_overlay,
}

missing_final_tree_layers = [
    layer_label
    for layer_label, layer_object
    in final_tree_required_layers.items()
    if layer_object.get_name()
    not in wui_dashboard_map._children
]

if missing_final_tree_layers:
    raise ValueError(
        'The following layers required by the final TreeLayerControl '
        'are not attached to the dashboard:\n'
        f'{missing_final_tree_layers}'
    )

final_dashboard_base_tree = {
    'label': 'Basemaps',
    'collapsed': True,
    'children': [
        {'label': 'Esri World Topographic', 'layer': esri_topographic_basemap},
        {'label': 'Esri World Imagery', 'layer': esri_imagery_basemap},
    ],
}

final_dashboard_overlay_tree = {
    'label': 'Map Layers',
    'children': [
        {
            'label': 'Reference & Facilities',
            'collapsed': True,
            'children': [
                {'label': 'Salt Lake County', 'layer': county_boundary_groups['SALT LAKE']},
                {'label': 'Utah County', 'layer': county_boundary_groups['UTAH']},
                {'label': 'Washington County', 'layer': county_boundary_groups['WASHINGTON']},
                {
                    'label': 'Hospitals / Medical Facilities',
                    'layer': facility_feature_groups['Hospitals and Medical Facilities'],
                },
                {'label': 'Fire Stations', 'layer': facility_feature_groups['Fire Stations']},
                {
                    'label': 'Police / Sheriff Stations',
                    'layer': facility_feature_groups['Police and Sheriff Stations'],
                },
                {'label': 'Public Schools', 'layer': facility_feature_groups['Public Schools']},
                {'label': 'Emergency Shelters', 'layer': facility_feature_groups['Emergency Shelters']},
            ],
        },
        {
            'label': 'Wildfire Analysis',
            'collapsed': False,
            'children': [
                {'label': 'Composite Fire Hazard', 'layer': fire_hazard_dashboard_overlay},
                {'label': 'High-Risk WUI', 'layer': wui_feature_group},
                {'label': 'Block Group WUI Exposure', 'layer': acs_exposure_feature_group},
            ],
        },
    ],
}

final_dashboard_tree_layer_control = TreeLayerControl(
    base_tree=final_dashboard_base_tree,
    overlay_tree=final_dashboard_overlay_tree,
    collapsed=True,
    position='topright',
    closed_symbol='▶',
    opened_symbol='▼',
    space_symbol='',
    select_all_checkbox=False,
)

# Folium's standard TreeLayerControl template creates the Leaflet control
# anonymously. Assign it to Folium's generated JavaScript name so the
# click-only interaction template below can safely remove the plugin's
# default hover and map-click listeners.
final_dashboard_tree_layer_control._template = FoliumTemplate(
    '''
    {% macro script(this, kwargs) %}
        var {{ this.get_name() }} = L.control.layers.tree(
            {{ this.base_tree | tojavascript }},
            {{ this.overlay_tree | tojavascript }},
            {{ this.options | tojavascript }}
        ).addTo({{ this._parent.get_name() }});
    {% endmacro %}
    '''
)

final_dashboard_tree_layer_control.add_to(
    wui_dashboard_map
)


# ----------------------------------------------------
# USE CLICK-TO-TOGGLE LAYER-CONTROL BEHAVIOR
# ----------------------------------------------------
# Leaflet normally expands a collapsed layer control on pointer hover and
# collapses it when the pointer leaves. Replace that desktop-only behavior
# with an intentional click toggle: the control begins collapsed, remains
# open while layers are changed, and closes only when its icon is clicked
# again. Keyboard users can perform the same action with Enter or Space.

layer_control_click_toggle_name = (
    'dashboard_layer_control_click_toggle'
)

layer_control_click_toggle_template = '''
{% macro header(this, kwargs) %}
<style>
    /* Keep the layers icon available as the close action while expanded. */
    .leaflet-control-layers-expanded > .leaflet-control-layers-toggle {
        display: block;
        float: right;
        margin: 0 0 5px 8px;
    }

    /* Place the layer tree below the retained toggle icon. */
    .leaflet-control-layers-expanded > .leaflet-control-layers-list {
        clear: both;
    }
</style>
{% endmacro %}

{% macro script(this, kwargs) %}
(function() {
    var layerTreeControl = {{ this.layer_control.get_name() }};
    var layerControlContainer = layerTreeControl.getContainer();
    var layerControlToggle = layerControlContainer.querySelector(
        '.leaflet-control-layers-toggle'
    );

    if (!layerControlToggle) {
        return;
    }

    // Remove Leaflet's pointer-enter and pointer-leave expansion behavior.
    L.DomEvent.off(
        layerControlContainer,
        'mouseenter',
        layerTreeControl.expand,
        layerTreeControl
    );
    L.DomEvent.off(
        layerControlContainer,
        'mouseleave',
        layerTreeControl.collapse,
        layerTreeControl
    );
    L.DomEvent.off(
        layerControlContainer,
        'mouseover',
        layerTreeControl.expand,
        layerTreeControl
    );
    L.DomEvent.off(
        layerControlContainer,
        'mouseout',
        layerTreeControl.collapse,
        layerTreeControl
    );

    // Prevent clicks elsewhere on the map from closing the control.
    // The layers icon is the only open/close action.
    layerTreeControl._map.off(
        'click',
        layerTreeControl.collapse,
        layerTreeControl
    );

    // Replace the default one-way click expansion with a two-way toggle.
    L.DomEvent.off(layerControlToggle, 'click');
    L.DomEvent.off(layerControlToggle, 'keydown');

    function layerControlIsExpanded() {
        return L.DomUtil.hasClass(
            layerControlContainer,
            'leaflet-control-layers-expanded'
        );
    }

    function updateLayerControlAccessibility() {
        var isExpanded = layerControlIsExpanded();
        layerControlToggle.setAttribute(
            'aria-expanded',
            isExpanded ? 'true' : 'false'
        );
        layerControlToggle.title = isExpanded
            ? 'Close map layers'
            : 'Open map layers';
        layerControlToggle.setAttribute(
            'aria-label',
            layerControlToggle.title
        );
    }

    function toggleLayerControl(event) {
        L.DomEvent.preventDefault(event);
        L.DomEvent.stopPropagation(event);

        if (layerControlIsExpanded()) {
            layerTreeControl.collapse();
        } else {
            layerTreeControl.expand();
        }

        updateLayerControlAccessibility();
    }

    L.DomEvent.on(layerControlToggle, 'click', toggleLayerControl);
    L.DomEvent.on(layerControlToggle, 'keydown', function(event) {
        if (event.key === 'Enter' || event.key === ' ') {
            toggleLayerControl(event);
        }
    });

    layerTreeControl.collapse();
    layerControlToggle.setAttribute('role', 'button');
    layerControlToggle.setAttribute('tabindex', '0');
    updateLayerControlAccessibility();
})();
{% endmacro %}
'''

wui_dashboard_map._children.pop(
    layer_control_click_toggle_name,
    None,
)

layer_control_click_toggle = MacroElement()
layer_control_click_toggle._name = (
    layer_control_click_toggle_name
)
layer_control_click_toggle.layer_control = (
    final_dashboard_tree_layer_control
)
layer_control_click_toggle._template = Template(
    layer_control_click_toggle_template
)

wui_dashboard_map.add_child(
    layer_control_click_toggle,
    name=layer_control_click_toggle_name,
)

layer_control_click_toggle_count = sum(
    child_name == layer_control_click_toggle_name
    for child_name in wui_dashboard_map._children
)

if layer_control_click_toggle_count != 1:
    raise ValueError(
        'The completed dashboard must contain exactly one '
        'click-toggle layer-control definition.'
    )


# ----------------------------------------------------
# ASSIGN AND VALIDATE THE FINAL MAP OBJECT
# ----------------------------------------------------

wui_exposure_fire_hazard_map = wui_dashboard_map

final_standard_layer_control_count = sum(
    isinstance(child_object, folium.map.LayerControl)
    for child_object
    in wui_exposure_fire_hazard_map._children.values()
)

final_tree_layer_control_count = sum(
    isinstance(child_object, TreeLayerControl)
    for child_object
    in wui_exposure_fire_hazard_map._children.values()
)

fire_hazard_overlay_count = sum(
    getattr(child_object, 'layer_name', None)
    == fire_hazard_dashboard_layer_name
    for child_object
    in wui_exposure_fire_hazard_map._children.values()
)

portfolio_legend_count = sum(
    child_name == portfolio_legend_name
    for child_name
    in wui_exposure_fire_hazard_map._children
)

legacy_dashboard_legend_count = sum(
    child_name == legacy_dashboard_legend_name
    for child_name
    in wui_exposure_fire_hazard_map._children
)

portfolio_title_count = sum(
    child_name == portfolio_title_name
    for child_name
    in wui_exposure_fire_hazard_map._children
)

if final_standard_layer_control_count != 0:
    raise ValueError(
        'The completed dashboard must contain zero standard '
        'Folium LayerControl objects.'
    )

if final_tree_layer_control_count != 1:
    raise ValueError(
        'The completed dashboard must contain exactly one TreeLayerControl.'
    )

if fire_hazard_overlay_count != 1:
    raise ValueError(
        'The completed dashboard must contain exactly one '
        'Composite Fire Hazard overlay.'
    )

if portfolio_legend_count != 1 or legacy_dashboard_legend_count != 0:
    raise ValueError(
        'The completed dashboard must contain one integrated legend '
        'and no legacy Phase 5 legend.'
    )

if portfolio_title_count != 1:
    raise ValueError(
        'The completed dashboard must contain exactly one portfolio title.'
    )


# ----------------------------------------------------
# RECORD AND REPORT PHASE 13 ASSEMBLY STATUS
# ----------------------------------------------------

fire_hazard_dashboard_integration_complete = True

print(f'-> Hazard overlay count: {fire_hazard_overlay_count}')
print(f'-> Integrated legend count: {portfolio_legend_count}')
print(f'-> Legacy legend count: {legacy_dashboard_legend_count}')
print(f'-> Portfolio title count: {portfolio_title_count}')
print(f'-> Standard LayerControl count: {final_standard_layer_control_count}')
print(f'-> TreeLayerControl count: {final_tree_layer_control_count}')
print('-> TreeLayerControl default state: collapsed')
print('-> Basemaps branch default state: collapsed')
print('-> Wildfire Analysis branch state when opened: expanded')
print('-> Dashboard has not been rendered, saved, or displayed yet.')

print(
    '\n=== INTEGRATED INTEGRATED DASHBOARD ASSEMBLY COMPLETE ==='
)


=== ASSEMBLING INTEGRATED INTEGRATED WILDFIRE DASHBOARD ===
-> Phase 5 WUI dashboard assembly: validated
-> Phase 13 hazard display preparation: validated
-> Preexisting layer controls removed: 0
-> Previous hazard overlays removed: 0
-> Composite Fire Hazard overlay added to shared dashboard.
-> One integrated analytical legend added in the lower-left corner.
-> County-boundary legend omitted as redundant.
-> Critical-facility key configured to appear contextually.
=== CREATING FINAL TREE LAYER CONTROL ===
-> Hazard overlay count: 1
-> Integrated legend count: 1
-> Legacy legend count: 0
-> Portfolio title count: 1
-> Standard LayerControl count: 0
-> TreeLayerControl count: 1
-> TreeLayerControl default state: collapsed
-> Basemaps branch default state: collapsed
-> Wildfire Analysis branch state when opened: expanded
-> Dashboard has not been rendered, saved, or displayed yet.

=== INTEGRATED INTEGRATED DASHBOARD ASSEMBLY COMPLETE ===


## Validate, Export, and Display the Final Dashboard


In [147]:
print(
    '=== VALIDATING, EXPORTING, AND DISPLAYING FINAL DASHBOARD ==='
)


# ----------------------------------------------------
# REQUIRE COMPLETED IN-MEMORY ASSEMBLY
# ----------------------------------------------------

if not globals().get(
    'fire_hazard_dashboard_integration_complete',
    False,
):
    raise RuntimeError(
        'Run the Phase 13 dashboard-assembly cell first.'
    )

if not isinstance(
    wui_exposure_fire_hazard_map,
    folium.Map,
):
    raise TypeError(
        'wui_exposure_fire_hazard_map is not a valid Folium Map.'
    )


# ----------------------------------------------------
# VALIDATE FINAL PYTHON MAP STATE
# ----------------------------------------------------

final_hazard_overlay_count = sum(
    getattr(child_object, 'layer_name', None)
    == fire_hazard_dashboard_layer_name
    for child_object
    in wui_exposure_fire_hazard_map._children.values()
)

final_portfolio_legend_count = sum(
    child_name == portfolio_legend_name
    for child_name
    in wui_exposure_fire_hazard_map._children
)

final_legacy_legend_count = sum(
    child_name == legacy_dashboard_legend_name
    for child_name
    in wui_exposure_fire_hazard_map._children
)

final_portfolio_title_count = sum(
    child_name == portfolio_title_name
    for child_name
    in wui_exposure_fire_hazard_map._children
)

final_standard_layer_control_count = sum(
    isinstance(child_object, folium.map.LayerControl)
    for child_object
    in wui_exposure_fire_hazard_map._children.values()
)

final_tree_layer_control_count = sum(
    isinstance(child_object, TreeLayerControl)
    for child_object
    in wui_exposure_fire_hazard_map._children.values()
)

if final_hazard_overlay_count != 1:
    raise ValueError('The final dashboard must contain one hazard overlay.')

if final_portfolio_legend_count != 1 or final_legacy_legend_count != 0:
    raise ValueError(
        'The final dashboard must contain one integrated legend '
        'and zero legacy legends.'
    )

if final_portfolio_title_count != 1:
    raise ValueError('The final dashboard must contain one portfolio title.')

if final_standard_layer_control_count != 0:
    raise ValueError('The final dashboard must contain no flat LayerControl.')

if final_tree_layer_control_count != 1:
    raise ValueError('The final dashboard must contain one TreeLayerControl.')


# ----------------------------------------------------
# VALIDATE FINAL TREE STRUCTURE
# ----------------------------------------------------

final_basemap_tree_count = len(
    final_dashboard_base_tree.get('children', [])
)

final_overlay_tree_categories = {
    category.get('label'): category
    for category
    in final_dashboard_overlay_tree.get('children', [])
}

final_reference_tree = final_overlay_tree_categories[
    'Reference & Facilities'
]

final_wildfire_analysis_tree = final_overlay_tree_categories[
    'Wildfire Analysis'
]

final_reference_layer_count = len(
    final_reference_tree.get('children', [])
)

final_wildfire_analysis_layer_count = len(
    final_wildfire_analysis_tree.get('children', [])
)

if final_dashboard_tree_layer_control.options.get('collapsed') is not True:
    raise ValueError(
        'The complete TreeLayerControl must begin collapsed.'
    )

if final_dashboard_base_tree.get('collapsed') is not True:
    raise ValueError('The Basemaps branch must begin collapsed.')

if final_reference_tree.get('collapsed') is not True:
    raise ValueError(
        'The Reference & Facilities branch must begin collapsed.'
    )

if final_wildfire_analysis_tree.get('collapsed') is not False:
    raise ValueError(
        'The Wildfire Analysis branch must be expanded when the control opens.'
    )

if final_basemap_tree_count != 2:
    raise ValueError('The Basemaps branch must contain two Esri choices.')

if final_reference_layer_count != 8:
    raise ValueError(
        'The Reference & Facilities branch must contain eight layers.'
    )

if final_wildfire_analysis_layer_count != 3:
    raise ValueError(
        'The Wildfire Analysis branch must contain three layers.'
    )


# ----------------------------------------------------
# VALIDATE CUSTOM NAVIGATION CONTROLS
# ----------------------------------------------------

custom_dashboard_controls_name = (
    'dashboard_custom_navigation_controls'
)

final_custom_controls_count = sum(
    child_name == custom_dashboard_controls_name
    for child_name
    in wui_exposure_fire_hazard_map._children
)

if final_custom_controls_count != 1:
    raise ValueError(
        'The final dashboard must contain exactly one '
        'fullscreen/mouse-position control definition.'
    )

layer_control_click_toggle_name = (
    'dashboard_layer_control_click_toggle'
)

final_layer_control_click_toggle_count = sum(
    child_name == layer_control_click_toggle_name
    for child_name in wui_exposure_fire_hazard_map._children
)

if final_layer_control_click_toggle_count != 1:
    raise ValueError(
        'The final dashboard must contain exactly one '
        'click-toggle layer-control definition.'
    )


# ----------------------------------------------------
# DEFINE FINAL DASHBOARD OUTPUTS
# ----------------------------------------------------

integrated_dashboard_directory = Path(
    'outputs/dashboard'
)

integrated_dashboard_directory.mkdir(
    parents=True,
    exist_ok=True,
)

integrated_dashboard_path = (
    integrated_dashboard_directory
    / 'Utah_WUI_Exposure_Composite_Fire_Hazard_Dashboard.html'
)

integrated_dashboard_summary_path = (
    integrated_dashboard_directory
    / 'Utah_WUI_Exposure_Composite_Fire_Hazard_Dashboard_Summary.csv'
)


# ----------------------------------------------------
# EXPORT THE FINAL DASHBOARD EXACTLY ONCE
# ----------------------------------------------------

print('\n=== EXPORTING FINAL DASHBOARD ===')

wui_exposure_fire_hazard_map.save(
    str(integrated_dashboard_path)
)

if (
    not integrated_dashboard_path.exists()
    or integrated_dashboard_path.stat().st_size <= 0
):
    raise IOError(
        'The integrated dashboard HTML file was not created successfully.'
    )

print(
    f'-> Final dashboard HTML created: '
    f'{integrated_dashboard_path}'
)

print(
    f'-> Final dashboard HTML size: '
    f'{integrated_dashboard_path.stat().st_size / (1024 ** 2):,.2f} MB'
)


# ----------------------------------------------------
# VALIDATE THE ACTUAL RENDERED HTML
# ----------------------------------------------------

final_dashboard_html = integrated_dashboard_path.read_text(
    encoding='utf-8',
)

rendered_tree_layer_control_count = final_dashboard_html.count(
    'L.control.layers.tree('
)

rendered_standard_layer_control_count = final_dashboard_html.count(
    'L.control.layers('
)

rendered_portfolio_legend_count = final_dashboard_html.count(
    'id="phase13-portfolio-legend"'
)

rendered_portfolio_title_count = final_dashboard_html.count(
    'id="phase13-portfolio-title"'
)

rendered_legacy_legend_present = (
    'County Boundaries</b>'
    in final_dashboard_html
)

rendered_contextual_facility_key_present = (
    'id="phase13-facility-key"'
    in final_dashboard_html
    and 'updatePortfolioFacilityKey'
    in final_dashboard_html
)

# Confirm that the reset-view control and its original-extent action were rendered.
rendered_reset_view_control_present = (
    'dashboard-reset-view-control'
    in final_dashboard_html
    and 'DashboardResetViewControl'
    in final_dashboard_html
    and 'dashboardResetViewBounds'
    in final_dashboard_html
)

# Confirm that the integrated legend uses the lower-left position requested for the final map.
rendered_legend_left_position = (
    '.portfolio-map-legend {'
    in final_dashboard_html
    and 'left: 10px;'
    in final_dashboard_html
)

rendered_collapsed_tree_control = (
    '"collapsed": true'
    in final_dashboard_html.lower()
    or 'collapsed: true'
    in final_dashboard_html.lower()
)

rendered_named_tree_control = (
    f'var {final_dashboard_tree_layer_control.get_name()} = '
    'L.control.layers.tree('
    in final_dashboard_html
)

rendered_click_toggle_layer_control = all(
    required_text in final_dashboard_html
    for required_text in [
        'function toggleLayerControl(event)',
        "L.DomEvent.off(layerControlToggle, 'click');",
        "L.DomEvent.off(layerControlToggle, 'keydown');",
        "L.DomEvent.on(layerControlToggle, 'click', toggleLayerControl);",
        "layerTreeControl._map.off(",
        'layerTreeControl.collapse();',
        "'aria-expanded'",
    ]
)

required_rendered_conditions = {
    'one tree layer control': rendered_tree_layer_control_count == 1,
    'no flat layer control': rendered_standard_layer_control_count == 0,
    'one integrated legend': rendered_portfolio_legend_count == 1,
    'one portfolio title': rendered_portfolio_title_count == 1,
    'no legacy county legend': not rendered_legacy_legend_present,
    'contextual facility key': rendered_contextual_facility_key_present,
    'legend in lower-left': rendered_legend_left_position,
    'collapsed layer control': rendered_collapsed_tree_control,
    'named tree-control reference': rendered_named_tree_control,
    'click-toggle layer control': rendered_click_toggle_layer_control,
    'fullscreen control': 'dashboard-fullscreen-control' in final_dashboard_html,
    'reset-view control': rendered_reset_view_control_present,
    'mouse-position control': 'dashboard-mouse-position' in final_dashboard_html,
}

failed_rendered_conditions = [
    condition_name
    for condition_name, condition_valid
    in required_rendered_conditions.items()
    if not condition_valid
]

if failed_rendered_conditions:
    raise ValueError(
        'The rendered dashboard failed the following interface checks:\n'
        f'{failed_rendered_conditions}'
    )

print(f'-> Rendered TreeLayerControl count: {rendered_tree_layer_control_count}')
print(f'-> Rendered standard LayerControl count: {rendered_standard_layer_control_count}')
print(f'-> Rendered integrated legend count: {rendered_portfolio_legend_count}')
print(f'-> Rendered portfolio title count: {rendered_portfolio_title_count}')
print('-> Rendered legacy county legend present: False')
print('-> Contextual critical-facility key present: True')
print('-> Integrated legend position: lower-left')
print('-> Reset-view control present: True')
print('-> Collapsed layer-control configuration present: True')
print('-> Named tree-control JavaScript reference present: True')
print('-> Click-to-toggle layer-control behavior present: True')


# ----------------------------------------------------
# CREATE FINAL DASHBOARD QA SUMMARY
# ----------------------------------------------------

integrated_dashboard_summary = pd.DataFrame(
    [
        {
            'PROJECT': (
                'Utah WUI Exposure and Composite Fire Hazard Grid'
            ),
            'MAP_OBJECT': 'wui_exposure_fire_hazard_map',
            'COMPOSITE_FIRE_HAZARD_INCLUDED': True,
            'PYTHON_STANDARD_LAYER_CONTROL_COUNT': (
                final_standard_layer_control_count
            ),
            'PYTHON_TREE_LAYER_CONTROL_COUNT': (
                final_tree_layer_control_count
            ),
            'RENDERED_STANDARD_LAYER_CONTROL_COUNT': (
                rendered_standard_layer_control_count
            ),
            'RENDERED_TREE_LAYER_CONTROL_COUNT': (
                rendered_tree_layer_control_count
            ),
            'BASEMAP_LAYER_COUNT': final_basemap_tree_count,
            'REFERENCE_FACILITY_LAYER_COUNT': (
                final_reference_layer_count
            ),
            'WILDFIRE_ANALYSIS_LAYER_COUNT': (
                final_wildfire_analysis_layer_count
            ),
            'INTEGRATED_LEGEND_COUNT': final_portfolio_legend_count,
            'LEGACY_LEGEND_COUNT': final_legacy_legend_count,
            'PORTFOLIO_TITLE_COUNT': final_portfolio_title_count,
            'CONTEXTUAL_FACILITY_KEY': True,
            'LEGEND_POSITION': 'lower-left',
            'RESET_VIEW_CONTROL': True,
            'LAYER_CONTROL_DEFAULT_COLLAPSED': True,
            'LAYER_CONTROL_INTERACTION': 'click toggle',
            'HAZARD_SOURCE': str(fire_hazard_dashboard_source_path),
            'HAZARD_DISPLAY_WIDTH': fire_hazard_dashboard_width,
            'HAZARD_DISPLAY_HEIGHT': fire_hazard_dashboard_height,
            'HAZARD_OVERLAY_OPACITY': fire_hazard_dashboard_opacity,
            'FINAL_HTML_PATH': str(integrated_dashboard_path),
            'FINAL_HTML_SIZE_BYTES': integrated_dashboard_path.stat().st_size,
            'VALID': True,
        }
    ]
)

integrated_dashboard_summary.to_csv(
    integrated_dashboard_summary_path,
    index=False,
)

if (
    not integrated_dashboard_summary_path.exists()
    or integrated_dashboard_summary_path.stat().st_size <= 0
):
    raise IOError(
        'The integrated dashboard QA summary was not created successfully.'
    )

print(
    f'-> Dashboard QA summary created: '
    f'{integrated_dashboard_summary_path}'
)


# ----------------------------------------------------
# DISPLAY THE ONE FINAL DASHBOARD
# ----------------------------------------------------

from IPython.display import IFrame, display

final_dashboard_iframe_height = 650

print('\n=== DISPLAYING FINAL DASHBOARD ===')

display(
    IFrame(
        src=str(integrated_dashboard_path),
        width='100%',
        height=final_dashboard_iframe_height,
    )
)

display(integrated_dashboard_summary)


# ----------------------------------------------------
# RECORD AND REPORT FINAL COMPLETION STATUS
# ----------------------------------------------------

fire_hazard_integrated_dashboard_finalized = True

print(f'-> Hazard overlay count: {final_hazard_overlay_count}')
print(f'-> Integrated legend count: {final_portfolio_legend_count}')
print(f'-> Legacy legend count: {final_legacy_legend_count}')
print(f'-> TreeLayerControl count: {final_tree_layer_control_count}')
print(f'-> Custom controls count: {final_custom_controls_count}')
print(f'-> Notebook dashboard height: {final_dashboard_iframe_height} px')
print(f'-> Final dashboard: {integrated_dashboard_path}')
print(f'-> Dashboard summary: {integrated_dashboard_summary_path}')
print(
    f'-> Integrated dashboard finalized: '
    f'{fire_hazard_integrated_dashboard_finalized}'
)

print(
    '\n=== CHRG PHASE 13 INTEGRATED DASHBOARD COMPLETE ==='
)


=== VALIDATING, EXPORTING, AND DISPLAYING FINAL DASHBOARD ===

=== EXPORTING FINAL DASHBOARD ===
-> Final dashboard HTML created: outputs\dashboard\Utah_WUI_Exposure_Composite_Fire_Hazard_Dashboard.html
-> Final dashboard HTML size: 21.97 MB
-> Rendered TreeLayerControl count: 1
-> Rendered standard LayerControl count: 0
-> Rendered integrated legend count: 1
-> Rendered portfolio title count: 1
-> Rendered legacy county legend present: False
-> Contextual critical-facility key present: True
-> Integrated legend position: lower-left
-> Reset-view control present: True
-> Collapsed layer-control configuration present: True
-> Named tree-control JavaScript reference present: True
-> Click-to-toggle layer-control behavior present: True
-> Dashboard QA summary created: outputs\dashboard\Utah_WUI_Exposure_Composite_Fire_Hazard_Dashboard_Summary.csv

=== DISPLAYING FINAL DASHBOARD ===


,PROJECT,MAP_OBJECT,COMPOSITE_FIRE_HAZARD_INCLUDED,PYTHON_STANDARD_LAYER_CONTROL_COUNT,PYTHON_TREE_LAYER_CONTROL_COUNT,RENDERED_STANDARD_LAYER_CONTROL_COUNT,RENDERED_TREE_LAYER_CONTROL_COUNT,BASEMAP_LAYER_COUNT,REFERENCE_FACILITY_LAYER_COUNT,WILDFIRE_ANALYSIS_LAYER_COUNT,...,RESET_VIEW_CONTROL,LAYER_CONTROL_DEFAULT_COLLAPSED,LAYER_CONTROL_INTERACTION,HAZARD_SOURCE,HAZARD_DISPLAY_WIDTH,HAZARD_DISPLAY_HEIGHT,HAZARD_OVERLAY_OPACITY,FINAL_HTML_PATH,FINAL_HTML_SIZE_BYTES,VALID
0,Utah WUI Exposure and Composite Fire Hazard Grid,wui_exposure_fire_hazard_map,True,0,1,0,1,2,8,3,...,True,True,click toggle,C:\Users\adamd\Projects\WUI\data\raw\fire_haza...,1192,1800,0.58,outputs\dashboard\Utah_WUI_Exposure_Composite_...,23033610,True


-> Hazard overlay count: 1
-> Integrated legend count: 1
-> Legacy legend count: 0
-> TreeLayerControl count: 1
-> Custom controls count: 1
-> Notebook dashboard height: 650 px
-> Final dashboard: outputs\dashboard\Utah_WUI_Exposure_Composite_Fire_Hazard_Dashboard.html
-> Dashboard summary: outputs\dashboard\Utah_WUI_Exposure_Composite_Fire_Hazard_Dashboard_Summary.csv
-> Integrated dashboard finalized: True

=== CHRG PHASE 13 INTEGRATED DASHBOARD COMPLETE ===


# 13. Findings

The exposure analysis identified **111 of 1,240 Census Block Groups** with measurable WUI overlap. Area-weighted allocation estimates **19,192 exposed residents**, **6,933 exposed housing units**, and **407.24 km²** of overlapping land.

The critical-facility inventory contains **1,823 facilities** across five categories. Of these, **27 are inside the WUI**, **178 are outside but within one mile**, **1,074 are between one and five miles**, and **544 are more than five miles away**.

The final 30-meter composite grid contains **14,112,977 valid study-area pixels**. Its normalized scores range from **0.019249 to 0.760326**, with a mean of **0.397139**. All six weighted components passed final validation, and no valid hazard pixels occurred outside the study-area mask.

Together, the exposure and hazard layers provide complementary evidence: the exposure analysis identifies people and assets associated with the WUI, while the composite surface characterizes relative landscape conditions.


# 14. Limitations

- Population and housing exposure is estimated by applying each block group's WUI-area proportion to its ACS totals; it does not locate individual residents or structures.
- ACS estimates contain sampling uncertainty, and proportional allocation introduces additional modeling uncertainty.
- The composite grid measures **relative hazard**, not ignition probability, fire intensity, structure loss, evacuation risk, or real-time fire behavior.
- Component weights are transparent analytical assumptions rather than coefficients calibrated against observed losses.
- Input datasets differ in vintage, resolution, collection method, and update schedule.
- Critical-facility results depend on the completeness and positional accuracy of the source inventories.
- The dashboard is a regional screening product and should not replace incident-command information, site assessment, or operational warning systems.


# 15. Next Steps

1. Validate hazard weights and class thresholds against independent fire occurrence, severity, or loss records.
2. Replace area-weighted demographic estimates with parcel, address-point, or building-footprint allocation where appropriate data are available.
3. Add uncertainty and sensitivity analysis for ACS margins of error and hazard-component weights.
4. Incorporate evacuation constraints, response-time accessibility, and social-vulnerability measures.
5. Expand the workflow to additional Utah counties and automate controlled data-version updates.
6. Publish the exported Folium dashboard as a standalone web artifact and connect it to the GitHub project documentation.
